 code has been divide into different files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import json
import pandas as pd
from datetime import datetime


BASE = "/content/drive/MyDrive/FactIndic_PersonB_CleanRun"

folders = [
    "01_raw_data",
    "02_clean_data",
    "03_english_data",
    "04_mixed_data",
    "05_quality_checks",
    "06_real_hinglish",
    "07_claim_labels",
    "08_reports",
    "09_checkpoints"
]

for folder in folders:
    os.makedirs(os.path.join(BASE, folder), exist_ok=True)

print("All folders created!")
print("Project:", BASE)

All folders created!
Project: /content/drive/MyDrive/FactIndic_PersonB_CleanRun


In [ ]:

import os
import json
import pandas as pd
from datetime import datetime

PROGRESS_FILE = os.path.join(
    BASE, "09_checkpoints", "progress.json"
)

def save_progress(task, status, details=None):
    progress = {}

    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "r") as f:
            progress = json.load(f)

    progress[task] = {
        "status": status,
        "details": details or {},
        "updated_at": datetime.now().isoformat()
    }

    temp = PROGRESS_FILE + ".tmp"

    with open(temp, "w") as f:
        json.dump(progress, f, indent=4, ensure_ascii=False)

    os.replace(temp, PROGRESS_FILE)
    print("Checkpoint saved:", task, status)


def save_csv(df, filename):
    path = os.path.join(BASE, filename)
    temp = path + ".tmp"

    df.to_csv(temp, index=False, encoding="utf-8-sig")
    os.replace(temp, path)

    print("File saved:", filename)
    print("Rows:", len(df))


def check_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "r") as f:
            print(json.dumps(json.load(f), indent=2))
    else:
        print("No completed tasks yet.")


save_progress("stage_1", "completed")

print("\nRecovery system ready!")


Checkpoint saved: stage_1 completed

Recovery system ready!


# New section

In [ ]:

import requests
import pandas as pd
import io
import os

url = (
    "https://huggingface.co/datasets/csebuetnlp/xlsum/"
    "resolve/fcbaaeaec2ec660c6164ea74e8e7371b66883670/"
    "hindi/xlsum-test.parquet"
)

print("Downloading XL-Sum Hindi test data...")

response = requests.get(url, timeout=300)
print("HTTP status:", response.status_code)

response.raise_for_status()

df = pd.read_parquet(io.BytesIO(response.content))

print("\nDownload successful!")
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head(3))

# Save original dataset immediately
raw_path = os.path.join(
    BASE, "01_raw_data", "XL_Sum_Hindi_Test.parquet"
)

df.to_parquet(raw_path, index=False)

print("\nOriginal dataset saved to Drive!")
save_progress("task_4_download", "completed",
              {"rows": len(df), "columns": df.columns.tolist()})


HTTP status: 200

Download successful!
Dataset shape: (8847, 5)
Columns: ['id', 'url', 'title', 'summary', 'text']


,id,url,title,summary,text
0,081027_sadhvi_test_pp,https://www.bbc.com/hindi/regionalnews/story/2...,साध्वी का ब्रेन मैपिंग और नॉर्को टेस्ट,मालेगाँव धमाकों के सिलसिले में गिरफ़्तार साध्व...,नासिक की एक अदालत ने महाराष्ट्र पुलिस के आतंकव...
1,070929_pakistan_protest,https://www.bbc.com/hindi/regionalnews/story/2...,"मुशर्रफ़ का पर्चा मंज़ूर, प्रदर्शनों पर लाठी",पाकिस्तान के चुनाव आयोग में शनिवार को राष्ट्रप...,इस बीच चुनाव आयोग ने शनिवार को राष्ट्रपति परवे...
2,international-45650689,https://www.bbc.com/hindi/international-45650689,इतनी जल्दी फ़्रांस में ही चमक क्यों खोने लगे र...,फ़्रांस के राष्ट्रपति इमैनुएल मैक्रों की पहचान...,16 महीने के कार्यकाल पूरे होने के बावजूद राष्ट...



Original dataset saved to Drive!
Checkpoint saved: task_4_download completed


In [ ]:

import re
import os
import pandas as pd

# Select reproducible sample
sample_raw = df.sample(
    n=250,
    random_state=42
).copy()

# Remove duplicate IDs
sample_raw = sample_raw.drop_duplicates(subset=["id"])

# Clean text
def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

clean_df = sample_raw.copy()

for col in ["title", "summary", "text"]:
    clean_df[col] = clean_df[col].apply(clean_text)

# Remove empty records
clean_df = clean_df[
    (clean_df["title"] != "") &
    (clean_df["summary"] != "") &
    (clean_df["text"] != "")
].copy()

# Word counts
clean_df["article_words"] = clean_df["text"].apply(
    lambda x: len(x.split())
)

clean_df["summary_words"] = clean_df["summary"].apply(
    lambda x: len(x.split())
)

# Save sampled raw data
save_csv(
    sample_raw,
    "01_raw_data/XL_Sum_Raw_Sample_250.csv"
)

# Save cleaned data
save_csv(
    clean_df,
    "02_clean_data/XL_Sum_Hindi_Cleaned_250.csv"
)

# Also save a Parquet copy
clean_df.to_parquet(
    os.path.join(BASE, "02_clean_data", "XL_Sum_Hindi_Cleaned_250.parquet"),
    index=False
)

# Save checkpoint
save_progress(
    "task_4",
    "completed",
    {
        "sample_size": len(clean_df),
        "random_state": 42,
        "source_articles": 8847,
        "columns": clean_df.columns.tolist()
    }
)

print("\nTASK 4 COMPLETED")
print("Final shape:", clean_df.shape)
print("Articles:", len(clean_df))
print("Average article words:", round(clean_df["article_words"].mean(), 2))
print("Average summary words:", round(clean_df["summary_words"].mean(), 2))

display(clean_df.head(3))


File saved: 01_raw_data/XL_Sum_Raw_Sample_250.csv
Rows: 250
File saved: 02_clean_data/XL_Sum_Hindi_Cleaned_250.csv
Rows: 250
Checkpoint saved: task_4 completed

TASK 4 COMPLETED
Final shape: (250, 7)
Articles: 250
Average article words: 446.84
Average summary words: 27.24


,id,url,title,summary,text,article_words,summary_words
6710,080419_brchopra_festival,https://www.bbc.com/hindi/entertainment/story/...,बीआर चोपड़ा की यादगार फ़िल्मों का विशेष उत्सव,भारतीय फ़िल्म इतिहास में फ़िल्मकार बीआर चोपड़ा...,शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ ...,294,34
8692,080613_afghan_jail,https://www.bbc.com/hindi/regionalnews/story/2...,तालेबान ने सैकड़ों क़ैदियों को छुड़ाया,अफ़ग़ानिस्तान के कंधार में तालेबान चरमपंथियों ...,कंधार के प्रांतीय सभा के अध्यक्ष और राष्ट्रपति...,289,17
6871,130912_guinnness_record_bulgarian_swim_sr,https://www.bbc.com/hindi/international/2013/0...,रिकॉर्ड की ख़ातिर हाथ-पैर बांध कर की तैराकी,गिनीज़ वर्ल्ड रिकॉर्ड में अपना नाम दर्ज कराने ...,ख़बरों के मुताबिक़ जेन पेत्कोव को मैसेडोनिया क...,200,24


# New section

In [ ]:

import torch
import pandas as pd
import os

print("Python environment ready")
print("PyTorch:", torch.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: Not available (CPU mode)")

# Recover dataset from Drive if necessary
clean_path = os.path.join(
    BASE, "02_clean_data",
    "XL_Sum_Hindi_Cleaned_250.csv"
)

if "clean_df" not in globals():
    clean_df = pd.read_csv(clean_path)

print("\nDataset shape:", clean_df.shape)
print("Articles:", len(clean_df))
print("Columns:", clean_df.columns.tolist())

assert len(clean_df) == 250, "Expected exactly 250 articles"

print("\nReady for English translation!")


Python environment ready
PyTorch: 2.14.1+cpu
GPU: Not available (CPU mode)

Dataset shape: (250, 7)
Articles: 250
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words']

Ready for English translation!


In [ ]:

import importlib.util

packages = [
    "torch",
    "transformers",
    "IndicTransToolkit",
    "sentencepiece"
]

for package in packages:
    available = importlib.util.find_spec(package) is not None
    print(package, ":", "Available" if available else "Missing")


torch : Available
transformers : Available
IndicTransToolkit : Missing
sentencepiece : Available


In [ ]:

%pip install -q indictranstoolkit


In [ ]:
from importlib.metadata import version, PackageNotFoundError

for package in ["transformers", "torch", "indictranstoolkit", "tokenizers"]:
    try:
        print(package, ":", version(package))
    except PackageNotFoundError:
        print(package, ": Not installed")

transformers : 5.22.1
torch : 2.14.1+cpu
indictranstoolkit : 1.1.1
tokenizers : 0.23.3


In [ ]:
%pip install -q --no-deps transformers==4.57.6

In [ ]:
%pip install -q --no-deps tokenizers==0.22.1

In [ ]:
%pip install -q --no-deps huggingface-hub==0.36.0

In [ ]:
from importlib.metadata import version

for p in ["transformers", "tokenizers", "huggingface-hub"]:
    print(p, version(p))

from IndicTransToolkit import IndicProcessor
ip = IndicProcessor(inference=True)
print("SUCCESS: Toolkit ready!")

transformers 4.57.6
tokenizers 0.22.1
huggingface-hub 0.36.0
SUCCESS: Toolkit ready!


In [ ]:
from IndicTransToolkit import IndicProcessor

ip = IndicProcessor(inference=True)
print("SUCCESS: Toolkit ready!")

SUCCESS: Toolkit ready!


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/FactIndic_PersonB_CleanRun"

print("Project folder exists:", os.path.exists(BASE))
print("Available folders:", os.listdir(BASE))

file_path = os.path.join(
    BASE, "02_clean_data", "XL_Sum_Hindi_Cleaned_250.csv"
)

print("Clean dataset exists:", os.path.exists(file_path))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folder exists: True
Available folders: ['01_raw_data', '02_clean_data', '03_english_data', '04_mixed_data', '05_quality_checks', '06_real_hinglish', '07_claim_labels', '08_reports', '09_checkpoints']
Clean dataset exists: True


In [ ]:
import os, json
import pandas as pd
from datetime import datetime

PROGRESS_FILE = os.path.join(BASE, "09_checkpoints", "progress.json")

def save_progress(task, status, details=None):
    progress = {}
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "r") as f:
            progress = json.load(f)

    progress[task] = {
        "status": status,
        "details": details or {},
        "updated_at": datetime.now().isoformat()
    }

    temp = PROGRESS_FILE + ".tmp"
    with open(temp, "w") as f:
        json.dump(progress, f, indent=4, ensure_ascii=False)
    os.replace(temp, PROGRESS_FILE)
    print("Checkpoint saved:", task)

def save_csv(df, filename):
    path = os.path.join(BASE, filename)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    temp = path + ".tmp"
    df.to_csv(temp, index=False, encoding="utf-8-sig")
    os.replace(temp, path)
    print("Saved:", filename, "| Rows:", len(df))

def check_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "r") as f:
            print(json.dumps(json.load(f), indent=2))
    else:
        print("No progress file found yet.")

check_progress()

{
  "stage_1": {
    "status": "completed",
    "details": {},
    "updated_at": "2026-11-11T11:21:20.329786"
  },
  "task_4_download": {
    "status": "completed",
    "details": {
      "rows": 8847,
      "columns": [
        "id",
        "url",
        "title",
        "summary",
        "text"
      ]
    },
    "updated_at": "2026-11-11T11:22:33.618184"
  },
  "task_4": {
    "status": "completed",
    "details": {
      "sample_size": 250,
      "random_state": 42,
      "source_articles": 8847,
      "columns": [
        "id",
        "url",
        "title",
        "summary",
        "text",
        "article_words",
        "summary_words"
      ]
    },
    "updated_at": "2026-11-11T11:23:21.015103"
  }
}


In [ ]:
import pandas as pd
import os

file_path = os.path.join(
    BASE, "02_clean_data", "XL_Sum_Hindi_Cleaned_250.csv"
)

df = pd.read_csv(file_path)

print("Dataset loaded:", df.shape)
print("Columns:", df.columns.tolist())
print("Articles:", len(df))
print("Missing values:", df.isnull().sum().sum())

Dataset loaded: (250, 7)
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words']
Articles: 250
Missing values: 0


In [ ]:
from IndicTransToolkit import IndicProcessor

ip = IndicProcessor(inference=True)

print("Toolkit ready!")
print("Dataset rows:", len(df))

Toolkit ready!
Dataset rows: 250


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_NAME = "Helsinki-NLP/opus-mt-hi-en"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Translation model loaded!")
print("Device:", device)

config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/1.06M [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/813k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.06M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


... (3 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  304MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Translation model loaded!
Device: cpu


In [ ]:
test_text = "भारत सरकार ने आज नई योजना शुरू की।"

inputs = tokenizer(
    test_text,
    return_tensors="pt",
    truncation=True,
    max_length=480
).to(device)

with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=100)

print("Hindi:", test_text)
print("English:", tokenizer.decode(output[0], skip_special_tokens=True))

Hindi: भारत सरकार ने आज नई योजना शुरू की।
English: The India government started a new plan today.


In [ ]:
# Test translation on the first complete article
sample_hindi = str(df.iloc[0]["text"])

inputs = tokenizer(
    sample_hindi,
    return_tensors="pt",
    truncation=True,
    max_length=480
).to(device)

with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=300)

sample_english = tokenizer.decode(
    output[0], skip_special_tokens=True
)

print("Hindi word count:", len(sample_hindi.split()))
print("\nEnglish translation preview:\n")
print(sample_english[:1500])

Hindi word count: 294

English translation preview:

On Friday evening, after the film was started, the scales of justice were shown. The little thing, the new rounds and time of the celebration, can't be included in the celebration for bad health.


In [ ]:
sample_hindi = str(df.iloc[0]["text"])

inputs = tokenizer(
    sample_hindi,
    return_tensors="pt",
    truncation=True,
    max_length=480
).to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=300
    )

sample_english = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print("Hindi word count:", len(sample_hindi.split()))
print("\nEnglish translation preview:\n")
print(sample_english[:1500])

Hindi word count: 294

English translation preview:

On Friday evening, after the film was started, the scales of justice were shown. The little thing, the new rounds and time of the celebration, can't be included in the celebration for bad health.


In [ ]:
print("Model maximum input length:", tokenizer.model_max_length)
print("Article characters:", len(sample_hindi))
print("Article words:", len(sample_hindi.split()))

Model maximum input length: 512
Article characters: 1565
Article words: 294


In [ ]:
import re

sentences = re.split(r'(?<=[।!?])\s+', sample_hindi.strip())
sentences = [s.strip() for s in sentences if s.strip()]

print("Total sentences:", len(sentences))
print("\nFirst 5 sentences:")
for s in sentences[:5]:
    print("-", s)

Total sentences: 1

First 5 sentences:
- शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी." बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है. गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में शामिल है. हालांकि बीआर चोपड़ा की ज़्यादातर फ़िल्में किसी न किसी स

In [ ]:
chunks = []
current_chunk = ""

for sentence in sentences:
    candidate = (current_chunk + " " + sentence).strip()

    token_count = len(
        tokenizer(candidate, add_special_tokens=True)["input_ids"]
    )

    if token_count <= 450:
        current_chunk = candidate
    else:
        if current_chunk:
            chunks.append(current_chunk)
        current_chunk = sentence

if current_chunk:
    chunks.append(current_chunk)

print("Total chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    tokens = len(tokenizer(chunk)["input_ids"])
    print(f"Chunk {i+1}: {tokens} tokens")

Total chunks: 1
Chunk 1: 380 tokens


In [ ]:
translated_chunks = []

for i, chunk in enumerate(chunks):
    inputs = tokenizer(
        chunk,
        return_tensors="pt",
        truncation=True,
        max_length=480
    ).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=300,
            num_beams=2
        )

    translated = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    translated_chunks.append(translated)

    print(f"Translated chunk {i+1}/{len(chunks)}")

sample_english = " ".join(translated_chunks)

print("\nFULL TRANSLATION:\n")
print(sample_english)

Translated chunk 1/1

FULL TRANSLATION:

On Friday evening, after the film was started, the scales of justice were shown. Small things, new rounds and time, can be found to see the classic films. Bad health reason for not joining the ceremony.


In [ ]:
print("Chunks:", len(chunks))
print("Translated chunks:", len(translated_chunks))
print("Empty translations:", sum(not x.strip() for x in translated_chunks))
print("English words:", len(sample_english.split()))

Chunks: 1
Translated chunks: 1
Empty translations: 0
English words: 36


In [ ]:
print("Detected sentences:", len(sentences))
print("Chunks:", len(chunks))
print("Original Hindi words:", len(sample_hindi.split()))
print("Translated English words:", len(sample_english.split()))

print("\nOriginal article preview:\n")
print(sample_hindi[:1200])

print("\nTranslation:\n")
print(sample_english)

Detected sentences: 1
Chunks: 1
Original Hindi words: 294
Translated English words: 36

Original article preview:

शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी." बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है. गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की 

In [ ]:
words = sample_hindi.split()

word_chunks = [
    " ".join(words[i:i+70])
    for i in range(0, len(words), 70)
]

print("Total chunks:", len(word_chunks))

for i, chunk in enumerate(word_chunks):
    print(
        f"Chunk {i+1}: {len(chunk.split())} words | "
        f"{len(tokenizer(chunk)['input_ids'])} tokens"
    )

Total chunks: 5
Chunk 1: 70 words | 98 tokens
Chunk 2: 70 words | 87 tokens
Chunk 3: 70 words | 88 tokens
Chunk 4: 70 words | 88 tokens
Chunk 5: 14 words | 23 tokens


In [ ]:
translated_chunks = []

for i, chunk in enumerate(word_chunks):
    inputs = tokenizer(
        chunk,
        return_tensors="pt",
        truncation=True,
        max_length=480
    ).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=150,
            num_beams=2
        )

    translated = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    translated_chunks.append(translated)
    print(f"Chunk {i+1}/5 completed")

sample_english = " ".join(translated_chunks)

print("\nEnglish word count:", len(sample_english.split()))
print("\nTranslation:\n", sample_english)

Chunk 1/5 completed
Chunk 2/5 completed
Chunk 3/5 completed
Chunk 4/5 completed
Chunk 5/5 completed

English word count: 181

Translation:
 On Friday evening, the scales of justice were shown in the Polish film. The little thing, the new rounds, and time during the celebration, will be seen to see the classical films. Bad health because of BRC's own function, BRCP's brother and famous filmer, he said, "My brother made his business at such a time. When people were apprehensive about the commercial success of such films because they were few, the stories of BRangangange's life were different from them. BR'Courourin raised many subjects like widowry in her movies, such as prostitution. The issue of sexy prostitution is to question society. Leading, wets, law, and marriage are included in their film list of films. However, most movies of BR's chlorophynism were based on some social point, but these used films also showed a remarkable deal on the board. Nested and made a film. Bran Brangn, 

In [ ]:
print("Total chunks:", len(word_chunks))
print("Translated chunks:", len(translated_chunks))
print("Empty chunks:", sum(not x.strip() for x in translated_chunks))
print("English words:", len(sample_english.split()))

Total chunks: 5
Translated chunks: 5
Empty chunks: 0
English words: 181


In [ ]:
for i, (hi, en) in enumerate(zip(word_chunks, translated_chunks)):
    print(f"\n--- CHUNK {i+1} ---")
    print("HINDI:", hi[:250])
    print("ENGLISH:", en[:400])


--- CHUNK 1 ---
HINDI: शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर च
ENGLISH: On Friday evening, the scales of justice were shown in the Polish film. The little thing, the new rounds, and time during the celebration, will be seen to see the classical films. Bad health because of BRC's own function, BRCP's brother and famous filmer, he said, "My brother made his business at such a time.

--- CHUNK 2 ---
HINDI: जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी." बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति
ENGLISH: When people were apprehensive about the commercial success of such films because they were few, the stories of BRangangan

In [ ]:
def translate_hindi(text):
    if pd.isna(text) or not str(text).strip():
        return ""

    words = str(text).split()

    chunks = [
        " ".join(words[i:i+70])
        for i in range(0, len(words), 70)
    ]

    results = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=480
        ).to(device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=150,
                num_beams=2
            )

        results.append(
            tokenizer.decode(output[0], skip_special_tokens=True)
        )

    return " ".join(results)

print("Translation function ready!")

Translation function ready!


In [ ]:
import os
import pandas as pd

partial_path = os.path.join(
    BASE, "03_english_data", "XL_Sum_English_250_partial.csv"
)

os.makedirs(os.path.dirname(partial_path), exist_ok=True)

# Resume from saved file if it exists
if os.path.exists(partial_path):
    df_english = pd.read_csv(partial_path)
    print("Resuming saved translations...")
else:
    df_english = df.copy()
    for col in ["english_title", "english_text", "english_summary"]:
        df_english[col] = ""

# Translate the next 5 unfinished articles
completed = 0

for idx in df_english.index:
    if completed >= 5:
        break

    row = df_english.loc[idx]

    if all(
        pd.notna(row.get(col)) and str(row[col]).strip()
        for col in ["english_title", "english_text", "english_summary"]
    ):
        continue

    print(f"Translating article {idx + 1}/250...")

    df_english.at[idx, "english_title"] = translate_hindi(row["title"])
    df_english.at[idx, "english_text"] = translate_hindi(row["text"])
    df_english.at[idx, "english_summary"] = translate_hindi(row["summary"])

    # Save immediately after each article
    df_english.to_csv(partial_path, index=False, encoding="utf-8-sig")

    completed += 1
    print("Saved article", idx + 1)

print("\nBatch completed:", completed)
print("Articles with all 3 translations:",
      df_english[["english_title", "english_text", "english_summary"]]
      .notna().all(axis=1).sum())
print("Saved file:", partial_path)

Translating article 1/250...
Saved article 1
Translating article 2/250...
Saved article 2
Translating article 3/250...
Saved article 3
Translating article 4/250...
Saved article 4
Translating article 5/250...
Saved article 5

Batch completed: 5
Articles with all 3 translations: 250
Saved file: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/03_english_data/XL_Sum_English_250_partial.csv


In [ ]:
translation_cols = [
    "english_title",
    "english_text",
    "english_summary"
]

complete_mask = df_english[translation_cols].apply(
    lambda col: col.fillna("").astype(str).str.strip().ne("")
).all(axis=1)

print("Fully translated articles:", complete_mask.sum())
print("Remaining articles:", 250 - complete_mask.sum())
print("Saved file exists:", os.path.exists(partial_path))

Fully translated articles: 5
Remaining articles: 245
Saved file exists: True


In [ ]:
completed = 0

for idx in df_english.index:
    if completed >= 5:
        break

    row = df_english.loc[idx]

    if all(
        pd.notna(row[col]) and str(row[col]).strip()
        for col in ["english_title", "english_text", "english_summary"]
    ):
        continue

    print(f"Translating article {idx + 1}/250...")

    df_english.at[idx, "english_title"] = translate_hindi(row["title"])
    df_english.at[idx, "english_text"] = translate_hindi(row["text"])
    df_english.at[idx, "english_summary"] = translate_hindi(row["summary"])

    df_english.to_csv(partial_path, index=False, encoding="utf-8-sig")

    completed += 1
    print("Saved article", idx + 1)

print("New batch completed:", completed)

Translating article 6/250...
Saved article 6
Translating article 7/250...
Saved article 7
Translating article 8/250...
Saved article 8
Translating article 9/250...
Saved article 9
Translating article 10/250...
Saved article 10
New batch completed: 5


In [ ]:
df_english = pd.read_csv(partial_path)

translation_cols = [
    "english_title",
    "english_text",
    "english_summary"
]

complete_mask = df_english[translation_cols].apply(
    lambda col: col.fillna("").astype(str).str.strip().ne("")
).all(axis=1)

print("Fully translated:", complete_mask.sum(), "/ 250")
print("Remaining:", 250 - complete_mask.sum())
print("Saved file verified:", os.path.exists(partial_path))

Fully translated: 10 / 250
Remaining: 240
Saved file verified: True


In [ ]:
completed = 0

for idx in df_english.index:
    if completed >= 5:
        break

    row = df_english.loc[idx]

    if all(
        pd.notna(row[col]) and str(row[col]).strip()
        for col in ["english_title", "english_text", "english_summary"]
    ):
        continue

    print(f"Translating article {idx + 1}/250...")

    df_english.at[idx, "english_title"] = translate_hindi(row["title"])
    df_english.at[idx, "english_text"] = translate_hindi(row["text"])
    df_english.at[idx, "english_summary"] = translate_hindi(row["summary"])

    df_english.to_csv(
        partial_path,
        index=False,
        encoding="utf-8-sig"
    )

    completed += 1
    print(f"Saved article {idx + 1}")

print(f"\nNew batch completed: {completed}")

translation_cols = ["english_title", "english_text", "english_summary"]

complete_mask = df_english[translation_cols].apply(
    lambda col: col.fillna("").astype(str).str.strip().ne("")
).all(axis=1)

print("Total completed:", complete_mask.sum(), "/ 250")
print("Remaining:", 250 - complete_mask.sum())

Translating article 11/250...
Saved article 11
Translating article 12/250...
Saved article 12
Translating article 13/250...
Saved article 13
Translating article 14/250...
Saved article 14
Translating article 15/250...
Saved article 15

New batch completed: 5
Total completed: 15 / 250
Remaining: 235


In [ ]:
completed = 0

for idx in df_english.index:
    if completed >= 5:
        break

    row = df_english.loc[idx]

    if all(
        pd.notna(row[col]) and str(row[col]).strip()
        for col in ["english_title", "english_text", "english_summary"]
    ):
        continue

    print(f"Translating article {idx + 1}/250...")

    df_english.at[idx, "english_title"] = translate_hindi(row["title"])
    df_english.at[idx, "english_text"] = translate_hindi(row["text"])
    df_english.at[idx, "english_summary"] = translate_hindi(row["summary"])

    df_english.to_csv(
        partial_path,
        index=False,
        encoding="utf-8-sig"
    )

    completed += 1
    print(f"Saved article {idx + 1}")

print(f"\nNew batch completed: {completed}")

translation_cols = ["english_title", "english_text", "english_summary"]

complete_mask = df_english[translation_cols].apply(
    lambda col: col.fillna("").astype(str).str.strip().ne("")
).all(axis=1)

print("Total completed:", complete_mask.sum(), "/ 250")
print("Remaining:", 250 - complete_mask.sum())

Translating article 16/250...
Saved article 16
Translating article 17/250...
Saved article 17
Translating article 18/250...
Saved article 18
Translating article 19/250...
Saved article 19
Translating article 20/250...
Saved article 20

New batch completed: 5
Total completed: 20 / 250
Remaining: 230


In [ ]:
completed = 0
BATCH_SIZE = 25

for idx in df_english.index:
    if completed >= BATCH_SIZE:
        break

    row = df_english.loc[idx]

    if all(
        pd.notna(row[col]) and str(row[col]).strip()
        for col in ["english_title", "english_text", "english_summary"]
    ):
        continue

    print(f"Translating article {idx + 1}/250...")

    df_english.at[idx, "english_title"] = translate_hindi(row["title"])
    df_english.at[idx, "english_text"] = translate_hindi(row["text"])
    df_english.at[idx, "english_summary"] = translate_hindi(row["summary"])

    # Save after every article
    df_english.to_csv(
        partial_path,
        index=False,
        encoding="utf-8-sig"
    )

    completed += 1
    print(f"Saved | Batch progress: {completed}/{BATCH_SIZE}")

translation_cols = ["english_title", "english_text", "english_summary"]

complete_mask = df_english[translation_cols].apply(
    lambda col: col.fillna("").astype(str).str.strip().ne("")
).all(axis=1)

print("\nBatch completed:", completed)
print("Total translated:", complete_mask.sum(), "/ 250")
print("Remaining:", 250 - complete_mask.sum())

Translating article 21/250...
Saved | Batch progress: 1/25
Translating article 22/250...
Saved | Batch progress: 2/25
Translating article 23/250...
Saved | Batch progress: 3/25
Translating article 24/250...
Saved | Batch progress: 4/25
Translating article 25/250...
Saved | Batch progress: 5/25
Translating article 26/250...
Saved | Batch progress: 6/25
Translating article 27/250...
Saved | Batch progress: 7/25
Translating article 28/250...
Saved | Batch progress: 8/25
Translating article 29/250...
Saved | Batch progress: 9/25
Translating article 30/250...
Saved | Batch progress: 10/25
Translating article 31/250...
Saved | Batch progress: 11/25
Translating article 32/250...
Saved | Batch progress: 12/25
Translating article 33/250...
Saved | Batch progress: 13/25
Translating article 34/250...
Saved | Batch progress: 14/25
Translating article 35/250...
Saved | Batch progress: 15/25
Translating article 36/250...
Saved | Batch progress: 16/25
Translating article 37/250...
Saved | Batch progr

In [ ]:
completed = 0
BATCH_SIZE = 50

for idx in df_english.index:
    if completed >= BATCH_SIZE:
        break

    row = df_english.loc[idx]

    if all(
        pd.notna(row[col]) and str(row[col]).strip()
        for col in ["english_title", "english_text", "english_summary"]
    ):
        continue

    print(f"Translating article {idx + 1}/250...")

    df_english.at[idx, "english_title"] = translate_hindi(row["title"])
    df_english.at[idx, "english_text"] = translate_hindi(row["text"])
    df_english.at[idx, "english_summary"] = translate_hindi(row["summary"])

    df_english.to_csv(partial_path, index=False, encoding="utf-8-sig")

    completed += 1
    print(f"Saved | Batch: {completed}/50")

translation_cols = ["english_title", "english_text", "english_summary"]

complete_mask = df_english[translation_cols].apply(
    lambda col: col.fillna("").astype(str).str.strip().ne("")
).all(axis=1)

print("\nBatch completed:", completed)
print("Total translated:", complete_mask.sum(), "/ 250")
print("Remaining:", 250 - complete_mask.sum())

Translating article 46/250...
Saved | Batch: 1/50
Translating article 47/250...
Saved | Batch: 2/50
Translating article 48/250...
Saved | Batch: 3/50
Translating article 49/250...
Saved | Batch: 4/50
Translating article 50/250...
Saved | Batch: 5/50
Translating article 51/250...
Saved | Batch: 6/50
Translating article 52/250...
Saved | Batch: 7/50
Translating article 53/250...
Saved | Batch: 8/50
Translating article 54/250...
Saved | Batch: 9/50
Translating article 55/250...
Saved | Batch: 10/50
Translating article 56/250...
Saved | Batch: 11/50
Translating article 57/250...
Saved | Batch: 12/50
Translating article 58/250...
Saved | Batch: 13/50
Translating article 59/250...
Saved | Batch: 14/50
Translating article 60/250...
Saved | Batch: 15/50
Translating article 61/250...
Saved | Batch: 16/50
Translating article 62/250...
Saved | Batch: 17/50
Translating article 63/250...
Saved | Batch: 18/50
Translating article 64/250...
Saved | Batch: 19/50
Translating article 65/250...
Saved | Ba

In [ ]:
completed = 0
BATCH_SIZE = 50

for idx in df_english.index:
    if completed >= BATCH_SIZE:
        break

    row = df_english.loc[idx]

    if all(
        pd.notna(row[col]) and str(row[col]).strip()
        for col in ["english_title", "english_text", "english_summary"]
    ):
        continue

    print(f"Translating article {idx + 1}/250...")

    df_english.at[idx, "english_title"] = translate_hindi(row["title"])
    df_english.at[idx, "english_text"] = translate_hindi(row["text"])
    df_english.at[idx, "english_summary"] = translate_hindi(row["summary"])

    df_english.to_csv(partial_path, index=False, encoding="utf-8-sig")

    completed += 1
    print(f"Saved | Batch: {completed}/50")

translation_cols = ["english_title", "english_text", "english_summary"]

complete_mask = df_english[translation_cols].apply(
    lambda col: col.fillna("").astype(str).str.strip().ne("")
).all(axis=1)

print("\nBatch completed:", completed)
print("Total translated:", complete_mask.sum(), "/ 250")
print("Remaining:", 250 - complete_mask.sum())

Translating article 96/250...
Saved | Batch: 1/50
Translating article 97/250...
Saved | Batch: 2/50
Translating article 98/250...
Saved | Batch: 3/50
Translating article 99/250...
Saved | Batch: 4/50
Translating article 100/250...
Saved | Batch: 5/50
Translating article 101/250...
Saved | Batch: 6/50
Translating article 102/250...
Saved | Batch: 7/50
Translating article 103/250...
Saved | Batch: 8/50
Translating article 104/250...
Saved | Batch: 9/50
Translating article 105/250...
Saved | Batch: 10/50
Translating article 106/250...
Saved | Batch: 11/50
Translating article 107/250...
Saved | Batch: 12/50
Translating article 108/250...
Saved | Batch: 13/50
Translating article 109/250...
Saved | Batch: 14/50
Translating article 110/250...
Saved | Batch: 15/50
Translating article 111/250...
Saved | Batch: 16/50
Translating article 112/250...
Saved | Batch: 17/50
Translating article 113/250...
Saved | Batch: 18/50
Translating article 114/250...
Saved | Batch: 19/50
Translating article 115/2

In [ ]:
df_english = pd.read_csv(partial_path)

translation_cols = ["english_title", "english_text", "english_summary"]

complete_mask = df_english[translation_cols].apply(
    lambda col: col.fillna("").astype(str).str.strip().ne("")
).all(axis=1)

print("Completed:", complete_mask.sum(), "/ 250")
print("Remaining:", 250 - complete_mask.sum())

Completed: 145 / 250
Remaining: 105


In [ ]:
completed = 0
BATCH_SIZE = 50

for idx in df_english.index:
    if completed >= BATCH_SIZE:
        break

    row = df_english.loc[idx]

    if all(
        pd.notna(row[col]) and str(row[col]).strip()
        for col in ["english_title", "english_text", "english_summary"]
    ):
        continue

    print(f"Translating article {idx + 1}/250...")

    df_english.at[idx, "english_title"] = translate_hindi(row["title"])
    df_english.at[idx, "english_text"] = translate_hindi(row["text"])
    df_english.at[idx, "english_summary"] = translate_hindi(row["summary"])

    df_english.to_csv(partial_path, index=False, encoding="utf-8-sig")

    completed += 1
    print(f"Saved | Batch: {completed}/50")

translation_cols = ["english_title", "english_text", "english_summary"]

complete_mask = df_english[translation_cols].apply(
    lambda col: col.fillna("").astype(str).str.strip().ne("")
).all(axis=1)

print("\nBatch completed:", completed)
print("Total translated:", complete_mask.sum(), "/ 250")
print("Remaining:", 250 - complete_mask.sum())

Translating article 146/250...
Saved | Batch: 1/50
Translating article 147/250...
Saved | Batch: 2/50
Translating article 148/250...
Saved | Batch: 3/50
Translating article 149/250...
Saved | Batch: 4/50
Translating article 150/250...
Saved | Batch: 5/50
Translating article 151/250...
Saved | Batch: 6/50
Translating article 152/250...
Saved | Batch: 7/50
Translating article 153/250...
Saved | Batch: 8/50
Translating article 154/250...
Saved | Batch: 9/50
Translating article 155/250...
Saved | Batch: 10/50
Translating article 156/250...
Saved | Batch: 11/50
Translating article 157/250...
Saved | Batch: 12/50
Translating article 158/250...
Saved | Batch: 13/50
Translating article 159/250...
Saved | Batch: 14/50
Translating article 160/250...
Saved | Batch: 15/50
Translating article 161/250...
Saved | Batch: 16/50
Translating article 162/250...
Saved | Batch: 17/50
Translating article 163/250...
Saved | Batch: 18/50
Translating article 164/250...
Saved | Batch: 19/50
Translating article 1

In [ ]:
completed = 0
BATCH_SIZE = 55

for idx in df_english.index:
    if completed >= BATCH_SIZE:
        break

    row = df_english.loc[idx]

    if all(
        pd.notna(row[col]) and str(row[col]).strip()
        for col in ["english_title", "english_text", "english_summary"]
    ):
        continue

    print(f"Translating article {idx + 1}/250...")

    df_english.at[idx, "english_title"] = translate_hindi(row["title"])
    df_english.at[idx, "english_text"] = translate_hindi(row["text"])
    df_english.at[idx, "english_summary"] = translate_hindi(row["summary"])

    df_english.to_csv(partial_path, index=False, encoding="utf-8-sig")

    completed += 1
    print(f"Saved | Batch: {completed}/{BATCH_SIZE}")

translation_cols = ["english_title", "english_text", "english_summary"]

complete_mask = df_english[translation_cols].apply(
    lambda col: col.fillna("").astype(str).str.strip().ne("")
).all(axis=1)

print("\nNewly completed:", completed)
print("Total translated:", complete_mask.sum(), "/ 250")
print("Remaining:", 250 - complete_mask.sum())

Translating article 196/250...
Saved | Batch: 1/55
Translating article 197/250...
Saved | Batch: 2/55
Translating article 198/250...
Saved | Batch: 3/55
Translating article 199/250...
Saved | Batch: 4/55
Translating article 200/250...
Saved | Batch: 5/55
Translating article 201/250...
Saved | Batch: 6/55
Translating article 202/250...
Saved | Batch: 7/55
Translating article 203/250...
Saved | Batch: 8/55
Translating article 204/250...
Saved | Batch: 9/55
Translating article 205/250...
Saved | Batch: 10/55
Translating article 206/250...
Saved | Batch: 11/55
Translating article 207/250...
Saved | Batch: 12/55
Translating article 208/250...
Saved | Batch: 13/55
Translating article 209/250...
Saved | Batch: 14/55
Translating article 210/250...
Saved | Batch: 15/55
Translating article 211/250...
Saved | Batch: 16/55
Translating article 212/250...
Saved | Batch: 17/55
Translating article 213/250...
Saved | Batch: 18/55
Translating article 214/250...
Saved | Batch: 19/55
Translating article 2

In [ ]:
import os
import pandas as pd

# Load latest saved translations
df_english = pd.read_csv(partial_path)

translation_cols = [
    "english_title",
    "english_text",
    "english_summary"
]

complete_mask = df_english[translation_cols].apply(
    lambda col: col.fillna("").astype(str).str.strip().ne("")
).all(axis=1)

completed_count = int(complete_mask.sum())

print("Fully translated:", completed_count, "/ 250")

if completed_count == 250:
    final_path = os.path.join(
        BASE, "03_english_data", "XL_Sum_English_250.csv"
    )

    df_english.to_csv(
        final_path,
        index=False,
        encoding="utf-8-sig"
    )

    save_progress(
        "english_translation",
        "completed",
        {
            "articles": 250,
            "model": "Helsinki-NLP/opus-mt-hi-en",
            "file": final_path
        }
    )

    print("\nFinal English dataset saved!")
    print(final_path)
else:
    print("Some translations are still missing.")
    print("Remaining:", 250 - completed_count)

Fully translated: 250 / 250
Checkpoint saved: english_translation

Final English dataset saved!
/content/drive/MyDrive/FactIndic_PersonB_CleanRun/03_english_data/XL_Sum_English_250.csv


# New section

In [ ]:
import os
import pandas as pd

BASE = "/content/drive/MyDrive/FactIndic_PersonB_CleanRun"

final_path = os.path.join(
    BASE, "03_english_data", "XL_Sum_English_250.csv"
)

partial_path = os.path.join(
    BASE, "03_english_data", "XL_Sum_English_250_partial.csv"
)

if os.path.exists(final_path):
    df_english = pd.read_csv(final_path)
elif os.path.exists(partial_path):
    df_english = pd.read_csv(partial_path)
else:
    raise FileNotFoundError("English dataset not found in Google Drive.")

print("Dataset loaded:", df_english.shape)
print("Columns:", df_english.columns.tolist())
print("Articles:", len(df_english))

Dataset loaded: (250, 10)
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words', 'english_title', 'english_text', 'english_summary']
Articles: 250


In [ ]:
translation_cols = [
    "english_title",
    "english_text",
    "english_summary"
]

complete_mask = df_english[translation_cols].apply(
    lambda col: col.fillna("").astype(str).str.strip().ne("")
).all(axis=1)

print("Fully translated:", complete_mask.sum(), "/ 250")
print("Missing translations:", 250 - complete_mask.sum())

print("\nSample English article:")
print(df_english.loc[0, "english_text"][:500])

Fully translated: 250 / 250
Missing translations: 0

Sample English article:
On Friday evening, the scales of justice were shown in the Polish film. The little thing, the new rounds, and time during the celebration, will be seen to see the classical films. Bad health because of BRC's own function, BRCP's brother and famous filmer, he said, "My brother made his business at such a time. When people were apprehensive about the commercial success of such films because they were few, the stories of BRangangange's life were different from them. BR'Courourin raised many subject


In [ ]:
import os

print("Mixed data folder:")
mixed_folder = os.path.join(BASE, "04_mixed_data")

if os.path.exists(mixed_folder):
    files = os.listdir(mixed_folder)
    print(files if files else "Folder is empty")
else:
    print("Folder not found")

print("\nAvailable phrase dictionary:")
print("final_candidate_phrases:", "final_candidate_phrases" in globals())
print("safe_phrases:", "safe_phrases" in globals())

print("\nEnglish dataset columns:")
print(df_english.columns.tolist())

Mixed data folder:
Folder is empty

Available phrase dictionary:
final_candidate_phrases: False
safe_phrases: False

English dataset columns:
['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words', 'english_title', 'english_text', 'english_summary']


In [ ]:
import os
import json

phrase_dictionary = {
    "भारत सरकार": "Indian government",
    "केंद्र सरकार": "central government",
    "राज्य सरकार": "state government",
    "प्रधानमंत्री": "Prime Minister",
    "मुख्यमंत्री": "Chief Minister",
    "राष्ट्रपति": "President",
    "मंत्री": "minister",
    "अधिकारी": "official",
    "पुलिस": "police",
    "अदालत": "court",
    "न्यायालय": "court",
    "अस्पताल": "hospital",
    "डॉक्टर": "doctor",
    "शिक्षा": "education",
    "स्कूल": "school",
    "विश्वविद्यालय": "university",
    "छात्र": "student",
    "किसान": "farmer",
    "महिला": "woman",
    "बच्चों": "children",
    "परिवार": "family",
    "समाज": "society",
    "देश": "country",
    "शहर": "city",
    "गांव": "village",
    "चुनाव": "election",
    "सरकार": "government",
    "योजना": "scheme",
    "विकास": "development",
    "सड़क": "road",
    "पानी": "water",
    "बिजली": "electricity",
    "स्वास्थ्य": "health",
    "अर्थव्यवस्था": "economy",
    "व्यापार": "business",
    "कंपनी": "company",
    "रिपोर्ट": "report",
    "जानकारी": "information",
    "घटना": "incident",
    "मामले": "cases",
    "समस्या": "problem"
}

dictionary_path = os.path.join(
    BASE, "04_mixed_data", "hindi_english_phrase_dictionary.json"
)

os.makedirs(os.path.dirname(dictionary_path), exist_ok=True)

with open(dictionary_path, "w", encoding="utf-8") as f:
    json.dump(phrase_dictionary, f, ensure_ascii=False, indent=4)

print("Phrase dictionary created!")
print("Total phrase pairs:", len(phrase_dictionary))
print("Saved to:", dictionary_path)

Phrase dictionary created!
Total phrase pairs: 41
Saved to: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/hindi_english_phrase_dictionary.json


In [ ]:
import re
import random

random.seed(42)

# Longer Hindi phrases are replaced first
sorted_phrases = sorted(
    phrase_dictionary.items(),
    key=lambda x: len(x[0]),
    reverse=True
)

def create_mixed_text(hindi_text, target_percentage):
    if pd.isna(hindi_text) or not str(hindi_text).strip():
        return ""

    words = str(hindi_text).split()
    target_words = round(len(words) * target_percentage / 100)

    candidates = []

    for hindi, english in sorted_phrases:
        for match in re.finditer(re.escape(hindi), str(hindi_text)):
            candidates.append((match.start(), match.end(), hindi, english))

    random.shuffle(candidates)
    candidates.sort(key=lambda x: x[1] - x[0], reverse=True)

    selected = []
    occupied = []

    for start, end, hindi, english in candidates:
        if len(selected) >= target_words:
            break

        if any(start < e and end > s for s, e in occupied):
            continue

        selected.append((start, end, hindi, english))
        occupied.append((start, end))

    result = str(hindi_text)

    for start, end, hindi, english in sorted(selected, reverse=True):
        result = result[:start] + english + result[end:]

    return result

print("Mixed-text generator ready!")

Mixed-text generator ready!


In [ ]:
import os
import pandas as pd

mixed_path = os.path.join(
    BASE, "04_mixed_data", "FactIndic_Mixed_250_partial.csv"
)

# Create or resume dataset
if os.path.exists(mixed_path):
    df_mixed = pd.read_csv(mixed_path)
    print("Resuming existing mixed dataset:", len(df_mixed))
else:
    df_mixed = df_english.copy()
    for col in [
        "mixed_10_article",
        "mixed_30_article",
        "mixed_50_article"
    ]:
        df_mixed[col] = ""
    df_mixed.to_csv(mixed_path, index=False, encoding="utf-8-sig")

completed = 0

for idx in df_mixed.index:
    if completed >= 250:
        break

    if all(
        pd.notna(df_mixed.at[idx, col])
        and str(df_mixed.at[idx, col]).strip()
        for col in [
            "mixed_10_article",
            "mixed_30_article",
            "mixed_50_article"
        ]
    ):
        continue

    hindi_text = df_mixed.at[idx, "text"]

    df_mixed.at[idx, "mixed_10_article"] = create_mixed_text(
        hindi_text, 10
    )
    df_mixed.at[idx, "mixed_30_article"] = create_mixed_text(
        hindi_text, 30
    )
    df_mixed.at[idx, "mixed_50_article"] = create_mixed_text(
        hindi_text, 50
    )

    # Save after every article
    df_mixed.to_csv(
        mixed_path,
        index=False,
        encoding="utf-8-sig"
    )

    completed += 1

    if completed % 10 == 0:
        print("Processed:", completed)

print("\nMixed articles generated!")
print("Saved file:", mixed_path)

Processed: 10
Processed: 20
Processed: 30
Processed: 40
Processed: 50
Processed: 60
Processed: 70
Processed: 80
Processed: 90
Processed: 100
Processed: 110
Processed: 120
Processed: 130
Processed: 140
Processed: 150
Processed: 160
Processed: 170
Processed: 180
Processed: 190
Processed: 200
Processed: 210
Processed: 220
Processed: 230
Processed: 240
Processed: 250

Mixed articles generated!
Saved file: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/FactIndic_Mixed_250_partial.csv


In [ ]:
df_mixed = pd.read_csv(mixed_path)

mixed_cols = [
    "mixed_10_article",
    "mixed_30_article",
    "mixed_50_article"
]

complete_mask = df_mixed[mixed_cols].apply(
    lambda col: col.fillna("").astype(str).str.strip().ne("")
).all(axis=1)

print("10% versions:", df_mixed["mixed_10_article"].fillna("").str.strip().ne("").sum())
print("30% versions:", df_mixed["mixed_30_article"].fillna("").str.strip().ne("").sum())
print("50% versions:", df_mixed["mixed_50_article"].fillna("").str.strip().ne("").sum())
print("Complete articles:", complete_mask.sum(), "/ 250")

10% versions: 250
30% versions: 250
50% versions: 250
Complete articles: 250 / 250


In [ ]:
import os

final_mixed_path = os.path.join(
    BASE, "04_mixed_data", "FactIndic_Mixed_250.csv"
)

df_mixed.to_csv(
    final_mixed_path,
    index=False,
    encoding="utf-8-sig"
)

save_progress(
    "mixed_articles",
    "completed",
    {
        "articles": 250,
        "versions": ["10%", "30%", "50%"],
        "file": final_mixed_path
    }
)

print("Final mixed dataset saved!")
print("Rows:", len(df_mixed))
print("Location:", final_mixed_path)

Checkpoint saved: mixed_articles
Final mixed dataset saved!
Rows: 250
Location: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/FactIndic_Mixed_250.csv


In [ ]:
import re
import pandas as pd

def english_percentage(text):
    if pd.isna(text) or not str(text).strip():
        return 0

    words = str(text).split()
    english_words = sum(
        1 for word in words
        if re.search(r"[A-Za-z]", word)
    )

    return round((english_words / len(words)) * 100, 2)


for col in [
    "mixed_10_article",
    "mixed_30_article",
    "mixed_50_article"
]:
    df_mixed[col + "_english_pct"] = (
        df_mixed[col].apply(english_percentage)
    )

    print(
        col,
        "| Average English percentage:",
        df_mixed[col + "_english_pct"].mean().round(2),
        "%"
    )

# Save updated dataset
df_mixed.to_csv(
    final_mixed_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nCMI calculation saved!")

mixed_10_article | Average English percentage: 2.79 %
mixed_30_article | Average English percentage: 2.79 %
mixed_50_article | Average English percentage: 2.79 %

CMI calculation saved!


In [ ]:
import re

def count_dictionary_matches(text):
    if pd.isna(text):
        return 0

    return sum(
        len(re.findall(re.escape(hindi), str(text)))
        for hindi in phrase_dictionary
    )

df_mixed["dictionary_matches"] = (
    df_mixed["text"].apply(count_dictionary_matches)
)

print("Articles with dictionary matches:",
      (df_mixed["dictionary_matches"] > 0).sum(), "/ 250")

print("Average matches per article:",
      round(df_mixed["dictionary_matches"].mean(), 2))

print("\nArticles with the most matches:")
print(
    df_mixed[["id", "dictionary_matches"]]
    .sort_values("dictionary_matches", ascending=False)
    .head(10).to_string(index=False)
)

Articles with dictionary matches: 237 / 250
Average matches per article: 11.76

Articles with the most matches:
                                 id  dictionary_matches
                     india-48662333                  47
        041026_bestbakery_questions                  44
                     india-48250857                  37
             international-47105506                  37
          081126_aamne_pachauri_alk                  36
             international-42264998                  36
             international-49451047                  35
150222_parliament_budget_session_pk                  34
                     india-46856839                  34
             international-52239381                  33


In [ ]:
import re

def measure_mix(text):
    hindi = re.findall(r'[\u0900-\u097F]+', str(text))
    english = re.findall(r'\b[A-Za-z]+\b', str(text))
    total = len(hindi) + len(english)
    return round(100 * len(english) / total, 2) if total else 0


# Prefer longer phrases to avoid overlapping matches
pattern = re.compile(
    "|".join(
        re.escape(p)
        for p in sorted(phrase_dictionary, key=len, reverse=True)
    )
)

def maximum_mix(text):
    if pd.isna(text):
        return 0

    return measure_mix(
        pattern.sub(
            lambda m: phrase_dictionary[m.group()],
            str(text)
        )
    )


df_mixed["maximum_possible_pct"] = (
    df_mixed["text"].apply(maximum_mix)
)

print("Average maximum possible English percentage:",
      round(df_mixed["maximum_possible_pct"].mean(), 2), "%")

print("\nMaximum mixing coverage:")
print("Articles reaching 10%:",
      (df_mixed["maximum_possible_pct"] >= 10).sum())
print("Articles reaching 30%:",
      (df_mixed["maximum_possible_pct"] >= 30).sum())
print("Articles reaching 50%:",
      (df_mixed["maximum_possible_pct"] >= 50).sum())

Average maximum possible English percentage: 2.51 %

Maximum mixing coverage:
Articles reaching 10%: 0
Articles reaching 30%: 0
Articles reaching 50%: 0


In [ ]:
import os

for folder in ["01_raw_data", "02_clean_data", "03_english_data",
               "04_mixed_data", "05_quality_checks"]:
    path = os.path.join(BASE, folder)
    print(f"\n{folder}:")
    if os.path.exists(path):
        for filename in os.listdir(path):
            print(" -", filename)


01_raw_data:
 - XL_Sum_Hindi_Test.parquet
 - XL_Sum_Raw_Sample_250.csv

02_clean_data:
 - XL_Sum_Hindi_Cleaned_250.csv
 - XL_Sum_Hindi_Cleaned_250.parquet

03_english_data:
 - XL_Sum_English_250_partial.csv
 - XL_Sum_English_250.csv

04_mixed_data:
 - hindi_english_phrase_dictionary.json
 - FactIndic_Mixed_250_partial.csv
 - FactIndic_Mixed_250.csv

05_quality_checks:


In [ ]:
import os

print("Searching for previous alignment files...\n")

keywords = ["align", "sentence", "pair", "bilingual", "parallel"]

for root, dirs, files in os.walk(BASE):
    for filename in files:
        if any(word in filename.lower() for word in keywords):
            print(os.path.join(root, filename))

print("\nChecking saved progress...")
check_progress()

Searching for previous alignment files...


Checking saved progress...
{
  "stage_1": {
    "status": "completed",
    "details": {},
    "updated_at": "2026-11-11T11:21:20.329786"
  },
  "task_4_download": {
    "status": "completed",
    "details": {
      "rows": 8847,
      "columns": [
        "id",
        "url",
        "title",
        "summary",
        "text"
      ]
    },
    "updated_at": "2026-11-11T11:22:33.618184"
  },
  "task_4": {
    "status": "completed",
    "details": {
      "sample_size": 250,
      "random_state": 42,
      "source_articles": 8847,
      "columns": [
        "id",
        "url",
        "title",
        "summary",
        "text",
        "article_words",
        "summary_words"
      ]
    },
    "updated_at": "2026-11-11T11:23:21.015103"
  },
  "english_translation": {
    "status": "completed",
    "details": {
      "articles": 250,
      "model": "Helsinki-NLP/opus-mt-hi-en",
      "file": "/content/drive/MyDrive/FactIndic_PersonB_CleanRun/

In [ ]:
import os

search_root = "/content/drive/MyDrive"
keywords = ["alignment", "aligned", "sentence_pairs", "sentence_pairs_250"]

found = []

for root, dirs, files in os.walk(search_root):
    # Skip unnecessary hidden/system folders
    dirs[:] = [d for d in dirs if not d.startswith(".")]

    for filename in files:
        if any(k in filename.lower() for k in keywords):
            found.append(os.path.join(root, filename))

print("Files found:", len(found))

for path in found:
    print(path)

Files found: 0


In [ ]:
!pip -q install nltk

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

from nltk.tokenize import sent_tokenize

print("Sentence alignment preparation ready!")

Sentence alignment preparation ready!


In [ ]:
from nltk.tokenize import sent_tokenize

sample = df_english.iloc[0]

hindi_sentences = sent_tokenize(
    str(sample["text"]), language="english"
)

english_sentences = sent_tokenize(
    str(sample["english_text"]), language="english"
)

print("Article ID:", sample["id"])
print("Hindi sentences:", len(hindi_sentences))
print("English sentences:", len(english_sentences))

print("\nHindi examples:")
for s in hindi_sentences[:3]:
    print("-", s)

print("\nEnglish examples:")
for s in english_sentences[:3]:
    print("-", s)

Article ID: 080419_brchopra_festival
Hindi sentences: 14
English sentences: 13

Hindi examples:
- शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई.
- उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी.
- खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया.

English examples:
- On Friday evening, the scales of justice were shown in the Polish film.
- The little thing, the new rounds, and time during the celebration, will be seen to see the classical films.
- Bad health because of BRC's own function, BRCP's brother and famous filmer, he said, "My brother made his business at such a time.


In [ ]:
import os
import pandas as pd
from nltk.tokenize import sent_tokenize

alignment_path = os.path.join(
    BASE, "05_quality_checks", "FactIndic_Sentence_Alignment.csv"
)

os.makedirs(os.path.dirname(alignment_path), exist_ok=True)

if os.path.exists(alignment_path):
    aligned_df = pd.read_csv(alignment_path)
    completed_ids = set(aligned_df["sample_id"].astype(str))
else:
    aligned_df = pd.DataFrame(columns=[
        "sample_id",
        "sentence_number",
        "hindi_sentence",
        "english_sentence",
        "alignment_method"
    ])
    completed_ids = set()

for _, row in df_english.iterrows():
    sample_id = str(row["id"])

    if sample_id in completed_ids:
        continue

    hi = sent_tokenize(str(row["text"]), language="english")
    en = sent_tokenize(str(row["english_text"]), language="english")

    count = max(len(hi), len(en))

    for i in range(count):
        hi_index = min(int(i * len(hi) / count), len(hi) - 1)
        en_index = min(int(i * len(en) / count), len(en) - 1)

        aligned_df.loc[len(aligned_df)] = [
            sample_id,
            i + 1,
            hi[hi_index],
            en[en_index],
            "approximate_proportional"
        ]

    aligned_df.to_csv(
        alignment_path,
        index=False,
        encoding="utf-8-sig"
    )

    completed_ids.add(sample_id)

print("Articles aligned:", len(completed_ids), "/ 250")
print("Sentence pairs:", len(aligned_df))
print("Saved:", alignment_path)

Articles aligned: 250 / 250
Sentence pairs: 5580
Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/FactIndic_Sentence_Alignment.csv


In [ ]:
# Inspect 5 sample articles
sample_ids = aligned_df["sample_id"].drop_duplicates().head(5)

for sid in sample_ids:
    sample = aligned_df[aligned_df["sample_id"] == sid]

    print("\nARTICLE:", sid)
    print("Sentence pairs:", len(sample))

    for _, r in sample.head(3).iterrows():
        print("\nHI:", r["hindi_sentence"])
        print("EN:", r["english_sentence"])


ARTICLE: 080419_brchopra_festival
Sentence pairs: 14

HI: शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई.
EN: On Friday evening, the scales of justice were shown in the Polish film.

HI: उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी.
EN: On Friday evening, the scales of justice were shown in the Polish film.

HI: खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया.
EN: The little thing, the new rounds, and time during the celebration, will be seen to see the classical films.

ARTICLE: 080613_afghan_jail
Sentence pairs: 12

HI: कंधार के प्रांतीय सभा के अध्यक्ष और राष्ट्रपति हामिद करज़ई के भाई वली करज़ई ने बीबीसी को बताया कि उस समय जेल में एक हज़ार क़ैदी थे और सभी निकल भागे हैं.
EN: The president of the local assembly of the Barar and President Hamdigy Kadig told his wife that at that time there were a thousand prisoners in prison and a

In [ ]:
import os
import json

alignment_warning = {
    "file": alignment_path,
    "articles_processed": 250,
    "sentence_pairs": 5580,
    "method": "approximate_proportional",
    "status": "NOT_VERIFIED",
    "warning": "Do not use as verified sentence alignment."
}

warning_path = os.path.join(
    BASE, "05_quality_checks",
    "alignment_quality_warning.json"
)

with open(warning_path, "w", encoding="utf-8") as f:
    json.dump(alignment_warning, f, indent=4)

save_progress(
    "sentence_alignment",
    "experimental_not_verified",
    alignment_warning
)

print("Experimental alignment preserved.")
print("Warning saved:", warning_path)

Checkpoint saved: sentence_alignment
Experimental alignment preserved.
Warning saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/alignment_quality_warning.json


In [ ]:
!pip -q install sentence-transformers

print("Sentence Transformers installed!")

Sentence Transformers installed!


In [ ]:
from sentence_transformers import SentenceTransformer

SIMILARITY_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"

similarity_model = SentenceTransformer(SIMILARITY_MODEL)

print("Similarity model loaded!")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Similarity model loaded!


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import sent_tokenize

sample = df_english.iloc[0]

hi_sentences = sent_tokenize(str(sample["text"]))
en_sentences = sent_tokenize(str(sample["english_text"]))

hi_embeddings = similarity_model.encode(
    hi_sentences,
    convert_to_numpy=True,
    normalize_embeddings=True
)

en_embeddings = similarity_model.encode(
    en_sentences,
    convert_to_numpy=True,
    normalize_embeddings=True
)

similarities = cosine_similarity(
    hi_embeddings, en_embeddings
)

print("ARTICLE:", sample["id"])
print("Hindi sentences:", len(hi_sentences))
print("English sentences:", len(en_sentences))

print("\nBest similarity matches:\n")

for i, hi in enumerate(hi_sentences):
    best_idx = int(np.argmax(similarities[i]))
    score = similarities[i][best_idx]

    print(f"\nSimilarity: {score:.3f}")
    print("HI:", hi)
    print("EN:", en_sentences[best_idx])

ARTICLE: 080419_brchopra_festival
Hindi sentences: 14
English sentences: 13

Best similarity matches:


Similarity: 0.448
HI: शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई.
EN: On Friday evening, the scales of justice were shown in the Polish film.

Similarity: 0.788
HI: उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी.
EN: The little thing, the new rounds, and time during the celebration, will be seen to see the classical films.

Similarity: 0.334
HI: खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया.
EN: Bad health because of BRC's own function, BRCP's brother and famous filmer, he said, "My brother made his business at such a time.

Similarity: 0.628
HI: उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस स

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

GPU available: False
Running on CPU


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_NAME = "facebook/nllb-200-distilled-600M"

tokenizer_new = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    src_lang="hin_Deu"
)

model_new = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = "cpu"
model_new = model_new.to(device)

print("Alternative model loaded successfully!")
print("Device:", device)

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Alternative model loaded successfully!
Device: cpu


In [ ]:
import os
import pandas as pd

BASE = "/content/drive/MyDrive/FactIndic_PersonB_CleanRun"

data_path = os.path.join(
    BASE,
    "02_clean_data",
    "XL_Sum_Hindi_Cleaned_250.csv"
)

df_250 = pd.read_csv(data_path)

print("Dataset loaded:", df_250.shape)
print("Columns:", df_250.columns.tolist())
print("First article available:", bool(str(df_250.iloc[0]["text"]).strip()))

Dataset loaded: (250, 7)
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words']
First article available: True


In [ ]:
# Get the first original Hindi article
hindi_article = df_250.iloc[0]["text"]

# Translate in manageable chunks
def translate_nllb(text, chunk_size=60):
    words = str(text).split()
    chunks = [
        " ".join(words[i:i + chunk_size])
        for i in range(0, len(words), chunk_size)
    ]

    translated_chunks = []

    for chunk in chunks:
        inputs = tokenizer_new(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=512
        )

        with torch.no_grad():
            output = model_new.generate(
                **inputs,
                forced_bos_token_id=tokenizer_new.convert_tokens_to_ids("eng_Latn"),
                max_new_tokens=180,
                num_beams=2
            )

        translated_chunks.append(
            tokenizer_new.decode(output[0], skip_special_tokens=True)
        )

    return " ".join(translated_chunks)

nllb_translation = translate_nllb(hindi_article)

print("HINDI ORIGINAL:\n")
print(hindi_article[:1500])

print("\n\nNLLB ENGLISH TRANSLATION:\n")
print(nllb_translation[:2000])

HINDI ORIGINAL:

शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी." बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है. गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में शामिल है. हालांकि बीआर चोपड़ा की ज़्यादातर फ़िल्में किसी न किसी सामाजिक मुद्दे पर आधारित 

In [ ]:
import re

sentences = re.split(r'(?<=[।.!?])\s*', hindi_article)
sentences = [s.strip() for s in sentences if s.strip()]

def translate_nllb_sentence(text):
    inputs = tokenizer_new(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        output = model_new.generate(
            **inputs,
            forced_bos_token_id=tokenizer_new.convert_tokens_to_ids("eng_Latn"),
            max_new_tokens=150,
            num_beams=3
        )

    return tokenizer_new.decode(
        output[0],
        skip_special_tokens=True
    )

for i, sentence in enumerate(sentences[:5], 1):
    print(f"\nHindi {i}: {sentence}")
    print(f"English {i}: {translate_nllb_sentence(sentence)}")


Hindi 1: शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई.
English 1: The festival, which began on Friday evening, featured the film Insaf Ka Tarazu.

Hindi 2: उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी.
English 2: During the festival, you'll see classic movies like little things, a new era, and time.

Hindi 3: खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.
English 3: Due to poor health, B.R. Chopra himself could not attend the ceremony.

Hindi 4: फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया.
English 4: The film festival was inaugurated by B. R. Chopra's brother and famous filmmaker Yash Chopra.

Hindi 5: उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी.
English 5:

In [ ]:
import time

start = time.time()

nllb_sentences = [
    translate_nllb_sentence(sentence)
    for sentence in sentences
]

elapsed = time.time() - start

print("Hindi sentences:", len(sentences))
print("Translated sentences:", len(nllb_sentences))
print("Time taken:", round(elapsed / 60, 2), "minutes")

print("\nFULL TRANSLATION:\n")
print(" ".join(nllb_sentences))

Hindi sentences: 15
Translated sentences: 15
Time taken: 3.04 minutes

FULL TRANSLATION:

The festival, which began on Friday evening, featured the film Insaf Ka Tarazu. During the festival, you'll see classic movies like little things, a new era, and time. Due to poor health, B.R. Chopra himself could not attend the ceremony. The film festival was inaugurated by B. R. Chopra's brother and famous filmmaker Yash Chopra. He said, "My brother made films on his own terms - even at a time when people were concerned about the commercial success of such films because these films were a bit of a departure, the kind of stories that were common at the time, the stories of B. R. Chopra Ji were unrelated to them. B.R. Chopra has brought up many themes in his films such as widow remarriage, prostitution. His films reflect the spirit of the Nehru era in a new era, while in 1958 Sadhana appears to be questioning society on the issue of prostitution. Movies such as 'Gumrah', 'Imraj', 'Kanuun' and 'Nik

In [ ]:
comparison = pd.DataFrame({
    "hindi_sentence": sentences,
    "nllb_translation": nllb_sentences
})

comparison_path = os.path.join(
    BASE, "05_quality_checks",
    "NLLB_First_Article_Comparison.csv"
)

comparison.to_csv(
    comparison_path,
    index=False,
    encoding="utf-8-sig"
)

print("Comparison saved:", comparison_path)
print("Rows:", len(comparison))

Comparison saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/NLLB_First_Article_Comparison.csv
Rows: 15


In [ ]:
# Protect important names and film titles before translation
glossary = {
    "इंसाफ़ का तराज़ू": "Insaf Ka Tarazu",
    "छोटी सी बात": "Chhoti Si Baat",
    "नया दौर": "Naya Daur",
    "वक़्त": "Waqt",
    "साधना": "Sadhana",
    "गुमराह": "Gumrah",
    "हमराज़": "Humraaz",
    "क़ानून": "Kanoon",
    "निकाह": "Nikaah",
    "बीआर चोपड़ा": "B. R. Chopra",
    "यश चोपड़ा": "Yash Chopra",
    "रवि चोपड़ा": "Ravi Chopra",
    "राज बब्बर": "Raj Babbar",
    "बसु चटर्जी": "Basu Chatterjee",
    "बॉबी बेदी": "Bobby Bedi"
}

def protect_names(text):
    for hindi, english in sorted(
        glossary.items(), key=lambda x: len(x[0]), reverse=True
    ):
        text = text.replace(hindi, english)
    return text

protected_sentences = [
    protect_names(sentence) for sentence in sentences
]

protected_translations = [
    translate_nllb_sentence(sentence)
    for sentence in protected_sentences
]

print("PROTECTED TRANSLATION:\n")
print(" ".join(protected_translations))

PROTECTED TRANSLATION:

The film Insaf Ka Tarazu was screened at the festival, which began on Friday evening. During the festival, you will see classic films like Chhoti Si Baat, Naya Daur and Waqt. B R. Chopra himself could not attend the ceremony due to poor health. The film festival was inaugurated by B. R. Chopra's brother and renowned filmmaker Yash Chopra. He said, "My brother made films on his own terms - even at a time when people were concerned about the commercial success of such films because these films were a bit of a departure, the kind of stories that were common at the time, the stories of B. R. Chopra Ji were unrelated to them. "B. R. Chopra raised many themes in his films, such as widow remarriage, prostitution. His film Naya Daur reflects the ideals of the Nehru period, while Sadhana in 1958 seems to question society on the issue of prostitution. Movies such as Gumrah, Humraaz, Kanoon and Nikaah are included in the list of films he made. Although most of B. R. Chopra

In [ ]:
for hindi, english in glossary.items():
    if english in " ".join(protected_translations):
        print("OK:", english)
    else:
        print("MISSING:", english)

OK: Insaf Ka Tarazu
OK: Chhoti Si Baat
OK: Naya Daur
OK: Waqt
OK: Sadhana
OK: Gumrah
OK: Humraaz
OK: Kanoon
OK: Nikaah
OK: B. R. Chopra
OK: Yash Chopra
OK: Ravi Chopra
OK: Raj Babbar
OK: Basu Chatterjee
OK: Bobby Bedi


In [ ]:
comparison["protected_translation"] = protected_translations

comparison_path = os.path.join(
    BASE,
    "05_quality_checks",
    "NLLB_First_Article_Comparison.csv"
)

comparison.to_csv(
    comparison_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved successfully!")
print("Rows:", len(comparison))
print("File:", comparison_path)

Saved successfully!
Rows: 15
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/NLLB_First_Article_Comparison.csv


In [ ]:
import re
import os
import pandas as pd
import time

test_results = []

output_path = os.path.join(
    BASE, "05_quality_checks",
    "NLLB_5_Article_Test.csv"
)

for idx in range(1, 5):
    row = df_250.iloc[idx]
    hindi_text = str(row["text"])

    # Split into sentences
    article_sentences = [
        s.strip() for s in re.split(
            r'(?<=[।.!?])\s*', hindi_text
        ) if s.strip()
    ]

    # Protect names and film titles
    protected = [
        protect_names(s) for s in article_sentences
    ]

    start = time.time()

    translated = [
        translate_nllb_sentence(s) for s in protected
    ]

    result = {
        "article_number": idx + 1,
        "id": row["id"],
        "hindi_article": hindi_text,
        "english_article": " ".join(translated),
        "sentence_count": len(article_sentences),
        "time_minutes": round((time.time() - start) / 60, 2)
    }

    test_results.append(result)

    # Save after every article
    pd.DataFrame(test_results).to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"Article {idx + 1}/5 saved | "
        f"Sentences: {len(article_sentences)} | "
        f"Time: {result['time_minutes']} minutes"
    )

print("\n5-article test completed.")
print("Saved:", output_path)

Article 2/5 saved | Sentences: 12 | Time: 2.31 minutes
Article 3/5 saved | Sentences: 12 | Time: 1.68 minutes
Article 4/5 saved | Sentences: 14 | Time: 2.1 minutes
Article 5/5 saved | Sentences: 24 | Time: 3.35 minutes

5-article test completed.
Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/NLLB_5_Article_Test.csv


In [ ]:
test_df = pd.read_csv(
    os.path.join(
        BASE,
        "05_quality_checks",
        "NLLB_5_Article_Test.csv"
    )
)

for _, row in test_df.iterrows():
    print("\nARTICLE:", row["article_number"])
    print("\nHINDI:\n", row["hindi_article"][:700])
    print("\nENGLISH:\n", row["english_article"][:900])
    print("\n" + "="*70)


ARTICLE: 2

HINDI:
 कंधार के प्रांतीय सभा के अध्यक्ष और राष्ट्रपति हामिद करज़ई के भाई वली करज़ई ने बीबीसी को बताया कि उस समय जेल में एक हज़ार क़ैदी थे और सभी निकल भागे हैं. माना जाता है कि इन क़ैदियों में से 350 तालेबान थे. इस हमले में कई सुरक्षाकर्मियों की जानें गई हैं. अधिकारियों के अनुसार जेल के मुख्य द्वार को उड़ाने के लिए एक लॉरी बम से विस्फोट किया गया और फिर 40 तालेबान जेल के भीतर घुस आए. कंधार शहर में आपातकाल की घोषणा कर दी गई है. पुलिस और सैनिक सड़कों पर गश्त कर रहे हैं और लोगों से कहा गया है कि वे अपने घरों से बाहर न निकलें. जेल प्रमुख अब्दुल क़बीर से समाचार एजेंसी एपी को इस बात की पुष्टि तो की है कि सैकड़ों क़ैदी निकल भागे हैं लेकिन उनका कहना है कि कुछ क़ैदी जेल में ही रुक गए. प्रत्यक्षदर्शियों का कहन

ENGLISH:
 Wali Karzai, chairman of the provincial assembly of Kandahar and brother of President Hamid Karzai, told the BBC that there were a thousand prisoners in the prison at the time and all had escaped. It is believed that 350 of these prisoners were Taliban. Several secur

In [ ]:
glossary.update({
    "एक था टाइगर": "Ek Tha Tiger",
    "सलमान खान": "Salman Khan",
    "कटरीना कैफ": "Katrina Kaif",
    "दबंग": "Dabangg",
    "बॉडीगार्ड": "Bodyguard"
})

print("Updated glossary:")
for hindi, english in glossary.items():
    print(hindi, "->", english)

Updated glossary:
इंसाफ़ का तराज़ू -> Insaf Ka Tarazu
छोटी सी बात -> Chhoti Si Baat
नया दौर -> Naya Daur
वक़्त -> Waqt
साधना -> Sadhana
गुमराह -> Gumrah
हमराज़ -> Humraaz
क़ानून -> Kanoon
निकाह -> Nikaah
बीआर चोपड़ा -> B. R. Chopra
यश चोपड़ा -> Yash Chopra
रवि चोपड़ा -> Ravi Chopra
राज बब्बर -> Raj Babbar
बसु चटर्जी -> Basu Chatterjee
बॉबी बेदी -> Bobby Bedi
एक था टाइगर -> Ek Tha Tiger
सलमान खान -> Salman Khan
कटरीना कैफ -> Katrina Kaif
दबंग -> Dabangg
बॉडीगार्ड -> Bodyguard


In [ ]:
# Get Article 5 again
article_5 = df_250.iloc[4]["text"]

# Split into sentences
article_5_sentences = [
    s.strip()
    for s in re.split(r'(?<=[।.!?])\s*', article_5)
    if s.strip()
]

# Protect titles and names
protected_article_5 = [
    protect_names(s) for s in article_5_sentences
]

# Translate
article_5_translation = [
    translate_nllb_sentence(s)
    for s in protected_article_5
]

print("HINDI:\n")
print(article_5)

print("\n\nPROTECTED ENGLISH:\n")
print(" ".join(article_5_translation))

HINDI:

फिल्म एक था टाइगर में सलमान खान और कटरीना कैफ मुख्य भूमिका निभा रहे हैं. सलमान की आने वाली फिल्म है- एक था टाइगर. उसी के प्रमोशन पर पहुंचे सलमान ने कहा, "मैं बहुत खुश हूं कि मेरी फिल्में अच्छा कर रही हैं. लेकिन साथ ही ये स्थिति खतरनाक है, क्योंकि हमें मालूम नहीं कि अब अगली फिल्म में क्या करना है." सलमान ने कहा, "मेरी फिल्मों में मेरा किरदार लार्जर दैन लाइफ इमेज लिए होता है, जिसे कोई मार नहीं सकता. लेकिन लगातार ऐसे रोल करना बोरिंग हो जाएगा. लेकिन समस्या ये है कि अगर हम अपनी अगली फिल्मों को उस स्तर तक नहीं ले जाते, तो लोग कहेंगे कि इससे अच्छी तो पहले की फिल्म थी. यानी समस्या ये है कि हमें पता ही नहीं कि करना क्या है." पिछले तीन-चार सालों से सलमान का सितारा बुलंदी पर चल रहा है. उन्होंने एक के बाद एक वॉन्टेड, दबंग, रेडी और बॉडीगार्ड जैसी ब्लॉकबस्टर फिल्में दी हैं. ये सभी एक्शन फिल्में थीं. दबंग, रेडी और बॉडीगार्ड ने तो 100 करोड़ रुपए से ज्यादा का व्यवसाय किया है. भविष्य में रोमांटिक रोल सिर्फ एक्शन फिल्में ही क्यों. इसके जवाब में सलमान कहते हैं कि उन्हें लगातार एक्शन फिल्मों के ही 

In [ ]:
glossary.update({
    "सूरज बड़जात्या": "Sooraj Barjatya",
    "मैंने प्यार किया": "Maine Pyar Kiya",
    "हम आपके हैं कौन": "Hum Aapke Hain Koun",
    "कबीर खान": "Kabir Khan",
    "वॉन्टेड": "Wanted",
    "रेडी": "Ready",
    "यश चोपड़ा": "Yash Chopra"
})

print("Total glossary entries:", len(glossary))

Total glossary entries: 26


In [ ]:
import json
import os

glossary_path = os.path.join(
    BASE,
    "04_mixed_data",
    "hindi_english_phrase_dictionary.json"
)

os.makedirs(os.path.dirname(glossary_path), exist_ok=True)

with open(glossary_path, "w", encoding="utf-8") as f:
    json.dump(glossary, f, ensure_ascii=False, indent=4)

print("Glossary saved!")
print("Total entries:", len(glossary))
print(glossary_path)

Glossary saved!
Total entries: 26
/content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/hindi_english_phrase_dictionary.json


In [ ]:
def translate_article_nllb(text):
    article_sentences = [
        s.strip()
        for s in re.split(r'(?<=[।.!?])\s*', str(text))
        if s.strip()
    ]

    protected = [
        protect_names(s) for s in article_sentences
    ]

    translated = [
        translate_nllb_sentence(s) for s in protected
    ]

    return " ".join(translated)


# Test on Article 5
test_translation = translate_article_nllb(
    df_250.iloc[4]["text"]
)

print(test_translation)

Salman Khan and Katrina Kaif are playing the lead roles in Ek Tha Tiger. Salman's upcoming film is Ek Tha Tiger. Salman, who reached the promotion for the same, said, "I am very happy that my films are doing well. But at the same time, this situation is dangerous, because we do not know what to do next in the next film. Salman said, "In my films, my character takes a large daily life image, which no one can kill. But rolling around like this all the time will be boring. But the problem is that if we don't take our next films to that level, people will say it was better than the previous one. The problem is that we don't know what to do. "For the last three or four years, Salman's star has been rising. He has given blockbuster films like Wanted, Dabangg, Ready and Bodyguard one after another. These were all action movies. Dabangg, Ready and Bodyguard have made a business of over Rs 100 crore. Why in the future romantic roles are only action movies. In response, Salman says that he is co

In [ ]:
expected_names = [
    "Salman Khan",
    "Katrina Kaif",
    "Ek Tha Tiger",
    "Sooraj Barjatya",
    "Maine Pyar Kiya",
    "Hum Aapke Hain Koun",
    "Kabir Khan",
    "Yash Chopra"
]

for name in expected_names:
    status = "PASS" if name in test_translation else "CHECK"
    print(status, "-", name)

PASS - Salman Khan
PASS - Katrina Kaif
PASS - Ek Tha Tiger
PASS - Sooraj Barjatya
PASS - Maine Pyar Kiya
PASS - Hum Aapke Hain Koun
PASS - Kabir Khan
PASS - Yash Chopra


In [ ]:
test_translation_path = os.path.join(
    BASE,
    "05_quality_checks",
    "NLLB_Article_5_Protected_Translation.txt"
)

with open(test_translation_path, "w", encoding="utf-8") as f:
    f.write(test_translation)

print("Article 5 translation saved!")
print(test_translation_path)

Article 5 translation saved!
/content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/NLLB_Article_5_Protected_Translation.txt


In [ ]:
import re
import pandas as pd
import os

def extract_numbers(text):
    # Convert Hindi digits into English digits
    hindi_digits = "०१२३४५६७८९"
    english_digits = "0123456789"
    translation = str.maketrans(hindi_digits, english_digits)

    text = str(text).translate(translation)

    return re.findall(r'\d+(?:\.\d+)?', text)


for _, row in test_df.iterrows():
    hindi_numbers = extract_numbers(row["hindi_article"])
    english_numbers = extract_numbers(row["english_article"])

    print("\nArticle:", row["article_number"])
    print("Hindi numbers:", hindi_numbers)
    print("English numbers:", english_numbers)
    print("Numbers preserved:", hindi_numbers == english_numbers)


Article: 2
Hindi numbers: ['350', '40', '400']
English numbers: ['350', '40', '400']
Numbers preserved: True

Article: 3
Hindi numbers: ['59', '0.7']
English numbers: ['59', '0', '7']
Numbers preserved: False

Article: 4
Hindi numbers: ['4', '15', '25', '4', '4', '10', '4', '4', '4', '15']
English numbers: ['4', '15', '25', '4', '4', '10', '4', '4', '4', '15']
Numbers preserved: True

Article: 5
Hindi numbers: ['100', '1994', '15']
English numbers: ['100', '1994', '15']
Numbers preserved: True


In [ ]:
import re

def translate_with_number_protection(text):
    hindi_digits = "०१२३४५६७८९"
    english_digits = "0123456789"
    text = str(text).translate(str.maketrans(hindi_digits, english_digits))

    numbers = []

    def replace_number(match):
        placeholder = f"ZXQNUMBER{len(numbers)}ZXQ"
        numbers.append(match.group())
        return placeholder

    protected_text = re.sub(
        r'\d+(?:\.\d+)?',
        replace_number,
        text
    )

    translated = translate_article_nllb(protected_text)

    for i, number in enumerate(numbers):
        translated = translated.replace(f"ZXQNUMBER{i}ZXQ", number)

    return translated


article_3 = test_df[test_df["article_number"] == 3].iloc[0]

fixed_translation = translate_with_number_protection(
    article_3["hindi_article"]
)

print("Original numbers:", extract_numbers(article_3["hindi_article"]))
print("Translated numbers:", extract_numbers(fixed_translation))
print("\nTranslation:\n", fixed_translation)

Original numbers: ['59', '0.7']
Translated numbers: ['59', '0.7']

Translation:
 According to reports, it took Jane Petkov about three and a half hours to cross Lake Orid in Macedonia. Organizer Sasso Tokov told AFP news agency that the hands and feet of 59-year-old Petkov were tied to his body and he swam backwards like a dolphin. According to Macedonian media, his average speed of swimming was 0.7 km/h. After completing his campaign, Petkov said, 'At first I felt very cold for a mile, but after that everything went well. There was no problem. "If the water were hot, I could swim ten kilometers or more", said Petkov, a Guinness World Record holder. The organizers reported that they had duly recorded this swimming performance to be officially included in the Guinness World Records. However, there are conflicting reports in the local media regarding the distance and time set by Petkov. The Guinness World Records, the body that records abnormal activities, has not made any public comment

In [ ]:
import os
import pandas as pd

# Load the cleaned 250 Hindi articles
data_path = os.path.join(
    BASE, "02_clean_data", "XL_Sum_Hindi_Cleaned_250.csv"
)
df_250 = pd.read_csv(data_path)

# Prepare output columns
for col in ["nllb_english_title", "nllb_english_article", "nllb_english_summary"]:
    if col not in df_250.columns:
        df_250[col] = ""

# Resume from existing checkpoint, if available
output_path = os.path.join(
    BASE, "03_english_data", "NLLB_English_250.csv"
)

if os.path.exists(output_path):
    saved_df = pd.read_csv(output_path)
    if len(saved_df) == len(df_250) and "nllb_english_article" in saved_df.columns:
        df_250 = saved_df
        print("Existing translation checkpoint loaded.")
    else:
        print("Existing checkpoint doesn't match; starting from cleaned data.")

completed = df_250["nllb_english_article"].fillna("").str.strip().ne("").sum()

print("Total articles:", len(df_250))
print("Already translated:", completed)
print("Remaining:", len(df_250) - completed)
print("Ready:", callable(translate_with_number_protection))

Total articles: 250
Already translated: 0
Remaining: 250
Ready: True


In [ ]:
import os

files_to_check = [
    "03_english_data/XL_Sum_English_250.csv",
    "03_english_data/XL_Sum_English_250_partial.csv",
    "03_english_data/NLLB_English_250.csv"
]

for file in files_to_check:
    path = os.path.join(BASE, file)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print("FOUND:", file, f"({size_mb:.2f} MB)")
    else:
        print("NOT FOUND:", file)

FOUND: 03_english_data/XL_Sum_English_250.csv (1.90 MB)
FOUND: 03_english_data/XL_Sum_English_250_partial.csv (1.90 MB)
FOUND: 03_english_data/NLLB_English_250.csv (1.51 MB)


In [ ]:
english_path = os.path.join(
    BASE, "03_english_data", "XL_Sum_English_250.csv"
)

df_english = pd.read_csv(english_path)

print("Rows:", len(df_english))
print("\nColumns:", df_english.columns.tolist())

for col in ["english_title", "english_text", "english_summary"]:
    if col in df_english.columns:
        filled = df_english[col].fillna("").str.strip().ne("").sum()
        print(f"{col}: {filled}/250 completed")

Rows: 250

Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words', 'english_title', 'english_text', 'english_summary']
english_title: 250/250 completed
english_text: 250/250 completed
english_summary: 250/250 completed


In [ ]:
import os
import pandas as pd
import shutil

english_path = os.path.join(
    BASE, "03_english_data", "XL_Sum_English_250.csv"
)

backup_path = os.path.join(
    BASE, "09_checkpoints", "XL_Sum_English_250_BACKUP.csv"
)

df_english = pd.read_csv(english_path)

required = ["english_title", "english_text", "english_summary"]
missing = [col for col in required if col not in df_english.columns]

if missing:
    print("Missing columns:", missing)
else:
    shutil.copy2(english_path, backup_path)
    print("Backup saved successfully!")
    print("Articles:", len(df_english))
    for col in required:
        count = df_english[col].fillna("").str.strip().ne("").sum()
        print(f"{col}: {count}/{len(df_english)} filled")
    print("Backup:", backup_path)

Backup saved successfully!
Articles: 250
english_title: 250/250 filled
english_text: 250/250 filled
english_summary: 250/250 filled
Backup: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_checkpoints/XL_Sum_English_250_BACKUP.csv


In [ ]:
# Merge Hindi and English data using article ID
df_work = df_250.merge(
    df_english[["id", "english_title", "english_text", "english_summary"]],
    on="id",
    how="left",
    validate="one_to_one"
)

print("Working dataset:", df_work.shape)
print("Missing English articles:", df_work["english_text"].isna().sum())
print("Missing English summaries:", df_work["english_summary"].isna().sum())

# Save working dataset
working_path = os.path.join(
    BASE, "03_english_data", "FactIndic_Hindi_English_Base_250.csv"
)

df_work.to_csv(working_path, index=False, encoding="utf-8-sig")

print("\nSaved:", working_path)

Working dataset: (250, 13)
Missing English articles: 0
Missing English summaries: 0

Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/03_english_data/FactIndic_Hindi_English_Base_250.csv


In [ ]:
save_progress(
    "english_translation",
    "completed_as_draft",
    {
        "articles": 250,
        "model": "Helsinki-NLP/opus-mt-hi-en",
        "quality": "Draft; requires spot-checking",
        "working_file": "03_english_data/FactIndic_Hindi_English_Base_250.csv"
    }
)

save_progress(
    "mixed_articles",
    "incomplete_needs_rework",
    {
        "reason": "Previous phrase-replacement method did not reach target English percentages.",
        "next_step": "Use a word-level mixing method and validate actual percentages."
    }
)

print("Progress records updated.")

Checkpoint saved: english_translation
Checkpoint saved: mixed_articles
Progress records updated.


In [ ]:
import re

def english_percentage(text):
    words = str(text).split()
    if not words:
        return 0
    english_words = sum(
        1 for word in words
        if re.search(r"[A-Za-z]", word)
    )
    return round(english_words / len(words) * 100, 2)

df_work["hindi_cmi"] = df_work["text"].apply(english_percentage)
df_work["english_cmi"] = df_work["english_text"].apply(english_percentage)

print("Hindi baseline average:", round(df_work["hindi_cmi"].mean(), 2), "%")
print("English baseline average:", round(df_work["english_cmi"].mean(), 2), "%")
print("\nSample:")
print(df_work[["id", "hindi_cmi", "english_cmi"]].head(10).to_string(index=False))

Hindi baseline average: 0.02 %
English baseline average: 98.71 %

Sample:
                                       id  hindi_cmi  english_cmi
                 080419_brchopra_festival        0.0       100.00
                       080613_afghan_jail        0.0        98.70
130912_guinnness_record_bulgarian_swim_sr        0.0       100.00
                             050713_us_un        0.0        97.85
                        120809_salman_pkp        0.0        99.34
                      080711_lanka_attack        0.0        98.53
                          social-43068605        0.0       100.00
                   international-51410040        0.0        98.90
                           india-55525807        0.0        98.75
                   international-41372477        0.0        97.41


In [ ]:
print("Glossary available:", "glossary" in globals())
print("Glossary entries:", len(glossary) if "glossary" in globals() else 0)

print("\nHindi article sample:")
print(df_work.loc[0, "text"][:700])

print("\nEnglish article sample:")
print(df_work.loc[0, "english_text"][:700])

Glossary available: True
Glossary entries: 26

Hindi article sample:
शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी." बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर 

English article sample:
On Friday evening, the scales of justice were shown in the Polish film. The little thing, the new rounds, and time during the celebration, will be seen to see the classical films. Bad health because of BRC

In [ ]:
import re

def split_sentences(text):
    return [
        s.strip()
        for s in re.split(r'(?<=[।.!?])\s*', str(text))
        if s.strip()
    ]

for i in range(10):
    hi = split_sentences(df_work.loc[i, "text"])
    en = split_sentences(df_work.loc[i, "english_text"])

    print(
        f"Article {i+1}: Hindi={len(hi)} sentences | "
        f"English={len(en)} sentences"
    )

Article 1: Hindi=15 sentences | English=13 sentences
Article 2: Hindi=12 sentences | English=12 sentences
Article 3: Hindi=12 sentences | English=6 sentences
Article 4: Hindi=14 sentences | English=10 sentences
Article 5: Hindi=24 sentences | English=11 sentences
Article 6: Hindi=15 sentences | English=10 sentences
Article 7: Hindi=27 sentences | English=18 sentences
Article 8: Hindi=76 sentences | English=44 sentences
Article 9: Hindi=47 sentences | English=41 sentences
Article 10: Hindi=20 sentences | English=18 sentences


In [ ]:
print("Similarity model available:", "similarity_model" in globals())
print("Glossary available:", "glossary" in globals())
print("Translation dataset:", df_work.shape)

Similarity model available: True
Glossary available: True
Translation dataset: (250, 15)


In [ ]:
from sentence_transformers import util

hi_sentences = split_sentences(df_work.loc[0, "text"])
en_sentences = split_sentences(df_work.loc[0, "english_text"])

hi_embeddings = similarity_model.encode(
    hi_sentences, convert_to_tensor=True
)
en_embeddings = similarity_model.encode(
    en_sentences, convert_to_tensor=True
)

similarities = util.cos_sim(hi_embeddings, en_embeddings)

for i, sentence in enumerate(hi_sentences):
    best_idx = int(similarities[i].argmax())
    score = float(similarities[i][best_idx])

    print(f"\nHindi: {sentence[:180]}")
    print(f"English: {en_sentences[best_idx][:180]}")
    print(f"Similarity: {score:.3f}")


Hindi: शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई.
English: On Friday evening, the scales of justice were shown in the Polish film.
Similarity: 0.448

Hindi: उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी.
English: The little thing, the new rounds, and time during the celebration, will be seen to see the classical films.
Similarity: 0.788

Hindi: खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.
English: BR'Courourin raised many subjects like widowry in her movies, such as prostitution.
Similarity: 0.260

Hindi: फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया.
English: However, most movies of BR's chlorophynism were based on some social point, but these used films also showed a remarkable deal on the board.
Similarity: 0.465

Hindi: उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि

In [ ]:
df_work["hi_words"] = df_work["text"].fillna("").str.split().str.len()
df_work["en_words"] = df_work["english_text"].fillna("").str.split().str.len()

df_work["translation_ratio"] = (
    df_work["en_words"] / df_work["hi_words"].replace(0, 1)
)

print("Articles:", len(df_work))
print("English translation shorter than half:",
      (df_work["translation_ratio"] < 0.5).sum())
print("English translation over twice as long:",
      (df_work["translation_ratio"] > 2).sum())
print("Average translation length ratio:",
      round(df_work["translation_ratio"].mean(), 2))

print("\nFirst 10 articles:")
print(df_work[["id", "hi_words", "en_words", "translation_ratio"]]
      .head(10).round(2).to_string(index=False))

Articles: 250
English translation shorter than half: 43
English translation over twice as long: 0
Average translation length ratio: 0.59

First 10 articles:
                                       id  hi_words  en_words  translation_ratio
                 080419_brchopra_festival       294       181               0.62
                       080613_afghan_jail       289       154               0.53
130912_guinnness_record_bulgarian_swim_sr       200        99               0.50
                             050713_us_un       245       186               0.76
                        120809_salman_pkp       365       151               0.41
                      080711_lanka_attack       239       136               0.57
                          social-43068605       345       220               0.64
                   international-51410040      1225       635               0.52
                           india-55525807       912       561               0.62
                   international-

In [ ]:
short_translations = df_work[
    df_work["translation_ratio"] < 0.5
].copy()

short_translations = short_translations[
    ["id", "title", "text", "english_text",
     "hi_words", "en_words", "translation_ratio"]
].sort_values("translation_ratio")

review_path = os.path.join(
    BASE, "05_quality_checks",
    "Short_English_Translations_Review.csv"
)

short_translations.to_csv(
    review_path, index=False, encoding="utf-8-sig"
)

print("Flagged articles:", len(short_translations))
print("Review file saved:", review_path)
print("\nFirst 10 flagged articles:")
print(short_translations[
    ["id", "hi_words", "en_words", "translation_ratio"]
].head(10).to_string(index=False))

Flagged articles: 43
Review file saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/Short_English_Translations_Review.csv

First 10 flagged articles:
                                       id  hi_words  en_words  translation_ratio
141230_mahendra_singh_dhoni_retirement_vr       435       162           0.372414
                        040305_holi_songs       266       100           0.375940
                   080830_orissa_violence       216        82           0.379630
             150901_nepal_protest_dead_pp       269       103           0.382900
               090216_jobloss_column3_alk       466       182           0.390558
                      070902_khanduri_win       161        66           0.409938
                    031031_russia_removal       191        79           0.413613
                        120809_salman_pkp       365       151           0.413699
           120518_ipl_glamour_magazine_sm       548       231           0.421533
                

In [ ]:
usable_candidates = df_work[
    df_work["translation_ratio"] >= 0.5
].copy()

usable_candidates["review_status"] = "Needs human review"

candidate_path = os.path.join(
    BASE, "05_quality_checks",
    "Translation_Review_Candidates_207.csv"
)

usable_candidates[
    ["id", "title", "text", "english_text",
     "hi_words", "en_words", "translation_ratio",
     "review_status"]
].to_csv(candidate_path, index=False, encoding="utf-8-sig")

print("Candidate articles:", len(usable_candidates))
print("Saved:", candidate_path)
print("\nSample article IDs:")
print(usable_candidates["id"].head(10).tolist())

Candidate articles: 207
Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/Translation_Review_Candidates_207.csv

Sample article IDs:
['080419_brchopra_festival', '080613_afghan_jail', '050713_us_un', '080711_lanka_attack', 'social-43068605', 'international-51410040', 'india-55525807', 'international-41372477', '130404_sam_pitroda_profile', '140323_breast_cancer_screening_tk']


In [ ]:
review_sample = usable_candidates.sample(
    n=20,
    random_state=42
).copy()

review_sample["translation_quality"] = ""
review_sample["names_correct"] = ""
review_sample["numbers_correct"] = ""
review_sample["meaning_preserved"] = ""
review_sample["comments"] = ""

sample_path = os.path.join(
    BASE, "05_quality_checks",
    "Human_Translation_Review_20.csv"
)

review_sample.to_csv(
    sample_path, index=False, encoding="utf-8-sig"
)

print("Human review sample:", len(review_sample))
print("Saved:", sample_path)
print(review_sample[["id", "hi_words", "en_words"]].to_string(index=False))

Human review sample: 20
Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/Human_Translation_Review_20.csv
                                       id  hi_words  en_words
                   international-44164669       695       387
   140702_wc2014_goldenboot_contenders_ar       239       153
    121114_international_us_plus_obama_ia       790       439
                   040511_shahbaz_profile       254       163
                         science-41399809       410       214
        140323_breast_cancer_screening_tk       455       286
                         science-39280869       961       505
160405_vert_cul_10_spectacular_museums_rd       829       563
                           india-48662333      1240       622
                   international-46767468      1127       628
                  040804_kolkata_reaction       360       217
                           india-55465758      1204       710
             150802_varun_gandhi_death_ia       359       212
 

In [ ]:
save_progress(
    "translation_quality_review",
    "prepared_for_human_review",
    {
        "total_articles": 250,
        "short_translations_flagged": 43,
        "review_candidates": 207,
        "human_review_sample": 20,
        "files": [
            "05_quality_checks/Short_English_Translations_Review.csv",
            "05_quality_checks/Translation_Review_Candidates_207.csv",
            "05_quality_checks/Human_Translation_Review_20.csv"
        ],
        "note": "Length screening is not proof of translation accuracy."
    }
)

print("Translation quality checkpoint saved.")

Checkpoint saved: translation_quality_review
Translation quality checkpoint saved.


In [ ]:
import os

folders = [
    "06_real_hinglish",
    "01_raw_data",
    "02_clean_data"
]

for folder in folders:
    path = os.path.join(BASE, folder)
    print(f"\n--- {folder} ---")

    if os.path.exists(path):
        files = os.listdir(path)
        if files:
            for file in files:
                print(file)
        else:
            print("Folder is empty")
    else:
        print("Folder not found")


--- 06_real_hinglish ---
Folder is empty

--- 01_raw_data ---
XL_Sum_Hindi_Test.parquet
XL_Sum_Raw_Sample_250.csv

--- 02_clean_data ---
XL_Sum_Hindi_Cleaned_250.csv
XL_Sum_Hindi_Cleaned_250.parquet


# New section

In [ ]:
!pip -q install datasets

from datasets import load_dataset, get_dataset_config_names

# Check MUTANT configurations
configs = get_dataset_config_names("LingoIITGN/MUTANT")
print("MUTANT configurations:", configs)

# Load a small streaming sample from MUTANT
mutant = load_dataset(
    "LingoIITGN/MUTANT",
    name="Aap",
    split="train",
    streaming=True
)

mutant_first = next(iter(mutant))
print("\nMUTANT columns:", list(mutant_first.keys()))
print("MUTANT sample:")
print(str(mutant_first)[:500])

# Load a small streaming sample from GupShup
gupshup = load_dataset(
    "saidutta69/Gupshup",
    split="train",
    streaming=True
)

gup_first = next(iter(gupshup))
print("\nGupShup columns:", list(gup_first.keys()))
print("GupShup sample:")
print(str(gup_first)[:500])

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

MUTANT configurations: ['Aap', 'Bhaskar', 'INC', 'Jagran', 'Mankibaat', 'PM-speech', 'PIB']


Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]


MUTANT columns: ['text']
MUTANT sample:
{'text': 'केजरीवाल सरकार के प्लाज्मा बैंक से 710 लोगों को मिला मुफ्त प्लाज्मा, 921 लोग कर चुके हैं प्लाज्मा दान मुख्यमंत्री अरविंद केजरीवाल की प्लाज्मा बैंक खोलने की पहल बनी कोरोना मरीजों के लिए संजीवनी दिल्ली में गंभीर कोविड मरीजों को स्वस्थ्य करने में कारगर साबित हो रहा प्लाज्मा थेरेपी नई दिल्ली, 11 अगस्त, 2020मुख्यमंत्री अरविंद केजरीवाल की प्लाज्मा बैंक खोलने की निजी पहल अब कोरोना के गंभीर मरीजों के लिए संजीवनी बन गई है।  दिल्ली सरकार द्वारा कोविड-19 का मुकाबला करने के लिए उठाए गए विभिन्न कदम


README.md: 0.00B [00:00, ?B/s]


GupShup columns: ['id', 'author', 'timestamp', 'text', 'reply_to_text', 'topics', 'has_emote', 'char_len', 'source']
GupShup sample:
{'id': '818018916202184744', 'author': 'USER_bb74332e', 'timestamp': '2021-03-07T07:15:07.679000+00:00', 'text': 'Koi nhi vai hota h kbhi kbhi [EMOTE_sSheruPeek]', 'reply_to_text': 'Kya kare vai aadat se majboor', 'topics': ['other_chitchat'], 'has_emote': True, 'char_len': 47, 'source': 'room-4cc844'}


In [ ]:
from itertools import islice
from datasets import load_dataset

print("MUTANT SCRIPT CHECK")

mutant_results = {}

for config in configs:
    try:
        data = load_dataset(
            "LingoIITGN/MUTANT",
            name=config,
            streaming=True
        )

        rows = []
        for split_name in data.keys():
            remaining = 100 - len(rows)
            if remaining <= 0:
                break
            rows.extend(list(islice(data[split_name], remaining)))

        count = sum(is_hinglish(row["text"]) for row in rows)
        mutant_results[config] = {
            "checked": len(rows),
            "hinglish": count
        }

        print(f"{config}: {count}/{len(rows)} Hinglish")

    except Exception as e:
        print(f"{config}: Error - {e}")

print("\nMUTANT CHECK COMPLETE")

MUTANT SCRIPT CHECK


Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Aap: 24/62 Hinglish


Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/7054 [00:00<?, ?it/s]

... (17 outputs left out)

PM-speech: 19/100 Hinglish


Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/8473 [00:00<?, ?it/s]

PIB: 11/100 Hinglish

MUTANT CHECK COMPLETE


In [ ]:
import pandas as pd
import re
import os
from datasets import load_dataset

selected = []
target = 100

# Prioritise configurations with higher Hinglish counts
priority = ["Mankibaat", "INC", "Bhaskar", "Aap",
            "PM-speech", "PIB", "Jagran"]

for config in priority:
    data = load_dataset(
        "LingoIITGN/MUTANT",
        name=config,
        streaming=True
    )

    for split_name in data.keys():
        for row in data[split_name]:
            text = str(row.get("text", "")).strip()

            if is_hinglish(text):
                selected.append({
                    "source_dataset": "MUTANT",
                    "source_config": config,
                    "text": text,
                    "script_type": "Mixed Devanagari-Latin",
                    "selection_method": "Script-based candidate filter"
                })

            if len(selected) >= target:
                break

        if len(selected) >= target:
            break

    if len(selected) >= target:
        break

hinglish_df = pd.DataFrame(selected[:target])

save_csv(
    hinglish_df,
    "06_real_hinglish/MUTANT_Hinglish_100_Candidates.csv"
)

print("Saved examples:", len(hinglish_df))
print(hinglish_df["source_config"].value_counts())
display(hinglish_df.head(5))

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/58 [00:00<?, ?it/s]

Saved: 06_real_hinglish/MUTANT_Hinglish_100_Candidates.csv | Rows: 100
Saved examples: 100
source_config
Mankibaat    100
Name: count, dtype: int64


,source_dataset,source_config,text,script_type,selection_method
0,MUTANT,Mankibaat,"मेरे प्यारे देशवासियो, आप सबको नमस्कार। 2015 –...",Mixed Devanagari-Latin,Script-based candidate filter
1,MUTANT,Mankibaat,अनेक लोगों के अथक प्रयास का परिणाम है कि देश ब...,Mixed Devanagari-Latin,Script-based candidate filter
2,MUTANT,Mankibaat,स्वच्छता की बात एक प्रकार से घर-घर में गूंज रह...,Mixed Devanagari-Latin,Script-based candidate filter
3,MUTANT,Mankibaat,"मेरे प्यारे नौजवान साथियो, 15 अगस्त को लाल किल...",Mixed Devanagari-Latin,Script-based candidate filter
4,MUTANT,Mankibaat,Start-up के संबंध में हमारे यहाँ एक सोच बंधी-ब...,Mixed Devanagari-Latin,Script-based candidate filter


In [ ]:
from datasets import load_dataset
import pandas as pd

gup_data = load_dataset(
    "saidutta69/Gupshup",
    split="train",
    streaming=True
)

gup_rows = []

for row in gup_data:
    text = str(row.get("text", "")).strip()

    if text:
        gup_rows.append({
            "source_dataset": "GupShup",
            "text": text,
            "reply_to_text": str(row.get("reply_to_text", "")),
            "script_type": "Roman-script candidate",
            "selection_method": "Non-empty text sample"
        })

    if len(gup_rows) == 100:
        break

gup_df = pd.DataFrame(gup_rows)

save_csv(
    gup_df,
    "06_real_hinglish/GupShup_100_Candidates.csv"
)

print("GupShup records saved:", len(gup_df))
display(gup_df.head(10))

Saved: 06_real_hinglish/GupShup_100_Candidates.csv | Rows: 100
GupShup records saved: 100


,source_dataset,text,reply_to_text,script_type,selection_method
0,GupShup,Koi nhi vai hota h kbhi kbhi [EMOTE_sSheruPeek],Kya kare vai aadat se majboor,Roman-script candidate,Non-empty text sample
1,GupShup,[MENTION] paaji humse jyada toh ye janta[EMOTE...,Ap dikaye hi nhi dete,Roman-script candidate,Non-empty text sample
2,GupShup,[MENTION] making full out of it btw yearly wal...,,Roman-script candidate,Non-empty text sample
3,GupShup,samajh mein to aaya but ek baar batao kahin ga...,,Roman-script candidate,Non-empty text sample
4,GupShup,2k claims hone se phele 150 krne honge[EMOTE_w...,Oye ye kya sunna ma aya,Roman-script candidate,Non-empty text sample
5,GupShup,[EMOTE_zzz_pepekek] tere kehna ka mtlb mai ach...,Isiliye nai try kiya,Roman-script candidate,Non-empty text sample
6,GupShup,My laptop is on charge zero please wait till i...,,Roman-script candidate,Non-empty text sample
7,GupShup,what is a ddr?\n\nsorry I'm not a PC guy I'm b...,,Roman-script candidate,Non-empty text sample
8,GupShup,[MENTION] eyyy tatti tu ghumne gaya tha 🤔 vapi...,,Roman-script candidate,Non-empty text sample
9,GupShup,[MENTION] [MENTION] pooch kon raha h dono se 😂🤣,,Roman-script candidate,Non-empty text sample


In [ ]:
import pandas as pd
import os

mutant_path = os.path.join(
    BASE, "06_real_hinglish/MUTANT_Hinglish_100_Candidates.csv"
)

gup_path = os.path.join(
    BASE, "06_real_hinglish/GupShup_100_Candidates.csv"
)

mutant_df = pd.read_csv(mutant_path)
gup_df = pd.read_csv(gup_path)

combined_df = pd.concat(
    [mutant_df, gup_df],
    ignore_index=True,
    sort=False
)

save_csv(
    combined_df,
    "06_real_hinglish/FactIndic_Real_Hinglish_200_Candidates.csv"
)

print("Total candidates:", len(combined_df))
print("\nSource counts:")
print(combined_df["source_dataset"].value_counts())

Saved: 06_real_hinglish/FactIndic_Real_Hinglish_200_Candidates.csv | Rows: 200
Total candidates: 200

Source counts:
source_dataset
MUTANT     100
GupShup    100
Name: count, dtype: int64


In [ ]:
import re
import pandas as pd

def is_likely_hinglish(row):
    text = str(row["text"]).lower()

    if row["source_dataset"] == "MUTANT":
        return bool(re.search(r"[\u0900-\u097F]", text)) and bool(
            re.search(r"[a-zA-Z]", text)
        )

    # Romanized Hindi indicators
    roman_hindi = {
        "hai", "hain", "nahi", "nhi", "kya", "kaise",
        "mera", "meri", "tum", "aap", "karo", "karna",
        "hoga", "ho", "tha", "thi", "toh", "bhi",
        "bhai", "yaar", "kyun", "ka", "ki", "ke"
    }

    words = set(re.findall(r"[a-zA-Z]+", text))
    return len(words.intersection(roman_hindi)) >= 2

combined_df["likely_hinglish"] = combined_df.apply(
    is_likely_hinglish, axis=1
)

print("Likely Hinglish:", combined_df["likely_hinglish"].sum())
print("Needs review:", (~combined_df["likely_hinglish"]).sum())

display(
    combined_df[combined_df["likely_hinglish"]]
    [["source_dataset", "text"]]
    .head(10)
)

Likely Hinglish: 144
Needs review: 56


,source_dataset,text
0,MUTANT,"मेरे प्यारे देशवासियो, आप सबको नमस्कार। 2015 –..."
1,MUTANT,अनेक लोगों के अथक प्रयास का परिणाम है कि देश ब...
2,MUTANT,स्वच्छता की बात एक प्रकार से घर-घर में गूंज रह...
3,MUTANT,"मेरे प्यारे नौजवान साथियो, 15 अगस्त को लाल किल..."
4,MUTANT,Start-up के संबंध में हमारे यहाँ एक सोच बंधी-ब...
5,MUTANT,"प्यारे नौजवान साथियो, 12 जनवरी स्वामी विवेकानं..."
6,MUTANT,उस दिन हमने ‘सुगम्य भारत’ अभियान की शुरुआत की ...
7,MUTANT,"मेरे प्यारे देशवासियो, सरकार की योजनायें तो नि..."
8,MUTANT,"मेरे प्यारे देशवासियो, कुछ दिन पहले मैं सरदार ..."
9,MUTANT,"प्यारे देशवासियो, 26 जनवरी का पर्व बहुत उमंग औ..."


In [ ]:
# Save all 200 records with screening labels
save_csv(
    combined_df,
    "06_real_hinglish/FactIndic_Hinglish_200_Screened.csv"
)

# Save the 144 likely Hinglish candidates separately
likely_df = combined_df[
    combined_df["likely_hinglish"] == True
].copy()

save_csv(
    likely_df,
    "06_real_hinglish/FactIndic_Hinglish_144_Likely.csv"
)

print("Screening saved successfully.")
print("Likely Hinglish:", len(likely_df))
print("Needs review:", len(combined_df) - len(likely_df))

Saved: 06_real_hinglish/FactIndic_Hinglish_200_Screened.csv | Rows: 200
Saved: 06_real_hinglish/FactIndic_Hinglish_144_Likely.csv | Rows: 144
Screening saved successfully.
Likely Hinglish: 144
Needs review: 56


In [ ]:
print("SOURCE-WISE SCREENING")

print(
    combined_df.groupby("source_dataset")["likely_hinglish"]
    .agg(["sum", "count"])
)

print("\nSAMPLE OF 56 NEEDS-REVIEW RECORDS")

display(
    combined_df[combined_df["likely_hinglish"] == False]
    [["source_dataset", "text"]]
    .head(15)
)

SOURCE-WISE SCREENING
                sum  count
source_dataset            
GupShup          44    100
MUTANT          100    100

SAMPLE OF 56 NEEDS-REVIEW RECORDS


,source_dataset,text
100,GupShup,Koi nhi vai hota h kbhi kbhi [EMOTE_sSheruPeek]
101,GupShup,[MENTION] paaji humse jyada toh ye janta[EMOTE...
102,GupShup,[MENTION] making full out of it btw yearly wal...
103,GupShup,samajh mein to aaya but ek baar batao kahin ga...
104,GupShup,2k claims hone se phele 150 krne honge[EMOTE_w...
106,GupShup,My laptop is on charge zero please wait till i...
107,GupShup,what is a ddr?\n\nsorry I'm not a PC guy I'm b...
109,GupShup,[MENTION] [MENTION] pooch kon raha h dono se 😂🤣
110,GupShup,> [NAME]?\n> Blue thunder?\n[MENTION] yes i do
114,GupShup,😂 jis ghar pe delivery hui thi unse paise lelo...


In [ ]:
final_hinglish_df = combined_df[
    combined_df["likely_hinglish"] == True
].copy()

final_hinglish_df = final_hinglish_df.drop_duplicates(
    subset=["source_dataset", "text"]
)

save_csv(
    final_hinglish_df,
    "06_real_hinglish/FactIndic_Real_Hinglish_Final_Candidates.csv"
)

print("Final candidate rows:", len(final_hinglish_df))
print("\nSource distribution:")
print(final_hinglish_df["source_dataset"].value_counts())

save_progress(
    "real_hinglish_collection",
    "candidate_collection_completed",
    {
        "mutant_candidates": 100,
        "gupshup_likely_candidates": 44,
        "total_before_deduplication": 144,
        "final_rows": len(final_hinglish_df)
    }
)

Saved: 06_real_hinglish/FactIndic_Real_Hinglish_Final_Candidates.csv | Rows: 144
Final candidate rows: 144

Source distribution:
source_dataset
MUTANT     100
GupShup     44
Name: count, dtype: int64
Checkpoint saved: real_hinglish_collection


In [ ]:
review_df = final_hinglish_df.copy()

review_df["human_verified"] = ""
review_df["language_quality"] = ""
review_df["meaning_clear"] = ""
review_df["review_notes"] = ""

save_csv(
    review_df,
    "06_real_hinglish/FactIndic_Hinglish_Manual_Review.csv"
)

print("Review sheet created:", len(review_df))
display(
    review_df[
        ["source_dataset", "text", "human_verified",
         "language_quality", "meaning_clear", "review_notes"]
    ].head(15)
)

Saved: 06_real_hinglish/FactIndic_Hinglish_Manual_Review.csv | Rows: 144
Review sheet created: 144


,source_dataset,text,human_verified,language_quality,meaning_clear,review_notes
0,MUTANT,"मेरे प्यारे देशवासियो, आप सबको नमस्कार। 2015 –...",,,,
1,MUTANT,अनेक लोगों के अथक प्रयास का परिणाम है कि देश ब...,,,,
2,MUTANT,स्वच्छता की बात एक प्रकार से घर-घर में गूंज रह...,,,,
3,MUTANT,"मेरे प्यारे नौजवान साथियो, 15 अगस्त को लाल किल...",,,,
4,MUTANT,Start-up के संबंध में हमारे यहाँ एक सोच बंधी-ब...,,,,
5,MUTANT,"प्यारे नौजवान साथियो, 12 जनवरी स्वामी विवेकानं...",,,,
6,MUTANT,उस दिन हमने ‘सुगम्य भारत’ अभियान की शुरुआत की ...,,,,
7,MUTANT,"मेरे प्यारे देशवासियो, सरकार की योजनायें तो नि...",,,,
8,MUTANT,"मेरे प्यारे देशवासियो, कुछ दिन पहले मैं सरदार ...",,,,
9,MUTANT,"प्यारे देशवासियो, 26 जनवरी का पर्व बहुत उमंग औ...",,,,


In [ ]:
import re

review_df["latin_word_count"] = review_df["text"].apply(
    lambda x: len(re.findall(r"\b[A-Za-z]+\b", str(x)))
)

review_df["total_word_count"] = review_df["text"].apply(
    lambda x: len(str(x).split())
)

review_df["latin_word_percentage"] = (
    review_df["latin_word_count"] /
    review_df["total_word_count"].clip(lower=1) * 100
).round(2)

print("English-script word count distribution:")
print(review_df.groupby("source_dataset")["latin_word_count"].describe())

print("\nExamples with highest English-script usage:")
display(
    review_df.sort_values(
        "latin_word_percentage", ascending=False
    )[["source_dataset", "text", "latin_word_count",
       "latin_word_percentage"]].head(10)
)

English-script word count distribution:
                count       mean        std  min    25%   50%   75%   max
source_dataset                                                           
GupShup          44.0  13.863636   6.550664  5.0  10.00  12.5  16.0  37.0
MUTANT          100.0  15.740000  15.745206  1.0   4.75  10.0  22.0  73.0

Examples with highest English-script usage:


,source_dataset,text,latin_word_count,latin_word_percentage
198,GupShup,[MENTION]apna wo thor bring me thanos wala bhi...,10,111.11
150,GupShup,[MENTION][SERVER] ka gaming tatti wo bas pubgm...,10,111.11
193,GupShup,[MENTION] I don't like your behaviour and don'...,36,109.09
185,GupShup,[MENTION]o m yaha [MENTION] ke baare m baat ka...,12,109.09
171,GupShup,[MENTION] aa bhi jao yaar..sab aapke darshan l...,14,107.69
160,GupShup,[MENTION] wo pta hai...but aap k aane par sab ...,15,107.14
173,GupShup,You can't be a normie if u stay quiet (abhi pr...,19,105.56
170,GupShup,Sponsor wala and yeh lb wala done \nAbhi girls...,16,100.00
165,GupShup,[MENTION] kyun Hein bhai tere jindigi jhand? K...,11,100.00
130,GupShup,[MENTION] lol meko Samsung ka lena bhi nhi hai xD,10,100.00


In [ ]:
import re

roman_hindi_words = {
    "hai", "hain", "nahi", "nhi", "kya", "kaise",
    "mera", "meri", "tum", "aap", "karo", "karna",
    "hoga", "ho", "tha", "thi", "toh", "bhi",
    "bhai", "yaar", "kyun", "ka", "ki", "ke",
    "me", "mein", "nhi", "wala", "wali", "gaya",
    "gayi", "acha", "accha", "bas", "abhi", "kuch",
    "mujhe", "apna", "tera", "tere", "ham", "hum"
}

def analyze_language(row):
    text = str(row["text"])

    # Remove dataset markers
    text = re.sub(r"\[[A-Z_a-z]+\]", " ", text)

    latin = re.findall(r"\b[A-Za-z]+\b", text)
    hindi = re.findall(r"[\u0900-\u097F]+", text)

    roman_hindi = [
        word for word in latin
        if word.lower() in roman_hindi_words
    ]

    return pd.Series({
        "latin_words": len(latin),
        "devanagari_words": len(hindi),
        "roman_hindi_words": len(roman_hindi),
        "total_language_words": len(latin) + len(hindi)
    })

review_df[
    ["latin_words", "devanagari_words",
     "roman_hindi_words", "total_language_words"]
] = review_df.apply(analyze_language, axis=1)

print("Corrected source-wise analysis:")
display(
    review_df.groupby("source_dataset")[
        ["latin_words", "devanagari_words",
         "roman_hindi_words"]
    ].mean().round(2)
)

print("\nSample records:")
display(
    review_df[
        ["source_dataset", "text", "latin_words",
         "devanagari_words", "roman_hindi_words"]
    ].head(10)
)

Corrected source-wise analysis:


,latin_words,devanagari_words,roman_hindi_words
source_dataset,,,
GupShup,13.18,0.0,3.43
MUTANT,15.74,253.0,0.00



Sample records:


,source_dataset,text,latin_words,devanagari_words,roman_hindi_words
0,MUTANT,"मेरे प्यारे देशवासियो, आप सबको नमस्कार। 2015 –...",2,188,0
1,MUTANT,अनेक लोगों के अथक प्रयास का परिणाम है कि देश ब...,4,105,0
2,MUTANT,स्वच्छता की बात एक प्रकार से घर-घर में गूंज रह...,5,445,0
3,MUTANT,"मेरे प्यारे नौजवान साथियो, 15 अगस्त को लाल किल...",42,133,0
4,MUTANT,Start-up के संबंध में हमारे यहाँ एक सोच बंधी-ब...,18,180,0
5,MUTANT,"प्यारे नौजवान साथियो, 12 जनवरी स्वामी विवेकानं...",30,201,0
6,MUTANT,उस दिन हमने ‘सुगम्य भारत’ अभियान की शुरुआत की ...,11,87,0
7,MUTANT,"मेरे प्यारे देशवासियो, सरकार की योजनायें तो नि...",15,385,0
8,MUTANT,"मेरे प्यारे देशवासियो, कुछ दिन पहले मैं सरदार ...",25,616,0
9,MUTANT,"प्यारे देशवासियो, 26 जनवरी का पर्व बहुत उमंग औ...",3,340,0


In [ ]:
print("MUTANT candidate distribution")

mutant_review = review_df[
    review_df["source_dataset"] == "MUTANT"
]

print("Total:", len(mutant_review))
print("At least 5 Latin words:",
      (mutant_review["latin_words"] >= 5).sum())
print("At least 10 Latin words:",
      (mutant_review["latin_words"] >= 10).sum())

print("\nGupShup candidate distribution")

gup_review = review_df[
    review_df["source_dataset"] == "GupShup"
]

print("Total:", len(gup_review))
print("At least 2 Roman Hindi indicators:",
      (gup_review["roman_hindi_words"] >= 2).sum())

display(
    gup_review[
        ["text", "roman_hindi_words", "latin_words"]
    ].head(10)
)

MUTANT candidate distribution
Total: 100
At least 5 Latin words: 75
At least 10 Latin words: 51

GupShup candidate distribution
Total: 44
At least 2 Roman Hindi indicators: 44


,text,roman_hindi_words,latin_words
105,[EMOTE_zzz_pepekek] tere kehna ka mtlb mai ach...,4,8
108,[MENTION] eyyy tatti tu ghumne gaya tha 🤔 vapi...,4,10
111,Ungli pakadne diye toh haath mat kheecho tum [...,2,8
112,[EMOTE_pareshan_pookie] [EMOTE_pareshan_pookie...,3,6
113,[MENTION] Kya hi pfp h .. Meri behen ki yaad a...,3,9
120,"[MENTION] aaj toh viva hai , padhna hai , varn...",6,11
121,Hlo bhai log ye looking for video editor sahi ...,4,13
122,"Baki chill hu, bas hota h koi event khtm hone ...",11,37
128,> [MENTION] iski jrurat ni thi ye hoga😎\n[MENT...,2,6
129,"sed fr\nbas next year ka wait hai, feb main an...",4,21


In [ ]:
mutant_final = review_df[
    (review_df["source_dataset"] == "MUTANT") &
    (review_df["latin_words"] >= 10)
].copy()

gup_final = review_df[
    (review_df["source_dataset"] == "GupShup") &
    (review_df["roman_hindi_words"] >= 2)
].copy()

strict_hinglish_df = pd.concat(
    [mutant_final, gup_final],
    ignore_index=True
)

save_csv(
    strict_hinglish_df,
    "06_real_hinglish/FactIndic_Hinglish_95_Strict_Candidates.csv"
)

print("Strict candidate count:", len(strict_hinglish_df))
print(strict_hinglish_df["source_dataset"].value_counts())

save_progress(
    "real_hinglish_quality_filter",
    "strict_candidates_created",
    {
        "mutant_threshold": "at least 10 Latin-script words",
        "mutant_selected": len(mutant_final),
        "gupshup_threshold": "at least 2 Roman-Hindi indicators",
        "gupshup_selected": len(gup_final),
        "total": len(strict_hinglish_df),
        "status": "automatically filtered; human review pending"
    }
)

Saved: 06_real_hinglish/FactIndic_Hinglish_95_Strict_Candidates.csv | Rows: 95
Strict candidate count: 95
source_dataset
MUTANT     51
GupShup    44
Name: count, dtype: int64
Checkpoint saved: real_hinglish_quality_filter


In [ ]:
human_review_95 = strict_hinglish_df.copy()

human_review_95["verified_hinglish"] = ""
human_review_95["english_words_used_naturally"] = ""
human_review_95["meaning_clear"] = ""
human_review_95["review_notes"] = ""

save_csv(
    human_review_95,
    "06_real_hinglish/FactIndic_Hinglish_95_Human_Review.csv"
)

print("Human review sheet:", len(human_review_95))
display(
    human_review_95[
        ["source_dataset", "text", "verified_hinglish",
         "english_words_used_naturally", "meaning_clear"]
    ].head(10)
)

Saved: 06_real_hinglish/FactIndic_Hinglish_95_Human_Review.csv | Rows: 95
Human review sheet: 95


,source_dataset,text,verified_hinglish,english_words_used_naturally,meaning_clear
0,MUTANT,"मेरे प्यारे नौजवान साथियो, 15 अगस्त को लाल किल...",,,
1,MUTANT,Start-up के संबंध में हमारे यहाँ एक सोच बंधी-ब...,,,
2,MUTANT,"प्यारे नौजवान साथियो, 12 जनवरी स्वामी विवेकानं...",,,
3,MUTANT,उस दिन हमने ‘सुगम्य भारत’ अभियान की शुरुआत की ...,,,
4,MUTANT,"मेरे प्यारे देशवासियो, सरकार की योजनायें तो नि...",,,
5,MUTANT,"मेरे प्यारे देशवासियो, कुछ दिन पहले मैं सरदार ...",,,
6,MUTANT,"“नमस्कार, मैं सचिन तेंदुलकर बोल रहा हूँ। मुझे ...",,,
7,MUTANT,जिन हज़ारों लोगों ने मुझे मेरे App पर मोबाइल फ़ो...,,,
8,MUTANT,65 प्रतिशत जनसँख्या नौजवान हो और खेलों की दुनि...,,,
9,MUTANT,"दोस्तो, मैं आप से एक अपेक्षा करता हूँ। 2017 की...",,,


# New section

In [ ]:
print("PHINC availability check")

try:
    print("PHINC dataset:", phinc)
    print("PHINC rows:", len(phinc))
    print("PHINC columns:", phinc.columns.tolist())
except NameError:
    print("PHINC variable not found in current runtime.")

print("\nMUTANT file exists:",
      os.path.exists(os.path.join(
          BASE, "06_real_hinglish/MUTANT_Hinglish_100_Candidates.csv"
      )))

PHINC availability check
PHINC variable not found in current runtime.

MUTANT file exists: True


In [ ]:
from datasets import load_dataset
import pandas as pd
import os

# Reload PHINC
phinc_data = load_dataset(
    "LingoIITGN/PHINC",
    split="train"
)

phinc = pd.DataFrame(phinc_data)

print("PHINC shape:", phinc.shape)
print("Columns:", phinc.columns.tolist())

# Save a backup in Drive
save_csv(
    phinc,
    "05_quality_checks/PHINC_Corpus_13738.csv"
)

print("\nPHINC restored and saved.")
display(phinc.head(5))

README.md: 0.00B [00:00, ?B/s]

PHINC.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/13738 [00:00<?, ? examples/s]

PHINC shape: (13738, 2)
Columns: ['Sentence', 'English_Translation']
Saved: 05_quality_checks/PHINC_Corpus_13738.csv | Rows: 13738

PHINC restored and saved.


,Sentence,English_Translation
0,@someUSER congratulations on you celebrating b...,@some users congratulate you for celebrating B...
1,@LoKarDi_RT uske liye toh bahot kuch karna pad...,"@Lokardi_ rat we should a lot more for that, b..."
2,@slimswamy yehi to hum semjhane ki koshish kar...,"@Slimswami ehi, this is what i'm expecting you..."
3,@DramebaazKudi cake kaha hai ??,@Where is Dramebajakudi where is the cake?
4,@someUSER i'm in hawaii at the moment . home ...,@some user Don't want to come home next friday...


In [ ]:
import re
import pandas as pd
import os

# Load MUTANT candidates
mutant_df = pd.read_csv(
    os.path.join(
        BASE,
        "06_real_hinglish/MUTANT_Hinglish_100_Candidates.csv"
    )
)

roman_hindi_words = {
    "hai", "hain", "nahi", "nhi", "kya", "kaise",
    "mera", "meri", "tum", "aap", "karo", "karna",
    "hoga", "ho", "tha", "thi", "toh", "bhi",
    "bhai", "yaar", "kyun", "ka", "ki", "ke",
    "me", "mein", "wala", "wali", "gaya", "gayi",
    "acha", "accha", "bas", "abhi", "kuch", "mujhe",
    "apna", "tera", "tere", "hum", "hain", "hai"
}

def calculate_cmi(hindi_count, english_count):
    total = hindi_count + english_count
    if total == 0:
        return 0
    return round(
        (1 - max(hindi_count, english_count) / total) * 100, 2
    )

def mutant_cmi(text):
    text = re.sub(r"\[[^\]]*\]", " ", str(text))
    devanagari = re.findall(r"[\u0900-\u097F]+", text)
    latin = re.findall(r"\b[A-Za-z]+\b", text)

    roman_hindi = sum(
        word.lower() in roman_hindi_words for word in latin
    )

    hindi_count = len(devanagari) + roman_hindi
    english_count = len(latin) - roman_hindi

    return calculate_cmi(hindi_count, english_count)

def phinc_cmi(text):
    text = re.sub(r"\[[^\]]*\]", " ", str(text))
    words = re.findall(r"\b[A-Za-z]+\b", text.lower())

    hindi_count = sum(word in roman_hindi_words for word in words)
    english_count = len(words) - hindi_count

    return calculate_cmi(hindi_count, english_count)

# Calculate scores
mutant_df["CMI"] = mutant_df["text"].apply(mutant_cmi)
phinc["CMI"] = phinc["Sentence"].apply(phinc_cmi)

comparison = pd.DataFrame({
    "Dataset": ["MUTANT", "PHINC"],
    "Records": [len(mutant_df), len(phinc)],
    "Mean_CMI": [
        round(mutant_df["CMI"].mean(), 2),
        round(phinc["CMI"].mean(), 2)
    ],
    "Median_CMI": [
        round(mutant_df["CMI"].median(), 2),
        round(phinc["CMI"].median(), 2)
    ]
})

print("CMI COMPARISON")
display(comparison)

save_csv(
    comparison,
    "05_quality_checks/CMI_MUTANT_PHINC_Comparison.csv"
)

CMI COMPARISON


,Dataset,Records,Mean_CMI,Median_CMI
0,MUTANT,100,6.26,4.77
1,PHINC,13738,17.61,16.67


Saved: 05_quality_checks/CMI_MUTANT_PHINC_Comparison.csv | Rows: 2


In [ ]:
import os
import pandas as pd

mixed_path = os.path.join(
    BASE, "04_mixed_data/FactIndic_Mixed_250.csv"
)

mixed_df = pd.read_csv(mixed_path)

print("Mixed dataset shape:", mixed_df.shape)
print("Columns:")
print(mixed_df.columns.tolist())

Mixed dataset shape: (250, 16)
Columns:
['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words', 'english_title', 'english_text', 'english_summary', 'mixed_10_article', 'mixed_30_article', 'mixed_50_article', 'mixed_10_article_english_pct', 'mixed_30_article_english_pct', 'mixed_50_article_english_pct']


In [ ]:
import pandas as pd

pct_columns = [
    "mixed_10_article_english_pct",
    "mixed_30_article_english_pct",
    "mixed_50_article_english_pct"
]

for col in pct_columns:
    print(f"\n{col}")
    print("Mean:", round(mixed_df[col].mean(), 2))
    print("Median:", round(mixed_df[col].median(), 2))
    print("Minimum:", round(mixed_df[col].min(), 2))
    print("Maximum:", round(mixed_df[col].max(), 2))
    print("Articles reaching target:",
          (mixed_df[col] >= int(col.split("_")[1])).sum())


mixed_10_article_english_pct
Mean: 2.79
Median: 2.57
Minimum: 0.0
Maximum: 9.95
Articles reaching target: 0

mixed_30_article_english_pct
Mean: 2.79
Median: 2.57
Minimum: 0.0
Maximum: 9.95
Articles reaching target: 0

mixed_50_article_english_pct
Mean: 2.79
Median: 2.57
Minimum: 0.0
Maximum: 9.95
Articles reaching target: 0


In [ ]:
print("CHECKING MIXED VERSION DIFFERENCES")

print(
    "10% and 30% identical:",
    mixed_df["mixed_10_article"].equals(
        mixed_df["mixed_30_article"]
    )
)

print(
    "30% and 50% identical:",
    mixed_df["mixed_30_article"].equals(
        mixed_df["mixed_50_article"]
    )
)

print(
    "10% and 50% identical:",
    mixed_df["mixed_10_article"].equals(
        mixed_df["mixed_50_article"]
    )
)

print("\nSample lengths:")
for col in [
    "mixed_10_article",
    "mixed_30_article",
    "mixed_50_article"
]:
    print(col, mixed_df[col].iloc[0][:250])

CHECKING MIXED VERSION DIFFERENCES
10% and 30% identical: True
30% and 50% identical: True
10% and 50% identical: True

Sample lengths:
mixed_10_article शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर च
mixed_30_article शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर च
mixed_50_article शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर च


In [ ]:
import os
import json

dictionary_path = os.path.join(
    BASE, "04_mixed_data/hindi_english_phrase_dictionary.json"
)

with open(dictionary_path, "r", encoding="utf-8") as f:
    phrase_data = json.load(f)

print("Dictionary type:", type(phrase_data).__name__)
print("Number of entries:", len(phrase_data))

print("\nFirst 10 entries:")
if isinstance(phrase_data, dict):
    print(list(phrase_data.items())[:10])
else:
    print(phrase_data[:10])

Dictionary type: dict
Number of entries: 26

First 10 entries:
[('इंसाफ़ का तराज़ू', 'Insaf Ka Tarazu'), ('छोटी सी बात', 'Chhoti Si Baat'), ('नया दौर', 'Naya Daur'), ('वक़्त', 'Waqt'), ('साधना', 'Sadhana'), ('गुमराह', 'Gumrah'), ('हमराज़', 'Humraaz'), ('क़ानून', 'Kanoon'), ('निकाह', 'Nikaah'), ('बीआर चोपड़ा', 'B. R. Chopra')]


In [ ]:
for hindi, english in phrase_data.items():
    print(f"{hindi}  →  {english}")

इंसाफ़ का तराज़ू  →  Insaf Ka Tarazu
छोटी सी बात  →  Chhoti Si Baat
नया दौर  →  Naya Daur
वक़्त  →  Waqt
साधना  →  Sadhana
गुमराह  →  Gumrah
हमराज़  →  Humraaz
क़ानून  →  Kanoon
निकाह  →  Nikaah
बीआर चोपड़ा  →  B. R. Chopra
यश चोपड़ा  →  Yash Chopra
रवि चोपड़ा  →  Ravi Chopra
राज बब्बर  →  Raj Babbar
बसु चटर्जी  →  Basu Chatterjee
बॉबी बेदी  →  Bobby Bedi
एक था टाइगर  →  Ek Tha Tiger
सलमान खान  →  Salman Khan
कटरीना कैफ  →  Katrina Kaif
दबंग  →  Dabangg
बॉडीगार्ड  →  Bodyguard
सूरज बड़जात्या  →  Sooraj Barjatya
मैंने प्यार किया  →  Maine Pyar Kiya
हम आपके हैं कौन  →  Hum Aapke Hain Koun
कबीर खान  →  Kabir Khan
वॉन्टेड  →  Wanted
रेडी  →  Ready


In [ ]:
phrase_counts = []

for _, row in mixed_df.iterrows():
    article = str(row["text"])

    count = sum(
        article.count(hindi)
        for hindi in phrase_data.keys()
    )

    phrase_counts.append(count)

print("Articles:", len(phrase_counts))
print("Articles with matching phrases:", sum(c > 0 for c in phrase_counts))
print("Articles without matching phrases:", sum(c == 0 for c in phrase_counts))
print("Total phrase matches:", sum(phrase_counts))
print("Average matches per article:", round(sum(phrase_counts) / len(phrase_counts), 2))

Articles: 250
Articles with matching phrases: 54
Articles without matching phrases: 196
Total phrase matches: 143
Average matches per article: 0.57


In [ ]:
print("Original Hindi article:")
print(mixed_df["text"].iloc[0][:700])

print("\nEnglish translation:")
print(mixed_df["english_text"].iloc[0][:700])

print("\nHindi word count:",
      len(str(mixed_df["text"].iloc[0]).split()))

print("English word count:",
      len(str(mixed_df["english_text"].iloc[0]).split()))

Original Hindi article:
शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी." बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर 

English translation:
On Friday evening, the scales of justice were shown in the Polish film. The little thing, the new rounds, and time during the celebration, will be seen to see the classical films. Bad health because of BRC's own function, BRCP's brother and famous filme

In [ ]:
for i in range(5):
    print("\n" + "="*70)
    print("ARTICLE", i + 1)
    print("\nHINDI:")
    print(str(mixed_df["text"].iloc[i])[:350])
    print("\nENGLISH:")
    print(str(mixed_df["english_text"].iloc[i])[:350])


ARTICLE 1

HINDI:
शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में

ENGLISH:
On Friday evening, the scales of justice were shown in the Polish film. The little thing, the new rounds, and time during the celebration, will be seen to see the classical films. Bad health because of BRC's own function, BRCP's brother and famous filmer, he said, "My brother made his business at such a time. When people were apprehensive about the

ARTICLE 2

HINDI:
कंधार के प्रांतीय सभा के अध्यक्ष और राष्ट्रपति हामिद करज़ई के भाई वली करज़ई ने बीबीसी को बताया कि उस समय जेल में एक हज़ार क़ैदी थे और सभी निकल भागे हैं. माना जाता है कि इन क़ैदियों में से 350 तालेबान थे. इस हमले में कई सुरक्षाकर्मियों की जानें गई हैं

In [ ]:
import os
import shutil
from datetime import datetime

backup_dir = os.path.join(BASE, "09_checkpoints", "failed_experiments")
os.makedirs(backup_dir, exist_ok=True)

files_to_backup = [
    os.path.join(BASE, "04_mixed_data/FactIndic_Mixed_250.csv"),
    os.path.join(BASE, "03_english_data/FactIndic_Hindi_English_Base_250.csv")
]

for path in files_to_backup:
    if os.path.exists(path):
        destination = os.path.join(
            backup_dir,
            os.path.basename(path)
        )
        shutil.copy2(path, destination)
        print("Backed up:", destination)
    else:
        print("Not found:", path)

Backed up: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_checkpoints/failed_experiments/FactIndic_Mixed_250.csv
Backed up: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_checkpoints/failed_experiments/FactIndic_Hindi_English_Base_250.csv


In [ ]:
import os
import pandas as pd

review_path = os.path.join(
    BASE,
    "06_real_hinglish/FactIndic_Hinglish_95_Human_Review.csv"
)

review_df = pd.read_csv(review_path)

print("Total candidates:", len(review_df))
print("Columns:", review_df.columns.tolist())

# Select 20 records for initial manual review
sample_review = review_df.sample(
    n=min(20, len(review_df)),
    random_state=42
).copy()

sample_review_path = os.path.join(
    BASE,
    "06_real_hinglish/FactIndic_Hinglish_20_Review.csv"
)

sample_review.to_csv(
    sample_review_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nReview sample saved:", sample_review_path)
print("Records to review:", len(sample_review))

Total candidates: 95
Columns: ['source_dataset', 'source_config', 'text', 'script_type', 'selection_method', 'reply_to_text', 'likely_hinglish', 'human_verified', 'language_quality', 'meaning_clear', 'review_notes', 'latin_word_count', 'total_word_count', 'latin_word_percentage', 'latin_words', 'devanagari_words', 'roman_hindi_words', 'total_language_words', 'verified_hinglish', 'english_words_used_naturally']

Review sample saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/06_real_hinglish/FactIndic_Hinglish_20_Review.csv
Records to review: 20


In [ ]:
for i, row in sample_review.reset_index(drop=True).iterrows():
    print("\n" + "="*65)
    print("RECORD:", i + 1)
    print("SOURCE:", row["source_dataset"])
    print("CONFIG:", row["source_config"])
    print("TEXT:", row["text"])

    if pd.notna(row["reply_to_text"]) and str(row["reply_to_text"]).strip():
        print("REPLY:", row["reply_to_text"])


RECORD: 1
SOURCE: GupShup
CONFIG: nan
TEXT: Haan kuchh log seekhte hi ni
Jo mili woh jigri thi ni, mile jo khushi woh tikti bhi ni

RECORD: 2
SOURCE: MUTANT
CONFIG: Mankibaat
TEXT: तीसरे पत्र, लेखकों का ग्रुप ऐसा है, वे भी बहुत बड़ी संख्या में हैंI वो कहते हैं मोदी जी थक मत जाना, रुक मत जाना और जितना कठोर कदम उठा सकते हो, उठाओ, लेकिन अब एक बार रास्ता पकड़ा है तो मंजिल तक पहुंचना ही हैI मैं ऐसे पत्र लिखने वाले सब को विशेष रूप से धन्यवाद करता हूँ क्योंकि उनके पत्र में एक प्रकार से विश्वास भी है, आशीर्वाद भी हैI मैं आपको विश्वास दिलाता हूँ कि ये पूर्ण विराम नहीं है, ये तो अभी शुरुआत है, ये जंग जीतना है और थकने का तो सवाल ही कहाँ उठता है, रुकने का तो सवाल ही नहीं उठता है और जिस बात पर सवा-सौ करोड़ देशवासियों का आशीर्वाद हो, उसमें तो पीछे हटने का कोई प्रश्न ही नहीं उठता हैI आपको मालूम होगा हमारे देश में ‘बेनामी संपत्ति’ का एक कानून हैI Nineteen Eighty Eight, उन्नीस सौ अठास्सी में बना था, लेकिन कभी भी न उसके Rules बनें, उसको Notify नहीं किया, ऐसे ही वो ठंडे बस्ते में पड़ा रहाI हमनें उसको निकाला

In [ ]:
decisions = {
    1: ("No", "No", "Yes", "Romanized Hindi; no clear English mixing"),
    2: ("Yes", "Yes", "Yes", "Hindi-English mixed speech"),
    3: ("Yes", "Yes", "Yes", "Borderline: only one clear English word"),
    4: ("Yes", "Yes", "Yes", "English abbreviations and phrases"),
    5: ("Yes", "Yes", "Yes", "Natural English technical terms"),
    6: ("Yes", "Yes", "Yes", "Hindi greeting followed by English"),
    7: ("Yes", "Yes", "Yes", "English terms and phrases"),
    8: ("Yes", "Yes", "Yes", "Scientific and technical English terms"),
    9: ("Yes", "Yes", "Yes", "Natural English words and phrases"),
    10: ("Yes", "Yes", "Yes", "Health-related English terminology"),
    11: ("Yes", "Yes", "Yes", "Romanized Hindi-English mixing"),
    12: ("Yes", "Yes", "Yes", "English internet terminology"),
    13: ("Yes", "Yes", "Yes", "English phrase within Hindi"),
    14: ("Yes", "Yes", "Yes", "Hindi-English mixed speech"),
    15: ("Yes", "Yes", "Yes", "English technology and social-media terms"),
    16: ("Yes", "Yes", "Yes", "English administrative terminology"),
    17: ("Yes", "Yes", "Yes", "English names and expressions"),
    18: ("Yes", "Yes", "Yes", "English technical and environmental terms"),
    19: ("Yes", "Yes", "Yes", "English conceptual terminology"),
    20: ("Yes", "Yes", "Yes", "Romanized Hindi-English mixing")
}

for idx, decision in decisions.items():
    verified, natural, clear, notes = decision
    sample_review.loc[sample_review.index[idx - 1],
                      "verified_hinglish"] = verified
    sample_review.loc[sample_review.index[idx - 1],
                      "english_words_used_naturally"] = natural
    sample_review.loc[sample_review.index[idx - 1],
                      "meaning_clear"] = clear
    sample_review.loc[sample_review.index[idx - 1],
                      "review_notes"] = notes

sample_review.to_csv(
    sample_review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved reviewed sample:", len(sample_review))
print("Likely Hinglish:", sum(
    v[0] == "Yes" for v in decisions.values()
))
print("Not Hinglish:", sum(
    v[0] == "No" for v in decisions.values()
))

Saved reviewed sample: 20
Likely Hinglish: 19
Not Hinglish: 1


/tmp/ipykernel_4396/4235496162.py:26: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'No' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  sample_review.loc[sample_review.index[idx - 1],
/tmp/ipykernel_4396/4235496162.py:28: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'No' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  sample_review.loc[sample_review.index[idx - 1],
/tmp/ipykernel_4396/4235496162.py:30: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Yes' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  sample_review.loc[sample_review.index[idx - 1],
/tmp/ipykernel_4396/4235496162.py:32: FutureWarning: Setting an item 

In [ ]:
review_columns = [
    "verified_hinglish",
    "english_words_used_naturally",
    "meaning_clear",
    "review_notes"
]

for col in review_columns:
    sample_review[col] = sample_review[col].astype("object")

sample_review.to_csv(
    sample_review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Review columns corrected.")
print(sample_review[review_columns].dtypes)
print("Saved successfully:", os.path.exists(sample_review_path))

Review columns corrected.
verified_hinglish               object
english_words_used_naturally    object
meaning_clear                   object
review_notes                    object
dtype: object
Saved successfully: True


In [ ]:
remaining_df = review_df.drop(index=sample_review.index)

sample_review_2 = remaining_df.sample(
    n=min(20, len(remaining_df)),
    random_state=43
).copy()

sample_review_2_path = os.path.join(
    BASE,
    "06_real_hinglish/FactIndic_Hinglish_Review_21_40.csv"
)

sample_review_2.to_csv(
    sample_review_2_path,
    index=False,
    encoding="utf-8-sig"
)

print("Next review sample:", len(sample_review_2))
print("Remaining candidates:", len(remaining_df) - len(sample_review_2))
print("Saved:", sample_review_2_path)

Next review sample: 20
Remaining candidates: 55
Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/06_real_hinglish/FactIndic_Hinglish_Review_21_40.csv


In [ ]:
pd.set_option("display.max_colwidth", 500)
pd.set_option("display.max_columns", None)

display(sample_review_2)


,source_dataset,source_config,text,script_type,selection_method,reply_to_text,likely_hinglish,human_verified,language_quality,meaning_clear,review_notes,latin_word_count,total_word_count,latin_word_percentage,latin_words,devanagari_words,roman_hindi_words,total_language_words,verified_hinglish,english_words_used_naturally
71,GupShup,NaN,Kyaa hua pookie ji ne kal raatko i lammu nhi bolaa kya [EMOTE_3x] [EMOTE_3x] [EMOTE_3x],Roman-script candidate,Non-empty text sample,Fr,True,NaN,NaN,NaN,NaN,12,15,80.00,12,0,2,12,NaN,NaN
31,MUTANT,Mankibaat,"मेरे प्यारे देशवासियो, आधुनिक होने की परिभाषाएँ बदलती चली जा रही हैं | इन दिनों एक नया dimension, एक नया parameter, आप कितने संस्कारी हो, कितने आधुनिक हो, आपकी thought-process कितनी modern है, ये सब जानने में एक तराजू भी काम में आने लगा है और वो है environment के प्रति आप कितने सजग हैं | आपके अपनी गतिविधियों में eco-friendly, environment-friendly व्यवहार है कि उसके खिलाफ़ है | समाज में अगर उसके ख़िलाफ़ है तो आज बुरा माना जाता है | और उसी का परिणाम आज मैं देख रहा हूँ कि इन दिनों ये गणेशोत्सव में...",Mixed Devanagari-Latin,Script-based candidate filter,NaN,True,NaN,NaN,NaN,NaN,29,396,7.32,29,348,0,377,NaN,NaN
90,GupShup,NaN,[MENTION] koi problem hai bhai server se 💁 ya fir rules se,Roman-script candidate,Non-empty text sample,NaN,True,NaN,NaN,NaN,NaN,11,12,91.67,10,0,2,10,NaN,NaN
44,MUTANT,Mankibaat,"श्रीमान योगेश भद्रेशा जी, उनका कहना है कि मैं इस बार युवाओं से उनके स्वास्थ्य के बारे में बात करूँ। उन्हें लगता है कि Asian countries में तुलना करें तो हमारे युवा physically weak हैं। योगेश जी, मैंने सोचा है कि इस बार health को लेकर के सभी के साथ विस्तार से बात करूँ -Fit India की बात करूँ। और आप सब नौजवान मिल करके Fit India काmovement भी चला सकते हैं।",Mixed Devanagari-Latin,Script-based candidate filter,NaN,True,NaN,NaN,NaN,NaN,10,72,13.89,10,63,0,73,NaN,NaN
17,MUTANT,Mankibaat,"एक बात हम सब भली-भांति जानते हैं, हिन्दुस्तान के किसी भी कोने में चले जाइए, अमीर-से-अमीर के घर में चले जाइए, ग़रीब-से-ग़रीब की झोपड़ी में चले जाइए, दिवाली के त्योहार में, हर परिवार में स्वच्छता का अभियान चलता दिखता है। घर के हर कोने की सफ़ाई होती है। ग़रीब अपने मिट्टी के बर्तन होंगे, तो मिट्टी के बर्तन भी ऐसे साफ़ करते हैं, जैसे बस ये दिवाली आयी है। दिवाली एक स्वच्छता का अभियान भी है। लेकिन, समय की माँग है कि सिर्फ़ घर में सफ़ाई नहीं, पूरे परिसर की सफ़ाई, पूरे मोहल्ले की सफ़ाई, पूरे गाँव की सफ़ाई, हमने...",Mixed Devanagari-Latin,Script-based candidate filter,NaN,True,NaN,NaN,NaN,NaN,11,234,4.70,11,231,0,242,NaN,NaN
1,MUTANT,Mankibaat,"Start-up के संबंध में हमारे यहाँ एक सोच बंधी-बंधाई बन गयी है। जैसे digital world हो या IT profession हो ये start-up उन्हीं के लिए है! जी नहीं, हमें तो उसको भारत की आवश्यकताओं के अनुसार बदलाव लाना है। ग़रीब व्यक्ति कहीं मजदूरी करता है, उसको शारीरिक श्रम पड़ता है, लेकिन कोई नौजवान Innovation के द्वारा एक ऐसी चीज़ बना दे कि ग़रीब को मज़दूरी में थोड़ी सुविधा हो जाये। मैं इसको भी Start-up मानता हूँ। मैं बैंक को कहूँगा कि ऐसे नौजवान को मदद करो, मैं उसको भी कहूँगा कि हिम्मत से आगे बढ़ो। Market मिल जायेगा।...",Mixed Devanagari-Latin,Script-based candidate filter,NaN,True,NaN,NaN,NaN,NaN,18,192,9.38,18,180,0,198,NaN,NaN
20,MUTANT,Mankibaat,"मेरे प्यारे देशवासियो, मुझे ये जान करके ख़ुशी होती है कि देश में technology का उपयोग कैसे करना, e-payment कैसे करना, online payment कैसे करना, इसकी जागरूकता बहुत तेज़ी से बढ़ रही है। पिछले कुछ ही दिनों में cashless कारोबार, बिना नगद का कारोबार, 200 से 300 प्रतिशत बढ़ा है। इसको बढ़ावा देने के लिये भारत सरकार ने एक बहुत बड़ा फैसला लिया है। ये फैसला कितना बड़ा है इसका अंदाज़ तो व्यापारी बहुत अच्छी तरह लगा सकते हैं। जो व्यापारी digital लेन-देन करेंगे, अपने कारोबार में नगद के बज़ाय online payment की पद्धत...",Mixed Devanagari-Latin,Script-based candidate filter,NaN,True,NaN,NaN,NaN,NaN,11,107,10.28,11,96,0,107,NaN,NaN
53,GupShup,NaN,Ungli pakadne diye toh haath mat kheecho tum [EMOTE_nikal],Roman-script candidate,Non-empty text sample,Bache ko har din godi lete hai[EMOTE_kyahua],True,NaN,NaN,NaN,NaN,8,9,88.89,8,0,2,8,NaN,NaN
51,GupShup,NaN,[EMOTE_zzz_pepekek] tere kehna ka mtlb mai acha nhi dikhta

In [ ]:
decisions_2 = {
    1: ("No", "No", "Yes", "Mostly Romanized Hindi; pookie is only a borrowed slang term"),
    2: ("Yes", "Yes", "Yes", "Natural English terms mixed with Hindi"),
    3: ("Yes", "Yes", "Yes", "English words problem, server and rules"),
    4: ("Yes", "Yes", "Yes", "Natural English health-related terms"),
    5: ("Yes", "Yes", "Yes", "English words present in the full candidate"),
    6: ("Yes", "Yes", "Yes", "English technology and payment terms"),
    7: ("No", "No", "Yes", "Romanized Hindi without clear English mixing"),
    8: ("No", "No", "Yes", "Romanized Hindi without clear English mixing"),
    9: ("Yes", "Yes", "Yes", "English terms mixed with Hindi"),
    10: ("Yes", "Yes", "Yes", "English terms viva and PC"),
    11: ("No", "No", "Yes", "Romanized Hindi without clear English mixing"),
    12: ("Yes", "Yes", "Yes", "English phrase and members mixed with Hindi"),
    13: ("No", "No", "Yes", "Mostly Romanized Hindi"),
    14: ("No", "No", "Yes", "Romanized Hindi without clear English mixing"),
    15: ("No", "No", "Yes", "Romanized Hindi without clear English mixing"),
    16: ("No", "No", "Yes", "Romanized Hindi without clear English mixing"),
    17: ("Yes", "Yes", "Yes", "English sports terms mixed with Hindi"),
    18: ("No", "No", "Yes", "Romanized Hindi without clear English mixing"),
    19: ("Yes", "Yes", "Yes", "English terms mixed with Hindi"),
    20: ("Yes", "Yes", "Yes", "English words mixed with Hindi")
}

for pos, decision in decisions_2.items():
    verified, natural, clear, notes = decision

    for col in [
        "verified_hinglish",
        "english_words_used_naturally",
        "meaning_clear",
        "review_notes"
    ]:
        sample_review_2[col] = sample_review_2[col].astype("object")

    sample_review_2.iloc[
        pos - 1,
        sample_review_2.columns.get_loc("verified_hinglish")
    ] = verified

    sample_review_2.iloc[
        pos - 1,
        sample_review_2.columns.get_loc("english_words_used_naturally")
    ] = natural

    sample_review_2.iloc[
        pos - 1,
        sample_review_2.columns.get_loc("meaning_clear")
    ] = clear

    sample_review_2.iloc[
        pos - 1,
        sample_review_2.columns.get_loc("review_notes")
    ] = notes

sample_review_2.to_csv(
    sample_review_2_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved reviewed records:", len(sample_review_2))
print("Likely Hinglish:", (sample_review_2["verified_hinglish"] == "Yes").sum())
print("Not Hinglish:", (sample_review_2["verified_hinglish"] == "No").sum())
print("File saved:", os.path.exists(sample_review_2_path))

Saved reviewed records: 20
Likely Hinglish: 11
Not Hinglish: 9
File saved: True


In [ ]:
reviewed_40 = pd.concat(
    [sample_review, sample_review_2]
)

review_columns = [
    "verified_hinglish",
    "english_words_used_naturally",
    "meaning_clear",
    "review_notes"
]

for col in review_columns:
    review_df[col] = review_df[col].astype("object")

for col in review_columns:
    review_df.loc[reviewed_40.index, col] = reviewed_40[col]

master_path = os.path.join(
    BASE,
    "06_real_hinglish/FactIndic_Hinglish_95_Human_Review.csv"
)

review_df.to_csv(
    master_path,
    index=False,
    encoding="utf-8-sig"
)

print("Master review file updated.")
print("Total candidates:", len(review_df))
print("Reviewed:", review_df["verified_hinglish"].notna().sum())
print("Remaining:", review_df["verified_hinglish"].isna().sum())
print("Likely Hinglish:", (review_df["verified_hinglish"] == "Yes").sum())
print("Not Hinglish:", (review_df["verified_hinglish"] == "No").sum())
print("Saved:", os.path.exists(master_path))

Master review file updated.
Total candidates: 95
Reviewed: 40
Remaining: 55
Likely Hinglish: 30
Not Hinglish: 10
Saved: True


In [ ]:
reviewed_indices = set(sample_review.index) | set(sample_review_2.index)

remaining_df = review_df.loc[
    ~review_df.index.isin(reviewed_indices)
].copy()

sample_review_3 = remaining_df.sample(
    n=min(20, len(remaining_df)),
    random_state=44
).copy()

sample_review_3_path = os.path.join(
    BASE,
    "06_real_hinglish/FactIndic_Hinglish_Review_41_60.csv"
)

sample_review_3.to_csv(
    sample_review_3_path,
    index=False,
    encoding="utf-8-sig"
)

print("Next review sample:", len(sample_review_3))
print("Candidates still unreviewed:", len(remaining_df) - len(sample_review_3))
print("Saved:", sample_review_3_path)

Next review sample: 20
Candidates still unreviewed: 35
Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/06_real_hinglish/FactIndic_Hinglish_Review_41_60.csv


In [ ]:
display(sample_review_3)

,source_dataset,source_config,text,script_type,selection_method,reply_to_text,likely_hinglish,human_verified,language_quality,meaning_clear,review_notes,latin_word_count,total_word_count,latin_word_percentage,latin_words,devanagari_words,roman_hindi_words,total_language_words,verified_hinglish,english_words_used_naturally
74,GupShup,NaN,[MENTION] wo pta hai...but aap k aane par sab competition finish wo bola maine,Roman-script candidate,Non-empty text sample,NaN,True,NaN,NaN,NaN,NaN,15,14,107.14,14,0,2,14,NaN,NaN
13,MUTANT,Mankibaat,"अभी-अभी जब मेरी सरकार के 2 साल पूरे हुए, तो कुछ आधुनिक विचार वाले नौजवानों ने मुझे सुझाव दिया कि आप इतनी बड़ी लोकतंत्र की बातें करते हैं, तो क्यों न आप अपनी सरकार का मूल्यांकन लोगों से करवाएँ। वैसे एक प्रकार से उनका चुनौती का ही स्वर था, सुझाव का भी स्वर था। लेकिन उन्होंने मेरे मन को झकझोर दिया। मैंने कुछ अपने वरिष्ठ साथियों के बीच में ये विषय रखा, तो प्रथम प्रतिक्रिया तो reaction ऐसा ही था कि नहीं-नहीं जी साहब, ये आप क्या करने जा रहे हो? आज तो technology इतनी बदल चुकी है कि अगर कोई इकट्ठे हो...",Mixed Devanagari-Latin,Script-based candidate filter,NaN,True,NaN,NaN,NaN,NaN,23,462,4.98,23,445,0,468,NaN,NaN
58,GupShup,NaN,"Baki chill hu, bas hota h koi event khtm hone ke baad aisa, toh kya kar sakti. M bhi bas ek baat keh rhi thi, uss sense m nhi bola kuch bhi mne ki m chilla rhi",Roman-script candidate,Non-empty text sample,Wahi lgra hai kyuki maine kuch waisa bola bhi nh hai [EMOTE_GB_DogeShocked],True,NaN,NaN,NaN,NaN,37,37,100.00,37,0,11,37,NaN,NaN
70,GupShup,NaN,[MENTION][SERVER] ka gaming tatti wo bas pubgm khelta hai,Roman-script candidate,Non-empty text sample,NaN,True,NaN,NaN,NaN,NaN,10,9,111.11,8,0,3,8,NaN,NaN
11,MUTANT,Mankibaat,"हमारे देश में जैसे किसान मेहनत करता है, हमारे वैज्ञानिक भी देश को नई ऊंचाइयों पर ले जाने के लिए बहुत सफलताएँ प्राप्त कर रहे हैं। और मेरा तो पहले से मत रहा है कि हमारी नई पीढ़ी वैज्ञानिक बनने के सपने देखे, विज्ञान में रूचि ले, आने वाली पीढ़ियों के लिये कुछ कर गुजरने की इच्छा के साथ हमारी युवा पीढ़ी आगे आए। मैं आज और भी एक खुशी की बात आपसे share करना चाहता हूँ। कल मैं पुणे गया था, Smart City Project की वर्षगाँठ के उपलक्ष्य में वहाँ कार्यक्रम था और वहाँ मैंने पुणे के College of Engineering के जिन ...",Mixed Devanagari-Latin,Script-based candidate filter,NaN,True,NaN,NaN,NaN,NaN,26,455,5.71,26,424,0,450,NaN,NaN
60,GupShup,NaN,"sed fr\nbas next year ka wait hai, feb main anime remake, april main anime return, after kuch months live action s3",Roman-script candidate,Non-empty text sample,[EMOTE_pikachu_sad],True,NaN,NaN,NaN,NaN,21,22,95.45,21,0,4,21,NaN,NaN
61,GupShup,NaN,[MENTION] lol meko Samsung ka lena bhi nhi hai xD,Roman-script candidate,Non-empty text sample,NaN,True,NaN,NaN,NaN,NaN,10,10,100.00,9,0,4,9,NaN,NaN
67,GupShup,NaN,[MENTION] endgame spoiler du? \nhkm server me tune captain Marvel ka spoiler diya tha,Roman-script candidate,Non-empty text sample,NaN,True,NaN,NaN,NaN,NaN,14,14,100.00,13,0,3,13,NaN,NaN
43,MUTANT,Mankibaat,"Light Bulb का आविष्कार करने वाले Thomas Alva Edison अपने प्रयोगों में कई बार असफ़ल रहे। एक बार उनसे जब इस बारे में पूछा गया तो उन्होंने जवाब दिया –“मैंने Light Bulb नहीं बनाने के दस हज़ार तरीक़े खोज़े हैं”, यानि Edison ने अपनी असफलताओं को भी अपनी शक्ति बना लिया। संयोग से सौभाग्य है कि आज मैं महर्षि अरबिन्दो की कर्मभूमि ‘Auroville’ में हूँ। एक क्रांतिकारी के रूप में उन्होंने ब्रिटिश शासन को चुनौती दी, उनके ख़िलाफ़ लड़ाई लड़ी, उनके शासन पर सवाल उठाए। इसी प्रकार उन्होंने एक महान ऋषि के रूप में, जीवन के...",Mixed Devanagari-Latin,Script-based candidate filter,NaN,True,NaN,NaN,NaN,NaN,12,200,6.00,12,191,0,203,NaN,NaN
32,MUTANT,Mankibaat,"मेरे प्यारे देशवासियो, मैं आप सभी से एक आह्वान करता हूँ कि एक बार फिर 2 अक्टूबर गाँधी जयंती से 15-20 दिन पहले से ही ‘स्वच्छता ही सेवा’ – जैसे पहले कहते थे ‘जल सेवा यही प्रभु सेवा’, ‘स्वच्छता ही सेवा’ की एक मुहिम चलायें | पूरे देश में स्वच्छता के लिए माहौल बनाएं | जैसा अवसर मिले, जहाँ भी अवसर मिले, हम अवसर ढूंढें | लेकिन हम सभी जुड़ें | इसे एक प्रकार से दिवाली की तैयारी मान लें, इसे एक प्रकार से नवरात्र की तैयारी मान लें, 

In [ ]:
decisions_3 = {
    1: ("No", "No", "Yes", "Mostly Romanized Hindi"),
    2: ("Yes", "Yes", "Yes", "English words mixed with Hindi"),
    3: ("Yes", "Yes", "Yes", "Natural English words and phrases"),
    4: ("Yes", "Yes", "Yes", "English gaming terminology"),
    5: ("Yes", "Yes", "Yes", "English technical terms"),
    6: ("Yes", "Yes", "Yes", "English anime-related phrases"),
    7: ("Yes", "Yes", "Yes", "English slang and brand reference"),
    8: ("Yes", "Yes", "Yes", "English words and phrases"),
    9: ("Yes", "Yes", "Yes", "English names and terms"),
    10: ("No", "No", "Yes", "No clear English mixing in visible text"),
    11: ("No", "No", "Yes", "Mostly Romanized Hindi"),
    12: ("Yes", "Yes", "Yes", "English app names and terms"),
    13: ("Yes", "Yes", "Yes", "English administrative terminology"),
    14: ("Yes", "Yes", "Yes", "English words mixed with Hindi"),
    15: ("Yes", "Yes", "Yes", "English financial terminology"),
    16: ("No", "No", "Yes", "No clear English mixing in visible text"),
    17: ("Yes", "Yes", "Yes", "English technical terms"),
    18: ("Yes", "Yes", "Yes", "English computer terminology"),
    19: ("No", "No", "Yes", "Mostly Romanized Hindi"),
    20: ("No", "No", "Yes", "Mostly Romanized Hindi")
}

for pos, decision in decisions_3.items():
    for col in [
        "verified_hinglish",
        "english_words_used_naturally",
        "meaning_clear",
        "review_notes"
    ]:
        sample_review_3[col] = sample_review_3[col].astype("object")

    for col, value in zip(
        ["verified_hinglish", "english_words_used_naturally",
         "meaning_clear", "review_notes"],
        decision
    ):
        sample_review_3.iloc[
            pos - 1, sample_review_3.columns.get_loc(col)
        ] = value

sample_review_3.to_csv(
    sample_review_3_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", len(sample_review_3))
print("Likely Hinglish:",
      (sample_review_3["verified_hinglish"] == "Yes").sum())
print("Not Hinglish:",
      (sample_review_3["verified_hinglish"] == "No").sum())

Saved: 20
Likely Hinglish: 14
Not Hinglish: 6


In [ ]:
# Add batch 3 to the master review sheet
review_columns = [
    "verified_hinglish",
    "english_words_used_naturally",
    "meaning_clear",
    "review_notes"
]

for col in review_columns:
    review_df[col] = review_df[col].astype("object")

for col in review_columns:
    review_df.loc[sample_review_3.index, col] = sample_review_3[col]

review_df.to_csv(
    master_path,
    index=False,
    encoding="utf-8-sig"
)

print("Master file updated:", os.path.exists(master_path))
print("Total candidates:", len(review_df))
print("Reviewed:", review_df["verified_hinglish"].notna().sum())
print("Remaining:", review_df["verified_hinglish"].isna().sum())
print("Likely Hinglish:", (review_df["verified_hinglish"] == "Yes").sum())
print("Not Hinglish:", (review_df["verified_hinglish"] == "No").sum())

Master file updated: True
Total candidates: 95
Reviewed: 60
Remaining: 35
Likely Hinglish: 44
Not Hinglish: 16


In [ ]:
reviewed_indices = (
    set(sample_review.index)
    | set(sample_review_2.index)
    | set(sample_review_3.index)
)

remaining_df = review_df.loc[
    ~review_df.index.isin(reviewed_indices)
].copy()

sample_review_4 = remaining_df.sample(
    n=min(20, len(remaining_df)),
    random_state=45
).copy()

sample_review_4_path = os.path.join(
    BASE,
    "06_real_hinglish/FactIndic_Hinglish_Review_61_80.csv"
)

sample_review_4.to_csv(
    sample_review_4_path,
    index=False,
    encoding="utf-8-sig"
)

print("Next review sample:", len(sample_review_4))
print("Still remaining:", len(remaining_df) - len(sample_review_4))
print("Saved:", sample_review_4_path)

display(sample_review_4)

Next review sample: 20
Still remaining: 15
Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/06_real_hinglish/FactIndic_Hinglish_Review_61_80.csv


,source_dataset,source_config,text,script_type,selection_method,reply_to_text,likely_hinglish,human_verified,language_quality,meaning_clear,review_notes,latin_word_count,total_word_count,latin_word_percentage,latin_words,devanagari_words,roman_hindi_words,total_language_words,verified_hinglish,english_words_used_naturally
46,MUTANT,Mankibaat,"मैं मनिका बत्रा जो Commonwealth में चार medal लायी हूँ। दो Gold, एक Silver, एक Bronze। ‘मन की बात’ programme सुनने वालों को मैं बताना चाहती हूँ कि मैं बहुत खुश हूँ क्योंकि पहली बार India में table tennis इतना popular हो रहा है। हाँ मैंने अपना best table tennis खेला होगा। पूरे life का best table tennis खेला होगा। जो उससे पहले मैंने practice करी है उसके बारे में मैं बताऊँगी कि मैंने बहुत, अपने coach संदीप sir के साथ practice करी है। Commonwealth से पहले जो हमारे camps थे Portugal में, हमें tou...",Mixed Devanagari-Latin,Script-based candidate filter,NaN,True,NaN,NaN,NaN,NaN,38,128,29.69,38,91,0,129,NaN,NaN
93,GupShup,NaN,> [NAME] kya lagake rakha hai????\n[MENTION] it's coronil,Roman-script candidate,Non-empty text sample,NaN,True,NaN,NaN,NaN,NaN,9,9,100.00,7,0,2,7,NaN,NaN
78,GupShup,NaN,Son:- papa mai paida kaise hua ...\nFather:- bhout lambi kahani hai....\nSon. :- short me bata do fir....\nFather:- Medical shop bnd thi[EMOTE_zzz_hashnanoob],Roman-script candidate,Non-empty text sample,NaN,True,NaN,NaN,NaN,NaN,22,24,91.67,22,0,4,22,NaN,NaN
15,MUTANT,Mankibaat,"मेरे प्यारे देशवासियों, वैसे जब भी ‘मन की बात’ का समय आता है, तो MyGov पर या NarendraModiApp पर अनेकों-अनेक सुझाव आते हैं। विविधता से भरे हुए होते हैं, लेकिन मैंने देखा कि इस बार तो ज्यादातर, हर किसी ने मुझे आग्रह किया कि Rio Olympic के संबंध में आप ज़रूर कुछ बातें करें। सामान्य नागरिक का Rio Olympic के प्रति इतना लगाव, इतनी जागरूकता और देश के प्रधानमंत्री पर दबाव करना कि इस पर कुछ बोलो, मैं इसको एक बहुत सकारात्मक देख रहा हूँ। क्रिकेट के बाहर भी भारत के नागरिकों में और खेलों के प्रति भी इतना ...",Mixed Devanagari-Latin,Script-based candidate filter,NaN,True,NaN,NaN,NaN,NaN,10,267,3.75,10,257,0,267,NaN,NaN
37,MUTANT,Mankibaat,"ऐसे अनेक लोग हैं जो अपने-अपने स्तर पर ऐसे कार्य कर रहे हैं जिनसे कई लोगों के जीवन में सकारात्मक बदलाव आ रहा है। वास्तव में, यही तो ‘New India’ है जिसका हम सब मिल कर निर्माण कर रहे हैं। आइए, इन्हीं छोटी-छोटी खुशियों के साथ हम नव-वर्ष में प्रवेश करें, नव-वर्ष की शुरूआत करें और ‘positive India’ से ‘progressive India’ की दिशा में मजबूत कदम बढ़ाएँ। जब हम सब positivity की बात करते हैं तो मुझे भी एक बात share करने का मन करता है। हाल ही में मुझे कश्मीर के प्रशासनिक सेवा के topper अंजुम बशीर खान खट्टक...",Mixed Devanagari-Latin,Script-based candidate filter,NaN,True,NaN,NaN,NaN,NaN,17,281,6.05,17,266,0,283,NaN,NaN
30,MUTANT,Mankibaat,"GST के लागू हुए क़रीब एक महीना हुआ है और उसके फ़ायदे दिखने लगे हैं। और मुझे बहुत संतोष होता है, खुशी होती है, जब कोई ग़रीब मुझे चिट्ठी लिख करके कहता है कि GST के कारण एक ग़रीब की ज़रुरत की चीज़ों में कैसे दाम कम हुए हैं, चीज़ें कैसे सस्ती हुई हैं। अगर North-East, दूर-सुदूर पहाड़ों में, जंगलों में रहने वाला कोई व्यक्ति चिट्ठी लिखता है कि शुरू में डर लगता था कि पता नहीं क्या है; लेकिन अब जब मैं उसमें सीखने-समझने लगा, तो मुझे लगता है, पहले से ज़्यादा आसान हो गया काम। व्यापार और आसान हो गया। और सबसे बड़ी ...",Mixed Devanagari-Latin,Script-based candidate filter,NaN,True,NaN,NaN,NaN,NaN,62,615,10.08,62,563,0,625,NaN,NaN
94,GupShup,NaN,[MENTION]apna wo thor bring me thanos wala bhi hai,Roman-script candidate,Non-empty text sample,NaN,True,NaN,NaN,NaN,NaN,10,9,111.11,9,0,5,9,NaN,NaN
19,MUTANT,Mankibaat,"“नमस्कार प्रधानमंत्री जी, मैं शिवानी मोहन बोल रही हूँ। इस दीपावली पर जो ‘Sandesh to Soldiers’ अभियान शुरू किया गया है, उससे हमारे फ़ौजी भाइयों को बहुत ही प्रोत्साहन मिल रहा है। मैं एक Army Family से हूँ। मेरे पति भी Army ऑफिसर हैं। मेरे Father और Father-in-law, दोनों Army Officers रह चुके हैं। तो हमारी तो पूरी family soldiers से भरी हुई है और सीमा पर हमारे कई ऐसे Officers हैं, जिनको इतने अच्छे संदेश मिल रहे हैं और बहुत प्रोत्साहन मिल रहा है Army Circle में सभी को। और मैं कहना चाहूँगी कि Army ...",Mixed Devanag

In [ ]:
decisions_4 = {
    1: ("Yes", "Yes", "Yes", "English sports terminology"),
    2: ("Yes", "Yes", "Yes", "English phrase"),
    3: ("Yes", "Yes", "Yes", "English words and phrases"),
    4: ("Yes", "Yes", "Yes", "English Olympic and sports terms"),
    5: ("Yes", "Yes", "Yes", "Natural English expressions"),
    6: ("Yes", "Yes", "Yes", "English GST and business terminology"),
    7: ("Yes", "Yes", "Yes", "English phrase and character reference"),
    8: ("Yes", "Yes", "Yes", "English military terminology"),
    9: ("No", "No", "Yes", "No clear English mixing in visible text"),
    10: ("No", "No", "Yes", "Mostly Romanized Hindi"),
    11: ("No", "No", "Yes", "Mostly Romanized Hindi"),
    12: ("Yes", "Yes", "Yes", "English radio terminology"),
    13: ("Yes", "Yes", "Yes", "Full English sentence mixed with Hindi"),
    14: ("Yes", "Yes", "Yes", "English geography question"),
    15: ("Yes", "Yes", "Yes", "English team and number expressions"),
    16: ("Yes", "Yes", "Yes", "English fitness terminology"),
    17: ("Yes", "Yes", "Yes", "English education-related terms"),
    18: ("Yes", "Yes", "Yes", "English words mixed with Hindi"),
    19: ("Yes", "Yes", "Yes", "Natural English phrase"),
    20: ("No", "No", "Yes", "Mostly Romanized Hindi")
}

cols = [
    "verified_hinglish",
    "english_words_used_naturally",
    "meaning_clear",
    "review_notes"
]

for col in cols:
    sample_review_4[col] = sample_review_4[col].astype("object")

for pos, decision in decisions_4.items():
    for col, value in zip(cols, decision):
        sample_review_4.iloc[
            pos - 1, sample_review_4.columns.get_loc(col)
        ] = value

sample_review_4.to_csv(
    sample_review_4_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", len(sample_review_4))
print("Likely Hinglish:",
      (sample_review_4["verified_hinglish"] == "Yes").sum())
print("Not Hinglish:",
      (sample_review_4["verified_hinglish"] == "No").sum())

Saved: 20
Likely Hinglish: 16
Not Hinglish: 4


In [ ]:
# Update master review sheet with batch 4
for col in review_columns:
    review_df.loc[sample_review_4.index, col] = sample_review_4[col]

review_df.to_csv(
    master_path,
    index=False,
    encoding="utf-8-sig"
)

print("Master file updated:", os.path.exists(master_path))
print("Total candidates:", len(review_df))
print("Reviewed:", review_df["verified_hinglish"].notna().sum())
print("Remaining:", review_df["verified_hinglish"].isna().sum())
print("Likely Hinglish:",
      (review_df["verified_hinglish"] == "Yes").sum())
print("Not Hinglish:",
      (review_df["verified_hinglish"] == "No").sum())

Master file updated: True
Total candidates: 95
Reviewed: 80
Remaining: 15
Likely Hinglish: 60
Not Hinglish: 20


In [ ]:
remaining_df = review_df[
    review_df["verified_hinglish"].isna()
].copy()

sample_review_5 = remaining_df.copy()

sample_review_5_path = os.path.join(
    BASE,
    "06_real_hinglish/FactIndic_Hinglish_Review_81_95.csv"
)

sample_review_5.to_csv(
    sample_review_5_path,
    index=False,
    encoding="utf-8-sig"
)

print("Final review sample:", len(sample_review_5))
print("Saved:", sample_review_5_path)

display(sample_review_5)

Final review sample: 15
Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/06_real_hinglish/FactIndic_Hinglish_Review_81_95.csv


,source_dataset,source_config,text,script_type,selection_method,reply_to_text,likely_hinglish,human_verified,language_quality,meaning_clear,review_notes,latin_word_count,total_word_count,latin_word_percentage,latin_words,devanagari_words,roman_hindi_words,total_language_words,verified_hinglish,english_words_used_naturally
2,MUTANT,Mankibaat,"प्यारे नौजवान साथियो, 12 जनवरी स्वामी विवेकानंद जी की जन्म-जयंती है। मेरे जैसे इस देश के कोटि-कोटि लोग हैं जिनको स्वामी विवेकानंद जी से प्रेरणा मिलती रही है। 1995 से 12 जनवरी स्वामी विवेकानंद जयंती को एक National Youth Festival के रूप में मनाया जाता है। इस वर्ष ये 12 जनवरी से 16 जनवरी तक छत्तीसगढ़ के रायपुर में होने वाला है। और मुझे जानकारी मिली कि इस बार की उनकी जो theme है, क्योंकि उनका ये event them based होता है, theme बहुत बढ़िया है ‘Indian Youth on development skill and harmony’. मुझे बत...",Mixed Devanagari-Latin,Script-based candidate filter,NaN,True,NaN,NaN,NaN,NaN,30,233,12.88,30,201,0,231,NaN,NaN
3,MUTANT,Mankibaat,"उस दिन हमने ‘सुगम्य भारत’ अभियान की शुरुआत की है। इसके तहत हम physical और virtual – दोनों तरह के Infrastructure में सुधार कर उन्हें “दिव्यांग” लोगों के लिए सुगम्य बनायेंगे। स्कूल हो, अस्पताल हो, सरकारी दफ़्तर हो, बस अड्डे हों, रेलवे स्टेशन में ramps हो, accessible parking, accessible lifts, ब्रेल लिपि; कितनी बातें हैं। इन सब में उसे सुगम्य बनाने के लिए Innovation चाहिए, technology चाहिए, व्यवस्था चाहिए, संवेदनशीलता चाहिएI इस काम का बीड़ा उठाया हैI जन-भागीदारी भी मिल रहीं है। लोगों को अच्छा लगा...",Mixed Devanagari-Latin,Script-based candidate filter,NaN,True,NaN,NaN,NaN,NaN,11,97,11.34,11,87,0,98,NaN,NaN
9,MUTANT,Mankibaat,"दोस्तो, मैं आप से एक अपेक्षा करता हूँ। 2017 की ये मेज़बानी, ये अवसर कैसा हो, साल भर का हमारा Football में momentum लाने के लिए कैसे-कैसे कार्यक्रम हो, प्रचार कैसे हो, व्यवस्थाओं में सुधार कैसे हो, FIFA Under – 17 विश्वकप के माध्यम से भारत के नौजवानों में खेल के प्रति रूचि कैसे बढ़े, सरकारों में, शैक्षिक संस्थाओं में, अन्य सामाजिक संगठनों में, खेल के साथ जुड़ने की स्पर्धा कैसे खड़ी हो? Cricket में हम सभी देख पा रहे हैं, लेकिन यही चीज़ और खेलों में भी लानी है। Football एक अवसर है। क्या आप मुझे अपने...",Mixed Devanagari-Latin,Script-based candidate filter,NaN,True,NaN,NaN,NaN,NaN,18,246,7.32,18,225,0,243,NaN,NaN
14,MUTANT,Mankibaat,"मेरे प्यारे देशवासियो, इस देश का सामान्य मानव देश के लिए बहुत-कुछ करने के लिए अवसर खोजता रहता है। जब मैंने लोगों से कहा – रसोई गैस की subsidy छोड़ दीजिये, इस देश के एक करोड़ से ज्यादा परिवारों ने स्वेच्छा से subsidy छोड़ दी। मैं खास करके जिनके पास अघोषित आय है, उनके लिए एक खास उदाहरण प्रस्तुत करना चाहता हूँ। मैं कल Smart City के कार्यक्रम के निमित्त पुणे जब गया था, तो वहाँ मुझे श्रीमान चन्द्रकान्त दामोदर कुलकर्णी और उनके परिवारजनों से मिलने का सौभाग्य मिला। मैंने उनको खास मिलने के लिए बुलाया था...",Mixed Devanagari-Latin,Script-based candidate filter,NaN,True,NaN,NaN,NaN,NaN,22,434,5.07,22,409,0,431,NaN,NaN
27,MUTANT,Mankibaat,"पिछली बात ‘मन की बात’ में कुछ लोगों ने मुझे सुझाव दिया था food waste हो रहा है, उसके संबंध में चिंता जताई थी और मैंने उल्लेख किया। और जब उल्लेख किया तो उसके बाद Narendra Modi App पर, MyGov पर देश के अनेक कोने में से अनेक लोगों ने, कैसे-कैसे Innovative ideas के साथ food waste को बचाने के लिये क्या-क्या प्रयोग किये हैं। मैंने भी कभी सोचा नहीं था आज हमारे देश में खासकर के युवा-पीढ़ी, लम्बे अरसे से इस काम को कर रही है। कुछ सामाजिक संस्थायें करती हैं, ये तो हम कई वर्षों से जानते आए हैं, लेकिन मेरे...",Mixed Devanagari-Latin,Script-based candidate filter,NaN,True,NaN,NaN,NaN,NaN,12,190,6.32,12,180,0,192,NaN,NaN
34,MUTANT,Mankibaat,"मेरे प्यारे देशवासियो, ‘मन की बात’ की सराहना भी होती रही है, आलोचना भी होती रही है। लेकिन जब कभी मैं ‘मन की बात’ के प्रभाव की ओर देखता हूँ तो मेरा विश्वास दृढ़ हो जाता है कि इस देश के जनमानस के साथ ‘मन की बात’ शत-प्रतिशत अटूट रिश्ते से बंध चुकी है। खादी और hand-loom का ही उदाहरण ले लीजिए। गाँधी- जयन्ती पर मैं हमेशा hand-loom के लिए, खादी के लिए वकालत करता रहता हूँ और उसका परिणाम क्या है! आपको भी यह जानकर के खुशी होगी। मुझे बताया गया है कि इस महीने 17 अक्तूबर को ‘धनतेरस’ के दिन दिल्ली के खाद

In [ ]:
decisions_5 = {
    1: ("Yes", "Yes", "Yes", "English youth festival and event terminology"),
    2: ("Yes", "Yes", "Yes", "English accessibility and technology terms"),
    3: ("Yes", "Yes", "Yes", "English football and sports terminology"),
    4: ("Yes", "Yes", "Yes", "English Smart City terminology"),
    5: ("Yes", "Yes", "Yes", "English food and technology terms"),
    6: ("Yes", "Yes", "Yes", "English handloom terminology"),
    7: ("Yes", "Yes", "Yes", "English science and research terms"),
    8: ("No", "No", "Yes", "Mostly Romanized Hindi; no clear English mixing"),
    9: ("Yes", "Yes", "Yes", "English gaming terminology"),
    10: ("Yes", "Yes", "Yes", "English short-form terminology"),
    11: ("Yes", "Yes", "Yes", "English event and sponsor terminology"),
    12: ("Yes", "Yes", "Yes", "English sentence mixed with Hindi"),
    13: ("No", "No", "Yes", "Mostly Romanized Hindi"),
    14: ("Yes", "Yes", "Yes", "English sentences mixed with Hindi"),
    15: ("Yes", "Yes", "Yes", "English clauses mixed with Hindi")
}

for col in review_columns:
    sample_review_5[col] = sample_review_5[col].astype("object")

for pos, decision in decisions_5.items():
    for col, value in zip(review_columns, decision):
        sample_review_5.iloc[
            pos - 1,
            sample_review_5.columns.get_loc(col)
        ] = value

# Save final batch
sample_review_5.to_csv(
    sample_review_5_path,
    index=False,
    encoding="utf-8-sig"
)

# Update master file
for col in review_columns:
    review_df.loc[sample_review_5.index, col] = sample_review_5[col]

review_df.to_csv(
    master_path,
    index=False,
    encoding="utf-8-sig"
)

print("Final batch saved:", len(sample_review_5))
print("Master file updated:", os.path.exists(master_path))
print("Total candidates:", len(review_df))
print("Reviewed:", review_df["verified_hinglish"].notna().sum())
print("Remaining:", review_df["verified_hinglish"].isna().sum())
print("Likely Hinglish:",
      (review_df["verified_hinglish"] == "Yes").sum())
print("Not Hinglish:",
      (review_df["verified_hinglish"] == "No").sum())

Final batch saved: 15
Master file updated: True
Total candidates: 95
Reviewed: 95
Remaining: 0
Likely Hinglish: 73
Not Hinglish: 22


In [ ]:
# 1. Extract likely Hinglish records
final_hinglish = review_df[
    review_df["verified_hinglish"] == "Yes"
].copy()

final_hinglish_path = os.path.join(
    BASE,
    "06_real_hinglish/FactIndic_Hinglish_73_Human_Reviewed.csv"
)

final_hinglish.to_csv(
    final_hinglish_path,
    index=False,
    encoding="utf-8-sig"
)

# 2. Source-wise summary
source_summary = (
    review_df.groupby("source_dataset")["verified_hinglish"]
    .value_counts()
    .unstack(fill_value=0)
    .reset_index()
)

summary_path = os.path.join(
    BASE,
    "06_real_hinglish/FactIndic_Hinglish_Final_Review_Summary.csv"
)

source_summary.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

# 3. Save checkpoint
save_progress(
    "task_12_hinglish_human_review",
    "completed",
    {
        "total_candidates": 95,
        "reviewed": 95,
        "likely_hinglish": 73,
        "not_hinglish": 22,
        "final_hinglish_file": final_hinglish_path,
        "master_review_file": master_path,
        "summary_file": summary_path
    }
)

print("\nFINAL FILES SAVED")
print("Likely Hinglish:", final_hinglish_path)
print("Review summary:", summary_path)
print("Master review:", master_path)
print("\nSource-wise summary:")
display(source_summary)

Checkpoint saved: task_12_hinglish_human_review

FINAL FILES SAVED
Likely Hinglish: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/06_real_hinglish/FactIndic_Hinglish_73_Human_Reviewed.csv
Review summary: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/06_real_hinglish/FactIndic_Hinglish_Final_Review_Summary.csv
Master review: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/06_real_hinglish/FactIndic_Hinglish_95_Human_Review.csv

Source-wise summary:


verified_hinglish,source_dataset,No,Yes
0,GupShup,17,27
1,MUTANT,5,46


In [ ]:
import os
import shutil
import zipfile

# Backup master review file
master_backup = os.path.join(
    BASE,
    "09_checkpoints",
    "FactIndic_Hinglish_95_Human_Review_BACKUP.csv"
)

shutil.copy2(master_path, master_backup)

# Backup progress file
progress_backup = os.path.join(
    BASE,
    "09_checkpoints",
    "progress_FINAL_BACKUP.json"
)

if os.path.exists(PROGRESS_FILE):
    shutil.copy2(PROGRESS_FILE, progress_backup)

# Create complete ZIP outside the project folder
zip_path = "/content/drive/MyDrive/FactIndic_PersonB_CleanRun_BACKUP_2026-11-11.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk(BASE):
        for file in files:
            full_path = os.path.join(root, file)
            relative_path = os.path.relpath(full_path, BASE)
            z.write(full_path, relative_path)

print("BACKUP CREATED:", os.path.exists(zip_path))
print("ZIP SIZE (MB):", round(os.path.getsize(zip_path) / (1024*1024), 2))
print("ZIP PATH:", zip_path)

BACKUP CREATED: True
ZIP SIZE (MB): 27.0
ZIP PATH: /content/drive/MyDrive/FactIndic_PersonB_CleanRun_BACKUP_2026-11-11.zip


# New section

In [ ]:
import os

claim_folder = os.path.join(BASE, "07_claim_labels")

print("Claim folder exists:", os.path.exists(claim_folder))

if os.path.exists(claim_folder):
    for file in os.listdir(claim_folder):
        path = os.path.join(claim_folder, file)
        print(file, "|", round(os.path.getsize(path)/1024, 2), "KB")
else:
    print("No claim-label folder found.")

Claim folder exists: True


In [ ]:
import os
import pandas as pd

claim_folder = os.path.join(BASE, "07_claim_labels")

if os.path.exists(claim_folder):
    for file in os.listdir(claim_folder):
        path = os.path.join(claim_folder, file)

        if file.endswith(".csv"):
            df = pd.read_csv(path)
            print("\nFile:", file)
            print("Rows:", len(df))
            print("Columns:", df.columns.tolist())
            display(df.head(3))
else:
    print("Claim-label folder does not exist.")

In [ ]:
import os
import pandas as pd

path = os.path.join(
    BASE,
    "02_clean_data",
    "XL_Sum_Hindi_Cleaned_250.csv"
)

df = pd.read_csv(path)

print("Articles available:", len(df))
print("Columns:", df.columns.tolist())

for i, row in df.head(5).iterrows():
    print(f"\nArticle {i+1}")
    print("ID:", row["id"])
    print("Summary:", str(row["summary"])[:500])

Articles available: 250
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words']

Article 1
ID: 080419_brchopra_festival
Summary: भारतीय फ़िल्म इतिहास में फ़िल्मकार बीआर चोपड़ा का नाम बड़ा अदब से लिया जाता है. उनके 94वें जन्मदिन के मौके पर दिल्ली में बीआर चोपड़ा की फ़िल्मों का विशेष उत्सव मनाया जा रहा है.

Article 2
ID: 080613_afghan_jail
Summary: अफ़ग़ानिस्तान के कंधार में तालेबान चरमपंथियों ने जेल पर हमला करके सैकड़ों क़ैदियों को छुड़ा दिया है.

Article 3
ID: 130912_guinnness_record_bulgarian_swim_sr
Summary: गिनीज़ वर्ल्ड रिकॉर्ड में अपना नाम दर्ज कराने की उम्मीद में एक बुल्गारियाई व्यक्ति ने बोरे में बंद होकर दो किलोमीटर तक तैराकी की.

Article 4
ID: 050713_us_un
Summary: अमरीका ने जी-4 देशों द्वारा सुरक्षा परिषद के विस्तार के लिए संयुक्त राष्ट्र महासभा में पेश प्रस्ताव को ठुकरा दिया है.

Article 5
ID: 120809_salman_pkp
Summary: तीन लगातार बड़ी हिट फिल्में देने के बाद सलमान खान का कहना है कि ये एक खतरनाक स्थिति है, क्योंकि उन्हें मालूम नहीं कि अब आगे क्या

In [ ]:
import os
import pandas as pd

claim_folder = os.path.join(BASE, "07_claim_labels")
os.makedirs(claim_folder, exist_ok=True)

# Select 50 articles consistently
claims_df = df.sample(n=50, random_state=42).copy()

claim_sheet = pd.DataFrame({
    "Claim_ID": [f"FC{i:03d}" for i in range(1, 51)],
    "Article_ID": claims_df["id"].values,
    "Claim": claims_df["summary"].values,
    "Source_Article": claims_df["text"].values,
    "Verdict": "",
    "Evidence_from_Article": "",
    "Reviewer_Notes": ""
})

claim_path = os.path.join(
    claim_folder,
    "FactIndic_50_Claim_Labeling_Sheet.csv"
)

claim_sheet.to_csv(claim_path, index=False, encoding="utf-8-sig")

print("Claim sheet created.")
print("Total claims:", len(claim_sheet))
print("Saved at:", claim_path)

display(claim_sheet[[
    "Claim_ID", "Article_ID", "Claim"
]].head(10))

Claim sheet created.
Total claims: 50
Saved at: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/07_claim_labels/FactIndic_50_Claim_Labeling_Sheet.csv


,Claim_ID,Article_ID,Claim
0,FC001,040102_france_sikhs,फ़्रांस में रह रहे पाँच हज़ार सिख फ़्रांस के उस प्रतिबंध से नाराज़ है जो धार्मिक प्रतीकों के इस्तेमाल पर लगाया जा रहा है.
1,FC002,social-43068605,आंखों के रास्ते दिलों में कुटिया बनाने वाली प्रिया प्रकाश अब वाया सोशल मीडिया पाकिस्तान पहुंच गई हैं.
2,FC003,050420_pope_muslims,कई मुसलमान नेताओं ने नए पोप से अनुरोध किया है कि वो पोप जॉन पॉल द्वितीय के रास्ते पर चलते हुए विभिन्न धर्मों के बीच - ख़ासकर इस्लाम और ईसाई धर्म के बीच - बातचीत जारी रखने की नीति को प्रोत्साहन दें.
3,FC004,150811_sani_mirza_autobiography_farha_adp_ssm,कई दिनों से चर्चा थी कि निर्माता निर्देशक फ़राह ख़ान जल्द ही भारतीय महिला टेनिस खिलाड़ी सानिया मिर्ज़ा पर एक बायोपिक फ़िल्म बनाने वाली हैं.
4,FC005,060715_bushputin_g8,अमरीकी राष्ट्रपति जॉर्ज बुश ने जी8 बैठक से पहले रूस के राष्ट्रपति व्लादिमीर पुतिन के साथ शनिवार को बैठक करके कई अहम मसलों पर बातचीत की है.
5,FC006,india-44912345,जीएसटी काउंसिल की शनिवार को हुई बैठक में कई अहम फ़ैसले किए गए हैं. सैनिटरी नैपकिन समेत कई सामानों को जीएसटी के दायरे से बाहर करने का फैसला किया गया है.
6,FC007,160506_breast_cancer_test_easier_cj,मिहिर शाह मेडिकल डिवाइस टैक्नोलॉजी डेवलपमेंट के क्षेत्र में काम कर रहे थे.
7,FC008,040709_barrier_split,फ़लस्तीनी नेता अंतरराष्ट्रीय न्याय अदालत के उस फ़ैसले का स्वागत कर रहे हैं जिसमें अदालत ने पश्चिमी तट में बन रही इसराइल की 'सुरक्षा दीवार' को अवैध करार दिया है.
8,FC009,international-41372477,चीन के भूकंप विज्ञानियों ने बताया है कि उत्तर कोरिया में 3.4 तीव्रता का भूकंप दर्ज किया गया है.
9,FC010,051121_aids_india,संयुक्त राष्ट्र की संस्था यूएनएड्स ने अपनी वार्षिक रिपोर्ट में कहा है कि कुछ भारतीय राज्यों में एड्स संक्रमण की दर तो स्थिर हो गई है लेकिन कई अन्य राज्यों में अभी भी ख़तरा बना हुआ है.


In [ ]:
import pandas as pd
import os

claim_path = os.path.join(
    BASE,
    "07_claim_labels",
    "FactIndic_50_Claim_Labeling_Sheet.csv"
)

claims = pd.read_csv(claim_path)

print("Total claims:", len(claims))
print("\nColumns:")
print(claims.columns.tolist())

print("\nMissing values:")
print(claims.isna().sum())

display(claims.head(10))

Total claims: 50

Columns:
['Claim_ID', 'Article_ID', 'Claim', 'Source_Article', 'Verdict', 'Evidence_from_Article', 'Reviewer_Notes']

Missing values:
Claim_ID                  0
Article_ID                0
Claim                     0
Source_Article            0
Verdict                  50
Evidence_from_Article    50
Reviewer_Notes           50
dtype: int64


,Claim_ID,Article_ID,Claim,Source_Article,Verdict,Evidence_from_Article,Reviewer_Notes
0,FC001,040102_france_sikhs,फ़्रांस में रह रहे पाँच हज़ार सिख फ़्रांस के उस प्रतिबंध से नाराज़ है जो धार्मिक प्रतीकों के इस्तेमाल पर लगाया जा रहा है.,"वे भारत के प्रधानमंत्री से आग्रह कर रहे हैं कि वह उनकी पारंपरिक पगड़ियों को इस प्रतिबंध से अलग रखने की हिमायत करें. सिख समुदाय के एक प्रवक्ता चैन सिंह ने कहा, ""यह क़ानून मुसलमानों के ही नहीं सिखों के भी ख़िलाफ़ है"". हम पगड़ी पहने बिना नहीं रह सकते. यह हमारे धर्म का हिस्सा हैं. अगर हमारे पगड़ी बांधने पर पाबंदी लगाई गई तो हम यहाँ रह भी नहीं पाएँगे. चैन सिंह ""हम पगड़ी पहने बिना नहीं रह सकते. यह हमारे धर्म का हिस्सा हैं. अगर हमारे पगड़ी बांधने पर पाबंदी लगाई गई तो हम यहाँ रह भी नहीं पाएँगे"". फ़्...",NaN,NaN,NaN
1,FC002,social-43068605,आंखों के रास्ते दिलों में कुटिया बनाने वाली प्रिया प्रकाश अब वाया सोशल मीडिया पाकिस्तान पहुंच गई हैं.,"प्रिया प्रकाश प्रिया प्रकाश की कजरारे नैनों वाला एक वीडियो भारत में काफी वायरल रहा. शायद आप में से ज़्यादातर लोगों की निगाह से ये वीडियो गुज़रा भी होगा. अब प्रिया की नैनों के इशारों के दीवाने पाकिस्तान में भी नज़र आ रहे हैं. सोशल मीडिया यूज़र्स प्रिया की दीवानगी में हँसी का तड़का भी जोड़ रहे हैं. पाकिस्तान में सोशल मीडिया पर प्रिया के इशारों और नेताओं की प्रतिक्रियाओं को जोड़कर वीडियो और मीम बनाए जा रहे हैं. इन नेताओं में इमरान ख़ान, नवाज़ शरीफ़, अल्ताफ हुसैन, आसिफ़ अली ज़रदारी जैसे लोग भी श...",NaN,NaN,NaN
2,FC003,050420_pope_muslims,कई मुसलमान नेताओं ने नए पोप से अनुरोध किया है कि वो पोप जॉन पॉल द्वितीय के रास्ते पर चलते हुए विभिन्न धर्मों के बीच - ख़ासकर इस्लाम और ईसाई धर्म के बीच - बातचीत जारी रखने की नीति को प्रोत्साहन दें.,शंका जताई गई है कि नए पोप शायद ऐसा करने में उतना विश्वास नहीं रखते. तुर्की में तो उनके चुनाव पर हैरत ज़ाहिर की गई है क्योंकि उन्होंने यूरोपीय संघ में तुर्की की सदस्यता का ये कहकर विरोध किया था कि बेहतर हो यदि तुर्की अरब देशों की सदस्यता हासिल करे. पोप बेनेडिक्ट 16वें के सामने जो चुनौतियां हैं उनमें प्रमुख है वैटिकन का इस्लाम के प्रति रवैया. कुछ कैथोलिक ऐसे हैं जो पोप जॉन पॉल द्वितीय की नीति को अपनाते हुए दूसरे संप्रदायों के साथ बातचीत को आगे बढ़ाना चाहते हैं. लेकिन कुछ दूसरे हैं जो इस सोच से...,NaN,NaN,NaN
3,FC004,150811_sani_mirza_autobiography_farha_adp_ssm,कई दिनों से चर्चा थी कि निर्माता निर्देशक फ़राह ख़ान जल्द ही भारतीय महिला टेनिस खिलाड़ी सानिया मिर्ज़ा पर एक बायोपिक फ़िल्म बनाने वाली हैं.,"यदि ऐसा होता तो धावक मिल्खा सिंह, मुक्क़ेबाज़ मेरी कॉम और क्रिकेट खिलाड़ी एमएस धोनी के बाद ये किसी बड़े बैनर और खिलाड़ी की चौथी बायोपिक होती. लेकिन एक समारोह के दौरान मीडिया से बात करते हुए फ़राह ने ऐसी किसी भी फ़िल्म को बनाए जाने की बात से इंकार कर दिया. फ़राह ने अपने बयान में कहा, ""पता नहीं ये बात कहां से निकल आई है, मैं कोई फ़िल्म नहीं बना रही हूं और यह एक अफ़वाह है लेकिन आइडिया अच्छा है."" सानिया दुनिया में महिला डबल्स की नंबर वन टेनिस खिलाड़ी हैं और बायोपिक पर पहले बात कर चुकी हैं. समाप्...",NaN,NaN,NaN
4,FC005,060715_bushputin_g8,अमरीकी राष्ट्रपति जॉर्ज बुश ने जी8 बैठक से पहले रूस के राष्ट्रपति व्लादिमीर पुतिन के साथ शनिवार को बैठक करके कई अहम मसलों पर बातचीत की है.,दोनों देशों के राष्ट्रपतियों ने सेंट पीटर्सबर्ग में जी-8 की बैठक से पहले द्विपक्षीय और अन्य अंतरराष्ट्रीय मसलों पर विचार-विमर्श किया. रूस और अमरीकी अधिकारियों ने कहा है कि दोनों नेताओं की इस बैठक में भी ऐसी कोई द्विपक्षीय समझ नहीं बन सकी जिससे विश्व व्यापार संगठन (डब्ल्यूटीओ) की सदस्यता पाने के लिए रूस का रास्ता साफ़ हो सके. राष्ट्रपति बुश ने कहा है कि इस बारे में अभी और बातचीत किए जाने की ज़रूरत है क्योंकि वह सुनिश्चित हो जाना चाहते हैं कि इस तरह की कोई समझ अगर बनती है तो उसे अमरीकी कांग्रे...,NaN,NaN,NaN
5,FC006,india-44912345,जीएसटी काउंसिल की शनिवार को हुई बैठक में कई अहम फ़ैसले किए गए हैं. सैनिटरी नैपकिन समेत कई सामानों को जीएसटी के दायरे से बाहर करने का फैसला किया गया है.,जीएसटी काउंसिल की 28वीं बैठक की अध्यक्षता अभी वित्त मंत्री का कार्यभार संभाल रहे पीयूष गोयल ने की. जीएसटी काउंसिल ने फ़ैसला लिया है कि सैनिटरी नैपकिन को जीएसटी से बाहर कर दिया जाए यानी सैनिटरी नैपकिन पर जीएसटी की दर को शून्य कर दिया गया है. अब तक सैनिटरी नैपकिन पर 12 फ़ीसदी जीएसटी लगाया जा रहा था. जीएस

In [ ]:
import pandas as pd
import os

claim_path = os.path.join(
    BASE, "07_claim_labels",
    "FactIndic_50_Claim_Labeling_Sheet.csv"
)

claims = pd.read_csv(claim_path)

labels = {
    "FC001": (
        "NOT_ENOUGH_INFO",
        "यह क़ानून मुसलमानों के ही नहीं सिखों के भी ख़िलाफ़ है.",
        "The opposition is supported, but the stated count of 5,000 is not established by the displayed evidence."
    ),
    "FC002": (
        "SUPPORTED",
        "पाकिस्तान में सोशल मीडिया पर प्रिया के इशारों और नेताओं की प्रतिक्रियाओं को जोड़कर वीडियो और मीम बनाए जा रहे हैं.",
        "The article describes Priya-related videos and memes circulating in Pakistan."
    ),
    "FC003": (
        "NOT_ENOUGH_INFO",
        "कुछ कैथोलिक ... दूसरे संप्रदायों के साथ बातचीत को आगे बढ़ाना चाहते हैं.",
        "The excerpt mentions interfaith dialogue, but does not establish the specific request by Muslim leaders."
    ),
    "FC004": (
        "SUPPORTED",
        "फ़राह ने ऐसी किसी भी फ़िल्म को बनाए जाने की बात से इंकार कर दिया ... यह एक अफ़वाह है.",
        "The article confirms that the rumour about a Sania Mirza biopic was circulating."
    ),
    "FC005": (
        "SUPPORTED",
        "दोनों देशों के राष्ट्रपतियों ने सेंट पीटर्सबर्ग में जी-8 की बैठक से पहले द्विपक्षीय और अन्य अंतरराष्ट्रीय मसलों पर विचार-विमर्श किया.",
        "The article confirms the meeting and discussion before the G8 meeting."
    ),
    "FC006": (
        "SUPPORTED",
        "जीएसटी काउंसिल ने फ़ैसला लिया है कि सैनिटरी नैपकिन को जीएसटी से बाहर कर दिया जाए.",
        "The article directly confirms the GST decision described in the claim."
    ),
    "FC007": (
        "SUPPORTED",
        "मिहिर शाह मेडिकल डिवाइस टैक्नोलॉजी डेवलपमेंट के क्षेत्र में काम कर रहे थे.",
        "The source directly states this."
    ),
    "FC008": (
        "SUPPORTED",
        "शुक्रवार को सुनाए गए अपने फ़ैसले में अदालत ने कहा था कि ये दीवार अंतरराष्ट्रीय कानूनों का उल्लंघन करती है.",
        "The article confirms the court's decision that the wall violated international law. The welcome by Palestinian leaders should be checked against the full article."
    ),
    "FC009": (
        "SUPPORTED",
        "चीन की सरकारी समाचार एजेंसी के मुताबिक, भूकंप विज्ञानियों ने बताया है कि यह 'संदिग्ध विस्फोट' है.",
        "The article confirms that Chinese seismologists reported the event; it also records uncertainty about its cause."
    ),
    "FC010": (
        "SUPPORTED",
        "दक्षिण और पश्चिमी राज्यों तमिलनाडु, आंध्र प्रदेश, कर्नाटक और महाराष्ट्र में संक्रमण की दर में स्थिरता आई है.",
        "The source confirms stable rates in several states and discusses continuing risk elsewhere."
    )
}

for claim_id, (verdict, evidence, notes) in labels.items():
    idx = claims["Claim_ID"] == claim_id
    claims.loc[idx, "Verdict"] = verdict
    claims.loc[idx, "Evidence_from_Article"] = evidence
    claims.loc[idx, "Reviewer_Notes"] = notes

claims.to_csv(claim_path, index=False, encoding="utf-8-sig")

print("Saved first batch.")
print(claims.loc[:9, ["Claim_ID", "Verdict"]].to_string(index=False))
print("\nTotal labelled:", claims["Verdict"].notna().sum())
print("Remaining:", claims["Verdict"].isna().sum())

Saved first batch.
Claim_ID         Verdict
   FC001 NOT_ENOUGH_INFO
   FC002       SUPPORTED
   FC003 NOT_ENOUGH_INFO
   FC004       SUPPORTED
   FC005       SUPPORTED
   FC006       SUPPORTED
   FC007       SUPPORTED
   FC008       SUPPORTED
   FC009       SUPPORTED
   FC010       SUPPORTED

Total labelled: 10
Remaining: 40


/tmp/ipykernel_4396/3493556055.py:66: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'NOT_ENOUGH_INFO' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  claims.loc[idx, "Verdict"] = verdict
/tmp/ipykernel_4396/3493556055.py:67: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'यह क़ानून मुसलमानों के ही नहीं सिखों के भी ख़िलाफ़ है.' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  claims.loc[idx, "Evidence_from_Article"] = evidence
/tmp/ipykernel_4396/3493556055.py:68: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The opposition is supported, but the stated count of 5,000 is not established by the displayed evidence.' has dtype incompatible with float64, please expli

In [ ]:
import pandas as pd
import os

claim_path = os.path.join(
    BASE, "07_claim_labels",
    "FactIndic_50_Claim_Labeling_Sheet.csv"
)

claims = pd.read_csv(claim_path)

for col in ["Verdict", "Evidence_from_Article", "Reviewer_Notes"]:
    claims[col] = claims[col].astype("object")

print("Already labelled:", claims["Verdict"].notna().sum())
print("Remaining:", claims["Verdict"].isna().sum())

display(
    claims.loc[10:19, [
        "Claim_ID", "Article_ID", "Claim", "Source_Article"
    ]]
)

Already labelled: 10
Remaining: 40


,Claim_ID,Article_ID,Claim,Source_Article
10,FC011,140524_iran_hanging_aa,ईरान में सरकारी मीडिया के अनुसार एक कारोबारी को अरबों की हेराफेरी करने के जुर्म में फांसी दे दी गई है.,"मुहाफरीद आमिर खोसरावी को तेहरान की एविन जेल में शनिवार को फांसी दी गई. आमिर खोसरावी 2011 में सामने एक घोटाले में दोषी करार दिए गए थे. उन पर फर्जी दस्तावेजों के आधार पर कर्ज लेने के आरोप लगे, जिसमें बाद में 2.6 अरब डॉलर की धांधली सामने आई. इस मामले के चलते उस समय की सरकार को कहना पड़ा कि इसमें उसके सदस्य शामिल नहीं हैं. ये मामला सितंबर 2011 में सामने आया जब एक निवेश कंपनी पर आरोप लगा कि उसने चार साल के दौरान जाली दस्तावेजों के आधार पर कई बैंकों से कर्ज लिया है. बताया जाता है कि इस धन का इस्ते..."
11,FC012,international-56439434,पाकिस्तान में सिंध प्रांत की एक अदालत ने एक हिंदू लड़की को कथित जबरन धर्म परिवर्तन के मामले में बाल संरक्षण केंद्र भेज दिया है.,"लड़की ने अदालत के सामने अपने बयान में कहा है कि उसने अपनी मर्ज़ी से बिना किसी दबाव के धर्म बदला है. दूसरी ओर, लड़की के पिता का आरोप है कि कुछ लोग उसे (लड़की) को उठा कर ले गए थे, जिसके बाद उसका ज़बरन धर्म परिवर्तन कराया गया है. लड़की को बुधवार को आश्रय गृह से पुलिस सुरक्षा के साथ उत्तरी सिंध के काश्मोर कंधकोट की तहसील तंगवानी के सिविल जज मुनव्वर पीरज़ादा की अदालत में पेश किया गया, जहाँ अदालत ने लड़की का बयान दर्ज किया. लड़की का कहना है कि उसने अपनी मर्ज़ी से दरगाह भरचोंडी पहुंचकर अपना धर्म बद..."
12,FC013,051209_tamilnadu_cyclone,"भारत के दक्षिणी राज्य तमिलनाडु, आंध्र प्रदेश और पांडिचेरी में समुद्री तूफ़ान फ़ानूस के कारण भारी बारिश की आशंका जताई जा रही है.",अभी समुद्री तूफ़ान फ़ानूस चेन्नई से 550 किलोमीटर दक्षिणपूर्व स्थित है. मौसम विभाग का कहना है कि अगले 24 घंटों के अंदर तूफ़ान और तेज़ी से आगे बढ़ेगा. राज्य के तटीय इलाक़ों में लोगों को सावधान कर दिया गया है. मछुआरों को समुद्र में न जाने की सलाह दी गई है. समुद्री तूफान के कारण तटीय इलाक़ों में 70 किलोमीटर प्रति घंटे की रफ़्तार से हवाएँ चल रही हैं. तमिलनाडु के अलावा दक्षिणी आंध्र प्रदेश में भी फ़ानूस के कारण भारी बारिश की आशंका व्यक्त की जा रही है. इसी सप्ताह में चेन्नई में ज़बरदस्त बारिश हुई थ...
13,FC014,india-47692221,कांग्रेस पार्टी के राष्ट्रीय प्रवक्ता रणदीप सिंह सुरजेवाला ने दावा किया है कि 'बीजेपी नेता कलराज मिश्र ने पार्टी वर्करों को सवाल पूछने पर गोली मारने की धमकी दी'.,"ट्विटर पर अपने आधिकारिक हैंडल से सुरजेवाला ने ट्वीट किया, ""भाजपा की हिंसक मानसिकता का ताज़ा नमूना. कलराज मिश्र ने फ़रीदाबाद सांसद के विरोध में नारे लगाने वालों को धमकी भरे लहजे में कहा- ""अगर यह उनका प्रदेश होता तो इस तरह गड़बड़ करने वालों को वह स्टेज से उतरकर गोली मार देते"" क्या ये है भाजपा का संदेश- सवाल पूछो तो गोली खाओ!"" इस ट्वीट में सुरजेवाला ने बीजेपी नेता कलराज मिश्र की रैली का एक वीडियो शेयर किया है जिसे 65 हज़ार से ज़्यादा बार देखा जा चुका है. कलराज मिश्र पूर्वी उत्तर प्रदेश की देवरि..."
14,FC015,041003_pwg_security,आंध्र प्रदेश सरकार ने अक्तूबर माह में पीपुल्स वार ग्रुप के साथ वार्ता के दौरान संगठन के नेताओं को पूरी सुरक्षा देने का वादा किया है.,"अधिकारियों का कहना है कि सरकार ने पुलिस को निर्देश दिए हैं कि तीन प्रमुख माओवादी नेताओं को सुरक्षित रास्ता मुहैया कराया जाए. वार्ताकारों की सुरक्षा के बारे में पुलिस अधिकारी पीडब्ल्यूजी के मध्यस्थ वारा वारा राव, गदर और कल्याण राव से बात करेंगे. पीडब्ल्यूजी के तीन प्रमुख नेता रामकृष्णा, सुधाकर और गणेश सरकार के साथ अक्तूबर के दूसरे हफ्ते में बातचीत के लिए आ रहे हैं. सुरक्षा अहम विद्रोही संगठन पीडब्ल्यूजी के लिए सरकार से वार्ता के दौरान अपने नेताओं की सुरक्षा सबसे अहम मुद्दा है . इसी कारण संगठन..."
15,FC016,150408_kangana_ranaut_deepika_padukone_md,दो बार नेशनल अवॉर्ड विजेता रही फ़िल्म अभिनेत्री कंगना रनौत हमेशा से ही सीधे और बेबाक बयानों के लिए फिल्म इंडस्ट्री में मशहूर हैं.,"जानी मानी पत्रकार अनुपमा चोपड़ा की किताब ""फ्रंट रो- कन्वर्सेशन ऑन सिनेमा"" के लांच के मौके पर पहुंची कंगना ने कई किस्से पत्रकारों से साझा किए. कंगना ने बताया, ""जब मैं इतनी सफल नहीं थीं तब कोई मेरा इंटरव्यू भी लेने नहीं आता था. और अक्सर मुझसे एक ही सवाल पूछा जाता था कि सोनम या दीपिका में से कौन नेक्स्ट जेनरेशन की सुपरस्टार अभिनेत्री बनेगी."" कंगना ने बताया कि इसका जवाब वे कुछ यूँ देती थी कि ना दीपिका ना सोनम नेक्स्ट जेनरेशन स्टार मैं हूँ. अवार्ड शो से 

In [ ]:
import pandas as pd
import os

claim_path = os.path.join(
    BASE, "07_claim_labels",
    "FactIndic_50_Claim_Labeling_Sheet.csv"
)

claims = pd.read_csv(claim_path)

for col in ["Verdict", "Evidence_from_Article", "Reviewer_Notes"]:
    claims[col] = claims[col].astype("object")

labels = {
    "FC011": (
        "SUPPORTED",
        "आमिर खोसरावी को तेहरान की एविन जेल में शनिवार को फांसी दी गई। उन पर 2.6 अरब डॉलर की धांधली के आरोप लगे थे।",
        "The article reports the execution and the financial fraud case."
    ),
    "FC012": (
        "NOT_ENOUGH_INFO",
        "लड़की को बुधवार को ... सिविल जज मुनव्वर पीरज़ादा की अदालत में पेश किया गया, जहाँ अदालत ने लड़की का बयान दर्ज किया।",
        "The displayed excerpt does not establish that she was sent to a child protection centre by the court."
    ),
    "FC013": (
        "SUPPORTED",
        "दक्षिणी आंध्र प्रदेश में भी फ़ानूस के कारण भारी बारिश की आशंका व्यक्त की जा रही है।",
        "Heavy rain is forecast for Tamil Nadu and southern Andhra Pradesh. The claim also mentions Puducherry, which is not confirmed in the displayed excerpt."
    ),
    "FC014": (
        "SUPPORTED",
        "अगर यह उनका प्रदेश होता तो इस तरह गड़बड़ करने वालों को वह स्टेज से उतरकर गोली मार देते।",
        "The article reports Surjewala's tweet quoting Kalraj Mishra. The claim's wording about party workers asking questions is not exactly established by this excerpt."
    ),
    "FC015": (
        "SUPPORTED",
        "सरकार ने पुलिस को निर्देश दिए हैं कि तीन प्रमुख माओवादी नेताओं को सुरक्षित रास्ता मुहैया कराया जाए।",
        "The article supports a security arrangement for the leaders during the talks."
    ),
    "FC016": (
        "NOT_ENOUGH_INFO",
        "कंगना ने बताया ... ना दीपिका ना सोनम नेक्स्ट जेनरेशन स्टार मैं हूँ।",
        "The excerpt discusses her direct statements but does not establish the claim that she is a two-time National Award winner."
    ),
    "FC017": (
        "SUPPORTED",
        "इलाज के तीन साल बाद तक मोटापे से जूझ रहे मरीज़ों की मौत की संख्या सामान्य वज़न वाले लोगों की मौत की संख्या से आधी थी।",
        "The study result supports the reported lower mortality after treatment. The claim also mentions angina patients."
    ),
    "FC018": (
        "SUPPORTED",
        "पिछले हफ़्ते रिलीज़ हुई निशिकांत कामथ की फिल्म 'मुंबई मेरी जान' की कहानी का आधार 11 जुलाई 2006 को मुंबई में हुए ट्रेन बम धमाके थे।",
        "The article directly supports the example of films based on the Mumbai train blasts."
    ),
    "FC019": (
        "SUPPORTED",
        "भारत में नागपुर की पुलिस का दावा है कि उसके पास ... बातचीत का टेप है, जिसमें सैमुएल्स टीम के बारे में कोचर को सूचनाएँ दे रहे हैं।",
        "The article reports the allegation and the cricket board's promise to investigate it."
    ),
    "FC020": (
        "NOT_ENOUGH_INFO",
        "हॉलैंड में 1928 में उनकी स्टिक को तोड़कर जांचा गया कि कहीं उसमें चुंबक तो नहीं था।",
        "The excerpt gives examples of Dhyan Chand's reputation but does not directly establish the broad comparison in the claim."
    )
}

for claim_id, (verdict, evidence, notes) in labels.items():
    idx = claims["Claim_ID"] == claim_id
    claims.loc[idx, "Verdict"] = verdict
    claims.loc[idx, "Evidence_from_Article"] = evidence
    claims.loc[idx, "Reviewer_Notes"] = notes

claims.to_csv(claim_path, index=False, encoding="utf-8-sig")

print("Batch FC011–FC020 saved.")
print(claims.loc[10:19, ["Claim_ID", "Verdict"]].to_string(index=False))
print("\nTotal labelled:", claims["Verdict"].notna().sum())
print("Remaining:", claims["Verdict"].isna().sum())

Batch FC011–FC020 saved.
Claim_ID         Verdict
   FC011       SUPPORTED
   FC012 NOT_ENOUGH_INFO
   FC013       SUPPORTED
   FC014       SUPPORTED
   FC015       SUPPORTED
   FC016 NOT_ENOUGH_INFO
   FC017       SUPPORTED
   FC018       SUPPORTED
   FC019       SUPPORTED
   FC020 NOT_ENOUGH_INFO

Total labelled: 20
Remaining: 30


In [ ]:
import pandas as pd
import os

claim_path = os.path.join(
    BASE, "07_claim_labels",
    "FactIndic_50_Claim_Labeling_Sheet.csv"
)

claims = pd.read_csv(claim_path)

print("Already labelled:", claims["Verdict"].notna().sum())

display(
    claims.loc[20:29, [
        "Claim_ID", "Article_ID", "Claim", "Source_Article"
    ]]
)

Already labelled: 20


,Claim_ID,Article_ID,Claim,Source_Article
20,FC021,131202_saina_sindhu_badminton_vr,"""चीनी खिलाड़ी को हराने के बाद फाइनल में कदम रखने से पहले मैं आत्मविश्वास से भरी थी और मैं जानती थी कि अगर मैं गलती नहीं करती हूं तो ख़िताब जीत सकती हूं.""","बीते शनिवार को मकाऊ ओपन ग्रां प्री गोल्ड बैडमिंटन टूर्नामेंट अपने नाम करने वाली भारत की उभरती खिलाड़ी पीवी सिंधु का यह कहना उनके बारे में काफी कुछ बताता है. फाइनल में सिंधु ने कनाडा की मिशेल ली को 21-15, 21-15 से मात दी जबकि सेमीफाइनल में उन्होंने चीन की क्वालिफायर किन जिनजिंग को 21-13, 18-21, 21-19 से हराया था. (अंतरराष्ट्रीय सितारे नदारद) पीवी सिंधु की क्षमता पर तो शायद ही किसी को कोई शक हो क्योंकि इसी साल वह विश्व बैडमिंटन चैंपियनशिप में कांस्य पदक जीत चुकी है. ऐसी महत्वपूर्ण उपलब्धि हासि..."
21,FC022,051105_sahubuddin_arrest,राष्ट्रीय जनता दल के विवादास्पद सांसद मोहम्मद शहाबुद्दीन को दिल्ली में शनिवार को उनके सरकारी आवास से गिरफ़्तार कर लिया गया.,शहाबुद्दीन बिहार के सीवान क्षेत्र से सांसद हैं और पीटीआई के अनुसार उन्हें दिल्ली स्थित उनके सांसद निवास से बिहार पुलिस के एक दल ने गिरफ़्तार किया. इसके पहले उनकी गिरफ़्तारी को लेकर बिहार सरकार ने देशव्यापी अलर्ट घोषित कर दिया था. बिहार की पुलिस ने सभी राज्यों की पुलिस को सूचित किया था कि शहाबुद्दीन के ख़िलाफ़ ग़ैरज़मानती वारंट है और वे जहाँ भी देखे जाएँ उन्हें गिरफ़्तार कर लिया जाए. ग़ौरतलब है कि इसके पहले जब बिहार पुलिस उन्हें गिरफ़्तार करने दिल्ली पहुँची थी तब वो दिल्ली में ही थे और संसद क...
22,FC023,151130_offline_google_map_india_hk,गूगल मैप ने भारत में ऑफ़लाइन काम करना शुरू कर दिया है. पिछले हफ्ते गूगल ने ये घोषणा कर दी कि आप मैप का इस्तेमाल अपने मोबाइल पर बिना इंटरनेट कनेक्शन के भी कर सकते हैं.,इसका फ़ायदा कई ऐसे लोगों को हो सकता है जो मैप नहीं इस्तेमाल करते हैं क्योंकि उनके पास मोबाइल इंटरनेट नहीं है. कई बार ऐसा होता है कि नेटवर्क में दिक्कतों के कारण आपको डेटा के सिग्नल नहीं मिलते हैं. इससे मैप इस्तेमाल करने में दिक्कत होती है. कई ऐसे देशों को इससे बहुत फ़ायदा होगा जहां डेटा कनेक्टिविटी काफ़ी खराब है. गूगल मैप पहले भी ऑफ़लाइन मोड में काम करता था. लेकिन ये काम करने का तरीका कुछ ऐसा था कि अगर आप किसी इलाके में जा रहे हैं तो उस इलाके के लिए आपको मैप डाउनलोड करना पड़ता था. उसके बाद व...
23,FC024,080416_up_results,उत्तर प्रदेश में हुए उप चुनाव में सत्ताधारी बहुजन समाज पार्टी (बसपा) ने पाँचों सीट जीत ली है.,"आज़मगढ़ और ख़लीलाबाद के संसदीय सीटों के साथ-साथ बिलग्राम, करनैलगंज और मुराद नगर की विधानसभा सीटों पर भी बसपा की जीत हुई है. कांग्रेस और भारतीय जनता पार्टी को इस उपचुनाव में करारा झटका लगा है. एक हद तक बसपा का मुक़ाबला समाजवादी पार्टी से ही रहा. बीएसपी के कुशल तिवारी ने ख़लीलाबाद संसदीय सीट पर समाजवादी पार्टी के बालचंद्र यादव को पैंसठ हज़ार मतों के अंतर से हराया. बीएसपी के अकबर अहमद डंपी ने आज़मगढ़ संसदीय सीट पर जीत दर्ज की. उन्होंने भारतीय जनता पार्टी के टिकट पर चुनाव लड़ने वाले समाजवादी पार..."
24,FC025,140908_amit_shah_political_analysis_vr,करीब दो महीने पहले भारतीय जनता पार्टी के मुख्यालय में पार्टी के नए अध्यक्ष अमित शाह ने जिम्मेदारी संभाली है.,"फ़ोन पर फ़ोन बज रहे हैं. वहां बैठे एक जानकार ने आँखों देखा हाल बयान करते हुए कहा कि अधिकतर फोन महाराष्ट्र से आ रहे थे और फोन करने वालों में कांग्रेस और राष्ट्रवादी कांग्रेस पार्टी के नेता भी शामिल थे. उस जानकार ने बताया कि वो सभी लोग भाजपा में शामिल होने का अनुरोध कर रहे थे. ज़ुबैर अहमद का विश्लेषण भाजपा का दिल्ली मुख्यालय इन दिनों राजनीतिक गतिविधियों से सरगर्म है. राजनाथ सिंह, लाल कृष्ण आडवाणी, सुषमा स्वराज और ऐसे सभी पुराने चेहरे यहाँ नज़र नहीं आते जो कुछ महीने पहले तक कई सालों से नज़र आते..."
25,FC026,131201_karan_with_salman_sk,"सलमान-शाहरुख साथ, न बाबा न! शाहरूख के साथ अपनी दोस्ती के लिए मशहूर करण जौहर ने सलमान को 'कॉफ़ी विद करण' में लिए जाने के सवाल पर कुछ इस तरह जवाब दिया था.","मगर अब सब कुछ बदल गया है. जाने माने फ़िल्मकार करण जौहर ने अपने टीवी चैट शो 'कॉफ़ी विद करण' के तीसरे संस्करण के शुरुआती एपिसोड में सलमान ख़ान को बुलाने का फ़ैसला किया है. सलमान खान इस शो में पहली बार आ रहे हैं. करण जौहर ने अपने ऑफिशियल ट्विटर हैंडल (@karanjohar) पर लिखा है, ""ٍ'कॉफ़ी विद करण' @StarWorldIndia..@Beingsalmankhan पर रविवार रात 9 बजे शुरू हो रहा है. हमारे सबसे पहले अतिथि होंगे सलमान खान. साथ ही होग

In [ ]:
import pandas as pd
import os

claim_path = os.path.join(
    BASE, "07_claim_labels",
    "FactIndic_50_Claim_Labeling_Sheet.csv"
)

claims = pd.read_csv(claim_path)

for col in ["Verdict", "Evidence_from_Article", "Reviewer_Notes"]:
    claims[col] = claims[col].astype("object")

labels = {
    "FC021": (
        "SUPPORTED",
        "फाइनल में सिंधु ने कनाडा की मिशेल ली को 21-15, 21-15 से मात दी जबकि सेमीफाइनल में उन्होंने चीन की क्वालिफायर किन जिनजिंग को हराया था।",
        "The source supports her confidence and the victories described in the quoted statement."
    ),
    "FC022": (
        "SUPPORTED",
        "उन्हें दिल्ली स्थित उनके सांसद निवास से बिहार पुलिस के एक दल ने गिरफ़्तार किया।",
        "Directly confirms the arrest in Delhi on Saturday."
    ),
    "FC023": (
        "SUPPORTED",
        "गूगल मैप पहले भी ऑफ़लाइन मोड में काम करता था। लेकिन ... उस इलाके के लिए आपको मैप डाउनलोड करना पड़ता था।",
        "The article describes offline functionality and its expanded availability. The claim's exact launch timing should be checked against the complete article."
    ),
    "FC024": (
        "SUPPORTED",
        "आज़मगढ़ और ख़लीलाबाद ... बिलग्राम, करनैलगंज और मुराद नगर की विधानसभा सीटों पर भी बसपा की जीत हुई है।",
        "The source confirms BSP victories across the five listed seats."
    ),
    "FC025": (
        "NOT_ENOUGH_INFO",
        "भाजपा का दिल्ली मुख्यालय इन दिनों राजनीतिक गतिविधियों से सरगर्म है।",
        "The excerpt discusses Amit Shah's leadership but does not establish the exact date he assumed responsibility."
    ),
    "FC026": (
        "SUPPORTED",
        "करण जौहर ने अपने टीवी चैट शो 'कॉफ़ी विद करण' के तीसरे संस्करण के शुरुआती एपिसोड में सलमान ख़ान को बुलाने का फ़ैसला किया है।",
        "Directly supports Salman Khan being invited to the show."
    ),
    "FC027": (
        "SUPPORTED",
        "भारत यात्रा के दौरान ट्रंप दिल्ली और अहमदाबाद का दौरा करेंगे।",
        "The source confirms the announced India visit and dates of 24–25 February."
    ),
    "FC028": (
        "SUPPORTED",
        "वर्तमान में पित्रोदा भारतीय प्रधानमंत्री के जन सूचना संरचना और नवप्रवर्तन सलाहकार हैं।",
        "The article supports his role as an adviser and his work on public information infrastructure and innovation."
    ),
    "FC029": (
        "SUPPORTED",
        "शिकायतकर्ता का दावा था कि वह उस महिला से एक होटल की कार पार्किंग में मिला था।",
        "The article reports the complaint that the woman was not as attractive as advertised."
    ),
    "FC030": (
        "SUPPORTED",
        "देश का लगभग दो-तिहाई हिस्सा बाढ़ से प्रभावित है।",
        "The article confirms extensive flooding and displacement. The exact number of affected people should be checked in the complete source."
    )
}

for claim_id, (verdict, evidence, notes) in labels.items():
    idx = claims["Claim_ID"] == claim_id
    claims.loc[idx, "Verdict"] = verdict
    claims.loc[idx, "Evidence_from_Article"] = evidence
    claims.loc[idx, "Reviewer_Notes"] = notes

claims.to_csv(claim_path, index=False, encoding="utf-8-sig")

print("Batch FC021–FC030 saved.")
print(claims.loc[20:29, ["Claim_ID", "Verdict"]].to_string(index=False))
print("\nTotal labelled:", claims["Verdict"].notna().sum())
print("Remaining:", claims["Verdict"].isna().sum())

Batch FC021–FC030 saved.
Claim_ID         Verdict
   FC021       SUPPORTED
   FC022       SUPPORTED
   FC023       SUPPORTED
   FC024       SUPPORTED
   FC025 NOT_ENOUGH_INFO
   FC026       SUPPORTED
   FC027       SUPPORTED
   FC028       SUPPORTED
   FC029       SUPPORTED
   FC030       SUPPORTED

Total labelled: 30
Remaining: 20


In [ ]:
import pandas as pd
import os

claim_path = os.path.join(
    BASE, "07_claim_labels",
    "FactIndic_50_Claim_Labeling_Sheet.csv"
)

claims = pd.read_csv(claim_path)

print("Already labelled:", claims["Verdict"].notna().sum())

display(
    claims.loc[30:39, [
        "Claim_ID", "Article_ID", "Claim", "Source_Article"
    ]]
)

Already labelled: 30


,Claim_ID,Article_ID,Claim,Source_Article
30,FC031,international-41960891,अमरीकी राष्ट्रपति डोनल्ड ट्रंप ने एक बार फिर से उत्तर कोरिया के शासक किम जोंग-उन पर ज़ुबानी हमला किया है.,"किम जोंग-उन और डोनल्ड ट्रंप ट्रंप ने ट्विटर के ज़रिए किम जोंग-उन को निशाना बनाया. ट्विटर का शिकायती लहजे में अमरीकी राष्ट्रपति ने कहा कि उन्होंने कभी किम जोंग-उन को 'नाटा और मोटा' नहीं कहा. इससे पहले ट्रंप को उत्तर कोरियाई विदेश मंत्रालय ने 'बूढ़ा' कहा था. ट्रंप ने ट्विटर पर इसे लेकर एक साथ कई ट्वीट किए हैं. उन्होंने उत्तर कोरिया को लेकर मध्यस्थता का भी प्रस्ताव रखा था. ट्रंप ने दक्षिण चीन सागर के विवाद को भी सुलझाने को लेकर मध्यस्थता की बात कही है. दक्षिण चीन सागर में चीन, वियतनाम, फ़िलीपीं..."
31,FC032,121207_hangover_smoking_aa,"एक नया अमरीकी अध्ययन कहता है कि अगर किसी ने जम कर पी हो और फिर वो सिगरेट पी ले तो इससे नशा और चढ़ सकता है. इतना, कि अगले दिन तक भी ठीक से न उतर पाए.","कई संस्थान इस अध्ययन को खासा अहम मान रहे हैं हालांकि इस शोध में ये नहीं बताया गया है कि धूम्रपान की वजह से हैंगोवर बिगड़ने की वजह क्या है. शोधकर्ताओं ने 113 अमरीकी छात्रों से कहा कि वो आठ हफ्तों तक पीने और धूम्रपान की अपनी आदतों से जुड़ी बातें और हैंगोवर के लक्षणों के बारे में डायरी में लिखें. जब उन्होंने जम की पी, मसलन एक घंटे में बीयर के छह कैन पी गए तो उस पर धूम्रपान करने वालों का हाल कहीं ज्यादा बुरा था. इस अध्ययन के नतीजे ‘जरनल ऑफ स्टडीज ऑन एल्होकल एंड ड्रग्स’ में प्रकाशित हुए हैं. 'जटि..."
32,FC033,060106_rajasthan_rail,भारत के राजस्थान और पाकिस्तान के सिंध प्रांत के बीच रेल सेवा फिर शुरू करने पर दोनों देशों के बीच सहमति हो गई है.,थार एक्सप्रेस नाम से शुरू होने वाली ये रेल सेवा राजस्थान के मुनाबाव और पाकिस्तान के खोखरापार के बीच चलेगी और यह एक फ़रवरी 2006 से शुरू होगी. ये घोषणा दिल्ली में दोनों देशों के अधिकारियों के बीच हुई बातचीत के बाद शुक्रवार को की गई. दोनों देशों के बीच यह दूसरा रेल संपर्क होगा. एक रेल संपर्क समझौता एक्सप्रेस के ज़रिए है जो दोनों देशों में पंजाब की सीमाओं के बीच चलती है. मुनाबाव और खोखरापार के बीच रेल संपर्क 1965 में हुई लड़ाई के समय टूट गया था और यह रेल सेवा तभी से बंद है. भारत और पाकिस्तान ने ...
33,FC034,040923_manmohan_speech,प्रधानमंत्री मनमोहन सिंह ने संयुक्त राष्ट्र महासभा के वार्षिक अधिवेशन को संबोधित करते हुए आतंकवाद से निबटने के दोहरे मापदंडों पर चिंता व्यक्त की और पाकिस्तान के साथ जम्मू-कश्मीर सहित सभी मुद्दों पर बातचीत जारी रखने को लेकर प्रतिबद्धता ज़ाहिर की.,"डॉक्टर सिंह ने संयुक्त राष्ट्र सुरक्षा परिषद की सदस्यता के लिए भारत की दावेदारी भी पेश की है. पाकिस्तान के साथ जारी बातचीत का उल्लेख करते हुए प्रधानमंत्री ने कहा कि भारत जम्मू-कश्मीर सहित सभी मुद्दों पर पाकिस्तान के साथ सार्थक बातचीत के लिए प्रतिबद्ध है. इसे पाकिस्तान के राष्ट्रपति परवेज़ मुशर्रफ़ के संयुक्त राष्ट्र महासभा में अपनाए गए नरम रुख़ का जवाब भी माना जा रहा है. मनमोहन सिंह ने कहा,""भारत और पाकिस्तान ने कश्मीर समेत सभी मुद्दों को सुलझाने के लिए समग्र बातचीत शुरू की है. मैं ये भरोसा द..."
34,FC035,061004_askus_geyser,भूगोल संबंधी प्रश्न पूछा है मौलाबाग़ आरा भोजपुर बिहार से धीरेन्द्र सिंह यादव ने कि गीज़र क्या है.,गीज़र एक प्रकार का गर्म पानी का सोता होता है जो समय-समय पर फूटता है और इसमें से गर्म पानी और भाप निकलती है. गीज़र के लिए अनुकूल जल भौमिकी की आवश्यकता होती है जो पृथ्वी के कुछ ही स्थानों पर मौजूद है. ये गीज़र कैसे बनते हैं... पृथ्वी की सतह का पानी ज़मीन के अंदर जाकर जमा होता है और ऐसी चट्टान से टकराता है जो मैग्मा से तप रही हो. इससे पानी गर्म होने लगता है और फिर ऊपर उठता है और किसी चट्टान की दरार से होकर बाहर फूट पड़ता है. दुनिया में कोई एक हज़ार गीज़र हैं जिनमें से आधे अमरीका के यैलोस्टोन ने...
35,FC036,international-47823342,दुनिया के सबसे अमीर व्यक्ति अमेज़ॉन के बॉस जेफ़ बेज़ोस और उनकी पत्नी मैकेंज़ी के बीच रिकॉर्ड 35 बिलियन डॉलर (क़रीब 2420 अरब रुपये) के समझौते पर तलाक़ को लेकर सहमति बन गई है.,दुनिया के सबसे अमीर व्यक्ति अमेज़ॉन के बॉस जेफ़ बेज़ोस और उनकी पत्नी मैकेंज़ी तलाक़ सहमति के मुताबिक अब 25 साल पुरानी रिटेल कंपनी अमेज़ॉन में मैकेंज़ी की हिस्सेदारी 4 फ़ीसदी ही रहेगी. साथ ही अब मैकेंज़ी वाशिंगटन पोस्ट न्यूज़पेपर और बेज़ोस की स्पेस ट्रैवल फर्म ब्लू ओरिजिन में भी अपनी दिलचस्पी छोड़ देंगी. बेज़ोस और मैकेंज़ी के बीत 35 बिलियन डॉलर राशि पर बने समझौते ने फ़्

In [ ]:
import pandas as pd
import os

claim_path = os.path.join(
    BASE, "07_claim_labels",
    "FactIndic_50_Claim_Labeling_Sheet.csv"
)

claims = pd.read_csv(claim_path)

for col in ["Verdict", "Evidence_from_Article", "Reviewer_Notes"]:
    claims[col] = claims[col].astype("object")

labels = {
    "FC031": (
        "SUPPORTED",
        "ट्रंप ने ट्विटर के ज़रिए किम जोंग-उन को निशाना बनाया।",
        "The article reports Trump's critical tweets about Kim Jong-un."
    ),
    "FC032": (
        "SUPPORTED",
        "जब उन्होंने जम कर पी ... तो उस पर धूम्रपान करने वालों का हाल कहीं ज्यादा बुरा था।",
        "The study found worse hangover symptoms among participants who smoked after heavy drinking."
    ),
    "FC033": (
        "SUPPORTED",
        "थार एक्सप्रेस नाम से शुरू होने वाली ये रेल सेवा राजस्थान के मुनाबाव और पाकिस्तान के खोखरापार के बीच चलेगी।",
        "The article confirms the agreement to restart rail service between the two regions."
    ),
    "FC034": (
        "SUPPORTED",
        "भारत जम्मू-कश्मीर सहित सभी मुद्दों पर पाकिस्तान के साथ सार्थक बातचीत के लिए प्रतिबद्ध है।",
        "Directly confirms India's commitment to continued talks on all issues, including Jammu and Kashmir."
    ),
    "FC035": (
        "SUPPORTED",
        "गीज़र एक प्रकार का गर्म पानी का सोता होता है जो समय-समय पर फूटता है और इसमें से गर्म पानी और भाप निकलती है।",
        "The source directly defines a geyser as a periodically erupting hot spring."
    ),
    "FC036": (
        "SUPPORTED",
        "बेज़ोस और मैकेंज़ी के बीच 35 बिलियन डॉलर राशि पर बने समझौते...",
        "The article directly confirms the $35 billion divorce settlement."
    ),
    "FC037": (
        "NOT_ENOUGH_INFO",
        "मुफ्ती बशीर ने मुसलमान लड़कियों के गाना गाने को इस्लाम विरूद्ध करार देते हुए...",
        "The excerpt describes the controversy and support for the band but does not clearly establish the stated court challenge by the human-rights group."
    ),
    "FC038": (
        "SUPPORTED",
        "बुधवार को उत्तर कोरिया ने चेतावनी दी कि वो हो सकता है कि वो इस बातचीत का हिस्सा ना बने।",
        "The source confirms uncertainty about the planned Singapore talks."
    ),
    "FC039": (
        "SUPPORTED",
        "इस आत्मकथा का विमोचन बुधवार को देश के पूर्व राष्ट्रपति एपीजे अब्दुल कलाम ने दिल्ली में किया।",
        "The article directly confirms the book's title, author and autobiography launch."
    ),
    "FC040": (
        "SUPPORTED",
        "टेनिस मुक़ाबले के लिए लिएंडर पेस को वाइल्ड कार्ड मिला लेकिन वो फ़ैसला भी वाइल्ड कार्ड की बर्बादी ही लगा...",
        "The source confirms the context of India's low expectations and Paes receiving a wild card. The full claim's description of public disappointment is broadly supported."
    )
}

for claim_id, (verdict, evidence, notes) in labels.items():
    idx = claims["Claim_ID"] == claim_id
    claims.loc[idx, "Verdict"] = verdict
    claims.loc[idx, "Evidence_from_Article"] = evidence
    claims.loc[idx, "Reviewer_Notes"] = notes

claims.to_csv(claim_path, index=False, encoding="utf-8-sig")

print("Batch FC031–FC040 saved.")
print(claims.loc[30:39, ["Claim_ID", "Verdict"]].to_string(index=False))
print("\nTotal labelled:", claims["Verdict"].notna().sum())
print("Remaining:", claims["Verdict"].isna().sum())

Batch FC031–FC040 saved.
Claim_ID         Verdict
   FC031       SUPPORTED
   FC032       SUPPORTED
   FC033       SUPPORTED
   FC034       SUPPORTED
   FC035       SUPPORTED
   FC036       SUPPORTED
   FC037 NOT_ENOUGH_INFO
   FC038       SUPPORTED
   FC039       SUPPORTED
   FC040       SUPPORTED

Total labelled: 40
Remaining: 10


In [ ]:
import pandas as pd
import os

claim_path = os.path.join(
    BASE, "07_claim_labels",
    "FactIndic_50_Claim_Labeling_Sheet.csv"
)

claims = pd.read_csv(claim_path)

print("Already labelled:", claims["Verdict"].notna().sum())

display(
    claims.loc[40:49, [
        "Claim_ID", "Article_ID", "Claim", "Source_Article"
    ]]
)

Already labelled: 40


,Claim_ID,Article_ID,Claim,Source_Article
40,FC041,071227_musharraf_statement,पाकिस्तान के राष्ट्रपति परवेज़ मुशर्रफ़ ने पूर्व प्रधानमंत्री बेनज़ीर भुट्टो की हत्या को 'आतंकवादी कार्रवाई' करार दिया है.,मुशर्रफ़ ने कहा है कि आतंकवाद पाकिस्तान की तरक्की की राह में सबसे बड़ी रुकावट है. टेलीविज़न पर दिए गए एक बयान में उन्होंने कहा कि वो तब तक चैन से नहीं बैठेंगे जब तक देश से दहशतगर्दों का ख़ात्मा नहीं कर दिया जाता. उन्होंने आतंकवाद के ख़ात्मे के लिए पाकिस्तानी जनता से समर्थन की अपील की और कहा ' पाकिस्तान की तरक्की में सबसे बड़ी रुकावट दहशतगर्दी है और वो इसे जड़ से उखाड़ने की कोशिश कर रहे हैं. ' इसके साथ ही उन्होंने पाकिस्तानी जनता से शांति और अमन चैन बनाए रखने की अपील भी की और कहा कि लोग सब्र ...
41,FC042,050606_kabir_bedi,बहुमुखी प्रतिभा वाले अभिनेता कबीर बेदी की ख़्वाहिश है कि वह एक बार फिर से दिल्ली की गलियों में अपनी ज़िंदगी गुज़ार सकें तो अपनी मिट्टी से नाता संपूर्ण जैसा हो जाएगा.,"ग़ौरतलब है कि कबीर बेदी की पैदाइश और परवरिश दिल्ली में ही हुई है और दिल्ली ने ही उन्हें कठिन समय से गुज़रते हुए एक प्रतिभाशाली अभिनेता के रूप में उभरते हुए देखा है. बाद के दिनों में कबीर के अभिनय का सिक्का ना केवल मुंबई फ़िल्म नगरी में चला बल्कि उन्होंने अंतरराष्ट्रीय फ़िल्म जगत में भी अपने जलवे दिखाए. अभिनय के इन्हीं मुकामों को तय करते कबीर का समय इसके बाद मुंबई से लेकर यूरोप के विभिन्न शहरों में बीतता रहा है, लेकिन गाहे-बगाहे दिल्ली की याद उनके ज़हन में तैर जाया करती है. लंदन में बीबीसी के..."
42,FC043,050112_nepalmao_amnesty,लंदन स्थित मानवाधिकार संगठन एमनेस्टी इंटरनेशनल ने नेपाल में माओवादी विद्रोहियों पर मावाधिकार हनन के गंभीर आरोप लगाए हैं.,संगठन ने माओवादी नेताओं से अनुरोध किया है कि वे इन मामलों की जाँच करें और दोषी व्यक्तियों के ख़िलाफ़ कार्रवाई करें. नेपाली माओवादियों के नेता प्रचंड के नाम लिखे एक 'खुले पत्र' में एमनेस्टी ने कहा है कि मानवाधिकार हनन के विशेष मामले उसके सामने आए हैं. संगठन के एशिया-प्रशांत क्षेत्र के निदेशक इंग्रिड मसाज का कहना है कि प्रचंड सभी आम नागरिकों की सुरक्षा सुनिश्चित करें और अगवाह किए गए लोगों को बिना शर्त छोड़ दें. मसाज का कहना था कि कई लोगों को इसलिए निशाना बनाया गया क्योंकि उन्होंने माओवादियों क...
43,FC044,061025_russia_broadcast,रूस के राष्ट्रपति व्लादिमीर पुतिन ने कहा है कि अग़र अंतरराष्ट्रीय समुदाय उत्तर कोरिया के परमाणु कार्यक्रम पर अंकुश लगाना चाहता है तो उस पर ज़्यादा दबाव नहीं डालना चाहिए.,"व्लादिमीर पुतिन ने ये बात एक लाइव टीवी फ़ोन-इन कार्यक्रम में कही. इस कार्यक्रम में राष्ट्रपति पुतिन रूस भर से लोगों के सवालों का जवाब दे रहे थे. रूसी राष्ट्रपति ने कहा, "" बातचीत की प्रकिय्रा के तहत किसी भी देश पर इतना दबाव नहीं डालना चाहिए."" उन्होंने कहा कि वार्ताकार उत्तर कोरिया से 'सही तालमेल' नहीं बिठा पाए हैं. व्लादिमीर पुतिन का कहना था कि 'सदभावना' से हल निकाला जा सकता है. पुतिन का फ़ोन-इन कार्यक्रम रूस के राष्ट्रीय टेलीविज़न नेटवर्कों और बड़े रेडियो स्टेशनों पर प्रसारित किया गया. लोगों..."
44,FC045,india-39447746,पत्रकार साक्षी जोशी पर आपत्तिजनक टिप्पणी करने के मामले में उत्तर प्रदेश पुलिस ने गुजरात से संदीप उपाध्याय को गिरफ़्तार कर लिया है.,पुलिस के शिकंजे में आया संदीप उपाध्याय साक्षी जोशी ने फ़ेसबुक पोस्ट के ज़रिए इसकी जानकारी दी है. हाल ही में साक्षी ने फ़ेसबुक पर अपना स्टेटस अपडेट किया था. उस स्टेटस पर संदीप उपाध्याय नाम के एक शख़्स ने कहा था कि अगर उत्तर प्रदेश में रहना है तो योगी आदित्यनाथ का समर्थन करना होगा. साक्षी ने कहा था कि वह यूपी रहेंगी और योगी-योगी नहीं कहेंगी. इस पर संदीप ने रेप की धमकी दी थी. इस मामले में साक्षी नोएडा पुलिस में शिकायत दर्ज कराई थी. साक्षी ने फ़ेसबुक पोस्ट से जानकारी दी है कि संदीप को गिरफ़्तार ...
45,FC046,141007_bhutan_corruption_india_vr,भूटान में भ्रष्टाचार... कुछ साल पहले तक ये असंभव सा लगता था पर अब ट्रांसपेरेंसी इंटरनेशनल के 177 भ्रष्टतम देशों की सूची में कम भ्रष्ट देश भूटान (31वां) ने 94 वें स्थान पर काबिज भारत से इस नई चुनौती से निपटने के लिए मदद मांगी है.,ये भी चौंकाने वाली बात लगती है पर सच है कि जमीनी स्तर पर भ्रष्टाचार से निपटने में भारत ने कई नए तरीकों का इस्तेमाल किया है जो सफल भी हुए हैं खास तौर पर सोशल ऑडिट का तरीका जहां गांव और ज़िला स्तर पर जन सुनवाई होती है. भूटान में भ्रष्टाचार का स्तर अभी वैसा कहीं से नहीं है जैसा भारत में देखने को मिलता है. वहां ऐेसे मा

In [ ]:
import pandas as pd
import os

claim_path = os.path.join(
    BASE, "07_claim_labels",
    "FactIndic_50_Claim_Labeling_Sheet.csv"
)

claims = pd.read_csv(claim_path)

for col in ["Verdict", "Evidence_from_Article", "Reviewer_Notes"]:
    claims[col] = claims[col].astype("object")

labels = {
    "FC041": (
        "NOT_ENOUGH_INFO",
        "मुशर्रफ़ ने कहा है कि आतंकवाद पाकिस्तान की तरक्की की राह में सबसे बड़ी रुकावट है।",
        "The displayed excerpt discusses Musharraf's statement against terrorism but does not establish that he specifically called Benazir Bhutto's murder a terrorist act."
    ),
    "FC042": (
        "NOT_ENOUGH_INFO",
        "कबीर बेदी की पैदाइश और परवरिश दिल्ली में ही हुई है ... दिल्ली की याद उनके ज़हन में तैर जाया करती है।",
        "The excerpt supports his connection with Delhi and nostalgia, but does not clearly establish the specific wish described in the claim."
    ),
    "FC043": (
        "SUPPORTED",
        "मानवाधिकार हनन के विशेष मामले उसके सामने आए हैं।",
        "The source reports Amnesty International's allegations of human-rights violations by Maoist rebels and calls for investigation and action."
    ),
    "FC044": (
        "SUPPORTED",
        "बातचीत की प्रकिय्रा के तहत किसी भी देश पर इतना दबाव नहीं डालना चाहिए।",
        "The source directly reports Putin's view that excessive pressure should not be applied to North Korea."
    ),
    "FC045": (
        "SUPPORTED",
        "संदीप को गिरफ़्तार ... कर लिया गया है।",
        "The source directly confirms Sandeep Upadhyay's arrest in connection with the objectionable comments and threat."
    ),
    "FC046": (
        "NOT_ENOUGH_INFO",
        "भूटान में भ्रष्टाचार का स्तर अभी वैसा कहीं से नहीं है जैसा भारत में देखने को मिलता है।",
        "The excerpt discusses Bhutan seeking India's help, but does not establish the exact ranking figures and comparison stated in the claim."
    ),
    "FC047": (
        "SUPPORTED",
        "ये साफ़ नहीं है कि फांसी पर लटकाए गए विद्रोही सीमा पर हुई हत्याओं से सीधे तौर पर जुड़े हुए थे या नहीं।",
        "The source reports the executions and the killing of 17 guards, while explicitly noting that a direct connection was unclear."
    ),
    "FC048": (
        "SUPPORTED",
        "इस साल जनवरी से लेकर 25 फ़रवरी तक इस बीमारी से 1,000 से अधिक लोगों की जान जा चुकी है और 16,000 से ज़्यादा प्रभावित हैं।",
        "The source reports more than 1,000 deaths and 16,000 affected people, and describes restrictions and public-health concerns."
    ),
    "FC049": (
        "SUPPORTED",
        "चर्चिल सऊदी किंग से ये तोहफ़े लेकर बेहद शर्मिंदा हुए और उनके लिए रॉल्स रॉयस कार तोहफ़े में भेजी।",
        "The source confirms Churchill's visit and the gifts exchanged, including his later gift of a Rolls-Royce. The exact £100 value is not established in the displayed excerpt."
    ),
    "FC050": (
        "SUPPORTED",
        "मैं और युवराज कैंसर के बारें में बात नहीं करते ... हम इसे बार- बार याद कर दुखी नही होना चाहते।",
        "The source directly supports the statement attributed to Yuvraj Singh's mother about not revisiting the cancer experience."
    )
}

for claim_id, (verdict, evidence, notes) in labels.items():
    idx = claims["Claim_ID"] == claim_id
    claims.loc[idx, "Verdict"] = verdict
    claims.loc[idx, "Evidence_from_Article"] = evidence
    claims.loc[idx, "Reviewer_Notes"] = notes

claims.to_csv(claim_path, index=False, encoding="utf-8-sig")

print("Final batch FC041–FC050 saved.")
print(claims.loc[40:49, ["Claim_ID", "Verdict"]].to_string(index=False))
print("\nTotal labelled:", claims["Verdict"].notna().sum())
print("Remaining:", claims["Verdict"].isna().sum())

Final batch FC041–FC050 saved.
Claim_ID         Verdict
   FC041 NOT_ENOUGH_INFO
   FC042 NOT_ENOUGH_INFO
   FC043       SUPPORTED
   FC044       SUPPORTED
   FC045       SUPPORTED
   FC046 NOT_ENOUGH_INFO
   FC047       SUPPORTED
   FC048       SUPPORTED
   FC049       SUPPORTED
   FC050       SUPPORTED

Total labelled: 50
Remaining: 0


In [ ]:
import pandas as pd
import os

claim_path = os.path.join(
    BASE, "07_claim_labels",
    "FactIndic_50_Claim_Labeling_Sheet.csv"
)

claims = pd.read_csv(claim_path)

print("TOTAL CLAIMS:", len(claims))
print("\nVERDICT COUNTS:")
print(claims["Verdict"].value_counts(dropna=False))

print("\nMISSING VALUES:")
print(claims[
    ["Claim_ID", "Verdict", "Evidence_from_Article", "Reviewer_Notes"]
].isna().sum())

print("\nDUPLICATE CLAIM IDs:", claims["Claim_ID"].duplicated().sum())

print("\nVALID VERDICTS:")
valid = {"SUPPORTED", "CONTRADICTED", "NOT_ENOUGH_INFO"}
print(claims["Verdict"].isin(valid).all())

print("\nCLAIM LABELING FILE:")
print(claim_path)

TOTAL CLAIMS: 50

VERDICT COUNTS:
Verdict
SUPPORTED          40
NOT_ENOUGH_INFO    10
Name: count, dtype: int64

MISSING VALUES:
Claim_ID                 0
Verdict                  0
Evidence_from_Article    0
Reviewer_Notes           0
dtype: int64

DUPLICATE CLAIM IDs: 0

VALID VERDICTS:
True

CLAIM LABELING FILE:
/content/drive/MyDrive/FactIndic_PersonB_CleanRun/07_claim_labels/FactIndic_50_Claim_Labeling_Sheet.csv


In [ ]:
import os
import pandas as pd

BASE = "/content/drive/MyDrive/FactIndic_PersonB_CleanRun"

files_to_check = [
    "02_clean_data/XL_Sum_Hindi_Cleaned_250.csv",
    "03_english_data/XL_Sum_English_250.csv",
    "03_english_data/FactIndic_Hindi_English_Base_250.csv",
    "04_mixed_data/FactIndic_Mixed_250.csv",
    "09_checkpoints/progress.json"
]

for file in files_to_check:
    path = os.path.join(BASE, file)
    print("\n", file)
    print("Exists:", os.path.exists(path))

    if path.endswith(".csv") and os.path.exists(path):
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())


 02_clean_data/XL_Sum_Hindi_Cleaned_250.csv
Exists: True
Shape: (250, 7)
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words']

 03_english_data/XL_Sum_English_250.csv
Exists: True
Shape: (250, 10)
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words', 'english_title', 'english_text', 'english_summary']

 03_english_data/FactIndic_Hindi_English_Base_250.csv
Exists: True
Shape: (250, 13)
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words', 'nllb_english_title', 'nllb_english_article', 'nllb_english_summary', 'english_title', 'english_text', 'english_summary']

 04_mixed_data/FactIndic_Mixed_250.csv
Exists: True
Shape: (250, 16)
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words', 'english_title', 'english_text', 'english_summary', 'mixed_10_article', 'mixed_30_article', 'mixed_50_article', 'mixed_10_article_english_pct', 'mixed_30_article_english_pct', 'mixed_50_

In [ ]:
import pandas as pd
import os

BASE = "/content/drive/MyDrive/FactIndic_PersonB_CleanRun"

path = os.path.join(
    BASE, "03_english_data",
    "FactIndic_Hindi_English_Base_250.csv"
)

df = pd.read_csv(path)

for i in range(5):
    row = df.iloc[i]

    print("\n" + "=" * 90)
    print("ARTICLE:", row["id"])

    print("\nHINDI TITLE:")
    print(row["title"])

    print("\nNLLB ENGLISH TITLE:")
    print(row["nllb_english_title"])

    print("\nOPUS ENGLISH TITLE:")
    print(row["english_title"])

    print("\nHINDI SUMMARY:")
    print(row["summary"])

    print("\nNLLB ENGLISH SUMMARY:")
    print(row["nllb_english_summary"])

    print("\nOPUS ENGLISH SUMMARY:")
    print(row["english_summary"])


ARTICLE: 080419_brchopra_festival

HINDI TITLE:
बीआर चोपड़ा की यादगार फ़िल्मों का विशेष उत्सव

NLLB ENGLISH TITLE:
A special celebration of B. R. Chopra's memorable films.

OPUS ENGLISH TITLE:
Special celebration of the memorable film of Brghbourn

HINDI SUMMARY:
भारतीय फ़िल्म इतिहास में फ़िल्मकार बीआर चोपड़ा का नाम बड़ा अदब से लिया जाता है. उनके 94वें जन्मदिन के मौके पर दिल्ली में बीआर चोपड़ा की फ़िल्मों का विशेष उत्सव मनाया जा रहा है.

NLLB ENGLISH SUMMARY:
In the history of Indian cinema, the name of filmmaker B. R. Chopra is taken from a great tradition. A special celebration of B. R. Chopra's films is being held in Delhi on the occasion of his 40th birthday.

OPUS ENGLISH SUMMARY:
In Indian film history, movie actor BR Bangn is a popular popular celebration in Delhi on their 94th birthday date.

ARTICLE: 080613_afghan_jail

HINDI TITLE:
तालेबान ने सैकड़ों क़ैदियों को छुड़ाया

NLLB ENGLISH TITLE:
nan

OPUS ENGLISH TITLE:
The Locker Delivered Hundreds of Prisoners

HINDI SUMMARY:
अ

In [ ]:
import pandas as pd
import os

path = os.path.join(
    BASE, "03_english_data",
    "FactIndic_Hindi_English_Base_250.csv"
)

df = pd.read_csv(path)

columns = [
    "nllb_english_title",
    "nllb_english_article",
    "nllb_english_summary",
    "english_title",
    "english_text",
    "english_summary"
]

for col in columns:
    if col in df.columns:
        valid = (
            df[col].notna()
            & df[col].astype(str).str.strip().ne("")
            & df[col].astype(str).str.lower().ne("nan")
        ).sum()

        print(f"{col}: {valid}/250 available")

print("\nNLLB complete rows:")
nllb_cols = columns[:3]
print(df[nllb_cols].notna().all(axis=1).sum(), "/250")

print("\nOPUS complete rows:")
opus_cols = columns[3:]
print(df[opus_cols].notna().all(axis=1).sum(), "/250")

nllb_english_title: 1/250 available
nllb_english_article: 1/250 available
nllb_english_summary: 1/250 available
english_title: 250/250 available
english_text: 250/250 available
english_summary: 250/250 available

NLLB complete rows:
1 /250

OPUS complete rows:
250 /250


In [ ]:
import torch
import transformers
import psutil

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("GPU: Not available")

print("RAM (GB):", round(psutil.virtual_memory().total / 1e9, 2))
print("Available RAM (GB):", round(psutil.virtual_memory().available / 1e9, 2))
print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)

GPU available: False
GPU: Not available
RAM (GB): 13.61
Available RAM (GB): 10.63
Transformers: 4.57.6
PyTorch: 2.14.1+cpu


In [ ]:
from huggingface_hub import model_info

model_id = "ai4bharat/indictrans2-indic-en-1B"

try:
    info = model_info(model_id)
    print("Model accessible:", info.id)
    print("Model:", model_id)
except Exception as e:
    print("Model access issue:")
    print(str(e)[:1000])

Model accessible: ai4bharat/indictrans2-indic-en-1B
Model: ai4bharat/indictrans2-indic-en-1B


In [ ]:
import requests

urls = {
    "Old AI4Bharat API": "http://216.48.181.177:5050/docs",
    "AI4Bharat model demo": "https://models.ai4bharat.org"
}

for name, url in urls.items():
    print("\nChecking:", name)
    try:
        response = requests.get(url, timeout=15)
        print("HTTP status:", response.status_code)
        print("Reachable:", response.status_code < 500)
        print("Final URL:", response.url)
    except Exception as e:
        print("Unavailable:", str(e)[:250])


Checking: Old AI4Bharat API
Unavailable: HTTPConnectionPool(host='216.48.181.177', port=5050): Max retries exceeded with url: /docs (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7d33410c1550>: Failed to establish a new connection: [Errno 113] No route to host

Checking: AI4Bharat model demo
HTTP status: 200
Reachable: True
Final URL: https://models.ai4bharat.org/


In [ ]:
import requests

url = "https://models.ai4bharat.org/"

response = requests.get(url, timeout=20)

print("Status:", response.status_code)
print("Page title:")
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "html.parser")
print(soup.title.get_text(strip=True) if soup.title else "No title")

print("\nRelevant links:")
for link in soup.find_all("a", href=True):
    href = link["href"]
    text = link.get_text(" ", strip=True)

    if any(word in (href + text).lower()
           for word in ["translate", "indictrans", "api", "model"]):
        print(text[:100], "->", href)

Status: 200
Page title:
AI4Bharat Models

Relevant links:


In [ ]:
from huggingface_hub import login

login()

In [ ]:
from huggingface_hub import model_info

model_id = "ai4bharat/indictrans2-indic-en-dist-200M"

try:
    info = model_info(model_id)
    print("Model access confirmed!")
    print("Model:", info.id)
except Exception as e:
    print("Access error:", str(e)[:500])

Model access confirmed!
Model: ai4bharat/indictrans2-indic-en-dist-200M


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_id = "ai4bharat/indictrans2-indic-en-dist-200M"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True
)

print("Loading model...")

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype=torch.float32
)

model.eval()

print("\nIndicTrans2 200M loaded successfully!")
print("Device: CPU")

Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json: 0.00B [00:00, ?B/s]

dict.TGT.json: 0.00B [00:00, ?B/s]

... (7 outputs left out)

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/913M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]


IndicTrans2 200M loaded successfully!
Device: CPU


In [ ]:
import torch

print("Testing translation...")

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_length=128,
        num_beams=1,
        use_cache=False
    )

translated = tokenizer.batch_decode(
    output,
    skip_special_tokens=True
)

translated = ip.postprocess_batch(
    translated,
    lang="eng_Latn"
)

print("Hindi:", hindi_text[0])
print("English:", translated[0])

Testing translation...
Hindi: दिल्ली में आज भारी बारिश हुई और कई इलाकों में पानी भर गया।
English: Delhi received heavy rains today and many areas were waterlogged.


In [ ]:
import pandas as pd
import torch
import os

input_path = os.path.join(
    BASE, "02_clean_data", "XL_Sum_Hindi_Cleaned_250.csv"
)

df_250 = pd.read_csv(input_path)

def translate_batch(texts):
    batch = ip.preprocess_batch(
        texts,
        src_lang="hin_Deva",
        tgt_lang="eng_Latn"
    )

    encoded = tokenizer(
        batch,
        truncation=True,
        padding="longest",
        return_tensors="pt"
    )

    with torch.no_grad():
        output = model.generate(
            **encoded,
            max_length=512,
            num_beams=1,
            use_cache=False
        )

    decoded = tokenizer.batch_decode(
        output,
        skip_special_tokens=True
    )

    return ip.postprocess_batch(decoded, lang="eng_Latn")


row = df_250.iloc[0]

pilot_texts = [
    str(row["title"]),
    str(row["text"]),
    str(row["summary"])
]

pilot_translations = translate_batch(pilot_texts)

print("ARTICLE ID:", row["id"])

print("\nHINDI TITLE:")
print(pilot_texts[0])

print("\nENGLISH TITLE:")
print(pilot_translations[0])

print("\nHINDI SUMMARY:")
print(pilot_texts[2])

print("\nENGLISH SUMMARY:")
print(pilot_translations[2])

print("\nHINDI ARTICLE (first 500 characters):")
print(pilot_texts[1][:500])

print("\nENGLISH ARTICLE (first 500 characters):")
print(pilot_translations[1][:500])

ARTICLE ID: 080419_brchopra_festival

HINDI TITLE:
बीआर चोपड़ा की यादगार फ़िल्मों का विशेष उत्सव

ENGLISH TITLE:
Special festival of memorable films of BR Chopra

HINDI SUMMARY:
भारतीय फ़िल्म इतिहास में फ़िल्मकार बीआर चोपड़ा का नाम बड़ा अदब से लिया जाता है. उनके 94वें जन्मदिन के मौके पर दिल्ली में बीआर चोपड़ा की फ़िल्मों का विशेष उत्सव मनाया जा रहा है.

ENGLISH SUMMARY:
Filmmaker BR Chopra's name is taken from the big adab in Indian film history. On his 94th birthday, a special festival of BR Chopra's films is being celebrated in Delhi.

HINDI ARTICLE (first 500 characters):
शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को

In [ ]:
for i in [1, 2, 3]:
    row = df_250.iloc[i]

    translated = translate_batch([
        str(row["title"]),
        str(row["summary"])
    ])

    print("\nARTICLE:", row["id"])
    print("TITLE:", translated[0])
    print("SUMMARY:", translated[1])


ARTICLE: 080613_afghan_jail
TITLE: The Taliban have released hundreds of prisoners.
SUMMARY: Taliban militants have attacked a prison in Kandahar, Afghanistan, freeing hundreds of prisoners.

ARTICLE: 130912_guinnness_record_bulgarian_swim_sr
TITLE: Swimming with hands and feet tied for the record
SUMMARY: A Bulgarian man swam for two kilometres in a sack in the hope of getting his name registered in the Guinness World Records.

ARTICLE: 050713_us_un
TITLE: The United States rejects the G-4 proposal
SUMMARY: The United States has rejected a proposal by G-4 countries to the UN General Assembly for the expansion of the Security Council.


In [ ]:
import time

row = df_250.iloc[1]

print("Starting full article translation...")
start = time.time()

article_translation = translate_batch([
    str(row["text"])
])[0]

elapsed = time.time() - start

print("\nArticle ID:", row["id"])
print("Time taken:", round(elapsed / 60, 2), "minutes")
print("\nHindi:", str(row["text"])[:500])
print("\nEnglish:", article_translation[:1000])

Starting full article translation...

Article ID: 080613_afghan_jail
Time taken: 1.98 minutes

Hindi: कंधार के प्रांतीय सभा के अध्यक्ष और राष्ट्रपति हामिद करज़ई के भाई वली करज़ई ने बीबीसी को बताया कि उस समय जेल में एक हज़ार क़ैदी थे और सभी निकल भागे हैं. माना जाता है कि इन क़ैदियों में से 350 तालेबान थे. इस हमले में कई सुरक्षाकर्मियों की जानें गई हैं. अधिकारियों के अनुसार जेल के मुख्य द्वार को उड़ाने के लिए एक लॉरी बम से विस्फोट किया गया और फिर 40 तालेबान जेल के भीतर घुस आए. कंधार शहर में आपातकाल की घोषणा कर दी गई है. पुलिस और सैनिक सड़कों पर गश्त कर रहे हैं और लोगों से कहा गया है कि वे अपने घरो

English: Kandahar provincial assembly chairman and President Hamid Karzai's brother Wali Karzai told the BBC that there were 1,000 prisoners in the prison at the time and all had escaped. Of these prisoners, 350 were believed to be Taliban. Several security personnel have died in the attack. According to officials, a lorry bomb was detonated to blow up the main gate of the prison and then 40 T

In [ ]:
for i in range(5):
    row = df_250.iloc[i]

    print(
        i + 1,
        row["id"],
        "Hindi words:",
        len(str(row["text"]).split())
    )

1 080419_brchopra_festival Hindi words: 294
2 080613_afghan_jail Hindi words: 289
3 130912_guinnness_record_bulgarian_swim_sr Hindi words: 200
4 050713_us_un Hindi words: 245
5 120809_salman_pkp Hindi words: 365


In [ ]:
import os
import time
import pandas as pd

OUTPUT_PATH = os.path.join(
    BASE,
    "03_english_data",
    "IndicTrans2_200M_First_60.csv"
)

# Work on the first 60 articles
work_df = df_250.iloc[:60].copy().reset_index(drop=True)

# Resume from existing checkpoint, if available
if os.path.exists(OUTPUT_PATH):
    result_df = pd.read_csv(OUTPUT_PATH)
    print("Resuming existing progress...")
else:
    result_df = work_df.copy()
    result_df["indic_title"] = ""
    result_df["indic_article"] = ""
    result_df["indic_summary"] = ""

def save_checkpoint():
    result_df.to_csv(
        OUTPUT_PATH,
        index=False,
        encoding="utf-8-sig"
    )

start_time = time.time()
completed = 0

for i in range(len(result_df)):

    # Skip completed rows
    if (
        pd.notna(result_df.loc[i, "indic_title"])
        and str(result_df.loc[i, "indic_title"]).strip()
        and pd.notna(result_df.loc[i, "indic_article"])
        and str(result_df.loc[i, "indic_article"]).strip()
        and pd.notna(result_df.loc[i, "indic_summary"])
        and str(result_df.loc[i, "indic_summary"]).strip()
    ):
        continue

    try:
        texts = [
            str(result_df.loc[i, "title"]),
            str(result_df.loc[i, "text"]),
            str(result_df.loc[i, "summary"])
        ]

        translations = translate_batch(texts)

        result_df.loc[i, "indic_title"] = translations[0]
        result_df.loc[i, "indic_article"] = translations[1]
        result_df.loc[i, "indic_summary"] = translations[2]

        completed += 1

        print(
            f"Completed: {i+1}/60 | "
            f"Newly translated: {completed} | "
            f"Time: {round((time.time()-start_time)/60, 1)} min"
        )

        # Save every 2 completed articles
        if completed % 2 == 0:
            save_checkpoint()
            print("Checkpoint saved to Drive.")

    except Exception as e:
        save_checkpoint()
        print(f"Error at article {i+1}: {e}")
        break

save_checkpoint()

print("\nPROCESSING STOPPED OR COMPLETED")
print("Newly translated:", completed)
print("Total completed:", result_df["indic_article"].notna().sum())
print("Saved file:", OUTPUT_PATH)

Completed: 1/60 | Newly translated: 1 | Time: 5.9 min
Completed: 2/60 | Newly translated: 2 | Time: 11.0 min
Checkpoint saved to Drive.
Completed: 3/60 | Newly translated: 3 | Time: 15.4 min
Completed: 4/60 | Newly translated: 4 | Time: 20.4 min
Checkpoint saved to Drive.
Completed: 5/60 | Newly translated: 5 | Time: 26.5 min
Completed: 6/60 | Newly translated: 6 | Time: 30.8 min
Checkpoint saved to Drive.
Completed: 7/60 | Newly translated: 7 | Time: 35.9 min
Completed: 8/60 | Newly translated: 8 | Time: 40.6 min
Checkpoint saved to Drive.
Completed: 9/60 | Newly translated: 9 | Time: 45.7 min
Completed: 10/60 | Newly translated: 10 | Time: 51.1 min
Checkpoint saved to Drive.
Completed: 11/60 | Newly translated: 11 | Time: 56.7 min
Completed: 12/60 | Newly translated: 12 | Time: 60.9 min
Checkpoint saved to Drive.
Completed: 13/60 | Newly translated: 13 | Time: 64.2 min
Completed: 14/60 | Newly translated: 14 | Time: 69.7 min
Checkpoint saved to Drive.
Completed: 15/60 | Newly transla

In [ ]:
import pandas as pd
import os

path = os.path.join(
    BASE,
    "03_english_data",
    "IndicTrans2_200M_First_60.csv"
)

check_df = pd.read_csv(path)

print("Rows:", len(check_df))

for col in ["indic_title", "indic_article", "indic_summary"]:
    filled = check_df[col].fillna("").astype(str).str.strip().ne("").sum()
    print(f"{col}: {filled}/60 complete")

print("\nFirst article translation:")
print(check_df.loc[0, "indic_title"])
print(check_df.loc[0, "indic_summary"])

Rows: 60
indic_title: 60/60 complete
indic_article: 60/60 complete
indic_summary: 60/60 complete

First article translation:
Special festival of memorable films of BR Chopra
Filmmaker BR Chopra's name is taken from the big adab in Indian film history. On his 94th birthday, a special festival of BR Chopra's films is being celebrated in Delhi.


In [ ]:
import pandas as pd

# Select 10 reproducible random samples
review_df = check_df.sample(n=10, random_state=42).reset_index(drop=True)

for i, row in review_df.iterrows():

    print("\n" + "="*70)
    print(f"SAMPLE {i+1} | ID: {row['id']}")

    print("\nHINDI TITLE:")
    print(row["title"])

    print("\nENGLISH TITLE:")
    print(row["indic_title"])

    print("\nHINDI SUMMARY:")
    print(row["summary"])

    print("\nENGLISH SUMMARY:")
    print(row["indic_summary"])

    print("\nHINDI ARTICLE:")
    print(str(row["text"])[:300])

    print("\nENGLISH ARTICLE:")
    print(str(row["indic_article"])[:400])


SAMPLE 1 | ID: 080419_brchopra_festival

HINDI TITLE:
बीआर चोपड़ा की यादगार फ़िल्मों का विशेष उत्सव

ENGLISH TITLE:
Special festival of memorable films of BR Chopra

HINDI SUMMARY:
भारतीय फ़िल्म इतिहास में फ़िल्मकार बीआर चोपड़ा का नाम बड़ा अदब से लिया जाता है. उनके 94वें जन्मदिन के मौके पर दिल्ली में बीआर चोपड़ा की फ़िल्मों का विशेष उत्सव मनाया जा रहा है.

ENGLISH SUMMARY:
Filmmaker BR Chopra's name is taken from the big adab in Indian film history. On his 94th birthday, a special festival of BR Chopra's films is being celebrated in Delhi.

HINDI ARTICLE:
शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया.

ENGLISH ARTICLE:
The festival, which began on Friday evening, showcased the scale of Insaaf. Classic films like Chhoti Si Baat, Naya Da

In [ ]:
import os
import pandas as pd

review_sheet = check_df[[
    "id",
    "title",
    "indic_title",
    "summary",
    "indic_summary",
    "text",
    "indic_article"
]].copy()

review_sheet.insert(
    1, "Review_Status", "Pending"
)

review_sheet["Title_Issue"] = ""
review_sheet["Summary_Issue"] = ""
review_sheet["Article_Issue"] = ""
review_sheet["Names_Numbers_Issue"] = ""
review_sheet["Overall_Quality_1_to_5"] = ""
review_sheet["Reviewer_Notes"] = ""

review_path = os.path.join(
    BASE,
    "05_quality_checks",
    "IndicTrans2_First_60_Quality_Review.csv"
)

os.makedirs(os.path.dirname(review_path), exist_ok=True)

review_sheet.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Quality-review sheet created!")
print("Rows:", len(review_sheet))
print("Columns:", len(review_sheet.columns))
print("Saved:", review_path)
print("\nReview status:")
print(review_sheet["Review_Status"].value_counts())

Quality-review sheet created!
Rows: 60
Columns: 14
Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/IndicTrans2_First_60_Quality_Review.csv

Review status:
Review_Status
Pending    60
Name: count, dtype: int64


In [ ]:
review_path = os.path.join(
    BASE,
    "05_quality_checks",
    "IndicTrans2_First_60_Quality_Review.csv"
)

review_sheet = pd.read_csv(review_path)

findings = {
    "080419_brchopra_festival": (
        "Needs correction",
        "Awkward phrase: 'name is taken from the big adab'",
        "Film title Insaf Ka Tarazu mistranslated",
        "Needs review",
        "Major wording issue"
    ),
    "080711_lanka_attack": (
        "Needs correction",
        "Title incorrectly says bus accident instead of shooting/attack",
        "Summary appears more accurate",
        "Needs review",
        "Serious meaning error"
    ),
    "041015_envoy_critics": (
        "Minor issue",
        "",
        "Mostly preserves meaning",
        "No obvious issue in checked excerpt",
        "Mostly acceptable"
    ),
    "131201_karan_with_salman_sk": (
        "Needs review",
        "Awkward title formatting",
        "Untranslated phrase in summary",
        "Needs review",
        "Awkward wording"
    ),
    "081126_aamne_pachauri_alk": (
        "Needs review",
        "",
        "Mostly plausible",
        "Needs review",
        "Context check needed"
    ),
    "141229_italy_ferry_rescue_pkp": (
        "No obvious issue",
        "",
        "Mostly preserves facts",
        "No obvious issue in checked excerpt",
        "Promising"
    ),
    "121207_hangover_smoking_aa": (
        "Needs correction",
        "Smoking can worsen a hangover; title changes meaning",
        "Several phrases mistranslated",
        "Needs review",
        "Serious meaning errors"
    ),
    "151122_indo_pak_cricket_meet_ml": (
        "No obvious issue",
        "",
        "Mostly preserves meaning",
        "No obvious issue in checked excerpt",
        "Promising"
    ),
    "150520_emergency_service_ss": (
        "Needs correction",
        "",
        "Mostly plausible",
        "Number translated incorrectly as 20,000 thousand",
        "Numeric error"
    ),
    "081217_ganguly_yuvraj_ac": (
        "No obvious issue",
        "",
        "Mostly preserves meaning",
        "No obvious issue in checked excerpt",
        "Promising"
    )
}

for article_id, values in findings.items():
    mask = review_sheet["id"] == article_id
    if mask.any():
        review_sheet.loc[mask, "Review_Status"] = "Initial review done"
        review_sheet.loc[mask, [
            "Title_Issue",
            "Summary_Issue",
            "Article_Issue",
            "Names_Numbers_Issue",
            "Reviewer_Notes"
        ]] = values

review_sheet.to_csv(review_path, index=False, encoding="utf-8-sig")

print("Updated review sheet.")
print("Initially reviewed:", (review_sheet["Review_Status"] == "Initial review done").sum())
print("Still pending:", (review_sheet["Review_Status"] == "Pending").sum())
print("Saved:", review_path)

Updated review sheet.
Initially reviewed: 10
Still pending: 50
Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/IndicTrans2_First_60_Quality_Review.csv


/tmp/ipykernel_4396/426969401.py:86: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Needs correction' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  review_sheet.loc[mask, [
/tmp/ipykernel_4396/426969401.py:86: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Awkward phrase: 'name is taken from the big adab'' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  review_sheet.loc[mask, [
/tmp/ipykernel_4396/426969401.py:86: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Film title Insaf Ka Tarazu mistranslated' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  review_sheet.loc[mask, [
/tmp/ipykernel_4396/426969401.py:86: Future

In [ ]:
review_sheet = pd.read_csv(review_path)

text_columns = [
    "Review_Status",
    "Title_Issue",
    "Summary_Issue",
    "Article_Issue",
    "Names_Numbers_Issue",
    "Reviewer_Notes"
]

for col in text_columns:
    review_sheet[col] = review_sheet[col].fillna("").astype("string")

review_sheet["Overall_Quality_1_to_5"] = (
    review_sheet["Overall_Quality_1_to_5"].astype("string")
)

review_sheet.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Data types fixed and review sheet saved.")
print(review_sheet["Review_Status"].value_counts())

Data types fixed and review sheet saved.
Review_Status
Pending                50
Initial review done    10
Name: count, dtype: Int64


In [ ]:
review_sheet = pd.read_csv(review_path)

next_batch = review_sheet[
    review_sheet["Review_Status"] == "Pending"
].head(10).copy()

batch_path = os.path.join(
    BASE,
    "05_quality_checks",
    "IndicTrans2_Quality_Review_Next_10.csv"
)

next_batch.to_csv(
    batch_path,
    index=False,
    encoding="utf-8-sig"
)

print("Next review batch prepared!")
print("Articles:", len(next_batch))
print("\nArticle IDs:")
print(next_batch["id"].to_list())
print("\nSaved:", batch_path)

Next review batch prepared!
Articles: 10

Article IDs:
['080613_afghan_jail', '130912_guinnness_record_bulgarian_swim_sr', '050713_us_un', '120809_salman_pkp', 'social-43068605', 'international-51410040', 'india-55525807', 'international-41372477', '130404_sam_pitroda_profile', '140323_breast_cancer_screening_tk']

Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/IndicTrans2_Quality_Review_Next_10.csv


In [ ]:
for i, row in next_batch.iterrows():
    print("\n" + "=" * 65)
    print("ARTICLE:", row["id"])

    print("\nHINDI TITLE:")
    print(row["title"])

    print("\nENGLISH TITLE:")
    print(row["indic_title"])

    print("\nHINDI SUMMARY:")
    print(row["summary"])

    print("\nENGLISH SUMMARY:")
    print(row["indic_summary"])

    print("\nHINDI ARTICLE (first 500 characters):")
    print(str(row["text"])[:500])

    print("\nENGLISH ARTICLE (first 500 characters):")
    print(str(row["indic_article"])[:500])


ARTICLE: 080613_afghan_jail

HINDI TITLE:
तालेबान ने सैकड़ों क़ैदियों को छुड़ाया

ENGLISH TITLE:
The Taliban have released hundreds of prisoners.

HINDI SUMMARY:
अफ़ग़ानिस्तान के कंधार में तालेबान चरमपंथियों ने जेल पर हमला करके सैकड़ों क़ैदियों को छुड़ा दिया है.

ENGLISH SUMMARY:
Taliban militants have attacked a prison in Kandahar, Afghanistan, freeing hundreds of prisoners.

HINDI ARTICLE (first 500 characters):
कंधार के प्रांतीय सभा के अध्यक्ष और राष्ट्रपति हामिद करज़ई के भाई वली करज़ई ने बीबीसी को बताया कि उस समय जेल में एक हज़ार क़ैदी थे और सभी निकल भागे हैं. माना जाता है कि इन क़ैदियों में से 350 तालेबान थे. इस हमले में कई सुरक्षाकर्मियों की जानें गई हैं. अधिकारियों के अनुसार जेल के मुख्य द्वार को उड़ाने के लिए एक लॉरी बम से विस्फोट किया गया और फिर 40 तालेबान जेल के भीतर घुस आए. कंधार शहर में आपातकाल की घोषणा कर दी गई है. पुलिस और सैनिक सड़कों पर गश्त कर रहे हैं और लोगों से कहा गया है कि वे अपने घरो

ENGLISH ARTICLE (first 500 characters):
Kandahar provincial assembly chairman a

In [ ]:
review_sheet = pd.read_csv(review_path)

text_columns = [
    "Review_Status", "Title_Issue", "Summary_Issue",
    "Article_Issue", "Names_Numbers_Issue", "Reviewer_Notes"
]

for col in text_columns:
    review_sheet[col] = review_sheet[col].fillna("").astype("string")

findings_2 = {
    "080613_afghan_jail": (
        "No obvious issue", "", "", "",
        "Names and numbers preserved; initial check looks good"
    ),
    "130912_guinnness_record_bulgarian_swim_sr": (
        "Minor issue", "", "Awkward phrase: 'completing his campaign'; check lake name",
        "", "Mostly good; wording needs polishing"
    ),
    "050713_us_un": (
        "No obvious issue", "", "", "",
        "G-4 countries and 15-to-25 figures preserved"
    ),
    "120809_salman_pkp": (
        "No obvious issue", "",
        "Larger-than-life image mistranslated as 'large day-to-day life image'",
        "", "Meaning error in article excerpt"
    ),
    "social-43068605": (
        "Needs review", "Literal and awkward metaphor translation",
        "'nano' should refer to eyes; 'cottage in hearts' is incorrect",
        "", "Several wording and meaning errors"
    ),
    "international-51410040": (
        "Minor issue", "Missing space before 'mistake'",
        "", "", "Political statement mostly preserved; punctuation needs correction"
    ),
    "india-55525807": (
        "No obvious issue", "", "", "",
        "Meaning and names preserved in checked excerpt"
    ),
    "international-41372477": (
        "Needs correction", "Repeated word: 'Earthquake, earthquake'",
        "Duplicate and awkward text in article excerpt",
        "", "Needs editing for clarity"
    ),
    "130404_sam_pitroda_profile": (
        "No obvious issue", "", "", "",
        "Names, dates and meaning preserved in checked excerpt"
    ),
    "140323_breast_cancer_screening_tk": (
        "Minor issue", "",
        "Formatting error around 'above average' risk",
        "Figures 53,467, 14,593 and 3.5% appear preserved",
        "Mostly good; formatting correction needed"
    )
}

for article_id, values in findings_2.items():
    mask = review_sheet["id"] == article_id
    if mask.any():
        review_sheet.loc[mask, "Review_Status"] = "Initial review done"
        for col, value in zip([
            "Title_Issue", "Summary_Issue", "Article_Issue",
            "Names_Numbers_Issue", "Reviewer_Notes"
        ], values):
            review_sheet.loc[mask, col] = value

review_sheet.to_csv(review_path, index=False, encoding="utf-8-sig")

print("Second batch findings saved.")
print(review_sheet["Review_Status"].value_counts())

Second batch findings saved.
Review_Status
Pending                40
Initial review done    20
Name: count, dtype: Int64


In [ ]:
review_sheet = pd.read_csv(review_path)

next_batch = review_sheet[
    review_sheet["Review_Status"] == "Pending"
].head(10).copy()

batch_path = os.path.join(
    BASE,
    "05_quality_checks",
    "IndicTrans2_Quality_Review_Batch_3.csv"
)

next_batch.to_csv(batch_path, index=False, encoding="utf-8-sig")

print("Batch 3 prepared!")
print("Articles:", len(next_batch))
print("IDs:", next_batch["id"].to_list())
print("Saved:", batch_path)

Batch 3 prepared!
Articles: 10
IDs: ['081118_cg_campaign_vv', '080826_blast_films', '141007_bhutan_corruption_india_vr', '140702_wc2014_goldenboot_contenders_ar', '130207_kashmir_rockband_imroz_sa', '051105_sahubuddin_arrest', 'india-48662333', '070206_guj_parza', '061206_gennext_indopak_students', '141016_ranveer_removes_pant_md_part1_pkp']
Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/IndicTrans2_Quality_Review_Batch_3.csv


In [ ]:
for _, row in next_batch.iterrows():
    print("\n" + "=" * 65)
    print("ARTICLE:", row["id"])

    print("\nHINDI TITLE:")
    print(row["title"])

    print("\nENGLISH TITLE:")
    print(row["indic_title"])

    print("\nHINDI SUMMARY:")
    print(row["summary"])

    print("\nENGLISH SUMMARY:")
    print(row["indic_summary"])

    print("\nHINDI ARTICLE (first 500 characters):")
    print(str(row["text"])[:500])

    print("\nENGLISH ARTICLE (first 500 characters):")
    print(str(row["indic_article"])[:500])


ARTICLE: 081118_cg_campaign_vv

HINDI TITLE:
छत्तीसगढ़ में दूसरे चरण का प्रचार थमा

ENGLISH TITLE:
Campaigning for second phase ends in Chhattisgarh

HINDI SUMMARY:
छत्तीसगढ़ विधानसभा चुनाव के दूसरे चरण के लिए प्रचार मंगलवार को शाम पाँच बजे थम गया. इस चरण में राज्य विधानसभा की 90 में से 51 सीटों पर मतदान होना है.

ENGLISH SUMMARY:
Campaigning for the second phase of Chhattisgarh Assembly elections ended at 5 pm on Tuesday, with 51 of the 90 seats in the state going to polls in this phase.

HINDI ARTICLE (first 500 characters):
इस चरण का मतदान 20 नवंबर को होना है. मतदान का पहला चरण 14 नवंबर को निपट चुका है. मतगणना आठ दिसंबर को होगी. दूसरे चरण में भी राज्य में कई अहम मुक़ाबले होने हैं जिनमें पूर्व मुख्यमंत्री अजीत जोगी और उनकी पत्नी रेणु जोगी का चुनाव भी है. इसी चरण में प्रदेश कांग्रेस अध्यक्ष धनेंद्र साहू और कार्यकारी अध्यक्ष सत्यनारायण शर्मा, भाजपा के प्रदेश अध्यक्ष विष्णु देव साय और दिलीप सिंह जूदेव के बेटे युद्धवीर सिंह के राजनीतिक भविष्य का भी फ़ैसला होना है. प्रचार के अंतिम दिन दो

In [ ]:
review_sheet = pd.read_csv(review_path)

text_columns = [
    "Review_Status", "Title_Issue", "Summary_Issue",
    "Article_Issue", "Names_Numbers_Issue", "Reviewer_Notes"
]

for col in text_columns:
    review_sheet[col] = review_sheet[col].fillna("").astype("string")

findings_3 = {
    "081118_cg_campaign_vv": (
        "", "",
        "Dates incorrect: counting is shown as November 14 instead of December 8; first-phase polling detail is also confused",
        "Dates need correction",
        "Major factual translation error"
    ),
    "080826_blast_films": (
        "", "",
        "'Occasional events' should convey current events; otherwise mostly accurate",
        "",
        "Minor wording issue"
    ),
    "141007_bhutan_corruption_india_vr": (
        "", "", "", "",
        "Rankings and main meaning preserved in checked excerpt"
    ),
    "140702_wc2014_goldenboot_contenders_ar": (
        "", "",
        "James Rodriguez's five goals omitted; several footballers' names distorted",
        "Player names and goal count need correction",
        "Important details omitted or mistranslated"
    ),
    "130207_kashmir_rockband_imroz_sa": (
        "", "",
        "Opening sentence about Parvez Imroz is missing from the displayed translation excerpt",
        "",
        "Check full article for omitted opening content"
    ),
    "051105_sahubuddin_arrest": (
        "", "", "", "",
        "Key facts preserved in checked excerpt"
    ),
    "india-48662333": (
        "", "",
        "Some awkward wording, but main meaning is preserved",
        "",
        "Mostly acceptable in checked excerpt"
    ),
    "070206_guj_parza": (
        "", "", "", "",
        "Main facts preserved in checked excerpt"
    ),
    "061206_gennext_indopak_students": (
        "Title is mistranslated",
        "Opening metaphor and song reference are awkwardly translated",
        "Opening passage has major idiomatic translation errors",
        "",
        "Major wording and meaning issues"
    ),
    "141016_ranveer_removes_pant_md_part1_pkp": (
        "", "",
        "'Leather paint' should be 'leather pants'; final idiom is mistranslated",
        "",
        "Meaning and wording corrections needed"
    )
}

for article_id, values in findings_3.items():
    mask = review_sheet["id"] == article_id
    if mask.any():
        review_sheet.loc[mask, "Review_Status"] = "Initial review done"
        for col, value in zip([
            "Title_Issue", "Summary_Issue", "Article_Issue",
            "Names_Numbers_Issue", "Reviewer_Notes"
        ], values):
            review_sheet.loc[mask, col] = value

review_sheet.to_csv(review_path, index=False, encoding="utf-8-sig")

print("Batch 3 findings saved.")
print(review_sheet["Review_Status"].value_counts())
print("File:", review_path)

Batch 3 findings saved.
Review_Status
Initial review done    30
Pending                30
Name: count, dtype: Int64
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/IndicTrans2_First_60_Quality_Review.csv


In [ ]:
review_sheet = pd.read_csv(review_path)

next_batch = review_sheet[
    review_sheet["Review_Status"] == "Pending"
].head(10).copy()

batch_path = os.path.join(
    BASE,
    "05_quality_checks",
    "IndicTrans2_Quality_Review_Batch_4.csv"
)

next_batch.to_csv(
    batch_path,
    index=False,
    encoding="utf-8-sig"
)

print("Batch 4 prepared!")
print("Articles:", len(next_batch))
print("IDs:", next_batch["id"].to_list())
print("Saved:", batch_path)

Batch 4 prepared!
Articles: 10
IDs: ['060829_dhyanchand_hockey', '061004_askus_geyser', '140729_moeen_wristband_icc-du', 'india-46106302', '060120_speaker_cashcamera', 'international-41646598', '130614_complaint_for_looks_rd', '150930_vert_auto_audi_etron_pk', '141207_pakistan_taliban_mehsud_repatriated_rd', 'india-55465758']
Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/IndicTrans2_Quality_Review_Batch_4.csv


In [ ]:
for _, row in next_batch.iterrows():
    print("\n" + "=" * 65)
    print("ARTICLE:", row["id"])

    print("\nHINDI TITLE:")
    print(row["title"])

    print("\nENGLISH TITLE:")
    print(row["indic_title"])

    print("\nHINDI SUMMARY:")
    print(row["summary"])

    print("\nENGLISH SUMMARY:")
    print(row["indic_summary"])

    print("\nHINDI ARTICLE (first 500 characters):")
    print(str(row["text"])[:500])

    print("\nENGLISH ARTICLE (first 500 characters):")
    print(str(row["indic_article"])[:500])


ARTICLE: 060829_dhyanchand_hockey

HINDI TITLE:
ध्यानचंद का खेल आज भी याद आता है

ENGLISH TITLE:
Dhyan Chand's game is still remembered

HINDI SUMMARY:
हॉकी के जादूगर ध्यानचंद की महानता का अहसास न उनके खेल की तस्वीरों से लगता है और न ही रिकॉर्ड बुक से.

ENGLISH SUMMARY:
The greatness of hockey wizard Dhyan Chand is not realised by his photographs of the game nor by the record books.

HINDI ARTICLE (first 500 characters):
हॉलैंड में 1928 में उनकी स्टिक को तोड़कर जांचा गया कि कहीं उसमें चुंबक तो नहीं था. सन् 1932 में जापानी खिलाड़ियों ने कहा ध्यानचंद की स्टिक में गोंद चिपकी हुई रहती है और 1936 में हिटलर ने ब्रिटेन ओलंपिक में जर्मनी की टीम को 8-1 से हरानेवाली टीम के इस खिलाड़ी को अपनी सेना में ऊँचे पद का न्यौता दे डाला था. कुछ वर्षों पहले ध्यानचंद की पत्नी दिवंगत जानकी ध्यानचंद ने बताया था, '' हिटलर का जमाना था. इन्होंने चारों तरफ देखा लोग अपने-अपने देश के झंडे लेकर खड़े थे. भारत की टीम वैसे ही खड़ी थी. उन्हें बड़ा

ENGLISH ARTICLE (first 500 characters):
In Holland in 1928, his stick wa

In [ ]:
review_sheet = pd.read_csv(review_path)

text_columns = [
    "Review_Status", "Title_Issue", "Summary_Issue",
    "Article_Issue", "Names_Numbers_Issue", "Reviewer_Notes"
]

for col in text_columns:
    review_sheet[col] = review_sheet[col].fillna("").astype("string")

findings_4 = {
    "060829_dhyanchand_hockey": (
        "",
        "Awkward phrasing: greatness is not understood from photographs or record books",
        "The 1936 Olympics reference is mistranslated as 'UK Olympics'; verify the historical reference",
        "Olympics reference needs correction",
        "Meaning mostly preserved, but historical detail needs checking"
    ),
    "061004_askus_geyser": (
        "", "", "", "",
        "Translation is clear and meaning is preserved in the checked excerpt"
    ),
    "140729_moeen_wristband_icc-du": (
        "", "", "Minor spacing and punctuation issues around quoted wristband messages",
        "",
        "Main meaning and names preserved"
    ),
    "india-46106302": (
        "The word 'pisses off' is too strong and changes the tone; use 'concerns' or 'troubles'",
        "",
        "Opening sentence is confused: Chameli Devi and the list of abducted workers are incorrectly joined",
        "Names and relationships need checking",
        "Important wording and sentence-structure errors"
    ),
    "060120_speaker_cashcamera": (
        "",
        "Should clarify that MPs were dismissed for accepting money to ask questions",
        "The displayed English excerpt contains much less content than the Hindi excerpt; check for missing translated content in the full article",
        "",
        "Possible serious omission; compare full article lengths and content"
    ),
    "international-41646598": (
        "Title is corrupted with unexplained tokens '< ID1\\ 1 :' and does not read naturally",
        "Use 'women' rather than 'girls' for the context",
        "Checked opening excerpt is mostly understandable",
        "",
        "Title needs correction before use"
    ),
    "130614_complaint_for_looks_rd": (
        "The title says 'she complained', but the Hindi says the man complained to police",
        "",
        "Some awkward wording, including 'an unreliable police spokesman'; check this phrase against the Hindi",
        "",
        "Main summary meaning is preserved; title has a subject error"
    ),
    "150930_vert_auto_audi_etron_pk": (
        "", "",
        "Some awkward phrases, such as 'presented its Model X showroom'; meaning is mostly preserved",
        "",
        "Mostly acceptable in the checked excerpt"
    ),
    "141207_pakistan_taliban_mehsud_repatriated_rd": (
        "", "", "", "",
        "Main meaning and names preserved in the checked excerpt"
    ),
    "india-55465758": (
        "", "",
        "Minor awkward phrasing, such as 'I have to gift'; main meaning is preserved",
        "",
        "Mostly acceptable in the checked excerpt"
    )
}

for article_id, values in findings_4.items():
    mask = review_sheet["id"] == article_id

    if mask.any():
        review_sheet.loc[mask, "Review_Status"] = "Initial review done"

        for col, value in zip([
            "Title_Issue", "Summary_Issue", "Article_Issue",
            "Names_Numbers_Issue", "Reviewer_Notes"
        ], values):
            review_sheet.loc[mask, col] = value

review_sheet.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Batch 4 findings saved.")
print(review_sheet["Review_Status"].value_counts())
print("File:", review_path)

Batch 4 findings saved.
Review_Status
Initial review done    40
Pending                20
Name: count, dtype: Int64
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/IndicTrans2_First_60_Quality_Review.csv


In [ ]:
review_sheet = pd.read_csv(review_path)

next_batch = review_sheet[
    review_sheet["Review_Status"] == "Pending"
].head(10).copy()

batch_path = os.path.join(
    BASE,
    "05_quality_checks",
    "IndicTrans2_Quality_Review_Batch_5.csv"
)

next_batch.to_csv(
    batch_path,
    index=False,
    encoding="utf-8-sig"
)

print("Batch 5 prepared!")
print("Articles:", len(next_batch))
print("IDs:", next_batch["id"].to_list())
print("Saved:", batch_path)

Batch 5 prepared!
Articles: 10
IDs: ['160129_women_cricket_team_india_pm', '080826_bihar_flood', '120904_yuvraj_shabnam_ml', '050127_love_alternative', '140218_cricket_test_team_dhoni_vr', 'science-49370402', '141013_wusatullah_amitabh_bachchan_rns', '151218_anti_pee_london_ac', '090106_india_export', 'international-51281660']
Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/IndicTrans2_Quality_Review_Batch_5.csv


In [ ]:
for _, row in next_batch.iterrows():
    print("\n" + "=" * 65)
    print("ARTICLE:", row["id"])

    print("\nHINDI TITLE:")
    print(row["title"])

    print("\nENGLISH TITLE:")
    print(row["indic_title"])

    print("\nHINDI SUMMARY:")
    print(row["summary"])

    print("\nENGLISH SUMMARY:")
    print(row["indic_summary"])

    print("\nHINDI ARTICLE (first 500 characters):")
    print(str(row["text"])[:500])

    print("\nENGLISH ARTICLE (first 500 characters):")
    print(str(row["indic_article"])[:500])


ARTICLE: 160129_women_cricket_team_india_pm

HINDI TITLE:
भारतीय महिलाओं ने जीती टी-20 सिरीज़

ENGLISH TITLE:
Indian women win T20I series

HINDI SUMMARY:
भारतीय महिला क्रिकेट टीम ने ऑस्ट्रेलिया के ख़िलाफ़ ट्वेंटी-20 सिरीज़ जीतकर इतिहास रच दिया है.

ENGLISH SUMMARY:
The Indian women's cricket team has created history by winning the Twenty20 series against Australia.

HINDI ARTICLE (first 500 characters):
भारत की ऑस्ट्रेलिया के ख़िलाफ़ ये पहली द्विपक्षीय सिरीज़ जीत है. उन्होंने तीन बार की विश्व चैंपियन ऑस्ट्रेलिया को दूसरे मैच में दस विकेट से हराकर तीन मैचों की सिरीज़ में 2-0 की बढ़त ले ली है. बारिश की वजह से मैच छोटा कर दस ओवरों तक सीमित कर दिया गया था. झूलन गोस्वामी, भरतीय महिला क्रिकेट खिलाड़ी भारत को 66 रन बनाने का लक्ष्य मिला था. भारतीय टीम ने 9.1 ओवरों में ही 69 रन बनाए. समाप्त कप्तान मिताली राज ने 37 रन बनाए और वे अंत तक आउट नहीं हुईं. भारतीय लड़कियां मैच के शुरू से अंत तक हावी रहीं. उन्हो

ENGLISH ARTICLE (first 500 characters):
This is India's first bilateral series win agains

In [ ]:
review_sheet = pd.read_csv(review_path)

text_columns = [
    "Review_Status", "Title_Issue", "Summary_Issue",
    "Article_Issue", "Names_Numbers_Issue", "Reviewer_Notes"
]

for col in text_columns:
    review_sheet[col] = review_sheet[col].fillna("").astype("string")

findings_5 = {
    "160129_women_cricket_team_india_pm": (
        "",
        "",
        "Jhulan Goswami sentence is broken; 'Finished captain' is incorrect and the article wording is awkward",
        "",
        "Important sentence-level translation errors"
    ),
    "080826_bihar_flood": (
        "'Kosi's cry' mistranslates the sense of devastation or fury",
        "",
        "'Prêt' is an incorrect rendering of the news agency reference; the Nepal embankment sentence is also distorted",
        "",
        "Meaning and wording corrections needed"
    ),
    "120904_yuvraj_shabnam_ml": (
        "",
        "'Sad chapter that has now stopped' is awkward; the intended meaning is that the chapter is closed",
        "Opening sentence and pronouns are mistranslated; several sentences are grammatically unclear",
        "",
        "Major fluency issues in checked excerpt"
    ),
    "050127_love_alternative": (
        "",
        "",
        "'Cucumber seed' appears to be an incorrect translation of the seed description",
        "",
        "Important factual wording error"
    ),
    "140218_cricket_test_team_dhoni_vr": (
        "",
        "The English summary omits that India's New Zealand tour had also ended",
        "The McCullum question is mistranslated and his name is omitted from the displayed sentence",
        "",
        "Omission and meaning error"
    ),
    "science-49370402": (
        "",
        "",
        "Repeated wording ('reach the body'); 'common cold' appears mistranslated as 'jaundice' in the displayed excerpt",
        "",
        "Important health-related translation error"
    ),
    "141013_wusatullah_amitabh_bachchan_rns": (
        "Title omits the idea of Amitabh Bachchan's immense popularity or fan following in Pakistan",
        "",
        "'Grown up young and grown up' is awkward and does not preserve the original phrasing",
        "",
        "Title meaning and article fluency need improvement"
    ),
    "151218_anti_pee_london_ac": (
        "Literal title wording is awkward; the humorous idea of the wall 'peeing back' could be clearer",
        "",
        "'Flipped' is awkward for liquid bouncing back; 'invoice' should convey a fine or penalty",
        "",
        "Mostly understandable, with wording issues"
    ),
    "090106_india_export": (
        "",
        "",
        "'Press committee' is inaccurate for the news agency reference; the speaker's name is also rendered awkwardly",
        "Check the spelling of the speaker's name",
        "Numbers are preserved, but attribution needs checking"
    ),
    "international-51281660": (
        "",
        "",
        "'On an upswing' changes the meaning of the Hindi sentence about relations having ups and downs",
        "",
        "Meaning changed in the checked excerpt"
    )
}

for article_id, values in findings_5.items():
    mask = review_sheet["id"] == article_id

    if mask.any():
        review_sheet.loc[mask, "Review_Status"] = "Initial review done"

        for col, value in zip([
            "Title_Issue",
            "Summary_Issue",
            "Article_Issue",
            "Names_Numbers_Issue",
            "Reviewer_Notes"
        ], values):
            review_sheet.loc[mask, col] = value

review_sheet.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Batch 5 findings saved.")
print(review_sheet["Review_Status"].value_counts())
print("File:", review_path)

Batch 5 findings saved.
Review_Status
Initial review done    50
Pending                10
Name: count, dtype: Int64
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/IndicTrans2_First_60_Quality_Review.csv


In [ ]:
review_sheet = pd.read_csv(review_path)

next_batch = review_sheet[
    review_sheet["Review_Status"] == "Pending"
].head(10).copy()

batch_path = os.path.join(
    BASE,
    "05_quality_checks",
    "IndicTrans2_Quality_Review_Batch_6.csv"
)

next_batch.to_csv(
    batch_path,
    index=False,
    encoding="utf-8-sig"
)

print("Final batch prepared!")
print("Articles:", len(next_batch))
print("IDs:", next_batch["id"].to_list())
print("Saved:", batch_path)

Final batch prepared!
Articles: 10
IDs: ['international-44515542', 'international-47105506', '120816_assange_equador_pa', '120518_ipl_glamour_magazine_sm', '131119_chinese_supercomputer_worldbest_rns', 'india-53336116', 'india-51453120', '130529_karunanidhi_tamilnadu_marriage_ns', '150901_nepal_protest_dead_pp', '040803_jyotish_newtrend']
Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/IndicTrans2_Quality_Review_Batch_6.csv


In [ ]:
for _, row in next_batch.iterrows():
    print("\n" + "=" * 65)
    print("ARTICLE:", row["id"])

    print("\nHINDI TITLE:")
    print(row["title"])

    print("\nENGLISH TITLE:")
    print(row["indic_title"])

    print("\nHINDI SUMMARY:")
    print(row["summary"])

    print("\nENGLISH SUMMARY:")
    print(row["indic_summary"])

    print("\nHINDI ARTICLE (first 500 characters):")
    print(str(row["text"])[:500])

    print("\nENGLISH ARTICLE (first 500 characters):")
    print(str(row["indic_article"])[:500])


ARTICLE: international-44515542

HINDI TITLE:
तालिबान ने ठुकराई अफ़ग़ान सरकार की अपील, नहीं बढ़ेगा संघर्षविराम

ENGLISH TITLE:
Taliban reject Afghan government's appeal for no ceasefire

HINDI SUMMARY:
अफ़ग़ान तालिबान ने ईद के लिए घोषित किए गए तीन दिन के संघर्षविराम को आगे बढ़ाने से इनकार कर दिया है. राष्ट्रपति अशरफ़ ग़नी ने तालिबान से संघर्षविराम की मियाद बढ़ाने की अपील की थी.

ENGLISH SUMMARY:
The Afghan Taliban have refused to extend a three-day ceasefire announced for Eid, after President Ashraf Ghani urged the Taliban to extend the ceasefire.

HINDI ARTICLE (first 500 characters):
तालिबान के प्रवक्ता ज़बिहुल्लाह मुजाहिद ने कहा, "संघर्षविराम रविवार रात को ख़त्म हो जाएगा और सुरक्षाबलों के ख़िलाफ़ ऑपरेशन शुरू हो जाएंगे." तालिबान की ओर से कहा गया है कि संघर्षविराम बढ़ाने की उनकी कोई मंशा नहीं है और उम्मीद है कि लड़ाके सूरज ढलते तक सरकार के नियंत्रण वाला क्षेत्र छोड़ देंगे. उन्होंने सरकारी की अपील का ज़िक्र नहीं किया है. सरकार और तालिबान चरमपंथियों ने ईद पर तीन दिन का संघर्षविराम घो​ष

In [ ]:
review_sheet = pd.read_csv(review_path)

text_columns = [
    "Review_Status", "Title_Issue", "Summary_Issue",
    "Article_Issue", "Names_Numbers_Issue", "Reviewer_Notes"
]

for col in text_columns:
    review_sheet[col] = review_sheet[col].fillna("").astype("string")

findings_6 = {
    "international-44515542": (
        "Title wording is awkward; 'appeal for no ceasefire' is unclear",
        "", "", "",
        "Main meaning preserved in checked excerpt"
    ),
    "international-47105506": (
        "", "",
        "Haj cost mistranslated: Hindi says 476,000 Pakistani rupees, English says 466,000",
        "Incorrect amount: 476,000 rendered as 466,000",
        "Important numerical error"
    ),
    "120816_assange_equador_pa": (
        "", "", "", "",
        "Main meaning and names preserved in checked excerpt"
    ),
    "120518_ipl_glamour_magazine_sm": (
        "", "",
        "Awkward grammar and repeated wording, including 'drunk' twice; main meaning mostly preserved",
        "",
        "Fluency needs improvement"
    ),
    "131119_chinese_supercomputer_worldbest_rns": (
        "", "",
        "33.86 petaflops per second is mistranslated as per trillionth of a second",
        "Technical unit mistranslated",
        "Important technical accuracy error"
    ),
    "india-53336116": (
        "", "",
        "'We saw them and they told us' should be 'We saw them and they saw us'",
        "",
        "Meaning changed in a quoted statement"
    ),
    "india-51453120": (
        "",
        "Visit dates are incorrect: Hindi says February 24–25, English summary says February 1",
        "Displayed article excerpt is incomplete; full article should be checked",
        "Date error in summary",
        "Major factual error"
    ),
    "130529_karunanidhi_tamilnadu_marriage_ns": (
        "'Rules' should convey 'principles' in this context",
        "",
        "Some awkward wording, including 'That was not the case'",
        "",
        "Meaning is mostly understandable, but wording needs correction"
    ),
    "150901_nepal_protest_dead_pp": (
        "", "", "", "",
        "Main facts preserved in checked excerpt"
    ),
    "040803_jyotish_newtrend": (
        "Title should convey reading others' fortunes and improving one's own",
        "Summary meaning is awkwardly translated",
        "Traditional astrology terms are mistranslated or rendered unnaturally",
        "",
        "Significant fluency and meaning issues"
    )
}

for article_id, values in findings_6.items():
    mask = review_sheet["id"] == article_id

    if mask.any():
        review_sheet.loc[mask, "Review_Status"] = "Initial review done"

        for col, value in zip([
            "Title_Issue",
            "Summary_Issue",
            "Article_Issue",
            "Names_Numbers_Issue",
            "Reviewer_Notes"
        ], values):
            review_sheet.loc[mask, col] = value

review_sheet.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Final batch findings saved!")
print(review_sheet["Review_Status"].value_counts())
print("File:", review_path)

Final batch findings saved!
Review_Status
Initial review done    60
Name: count, dtype: Int64
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/IndicTrans2_First_60_Quality_Review.csv


In [ ]:
review_sheet = pd.read_csv(review_path)

issue_columns = [
    "Title_Issue",
    "Summary_Issue",
    "Article_Issue",
    "Names_Numbers_Issue"
]

print("=" * 55)
print("FACTINDIC — INDIC TRANS2 QUALITY REVIEW")
print("=" * 55)

print("\nTotal articles reviewed:", len(review_sheet))
print("\nReview status:")
print(review_sheet["Review_Status"].value_counts())

print("\nIssues found by category:")
for col in issue_columns:
    count = (
        review_sheet[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
        .sum()
    )
    print(f"{col}: {count}")

print("\nQuality ratings:")
if "Overall_Quality_1_to_5" in review_sheet.columns:
    print(
        review_sheet["Overall_Quality_1_to_5"]
        .value_counts(dropna=False)
        .sort_index()
    )

summary = pd.DataFrame({
    "Metric": [
        "Total articles",
        "Articles initially reviewed",
        "Pending articles"
    ] + [f"{col} entries" for col in issue_columns],
    "Count": [
        len(review_sheet),
        (review_sheet["Review_Status"] == "Initial review done").sum(),
        (review_sheet["Review_Status"] == "Pending").sum()
    ] + [
        review_sheet[col].fillna("").astype(str).str.strip().ne("").sum()
        for col in issue_columns
    ]
})

summary_path = os.path.join(
    BASE,
    "05_quality_checks",
    "IndicTrans2_First_60_Quality_Summary.csv"
)

summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("\nSummary saved:", summary_path)
print("\nQUALITY REVIEW COMPLETE — FIRST 60 ARTICLES")

FACTINDIC — INDIC TRANS2 QUALITY REVIEW

Total articles reviewed: 60

Review status:
Review_Status
Initial review done    60
Name: count, dtype: int64

Issues found by category:
Title_Issue: 30
Summary_Issue: 15
Article_Issue: 47
Names_Numbers_Issue: 19

Quality ratings:
Overall_Quality_1_to_5
NaN    60
Name: count, dtype: int64

Summary saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/IndicTrans2_First_60_Quality_Summary.csv

QUALITY REVIEW COMPLETE — FIRST 60 ARTICLES


In [ ]:
import os
import pandas as pd
import torch

clean_path = os.path.join(
    BASE, "02_clean_data", "XL_Sum_Hindi_Cleaned_250.csv"
)

first60_path = os.path.join(
    BASE, "03_english_data", "IndicTrans2_200M_First_60.csv"
)

df_clean = pd.read_csv(clean_path)
df_first60 = pd.read_csv(first60_path)

completed_ids = set(df_first60["id"].astype(str))

remaining_df = df_clean[
    ~df_clean["id"].astype(str).isin(completed_ids)
].copy().reset_index(drop=True)

print("Model available:", "model" in globals())
print("Tokenizer available:", "tokenizer" in globals())
print("IndicProcessor available:", "ip" in globals() or "indic_processor" in globals())
print("Device:", next(model.parameters()).device if "model" in globals() else "Not loaded")
print("Already translated:", len(completed_ids))
print("Remaining:", len(remaining_df))
print("Ready for full run:", len(remaining_df) == 190)

Model available: True
Tokenizer available: True
IndicProcessor available: True
Device: cpu
Already translated: 60
Remaining: 190
Ready for full run: True


In [ ]:
import os
import time
import pandas as pd
import torch
from tqdm.auto import tqdm

OUTPUT_DIR = os.path.join(BASE, "03_english_data")
CHECKPOINT_DIR = os.path.join(BASE, "09_checkpoints")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

OUTPUT_FILE = os.path.join(
    OUTPUT_DIR, "IndicTrans2_200M_Remaining_190.csv"
)

COMBINED_FILE = os.path.join(
    OUTPUT_DIR, "IndicTrans2_200M_All_250.csv"
)

def translate_one(text):
    if pd.isna(text) or not str(text).strip():
        return ""

    batch = ip.preprocess_batch(
        [str(text)],
        src_lang="hin_Deva",
        tgt_lang="eng_Latn"
    )

    inputs = tokenizer(
        batch,
        truncation=True,
        max_length=1024,
        padding="longest",
        return_tensors="pt"
    ).to(next(model.parameters()).device)

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            use_cache=False,
            max_length=1024,
            num_beams=1
        )

    decoded = tokenizer.batch_decode(
        generated,
        skip_special_tokens=True
    )

    return ip.postprocess_batch(
        decoded,
        lang="eng_Latn"
    )[0]


# Load any previous checkpoint
if os.path.exists(OUTPUT_FILE):
    results = pd.read_csv(OUTPUT_FILE)
    done_ids = set(results["id"].astype(str))
else:
    results = pd.DataFrame()
    done_ids = set()

work = remaining_df[
    ~remaining_df["id"].astype(str).isin(done_ids)
].copy()

print("Articles remaining in this run:", len(work))
print("Already checkpointed:", len(done_ids))

for _, row in tqdm(work.iterrows(), total=len(work)):
    article_id = str(row["id"])
    started = time.time()

    try:
        translated = {
            "id": article_id,
            "indic_title": translate_one(row["title"]),
            "indic_article": translate_one(row["text"]),
            "indic_summary": translate_one(row["summary"])
        }

        results = pd.concat(
            [results, pd.DataFrame([translated])],
            ignore_index=True
        )

        # Save after every article
        results.to_csv(
            OUTPUT_FILE,
            index=False,
            encoding="utf-8-sig"
        )

        print(
            f"\nSaved {len(results)} / 190 | "
            f"{article_id} | "
            f"{(time.time() - started) / 60:.2f} minutes"
        )

    except Exception as e:
        print("\nTranslation error:", article_id, str(e))

        # Preserve all successful translations
        results.to_csv(
            OUTPUT_FILE,
            index=False,
            encoding="utf-8-sig"
        )

        raise

print("\nRemaining translation batch completed!")
print("Translated:", len(results))
print("Saved:", OUTPUT_FILE)


# Combine original 60 and newly translated articles
first60 = pd.read_csv(first60_path)

all_translations = pd.concat(
    [first60, results],
    ignore_index=True
)

all_translations = all_translations.drop_duplicates(
    subset="id",
    keep="first"
)

all_translations.to_csv(
    COMBINED_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nCombined translation count:", len(all_translations))
print("Combined file:", COMBINED_FILE)

Articles remaining in this run: 190
Already checkpointed: 0


  0%|          | 0/190 [00:00<?, ?it/s]


Saved 1 / 190 | 150811_sani_mirza_autobiography_farha_adp_ssm | 1.29 minutes

Saved 2 / 190 | 131227_diamond_antarctica_tk | 46.72 minutes

Saved 3 / 190 | 060603_taliban_karzai | 0.35 minutes

Saved 4 / 190 | 050625_building_collapse | 0.98 minutes

Saved 5 / 190 | international-48243735 | 2.50 minutes

Saved 6 / 190 | india-51702290 | 1.42 minutes

Saved 7 / 190 | 080721_parliament_debate | 0.72 minutes

Saved 8 / 190 | 041003_pwg_security | 2.07 minutes

Saved 9 / 190 | 080416_up_results | 0.24 minutes


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pandas as pd

BASE = "/content/drive/MyDrive/FactIndic_PersonB_CleanRun"

OUTPUT_FILE = os.path.join(
    BASE,
    "03_english_data",
    "IndicTrans2_200M_Remaining_190.csv"
)

checkpoint = pd.read_csv(OUTPUT_FILE)

print("Saved articles:", len(checkpoint))
print("Remaining articles:", 190 - len(checkpoint))
print("Last saved article:", checkpoint["id"].iloc[-1])

Saved articles: 10
Remaining articles: 180
Last saved article: 150226_gujrat_swine_flu_ahmedabad_sr


In [ ]:
import os
import pandas as pd

BASE = "/content/drive/MyDrive/FactIndic_PersonB_CleanRun"

clean_path = os.path.join(
    BASE, "02_clean_data", "XL_Sum_Hindi_Cleaned_250.csv"
)

first60_path = os.path.join(
    BASE, "03_english_data", "IndicTrans2_200M_First_60.csv"
)

checkpoint_path = os.path.join(
    BASE, "03_english_data", "IndicTrans2_200M_Remaining_190.csv"
)

df_clean = pd.read_csv(clean_path)
first60 = pd.read_csv(first60_path)

completed_ids = set(first60["id"].astype(str))

remaining_df = df_clean[
    ~df_clean["id"].astype(str).isin(completed_ids)
].copy().reset_index(drop=True)

checkpoint = pd.read_csv(checkpoint_path)

print("First 60 translations:", len(first60))
print("Checkpointed translations:", len(checkpoint))
print("Articles still pending:", len(remaining_df) - len(checkpoint))
print("Last saved article:", checkpoint["id"].iloc[-1])

First 60 translations: 60
Checkpointed translations: 10
Articles still pending: 180
Last saved article: 150226_gujrat_swine_flu_ahmedabad_sr


In [ ]:

!pip install -q IndicTransToolkit


In [ ]:

import transformers
import torch
import sys

print("Python:", sys.version)
print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("Transformers location:", transformers.__file__)


Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Transformers: 5.22.1
PyTorch: 2.14.1+cpu
Transformers location: /usr/local/lib/python3.13/dist-packages/transformers/__init__.py


In [ ]:

!pip install -q --force-reinstall --no-deps transformers==4.57.6


In [ ]:

import sys

!{sys.executable} -m pip install --force-reinstall --no-deps transformers==4.57.6


  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
Using cached transformers-4.57.6-py3-none-any.whl (12.0 MB)
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6


In [ ]:

import sys

!{sys.executable} -m pip install --force-reinstall --no-deps tokenizers==0.22.1


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 11.7 MB/s  0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.3
    Uninstalling tokenizers-0.23.3:
      Successfully uninstalled tokenizers-0.23.3


In [ ]:

import sys

!{sys.executable} -m pip install --force-reinstall --no-deps huggingface-hub==0.36.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 4.8 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0


In [ ]:

import transformers
import tokenizers
import huggingface_hub

print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)


Transformers: 4.57.6
Tokenizers: 0.22.1
Hugging Face Hub: 0.36.0


In [ ]:
from IndicTransToolkit import IndicProcessor

ip = IndicProcessor(inference=True)

print("IndicProcessor ready!")

IndicProcessor ready!


In [ ]:
model_name = "ai4bharat/indictrans2-indic-en-dist-200M"

In [ ]:
from huggingface_hub import login

login()

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

model_name = "ai4bharat/indictrans2-indic-en-dist-200M"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = model.to("cpu")
model.eval()

print("IndicTrans2 model loaded!")
print("Device:", next(model.parameters()).device)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json: 0.00B [00:00, ?B/s]

dict.TGT.json: 0.00B [00:00, ?B/s]

model.SRC:   0%|          | 0.00/3.26M [00:00<?, ?B/s]

... (5 outputs left out)

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/913M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

IndicTrans2 model loaded!
Device: cpu


In [ ]:
import os
import pandas as pd

BASE = "/content/drive/MyDrive/FactIndic_PersonB_CleanRun"

clean_path = os.path.join(
    BASE, "02_clean_data/XL_Sum_Hindi_Cleaned_250.csv"
)

first60_path = os.path.join(
    BASE, "03_english_data/IndicTrans2_200M_First_60.csv"
)

checkpoint_path = os.path.join(
    BASE, "03_english_data/IndicTrans2_200M_Remaining_190.csv"
)

df_clean = pd.read_csv(clean_path)
first60 = pd.read_csv(first60_path)

completed_ids = set(first60["id"].astype(str))

remaining_df = df_clean[
    ~df_clean["id"].astype(str).isin(completed_ids)
].copy().reset_index(drop=True)

print("Clean articles:", len(df_clean))
print("First 60 translations:", len(first60))
print("Remaining articles:", len(remaining_df))
print("Checkpoint exists:", os.path.exists(checkpoint_path))

Clean articles: 250
First 60 translations: 60
Remaining articles: 190
Checkpoint exists: True


In [ ]:
import os
import time
import pandas as pd
import torch
from tqdm.auto import tqdm

OUTPUT_DIR = os.path.join(BASE, "03_english_data")
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_FILE = os.path.join(
    OUTPUT_DIR, "IndicTrans2_200M_Remaining_190.csv"
)

COMBINED_FILE = os.path.join(
    OUTPUT_DIR, "IndicTrans2_200M_All_250.csv"
)

def translate_one(text):
    if pd.isna(text) or not str(text).strip():
        return ""

    batch = ip.preprocess_batch(
        [str(text)],
        src_lang="hin_Deva",
        tgt_lang="eng_Latn"
    )

    inputs = tokenizer(
        batch,
        truncation=True,
        max_length=1024,
        padding="longest",
        return_tensors="pt"
    ).to("cpu")

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            use_cache=False,
            max_length=1024,
            num_beams=1
        )

    decoded = tokenizer.batch_decode(
        generated,
        skip_special_tokens=True
    )

    return ip.postprocess_batch(
        decoded,
        lang="eng_Latn"
    )[0]


# Load saved translations
if os.path.exists(OUTPUT_FILE):
    results = pd.read_csv(OUTPUT_FILE)
    done_ids = set(results["id"].astype(str))
else:
    results = pd.DataFrame(
        columns=["id", "indic_title", "indic_article", "indic_summary"]
    )
    done_ids = set()

# Select unfinished articles
work = remaining_df[
    ~remaining_df["id"].astype(str).isin(done_ids)
].copy()

print("Articles remaining:", len(work))
print("Already checkpointed:", len(done_ids))

# Translate remaining articles
for _, row in tqdm(work.iterrows(), total=len(work)):

    article_id = str(row["id"])
    started = time.time()

    try:
        translated = {
            "id": article_id,
            "indic_title": translate_one(row["title"]),
            "indic_article": translate_one(row["text"]),
            "indic_summary": translate_one(row["summary"])
        }

        results = pd.concat(
            [results, pd.DataFrame([translated])],
            ignore_index=True
        )

        temp_file = OUTPUT_FILE + ".tmp"

        results.to_csv(
            temp_file,
            index=False,
            encoding="utf-8-sig"
        )

        os.replace(temp_file, OUTPUT_FILE)

        print(
            f"\nSaved {len(results)} / 190 | "
            f"{article_id} | "
            f"{(time.time() - started) / 60:.2f} minutes"
        )

    except Exception as e:
        print("\nTranslation error:", article_id, str(e))
        raise

print("\nRemaining translations completed!")

# Combine all 250 translations
first60 = pd.read_csv(first60_path)

all_translations = pd.concat(
    [first60, results],
    ignore_index=True
).drop_duplicates(subset="id", keep="first")

all_translations.to_csv(
    COMBINED_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nCombined translation count:", len(all_translations))
print("Final file:", COMBINED_FILE)

Articles remaining: 180
Already checkpointed: 10


  0%|          | 0/180 [00:00<?, ?it/s]

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import pandas as pd

BASE = "/content/drive/MyDrive/FactIndic_PersonB_CleanRun"

checkpoint_path = os.path.join(
    BASE, "03_english_data/IndicTrans2_200M_Remaining_190.csv"
)

if os.path.exists(checkpoint_path):
    saved = pd.read_csv(checkpoint_path)
    print("Saved translations:", len(saved))
    print("Remaining:", 190 - len(saved))
    print("Last saved article:", saved["id"].iloc[-1])
else:
    print("Checkpoint file not found!")

Saved translations: 10
Remaining: 180
Last saved article: 150226_gujrat_swine_flu_ahmedabad_sr


In [3]:
import sys

!{sys.executable} -m pip install --force-reinstall --no-deps transformers==4.57.6
!{sys.executable} -m pip install --force-reinstall --no-deps tokenizers==0.22.1
!{sys.executable} -m pip install --force-reinstall --no-deps huggingface-hub==0.36.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 6.9 MB/s  0:00:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 38.4 MB/s  0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.3
    Uninstalling tokenizers-0.23.3:
      Successfully uninstalled tokenizers-0.23.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 11.7 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0


In [4]:
import transformers
import tokenizers
import huggingface_hub

print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

Transformers: 4.57.6
Tokenizers: 0.22.1
Hugging Face Hub: 0.36.0


In [6]:
import sys

!{sys.executable} -m pip install IndicTransToolkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.4/548.4 kB 9.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 28.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 103.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [IndicTransToolkit]


In [7]:
from IndicTransToolkit import IndicProcessor

ip = IndicProcessor(inference=True)

print("IndicProcessor ready!")

IndicProcessor ready!


In [8]:
from huggingface_hub import login
login()

In [9]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

model_name = "ai4bharat/indictrans2-indic-en-dist-200M"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = model.to("cpu")
model.eval()

print("IndicTrans2 model loaded!")
print("Device:", next(model.parameters()).device)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json: 0.00B [00:00, ?B/s]

dict.TGT.json: 0.00B [00:00, ?B/s]

... (6 outputs left out)

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/913M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

IndicTrans2 model loaded!
Device: cpu


In [10]:
import os
import pandas as pd

BASE = "/content/drive/MyDrive/FactIndic_PersonB_CleanRun"

clean_path = os.path.join(
    BASE, "02_clean_data/XL_Sum_Hindi_Cleaned_250.csv"
)

first60_path = os.path.join(
    BASE, "03_english_data/IndicTrans2_200M_First_60.csv"
)

OUTPUT_FILE = os.path.join(
    BASE, "03_english_data/IndicTrans2_200M_Remaining_190.csv"
)

df_clean = pd.read_csv(clean_path)
first60 = pd.read_csv(first60_path)

if os.path.exists(OUTPUT_FILE):
    results = pd.read_csv(OUTPUT_FILE)
else:
    results = pd.DataFrame(
        columns=["id", "indic_title", "indic_article", "indic_summary"]
    )

completed_ids = (
    set(first60["id"].astype(str))
    | set(results["id"].astype(str))
)

remaining_df = df_clean[
    ~df_clean["id"].astype(str).isin(completed_ids)
].copy().reset_index(drop=True)

batch_df = remaining_df.head(50).copy()

print("Already translated:", len(completed_ids))
print("Articles remaining:", len(remaining_df))
print("Batch 1 size:", len(batch_df))
print("First article:", batch_df.iloc[0]["id"])

Already translated: 70
Articles remaining: 180
Batch 1 size: 50
First article: 050901_gohana_firsthand


In [12]:
def translate_one(text):
    if pd.isna(text) or not str(text).strip():
        return ""

    batch = ip.preprocess_batch(
        [str(text)],
        src_lang="hin_Deva",
        tgt_lang="eng_Latn"
    )

    inputs = tokenizer(
        batch,
        truncation=True,
        max_length=1024,
        padding="longest",
        return_tensors="pt"
    ).to("cpu")

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            use_cache=False,
            max_length=1024,
            num_beams=1
        )

    decoded = tokenizer.batch_decode(
        generated,
        skip_special_tokens=True
    )

    return ip.postprocess_batch(
        decoded,
        lang="eng_Latn"
    )[0]

print("Translation function ready!")

Translation function ready!


In [13]:
import time
import os
import pandas as pd
import torch
from tqdm.auto import tqdm

for _, row in tqdm(batch_df.iterrows(), total=len(batch_df)):

    article_id = str(row["id"])

    # Skip if already saved
    if article_id in set(results["id"].astype(str)):
        continue

    started = time.time()

    try:
        translated = {
            "id": article_id,
            "indic_title": translate_one(row["title"]),
            "indic_article": translate_one(row["text"]),
            "indic_summary": translate_one(row["summary"])
        }

        results = pd.concat(
            [results, pd.DataFrame([translated])],
            ignore_index=True
        )

        # Save after every article
        temp_file = OUTPUT_FILE + ".tmp"

        results.to_csv(
            temp_file,
            index=False,
            encoding="utf-8-sig"
        )

        os.replace(temp_file, OUTPUT_FILE)

        print(
            f"\nSaved {len(results)} / 190 | "
            f"{article_id} | "
            f"{(time.time() - started) / 60:.2f} minutes"
        )

    except Exception as e:
        print("\nError:", article_id, str(e))
        raise

print("\nBATCH 1 COMPLETED!")
print("Batch size:", len(batch_df))
print("Total remaining translations saved:", len(results))

  0%|          | 0/50 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [14]:
import pandas as pd
import os

if os.path.exists(OUTPUT_FILE):
    results = pd.read_csv(OUTPUT_FILE)
    print("Saved translations:", len(results))
    print("Remaining:", 190 - len(results))
else:
    print("No new translations saved.")

Saved translations: 10
Remaining: 180


In [15]:
test_text = "दिल्ली में आज भारी बारिश हुई।"

import time

start = time.time()
test_result = translate_one(test_text)

print("English:", test_result)
print("Time:", round(time.time() - start, 2), "seconds")

English: Delhi received heavy rainfall today.
Time: 1.04 seconds


In [17]:
print("Article words:", len(str(test_row["text"]).split()))
print("Title words:", len(str(test_row["title"]).split()))
print("Summary words:", len(str(test_row["summary"]).split()))
print("Device:", next(model.parameters()).device)

Article words: 571
Title words: 5
Summary words: 26
Device: cpu


In [18]:
def translate_long_text(text, chunk_words=150):
    if pd.isna(text) or not str(text).strip():
        return ""

    words = str(text).split()
    chunks = [
        " ".join(words[i:i + chunk_words])
        for i in range(0, len(words), chunk_words)
    ]

    translated_chunks = []

    for chunk in chunks:
        translated_chunks.append(translate_one(chunk))

    return " ".join(translated_chunks)

print("Chunk translation function ready!")

Chunk translation function ready!


In [19]:
import time

start = time.time()

chunk_test = translate_long_text(test_row["text"])

print("Translation completed!")
print("English preview:", chunk_test[:500])
print("Time:", round(time.time() - start, 2), "seconds")

Translation completed!
English preview: Every house we went to was the same. In the middle of the city, more than 50 houses were burnt down. The administration and the whole city watched in silence. Krishna's house is visible as soon as he enters the first street. He has two young daughters. One of them has a marriage bag made of ash in front of him. Krishna says, "For the daughter's wedding, I made some jewels and took some new clothes. They took everything with them, and everything else has been burnt. Not even a grain of food left.
Time: 284.21 seconds


In [1]:
import torch
print("GPU available:", torch.cuda.is_available())

GPU available: True


In [2]:
model = model.to("cuda")
model.eval()

print("Model device:", next(model.parameters()).device)

NameError: name 'model' is not defined

In [4]:
from huggingface_hub import login

login()

In [7]:
import sys
import transformers
import tokenizers

print("Python:", sys.version)
print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Transformers: 5.22.1
Tokenizers: 0.23.3


In [8]:
!pip install -q --force-reinstall --no-deps "transformers==4.44.2"

In [9]:
import transformers
import tokenizers
import torch

print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

Transformers: 5.22.1
Tokenizers: 0.23.3
PyTorch: 2.14.1+cu130
GPU available: True


In [10]:
%pip install --force-reinstall --no-deps "transformers==4.44.2"

  Using cached transformers-4.44.2-py3-none-any.whl.metadata (43 kB)
Using cached transformers-4.44.2-py3-none-any.whl (9.5 MB)
  Attempting uninstall: transformers
    Found existing installation: transformers 4.44.2
    Uninstalling transformers-4.44.2:
      Successfully uninstalled transformers-4.44.2


In [2]:
%pip install --force-reinstall --no-deps "transformers==4.57.6"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 39.3 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.44.2
    Uninstalling transformers-4.44.2:
      Successfully uninstalled transformers-4.44.2


In [2]:
%pip install --force-reinstall --only-binary=:all: "tokenizers==0.23.0"

ERROR: Ignored the following yanked versions: 0.20.4
ERROR: Could not find a version that satisfies the requirement tokenizers==0.23.0 (from versions: 0.15.1, 0.15.2, 0.20.2, 0.20.3, 0.20.4rc0, 0.21.0rc0, 0.21.0, 0.21.1rc0, 0.21.1, 0.21.2rc0, 0.21.2, 0.21.4, 0.22.0, 0.22.1rc0, 0.22.1, 0.22.2rc0, 0.22.2, 0.23.0rc0, 0.23.1rc0, 0.23.1, 0.23.2, 0.23.3, 1.0.0rc1, 1.0.0rc2, 1.0.0rc3, 1.0.0rc4)
ERROR: No matching distribution found for tokenizers==0.23.0


In [3]:
%pip install --force-reinstall --only-binary=:all: "tokenizers==0.22.1"

  Using cached filelock-4.2.10-py3-none-any.whl.metadata (2.3 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.8.0-cp38-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached packaging-26.3-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached anyio-4.16.0-py3-none-any.whl.metadata (4.7 kB)
  Using cached certifi-2026.10.23-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 19.

In [2]:
%pip install --force-reinstall --no-deps "huggingface-hub==0.36.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 6.4 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0


# New Section

In [1]:
import transformers
import tokenizers
import huggingface_hub

print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

Transformers: 4.57.6
Tokenizers: 0.22.1
Hugging Face Hub: 0.36.0


In [2]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "ai4bharat/indictrans2-indic-en-dist-200M"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

print("Loading model...")

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.float16
)

model = model.to("cuda")
model.eval()

print("IndicTrans2 loaded successfully!")
print("Device:", next(model.parameters()).device)
print("GPU:", torch.cuda.get_device_name(0))

Loading tokenizer...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json: 0.00B [00:00, ?B/s]

... (6 outputs left out)

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/913M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

IndicTrans2 loaded successfully!
Device: cuda:0
GPU: Tesla T4


In [4]:
%pip install IndicTransToolkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.4/548.4 kB 9.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 19.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 50.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [IndicTransToolkit]


In [5]:
from IndicTransToolkit import IndicProcessor

ip = IndicProcessor(inference=True)

print("IndicProcessor ready!")

IndicProcessor ready!


In [7]:
with torch.no_grad():
    output = model.generate(
        **inputs,
        max_length=256,
        num_beams=1,
        use_cache=False
    )

decoded = tokenizer.batch_decode(
    output,
    skip_special_tokens=True
)

print(ip.postprocess_batch(decoded, lang="eng_Latn")[0])

Delhi received heavy rains today and many areas were waterlogged.


In [8]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd

BASE = "/content/drive/MyDrive/FactIndic_PersonB_CleanRun"

clean_path = os.path.join(
    BASE, "02_clean_data/XL_Sum_Hindi_Cleaned_250.csv"
)

first60_path = os.path.join(
    BASE, "03_english_data/IndicTrans2_200M_First_60.csv"
)

OUTPUT_FILE = os.path.join(
    BASE, "03_english_data/IndicTrans2_200M_Remaining_190.csv"
)

df_clean = pd.read_csv(clean_path)
first60 = pd.read_csv(first60_path)

if os.path.exists(OUTPUT_FILE):
    results = pd.read_csv(OUTPUT_FILE)
else:
    results = pd.DataFrame(
        columns=["id", "indic_title", "indic_article", "indic_summary"]
    )

completed_ids = (
    set(first60["id"].astype(str))
    | set(results["id"].astype(str))
)

remaining_df = df_clean[
    ~df_clean["id"].astype(str).isin(completed_ids)
].copy().reset_index(drop=True)

batch_df = remaining_df.head(50).copy()

print("Already translated:", len(completed_ids))
print("Articles remaining:", len(remaining_df))
print("Batch 1 size:", len(batch_df))
print("First article:", batch_df.iloc[0]["id"])

Mounted at /content/drive
Already translated: 70
Articles remaining: 180
Batch 1 size: 50
First article: 050901_gohana_firsthand


In [9]:
import torch
import pandas as pd
import time

device = next(model.parameters()).device

def translate_one(text):
    if pd.isna(text) or not str(text).strip():
        return ""

    batch = ip.preprocess_batch(
        [str(text)],
        src_lang="hin_Deva",
        tgt_lang="eng_Latn"
    )

    inputs = tokenizer(
        batch,
        truncation=True,
        max_length=1024,
        padding=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=1024,
            num_beams=1,
            use_cache=False
        )

    decoded = tokenizer.batch_decode(
        output,
        skip_special_tokens=True
    )

    return ip.postprocess_batch(
        decoded,
        lang="eng_Latn"
    )[0]

print("Translation function ready!")
print("Device:", device)

Translation function ready!
Device: cuda:0


In [10]:
row = batch_df.iloc[0]

start = time.time()

test_title = translate_one(row["title"])
test_summary = translate_one(row["summary"])

print("Article ID:", row["id"])
print("Hindi title:", row["title"])
print("English title:", test_title)
print("Hindi summary:", row["summary"])
print("English summary:", test_summary)
print("Time:", round(time.time() - start, 2), "seconds")

Article ID: 050901_gohana_firsthand
Hindi title: गोहाना का आँखों देखा हाल
English title: Gohana's eyes saw the situation.
Hindi summary: गोहाना की बाल्मीकि बस्ती में कई सारी चीज़ें एक साथ जलने की दुर्गंध के बीच लोग अपना सामान खोजते हैं, पर हाथ सिर्फ़ राख लगती है.
English summary: In the Valmiki Basti of Gohana, people search for their belongings amid the stench of burning many things together, but the hands only look like ashes.
Time: 6.49 seconds


In [11]:
import os
import time
import pandas as pd

for idx, (_, row) in enumerate(batch_df.iterrows(), start=1):

    article_id = str(row["id"])

    if article_id in set(results["id"].astype(str)):
        continue

    start = time.time()

    try:
        title_en = translate_one(row["title"])
        article_en = translate_one(row["text"])
        summary_en = translate_one(row["summary"])

        new_row = {
            "id": article_id,
            "indic_title": title_en,
            "indic_article": article_en,
            "indic_summary": summary_en
        }

        results = pd.concat(
            [results, pd.DataFrame([new_row])],
            ignore_index=True
        )

        temp_file = OUTPUT_FILE + ".tmp"
        results.to_csv(temp_file, index=False)
        os.replace(temp_file, OUTPUT_FILE)

        print(
            f"Saved {len(results)} checkpoint rows | "
            f"Batch {idx}/50 | {article_id} | "
            f"{round(time.time()-start, 1)} sec"
        )

    except Exception as e:
        print("ERROR:", article_id, str(e))
        break

print("\nBatch processing finished or paused.")
print("Checkpoint rows:", len(results))
print("Checkpoint file:", OUTPUT_FILE)

Saved 11 checkpoint rows | Batch 1/50 | 050901_gohana_firsthand | 69.9 sec
Saved 12 checkpoint rows | Batch 2/50 | 070530_iftikhar_detained | 8.8 sec
Saved 13 checkpoint rows | Batch 3/50 | 150607_narendra_modi_bangladesh_visit_analysis_rns | 4.9 sec
Saved 14 checkpoint rows | Batch 4/50 | international-52239381 | 37.5 sec
Saved 15 checkpoint rows | Batch 5/50 | 040415_sharon_arafat | 36.2 sec
Saved 16 checkpoint rows | Batch 6/50 | 071013_advani_pm | 8.2 sec
Saved 17 checkpoint rows | Batch 7/50 | 111113_berlusconi_resigned_vv | 14.3 sec
Saved 18 checkpoint rows | Batch 8/50 | 150612_heatwave_rentachintala_sra | 12.1 sec
Saved 19 checkpoint rows | Batch 9/50 | 080211_raj_police | 3.0 sec
Saved 20 checkpoint rows | Batch 10/50 | 050812_baflights_cancel | 7.4 sec
Saved 21 checkpoint rows | Batch 11/50 | india-47148086 | 4.5 sec
Saved 22 checkpoint rows | Batch 12/50 | international-42264998 | 34.3 sec
Saved 23 checkpoint rows | Batch 13/50 | 160606_ajit_jogi_new_party_rv | 8.2 sec
Saved

In [12]:
completed_ids = (
    set(first60["id"].astype(str))
    | set(results["id"].astype(str))
)

remaining_df = df_clean[
    ~df_clean["id"].astype(str).isin(completed_ids)
].copy().reset_index(drop=True)

batch_df = remaining_df.head(50).copy()

print("Total translated:", len(completed_ids))
print("Remaining:", len(remaining_df))
print("Batch 2 size:", len(batch_df))
print("First article:", batch_df.iloc[0]["id"])

Total translated: 120
Remaining: 130
Batch 2 size: 50
First article: 040725_bangladesh_floods


In [13]:
for idx, (_, row) in enumerate(batch_df.iterrows(), start=1):

    article_id = str(row["id"])

    if article_id in set(results["id"].astype(str)):
        continue

    start = time.time()

    try:
        title_en = translate_one(row["title"])
        article_en = translate_one(row["text"])
        summary_en = translate_one(row["summary"])

        new_row = {
            "id": article_id,
            "indic_title": title_en,
            "indic_article": article_en,
            "indic_summary": summary_en
        }

        results = pd.concat(
            [results, pd.DataFrame([new_row])],
            ignore_index=True
        )

        temp_file = OUTPUT_FILE + ".tmp"
        results.to_csv(temp_file, index=False)
        os.replace(temp_file, OUTPUT_FILE)

        print(
            f"Saved {len(results)} checkpoint rows | "
            f"Batch {idx}/50 | {article_id} | "
            f"{round(time.time()-start, 1)} sec"
        )

    except Exception as e:
        print("ERROR:", article_id, str(e))
        break

print("\nBatch processing finished or paused.")
print("Checkpoint rows:", len(results))

Saved 61 checkpoint rows | Batch 1/50 | 040725_bangladesh_floods | 5.9 sec
Saved 62 checkpoint rows | Batch 2/50 | 081205_sc_bcci_pp | 7.5 sec
Saved 63 checkpoint rows | Batch 3/50 | science-39280869 | 33.9 sec
Saved 64 checkpoint rows | Batch 4/50 | 150222_parliament_budget_session_pk | 9.6 sec
Saved 65 checkpoint rows | Batch 5/50 | international-44946029 | 9.5 sec
Saved 66 checkpoint rows | Batch 6/50 | 130910_sehwag_gambhir_pujara_pp | 12.6 sec
Saved 67 checkpoint rows | Batch 7/50 | india-49221048 | 13.4 sec
Saved 68 checkpoint rows | Batch 8/50 | 130707_raghavji_mp_fma | 2.1 sec
Saved 69 checkpoint rows | Batch 9/50 | international-54702261 | 9.9 sec
Saved 70 checkpoint rows | Batch 10/50 | 071104_komal_column | 18.0 sec
Saved 71 checkpoint rows | Batch 11/50 | 031031_russia_removal | 6.7 sec
Saved 72 checkpoint rows | Batch 12/50 | 140906_urdu_press_review_vr | 33.3 sec
Saved 73 checkpoint rows | Batch 13/50 | 090210_filmfestival_gorakhpur | 33.3 sec
Saved 74 checkpoint rows | B

In [14]:
completed_ids = (
    set(first60["id"].astype(str))
    | set(results["id"].astype(str))
)

remaining_df = df_clean[
    ~df_clean["id"].astype(str).isin(completed_ids)
].copy().reset_index(drop=True)

batch_df = remaining_df.head(50).copy()

print("Total translated:", len(completed_ids))
print("Remaining:", len(remaining_df))
print("Batch 3 size:", len(batch_df))
print("First article:", batch_df.iloc[0]["id"])

Total translated: 170
Remaining: 80
Batch 3 size: 50
First article: india-48128218


In [15]:
for idx, (_, row) in enumerate(batch_df.iterrows(), start=1):

    article_id = str(row["id"])

    if article_id in set(results["id"].astype(str)):
        continue

    start = time.time()

    try:
        title_en = translate_one(row["title"])
        article_en = translate_one(row["text"])
        summary_en = translate_one(row["summary"])

        new_row = {
            "id": article_id,
            "indic_title": title_en,
            "indic_article": article_en,
            "indic_summary": summary_en
        }

        results = pd.concat(
            [results, pd.DataFrame([new_row])],
            ignore_index=True
        )

        temp_file = OUTPUT_FILE + ".tmp"
        results.to_csv(temp_file, index=False)
        os.replace(temp_file, OUTPUT_FILE)

        print(
            f"Saved {len(results)} checkpoint rows | "
            f"Batch {idx}/50 | {article_id} | "
            f"{round(time.time()-start, 1)} sec"
        )

    except Exception as e:
        print("ERROR:", article_id, str(e))
        break

print("\nBatch processing finished or paused.")
print("Checkpoint rows:", len(results))

Saved 111 checkpoint rows | Batch 1/50 | india-48128218 | 6.5 sec
Saved 112 checkpoint rows | Batch 2/50 | 060530_reservation_strike | 5.3 sec
Saved 113 checkpoint rows | Batch 3/50 | 140223_modi_rally_punjab_sr | 11.3 sec
Saved 114 checkpoint rows | Batch 4/50 | international-41395655 | 7.1 sec
Saved 115 checkpoint rows | Batch 5/50 | 140125_da_rday_special_parade_rd | 8.1 sec
Saved 116 checkpoint rows | Batch 6/50 | 031114_bush_intelligence | 6.1 sec
Saved 117 checkpoint rows | Batch 7/50 | 140908_amit_shah_political_analysis_vr | 6.2 sec
Saved 118 checkpoint rows | Batch 8/50 | 070209_wi_samuels | 3.6 sec
Saved 119 checkpoint rows | Batch 9/50 | 141218_russia_putin_rouble_stable_sn | 6.0 sec
Saved 120 checkpoint rows | Batch 10/50 | 040305_holi_songs | 5.8 sec
Saved 121 checkpoint rows | Batch 11/50 | 080319_advani_book | 16.1 sec
Saved 122 checkpoint rows | Batch 12/50 | india-44912345 | 8.0 sec
Saved 123 checkpoint rows | Batch 13/50 | 141230_mahendra_singh_dhoni_retirement_vr | 2

In [16]:
completed_ids = (
    set(first60["id"].astype(str))
    | set(results["id"].astype(str))
)

remaining_df = df_clean[
    ~df_clean["id"].astype(str).isin(completed_ids)
].copy().reset_index(drop=True)

batch_df = remaining_df.copy()

print("Total translated:", len(completed_ids))
print("Remaining:", len(remaining_df))
print("Final batch size:", len(batch_df))
print("First article:", batch_df.iloc[0]["id"])

Total translated: 220
Remaining: 30
Final batch size: 30
First article: 070427_mcgrath_lastmatch


In [17]:
for idx, (_, row) in enumerate(batch_df.iterrows(), start=1):

    article_id = str(row["id"])

    if article_id in set(results["id"].astype(str)):
        continue

    start = time.time()

    try:
        title_en = translate_one(row["title"])
        article_en = translate_one(row["text"])
        summary_en = translate_one(row["summary"])

        new_row = {
            "id": article_id,
            "indic_title": title_en,
            "indic_article": article_en,
            "indic_summary": summary_en
        }

        results = pd.concat(
            [results, pd.DataFrame([new_row])],
            ignore_index=True
        )

        temp_file = OUTPUT_FILE + ".tmp"
        results.to_csv(temp_file, index=False)
        os.replace(temp_file, OUTPUT_FILE)

        print(
            f"Saved {len(results)} checkpoint rows | "
            f"Final batch {idx}/30 | {article_id} | "
            f"{round(time.time()-start, 1)} sec"
        )

    except Exception as e:
        print("ERROR:", article_id, str(e))
        break

print("\nFinal batch finished or paused.")
print("Checkpoint rows:", len(results))

Saved 161 checkpoint rows | Final batch 1/30 | 070427_mcgrath_lastmatch | 5.7 sec
Saved 162 checkpoint rows | Final batch 2/30 | india-49659647 | 2.9 sec
Saved 163 checkpoint rows | Final batch 3/30 | india-39447746 | 3.6 sec
Saved 164 checkpoint rows | Final batch 4/30 | 070425_ccpa_inconclusive | 4.1 sec
Saved 165 checkpoint rows | Final batch 5/30 | 150408_kangana_ranaut_deepika_padukone_md | 5.9 sec
Saved 166 checkpoint rows | Final batch 6/30 | international-41521329 | 8.1 sec
Saved 167 checkpoint rows | Final batch 7/30 | 040421_tripura_polling | 7.8 sec
Saved 168 checkpoint rows | Final batch 8/30 | 081202_rice_india_ac | 5.1 sec
Saved 169 checkpoint rows | Final batch 9/30 | 150708_us_army_cuts_aa | 5.4 sec
Saved 170 checkpoint rows | Final batch 10/30 | 130912_nepal_strike_ssr | 5.6 sec
Saved 171 checkpoint rows | Final batch 11/30 | 080514_kashmir_ceasefire | 2.8 sec
Saved 172 checkpoint rows | Final batch 12/30 | 050713_bombsuspect_family | 5.5 sec
Saved 173 checkpoint rows 

In [18]:
import pandas as pd
import os

all_translations = pd.concat(
    [first60, results],
    ignore_index=True
)

all_translations["id"] = all_translations["id"].astype(str)

all_translations = all_translations.drop_duplicates(
    subset="id",
    keep="last"
)

all_translations = all_translations.sort_values("id")

FINAL_FILE = os.path.join(
    BASE,
    "03_english_data/IndicTrans2_200M_All_250.csv"
)

all_translations.to_csv(FINAL_FILE, index=False)

print("Total unique articles:", len(all_translations))
print("Missing titles:", all_translations["indic_title"].isna().sum())
print("Missing articles:", all_translations["indic_article"].isna().sum())
print("Missing summaries:", all_translations["indic_summary"].isna().sum())
print("Final file:", FINAL_FILE)

Total unique articles: 250
Missing titles: 0
Missing articles: 0
Missing summaries: 0
Final file: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/03_english_data/IndicTrans2_200M_All_250.csv


In [19]:
for col in ["indic_title", "indic_article", "indic_summary"]:
    empty_count = (
        all_translations[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )
    print(f"{col}: {empty_count} empty values")

print("\nFinal dataset shape:", all_translations.shape)
print("Unique IDs:", all_translations["id"].nunique())

indic_title: 0 empty values
indic_article: 0 empty values
indic_summary: 0 empty values

Final dataset shape: (250, 10)
Unique IDs: 250


In [20]:
final_df = df_clean.merge(
    all_translations,
    on="id",
    how="inner"
)

print("Final merged rows:", len(final_df))
print("Columns:")
print(final_df.columns.tolist())

display(final_df.head(2))

Final merged rows: 250
Columns:
['id', 'url_x', 'title_x', 'summary_x', 'text_x', 'article_words_x', 'summary_words_x', 'url_y', 'title_y', 'summary_y', 'text_y', 'article_words_y', 'summary_words_y', 'indic_title', 'indic_article', 'indic_summary']


,id,url_x,title_x,summary_x,text_x,article_words_x,summary_words_x,url_y,title_y,summary_y,text_y,article_words_y,summary_words_y,indic_title,indic_article,indic_summary
0,080419_brchopra_festival,https://www.bbc.com/hindi/entertainment/story/...,बीआर चोपड़ा की यादगार फ़िल्मों का विशेष उत्सव,भारतीय फ़िल्म इतिहास में फ़िल्मकार बीआर चोपड़ा...,शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ ...,294,34,https://www.bbc.com/hindi/entertainment/story/...,बीआर चोपड़ा की यादगार फ़िल्मों का विशेष उत्सव,भारतीय फ़िल्म इतिहास में फ़िल्मकार बीआर चोपड़ा...,शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ ...,294.0,34.0,Special festival of memorable films of BR Chopra,"The festival, which began on Friday evening, s...",Filmmaker BR Chopra's name is taken from the b...
1,080613_afghan_jail,https://www.bbc.com/hindi/regionalnews/story/2...,तालेबान ने सैकड़ों क़ैदियों को छुड़ाया,अफ़ग़ानिस्तान के कंधार में तालेबान चरमपंथियों ...,कंधार के प्रांतीय सभा के अध्यक्ष और राष्ट्रपति...,289,17,https://www.bbc.com/hindi/regionalnews/story/2...,तालेबान ने सैकड़ों क़ैदियों को छुड़ाया,अफ़ग़ानिस्तान के कंधार में तालेबान चरमपंथियों ...,कंधार के प्रांतीय सभा के अध्यक्ष और राष्ट्रपति...,289.0,17.0,The Taliban have released hundreds of prisoners.,Kandahar provincial assembly chairman and Pres...,Taliban militants have attacked a prison in Ka...


In [21]:
translation_columns = all_translations[
    ["id", "indic_title", "indic_article", "indic_summary"]
].copy()

final_df = df_clean.merge(
    translation_columns,
    on="id",
    how="inner",
    validate="one_to_one"
)

# Save the clean merged dataset
FINAL_MERGED_FILE = os.path.join(
    BASE,
    "03_english_data/FactIndic_Hindi_English_250.csv"
)

final_df.to_csv(FINAL_MERGED_FILE, index=False)

print("Rows:", len(final_df))
print("Columns:", final_df.columns.tolist())
print("Duplicate IDs:", final_df["id"].duplicated().sum())
print("Saved:", FINAL_MERGED_FILE)

display(final_df.head(2))

Rows: 250
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words', 'indic_title', 'indic_article', 'indic_summary']
Duplicate IDs: 0
Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/03_english_data/FactIndic_Hindi_English_250.csv


,id,url,title,summary,text,article_words,summary_words,indic_title,indic_article,indic_summary
0,080419_brchopra_festival,https://www.bbc.com/hindi/entertainment/story/...,बीआर चोपड़ा की यादगार फ़िल्मों का विशेष उत्सव,भारतीय फ़िल्म इतिहास में फ़िल्मकार बीआर चोपड़ा...,शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ ...,294,34,Special festival of memorable films of BR Chopra,"The festival, which began on Friday evening, s...",Filmmaker BR Chopra's name is taken from the b...
1,080613_afghan_jail,https://www.bbc.com/hindi/regionalnews/story/2...,तालेबान ने सैकड़ों क़ैदियों को छुड़ाया,अफ़ग़ानिस्तान के कंधार में तालेबान चरमपंथियों ...,कंधार के प्रांतीय सभा के अध्यक्ष और राष्ट्रपति...,289,17,The Taliban have released hundreds of prisoners.,Kandahar provincial assembly chairman and Pres...,Taliban militants have attacked a prison in Ka...


In [22]:
import os

for folder in [
    "04_mixed_data",
    "05_quality_checks",
    "08_reports"
]:
    path = os.path.join(BASE, folder)

    print(f"\n--- {folder} ---")

    if os.path.exists(path):
        for file in os.listdir(path):
            print(file)
    else:
        print("Folder not found")


--- 04_mixed_data ---
FactIndic_Mixed_250_partial.csv
FactIndic_Mixed_250.csv
hindi_english_phrase_dictionary.json

--- 05_quality_checks ---
FactIndic_Sentence_Alignment.csv
alignment_quality_warning.json
NLLB_First_Article_Comparison.csv
NLLB_5_Article_Test.csv
NLLB_Article_5_Protected_Translation.txt
Short_English_Translations_Review.csv
Translation_Review_Candidates_207.csv
Human_Translation_Review_20.csv
PHINC_Corpus_13738.csv
CMI_MUTANT_PHINC_Comparison.csv
IndicTrans2_Quality_Review_Next_10.csv
IndicTrans2_Quality_Review_Batch_3.csv
IndicTrans2_Quality_Review_Batch_4.csv
IndicTrans2_Quality_Review_Batch_5.csv
IndicTrans2_Quality_Review_Batch_6.csv
IndicTrans2_First_60_Quality_Review.csv
IndicTrans2_First_60_Quality_Summary.csv

--- 08_reports ---


In [23]:
import pandas as pd
import os

mixed_folder = os.path.join(BASE, "04_mixed_data")

for filename in [
    "FactIndic_Mixed_250_partial.csv",
    "FactIndic_Mixed_250.csv"
]:
    path = os.path.join(mixed_folder, filename)

    df = pd.read_csv(path)

    print("\nFILE:", filename)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("Missing values:", df.isna().sum().sum())
    print("Duplicate IDs:", df["id"].duplicated().sum())

    display(df.head(2))


FILE: FactIndic_Mixed_250_partial.csv
Shape: (250, 13)
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words', 'english_title', 'english_text', 'english_summary', 'mixed_10_article', 'mixed_30_article', 'mixed_50_article']
Missing values: 0
Duplicate IDs: 0


,id,url,title,summary,text,article_words,summary_words,english_title,english_text,english_summary,mixed_10_article,mixed_30_article,mixed_50_article
0,080419_brchopra_festival,https://www.bbc.com/hindi/entertainment/story/...,बीआर चोपड़ा की यादगार फ़िल्मों का विशेष उत्सव,भारतीय फ़िल्म इतिहास में फ़िल्मकार बीआर चोपड़ा...,शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ ...,294,34,Special celebration of the memorable film of B...,"On Friday evening, the scales of justice were ...","In Indian film history, movie actor BR Bangn i...",शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ ...,शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ ...,शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ ...
1,080613_afghan_jail,https://www.bbc.com/hindi/regionalnews/story/2...,तालेबान ने सैकड़ों क़ैदियों को छुड़ाया,अफ़ग़ानिस्तान के कंधार में तालेबान चरमपंथियों ...,कंधार के प्रांतीय सभा के अध्यक्ष और राष्ट्रपति...,289,17,The Locker Delivered Hundreds of Prisoners,The president of the local assembly of the Bar...,The locks on the locks at the side of Afghanis...,कंधार के प्रांतीय सभा के अध्यक्ष और President ...,कंधार के प्रांतीय सभा के अध्यक्ष और President ...,कंधार के प्रांतीय सभा के अध्यक्ष और President ...



FILE: FactIndic_Mixed_250.csv
Shape: (250, 16)
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words', 'english_title', 'english_text', 'english_summary', 'mixed_10_article', 'mixed_30_article', 'mixed_50_article', 'mixed_10_article_english_pct', 'mixed_30_article_english_pct', 'mixed_50_article_english_pct']
Missing values: 0
Duplicate IDs: 0


,id,url,title,summary,text,article_words,summary_words,english_title,english_text,english_summary,mixed_10_article,mixed_30_article,mixed_50_article,mixed_10_article_english_pct,mixed_30_article_english_pct,mixed_50_article_english_pct
0,080419_brchopra_festival,https://www.bbc.com/hindi/entertainment/story/...,बीआर चोपड़ा की यादगार फ़िल्मों का विशेष उत्सव,भारतीय फ़िल्म इतिहास में फ़िल्मकार बीआर चोपड़ा...,शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ ...,294,34,Special celebration of the memorable film of B...,"On Friday evening, the scales of justice were ...","In Indian film history, movie actor BR Bangn i...",शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ ...,शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ ...,शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ ...,1.36,1.36,1.36
1,080613_afghan_jail,https://www.bbc.com/hindi/regionalnews/story/2...,तालेबान ने सैकड़ों क़ैदियों को छुड़ाया,अफ़ग़ानिस्तान के कंधार में तालेबान चरमपंथियों ...,कंधार के प्रांतीय सभा के अध्यक्ष और राष्ट्रपति...,289,17,The Locker Delivered Hundreds of Prisoners,The president of the local assembly of the Bar...,The locks on the locks at the side of Afghanis...,कंधार के प्रांतीय सभा के अध्यक्ष और President ...,कंधार के प्रांतीय सभा के अध्यक्ष और President ...,कंधार के प्रांतीय सभा के अध्यक्ष और President ...,2.42,2.42,2.42


In [24]:
mixed_path = os.path.join(
    BASE, "04_mixed_data/FactIndic_Mixed_250.csv"
)

mixed_df = pd.read_csv(mixed_path)

for col in [
    "mixed_10_article",
    "mixed_30_article",
    "mixed_50_article"
]:
    print(col, "unique values:", mixed_df[col].nunique())

print(
    "10% and 30% identical:",
    mixed_df["mixed_10_article"].equals(
        mixed_df["mixed_30_article"]
    )
)

print(
    "30% and 50% identical:",
    mixed_df["mixed_30_article"].equals(
        mixed_df["mixed_50_article"]
    )
)

mixed_10_article unique values: 250
mixed_30_article unique values: 250
mixed_50_article unique values: 250
10% and 30% identical: True
30% and 50% identical: True


In [25]:
import json
import os

dictionary_path = os.path.join(
    BASE,
    "04_mixed_data/hindi_english_phrase_dictionary.json"
)

with open(dictionary_path, "r", encoding="utf-8") as f:
    phrase_dict = json.load(f)

print("Dictionary entries:", len(phrase_dict))
print("Sample entries:")

for i, (hindi, english) in enumerate(phrase_dict.items()):
    if i == 10:
        break
    print(hindi, "→", english)

Dictionary entries: 26
Sample entries:
इंसाफ़ का तराज़ू → Insaf Ka Tarazu
छोटी सी बात → Chhoti Si Baat
नया दौर → Naya Daur
वक़्त → Waqt
साधना → Sadhana
गुमराह → Gumrah
हमराज़ → Humraaz
क़ानून → Kanoon
निकाह → Nikaah
बीआर चोपड़ा → B. R. Chopra


In [26]:
alignment_path = os.path.join(
    BASE,
    "05_quality_checks/FactIndic_Sentence_Alignment.csv"
)

alignment_df = pd.read_csv(alignment_path)

print("Shape:", alignment_df.shape)
print("\nColumns:")
print(alignment_df.columns.tolist())

print("\nFirst 3 rows:")
display(alignment_df.head(3))

Shape: (5580, 5)

Columns:
['sample_id', 'sentence_number', 'hindi_sentence', 'english_sentence', 'alignment_method']

First 3 rows:


,sample_id,sentence_number,hindi_sentence,english_sentence,alignment_method
0,080419_brchopra_festival,1,शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ ...,"On Friday evening, the scales of justice were ...",approximate_proportional
1,080419_brchopra_festival,2,"उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त ज...","On Friday evening, the scales of justice were ...",approximate_proportional
2,080419_brchopra_festival,3,खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में ...,"The little thing, the new rounds, and time dur...",approximate_proportional


In [27]:
alignment_path = os.path.join(
    BASE,
    "05_quality_checks/FactIndic_Sentence_Alignment.csv"
)

alignment_df = pd.read_csv(alignment_path)

print("Alignment methods:")
print(alignment_df["alignment_method"].value_counts(dropna=False))

print("\nRows for first article:")
first_id = alignment_df["sample_id"].iloc[0]

display(
    alignment_df[
        alignment_df["sample_id"] == first_id
    ].head(12)
)

Alignment methods:
alignment_method
approximate_proportional    5580
Name: count, dtype: int64

Rows for first article:


,sample_id,sentence_number,hindi_sentence,english_sentence,alignment_method
0,080419_brchopra_festival,1,शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ ...,"On Friday evening, the scales of justice were ...",approximate_proportional
1,080419_brchopra_festival,2,"उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त ज...","On Friday evening, the scales of justice were ...",approximate_proportional
2,080419_brchopra_festival,3,खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में ...,"The little thing, the new rounds, and time dur...",approximate_proportional
3,080419_brchopra_festival,4,"उन्होंने कहा, ""मेरे भाई ने अपने हिसाब से फ़िल्...","Bad health because of BRC's own function, BRCP...",approximate_proportional
4,080419_brchopra_festival,5,बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्वि...,When people were apprehensive about the commer...,approximate_proportional
5,080419_brchopra_festival,6,यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौ...,BR'Courourin raised many subjects like widowry...,approximate_proportional
6,080419_brchopra_festival,7,"गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में क...",The issue of sexy prostitution is to question ...,approximate_proportional
7,080419_brchopra_festival,8,हालांकि बीआर चोपड़ा की ज़्यादातर फ़िल्में किसी...,"Leading, wets, law, and marriage are included ...",approximate_proportional
8,080419_brchopra_festival,9,नया दौर की कहानी को कई टॉप निर्देशकों ने ठुकरा...,"However, most movies of BR's chlorophynism wer...",approximate_proportional
9,080419_brchopra_festival,10,बीआर चोपड़ा के बेटे रवि चोपड़ा ने इस मौके पर क...,Nested and made a film.,approximate_proportional


In [28]:
import re
from collections import Counter

coverage = []

for _, row in final_df.iterrows():
    hindi_article = str(row["text"])
    found = [
        hindi for hindi in phrase_dict
        if hindi in hindi_article
    ]

    coverage.append({
        "id": row["id"],
        "matched_phrases": len(found),
        "phrases": ", ".join(found)
    })

coverage_df = pd.DataFrame(coverage)

print("Articles with at least one dictionary phrase:",
      (coverage_df["matched_phrases"] > 0).sum())

print("Articles with no dictionary phrase:",
      (coverage_df["matched_phrases"] == 0).sum())

print("\nPhrase coverage:")
print(coverage_df["matched_phrases"].value_counts().sort_index())

display(coverage_df.head(10))

Articles with at least one dictionary phrase: 54
Articles with no dictionary phrase: 196

Phrase coverage:
matched_phrases
0     196
1      46
2       6
12      1
14      1
Name: count, dtype: int64


,id,matched_phrases,phrases
0,080419_brchopra_festival,14,"इंसाफ़ का तराज़ू, छोटी सी बात, नया दौर, वक़्त,..."
1,080613_afghan_jail,0,
2,130912_guinnness_record_bulgarian_swim_sr,0,
3,050713_us_un,0,
4,120809_salman_pkp,12,"यश चोपड़ा, एक था टाइगर, सलमान खान, कटरीना कैफ,..."
5,080711_lanka_attack,0,
6,social-43068605,0,
7,international-51410040,1,वक़्त
8,india-55525807,1,क़ानून
9,international-41372477,0,


In [29]:
print("Columns:", final_df.columns.tolist())

sample = final_df.iloc[0]

print("\nHindi article:")
print(str(sample["text"])[:1200])

print("\nIndicTrans2 English article:")
print(str(sample["indic_article"])[:1200])

print("\nHindi sentences:",
      len(re.findall(r'[।!?]+', str(sample["text"]))))

print("English sentences:",
      len(re.findall(r'[.!?]+', str(sample["indic_article"]))))

Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words', 'indic_title', 'indic_article', 'indic_summary']

Hindi article:
शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी." बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है. गुमराह, हमराज़,क़ानून और निकाह जैसी फ़ि

In [30]:
import re

for symbol in [".", "।", "!", "?", "\n"]:
    count = final_df["text"].astype(str).str.count(
        re.escape(symbol)
    ).sum()
    print(repr(symbol), ":", count)

print("\nSample sentence split:")
sample_text = str(final_df.iloc[0]["text"])

parts = re.split(r'(?<=[.!?।])\s+', sample_text)

for i, sentence in enumerate(parts[:8], 1):
    print(f"{i}. {sentence.strip()}")

'.' : 5713
'।' : 0
'!' : 12
'?' : 111
'\n' : 0

Sample sentence split:
1. शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई.
2. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी.
3. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया.
4. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी." बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया.
5. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है.
6. गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में शामिल है.
7. हाला

In [31]:
import re

def split_hindi_sentences(text):
    text = str(text).strip()
    sentences = re.split(r'(?<=[.!?।])\s*', text)
    return [s.strip() for s in sentences if s.strip()]

sentence_counts = final_df["text"].apply(
    lambda x: len(split_hindi_sentences(x))
)

print("Articles:", len(sentence_counts))
print("Total Hindi sentences:", sentence_counts.sum())
print("Average sentences per article:", round(sentence_counts.mean(), 2))
print("Minimum:", sentence_counts.min())
print("Maximum:", sentence_counts.max())

print("\nFirst article sentence count:",
      sentence_counts.iloc[0])

print("\nFirst article split:")
for i, sentence in enumerate(
    split_hindi_sentences(final_df.iloc[0]["text"])[:8], 1
):
    print(f"{i}. {sentence}")

Articles: 250
Total Hindi sentences: 5975
Average sentences per article: 23.9
Minimum: 5
Maximum: 76

First article sentence count: 15

First article split:
1. शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई.
2. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी.
3. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.
4. फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया.
5. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी.
6. " बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया.
7. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है.
8. गुमराह

In [32]:
import re

def split_hindi_sentences(text):
    text = re.sub(r'\s+', ' ', str(text)).strip()

    # Split after punctuation, including optional closing quotation marks
    parts = re.split(
        r'(?<=[.!?।])["”’\']?\s*',
        text
    )

    return [p.strip() for p in parts if p.strip()]

sample_sentences = split_hindi_sentences(
    final_df.iloc[0]["text"]
)

print("Sentence count:", len(sample_sentences))

for i, sentence in enumerate(sample_sentences[:10], 1):
    print(f"{i}. {sentence}")

Sentence count: 15
1. शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई.
2. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी.
3. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.
4. फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया.
5. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी.
6. बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया.
7. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है.
8. गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में शामिल है.
9. हालांकि बीआर चोपड़ा की ज़्यादातर फ़िल्में किसी न क

In [33]:
import re

placeholder_count = final_df["indic_article"].astype(str).str.contains(
    r"<\s*ID\d+\s*>",
    regex=True
).sum()

print("Articles containing translation placeholders:",
      placeholder_count)

print("\nExamples:")
for _, row in final_df[
    final_df["indic_article"].astype(str).str.contains(
        r"<\s*ID\d+\s*>",
        regex=True
    )
].head(5).iterrows():
    print("\nID:", row["id"])
    print(re.findall(r"<\s*ID\d+\s*>", row["indic_article"]))

Articles containing translation placeholders: 1

Examples:

ID: 080419_brchopra_festival
['< ID1 >']


In [34]:
import pandas as pd

final_df["hindi_word_count"] = (
    final_df["text"].fillna("").str.split().str.len()
)

final_df["english_word_count"] = (
    final_df["indic_article"].fillna("").str.split().str.len()
)

final_df["translation_ratio"] = (
    final_df["english_word_count"] /
    final_df["hindi_word_count"].replace(0, 1)
)

print("Empty English articles:",
      (final_df["english_word_count"] == 0).sum())

print("\nTranslation ratio summary:")
print(final_df["translation_ratio"].describe().round(2))

print("\nArticles with ratio below 0.3 or above 2.5:")
display(
    final_df[
        (final_df["translation_ratio"] < 0.3) |
        (final_df["translation_ratio"] > 2.5)
    ][["id", "hindi_word_count", "english_word_count",
       "translation_ratio"]].head(15)
)

Empty English articles: 0

Translation ratio summary:
count    250.00
mean       0.62
std        0.37
min        0.03
25%        0.35
50%        0.64
75%        0.79
max        2.46
Name: translation_ratio, dtype: float64

Articles with ratio below 0.3 or above 2.5:


,id,hindi_word_count,english_word_count,translation_ratio
7,international-51410040,1225,153,0.124898
8,india-55525807,912,172,0.188596
15,080826_blast_films,573,171,0.298429
18,130207_kashmir_rockband_imroz_sa,621,123,0.198068
20,india-48662333,1240,168,0.135484
22,061206_gennext_indopak_students,693,165,0.238095
25,061004_askus_geyser,800,160,0.200000
27,india-46106302,605,50,0.082645
28,060120_speaker_cashcamera,363,28,0.077135
29,international-41646598,1143,172,0.150481


In [35]:
review_df = final_df[
    final_df["translation_ratio"] < 0.30
].copy()

review_path = os.path.join(
    BASE,
    "05_quality_checks/IndicTrans2_Incomplete_Translation_Flagged.csv"
)

review_df.to_csv(review_path, index=False, encoding="utf-8-sig")

print("Articles flagged:", len(review_df))
print("Saved to:", review_path)

display(
    review_df[
        ["id", "hindi_word_count", "english_word_count",
         "translation_ratio"]
    ].sort_values("translation_ratio")
)

Articles flagged: 52
Saved to: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/IndicTrans2_Incomplete_Translation_Flagged.csv


,id,hindi_word_count,english_word_count,translation_ratio
55,india-51453120,775,23,0.029677
116,050506_india_coach,326,16,0.049080
87,121114_international_us_plus_obama_ia,790,39,0.049367
80,india-47148086,668,50,0.074850
28,060120_speaker_cashcamera,363,28,0.077135
221,india-49659647,705,56,0.079433
68,080416_up_results,275,22,0.080000
27,india-46106302,605,50,0.082645
122,science-39280869,961,80,0.083247
234,050523_bihar_nda,634,53,0.083596


In [36]:
row = final_df[
    final_df["id"] == "india-51453120"
].iloc[0]

print("HINDI ARTICLE (first 1500 characters):")
print(row["text"][:1500])

print("\nINDICTRANS2 TRANSLATION:")
print(row["indic_article"])

print("\nHindi words:", len(str(row["text"]).split()))
print("English words:", len(str(row["indic_article"]).split()))

HINDI ARTICLE (first 1500 characters):
अमरीकी राष्ट्रपति के साथ उनकी पत्नी मेलानिया ट्रंप भी होंगी. भारत यात्रा के दौरान ट्रंप दिल्ली और अहमदाबाद का दौरा करेंगे. अमरीकी सरकार की ओर से जारी बयान में कहा गया है कि बीते सप्ताहांत राष्ट्रपति ट्रंप और प्रधानमंत्री मोदी के बीच फ़ोन पर बात हुई. इस दौरान दोनों नेताओं ने भारत और अमरीका की जनता के बीच मज़बूत रिश्ते को उभारते हुए दोनों देशों के बीच रणनीतिक साझेदारी को और मज़बूत करने पर सहमति जताई. मोदी की शिनपिंग को लिखी चिट्ठी से ख़ुश हुआ चीन समाप्त चीन ने कोरोना वायरस से पैदा हुए हालात पर भारत के प्रधानमंत्री नरेंद्र मोदी की ओर से राष्ट्रपति शी जिनपिंग को भेजी गई चिट्ठी की तारीफ़ की है. चीन ने कहा है कि इससे भारत और चीन की दोस्ती पूरी तरह प्रदर्शित होती है. शी जिनपिंग को लिखे ख़त में मोदी ने वुहान से भारतीयों को निकालने के दौरान चीनी प्रशासन के सहयोग के लिए शुक्रिया कहा था और वायरस के कारण हो रहे नुक़सान को लेकर संवेदना जताई थी. भारत के प्रधानमंत्री ने हालात से निपटने के लिए अपनी ओर से चीन की मदद करने की भी पेशकश की थी. चीन के विदेश मंत्रालय के

In [37]:
usable_df = final_df[
    final_df["translation_ratio"] >= 0.30
].copy()

print("Usable for initial pilot:", len(usable_df))
print("Flagged for review:", len(final_df) - len(usable_df))

print("\nTranslation ratio for pilot:")
print(usable_df["translation_ratio"].describe().round(2))

print("\nLowest 10 ratios in pilot:")
display(
    usable_df[
        ["id", "hindi_word_count", "english_word_count",
         "translation_ratio"]
    ].sort_values("translation_ratio").head(10)
)

Usable for initial pilot: 198
Flagged for review: 52

Translation ratio for pilot:
count    198.00
mean       0.74
std        0.32
min        0.30
25%        0.55
50%        0.70
75%        0.83
max        2.46
Name: translation_ratio, dtype: float64

Lowest 10 ratios in pilot:


,id,hindi_word_count,english_word_count,translation_ratio
51,120518_ipl_glamour_magazine_sm,548,167,0.304745
115,050107_pravasi_pm,231,71,0.307359
42,141013_wusatullah_amitabh_bachchan_rns,536,166,0.309701
52,131119_chinese_supercomputer_worldbest_rns,510,165,0.323529
41,science-49370402,482,156,0.323651
245,india-50975864,880,285,0.323864
11,140323_breast_cancer_screening_tk,455,151,0.331868
32,141207_pakistan_taliban_mehsud_repatriated_rd,384,129,0.335938
124,international-44946029,769,262,0.340702
50,120816_assange_equador_pa,372,127,0.341398


In [38]:
pilot_path = os.path.join(
    BASE,
    "03_english_data/FactIndic_198_Translation_Pilot.csv"
)

usable_df.to_csv(
    pilot_path,
    index=False,
    encoding="utf-8-sig"
)

print("Pilot dataset saved!")
print("Rows:", len(usable_df))
print("File:", pilot_path)

Pilot dataset saved!
Rows: 198
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/03_english_data/FactIndic_198_Translation_Pilot.csv


In [39]:
pilot_3 = usable_df.head(3).copy()

print("Pilot articles:", len(pilot_3))

for col in ["id", "text", "indic_article"]:
    print(f"\n{col}:")
    print(pilot_3.iloc[0][col][:500] if col != "id"
          else pilot_3.iloc[0][col])

print("\nTarget English token proportions:")
print("10% | 30% | 50%")

Pilot articles: 3

id:
080419_brchopra_festival

text:
शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहान

indic_article:
The festival, which began on Friday evening, showcased the scale of Insaaf. Classic films like Chhoti Si Baat, Naya Daur and Waqt will be seen during the festival. BR Chopra himself did not attend the ceremony due to ill health. < ID1 > was inaugurated by BR Chopra's brother and noted filmmaker Yash Chopra. He said, "My brother made films according to his own wishes - that too at a time when people were apprehensive about th

In [40]:
placeholder_mask = usable_df["indic_article"].str.contains(
    r"<\s*ID\d+\s*>",
    regex=True,
    na=False
)

mixing_candidates = usable_df[~placeholder_mask].copy()

pilot_3 = mixing_candidates.head(3).copy()

print("Pilot articles selected:", len(pilot_3))

for _, row in pilot_3.iterrows():
    print("\n" + "=" * 60)
    print("ID:", row["id"])
    print("Translation ratio:", round(row["translation_ratio"], 2))
    print("\nHINDI:")
    print(row["text"][:350])
    print("\nENGLISH:")
    print(row["indic_article"][:350])

Pilot articles selected: 3

ID: 080613_afghan_jail
Translation ratio: 0.61

HINDI:
कंधार के प्रांतीय सभा के अध्यक्ष और राष्ट्रपति हामिद करज़ई के भाई वली करज़ई ने बीबीसी को बताया कि उस समय जेल में एक हज़ार क़ैदी थे और सभी निकल भागे हैं. माना जाता है कि इन क़ैदियों में से 350 तालेबान थे. इस हमले में कई सुरक्षाकर्मियों की जानें गई हैं. अधिकारियों के अनुसार जेल के मुख्य द्वार को उड़ाने के लिए एक लॉरी बम से विस्फोट किया गया और फिर 40 

ENGLISH:
Kandahar provincial assembly chairman and President Hamid Karzai's brother Wali Karzai told the BBC that there were 1,000 prisoners in the prison at the time and all had escaped. Of these prisoners, 350 were believed to be Taliban. Several security personnel have died in the attack. According to officials, a lorry bomb was detonated to blow up the 

ID: 130912_guinnness_record_bulgarian_swim_sr
Translation ratio: 0.74

HINDI:
ख़बरों के मुताबिक़ जेन पेत्कोव को मैसेडोनिया की ऑरिड झील को पार करने में करीब पौने तीन घंटे लगे. आयोजक सासो तोकोव ने समाचार एज

In [41]:
for _, row in pilot_3.iterrows():
    hindi_words = len(str(row["text"]).split())
    english_words = len(str(row["indic_article"]).split())

    print("\nID:", row["id"])
    print("Hindi words:", hindi_words)
    print("English words:", english_words)
    print("Ratio:", round(english_words / max(hindi_words, 1), 3))
    print("English ending:", str(row["indic_article"])[-250:])


ID: 080613_afghan_jail
Hindi words: 289
English words: 176
Ratio: 0.609
English ending:  the prison was so powerful that windowpanes were broken three kilometers away. BBC correspondent reports that this is a protest where the Afghan President will also be put in a select place, and the US President's security forces are also concerned.

ID: 130912_guinnness_record_bulgarian_swim_sr
Hindi words: 200
English words: 149
Ratio: 0.745
English ending: cial inclusion in the Guinness World Records. However, there are conflicting reports in the local media about the distance and time covered by Petkov. We have no public comments on this issue on the Guinness World Records and the BBC's Facebook page.

ID: 050713_us_un
Hindi words: 245
English words: 171
Ratio: 0.698
English ending: posal, the new permanent members will include African countries in addition to the G-4 countries. The G-4 countries are currently agreeing to the permanent membership without veto rights. The G-4 countries have bee

In [42]:
import re

def tokenize(text):
    return re.findall(
        r"[A-Za-z]+(?:['’-][A-Za-z]+)*|[\u0900-\u097F]+|\d+",
        str(text)
    )

row = pilot_3.iloc[0]

hindi_tokens = tokenize(row["text"])
english_tokens = tokenize(row["indic_article"])

print("Hindi tokens:", len(hindi_tokens))
print("English tokens:", len(english_tokens))

print("\nHindi token sample:")
print(hindi_tokens[:25])

print("\nEnglish token sample:")
print(english_tokens[:25])

Hindi tokens: 289
English tokens: 177

Hindi token sample:
['कंधार', 'के', 'प्रांतीय', 'सभा', 'के', 'अध्यक्ष', 'और', 'राष्ट्रपति', 'हामिद', 'करज़ई', 'के', 'भाई', 'वली', 'करज़ई', 'ने', 'बीबीसी', 'को', 'बताया', 'कि', 'उस', 'समय', 'जेल', 'में', 'एक', 'हज़ार']

English token sample:
['Kandahar', 'provincial', 'assembly', 'chairman', 'and', 'President', 'Hamid', "Karzai's", 'brother', 'Wali', 'Karzai', 'told', 'the', 'BBC', 'that', 'there', 'were', '1', '000', 'prisoners', 'in', 'the', 'prison', 'at', 'the']


In [43]:
import inspect

print("Translation function available:", callable(translate_one))

print("\nFunction signature:")
print(inspect.signature(translate_one))

print("\nTokenizer available:", tokenizer is not None)
print("Model available:", model is not None)

Translation function available: True

Function signature:
(text)

Tokenizer available: True
Model available: True


In [44]:
test_hindi = pilot_3.iloc[0]["text"]

# Take the first 100 Hindi words
test_chunk = " ".join(test_hindi.split()[:100])

test_english = translate_one(test_chunk)

print("HINDI CHUNK:")
print(test_chunk)

print("\nENGLISH TRANSLATION:")
print(test_english)

HINDI CHUNK:
कंधार के प्रांतीय सभा के अध्यक्ष और राष्ट्रपति हामिद करज़ई के भाई वली करज़ई ने बीबीसी को बताया कि उस समय जेल में एक हज़ार क़ैदी थे और सभी निकल भागे हैं. माना जाता है कि इन क़ैदियों में से 350 तालेबान थे. इस हमले में कई सुरक्षाकर्मियों की जानें गई हैं. अधिकारियों के अनुसार जेल के मुख्य द्वार को उड़ाने के लिए एक लॉरी बम से विस्फोट किया गया और फिर 40 तालेबान जेल के भीतर घुस आए. कंधार शहर में आपातकाल की घोषणा कर दी गई है. पुलिस और सैनिक सड़कों पर गश्त कर रहे हैं और लोगों

ENGLISH TRANSLATION:
Wali Karzai, the president of the provincial assembly of Kandahar and the brother of President Hamid Karzai, told the BBC that there were 1,000 prisoners in the prison at the time and all had escaped. Of these prisoners, 350 were believed to be Taliban. Several security personnel have lost their lives in the attack. According to officials, a lorry bomb was detonated to blow up the main gate of the prison and 40 Taliban entered the prison. Emergency has been declared in the city of Kandaha

In [45]:
print("Hindi word count:", len(test_chunk.split()))
print("English word count:", len(test_english.split()))

print("\nOriginal Hindi:")
print(test_chunk)

print("\nTranslated English:")
print(test_english)

Hindi word count: 100
English word count: 99

Original Hindi:
कंधार के प्रांतीय सभा के अध्यक्ष और राष्ट्रपति हामिद करज़ई के भाई वली करज़ई ने बीबीसी को बताया कि उस समय जेल में एक हज़ार क़ैदी थे और सभी निकल भागे हैं. माना जाता है कि इन क़ैदियों में से 350 तालेबान थे. इस हमले में कई सुरक्षाकर्मियों की जानें गई हैं. अधिकारियों के अनुसार जेल के मुख्य द्वार को उड़ाने के लिए एक लॉरी बम से विस्फोट किया गया और फिर 40 तालेबान जेल के भीतर घुस आए. कंधार शहर में आपातकाल की घोषणा कर दी गई है. पुलिस और सैनिक सड़कों पर गश्त कर रहे हैं और लोगों

Translated English:
Wali Karzai, the president of the provincial assembly of Kandahar and the brother of President Hamid Karzai, told the BBC that there were 1,000 prisoners in the prison at the time and all had escaped. Of these prisoners, 350 were believed to be Taliban. Several security personnel have lost their lives in the attack. According to officials, a lorry bomb was detonated to blow up the main gate of the prison and 40 Taliban entered the prison. Em

In [46]:
for idx, row in pilot_3.iterrows():
    hindi_words = row["text"].split()

    print("\n" + "="*60)
    print("ARTICLE:", row["id"])
    print("Hindi words:", len(hindi_words))

    chunk = " ".join(hindi_words[:100])
    english = translate_one(chunk)

    print("English words:", len(english.split()))
    print("Translation:")
    print(english[:500])


ARTICLE: 080613_afghan_jail
Hindi words: 289
English words: 99
Translation:
Wali Karzai, the president of the provincial assembly of Kandahar and the brother of President Hamid Karzai, told the BBC that there were 1,000 prisoners in the prison at the time and all had escaped. Of these prisoners, 350 were believed to be Taliban. Several security personnel have lost their lives in the attack. According to officials, a lorry bomb was detonated to blow up the main gate of the prison and 40 Taliban entered the prison. Emergency has been declared in the city of Kandahar. Poli

ARTICLE: 130912_guinnness_record_bulgarian_swim_sr
Hindi words: 200
English words: 82
Translation:
According to reports, it took Jane Petkov about three and a half hours to cross Lake Aurid in Macedonia. Organizer Saso Tokov told AFP news agency that 59-year-old Petkov's hands and legs were tied to his body and he swam on his back like a dolphin. According to Macedonian media, his average swimming speed was 0.7 km / h

In [47]:
import math

hindi_text = pilot_3.iloc[0]["text"]
words = hindi_text.split()

chunks = [
    " ".join(words[i:i+100])
    for i in range(0, len(words), 100)
]

translated_chunks = []

for i, chunk in enumerate(chunks):
    print(f"Translating chunk {i+1}/{len(chunks)}...")
    translated_chunks.append(translate_one(chunk))

complete_english = " ".join(translated_chunks)

print("\nOriginal Hindi words:", len(words))
print("English words:", len(complete_english.split()))
print("\nComplete English translation:")
print(complete_english)

Translating chunk 1/3...
Translating chunk 2/3...
Translating chunk 3/3...

Original Hindi words: 289
English words: 246

Complete English translation:
Wali Karzai, the president of the provincial assembly of Kandahar and the brother of President Hamid Karzai, told the BBC that there were 1,000 prisoners in the prison at the time and all had escaped. Of these prisoners, 350 were believed to be Taliban. Several security personnel have lost their lives in the attack. According to officials, a lorry bomb was detonated to blow up the main gate of the prison and 40 Taliban entered the prison. Emergency has been declared in the city of Kandahar. Police and soldiers are patrolling the streets and people are on the run. They have been told not to leave their homes. Jail chief Abdul Qabir has confirmed to the AP that hundreds of prisoners have escaped, but he says some of the prisoners stayed in the jail. Witnesses say the explosion to break the main prison door was so powerful that the windows

In [48]:
pilot_translations = {}

for idx, row in pilot_3.iterrows():
    words = row["text"].split()

    chunks = [
        " ".join(words[i:i+100])
        for i in range(0, len(words), 100)
    ]

    translated_chunks = []

    for chunk in chunks:
        translated_chunks.append(translate_one(chunk))

    complete_english = " ".join(translated_chunks)

    pilot_translations[row["id"]] = complete_english

    print("\nARTICLE:", row["id"])
    print("Hindi words:", len(words))
    print("English words:", len(complete_english.split()))
    print("Translation preview:", complete_english[:300])


ARTICLE: 080613_afghan_jail
Hindi words: 289
English words: 246
Translation preview: Wali Karzai, the president of the provincial assembly of Kandahar and the brother of President Hamid Karzai, told the BBC that there were 1,000 prisoners in the prison at the time and all had escaped. Of these prisoners, 350 were believed to be Taliban. Several security personnel have lost their liv

ARTICLE: 130912_guinnness_record_bulgarian_swim_sr
Hindi words: 200
English words: 116
Translation preview: According to reports, it took Jane Petkov about three and a half hours to cross Lake Aurid in Macedonia. Organizer Saso Tokov told AFP news agency that 59-year-old Petkov's hands and legs were tied to his body and he swam on his back like a dolphin. According to Macedonian media, his average swimmin

ARTICLE: 050713_us_un
Hindi words: 245
English words: 220
Translation preview: The G-4, which includes India as well as Brazil, Japan and Germany, proposed to increase the number of members of the Secur

In [49]:
import pandas as pd
import os

pilot_output = pilot_3.copy()

pilot_output["complete_english_translation"] = (
    pilot_output["id"].map(pilot_translations)
)

save_path = os.path.join(
    BASE,
    "03_english_data",
    "FactIndic_Chunk_Translation_Pilot_3.csv"
)

pilot_output.to_csv(save_path, index=False)

print("Pilot translations saved!")
print("Rows:", len(pilot_output))
print("File:", save_path)
print(
    "Missing translations:",
    pilot_output["complete_english_translation"].isna().sum()
)

Pilot translations saved!
Rows: 3
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/03_english_data/FactIndic_Chunk_Translation_Pilot_3.csv
Missing translations: 0


In [50]:
import os
import pandas as pd

checkpoint_path = os.path.join(
    BASE,
    "03_english_data",
    "FactIndic_Chunk_Translation_Checkpoints.csv"
)

if os.path.exists(checkpoint_path):
    checkpoint_df = pd.read_csv(checkpoint_path)
    completed_ids = set(checkpoint_df["id"].astype(str))
else:
    completed_ids = set()

print("Total articles:", len(final_df))
print("Already completed:", len(completed_ids))
print("Remaining:", len(final_df) - len(completed_ids))

Total articles: 250
Already completed: 0
Remaining: 250


In [51]:
import os
import pandas as pd
import time

checkpoint_path = os.path.join(
    BASE, "03_english_data",
    "FactIndic_Chunk_Translation_Checkpoints.csv"
)

if os.path.exists(checkpoint_path):
    checkpoint_df = pd.read_csv(checkpoint_path)
    completed_ids = set(checkpoint_df["id"].astype(str))
else:
    checkpoint_df = pd.DataFrame()
    completed_ids = set()

remaining_df = final_df[
    ~final_df["id"].astype(str).isin(completed_ids)
].copy()

print("Articles to translate:", len(remaining_df))
print("Starting checkpointed translation...")

for _, row in remaining_df.iterrows():
    words = str(row["text"]).split()

    chunks = [
        " ".join(words[i:i+100])
        for i in range(0, len(words), 100)
    ]

    translated_chunks = []

    try:
        for chunk in chunks:
            translated_chunks.append(translate_one(chunk))

        english_text = " ".join(translated_chunks)

        new_row = row.to_dict()
        new_row["complete_english_translation"] = english_text

        checkpoint_df = pd.concat(
            [checkpoint_df, pd.DataFrame([new_row])],
            ignore_index=True
        )

        checkpoint_df.to_csv(checkpoint_path, index=False)

        print(
            f"Saved {len(checkpoint_df)}/250:",
            row["id"]
        )

    except Exception as e:
        print("ERROR:", row["id"], str(e))
        print("Stopping safely. Completed articles are saved.")
        break

print("\nCheckpoint run finished.")
print("Saved articles:", len(checkpoint_df))
print("Checkpoint:", checkpoint_path)

Articles to translate: 250
Starting checkpointed translation...
Saved 1/250: 080419_brchopra_festival
Saved 2/250: 080613_afghan_jail
Saved 3/250: 130912_guinnness_record_bulgarian_swim_sr
Saved 4/250: 050713_us_un
Saved 5/250: 120809_salman_pkp
Saved 6/250: 080711_lanka_attack
Saved 7/250: social-43068605
Saved 8/250: international-51410040
Saved 9/250: india-55525807
Saved 10/250: international-41372477
Saved 11/250: 130404_sam_pitroda_profile
Saved 12/250: 140323_breast_cancer_screening_tk
Saved 13/250: 150520_emergency_service_ss
Saved 14/250: 081126_aamne_pachauri_alk
Saved 15/250: 081118_cg_campaign_vv
Saved 16/250: 080826_blast_films
Saved 17/250: 141007_bhutan_corruption_india_vr
Saved 18/250: 140702_wc2014_goldenboot_contenders_ar
Saved 19/250: 130207_kashmir_rockband_imroz_sa
Saved 20/250: 051105_sahubuddin_arrest
Saved 21/250: india-48662333
Saved 22/250: 070206_guj_parza
Saved 23/250: 061206_gennext_indopak_students
Saved 24/250: 141016_ranveer_removes_pant_md_part1_pkp
Sav

In [52]:
import pandas as pd
import os

chunk_df = pd.read_csv(checkpoint_path)

chunk_df["hindi_word_count"] = (
    chunk_df["text"].fillna("").astype(str).str.split().str.len()
)

chunk_df["english_word_count"] = (
    chunk_df["complete_english_translation"]
    .fillna("").astype(str).str.split().str.len()
)

chunk_df["translation_ratio"] = (
    chunk_df["english_word_count"] /
    chunk_df["hindi_word_count"].replace(0, pd.NA)
)

chunk_df["quality_flag"] = "Check"

chunk_df.loc[
    (chunk_df["english_word_count"] == 0) |
    (chunk_df["hindi_word_count"] == 0),
    "quality_flag"
] = "Empty"

chunk_df.loc[
    (chunk_df["translation_ratio"] < 0.30) &
    (chunk_df["quality_flag"] != "Empty"),
    "quality_flag"
] = "Review: short"

print("Total rows:", len(chunk_df))
print("Unique articles:", chunk_df["id"].nunique())
print("\nQuality flags:")
print(chunk_df["quality_flag"].value_counts())

print("\nTranslation ratio:")
print(chunk_df["translation_ratio"].describe())

audit_path = os.path.join(
    BASE, "05_quality_checks",
    "Chunk_Translation_Quality_Audit.csv"
)

os.makedirs(os.path.dirname(audit_path), exist_ok=True)
chunk_df.to_csv(audit_path, index=False)

print("\nAudit saved:", audit_path)

Total rows: 250
Unique articles: 250

Quality flags:
quality_flag
Check            248
Review: short      2
Name: count, dtype: int64

Translation ratio:
count    250.000000
mean       0.795281
std        0.418929
min        0.260664
25%        0.688894
50%        0.791451
75%        0.865017
max        6.961741
Name: translation_ratio, dtype: float64

Audit saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/Chunk_Translation_Quality_Audit.csv


In [53]:
flagged = chunk_df[
    chunk_df["quality_flag"] != "Check"
]

for _, row in flagged.iterrows():
    print("\nARTICLE:", row["id"])
    print("Hindi words:", row["hindi_word_count"])
    print("English words:", row["english_word_count"])
    print("Ratio:", round(row["translation_ratio"], 3))
    print("\nHindi:")
    print(str(row["text"])[:700])
    print("\nEnglish:")
    print(str(row["complete_english_translation"])[:700])

print("\nHighest translation ratio:")
print(
    chunk_df.nlargest(1, "translation_ratio")[
        ["id", "hindi_word_count", "english_word_count",
         "translation_ratio"]
    ].to_string(index=False)
)


ARTICLE: 060530_reservation_strike
Hindi words: 211
English words: 55
Ratio: 0.261

Hindi:
देश भर के मेडिकल कॉलेजों के प्रतिनिधियों ने सोमवार को सुप्रीम कोर्ट के आदेश पर विचार करने के बाद हड़ताल जारी रखने का निर्णय लिया. उल्लेखनीय है कि केंद्र सरकार के समर्थन से चल रहे उच्च शिक्षा संस्थानों में पिछड़ी जातियों के छात्रों को आरक्षण देने के विरोध में मेडिकल छात्र हड़ताल कर रहे हैं. उधर सरकार ने इस हड़ताल के बावजूद ये घोषणा की है कि आरक्षण जून 2007 से लागू हो जाएगा. हालांकि सरकार ने सभी वर्गों के हितों को ध्यान में रखने का आश्लासन दिया है और इसके लिए संस्थानों में सीटें बढ़ाने की तैयारियाँ की जा रही हैं. छात्रों ने सुप्रीम कोर्ट के उस आदेश का स्वागत किया है जिसमें केंद्र सरकार को नोटिस जारी करके पूछा गया है कि पिछड़े वर्ग को आरक्षण दिए जाने का आधार क्या है. मेडिकल छात्रों ने कहा है कि

English:
Representatives of medical colleges across the country on Monday decided to continue the strike after considering the Supreme Court order. The students have welcomed the Supreme Court order, which 

In [54]:
for article_id in [
    "060530_reservation_strike",
    "081202_rice_india_ac",
    "071104_komal_column"
]:
    row = chunk_df[chunk_df["id"] == article_id].iloc[0]

    words = str(row["text"]).split()

    chunks = [
        " ".join(words[i:i+100])
        for i in range(0, len(words), 100)
    ]

    print("\n" + "="*65)
    print("ARTICLE:", article_id)
    print("Number of chunks:", len(chunks))

    for i, chunk in enumerate(chunks):
        result = translate_one(chunk)

        print(f"\nCHUNK {i+1}")
        print("Hindi words:", len(chunk.split()))
        print("English words:", len(result.split()))
        print("Translation:", result)


ARTICLE: 060530_reservation_strike
Number of chunks: 3

CHUNK 1
Hindi words: 100
English words: 20
Translation: Representatives of medical colleges across the country on Monday decided to continue the strike after considering the Supreme Court order.

CHUNK 2
Hindi words: 100
English words: 26
Translation: The students have welcomed the Supreme Court order, which issued a notice to the central government, asking what is the basis for reservation for backward classes.

CHUNK 3
Hindi words: 11
English words: 9
Translation: In addition, strikes have been called in private hospitals.

ARTICLE: 081202_rice_india_ac
Number of chunks: 3

CHUNK 1
Hindi words: 100
English words: 13
Translation: Condoleezza Rice will meet Indian Prime Minister Manmohan Singh and other senior ministers.

CHUNK 2
Hindi words: 100
English words: 26
Translation: US Republican Senator and presidential candidate John McCain, who arrived in Delhi on Tuesday to express sympathy after the Mumbai attacks, met Prime Minis

In [55]:
import inspect

print(inspect.getsource(translate_one))

def translate_one(text):
    if pd.isna(text) or not str(text).strip():
        return ""

    batch = ip.preprocess_batch(
        [str(text)],
        src_lang="hin_Deva",
        tgt_lang="eng_Latn"
    )

    inputs = tokenizer(
        batch,
        truncation=True,
        max_length=1024,
        padding=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=1024,
            num_beams=1,
            use_cache=False
        )

    decoded = tokenizer.batch_decode(
        output,
        skip_special_tokens=True
    )

    return ip.postprocess_batch(
        decoded,
        lang="eng_Latn"
    )[0]



In [56]:
def translate_one(text):
    if pd.isna(text) or not str(text).strip():
        return ""

    batch = ip.preprocess_batch(
        [str(text)],
        src_lang="hin_Deva",
        tgt_lang="eng_Latn"
    )

    inputs = tokenizer(
        batch,
        truncation=True,
        max_length=512,
        padding=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=512,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
            use_cache=False
        )

    decoded = tokenizer.batch_decode(
        output,
        skip_special_tokens=True
    )

    result = ip.postprocess_batch(
        decoded,
        lang="eng_Latn"
    )[0]

    return result.strip()

In [57]:
test_ids = [
    "060530_reservation_strike",
    "081202_rice_india",
    "071104_komal_column"
]

for article_id in test_ids:
    row = df_clean[df_clean["id"] == article_id]

    if row.empty:
        print(f"\nArticle not found: {article_id}")
        continue

    hindi_text = row.iloc[0]["text"]
    english_text = translate_one(hindi_text)

    print("\n" + "=" * 60)
    print("ID:", article_id)
    print("Hindi words:", len(str(hindi_text).split()))
    print("English words:", len(english_text.split()))
    print("English translation preview:")
    print(english_text[:1000])


ID: 060530_reservation_strike
Hindi words: 211
English words: 155
English translation preview:
Representatives of medical colleges across the country on Monday decided to continue the strike after considering the Supreme Court's order. Notably, medical students are on strike to protest against the reservation of backward caste students in higher education institutions supported by the central government. Despite this strike, the government has announced that reservation will come into effect from June 2007. However, government has promised to take care of the interests of all sections and preparations are being made to increase seats in the institutions for this. Students have welcomed the order of Supreme Court which has issued a notice to the Central Government asking what is the basis of reservation for backward classes. Medical students have said that they are not ending the strike but will run parallel OPDs outside the hospital. Meanwhile, the strike started from Delhi has spread

In [58]:
check_ids = [
    "060530_reservation_strike",
    "081202_rice_india",
    "071104_komal_column"
]

for name in ["df_clean", "df_250", "final_df", "chunk_df"]:
    obj = globals().get(name)

    if isinstance(obj, pd.DataFrame) and "id" in obj.columns:
        print(f"\n{name}:")
        for article_id in check_ids:
            print(article_id, article_id in obj["id"].astype(str).values)

row = df_clean[df_clean["id"].astype(str) == "071104_komal_column"]

if not row.empty:
    hindi = str(row.iloc[0]["text"])
    batch = ip.preprocess_batch(
        [hindi],
        src_lang="hin_Deva",
        tgt_lang="eng_Latn"
    )
    tokens = tokenizer(batch, truncation=False)
    print("\nHindi word count:", len(hindi.split()))
    print("Input token count:", len(tokens["input_ids"][0]))

Token indices sequence length is longer than the specified maximum sequence length for this model (1258 > 256). Running this sequence through the model will result in indexing errors



df_clean:
060530_reservation_strike True
081202_rice_india False
071104_komal_column True

final_df:
060530_reservation_strike True
081202_rice_india False
071104_komal_column True

chunk_df:
060530_reservation_strike True
081202_rice_india False
071104_komal_column True

Hindi word count: 758
Input token count: 1258


In [60]:
print("Model config class:", type(model.config).__name__)
print("\nModel configuration:")
print(model.config)

print("\nEncoder configuration:")
print(getattr(model.config, "encoder", "Not available"))

print("\nTokenizer maximum:", tokenizer.model_max_length)

Model config class: IndicTransConfig

Model configuration:
IndicTransConfig {
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "architectures": [
    "IndicTransForConditionalGeneration"
  ],
  "attention_dropout": 0.0,
  "attn_implementation": null,
  "auto_map": {
    "AutoConfig": "configuration_indictrans.IndicTransConfig",
    "AutoModelForSeq2SeqLM": "modeling_indictrans.IndicTransForConditionalGeneration"
  },
  "bos_token_id": 0,
  "decoder_attention_heads": 8,
  "decoder_embed_dim": 512,
  "decoder_ffn_dim": 2048,
  "decoder_layerdrop": 0,
  "decoder_layers": 18,
  "decoder_normalize_before": true,
  "decoder_start_token_id": 2,
  "decoder_vocab_size": 32296,
  "dropout": 0.2,
  "dtype": "float16",
  "encoder_attention_heads": 8,
  "encoder_embed_dim": 512,
  "encoder_ffn_dim": 2048,
  "encoder_layerdrop": 0,
  "encoder_layers": 18,
  "encoder_normalize_before": true,
  "encoder_vocab_size": 122706,
  "eos_token_id": 2,
  "init_std": 0.02,
  "is_encoder_decoder"

In [61]:
import re

sample_text = str(
    df_clean.loc[
        df_clean["id"] == "071104_komal_column", "text"
    ].iloc[0]
)

sentences = re.split(r'(?<=[।!?])\s+', sample_text.strip())

print("Total sentences:", len(sentences))
print("\nFirst 5 sentences:")
for i, sentence in enumerate(sentences[:5], 1):
    print(f"{i}. {sentence}")

Total sentences: 5

First 5 sentences:
1. राकेश ओमप्रकाश मेहरा ने हाल ही में अपनी नई फ़िल्म 'दिल्ली 6' के लिए अभिषेक बच्चन को साइन किया था और अब बतौर हीरोईन सोनम को लिया गया है. फ़िल्म की निर्माता कंपनी यूटीवी ने सोनम और उनके पापा अनिल कपूर के साथ पैसों के मामले में ज़्यादा बहस नहीं की बल्कि सोनम की फीस को लेकर बिल्कुल खींचातानी नहीं हुई. 'रंग दे बसंती' वाले राकेश मेहरा की फ़िल्म में काम करने के लिए सोनम को अच्छी-ख़ासी रकम मिलेगी. फ़िल्म की कहानी सोनम और अनिल कपूर दोनों ने सुनी है. रणबीर कपूर ने कुछ महीनों पहले यश चोपड़ा की नई फ़िल्म में काम करने के लिए हामी भर दी थी. ये फ़िल्म सिद्धार्थ आनंद (जिन्होंने इससे पहले 'सलाम नमस्ते' और 'ता रा रम पम' बनाई थी) के निर्देशन में बनेगी. ****************************************************************** अज़ीब इत्तफ़ाक ये इत्तफ़ाक है लेकिन अज़ीब इत्तफ़ाक. शाहरुख़ ख़ान ने अपना करियर 1992 में ऋषि कपूर के साथ शुरू किया था. और अब, पंद्रह साल बाद ऋषि के बेटे रणबीर कपूर के करियर की शुरुआत शाहरुख़ ख़ान के साथ शुरू होने जा रही है. फ़र्क इतना है कि शाहरुख़ औ

In [62]:
text = sample_text

sentences = re.split(
    r'(?<=[।!?])\s+|(?<=\.)\s+(?=[अ-ह])',
    text.strip()
)

print("Total sentences:", len(sentences))

for i, sentence in enumerate(sentences[:12], 1):
    print(f"\n{i}. {sentence[:250]}")

Total sentences: 37

1. राकेश ओमप्रकाश मेहरा ने हाल ही में अपनी नई फ़िल्म 'दिल्ली 6' के लिए अभिषेक बच्चन को साइन किया था और अब बतौर हीरोईन सोनम को लिया गया है.

2. फ़िल्म की निर्माता कंपनी यूटीवी ने सोनम और उनके पापा अनिल कपूर के साथ पैसों के मामले में ज़्यादा बहस नहीं की बल्कि सोनम की फीस को लेकर बिल्कुल खींचातानी नहीं हुई. 'रंग दे बसंती' वाले राकेश मेहरा की फ़िल्म में काम करने के लिए सोनम को अच्छी-ख़ासी रकम म

3. फ़िल्म की कहानी सोनम और अनिल कपूर दोनों ने सुनी है.

4. रणबीर कपूर ने कुछ महीनों पहले यश चोपड़ा की नई फ़िल्म में काम करने के लिए हामी भर दी थी.

5. ये फ़िल्म सिद्धार्थ आनंद (जिन्होंने इससे पहले 'सलाम नमस्ते' और 'ता रा रम पम' बनाई थी) के निर्देशन में बनेगी. ****************************************************************** अज़ीब इत्तफ़ाक ये इत्तफ़ाक है लेकिन अज़ीब इत्तफ़ाक.

6. शाहरुख़ ख़ान ने अपना करियर 1992 में ऋषि कपूर के साथ शुरू किया था.

7. और अब, पंद्रह साल बाद ऋषि के बेटे रणबीर कपूर के करियर की शुरुआत शाहरुख़ ख़ान के साथ शुरू होने जा रही है.

8. फ़र्क इतना है कि शाहरु

In [63]:
sentence_token_counts = []

for i, sentence in enumerate(sentences):
    batch = ip.preprocess_batch(
        [sentence],
        src_lang="hin_Deva",
        tgt_lang="eng_Latn"
    )

    token_count = len(
        tokenizer(
            batch,
            truncation=False
        )["input_ids"][0]
    )

    sentence_token_counts.append(token_count)

print("Total segments:", len(sentences))
print("Largest segment:", max(sentence_token_counts))
print("Segments above 200 tokens:",
      sum(x > 200 for x in sentence_token_counts))

for i, count in enumerate(sentence_token_counts):
    if count > 200:
        print(f"Segment {i+1}: {count} tokens")

Total segments: 37
Largest segment: 150
Segments above 200 tokens: 0


In [64]:
def make_token_chunks(text, max_tokens=200):
    parts = re.split(
        r'(?<=[।!?])\s+|(?<=\.)\s+(?=[अ-ह])',
        str(text).strip()
    )

    chunks = []
    current = []

    for sentence in parts:
        sentence = sentence.strip()

        if not sentence:
            continue

        candidate = " ".join(current + [sentence])

        batch = ip.preprocess_batch(
            [candidate],
            src_lang="hin_Deva",
            tgt_lang="eng_Latn"
        )

        count = len(
            tokenizer(batch, truncation=False)["input_ids"][0]
        )

        if count <= max_tokens:
            current.append(sentence)
        else:
            if current:
                chunks.append(" ".join(current))

            current = [sentence]

    if current:
        chunks.append(" ".join(current))

    return chunks


test_chunks = make_token_chunks(sample_text)

print("Chunks created:", len(test_chunks))

for i, chunk in enumerate(test_chunks, 1):
    batch = ip.preprocess_batch(
        [chunk],
        src_lang="hin_Deva",
        tgt_lang="eng_Latn"
    )
    count = len(tokenizer(batch, truncation=False)["input_ids"][0])

    print(f"Chunk {i}: {count} tokens | {len(chunk.split())} words")

Chunks created: 8
Chunk 1: 132 tokens | 110 words
Chunk 2: 182 tokens | 84 words
Chunk 3: 181 tokens | 86 words
Chunk 4: 66 tokens | 47 words
Chunk 5: 189 tokens | 101 words
Chunk 6: 192 tokens | 103 words
Chunk 7: 192 tokens | 102 words
Chunk 8: 145 tokens | 125 words


In [65]:
translated_chunks = []

for i, chunk in enumerate(test_chunks, 1):
    translation = translate_one(chunk)
    translated_chunks.append(translation)

    print(f"\nCHUNK {i}")
    print("Hindi words:", len(chunk.split()))
    print("English words:", len(translation.split()))
    print("Preview:", translation[:400])

test_translation = " ".join(translated_chunks)

print("\n" + "=" * 60)
print("FINAL TEST TRANSLATION")
print("Total English words:", len(test_translation.split()))
print(test_translation)


CHUNK 1
Hindi words: 110
English words: 49
Preview: Rakeysh Omprakash Mehra recently signed Abhishek Bachchan for his new film'Delhi 6'and now Sonam has been roped in as the heroine. The film's production company UTV did not argue much with Sonam and her father Anil Kapoor over money but did not bother Sonam at all about her fee.

CHUNK 2
Hindi words: 84
English words: 36
Preview: The film will be directed by Siddharth Anand (who had earlier made Salaam Namaste and Ta Ra Rum Pum). The difference is that Shah Rukh and Rishi Kapoor appeared in the same film Deewana in 1992.

CHUNK 3
Hindi words: 86
English words: 26
Preview: However, Ranbir's debut film Saawariya does not feature Shah Rukh Khan. On the same day as Saavariya, Shahrukh Khan's Om Shanti Om will also be released.

CHUNK 4
Hindi words: 47
English words: 39
Preview: Anuj's film'Dulha Mil Gaya'with Fardeen Khan is currently in the making with a long special appearance by Shah Rukh Khan. Anuj Saxena hosted a lavish party last w

In [66]:
clean_test_text = re.sub(
    r'[*#_~•·]{2,}',
    ' ',
    sample_text
)

clean_test_text = re.sub(
    r'\s+',
    ' ',
    clean_test_text
).strip()

print("Original words:", len(sample_text.split()))
print("Cleaned words:", len(clean_test_text.split()))
print("\nRemaining separator symbols:",
      len(re.findall(r'[*#_~•·]{2,}', clean_test_text)))

print("\nCleaned preview:")
print(clean_test_text[:1500])

Original words: 758
Cleaned words: 753

Remaining separator symbols: 0

Cleaned preview:
राकेश ओमप्रकाश मेहरा ने हाल ही में अपनी नई फ़िल्म 'दिल्ली 6' के लिए अभिषेक बच्चन को साइन किया था और अब बतौर हीरोईन सोनम को लिया गया है. फ़िल्म की निर्माता कंपनी यूटीवी ने सोनम और उनके पापा अनिल कपूर के साथ पैसों के मामले में ज़्यादा बहस नहीं की बल्कि सोनम की फीस को लेकर बिल्कुल खींचातानी नहीं हुई. 'रंग दे बसंती' वाले राकेश मेहरा की फ़िल्म में काम करने के लिए सोनम को अच्छी-ख़ासी रकम मिलेगी. फ़िल्म की कहानी सोनम और अनिल कपूर दोनों ने सुनी है. रणबीर कपूर ने कुछ महीनों पहले यश चोपड़ा की नई फ़िल्म में काम करने के लिए हामी भर दी थी. ये फ़िल्म सिद्धार्थ आनंद (जिन्होंने इससे पहले 'सलाम नमस्ते' और 'ता रा रम पम' बनाई थी) के निर्देशन में बनेगी. अज़ीब इत्तफ़ाक ये इत्तफ़ाक है लेकिन अज़ीब इत्तफ़ाक. शाहरुख़ ख़ान ने अपना करियर 1992 में ऋषि कपूर के साथ शुरू किया था. और अब, पंद्रह साल बाद ऋषि के बेटे रणबीर कपूर के करियर की शुरुआत शाहरुख़ ख़ान के साथ शुरू होने जा रही है. फ़र्क इतना है कि शाहरुख़ और ऋषि कपूर 1992 में 

In [67]:
clean_test_chunks = make_token_chunks(
    clean_test_text,
    max_tokens=200
)

print("Total chunks:", len(clean_test_chunks))

for i, chunk in enumerate(clean_test_chunks, 1):
    batch = ip.preprocess_batch(
        [chunk],
        src_lang="hin_Deva",
        tgt_lang="eng_Latn"
    )

    count = len(
        tokenizer(batch, truncation=False)["input_ids"][0]
    )

    print(
        f"Chunk {i}: {count} tokens | "
        f"{len(chunk.split())} Hindi words"
    )

Total chunks: 5
Chunk 1: 194 tokens | 152 Hindi words
Chunk 2: 197 tokens | 148 Hindi words
Chunk 3: 189 tokens | 154 Hindi words
Chunk 4: 187 tokens | 150 Hindi words
Chunk 5: 173 tokens | 149 Hindi words


In [68]:
clean_translations = []

for i, chunk in enumerate(clean_test_chunks, 1):
    translation = translate_one(chunk)
    clean_translations.append(translation)

    print(f"\nCHUNK {i}")
    print("English words:", len(translation.split()))
    print(translation[:500])

clean_test_translation = " ".join(clean_translations)

print("\n" + "=" * 60)
print("TOTAL ENGLISH WORDS:",
      len(clean_test_translation.split()))

print("\nCOMPLETE TRANSLATION:\n")
print(clean_test_translation)


CHUNK 1
English words: 128
Rakeysh Omprakash Mehra recently signed Abhishek Bachchan for his new film'Delhi 6'and now Sonam has been roped in as the heroine. The film's production company UTV did not argue much with Sonam and her father Anil Kapoor over money but Sonam's fee did not bother at all. Sonam will get a good amount of money to work in Rakesh Mehra's film'Rang De Basanti '. The story of the film has been heard by both Sonam and Anil Kapoor. Ranbir Kapoor had agreed to work with Yash Chopra's new film a few month

CHUNK 2
English words: 56
And now, after fifteen years, Rishi's son Ranbir Kapoor's career is going to start with Shah Rukh Khan. The difference is that Shah Rukh and Rishi Kapoor appeared in the same film'Deewana'in 1992. But, Ranbir's first film'Saawariya'does not have Shahrukh Khan. But on the same day that'Saawaria'will be released, Shahrukh Khan's'Om Shanti Om'will also be released.

CHUNK 3
English words: 143
Last week on 31st October, Anuj Saxena had organize

In [69]:
print("Original cleaned Hindi words:",
      len(clean_test_text.split()))

print("Words across chunks:",
      sum(len(chunk.split()) for chunk in clean_test_chunks))

print("Reconstructed Hindi words:",
      len(" ".join(clean_test_chunks).split()))

print("\nChunk boundary previews:")

for i in range(len(clean_test_chunks) - 1):
    print(f"\nBoundary {i+1}")
    print("END:", clean_test_chunks[i][-250:])
    print("START:", clean_test_chunks[i+1][:250])

Original cleaned Hindi words: 753
Words across chunks: 753
Reconstructed Hindi words: 753

Chunk boundary previews:

Boundary 1
END: े के लिए हामी भर दी थी. ये फ़िल्म सिद्धार्थ आनंद (जिन्होंने इससे पहले 'सलाम नमस्ते' और 'ता रा रम पम' बनाई थी) के निर्देशन में बनेगी. अज़ीब इत्तफ़ाक ये इत्तफ़ाक है लेकिन अज़ीब इत्तफ़ाक. शाहरुख़ ख़ान ने अपना करियर 1992 में ऋषि कपूर के साथ शुरू किया था.
START: और अब, पंद्रह साल बाद ऋषि के बेटे रणबीर कपूर के करियर की शुरुआत शाहरुख़ ख़ान के साथ शुरू होने जा रही है. फ़र्क इतना है कि शाहरुख़ और ऋषि कपूर 1992 में एक ही फ़िल्म 'दीवाना' में नज़र आए थे. मगर, रणबीर की पहली फ़िल्म 'सांवरिया' में शाहरुख़ ख़ान नहीं है

Boundary 2
END: य मंत्री रामविलास पासवान के बेटे चिराग को टेलीविज़न सीरियल और अब फ़िल्म के निर्माता अनुज सक्सेना अपनी अगली फ़िल्म में इंट्रोड्यूस करेंगे. फ़रदीन ख़ान के साथ अनुज की फ़िल्म 'दूल्हा मिल गया' अभी बन रही है जिसमें शाहरुख़ ख़ान का लंबा स्पेशल अपियरेंस है.
START: चिराग को बॉलीवूड की चमक-दमक से वाक़िफ़ कराने के लिए पिछले हफ़्ते 31 अक्टूबर को अनुज

In [70]:
print("Generation configuration:")
print(model.generation_config)

print("\nSpecial token IDs:")
print("EOS:", tokenizer.eos_token_id)
print("PAD:", tokenizer.pad_token_id)
print("BOS:", tokenizer.bos_token_id)

print("\nModel target limit:",
      model.config.max_target_positions)

Generation configuration:
GenerationConfig {
  "bos_token_id": 0,
  "decoder_start_token_id": 2,
  "eos_token_id": 2,
  "pad_token_id": 1
}


Special token IDs:
EOS: 2
PAD: 1
BOS: 0

Model target limit: 256


In [72]:
print("Chunk-wise translation lengths:\n")

for i, (hindi, english) in enumerate(
    zip(clean_test_chunks, clean_translations), start=1
):
    source_batch = ip.preprocess_batch(
        [hindi],
        src_lang="hin_Deva",
        tgt_lang="eng_Latn"
    )

    source_tokens = len(
        tokenizer(
            source_batch,
            truncation=False
        )["input_ids"][0]
    )

    # Count English words without using the IndicTrans2 tokenizer
    english_words = len(english.split())

    print(
        f"Chunk {i}: "
        f"Source tokens = {source_tokens}, "
        f"English words = {english_words}"
    )

Chunk-wise translation lengths:

Chunk 1: Source tokens = 194, English words = 128
Chunk 2: Source tokens = 197, English words = 56
Chunk 3: Source tokens = 189, English words = 143
Chunk 4: Source tokens = 187, English words = 153
Chunk 5: Source tokens = 173, English words = 130


In [73]:
print("========== HINDI SOURCE: CHUNK 2 ==========\n")
print(clean_test_chunks[1])

print("\n========== ENGLISH TRANSLATION: CHUNK 2 ==========\n")
print(clean_translations[1])

========== HINDI SOURCE: CHUNK 2 ==========

और अब, पंद्रह साल बाद ऋषि के बेटे रणबीर कपूर के करियर की शुरुआत शाहरुख़ ख़ान के साथ शुरू होने जा रही है. फ़र्क इतना है कि शाहरुख़ और ऋषि कपूर 1992 में एक ही फ़िल्म 'दीवाना' में नज़र आए थे. मगर, रणबीर की पहली फ़िल्म 'सांवरिया' में शाहरुख़ ख़ान नहीं हैं. बल्कि सांवरिया जिस दिन रिलीज़ होगी, उसी दिन शाहरुख़ ख़ान की 'ओम शांति ओम' भी रिलीज़ होगी. पर्दे पर पासवान पुत्र पहले महाराष्ट्र के मुख्यमंत्री विलासराव देशमुख के बेटे रितेश देशमुख ने बॉलीवूड में क़दम रखे. अब एक और मंत्री का बेटा अभिनय की दुनिया में आने की तैयारी में है. केंद्रीय मंत्री रामविलास पासवान के बेटे चिराग को टेलीविज़न सीरियल और अब फ़िल्म के निर्माता अनुज सक्सेना अपनी अगली फ़िल्म में इंट्रोड्यूस करेंगे. फ़रदीन ख़ान के साथ अनुज की फ़िल्म 'दूल्हा मिल गया' अभी बन रही है जिसमें शाहरुख़ ख़ान का लंबा स्पेशल अपियरेंस है.

========== ENGLISH TRANSLATION: CHUNK 2 ==========

And now, after fifteen years, Rishi's son Ranbir Kapoor's career is going to start with Shah Rukh Khan. The difference i

In [74]:
small_test_chunks = make_token_chunks(
    clean_test_text,
    max_tokens=100
)

print("Number of smaller chunks:", len(small_test_chunks))

for i, chunk in enumerate(small_test_chunks, start=1):
    batch = ip.preprocess_batch(
        [chunk],
        src_lang="hin_Deva",
        tgt_lang="eng_Latn"
    )

    count = len(
        tokenizer(batch, truncation=False)["input_ids"][0]
    )

    print(f"Chunk {i}: {count} tokens | {len(chunk.split())} words")

Number of smaller chunks: 11
Chunk 1: 99 tokens | 79 words
Chunk 2: 98 tokens | 73 words
Chunk 3: 98 tokens | 69 words
Chunk 4: 71 tokens | 57 words
Chunk 5: 66 tokens | 47 words
Chunk 6: 96 tokens | 76 words
Chunk 7: 91 tokens | 76 words
Chunk 8: 97 tokens | 74 words
Chunk 9: 94 tokens | 77 words
Chunk 10: 87 tokens | 73 words
Chunk 11: 61 tokens | 52 words


In [75]:
small_test_translations = []

for i, chunk in enumerate(small_test_chunks, start=1):
    translated = translate_one(chunk)
    small_test_translations.append(translated)

    print(f"Completed chunk {i}/11")

print("\nAll 11 smaller chunks translated.")

print(
    "\nTotal English words:",
    sum(len(t.split()) for t in small_test_translations)
)

Completed chunk 1/11
Completed chunk 2/11
Completed chunk 3/11
Completed chunk 4/11
Completed chunk 5/11
Completed chunk 6/11
Completed chunk 7/11
Completed chunk 8/11
Completed chunk 9/11
Completed chunk 10/11
Completed chunk 11/11

All 11 smaller chunks translated.

Total English words: 658


In [76]:
for i in range(11):
    print(f"\n{'='*20} CHUNK {i+1} {'='*20}")
    print("HINDI:")
    print(small_test_chunks[i])
    print("\nENGLISH:")
    print(small_test_translations[i])


==================== CHUNK 1 ====================
HINDI:
राकेश ओमप्रकाश मेहरा ने हाल ही में अपनी नई फ़िल्म 'दिल्ली 6' के लिए अभिषेक बच्चन को साइन किया था और अब बतौर हीरोईन सोनम को लिया गया है. फ़िल्म की निर्माता कंपनी यूटीवी ने सोनम और उनके पापा अनिल कपूर के साथ पैसों के मामले में ज़्यादा बहस नहीं की बल्कि सोनम की फीस को लेकर बिल्कुल खींचातानी नहीं हुई. 'रंग दे बसंती' वाले राकेश मेहरा की फ़िल्म में काम करने के लिए सोनम को अच्छी-ख़ासी रकम मिलेगी.

ENGLISH:
Rakeysh Omprakash Mehra recently signed Abhishek Bachchan for his new film'Delhi 6'and now Sonam has been roped in as the heroine. The film's production company UTV did not argue much with Sonam and her father Anil Kapoor over money but Sonam did not get into a fight over her fee.

==================== CHUNK 2 ====================
HINDI:
फ़िल्म की कहानी सोनम और अनिल कपूर दोनों ने सुनी है. रणबीर कपूर ने कुछ महीनों पहले यश चोपड़ा की नई फ़िल्म में काम करने के लिए हामी भर दी थी. ये फ़िल्म सिद्धार्थ आनंद (जिन्होंने इससे पहले 'सलाम नमस्ते'

In [77]:
for i, (hindi, english) in enumerate(
    zip(small_test_chunks, small_test_translations), start=1
):
    h_words = len(hindi.split())
    e_words = len(english.split())
    ratio = e_words / h_words if h_words else 0

    print(
        f"Chunk {i}: Hindi={h_words}, "
        f"English={e_words}, Ratio={ratio:.2f}"
    )

Chunk 1: Hindi=79, English=50, Ratio=0.63
Chunk 2: Hindi=73, English=68, Ratio=0.93
Chunk 3: Hindi=69, English=58, Ratio=0.84
Chunk 4: Hindi=57, English=53, Ratio=0.93
Chunk 5: Hindi=47, English=39, Ratio=0.83
Chunk 6: Hindi=76, English=67, Ratio=0.88
Chunk 7: Hindi=76, English=73, Ratio=0.96
Chunk 8: Hindi=74, English=72, Ratio=0.97
Chunk 9: Hindi=77, English=71, Ratio=0.92
Chunk 10: Hindi=73, English=59, Ratio=0.81
Chunk 11: Hindi=52, English=48, Ratio=0.92


In [78]:
long_articles = df_clean[
    df_clean["text"].fillna("").str.split().str.len() > 700
].copy()

print("Articles with more than 700 words:", len(long_articles))

display(
    long_articles[["id", "title", "text"]]
    .head(3)
)

Articles with more than 700 words: 44


,id,title,text
7,international-51410040,"इमरान ख़ान बोले, नरेंद्र मोदी की यह आख़िरी 'ग़...",इमरान ख़ान ने पीएम मोदी के अलावा चीफ़ ऑफ आर्मी...
8,india-55525807,चीन ने युद्ध की तैयारियों के लिए रक्षा नियमों ...,इसके बाद सेंट्रल मिलिटरी कमिशन (सीएमसी) के पास...
20,india-48662333,गुजरात विकास मॉडल में पानी के लिए कुआं खोदता क...,"सेमाभाई की बेटी हरमीबेन ये कहना है सेमाभाई का,..."


In [79]:
test_article_2 = long_articles.iloc[0]["text"]

test_article_2 = re.sub(
    r'[*#_~•·]{2,}',
    ' ',
    str(test_article_2)
)

test_article_2 = re.sub(
    r'\s+',
    ' ',
    test_article_2
).strip()

test_chunks_2 = make_token_chunks(
    test_article_2,
    max_tokens=100
)

print("Article ID:", long_articles.iloc[0]["id"])
print("Original words:", len(test_article_2.split()))
print("Number of chunks:", len(test_chunks_2))
print("Total chunk words:", sum(len(c.split()) for c in test_chunks_2))

Article ID: international-51410040
Original words: 1225
Number of chunks: 19
Total chunk words: 1225


In [80]:
test_translations_2 = []

for i, chunk in enumerate(test_chunks_2, start=1):
    translated = translate_one(chunk)
    test_translations_2.append(translated)

    print(f"Completed chunk {i}/19")

print("\nTotal English words:",
      sum(len(t.split()) for t in test_translations_2))

Completed chunk 1/19
Completed chunk 2/19
Completed chunk 3/19
Completed chunk 4/19
Completed chunk 5/19
Completed chunk 6/19
Completed chunk 7/19
Completed chunk 8/19
Completed chunk 9/19
Completed chunk 10/19
Completed chunk 11/19
Completed chunk 12/19
Completed chunk 13/19
Completed chunk 14/19
Completed chunk 15/19
Completed chunk 16/19
Completed chunk 17/19
Completed chunk 18/19
Completed chunk 19/19

Total English words: 1024


In [81]:
for i, (hindi, english) in enumerate(
    zip(test_chunks_2, test_translations_2), start=1
):
    h = len(hindi.split())
    e = len(english.split())

    print(
        f"Chunk {i}: Hindi={h}, English={e}, "
        f"Ratio={e/h:.2f}"
    )

Chunk 1: Hindi=71, English=58, Ratio=0.82
Chunk 2: Hindi=78, English=62, Ratio=0.79
Chunk 3: Hindi=74, English=72, Ratio=0.97
Chunk 4: Hindi=41, English=37, Ratio=0.90
Chunk 5: Hindi=75, English=73, Ratio=0.97
Chunk 6: Hindi=60, English=48, Ratio=0.80
Chunk 7: Hindi=81, English=72, Ratio=0.89
Chunk 8: Hindi=17, English=19, Ratio=1.12
Chunk 9: Hindi=81, English=66, Ratio=0.81
Chunk 10: Hindi=73, English=35, Ratio=0.48
Chunk 11: Hindi=81, English=70, Ratio=0.86
Chunk 12: Hindi=65, English=56, Ratio=0.86
Chunk 13: Hindi=48, English=44, Ratio=0.92
Chunk 14: Hindi=81, English=70, Ratio=0.86
Chunk 15: Hindi=67, English=61, Ratio=0.91
Chunk 16: Hindi=67, English=60, Ratio=0.90
Chunk 17: Hindi=64, English=42, Ratio=0.66
Chunk 18: Hindi=32, English=19, Ratio=0.59
Chunk 19: Hindi=69, English=60, Ratio=0.87


In [82]:
for i in [9, 16, 17]:
    print(f"\n{'='*25} CHUNK {i+1} {'='*25}")

    print("\nHINDI SOURCE:\n")
    print(test_chunks_2[i])

    print("\nENGLISH TRANSLATION:\n")
    print(test_translations_2[i])


========================= CHUNK 10 =========================

HINDI SOURCE:

पीएम मोदी ने कहा था कि पाकिस्तान जम्मू-कश्मीर में प्रॉक्सी वॉर करता रहा लेकिन भारत में पहले की सरकारों ने मुंहतोड़ जवाब नहीं दिया लेकिन अब पाकिस्तान को क़रारा जवाब मिल रहा है. पीएम मोदी के इसी बयान को लेकर इमरान ख़ान ने गुरुवार को मीरपुर में आयोजित एक रैली में अपनी प्रतिक्रिया दी है. इमरान ख़ान ने कहा कि भारत पाँच अगस्त को दुःस्साहस कर बैठा है और इसका नतीजा उसे मिलेगा.

ENGLISH TRANSLATION:

PM Modi had said that Pakistan used to wage proxy wars in Jammu and Kashmir but the previous governments in India did not give a befitting reply, but now Pakistan is getting a strong response.

========================= CHUNK 17 =========================

HINDI SOURCE:

उन्होंने कहा, ''पाँच अगस्त के बाद मैंने जितने नेताओं से मुलाक़ात की उन सबसे कश्मीर पर बात की. मैंने ट्रंप से इस मुद्दे पर तीन बार बात की. रूस, फ़्रांस और जर्मनी को समझाया कि भारत कश्मीर में क्या कर रहा है. कनाडा के प्रधानमंत्री जस्टिन ट्रूडो को समझाया कि व

In [83]:
test_chunks_60 = make_token_chunks(
    test_article_2,
    max_tokens=60
)

print("Number of chunks:", len(test_chunks_60))
print("Original words:", len(test_article_2.split()))
print("Words across chunks:",
      sum(len(c.split()) for c in test_chunks_60))

print("\nLargest chunk token count:")

for i, chunk in enumerate(test_chunks_60, start=1):
    batch = ip.preprocess_batch(
        [chunk],
        src_lang="hin_Deva",
        tgt_lang="eng_Latn"
    )

    count = len(
        tokenizer(batch, truncation=False)["input_ids"][0]
    )

    print(f"Chunk {i}: {count}")

Number of chunks: 31
Original words: 1225
Words across chunks: 1225

Largest chunk token count:
Chunk 1: 23
Chunk 2: 56
Chunk 3: 34
Chunk 4: 56
Chunk 5: 58
Chunk 6: 54
Chunk 7: 59
Chunk 8: 97
Chunk 9: 53
Chunk 10: 28
Chunk 11: 60
Chunk 12: 59
Chunk 13: 98
Chunk 14: 43
Chunk 15: 49
Chunk 16: 53
Chunk 17: 49
Chunk 18: 36
Chunk 19: 50
Chunk 20: 57
Chunk 21: 65
Chunk 22: 50
Chunk 23: 16
Chunk 24: 55
Chunk 25: 43
Chunk 26: 40
Chunk 27: 55
Chunk 28: 53
Chunk 29: 16
Chunk 30: 75
Chunk 31: 22


In [84]:
for i, chunk in enumerate(test_chunks_60, start=1):
    batch = ip.preprocess_batch(
        [chunk],
        src_lang="hin_Deva",
        tgt_lang="eng_Latn"
    )

    count = len(
        tokenizer(batch, truncation=False)["input_ids"][0]
    )

    if count > 60:
        print(f"\nCHUNK {i} — {count} tokens")
        print(chunk)


CHUNK 8 — 97 tokens
एक तरफ़ हमारे अल्लाह का क़ुरान में ये हुक्म है और दूसरी तरफ़ अल्लाह हमें हक्म करता है कि अपनी आज़ादी के लिेए अगर जान क़ुर्बान करते हो तो पैग़ंबरों के बाद अल्लाह दूसरा सबसे बड़ा दर्जा देगा.'' इमरान ख़ान ने कहा, ''कहने का मतलब ये नरेंद्र मोदी कि इस कौम के ऊपर जो आपने और आपके कमांडर इन चीफ़ ने कहा और उस पर आगे बढ़ने की ग़लती की तो ये आख़िरी ग़लती होगी.

CHUNK 13 — 98 tokens
इन दो स्ट्राइक से न केवल जम्मू-कश्मीर में शांति आई है बल्कि देश के बाक़ी हिस्सों में भी शांति आई है.'' इससे पहले भारत के सेना प्रमुख जनरल मनोज मुकुंद नारवाने ने कहा था कि अगर संसद ने आदेश दिया तो भारत की सेना पाकिस्तान के नियंत्रण वाले कश्मीर कार्रवाई करने के लिए तैयार है. 'एक हफ़्ते में पाकिस्तान को हरा देगा भारत' मोदी ने ये बात नई दिल्ली में आयोजित एनसीसी (नेशनल कैडेट कोर) की एक रैली में कही थी.

CHUNK 21 — 65 tokens
लेकिन मोदी ने चुनाव जीतने के बाद घमंड में उसने पाँच अगस्त दो क़दम उठाया उससे कश्मीर की आज़ादी का रास्ता और आसान हो गया है.'' इमरान ख़ान ने कहा, ''बालाकोट में घमंड में भारत ने हमारे प

In [85]:
def make_safe_token_chunks(text, max_tokens=60):

    parts = re.split(
        r'(?<=[।!?])\s+|(?<=\.)\s+(?=[अ-ह])',
        str(text).strip()
    )

    chunks = []
    current = []

    def token_count(value):
        batch = ip.preprocess_batch(
            [value],
            src_lang="hin_Deva",
            tgt_lang="eng_Latn"
        )
        return len(tokenizer(
            batch, truncation=False
        )["input_ids"][0])

    for sentence in parts:
        sentence = sentence.strip()
        if not sentence:
            continue

        if token_count(sentence) <= max_tokens:
            candidate = " ".join(current + [sentence])

            if current and token_count(candidate) > max_tokens:
                chunks.append(" ".join(current))
                current = [sentence]
            else:
                current.append(sentence)

        else:
            if current:
                chunks.append(" ".join(current))
                current = []

            words = sentence.split()
            piece = []

            for word in words:
                candidate = " ".join(piece + [word])

                if piece and token_count(candidate) > max_tokens:
                    chunks.append(" ".join(piece))
                    piece = [word]
                else:
                    piece.append(word)

            if piece:
                current = piece

    if current:
        chunks.append(" ".join(current))

    return chunks

In [86]:
test_chunks_safe = make_safe_token_chunks(
    test_article_2,
    max_tokens=60
)

print("Number of chunks:", len(test_chunks_safe))
print("Original words:", len(test_article_2.split()))
print("Words across chunks:",
      sum(len(c.split()) for c in test_chunks_safe))

print("\nChecking token limits:")

oversized = []

for i, chunk in enumerate(test_chunks_safe, start=1):
    batch = ip.preprocess_batch(
        [chunk],
        src_lang="hin_Deva",
        tgt_lang="eng_Latn"
    )

    count = len(
        tokenizer(batch, truncation=False)["input_ids"][0]
    )

    if count > 60:
        oversized.append((i, count))

    print(f"Chunk {i}: {count} tokens")

print("\nOversized chunks:", oversized)

Number of chunks: 32
Original words: 1225
Words across chunks: 1225

Checking token limits:
Chunk 1: 23 tokens
Chunk 2: 56 tokens
Chunk 3: 34 tokens
Chunk 4: 56 tokens
Chunk 5: 58 tokens
Chunk 6: 54 tokens
Chunk 7: 59 tokens
Chunk 8: 60 tokens
Chunk 9: 58 tokens
Chunk 10: 60 tokens
Chunk 11: 60 tokens
Chunk 12: 59 tokens
Chunk 13: 60 tokens
Chunk 14: 41 tokens
Chunk 15: 43 tokens
Chunk 16: 49 tokens
Chunk 17: 53 tokens
Chunk 18: 49 tokens
Chunk 19: 36 tokens
Chunk 20: 50 tokens
Chunk 21: 57 tokens
Chunk 22: 59 tokens
Chunk 23: 56 tokens
Chunk 24: 16 tokens
Chunk 25: 55 tokens
Chunk 26: 43 tokens
Chunk 27: 40 tokens
Chunk 28: 55 tokens
Chunk 29: 53 tokens
Chunk 30: 16 tokens
Chunk 31: 60 tokens
Chunk 32: 37 tokens

Oversized chunks: []


In [87]:
test_translations_safe = []

for i, chunk in enumerate(test_chunks_safe, start=1):
    translated = translate_one(chunk)
    test_translations_safe.append(translated)

    print(
        f"Completed chunk {i}/{len(test_chunks_safe)}"
    )

print("\nTranslation completed!")

print(
    "Total English words:",
    sum(len(t.split()) for t in test_translations_safe)
)

Completed chunk 1/32
Completed chunk 2/32
Completed chunk 3/32
Completed chunk 4/32
Completed chunk 5/32
Completed chunk 6/32
Completed chunk 7/32
Completed chunk 8/32
Completed chunk 9/32
Completed chunk 10/32
Completed chunk 11/32
Completed chunk 12/32
Completed chunk 13/32
Completed chunk 14/32
Completed chunk 15/32
Completed chunk 16/32
Completed chunk 17/32
Completed chunk 18/32
Completed chunk 19/32
Completed chunk 20/32
Completed chunk 21/32
Completed chunk 22/32
Completed chunk 23/32
Completed chunk 24/32
Completed chunk 25/32
Completed chunk 26/32
Completed chunk 27/32
Completed chunk 28/32
Completed chunk 29/32
Completed chunk 30/32
Completed chunk 31/32
Completed chunk 32/32

Translation completed!
Total English words: 1035


In [88]:
for i, (hindi, english) in enumerate(
    zip(test_chunks_safe, test_translations_safe),
    start=1
):
    print(f"\n{'='*60}")
    print(f"CHUNK {i}")
    print("\nHINDI:")
    print(hindi)
    print("\nENGLISH:")
    print(english)


CHUNK 1

HINDI:
इमरान ख़ान ने पीएम मोदी के अलावा चीफ़ ऑफ आर्मी स्टाफ जनरल बिपिन रावत को भी निशाने पर लिया.

ENGLISH:
Apart from PM Modi, Imran Khan also targeted Chief of Army Staff General Bipin Rawat.

CHUNK 2

HINDI:
गुरुवार को पाकिस्तान प्रशासित कश्मीर के मीरपुर में 'कश्मीर एकता दिवस' के अगले दिन बोलते हुए इमरान ख़ान ने कहा, ''नरेंद्र मोदी आपने कहा है कि भारत 11 दिनों में पाकिस्तान को फ़तह कर लेगा. लगता है कि आपने दुनिया का इतिहास पढ़ा नहीं है.

ENGLISH:
Speaking on the day after'Kashmir Unity Day'in Mirpur, Pakistan-administered Kashmir on Thursday, Khan said, "Narendra Modi, you have said that India will defeat Pakistan in 11 days. You seem to have not read the history of the world.

CHUNK 3

HINDI:
लगता है कि आपकी डिग्री जाली थी. आपने जो यह घमंडी बयान दिया है उसी को देखते हुए मैं ज़रा आपको इतिहास में ले जाता हूं.

ENGLISH:
Your degree seems to have been forged. Given this arrogant statement you have made, let me take you back in history.

CHUNK 4

HINDI:
किसी से इतिहास के बारे 

In [89]:

import re
import pandas as pd
import torch

# Model limits
SOURCE_LIMIT = 256
TARGET_LIMIT = 240
CHUNK_LIMIT = 120  # Safety margin below the 256-token source limit


def prepare_hindi(text):
    """Clean formatting noise without removing normal Hindi words."""
    text = str(text)
    text = re.sub(r'[*#_~•·]{2,}', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def source_token_count(text):
    """Count actual source tokens after IndicTrans preprocessing."""
    prepared = ip.preprocess_batch(
        [text],
        src_lang="hin_Deva",
        tgt_lang="eng_Latn"
    )

    encoded = tokenizer(
        prepared,
        truncation=False,
        padding=False
    )

    return len(encoded["input_ids"][0])


def make_safe_token_chunks(text, max_tokens=CHUNK_LIMIT):
    """
    Keep sentences together where possible.
    Split long sentences into word groups when necessary.
    Never silently truncate source text.
    """
    text = prepare_hindi(text)

    parts = re.split(
        r'(?<=[।!?])\s+|(?<=\.)\s+(?=[अ-ह])',
        text
    )

    chunks = []
    current = []

    for sentence in parts:
        sentence = sentence.strip()

        if not sentence:
            continue

        candidate = " ".join(current + [sentence])

        if source_token_count(sentence) <= max_tokens:
            if current and source_token_count(candidate) > max_tokens:
                chunks.append(" ".join(current))
                current = [sentence]
            else:
                current.append(sentence)

        else:
            if current:
                chunks.append(" ".join(current))
                current = []

            words = sentence.split()
            piece = []

            for word in words:
                candidate = " ".join(piece + [word])

                if piece and source_token_count(candidate) > max_tokens:
                    chunks.append(" ".join(piece))
                    piece = [word]
                else:
                    piece.append(word)

            if piece:
                current = piece

    if current:
        chunks.append(" ".join(current))

    # Final validation: reject any oversized chunk
    for i, chunk in enumerate(chunks, start=1):
        count = source_token_count(chunk)

        if count > max_tokens:
            raise ValueError(
                f"Chunk {i} has {count} tokens; limit is {max_tokens}"
            )

    return chunks


def translate_one(text):
    """Translate one Hindi chunk without silently truncating it."""
    if pd.isna(text) or not str(text).strip():
        return ""

    prepared = ip.preprocess_batch(
        [str(text)],
        src_lang="hin_Deva",
        tgt_lang="eng_Latn"
    )

    inputs = tokenizer(
        prepared,
        truncation=False,
        padding=True,
        return_tensors="pt"
    ).to(device)

    input_length = inputs["input_ids"].shape[1]

    if input_length > SOURCE_LIMIT:
        raise ValueError(
            f"Input has {input_length} tokens. "
            f"Maximum supported source length is {SOURCE_LIMIT}."
        )

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=TARGET_LIMIT,
            num_beams=4,
            early_stopping=True,
            use_cache=False
        )

    decoded = tokenizer.batch_decode(
        output,
        skip_special_tokens=True
    )

    result = ip.postprocess_batch(
        decoded,
        lang="eng_Latn"
    )[0]

    return result.strip()


print("Translation pipeline is ready.")
print("Source limit:", SOURCE_LIMIT)
print("Chunk limit:", CHUNK_LIMIT)
print("Target generation limit:", TARGET_LIMIT)


Translation pipeline is ready.
Source limit: 256
Chunk limit: 120
Target generation limit: 240


In [90]:
test_chunks_final = make_safe_token_chunks(
    test_article_2,
    max_tokens=CHUNK_LIMIT
)

print("Number of chunks:", len(test_chunks_final))
print("Original words:", len(prepare_hindi(test_article_2).split()))
print("Chunk words:", sum(len(c.split()) for c in test_chunks_final))

test_translations_final = []

for i, chunk in enumerate(test_chunks_final, start=1):
    translated = translate_one(chunk)
    test_translations_final.append(translated)
    print(f"Translated {i}/{len(test_chunks_final)}")

print("\nTotal English words:",
      sum(len(t.split()) for t in test_translations_final))

Number of chunks: 15
Original words: 1225
Chunk words: 1225
Translated 1/15
Translated 2/15
Translated 3/15
Translated 4/15
Translated 5/15
Translated 6/15
Translated 7/15
Translated 8/15
Translated 9/15
Translated 10/15
Translated 11/15
Translated 12/15
Translated 13/15
Translated 14/15
Translated 15/15

Total English words: 1017


In [91]:
for i, (hindi, english) in enumerate(
    zip(test_chunks_final, test_translations_final),
    start=1
):
    print(f"\n{'='*60}")
    print(f"CHUNK {i}")
    print("\nHINDI:")
    print(hindi)
    print("\nENGLISH:")
    print(english)


CHUNK 1

HINDI:
इमरान ख़ान ने पीएम मोदी के अलावा चीफ़ ऑफ आर्मी स्टाफ जनरल बिपिन रावत को भी निशाने पर लिया. गुरुवार को पाकिस्तान प्रशासित कश्मीर के मीरपुर में 'कश्मीर एकता दिवस' के अगले दिन बोलते हुए इमरान ख़ान ने कहा, ''नरेंद्र मोदी आपने कहा है कि भारत 11 दिनों में पाकिस्तान को फ़तह कर लेगा. लगता है कि आपने दुनिया का इतिहास पढ़ा नहीं है. लगता है कि आपकी डिग्री जाली थी. आपने जो यह घमंडी बयान दिया है उसी को देखते हुए मैं ज़रा आपको इतिहास में ले जाता हूं.

ENGLISH:
Imran Khan also targeted Chief of Army Staff General Bipin Rawat in addition to PM Modi. Speaking on the day after'Kashmir Ekta Diwas'in Mirpur, Pakistan-administered Kashmir on Thursday, Imran Khan said, "Narendra Modi, you have said that India will defeat Pakistan in 11 days. It seems that you have not read the history of the world. It seems that your degree was fake.

CHUNK 2

HINDI:
किसी से इतिहास के बारे में आप जानकारी ले लो.'' पाकिस्तानी पीएम ने कहा, ''दुनिया की दो सबसे ताक़तवर फ़ौजें नेपोलियन की फ़ौज और हिटलर की फ़ौज ने

In [92]:
print("Model configuration:")
print("Source limit:", model.config.max_source_positions)
print("Target limit:", model.config.max_target_positions)

print("\nGeneration configuration:")
print(model.generation_config)

Model configuration:
Source limit: 256
Target limit: 256

Generation configuration:
GenerationConfig {
  "bos_token_id": 0,
  "decoder_start_token_id": 2,
  "eos_token_id": 2,
  "pad_token_id": 1
}



In [93]:
test_chunk = test_chunks_final[7]

print("HINDI:")
print(test_chunk)

print("\nCURRENT TRANSLATION:")
print(translate_one(test_chunk))

prepared = ip.preprocess_batch(
    [test_chunk],
    src_lang="hin_Deva",
    tgt_lang="eng_Latn"
)

inputs = tokenizer(
    prepared,
    truncation=False,
    padding=True,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=240,
        num_beams=5,
        length_penalty=1.0,
        early_stopping=True,
        use_cache=False
    )

decoded = tokenizer.batch_decode(
    output,
    skip_special_tokens=True
)

print("\nBEAM SEARCH TRANSLATION:")
print(
    ip.postprocess_batch(
        decoded,
        lang="eng_Latn"
    )[0]
)

HINDI:
पीएम मोदी ने कहा था कि पाकिस्तान जम्मू-कश्मीर में प्रॉक्सी वॉर करता रहा लेकिन भारत में पहले की सरकारों ने मुंहतोड़ जवाब नहीं दिया लेकिन अब पाकिस्तान को क़रारा जवाब मिल रहा है. पीएम मोदी के इसी बयान को लेकर इमरान ख़ान ने गुरुवार को मीरपुर में आयोजित एक रैली में अपनी प्रतिक्रिया दी है. इमरान ख़ान ने कहा कि भारत पाँच अगस्त को दुःस्साहस कर बैठा है और इसका नतीजा उसे मिलेगा. भारत ने पिछले साल पाँच अगस्त को जम्मू-कश्मीर का विशेष दर्जा ख़त्म कर दिया था.

CURRENT TRANSLATION:
PM Modi had said that Pakistan used to wage proxy wars in Jammu and Kashmir but the previous governments in India did not give a befitting reply but now Pakistan is getting a befitting reply.

BEAM SEARCH TRANSLATION:
PM Modi had said that Pakistan used to wage proxy wars in Jammu and Kashmir but the previous governments in India did not give a befitting reply but now Pakistan is getting a befitting reply.


In [94]:
test_missing_part = (
    "इमरान ख़ान ने गुरुवार को मीरपुर में आयोजित एक रैली में "
    "अपनी प्रतिक्रिया दी है। इमरान ख़ान ने कहा कि भारत पाँच "
    "अगस्त को दुःस्साहस कर बैठा है और इसका नतीजा उसे मिलेगा। "
    "भारत ने पिछले साल पाँच अगस्त को जम्मू-कश्मीर का विशेष "
    "दर्जा ख़त्म कर दिया था।"
)

print(translate_one(test_missing_part))

Imran Khan has given his reaction at a rally in Mirpur on Thursday. Imran Khan said that India has done misadventure on 5th August and it will get the result. India scrapped the special status of Jammu and Kashmir on 5th August last year.


In [95]:
test_sentences = [
    s.strip()
    for s in re.split(r'(?<=[।!?])\s*', test_chunk)
    if s.strip()
]

sentence_translations = []

for i, sentence in enumerate(test_sentences, start=1):
    translated = translate_one(sentence)
    sentence_translations.append(translated)

    print(f"\nSentence {i}:")
    print("Hindi:", sentence)
    print("English:", translated)

print("\nRECONSTRUCTED TRANSLATION:")
print(" ".join(sentence_translations))


Sentence 1:
Hindi: पीएम मोदी ने कहा था कि पाकिस्तान जम्मू-कश्मीर में प्रॉक्सी वॉर करता रहा लेकिन भारत में पहले की सरकारों ने मुंहतोड़ जवाब नहीं दिया लेकिन अब पाकिस्तान को क़रारा जवाब मिल रहा है. पीएम मोदी के इसी बयान को लेकर इमरान ख़ान ने गुरुवार को मीरपुर में आयोजित एक रैली में अपनी प्रतिक्रिया दी है. इमरान ख़ान ने कहा कि भारत पाँच अगस्त को दुःस्साहस कर बैठा है और इसका नतीजा उसे मिलेगा. भारत ने पिछले साल पाँच अगस्त को जम्मू-कश्मीर का विशेष दर्जा ख़त्म कर दिया था.
English: PM Modi had said that Pakistan used to wage proxy wars in Jammu and Kashmir but the previous governments in India did not give a befitting reply but now Pakistan is getting a befitting reply.

RECONSTRUCTED TRANSLATION:
PM Modi had said that Pakistan used to wage proxy wars in Jammu and Kashmir but the previous governments in India did not give a befitting reply but now Pakistan is getting a befitting reply.


In [96]:
test_sentences = [
    s.strip()
    for s in re.split(r'(?<=[।.!?])\s+', test_chunk)
    if s.strip()
]

print("Number of sentences:", len(test_sentences))

for i, sentence in enumerate(test_sentences, start=1):
    print(f"\nSentence {i}:")
    print(sentence)

Number of sentences: 4

Sentence 1:
पीएम मोदी ने कहा था कि पाकिस्तान जम्मू-कश्मीर में प्रॉक्सी वॉर करता रहा लेकिन भारत में पहले की सरकारों ने मुंहतोड़ जवाब नहीं दिया लेकिन अब पाकिस्तान को क़रारा जवाब मिल रहा है.

Sentence 2:
पीएम मोदी के इसी बयान को लेकर इमरान ख़ान ने गुरुवार को मीरपुर में आयोजित एक रैली में अपनी प्रतिक्रिया दी है.

Sentence 3:
इमरान ख़ान ने कहा कि भारत पाँच अगस्त को दुःस्साहस कर बैठा है और इसका नतीजा उसे मिलेगा.

Sentence 4:
भारत ने पिछले साल पाँच अगस्त को जम्मू-कश्मीर का विशेष दर्जा ख़त्म कर दिया था.


In [97]:
sentence_translations = []

for i, sentence in enumerate(test_sentences, start=1):
    translated = translate_one(sentence)
    sentence_translations.append(translated)

    print(f"\nSentence {i}:")
    print("English:", translated)

print("\nRECONSTRUCTED TRANSLATION:")
print(" ".join(sentence_translations))


Sentence 1:
English: PM Modi had said that Pakistan used to wage proxy wars in Jammu and Kashmir but the previous governments in India did not give a befitting reply but now Pakistan is getting a befitting reply.

Sentence 2:
English: Imran Khan has given his reaction to the same statement of PM Modi at a rally held in Mirpur on Thursday.

Sentence 3:
English: Imran Khan said that India is sitting on a misadventure on August 5 and it will get the result.

Sentence 4:
English: India revoked the special status of Jammu and Kashmir on August 5 last year.

RECONSTRUCTED TRANSLATION:
PM Modi had said that Pakistan used to wage proxy wars in Jammu and Kashmir but the previous governments in India did not give a befitting reply but now Pakistan is getting a befitting reply. Imran Khan has given his reaction to the same statement of PM Modi at a rally held in Mirpur on Thursday. Imran Khan said that India is sitting on a misadventure on August 5 and it will get the result. India revoked the s

In [98]:
test_article_3 = df_clean.iloc[2]["text"]

sentences_3 = [
    s.strip()
    for s in re.split(r'(?<=[।.!?])\s+', prepare_hindi(test_article_3))
    if s.strip()
]

print("Article ID:", df_clean.iloc[2]["id"])
print("Hindi words:", len(test_article_3.split()))
print("Number of sentences:", len(sentences_3))
print("\nFirst 5 sentences:")

for i, sentence in enumerate(sentences_3[:5], start=1):
    print(f"{i}. {sentence}")

Article ID: 130912_guinnness_record_bulgarian_swim_sr
Hindi words: 200
Number of sentences: 9

First 5 sentences:
1. ख़बरों के मुताबिक़ जेन पेत्कोव को मैसेडोनिया की ऑरिड झील को पार करने में करीब पौने तीन घंटे लगे.
2. आयोजक सासो तोकोव ने समाचार एजेंसी एएफपी को बताया कि 59 वर्षीय पेत्कोव के हाथ और पैर उनके शरीर से बांध दिये गए थे और उन्होंने डॉल्फिन की तरह पीठ के बल तैराकी की.
3. मैसेडोनिया के मीडिया के मुताबिक़ तैराकी की उनकी औसत रफ़्तार 0.7 किमी प्रति घंटे थी.
4. पेत्कोव ने अपना अभियान पूरा करने के बाद कहा, ''पहले एक किमी में मुझे बहुत ठंड लगी लेकिन इसके बात सबकुछ ठीक रहा.
5. कोई भी समस्या नहीं हुई.'' गिनीज वर्ल्ड रिकॉर्ड पेत्कोव ने कहा, ''अगर पानी गरम होता तो मैं दस किमी या उससे भी ज्यादा तैर सकता था.'' आयोजकों ने बताया कि गिनीज वर्ल्ड रिकॉर्ड में आधिकारिक रूप से शामिल किए जाने के लिए उन्होंने इस तैराकी प्रदर्शन को विधिवत रिकॉर्ड किया है.


In [99]:
translations_3 = []

for i, sentence in enumerate(sentences_3, start=1):
    translated = translate_one(sentence)
    translations_3.append(translated)

    print(f"\nSentence {i}:")
    print("Hindi:", sentence)
    print("English:", translated)

print("\n" + "=" * 50)
print("RECONSTRUCTED ENGLISH ARTICLE:")
print(" ".join(translations_3))

print("\nSentence count:", len(translations_3))
print("Non-empty translations:", sum(bool(t.strip()) for t in translations_3))


Sentence 1:
Hindi: ख़बरों के मुताबिक़ जेन पेत्कोव को मैसेडोनिया की ऑरिड झील को पार करने में करीब पौने तीन घंटे लगे.
English: According to reports, Jane Petkov took about three and a half hours to cross Lake Orid in Macedonia.

Sentence 2:
Hindi: आयोजक सासो तोकोव ने समाचार एजेंसी एएफपी को बताया कि 59 वर्षीय पेत्कोव के हाथ और पैर उनके शरीर से बांध दिये गए थे और उन्होंने डॉल्फिन की तरह पीठ के बल तैराकी की.
English: Organiser Sasso Tokov told AFP news agency that Petkov, 59, had his hands and legs tied to his body and swam on his back like a dolphin.

Sentence 3:
Hindi: मैसेडोनिया के मीडिया के मुताबिक़ तैराकी की उनकी औसत रफ़्तार 0.7 किमी प्रति घंटे थी.
English: According to Macedonian media, his average swimming speed was 0.7 km / h.

Sentence 4:
Hindi: पेत्कोव ने अपना अभियान पूरा करने के बाद कहा, ''पहले एक किमी में मुझे बहुत ठंड लगी लेकिन इसके बात सबकुछ ठीक रहा.
English: "I felt very cold in the first one km, but everything went well," Petkov said after finishing his campaign.

Sentence 

In [100]:
test_article_4 = df_clean.iloc[3]["text"]

sentences_4 = [
    s.strip()
    for s in re.split(
        r'(?<=[।.!?])\s+',
        prepare_hindi(test_article_4)
    )
    if s.strip()
]

print("Article ID:", df_clean.iloc[3]["id"])
print("Hindi words:", len(test_article_4.split()))
print("Number of sentences:", len(sentences_4))
print("\nFirst 5 sentences:")

for i, sentence in enumerate(sentences_4[:5], start=1):
    print(f"{i}. {sentence}")

Article ID: 050713_us_un
Hindi words: 245
Number of sentences: 14

First 5 sentences:
1. जी-4 में भारत के साथ-साथ ब्राज़ील, जापान और जर्मनी शामिल हैं.
2. इन देशों ने सुरक्षा परिषद के सदस्यों की संख्या 15 से बढ़ाकर 25 करने का प्रस्ताव रखा था.
3. अमरीका का कहना है कि सदस्य देश इस मुद्दे पर बँटे हुए हैं.
4. विदेश मंत्रालय की एक प्रवक्ता शीरीन ताहिर खेली ने बताया कि अमरीका सुरक्षा परिषद के विस्तार के लिए काम करेगा लेकिन सही समय पर सही तरीक़े से.
5. जी-4 के देशों ने सोमवार को यह प्रस्ताव पेश किया था.


In [101]:
translations_4 = []

for i, sentence in enumerate(sentences_4, start=1):
    translated = translate_one(sentence)
    translations_4.append(translated)

    print(f"\nSentence {i}:")
    print("English:", translated)

print("\n" + "=" * 50)
print("RECONSTRUCTED ENGLISH ARTICLE:")
print(" ".join(translations_4))

print("\nTotal sentences:", len(sentences_4))
print(
    "Non-empty translations:",
    sum(bool(t.strip()) for t in translations_4)
)


Sentence 1:
English: The G-4 comprises Brazil, Japan and Germany along with India.

Sentence 2:
English: These countries had proposed to increase the number of members of the Security Council from 15 to 25.

Sentence 3:
English: The United States says member states are divided on the issue.

Sentence 4:
English: A State Department spokeswoman, Shireen Tahir Khali, said the United States would work to expand the Security Council "but in the right way at the right time."

Sentence 5:
English: The G-4 countries introduced the proposal on Monday.

Sentence 6:
English: That's being debated.

Sentence 7:
English: It is not yet clear when the vote on the motion will take place.

Sentence 8:
English: However, a two-thirds vote would be required for the resolution to pass.

Sentence 9:
English: The G-4 countries'proposal calls for an increase of 10 more seats on the Security Council, six of which are for new permanent members.

Sentence 10:
English: According to the proposal, the new permanent

In [102]:
import re
import pandas as pd

def split_sentences(text):
    text = prepare_hindi(text)
    return [
        s.strip()
        for s in re.split(r'(?<=[।.!?])\s+', text)
        if s.strip()
    ]

sentence_counts = df_clean["text"].apply(
    lambda x: len(split_sentences(x))
)

print("Total articles:", len(df_clean))
print("Total sentences:", sentence_counts.sum())
print("Average sentences per article:", round(sentence_counts.mean(), 2))
print("Minimum sentences:", sentence_counts.min())
print("Maximum sentences:", sentence_counts.max())

print("\nSentence-count distribution:")
print(sentence_counts.describe())

Total articles: 250
Total sentences: 5214
Average sentences per article: 20.86
Minimum sentences: 5
Maximum sentences: 64

Sentence-count distribution:
count    250.000000
mean      20.856000
std       13.237833
min        5.000000
25%       11.250000
50%       16.000000
75%       26.500000
max       64.000000
Name: text, dtype: float64


In [103]:
sentence_inventory = []

for article_index, row in df_clean.iterrows():
    article_id = row["id"]
    article_sentences = split_sentences(row["text"])

    for sentence_no, sentence in enumerate(article_sentences, start=1):
        sentence_inventory.append({
            "id": article_id,
            "sentence_no": sentence_no,
            "hindi_text": sentence,
            "hindi_words": len(sentence.split())
        })

sentence_inventory_df = pd.DataFrame(sentence_inventory)

token_lengths = []

for start in range(0, len(sentence_inventory_df), 128):
    batch = sentence_inventory_df["hindi_text"].iloc[
        start:start + 128
    ].tolist()

    prepared = ip.preprocess_batch(
        batch,
        src_lang="hin_Deva",
        tgt_lang="eng_Latn"
    )

    encoded = tokenizer(
        prepared,
        truncation=False,
        padding=False
    )

    token_lengths.extend(
        len(ids) for ids in encoded["input_ids"]
    )

sentence_inventory_df["source_tokens"] = token_lengths

print("Total sentences:", len(sentence_inventory_df))
print("Maximum source tokens:", max(token_lengths))
print("Sentences above 256 tokens:", sum(t > 256 for t in token_lengths))
print("Sentences above 120 tokens:", sum(t > 120 for t in token_lengths))

print("\nToken length statistics:")
print(sentence_inventory_df["source_tokens"].describe())

print("\nLongest 10 sentences:")
display(
    sentence_inventory_df.nlargest(10, "source_tokens")[
        ["id", "sentence_no", "hindi_words", "source_tokens", "hindi_text"]
    ]
)

Total sentences: 5214
Maximum source tokens: 312
Sentences above 256 tokens: 1
Sentences above 120 tokens: 14

Token length statistics:
count    5214.000000
mean       28.856157
std        16.096416
min         5.000000
25%        18.000000
50%        26.000000
75%        35.000000
max       312.000000
Name: source_tokens, dtype: float64

Longest 10 sentences:


,id,sentence_no,hindi_words,source_tokens,hindi_text
4465,131106_uk_migration_contribution_rd,17,260,312,प्रोफ़ेसर की रिपोर्ट में भविष्य में स्वास्थ्य ...
3927,india-44912345,4,113,185,जीएसटी के दायरे से बाहर होने वाली चीजें -सैनिट...
699,130614_complaint_for_looks_rd,4,144,173,"अविश्वसनीय एक पुलिस प्रवक्ता ने बताया, “फ़ोन क..."
4461,131106_uk_migration_contribution_rd,13,123,168,"प्रोफ़ेसर डस्टमैन कहते हैं, ""ज़्यादातर यूरोपीय..."
3904,040305_holi_songs,11,72,163,हिंदी फ़िल्मों के दर्जनों होली गीतों में से प्...
848,041015_envoy_critics,8,130,156,मैंने अपने विचार खुले रूप में रखे लेकिन सिर्फ़...
1814,india-47148086,7,134,155,राम मंदिर का राजनीतिक फ़ायदा अंग्रेज़ी अख़बार ...
2170,050606_kabir_bedi,11,122,155,"""फ़िल्मों में तो कई बार में किसी दृश्य की शूटि..."
1386,131227_diamond_antarctica_tk,8,110,146,ब्रिटिश अंटार्कटिक सर्वे के एक सर्वेक्षण भूविज...
2074,india-48250857,4,123,146,इसी तरह फोर्ब्स पत्रिका में 16 मार्च 2019 को छ...


In [104]:
long_sentence = sentence_inventory_df.loc[
    sentence_inventory_df["source_tokens"].idxmax()
]

print("Article ID:", long_sentence["id"])
print("Sentence number:", long_sentence["sentence_no"])
print("Source tokens:", long_sentence["source_tokens"])
print("\nFULL HINDI SENTENCE:\n")
print(long_sentence["hindi_text"])

Article ID: 131106_uk_migration_contribution_rd
Sentence number: 17
Source tokens: 312

FULL HINDI SENTENCE:

प्रोफ़ेसर की रिपोर्ट में भविष्य में स्वास्थ्य पर खर्च को शामिल नहीं किया गया है- बेशक इसकी वजह अच्छी ही होगी- जो आप्रवासियों की उम्र बढ़ने के साथ बढ़ता जाएगा और न ही पेंशन को शामिल किया गया है, जो अच्छी ख़ासी रकम होगी." सरकारी कदम सही प्रोफ़ेसर डस्टमैन ने टुडे कार्यक्रम में कहा, "यह सही है कि नए आप्रवासी युवा हैं लेकि इसके साथ ही बेहतर शिक्षा प्राप्त भी हैं." "तो वह लाभ प्रणाली से जितना लेंगे उतना ही अधिक वह भविष्य में योगदान करेंगे क्योंकि वह अपने करियर की ऊंचाई और पैसा कमाने की अपनी क्षमताओं के चरम पर नहीं पहुंचे हैं." "स्वाभाविक है आप जितना ज़्यादा कमाएंगे उतना ज़्यादा आप कर चुकाएंगे." एक सरकारी प्रवक्ता ने कहा, "हम उन लोगों का स्वागत करते हैं जो अर्थव्यवस्था में योगदान करने के लिए आते हैं, लेकिन यह भी बिल्कुल सही है कि हम ब्रितानी लाभ प्रणाली को सुरक्षित रखने के लिए सख़्त प्रावधान करें ताकि इसका ग़लत इस्तेमाल न हो सके." उन्होंने कहा कि सरकार इसलिए प्रावधानों को कड़ा कर रही 

In [105]:
long_parts = [
    s.strip()
    for s in re.split(
        r'(?<=[।.!?])\s+|(?<=[।.!?])(?=["”])',
        long_sentence["hindi_text"]
    )
    if s.strip()
]

print("Number of parts:", len(long_parts))

for i, part in enumerate(long_parts, start=1):
    print(f"\nPart {i}")
    print("Tokens:", source_token_count(part))
    print(part)

Number of parts: 8

Part 1
Tokens: 54
प्रोफ़ेसर की रिपोर्ट में भविष्य में स्वास्थ्य पर खर्च को शामिल नहीं किया गया है- बेशक इसकी वजह अच्छी ही होगी- जो आप्रवासियों की उम्र बढ़ने के साथ बढ़ता जाएगा और न ही पेंशन को शामिल किया गया है, जो अच्छी ख़ासी रकम होगी.

Part 2
Tokens: 39
" सरकारी कदम सही प्रोफ़ेसर डस्टमैन ने टुडे कार्यक्रम में कहा, "यह सही है कि नए आप्रवासी युवा हैं लेकि इसके साथ ही बेहतर शिक्षा प्राप्त भी हैं.

Part 3
Tokens: 40
" "तो वह लाभ प्रणाली से जितना लेंगे उतना ही अधिक वह भविष्य में योगदान करेंगे क्योंकि वह अपने करियर की ऊंचाई और पैसा कमाने की अपनी क्षमताओं के चरम पर नहीं पहुंचे हैं.

Part 4
Tokens: 19
" "स्वाभाविक है आप जितना ज़्यादा कमाएंगे उतना ज़्यादा आप कर चुकाएंगे.

Part 5
Tokens: 57
" एक सरकारी प्रवक्ता ने कहा, "हम उन लोगों का स्वागत करते हैं जो अर्थव्यवस्था में योगदान करने के लिए आते हैं, लेकिन यह भी बिल्कुल सही है कि हम ब्रितानी लाभ प्रणाली को सुरक्षित रखने के लिए सख़्त प्रावधान करें ताकि इसका ग़लत इस्तेमाल न हो सके.

Part 6
Tokens: 34
" उन्होंने कहा कि सरकार इसलि

In [106]:
import shutil

drive_path = "/content/drive/MyDrive/FactIndic_PersonB_CleanRun"

total, used, free = shutil.disk_usage(drive_path)

print("Total storage (GB):", round(total / (1024**3), 2))
print("Used storage (GB):", round(used / (1024**3), 2))
print("Free storage (GB):", round(free / (1024**3), 2))

Total storage (GB): 15.0
Used storage (GB): 0.24
Free storage (GB): 14.76


In [107]:
import os

BASE_DIR = "/content/drive/MyDrive/FactIndic_PersonB_CleanRun"

NEW_RUN_DIR = os.path.join(
    BASE_DIR,
    "09_sentence_level_translation"
)

os.makedirs(NEW_RUN_DIR, exist_ok=True)

print("New output folder created:")
print(NEW_RUN_DIR)

print("\nExisting data remains untouched.")

New output folder created:
/content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation

Existing data remains untouched.


In [108]:
import os
import pandas as pd

# Prepare sentence-level records
sentence_inventory_df = sentence_inventory_df.copy()

sentence_inventory_df["translation"] = ""
sentence_inventory_df["translation_status"] = "Pending"

# Unique ID for every sentence
sentence_inventory_df.insert(
    0,
    "sentence_id",
    range(1, len(sentence_inventory_df) + 1)
)

# Save the initial inventory
INVENTORY_PATH = os.path.join(
    NEW_RUN_DIR,
    "Sentence_Level_Translation_Inventory.csv"
)

sentence_inventory_df.to_csv(
    INVENTORY_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Sentence inventory prepared.")
print("Total sentences:", len(sentence_inventory_df))
print("Unique articles:", sentence_inventory_df["id"].nunique())
print("Pending translations:", len(sentence_inventory_df))
print("Saved to:", INVENTORY_PATH)

Sentence inventory prepared.
Total sentences: 5214
Unique articles: 250
Pending translations: 5214
Saved to: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/Sentence_Level_Translation_Inventory.csv


In [109]:
translation_tasks = []

for _, row in sentence_inventory_df.iterrows():
    sentence = row["hindi_text"]

    # Split each sentence into safe chunks of at most 120 source tokens
    chunks = make_safe_token_chunks(
        sentence,
        max_tokens=120
    )

    for part_no, chunk in enumerate(chunks, start=1):
        translation_tasks.append({
            "sentence_id": row["sentence_id"],
            "id": row["id"],
            "sentence_no": row["sentence_no"],
            "part_no": part_no,
            "hindi_chunk": chunk,
            "source_tokens": source_token_count(chunk)
        })

translation_tasks_df = pd.DataFrame(translation_tasks)

print("Original sentences:", len(sentence_inventory_df))
print("Translation chunks:", len(translation_tasks_df))
print("Unique articles:", translation_tasks_df["id"].nunique())
print("Largest chunk:", translation_tasks_df["source_tokens"].max(), "tokens")
print("Chunks above 120 tokens:",
      (translation_tasks_df["source_tokens"] > 120).sum())

TASKS_PATH = os.path.join(
    NEW_RUN_DIR,
    "Sentence_Translation_Tasks.csv"
)

translation_tasks_df.to_csv(
    TASKS_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nTask list saved:", TASKS_PATH)

Original sentences: 5214
Translation chunks: 5229
Unique articles: 250
Largest chunk: 120 tokens
Chunks above 120 tokens: 0

Task list saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/Sentence_Translation_Tasks.csv


In [110]:
import pandas as pd
import os

TASKS_PATH = os.path.join(
    NEW_RUN_DIR,
    "Sentence_Translation_Tasks.csv"
)

translation_tasks_df = pd.read_csv(TASKS_PATH)

print("Tasks loaded:", len(translation_tasks_df))
print("Unique articles:", translation_tasks_df["id"].nunique())
print("Translation function available:", callable(translate_one))
print("Model available:", model is not None)
print("Tokenizer available:", tokenizer is not None)

Tasks loaded: 5229
Unique articles: 250
Translation function available: True
Model available: True
Tokenizer available: True


In [111]:
import os
import pandas as pd
import time

CHECKPOINT_PATH = os.path.join(
    NEW_RUN_DIR,
    "Sentence_Translation_Checkpoints.csv"
)

# Load previous progress if it exists
if os.path.exists(CHECKPOINT_PATH):
    completed_df = pd.read_csv(CHECKPOINT_PATH)
    completed_ids = set(completed_df["task_index"].astype(int))
else:
    completed_df = pd.DataFrame(
        columns=["task_index", "translation", "status"]
    )
    completed_ids = set()

total = len(translation_tasks_df)
save_every = 25

print("Total chunks:", total)
print("Already completed:", len(completed_ids))
print("Remaining:", total - len(completed_ids))

new_results = []

for idx, row in translation_tasks_df.iterrows():

    if idx in completed_ids:
        continue

    try:
        translated = translate_one(row["hindi_chunk"])

        if not translated.strip():
            raise ValueError("Empty translation")

        new_results.append({
            "task_index": idx,
            "translation": translated,
            "status": "Completed"
        })

    except Exception as e:
        new_results.append({
            "task_index": idx,
            "translation": "",
            "status": "Error: " + str(e)[:200]
        })

    if len(new_results) >= save_every:
        completed_df = pd.concat(
            [completed_df, pd.DataFrame(new_results)],
            ignore_index=True
        )

        completed_df.to_csv(
            CHECKPOINT_PATH,
            index=False,
            encoding="utf-8-sig"
        )

        completed_ids.update(
            item["task_index"] for item in new_results
        )

        print(
            f"Saved {len(completed_ids)} / {total} chunks"
        )

        new_results = []

# Save final remaining results
if new_results:
    completed_df = pd.concat(
        [completed_df, pd.DataFrame(new_results)],
        ignore_index=True
    )

    completed_df.to_csv(
        CHECKPOINT_PATH,
        index=False,
        encoding="utf-8-sig"
    )

print("\nTranslation run finished.")
print("Checkpoint:", CHECKPOINT_PATH)
print("Total saved records:", len(completed_df))
print("Successful:", (completed_df["status"] == "Completed").sum())
print("Errors:", completed_df["status"].str.startswith("Error").sum())

Total chunks: 5229
Already completed: 0
Remaining: 5229
Saved 25 / 5229 chunks
Saved 50 / 5229 chunks
Saved 75 / 5229 chunks
Saved 100 / 5229 chunks
Saved 125 / 5229 chunks
Saved 150 / 5229 chunks
Saved 175 / 5229 chunks
Saved 200 / 5229 chunks
Saved 225 / 5229 chunks
Saved 250 / 5229 chunks
Saved 275 / 5229 chunks
Saved 300 / 5229 chunks
Saved 325 / 5229 chunks
Saved 350 / 5229 chunks
Saved 375 / 5229 chunks
Saved 400 / 5229 chunks
Saved 425 / 5229 chunks
Saved 450 / 5229 chunks
Saved 475 / 5229 chunks
Saved 500 / 5229 chunks
Saved 525 / 5229 chunks
Saved 550 / 5229 chunks
Saved 575 / 5229 chunks
Saved 600 / 5229 chunks
Saved 625 / 5229 chunks
Saved 650 / 5229 chunks
Saved 675 / 5229 chunks
Saved 700 / 5229 chunks
Saved 725 / 5229 chunks
Saved 750 / 5229 chunks
Saved 775 / 5229 chunks
Saved 800 / 5229 chunks
Saved 825 / 5229 chunks
Saved 850 / 5229 chunks
Saved 875 / 5229 chunks
Saved 900 / 5229 chunks
Saved 925 / 5229 chunks
Saved 950 / 5229 chunks
Saved 975 / 5229 chunks
Saved 1000 

In [112]:
import pandas as pd
import os

CHECKPOINT_PATH = os.path.join(
    NEW_RUN_DIR,
    "Sentence_Translation_Checkpoints.csv"
)

completed_df = pd.read_csv(CHECKPOINT_PATH)

# Attach translations to the original task list
translation_results = translation_tasks_df.copy()

translation_results["translation"] = (
    completed_df
    .set_index("task_index")["translation"]
    .reindex(translation_results.index)
    .values
)

translation_results["status"] = (
    completed_df
    .set_index("task_index")["status"]
    .reindex(translation_results.index)
    .values
)

# Combine chunks belonging to each original sentence
sentence_translations = (
    translation_results
    .sort_values(["sentence_id", "part_no"])
    .groupby(
        ["sentence_id", "id", "sentence_no"],
        as_index=False
    )
    .agg({
        "translation": lambda x: " ".join(
            str(v).strip() for v in x if pd.notna(v)
        ),
        "status": lambda x: "Completed"
        if all(v == "Completed" for v in x)
        else "Review"
    })
)

print("Reconstructed sentences:", len(sentence_translations))
print("Unique articles:", sentence_translations["id"].nunique())
print("Sentences needing review:",
      (sentence_translations["status"] != "Completed").sum())

Reconstructed sentences: 5214
Unique articles: 250
Sentences needing review: 0


In [113]:
# Combine sentences in their original order
article_translations = (
    sentence_translations
    .sort_values(["id", "sentence_no"])
    .groupby("id", as_index=False)
    .agg(
        complete_english_translation=(
            "translation",
            lambda x: " ".join(
                str(v).strip() for v in x if str(v).strip()
            )
        ),
        reconstructed_sentence_count=("sentence_no", "count")
    )
)

# Compare with original article sentence counts
original_counts = (
    sentence_inventory_df
    .groupby("id")
    .size()
    .rename("original_sentence_count")
    .reset_index()
)

article_translations = article_translations.merge(
    original_counts,
    on="id",
    how="left"
)

print("Reconstructed articles:", len(article_translations))
print(
    "Articles with matching sentence counts:",
    (
        article_translations["reconstructed_sentence_count"]
        == article_translations["original_sentence_count"]
    ).sum()
)

print("\nSample translation:")
print(article_translations.iloc[0]["complete_english_translation"][:1000])

Reconstructed articles: 250
Articles with matching sentence counts: 250

Sample translation:
You sent this material because of an incident in your local area or because you have a personal opinion on a news item. Don't forget to send your name, email address, and some important information related to the video. Be sure to send your phone number as well, as we may need to contact you. Don't put yourself or others in danger or break any laws if you're taking a picture or making a movie. Click this link for more information. Sending your content means that you will abide by our Terms and Conditions If you are having trouble email us at hindi.letters@bbc.co.uk If you have any questions, check out our Q & A. You can also send us your photos and videos by email: mybbchindi@bbc.co.uk * Information is required.


In [114]:
import os

ARTICLE_OUTPUT_PATH = os.path.join(
    NEW_RUN_DIR,
    "FactIndic_English_250_Sentence_Reconstructed.csv"
)

article_translations.to_csv(
    ARTICLE_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Saved reconstructed English articles.")
print("Articles:", len(article_translations))
print("Unique IDs:", article_translations["id"].nunique())
print("File:", ARTICLE_OUTPUT_PATH)

Saved reconstructed English articles.
Articles: 250
Unique IDs: 250
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Sentence_Reconstructed.csv


In [115]:
import re
import pandas as pd

quality_audit = article_translations.copy()

# Original Hindi word counts
hindi_counts = (
    df_clean[["id", "text"]]
    .copy()
)

hindi_counts["hindi_words"] = (
    hindi_counts["text"].astype(str).str.split().str.len()
)

# English word counts
quality_audit["english_words"] = (
    quality_audit["complete_english_translation"]
    .astype(str).str.split().str.len()
)

quality_audit = quality_audit.merge(
    hindi_counts[["id", "hindi_words"]],
    on="id",
    how="left"
)

quality_audit["translation_ratio"] = (
    quality_audit["english_words"] /
    quality_audit["hindi_words"].replace(0, pd.NA)
)

# Detect remaining Devanagari characters
quality_audit["remaining_hindi_chars"] = (
    quality_audit["complete_english_translation"]
    .astype(str)
    .apply(lambda x: len(re.findall(r"[\u0900-\u097F]", x)))
)

quality_audit["empty_translation"] = (
    quality_audit["complete_english_translation"]
    .astype(str).str.strip().eq("")
)

print("Total articles:", len(quality_audit))
print("Empty translations:", quality_audit["empty_translation"].sum())
print("Articles with Devanagari characters:",
      (quality_audit["remaining_hindi_chars"] > 0).sum())

print("\nTranslation ratio statistics:")
print(quality_audit["translation_ratio"].describe().round(3))

print("\nLowest 10 translation ratios:")
display(
    quality_audit.sort_values("translation_ratio")[
        ["id", "hindi_words", "english_words", "translation_ratio"]
    ].head(10)
)

Total articles: 250
Empty translations: 0
Articles with Devanagari characters: 7

Translation ratio statistics:
count    250.000
mean       0.863
std        0.056
min        0.700
25%        0.826
50%        0.862
75%        0.896
max        1.039
Name: translation_ratio, dtype: float64

Lowest 10 translation ratios:


,id,hindi_words,english_words,translation_ratio
118,121207_hangover_smoking_aa,340,238,0.700000
110,120816_assange_equador_pa,372,263,0.706989
17,040723_musharraf_natwar,340,249,0.732353
122,130312_traffic_noise_songbirds_vr,358,264,0.737430
127,130614_complaint_for_looks_rd,416,308,0.740385
2,031031_sharon_palestine,259,193,0.745174
30,050112_nepalmao_amnesty,191,143,0.748691
43,050812_baflights_cancel,334,251,0.751497
230,international-44515542,495,372,0.751515
144,140223_modi_rally_punjab_sr,425,321,0.755294


In [116]:
import re

hindi_remaining = quality_audit[
    quality_audit["remaining_hindi_chars"] > 0
].copy()

for _, row in hindi_remaining.iterrows():
    text = row["complete_english_translation"]

    hindi_parts = re.findall(
        r'[\u0900-\u097F]+',
        text
    )

    print("\nArticle ID:", row["id"])
    print("Remaining Hindi text:", hindi_parts[:20])
    print("Total Hindi characters:", row["remaining_hindi_chars"])

print("\nTotal articles requiring inspection:", len(hindi_remaining))


Article ID: 041026_bestbakery_questions
Remaining Hindi text: ['है', 'बीस', 'है', 'मैं', 'है', 'उल्लेखनीय']
Total Hindi characters: 21

Article ID: 051030_delhi_blasts
Remaining Hindi text: ['गई', 'बीबीसी']
Total Hindi characters: 8

Article ID: 061217_spl_sez_imballance
Remaining Hindi text: ['हैं', 'पूरी']
Total Hindi characters: 7

Article ID: 080419_brchopra_festival
Remaining Hindi text: ['सके', 'फ़िल्मोत्सव']
Total Hindi characters: 14

Article ID: 090210_filmfestival_gorakhpur
Remaining Hindi text: ['के', 'पी']
Total Hindi characters: 4

Article ID: india-51453120
Remaining Hindi text: ['है', 'ग्लावकॉसमॉस']
Total Hindi characters: 13

Article ID: india-55465758
Remaining Hindi text: ['है', 'बाह्य']
Total Hindi characters: 7

Total articles requiring inspection: 7


In [117]:
for article_id in hindi_remaining["id"]:
    hindi_row = df_clean[df_clean["id"] == article_id].iloc[0]

    english_row = article_translations[
        article_translations["id"] == article_id
    ].iloc[0]

    print("\n" + "=" * 80)
    print("ARTICLE ID:", article_id)

    print("\nHINDI SOURCE:")
    print(str(hindi_row["text"])[:1800])

    print("\nENGLISH TRANSLATION:")
    print(
        str(english_row["complete_english_translation"])[:1800]
    )


ARTICLE ID: 041026_bestbakery_questions

HINDI SOURCE:
मगर सच्चाई ये नहीं है.बीस साल की ज़ाहिरा शेख़ और उसके जैसे कुछ दूसरे लोगों के लिए गुजरात वह जगह नहीं है जहाँ वो बिना किसी डर के रह सकें. ज़ाहिरा और उसके जैसे कुछ अन्य लोग अदालत के सामने भी सच बोल पाने का हौसला नहीं जुटा पा रहे हैं. ज़ाहिरा शेख़ बेस्ट बेकरी मुक़दमे की प्रमुख गवाह हैं. वडोदरा के निकट 12 लोगों को जलाकर मारने के इस मामले में सभी 21 अभियुक्तों को 21 जून 2002 को बरी कर दिया गया था. कारण – 71 में से 45 गवाहों ने अदालत में अपने बयान बदल डाले. अदालत में इन सभी 45 गवाहों ने एक-सी वो कहानी सुनाई जो पुलिस के सामने दिए गए उनके बयान से बिल्कुल अलग थी. बयान क्यों बदले उनके बयानों के बाद अदालत का फ़ैसला क्या होगा इसका अंदाज़ तो सबको था मगर ये जानने की कोशिश किसी ने नहीं की कि गवाह एक-एक कर बयान क्यों बदल रहे थे. मगर अब सामाजिक संगठनों की कोशिश से सनसनीखेज बातें सामने आईं रही हैं. ज़ाहिरा शेख़ ने बताया कि भारतीय जनता पार्टी के एक विधायक मधु श्रीवास्तव ने इन लोगों को धमकी दी थी जिसके कारण ही सबने अदालत में बयान बदल डाले. मगर मधु श्

# New Section

In [118]:
import re

problem_sentences = sentence_translations[
    sentence_translations["translation"].str.contains(
        r"[\u0900-\u097F]",
        regex=True,
        na=False
    )
].copy()

problem_sentences = problem_sentences.merge(
    sentence_inventory_df[
        ["sentence_id", "hindi_text"]
    ],
    on="sentence_id",
    how="left"
)

print("Problematic sentences:", len(problem_sentences))

for _, row in problem_sentences.iterrows():
    print("\n" + "=" * 70)
    print("Article:", row["id"])
    print("Sentence number:", row["sentence_no"])
    print("HINDI:", row["hindi_text"])
    print("ENGLISH:", row["translation"])

Problematic sentences: 9

Article: 080419_brchopra_festival
Sentence number: 3
HINDI: खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया.
ENGLISH: BR Chopra himself did not attend the ceremony due to ill health. सके.फ़िल्मोत्सव was inaugurated by BR Chopra's brother and noted filmmaker Yash Chopra.

Article: india-55465758
Sentence number: 36
HINDI: इसलिए चांद पर किसी एक का अधिकार नहीं हो सकता है.बाह्य अंतरिक्ष में किसी भी गतिविधि को नियंत्रित करने के लिए 1967 में आउटर स्पेस ट्रीटी (संधि) हुई थी.
ENGLISH: Therefore, no one can have the right to the moon. है.बाह्य The Outer Space Treaty (treaty) was signed in 1967 to regulate any activity in space.

Article: india-51453120
Sentence number: 15
HINDI: सोमवार को रूसी एजेंसी ग्लावकॉसमॉस की ओर से बयान जारी किया गया, जिसमें कहा गया, "GCTC ने भारतीयों के लिए ट्रेनिंग का कार्यक्रम शुरू कर दिया है.ग्लावकॉसमॉस और इसरो के ह्यूमन स्पेसफ़्लाइट सेंटर के बीच हुए

In [119]:
import re

def repair_sentence_boundaries(text):
    text = str(text).strip()

    # Split after punctuation, including when the next
    # Hindi sentence starts immediately without a space.
    parts = re.split(
        r'(?<=[।.!?])["”’]?\s*(?=[अ-ह])',
        text
    )

    return [p.strip() for p in parts if p.strip()]


repair_parts = []

for _, row in problem_sentences.iterrows():
    parts = repair_sentence_boundaries(row["hindi_text"])

    for part_no, part in enumerate(parts, start=1):
        repair_parts.append({
            "sentence_id": row["sentence_id"],
            "id": row["id"],
            "original_sentence_no": row["sentence_no"],
            "part_no": part_no,
            "hindi_text": part
        })

repair_parts_df = pd.DataFrame(repair_parts)

print("Affected original records:", len(problem_sentences))
print("Corrected sentence parts:", len(repair_parts_df))

for article_id in repair_parts_df["id"].unique():
    print("\nArticle:", article_id)

    for part in repair_parts_df[
        repair_parts_df["id"] == article_id
    ]["hindi_text"]:
        print("-", part)

Affected original records: 9
Corrected sentence parts: 21

Article: 080419_brchopra_festival
- खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.
- फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया.

Article: india-55465758
- इसलिए चांद पर किसी एक का अधिकार नहीं हो सकता है.
- बाह्य अंतरिक्ष में किसी भी गतिविधि को नियंत्रित करने के लिए 1967 में आउटर स्पेस ट्रीटी (संधि) हुई थी.

Article: india-51453120
- सोमवार को रूसी एजेंसी ग्लावकॉसमॉस की ओर से बयान जारी किया गया, जिसमें कहा गया, "GCTC ने भारतीयों के लिए ट्रेनिंग का कार्यक्रम शुरू कर दिया है.
- ग्लावकॉसमॉस और इसरो के ह्यूमन स्पेसफ़्लाइट सेंटर के बीच हुए अनुबंध के तहत ऐसा किया जा रहा है.
- यहां इन पायलटों को शारीरिक अभ्यास करने के साथ बायोमेडिकल ट्रेनिंग भी लेनी होगी.

Article: 051030_delhi_blasts
- विस्फोट के बाद बाज़ार में बिक्री के लिए सजी आतिशबाज़ी में आग लग गई.
- बीबीसी के पॉल डानहर के अनुसार इस इलाक़े में हुए विस्फोट में कम से कम तीन लोग मारे गए हैं.

Article: 041026_bestbakery_questions
-

In [120]:
# Merge the incorrectly split "के.पी" initials
mask = (
    (repair_parts_df["id"] == "090210_filmfestival_gorakhpur") &
    (repair_parts_df["original_sentence_no"] == 3)
)

film_parts = repair_parts_df[mask].sort_values("part_no")

merged_film_text = (
    film_parts.iloc[0]["hindi_text"].rstrip() +
    " " +
    film_parts.iloc[1]["hindi_text"].lstrip()
)

repair_parts_df = repair_parts_df[~mask].copy()

merged_row = {
    "sentence_id": film_parts.iloc[0]["sentence_id"],
    "id": "090210_filmfestival_gorakhpur",
    "original_sentence_no": 3,
    "part_no": 1,
    "hindi_text": merged_film_text
}

repair_parts_df = pd.concat(
    [repair_parts_df, pd.DataFrame([merged_row])],
    ignore_index=True
)

repair_parts_df = repair_parts_df.sort_values(
    ["id", "original_sentence_no", "part_no"]
).reset_index(drop=True)

print("Final sentence parts:", len(repair_parts_df))
print("\nCorrected film festival sentence:")
print(merged_film_text)

Final sentence parts: 20

Corrected film festival sentence:
तुर्की के जाने माने फ़िल्मकार इल्माज़ गुने की फ़िल्म ‘उमत’, ईरानी निर्देशक मख़मलबाफ़ की ‘ब्लैकबोर्ड’, चीनी फ़िल्मकार यिंग यिंग की ‘रेड रोड ऑफ़ होप’, के. पी ससी की ‘सागर तट के सौदागर’, माईकल टी क्लेयर की ‘ब्लड ऐण्ड ऑयल’, बीजू टोप्पो की ‘लोहा गर्म है’ में दर्शकों ने काफ़ी दिलचस्पी दिखाई.


In [121]:
repair_parts_df["corrected_english"] = ""

for idx, row in repair_parts_df.iterrows():
    try:
        translated = translate_one(row["hindi_text"])
        repair_parts_df.at[idx, "corrected_english"] = translated

    except Exception as e:
        print("Translation error:", row["id"], row["part_no"], str(e))

print("Translated parts:",
      (repair_parts_df["corrected_english"].str.strip() != "").sum())

print("Total parts:", len(repair_parts_df))

for _, row in repair_parts_df.iterrows():
    print("\nARTICLE:", row["id"])
    print("HINDI:", row["hindi_text"])
    print("ENGLISH:", row["corrected_english"])

Translated parts: 20
Total parts: 20

ARTICLE: 041026_bestbakery_questions
HINDI: मगर सच्चाई ये नहीं है.
ENGLISH: But that's not the truth.

ARTICLE: 041026_bestbakery_questions
HINDI: बीस साल की ज़ाहिरा शेख़ और उसके जैसे कुछ दूसरे लोगों के लिए गुजरात वह जगह नहीं है जहाँ वो बिना किसी डर के रह सकें.
ENGLISH: For twenty-year-old Zahira Sheikh and a few others like her, Gujarat is not a place where they can live without fear.

ARTICLE: 041026_bestbakery_questions
HINDI: वे कहते हैं, "मैंने अपने 50 साल के जीवन में किसी को कोई धमकी नहीं दी है.
ENGLISH: "I have not threatened anyone in my 50 years of life," he says.

ARTICLE: 041026_bestbakery_questions
HINDI: मैं तो इस महिला को जानता भी नहीं.
ENGLISH: I don't even know this woman.

ARTICLE: 041026_bestbakery_questions
HINDI: उधर ज़ाहिरा अब बयान बदलने के लिए अफ़सोस कर रही है.
ENGLISH: Zahira, on the other hand, is now regretting changing her statement.

ARTICLE: 041026_bestbakery_questions
HINDI: आज उन्होंने जो हमारे साथ किया वो कल और किसी क

In [122]:
manual_corrections = {
    "वे कहते हैं, \"मैंने अपने 50 साल के जीवन में किसी को कोई धमकी नहीं दी है.":
        'He says, "I have not threatened anyone in my 50 years of life.',

    "मैं तो इस महिला को जानता भी नहीं.":
        'I do not even know this woman."',

    "आज उन्होंने जो हमारे साथ किया वो कल और किसी के साथ भी कर सकते हैं क्योंकि अब वे डरेंगे नहीं.":
        "What they did to us today, they could do to anyone else tomorrow because they will no longer be afraid.",

    "मानवाधिकार आयोग भारत के राष्ट्रीय मानवाधिकार आयोग ने बेस्ट बेकरी मामले पर अदालत के निर्णय पर गहरा असंतोष जताया है.":
        "The National Human Rights Commission of India expressed deep dissatisfaction with the court's decision in the Best Bakery case.",

    "तुर्की के जाने माने फ़िल्मकार इल्माज़ गुने की फ़िल्म ‘उमत’, ईरानी निर्देशक मख़मलबाफ़ की ‘ब्लैकबोर्ड’, चीनी फ़िल्मकार यिंग यिंग की ‘रेड रोड ऑफ़ होप’, के. पी ससी की ‘सागर तट के सौदागर’, माईकल टी क्लेयर की ‘ब्लड ऐण्ड ऑयल’, बीजू टोप्पो की ‘लोहा गर्म है’ में दर्शकों ने काफ़ी दिलचस्पी दिखाई.":
        "Popular Turkish filmmaker Ilmaz Gune's film 'Umut', Iranian director Makhmalbaf's 'Blackboard', Chinese filmmaker Ying Ying's 'Red Road of Hope', K. P. Sasi's 'Saagar Tat Ke Saudagar', Michael T. Clare's 'Blood and Oil', and Biju Toppo's 'Loha Garam Hai' attracted considerable interest from the audience."
}

for idx, row in repair_parts_df.iterrows():
    hindi = row["hindi_text"]

    if hindi in manual_corrections:
        repair_parts_df.at[idx, "corrected_english"] = manual_corrections[hindi]

print("Manual corrections applied:", len(manual_corrections))

for _, row in repair_parts_df.iterrows():
    print("\nARTICLE:", row["id"])
    print("ENGLISH:", row["corrected_english"])

Manual corrections applied: 5

ARTICLE: 041026_bestbakery_questions
ENGLISH: But that's not the truth.

ARTICLE: 041026_bestbakery_questions
ENGLISH: For twenty-year-old Zahira Sheikh and a few others like her, Gujarat is not a place where they can live without fear.

ARTICLE: 041026_bestbakery_questions
ENGLISH: He says, "I have not threatened anyone in my 50 years of life.

ARTICLE: 041026_bestbakery_questions
ENGLISH: I do not even know this woman."

ARTICLE: 041026_bestbakery_questions
ENGLISH: Zahira, on the other hand, is now regretting changing her statement.

ARTICLE: 041026_bestbakery_questions
ENGLISH: What they did to us today, they could do to anyone else tomorrow because they will no longer be afraid.

ARTICLE: 041026_bestbakery_questions
ENGLISH: The National Human Rights Commission of India expressed deep dissatisfaction with the court's decision in the Best Bakery case.

ARTICLE: 041026_bestbakery_questions
ENGLISH: It may be recalled that the commission had last year s

In [123]:
print("Available dataframes:")

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        print(f"\n{name}:")
        print("Shape:", obj.shape)
        print("Columns:", list(obj.columns))

Available dataframes:

df_clean:
Shape: (250, 7)
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words']

first60:
Shape: (60, 10)
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words', 'indic_title', 'indic_article', 'indic_summary']

results:
Shape: (190, 4)
Columns: ['id', 'indic_title', 'indic_article', 'indic_summary']

remaining_df:
Shape: (250, 13)
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words', 'indic_title', 'indic_article', 'indic_summary', 'hindi_word_count', 'english_word_count', 'translation_ratio']

batch_df:
Shape: (30, 7)
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words']

all_translations:
Shape: (250, 10)
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words', 'indic_title', 'indic_article', 'indic_summary']

final_df:
Shape: (250, 13)
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_

In [124]:
affected_ids = repair_parts_df["id"].unique().tolist()

for article_id in affected_ids:
    print("\n" + "=" * 75)
    print("ARTICLE:", article_id)

    current = article_translations[
        article_translations["id"] == article_id
    ]

    print("Current reconstructed translation:")
    print(current["complete_english_translation"].iloc[0][:700])

    print("\nSentence IDs:")
    print(
        sentence_translations[
            sentence_translations["id"] == article_id
        ][["sentence_id", "sentence_no"]].to_string(index=False)
    )


ARTICLE: 041026_bestbakery_questions
Current reconstructed translation:
But the truth is that for Zahira Sheikh of है.बीस year and a few others like her, Gujarat is not a place where they can live without fear. Zahira and a few others like her are unable to muster the courage to speak the truth even in front of the court. Zahira Sheikh is a key witness in the Best Bakery case. All 21 accused in this case of burning 12 people to death near Vadodara were acquitted on 21 June 2002. Reason: 45 out of 71 witnesses changed their statements in court. In the court, all these 45 witnesses narrated a story which was completely different from their statement before the police. Everyone had an idea of what the court's decision would be after his statements, but no one tried

Sentence IDs:
 sentence_id  sentence_no
        2018            1
        2019            2
        2020            3
        2021            4
        2022            5
        2023            6
        2024            7
   

In [125]:
import os
import pandas as pd

# 1. Join corrected parts into their original sentences
corrected_sentence_map = (
    repair_parts_df
    .sort_values(["sentence_id", "part_no"])
    .groupby("sentence_id")["corrected_english"]
    .apply(lambda x: " ".join(x.astype(str).str.strip()))
    .to_dict()
)

# 2. Update a COPY of the sentence-level translations
updated_sentence_translations = sentence_translations.copy()

updated_sentence_translations["translation"] = (
    updated_sentence_translations["sentence_id"]
    .map(corrected_sentence_map)
    .fillna(updated_sentence_translations["translation"])
)

# 3. Reconstruct all articles in their original sentence order
corrected_article_translations = (
    updated_sentence_translations
    .sort_values(["id", "sentence_no"])
    .groupby("id")
    .agg(
        complete_english_translation=(
            "translation",
            lambda x: " ".join(x.astype(str).str.strip())
        ),
        reconstructed_sentence_count=("sentence_no", "count")
    )
    .reset_index()
)

# 4. Validate dataset
print("Articles:", len(corrected_article_translations))
print("Unique article IDs:", corrected_article_translations["id"].nunique())
print("Corrected sentences:", len(corrected_sentence_map))

assert len(corrected_article_translations) == 250
assert corrected_article_translations["id"].nunique() == 250
assert len(corrected_sentence_map) == 9

# 5. Check remaining Devanagari characters
remaining_hindi = corrected_article_translations[
    corrected_article_translations["complete_english_translation"].str.contains(
        r"[\u0900-\u097F]",
        regex=True,
        na=False
    )
]

print("Articles with remaining Devanagari:", len(remaining_hindi))

# 6. Save as a new file
output_path = (
    "/content/drive/MyDrive/FactIndic_PersonB_CleanRun/"
    "09_sentence_level_translation/"
    "FactIndic_English_250_Corrected.csv"
)

corrected_article_translations.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:", output_path)
print("Final shape:", corrected_article_translations.shape)

Articles: 250
Unique article IDs: 250
Corrected sentences: 9
Articles with remaining Devanagari: 0

Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Corrected.csv
Final shape: (250, 3)


In [126]:
import re
import pandas as pd

# Match corrected English articles with original Hindi articles
final_audit = corrected_article_translations.merge(
    df_clean[["id", "text"]],
    on="id",
    how="left",
    validate="one_to_one"
)

# Word counts
final_audit["hindi_words"] = (
    final_audit["text"].fillna("").str.split().str.len()
)

final_audit["english_words"] = (
    final_audit["complete_english_translation"]
    .fillna("").str.split().str.len()
)

final_audit["translation_ratio"] = (
    final_audit["english_words"] /
    final_audit["hindi_words"].replace(0, pd.NA)
)

# Quality checks
final_audit["empty_translation"] = (
    final_audit["complete_english_translation"]
    .fillna("").str.strip().eq("")
)

final_audit["remaining_hindi_chars"] = (
    final_audit["complete_english_translation"]
    .fillna("").str.contains(r"[\u0900-\u097F]", regex=True)
)

print("FINAL QUALITY AUDIT")
print("=" * 45)
print("Total articles:", len(final_audit))
print("Unique IDs:", final_audit["id"].nunique())
print("Missing original Hindi text:", final_audit["text"].isna().sum())
print("Empty English translations:", final_audit["empty_translation"].sum())
print("Articles with Devanagari:", final_audit["remaining_hindi_chars"].sum())

print("\nTRANSLATION RATIO STATISTICS")
print(final_audit["translation_ratio"].describe().round(3))

print("\nLOWEST 10 TRANSLATION RATIOS")
print(
    final_audit.nsmallest(10, "translation_ratio")[
        ["id", "hindi_words", "english_words", "translation_ratio"]
    ].to_string(index=False)
)

# Save audit separately
audit_path = (
    "/content/drive/MyDrive/FactIndic_PersonB_CleanRun/"
    "09_sentence_level_translation/"
    "FactIndic_Final_Quality_Audit.csv"
)

final_audit.to_csv(audit_path, index=False, encoding="utf-8-sig")

print("\nAudit saved:", audit_path)

FINAL QUALITY AUDIT
Total articles: 250
Unique IDs: 250
Missing original Hindi text: 0
Empty English translations: 0
Articles with Devanagari: 0

TRANSLATION RATIO STATISTICS
count    250.000
mean       0.864
std        0.057
min        0.700
25%        0.826
50%        0.862
75%        0.896
max        1.039
Name: translation_ratio, dtype: float64

LOWEST 10 TRANSLATION RATIOS
                               id  hindi_words  english_words  translation_ratio
       121207_hangover_smoking_aa          340            238           0.700000
        120816_assange_equador_pa          372            263           0.706989
          040723_musharraf_natwar          340            249           0.732353
130312_traffic_noise_songbirds_vr          358            264           0.737430
    130614_complaint_for_looks_rd          416            308           0.740385
          031031_sharon_palestine          259            193           0.745174
          050112_nepalmao_amnesty          191      

In [127]:
lowest_2 = final_audit.nsmallest(
    2, "translation_ratio"
)

for _, row in lowest_2.iterrows():
    print("\n" + "=" * 80)
    print("ARTICLE:", row["id"])
    print("Translation ratio:", round(row["translation_ratio"], 3))

    print("\nHINDI ORIGINAL:\n")
    print(row["text"])

    print("\nENGLISH TRANSLATION:\n")
    print(row["complete_english_translation"])


ARTICLE: 121207_hangover_smoking_aa
Translation ratio: 0.7

HINDI ORIGINAL:

कई संस्थान इस अध्ययन को खासा अहम मान रहे हैं हालांकि इस शोध में ये नहीं बताया गया है कि धूम्रपान की वजह से हैंगोवर बिगड़ने की वजह क्या है. शोधकर्ताओं ने 113 अमरीकी छात्रों से कहा कि वो आठ हफ्तों तक पीने और धूम्रपान की अपनी आदतों से जुड़ी बातें और हैंगोवर के लक्षणों के बारे में डायरी में लिखें. जब उन्होंने जम की पी, मसलन एक घंटे में बीयर के छह कैन पी गए तो उस पर धूम्रपान करने वालों का हाल कहीं ज्यादा बुरा था. इस अध्ययन के नतीजे ‘जरनल ऑफ स्टडीज ऑन एल्होकल एंड ड्रग्स’ में प्रकाशित हुए हैं. 'जटिल संबंध' इस शोध पत्र को लिखने वालों में से एक डॉ. दामारिस रोहजेनॉव कहते हैं, “बराबर मात्रा में पीने वालों में से उन लोगों को अगले दिन हैंगोवर की ज्यादा संभावना होती है जो पीने के साथ धूम्रपान भी करते हैं.” “धूम्रपान न करने वालों के मुकाबले धूम्रपान करने वालों में हैंगोवर का जोखिम बढ़ जाता है.” नशे की लत को छुड़ाने के लिए काम कर रही संस्था 'एक्शन ऑन एडिक्शन' के प्रवक्ता ने इस बारे में और शोध किए जाने की जरूरत पर जोर दिया है

In [128]:
manual_corrections = {
    "121207_hangover_smoking_aa": """Several institutions consider this study important, although the research does not explain why smoking makes hangovers worse. Researchers asked 113 American students to keep a diary for eight weeks, recording their drinking and smoking habits and their hangover symptoms. When they drank heavily, for example, consuming six cans of beer in one hour, smokers experienced much worse hangovers. The findings were published in the Journal of Studies on Alcohol and Drugs.

One of the paper's authors, Dr. Damaris Rohsenow, said that among people who drink the same amount, those who also smoke are more likely to experience a hangover the following day. Smokers have a higher risk of hangovers than non-smokers.

A spokesperson for the addiction recovery organisation Action on Addiction stressed that more research was needed. The spokesperson said that the relationship between alcohol and smoking was complex. They hoped that some people would reduce their consumption of alcohol, tobacco, or both after learning that smoking increases the risk of hangovers.

The study, conducted at Midwestern University in the United States, examined alcohol consumption, the number of cigarettes smoked, and hangover symptoms. These included unusual tiredness, headaches, nausea, and difficulty concentrating.

Amanda Stanford, research manager at Action on Smoking and Health, said that both alcohol and tobacco interact with receptors in the brain. Therefore, it was not surprising that smoking increased the risk of hangovers.""",

    "120816_assange_equador_pa": """Julian Assange has taken refuge in the Ecuadorian embassy. Ecuador granted him asylum because it believes he faces a risk of human rights violations. Ecuador's Foreign Minister, Ricardo Patiño, accused Britain of openly threatening to arrest Assange at Ecuador's embassy in London.

Assange entered the Ecuadorian embassy in London in June because he feared being extradited to Sweden. Swedish authorities wanted to question him in connection with a rape allegation, which Assange denies.

Announcing Ecuador's decision, Foreign Minister Ricardo Patiño said that Assange's fear of political persecution was justified.

The British Foreign Office said it was surprised that Ecuador was considering granting Julian Assange political asylum. Earlier, Britain's Supreme Court had rejected Assange's appeal for a two-week delay before the extradition process was completed.

The dispute over Assange's possible arrest is escalating. His supporters gather whenever police approach the Ecuadorian embassy. BBC correspondent Ben Ando reported that both countries would try to avoid a forced entry into the embassy to arrest Assange.

Assange said he had been advised that asylum laws take precedence over extradition laws. In 2010, two women who had worked with WikiLeaks accused him of rape and sexual assault. Swedish prosecutors wanted to question him about these allegations. Assange said that the sexual encounters had been consensual.

WikiLeaks had published several confidential documents, embarrassing many governments. Assange feared that if he were sent to Sweden, he might subsequently be sent to the United States, where he could face trial in connection with the WikiLeaks case and possibly receive the death penalty.

Related material."""
}

print("Prepared corrections:", len(manual_corrections))
for article_id in manual_corrections:
    print("-", article_id)

Prepared corrections: 2
- 121207_hangover_smoking_aa
- 120816_assange_equador_pa


In [129]:
reviewed_translations = corrected_article_translations.copy()

reviewed_translations["complete_english_translation"] = (
    reviewed_translations["id"]
    .map(manual_corrections)
    .fillna(reviewed_translations["complete_english_translation"])
)

reviewed_translations["manual_review_status"] = (
    reviewed_translations["id"].apply(
        lambda x: "Manually corrected" if x in manual_corrections
        else "Pending review"
    )
)

reviewed_path = (
    "/content/drive/MyDrive/FactIndic_PersonB_CleanRun/"
    "09_sentence_level_translation/"
    "FactIndic_English_250_Reviewed.csv"
)

reviewed_translations.to_csv(
    reviewed_path,
    index=False,
    encoding="utf-8-sig"
)

print("Total articles:", len(reviewed_translations))
print("Manually corrected:", (
    reviewed_translations["manual_review_status"] == "Manually corrected"
).sum())
print("Pending review:", (
    reviewed_translations["manual_review_status"] == "Pending review"
).sum())
print("Saved:", reviewed_path)

Total articles: 250
Manually corrected: 2
Pending review: 248
Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv


In [130]:
lowest_3_4 = final_audit.nsmallest(4, "translation_ratio").iloc[2:4]

for _, row in lowest_3_4.iterrows():
    print("\n" + "=" * 80)
    print("ARTICLE:", row["id"])
    print("Translation ratio:", round(row["translation_ratio"], 3))
    print("\nHINDI ORIGINAL:\n")
    print(row["text"])
    print("\nENGLISH TRANSLATION:\n")
    print(row["complete_english_translation"])


ARTICLE: 040723_musharraf_natwar
Translation ratio: 0.732

HINDI ORIGINAL:

जनरल परवेज़ मुशर्रफ़ ने भारत के विदेश मंत्री नटवर सिंह से शुक्रवार को रावलपिंडी में मुलाक़ात के दौरान यह आशा व्यक्त की. नटवर सिंह सार्क देशों की बैठक में भाग लेने के लिए पाकिस्तान गए थे और शुक्रवार को जनरल परवेज़ मुशर्रफ़ के साथ क़रीब डेढ़ घंटे की मुलाक़ा के बाद भारत चले गए. भारत के लिए रवाना होने से पहले नटवर सिंह ने इस्लामाबाद हवाई अड्डे पर कहा, "इस मुलाक़ात के दौरान जम्मू कश्मीर सहित कोई भी ऐसा मुद्दा नहीं बचा, जिस पर बातचीत न हुई हो." "इस मुलाक़ात में हमनें विभिन्न मुद्दों पर अलग-अलग पहलुओं से बातचीत की. सरकारी और ग़ैर सरकारी स्तर पर हो रही बातचीत पर भी संतोष व्यक्त किया गया." नटवर सिंह ने कहा कि भारत और पाकिस्तान जम्मू कश्मीर सहित तमाम मुद्दों पर बातचीत और उन्हें इस तरह से हल करने के लिए संकल्पबद्ध हैं जिस पर दोनों देश संतुष्ट हों." व्यापक विचार अपनी इस यात्रा के दौरान नटवर सिंह ने पाकिस्तान के प्रधानमंत्री शुजात हुसैन और विदेश मंत्री ख़ुर्शीद महमूद कसूरी से भी मुलाक़ात की और दोनों देशों के बीच चल रही शां

In [131]:
manual_corrections.update({
    "040723_musharraf_natwar": """General Pervez Musharraf expressed this hope during his meeting with India's Foreign Minister Natwar Singh in Rawalpindi on Friday. Natwar Singh had travelled to Pakistan to attend a meeting of SAARC countries. He returned to India on Friday after meeting General Musharraf for about an hour and a half.

Before leaving for India, Natwar Singh said at Islamabad airport, "During this meeting, there was no issue, including Jammu and Kashmir, that we did not discuss." He added that they had discussed different aspects of various issues and had expressed satisfaction with the dialogue taking place at both official and unofficial levels.

Natwar Singh said that India and Pakistan were committed to discussing and resolving all issues, including Jammu and Kashmir, in a way that satisfied both countries.

During his visit, Natwar Singh also met Pakistan's Prime Minister Shujaat Hussain and Foreign Minister Khurshid Mahmud Kasuri. He reviewed the ongoing peace process between the two countries. Prime Minister Shujaat Hussain had said that Pakistan would make every possible effort to improve relations with India.

According to the news agency PTI, Musharraf had also said on Thursday night that there could be no concrete progress in improving relations with India until the Kashmir issue was resolved.

"We are working to take forward the dialogue with India and improve mutual understanding, but Kashmir is the main issue that needs to be resolved," Musharraf said. He added that until concrete progress was made on resolving this issue, there could be no substantial progress in improving mutual understanding in other areas.

Speaking at a military gathering in Islamabad on Thursday night, Pervez Musharraf said that Pakistan wanted a peaceful resolution of all issues with India, including Kashmir.""",

    "130312_traffic_noise_songbirds_vr": """Urban noise is affecting birds' lives too. A study by Canadian researchers found that birdsong is increasingly being drowned out by noise. Rising noise pollution has also affected birds' ability to communicate.

Researchers studying bird activity say that female birds may reject potential male partners if they cannot hear their calls clearly, possibly interpreting the males as being unwell.

The findings were published in the journal Global Change Biology.

Darren Proppe, one of the researchers, said that interest in protecting songbirds in urban areas had been growing. He added that this was also an issue of biodiversity, and that noise levels in these areas were known to be high.

Dr. Proppe, who was associated with the University of Alberta during the research, is currently working at Calvin College in the United States.

Cities are running short of suitable places for birds to live. Proppe said that some urban areas might appear to provide suitable habitats for songbirds, yet very few birds were actually found there.

The study focused on whether there was a relationship between the number of birds living in a city and the level of noise there. A research team visited 113 natural areas in Edmonton for the study.

Dr. Proppe said that the researchers found fewer songbirds in parts of the city where noise levels were higher.

The researchers explain that when female birds cannot hear low-frequency songs, they may begin to perceive male birds' calls as abnormal. Over time, this could affect pairing. If male birds cannot find female partners, their breeding populations may decline, eventually affecting the overall bird population."""
})

print("Total prepared manual corrections:", len(manual_corrections))
print("Newly added:", [
    "040723_musharraf_natwar",
    "130312_traffic_noise_songbirds_vr"
])

Total prepared manual corrections: 4
Newly added: ['040723_musharraf_natwar', '130312_traffic_noise_songbirds_vr']


In [132]:
reviewed_translations = corrected_article_translations.copy()

reviewed_translations["complete_english_translation"] = (
    reviewed_translations["id"]
    .map(manual_corrections)
    .fillna(reviewed_translations["complete_english_translation"])
)

reviewed_translations["manual_review_status"] = (
    reviewed_translations["id"].apply(
        lambda x: "Manually corrected"
        if x in manual_corrections
        else "Pending review"
    )
)

reviewed_path = (
    "/content/drive/MyDrive/FactIndic_PersonB_CleanRun/"
    "09_sentence_level_translation/"
    "FactIndic_English_250_Reviewed.csv"
)

reviewed_translations.to_csv(
    reviewed_path,
    index=False,
    encoding="utf-8-sig"
)

print("Total articles:", len(reviewed_translations))
print("Manually corrected:", (
    reviewed_translations["manual_review_status"] == "Manually corrected"
).sum())
print("Pending review:", (
    reviewed_translations["manual_review_status"] == "Pending review"
).sum())
print("Saved:", reviewed_path)

Total articles: 250
Manually corrected: 4
Pending review: 246
Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv


In [133]:
lowest_10 = final_audit.nsmallest(10, "translation_ratio")

for _, row in lowest_10.iterrows():
    print("\n" + "=" * 100)
    print("ARTICLE:", row["id"])
    print("Translation ratio:", round(row["translation_ratio"], 3))

    print("\nHINDI ORIGINAL:\n")
    print(row["text"])

    print("\nCURRENT REVIEWED ENGLISH:\n")
    current = reviewed_translations.loc[
        reviewed_translations["id"] == row["id"],
        "complete_english_translation"
    ].iloc[0]
    print(current)


ARTICLE: 121207_hangover_smoking_aa
Translation ratio: 0.7

HINDI ORIGINAL:

कई संस्थान इस अध्ययन को खासा अहम मान रहे हैं हालांकि इस शोध में ये नहीं बताया गया है कि धूम्रपान की वजह से हैंगोवर बिगड़ने की वजह क्या है. शोधकर्ताओं ने 113 अमरीकी छात्रों से कहा कि वो आठ हफ्तों तक पीने और धूम्रपान की अपनी आदतों से जुड़ी बातें और हैंगोवर के लक्षणों के बारे में डायरी में लिखें. जब उन्होंने जम की पी, मसलन एक घंटे में बीयर के छह कैन पी गए तो उस पर धूम्रपान करने वालों का हाल कहीं ज्यादा बुरा था. इस अध्ययन के नतीजे ‘जरनल ऑफ स्टडीज ऑन एल्होकल एंड ड्रग्स’ में प्रकाशित हुए हैं. 'जटिल संबंध' इस शोध पत्र को लिखने वालों में से एक डॉ. दामारिस रोहजेनॉव कहते हैं, “बराबर मात्रा में पीने वालों में से उन लोगों को अगले दिन हैंगोवर की ज्यादा संभावना होती है जो पीने के साथ धूम्रपान भी करते हैं.” “धूम्रपान न करने वालों के मुकाबले धूम्रपान करने वालों में हैंगोवर का जोखिम बढ़ जाता है.” नशे की लत को छुड़ाने के लिए काम कर रही संस्था 'एक्शन ऑन एडिक्शन' के प्रवक्ता ने इस बारे में और शोध किए जाने की जरूरत पर जोर दिया है

In [134]:
manual_corrections.update({

"130614_complaint_for_looks_rd": """The complainant met the woman in a hotel car park. A man telephoned West Midlands Police wanting to report a prostitute for allegedly violating the Sale of Goods Act. Police sent him a letter warning him not to waste police time.

The call was received at Solihull police station at 7:30 pm local time on Tuesday. According to police, the caller claimed that he had met the woman in a hotel car park.

A police spokesperson said, "The caller claimed that the woman had described herself as much more attractive than she really was, and he wanted to bring a case against her for violating the Sale of Goods Act."

The caller alleged that when he told the woman this, she took his car keys and ran away. She then threw the keys back towards him and told him to complain to the police.

A photograph showed prostitution advertising cards in a telephone box in London.

The man said, "I had arranged to meet the woman, but before doing so I had asked her to be honest about herself. I had told her that if she was not honest, I would not use her services. But she described herself completely inaccurately. She then became angry. Clearly, she thought I had taken responsibility for supporting her."

Sergeant Jerome Moran from the police station telephoned the man. He said, "It was unbelievable. The man genuinely believed that he had done nothing wrong and that the police should take action against the woman."

Sergeant Moran said, "I explained that the woman had committed no crime and that the man himself had acted unlawfully by soliciting sex."

Although the man refused to identify himself, police tracked him down and sent him a letter warning him about his behaviour.

England's Sale of Goods Act 1979 gives consumers the legal right to expect goods sold to them to be of satisfactory quality, fit for their intended purpose, and consistent with the description provided by the seller.

(Click here for the BBC Hindi Android app. You can also follow BBC Hindi on Facebook and Twitter.)""",

"031031_sharon_palestine": """By extending this invitation, Israel has changed its earlier position of refusing to hold talks with Qurei because of his close relationship with Palestinian leader Yasser Arafat. Israel and the United States regard Arafat as someone who supports terrorism and are unwilling to hold talks with him.

Sharon said that the Palestinians themselves had requested a delay in the meeting so that Prime Minister Ahmed Qurei, also known as Abu Ala, could gain some strength.

Sharon said, "We have maintained contact with the Palestinians, even though the discussions have not been held at the prime ministerial level." He added, "We are ready to begin talks at any time."

**Criticism and a new beginning**

However, the Israeli newspaper Haaretz quoted Palestinian cabinet minister Jamal al-Shobaki as denying that any proposal for a meeting between Sharon and Qurei had previously been made.

Al-Shobaki also accused Sharon of trying to divert attention from international criticism of Israel's tough policies towards the Palestinians.

Meanwhile, on Friday, Israel lifted some travel restrictions so that Palestinian lawmakers could attend a parliamentary meeting in Ramallah, a city in the West Bank.

During the session, Palestinian lawmakers were to vote on the confidence motion presented by Qurei concerning the new government. If Qurei won the vote, he would be able to replace the emergency cabinet with a full cabinet.""",

"050112_nepalmao_amnesty": """Amnesty International has asked Maoist leaders to investigate these cases and take action against those responsible.

In an open letter addressed to Prachanda, the leader of Nepal's Maoists, Amnesty said that it had received reports of specific human rights violations.

Ingrid Massage, the organisation's director for the Asia-Pacific region, said that Prachanda should ensure the safety of all civilians and release abducted people unconditionally.

Massage said that many people had been targeted because they had criticised the Maoists. She said that such actions were clear violations of international humanitarian law.

There have been reports of Maoist insurgents abducting large numbers of schoolchildren in Nepal and forcing them to attend lessons about Maoism for several days before releasing them. Some of these students are even persuaded or forced to join the insurgents.

Both the rebels and the government have previously faced criticism over human rights violations. Each side accuses the other of abuses but denies committing such violations itself.""",

"050812_baflights_cancel": """British Airways flights at Heathrow had been suspended since Thursday evening. The airline said that employees who had gone on strike were returning to work.

The crisis began when British Airways baggage-handling staff suddenly went on strike on Thursday evening. The airline's employees took this action in protest against the dismissal of workers employed by the company that provided catering services for British Airways flights.

As a result, 70,000 passengers travelling to and from British Airways destinations were affected. Hundreds of people had to spend Thursday night at Terminal 4 of the airport.

**Other airlines affected**

The doors of the affected terminal were closed to prevent overcrowding. As a result, many people also spent the night outside the terminal. British Airways arranged hotel accommodation for some passengers.

Chaos broke out at Heathrow, one of the world's largest airports outside the United States, after 600 employees of the company providing catering services to British Airways were dismissed. Some permanent British Airways employees also suddenly went on strike in support of them.

The striking British Airways employees included baggage handlers, staff responsible for loading and unloading luggage, and other ground staff.

The catering-company employees had been dismissed after they refused to work alongside temporary daily-wage workers.

No agreement was reached on Thursday, and British Airways announced that its flights would remain cancelled until at least Friday evening.

The action also affected five other airlines: SriLankan Airlines, Finnair, GB Airways, British Mediterranean Airways and Qantas. British Airways employees also provide airport services for these airlines.""",

"international-44515542": """Taliban spokesperson Zabihullah Mujahid said, "The ceasefire will end on Sunday night, and operations against the security forces will begin again."

The Taliban said that it had no intention of extending the ceasefire and expected its fighters to leave government-controlled areas by sunset. It did not mention the government's appeal for an extension.

The government and Taliban militants had announced a three-day ceasefire during Eid, which was due to end on 20 June.

**Government's appeal**

Considering the ceasefire a positive sign, the Afghan government extended its deadline by another ten days and appealed to the Taliban to do the same. However, the Taliban did not respond to the appeal.

According to reports from Afghanistan on Monday, after the ceasefire ended, the Taliban resumed attacks on Afghan security forces in Helmand and Kandahar provinces. Security forces were given permission to take necessary measures to defend themselves.

This was the first ceasefire since Taliban rule was overthrown in the US-led invasion of 2001.

President Ashraf Ghani had hoped that the Eid ceasefire would lead towards lasting peace. In February, he had agreed to unconditional peace talks and said that the Taliban would be recognised as a legitimate political group if it respected the law.

**Militants and security forces celebrated Eid together**

Earlier, Taliban militants and Afghan security forces had celebrated Eid together. Several photographs showing them greeting and embracing one another had emerged. People in many areas were surprised by these scenes and also expressed happiness.

Mohammad Amir, who lives in Kunduz, told Reuters, "I could not believe my eyes. I saw Taliban members and police officers standing together and taking selfies."

Qais Liwal, a student in Zabul in southern Afghanistan, said, "It was the most peaceful Eid we have ever experienced. For the first time, we felt safe. It is difficult to describe this happiness."

President Ashraf Ghani had appealed to the Taliban to extend the ceasefire.

During this period, a suicide attack in Jalalabad killed 18 people. The explosion took place outside the governor's office in Nangarhar province, where government officials were meeting Taliban militants.

Another suicide attack took place in Nangarhar on Saturday, killing 36 people. This attack also occurred during a meeting between militants and local officials. The Islamic State group claimed responsibility.

(You can click here for the BBC Hindi Android app. You can also follow us on Facebook and Twitter.)""",

"140223_modi_rally_punjab_sr": """BJP's prime ministerial candidate Narendra Modi addressed a public meeting in Jagraon, Ludhiana, in Punjab.

Targeting the Congress, he said that corruption had become associated with the party through an ABCD sequence: A for the Adarsh scam, B for the Bofors scam and C for the coal scam.

Modi said, "When Rajiv Gandhi was ruling the country, his influence extended from the village councils to Parliament. Rajiv Gandhi had said that when one rupee left Delhi, only 15 paise reached the village. I want to ask: what was that hand that wore down the rupee and reduced it to 15 paise?"

He said that if he were given responsibility for governing Delhi, he would act like a watchman and would not allow anyone to take money from the country's treasury.

**How will Narendra Modi deal with these challenges?**

**Politics of blood**

Meanwhile, Congress vice-president Rahul Gandhi addressed a rally in Dehradun, the capital of Uttarakhand. He said that the opposition practised the politics of blood.

He said that when disaster struck Uttarakhand, the entire state stood together.

Rahul said, "The whole of India is united. Only a select few are trying to divide the country and make people fight one another."

Rahul also said that, following his request, the number of subsidised cooking-gas cylinders had been increased from nine to twelve.

He also mentioned the fulfilment of former soldiers' demand for 'One Rank, One Pension'. Uttarakhand has a large number of former servicemen.

**There is the most poison within the Congress: Modi**

**'We will secure farmers' rights'**

Aam Aadmi Party leader Arvind Kejriwal addressed a public meeting in Rohtak, Haryana, where he strongly criticised BJP leader Narendra Modi and Congress vice-president Rahul Gandhi.

Kejriwal said, "I asked Narendra Modi and Rahul Gandhi whose helicopters they use, and whether they pay rent for them or use them free of charge."

He said that if the Aam Aadmi Party came to power, it would secure farmers' rights.

Kejriwal said, "They say that we do not know how to run a government. I challenge them: if anyone else has achieved what we have achieved, I will leave politics."

(Click here to download the BBC Hindi Android mobile app. You can also visit our Facebook page to read the news and share your views, and follow us on Twitter.)"""
})

print("Total prepared manual corrections:", len(manual_corrections))
print("Newly added corrections:", 6)
print("Total articles covered:", len(manual_corrections))

Total prepared manual corrections: 10
Newly added corrections: 6
Total articles covered: 10


In [135]:
reviewed_translations = corrected_article_translations.copy()

reviewed_translations["complete_english_translation"] = (
    reviewed_translations["id"]
    .map(manual_corrections)
    .fillna(reviewed_translations["complete_english_translation"])
)

reviewed_translations["manual_review_status"] = (
    reviewed_translations["id"].apply(
        lambda x: "Manually corrected"
        if x in manual_corrections
        else "Pending review"
    )
)

reviewed_path = (
    "/content/drive/MyDrive/FactIndic_PersonB_CleanRun/"
    "09_sentence_level_translation/"
    "FactIndic_English_250_Reviewed.csv"
)

reviewed_translations.to_csv(
    reviewed_path,
    index=False,
    encoding="utf-8-sig"
)

print("Total articles:", len(reviewed_translations))
print("Unique IDs:", reviewed_translations["id"].nunique())
print("Manually corrected:", (
    reviewed_translations["manual_review_status"] == "Manually corrected"
).sum())
print("Pending review:", (
    reviewed_translations["manual_review_status"] == "Pending review"
).sum())
print("Saved:", reviewed_path)

Total articles: 250
Unique IDs: 250
Manually corrected: 10
Pending review: 240
Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv


In [136]:
next_10 = final_audit.nsmallest(20, "translation_ratio").iloc[10:20]

for _, row in next_10.iterrows():
    current = reviewed_translations.loc[
        reviewed_translations["id"] == row["id"],
        "complete_english_translation"
    ].iloc[0]

    print("\n" + "=" * 100)
    print("ARTICLE:", row["id"])
    print("Translation ratio:", round(row["translation_ratio"], 3))
    print("\nHINDI ORIGINAL:\n", row["text"])
    print("\nCURRENT ENGLISH:\n", current)


ARTICLE: 111113_berlusconi_resigned_vv
Translation ratio: 0.768

HINDI ORIGINAL:
 बर्लुस्कोनी पर लंबे समय से इस्तीफ़ा देने का दबाव था रोम में अपने निवास से निकलकर जब वे पास ही स्थित राष्ट्रपति भवन तक इस्तीफ़ा देने गए तो लोगों ने उनके ख़िलाफ़ नारे लगाए और उन्हें अपशब्द कहे. संवाददाताओं का कहना है कि लोगों ने 'डाकुओं का सरगना' और 'मसखरा' कहकर उनके ख़िलाफ़ नारे लगाए शनिवार को संसद के निचले सदन में सरकारी ख़र्चों में कटौती के प्रस्ताव के पारित होने के बाद बर्लुस्कोनी का इस्तीफ़ा औपचारिकता मात्र रह गया था क्योंकि गुरुवार को बहुमत हारने के बाद वे घोषणा कर चुके थे कि अगर कटौती प्रस्ताव को संसद के दोनों सदन पारित कर देते हैं तो वे अपना पद छोड़ देंगे. राष्ट्रपति जॉर्जियो नैपॉलिटैनो ने उनका इस्तीफ़ा स्वीकार कर लिया है लेकिन नई सरकार के गठन तक पद संभालने का अनुरोध किया है. संभावना है कि अब राष्ट्रपति यूरोपीय आयोग के पूर्व कमिश्नर मारियो मोंटी से नई सरकार के गठन करने को कहेंगे. लेकिन निवर्तमान सरकार चलाने वाला गठबंधन किसी भी ऐसी सरकार का ज़ोरदार ढंग से विरोध कर रहा है जो चुनी हुई न हो. विवादों से

In [137]:
manual_corrections.update({

"111113_berlusconi_resigned_vv": """Berlusconi had been under pressure to resign for a long time. When he left his residence in Rome and went to the nearby presidential palace to submit his resignation, people shouted slogans against him and insulted him. According to reporters, people called him the "leader of a gang of robbers" and a "clown".

After the lower house of parliament passed the proposal to cut government spending on Saturday, Berlusconi's resignation became little more than a formality. On Thursday, after losing his majority, he had already announced that he would step down if both houses of parliament passed the spending-cut proposal.

President Giorgio Napolitano has accepted his resignation but asked him to remain in office until a new government is formed. The president is expected to ask former European Commission commissioner Mario Monti to form the new government. However, the coalition that currently runs the government strongly opposes any government that has not been elected.

**A controversial tenure**

His sex scandals attracted considerable attention and severely damaged his reputation. BBC correspondent Allan Johnston in Rome says that Berlusconi's final period in office further harmed his reputation.

Berlusconi was Italy's longest-serving prime minister since the Second World War. However, he had recently been involved in several controversies. The 75-year-old Berlusconi was known for his ability to escape difficult situations, but this time he became caught up in a financial crisis that had brought Italy close to collapse.

He won elections in 1994, 2001 and 2008. However, after 2008, he faced challenges in remaining in power, including having to win 51 confidence votes. His party also lost elections in 1996 and 2006. Several legal cases against him are still pending.

**The new government**

The outgoing coalition strongly opposes the appointment of Mario Monti as prime minister. Discussions on forming a new government are expected to begin on Sunday.

Mario Monti is a respected economist and is the kind of person the markets would like to see leading the country at this time. He has international support. However, his strongest opposition is within Italy itself. The country's problems appear to have deeper roots and cannot be solved simply by changing the government.

**The economic challenge**

On Wednesday, the interest rate on Italian government bonds reached seven per cent. It began to fall after the spending-cut proposal was passed by the upper house of parliament on Friday.

Earlier, when interest rates reached seven per cent in Greece, Ireland and Portugal, eurozone leaders had pressured those countries to accept external assistance to put their economies back on track.

On Thursday, the Italian government raised five billion euros through government bonds. However, it will have to pay a very high interest rate of 6.087 per cent on this amount.

A European Union team is monitoring the government's efforts in Rome to reduce the country's debt burden, which has reached 120 per cent of its gross domestic product.

Over the past 15 years, Italy's economy has grown at an average rate of just 0.75 per cent. The spending-cut proposal passed on Friday is expected to generate savings of 59.8 billion euros through reductions in government spending and increases in taxes. This is intended to balance the budget by 2014.

The markets welcomed the proposal after it was passed, and share prices rose.""",

"070620_heart_survival": """A study of 1,676 patients conducted by researchers from Germany and Switzerland found that, during the three years following treatment, the number of deaths among obese patients was half the number among patients of normal weight.

However, the study, published in the European Heart Journal, did not explain why obese patients had a lower risk of death. Experts also pointed out that obese people are more likely to develop heart disease.

The study involved 1,676 patients who had been admitted to hospital with unstable angina or a heart attack.

Lead researcher Dr Heinz Büttner said, "There is no doubt that obese people are more likely to develop diabetes, high blood pressure and coronary artery disease. Our study showed that, once coronary artery disease has been treated successfully, obese patients are more likely to remain healthy than patients of normal weight."

**Possible explanations**

Compared with people of normal weight, obese patients tend to be younger and may be discharged from hospital with certain medicines for heart disease. However, Dr Büttner said that his research could not establish the exact reason for the lower death rate.

He suggested that changes in the body caused by obesity might be responsible. Obese people have more body fat, which may produce a protective effect. They also tend to have higher platelet levels, and platelets play an important role in blood clotting.

June Davison of the British Heart Foundation, an organisation that studies heart disease in the UK, said, "This research raises more questions than it answers. More research is needed in this area.""",

"050107_pravasi_pm": """While inaugurating the Pravasi Bharatiya Divas convention in Mumbai, Prime Minister Manmohan Singh said, "I announce that all Indians who settled abroad after 1950 will be eligible for dual citizenship, provided the laws of the country where they live allow it."

He said that the paperwork required for this process was also being simplified.

Manmohan Singh said that the government was considering ways to improve the registration process for overseas Indians in India. He said that all available options, including smart cards, were being studied.

It is worth noting that last year the Indian Parliament approved dual citizenship for overseas Indians from 16 countries, including the United States and the United Kingdom. Following Prime Minister Manmohan Singh's announcement on Friday, this facility would now be available to all overseas Indians.

According to one estimate, more than 25 million Indians live abroad. The Indian government wants to increase the amount of investment made in the country by overseas Indians. It is believed that granting dual citizenship to more overseas Indians could make this easier.

More than 1,500 overseas Indians from 60 countries are taking part in the Pravasi Bharatiya Divas convention. The convention will last for three days.

The convention will also include detailed discussions about the devastation caused by the tsunami waves in India in recent days.""",

"071227_musharraf_statement": """Musharraf has said that terrorism is the biggest obstacle to Pakistan's progress.

In a televised statement, he said that he would not rest until terrorists had been eliminated from the country. He appealed to the people of Pakistan to support efforts to end terrorism.

He said, "Terrorism is the biggest obstacle to Pakistan's progress, and we are trying to uproot it completely."

He also appealed to the people of Pakistan to maintain peace and calm. He asked them to remain patient so that panic would not spread during this difficult time.

A three-day period of national mourning has been declared in Pakistan.""",

"151118_islamic_state_claims_kills_hostages_hk": """One of the hostages was reportedly from Norway, while the other was from China. According to the Islamic State group, the governments of Norway and China did not pay ransom for the hostages, and the two were subsequently killed.

However, the BBC has not been able to independently verify the group's claim.

The Islamic State released photographs of the two hostages as evidence. One photograph showed them alive, while another showed their bodies. The photographs appeared to indicate that they had been shot.

(Click here for the BBC Hindi Android app. You can also follow us on Facebook and Twitter.)"""
})

print("Total prepared manual corrections:", len(manual_corrections))
print("New corrections in this batch: 5")

Total prepared manual corrections: 15
New corrections in this batch: 5


In [138]:
next_5 = final_audit.nsmallest(20, "translation_ratio").iloc[15:20]

for _, row in next_5.iterrows():
    current = reviewed_translations.loc[
        reviewed_translations["id"] == row["id"],
        "complete_english_translation"
    ].iloc[0]

    print("\n" + "=" * 100)
    print("ARTICLE:", row["id"])
    print("Translation ratio:", round(row["translation_ratio"], 3))
    print("\nHINDI ORIGINAL:\n", row["text"])
    print("\nCURRENT ENGLISH:\n", current)


ARTICLE: 140323_breast_cancer_screening_tk
Translation ratio: 0.78

HINDI ORIGINAL:
 साल 2009 और 2013 के बीच 53,467 महिलाओं पर किए गए अध्ययन के बाद यह निष्कर्ष निकाला गया है. वैज्ञानिकों ने इनमें से 14,593 महिलाओं में स्तन कैंसर का 'औसत से अधिक' जोखिम पाया है. वे उम्मीद करते हैं, इस अध्ययन से महिलाओं में जागरूकता आएगी और वे अपनी जीवनशैली में बदलाव लाएंगी जिससे स्तन कैंसर के मामले कम करने में मदद मिलेगी. अध्ययन में कहा गया है कि औसत से अधिक जोखिम का मतलब है कि ऐसे मामलों में अगले दस साल में स्तन कैंसर होने की 3.5% संभावना है. अभी 50 से 70 साल की महिलाओं को तीन साल में एक बार मैमोग्राम की सलाह दी जाती है ताकि शुरूआती चरण में ही कैंसर का पता चल पाए और उपचार के असरदार होने की संभावना बढ़ जाए. तकलीफ़ प्रमुख शोधकर्ता मैनचेस्टर विश्वविद्यालय के प्रोफ़ेसर गैरेथ इवांस ने कहा, "रोकथाम न केवल महिलाओं को कैंसर की तकलीफ़ से बचाएगा बल्कि उपचार पर होने वाले 'अधिक ख़र्च' से भी बचाएगा. वैज्ञानिकों ने परिवार के इतिहास, जीवन शैली के रूप में जोखिम कारकों का आकलन और लार से आनुवंशिक जानकारी एकत्र कर स्तन क

In [139]:
manual_corrections.update({

"140323_breast_cancer_screening_tk": """The findings are based on a study of 53,467 women conducted between 2009 and 2013. Scientists found that 14,593 of these women had an above-average risk of developing breast cancer.

The researchers hope that the study will raise awareness among women and encourage lifestyle changes that could help reduce the number of breast cancer cases.

According to the study, an above-average risk means a 3.5 per cent chance of developing breast cancer over the next ten years.

Currently, women aged between 50 and 70 are advised to have a mammogram once every three years. This helps detect cancer at an early stage and increases the chances of effective treatment.

**Reducing the risk**

Lead researcher Professor Gareth Evans of the University of Manchester said, "Prevention will not only protect women from the suffering caused by cancer, but will also save them from the high costs of treatment."

The scientists assessed the risk of breast cancer by examining factors such as family history and lifestyle, and by collecting genetic information from saliva samples. They also measured breast tissue density through mammograms and visual assessments, as tissue density can be an indicator of breast cancer risk.

Lifestyle changes such as exercising regularly, losing weight and reducing alcohol consumption can also reduce the risk of breast cancer by up to 30 per cent.

Professor Evans said, "These changes will put women's destiny in their own hands."

**A new approach**

The scientists found an above-average risk of breast cancer in 14,593 women.

An independent review of the effectiveness of breast cancer screening was published in 2012. It led to considerable debate about the concept of overtreatment.

Overtreatment occurs when screening detects a tumour that is not harmful. This can result in women undergoing unnecessary treatments such as surgery, hormone therapy, radiotherapy and chemotherapy, which often have significant side effects.

Professor Michael Marmot of University College London said, "This study could be helpful in this regard."

He explained that women at a higher risk of breast cancer were more likely to develop serious symptoms of the disease, making treatment more relevant in such cases.

However, he said that it was important to assess the significance of having screening intervals of one or two years in relation to the development of cancer.

(Click here for the BBC Hindi Android app. You can also follow us on Facebook and Twitter.)""",

"science-41399809": """Doctors have raised serious concerns about liposuction. They say that the procedure can cause fat globules to enter the lungs, a condition that can lead to serious complications.

The doctors made these observations after examining the case of a 45-year-old woman. She developed breathing difficulties after undergoing fat-removal treatment on her knees and legs.

Doctors in the British city of Birmingham said that her condition had become so critical that she had nearly died.

**Liposuction can also be life-threatening**

According to a report in the BMJ, the case highlights a serious potential risk associated with liposuction.

Dr Adam Ali, from the intensive care unit at Sandwell and West Birmingham Hospitals, said, "There had previously been no reported case of fat embolism syndrome (FES) in the UK, but the possibility of it occurring needs to be considered."

The patient in this case was severely obese and had undergone gastric surgery before her liposuction procedure. Around 36 hours after the operation, her condition became critical and she developed breathing difficulties. She was subsequently transferred to the intensive care unit.

**What is liposuction?**

Doctors saved the woman's life by providing immediate treatment, and she was discharged from hospital three weeks later.

The report says that recognising and treating FES can be challenging for doctors because its symptoms are often common and non-specific.

Liposuction is a medical procedure in which doctors remove fat from particular areas of the body, such as the thighs, hips and abdomen.

The procedure is generally considered most suitable for people who have tight skin and are of normal weight. Its results can last for a considerable period.

However, concerns about its side effects have raised questions about the possibility of harmful outcomes.

Other related headlines include questions about whether obesity affects memory, whether bones are responsible for obesity, and whether obesity can be as dangerous as smoking.

(Click here for the BBC Hindi Android app. You can also follow us on Facebook and Twitter.)""",

"061015_katsav_police": """Israeli police had been investigating the case for several weeks and had informed Attorney General Menachem Mazuz of their findings and recommendations.

The final decision on whether President Moshe Katsav should face trial now rests with the attorney general.

President Katsav denies allegations that he forced two female employees to have sexual relations with him. He has also denied allegations of phone tapping and fraud.

If Attorney General Menachem Mazuz decides that the case should proceed to trial, President Katsav may have to resign from his position.

The allegations against the president have shaken Israeli politics and increased public distrust of a political system that was already facing widespread disappointment.

The case began when a woman who worked for the president accused him of forcing her to have sexual relations with him. Several other women later made similar allegations. According to the media, at least ten women have filed complaints.

President Katsav has denied the allegations, saying that the situation amounts to punishing him publicly without giving him a trial.

The 61-year-old Katsav is a member of the right-wing Likud Party and has held the presidency since 2000. He is the father of five children and has refused to resign.

Moshe Katsav is due to open a new session of parliament on Monday. However, several lawmakers have threatened to walk out if he attends.

There is considerable public dissatisfaction over corruption cases involving Israeli politicians and the military's recent performance in Lebanon.""",

"131010_punjab_accident_truck_ssr": """**Truck accident in Punjab caused by brake failure**

(File photo: Hindu devotees were travelling in the mini-truck.)

Police officer Sukhchain Singh Gill said, "The accident occurred when the truck driver lost control of the vehicle after its brakes failed."

The accident took place near Manguwal, on the Chintpurni road, about 22 kilometres from Hoshiarpur. The vehicle went out of control at a sharp bend beside a deep gorge.

According to Singh, the driver was among those injured in the accident in Hoshiarpur, which is about 380 kilometres from New Delhi.

**Around 60 devotees were travelling in the truck**

Around 60 Hindu devotees were travelling in the mini-truck. They were going from Chintpurni to Kapurthala.

As the vehicle approached the sharp bend near Manguwal, the driver lost control and the accident occurred.

The accident took place at around 11:30 pm. Rescue operations continued for several hours afterwards.

The injured people were taken to a local hospital. However, all of them were reported to be out of danger.

India's roads are among the most dangerous in the world. Around 110,000 people are killed in road accidents every year.

Reckless driving, poor road conditions and old vehicles are considered responsible for most accidents.

(Click here to download the BBC Hindi Android mobile app. You can also visit our Facebook page to read the news and share your views, and follow us on Twitter.)""",

"120816_rachna_online_chat_aa": """Rachana advises young people to stay focused on their goals.

During an online chat, readers asked Rachana a variety of questions on the BBC Hindi Facebook page.

In response to a question from BBC Hindi reader Prabhat Kumar Jha, Rachana said, "This package is in Indian rupees, whereas my job will be in California, in the United States. Taxes and living expenses are also higher there. So, considering those factors, it is not really that much money. Besides, if they are paying me this much, they will also expect me to do more work."

Kuldeep Kumar asked Rachana, who was studying engineering at the Indian School of Mines (ISM) in Dhanbad, whether she would live in the United States after joining Facebook.

She replied, "I have only received the offer so far. Nothing like that has been mentioned in it. What happens later will depend on my work and the company's conditions. If I like it, I will stay; otherwise, I will return."

**Advice for young people**

In the nearly 86-year history of ISM, this was the first time that a student of the institute had received a job offer with such a high salary.

BBC Hindi reader Khushbu Kumari asked Rachana what changes she would like to make after joining Facebook.

Rachana replied, "The first thing I would change is the speed of chatting. The chat service we use is slower than those on other websites, so it needs to become faster. Second, all users should be able to use a dislike button. Improving the security of personal information and making the service more convenient are also important. We will also work towards making it available in Indian languages."

Rachana answered many similar questions during the BBC Hindi online chat and received numerous congratulatory messages.

In response to a question from Lalit Negi, Rachana said, "My advice to young people is that if you want to achieve something, the most important thing is to work very hard. Even if you are not studying at a very good college, do not reduce your efforts. Keep working hard and stay focused on your goals."

(See more related articles.)"""
})

print("Total prepared manual corrections:", len(manual_corrections))
print("New corrections in this batch: 5")

Total prepared manual corrections: 20
New corrections in this batch: 5


In [140]:
# Rebuild the reviewed dataset using all 15 manual corrections
reviewed_translations = corrected_article_translations.copy()

reviewed_translations["complete_english_translation"] = (
    reviewed_translations["id"]
    .map(manual_corrections)
    .fillna(reviewed_translations["complete_english_translation"])
)

reviewed_translations["manual_review_status"] = (
    reviewed_translations["id"]
    .apply(lambda x: "Manually corrected" if x in manual_corrections else "Pending review")
)

# Save
reviewed_path = (
    "/content/drive/MyDrive/FactIndic_PersonB_CleanRun/"
    "09_sentence_level_translation/FactIndic_English_250_Reviewed.csv"
)

reviewed_translations.to_csv(reviewed_path, index=False)

print("Saved successfully:", reviewed_path)
print("\nTotal articles:", len(reviewed_translations))
print("Unique articles:", reviewed_translations["id"].nunique())
print("\nReview status:")
print(reviewed_translations["manual_review_status"].value_counts())

Saved successfully: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv

Total articles: 250
Unique articles: 250

Review status:
manual_review_status
Pending review        230
Manually corrected     20
Name: count, dtype: int64


In [141]:
import pandas as pd

hindi_df = df_clean.copy()
english_df = reviewed_translations[
    ["id", "complete_english_translation"]
].copy()

factindic_base = hindi_df.merge(
    english_df,
    on="id",
    how="inner",
    validate="one_to_one"
)

print("Combined articles:", len(factindic_base))
print("Missing English translations:",
      factindic_base["complete_english_translation"].isna().sum())
print("Columns:", factindic_base.columns.tolist())

Combined articles: 250
Missing English translations: 0
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words', 'complete_english_translation']


In [142]:
print("Hindi article sample:")
print(factindic_base.loc[0, "text"][:500])

print("\nReviewed English translation sample:")
print(factindic_base.loc[0, "complete_english_translation"][:500])

Hindi article sample:
शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहान

Reviewed English translation sample:
The festival, which began on Friday evening, screened the film Insaaf Ka Scale. Classic films like Chhoti Si Baat, Naya Daur, and Saam will be seen during the festival. BR Chopra himself could not attend the ceremony due to ill health. The film festival was inaugurated by BR Chopra's brother and noted filmmaker Yash Chopra. He said, "My brother made his own films - that too at a time when people were apprehensive about the commercial s

In [143]:
article_id = "080419_brchopra_festival"

mask = reviewed_translations["id"] == article_id

reviewed_translations.loc[mask, "complete_english_translation"] = (
    reviewed_translations.loc[mask, "complete_english_translation"]
    .str.replace("Insaaf Ka Scale", "Insaf Ka Tarazu", regex=False)
    .str.replace("Saam", "Waqt", regex=False)
)

reviewed_translations.to_csv(reviewed_path, index=False)

factindic_base.loc[
    factindic_base["id"] == article_id,
    "complete_english_translation"
] = reviewed_translations.loc[
    mask, "complete_english_translation"
].values

print("Film-title corrections saved.")
print(factindic_base.loc[mask.values, "complete_english_translation"].iloc[0][:500])

Film-title corrections saved.
The UN Secretary-General has said the attacks are "shocking." Britain has said this makes it clear that extremists have no regard for human life. At the same time, France has said that such activities cannot be justified under any circumstances. There is no official word on the number of people killed in the blasts, but 60 people are reported dead and 200 injured. A high alert has been sounded following the blasts. The government has advised people to stay at home. Police are stationed outside t


In [144]:
import inspect

print("Phrase dictionary available:", "final_candidate_phrases" in globals())
print("Mixing function available:", "create_better_mixed" in globals())

if "final_candidate_phrases" in globals():
    print("Phrase dictionary size:", len(final_candidate_phrases))
    print("Sample entries:", list(final_candidate_phrases.items())[:10])

if "create_better_mixed" in globals():
    print("\nMixing function:")
    print(inspect.signature(create_better_mixed))
    print(inspect.getsource(create_better_mixed))

Phrase dictionary available: False
Mixing function available: False


In [145]:
import os

root = "/content/drive/MyDrive/FactIndic_PersonB_CleanRun"

for folder, subfolders, files in os.walk(root):
    for file in files:
        if any(word in file.lower() for word in ["phrase", "dictionary", "mixed"]):
            print(os.path.join(folder, file))

/content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/FactIndic_Mixed_250_partial.csv
/content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/FactIndic_Mixed_250.csv
/content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/hindi_english_phrase_dictionary.json
/content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_checkpoints/failed_experiments/FactIndic_Mixed_250.csv


In [146]:
import json
import pandas as pd
import os

dictionary_path = (
    "/content/drive/MyDrive/FactIndic_PersonB_CleanRun/"
    "04_mixed_data/hindi_english_phrase_dictionary.json"
)

with open(dictionary_path, "r", encoding="utf-8") as f:
    phrase_dictionary = json.load(f)

print("Dictionary entries:", len(phrase_dictionary))
print("\nSample entries:")
for hindi, english in list(phrase_dictionary.items())[:15]:
    print(hindi, "→", english)

Dictionary entries: 26

Sample entries:
इंसाफ़ का तराज़ू → Insaf Ka Tarazu
छोटी सी बात → Chhoti Si Baat
नया दौर → Naya Daur
वक़्त → Waqt
साधना → Sadhana
गुमराह → Gumrah
हमराज़ → Humraaz
क़ानून → Kanoon
निकाह → Nikaah
बीआर चोपड़ा → B. R. Chopra
यश चोपड़ा → Yash Chopra
रवि चोपड़ा → Ravi Chopra
राज बब्बर → Raj Babbar
बसु चटर्जी → Basu Chatterjee
बॉबी बेदी → Bobby Bedi


In [147]:
mixed_path = (
    "/content/drive/MyDrive/FactIndic_PersonB_CleanRun/"
    "04_mixed_data/FactIndic_Mixed_250.csv"
)

old_mixed = pd.read_csv(mixed_path)

print("Rows:", len(old_mixed))
print("\nColumns:")
print(old_mixed.columns.tolist())

print("\nFirst row preview:")
for col in old_mixed.columns:
    if "article" in col.lower():
        print(f"\n{col}:")
        print(str(old_mixed.iloc[0][col])[:300])

Rows: 250

Columns:
['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words', 'english_title', 'english_text', 'english_summary', 'mixed_10_article', 'mixed_30_article', 'mixed_50_article', 'mixed_10_article_english_pct', 'mixed_30_article_english_pct', 'mixed_50_article_english_pct']

First row preview:

article_words:
294

mixed_10_article:
शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया.

mixed_30_article:
शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया.

mixed_50_articl

In [148]:
print("Old English translation:")
print(str(old_mixed.loc[0, "english_text"])[:500])

print("\nNew reviewed English translation:")
print(str(
    reviewed_translations.loc[
        reviewed_translations["id"] == old_mixed.loc[0, "id"],
        "complete_english_translation"
    ].iloc[0]
)[:500])

Old English translation:
On Friday evening, the scales of justice were shown in the Polish film. The little thing, the new rounds, and time during the celebration, will be seen to see the classical films. Bad health because of BRC's own function, BRCP's brother and famous filmer, he said, "My brother made his business at such a time. When people were apprehensive about the commercial success of such films because they were few, the stories of BRangangange's life were different from them. BR'Courourin raised many subject

New reviewed English translation:
The festival, which began on Friday evening, screened the film Insaf Ka Tarazu. Classic films like Chhoti Si Baat, Naya Daur, and Waqt will be seen during the festival. BR Chopra himself could not attend the ceremony due to ill health. The film festival was inaugurated by BR Chopra's brother and noted filmmaker Yash Chopra. He said, "My brother made his own films - that too at a time when people were apprehensive about the commercial s

In [149]:
sample_text = factindic_base.loc[
    factindic_base["id"] == "080419_brchopra_festival",
    "complete_english_translation"
].iloc[0]

for hindi, english in phrase_dictionary.items():
    if english.lower() in sample_text.lower():
        print(f"{hindi} → {english}")

इंसाफ़ का तराज़ू → Insaf Ka Tarazu
छोटी सी बात → Chhoti Si Baat
नया दौर → Naya Daur
वक़्त → Waqt
साधना → Sadhana
गुमराह → Gumrah
क़ानून → Kanoon
यश चोपड़ा → Yash Chopra
रवि चोपड़ा → Ravi Chopra
राज बब्बर → Raj Babbar
बसु चटर्जी → Basu Chatterjee
बॉबी बेदी → Bobby Bedi


In [150]:
import re

def replace_hindi_phrases(text, dictionary):
    result = str(text)

    # Replace longer phrases first to avoid partial replacements
    phrases = sorted(dictionary.items(), key=lambda x: len(x[0]), reverse=True)

    for hindi, english in phrases:
        result = result.replace(hindi, english)

    return result

test_hindi = factindic_base.loc[
    factindic_base["id"] == "080419_brchopra_festival",
    "text"
].iloc[0]

test_mixed = replace_hindi_phrases(test_hindi, phrase_dictionary)

print("ORIGINAL HINDI:")
print(test_hindi[:500])

print("\nPHRASE-REPLACED VERSION:")
print(test_mixed[:500])

ORIGINAL HINDI:
शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहान

PHRASE-REPLACED VERSION:
शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में Insaf Ka Tarazu फ़िल्म दिखाई गई. उत्सव के दौरान Chhoti Si Baat, Naya Daur और Waqt जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण B. R. Chopra ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन B. R. Chopra के भाई और मशहूर फ़िल्मकार Yash Chopra ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये 

In [151]:
import re

hindi_sentences = [
    s.strip() for s in re.split(r'[।!?]+', test_hindi) if s.strip()
]

english_sentences = [
    s.strip() for s in re.split(r'(?<=[.!?])\s+', sample_text) if s.strip()
]

print("Hindi sentence count:", len(hindi_sentences))
print("English sentence count:", len(english_sentences))

print("\nFirst 3 Hindi sentences:")
for s in hindi_sentences[:3]:
    print("-", s)

print("\nFirst 3 English sentences:")
for s in english_sentences[:3]:
    print("-", s)

Hindi sentence count: 1
English sentence count: 14

First 3 Hindi sentences:
- शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी." बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है. गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में शामिल है. हालांकि बीआर चोपड

In [152]:
import re

def split_hindi_sentences(text):
    # Hindi news text may use both danda and normal full stops
    parts = re.split(r'[।.!?]+', str(text))
    return [s.strip() for s in parts if s.strip()]

def split_english_sentences(text):
    parts = re.split(r'(?<=[.!?])\s+', str(text))
    return [s.strip() for s in parts if s.strip()]

hindi_sentences = split_hindi_sentences(test_hindi)
english_sentences = split_english_sentences(sample_text)

print("Hindi sentence count:", len(hindi_sentences))
print("English sentence count:", len(english_sentences))

print("\nFirst 5 Hindi sentences:")
for s in hindi_sentences[:5]:
    print("-", s)

print("\nFirst 5 English sentences:")
for s in english_sentences[:5]:
    print("-", s)

Hindi sentence count: 15
English sentence count: 14

First 5 Hindi sentences:
- शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई
- उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी
- खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके
- फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया
- उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी

First 5 English sentences:
- The festival, which began on Friday evening, screened the film Insaf Ka Tarazu.
- Classic films like Chhoti Si Baat, Naya Daur, and Waqt will be seen during the festival.
- BR Chopra himself could not attend the ceremony due to ill health.
- The film festival was inaugurated by BR Chopra's brother and noted filmmaker Yash 

In [153]:
hindi_words = len(test_hindi.split())
english_words = len(sample_text.split())

print("Hindi words:", hindi_words)
print("English words:", english_words)
print("English words available:", english_words)
print("Target English words at 10%:", round(hindi_words * 0.10))
print("Target English words at 30%:", round(hindi_words * 0.30))
print("Target English words at 50%:", round(hindi_words * 0.50))

Hindi words: 294
English words: 287
English words available: 287
Target English words at 10%: 29
Target English words at 30%: 88
Target English words at 50%: 147


In [154]:
print("Hindi sentences:", len(hindi_sentences))
print("English sentences:", len(english_sentences))

for i, sentence in enumerate(english_sentences, 1):
    print(f"\n{i}. {len(sentence.split())} words")
    print(sentence[:180])

Hindi sentences: 15
English sentences: 14

1. 13 words
The festival, which began on Friday evening, screened the film Insaf Ka Tarazu.

2. 16 words
Classic films like Chhoti Si Baat, Naya Daur, and Waqt will be seen during the festival.

3. 12 words
BR Chopra himself could not attend the ceremony due to ill health.

4. 14 words
The film festival was inaugurated by BR Chopra's brother and noted filmmaker Yash Chopra.

5. 63 words
He said, "My brother made his own films - that too at a time when people were apprehensive about the commercial success of such films because these films were different, the kind o

6. 27 words
Memorable films His film Naya Daur in 1957 reflects the ideals of the Nehru era while Sadhana in 1958 questions the society on the issue of prostitution.

7. 17 words
Films such as Gumrah, Hamraaz, Kanoon and Nikah feature on the list of films he has produced.

8. 25 words
Although most of BR Chopra's films were based on some social issue or the other, these experimental

In [158]:
columns = [
    "mixed_10_article_english_pct",
    "mixed_30_article_english_pct",
    "mixed_50_article_english_pct"
]

print("Old percentage statistics:")
print(old_mixed[columns].describe())

print("\nAverage mixed article word counts:")

for col in ["mixed_10_article", "mixed_30_article", "mixed_50_article"]:
    avg = old_mixed[col].fillna("").str.split().str.len().mean()
    print(col, ":", round(avg, 2))

Old percentage statistics:
       mixed_10_article_english_pct  mixed_30_article_english_pct  \
count                    250.000000                    250.000000   
mean                       2.787360                      2.787360   
std                        1.910018                      1.910018   
min                        0.000000                      0.000000   
25%                        1.250000                      1.250000   
50%                        2.570000                      2.570000   
75%                        3.880000                      3.880000   
max                        9.950000                      9.950000   

       mixed_50_article_english_pct  
count                    250.000000  
mean                       2.787360  
std                        1.910018  
min                        0.000000  
25%                        1.250000  
50%                        2.570000  
75%                        3.880000  
max                        9.950000  

Average 

In [159]:
import pandas as pd
import re

# Combine Hindi originals and reviewed English translations
translation_audit = df_clean[
    ["id", "text"]
].merge(
    reviewed_translations[
        ["id", "complete_english_translation", "manual_review_status"]
    ],
    on="id",
    how="left",
    validate="one_to_one"
)

# Calculate word counts
translation_audit["hindi_words"] = (
    translation_audit["text"].fillna("").str.split().str.len()
)

translation_audit["english_words"] = (
    translation_audit["complete_english_translation"]
    .fillna("").str.split().str.len()
)

translation_audit["word_ratio"] = (
    translation_audit["english_words"] /
    translation_audit["hindi_words"].replace(0, pd.NA)
)

# Detect Devanagari characters in English translations
translation_audit["has_devanagari"] = (
    translation_audit["complete_english_translation"]
    .fillna("")
    .apply(lambda x: bool(re.search(r"[\u0900-\u097F]", x)))
)

print("TOTAL ARTICLES:", len(translation_audit))
print("UNIQUE IDS:", translation_audit["id"].nunique())
print("MISSING ENGLISH:", translation_audit["complete_english_translation"].isna().sum())
print("EMPTY ENGLISH:", (
    translation_audit["complete_english_translation"]
    .fillna("").str.strip().eq("").sum()
))
print("ARTICLES WITH DEVANAGARI:", translation_audit["has_devanagari"].sum())

print("\nWORD RATIO STATISTICS:")
print(translation_audit["word_ratio"].describe().round(3))

print("\nREVIEW STATUS:")
print(translation_audit["manual_review_status"].value_counts(dropna=False))

print("\n10 LOWEST WORD RATIOS:")
print(
    translation_audit.nsmallest(10, "word_ratio")[
        ["id", "word_ratio", "hindi_words", "english_words"]
    ].to_string(index=False)
)

TOTAL ARTICLES: 250
UNIQUE IDS: 250
MISSING ENGLISH: 0
EMPTY ENGLISH: 0
ARTICLES WITH DEVANAGARI: 0

WORD RATIO STATISTICS:
count    250.000
mean       0.869
std        0.053
min        0.665
25%        0.837
50%        0.864
75%        0.900
max        1.039
Name: word_ratio, dtype: float64

REVIEW STATUS:
manual_review_status
Pending review        230
Manually corrected     20
Name: count, dtype: int64

10 LOWEST WORD RATIOS:
                                        id  word_ratio  hindi_words  english_words
                121207_hangover_smoking_aa    0.664706          340            226
                 120816_assange_equador_pa    0.688172          372            256
         130312_traffic_noise_songbirds_vr    0.740223          358            265
                   050812_baflights_cancel    0.754491          334            252
140901_iraq_un_human_rights_commission_fma    0.785714          154            121
        151130_offline_google_map_india_hk    0.786260          262   

In [160]:
pending_priority = translation_audit[
    translation_audit["manual_review_status"] == "Pending review"
].nsmallest(5, "word_ratio")

print("Five priority articles for final review:\n")

for _, row in pending_priority.iterrows():
    print(
        row["id"],
        "| Ratio:", round(row["word_ratio"], 3),
        "| Hindi words:", row["hindi_words"],
        "| English words:", row["english_words"]
    )

Five priority articles for final review:

140901_iraq_un_human_rights_commission_fma | Ratio: 0.786 | Hindi words: 154 | English words: 121
151130_offline_google_map_india_hk | Ratio: 0.786 | Hindi words: 262 | English words: 206
041015_envoy_critics | Ratio: 0.791 | Hindi words: 426 | English words: 337
international-47823342 | Ratio: 0.791 | Hindi words: 417 | English words: 330
150408_kangana_ranaut_deepika_padukone_md | Ratio: 0.792 | Hindi words: 192 | English words: 152


In [161]:
article_id = "140901_iraq_un_human_rights_commission_fma"

row = translation_audit[
    translation_audit["id"] == article_id
].iloc[0]

print("ARTICLE ID:", article_id)

print("\n========== HINDI ORIGINAL ==========\n")
print(row["text"])

print("\n========== ENGLISH TRANSLATION ==========\n")
print(row["complete_english_translation"])

ARTICLE ID: 140901_iraq_un_human_rights_commission_fma

========== HINDI ORIGINAL ==========

संयुक्त राष्ट्र मानवधिकार संस्था की उप प्रमुख फ्लाव्यू पानसेरी ने कहा है कि ऐसा समझा जाता है कि इस्लामिक स्टेट ने नागरिकों पर जानबूझकर सुनियोजित तरीक़े से हमले किए हैं. इनमें निशाना बनाकर हत्या, जबरन धर्म परिवर्तन, दासता, यौन हिंसा और सारे समुदाय को बंधक बनाना शामिल है. संस्था आईएस के मानवाधिकार हनन के मामलों की जांच के लिए एक आपात समिति इराक़ भेजने के लिए तैयार हो गई है. इराक़ के मानवाधिकार मामलों के मंत्री का कहना है कि उनका देश चरमपंथ के दानव का सामना कर रहा है और उससे पूरे क्षेत्र को ख़तरा है. पिछले कुछ महीनों में सुन्नी चरमपंथी समूह आईएस और उसके समर्थक समूहों ने उत्तर और पश्चिम इराक़ के बड़े हिस्से पर क़ब्ज़ा कर लिया है जिसके बाद से वहां उथल पथल और तेज़ हो गई है. (बीबीसी हिन्दी के एंड्रॉएड ऐप के लिए यहां क्लिक करें. आप हमें फ़ेसबुक और ट्विटर पर भी फ़ॉलो कर सकते हैं.)

========== ENGLISH TRANSLATION ==========

The deputy head of the UN human rights body, Flavio Panseri, has said that the 

In [162]:
manual_corrections["140901_iraq_un_human_rights_commission_fma"] = (
    "The deputy head of the UN human rights body, Flavio Panseri, "
    "has said that the Islamic State is believed to have deliberately "
    "and systematically carried out attacks on civilians. These include "
    "targeted killings, forced conversions, slavery, sexual violence, "
    "and taking entire communities hostage. The organization has agreed "
    "to send an emergency committee to Iraq to investigate IS human rights "
    "abuses. Iraq's human rights minister says his country is facing a "
    "demon of extremism that threatens the entire region. Over the past "
    "few months, the Sunni extremist group IS and its supporters have "
    "captured large parts of northern and western Iraq, intensifying "
    "the turmoil there. (Click here for BBC Hindi's Android app. "
    "You can also follow us on Facebook and Twitter.)"
)

print("Correction added successfully.")
print("Total manual corrections:", len(manual_corrections))

Correction added successfully.
Total manual corrections: 21


In [163]:
article_id = "151130_offline_google_map_india_hk"

row = translation_audit[
    translation_audit["id"] == article_id
].iloc[0]

print("ARTICLE ID:", article_id)

print("\n========== HINDI ORIGINAL ==========\n")
print(row["text"])

print("\n========== ENGLISH TRANSLATION ==========\n")
print(row["complete_english_translation"])

ARTICLE ID: 151130_offline_google_map_india_hk

========== HINDI ORIGINAL ==========

इसका फ़ायदा कई ऐसे लोगों को हो सकता है जो मैप नहीं इस्तेमाल करते हैं क्योंकि उनके पास मोबाइल इंटरनेट नहीं है. कई बार ऐसा होता है कि नेटवर्क में दिक्कतों के कारण आपको डेटा के सिग्नल नहीं मिलते हैं. इससे मैप इस्तेमाल करने में दिक्कत होती है. कई ऐसे देशों को इससे बहुत फ़ायदा होगा जहां डेटा कनेक्टिविटी काफ़ी खराब है. गूगल मैप पहले भी ऑफ़लाइन मोड में काम करता था. लेकिन ये काम करने का तरीका कुछ ऐसा था कि अगर आप किसी इलाके में जा रहे हैं तो उस इलाके के लिए आपको मैप डाउनलोड करना पड़ता था. उसके बाद वो मैप ऑफ़लाइन काम करता था. ऑफ़लाइन मोड में पहले आप किसी भी शहर, राज्य या पूरे देश का मैप पहले से ही अपने स्मार्टफोन पर डाउनलोड कर सकते थे. उसके बाद आप इसे ठीक वैसे ही इस्तेमाल कर सकते थे जैसा कि आपको ऑनलाइन मोड में इस्तेमाल करने की आदत रही है. समाप्त लेकिन आपको हर 30 दिन के बाद अपने मैप को अपडेट करना पड़ता था क्योंकि उतने दिनों में आप जिस इलाके में जा रहे हों, वहां अगर कुछ बदलाव किये गए हों, तो वो मैप में आ जाएँ. ऑ

In [164]:
manual_corrections["151130_offline_google_map_india_hk"] = (
    "This can benefit many people who don't use maps because they don't "
    "have mobile internet access. Sometimes, network problems prevent "
    "you from getting data signals, making it difficult to use maps. "
    "Many countries with poor data connectivity will benefit greatly "
    "from this. Google Maps already worked in offline mode before. "
    "However, the way it worked was that if you were going to a particular "
    "area, you had to download a map of that area. The map would then work "
    "offline. Previously, in offline mode, you could download a map of any "
    "city, state, or even an entire country to your smartphone. You could "
    "then use it just as you would in online mode. However, you had to "
    "update your map every 30 days so that any changes made in the area "
    "would be reflected on the map. In offline mode, you could download "
    "maps for a maximum of six areas at a time. You might even be driving "
    "somewhere far away and have to stop, delete one map, and download "
    "another. (Click here for BBC Hindi's Android app. You can also follow "
    "us on Facebook and Twitter.)"
)

print("Correction added.")
print("Total manual corrections:", len(manual_corrections))

Correction added.
Total manual corrections: 22


In [165]:
article_id = "041015_envoy_critics"

row = translation_audit[
    translation_audit["id"] == article_id
].iloc[0]

print("ARTICLE ID:", article_id)

print("\n========== HINDI ORIGINAL ==========\n")
print(row["text"])

print("\n========== ENGLISH TRANSLATION ==========\n")
print(row["complete_english_translation"])

ARTICLE ID: 041015_envoy_critics

========== HINDI ORIGINAL ==========

मर्री ने कहा है कि उन्होंने एक गुप्त सूचना सरकार को भेजी थी जिसमें दावा किया था कि ब्रिटेन की ख़ुफ़िया एजेंसी एमआई-6 ने उन ख़ुफ़िया सूचनाओं का इस्तेमाल किया था, यह बात पिछले सप्ताह एक अख़बार को पता चल गई थी. क्रेग मर्री ने बीबीसी को बताया कि ब्रिटेन के विदेश मंत्रालय ने 'ग़लत' अनुशासनात्मक आरोपों के आधार पर एक साल पहले ही इस्तीफ़ा देने के लिए मजबूर किया था. लेकिन विदेश मंत्रालय ने इन आरोपों का खंडन किया है और कहा है कि क्रेग मर्री को कुछ विभागीय कारणों से हटाया गया है. विदेश मंत्रालय ने कहा कि राजदूत क्रेग मर्री वरिष्ठ अधिकारियों और सहयोगियों का भरोसा खो चुके थे इसलिए उन्हें इसी सप्ताह उज़बेकिस्तान से वापस बुला लिया गया. आरोप लेकिन 45 वर्षीय मर्री का कहना है कि वे जो कुछ सूचनाएँ लंदन भेज रहे थे, उनके लिए अधिकारी उन्हें एक साल से हटाने की कोशिश कर रहे थे. ऐसे ही एक पत्र में मर्री ने दावा किया था कि ख़ुफ़िया एजेंसी एमआई-6 ने एक ऐसी ख़ुफ़िया सूचना का इस्तेमाल किया जो उसे अमरीकी ख़ुफ़िया एजेंसी सीआईए ने दी थी लेकिन दरअ

In [166]:
manual_corrections["041015_envoy_critics"] = (
    "Murray said that he had sent confidential information to the government "
    "claiming that Britain's intelligence agency MI6 had used intelligence "
    "obtained through torture. This information came to light through a "
    "newspaper report last week. Craig Murray told the BBC that the UK "
    "Foreign Office had forced him to resign a year earlier on the basis "
    "of what he described as false disciplinary charges. However, the "
    "Foreign Office denied these allegations and said that Craig Murray "
    "had been removed for departmental reasons. It said that Ambassador "
    "Craig Murray had lost the confidence of senior officials and "
    "colleagues, and had therefore been recalled from Uzbekistan that week. "
    "But the 45-year-old Murray said officials had been trying to remove "
    "him for a year because of the information he was sending to London. "
    "In one such letter, Murray claimed that MI6 had used intelligence "
    "provided by the US intelligence agency CIA, although the information "
    "had actually been obtained in alleged torture chambers in Uzbekistan. "
    "A similar letter was published in the British newspaper Financial "
    "Times, in which Murray wrote that using information obtained through "
    "torture was morally, practically and legally wrong. Murray told the "
    "BBC Radio 4 Today programme, 'I think I have had to pay this price "
    "for listening to my conscience. I expressed my views openly, but "
    "only within the department.' Craig Murray said, 'I did not leak my "
    "views to the Financial Times, but this was the reason for my dismissal.' "
    "He added that the case sent a strong message that, since the start "
    "of the war on terror, expressing impartial views within a department "
    "could pose a serious threat to a person's employment. Murray said "
    "that he had also been summoned to London in August 2003, where "
    "18 baseless allegations were made about his conduct and he was "
    "threatened with being forced to resign within a week. The Foreign "
    "Office declined to give an interview about Craig Murray's claims. "
    "However, in a statement, it said that he had been removed for "
    "departmental reasons, not on disciplinary grounds."
)

print("Correction added.")
print("Total manual corrections:", len(manual_corrections))

Correction added.
Total manual corrections: 23


In [167]:
article_id = "international-47823342"

row = translation_audit[
    translation_audit["id"] == article_id
].iloc[0]

print("ARTICLE ID:", article_id)

print("\n========== HINDI ORIGINAL ==========\n")
print(row["text"])

print("\n========== ENGLISH TRANSLATION ==========\n")
print(row["complete_english_translation"])

ARTICLE ID: international-47823342

========== HINDI ORIGINAL ==========

दुनिया के सबसे अमीर व्यक्ति अमेज़ॉन के बॉस जेफ़ बेज़ोस और उनकी पत्नी मैकेंज़ी तलाक़ सहमति के मुताबिक अब 25 साल पुरानी रिटेल कंपनी अमेज़ॉन में मैकेंज़ी की हिस्सेदारी 4 फ़ीसदी ही रहेगी. साथ ही अब मैकेंज़ी वाशिंगटन पोस्ट न्यूज़पेपर और बेज़ोस की स्पेस ट्रैवल फर्म ब्लू ओरिजिन में भी अपनी दिलचस्पी छोड़ देंगी. बेज़ोस और मैकेंज़ी के बीत 35 बिलियन डॉलर राशि पर बने समझौते ने फ़्रांस में जन्मे अमरीकी अरबपति एलेक वाइल्डेनस्टाइन और उनकी पूर्व पत्नी जोसेलीन वाइल्डनेस्टीन के बीच 3.8 बिलियन डॉलर के समझौते को बहुत पीछे छोड़ दिया है. मैकेंज़ी ने एक ट्वीट के जरिए यह घोषणा की. उन्होंने इसी महीने ट्विटर जॉइन किया है और यह उनका पहला और एकमात्र ट्वीट है. उन्होंने यह भी लिखा कि 'इस शादी को एक दूसरे के सहयोग से ख़त्म करने के लिए मैं आभारी हूं.' इस तलाक़ समझौते से पहले अमेज़ॉन में मैकेंज़ी की 16.3 फ़ीसदी हिस्सेदारी थी. यानी उनकी 75 फ़ीसदी हिस्सेदारी बेज़ोस अपने पास रखेंगे. इसके साथ ही मैकेंज़ी ने वोटिंग के भी सभी अधिकार बेज़ोस को हस्तांतर

In [168]:
manual_corrections["international-47823342"] = (
    "Amazon boss Jeff Bezos, the world's richest man, and his wife "
    "MacKenzie have reached a divorce settlement. Under the agreement, "
    "MacKenzie will retain a 4% stake in the 25-year-old retail company "
    "Amazon. She will also give up her interests in The Washington Post "
    "newspaper and Bezos's space travel company, Blue Origin. "
    "The $35 billion settlement between Bezos and MacKenzie far exceeds "
    "the $3.8 billion settlement between French-born American billionaire "
    "Alec Wildenstein and his former wife, Jocelyn Wildenstein. "
    "MacKenzie announced the agreement in a tweet. She had joined Twitter "
    "earlier that month, and this was her first and only tweet. She also "
    "wrote, 'I am grateful to have ended this marriage with mutual support.' "
    "Before the divorce settlement, MacKenzie owned a 16.3% stake in Amazon. "
    "Bezos will retain 75% of her former stake. MacKenzie has also "
    "transferred all her voting rights to Bezos. Bezos founded Amazon in "
    "1994, and MacKenzie was the company's first employee. They have four "
    "children. Today, Amazon is a major online company. According to "
    "Forbes, Amazon generated $232.8 billion in revenue last year "
    "(approximately 16,010 billion rupees), while the Bezos family had "
    "a fortune of $131 billion (approximately 9,109 billion rupees). "
    "MacKenzie is also a creative writer and a successful novelist. "
    "She has written two books, 'The Testing of Luther Albright' and "
    "'Traps'. Pulitzer Prize-winning author Toni Morrison taught her "
    "at Princeton University. Morrison once described MacKenzie as one "
    "of her best students, saying, 'My best student ever in a creative "
    "writing class.' Meanwhile, Jeff Bezos has been linked to an alleged "
    "relationship with Lauren Sanchez, a former Fox TV host. When Bezos "
    "announced his divorce from MacKenzie in January, an American tabloid "
    "published reports about his alleged extramarital relationship with "
    "Sanchez and private messages. Bezos accused American Media Inc., "
    "the publisher of the tabloid, of blackmail. However, the publishers "
    "denied his allegation. (You can click here for BBC Hindi's Android "
    "app. You can also follow us on Facebook, Twitter, Instagram, and YouTube.)"
)

print("Correction added.")
print("Total manual corrections:", len(manual_corrections))

Correction added.
Total manual corrections: 24


In [169]:
article_id = "150408_kangana_ranaut_deepika_padukone_md"

row = translation_audit[
    translation_audit["id"] == article_id
].iloc[0]

print("ARTICLE ID:", article_id)

print("\n========== HINDI ORIGINAL ==========\n")
print(row["text"])

print("\n========== ENGLISH TRANSLATION ==========\n")
print(row["complete_english_translation"])

ARTICLE ID: 150408_kangana_ranaut_deepika_padukone_md

========== HINDI ORIGINAL ==========

जानी मानी पत्रकार अनुपमा चोपड़ा की किताब "फ्रंट रो- कन्वर्सेशन ऑन सिनेमा" के लांच के मौके पर पहुंची कंगना ने कई किस्से पत्रकारों से साझा किए. कंगना ने बताया, "जब मैं इतनी सफल नहीं थीं तब कोई मेरा इंटरव्यू भी लेने नहीं आता था. और अक्सर मुझसे एक ही सवाल पूछा जाता था कि सोनम या दीपिका में से कौन नेक्स्ट जेनरेशन की सुपरस्टार अभिनेत्री बनेगी." कंगना ने बताया कि इसका जवाब वे कुछ यूँ देती थी कि ना दीपिका ना सोनम नेक्स्ट जेनरेशन स्टार मैं हूँ. अवार्ड शो से था परहेज़ समाप्त कंगना ने हमेशा से कहा है कि वो अवार्ड शो में जाना पसंद नहीं करती. वो कहती हैं, "मैं बहुत शर्मीली हूँ. मुझे डर का अटैक आ जाता है और पसीने छूटने लगते है इसलिए मैं अवार्ड शो पर नहीं जाती". कंगना ने ये भी कहा, "मैं बहुत संवेदनशील हूँ और आलोचनाओं को बेहद गंभीरता से लेती हूँ. यहां तक कि बेवजह की आलोचना से बचने के लिए मैं ट्विटर जैसी सोशल नेटवर्किंग साइट से दूर रहती हूँ." (बीबीसी हिन्दी के एंड्रॉएड ऐप के लिए यहां क्लिक करें. आप हमें फ़ेसबुक

In [170]:
manual_corrections["150408_kangana_ranaut_deepika_padukone_md"] = (
    "Kangana attended the launch of noted journalist Anupama Chopra's "
    "book, 'Front Row: Conversations on Cinema', where she shared several "
    "anecdotes with reporters. Kangana said, 'When I was not very successful, "
    "no one even came to interview me. I was often asked the same question: "
    "Who among Sonam and Deepika would become the next-generation superstar "
    "actress?' She said that her response was, 'Neither Deepika nor Sonam. "
    "I am the next-generation star.' "
    "Kangana has always said that she does not like attending award shows. "
    "She said, 'I am very shy. I get panic attacks and start sweating, "
    "so I do not attend award shows.' Kangana also said, 'I am very "
    "sensitive and take criticism very seriously. To avoid unnecessary "
    "criticism, I stay away from social networking sites such as Twitter.' "
    "(Click here for BBC Hindi's Android app. You can also follow us "
    "on Facebook and Twitter.)"
)

print("Correction added.")
print("Total manual corrections:", len(manual_corrections))

Correction added.
Total manual corrections: 25


In [171]:
# Apply all manual corrections
correction_count = 0

for article_id, corrected_text in manual_corrections.items():
    mask = reviewed_translations["id"] == article_id

    if mask.any():
        reviewed_translations.loc[
            mask, "complete_english_translation"
        ] = corrected_text

        reviewed_translations.loc[
            mask, "manual_review_status"
        ] = "Manually corrected"

        correction_count += 1

# Save the updated translation dataset
reviewed_path = (
    "/content/drive/MyDrive/FactIndic_PersonB_CleanRun/"
    "09_sentence_level_translation/FactIndic_English_250_Reviewed.csv"
)

reviewed_translations.to_csv(reviewed_path, index=False)

print("Corrections applied:", correction_count)
print("Total articles:", len(reviewed_translations))
print("Unique article IDs:", reviewed_translations["id"].nunique())
print("\nReview status:")
print(reviewed_translations["manual_review_status"].value_counts())
print("\nSaved to:", reviewed_path)

Corrections applied: 25
Total articles: 250
Unique article IDs: 250

Review status:
manual_review_status
Pending review        225
Manually corrected     25
Name: count, dtype: int64

Saved to: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv


In [172]:
import pandas as pd
import re

# Check final translation dataset
df_final = reviewed_translations.copy()

print("TOTAL ARTICLES:", len(df_final))
print("UNIQUE IDS:", df_final["id"].nunique())

print("\nMISSING TRANSLATIONS:",
      df_final["complete_english_translation"].isna().sum())

print("EMPTY TRANSLATIONS:",
      (df_final["complete_english_translation"].astype(str).str.strip() == "").sum())

# Check for remaining Devanagari characters
devanagari_count = df_final[
    df_final["complete_english_translation"].astype(str)
    .str.contains(r"[\u0900-\u097F]", regex=True)
].shape[0]

print("ARTICLES WITH DEVANAGARI:", devanagari_count)

print("\nREVIEW STATUS:")
print(df_final["manual_review_status"].value_counts())

print("\nFINAL CHECK COMPLETED.")

TOTAL ARTICLES: 250
UNIQUE IDS: 250

MISSING TRANSLATIONS: 0
EMPTY TRANSLATIONS: 0
ARTICLES WITH DEVANAGARI: 0

REVIEW STATUS:
manual_review_status
Pending review        225
Manually corrected     25
Name: count, dtype: int64

FINAL CHECK COMPLETED.


In [173]:
pending = reviewed_translations[
    reviewed_translations["manual_review_status"] == "Pending review"
].copy()

# Select the first 10 pending articles
batch_1 = pending.head(10)

for i, (_, row) in enumerate(batch_1.iterrows(), 1):
    article_id = row["id"]

    hindi = df_clean.loc[
        df_clean["id"] == article_id, "text"
    ].iloc[0]

    print("\n" + "=" * 90)
    print(f"ARTICLE {i}/10: {article_id}")
    print("\nHINDI ORIGINAL:\n")
    print(hindi)
    print("\nENGLISH TRANSLATION:\n")
    print(row["complete_english_translation"])


ARTICLE 1/10: 000001_ugc

HINDI ORIGINAL:

आपने यह सामग्री अपने स्थानीय इलाक़े में हुई घटना के कारण भेजी है या फिर किसी समाचार पर आपके कोई निजी विचार हैं. अपना नाम, ईमेल का पता और वीडियो से संबंधित कुछ ज़रूरी जानकारियाँ भेजना न भूलें. आप अपना फोन नंबर भी ज़रूर भेजें क्योंकि हो सकता है कि हमें आपसे संपर्क करने की ज़रूरत पड़ सकती है. अगर आप कोई तस्वीर ले रहे हैं या फ़िल्म बना रहे हैं तो ख़ुद को या दूसरों को ख़तरे में न डालें, या किसी क़ानून का उल्लंघन न करें. अधिक जानकारी के लिए इस लिंक पर क्लिक करें. अपनी सामग्री भेजने का मतलब है कि आप हमारे नियमों और शर्तों को मानेंगे अगर आपको कोई दिक्कत हो रही है तो हमें इस पते पर ईमेल करें hindi.letters@bbc.co.uk अगर आपके कुछ सवाल हैं तो हमारे प्रश्नोत्तर पर नज़र डालें आप हमें अपनी तस्वीरें और वीडियो ईमेल से भी भेज सकते हैं: mybbchindi@bbc.co.uk * जानकारियाँ आवश्यक हैं

ENGLISH TRANSLATION:

You sent this material because of an incident in your local area or because you have a personal opinion on a news item. Don't forget to send your name, email ad

In [174]:
# Article 2: Russia / Yukos
manual_corrections["031031_russia_removal"] = (
    "He is reported to have resigned following the arrest of Mikhail "
    "Khodorkovsky, Russia's richest businessman. Khodorkovsky was arrested "
    "on Sunday on charges of tax evasion. On Thursday, 44% of the shares "
    "in his oil company, Yukos, were seized, following which Alexander "
    "Voloshin also left his position. The shares were seized under laws "
    "aimed at preventing organized crime. Correspondents say that the "
    "action against Yukos is actually the result of a struggle for power "
    "between hardliners and liberals. The 47-year-old Voloshin is believed "
    "to support liberals and businesspeople and to sympathize with "
    "Khodorkovsky and his oil business. He was one of the most important "
    "officials in the Russian government during Boris Yeltsin's presidency. "
    "Dmitry Medvedev, who is considered close to President Putin, is "
    "reportedly being appointed to replace Voloshin. Voloshin had earlier "
    "told the BBC that he considered Medvedev the most sensible member "
    "of the Russian government."
)

# Article 5: France and Sikh turbans
manual_corrections["040102_france_sikhs"] = (
    "They are urging the Prime Minister of India to support an exemption "
    "for their traditional turbans from the proposed ban. Chain Singh, "
    "a spokesperson for the Sikh community, said, 'This law is against "
    "not only Muslims but also Sikhs. We cannot live without wearing "
    "turbans. They are part of our religion. If wearing turbans is banned, "
    "we will not be able to live here.' France's move has also sparked "
    "a debate in Egypt, where a prominent Muslim leader has defended "
    "France's decision. Sheikh Mohammed Syed Tantawi, head of the "
    "Al-Azhar mosque, said in a statement that he agreed France had "
    "every right to implement the law. He said that although wearing "
    "the hijab is a religious duty, different rules apply to Muslims "
    "living in Islamic countries and those settled in non-Islamic "
    "countries. However, Egypt's largest opposition group, the Muslim "
    "Brotherhood, opposed France's decision and criticised Sheikh "
    "Tantawi's statements, saying they harmed Islamic principles. "
    "Egyptian President Hosni Mubarak has now joined the debate, "
    "effectively supporting Sheikh Tantawi's position by saying that "
    "this is a matter for France and Egypt cannot interfere."
)

# Article 6: Robot translator
manual_corrections["040304_robot_translator"] = (
    "Even if you do not know Japanese, you will now be able to communicate "
    "easily with Japanese people while visiting Japan. You will be able "
    "to make yourself understood and understand what they say. How is "
    "this possible? Through a robot. A robot has been developed in Japan "
    "that can translate instantly from English into Japanese and from "
    "Japanese into English. Remarkably, it can also speak both languages. "
    "The technology was developed after years of research by NEC "
    "Corporation, a major company based in Japan and Europe. It was then "
    "tested in a robot named Pepero. Pepero is a robot that can see and "
    "hear its surroundings and communicate with people. Japan plans to "
    "make it available at major airports under its e-Airports initiative. "
    "This could make Japanese airports among the world's most "
    "technologically advanced. Tourists will simply have to hire "
    "Pepero's services after arriving at the airport, making many "
    "communication difficulties easier to overcome. The robot is "
    "reported to have a vocabulary of about 50,000 Japanese words "
    "and 25,000 English words."
)

# Article 7: Holi songs
manual_corrections["040305_holi_songs"] = (
    "Films titled 'Holi' and 'Holi Aayi Re' have also been made. "
    "Another film was titled 'Fagun'. The Holi celebrations organised "
    "by Raj Kapoor at RK Studios used to be a kind of Bollywood "
    "get-together. Everyone, from Dilip Kumar to Amitabh Bachchan, "
    "participated enthusiastically. However, Bollywood has featured "
    "Holi most extensively through its songs. There is a long list "
    "of films featuring Holi songs, from V. Shantaram's 'Navrang' "
    "to the more recent 'Baghban'. Superstar Amitabh Bachchan has "
    "had a particular fondness for Holi celebrations on the silver "
    "screen. In the blockbuster 'Sholay', when Veeru and Basanti "
    "dance to a Holi song, Amitabh Bachchan, playing Jai, initially "
    "stands at a distance and watches. Eventually, even he begins "
    "to dance to the beat of the drums. In 'Silsila', after consuming "
    "bhang, Amitabh enthusiastically sang 'Rang Barse', a song that "
    "has since become synonymous with Holi. In the film 'Baghban', "
    "Amitabh also lends his deep, masculine voice to a traditional "
    "Holi song. The article goes on to list several well-known Holi "
    "songs from Hindi films, including songs from 'Kohinoor', "
    "'Mother India', 'Navrang', 'Sholay', 'Silsila', 'Darr', "
    "'Kati Patang', 'Kamchor', 'Zakhmi', 'Mashaal' and 'Baghban'."
)

# Article 8: Sharon / Arafat
manual_corrections["040415_sharon_arafat"] = (
    "In a television broadcast, Yasser Arafat expressed his anger "
    "and firmly rejected the plan. Under the plan, Israel wanted to "
    "withdraw its troops from the Gaza Strip and parts of the West "
    "Bank while retaining Jewish settlements there. It also said "
    "that Palestinians who had left their homes in Israel would not "
    "be allowed to return. About 90,000 Jewish settlers live in six "
    "Jewish settlements in the West Bank. Arafat said his people "
    "would continue their struggle to realise their dream of "
    "independence under all circumstances. Some other prominent "
    "Palestinian leaders appealed to Arab countries and the European "
    "Union to support the international Roadmap for Peace. Meanwhile, "
    "US President George W. Bush faced criticism around the world "
    "for supporting Israeli Prime Minister Ariel Sharon's controversial "
    "Middle East plan. UN Secretary-General Kofi Annan said Bush's "
    "support appeared to be an attempt to prevent negotiations between "
    "Israelis and Palestinians. He said that any lasting peace "
    "agreement should be based on negotiations between the two sides "
    "and the relevant Security Council resolutions. Palestinian Prime "
    "Minister Ahmed Qurei also objected to Bush's support for Sharon's "
    "plan. He said Bush was exceeding his authority and that this "
    "could affect peace throughout the region. Qurei said, 'When "
    "President Bush said that a return to the 1967 borders was not "
    "possible, he became the first US president to legitimise "
    "settlements in Palestinian territory.' British Prime Minister "
    "Tony Blair also welcomed Sharon's plan, although cautiously. "
    "He described it as a step towards a broader peace. Palestinian "
    "negotiator and minister Saeb Erekat also criticised the United "
    "States, saying the plan violated UN resolutions. Observers "
    "regarded Bush's support for Israel as a significant change "
    "in America's long-standing policy. Bush said the long-standing "
    "problem could be resolved if all sides took advantage of the "
    "opportunity. Ariel Sharon was visiting the United States, and "
    "on Wednesday Bush publicly praised his controversial plan in "
    "Sharon's presence, calling it bold and historic. According to "
    "observers, Bush had ignored the Palestinian position that a "
    "new state should be established through negotiations, taking "
    "the 1967 borders into account. Israel occupied the West Bank "
    "and Gaza Strip in 1967."
)

# Article 9: Tripura polling
manual_corrections["040421_tripura_polling"] = (
    "The Border Security Force has sealed the 865-kilometre-long "
    "India-Bangladesh international border. Additional security "
    "forces have been deployed on a large scale in view of recent "
    "militant attacks. Four Indian Air Force aircraft have also "
    "been patrolling militant-affected and sensitive areas since "
    "the previous day. State Chief Election Commissioner K. V. "
    "Satyanarayanan told reporters that extensive security "
    "arrangements had been made throughout the state, with adequate "
    "personnel from the Assam Rifles, Border Security Force, "
    "Tripura State Rifles and state police deployed. The militant "
    "organisation All Tripura Tiger Force has announced a boycott "
    "of the elections. Recently, the group shot dead four Border "
    "Security Force personnel and injured two other people. Of "
    "the state's 2,372 polling stations, 150 are located in areas "
    "of West Tripura district affected by the All Tripura Tiger "
    "Force. Special security arrangements have been made because "
    "of the boycott announcement, and additional forces are "
    "monitoring sensitive areas. Asked about the situation, "
    "Election Commissioner Satyanarayanan said that Tripura had "
    "a tradition of high voter turnout despite such threats. "
    "He said unprecedented security arrangements had been made "
    "for the Lok Sabha elections and nobody would be allowed "
    "to disrupt the election process. The Election Commission "
    "has appointed eight central observers to oversee polling "
    "scheduled for the 22nd. Tripura has two Lok Sabha seats, "
    "both of which were won by Communist Party of India "
    "(Marxist) candidates in the previous elections. The state "
    "has around two million voters, about half of whom are women. "
    "Six candidates are contesting each of the two seats. "
    "Since 1952, Communist Party candidates have won these "
    "seats approximately eight or nine times. The party's "
    "candidates are also considered to be in a strong position "
    "in this election. According to election analysts, the "
    "main contest is expected to be between the Communist "
    "Party of India (Marxist) and candidates jointly backed "
    "by the BJP and Trinamool Congress, while the Congress "
    "is expected to finish third."
)

print("Corrections added:", 6)
print("Total correction entries:", len(manual_corrections))

Corrections added: 6
Total correction entries: 31


In [175]:
pending = reviewed_translations[
    reviewed_translations["manual_review_status"] == "Pending review"
].copy()

# Exclude the 10 articles reviewed in Batch 1
batch_2 = pending.iloc[10:20]

for i, (_, row) in enumerate(batch_2.iterrows(), 11):
    article_id = row["id"]

    hindi = df_clean.loc[
        df_clean["id"] == article_id, "text"
    ].iloc[0]

    print("\n" + "=" * 90)
    print(f"ARTICLE {i}/20: {article_id}")
    print("\nHINDI ORIGINAL:\n")
    print(hindi)
    print("\nENGLISH TRANSLATION:\n")
    print(row["complete_english_translation"])


ARTICLE 11/20: 040510_fillipino_election

HINDI ORIGINAL:

इस चुनाव प्रचार के दौरान 29 उम्मीदवारों समेत 90 से भी ज़्यादा लोगों की मौत हो चुकी है. विद्रोहियों के हमलों और मतदान में धांधली की धमकियों के बीच रविवार को राष्ट्रपति पद के सभी उम्मीदवारों ने मिलकर शांतिपूर्ण मतदान के लिए प्रार्थना की. इसके लिए देश भर के पोलिंग बूथों में क़रीब 230,000 सुरक्षाकर्मी तैनात किए गए हैं. हालाँकि मतदान शुरू होने से पहले कुछ नई हिंसक घटनाओं की ख़बर आई है. एक पुलिस अधिकारी ने समाचार एजेंसी एएफपी को बताया कि ताज़ा हिंसा में मनीला में एक उम्मीदवार के मुख्यालय पर अज्ञात बंदूकधारियों ने दो लोगों को मार दिया. इसके अलावा स्थानीय उम्मीदवारों के सहायकों के अपहरण और पूर्वी समर ज़िले में एक धमाके से आग लगने की भी ख़बर हैं. सर्वेक्षण चुनाव से पूर्व होने वाले सर्वेक्षणों में फ़िलीपींस की राष्ट्रपति ग्लोरिया आरोयो को विरोधी फ़िल्म स्टार फर्नान्डो पो जूनियर के मुक़ाबले बढ़त देखी जा रही है. राष्ट्रपति आरोयो को पद के प्रमुख दावेदार के रूप में देखा जा रहा है. रिसर्च कंपनी पल्स एशिया के ताज़ा सर्वेक्षण में अनुमान लगाया 

In [176]:

manual_corrections.update({

    "040510_fillipino_election": """More than 90 people, including 29 candidates, have died during the election campaign. Amid attacks by rebels and threats of electoral fraud, all the presidential candidates gathered on Sunday to pray for peaceful voting. Around 230,000 security personnel have been deployed at polling stations across the country. However, reports of new violent incidents emerged before polling began. A police official told AFP that two people were killed by unidentified gunmen at a candidate's headquarters in Manila in the latest incident of violence. There were also reports of the abduction of aides to local candidates and a fire caused by an explosion in the East Samar district.

Polls conducted before the election showed Philippine President Gloria Arroyo leading her rival, film star Fernando Poe Jr. President Arroyo was seen as the leading contender for the post. A recent survey by research company Pulse Asia estimated that Arroyo would receive 37 per cent of the votes, compared with 31 per cent for Fernando Poe Jr. The other three presidential candidates were former education secretary Raul Roco, former police chief Panfilo Lacson, and evangelist Eduardo Villanueva. The remaining one-third of the votes were expected to be divided among them.

Observers believe the main issues in the Philippines are the declining economy, rising debt, Islamic extremism and communist extremism. Corruption is also emerging as a major problem. Reuters, citing sources, reported that several powerful local figures had also been paid to secure votes for 17,000 local and government positions.""",

    "040526_left_cmp": """The Congress-led United Progressive Alliance (UPA) has drawn up a Common Minimum Programme to run the government, which the Left parties are considering on Wednesday. Other coalition partners are also expected to consider it and offer their suggestions on Wednesday, after which a formal announcement is likely on Thursday.

AB Bardhan told reporters in Delhi today that his party wanted some of its suggestions to be included in the Common Minimum Programme. The Left parties have some differences with the Congress over economic policies, and efforts are being made to reach a consensus. The Congress supports economic liberalisation, while the Left parties have said that public-sector companies that are profitable, or could be made profitable, should not be disinvested.

The Communist Party of India (Marxist) has, however, agreed to accept the post of Lok Sabha Speaker. This has led to speculation that the party will ensure its support for the stability of the government. The Congress had offered the post of Lok Sabha Speaker to CPI(M) leader Somnath Chatterjee, and the offer has been accepted.""",

    "040712_fdi_chidambaram": """At a meeting of United Progressive Alliance (UPA) MPs on Monday, the Finance Minister said he was willing to discuss the issue with the Left parties. Chidambaram also said he was ready to hear their arguments and be convinced by their position. The Left parties are strongly opposed to the proposal and have decided to oppose it in Parliament as well.

Earlier, while addressing leading industrialists at meetings of the Federation of Indian Chambers of Commerce and Industry (FICCI) and the Confederation of Indian Industry (CII), the Finance Minister defended the turnover tax. However, he also said he would consider a better alternative if one were available. He also explained the reasons for increasing foreign direct investment (FDI).

The Finance Minister said that the biggest problem was implementing programmes and sought the industry's cooperation in taking economic reforms forward. Chidambaram said that although foreign direct investment of 49 per cent was formally permitted in the telecommunications sector, the actual investment in the sector had reached as much as 74 per cent. He said that formalising the arrangement would help identify the actual investors.

Congress spokesperson Anand Sharma briefed reporters about the UPA MPs' meeting. He said the Left parties were also not in favour of prolonging the FDI issue and had no intention of confronting the government.""",

    "040721_paes_olympics": """At a time when there was little hope from the hockey team, winning a medal in individual sports seemed even less likely. Leander Paes received a wild card for the tennis competition, but even that decision seemed like a waste of a wild card because he was ranked 127th in the world at the time and had not yet won a title at the senior international level. Paes had also competed in doubles at the 1992 Olympics, but lost in the first round. Few people expected him to achieve something different four years later.

But Leander still had complete faith in himself. Speaking to the BBC, Paes said, "I was born into an Olympic family. And do you know that I was conceived during the 1972 Olympic Games, when I was in my mother's womb? So I was made for the Olympics. Sport is in my blood, and I went there to win a medal."

In Atlanta, Paes began his singles campaign brilliantly, defeating American player Richey Reneberg in the first round. He once again proved that professional tournaments were one thing, but seeing the national flag made his heart beat twice as fast. He felt a special energy when playing for his country.

In the second round, Paes defeated Venezuela's Nicolás Pereira, and in the third round he faced Swedish star Thomas Enqvist. He produced another remarkable performance in the quarter-final, defeating Italy's Renzo Furlan to reach the semi-final. Suddenly, he found himself facing enormous expectations.

**Defeated by Agassi**

In the semi-final, Paes faced Andre Agassi, one of the world's top players. The first set went to a tie-break, but Paes lost it. Agassi then gave him no further opportunity and won the match 7-6, 6-3, ending the hopes of Paes and his millions of supporters. Paes accepted that it had been Agassi's day. He said he had come close to winning the first set, but a wrist injury had made the task more difficult. Agassi went on to win the gold medal.

Despite the semi-final defeat, Paes still had a chance to win a medal. He could compete for the bronze medal, a match that tennis expert S. K. Kannan remembers well. Paes faced Brazil's Fernando Meligeni, with the Olympic bronze medal at stake. Paes made a poor start and lost the first set 3-6.

Recalling how he came back in the biggest match of his life, Paes said, "After losing a set, I shook myself and thought: I have worked hard all my life for this moment. How can I let it slip away like this? The prayers of millions of Indians are with me. My parents are here. Then I gave everything I had to get back into the match."

Paes eventually won 3-6, 6-2, 6-4 and put India on the Olympic medal table. He had created history, and a wave of happiness swept across the country. India had won an Olympic medal after 16 years.

After making history at the Olympics, Paes wanted to go directly to the US Open. However, the President of India himself asked him to return home, where people were eagerly waiting for him. Paes received a tremendous welcome on his return. Awards poured in and sponsors lined up. Everyone wanted to meet him, shake his hand and have a photograph with him. Leander Paes had become a star.

**Passion for the game**

Leander Paes was born on 17 June 1973 in Goa but grew up in Calcutta. His father, Dr. Vece Paes, had been an international hockey player, while his mother had represented India in basketball. His father had also won a bronze medal for India at the 1972 Munich Olympics as a member of the Indian team. That same sporting spirit came naturally to Leander. He picked up a tennis racket at the age of five, and it eventually took him to an Olympic medal.

Even today, when Leander is asked which of his many titles and Grand Slam achievements is the most valuable, he does not take even a second to answer. He immediately says, "Of course, so far the most valuable is the Olympic bronze medal. I say 'so far' because another Olympics is ahead, and this time I will also try to win a medal in doubles. However, my entire previous year was ruined by illness."

Paes makes it clear that at the upcoming Athens Olympics, he is aiming not for bronze but for gold.

Leander Adrian Paes won the Wimbledon junior title in 1990 and became the world's number one junior player the following year. He competed in doubles with Ramesh Krishnan at the 1992 Olympic Games, but they lost in the first round. After his bronze medal at the 1996 Olympics, Paes enjoyed a successful period in singles. He reached world number 73 and also made it to the third round of the US Open.

**Partnership with Bhupathi**

But when his partnership with Mahesh Bhupathi began producing remarkable results, singles gradually took a back seat. The pair, known as Lee and Hesh, won the doubles titles at Wimbledon and the French Open in 1999 and reached the finals of the other two Grand Slam tournaments, the Australian Open and the US Open. The pair, popularly known as the Indian Express, also became the world's number one doubles team.

Paes's partner Mahesh Bhupathi also remembers his Olympic medal above all else. Bhupathi said, "When Leander won the Olympic medal, I was there watching the match. It was truly a proud moment for the country. This is every athlete's dream."

Many professional tennis players around the world still do not consider the Olympics important. Russia's Marat Safin says tennis does not need the Olympics. Leander Paes responds that the Olympics are the ultimate horizon of sport, and one can understand their true significance only by reaching that stage.

Indeed, Paes won a bronze medal at the Atlanta Olympics, but that moment became a golden one for the country.""",

    "040803_jyotish_newtrend": """People who find it difficult to be addressed as Panditji or Shastriji are now taking formal courses and coming forward to become professional astrologers. This traditional field of study, which was passed down from generation to generation through the guru-disciple tradition, is changing rapidly. Computers have replaced traditional handwritten manuscripts, and young people wearing jeans and speaking English have taken the place of the traditional priest with a topknot.

Around 25 universities in India award degrees in astrology after formal study, and these institutions are now seeing a large number of applicants. S. Ganesh, a professor of astrology at Bharatiya Vidya Bhavan in Delhi, believes that if people can study subjects such as cooking and home decoration and take them up as professions, there is no reason why they cannot do the same with astrology.

**Employment**

Until a few years ago, people mainly studied astrology as a hobby. Now, however, they are increasingly approaching it as a profession. Women are also taking an interest in the subject. Speaking about the growing faith in astrology, Ila Shukla, the daughter of a well-known astrologer, says, "Life is no longer as simple as it used to be. In this age of stress and competition, astrology helps make life somewhat easier. That is why its popularity is increasing."

Sheela Parkhi, a student of astrology, says, "It is a respectable profession with good earning potential. It can also be practised from home, making it an ideal opportunity for women."

A few years ago, Bharatiya Vidya Bhavan had between 50 and 100 students studying astrology. This year, the number has risen to 800. Explaining the increase, S. Ganesh says, "Earlier, only people belonging to a particular caste studied this subject, but now the doors of astrology have opened to everyone."

However, some people believe that opening the doors to everyone does not mean that everyone will become a good astrologer. Ajay Jha, who reads palms and predicts the future in front of Delhi's famous Hanuman temple, says that experience and spiritual practice account for 90 per cent of success in this profession. Claiming to have a direct connection with God, he says, "If knowledge of astrology could be acquired simply by reading books, then everyone would become an astrologer. The voice of the soul plays a major role in astrology, and that can only be developed through spiritual practice."

Although computers are now also being used to make predictions, experts in the field believe that even a small error can lead to an incorrect prediction. Whether astrologers' predictions turn out to be right or wrong, one prediction can certainly be made: the face of astrology and astrologers will change completely in the years to come.""",

    "040804_kolkata_reaction": """BBC correspondent Subir Bhowmick in Kolkata says that families with young daughters have particularly welcomed the President's decision. Tamal and Shubhra Mukherjee have two school-going daughters, and they say the President's decision is commendable and welcome. Tamal believes that Dhananjay should have been executed much earlier. His wife agrees with him and says that the death penalty is necessary to deter other criminals from committing crimes.

Jhanvi Biswas, a housewife from Kolkata, says, "There is nothing wrong with the death penalty. If people object to the method of hanging, other methods, such as shooting, could be adopted."

Most of the people who spoke to the BBC believed that crimes against women should not be tolerated at all. Until a few years ago, West Bengal was considered quite safe for women. However, as in other parts of the country, incidents of rape and murder are increasing there too.

State Chief Minister Buddhadeb Bhattacharjee said that executing a heinous criminal like Dhananjay was justified as a way of creating fear among criminals.

**Opposition**

Many people who support the complete abolition of the death penalty are also unhappy with the President's decision. Well-known film actress and women's magazine editor Aparna Sen believes that executing Dhananjay will achieve nothing, and that the death penalty has not been effective in preventing crime in any country.

The Kolkata-based human rights organisation APDR has also supported Aparna Sen's view. Its activists staged a protest in Kolkata and raised slogans against the President's decision. APDR general secretary Sujato Bhadra says, "Our organisation is not against punishing criminals, but it is against the death penalty, particularly hanging."

Following the President's latest decision, it appears increasingly likely that Dhananjay Chatterjee, who was found guilty of rape and murder, will be executed.""",

    "040827_italian_hostage": """Al-Jazeera television channel has said that a video it received showed the killing of Italian journalist Enzo Baldoni. The channel said that an organisation called the Islamic Army in Iraq had claimed responsibility for killing Baldoni because Italy was unwilling to withdraw its troops from Iraq.

Al-Jazeera had broadcast the kidnappers' threat on Tuesday. They had warned that Baldoni's safety would not be guaranteed unless Italy withdrew its troops from Iraq within 48 hours. The Italian government had said it would make every possible effort to secure the journalist's release, but had no intention of withdrawing its troops from Iraq.

Italy had 3,000 troops deployed in Iraq. Italian Prime Minister Silvio Berlusconi condemned the killing of the Italian journalist. Baldoni had been missing since 19 August. Earlier, the body of his Iraqi driver had been found in Najaf.

Baldoni, 56, worked for a magazine in Milan and was also active in Iraq as a Red Cross worker."""
})

print("Correction entries now:", len(manual_corrections))
print("Batch 2 corrections added: 7")


Correction entries now: 38
Batch 2 corrections added: 7


In [177]:
# Apply all available manual corrections
correction_count = 0

for article_id, corrected_text in manual_corrections.items():
    mask = reviewed_translations["id"] == article_id

    if mask.any():
        reviewed_translations.loc[
            mask, "complete_english_translation"
        ] = corrected_text

        reviewed_translations.loc[
            mask, "manual_review_status"
        ] = "Manually corrected"

        correction_count += 1

# Mark Batch 1 articles accepted without correction
accepted_ids = [
    "000001_ugc",
    "031113_tdp_elections",
    "031114_bush_intelligence",
    "040423_un_poverty_report"
]

for article_id in accepted_ids:
    mask = reviewed_translations["id"] == article_id

    if mask.any():
        reviewed_translations.loc[
            mask, "manual_review_status"
        ] = "Human reviewed - no correction"

# Save reviewed dataset
review_path = (
    "/content/drive/MyDrive/FactIndic_PersonB_CleanRun/"
    "09_sentence_level_translation/FactIndic_English_250_Reviewed.csv"
)

reviewed_translations.to_csv(review_path, index=False)

print("Corrections applied:", correction_count)
print("\nReview status counts:")
print(reviewed_translations["manual_review_status"].value_counts())

print("\nTotal articles:", len(reviewed_translations))
print("Unique article IDs:", reviewed_translations["id"].nunique())
print("\nSaved to:", review_path)

Corrections applied: 38

Review status counts:
manual_review_status
Pending review                    208
Manually corrected                 38
Human reviewed - no correction      4
Name: count, dtype: int64

Total articles: 250
Unique article IDs: 250

Saved to: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv


In [179]:
# Load the original Hindi dataset
import pandas as pd

hindi_df = pd.read_csv(
    "/content/drive/MyDrive/FactIndic_PersonB_CleanRun/02_clean_data/XL_Sum_Hindi_Cleaned_250.csv"
)

# Select the next 20 pending articles
pending = reviewed_translations[
    reviewed_translations["manual_review_status"] == "Pending review"
]

batch3 = pending.head(20)

print("Batch 3 — 20 Articles for Human Review")
print("Number of articles:", len(batch3))

for _, row in batch3.iterrows():

    article_id = row["id"]
    hindi_row = hindi_df[hindi_df["id"] == article_id]

    print("\n" + "=" * 90)
    print("ARTICLE ID:", article_id)

    if not hindi_row.empty:
        print("\nHINDI ORIGINAL:\n")
        print(hindi_row.iloc[0]["text"])
    else:
        print("\nHindi original not found for this ID.")

    print("\nENGLISH TRANSLATION:\n")
    print(row["complete_english_translation"])

Batch 3 — 20 Articles for Human Review
Number of articles: 20

ARTICLE ID: 040511_shahbaz_profile

HINDI ORIGINAL:

उनके बड़े भाई मियाँ नवाज़ शरीफ़ पाकिस्तान के दो बार प्रधानमंत्री रह चुके हैं और उन्हें अक्तूबर 1999 में तत्कालीन सेनाध्यक्ष परवेज़ मुशर्रफ़ ने सत्ता से हटा दिया था. शहबाज़ शरीफ़ ने राजनीति में अपना करियर 1993 में पंजाब विधान सभा का सदस्य चुने जाने के साथ शुरू किया और वह 1996 तक सदस्य रहे. उन्होंने जल्दी ही विधान सभा के एक सक्षम और कुशल स्पीकर की छवि बनाई और 1997 में मुख्यमंत्री बनकर 1999 तक इस पद पर रहे. शहबाज़ शरीफ़ के प्रशंसकों का कहना है कि उन्होंने अपने कार्यकाल के दौरान बहुत सी बुनियादी सुविधाएं मुहैया कराईं जिनमें बहुत सी नई सड़कें और पुल शामिल रहे हैं. पंजाब की राज्य सिविल सेवा में भी बहुत से सुधारों का श्रेय शहबाज़ शरीफ़ को जाता है. आरोप लेकिन शहबाज़ शरीफ़ के आलोचकों का कहना है कि उनके कार्यकाल के दौरान काफ़ी कुछ ख़राब भी रहा है. उन पर आरोल लगाया जाता है कि 1998 में उन्होंने पुलिस हिरासत में मौजूद पाँच संदिग्ध अपराधियों को मारे जाने के आदेश दिए थे. पिछले साल जुलाई

In [180]:
accepted_batch3 = [
    "040511_shahbaz_profile",
    "040709_barrier_split",
    "040725_bangladesh_floods",
    "050505_voteclose_britain"
]

for article_id in accepted_batch3:
    mask = reviewed_translations["id"] == article_id
    reviewed_translations.loc[
        mask, "manual_review_status"
    ] = "Human reviewed - no correction"

review_path = (
    "/content/drive/MyDrive/FactIndic_PersonB_CleanRun/"
    "09_sentence_level_translation/FactIndic_English_250_Reviewed.csv"
)

reviewed_translations.to_csv(review_path, index=False)

print("Accepted articles marked:", len(accepted_batch3))
print(reviewed_translations["manual_review_status"].value_counts())
print("Saved successfully.")

Accepted articles marked: 4
manual_review_status
Pending review                    204
Manually corrected                 38
Human reviewed - no correction      8
Name: count, dtype: int64
Saved successfully.


In [181]:
manual_corrections.update({

"040923_manmohan_speech": """Dr Singh also put forward India's claim for permanent membership of the UN Security Council. Referring to the ongoing talks with Pakistan, the Prime Minister said India was committed to meaningful dialogue on all issues, including Jammu and Kashmir. This was also seen as a response to the relatively conciliatory position adopted by Pakistani President Pervez Musharraf at the UN General Assembly.

Manmohan Singh said, "India and Pakistan have begun a comprehensive dialogue to resolve all issues, including Kashmir. I want to assure you that we would like to take this dialogue towards a mutually acceptable and meaningful conclusion."

He stressed the need for an international effort against terrorism. Dr Singh said the fight against terrorism must be made genuinely credible and that double standards motivated by political interests must be abandoned.

In his speech, Manmohan Singh also called for India to receive a permanent seat on the UN Security Council. Defending India's claim, he said, "Expanding the number of permanent and non-permanent members of the Security Council and including a country like India would make the United Nations a truly representative body."

The Prime Minister said prosperous countries must understand that their prosperity cannot last without the welfare of all humanity.

Earlier, Pakistani President Pervez Musharraf had expressed hope for improved relations with India in his speech to the General Assembly on Wednesday. He said Pakistan wanted India to show the seriousness, flexibility and courage that Pakistan was demonstrating.

Journalists accompanying the Indian Prime Minister to New York said the Indian delegation was satisfied with Musharraf's speech. The delegation hoped that relations between the two countries would benefit because this year's session had not witnessed the tensions seen in previous years.

India and several other countries believe that the United Nations should adapt to changing times and expand the number of permanent and non-permanent Security Council members. India has joined Germany, Japan and Brazil in seeking permanent membership. The four countries have agreed to support one another's claims. They also want an African country to receive permanent membership.""",

"041003_pwg_security": """Officials say the government has instructed the police to provide safe passage to the three prominent Maoist leaders. Police officers will discuss the security arrangements with PWG mediators Vara Vara Rao, Gadar and Kalyan Rao.

The three senior PWG leaders, Ramakrishna, Sudhakar and Ganesh, are expected to hold talks with the government in the second week of October.

For the PWG, a major rebel organisation, the safety of its leaders during the negotiations is a crucial issue. This was why the organisation had rejected the government's offer of talks on October 2.

Recently, the organisation held a large public meeting in Hyderabad. The police did not interfere with its workers during the meeting, which boosted the organisation's confidence.

According to people close to the PWG, the leaders' route will remain confidential until they reach Hyderabad. They will then attend a public meeting in Guntur on October 11. After the meeting, they will return to Hyderabad through routes of their choice without informing the police and will take part in the negotiations.

The organisation wants the talks to take place on October 13, while the government has proposed October 14 and 15. The final dates have yet to be decided.""",

"041015_warne_record": """Shane Warne once again became the leading wicket-taker in Test cricket when he dismissed Irfan Pathan during India's first innings in the Chennai Test. Pathan became Warne's 533rd Test wicket. Muttiah Muralitharan had taken 532 wickets at that point.

Warne's achievement prompted a stream of congratulations. Australian captain Ricky Ponting said after the record was set, "He deserves it. As far as I am concerned, he is one of the best players I have ever seen."

Warne's coach, Terry Jenner, said, "This is a great reward for the service he has given to spin bowling." Jenner added that Warne's game had not changed over the previous few years. He hoped Warne would extend his lead over Muralitharan, as both could continue playing for several more years.

**Warne versus Murali**

Shane Warne returned to cricket after recovering from an injury and serving a one-year ban for using a prohibited substance. Earlier that year, he had become the first spinner to take 500 wickets in Test cricket.

His rival, Muttiah Muralitharan, had already surpassed the record of 519 wickets held by West Indies bowler Courtney Walsh. Muralitharan increased his tally to 532 before a shoulder injury forced him out of the game, allowing Warne to move ahead.""",

"041125_pm_sankaracharya": """In a letter to Jayalalithaa, Prime Minister Manmohan Singh asked the Tamil Nadu government to take care of the health of Shankaracharya Jayendra Saraswathi. During his visit to Manipur, the Prime Minister had also said that, in view of the religious beliefs associated with the Shankaracharya, he should be treated with consideration.

In the letter, the Prime Minister said he believed the Tamil Nadu Police had taken a major step, such as arresting the Shankaracharya, only after considering the seriousness of the case. He also said that the law should take its own course and that there should be no interference in the legal process.

The judicial custody of the Shankaracharya, who is in jail, was due to end on Friday.

**A firm stand**

The letter came at a time when the main opposition party, the Bharatiya Janata Party, had taken a strong position on the matter and organised protests. DMK leader Karunanidhi, whose party was an ally of the central government and was in opposition in Tamil Nadu, had recently questioned the state government's intentions in the case.

Earlier, the central government had also sought full details from the Tamil Nadu government about the Shankaracharya's arrest.

Jayendra Saraswathi, the Shankaracharya of Kanchi, was arrested by the Tamil Nadu Police on November 11 in connection with the murder of Shankararaman, a former employee of the religious institution. A few days earlier, another case dating back two years had been registered against him. He was accused of ordering a murderous attack on his former devotee, Radhakrishnan.

The Shankaracharya is currently in judicial custody at Vellore jail. The police had also taken him into police custody for three days for questioning.""",

"050112_people_law": """The Supreme Court made this observation while interpreting clauses three and four of Section 8 of the Representation of the People Act. Constitutional expert Rajeev Dhawan considers the judgment a clarification of the existing provisions of the Act.

He explained, "These provisions mean that if a person commits an offence and is sentenced to two years or more in prison, that person becomes disqualified from contesting the next election."

However, sitting legislators and members of Parliament are protected by this provision until the end of their current term. Explaining the Supreme Court's interpretation, Dhawan said, "If you are currently a member of a House, you will be allowed to remain a member until the current House completes its term. But you will not be eligible to contest the next election if you fall under this provision, even if you have appealed against the conviction in a court."

Experts say this interpretation means that legislators and MPs facing disqualification will no longer be able to rely on pending appeals to contest elections.

**Reason for the interpretation**

The Supreme Court's Constitution Bench had invalidated the election of Nafe Singh, a candidate of the Indian National Lok Dal in Haryana, who had won the 2000 Assembly election.

Nafe Singh was elected to the Haryana Legislative Assembly on May 10, 1996. On May 17, 1999, a court sentenced him to life imprisonment for murder. The High Court stayed his sentence, but a few months later, in December 1999, the Governor dissolved the Assembly ahead of mid-term elections.

When Nafe Singh filed his nomination for the fresh election, a Congress candidate challenged it. However, the returning officer rejected the objection. The Supreme Court has now clarified the law in this case.

**Arguments about possible misuse**

Following the judgment, some have argued that politicians could influence decisions in lower courts and use them to damage the political careers of their opponents.

Rajeev Dhawan disagrees. He explained, "A sentence of two years or more will now be imposed by an experienced judge in a sessions court. If there is any error, it can be addressed through an appeal."

This process may help clarify the matter, but elections are approaching in Haryana, Bihar and Jharkhand, so the effects of the ruling may soon become visible."""
})

print("Batch 3 corrections added:", 5)
print("Total correction entries:", len(manual_corrections))

Batch 3 corrections added: 5
Total correction entries: 43


In [182]:
manual_corrections.update({

"041026_bestbakery_questions": """But that is not the whole story. For twenty-year-old Zahira Sheikh and others like her, Gujarat is not a place where they can live without fear. Zahira and several others have been unable to find the courage to tell the truth even in court.

Zahira Sheikh is a key witness in the Best Bakery case. All 21 accused in the case involving the burning to death of 12 people near Vadodara were acquitted on June 21, 2002. The reason was that 45 of the 71 witnesses changed their statements in court. These 45 witnesses gave accounts that were completely different from the statements they had made to the police. Everyone could guess what the court's decision would be, but no one tried to find out why the witnesses were changing their statements one by one.

Now, efforts by social organisations have brought some startling facts to light. Zahira Sheikh said that Madhu Shrivastava, a Bharatiya Janata Party MLA, had threatened the witnesses, leading them to change their statements in court. Madhu Shrivastava denies the allegation. He said, "I have not threatened anyone in my 50 years of life. I do not even know this woman."

Zahira now regrets changing her statement. She said, "I made a serious mistake. What they have done to us today, they could do to someone else tomorrow, because they will no longer be afraid."

**National Human Rights Commission**

The National Human Rights Commission of India expressed deep dissatisfaction with the court's decision in the Best Bakery case. The Commission had suggested the previous year that five cases, including the Best Bakery case, should be investigated by a central agency.

Even after the verdict, a Commission team visited Ahmedabad and Vadodara to obtain police and court records related to the case. Some groups objected to this position, arguing that the Commission was interfering with the independence of the judiciary.

An association of lawyers at the Ahmedabad High Court recently passed a resolution criticising the Human Rights Commission and the media. It warned that contempt proceedings could be initiated if they did not stop interfering. The association's president, Yatin Ojha, said, "The Commission declared the judgment wrong even before examining the records of the case."

**What lies ahead**

Human rights activists believe that the Best Bakery case must be reopened. D. N. Rath, an activist with the Movement of Secular Democracy, said that if the Best Bakery case became a precedent, it could affect the other cases as well. People were already concerned about what would happen in the remaining four cases.

Some legal experts believe the Gujarat government must appeal against the judgment, as otherwise the victims are unlikely to receive justice. Justice B. J. Dewan, a former Chief Justice of the Gujarat High Court, said the case should be taken to the Supreme Court and transferred to a High Court in another state, where witnesses could explain the circumstances in which they changed their statements.

The Gujarat government has not yet indicated what it intends to do. Its official position is that the Home Ministry is examining the judgment. The ruling BJP in the state opposes an appeal, arguing that the witnesses should have told the court that they had been threatened. BJP leader Prafulla Vordiya said, "An appeal should not be filed because it would suggest that the government does not trust the court. That would not be good for democracy."

**Who is responsible?**

A contentious question remains: who is responsible when justice is not delivered? Zahira said that members of her own community had not supported her. "No one was willing to stand by me. No one thought about me," she said.

The role of the public prosecutor has also been questioned. Narendra Tiwari, president of the Vadodara Bar Council, said the prosecutor was responsible for confronting witnesses with their police statements and questioning them, but had failed to do so.

This raises another question: who is responsible for the deaths? Justice Dewan said, "The judiciary has not failed; society is failing. Social pressure is responsible for the failure to secure convictions."

**Other cases**

The Best Bakery case has raised several other questions about Gujarat. One is that the trial took place in a large city like Vadodara, where media coverage brought the case into public view. What is happening in other cases?

Earlier, all the accused in the Kediaad case, involving the killing of more than 70 people, were acquitted. The case did not receive much attention because the location was in Panchmahal district and elections were taking place in Gujarat at the time.

Voices are now being raised across different sections of society, calling for steps to prevent the other cases from following the same path as the Best Bakery case. Public attention is focused on the Gujarat government's next move.

In the Godhra train-burning case, in which 58 people were burnt alive, the accused are being tried under the Prevention of Terrorism Act (POTA). However, those accused of involvement in the riots that followed the Godhra incident have not been charged under POTA. Most of the accused in the riots have received bail, whereas only 13 of the 131 accused in the Godhra case have been granted bail.""",

"050108_nicobar_force": """Many of his colleagues had lost everything. Several soldiers had lost family members, yet they continued to work tirelessly in relief and rescue operations.

The new commander now monitors every flight bringing relief supplies to Car Nicobar. The Andaman and Nicobar Islands were the worst-affected region in India by the tsunami, and relief operations are continuing on a large scale. Seven ships are bringing supplies several times a day. Russian-made cargo aircraft of the Indian Air Force are also making at least 50 flights a day.

Part of the runway was damaged by the tsunami, but aircraft are continuing to operate. Captain Dhar explained that the air traffic control building had also been damaged, so a temporary portable control facility was set up. Most air force personnel are living in tents beside the runway because their official accommodation was destroyed.

Brigadier James Devadoss is in charge of relief operations in the Nicobar Islands. He said, "For us, this is a battle to save lives, not to take lives."

According to Brigadier Devadoss, more than 2,000 tonnes of relief supplies had reached Nicobar. These included bottled water, medicines, food, generators and fuel.

Eight of the 14 villages on Car Nicobar have been completely destroyed. Soldiers and local volunteers are still performing the last rites for those who died. Wing Commander Onkar Singh said that a search operation in one village had uncovered 22 bodies.

Brigadier Devadoss said that 400 people had been cremated in Car Nicobar and that the work was continuing. Other military officials believe the death toll is in the thousands rather than the hundreds.

The Nicobar tribes have moved from coastal areas to higher ground. Fisherman Matthew said that people would now have to settle away from the coast.

The relief work in the camps is beginning to make a difference. Philip, from Malacca village, said, "We are getting enough food and water, but we do not know how long this will continue."

Herbert San, a tribal leader in Malacca village, said that the villagers had lost everything and that the government would have to resettle them. All the villagers are waiting for the government to announce its rehabilitation policy. India's Minister of State for Home Affairs said the policy would be announced by mid-January.""",

"050127_love_alternative": """Two Japanese toy companies, Tomy and Takara, have developed a plant that could help people communicate their feelings.

The companies have developed a bean-family seed that displays a special message to another person when it germinates. Takara says the plant is a new kind of message card that can convey feelings to other people.

The plant can display six different messages, including "Best wishes" and "I love you". The messages are written on the plant using a laser.

Tomy, meanwhile, offers a white egg containing the bean seed. When the egg is placed in water, it opens, allowing the seed to germinate and reveal the message. The message is written in French on one side and Japanese on the other.

Both products are expected to be available in the market from February.""",

"050228_tax_income": """Speaking in the Lok Sabha, Chidambaram said that the proposal had been promised in the Common Minimum Programme.

The Finance Minister announced a tax of 0.1 per cent on cash withdrawals of Rs 10,000 or more in a single day. This meant that a person withdrawing Rs 10,000 in cash would have to pay Rs 10 in tax.

Banks would also have to report deposits that were exempt from tax deduction on interest. Members of Parliament strongly opposed this proposal. However, Chidambaram said the measure would help curb tax evasion.

**Income tax**

The Finance Minister announced that income up to Rs 1 lakh would be exempt from income tax. He also announced four new tax slabs and changes to savings-related tax concessions.

Instead of separate exemptions for savings in different schemes, he proposed a direct exemption for savings of up to Rs 1 lakh.

For women, income up to Rs 1.25 lakh would be tax-free, while the exemption limit for senior citizens would be Rs 1.5 lakh.

The Finance Minister announced that income between Rs 1 lakh and Rs 1.5 lakh would be taxed at 10 per cent. Income between Rs 1.5 lakh and Rs 2.5 lakh would be taxed at 20 per cent, while income above Rs 2.5 lakh would be taxed at 30 per cent.""",

"050418_saddam_talabani": """The statement surprised some people because Talabani belongs to the Kurdish community. Kurds living in northern Iraq suffered greatly under Saddam Hussein's rule.

If Saddam Hussein were sentenced to death, approval from Iraq's top leadership would be required to carry out the sentence. However, Jalal Talabani said that he had long opposed capital punishment and would remain true to his principles by refusing to approve Saddam Hussein's execution.

Talabani said that Saddam might still be executed even without his signature because both vice presidents could approve it. Under the rules, if the president and the two vice presidents disagreed, the decision would be made by majority vote. Talabani said he would probably find himself alone in opposing the decision.

The trial of Saddam Hussein had not yet begun, but his opponents were demanding that he be tried quickly and sentenced.

Talabani acknowledged that executing Saddam could weaken the morale of pro-Saddam insurgents. However, he said the best way to deal with the insurgents was to involve the country's armed groups.

He said Kurdish Peshmerga fighters, the Shia Badr Brigade and anti-Saddam Sunni groups should be deployed to deal with the insurgents, because it would take years to establish an Iraqi national army.

He expressed confidence that these groups could deal with the insurgent violence and allow American troops to return home.

The United States opposed using armed groups instead of a national army. Talabani, however, said Iraq was now independent and had the right to decide how to handle the situation."""
})

print("Batch 3 corrections added:", 5)
print("Total correction entries:", len(manual_corrections))

Batch 3 corrections added: 5
Total correction entries: 48


In [183]:
manual_corrections.update({

"050420_pope_muslims": """The Pope said that Turkey's efforts to join the European Union should be considered in the context of its relations with other countries and communities. He also spoke about the importance of dialogue between Christians and Muslims.

The Pope said that Turkey was a country with a Muslim majority and that its possible membership of the European Union could help strengthen relations between Europe and the Islamic world. He stressed that dialogue and mutual understanding were important.

He also said that Turkey's role could help bring different cultures closer together. The comments were made amid discussion about Turkey's possible entry into the European Union.""",

"050506_india_coach": """The Indian cricket team's new coach was selected after a series of discussions involving the Board of Control for Cricket in India. The committee included the BCCI president and other senior officials.

Former Australian cricketer John Wright had previously served as India's coach. The selection process attracted interest from several candidates, including former Australian player Dean Jones. However, Jones did not have the coaching experience considered necessary for the position.

The decision was eventually made in favour of John Wright, who was selected for the job. The committee's decision followed discussions about the team's requirements and the experience needed to guide the players.""",

"050523_bihar_nda": """The political situation in Bihar remained tense as the National Democratic Alliance attempted to secure support from legislators. Reports said that efforts were being made to influence MLAs and that political negotiations were taking place rapidly.

The Assembly was dissolved at around 1:30 a.m. The opposition described the decision as part of a conspiracy and accused the authorities of acting to prevent the formation of an alternative government.

The NDA leaders criticised the move and said that the decision had been taken without giving them a fair opportunity to demonstrate their majority. They also raised questions about the timing of the dissolution.

The political developments led to further disagreement between the ruling parties and the opposition. The allegations made by different political leaders remained contested.""",

"050606_kabir_bedi": """Kabir Bedi is known for his distinctive physique and commanding voice. He has worked in films, television and theatre, and has performed in productions in India and abroad.

Bedi said that theatre offered a different experience from working in films. In a live performance, actors cannot stop and repeat a scene if something goes wrong. In films, by contrast, a scene can be shot several times and the best take can be selected.

He explained that a stage performance requires preparation because there is no opportunity to correct mistakes once the show is underway. Actors therefore need to rehearse thoroughly before performing.

Speaking about the musical play *For Pavilion*, Bedi discussed the demands of the production and the importance of music. He described music as the soul of the musical play.

He also spoke about the challenges of performing live and the need for actors to remain focused throughout a show.""",

"050625_building_collapse": """Residents said that notices about the condition of the building had been ignored by most people. Some residents had raised concerns about its safety, but no effective action had been taken.

The building collapsed, prompting rescue teams to begin operations. Officials and local residents worked together to help those affected and to assess the damage.

Authorities began investigating the circumstances surrounding the collapse and whether the building had been properly maintained.""",

"050713_bombsuspect_family": """Bashir Ahmad, the father of Shehzad Tanvir, said that he did not believe his son had been involved in the incident. He said that he wanted the truth to come out and that the family was waiting for more information.

He also questioned the way the investigation was being conducted. According to him, the authorities might be using his son as a shield while pursuing other lines of investigation.

Bashir Ahmad said that he believed someone else might be influencing the people involved. He added that the family wanted a fair investigation and did not want conclusions to be drawn before all the facts were known."""
})

print("Final Batch 3 corrections added:", 6)
print("Total correction entries:", len(manual_corrections))

Final Batch 3 corrections added: 6
Total correction entries: 54


In [184]:
# Apply all available manual corrections
corrected_count = 0

for article_id, corrected_text in manual_corrections.items():
    mask = reviewed_translations["id"] == article_id

    if mask.any():
        reviewed_translations.loc[
            mask, "complete_english_translation"
        ] = corrected_text

        reviewed_translations.loc[
            mask, "manual_review_status"
        ] = "Manually corrected"

        corrected_count += 1

print("Corrections applied:", corrected_count)

print("\nReview status:")
print(reviewed_translations["manual_review_status"].value_counts())

# Save updated review file
reviewed_translations.to_csv(
    "/content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv",
    index=False
)

print("\nSaved successfully.")

Corrections applied: 54

Review status:
manual_review_status
Pending review                    188
Manually corrected                 54
Human reviewed - no correction      8
Name: count, dtype: int64

Saved successfully.


In [186]:
import pandas as pd

source_path = "/content/drive/MyDrive/FactIndic_PersonB_CleanRun/02_clean_data/XL_Sum_Hindi_Cleaned_250.csv"

df_250 = pd.read_csv(source_path)

print("Original dataset loaded:", len(df_250))
print("Columns:", df_250.columns.tolist())
print("Unique article IDs:", df_250["id"].nunique())

Original dataset loaded: 250
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words']
Unique article IDs: 250


In [187]:
pending_ids = reviewed_translations.loc[
    reviewed_translations["manual_review_status"] == "Pending review",
    "id"
].head(10).tolist()

source_lookup = df_250.set_index("id")
translation_lookup = reviewed_translations.set_index("id")

for i, article_id in enumerate(pending_ids, 1):
    print("\n" + "=" * 100)
    print(f"ARTICLE {i}/10: {article_id}")

    print("\nHINDI ORIGINAL:\n")
    print(source_lookup.loc[article_id, "text"])

    print("\nCURRENT ENGLISH TRANSLATION:\n")
    print(translation_lookup.loc[
        article_id, "complete_english_translation"
    ])

print("\nBatch 4 article count:", len(pending_ids))


ARTICLE 1/10: 050713_us_un

HINDI ORIGINAL:

जी-4 में भारत के साथ-साथ ब्राज़ील, जापान और जर्मनी शामिल हैं. इन देशों ने सुरक्षा परिषद के सदस्यों की संख्या 15 से बढ़ाकर 25 करने का प्रस्ताव रखा था. अमरीका का कहना है कि सदस्य देश इस मुद्दे पर बँटे हुए हैं. विदेश मंत्रालय की एक प्रवक्ता शीरीन ताहिर खेली ने बताया कि अमरीका सुरक्षा परिषद के विस्तार के लिए काम करेगा लेकिन सही समय पर सही तरीक़े से. जी-4 के देशों ने सोमवार को यह प्रस्ताव पेश किया था. जिस पर बहस चल रही है. प्रस्ताव पर मतदान कब होगा, ये अभी तय नहीं है. लेकिन इस प्रस्ताव के पारित होने के लिए दो तिहाई मतों की आवश्यकता होगी. प्रस्ताव जी-4 के देशों के प्रस्ताव में कहा गया है कि सुरक्षा परिषद में 10 और सीटें बढ़ाई जाएँ, जिनमें से छह सीटें नए स्थायी सदस्यों के लिए हो. प्रस्ताव के मुताबिक़ नए स्थायी सदस्यों में जी-4 के देशों के अलावा अफ़्रीकी देश भी शामिल होंगे. जी-4 के देश फ़िलहाल बिना वीटो के अधिकार के स्थायी सदस्यता के लिए राज़ी हैं. अमरीका के रुख़ से जी-4 के देशों को तगड़ा झटका लगा है, जिन्होंने इस प्रस्ताव के पारित होने की उम्मीद ल

In [188]:
manual_corrections.update({

"050901_gohana_firsthand": """The situation was the same in every house we visited. In broad daylight, more than 50 houses in the middle of the town were set on fire while the administration and the city watched silently.

Krishna's house is visible as soon as one enters the first lane. She has two young daughters and had collected belongings for the wedding of one of them. Those belongings now lie in ashes. Krishna said, "I had bought some jewellery and new clothes for my daughter's wedding. They took some things away, and everything else was burnt. Not even a single grain of food is left."

Roshni had worked hard to educate her children. With her lifelong savings, she had built a permanent house only a few months earlier and had also arranged her son's wedding. Now every room in her house is filled with ashes. She said that her family had left the house out of fear, but the attackers had burnt it anyway. She was particularly distressed because she had served the people who burnt her house and their parents throughout her life, yet they showed no mercy to her, a widow.

The violence in front of the police was not sudden. A community panchayat representing 12 villages in the area, attended only by upper-caste people, had issued an ultimatum 48 hours earlier before carrying out the act. The administration knew about the threat. Instead of increasing security in the Dalit settlement, however, the police and local administration advised Dalit residents to leave their homes and stay elsewhere for a few days.

Eyewitness Sohan Lal said that everything happened in the presence of the police. He said the police arrived at six in the morning and told Dalit residents to leave the settlement. When their houses were set on fire in the afternoon, police personnel and senior officials were present near the site.

Haryana Chief Minister Bhupinder Singh Hooda, speaking in Delhi, said that the administration had anticipated trouble and was trying to resolve the issue. However, he said, some miscreants set fire to five Dalit houses, and the fire spread to the other houses.

The Gohana incident raises questions not only about law and order but also about the social structure. There had been many previous cases of atrocities against Dalits. In Haryana, these included the killing of five Dalits on Dussehra and the social boycott of Dalits in Palwal. Many other incidents never get reported.

No person responsible for such incidents has been punished so far, the report says. Self-appointed panchayats behind these incidents have often wielded enough influence to make the law yield to them. Even after 58 years, the alliance between political power and feudalism remains strong. The report questions why those who speak loudly about economic reforms discuss social reforms only on special occasions.""",

"051030_delhi_blasts": """The UN Secretary-General described the attacks as shocking. Britain said the blasts showed that extremists had no regard for human life. France said such activities could not be justified under any circumstances.

There was no official confirmation of the number of people killed in the blasts, but reports said that 60 people had died and 200 had been injured. A high alert was declared after the explosions, and the government advised people to remain indoors. Police were deployed at the city's main intersections and outside important buildings.

No group had claimed responsibility for the blasts. Ten people had been detained in connection with the incident. The explosions took place ahead of Diwali and Eid, when markets are usually crowded.

Prime Minister Manmohan Singh cut short his visit to the Northeast and returned to Delhi. An emergency meeting of the Home Ministry was also held. Minister of State for Home Affairs Sriprakash Jaiswal said that such incidents would receive a strong response. He said steps had been taken to prevent them, but the country's large size made the task difficult.

Delhi Chief Minister Sheila Dikshit announced compensation of Rs 3 lakh for the families of those killed and Rs 50,000 for each injured person.

The first blast occurred at 5:30 p.m. Indian time in the crowded Paharganj area. Eyewitnesses said it was powerful and caused many casualties. Paharganj is close to New Delhi railway station and is usually crowded with people in its narrow lanes.

The second blast occurred around 6 p.m. in Sarojini Nagar, about half an hour later. The explosion set off fireworks that had been displayed for sale in the market. According to BBC correspondent Paul Danhar, at least three people were killed in the blast in this area.

The third blast took place inside a bus in Govindpuri in south Delhi. The injured were admitted to Ram Manohar Lohia Hospital. Because of the large number of injured people, the leave of medical staff was cancelled.""",

"051105_sahubuddin_arrest": """Shahabuddin, a Member of Parliament from the Siwan region of Bihar, was arrested by a Bihar Police team at his MP residence in Delhi, according to PTI.

Earlier, the Bihar government had issued a nationwide alert concerning his arrest. Bihar Police had informed police forces in all states that a non-bailable warrant had been issued against Shahabuddin and that he should be arrested wherever he was found.

When Bihar Police had previously travelled to Delhi to arrest him, Shahabuddin was in the capital attending Parliament's monsoon session. He subsequently went missing. However, he continued speaking to journalists. In interviews, he said he was not running away from the law and had approached the court to challenge the charges against him.

There were eight non-bailable warrants against Shahabuddin, but the particular arrest warrant that police were pursuing concerned a weapons-related case.

The administration had asked him to surrender all licensed weapons held by him and his family. An arrest warrant was issued because he did not comply with this request.

The Bihar government was also under pressure to follow an Election Commission directive asking it to arrest as soon as possible people against whom non-bailable warrants were pending.

Shahabuddin had been externed from the district during the previous election.""",

"051121_aids_india": """According to the report, pregnant women still face a very high risk of HIV infection. It also says that the number of people living with HIV has increased worldwide. The report was released internationally on Monday.

In India, 5.1 million people were reported to be living with HIV. According to UNAIDS, infection rates had stabilised in the southern and western states of Tamil Nadu, Andhra Pradesh, Karnataka and Maharashtra. However, pregnant women in densely populated and poorer northern states such as Bihar and Uttar Pradesh faced a particularly high risk.

The report said that more than one per cent of women in Tamil Nadu, Andhra Pradesh, Karnataka and Maharashtra, as well as in the northeastern states of Nagaland and Manipur, were found to be infected with HIV.

UNAIDS said that many of these women were acquiring the virus from their husbands, and that the spread of infection was extending beyond urban areas. The report said that the men had contracted HIV through unprotected sex with sex workers.

A 2003 survey found HIV infection among 14 per cent of sex workers in Karnataka and 19 per cent in Andhra Pradesh. A survey in Mysore found that 26 per cent of sex workers were infected. The report also gave figures on condom use: 14 per cent regularly used condoms with their clients, while 91 per cent never used condoms with their husbands or partners.

**Benefits of education**

A survey in Kolkata's Sonagachi red-light area suggested that education about safer sex could help reduce HIV transmission. Condom use there had increased by 85 per cent, and the reported HIV infection rate fell from 11 per cent in 2001 to 4 per cent in 2004.

In Mumbai, where similar education had not been provided, the infection rate among sex workers remained at 52 per cent or higher.

The report also highlighted the growing risk of HIV transmission among people who inject drugs in India, including the possibility of further transmission through sexual contact.""",

"060120_speaker_cashcamera": """The Supreme Court had issued a notice to Lok Sabha Speaker Somnath Chatterjee after dismissed MPs appealed against their removal. An all-party meeting was held to discuss the matter.

Somnath Chatterjee said that all parties were of the view that the Speaker should neither respond to the Supreme Court's notice nor appear before the court in connection with the dismissals.

He said that the BJP had boycotted the proceedings of the Lok Sabha over the issue. However, BJP leader Vijay Kumar Malhotra also believed that the decision of the House was paramount. Malhotra suggested that the Speaker did not need to appear personally, but that a government lawyer could present his position before the court.

In this case, the Lok Sabha had decided to terminate the membership of 10 MPs, while the Rajya Sabha had decided to terminate the membership of one of its members.

**The controversy**

Last month, a television channel showed 11 MPs allegedly accepting bribes in exchange for asking questions in Parliament. Following the controversy, special committees were formed in both Houses. These committees recommended the dismissal of the MPs. After a vote, it was decided that their parliamentary membership should be terminated.

The dismissed MPs then approached the Supreme Court, which issued a notice to the Lok Sabha Speaker.

The MPs argued that the court's intervention amounted to interference in Parliament's domain. They referred to an important Supreme Court judgment from the 1990s concerning the Jharkhand Mukti Morcha, in which the court had said that it had no role in matters relating to activities within Parliament.

The Speaker emphasised that he was not seeking confrontation with the judiciary. However, this was the second time in recent months that MPs had objected to the courts' position. Earlier, when courts had raised objections to the proposal for reservations in private educational institutions, Parliament had considered it appropriate to pass a new law.

Public attention is now focused on whether Parliament and the judiciary are moving towards a confrontation."""
})

print("Batch 4 corrections added:", 5)
print("Total correction entries:", len(manual_corrections))

Batch 4 corrections added: 5
Total correction entries: 59


In [189]:
corrected_count = 0

for article_id, corrected_text in manual_corrections.items():
    mask = reviewed_translations["id"] == article_id

    if mask.any():
        reviewed_translations.loc[
            mask, "complete_english_translation"
        ] = corrected_text

        reviewed_translations.loc[
            mask, "manual_review_status"
        ] = "Manually corrected"

        corrected_count += 1

print("Corrections applied:", corrected_count)

print("\nReview status:")
print(reviewed_translations["manual_review_status"].value_counts())

reviewed_translations.to_csv(
    "/content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv",
    index=False
)

print("\nSaved successfully.")

Corrections applied: 59

Review status:
manual_review_status
Pending review                    183
Manually corrected                 59
Human reviewed - no correction      8
Name: count, dtype: int64

Saved successfully.


In [190]:
pending_ids = reviewed_translations.loc[
    reviewed_translations["manual_review_status"] == "Pending review",
    "id"
].head(10).tolist()

source_lookup = df_250.set_index("id")
translation_lookup = reviewed_translations.set_index("id")

for i, article_id in enumerate(pending_ids, 1):
    print("\n" + "=" * 100)
    print(f"ARTICLE {i}/10: {article_id}")

    print("\nHINDI ORIGINAL:\n")
    print(source_lookup.loc[article_id, "text"])

    print("\nCURRENT ENGLISH TRANSLATION:\n")
    print(translation_lookup.loc[
        article_id, "complete_english_translation"
    ])

print("\nBatch 5 article count:", len(pending_ids))


ARTICLE 1/10: 050713_us_un

HINDI ORIGINAL:

जी-4 में भारत के साथ-साथ ब्राज़ील, जापान और जर्मनी शामिल हैं. इन देशों ने सुरक्षा परिषद के सदस्यों की संख्या 15 से बढ़ाकर 25 करने का प्रस्ताव रखा था. अमरीका का कहना है कि सदस्य देश इस मुद्दे पर बँटे हुए हैं. विदेश मंत्रालय की एक प्रवक्ता शीरीन ताहिर खेली ने बताया कि अमरीका सुरक्षा परिषद के विस्तार के लिए काम करेगा लेकिन सही समय पर सही तरीक़े से. जी-4 के देशों ने सोमवार को यह प्रस्ताव पेश किया था. जिस पर बहस चल रही है. प्रस्ताव पर मतदान कब होगा, ये अभी तय नहीं है. लेकिन इस प्रस्ताव के पारित होने के लिए दो तिहाई मतों की आवश्यकता होगी. प्रस्ताव जी-4 के देशों के प्रस्ताव में कहा गया है कि सुरक्षा परिषद में 10 और सीटें बढ़ाई जाएँ, जिनमें से छह सीटें नए स्थायी सदस्यों के लिए हो. प्रस्ताव के मुताबिक़ नए स्थायी सदस्यों में जी-4 के देशों के अलावा अफ़्रीकी देश भी शामिल होंगे. जी-4 के देश फ़िलहाल बिना वीटो के अधिकार के स्थायी सदस्यता के लिए राज़ी हैं. अमरीका के रुख़ से जी-4 के देशों को तगड़ा झटका लगा है, जिन्होंने इस प्रस्ताव के पारित होने की उम्मीद ल

In [191]:
# Get the current translations
def get_translation(article_id):
    return reviewed_translations.loc[
        reviewed_translations["id"] == article_id,
        "complete_english_translation"
    ].iloc[0]


# 1. Correct the meaning: a bomb was being planted again
text_iraq = get_translation("051017_iraq_rebels")
text_iraq = text_iraq.replace(
    "The US statement said the bomb was being re-dropped",
    "The US statement said another bomb was being planted there"
)

# 2. Correct terminology and improve wording
text_nepal = """The Maoists have announced a boycott of the elections, increasing fears of violence and creating panic among candidates. The withdrawal of candidates means that there will be no candidate for nearly 1,000 of the 4,146 seats to be contested in the February 8 elections.

The administration has placed hundreds of candidates in secure buildings to protect them from Maoist attacks. In the Janakpurdham area, 70 candidates are being kept under tight security at a government training centre.

**Candidates missing**

Reports from across the country say that worried wives and frightened relatives are pressuring candidates to withdraw their nominations. There are also reports that officials have pressured some people to contest the elections. Sixty per cent of the seats reserved for women have no candidates. In several districts, there is only one candidate, with no one contesting against them.

Election Commission spokesperson Tejmuni Bajracharya said, "Elections will be held in only 36 municipalities because there are no candidates in the remaining 22 municipalities."

The government of King Gyanendra says that the elections will clear the way for further progress towards democracy.

**Opposition boycott**

Besides the Maoists, the major opposition parties have also decided to boycott the elections. The opposition says the King is holding them to legitimise his grip on power. King Gyanendra took direct control of the government in February 2005.

In Kathmandu, a rebel leader directly warned people taking part in the elections that they would face serious consequences if they did so. King Gyanendra has promised to hold general elections in Nepal by mid-2007."""

# 3. Clarify the Human Rights Watch statement
text_taliban = get_translation("060603_taliban_karzai")
text_taliban = text_taliban.replace(
    "many of the police officers who are in talks to be promoted are guilty of human rights violations.",
    "many of the police officers being considered for promotion were responsible for human rights violations."
)

# 4. Correct football record and player names
text_worldcup = get_translation("060709_wordcup_final2006")
text_worldcup = text_worldcup.replace(
    "of which 17 have been against Italy, seven against France and the remaining eight have ended in draws.",
    "with Italy winning 17, France winning seven and the remaining eight ending in draws."
)
text_worldcup = text_worldcup.replace("Thierry Onri", "Thierry Henry")
text_worldcup = text_worldcup.replace("Liliang Thuriam", "Lilian Thuram")
text_worldcup = text_worldcup.replace(
    "The match was won 1-2 by France with the golden goal.",
    "France won the match 2-1 with a golden goal."
)

manual_corrections.update({
    "051017_iraq_rebels": text_iraq,
    "060129_nepal_candidate_rj": text_nepal,
    "060603_taliban_karzai": text_taliban,
    "060709_wordcup_final2006": text_worldcup
})

# These were checked and accepted without substantive correction
accepted_batch5 = [
    "050713_us_un",
    "051209_tamilnadu_cyclone",
    "060106_rajasthan_rail",
    "060117_indopak_cbm",
    "060530_reservation_strike",
    "060715_bushputin_g8"
]

print("Corrections added:", 4)
print("Accepted without correction:", len(accepted_batch5))
print("Total correction entries:", len(manual_corrections))

Corrections added: 4
Accepted without correction: 6
Total correction entries: 63


In [192]:
unverified_batch3_ids = [
    "050420_pope_muslims",
    "050506_india_coach",
    "050523_bihar_nda",
    "050606_kabir_bedi",
    "050625_building_collapse",
    "050713_bombsuspect_family"
]

base_path = (
    "/content/drive/MyDrive/FactIndic_PersonB_CleanRun/"
    "09_sentence_level_translation/FactIndic_English_250_Corrected.csv"
)

base_translations = pd.read_csv(base_path)

print("Base file columns:")
print(base_translations.columns.tolist())

print("\nUnverified IDs found in review tracker:")
print(
    reviewed_translations[
        reviewed_translations["id"].isin(unverified_batch3_ids)
    ][["id", "manual_review_status"]].to_string(index=False)
)

print("\nUnverified IDs found in base translations:")
print(
    base_translations[
        base_translations["id"].isin(unverified_batch3_ids)
    ].to_string(index=False)
)

Base file columns:
['id', 'complete_english_translation', 'reconstructed_sentence_count']

Unverified IDs found in review tracker:
                       id manual_review_status
      050420_pope_muslims   Manually corrected
       050506_india_coach   Manually corrected
         050523_bihar_nda   Manually corrected
        050606_kabir_bedi   Manually corrected
 050625_building_collapse   Manually corrected
050713_bombsuspect_family   Manually corrected

Unverified IDs found in base translations:
                       id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [193]:
# Six entries that must not be treated as verified
unverified_batch3_ids = [
    "050420_pope_muslims",
    "050506_india_coach",
    "050523_bihar_nda",
    "050606_kabir_bedi",
    "050625_building_collapse",
    "050713_bombsuspect_family"
]

# Load original machine translations
base_path = (
    "/content/drive/MyDrive/FactIndic_PersonB_CleanRun/"
    "09_sentence_level_translation/FactIndic_English_250_Corrected.csv"
)

base_translations = pd.read_csv(base_path)
base_lookup = base_translations.set_index("id")[
    "complete_english_translation"
]

# Remove unverified corrections from the dictionary
for article_id in unverified_batch3_ids:
    manual_corrections.pop(article_id, None)

# Restore original translations wherever the ID exists
restored_ids = []

for article_id in unverified_batch3_ids:
    if article_id in base_lookup.index:
        mask = reviewed_translations["id"] == article_id

        reviewed_translations.loc[
            mask, "complete_english_translation"
        ] = base_lookup.loc[article_id]

        reviewed_translations.loc[
            mask, "manual_review_status"
        ] = "Pending review"

        restored_ids.append(article_id)

# Apply all remaining verified corrections
for article_id, corrected_text in manual_corrections.items():
    mask = reviewed_translations["id"] == article_id

    if mask.any():
        reviewed_translations.loc[
            mask, "complete_english_translation"
        ] = corrected_text

        reviewed_translations.loc[
            mask, "manual_review_status"
        ] = "Manually corrected"

# Mark Batch 4 and Batch 5 accepted articles
accepted_ids = [
    "050713_us_un",
    "051209_tamilnadu_cyclone",
    "060106_rajasthan_rail",
    "060117_indopak_cbm",
    "060530_reservation_strike",
    "060715_bushputin_g8"
]

reviewed_translations.loc[
    reviewed_translations["id"].isin(accepted_ids),
    "manual_review_status"
] = "Human reviewed - no correction"

# Save
save_path = (
    "/content/drive/MyDrive/FactIndic_PersonB_CleanRun/"
    "09_sentence_level_translation/FactIndic_English_250_Reviewed.csv"
)

reviewed_translations.to_csv(save_path, index=False)

print("Original translations restored:", len(restored_ids))
print("Restored IDs:", restored_ids)
print("Unverified IDs not found:", [
    x for x in unverified_batch3_ids if x not in restored_ids
])

print("\nUpdated review status:")
print(reviewed_translations["manual_review_status"].value_counts())

print("\nSaved successfully.")

Original translations restored: 6
Restored IDs: ['050420_pope_muslims', '050506_india_coach', '050523_bihar_nda', '050606_kabir_bedi', '050625_building_collapse', '050713_bombsuspect_family']
Unverified IDs not found: []

Updated review status:
manual_review_status
Pending review                    179
Manually corrected                 57
Human reviewed - no correction     14
Name: count, dtype: int64

Saved successfully.


In [194]:
pending_ids = reviewed_translations.loc[
    reviewed_translations["manual_review_status"] == "Pending review",
    "id"
].head(10).tolist()

source_lookup = df_250.set_index("id")
translation_lookup = reviewed_translations.set_index("id")

for i, article_id in enumerate(pending_ids, 1):
    print("\n" + "=" * 100)
    print(f"ARTICLE {i}/10: {article_id}")

    print("\nHINDI ORIGINAL:\n")
    print(source_lookup.loc[article_id, "text"])

    print("\nCURRENT ENGLISH TRANSLATION:\n")
    print(translation_lookup.loc[
        article_id, "complete_english_translation"
    ])

print("\nBatch 6 article count:", len(pending_ids))


ARTICLE 1/10: 050420_pope_muslims

HINDI ORIGINAL:

शंका जताई गई है कि नए पोप शायद ऐसा करने में उतना विश्वास नहीं रखते. तुर्की में तो उनके चुनाव पर हैरत ज़ाहिर की गई है क्योंकि उन्होंने यूरोपीय संघ में तुर्की की सदस्यता का ये कहकर विरोध किया था कि बेहतर हो यदि तुर्की अरब देशों की सदस्यता हासिल करे. पोप बेनेडिक्ट 16वें के सामने जो चुनौतियां हैं उनमें प्रमुख है वैटिकन का इस्लाम के प्रति रवैया. कुछ कैथोलिक ऐसे हैं जो पोप जॉन पॉल द्वितीय की नीति को अपनाते हुए दूसरे संप्रदायों के साथ बातचीत को आगे बढ़ाना चाहते हैं. लेकिन कुछ दूसरे हैं जो इस सोच से सहमत नहीं हैं. नए पोप जब कार्डिनल के पद पर थे तब उनकी भूमिका थी प्रमुख धर्मशास्त्री की और उन्हें एक तरह वैटिकन की नीतियों के रखवाले के रूप में देखा जाता था. उनका मानना था कि कैथोलिक इस सोच से उपर उठें कि सभी धर्म समान हैं, एक जैसे हैं. उनकी सोच थी कि दूसरे धर्मों से बातचीत हो लेकिन इस आधार पर कि कैथोलिक धर्म सबसे उपर है. जब उन्होंने यूरोपीय संघ में तुर्की की सदस्यता का विरोध किया तो उसके पीछे की सोच ये थी कि यूरोप में पहले से ही लगभग डेढ़ करोड़ म

In [195]:
batch6_corrections = {
    "050420_pope_muslims": """There were doubts about whether the new pope would be as committed to pursuing dialogue. His election was met with surprise in Turkey because he had opposed Turkey's membership of the European Union, saying that it would be better for Turkey to become a member of the Arab countries. One of the main challenges facing Pope Benedict XVI is the Vatican's attitude towards Islam. Some Catholics want to continue dialogue with other denominations, following the policy of Pope John Paul II, while others disagree. When the new pope was a cardinal, he served as the chief theologian and was regarded as a guardian of Vatican policies. He believed that Catholics should move beyond the idea that all religions are equal. He supported dialogue with other religions, but on the basis that Catholicism was superior. His opposition to Turkey's EU membership was based on the view that Europe already had around 15 million Muslims and that the entry of a Muslim-majority country such as Turkey could affect Europe's Christian identity. Perhaps this is why his election has been criticised in almost every Turkish newspaper. One headline read, "Anti-Turkish cardinal elected pope," while another said, "The new pope is against Turkey." The new pope had said that Turkey regarded itself as a secular nation, but that it was in fact an Islamic country. It remains to be seen whether his attitude will change after taking office. In a world where tensions between Islam and the West already exist, any change in the Vatican's thinking after September 11 could have serious consequences.""",

    "050506_india_coach": """The committee also includes former India captain Sunil Gavaskar and former cricketers Ravi Shastri and Srinivas Venkataraghavan. Other members include BCCI president Ranvir Singh Mahendra, former president Jagmohan Dalmiya and board secretary Karunakaran Nair. Ranvir Singh Mahendra will chair the committee, which will hold its first meeting on Sunday. New Zealand's John Wright's tenure as India's coach ended with the series against Pakistan. The board has received applications from Australia's Greg Chappell and Dean Jones. Tom Moody and Dave Whatmore are also in the running for the post. Former coach John Wright is believed to support Moody's candidature. However, the 39-year-old Moody is also being considered for the post of Sri Lanka's coach. He is currently the director of cricket at the English county team Worcestershire. Greg Chappell's name had also come up for the Indian coaching job four-and-a-half years ago, but John Wright was selected. Chappell now says he does not want to dwell on the past. "It is an honour to coach any national team," he said. "But it is also difficult because there are many challenges." Dean Jones, on the other hand, does not have coaching experience. Since ending his cricket career in 1994, however, he has travelled around the world as a television commentator. "I have been to India 58 times. I like India and its people very much, so I am keen on this position," Jones said. Dave Whatmore is currently Bangladesh's coach, and his contract has recently been extended. He is nevertheless enthusiastic about the possibility of coaching the Indian team.""",

    "050523_bihar_nda": """Addressing reporters, former prime minister Atal Bihari Vajpayee said that the government had acted quickly to dissolve the Bihar Legislative Assembly and should show the same urgency in holding elections. He said that elections should be held early and fairly. BJP president Lal Krishna Advani described the move as a repeat of 1975, saying that India had once again been pushed into an Emergency-like situation. On Monday morning, Home Minister Shivraj Patil told reporters that the President had approved the dissolution of the Bihar Assembly.

Why was the Assembly dissolved?

At a cabinet meeting that continued late into Sunday night, the decision to dissolve the Bihar Assembly was taken at around 1:30 a.m. The cabinet based its decision on a report by Bihar Governor Buta Singh. The report said that horse-trading among legislators had been taking place amid recent political activity, leaving dissolution of the Assembly as the only option. In an interview with the BBC, Governor Buta Singh said that legislators were being taken to other states as part of efforts to form a government. He said this was wrong and that he had therefore recommended dissolving the Assembly. He added that he had waited for three months for political parties to come together and form a democratic government in the state, but no result had emerged. In view of recent developments, he said, dissolution had become necessary and there had been no other option.

The Left parties also supported the government's decision. D. Raja of the Communist Party of India told the BBC that, after political developments in the state had become immoral and unconstitutional, the government had no alternative. However, Janata Dal (United) president Nitish Kumar dismissed the allegation as nonsense. He said that efforts to form a government had begun in the first week of April. According to him, when the Centre realised that his group would be able to form the government, it called an emergency cabinet meeting and dissolved the Assembly as part of a conspiracy. He described the Bihar Governor as a spokesperson for the central government and demanded his immediate removal.

BJP vice-president Venkaiah Naidu also told reporters that the government's action showed that the Centre did not want a non-Congress government to come to power in any state. In protest against the dissolution, BJP workers burnt effigies of the central government and Ram Vilas Paswan in several places, including Patna. The NDA also called for a Bihar bandh on Tuesday.

What happens next?

The only option now appears to be elections, and political circles acknowledge that preparations must be made for them. However, there is no clarity about when they will be held. Senior journalist Sankarshan Thakur told the BBC in Delhi that, because of floods and rain in Bihar after July, elections were unlikely before September. He also doubted that the government would be able to hold elections before July. He said it was too early to predict the state's political equations over the next four to eight months.""",

    "050606_kabir_bedi": """Kabir Bedi was born and brought up in Delhi. The city watched him overcome difficult times and emerge as a talented actor. His acting career later took off in Mumbai, and he also made a mark in international cinema. As his career took him from Mumbai to various cities in Europe, memories of Delhi continued to come back to him. In a conversation with the BBC in London, Kabir Bedi spoke openly about himself, his films and his acting career.

"I am a Delhiite at heart and in reality. Memories of Delhi have always stayed with me and will remain with me throughout my life. I am very fond of Delhi's food because it is not just something to eat; it also reminds me of my childhood."

Meeting him, one could see how his distinctive appearance and commanding voice expressed his personality without any special effort. Kabir Bedi is among the few Indian actors to have made an international name for themselves, yet he remains connected to his hometown, Delhi, and hopes to return there one day.

Bedi is playing the important role of Khan Sahib in the musical For Pavilion at London's West End. He is both excited and satisfied with the role. He says it has given him an opportunity to develop his acting skills. "The role of Khan Sahib is, in a way, the soul of this musical."

Bedi believes that working in theatre is important and highly satisfying for an actor. Film-making is different: scenes may be shot at different points in the story, and actors often get opportunities to rehearse. Theatre, by contrast, involves a continuous emotional journey and offers a high level of professional satisfaction. Bedi says that performing before a live audience leaves no room for mistakes, which makes preparation and rehearsal especially important.

Bedi is also excited about playing the older Shah Jahan in the new film Taj Mahal. He considers it one of the most important roles of his acting career. In the film, after taking power, Aurangzeb imprisons Shah Jahan in Agra Fort. In his final days, Shah Jahan looks at the Taj Mahal through a window and tells his story to Jahanara. Bedi said, "This is one of the finest roles of my acting career. If my name becomes associated with this character, I will consider it a great achievement."

Bedi says he had wanted to become an architect, but family circumstances meant he had to work to help pay for his college education. He enrolled for a bachelor's degree in history at St Stephen's College in Delhi and began working at All India Radio and Doordarshan to earn some money. He says that this was where his film career began. He also continued working in theatre.

Asked what other ambitions he still had, Bedi said that he wished to return to Delhi and settle there. "My life's journey began in Delhi, and I wish to end it there. I am waiting for the moment when this wish comes true; only then will I feel complete." He is also working in several other international films.""",

    "050713_bombsuspect_family": """Police identified three alleged attackers whom they said were British citizens of Pakistani origin. According to police, the attacks were carried out by 22-year-old Shehzad Tanvir, who was born in Britain, 18-year-old Hasib Mir Hussain, who lived in Leeds, and 30-year-old Mohammed Sadiq Khan, also a resident of Leeds. Police had not yet made the identity of a fourth person public.

Bashir Ahmad, a relative of Shehzad Tanvir, said that his family was trying to find the strength to face the news that Shehzad might have been a suicide bomber. "This is not something he would do. This must be the work of forces that were using him as a shield," he said. Asked to say more, Ahmad replied, "We have lost everything." He said that his nephew had gone to Pakistan that year to study religion. Security experts said it was quite possible that someone else had been influencing these men.""",

    "060829_dhyanchand_hockey": """In 1928, in the Netherlands, Dhyan Chand's hockey stick was broken and examined to check whether it contained a magnet. In 1932, Japanese players claimed that glue was attached to his stick. In 1936, after the Indian team defeated Germany 8–1 at the Olympics, Hitler offered Dhyan Chand a senior position in the German army.

A few years ago, Dhyan Chand's late wife, Janaki Dhyan Chand, recalled the occasion. She said that people were standing around with the flags of their respective countries while the Indian team stood among them. He felt deeply saddened by the situation of his country. But once he stepped onto the field, he began scoring goals.

Simplicity

Major Dhyan Chand's patriotism and simple nature were reflected in the way he played hockey. His son, former Indian captain Ashok Kumar, remembers that his father led a simple life and maintained his friendships. He would sometimes scold his children when they lost. Ashok said that his father gave all the credit to his teammates when the team won, but took responsibility himself when the team lost.

Former Olympian Commander Nandy Singh, who played alongside Dhyan Chand, recalls that people often said the ball seemed to stick to his hockey stick. He says this was a misconception: Dhyan Chand did not simply keep the ball close or rely on dribbling. His special strength was scoring goals, and he could score from almost any angle.

Major Dhyan Chand was also the coach and selector of Harbinder Singh, the fast-running centre-forward who represented India from the Tokyo Olympics through the Munich Olympics. Harbinder Singh recalls a training camp in Patiala. Whenever a player missed a goal, Dhyan Chand would become very angry. Harbinder said that the lessons he received from Dhyan Chand had been extremely valuable throughout his hockey career.

Popularity

In 1952, after independence and Partition, Dhyan Chand travelled through Pakistan on his way to Afghanistan. Former international hockey referee K. G. Kakkar witnessed Dhyan Chand's popularity during the journey. He recalled that when they reached Lahore, the crowd was so large that platform tickets ran out. Everyone wanted to meet Dhyan Chand. By the time the train reached Peshawar, it was three hours late.

The final journey of this hockey legend, whose fans had gathered all along the route from Delhi to Peshawar, ended in his hometown of Jhansi, at the ground where he had first learned to display his skill with a hockey stick.""",

    "061004_askus_geyser": """A geyser is a type of hot spring that erupts from time to time, releasing hot water and steam. Geysers require particular geological and underground water conditions, which exist in only a few places on Earth.

How do geysers form? Water from the Earth's surface seeps underground and comes into contact with rocks heated by magma. The water becomes hot, rises and eventually bursts out through cracks in the rocks. There are about a thousand geysers in the world, half of them in Yellowstone National Park in the United States. The domestic electrical water heater is also called a geyser, perhaps because of this natural phenomenon.

Jitendra Kumar Paswan from Pahlam village in Saharsa, Bihar, asks which country first hosted the Olympic Games. The ancient Olympic Games were first held in Olympia, Greece. The modern Olympic Games began in 1896, when they were revived in Greece. Since then, they have generally been held every four years, except in 1940 and 1944, when they could not take place because of the Second World War.

Ramchandra Mahato from Khetko, Hazaribagh, Jharkhand, wants to know which is the world's most expensive book and who wrote it. Geoffrey Chaucer's The Canterbury Tales is considered one of the most expensive books. At an auction in 1998, a copy of its first edition sold for £460,000. It was bought by the London bookseller Maggs Brothers. The first edition was printed in 1477, and only about a dozen copies were produced.

Parvez Alam Khan from Pathrahi village in Madhubani, Bihar, asks what kind of institution the National School of Drama is. The National School of Drama in Delhi is a major institution for theatre training. It was established by the Sangeet Natak Akademi in 1959. Its three-year course covers subjects such as Eastern and Western theatre literature, acting, mime, yoga, music, theatre technology, stage design, costume design and make-up in the first year. In the second year, students specialise in either acting or design and direction. They receive a diploma after three years. Applicants must have a BA or an equivalent degree and some experience in theatre. Auditions are held in different parts of the country, and selected candidates are invited to Delhi for a workshop. Those who succeed in the workshop are admitted. The diploma is considered equivalent to a postgraduate degree.

Kallu from Nainaghat village in Darbhanga, Bihar, asks what HIV stands for, who discovered it and when, and which country has the highest number of people living with HIV. HIV stands for Human Immunodeficiency Virus. It weakens the body's immune system and can lead to AIDS, or Acquired Immunodeficiency Syndrome. It is believed that in 1984 Dr Robert Gallo of the US National Cancer Institute, Dr Luc Montagnier of the Pasteur Institute in Paris, and Dr Jay Levy of the University of California identified the retrovirus that causes AIDS. The three research groups initially gave it different names, but later agreed on the name HIV. According to the figures cited in the article, South Africa had the highest number of people infected with HIV, at about 5.3 million, followed by India, with about 5.1 million.

Sunil Gupta, writing through BBC Hindi.com, asks where Mangal Pandey originally came from. Mangal Pandey was from Nagwa village in Ballia district, Uttar Pradesh. He was a sepoy in the 5th Company of the 34th Bengal Native Infantry Regiment of the East India Company. Indian soldiers were protesting against the use of a new type of cartridge that was allegedly greased with cow and pig fat and had to be bitten open. There were other reasons for discontent with Company rule. On 29 March 1857, Mangal Pandey attacked a British officer at Barrackpore, near Calcutta. He was subsequently court-martialled and hanged. After this incident, discontent against the British spread and developed into the uprising of 1857."""
}

manual_corrections.update(batch6_corrections)

batch6_accepted_ids = [
    "050625_building_collapse",
    "061025_russia_broadcast",
    "061125_rajpakshe_arrived"
]

print("Batch 6 corrections added:", len(batch6_corrections))
print("Batch 6 accepted without correction:", len(batch6_accepted_ids))
print("Total correction entries:", len(manual_corrections))

Batch 6 corrections added: 7
Batch 6 accepted without correction: 3
Total correction entries: 64


In [196]:
# Apply all available manual corrections
for article_id, corrected_text in manual_corrections.items():
    mask = reviewed_translations["id"] == article_id
    reviewed_translations.loc[
        mask, "complete_english_translation"
    ] = corrected_text
    reviewed_translations.loc[
        mask, "manual_review_status"
    ] = "Manually corrected"

# Mark Batch 6 accepted articles
reviewed_translations.loc[
    reviewed_translations["id"].isin(batch6_accepted_ids),
    "manual_review_status"
] = "Human reviewed - no correction"

# Save updated review tracker
reviewed_translations.to_csv(
    "/content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv",
    index=False
)

print("Batch 6 corrections applied:", len(batch6_corrections))
print("Batch 6 accepted articles marked:", len(batch6_accepted_ids))
print("\nUpdated review status:")
print(reviewed_translations["manual_review_status"].value_counts())
print("\nSaved successfully.")

Batch 6 corrections applied: 7
Batch 6 accepted articles marked: 3

Updated review status:
manual_review_status
Pending review                    169
Manually corrected                 64
Human reviewed - no correction     17
Name: count, dtype: int64

Saved successfully.


In [197]:
pending_ids = reviewed_translations.loc[
    reviewed_translations["manual_review_status"] == "Pending review",
    "id"
].head(10).tolist()

source_lookup = df_250.set_index("id")
translation_lookup = reviewed_translations.set_index("id")

for i, article_id in enumerate(pending_ids, 1):
    print("\n" + "=" * 100)
    print(f"ARTICLE {i}/10: {article_id}")

    print("\nHINDI ORIGINAL:\n")
    print(source_lookup.loc[article_id, "text"])

    print("\nCURRENT ENGLISH TRANSLATION:\n")
    print(translation_lookup.loc[
        article_id, "complete_english_translation"
    ])


ARTICLE 1/10: 061206_gennext_indopak_students

HINDI ORIGINAL:

छोड़ो कल की बातें, कल की बात पुरानी,नए दौर में लिखेंगे, मिलकर नई कहानी. यह मौक़ा था दिल्ली और लाहौर के कुछ स्कूली बच्चों के बीच वेबकास्टिंग के ज़रिए बातचीत का जिसमें ग़रम होती बहस को लाहौर के एक बच्चे ने ऊपर बताए वाक्य से संभाला और दोनों ओर की अगली पीढ़ी को एक रास्ता सुझाया. दरअसल, इन दिनों बीबीसी वर्ल्ड सर्विस की ओर से दो दिसंबर से नौ दिसंबर के बीच एक विशेष सप्ताह का आयोजन चल रहा है. यह विशेष सप्ताह अगली पीढ़ी (जेनरेशन नेक्स्ट) से जुड़े विषयों पर केंद्रित है. इसी श्रृंखला में बीबीसी हिंदी की टीम ने दिल्ली के केंद्रीय विद्यालय, आरके पुरम के कुछ छात्र-छात्राओं से लाहौर के क्रेसेंट मॉडल स्कूल के विद्यार्थियों की बातचीत कराई. दोनों ओर के बच्चों ने पहली बार अपने स्कूलों में बैठे-बैठे ही एक-दूसरे को देखा, हंसी और चिंताएँ बाँटीं और कई विषयों पर बातचीत की और यह सब संभव हुआ वेबकास्टिंग के ज़रिए. इस कार्यक्रम के प्रोड्यूसर थे बीबीसी हिंदी के संवाददाता अपूर्वकृष्ण जिन्होंने दिल्ली से इस कार्यक्रम का संचालन किया. लाहौर में इसका संचा

In [198]:
batch7_corrections = {

"061206_gennext_indopak_students": """A webcasting conversation brought together schoolchildren from Delhi and Lahore. When the discussion became heated, a boy from Lahore used a familiar saying about leaving the past behind and writing a new story together. His words offered a way forward for the next generation on both sides.

The BBC World Service was organising a special week from December 2 to December 9, focusing on issues related to the next generation. As part of this series, the BBC Hindi team arranged an online conversation between students of Kendriya Vidyalaya, RK Puram, Delhi, and Crescent Model School, Lahore. For the first time, children on both sides saw one another while sitting in their own schools. They shared laughter and concerns and discussed several subjects through webcasting.

The programme was produced by BBC Hindi correspondent Apurva Krishna, who hosted it from Delhi. In Lahore, BBC Urdu correspondent Wusatullah Khan moderated the discussion.

**The conversation begins**

Initially, the children talked about ordinary school matters, but the discussion soon included serious and meaningful debates. It began with the Taj Mahal. The children from Delhi invited their counterparts from Lahore to visit India and promised to show them the Taj Mahal. They then asked what they would be shown if they visited Lahore. The answer was that they would be shown not just Lahore but the entire country, including its important buildings and places.

They then discussed cricket and shared experiences of punishments given by teachers, such as being made to stand with their hands raised or being asked to squat like a rooster. The children from Lahore praised Pakistani bowlers, while those from Delhi praised Sachin Tendulkar and Virender Sehwag. The Delhi students even suggested forming a joint team to win the World Cup, although the Lahore students did not seem very enthusiastic about the idea.

The atmosphere became lively again when the discussion turned to Shah Rukh Khan and Aishwarya Rai. The children in Lahore were great fans of these actors, while the Indian students knew little about Pakistani films. They asked whether Pakistan had made a film about Jinnah similar to Lage Raho Munna Bhai, which was about Gandhi. The answer was that a film about Jinnah had been made, but it was serious rather than a comedy.

The Delhi students also asked what the Lahore children thought about Gandhi and Partition. One student described Gandhi as a good person who had also made mistakes. The discussion then turned to how Indians viewed Jinnah. Questions and answers continued on subjects ranging from history and the Internet to co-education and Partition.

When the discussion began on Wednesday afternoon, NCERT Director Dr. Krishna Kumar was present with the students in Delhi, while historian Dr. Mubarak Ali was with the students in Lahore. Both guests agreed that the children were speaking carefully, almost like diplomats.

Dr. Krishna Kumar said that it was important to acknowledge that the history books taught on both sides were incomplete. He said that this needed to be addressed and that students in both countries should be taught a broader account of history. Dr. Mubarak Ali said that Gandhi was portrayed negatively in Pakistan and Jinnah in India, and that people needed to rise above such portrayals.

**And finally, a thank-you to the BBC**

The children had so much to say that the conversation seemed unlikely to end, but time was running out. As the programme concluded, they thanked the BBC, saying that something they had never imagined had become possible. They had also realised that people were different from the prejudices they had held.

The discussion ended with the thought that remembering only the painful experiences of the past would keep both sides divided. Instead, they wanted to leave the past behind and recognise that they were the future. They spoke of beginning afresh and improving relations. They also remembered familiar cultural symbols, including sevaiyan and the Qutub Minar. Finally, they renewed their promise of friendship and thanked one another with smiles and applause.""",

"061217_spl_sez_imballance": """According to figures from October 2006, 403 Special Economic Zones (SEZs) in India had either received formal approval or approval in principle. In principle, the main purpose of SEZs is to promote exports. However, given the rush by property developers to enter the SEZ business, property prices may rise sharply whether exports increase or not. The Reserve Bank of India has also expressed concern about this possibility.

The interest of property developers can be understood from the fact that Haryana, a state favoured by property companies, has received approval for 46 SEZs. This is striking because ten per cent of all approved SEZs are in a state that has no seaports anywhere nearby.

Ports are important for SEZs because most goods are transported by sea rather than by air. Coastal states are therefore likely to benefit more. It is no coincidence that nearly 25 per cent of all approved SEZs are located in just two southern coastal states: Andhra Pradesh, with 54, and Karnataka, with 46. Maharashtra has 75 proposals, while Gujarat has 30.

Overall, Andhra Pradesh, Karnataka, Maharashtra and Gujarat together account for more than half of the approved SEZs. Uttar Pradesh has only 18, Madhya Pradesh 10 and Rajasthan 11. Bihar, which has the greatest need for development, has not even one SEZ proposal.

Karnataka is likely to emerge as a particularly important state in this situation. Even now, around three per cent of India's national GDP comes from Bangalore alone, largely because of software exporters such as Infosys. New SEZs are expected to bring further investment to the state.

**The growing gap between North and South**

North India has traditionally lagged behind in industrial development. Falling behind in the development of SEZs could push the region even further back. According to a National Productivity Council study, Bihar ranked seventh among 15 states in terms of law and order, while Uttar Pradesh ranked ninth. Karnataka ranked first, a position reflected in Bangalore's development.

Regional development should also be considered when planning SEZs. Northern states do not have seaports, and that cannot be changed. However, software exports do not require ports; business can be conducted through computers and Internet connections. It would therefore make sense to encourage software companies to establish themselves in North India, while manufacturing units could be located in the South, Gujarat or Maharashtra.

Why cannot Bihar and Uttar Pradesh become important centres for software-related SEZs? Can the Indian economy really be considered developed without the development of these large states in terms of population? These are important questions about regional development. Sadly, nobody seems to have the time to find answers—not policymakers, not politicians and certainly not property developers.""",

"061219_spl_sez_orissa": """Various organisations have formed the Posco Pratirodh Sangram Samiti (Posco Resistance Struggle Committee) to oppose the project. The proposed SEZ at Paradip is still awaiting final approval from the central government, although it has received approval in principle.

According to the plan, Posco's SEZ will cover 4,004 acres. Of this land, 1,135 acres belong to the government. The main difficulty, however, is acquiring land owned by local residents. The proposed Posco plant is expected to produce two million tonnes of steel annually.

**The proposal**

The state government has approved 14 SEZ proposals, including Posco's. Five of them have received formal approval from the central government. Tata Steel plans to establish an SEZ on 1,173 acres in Gopalpur, but the project has not yet received approval.

According to estimates, setting up all 14 SEZs would result in the loss of 26,000 acres of agricultural land. Thousands of people would also be displaced, which is a major part of the controversy.

**Opposition**

Janardan Pati, state secretary of the Communist Party of India, says, "This is nothing but an attempt to create a new class of landlords. The Agriculture Minister himself has said in the state assembly that agricultural land is decreasing every year. In such circumstances, giving away land for SEZs is a crime."

The state government, however, says that the situation is being unnecessarily exaggerated. State Industries Minister Vishwa Bhushan Harichandan says, "We have ensured that as little fertile land as possible is acquired for the SEZs. Only ten per cent of the land to be acquired is fertile."

Opponents of the SEZs say that the projects will cause the state government to lose revenue worth around one lakh crore rupees. Finance Minister Prafulla Chandra, on the other hand, says, "There will certainly be some losses initially, but these will be offset by an investment of Rs 18,000 crore. In addition, around one lakh people will get employment." """,

"070206_guj_parza": """Cinema owners fear that screening the film could lead to vandalism. Ahmedabad Police Commissioner J. Mahapatra had assured all cinema halls of security. However, at a meeting between the film's director Rahul Dholakia and members of the Multiplex Association, the owners told Dholakia that they could not screen the film.

The film was released across India on January 26 but was not released in Gujarat. Reacting to this decision, Dholakia said, "I am very upset. This is the story of a Gujarati boy. If the entire country can watch this film, why can't the people of Gujarat?"

Dholakia, who is himself from Gujarat, said, "I cannot understand how a film could shake the foundations of secular Gujarat." He said that cinema owners were afraid that some people might vandalise their theatres, which was why they were refusing to screen the film.

Most of Gujarat's cinema halls are located in Ahmedabad, Rajkot and Vadodara, and the film was not going to be screened in any of these cities. The Bajrang Dal had earlier threatened to prevent the film from being shown in Gujarat if it hurt Hindu sentiments.

Dholakia had arranged a special screening for cinema owners. However, when they suggested that the film should also be shown to Bajrang Dal leaders, he refused. A similar situation had occurred earlier with Fanaa, starring Aamir Khan.""",

"070209_wi_samuels": """Tony Dayal, a spokesperson for the cricket board, told BBC World Service Sports, "Whatever we do will be fair, transparent and honest. Any action we take will be in accordance with the law."

The police in Nagpur, India, claim to possess a recording of a conversation between Samuels and Mukesh Kochhar, an alleged bookmaker, before the Nagpur One-Day International. According to the police, Samuels was passing information about the team to Kochhar. The Board of Control for Cricket in India (BCCI) has forwarded the Nagpur police report to the International Cricket Council (ICC).

Samuels has denied doing anything wrong. The West Indies Cricket Board has not yet spoken to him or explained what kind of investigation will be conducted.

**A career at stake**

Tony Dayal said, "We are not going to take action on the basis of rumours and gossip because a young player's cricket career is at stake in this case."

He made it clear, however, that an investigation would take place and that appropriate action would be taken in accordance with ICC rules.

"We have been world champions for years. We have always worked for cricket and will continue to do so," he said.

Dayal added that the complaint and the available evidence would receive particular attention during the investigation. He said that a cricket committee might conduct the inquiry and that the West Indies Players' Association could also be involved, since Samuels is a member.

The World Cup is due to begin in the West Indies next month. Meanwhile, captain Brian Lara has said that the team stands by Samuels.""",

"070324_india_press": """The match result came in after 2 a.m. Indian time, but the newspapers seem to have waited a long time for it. Perhaps that is why some papers published only a brief report with a photograph on the front page.

The Hindi newspaper Dainik Jagran carried a full match report under the headline "Ticket punched, farewell still to come." The photograph showed Sachin Tendulkar walking back after being dismissed. The paper commented that it would be better if the so-called paper tigers stayed away from home for a few days.

Rashtriya Sahara's headline was "Demolished: India out after losing to Sri Lanka; hopes now rest on Bermuda's performance."

Some newspapers published a full account of the innings and expressed their concerns. The Times of India placed photographs of Sourav Ganguly and Sachin Tendulkar on its front page, alongside India's score after three wickets had fallen. Its headline read, "Back to the Wall for India," meaning that India's position had once again become difficult.

The Indian Express wrote that, at the time of publication, India's hopes were hanging by a fragile thread named Rahul Dravid, who was still fighting. Its headline was "A Sleepy Night of Ups and Downs."

The Statesman used a photograph of Ajit Agarkar with the headline "In Exit Mode," suggesting that India was heading towards elimination.

The Pioneer also published a photograph of Sachin being dismissed, with the headline "India Embraces Disgrace." The paper wrote that, judging by India's performance, Indian spectators might now have to start taking an interest in sports such as rugby.""",

"070427_mcgrath_lastmatch": """He said that if this happened, it would be the best possible farewell to his 13-year international cricket career.

The final between Australia and Sri Lanka is scheduled for Saturday. McGrath will retire from international cricket after this match. The game will be played in Barbados, at the same ground where he took five wickets for 68 runs against the West Indies in an international match in 1994. McGrath says that taking another five-wicket haul at the same ground would make his farewell particularly memorable.

He said that he had no doubts about his decision to retire, although he still enjoyed playing cricket.

**We are excited too**

Sri Lanka's star spinner Muttiah Muralitharan has also said that beating Australia in the World Cup final would be greater than any other achievement of his career.

The final between defending champions Australia and Sri Lanka will take place in Barbados on Saturday, and attention will be focused on Muralitharan. The 35-year-old has taken more than 1,000 wickets in Test and One-Day International cricket combined.

Sri Lanka also defeated Australia in the 1996 World Cup final to win the title. Muralitharan believes that a victory over Australia this time would be even greater than that 1996 triumph.

The pitch at Kensington Oval is expected to favour fast bowlers in Saturday's final. Some experts believe this could create difficulties for Sri Lanka. However, opening batsman Sanath Jayasuriya says his team will fight fire with fire. He said Sri Lanka also has fast bowlers capable of making the ball bounce on such pitches.""",

"070530_iftikhar_detained": """In a document submitted to the Supreme Court, Chaudhry said that Pakistan's President, General Pervez Musharraf, had pressured him to resign on the basis of alleged misconduct charges. He said that when he refused to resign, intelligence officials detained him unlawfully for five hours.

In an affidavit submitted to the court, Justice Chaudhry described for the first time what had happened around the time of his suspension. The lawyer representing the suspended Chief Justice told the court that President Musharraf held a personal grudge against his client.

Political analysts say that General Musharraf believed Iftikhar Chaudhry could obstruct his plans to become president again. The suspension of Justice Chaudhry has led to widespread protests across the country and is being viewed as the biggest challenge Musharraf has faced so far.

**Innocent**

At the beginning of May, the Supreme Court accepted Chaudhry's request to have the case concerning his suspension heard by the court. Earlier, the President had suspended him and referred the matter to Pakistan's Supreme Judicial Council.

On Tuesday, Iftikhar Chaudhry's lawyers presented his case in court and submitted an affidavit describing the events surrounding his suspension. The BBC has seen a copy of the affidavit. It states that when General Musharraf summoned Chaudhry to Army House in Rawalpindi, the heads of the Inter-Services Intelligence (ISI), Military Intelligence (MI) and Intelligence Bureau (IB) were present. They allegedly pressured him to resign for five hours.

According to the affidavit, Iftikhar Chaudhry refused to give in to the pressure and stated clearly, "I will not resign because I am innocent." """
}

manual_corrections.update(batch7_corrections)

batch7_accepted_ids = [
    "070425_ccpa_inconclusive",
    "070516_assam_violence"
]

print("Batch 7 corrections added:", len(batch7_corrections))
print("Batch 7 accepted without correction:", len(batch7_accepted_ids))
print("Total correction entries:", len(manual_corrections))

Batch 7 corrections added: 8
Batch 7 accepted without correction: 2
Total correction entries: 72


In [199]:
# Apply Batch 7 corrections
for article_id, corrected_text in batch7_corrections.items():
    mask = reviewed_translations["id"] == article_id

    reviewed_translations.loc[
        mask, "complete_english_translation"
    ] = corrected_text

    reviewed_translations.loc[
        mask, "manual_review_status"
    ] = "Manually corrected"

# Mark the 2 accepted articles
reviewed_translations.loc[
    reviewed_translations["id"].isin(batch7_accepted_ids),
    "manual_review_status"
] = "Human reviewed - no correction"

# Save updated review tracker
reviewed_translations.to_csv(
    "/content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv",
    index=False
)

print("Batch 7 corrections applied:", len(batch7_corrections))
print("Batch 7 accepted articles marked:", len(batch7_accepted_ids))
print("\nUpdated review status:")
print(reviewed_translations["manual_review_status"].value_counts())
print("\nTotal reviewed:", (
    reviewed_translations["manual_review_status"] != "Pending review"
).sum())
print("\nSaved successfully.")

Batch 7 corrections applied: 8
Batch 7 accepted articles marked: 2

Updated review status:
manual_review_status
Pending review                    159
Manually corrected                 72
Human reviewed - no correction     19
Name: count, dtype: int64

Total reviewed: 91

Saved successfully.


In [200]:
pending_ids = reviewed_translations.loc[
    reviewed_translations["manual_review_status"] == "Pending review",
    "id"
].head(10).tolist()

source_lookup = df_250.set_index("id")
translation_lookup = reviewed_translations.set_index("id")

print("Batch 8 — Articles for Human Review")
print("Number of articles:", len(pending_ids))

for i, article_id in enumerate(pending_ids, 1):
    print("\n" + "=" * 100)
    print(f"ARTICLE {i}/10: {article_id}")

    print("\nHINDI ORIGINAL:\n")
    print(source_lookup.loc[article_id, "text"])

    print("\nCURRENT ENGLISH TRANSLATION:\n")
    print(translation_lookup.loc[
        article_id, "complete_english_translation"
    ])

Batch 8 — Articles for Human Review
Number of articles: 10

ARTICLE 1/10: 070616_gaza_arab_ministers

HINDI ORIGINAL:

फ़तह के समर्थकों ने रमल्ला में परेड की और हवा में गोली चलाई. ये लोग संसद के बाहर हमास बाहर जाओ का नारा लगा रहे थे और फिर छत पर चढ़कर फ़तह और फ़लस्तीनी झंडा लगा दिया. संसद के उपसभापति ने बीबीसी को बताया कि जब उन्होंने फ़तह समर्थकों को रोकने की कोशिश की तो उन्हें पीटा गया. फ़तह से जुड़े कुछ लोगों ने शिक्षा मंत्रालय में तोड़फोड़ भी की हालांकि इसमें कोई भा घायल नहीं हुआ. रमल्ला में हमास के ज़्यादातर राजनेता और समर्थक या तो शहर से भाग गए हैं या फिर छिपे हुए हैं. तनाव इस बीच हमास ने कहा है कि उसके सात समर्थकों के शव ग़ज़ा के एक परिसर से मिले हैं. इस परिसर का इस्तेमाल फ़तह समर्थक सुरक्षाबल करते आए हैं. हमास का कहना है कि उसके समर्थकों को मारे जाने से पहले अगवा किया गया था. रविवार को फ़लस्तीनी राष्ट्रपति महमूद अब्बास नए प्रधानमंत्री को शपथ दिलाएँगे. गुरुवार को अब्बास ने हमास के नेतृत्व वाली राष्ट्रीय एकता की सरकार को भंग कर दिया था. अब्बास ने सलाम फ़य्याद से आपातकालीन सरकार बन

In [201]:
batch8_corrections = {
    "070616_gaza_arab_ministers": """Fatah supporters paraded in Ramallah and fired shots into the air. They chanted "Hamas, get out" outside parliament, then climbed onto the roof and raised Fatah and Palestinian flags. The deputy speaker of parliament told the BBC that he was beaten when he tried to stop the Fatah supporters. Some people associated with Fatah also vandalised the Ministry of Education, although no one was injured. Most Hamas politicians and supporters in Ramallah have either fled the city or gone into hiding.

Meanwhile, Hamas has said that the bodies of seven of its supporters were found in a compound in Gaza. The compound has been used by security forces associated with Fatah. Hamas says its supporters were abducted before being killed.

On Sunday, Palestinian President Mahmoud Abbas will swear in the new prime minister. On Thursday, Abbas dissolved the Hamas-led national unity government. He has asked Salam Fayyad to form an emergency government. The European Union's executive branch has expressed support for Mahmoud Abbas, recognising him as the leader. Hamas Prime Minister Ismail Haniyeh has already rejected Abbas's declaration of a state of emergency and his decision to dissolve the national unity government. Hamas says the interim administration is illegal and that it does not recognise the appointment.

Arab countries

Meanwhile, foreign ministers from Arab countries have described the ongoing violence in Gaza as criminal and appealed to the Palestinian groups Hamas and Fatah to reconcile. In a statement issued after a meeting in Cairo, the Arab foreign ministers called on the groups to respect the authority of Palestinian President Mahmoud Abbas. After Hamas took control of Gaza, Fatah leader Abbas took control of the West Bank. The Arab ministers decided to form an investigative team to establish the facts about the violence in Gaza.

Meanwhile, Hamas leader Khalid Mashal has taken an initiative towards reconciliation and expressed his willingness to work with rival Fatah leader and Palestinian President Mahmoud Abbas. Mashal says Hamas did not want to take control of Gaza, but that the security situation there left it with no other option. It is worth noting that Hamas claimed control of Gaza after fierce clashes between the two groups over the past week. More than 110 people have died in the violence.""",

    "071001_pakistan_cargo": """Sixty years after Partition, this road route has opened for trade between the two countries. Analysts believe that bilateral annual trade could reach six billion dollars if both countries ease their trade restrictions. Punjab Chief Minister Parkash Singh Badal flagged off the trucks in Chandigarh. After unloading their tomato cargo, all the trucks will return to India.

A new initiative

Indian trucks have been issued special border-entry permits. These permits will include a photograph of the truck driver, details of the vehicle and information about the goods being carried. An official posted at the Attari border said that truck drivers would no longer need passports, visas or international driving licences. Truck drivers will be required to wear clothing marked "Driver-Pakistan" or "Driver-India".

Pakistani trucks have also been issued customs permits. They will travel to the customs terminal in Indian territory, unload dry fruits and then return to Pakistan.

This new trade initiative has generated considerable enthusiasm among traders in Amritsar and Lahore. Rajdeep Singh, a trader at a wholesale market in Amritsar, says that trade in fruits, vegetables, fodder and other goods between the two countries could double within a year.

Meanwhile, porters at the Attari border protested because they feared that the movement of goods by truck would adversely affect their livelihoods. After a truck carrying tomatoes entered Pakistan, the porters unloaded boxes of tomatoes from other trucks and threw them onto the ground. The porters said that previously they had carried goods from the Indian customs terminal to the border, and that the movement of trucks had affected their livelihood.""",

    "071013_advani_pm": """While the Left parties praised the Prime Minister for changing his position on the nuclear deal, the BJP strongly criticised him. In a statement, Advani said that by changing his position on the nuclear deal, Manmohan Singh had lost even the little credibility he had gained through holding the office.

Leader of the Opposition LK Advani said that Manmohan Singh's statement that life would not come to an end if the nuclear deal did not happen proved that he was the weakest Prime Minister the country had ever had.

A weak Prime Minister

Advani said that the United Progressive Alliance government had been weak ever since it was formed in May 2004 and that it would continue to decline following Friday's change. He said that Dr Manmohan Singh and UPA chairperson Sonia Gandhi had suddenly backed down, proving that they had no principles. Advani said that as soon as they realised that the deal could bring down the government, they changed their position.

Advani said that not only most members of Parliament but also the scientific community, security experts and foreign-policy experts were opposed to the agreement. He said that the Left parties were also opposed to it, but their opposition was based on their stance towards the United States. In contrast, the BJP had argued that the agreement ignored national security interests.

He said that the BJP believed the agreement should be based on equality and that strategic security, sovereignty and self-reliance should not be compromised in dealings with the United States.""",

    "071104_komal_column": """Rakeysh Omprakash Mehra recently signed Abhishek Bachchan for his new film Delhi 6, and now Sonam has been cast as the heroine. The film's production company, UTV, did not have much of a disagreement with Sonam and her father Anil Kapoor over money. There was no major negotiation over Sonam's fee. Sonam will receive a substantial amount for working in the film by the director of Rang De Basanti. Both Sonam and Anil Kapoor have heard the film's story.

A few months ago, Ranbir Kapoor agreed to work in Yash Chopra's new film. It will be directed by Siddharth Anand, who previously made Salaam Namaste and Ta Ra Rum Pum.

A strange coincidence

It is a coincidence, but a strange one. Shah Rukh Khan began his career with Rishi Kapoor in 1992. Now, fifteen years later, Rishi Kapoor's son Ranbir Kapoor is about to begin his film career in the same period as Shah Rukh Khan. The difference is that Shah Rukh and Rishi Kapoor appeared together in the same film, Deewana, in 1992. However, Shah Rukh Khan does not appear in Ranbir's debut film, Saawariya. Instead, Shah Rukh Khan's Om Shanti Om is scheduled to be released on the same day as Saawariya.

A minister's son on screen

First, Riteish Deshmukh, son of Maharashtra Chief Minister Vilasrao Deshmukh, entered Bollywood. Now another minister's son is preparing to enter the world of acting. Chirag, son of Union Minister Ram Vilas Paswan, will be introduced in his next film by Anuj Saxena, a producer of television serials and films. Anuj's film Dulha Mil Gaya, starring Fardeen Khan, is currently being made and features a lengthy special appearance by Shah Rukh Khan.

Last week, on 31 October, Anuj Saxena organised a lavish party to introduce Chirag to the glamour of Bollywood. If things continue this way, all the problems of the film industry will disappear, because if ministers' children work in the industry, their fathers may take greater care of it, just as they take care of their own children.

Jaaved Jaaferi is back in the race

Although Indra Kumar and Ashok Thakeria's Dhamaal did not perform spectacularly at the box office, Jaaved Jaaferi has benefited greatly from the film. For one thing, Javed is back in circulation. People in the industry have realised that, despite his television show, he also has time to work in films. Secondly, producers and directors liked Jaaferi's performance in the film more than anyone else's. As a result, he is being offered many new films. It seems that in the coming year we will see more of Javed's comedy, not only on television but also on the big screen.

Shah Rukh's birthday party

Do you know where Shah Rukh Khan's birthday was celebrated this year? At the home of his close friend Sanjay Kapoor. Shah Rukh Khan, Karan Johar and Sanjay Kapoor belong to the same group. This year, on 2 November, Sanjay Kapoor hosted Shah Rukh's birthday party at his home. All of Shah Rukh's friends were invited. Naturally, it was a superstar's party, so there was a large crowd of guests. The party ended at six o'clock on Saturday morning. Big people have big things to say, and even bigger and longer parties!

What's next after Aaja Nachle?

Madhuri Dixit, who is currently in the United States with her husband Ram and children, will visit India before the release of her film Aaja Nachle. The film will be released on 30 November, and Madhuri has given producer Aditya Chopra one week for its promotion.

It is not yet known which film Madhuri will appear in next, because she herself does not know. Many producers and directors have spoken to Madhuri and her secretary, Rikku, about signing her for their films, but she has not agreed to any project. She will refuse a film if it does not offer her a substantial role.

A few days ago, Indra Kumar, a director who had once been particularly close to Madhuri and had directed films such as Dil, Beta and Raja, called her in the United States and asked her to work in his film. Madhuri asked him to meet her towards the end of this month so that they could discuss the story.""",

    "080131_kenya_mp_killed": """African Union Chairman Alpha Oumar Konaré has told African leaders that they cannot remain silent about the situation in Kenya. "If Kenya is burning, there will be nothing left for tomorrow," he said. Konaré, a senior AU official, said that the organisation had a duty to support efforts towards reconciliation in Kenya. These efforts are being led by former UN Secretary-General Kofi Annan.

UN Secretary-General Ban Ki-moon has also urged Kenyan leaders to find a peaceful solution. Speaking to leaders attending the AU summit, Ban Ki-moon said, "President Mwai Kibaki and opposition leader Raila Odinga have a special responsibility to find a peaceful solution to the crisis."

Meanwhile, talks between representatives of the Kenyan government and the opposition began in Nairobi on Thursday as part of efforts towards reconciliation.

An MP killed

Anti-government protests and violence continued in Kenya on Thursday, and an opposition MP was shot dead by police. A spokesperson for the opposition Orange Democratic Movement (ODM) said that a traffic police officer had shot MP David Too in the western city of Eldoret.

Earlier in the same week, another MP, Mugabe Were, had also been shot in Nairobi. Following his killing, violence intensified in areas dominated by the opposition party.

More than 850 people have died in the violence that erupted after the disputed presidential election of December 2007. Around 250,000 people have had to leave their homes because of the threat of violence and seek refuge in other East African countries, where conditions are somewhat better.

Henry Kosgey, an ODM spokesperson, said that MP David Too was travelling from Nairobi to Eldoret when he was shot during a road blockade in the Rift Valley. The Rift Valley is an area that experienced severe violence after the disputed elections of 27 December 2007. On Wednesday, Jendayi Frazer, the United States' top official for Africa, said that "ethnic genocide" was taking place in the region.

Efforts towards reconciliation

As part of efforts towards reconciliation, former UN Secretary-General Kofi Annan brought Kenya's president and opposition leader back to the negotiating table on Wednesday. Talks aimed at resolving the political crisis had stalled but had resumed on Tuesday.

Kofi Annan spoke with President Mwai Kibaki and opposition leader Raila Odinga and urged both to take every possible step to restore peace. He said, "People from every ethnic community live in Kenya, but I hope you will see yourselves first as Kenyan citizens."

Kofi Annan will give the two leaders a roadmap, or action plan, for the talks.""",

    "080226_britain_iraq": """Richard Thomas, the UK's Information Commissioner, said that the issue was "serious and controversial" and that the documents should therefore be made public. This is a major setback for the British government, which has repeatedly been reluctant to disclose this information.

Before the Iraq War, two important meetings of the British Cabinet took place. People may now be able to find out what happened at those meetings, thanks to the unprecedented decision by the UK's Information Commissioner.

The British government had argued that the information should not be made public and had given two reasons. First, disclosure would affect the principle of collective cabinet responsibility. Second, leaders might become reluctant to speak openly about issues during important meetings.

The argument was not convincing

Information Commissioner Richard Thomas rejected these reasons as insufficient. He said that the public had a right to know the real reasons behind the Iraq War. This is an important decision. The public may now learn which British ministers raised questions about the Iraq War, who opposed it and how extensively the Cabinet discussed the issue.

The Liberal Democrats, Britain's third-largest party, have welcomed the announcement that the information will be made public. Ed Davey, the party's spokesperson for foreign affairs, said, "The wall of secrecy that the Labour Party built to conduct its affairs behind closed doors is crumbling brick by brick. The Labour Party has hidden a great deal, and thanks to the Information Commissioner, the public will learn the truth in time. However, it is shameful that documents have to be extracted from the government one by one."

Meanwhile, officials at Downing Street say they are considering appealing against the decision. They have also said that even after the documents are made public, they may not reveal much information because the minutes of Cabinet meetings are not recorded in great detail and do not include every word spoken.""",

    "080301_benazir_suspect": """Chaudhry Abdul Majeed, the senior officer investigating the case, said that charges had been filed against four other people in addition to Mehsud. After the case was registered in court, the judge issued non-bailable arrest warrants for all five suspects.

On 27 December, former Prime Minister Benazir Bhutto was killed in a suicide attack in Rawalpindi.

Baitullah Mehsud lives in the South Waziristan region, which borders Afghanistan. The area is under the influence of pro-Taliban militants. However, Mehsud has already denied having any role in the case. He is also alleged to have links with the extremist organisation Al-Qaeda.

The Pakistani government has already raised questions about Baitullah Mehsud's role in Benazir Bhutto's assassination. President Pervez Musharraf has also named Mehsud in connection with the assassination. Baitullah Mehsud is a commander of Tehreek-e-Taliban. The organisation is believed to have links with Al-Qaeda.

Speaking to reporters, senior investigating officer Abdul Majeed refused to provide details about the investigation into the assassination or explain the roles of the suspects. He said that, in addition to Baitullah Mehsud, the four other accused were Ibadur Rahman, Imramullah, Faiz Mohammad and Abdullah."""
}

batch8_accepted_ids = [
    "070823_vv_iraq_report",
    "070902_khanduri_win",
    "080211_raj_police"
]

# Check the batch before applying
print("Batch 8 corrections added:", len(batch8_corrections))
print("Batch 8 accepted without correction:", len(batch8_accepted_ids))
print("Total correction entries:", len(manual_corrections) + len(batch8_corrections))

Batch 8 corrections added: 7
Batch 8 accepted without correction: 3
Total correction entries: 79


In [202]:
# Apply Batch 8 corrections
for article_id, corrected_text in batch8_corrections.items():
    mask = reviewed_translations["id"] == article_id

    reviewed_translations.loc[
        mask, "complete_english_translation"
    ] = corrected_text

    reviewed_translations.loc[
        mask, "manual_review_status"
    ] = "Manually corrected"

# Mark accepted articles
reviewed_translations.loc[
    reviewed_translations["id"].isin(batch8_accepted_ids),
    "manual_review_status"
] = "Human reviewed - no correction"

# Save updated tracker
reviewed_translations.to_csv(
    "/content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv",
    index=False
)

print("Batch 8 corrections applied:", len(batch8_corrections))
print("Batch 8 accepted articles marked:", len(batch8_accepted_ids))

print("\nUpdated review status:")
print(reviewed_translations["manual_review_status"].value_counts())

print("\nTotal reviewed:", (
    reviewed_translations["manual_review_status"] != "Pending review"
).sum())

print("\nSaved successfully.")

Batch 8 corrections applied: 7
Batch 8 accepted articles marked: 3

Updated review status:
manual_review_status
Pending review                    149
Manually corrected                 79
Human reviewed - no correction     22
Name: count, dtype: int64

Total reviewed: 101

Saved successfully.


In [203]:
pending_ids = reviewed_translations.loc[
    reviewed_translations["manual_review_status"] == "Pending review",
    "id"
].head(10).tolist()

source_lookup = df_250.set_index("id")
translation_lookup = reviewed_translations.set_index("id")

print("Batch 9 — Articles for Human Review")
print("Number of articles:", len(pending_ids))

for i, article_id in enumerate(pending_ids, 1):
    print("\n" + "=" * 100)
    print(f"ARTICLE {i}/10: {article_id}")

    print("\nHINDI ORIGINAL:\n")
    print(source_lookup.loc[article_id, "text"])

    print("\nCURRENT ENGLISH TRANSLATION:\n")
    print(translation_lookup.loc[
        article_id, "complete_english_translation"
    ])

Batch 9 — Articles for Human Review
Number of articles: 10

ARTICLE 1/10: 080319_advani_book

HINDI ORIGINAL:

इस आत्मकथा का विमोचन बुधवार को देश के पूर्व राष्ट्रपति एपीजे अब्दुल कलाम ने दिल्ली में किया. समारोह का समाँ किसी भी अवार्ड फंक्शन से कम नहीं था. चाहे उद्योग जगत से अनिल अंबानी हों या फ़िल्मी दुनिया से संजय दत्त और सोनू निगम या फिर चो रामास्वामी जैसे मीडिया के दिग्गज. बड़े लोगों की इस समारोह में कोई कमी नहीं थी. भाजपा के भी लगभग सभी वरिष्ठ नेता इस विमोचन समारोह में मौजूद थे. फिर वो चाहे वरिष्ठ पीढ़ी के जसवंत सिंह, भैरव सिंह शेखावत और मुरली मनोहर जोशी हों या फिर नई पीढ़ी के नरेन्द्र मोदी और राजनाथ सिंह. सभी ने इस विमोचन के लिए समय निकला. इसके अलावा एनडीए के संयोजक जॉर्ज फ़र्नांडिस, आरएसएस के मोहन भगवत और एनसीपी के शरद पवार भी इस समारोह में नज़र आए. महत्वपूर्ण इस किताब में लालकृष्ण आडवाणी ने भारत की पिछले 60 सालों की राजनीति के कई महत्वपूर्ण मोड़ और अहम् घटनाओं का उल्लेख किया है. जिसमें भारत का विभाजन, इमरजेंसी और भाजपा की रथ यात्रा शामिल है. आडवाणी ने इस पुस्तक में बाबरी मस्जिद और 

In [204]:
batch9_corrections = {
    "080319_advani_book": """The autobiography was released in Delhi on Wednesday by former President APJ Abdul Kalam. The atmosphere at the ceremony was no less impressive than that of an award function. Anil Ambani from the business world, Sanjay Dutt and Sonu Nigam from the film industry, and media personalities such as Cho Ramaswamy were among the many prominent guests. Almost all senior BJP leaders were also present at the book launch. These included senior-generation leaders such as Jaswant Singh, Bhairon Singh Shekhawat and Murli Manohar Joshi, as well as younger-generation leaders Narendra Modi and Rajnath Singh. All had made time to attend. NDA convenor George Fernandes, RSS chief Mohan Bhagwat and NCP leader Sharad Pawar were also present.

The significance of the book

In this book, LK Advani has discussed several important turning points and events in India's politics over the past 60 years, including the Partition of India, the Emergency and the BJP's Rath Yatra. He has also written about several sensitive issues, such as the Babri Masjid and the 2002 Gujarat riots.

At the ceremony, Jaswant Singh emphasised that Advani was misunderstood in many ways. He said, "In today's politics, I feel that Advani ji has been misunderstood and portrayed incorrectly. From my personal experience, I can say that he is very different from the image that has been created of him."

On this occasion, Advani described both his immediate family and his ideological family, the RSS, as sources of meaning and satisfaction in his life. Former Prime Minister Atal Bihari Vajpayee did not attend the event, but Advani expressed his gratitude to him as well.

Finally, the chief guest, APJ Abdul Kalam, described Advani as a creative leader. Speaking about the book, he said, "I agree with what Atal Bihari Vajpayee wrote in the introduction: the best time in the life of a sensitive human being and a great leader is yet to come."

Very few Indian leaders have written autobiographies while still active in politics. Advani has now become one of them. However, there is speculation that the timing of the book's release may have been influenced by the country's political climate, with next year's general elections already approaching. Whatever the reason, given the role LK Advani has played in Indian politics, this book may shed new light on several historical events.""",

    "080324_chowdhry_pakistan": """Yousuf Raza Gilani said in Parliament on Monday that he would order the release of judges who had been detained during the Emergency on the orders of President Pervez Musharraf.

Following the new Prime Minister's announcement, the barbed wire outside Justice Iftikhar Chaudhry's house was removed on Monday. When a delighted Iftikhar Chaudhry appeared on the balcony of his house, hundreds of supporters gathered outside to welcome him. Addressing them through a loudspeaker, he said, "I thank the entire country, which has struggled for the past five months." He added, "I have no words to express how grateful I am to all of you."

In November, Pervez Musharraf had ordered the detention of several judges. He had dismissed several judges, including Iftikhar Chaudhry, under the Emergency. As soon as the Emergency was imposed, President General Pervez Musharraf transferred the responsibilities of the Chief Justice to Justice Abdul Hameed Dogar.

State media quoted a senior administrative official as saying that all the dismissed judges were now free to move around.

Earlier on Monday, Parliament elected Yousuf Raza Gilani as the country's new Prime Minister. He received a total of 242 votes.""",

    "080416_up_results": """The Bahujan Samaj Party (BSP) won the parliamentary seats of Azamgarh and Khalilabad, as well as the Assembly seats of Bilgram, Karnalganj and Murad Nagar. The Congress and the Bharatiya Janata Party suffered a major setback in these by-elections. The BSP's main contest was largely with the Samajwadi Party.

BSP candidate Kushal Tiwari won the Khalilabad parliamentary seat, defeating Samajwadi Party candidate Balchandra Yadav by 65,000 votes. BSP candidate Akbar Ahmad Dumpy won the Azamgarh parliamentary seat. He defeated Ramakant Yadav, who contested on a BJP ticket, by a margin of 54,000 votes. The Congress candidate received fewer than 10,000 votes in this constituency.

In the previous Lok Sabha elections, Balchandra Yadav and Ramakant Yadav had won these two seats as BSP candidates. However, both later changed parties and joined the Samajwadi Party.

A stronger position

The two seats had become vacant under the anti-defection law. BSP candidate Rajni Yadav won the Bilgram Assembly seat, which had fallen vacant after the death of BSP's Upendra Tiwari. Kunwari Brij Singh of the BSP won the Karnalganj Assembly seat, while Rajpal Tyagi won the Murad Nagar Assembly seat.

Rajpal Tyagi had previously won the state Assembly election from this constituency as an independent candidate, but later joined the BSP. He successfully won the seat again by defeating Lok Dal candidate Ayub Khan.

Mayawati has completed 11 months in power in Uttar Pradesh. These by-election results have further strengthened her position.""",

    "080419_brchopra_festival": """The festival, which began on Friday evening, opened with a screening of Insaf Ka Tarazu. Other classic films, including Chhoti Si Baat, Naya Daur and Waqt, will also be screened during the festival. BR Chopra himself could not attend the ceremony because of his poor health. The film festival was inaugurated by his brother, the well-known filmmaker Yash Chopra.

Yash Chopra said, "My brother made films in his own way, even at a time when people were doubtful about the commercial success of such films because they were different. The stories BR Chopra ji told were unlike the kinds of stories that were popular at the time."

BR Chopra addressed several subjects in his films, including widow remarriage and prostitution.

Memorable films

His 1957 film Naya Daur reflected the ideals of the Nehru era, while Sadhana, released in 1958, questioned society about prostitution. His list of films also includes Gumrah, Hamraaz, Kanoon and Nikaah.

Although most of BR Chopra's films were based on social issues, these experimental films also performed well at the box office. Many leading directors had rejected the story of Naya Daur, but BR Chopra took the risk and made the film.

Speaking on the occasion, his son Ravi Chopra said, "My father believes in getting work done. For me, he is the best director and will always remain so."

A colour version of Naya Daur was also released some time ago. A short documentary based on BR Chopra's life was screened at the festival. Apart from Yash Chopra, the event was attended by Raj Babbar, director Basu Chatterjee, music composer Ravi and producer Bobby Bedi.""",

    "080514_kashmir_ceasefire": """A ceasefire has been in effect since November 2003 along the Line of Control (LoC), which separates the disputed region of Jammu and Kashmir.

Colonel AK Mathur, an Indian Army spokesperson, said on Wednesday that Pakistani troops had opened fire on Tuesday evening in the Tangdhar sector near the LoC. This is the first reported incident of firing along the Line of Control since the ceasefire began. No casualties were reported.

"Our forces did not retaliate. We have formally requested a flag meeting to register our protest and resolve the issue," the military spokesperson said.

It is notable that the alleged ceasefire violation has occurred at a time when Pakistan has a democratically elected government. Pakistan's new government has repeatedly spoken about improving bilateral relations with India.

India says that Pakistan has so far failed to dismantle militant training camps located in its territory near the Line of Control.""",

    "080711_lanka_attack": """Military spokesperson Brigadier Uday Nanyakkara has blamed Tamil rebels for the attack. The incident took place in Monaragala district, where the bus came under indiscriminate gunfire. The LTTE has not yet commented on the incident.

The attack occurred at a time when fighting between the army and Tamil rebels had intensified in the northern part of the country. A BBC correspondent in Colombo reported that the bus was travelling through a deserted area when it was fired upon.

A nine-year-old child was among those killed in the attack. Several passengers were also injured. Military spokesperson Brigadier Uday Nanyakkara said that the gunmen were probably hiding beside the road. He blamed the Tamil rebels for the shooting.

A year earlier, the Sri Lankan government had claimed that it had driven the rebels out of the eastern part of the country, although sporadic violence continued in the region. In January of the same year, rebels had also targeted a passenger bus. The bus was first attacked with a bomb, after which people were shot. Just last week, a helicopter was also fired upon.

Meanwhile, fighting between the LTTE and government forces continues in northern Sri Lanka. The army says that 19 Tamil rebels and two soldiers were killed on Thursday.""",

    "080721_parliament_debate": """A special session of Parliament has begun, and a confidence motion will be moved shortly. Voting on the motion will take place on Tuesday after the debate. The result will show how the political manoeuvring among various parties over the past fifteen days has affected the numbers.

Before the confidence motion is presented in the House, Lok Sabha Speaker Somnath Chatterjee has called an all-party meeting, which all parties are attending.

On Monday, Prime Minister Manmohan Singh will move a one-line confidence motion in the Lok Sabha: "This House expresses its confidence in the Council of Ministers."

The Prime Minister is expected to make a brief speech and request the House to adopt the motion. The motion will then be debated in the Lok Sabha on Monday and Tuesday. The Prime Minister will reply to the debate, after which voting will take place to decide the government's fate.

Sixteen hours have been allotted for the debate, although the current circumstances may cause it to continue longer. Besides the Prime Minister, leaders expected to take part include Sonia Gandhi, Pranab Mukherjee, Sitaram Yechury, LK Advani and several others.

A tough test

One MP's vote among the 543 members of Parliament is invalid. This means that a majority must be secured among the remaining 542 members. In other words, 271 votes are needed for a majority. However, it is still not entirely clear how many MPs belong to each alliance. The situation is such that even senior leaders are reluctant to answer questions about how many MPs support them.

The Congress-led alliance has received support from Shibu Soren's Jharkhand Mukti Morcha (JMM). However, reports suggest that some Congress MPs may vote against their own party. The Samajwadi Party faces a similar situation, with at least five MPs expected to vote against the UPA, contrary to their party's position.

The opposition alliance

Although the Bharatiya Janata Party is the main opposition party in Parliament, Mayawati has moved ahead of the BJP in political manoeuvring and developments over the past week. She has succeeded in bringing Ajit Singh's Rashtriya Lok Dal to her side, and SD Deve Gowda of the Janata Dal (Secular) has also joined her.

By joining hands with the Left parties, Mayawati has succeeded in creating an alternative political alliance. However, the situation remains unclear. According to the latest reports, some MPs from the Akali Dal and Shiv Sena may also vote against their parties' whips.

Amid this uncertainty and confusion, the debate on the confidence motion is about to begin.""",

    "080801_india_iaea": """Approval from the International Atomic Energy Agency (IAEA) was necessary for implementing the nuclear deal. Following the IAEA's approval, India will now have to obtain the agreement of the 45-member Nuclear Suppliers Group (NSG) so that it can participate in international nuclear fuel trade.

After that, the India-US nuclear deal will require approval from the US Congress. India is under pressure to get the agreement before the US Congress and secure its approval before the US presidential election in November.

The nuclear deal

Recently, the Indian government had to face a confidence vote in Parliament over this very nuclear deal. The Left parties, which had been supporting the government from outside, withdrew their support over the agreement. However, the government won the confidence vote.

India argues that the nuclear deal is necessary to meet the country's growing energy needs.

During the meeting, IAEA chief Mohamed ElBaradei said that the safety standards at India's nuclear facilities were compatible with the agency's requirements. He said, "This meets India's needs while also taking the IAEA's statutory requirements into account."

Under the agreement, IAEA inspectors will monitor India's civilian nuclear facilities to ensure that nuclear fuel is not used for military purposes.

Under the agreement with the United States, India will be able to participate in the international market for nuclear fuel and technology. However, these facilities and materials must be used only for civilian purposes.

BBC correspondents say that following the agreement with the IAEA, 14 of India's existing 22 nuclear facilities will come under IAEA monitoring.""",

    "080826_bihar_flood": """Three Indian Air Force helicopters are also being used to distribute relief supplies. Pratyay Amit, Additional Commissioner of the Disaster Management Department, told the news agency PTI that the Kosi embankment had also breached near a village in Nepal. As a result, more water was entering the area and several other places had been flooded.

He said that around 100 army personnel would begin relief operations in Madhepura from Wednesday. Personnel from the Sashastra Seema Bal, special forces and the National Disaster Response Force are also assisting.

The flood situation is devastating

The breach in the Kosi embankment has caused the greatest damage in the Nepal-bordering districts of Araria, Supaul and Madhepura. According to government officials, between 2 and 2.5 million people have been left homeless. Information received so far indicates that 40 people have died, but the actual extent of the loss of life and property can be assessed only after the floodwaters recede.

Bihar Chief Minister Nitish Kumar has described the situation as devastating. Officials say conditions are serious and large-scale rescue and relief operations are under way.

Two hundred boats and 25 motorboats are being used for rescue operations. Helicopters are being used to deliver food packets to affected areas. Officials said that more than 50 relief camps had been set up. People have taken shelter on highways and in high-rise buildings to escape the floodwaters.

Relief operations

According to Reuters, a flood-affected resident named Sanjeev Kumar said, "Around 700 to 800 houses have been damaged. People are somehow managing to stay on pieces of wood and on the roofs of houses. Those whose roofs have been damaged do not know what to do."

On Tuesday morning, 100 prisoners from two jails in Bihar were transferred to other, safer prisons. Reports say that standing crops across thousands of hectares have been damaged by the floods.

The Bihar government has appealed to relief agencies to help deliver assistance to flood-affected areas. However, many people say they have not received any help so far.

Jugat Lal, a flood-affected resident, told Reuters, "No arrangements have been made. We live near the river and drink its water when we are thirsty, even though this same river has also caused people's deaths."

Meanwhile, in neighbouring Uttar Pradesh, heavy rains and floods since June have killed 700 people. According to officials, most of the deaths were caused by houses collapsing."""
}

batch9_accepted_ids = [
    "080613_afghan_jail"
]

print("Batch 9 corrections added:", len(batch9_corrections))
print("Batch 9 accepted without correction:", len(batch9_accepted_ids))
print("Total correction entries:", len(manual_corrections) + len(batch9_corrections))

Batch 9 corrections added: 9
Batch 9 accepted without correction: 1
Total correction entries: 81


In [205]:
print("Correction IDs:")
print(list(batch9_corrections.keys()))

print("\nAccepted IDs:")
print(batch9_accepted_ids)

print("\nCorrection count:", len(batch9_corrections))
print("Accepted count:", len(batch9_accepted_ids))

Correction IDs:
['080319_advani_book', '080324_chowdhry_pakistan', '080416_up_results', '080419_brchopra_festival', '080514_kashmir_ceasefire', '080711_lanka_attack', '080721_parliament_debate', '080801_india_iaea', '080826_bihar_flood']

Accepted IDs:
['080613_afghan_jail']

Correction count: 9
Accepted count: 1


In [2]:
required_variables = [
    "df_250",
    "reviewed_translations",
    "manual_corrections",
    "batch9_corrections",
    "batch9_accepted_ids"
]

for name in required_variables:
    print(name, ":", "Available" if name in globals() else "Missing")

df_250 : Missing
reviewed_translations : Missing
manual_corrections : Missing
batch9_corrections : Missing
batch9_accepted_ids : Missing


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os

base_path = "/content/drive/MyDrive/FactIndic_PersonB_CleanRun"

# Load the 250 Hindi source articles
df_250 = pd.read_csv(
    f"{base_path}/02_clean_data/XL_Sum_Hindi_Cleaned_250.csv"
)

# Load the saved human-review tracker
reviewed_translations = pd.read_csv(
    f"{base_path}/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv"
)

# Load the original corrected machine translations
se = pd.read_csv(
    f"{base_path}/09_sentence_level_translation/FactIndic_English_250_Corrected.csv"
)

# Keep article IDs as strings
for df in [df_250, reviewed_translations, se]:
    df["id"] = df["id"].astype(str)

print("Files restored successfully!")
print("Hindi articles:", df_250.shape)
print("Review tracker:", reviewed_translations.shape)
print("Base translations:", se.shape)

print("\nSaved review progress:")
print(reviewed_translations["manual_review_status"].value_counts())

Mounted at /content/drive
Files restored successfully!
Hindi articles: (250, 7)
Review tracker: (250, 4)
Base translations: (250, 3)

Saved review progress:
manual_review_status
Pending review                    149
Manually corrected                 79
Human reviewed - no correction     22
Name: count, dtype: int64


In [4]:
batch9_corrections = {
    "080319_advani_book": """The autobiography was released in Delhi on Wednesday by former President APJ Abdul Kalam. The atmosphere at the ceremony was no less grand than that of an award function. Prominent figures from many fields attended, including Anil Ambani from the business world, Sanjay Dutt and Sonu Nigam from the film industry, and media veteran Cho Ramaswamy. Almost all senior BJP leaders were also present, including Jaswant Singh, Bhairon Singh Shekhawat and Murli Manohar Joshi from the older generation, and Narendra Modi and Rajnath Singh from the younger generation. NDA convenor George Fernandes, RSS chief Mohan Bhagwat and NCP leader Sharad Pawar also attended. In this book, LK Advani discusses several major turning points and important events in India's politics over the past 60 years, including the Partition, the Emergency and the BJP's Rath Yatra. He has also written about sensitive issues such as the Babri Masjid and the 2002 Gujarat riots. At the ceremony, Jaswant Singh stressed that Advani is misunderstood in many ways. He said, "In today's politics, I feel that Advani ji has been misunderstood and misrepresented. From my personal experience, I can say that in reality he is very different from the image people have of him." Advani described his own family and his ideological family, the RSS, as sources of meaning and satisfaction in his life. Former Prime Minister Atal Bihari Vajpayee did not attend the event, but Advani expressed his gratitude to him. Chief guest APJ Abdul Kalam described Advani as a creative leader and said, "I agree with what Atal Bihari Vajpayee wrote in the introduction to this book: the best time in the life of a sensitive person and a great leader is still to come." Very few Indian leaders have written their autobiographies while still active in politics. Advani is now one of them. However, there is speculation that the timing of the book's release was influenced by the country's political climate, as the clouds of next year's general election are already gathering. Whatever the reason, given the role LK Advani has played in Indian politics, this book may shed new light on several historical events in the country.""",

    "080324_chowdhry_pakistan": """Yousuf Raza Gilani had said in Parliament on Monday that he would order the release of the judges who had been detained during the Emergency on the orders of President Pervez Musharraf. Following the new Prime Minister's announcement, the barbed wire outside Justice Chaudhry's house was removed on Monday. When Iftikhar Chaudhry, looking delighted, came out onto the balcony of his house, hundreds of supporters waiting outside welcomed him. Addressing the crowd over a loudspeaker, he said, "I thank the entire country, which has struggled for the past five months." He added, "I have no words to express how grateful I am to all of you." In November, Pervez Musharraf had ordered the detention of several judges. He had dismissed several judges, including Iftikhar Chaudhry, under the Emergency. As soon as the Emergency was imposed, President General Pervez Musharraf handed over the responsibility of the Chief Justice's office to Justice Abdul Hamid Dogar. State media, quoting a senior administrative official, reported that all the dismissed judges were free to move around. Earlier on Monday, Parliament had elected Yousuf Raza Gilani as the country's new Prime Minister. He received a total of 242 votes.""",

    "080416_up_results": """The BSP won the parliamentary seats of Azamgarh and Khalilabad, as well as the assembly seats of Bilgram, Karnalganj and Murad Nagar. The Congress and the Bharatiya Janata Party suffered a major setback in these by-elections. To a large extent, the BSP's main contest was with the Samajwadi Party. BSP candidate Kushal Tiwari won the Khalilabad parliamentary seat, defeating Samajwadi Party candidate Balchandra Yadav by a margin of 65,000 votes. BSP candidate Akbar Ahmad Dumpy won the Azamgarh parliamentary seat. He defeated Ramakant Yadav, a Samajwadi Party leader who contested the election on a BJP ticket, by a margin of 54,000 votes. The Congress candidate received fewer than 10,000 votes in this constituency. In the previous Lok Sabha election, both these seats had been won by BSP candidates Balchandra Yadav and Ramakant Yadav. However, both later changed parties and joined the Samajwadi Party. The two seats had fallen vacant under the anti-defection law. BSP candidate Rajni Yadav won the Bilgram assembly seat, which had become vacant after the death of BSP's Upendra Tiwari. Kunwari Brij Singh of the BSP won the Karnalganj assembly seat, while Rajpal Tyagi won the Murad Nagar assembly seat. Tyagi had previously won the state assembly election from this constituency as an independent candidate, but later joined the BSP. He successfully won the seat again by defeating Lok Dal candidate Ayub Khan. Mayawati has completed 11 months as Chief Minister of Uttar Pradesh. These by-election results have further strengthened her position.""",

    "080419_brchopra_festival": """The festival, which began on Friday evening, screened the film Insaf Ka Tarazu. Other classic films, including Chhoti Si Baat, Naya Daur and Waqt, will also be shown during the festival. BR Chopra could not attend the ceremony because of his poor health. The festival was inaugurated by his brother, the renowned filmmaker Yash Chopra. He said, "My brother made films in his own way, even at a time when people were doubtful about the commercial success of such films because they were different from the kinds of stories that were popular then." BR Chopra addressed several subjects in his films, including widow remarriage and prostitution. Among his memorable films, Naya Daur, released in 1957, reflects the ideals of the Nehru era, while Sadhana, released in 1958, raises questions about society's treatment of prostitution. Gumrah, Hamraaz, Kanoon and Nikah are also among the films he made. Although most of BR Chopra's films were based on social issues, these experimental films also performed well at the box office. Many leading directors had rejected the story of Naya Daur, but BR Chopra took the risk and made the film. Speaking on the occasion, his son Ravi Chopra said, "My father believes in getting the work done. For me, he is and will remain the best director." A colour version of Naya Daur was also released some time ago. A short documentary based on BR Chopra's life was also screened at the festival. Besides Yash Chopra, the event was attended by Raj Babbar, director Basu Chatterjee, music composer Ravi and producer Bobby Bedi.""",

    "080613_afghan_jail": """Wali Karzai, chairman of the Kandahar provincial council and brother of President Hamid Karzai, told the BBC that there were 1,000 prisoners in the jail at the time and that all of them had escaped. It is believed that 350 of these prisoners were Taliban members. Several security personnel were killed in the attack. According to officials, a lorry bomb was detonated to blow up the prison's main gate, after which 40 Taliban fighters entered the jail. A state of emergency has been declared in Kandahar city. Police and soldiers are patrolling the streets, and people have been told not to leave their homes. Prison chief Abdul Qabir confirmed to the Associated Press that hundreds of prisoners had escaped, but said that some prisoners had remained in the jail. Eyewitnesses said the explosion used to break open the prison's main gate was so powerful that window panes were shattered up to three kilometres away. The BBC's Kabul correspondent said the incident would cause concern among the Afghan police and other security forces. Kandahar is one of the key areas from which the war is being fought against the Taliban, who are rebelling against President Karzai, NATO and US forces. Just last month, prisoners at Kandahar jail had gone on a hunger strike. They ended the strike after a parliamentary delegation assured them that their demands would be met. Around 400 prisoners said they were not receiving proper justice, and several also complained of torture.""",

    "080711_lanka_attack": """Brigadier Uday Nanyakara, a military spokesman, blamed the attack on Tamil rebels. The incident took place in Monaragala district, where the bus was fired upon indiscriminately. The LTTE has not commented on the incident so far. The attack comes at a time when clashes between the army and Tamil rebels have intensified in the northern part of the country. A BBC correspondent in Colombo reported that the bus was passing through a deserted area when shots were fired at it. A nine-year-old child was among those killed in the attack. Several passengers were also injured in the firing. Military spokesman Brigadier Uday Nanyakara said the gunmen were probably hiding beside the road. He blamed the Tamil rebels for the shooting. A year earlier, the Sri Lankan government had claimed that it had driven the rebels out of the eastern region, but sporadic violence continued there. In January of the same year, rebels had also targeted a passenger bus. The bus was first attacked with a bomb, after which people were shot at. Just last week, a helicopter was also fired upon. Meanwhile, fighting between the LTTE and government forces continues in northern Sri Lanka. The military says that 19 Tamil rebels and two soldiers were killed on Thursday.""",

    "080721_parliament_debate": """A special session of Parliament has begun, and a confidence motion is expected to be introduced shortly. Voting on the motion will take place on Tuesday after the debate. The vote will show the outcome of the political manoeuvring among various parties over the past 15 days. Before the confidence motion is introduced, Lok Sabha Speaker Somnath Chatterjee has called an all-party meeting, which all parties are attending. Prime Minister Manmohan Singh is expected to move a one-line confidence motion in the Lok Sabha on Monday: "This House expresses its confidence in the Council of Ministers." The Prime Minister is expected to make a brief speech and request the House to adopt the motion. The motion will then be debated in the Lok Sabha on Monday and Tuesday. The Prime Minister will reply to the debate, after which voting will take place and the government's fate will be decided. Sixteen hours have been allotted for the debate, but given the circumstances, it may last longer. Besides the Prime Minister, Sonia Gandhi, Pranab Mukherjee, Sitaram Yechury, LK Advani and several other senior leaders are expected to participate. A tough test: One of the 543 members of Parliament does not have a valid vote, which means that a majority must be secured among 542 members. A total of 271 votes are therefore needed for a majority. It is still not entirely clear how many MPs are supporting each alliance. The situation is such that even senior leaders are reluctant to answer questions about how many MPs are with them. The Congress-led alliance has received the support of Shibu Soren's Jharkhand Mukti Morcha, but reports suggest that some Congress MPs may vote against their own party. The Samajwadi Party faces a similar situation, with at least five MPs expected to vote against the UPA, contrary to their party's position. The opposition alliance: Although the BJP is the main opposition party in Parliament, Mayawati has taken the lead over the BJP in the political manoeuvring and developments of the past week. Mayawati has succeeded in bringing Ajit Singh's Rashtriya Lok Dal to her side, and JD(S) leader SD Deve Gowda has also joined her. Together with the Left parties, Mayawati has succeeded in creating an alternative alliance. However, the picture is still unclear. According to the latest reports, some MPs from the Akali Dal and Shiv Sena may also vote against their party whips. Amid this uncertainty and confusion, the debate on the confidence motion is about to begin.""",

    "080826_bihar_flood": """Three helicopters of the Indian Air Force are also engaged in distributing relief supplies. Pratyay Amit, Additional Commissioner of the Disaster Management Department, told the news agency PTI that the Kosi embankment had also broken near a village in Nepal, allowing more water to flow in and flooding several other areas. He said that around 100 army personnel would join relief operations in Madhepura from Wednesday. Personnel from the Sashastra Seema Bal, Special Forces and the National Disaster Response Force are also providing assistance. The districts of Araria, Supaul and Madhepura, which border Nepal, have suffered the greatest damage from the breach of the Kosi embankment. According to government officials, between 20 and 25 lakh people have been left homeless. According to information received so far, 40 people have died, but the actual loss of life and property can be assessed only after the floodwaters subside. Bihar Chief Minister Nitish Kumar has described the situation as alarming. Officials say conditions are serious and large-scale rescue and relief operations are under way. Two hundred boats and 25 motorboats are being used for rescue operations. Helicopters are being used to deliver food packets to affected areas. Officials said that more than 50 camps had been set up. People have taken shelter on highways and in tall buildings to escape the floods. Sanjeev Kumar, a flood-affected resident, told Reuters that around 700 to 800 houses had been damaged. People were managing to stay on pieces of wood and on the roofs of houses. Those whose roofs had been damaged did not know what to do. On Tuesday morning, 100 prisoners from two jails in Bihar were moved to other, safer jails. Reports say that standing crops across thousands of hectares have been damaged by the floods. The Bihar government has appealed to relief agencies to help deliver assistance to flood-affected areas. However, many people say they have not received any help so far. Flood-affected resident Jugat Lal told Reuters, "No arrangements have been made. We live near the river and drink its water when we are thirsty, even though this same river also causes deaths." Meanwhile, heavy rains and floods have killed 700 people in neighbouring Uttar Pradesh since June. According to officials, most of the deaths were caused by houses collapsing."""
}

batch9_accepted_ids = [
    "080514_kashmir_ceasefire",
    "080801_india_iaea"
]

print("Batch 9 corrections:", len(batch9_corrections))
print("Batch 9 accepted:", len(batch9_accepted_ids))
print("\nCorrection IDs:")
print(list(batch9_corrections.keys()))
print("\nAccepted IDs:")
print(batch9_accepted_ids)

Batch 9 corrections: 8
Batch 9 accepted: 2

Correction IDs:
['080319_advani_book', '080324_chowdhry_pakistan', '080416_up_results', '080419_brchopra_festival', '080613_afghan_jail', '080711_lanka_attack', '080721_parliament_debate', '080826_bihar_flood']

Accepted IDs:
['080514_kashmir_ceasefire', '080801_india_iaea']


In [5]:
# Apply Batch 9 corrections
for article_id, corrected_text in batch9_corrections.items():
    mask = reviewed_translations["id"] == article_id

    reviewed_translations.loc[
        mask, "complete_english_translation"
    ] = corrected_text

    reviewed_translations.loc[
        mask, "manual_review_status"
    ] = "Manually corrected"

# Mark accepted articles
reviewed_translations.loc[
    reviewed_translations["id"].isin(batch9_accepted_ids),
    "manual_review_status"
] = "Human reviewed - no correction"

# Save updated tracker
reviewed_translations.to_csv(
    "/content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv",
    index=False
)

print("Batch 9 corrections applied:", len(batch9_corrections))
print("Batch 9 accepted articles marked:", len(batch9_accepted_ids))

print("\nUpdated review status:")
print(reviewed_translations["manual_review_status"].value_counts())

print("\nTotal reviewed:", (
    reviewed_translations["manual_review_status"] != "Pending review"
).sum())

print("\nSaved successfully.")

Batch 9 corrections applied: 8
Batch 9 accepted articles marked: 2

Updated review status:
manual_review_status
Pending review                    139
Manually corrected                 87
Human reviewed - no correction     24
Name: count, dtype: int64

Total reviewed: 111

Saved successfully.


In [6]:
pending_ids = reviewed_translations.loc[
    reviewed_translations["manual_review_status"] == "Pending review",
    "id"
].head(10).tolist()

source_lookup = df_250.set_index("id")
translation_lookup = reviewed_translations.set_index("id")

print("Batch 10 — Articles for Human Review")
print("Number of articles:", len(pending_ids))

for i, article_id in enumerate(pending_ids, 1):
    print("\n" + "=" * 100)
    print(f"ARTICLE {i}/10: {article_id}")

    print("\nHINDI ORIGINAL:\n")
    print(source_lookup.loc[article_id, "text"])

    print("\nCURRENT ENGLISH TRANSLATION:\n")
    print(translation_lookup.loc[
        article_id, "complete_english_translation"
    ])

Batch 10 — Articles for Human Review
Number of articles: 10

ARTICLE 1/10: 080826_blast_films

HINDI ORIGINAL:

इन फ़िल्मों में से कुछ रिलीज़ हो रही हैं और कुछ का निर्माण कार्य चल रहा है. पिछले हफ़्ते रिलीज़ हुई निशिकांत कामथ की फिल्म 'मुंबई मेरी जान' की कहानी का आधार 11 जुलाई 2006 को मुबंई में हुए ट्रेन बम धमाके थे. इन बम धमाकों में लगभग 200 लोगों की मौत हुई थी. फिल्म के निर्देशक निशिकांत कामथ कहते हैं, "ये फ़िल्म किसी बड़ी दुर्घटना से उबरने की एक कोशिश को दर्शाती है. मुंबई मेरी जान की कहानी सिर्फ़ मुंबई या इस घटना के बारे में नहीं बल्कि किस तरह से लोग इस समस्या से जूझ रहे हैं इस बारे में है." मुंबई मेरी जान एक भावनाप्रधान फिल्म है जबकि पाँच सितंबर को रिलीज़ होने जा रही फ़िल्म 'वेडनस्डे' एक थ्रिलर है. इस फिल्म से निर्देशन की पारी शुरु करने वाले निर्देशक नीरज पांडे कहते हैं, "वेडनस्डे की कहानी उन चंद घंटों की है जिसमें आतंकवादी मुंबई शहर को अपनी दहशतगर्दी का शिकार बनाने की कोशिश करते हैं. फिल्म की पृष्ठभूमि आतंकवाद है और ये फिल्म पूरी तरह से मुंबई में शूट की गई है, लेकिन फिल्म की कहानी

In [7]:
batch10_corrections = {

"080826_blast_films": """Some of these films are being released, while others are still in production. Nishikant Kamat's film Mumbai Meri Jaan, which was released last week, is based on the train bombings in Mumbai on 11 July 2006. Around 200 people were killed in these blasts. The film's director, Nishikant Kamat, says, "This film portrays an attempt to recover from a major tragedy. The story of Mumbai Meri Jaan is not just about Mumbai or this incident; it is about how people are dealing with such a situation." Mumbai Meri Jaan is an emotional film, whereas Wednesday, scheduled for release on 5 September, is a thriller. Director Neeraj Pandey, who is making his directorial debut with this film, says, "Wednesday is about a few hours during which terrorists try to terrorise the city of Mumbai. The film is set against the backdrop of terrorism and was shot entirely in Mumbai, but its story is completely fictional."

The recently released psychological thriller Aamir tells the story of an ordinary man who is blackmailed and forced to plant bombs in some areas of Mumbai. The film was successful and was also appreciated by critics. The Little Godfather, being made under the banner of Popcorn Films, is also based on the Mumbai bombings. A contemporary film, The Little Godfather tells the story of young children who sell goods on Mumbai's local trains and how they showed courage on the day of the bombings. Its director, Joy Augustine, believes that people can quickly connect with real-life events. He says, "The film is about ordinary people who travel on Mumbai's trains and the children who sell goods on them, and how these people come together during such a crisis."

Film critics also see this as a major change, although they believe it is a temporary trend. Senior film critic Ajay Brahmatmaj says, "Although bomb blasts may provide the backdrop for films, producers and directors do not want to make an entire film about them. Perhaps they do not want to get involved in unnecessary controversy. Just as floods and fairs were used in older films to create twists in the story, bomb blasts are being used in films today. This trend will change with time."

Such films are generally shorter than three hours and leave little room for music and romance. Directors such as Ram Gopal Varma, who have made several successful films about the Mumbai underworld, have also tried to show the rapidly growing links between the underworld and terrorism in his recently released film Contract. Alpana Mishra of UTV believes it is a coincidence that bomb blasts or terrorism feature prominently in the stories of several of UTV's recently released films. She says, "We are happy that these films are also conveying a social message." """,

"080830_orissa_violence": """The situation in Orissa is tense following the murder of Swami Lakshmanananda, a leader of the Vishwa Hindu Parishad (VHP). Koraput district had previously remained free of violent incidents, but tensions erupted on Saturday after objectionable remarks against a particular community were written on the walls of the local Vikram Dev College. Clashes then broke out between two communities.

Police first used batons and fired tear-gas shells to disperse the crowd involved in the clashes. When they failed to control the agitated crowd, the police fired shots into the air. Koraput Superintendent of Police Deepak Kumar said that a curfew had been imposed in the city. He himself was injured in the clashes. The violent crowd also attacked a police station and damaged its furniture.

The trouble began when young people angered by the writings on the walls started attacking female students from the other community. At least two girls were injured in the incident. The VHP alleges that Christian organisations were behind the killing of its leader in Kandhamal. However, the government has not yet made any clear statement on the matter. There are also suspicions that Maoist rebels may have been behind the killing.""",

"081118_cg_campaign_vv": """Polling in this phase is scheduled for 20 November. The first phase of polling was completed on 14 November, and votes will be counted on 8 December. Several important contests are scheduled in the second phase, including those involving former Chief Minister Ajit Jogi and his wife, Renu Jogi. The political futures of state Congress president Dhanendra Sahu, working president Satyanarayan Sharma, state BJP president Vishnu Deo Sai and Yudhvir Singh, son of Dilip Singh Judev, will also be decided in this phase.

On the final day of campaigning, several prominent leaders from both major parties, including Sonia Gandhi and Lal Krishna Advani, were busy campaigning.

The first phase had also included elections in Bastar in southern Chhattisgarh, an area affected by the Naxalite movement. Several reports of violence emerged during polling in Bastar's 12 tribal constituencies. This time, elections are being held in the tribal constituencies of northern Chhattisgarh. Many areas bordering Jharkhand are also considered affected by the Naxalite or Maoist movement, although Maoist influence there is not as strong as in the areas of Bastar.

In this phase, elections will be held in 15 seats reserved for Scheduled Tribes, six seats reserved for Scheduled Castes and 30 general seats. The political future of 687 candidates will be decided across these 51 constituencies. They include Home Minister Ramvichar Netam, Health Minister Amar Agarwal, former Home Minister Nandkumar Patel and NCP state president Nobel Verma.

The Congress has left three of these 51 seats for its political ally, the NCP, while the BJP and BSP have fielded candidates in all 51 seats. Since Chief Minister Raman Singh's election was completed in the first phase, he campaigned vigorously for the second phase. Several other leaders, including BJP president Rajnath Singh, MP Shatrughan Sinha and Hema Malini, addressed election meetings. On behalf of the Congress, Rahul Gandhi, Vidya Charan Shukla, Motilal Vora and Haryana Chief Minister Bhupinder Singh Hooda addressed election rallies.

In these elections, the BJP has raised issues including development and providing rice to poor people at Rs 3 per kilogram. The Congress, meanwhile, has highlighted corruption in development work and promised to provide rice at Rs 2 per kilogram if it forms the government.""",

"081126_aamne_pachauri_alk": """The BJP won the previous election on the issues of electricity, water and roads. However, during its five-year term, the party has achieved little apart from changing the chief minister three times. Chief Minister Shivraj Singh says that electricity generation under his government has increased to 6,200 megawatts, but he forgets to mention that more than 1,000 megawatts of this electricity comes from the central government.

What could be a greater tragedy than the fact that electricity produced in Madhya Pradesh does not reach the state's farmers but is sent to other states? This gives an idea of the situation: while the national growth rate is around nine per cent, Madhya Pradesh's growth rate is only three and a half per cent. When Digvijaya Singh's government was in power, the state ranked 12th in development; it is now in 14th place. In agriculture, it was ranked ninth and is now 12th. In industrial development, it was eighth and is now 15th.

Law and order and corruption will be major issues in this election. The deteriorating law-and-order situation and corruption in the state are serious concerns. The state's minority communities are frightened by the BJP's communal politics and feel insecure. Our slogan is that, if we come to power, we will provide the people with an accountable and honest government.

The Shivraj government has broken all previous records when it comes to corruption. Perhaps for the first time in democratic history, corruption cases have been registered against 12 or 13 ministers of a state. Even the Chief Minister has allegations against him, and the public understands this very well.

There is no threat from the BSP. Inflation is certainly an issue, but it would not be fair to blame the UPA government at the Centre alone. Prices have risen across the world, and state governments also have a role in controlling them. We have promised in our manifesto to provide food grains at affordable prices. Some people say that this is not an economically sensible decision, but I would like to point out that it was taken only after consulting economic experts.

I am confident that the Congress party will win a majority and form the government on its own.

(Based on an interview with BBC correspondent Alok Kumar.)""",

"081205_sc_bcci_pp": """The Calcutta High Court had directed that a criminal case be registered against these people following a petition by Jagmohan Dalmiya. They were accused of filing false affidavits. A Supreme Court division bench headed by Chief Justice K. G. Balakrishnan issued this direction while hearing a petition filed by Sharad Pawar and others. In their petition, they said that the Calcutta High Court had delivered its judgment without hearing their side.

Apart from Sharad Pawar, the other people accused in the case are current BCCI president Shashank Manohar, former secretary Niranjan Shah, senior official Ratnakar Shetty, current secretary N. Srinivasan and chairman of the junior cricket committee Chirayu Amin.

The Calcutta High Court had issued the direction in connection with allegations that a false affidavit had been filed in the case involving former BCCI president Jagmohan Dalmiya. Dalmiya was suspended from the BCCI in 2006. He was accused of financial irregularities during the Cricket World Cup tournament held during his tenure. Dalmiya challenged the board's decision in court. The court later stayed his suspension, and in the election held that year, Dalmiya was once again elected president of the Cricket Association of Bengal.

A few months ago, Dalmiya had approached the court over the alleged filing of a false affidavit. Following this, in its order dated 12 November, the Calcutta High Court directed that criminal proceedings be initiated against six board officials.""",

"081217_ganguly_yuvraj_ac": """Sourav Ganguly, who had come to Delhi to play a Ranji Trophy match, told reporters that Yuvraj is an outstanding one-day cricket player. He said, "I am confident that he will prove himself in Test cricket as well."

It is worth noting that Yuvraj Singh took Ganguly's place after Ganguly retired. Yuvraj returned to Test cricket after eight months and scored 14 and 85 runs in the first Test against England in Chennai. England had set India a target of 387 runs to win. On the final day, India achieved the target with the loss of four wickets, thanks to an excellent partnership between Sachin Tendulkar and Yuvraj. Tendulkar scored an outstanding unbeaten century, while Yuvraj remained unbeaten on 85.

Indian captain Mahendra Singh Dhoni is very pleased with Yuvraj Singh's current form. During the one-day series against England, Dhoni said that it was pleasant to watch Sehwag and Sachin bat on the field, but Yuvraj had a different presence at the crease. According to Dhoni, when Yuvraj is in rhythm, he appears more aggressive than Sachin and Sehwag.""",

"090106_india_export": """According to the Federation of Indian Export Organisations (FIEO), the international market has deteriorated so badly that, for the first time since 1929, there is hardly any demand for Indian goods. According to the Press Trust of India, the federation's president, A. Sakthivel, told reporters in New Delhi, "At least one crore jobs will be lost by March."

It is worth noting that the export sector accounts for only 20 per cent of India's gross domestic product, but it provides employment to many people. During the first six months of the previous year, exports recorded a strong growth rate of 30.9 per cent. However, after October, this rate fell to just 12 per cent. This is the first time in five years that such a sharp decline in export growth has been seen. The rate fell even further in November.

Sakthivel said, "I can say that growth will remain negative in December and there will be no improvement over the next few months." He said that the measures announced by the government last week to improve the economy did not include any steps for the export sector.

The export target for the previous year was 200 billion dollars, but it is now expected that exports will reach only 180 billion dollars. The figures for December are still awaited. Europe and North America are the biggest markets for Indian goods, but both regions are facing the effects of the recession.""",

"090210_filmfestival_gorakhpur": """The festival, which has been held every year since 2006, was moved outside the university this time and organised among the wider city population. The four-day film festival, held last week, was organised by Jan Sanskriti Manch and the Expression Gorakhpur Film Society. Students and local residents showed considerable interest.

Audiences showed great interest in films such as Umut by renowned Turkish filmmaker Yilmaz Guney, Blackboard by Iranian director Makhmalbaf, Red Road of Hope by Chinese filmmaker Ying Ying, Saagar Tat Ke Saudagar by K. P. Sasi, Blood and Oil by Michael T. Clare, and Loha Garam Hai by Biju Toppo.

Gulabi Talkies and the popularity of MST

Gulabi Talkies, the award-winning film by well-known Kannada filmmaker Girish Kasaravalli, touched the hearts of Hindi-speaking audiences. Kasaravalli said, "Our problems are not limited by language." According to him, Gulabi Talkies, based on a story by feminist writer Vaidehi, explores the social and political implications of problems connected with ordinary life. While writing the screenplay, Kasaravalli made the main character a Muslim. Through the film, he also showed the dangers associated with consumerism and economic reforms.

The film MST by British filmmaker Gibby Zovell is based on the historic march organised by the Brazilian landless farmers' movement, Movimento Sem Terra. Gibby said, "The film is our journey alongside that historic march, in which 12,000 people travelled 238 kilometres over 17 days to demonstrate in Brazil's capital." Gibby had come from Britain at his own expense to show the film. He said that land is an issue throughout the world, including India, and that this may be why people in India showed interest in the film.

While inaugurating the Slumdog Millionaire film festival, writer Arundhati Roy said that, just as an American pilot had recently made an emergency landing in the Hudson River, President Barack Obama had been elected to make an emergency landing for American imperialism.

Responding to a question about Slumdog Millionaire, Roy said that the film was well packaged, but that the boy playing the main character as an adult appeared more confident than Anil Kapoor. She said this was mainly because he had been brought from abroad. She added, "Poverty in our country is being commercialised, but no significant steps are being taken to improve the condition of the poor."

Pranay Krishna Srivastava, general secretary of Jan Sanskriti Manch, said that organising the film festival was an initiative to stand against Hindutva and communal forces at the cultural level. Sanjay Joshi, convenor of the Expression Film Society and the film festival, believed that the event would help create a particular audience for this kind of cinema.""",

"090216_jobloss_column3_alk": """Finding a new job while you are still employed is relatively easy, but looking for work after losing your job is much more difficult. That is why I had also made a request to the HR department: give me a three-month notice period. You do not have to pay me a salary, but at least give me an opportunity to find another job. However, that did not happen.

In fact, the people at my office knew that my father was a retired judge. I do not know why, but they seemed concerned that I might take them to court. Perhaps that is why they followed every formal procedure related to my resignation exactly as required. I was given no concession.

My job search began just two days later. I started by informing friends and trying to find a job through my own network. Initially, some friends promised to arrange a job for me at their workplaces, but 15 days passed and nothing worked out. All my friends advised me to wait a few more days. Some said, "There was an opening just five days ago. You were a little late, my friend."

I had already understood what was happening, so I started visiting offices myself. Before doing that, I registered on a few websites. After that, waiting was the only thing I could do.

Ten days later, I received a call from a placement agency. For the first time in my career, apart from the initial days, I felt nervous before an interview. Perhaps this is also one of the side effects of having to look for a job after losing your position.

I had gone expecting an interview, but instead I was handed a written test paper. To be honest, I felt embarrassed about having to take a written test at this stage of my career. But I had no choice.

The interview was fine, but...

When I returned home, I kept wondering whether I would succeed. For the first time, I was afraid of failing. And that is exactly what happened. Before leaving, I had asked about the result and was told that I would know within two or three days. But by the morning of the fourth day, I had run out of patience and called them myself.

The reply was, "We haven't called you... which means they must have hired someone else." I said that I thought my interview had gone well. They replied, "Yes, your interview was good, but your profile did not match."

After that, I tried to prove myself at several offices, but I could not succeed. My search continues.

(Ajay's personal account was based on a conversation with BBC correspondent Alok Kumar. From next Friday, Vikas Shankar will share his story. He was a manager at a private bank and also lost his job.)"""
}

batch10_accepted_ids = [
    "081202_rice_india_ac"
]

print("Batch 10 corrections:", len(batch10_corrections))
print("Batch 10 accepted:", len(batch10_accepted_ids))
print("\nCorrection IDs:")
print(list(batch10_corrections.keys()))
print("\nAccepted IDs:")
print(batch10_accepted_ids)

Batch 10 corrections: 9
Batch 10 accepted: 1

Correction IDs:
['080826_blast_films', '080830_orissa_violence', '081118_cg_campaign_vv', '081126_aamne_pachauri_alk', '081205_sc_bcci_pp', '081217_ganguly_yuvraj_ac', '090106_india_export', '090210_filmfestival_gorakhpur', '090216_jobloss_column3_alk']

Accepted IDs:
['081202_rice_india_ac']


In [8]:
batch10_corrections = {
    "080826_blast_films": """PASTE THE CORRECTED ENGLISH TRANSLATION FOR ARTICLE 1 HERE""",
    "080830_orissa_violence": """PASTE THE CORRECTED ENGLISH TRANSLATION FOR ARTICLE 2 HERE""",
    "081118_cg_campaign_vv": """PASTE THE CORRECTED ENGLISH TRANSLATION FOR ARTICLE 3 HERE""",
    "081126_aamne_pachauri_alk": """PASTE THE CORRECTED ENGLISH TRANSLATION FOR ARTICLE 4 HERE""",
    "081205_sc_bcci_pp": """PASTE THE CORRECTED ENGLISH TRANSLATION FOR ARTICLE 6 HERE""",
    "081217_ganguly_yuvraj_ac": """PASTE THE CORRECTED ENGLISH TRANSLATION FOR ARTICLE 7 HERE""",
    "090106_india_export": """PASTE THE CORRECTED ENGLISH TRANSLATION FOR ARTICLE 8 HERE""",
    "090210_filmfestival_gorakhpur": """PASTE THE CORRECTED ENGLISH TRANSLATION FOR ARTICLE 9 HERE""",
    "090216_jobloss_column3_alk": """PASTE THE CORRECTED ENGLISH TRANSLATION FOR ARTICLE 10 HERE"""
}

batch10_accepted_ids = [
    "081202_rice_india_ac"
]

print("Batch 10 corrections:", len(batch10_corrections))
print("Batch 10 accepted:", len(batch10_accepted_ids))
print("Correction IDs:", list(batch10_corrections.keys()))
print("Accepted IDs:", batch10_accepted_ids)

Batch 10 corrections: 9
Batch 10 accepted: 1
Correction IDs: ['080826_blast_films', '080830_orissa_violence', '081118_cg_campaign_vv', '081126_aamne_pachauri_alk', '081205_sc_bcci_pp', '081217_ganguly_yuvraj_ac', '090106_india_export', '090210_filmfestival_gorakhpur', '090216_jobloss_column3_alk']
Accepted IDs: ['081202_rice_india_ac']


In [9]:
batch10_corrections = {
    "080826_blast_films": """Some of these films are being released, while others are still in production. Nishikant Kamat's film Mumbai Meri Jaan, released last week, is based on the train bombings that took place in Mumbai on 11 July 2006, in which around 200 people were killed.

The film's director, Nishikant Kamat, says, "This film portrays an attempt to recover from a major tragedy. The story of Mumbai Meri Jaan is not just about Mumbai or this incident; it is about how people are coping with such a crisis."

Mumbai Meri Jaan is an emotional film, whereas Wednesday, scheduled for release on 5 September, is a thriller. Director Neeraj Pandey, making his directorial debut with this film, says, "Wednesday tells the story of a few hours during which terrorists attempt to terrorise the city of Mumbai. The film is set against the backdrop of terrorism and was shot entirely in Mumbai, but its story is completely fictional."

The recently released psychological thriller Aamir tells the story of an ordinary man who is blackmailed and forced to plant bombs in certain areas of Mumbai. The film performed well and was also appreciated by critics.

The Little Godfather, being produced under the Popcorn Films banner, is also based on the Mumbai bombings. It tells the story of young children who sell goods on Mumbai's local trains and how they showed courage on the day of the bombings. Its director, Joy Augustine, believes that people quickly connect with real-life events. He says, "The film is about ordinary people who travel on Mumbai's trains and the children who sell goods on them, and how these people come together during such a crisis."

Film critics also see this as a significant change, although they consider it a temporary trend. Senior film critic Ajay Brahmatmaj says, "Although bombings may provide the backdrop for films, producers and directors do not want to make films entirely about them. Perhaps they do not want to become involved in unnecessary controversies. In older films, incidents such as floods and fairs were used to introduce twists in the story. Today, bombings are being used in a similar way, but this will change with time."

Such films are generally shorter than three hours and leave little room for music and romance. Directors such as Ram Gopal Varma, who have made several successful films about the Mumbai underworld, have also attempted to portray the rapidly growing connection between the underworld and terrorism in his recently released film Contract.

Alpana Mishra of UTV believes it is coincidental that several of the company's recently released films have stories in which bombings or terrorism play an important role. She says, "We are happy that these films are also conveying a social message.""",

    "080830_orissa_violence": """The situation in Orissa remains tense following the murder of Vishwa Hindu Parishad leader Swami Lakshmanananda. Koraput district had previously remained free from violent incidents, but tensions erupted on Saturday over objectionable remarks written against a particular community on the walls of the local Vikram Dev College.

This led to clashes between two communities. Police first used batons and fired tear gas shells to disperse the crowd involved in the clashes. When they failed to control the agitated crowd, police fired shots into the air.

Koraput Superintendent of Police Deepak Kumar said that a curfew had been imposed in the city. He himself was injured in the clashes. The violent crowd also attacked a police station and damaged its furniture.

The trouble began when youths angered by the writings on the walls started attacking female students from another community. At least two girls were injured in the incident.

The VHP alleges that Christian organisations were behind the murder of its leader in Kandhamal. However, the government has not yet made any clear statement on the matter. There are also suspicions that Maoist rebels may have been responsible for the murder.""",

    "081118_cg_campaign_vv": """Polling in this phase is scheduled for 20 November. The first phase of polling was completed on 14 November, and votes will be counted on 8 December.

Several important contests are scheduled in the second phase, including those involving former Chief Minister Ajit Jogi and his wife, Renu Jogi. The political futures of state Congress president Dhanendra Sahu, working president Satyanarayan Sharma, state BJP president Vishnu Deo Sai and Yudhvir Singh, son of Dilip Singh Judev, will also be decided in this phase.

On the final day of campaigning, several senior leaders from both major parties, including Sonia Gandhi and Lal Krishna Advani, remained busy campaigning.

The first phase of elections had also been held in Bastar in southern Chhattisgarh, an area affected by the Naxalite movement. Several reports of violence emerged during polling in Bastar's 12 tribal constituencies.

This time, elections are being held in the tribal constituencies of northern Chhattisgarh. Many areas bordering Jharkhand are also considered affected by the Naxalite or Maoist movement. However, Maoist influence here is not as strong as it is in parts of Bastar.

A total of 15 seats reserved for Scheduled Tribes are being contested in this phase, while six seats are reserved for Scheduled Castes and 30 are general constituencies. The political future of 687 candidates will be decided across these 51 seats. They include Home Minister Ramvichar Netam, Health Minister Amar Agarwal, former Home Minister Nandkumar Patel and NCP state president Nobel Verma.

The Congress has left three of these 51 seats for its political ally, the NCP, while the BJP and BSP have fielded candidates in all 51 constituencies.

Since Chief Minister Raman Singh's election was completed in the first phase, he campaigned vigorously for the second phase. BJP president Rajnath Singh, MP Shatrughan Sinha, Hema Malini and several other leaders also addressed election meetings.

On the Congress side, Rahul Gandhi, Vidya Charan Shukla, Motilal Vora and Haryana Chief Minister Bhupinder Singh Hooda addressed election rallies.

In these elections, the BJP has raised issues such as development and providing rice to poor people at Rs 3 per kilogram. The Congress, meanwhile, has highlighted corruption in development projects and promised to provide rice at Rs 2 per kilogram if it forms the government.""",

    "081126_aamne_pachauri_alk": """The BJP won the previous election on the issues of electricity, water and roads. However, during its five-year term, the party has achieved little apart from changing chief ministers three times.

Chief Minister Shivraj Singh says that electricity generation under his government has increased to 6,200 megawatts. However, he forgets to mention that more than 1,000 megawatts of this electricity comes from the central government. What could be a greater tragedy than the fact that electricity generated in Madhya Pradesh does not reach the state's farmers but is supplied to other states?

Consider this: while the national growth rate is around nine per cent, Madhya Pradesh's growth rate is only 3.5 per cent. When Digvijaya Singh's government was in power, the state ranked 12th in development. It is now in 14th place. In agriculture, it ranked ninth and is now 12th. In industrial development, it ranked eighth and is now 15th.

Law and order and corruption

The deteriorating law and order situation and corruption in the state are likely to become major issues in this election. The state's minority communities are frightened by the BJP's communal politics and feel unsafe.

Our slogan is that, if we come to power, we will provide the people with an accountable and honest government. The Shivraj government has broken all previous records when it comes to corruption. Perhaps, for the first time in democratic history, corruption cases have been registered against 12 or 13 ministers of a state. Even the Chief Minister has allegations against him, and the public understands this very well.

No threat from the BSP

Inflation is certainly an issue, but it would not be fair to blame the UPA government at the Centre for it. Prices have risen across the world, and state governments also have a role in controlling them.

We have promised in our manifesto to provide food grains at affordable prices. Some people say this is not an economically sensible decision, but I would like to point out that we made this decision only after consulting economic experts.

I am confident that the Congress party will win a majority and form the government on its own.

(Based on a conversation with BBC correspondent Alok Kumar.)""",

    "081205_sc_bcci_pp": """The Calcutta High Court had directed that criminal proceedings be initiated against these individuals following a petition filed by Jagmohan Dalmiya. They were accused of filing false affidavits.

A Supreme Court bench headed by Chief Justice K. G. Balakrishnan issued this direction while hearing a petition filed by Sharad Pawar and others. In their petition, they argued that the Calcutta High Court had delivered its judgment without hearing their side.

Apart from Sharad Pawar, the other people accused in the case are current BCCI president Shashank Manohar, former secretary Niranjan Shah, senior official Ratnakar Shetty, current secretary N. Srinivasan and junior cricket committee chairman Chirayu Amin.

The case

The Calcutta High Court had issued the direction over allegations that a false affidavit had been filed in connection with the case involving former BCCI president Jagmohan Dalmiya.

Dalmiya was suspended from the BCCI in 2006. He was accused of financial irregularities during the World Cup cricket tournament held during his tenure. Dalmiya challenged the board's decision in court. The court later stayed his suspension, and in the election held this year, Dalmiya was once again elected president of the Bengal Cricket Association.

A few months ago, Dalmiya approached the court over the alleged filing of a false affidavit. Following this, the Calcutta High Court, in its order dated 12 November, directed that criminal proceedings be initiated against six board officials.""",

    "081217_ganguly_yuvraj_ac": """Sourav Ganguly, who had come to Delhi to play a Ranji Trophy match, told reporters that Yuvraj was an outstanding one-day cricket player. He said, "I am confident that he will prove himself in Test cricket as well."

It is worth noting that Yuvraj Singh took Ganguly's place after Ganguly retired. Yuvraj returned to Test cricket after eight months and scored 14 and 85 runs in the first Test against England in Chennai.

England had set India a target of 387 runs for victory. On the final day, India achieved the target after losing four wickets, thanks to an excellent partnership between Sachin Tendulkar and Yuvraj. Tendulkar scored an impressive unbeaten century, while Yuvraj remained unbeaten on 85.

Indian captain Mahendra Singh Dhoni is very pleased with Yuvraj Singh's current form. During the ODI series against England, Dhoni said that it was a pleasure to watch Sehwag and Sachin bat on the field, but Yuvraj brought a different presence to the crease.

According to Dhoni, when Yuvraj is in rhythm, he appears more aggressive than Sachin and Sehwag.""",

    "090106_india_export": """According to the Federation of Indian Export Organisations (FIEO), the international market has not been in such a poor condition since 1929. Indian goods are finding almost no buyers.

According to the news agency Press Trust of India, federation president E. Sakthivel told reporters in New Delhi that at least 10 million jobs would be lost by March.

The export sector accounts for only 20 per cent of India's gross domestic product, but it provides employment to many people.

Exports recorded strong growth of 30.9 per cent during the first six months of last year. However, after October, the growth rate fell to just 12 per cent. This is the sharpest decline in export growth seen in the past five years. The rate fell even further in November.

"I can say that growth will remain negative in December and that there will be no improvement over the next few months," Sakthivel said.

He added that the measures announced by the government last week to revive the economy did not include any steps specifically for the export sector.

Last year, the export target was 200 billion dollars, but it is now expected that exports will reach only 180 billion dollars. December's figures are still awaited.

Europe and North America are the largest markets for Indian goods, but both regions are facing a recession.""",

    "090210_filmfestival_gorakhpur": """Held every year since 2006, the festival was moved outside the university this time and organised among the general public in the city.

The four-day film festival, held last week, was organised by Jan Sanskriti Manch and the Expression Gorakhpur Film Society. Students and local residents showed considerable interest.

The audience showed great interest in several films, including Umut by renowned Turkish filmmaker Yilmaz Guney, Blackboard by Iranian director Makhmalbaf, Red Road of Hope by Chinese filmmaker Ying Ying, Saagar Tat Ke Saudagar by K. P. Sasi, Blood and Oil by Michael T. Clare, and Loha Garam Hai by Biju Toppo.

Gulabi Talkies and the popularity of MST

Gulabi Talkies, the award-winning film by renowned Kannada filmmaker Girish Kasaravalli, touched the hearts of the largely Hindi-speaking audience.

Kasaravalli said, "Our problems are not limited by language." According to him, Gulabi Talkies, based on a story by feminist writer Vaidehi, explores the social and political implications of problems connected with everyday life.

While writing the screenplay, Kasaravalli made the main character a Muslim. Through the film, he also highlighted the dangers associated with consumerism and economic reforms.

The film MST by British filmmaker Gibby Zobel is based on the historic march organised by Brazil's landless farmers' movement, Movimento Sem Terra.

Gibby said, "The film follows our journey alongside that historic march, in which 12,000 people travelled 238 kilometres over 17 days to stage a demonstration in Brazil's capital."

Gibby had travelled from Britain at his own expense to show the film. He said that land is an issue throughout the world, including India, and this may be why Indian audiences showed interest in the film.

While inaugurating the Slumdog Millionaire film festival, writer Arundhati Roy said that, just as an American pilot had recently made an emergency landing on the Hudson River, President Barack Obama had been elected to make an emergency landing for American imperialism.

Responding to a question about Slumdog Millionaire, Roy said that the film was well packaged. However, she felt that the boy playing the grown-up version of the main character appeared even more confident than Anil Kapoor, mainly because he had been brought from abroad.

She said, "Poverty in our country is being commercialised, but no significant steps are being taken to improve the condition of poor people."

Pranay Krishna Srivastava, general secretary of Jan Sanskriti Manch, said that organising the festival was an initiative to oppose Hindutva and communal forces at the cultural level.

Sanjay Joshi, convenor of the Expression Film Society and the film festival, believed that the festival would help create an audience interested in a particular kind of cinema.""",

    "090216_jobloss_column3_alk": """I had never imagined that I would lose my job so suddenly. My father had retired as a judge, and I had always believed that my career was secure. When I received the notice, I asked for some time to look for another job.

I started searching for work immediately. I contacted friends and acquaintances, hoping that someone might be able to help me find a suitable position. I also approached a placement agency and registered with them.

After some time, I received a call for a written test. I prepared for it and appeared for the test, but I was not selected. I felt disappointed, but I continued looking for other opportunities.

I kept contacting people and applying for jobs. Sometimes I would receive a response, but often there was no reply. Even when I managed to speak to someone on the phone, it did not necessarily lead to an interview or an offer.

I will write more about this next Friday. Vikas Shankar's account will also appear then."""
}

batch10_accepted_ids = ["081202_rice_india_ac"]

print("Corrections:", len(batch10_corrections))
print("Accepted:", len(batch10_accepted_ids))
print("Correction IDs:", list(batch10_corrections.keys()))

Corrections: 9
Accepted: 1
Correction IDs: ['080826_blast_films', '080830_orissa_violence', '081118_cg_campaign_vv', '081126_aamne_pachauri_alk', '081205_sc_bcci_pp', '081217_ganguly_yuvraj_ac', '090106_india_export', '090210_filmfestival_gorakhpur', '090216_jobloss_column3_alk']


In [10]:
for article_id, corrected_text in batch10_corrections.items():
    mask = reviewed_translations["id"] == article_id
    reviewed_translations.loc[mask, "complete_english_translation"] = corrected_text
    reviewed_translations.loc[mask, "manual_review_status"] = "Manually corrected"

reviewed_translations.loc[
    reviewed_translations["id"].isin(batch10_accepted_ids),
    "manual_review_status"
] = "Human reviewed - no correction"

reviewed_translations.to_csv(
    f"{base_path}/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv",
    index=False
)

print("Batch 10 corrections applied:", len(batch10_corrections))
print("Batch 10 accepted articles marked:", len(batch10_accepted_ids))
print("\nUpdated review status:")
print(reviewed_translations["manual_review_status"].value_counts())
print("\nTotal reviewed:", (
    reviewed_translations["manual_review_status"] != "Pending review"
).sum())
print("\nSaved successfully.")

Batch 10 corrections applied: 9
Batch 10 accepted articles marked: 1

Updated review status:
manual_review_status
Pending review                    129
Manually corrected                 96
Human reviewed - no correction     25
Name: count, dtype: int64

Total reviewed: 121

Saved successfully.


In [11]:
pending_ids = reviewed_translations.loc[
    reviewed_translations["manual_review_status"] == "Pending review",
    "id"
].head(10).tolist()

source_lookup = df_250.set_index("id")
translation_lookup = reviewed_translations.set_index("id")

print("Batch 11 — Articles for Human Review")
print("Number of articles:", len(pending_ids))

for i, article_id in enumerate(pending_ids, 1):
    print("\n" + "=" * 100)
    print(f"ARTICLE {i}/10: {article_id}")

    print("\nHINDI ORIGINAL:\n")
    print(source_lookup.loc[article_id, "text"])

    print("\nCURRENT ENGLISH TRANSLATION:\n")
    print(translation_lookup.loc[
        article_id, "complete_english_translation"
    ])

Batch 11 — Articles for Human Review
Number of articles: 10

ARTICLE 1/10: 100418_tharoor_controversy_vv

HINDI ORIGINAL:

थरूर अक्सर अपनी टिप्पणियों के लिए सफ़ाई देते नज़र आए लेकिन एक साल से भी कम समय के अपने कार्यकाल में शशि थरूर जितने विवादों से घिरे रहे उसने राज्यमंत्री बनाए जाने के प्रधानमंत्री मनमोहन सिंह के निर्णय को एक तरह से सही ही साबित किया. आख़िर एक ऐसे विवाद ने ही उन्हें इस्तीफ़ा देने पर मजबूर कर दिया जिसका उनके विभागीय कार्य से कोई लेना देना नहीं था. विवाद दर विवाद शशि थरूर को लेकर पहला विवाद तब शुरु हुआ जब यह ख़बर प्रकाशित हुई कि विदेश मंत्री एसएम कृष्णा और शशि थरूर दोनों अपने सरकारी आवास की जगह पाँच सितारा होटल में रह रहे हैं. आम आदमी के मुद्दे पर चुनाव लड़ने वाली कांग्रेस पार्टी के लिए यह ख़बर शर्मिंदगी पैदा करने वाली थी, ख़ासकर तब जब वित्तमंत्री खर्चों में कटौती की बात कर रहे थे. लेकिन शशि थरूर ने तर्क दिया कि एक तो उनका सरकारी आवास तैयार नहीं है दूसरा किसी सरकारी अतिथि गृह में जिम नहीं है इसलिए वे अपने खर्चे पर होटल में रह रहे हैं और इस पर किसी को आपत्ति नहीं होनी चा

In [12]:
batch11_corrections = {
    "100418_tharoor_controversy_vv": """Shashi Tharoor often had to explain his comments. However, the number of controversies surrounding him during less than a year in office seemed to justify Prime Minister Manmohan Singh's decision to appoint him as a Minister of State. Eventually, a controversy unrelated to his ministerial work forced him to resign.

Controversy after controversy

The first controversy began when reports emerged that External Affairs Minister S. M. Krishna and Shashi Tharoor were staying in five-star hotels instead of their official residences. This was embarrassing for the Congress party, which had contested the elections on the issue of the common man, especially when the Finance Minister was calling for spending cuts.

Tharoor argued that his official accommodation was not ready and that government guest houses did not have gyms. He said he was staying in a hotel at his own expense and that no one should object to it.

Manmohan Singh always appeared somewhat lenient towards Tharoor. Later, when the Finance Minister advised ministers to travel by economy class as part of spending cuts, Tharoor created another controversy by referring to the class as "cattle class". He explained that people had failed to understand a common English expression. However, he had to appear before the party leadership and explain his remark.

Before this controversy had died down, the Minister of State for External Affairs, who was already known for his Twitter messages, caused another controversy by saying that people should work on Gandhi Jayanti instead of staying at home and relaxing.

After the case involving David Headley and Tahawwur Rana, linked to the Mumbai terrorist attacks, came to light, Tharoor again made headlines by publicly criticising the Indian government's decision to tighten visa rules.

Meanwhile, he once said that Saudi Arabia could play the role of an "interlocutor" between India and Pakistan. This led to a media controversy, with questions being raised about whether the Minister of State for External Affairs was speaking against India's policy of not accepting mediation by any third country between India and Pakistan. Tharoor again said that people had misunderstood his statement and that by "interlocutor" he did not mean mediator.

The latest controversy concerned the Kochi IPL team. It began with a Twitter exchange with IPL commissioner Lalit Modi and grew so serious that Tharoor eventually had to resign. Both the Congress party and the government had to explain their position on the controversy. Tharoor's friend Sunanda Pushkar also became a subject of media attention.""",

    "120518_ipl_glamour_magazine_sm": """The IPL is widely discussed because of its money and glamour. On the other hand, Mumbai Cricket Association officials have alleged that Shah Rukh Khan was drunk and abused security guards. What actually happened will become clear after the investigation. However, one thing is certain: Shah Rukh Khan himself has admitted that he used abusive language. As a respected citizen, he should apologise for it. His statement at the press conference that whatever he did was right was not appropriate.

Another issue is that players often seem to disappear into the background during the IPL. It is difficult to understand where the boundaries lie between team owners, coaches and players.

The English Premier League

It is worth mentioning that in the English Premier League, people generally do not know who owns each football team. Since IPL organisers say that the league was started on similar lines, they should learn from that model.

In fact, the combination of glamour, cricket, corporate interests and money is taking the game in the wrong direction. The IPL has also copied the American practice of bringing in cheerleaders. But the system in the United States is different. There, cheerleaders are properly trained, and both men and women perform this role. Here, however, inviting women to dance seems mainly intended to attract crowds.

The biggest harm caused by the IPL is that, in the pursuit of money, it has mixed sport with glamour and Bollywood and damaged the basic spirit of the game. Players sometimes even hide their injuries during matches, and this has affected the national team on several occasions.

The IPL may be a good form of cricket-based entertainment, but what happens in it should not come to be regarded as real cricket, while people lose interest in traditional cricket.

A ban imposed in haste?

As far as the Mumbai Cricket Association's (MCA) ban on Shah Rukh Khan is concerned, it should have been imposed only after he was found guilty. If he is indeed guilty, he should be punished. But the MCA should have waited until then. It is difficult to think of another example of someone being banned simply for entering a ground, especially a celebrity. At least, this has not happened in India before.

Before the IPL, celebrities did not usually come to watch matches in this manner. They would arrive quietly, watch the match and leave. In the IPL, however, celebrities own teams and behave as though cricket exists because of them. Such behaviour is a result of the IPL culture.

The ban does not seem likely to last very long, and perhaps it should not. The BCCI also needs to define the rules for franchises and clearly establish their limits so that such situations do not arise again.""",

    "120521_pak_mandir_burnt_ia": """The police had taken control of the temple, but in September 2011 a court allowed Hindus to worship there.

Peshawar police officer Shahid Khan told BBC correspondent Dilawar Khan Wazir that unidentified people set fire to a temple calendar and some other papers inside the Gorakhtari tehsil office on Sunday evening.

According to the police officer, preliminary information suggested that people responsible for looking after the temple might also have been involved in the incident. He said that a dispute had been going on for several days between the local archaeology department and some Hindus who looked after the temple over who should keep its keys.

The police said that the temple itself had not been damaged. The locks had not been broken, and no attempt had been made to break down the doors.

The investigation

According to Shahid Khan, a large number of police officers reached the site on Monday morning as soon as the incident was reported. Archaeology department officials also arrived near the temple, and an investigation is under way. No one has been arrested so far.

An archaeology department official said that only some papers inside the temple had been set on fire. According to the official, a man named Ramesh had set the papers alight himself and had called the media before informing the police.

It is worth noting that the Peshawar High Court allowed Hindus to worship at this temple in September 2011. However, a dispute continues between the archaeology department and the Hindus looking after the temple over who should keep its keys.

The temple is around 160 years old. It is historically important to local Hindus, and its distinctive feature is that it is located inside the tehsil office.""",

    "120809_salman_pkp": """Salman Khan and Katrina Kaif play the lead roles in Ek Tha Tiger, Salman's upcoming film. Speaking at the film's promotional event, Salman said, "I am very happy that my films are doing well. But at the same time, this situation is dangerous because we do not know what to do with the next film."

Salman said, "The characters I play in my films have a larger-than-life image, and nobody can kill them. But continuing to play such roles would become boring. The problem is that if we do not make our next films match that level, people will say that the earlier film was better. So the problem is that we ourselves do not know what to do."

Salman's career has been on the rise for the past three or four years. He has delivered one blockbuster after another, including Wanted, Dabangg, Ready and Bodyguard. All of these were action films. Dabangg, Ready and Bodyguard each earned more than Rs 100 crore.

Why only action films? What about romantic roles in the future?

Salman says that he keeps receiving offers for action films. However, he made it clear that he will also act in romantic films in the future. He will be seen in a romantic role in an upcoming film directed by Sooraj Barjatya.

Salman's remarkable run of success began with Maine Pyar Kiya, directed by Sooraj Barjatya. He later played the lead role in Hum Aapke Hain Koun, another film directed by Sooraj and released in 1994. That film was also a huge success and became the biggest Hindi film hit up to that time.

Ek Tha Tiger stars Salman Khan and Katrina Kaif. The film is being made under the Yash Chopra banner and is directed by Kabir Khan. Salman plays a RAW agent in the film. It is scheduled for release on 15 August. Given the excitement among Salman's fans, it is expected to have a very big opening.""",

    "120831_filmreview_joker_dk": """Arnab Banerjee reviews Joker for BBC Hindi.com.

Joker is Akshay Kumar's 100th film. It is Shirish Kunder's second film as a director, while Akshay Kumar completes his century in Bollywood with this film. As the title suggests, the film seems to be an attempt to turn its audience into jokers.

Everyone is expected to do their work with sincerity and dedication. As a film critic, my job is to watch every film in full and then give my opinion. But believe me, watching Joker from beginning to end was not only a test of my patience; it also made me wonder whether the film was making a joke of the audience's intelligence.

Joker is supposed to be a comedy, but its humour offers nothing new. In fact, I would say that the film reaches the heights of stupidity.

Akshay Kumar plays a scientist working for the American space agency NASA, where he has been researching aliens. He has to build a device that will allow humans to communicate with beings from another world. When he fails to build it, NASA officials give him one month to complete the task.

Meanwhile, Akshay has to return to his village, Paglapur, because he is told that his father has suddenly fallen ill. Paglapur is a village in India that does not appear on the map of the country.

As soon as Akshay, whom the film calls Joker, arrives in Paglapur with his girlfriend Sonakshi Sinha, one problem after another begins. He discovers that his father's illness was made up simply to bring him back to the village.

The village is called Paglapur because everyone living there is a little unusual and strange. There is neither electricity nor water, and nobody seems to have a solution to these problems.

Akshay takes responsibility for improving the village and comes up with an idea that draws the attention of the whole world to Paglapur. This is where the aliens enter the story. These aliens understand human language and dance and sing just like humans.

If you think real aliens appear in the film, let me clarify that this is actually Akshay Kumar's trick to attract the world's attention. Joker is Akshay and Sonakshi's second film together after Rowdy Rathore. Akshay does this because he also wants his village to enjoy modern facilities.

The mere mention of aliens is enough to bring media organisations from around the world to Paglapur. Now imagine sitting in a cinema where every character laughs, sings and dances without reason and behaves in an excessively artificial and dramatic manner. What would you do? Perhaps pull your hair out.

Director Shirish Kunder seems to be trying to convince us that a film can work without a story, provided you add a little comedy to it.

I think that after the success of Rowdy Rathore, Akshay Kumar himself became worried when he saw Joker. Perhaps he did not like the film, which may explain why he did not appear in the media to promote it.

The film is extremely disappointing. Neither the story has any substance nor do the characters have anything interesting to offer. Even Chitrangada Singh's special dance number, Kafirana Dil, has little to offer.

In the end, my advice is simple: even if you are a huge Akshay Kumar fan, you can easily manage without watching this film.

Related material.""",

    "120904_yuvraj_shabnam_ml": """Shabnam Singh, Yuvraj Singh's mother:

"It was a very difficult time when I took Yuvraj to America. I have great faith in my Guruji. He told me to take Yuvraj to America, and I became convinced that he would return completely cured. So I did exactly that. We had already sent all the test reports.

"The doctors said that the disease would be difficult to treat, but that it was not impossible. I treated the illness as a challenge and was fully prepared to face it. I had only one thought in my mind: whatever happened, I had to help my child and free him from cancer.

"At first, it was extremely difficult for both Yuvraj and me to accept that his illness was not an ordinary one but cancer. When Yuvraj returned to India after receiving cancer treatment in the United States, people welcomed him. They knew that he had been suffering from some problem, but he had never allowed anyone to see what he was going through.

Criticism

"There was also a period of considerable criticism. People said that Yuvraj had become fat and did not want to play. They said he did not want to play under someone else's captaincy. All these comments deeply affected him.

"But only he and I know how terrible that period was. He had to endure 30 or 40 injections, yet he would still play the next day. He often began to cry during that time, but I encouraged him because if I had started crying too, who would have given him strength?

"While he was undergoing treatment, I came to India. A television channel showed a programme about Yuvraj, and I became so emotional watching it that I started crying. At that time, I did not want to speak to anyone. I told people not to show sympathy towards Yuvraj and me.

Encouragement, not sympathy

"There were many moments when Yuvraj felt extremely weak. But people continued to encourage him on Twitter and other social networking sites. He kept drawing strength from their support.

"False and misleading reports were appearing in the media. Some people said that Yuvraj had lung cancer, while others made different claims. In this situation, Yuvraj decided that he would tell people himself what had happened to him and how he was being treated.

"His eyebrows had fallen out, his eyes looked unusually large, and the effects of steroids could be seen throughout his body. Standing in front of the mirror, Yuvraj would ask me whether he was the ugliest person in the world."

Shabnam Singh says that Yuvraj asked the doctors whether he would lose his hair. The doctors said yes, but assured him that it would grow back. This was a source of comfort to him. However, when his hair began falling out, he experienced a severe burning sensation. So Yuvraj decided to shave his head.

"His eyebrows had fallen out, his eyes looked unusually large, and the effects of steroids were visible throughout his body. He would stand in front of the mirror and ask me whether he was the ugliest person in the world."

Yuvraj returns to the field

As for his return to the field, even the doctors were asking why Yuvraj had not started playing again. According to the programme prepared by the doctors treating him, the side effects of cancer treatment remain in the body for around six weeks. He is now returning to the field.

Cancer had frightened Yuvraj. That fear was present then and remains even today, because the whole world would be watching him when he returned to the field. He knows that cricket is his life and that he cannot stay away from it.

(Based on a conversation with BBC correspondent Saima Iqbal.)""",

    "121009_rushdie_film_da": """PVR Pictures says that it has acquired the rights to the film in India and plans to release it in December. Earlier, the film's director, Deepa Mehta, had said that she was having difficulty finding a distributor in India.

Salman Rushdie's book The Satanic Verses was banned in India in 1988, and the ban is still in force. In January of this year, after receiving death threats, Rushdie declined to attend the Jaipur Literature Festival. The atmosphere of protest was so tense that he could not even address the gathering through a video link.

The film Midnight's Children, based on the book of the same name, was screened last month at the Toronto International Film Festival in Canada. PVR Pictures head Kamal Gianchandani told the news website Livemint.com, "The release date will be decided later, but it is certain that the scale of its release and marketing will be no less than that of a Bollywood film."

Earlier, some Hindu organisations had protested against Deepa Mehta's film Water being shot in Varanasi. She later filmed that movie in Sri Lanka. Midnight's Children was also shot there.""",

    "121028_international_others_thai_women_aa": """Kreatae had the responsibility of supporting her parents and siblings. There was good money in this profession, but it also brought the constant challenge of looking beautiful, which often proved very costly for women known as "pretties".

Thirty-two-year-old Athitiya Iyamayi had reached the age at which many women in this profession begin looking for other work. But for Athitiya, better known as Kreatae, giving up a job that paid 100 dollars, or around 6,000 rupees, a day was not easy. She was responsible not only for her parents but also for her brothers and sisters.

Kreatae had already spent thousands of dollars on improving her appearance. She had undergone several operations: changing the shape of her nose, narrowing her jaw and enlarging her breasts.

The painful pursuit of beauty

Kreatae's friend and fellow beauty worker, Pim Saisanard, says that Kreatae was not happy with her appearance.

"She used to say that if you wanted to be beautiful, you had to endure pain. She wanted bigger hips so that they would match her large breasts. She would say that she was going to make herself perfect."

There are two ways to increase the size of the hips. Most plastic surgeons recommend transferring a person's abdominal fat to the hips. Since Kreatae was very thin, she could not use this method. She chose another option involving the injection of artificial substances known as fillers into the hips.

These fillers are expensive. Even in Bangkok, one of the cheapest places for surgery, they cost 80,000 baht, or around 140,000 rupees.

Kreatae's friend introduced her to a man named Thanat Natwirakul, who called himself Dr Pop.

Cheap surgery proves costly

These young women are in high demand at international fairs, exhibitions and product launches held in Thailand.

Dr Pop was not a doctor and had very limited experience in cosmetic surgery. Kreatae agreed to undergo the procedure with him for half the usual price: 40,000 baht.

It is not clear exactly what went wrong during the procedure, but Kreatae began having difficulty breathing soon after the fillers were injected. It is possible that Dr Pop injected the filler into a blood vessel instead of the fatty tissue. The gel-like substance then reached her lungs and accumulated there, preventing oxygen from reaching her brain.

In the confusion that followed, Dr Pop took Kreatae to hospital. She fell into a coma and never regained consciousness. She died three weeks later.

How important is beauty?

Alyssa Fiyabunnantanpong also works in this beauty-driven profession. She believes Kreatae was simply unlucky; otherwise, many people undergo such inexpensive procedures.

Alyssa has undergone several operations herself. At the age of 25, she had eight teeth removed to give her jaw the desired shape.

Women in this profession usually get work through personal recommendations. Maintaining contact with other women in the same profession is therefore very important.

Pim Saisanard says, "If one woman gets a particular operation, the others do the same. If you do not undergo surgery, you look different from the others and do not get work."

It is true that not every illegal operation ends badly, but when something does go wrong, tragic cases such as Kreatae's occur.

Corruption

Dr Thep Vechavisit, who works at a private hospital in Bangkok, says that many people come to him wanting to improve their appearance through surgery. He attributes the increasing number of illegal procedures to corruption.

Dr Vechavisit says that people performing such operations have links with the police. They pay bribes, and the police do not take action against them.

Dr Jinda Rojanamatin, director of the dermatology department at a government hospital, says, "Many people who have worked with doctors start performing cosmetic surgery themselves, even though it is illegal."

Dr Rojanamatin says that the Public Health Ministry is aware of the problem but does not have enough personnel to deal with it.

Kreatae's death received widespread media attention, after which Dr Pop was arrested. He could face up to 13 years in prison and a fine of up to 50,000 baht.""",

    "121114_international_us_plus_obama_ia": """The CIA chief had to resign because of an extramarital relationship.

Addressing his first press conference at the White House since his election victory, on Wednesday President Barack Obama said, "I do not yet have any evidence. Based on what I have seen, I can say that no classified secrets were revealed in this matter and national security was not harmed."

When asked whether he would order an investigation into the FBI for informing him late about the matter, Obama said he did not want to interfere with the investigation and was therefore not taking any action at present.

"I have not made any decision yet because the investigation is still under way and we do not have all the information. The FBI has its own legal procedures and is conducting the investigation. It is better for us to wait."

However, President Obama strongly praised former CIA chief David Petraeus, who was facing a scandal.

"General Petraeus has had an outstanding career. He has served his country extremely well in Iraq, Afghanistan and as head of the CIA," Obama said.

David Petraeus resigned as CIA director on 8 November after the FBI had spent several months investigating his extramarital relationship with his friend Paula Broadwell.

In an earlier photograph, Obama, Panetta and Petraeus are seen together.

The FBI had informed US Attorney General Eric Holder about the matter in July. However, President Barack Obama was not informed until 7 November. This raised questions about why the country's president had not been told earlier about the investigation involving the CIA chief.

No tax relief for the wealthy

President Obama made his position clear on reducing spending and increasing taxes on the wealthy to put the economy back on track. He again stressed that taxes on the middle class should not be increased.

"Nobody should be surprised that I want to raise taxes on the wealthy. If their taxes increase a little, they will not suffer much; they will still remain wealthy. I want to provide tax relief to the middle class. But I am willing to listen to better ideas on this issue," Obama said.

He also said that he would seek advice from Mitt Romney, the Republican candidate who had lost the election.

"I will meet Romney as well. The election has only just ended, and I imagine Mitt Romney would also like to spend some time with his family. I would like to hear his views on improving the economy and tackling unemployment."

Obama said that he also wanted to work with members of the Republican Party, which held a majority in the House of Representatives, the lower chamber of the US Congress. He said he wanted to work together to address the problems faced by all Americans.

He also repeated his promise to reform immigration laws.

Iran and international issues

On Iran's nuclear programme and the violence in Syria, Obama reiterated his position.

He again warned Iran that it would not be allowed to develop a nuclear bomb.

"I will not allow Iran to acquire a nuclear bomb. I want to try diplomatic methods first, and we have seen that economic sanctions are having an effect," Obama said.

Regarding Syria, Obama stressed the need for the Syrian opposition to unite. He said that he had been the first world leader to call for a change of government in Syria.

However, he also said that the United States did not want American weapons to reach extremists in Syria who could later pose a threat to the United States.

On climate change, Obama said that he believed it was affecting the entire world and would have harmful consequences for future generations. He said that action was necessary.

He added that development and environmental protection should go hand in hand. Obama said he would meet scientists and environmental experts to find out what could be done to prevent the situation from becoming worse.""",

    "121115_2g_auction_ac": """Telecom companies have shown little interest in the auction.

The Indian government, which had incurred an estimated loss of Rs 1.76 lakh crore in the allocation of 2G spectrum, had hoped that the auction would bring at least Rs 40,000 crore into the government treasury. However, telecom companies showed little interest. After the first day of the auction, the government had received only Rs 9,200 crore.

BBC correspondent Samir Hashmi, reporting from Mumbai, explains why the auction failed and what its possible impact could be.

How do you view the 2G auction?

Wednesday was the second day of the auction. The Indian government had received a poor response on the first day, with only five companies participating. The government had managed to collect only Rs 9,200 crore.

One reason was that the base price for a licence covering the entire country was Rs 14,000 crore, which was considered very high. Companies felt that it was too expensive when compared with prices in 2001 or the cost of 3G licences. That is why the response on the first day was poor.

What was the response in different cities?

One notable feature of the auction was that no company applied for licences in major cities or circles such as Delhi, Mumbai, Karnataka and Rajasthan.

Wednesday's auction covered smaller areas. Apart from Bihar, there was little demand anywhere. The number of mobile phone users in Bihar is increasing rapidly, which is why some companies showed interest there. The other cities and states did not attract much enthusiasm.

How did the auction go for the government?

This is very bad news for the Indian government. When the auction began, the government had expected to earn Rs 40,000 crore. However, it is now being said that it may not even earn Rs 20,000 crore.

Will the auction affect ordinary people?

It could affect both consumers and the telecom industry because the bidding price is quite high. Telecom companies may increase mobile tariffs, even though prices in India are already quite low."""
}

batch11_accepted_ids = []

In [13]:
for article_id, corrected_text in batch11_corrections.items():
    mask = reviewed_translations["id"] == article_id
    reviewed_translations.loc[mask, "complete_english_translation"] = corrected_text
    reviewed_translations.loc[mask, "manual_review_status"] = "Manually corrected"

reviewed_translations.loc[
    reviewed_translations["id"].isin(batch11_accepted_ids),
    "manual_review_status"
] = "Human reviewed - no correction"

reviewed_translations.to_csv(
    f"{base_path}/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv",
    index=False
)

print("Batch 11 corrections applied:", len(batch11_corrections))
print("Batch 11 accepted articles:", len(batch11_accepted_ids))
print("\nUpdated review status:")
print(reviewed_translations["manual_review_status"].value_counts())
print("\nTotal reviewed:", (
    reviewed_translations["manual_review_status"] != "Pending review"
).sum())
print("\nSaved successfully.")

Batch 11 corrections applied: 10
Batch 11 accepted articles: 0

Updated review status:
manual_review_status
Pending review                    119
Manually corrected                106
Human reviewed - no correction     25
Name: count, dtype: int64

Total reviewed: 131

Saved successfully.


In [14]:
pending_ids = reviewed_translations.loc[
    reviewed_translations["manual_review_status"] == "Pending review",
    "id"
].head(10).tolist()

source_lookup = df_250.set_index("id")
translation_lookup = reviewed_translations.set_index("id")

print("Batch 12 — Articles for Human Review")
print("Number of articles:", len(pending_ids))

for i, article_id in enumerate(pending_ids, 1):
    print("\n" + "=" * 100)
    print(f"ARTICLE {i}/10: {article_id}")

    print("\nHINDI ORIGINAL:\n")
    print(source_lookup.loc[article_id, "text"])

    print("\nCURRENT ENGLISH TRANSLATION:\n")
    print(translation_lookup.loc[
        article_id, "complete_english_translation"
    ])

Batch 12 — Articles for Human Review
Number of articles: 10

ARTICLE 1/10: 121217_greek_football_brothel_sponsor_sdp

HINDI ORIGINAL:

स्थानीय वेश्यालय 'विला इरोटिका' फुटबॉल क्लब वूकेफलास एफसी को प्रायोजित कर रहा है और क्लब के खिलाड़ी गुलाबी शर्ट पहनकर उसका प्रचार कर रहे हैं. कुछ लोगों को भले ही इसमें हैरानी हो लेकिन टीम के कोच इयानिस बात्ज़ियोलास का कहना है कि इसमें शर्म की कोई बात नहीं है. वे कहते हैं, ''पेशेवर फुटबॉल में आप देखिए, वहां तो शराब का विज्ञापन होता है. मुझे लगता है कि वो इससे कहीं ज्यादा अनैतिक है.'' 'इंसानी जरूरत' टीम के कोच इयानिस बात्ज़ियोलास प्रायोजक के चयन के फैसले का बचाव करते हुए कहते हैं, ''ग्रीस में सेक्स के लिए वेश्यालय जाना कोई हैरानी की बात नहीं है. मुझे लगता है कि ज्यादातर युवा वहां जाते हैं क्योंकि ये इंसानी जरूरत है.'' वे कहते हैं, ''मुझे उस महिला पर गर्व है जो इस वेश्यालय की मालकिन है, गर्व इसलिए कि उन्होंने हमारा फुटबॉल खेलने का सपना सच करने में मदद की है.'' 'विला इरोटिका' की मालकिन साउला एलिव्रिडो को भी अपनी इस पहल पर गर्व है. मर्दाना कपड़े पहनने वाली स

In [15]:
batch12_corrections = {
    "121217_greek_football_brothel_sponsor_sdp": """The local brothel Villa Erotica is sponsoring the football club Voukefalas FC, and the club's players are promoting it by wearing pink shirts. Some people may find this surprising, but the team's coach, Ianis Batziolas, says there is nothing to be ashamed of. "In professional football, you see advertisements for alcohol. I think that is far more immoral than this," he says.

Human Needs

Defending the decision to choose this sponsor, coach Ianis Batziolas says, "In Greece, going to a brothel for sex is not unusual. I think most young people go there because it is a human need." He adds, "I am proud of the woman who owns this brothel because she has helped make our dream of playing football come true."

Saula Elivrido, the owner of Villa Erotica, is also proud of her initiative. Known for wearing masculine clothes, Saula has the image of a wealthy businesswoman. "I am doing this because I love Greece and enjoy helping young people," she says. "The authorities here do not provide much funding for sports and education. So why should a poor football club not come to me for help?"

Voices of Opposition

The local football association has protested against linking sex with sports. It has insisted that the business of selling sex should be kept away from sport. Under the agreement between the club and the brothel, team members must display the brothel's logo on their clothing.

"We cannot accept the promotion of prostitution," says the association's head, Marios Spirtos. "We know it is a part of life, but it is not appropriate for young players. Some of the players are even under 18."

The state of sports in Greece is not very good, and the future of players looks uncertain because of inadequate sports funding. Just eight years ago, Greece hosted the Olympic Games, and its national football team had also become European champions. But now both sports and players are struggling, which is why they are trying every possible way to survive. Perhaps both Villa Erotica and Voukefalas FC will benefit from this arrangement: the club may achieve something worthwhile, while the brothel may gain more customers.""",

    "130207_kashmir_rockband_imroz_sa": """Pervez Imroz has come forward in support of the girls' rock band Pragash. This could become the first legal battle between civil society and the influential clerical class in Indian-administered Kashmir.

The girls' rock band Pragash has already announced that it is disbanding. The announcement followed comments against the group on Facebook and a fatwa issued by Mufti Bashir, who is considered to be supportive of the government. A few days earlier, Mufti Bashir had issued a fatwa declaring that Muslim girls singing was against Islam.

In an exclusive conversation with the BBC, Pervez Imroz, head of the Coalition of Civil Society, said, "We are filing a public interest litigation in the High Court against Mufti Bashir. Civil society will not only question his legal authority but will also ask him what right he has to question people's likes and dislikes."

Alleged Collusion

Pervez Imroz has described Mufti Bashir as part of a "government agency" that often raises unnecessary issues. According to him, these agencies help the government divert attention from real problems by giving prominence to such issues. Human rights violations in Kashmir are among the most important of these problems.

According to Imroz, these religious leaders remain silent when graves are dug up in Kashmir and war-related crimes are committed. He adds, "Unfortunately, the Indian media gives such issues extensive coverage, allowing Kashmir to be portrayed inaccurately across India and in the West."

Imroz says he has no objection to providing legal assistance to the three boys arrested for posting comments against the girls in the rock band on Facebook.

Contradictory Attitude

According to Pervez, "The government's attitude here is completely contradictory. On the one hand, it says that it supports freedom of expression, but on the other hand, it turns against those who openly express their feelings."

The girls in the rock band Pragash had received threats. Pervez Imroz claims that the local administration, the Indian media and some Kashmiri religious leaders have joined forces to give the issue prominence in order to portray the entire Kashmiri society as conservative and backward.

Earlier, the Hurriyat Conference, the main separatist group in the Kashmir Valley, had accused the Indian media of deliberately exaggerating the issue.

But can opposing Mufti Bashir mean supporting the girls? Responding to this question, Pervez Imroz says that people will understand their intentions clearly.

Asked whether Kashmiri society has become more intolerant in recent times, Imroz says, "People here have traditionally organised musical evenings in accordance with their customs and culture. They were unhappy with the government's decision to allow filmmakers to shoot films in the valley. If people who favour violence organise musical events here, people are bound to react."

Meanwhile, police have raided several places in the Kashmir Valley and arrested three young men accused of threatening the three girls in the rock band.

Speaking to the media, the mother of one of the arrested youths said, "If my son commits a crime by condemning those girls, why is no action taken against those religious leaders? Why are clerics like Mufti Bashir becoming famous?""",

    "130220_dhoni_clarke_vd": """In 2011–12, Australia defeated India 4–0 on Australian soil. While Harbhajan Singh predicted that India would win the series 4–0, Australian captain Michael Clarke says his team will not play out of fear of India.

After losing 10 of their last 17 Test matches, Indian captain Mahendra Singh Dhoni will be under pressure. However, senior cricket journalist Pradeep Magazine says Dhoni is a player who does not get affected by pressure.

Speaking to the BBC, he said, "Considering the way India lost to England at home, another captain might not have survived. Many people say that Dhoni has continued as Test captain only because he has the backing of the board. But Dhoni has several strengths. He is an excellent captain in one-day cricket and a good player, although he has not made the same impact in Tests."

Asked how the series result would affect Dhoni's captaincy, Magazine says that the Board president would be under greater pressure than Dhoni to ensure a good performance, so that people do not get an opportunity to criticise him.

Michael Clarke

Australian captain Michael Clarke is himself in good form, but several of Australia's star players have recently retired. This team is being considered one of the weaker Australian sides to tour India. Magazine also believes that playing in India will not be easy for Australia.

He said, "Clarke is a great batsman and is in good form. He will want to show that he can build another world-beating team. If he manages to defeat India, it will be a major achievement. But Australia no longer has Ponting, and Hussey has also retired, which has weakened its batting. Australia will also miss having quality spin bowlers."

Sachin Tendulkar

Magazine believes Sachin Tendulkar has shown that he is still hungry for runs. He has played domestic cricket and does not want to retire from Test cricket just yet. However, Tendulkar himself knows that scoring a century in the Ranji Trophy is very different from scoring one against international teams.

He says, "If the opening batsmen perform well, there will be less pressure on Tendulkar. If he scores runs, he will certainly play the next series. But if he fails, the selectors will ask him some tough questions."

India and Australia will play a four-Test series. The first Test will begin in Chennai on 22 February."""
}

batch12_accepted_ids = []

In [16]:
batch12_corrections.update({
    "130404_sam_pitroda_profile": """Sam Pitroda founded C-DOT after returning to India in 1984. He is currently an adviser to the Indian Prime Minister on Public Information Infrastructure and Innovation. His role is to make citizen services more effective across different sectors and prepare a roadmap for innovation.

He is also the chairman of the National Innovation Council. Earlier, from 2005 to 2009, he served as chairman of the National Knowledge Commission. This high-level advisory body was formed to advise the government on improving knowledge-based institutions and infrastructure. During his tenure, he made around 300 recommendations in 27 different areas.

After completing his postgraduate degree in physics, Pitroda went to the Illinois Institute of Technology in Chicago, where he earned a master's degree in electronics. During the 1960s and 1970s, he worked on telecommunications and computing technology and holds more than 100 patents.

In 1984, Prime Minister Rajiv Gandhi invited him back to India. Pitroda established C-DOT (Centre for Development of Telematics) to carry out independent research and development in telecommunications. In recognition of his contribution to science, he was awarded the Padma Bhushan in 2009. As an adviser to Rajiv Gandhi, he also helped shape India's domestic and international telecommunications policies.

In the 1990s, Pitroda returned to the United States to pursue business interests and spent a long period doing technical research. In 2004, when the UPA came to power, Prime Minister Manmohan Singh invited him back to India and appointed him chairman of the National Knowledge Commission.

He has received several national and international awards. He was also an adviser to the United Nations in 1992.

During the previous year's Uttar Pradesh Assembly election, the Congress party projected him as an OBC 'poster boy' in an attempt to create a new electoral equation.""",

    "130528_speed_breaker_cars_impact_sy": """A car's radiator was damaged after it scraped against a speed breaker. The owner was told that repairs would take at least two days.

Roshni has been running an automobile service centre in Delhi for the past 15 years. She says speed breakers are good for her business because two or three damaged cars arrive at her workshop every day. Most of them are expensive cars. Authorised company service centres charge a lot for repairs, while her workshop charges only half as much.

Read what an ideal speed breaker should be.

Many people install CNG cylinders in large cars to save money. This makes the cars sit lower. At Roshni's service centre, mechanics increase the height of the shock absorber pads to improve ground clearance to some extent. However, some speed breakers in India are so high and narrow that even these adjustments are not enough.

Gyan Prakash, who owns a Honda Civic, says there are speed breakers everywhere. Even when he drives carefully, the underside of his car scrapes against them. Once, oil leaked from his car after it hit a speed breaker, and the vehicle stopped. He accepts that speed breakers improve safety, but says that having too many of them damages cars.

Owners of the Lancer, Linea, Dzire, Esteem and Verna have similar complaints.

How speed breakers put the brakes on life

Roads for cars, or cars for roads?

India does not follow the prescribed standards for speed breakers. When Maruti Suzuki decided to enter the Indian market decades ago, its team travelled across the country, from north to south and east to west, to study the roads. The company realised that it would need special technology and design for Indian conditions. It therefore developed cars with strong suspension systems and underbodies. Maruti's ground clearance became an important example in the Indian market, but customers still complain about damage caused by speed breakers.

CV Raman, Director of Engineering at Maruti Suzuki India Pvt Ltd, says that lower ground clearance generally improves a car's performance on roads. However, Indian roads are not suited to these cars, or perhaps these cars are not suited to Indian roads. At the same time, demand for long and large sedans is increasing.

Sedan's loss becomes SUV's gain

The undersides of sedans often scrape against speed breakers. Demand for luxury cars is rising, but the roads do not meet the expectations of their owners. Experts say that many people are switching from sedans to SUVs because of the damage caused by poor roads and speed breakers.

Amitabh Mukherjee, general manager of a Chevrolet service centre, says speed breakers can cause serious damage to long cars, and repairs may cost lakhs of rupees. The condition of Indian roads makes it seem as though sedans should not be manufactured here. Even villages connected to Delhi have poor roads, so people often prefer SUVs.

Jaguar receives similar complaints from its customers. However, the company cannot change the basic technology of its cars simply to suit Indian roads.

What can luxury car companies do?

Automobile expert Hormazd Sorabjee says that bringing Indian roads up to European standards would take decades. Car manufacturers can increase suspension height and ground clearance or use bigger tyres, but they should avoid making too many changes to the original design.

Until authorities crack down on illegal speed breakers, cars will continue to bear the brunt.

Readers are invited to send photographs of unusually high and uncomfortable speed breakers to hindi.desk@bbc.co.uk or hindi.letters@bbc.co.uk, or share them on Facebook and Twitter. Some of the photographs will be published on the BBC Hindi website.""",

    "130529_karunanidhi_tamilnadu_marriage_ns": """M. Karunanidhi says that practising yoga regularly is the secret of his good health.

PTI reports that M. Karunanidhi, a supporter of self-respect marriages, recalled moments from his life that have now become history.

In 1944, the family of the woman he loved wanted them to have a traditional wedding. However, his principles did not allow him to agree to such a marriage, and the wedding did not take place.

The 89-year-old former Tamil Nadu Chief Minister was speaking at a wedding ceremony in Chennai. He recalled that the woman's family had rejected his wish for a simple marriage. He was opposed to traditional rituals such as the chanting of mantras and tying of the mangalsutra. Her family, however, did not want their daughter to be married in that way.

Life

Karunanidhi has appointed his son M. K. Stalin as his political successor.

The former Tamil Nadu Chief Minister said that he had always acted according to his principles and therefore could not marry the woman he loved. He later married Dayalu, and they had four children.

Dravidian thinkers E. V. Ramasamy Periyar and C. N. Annadurai also supported self-respect marriages, and they did not face any problems in their personal lives because of their views.

Karunanidhi has had a political career spanning 60 years and has served as Chief Minister of Tamil Nadu five times. He has also set a record by winning every election he has contested so far.

For more news, download the BBC Hindi app and follow BBC Hindi on Facebook and Twitter."""
})

In [17]:
batch12_corrections.update({
    "130606_china_gender_ss": """A man in China who went to hospital with abdominal pain caused by an ovarian cyst discovered that, genetically, he was female and not male.

Doctors found that he had a rare combination of two genetic conditions: Turner syndrome and congenital adrenal hyperplasia.

Turner syndrome affects certain female physical characteristics and can prevent a person from becoming a mother. People with this condition may appear female but may not develop typical female physical characteristics. Congenital adrenal hyperplasia, meanwhile, can cause the body to produce excessive male hormones. The combination of these two conditions resulted in an unusual situation.

The person is 4 feet 5 inches tall and has a slim build. He looked and behaved like a man and did not know about the female characteristics he had. He had also experienced difficulty urinating and other health problems. He had always considered himself male.

Doctors at Kwong Wah Hospital and Queen Elizabeth Hospital in Hong Kong said that he would have had ovaries and female characteristics. The person identifies as a man and will receive male hormone treatment.""",

    "130707_raghavji_mp_fma": """The Congress party has demanded the resignation of Madhya Pradesh Chief Minister Shivraj Singh Chouhan after police registered a case against former minister Raghavji.

The complainant, Rajkumar Dangi, who was an associate of Raghavji, appeared before the police and gave his statement. Habibganj police in Bhopal registered a case against the former minister under sections of the Indian Penal Code relating to unnatural acts, intimidation and other offences.

The BJP took the decision after discussions among its senior leaders. A meeting was held at the Chief Minister's residence on Sunday evening. Those present included state BJP president Narendra Singh Tomar, Prabhat Jha and former Chief Ministers Sunderlal Patwa and Kailash Joshi.

While the party decided to expel Raghavji, he celebrated his 78th birthday in Vidisha.

On Saturday evening, the BJP suspended Shivshankar Pateria from its primary membership. Pateria had taken responsibility for preparing the CD related to the case.

The winter session of the Madhya Pradesh Assembly begins on Monday, and the issue is expected to cause an uproar. The Congress is preparing to bring a no-confidence motion against the Shivraj Singh Chouhan government.

The government is now facing both a no-confidence motion and the case involving allegations of sexual abuse against Raghavji.

Leader of the Opposition Ajay Singh asked Chief Minister Shivraj Singh Chouhan to resign on Sunday.""",

    "130910_sehwag_gambhir_pujara_pp": """Virender Sehwag and Gautam Gambhir have been included in the India A squad for the second and third four-day matches against West Indies A. However, the selectors have not considered either of them for the captaincy. The less experienced Cheteshwar Pujara has been named captain.

Fast bowler Zaheer Khan, India's pace spearhead, has also been included in the squad.

Sehwag last played a Test match against Australia in March this year. He has since been selected for the India A team. His last One Day International was against Pakistan in Kolkata in January, and he has not played an ODI since then.

Yuvraj Singh will captain India in the one-day matches against West Indies A. Gambhir last played a Test match in December last year. His most recent ODI was against England in Dharamsala in January.

Yuvraj Singh has also been out of the Indian team for a long time.

The selection meeting was held in Visakhapatnam, where squads were announced for the ODI, T20 and four-day matches against West Indies A. Teams India Red and India Blue were also announced for the NKP Salve Challenger Trophy.

Irfan Pathan will captain India Red, while Yuvraj Singh will lead India Blue.

For more news, download the BBC Hindi app and follow BBC Hindi on Facebook and Twitter."""
})

In [18]:
batch12_corrections.update({
    "130912_guinnness_record_bulgarian_swim_sr": """A 59-year-old Bulgarian man, Jane Petkov, crossed Lake Ohrid in Macedonia in about two hours and 45 minutes while his arms and legs were tied to his body.

The organiser, Saso Tokov, told AFP that Petkov swam on his back, moving through the water like a dolphin. According to Macedonian media reports, his average speed was 0.7 kilometres per hour.

Petkov said that the water was extremely cold during the first kilometre, but after that everything went well and he faced no problems.

He added that if the water had been warm, he could have swum 10 kilometres or even more.

The organisers formally recorded his performance for consideration by Guinness World Records.

However, local media reports have differed about the distance covered and the time taken. Guinness World Records has not yet made any public comment on the achievement.

For more news, download the BBC Hindi app and follow BBC Hindi on Facebook and Twitter."""
})

print("Batch 12 corrections ready:", len(batch12_corrections))
print("Accepted without correction:", len(batch12_accepted_ids))

Batch 12 corrections ready: 10
Accepted without correction: 0


In [19]:
corrections_df = pd.DataFrame(
    list(batch12_corrections.items()),
    columns=["id", "complete_english_translation"]
)

if batch12_accepted_ids:
    reviewed_translations.loc[
        reviewed_translations["id"].isin(batch12_accepted_ids),
        "manual_review_status"
    ] = "Human reviewed - no correction"

for _, row in corrections_df.iterrows():
    reviewed_translations.loc[
        reviewed_translations["id"] == row["id"],
        "complete_english_translation"
    ] = row["complete_english_translation"]

reviewed_translations.loc[
    reviewed_translations["id"].isin(batch12_corrections.keys()),
    "manual_review_status"
] = "Manually corrected"

output_path = (
    f"{base_path}/09_sentence_level_translation/"
    "FactIndic_English_250_Reviewed.csv"
)

reviewed_translations.to_csv(output_path, index=False)

print("Batch 12 saved successfully.")
print("Corrections applied:", len(batch12_corrections))
print("Accepted without correction:", len(batch12_accepted_ids))
print("\nUpdated review status:")
print(reviewed_translations["manual_review_status"].value_counts())
print("\nSaved to:", output_path)

Batch 12 saved successfully.
Corrections applied: 10
Accepted without correction: 0

Updated review status:
manual_review_status
Manually corrected                116
Pending review                    109
Human reviewed - no correction     25
Name: count, dtype: int64

Saved to: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv


In [20]:
corrections_df = pd.DataFrame(
    list(batch12_corrections.items()),
    columns=["id", "complete_english_translation"]
)

if batch12_accepted_ids:
    reviewed_translations.loc[
        reviewed_translations["id"].isin(batch12_accepted_ids),
        "manual_review_status"
    ] = "Human reviewed - no correction"

for _, row in corrections_df.iterrows():
    reviewed_translations.loc[
        reviewed_translations["id"] == row["id"],
        "complete_english_translation"
    ] = row["complete_english_translation"]

reviewed_translations.loc[
    reviewed_translations["id"].isin(batch12_corrections.keys()),
    "manual_review_status"
] = "Manually corrected"

output_path = (
    f"{base_path}/09_sentence_level_translation/"
    "FactIndic_English_250_Reviewed.csv"
)

reviewed_translations.to_csv(output_path, index=False)

print("Batch 12 saved successfully.")
print("Corrections applied:", len(batch12_corrections))
print("Accepted without correction:", len(batch12_accepted_ids))
print("\nUpdated review status:")
print(reviewed_translations["manual_review_status"].value_counts())
print("\nSaved to:", output_path)

Batch 12 saved successfully.
Corrections applied: 10
Accepted without correction: 0

Updated review status:
manual_review_status
Manually corrected                116
Pending review                    109
Human reviewed - no correction     25
Name: count, dtype: int64

Saved to: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv


In [21]:
pending_ids = reviewed_translations.loc[
    reviewed_translations["manual_review_status"] == "Pending review",
    "id"
].head(10).tolist()

source_lookup = df_250.set_index("id")
translation_lookup = reviewed_translations.set_index("id")

print("Batch 13 — Articles for Human Review")
print("Number of articles:", len(pending_ids))

for i, article_id in enumerate(pending_ids, 1):
    print("\n" + "=" * 100)
    print(f"ARTICLE {i}/10: {article_id}")

    print("\nHINDI ORIGINAL:\n")
    print(source_lookup.loc[article_id, "text"])

    print("\nCURRENT ENGLISH TRANSLATION:\n")
    print(translation_lookup.loc[
        article_id, "complete_english_translation"
    ])

Batch 13 — Articles for Human Review
Number of articles: 10

ARTICLE 1/10: 130912_nepal_strike_ssr

HINDI ORIGINAL:

नेपाल में हड़ताल के चलते सड़कें सूनी दिखीं सरकार द्वारा नवंबर में चुनाव कराने की योजना के विरोध में ये हड़ताल की गई. स्कूल और बाजार बंद हैं, सड़कें सूनी पड़ी हैं और वाहन चलते नहीं दिखे. आह्वान ये हड़ताल नेपाल में 33 पार्टियों के गठबंधन द्वारा बुलाई गई थी, जिसमें नेपाल कम्युनिस्ट पार्टी - माओवादी भी शामिल है. मार्च में एक सीनियर जज की अगुवाई वाली सरकार को चुनाव कराने के लिए नियुक्त किया गया था. नेपाल सुप्रीम कोर्ट के मुख्य न्यायाधीश खिलराज रेजमी प्रशासन पर ताजा चुनाव 19 नवंबर को कराने की जिम्मेदारी है. नेपाल में आखिरी बार संसदीय चुनाव 2008 में कराए गए थे. नये संविधान का मसौदा तय करने के बाद संवैधानिक संसद का कार्यकाल मई 2012 में खत्म हो गया. (बीबीसी हिन्दी के एंड्रॉएड ऐप के लिए यहां क्लिक करें. आप हमें फ़ेसबुक और ट्विटर पर फ़ॉलो भी कर सकते हैं.)

CURRENT ENGLISH TRANSLATION:

In Nepal, the strike hit the roads as a protest against the government's plan to hold elections i

In [22]:
batch13_corrections = {
    "130912_nepal_strike_ssr": """Streets in Nepal were deserted during a nationwide strike called to protest against the government's plan to hold elections in November. Schools and markets remained closed, roads were empty and vehicles were not seen on the streets.

The strike was called by an alliance of 33 political parties in Nepal, including the Nepal Communist Party-Maoist.

In March, a government led by a senior judge was appointed to conduct the elections. The administration headed by Nepal's Supreme Court Chief Justice Khil Raj Regmi is responsible for holding fresh elections on 19 November.

The last parliamentary elections in Nepal were held in 2008. The term of the Constituent Assembly ended in May 2012 after it failed to complete the drafting of a new constitution.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "131026_iran_hanging_akd": """Iranian soldiers deployed along the border (file photo)

Mohammad Marzia, the attorney general of Sistan-Baluchistan province, was quoted as saying that the rebels belonged to a group considered a threat to the authorities.

The rebels were hanged at a prison in Zahedan, northwest of Saravan. This is the same area where the killings took place along the border the previous night. It is not clear whether those executed were directly connected to the border killings.

Hayatullah Mir Moradzehi, a member of parliament from the Saravan region, blamed anti-revolutionary guerrillas for the attacks. However, he also told a local news agency that it had not been confirmed whether a group called Jaish al-Adl had claimed responsibility for the attacks.

Discrimination

In recent years, the Sunni armed group Jundallah has carried out several attacks against the government. The Sunni Muslim population of Sistan-Baluchistan has repeatedly accused Iran's Shia-led government of discrimination.

The Iranian Students' News Agency (ISNA) also reported the executions.

Regarding Friday's violence along the border, the Fars News Agency quoted the province's deputy governor general, Rajabali Sheikhzada, as saying that those responsible for the border attack had fled to Pakistan.

There have been several deadly clashes in this region in recent years. Iran lies on a major drug-trafficking route between Afghanistan and Europe. According to AFP, which quoted officials, more than 4,000 police officers and soldiers have been killed in clashes with drug traffickers over the past three decades.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "131106_uk_migration_contribution_rd": """A study says that recent immigrants to the UK are less likely than people born in the country to live in social housing or claim benefits. The researchers say that immigrants' contribution is "significantly higher" than what they receive. The British government says that stricter rules were necessary to protect the benefits system.

According to a report by Dr Tommaso Frattini and Professor Christian Dustmann of University College London's Centre for Research and Analysis of Migration, immigrants who arrived after 1999 were 45 per cent less likely than British citizens to receive government benefits and tax credits in 2000–2001. They were also 3 per cent less likely to live in social housing.

Negative Contributions

The researchers say that, after comparing immigrants and local residents by age, gender and education, recent immigrants were 21 per cent less likely to receive benefits.

Immigrants from the European Economic Area (EEA, which includes the European Union as well as Norway, Iceland and Liechtenstein) paid 34 per cent more in taxes than the value of the benefits they received.

Professor Christian Dustmann says that immigrants contribute more. Contributors from outside the EEA paid 2 per cent more in taxes than the value of the benefits they received. By comparison, British people paid 11 per cent less in taxes than the value of the benefits they received during the same period.

According to the study, immigrants from non-EEA countries paid less tax and received more benefits between 1995 and 2001. This was mainly because they had more children than British people.

The researchers collected data from the British Labour Force Survey and government reports.

Professor Dustmann says, "Unlike most European countries, the UK receives highly skilled and highly educated immigrants from the EEA and elsewhere."

He adds, "Our research shows that over the past decade or so, the UK has benefited economically from immigrants from the EEA."

"Given these facts, claims that EEA immigrants are engaging in 'benefit tourism' appear to be far removed from reality."

Sir Andrew Green of the pressure group Migration Watch says the report has been "distorted".

Speaking on BBC Radio 4's Today programme, he said, "Under the previous government, there were around four million immigrants here, two-thirds of whom came from outside the European Union."

"Since 1995, they have made a negative contribution overall. So the contribution of non-EU immigrants is either very small or negative."

However, he accepts that if the entire European Union is considered, the contribution is "clearly positive". Andrew says this is to be expected because the figures include "German engineers, French fashion designers and Swiss bankers from the EEA".

"The main issue for the future is low-income immigrants from Eastern Europe," he says.

"The report looks backwards, not forwards. The professor's report does not include future healthcare spending, which will increase as immigrants grow older, nor does it include pensions, which will involve considerable costs."

Government Action Is Right

Professor Dustmann said on the Today programme, "It is true that recent immigrants are young, but they are also better educated."

"So although they may receive benefits now, they will contribute more in the future as they progress in their careers and reach their full earning potential."

"Naturally, the more you earn, the more tax you pay."

A government spokesperson said, "We welcome people who come here to contribute to the economy, but it is also right that we introduce strict measures to protect the British benefits system from misuse."

The spokesperson added that the government was tightening the rules so that benefits would go only to those who were "legally allowed to live in the UK".

Meanwhile, a separate UCL study released on Tuesday warned that the government's target of reducing the number of immigrants by millions was "neither useful nor effective as a matter of policy".

The report said that measures to restrict work-related, student and family immigration had damaged Britain's image as a good place to work and study.

According to the 2011 census, 13 per cent of the population of England and Wales was born outside the UK.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)"""
}

batch13_accepted_ids = []

In [23]:
batch13_corrections.update({
    "131202_saina_sindhu_badminton_vr": """What India's rising badminton star PV Sindhu said after winning the Macau Open Grand Prix Gold tournament last Saturday reveals a lot about her growing confidence.

In the final, Sindhu defeated Canada's Michelle Li 21-15, 21-15. In the semi-final, she beat Chinese qualifier Qin Jinjing 21-13, 18-21, 21-19.

(International stars absent)

There is little doubt about PV Sindhu's ability. Earlier this year, she won a bronze medal at the World Badminton Championships, becoming the first Indian woman badminton player to achieve this feat.

Despite this achievement, Sindhu still has a long way to go in international badminton. She was the top seed at the Macau Open and lived up to that position. Before this, she had also won the Malaysian Open.

After her Macau Open victory, all eyes will be on Sindhu because the fortunes of another Indian badminton star, Saina Nehwal, have not been very good recently.

In fact, Saina Nehwal has not won a title this year. However, she performed brilliantly in the first Indian Badminton League. She played like a champion and led her franchise to the title.

The IBL was the first tournament in which Saina Nehwal and PV Sindhu faced each other. In their match in Delhi, Sindhu could not match Saina's performance, and the same happened in their second meeting.

A series of setbacks

After the IBL, many people expected Saina Nehwal and PV Sindhu to transform Indian badminton, but that has not happened.

Although Saina did not participate in the Macau Open, she had earlier been knocked out in the second round of the Hong Kong Open. Sindhu also failed to make an impact there and was eliminated in the first round.

(Saina avenges her defeat)

Before the Hong Kong Open, Saina had been eliminated in the second round of the China Open. She also lost in the second round of the French Open Super Series and failed to defend her title at the Denmark Open.

At the French Open, Sindhu was eliminated in the second round, while at the Denmark Open she lost in the first round. At the Japan Open, Sindhu was knocked out in the second round, while Saina's disappointing run continued.

At the Singapore Open, Saina could reach only the third round. At the Indian Open, Sindhu managed to reach the semi-finals, but Saina was eliminated in the second round.

Earlier this year, Sindhu also took part in the Asian Badminton Championships, where she was eliminated in the quarter-finals. At the Swiss Open, Sindhu lost in the second round, while Saina reached the semi-finals.

At the All England Badminton Championships, one of the most prestigious tournaments of 2013, Sindhu was eliminated in the second round.

The road to success

Saina, however, produced an excellent performance and reached the semi-finals. At that time, Saina was ranked second in the world, while Sindhu was still waiting to receive a world ranking.

Saina is currently ranked sixth in the world. At the Malaysian Open held in January this year, Saina lost in the semi-finals, while Sindhu was eliminated in the first round.

(How Saina's fortunes changed)

Earlier, at the Korea Open, Sindhu lost in the second round and Saina was eliminated in the quarter-finals.

PV Sindhu's path to success opened at the Malaysian Open Grand Prix Gold tournament, where she defeated Singapore's Juan Gu 21-17, 17-21, 21-19 to win the title.

Overall, the statistics show that neither PV Sindhu nor Saina Nehwal performed at the level expected of them this year. However, Sindhu is only 18 years old, and her future looks very promising.

Although Saina has not been very successful this year, it is also true that she came close to winning several titles but narrowly missed out.

Sindhu's journey to success has only just begun, while Saina can use her experience to make the kind of comeback expected of her next year.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "131227_diamond_antarctica_tk": """Commercial exploitation of any kind of mineral resources is prohibited in the Antarctic region. The research has been published in the journal Nature Communications.

Diamonds are formed from pure carbon under extreme heat and pressure at a depth of around 150 kilometres beneath the Earth's surface.

Kimberlite

Diamonds are found in kimberlite. Volcanic eruptions bring diamonds towards the Earth's surface, where they are usually found embedded in a type of blue rock called kimberlite.

The presence of kimberlite has helped identify diamond-bearing regions in several parts of the world, including Africa, Siberia and Australia.

Researchers have found evidence of kimberlite in Antarctica for the first time. The research team discovered three samples on the slopes of Mount Meredith in the northern Prince Charles Mountains.

Dr Teal Riley, a survey geologist with the British Antarctic Survey, said, "The discovery of Group One kimberlite is significant because this type of kimberlite has a much higher likelihood of containing diamonds."

He added, "However, only around 10 per cent of Group One kimberlite deposits are economically viable. Therefore, this discovery is still a long way from encouraging any diamond-mining activity in Antarctica."

Legal barriers

Even if diamonds are found in large quantities in this remote region, there are still significant legal barriers to extracting them.

There may be diamonds beneath Antarctica's frozen landscape. The 1991 Protocol on Environmental Protection to the Antarctic Treaty explicitly prohibits mining activities involving mineral resources, except for scientific research.

However, the protocol is due to be reviewed in 2041, and its provisions could be changed at that time.

Dr Kevin Hughes of the Scientific Committee on Antarctic Research said, "It is not known what position the countries that are parties to the treaty will take on mineral extraction in Antarctica after 2041. By then, technology may have developed to make mining in Antarctica economically viable."

He added, "Another issue is that the provisions of the protocol do not apply to countries that are not parties to it. This includes the provisions prohibiting mining."

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "131231_drone_test_site_sr": """The global market for drones is expected to reach $89 billion over the next decade.

According to the Federal Aviation Administration (FAA), the six states selected are Alaska, Nevada, New York, North Dakota, Texas and Virginia.

The selection process is part of a project to establish safety and operating regulations for drones by the end of 2015. Until now, drones have mainly been used by the military. However, real estate agents, farmers and service companies are now exploring their potential commercial uses.

Safety is the main challenge

According to FAA chief Michael Huerta, approval is being sought to allow unmanned aircraft to fly in US airspace, but safety will remain the top priority.

Pilots will be informed through routine announcements about where drones are scheduled to fly.

An FAA statement said that it took around 10 months to reach the decision. Proposals were received from 24 states.

Before making its decision, the agency examined the geographical conditions of the proposed test sites, ground infrastructure, research requirements, air traffic density and potential risks.

A flight for $25

Commercial use of drones in agriculture and law enforcement is expected to develop on a large scale.

Police departments have to spend hundreds of dollars per hour to deploy a helicopter. However, sending an unmanned aircraft into the sky would cost only $25 (around 1,500 rupees).

Farmers could also use drones to spray pesticides on their crops or monitor livestock.

The FAA estimates that five years after drone operations are legalised, the number of drones in the sky could reach around 7,500.

70,000 jobs

Plans for the commercial use of drones have faced considerable criticism from both right-wing and liberal groups. However, leaders of the states selected for the trials are pleased with the decision.

Democratic Senator Harry Reid of Nevada said, "This is very good news for Nevada and will provide a major opportunity for our economy."

An industry-sponsored study estimates that easing restrictions on drones could create around 70,000 jobs, including jobs for drone operators, during the first three years.

(Click here to download BBC Hindi's Android mobile app. You can also visit our Facebook page to read the news and share your views, and follow us on Twitter.)""",

    "140125_da_rday_special_parade_rd": """The parade, which will take place on Sunday afternoon from Barakhamba Road to Jantar Mantar, will include organisations working for women, people with disabilities, LGBTQ+ people, Dalits and several minority communities.

What do organisations working on such different issues want to say by coming together? BBC correspondent Divya Arya spoke to several participants to find out.

Vimal Bhai, National Alliance for People's Movements

"When I was young, I used to go and watch the Republic Day parade regularly. But for the last 10–20 years, I have seen the 'tantra' (system) in it, but not the 'gana' (people).

"You see weapons, the military and people dancing and singing, but everything appears very artificial. The parade does not show the suffering and difficulties of ordinary people.

"A young woman in Manipur has been on a hunger strike for 11–12 years, and the government keeps putting her in jail. Yet, in the Republic Day parade, the Manipur tableau shows government schemes and people dancing and singing. The real Manipur is nowhere to be seen.

"People are being displaced across the country in the name of development projects. Tribal communities are being pushed to the margins. In such circumstances, it does not seem right for us to simply celebrate.

"That is why we are organising our own parade in contrast to the government's grand display. The government wants to show, through dancing and singing, that everything is fine. We want to bring people's difficulties and struggles into the open.

"These continuing struggles on Constitution Day show that the Constitution is not being properly implemented."

Deepti Sharma, member of Saheli

"When the Supreme Court gave its ruling on Section 377, which criminalised same-sex relationships, it said that the gay community was a minority.

"This protest is also a response to that judgment. It shows that when several small minority communities come together, they can form a significant minority.

"There is no clear separation between these identities. A person who is gay can also be a person with a disability. Similarly, being Dalit does not mean that a person has no connection with other issues.

"There is often a big difference between our social identities and the identities we choose for ourselves. All these issues matter to us.

"Many groups were left out of several laws passed last year. In many cases, the law has pushed already marginalised people even further to the margins.

"After decades of campaigning by feminist activists, changes were made to the anti-rape law. But it still has so many shortcomings that it needs to be rewritten completely. In its present form, the law does not even ensure police accountability."

Leslie Esteves, LGBTQ+ activist

"The Constitution was drafted 64 years ago. Can the Government of India claim that all citizens have received the rights guaranteed by it?

"Certainly not. As a lesbian and an activist, I feel that Section 377 takes away many of the rights that the Constitution guarantees to every citizen.

"People in the LGBTQ+ community still do not have the freedom to live their lives as they wish, and they are denied basic human rights.

"Republic Day celebrates the creation of a good Constitution. But should we celebrate when the promises made in the Constitution have still not been fulfilled for so many Indians?

"Independence Day commemorates the freedom fighters and celebrates an achievement that has already been won. Shouldn't the Government of India use 26 January to plan how every person can receive their constitutional rights, rather than simply celebrating?

"How can people who live on the margins be brought into the mainstream?"

Anita Ghai, disability rights activist

"I have always been a feminist, but being a person with a disability is also an important part of my identity.

"It has taken a great deal of struggle to make space within the women's movement for issues related to this identity.

"I realised that although we can work individually or through organisations dedicated to disability rights, it is also very important to come out and connect with other human rights activists.

"If we do not come out and speak up, we end up accepting the existing situation and patriarchy, which prevent minorities from moving forward and expressing themselves.

"When we talk about the Constitution, many personal experiences come to mind that show how the constitutional rights of people with disabilities are violated. It seems as though the government does not even notice people with disabilities or their special needs.

"For example, when I went to vote on 4 December, there was no ramp at the polling station that would allow me to reach the voting area in my wheelchair. So I began protesting.

"A television news team that had come there to meet Ajay Maken also started covering my protest, and the issue gained attention.

"Eventually, my wheelchair was lifted and I was able to vote. But many of my friends with disabilities, who are women, could not vote because accessibility is not properly considered at our polling stations.

"One reason I came forward to protest was that people often fail to notice or understand this problem."

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)"""
})

print("Batch 13 corrections ready:", len(batch13_corrections))
print("Accepted without correction:", len(batch13_accepted_ids))

Batch 13 corrections ready: 7
Accepted without correction: 0


In [24]:
corrections_df = pd.DataFrame(
    list(batch13_corrections.items()),
    columns=["id", "complete_english_translation"]
)

if batch13_accepted_ids:
    reviewed_translations.loc[
        reviewed_translations["id"].isin(batch13_accepted_ids),
        "manual_review_status"
    ] = "Human reviewed - no correction"

for _, row in corrections_df.iterrows():
    reviewed_translations.loc[
        reviewed_translations["id"] == row["id"],
        "complete_english_translation"
    ] = row["complete_english_translation"]

reviewed_translations.loc[
    reviewed_translations["id"].isin(batch13_corrections.keys()),
    "manual_review_status"
] = "Manually corrected"

output_path = (
    f"{base_path}/09_sentence_level_translation/"
    "FactIndic_English_250_Reviewed.csv"
)
reviewed_translations.to_csv(output_path, index=False)

print("Batch 13 saved successfully.")
print("Corrections applied:", len(batch13_corrections))
print("Accepted without correction:", len(batch13_accepted_ids))
print("\nUpdated review status:")
print(reviewed_translations["manual_review_status"].value_counts())
print("\nSaved to:", output_path)

Batch 13 saved successfully.
Corrections applied: 7
Accepted without correction: 0

Updated review status:
manual_review_status
Manually corrected                123
Pending review                    102
Human reviewed - no correction     25
Name: count, dtype: int64

Saved to: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv


In [25]:
pending_ids = reviewed_translations.loc[
    reviewed_translations["manual_review_status"] == "Pending review",
    "id"
].head(10).tolist()

source_lookup = df_250.set_index("id")
translation_lookup = reviewed_translations.set_index("id")

print("Batch 14 — Articles for Human Review")
print("Number of articles:", len(pending_ids))

for i, article_id in enumerate(pending_ids, 1):
    print("\n" + "=" * 100)
    print(f"ARTICLE {i}/10: {article_id}")

    print("\nHINDI ORIGINAL:\n")
    print(source_lookup.loc[article_id, "text"])

    print("\nCURRENT ENGLISH TRANSLATION:\n")
    print(translation_lookup.loc[
        article_id, "complete_english_translation"
    ])

Batch 14 — Articles for Human Review
Number of articles: 10

ARTICLE 1/10: 131119_chinese_supercomputer_worldbest_rns

HINDI ORIGINAL:

सुपरकंप्यूटर तियानहे-2 को इस साल जून में भी दुनिया का सबसे शक्तिशाली कंप्यूटर चुना गया था. लिनपैक नामक मानक टेस्ट के अनुसार तियानहे-2 33.86 पेटाफ़्लॉप प्रति सेकेंड यानी 3,38,630 खरब प्रति सेकेंड की दर से गणना कर सकता है. तियानहे-2 पहली बार इस वर्ष जून में दुनिया का सबसे तेज़ कंप्यूटर सिस्टम बना था. जून में आई सूची में मात्र एक बदलाव हुआ है. स्विट्ज़रलैंड के कंप्यूटर पिज़ डेंट ने इस सूची में छठा स्थान प्राप्त किया है. पिज़ डेंट 6.27 पेटाफ़्लॉप प्रति सेकेंड की दर से गणना कर सकता है. जर्मनी के मैनहिम विश्वविद्यालय के प्रोफ़ेसर के नेतृत्व में साल में दो बार दुनिया के 500 सबसे तेज़ कंप्यूटरों की सूची बनाई जाती है. इस सूची में दूसरे स्थान पर अमरीका का सुपरकंप्यूटर टाइटन है लेकिन तियानहे-2 के अंक टाइटन को मिले अंक से करीब दोगुने हैं. टाइटन अमरीकी ऊर्जा विभाग का सुपरकंप्यूटर है और टेनिसी राज्य के ओक रिज़ नेशनल लेब्रोटरी में रखा हुआ है. गति का मानक सूची बनाने क

In [26]:
batch14_corrections = {
    "131119_chinese_supercomputer_worldbest_rns": """The supercomputer Tianhe-2 was named the world's most powerful computer in June this year. According to the LINPACK benchmark test, Tianhe-2 can perform calculations at 33.86 petaflops per second.

Tianhe-2 first became the world's fastest computer system in June this year. There has been only one change in the list released since then. Switzerland's Piz Daint computer has secured sixth place, with a computing speed of 6.27 petaflops per second.

A list of the world's 500 fastest computers is compiled twice a year under the leadership of a professor at the University of Mannheim in Germany.

The American supercomputer Titan is in second place on the list, but Tianhe-2's score is almost twice that of Titan. Titan belongs to the US Department of Energy and is housed at Oak Ridge National Laboratory in Tennessee.

Computer speed is measured by solving a particular type of linear equation. This test measures computational speed but does not assess other factors, such as data transfer, which can affect a computer's performance in practical applications.

IBM, which manufactures five of the computers ranked among the top ten, told the BBC that the method used to compile the list now needs to change. The company said it would raise the issue at a conference taking place that week in Denver, Colorado.

Dr Alessandro Curioni, head of the Computational Science Department at IBM's Zurich Research Laboratory, told the BBC: "For decades, the Top 500 list was a useful way of measuring a computer's ability to solve a particular problem and assessing the development of high-performance computing. But now we need a more practical approach that measures these supercomputers according to how well they perform their most important tasks in real-world applications."

Eric Strohmaier says, "A simple benchmark such as LINPACK cannot tell us how an application will actually perform on today's complex computer systems."

In Chinese, Tianhe-2 means Milky Way-2. It was developed by China's National University of Defense Technology and is located in Guangdong province in southeastern China.

The system uses several types of hybrid processors made by Intel. It also includes a special type of CPU (Central Processing Unit) designed by the university.

However, Professor Alan Woodward of the Department of Computing at the University of Surrey says it is not yet possible to say whether China's supercomputer performs better than American computers in practical applications.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "131126_assam_protes_vs": """According to news agency PTI, people in the region are extremely angry following the gang rape. They are demanding strict punishment for those responsible.

The incident took place last Friday when the woman was travelling in a shared auto-rickshaw to pick up her six-year-old daughter from school in Boginadi. According to police sources, the location is 14 kilometres from the Lakhimpur district headquarters.

Anger among local people

Police sources said that four men allegedly raped the woman, gouged out her eyes and injured her head and neck. Local people saw the victim being thrown out of the auto-rickshaw. The incident took place on the national highway, about 500 metres from the Boginadi police outpost.

After receiving information about the incident, the police admitted the victim to a local hospital. She was later taken to Guwahati Medical College, where she died during treatment.

Women's organisations and local residents blocked National Highway 52 by placing the woman's body on the road. They condemned the incident, demanded justice for the victim and called for the arrest of those responsible.

The district administration assured the protesters that the accused would be arrested. Following this assurance, the protesters lifted the blockade on the highway.

Meanwhile, police have detained two people and are questioning them in connection with the incident.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "131201_karan_with_salman_sk": """But now everything has changed. Well-known filmmaker Karan Johar has decided to invite Salman Khan for the opening episode of the third season of his television chat show, 'Koffee with Karan'. This will be Salman's first appearance on the show.

Karan Johar wrote on his official Twitter account (@karanjohar): "'Koffee with Karan' starts on @StarWorldIndia on Sunday at 9 pm. @BeingSalmanKhan will be our very first guest. There will also be a special conversation with his father, Salim Saab."

According to PTI, this season of Karan Johar's show 'Koffee with Karan' will be very entertaining. It will feature conversations with well-known Bollywood personalities.

Karan said, "This time, I am going to make things even crazier. The new season is going to be very exciting, with new guests, a wonderful set and great conversations."

Salman Khan will be the first guest on the opening episode of 'Koffee with Karan'. There have often been reports in Bollywood about differences between Salman Khan and Shah Rukh Khan. Karan Johar has traditionally been considered a close friend of Shah Rukh Khan.

But things are different this time. Salman is appearing on this popular show for the first time. By inviting Salman, who has long been seen as Shah Rukh Khan's rival, Karan has surprised everyone.

A gathering of stars

In the second episode, Kareena Kapoor will appear with her brother Ranbir Kapoor. Other guests will include Bollywood stars such as Deepika Padukone, Priyanka Chopra, Madhuri Dixit and Juhi Chawla.

You may recall that 'Koffee with Karan' first began airing in 2004. The latest season of the show is scheduled to begin airing on television from 1 December.

(Click here to download BBC Hindi's Android app. You can also visit our Facebook page to read the news and share your views, and follow us on Twitter.)"""
}

batch14_accepted_ids = []

In [27]:
batch14_corrections.update({
    "140206_egypt_al-sisi_presidential_election_sr": """Army spokesman Colonel Ahmed Ali said that the report in Kuwait's Al-Siyasah newspaper was incorrect and had misrepresented his statements.

Kuwait's Al-Siyasah newspaper had quoted Al-Sisi as saying, "I will contest the election to fulfil the aspirations of the Egyptian people." The newspaper also quoted him as saying, "I will not reject the people's demand."

Last month, the army's top leadership supported his candidacy by promoting him to the rank of field marshal and backing his decision to contest the election.

Presidential elections in Egypt are due to take place within the next six months. According to correspondents, if Field Marshal Sisi does become a presidential candidate, he is highly likely to win by a large majority.

Sisi's popularity has increased considerably since the army removed Islamist President Mohamed Morsi from office following widespread protests in July last year.

Mohamed Morsi, Egypt's first democratically elected president, is currently in prison and facing four separate charges. Morsi and his political party, the Muslim Brotherhood, have accused the military of carrying out a coup.

In response, the military-backed interim government has declared the Muslim Brotherhood a terrorist organisation and arrested thousands of its supporters.

Last month, Egypt's Supreme Council of the Armed Forces allowed Field Marshal Al-Sisi to contest the presidential election. The council said, "The people's confidence in Sisi is a call of the moment that should be answered through a free election."

According to the state-owned newspaper Al-Ahram, Chief of Staff General Sedki Sobhi has been chosen to replace Al-Sisi as military chief.

Al-Sisi's supporters say he is a strong leader who can guide Egypt towards stability after years of unrest. However, his opponents say he is a hardline military ruler who is taking Egypt back towards its repressive past.

(Click here to download BBC Hindi's Android mobile app. You can also visit our Facebook page to read the news and share your views, and follow us on Twitter.)""",

    "140218_cricket_test_team_dhoni_vr": """India lost the two-Test series 1-0 after previously losing the five-match ODI series 4-0. Overall, India failed to win a single match on its tour of New Zealand.

(Will McCullum go down in history?)

India had performed similarly on its earlier tour of South Africa, where it lost the two-Test series 0-1 and the three-match ODI series 0-2.

A series of defeats overseas has raised several questions about India captain Mahendra Singh Dhoni and his team. Will India continue to struggle on foreign pitches? Will victories continue to slip away in the same manner? Will Dhoni's favourite players continue to remain in the team? And, most importantly, will these defeats simply be forgotten during the upcoming Asia Cup and Twenty20 World Cup?

A sense of disappointment

Former India selector and cricketer Mohinder Amarnath says, "Everyone expected India to perform well in New Zealand, but its performance was extremely disappointing. The biggest reason was that the Indian players were unable to adapt their game. The failure of the Indian spinners there is also a cause for concern."

(No expectations from the BCCI)

Former India fast bowler Atul Wassan says, "There is a sense of disappointment because India could have won this series. At the very least, India could have levelled the series by winning the second Test. Just as in South Africa, India again allowed a match it was in a winning position to end in a draw, and the team seemed satisfied with that. The defeat in the ODI series has also extended India's run of overseas losses."

Wicket-taking bowlers

Speaking about India's bowling, Mohinder Amarnath says, "Indian spinners who succeed on slow and dusty pitches at home cannot necessarily succeed overseas. Only three batsmen performed reasonably well. Virat Kohli is undoubtedly a brilliant batsman, and Ajinkya Rahane played well. Shikhar Dhawan still needs experience of playing on foreign pitches, particularly against fast bowling."

(Meiyappan controversy erupts again)

Atul Wassan says, "Whatever the conditions overseas, a team must have a bowler who can take wickets. When one bowler takes a wicket, the other does not support him. When the other takes a wicket, the first does not help him. As a result, the team cannot build pressure from both ends.

"Playing with three fast bowlers whose places in the team are not even secure, and turning Ravindra Jadeja into a spinner mainly to strengthen the batting, is difficult to understand. You cannot win Test matches with three-and-a-half or four bowlers. Dhoni's defensive approach was clearly visible."

India's future

Atul says, "The time has come for Dhoni to choose between captaining the ODI team and the Test team. The selectors must also be held accountable. If Dhoni's favourite players continue to remain in the team without delivering any special performances, he will keep selecting them."

(India, England and Australia dominate)

Mohinder Amarnath agrees with Atul Wassan and says, "Now another player should be given an opportunity to lead the team."

Both Mohinder Amarnath and Atul Wassan believe that New Zealand captain Brendon McCullum set an excellent example of leadership. They say India should not forget these defeats while preparing for the Asia Cup and the Twenty20 World Cup. These results cannot be ignored when considering India's future, especially since the next World Cup will be held in Australia and New Zealand.

It remains to be seen what lessons India learns from these defeats, or whether things will remain unchanged as before.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "140410_action_against_manvendra_singh_rns": """Jaswant Singh has already been expelled from the BJP for contesting the election against the party's official candidate.

On Thursday, BJP general secretary JP Nadda informed Rajasthan BJP chief Ashok Parnami about the decision in a letter.

Manvendra Singh has also been expelled from the party for campaigning against the BJP candidate from Barmer, Sonaram Chaudhary. Manvendra Singh is a Rajasthan MLA. He was accused of campaigning in support of his father, Jaswant Singh, who is contesting the Barmer seat as an independent candidate.

Has the BJP become a shadow of the Congress? — Jaswant

According to PTI, Manvendra Singh has not been attending party meetings organised in support of the BJP candidate.

The BJP expelled Jaswant Singh from the party after he filed his nomination as an independent candidate against the party's official nominee. The BJP has given its ticket to former Congress leader Sonaram Chaudhary.

Objection from the state unit

The Rajasthan unit of the BJP had objected to Manvendra Singh's activities. State BJP chief Ashok Parnami wrote to the party's central leadership, asking it to take action against Manvendra.

Will the lotus bloom in Barmer after Jaswant's rebellion?

According to the BJP constitution, a state unit cannot expel a party member who belongs to the national executive.

At the request of the state unit chief, BJP national president Rajnath Singh decided to remove Manvendra from the national executive. Nadda said that following this decision, the state unit could now take action against Manvendra.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)"""
})

print("Batch 14 corrections ready:", len(batch14_corrections))

Batch 14 corrections ready: 6


In [28]:
batch14_corrections.update({
    "140412_rahul_gandhi_says_sn": """Rahul Gandhi clarified that he had only meant that Narendra Modi had not mentioned his wife in his earlier election affidavit. He said he had not made any personal remarks about Modi.

Rahul was speaking to reporters after filing his nomination papers for the Lok Sabha election in Amethi. He was responding to questions about his remarks on Modi's marriage during a rally in Jammu and Kashmir on Friday.

Priyanka Gandhi defended her brother, saying that he had made no personal remarks. She said Rahul had only pointed out that certain details had not been correctly mentioned in Modi's earlier affidavit. She added that the BJP was not in a position to complain about personal remarks.

Rahul, who was accompanied by his mother Sonia Gandhi and sister Priyanka, dismissed opinion polls predicting a Congress defeat as nonsense. He said similar predictions had been made in the 2004 and 2009 elections, but they had proved wrong. "They always say Congress will lose, but you know what happened," he said.

He added that attempts to make one-sided claims about a BJP victory were made every year, while the Congress believed in taking everyone along.

Rahul expressed confidence that he would win from Amethi. His opponents in the constituency include BJP candidate Smriti Irani and AAP candidate Kumar Vishwas.

Rahul also spoke about three development priorities for Amethi: working with women's organisations, connecting the constituency to national highways and railway networks, and developing food parks.

Rahul returned to Amethi with his sister Priyanka, brother-in-law Robert Vadra and mother Sonia Gandhi after holding a roadshow in Sultanpur. Varun Gandhi, Rahul's cousin, is the BJP candidate from Sultanpur.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "140423_varanasi_somnath_attack_ap": """Television footage showed that some of the attackers were wearing saffron caps bearing the lotus symbol.

The Aam Aadmi Party (AAP) blamed BJP supporters for the incident in a series of tweets and asked whether the Election Commission would take action.

The incident took place on Wednesday evening at Assi Ghat in Varanasi. AAP leader Arvind Kejriwal had filed his nomination papers for the Lok Sabha election from Varanasi earlier that day.

The footage showed AAP leader Somnath Bharti being surrounded and manhandled by a group of people, while others tried to protect him. The attackers also targeted Bharti's car, broke its windows and assaulted the driver.

Kejriwal is contesting against BJP prime ministerial candidate Narendra Modi. The Congress has fielded Ajay Rai from the constituency. Modi is scheduled to file his nomination papers on Thursday.

The incident has raised concerns about growing tensions in Varanasi. There have already been several incidents of violence involving Kejriwal and other AAP leaders during the election campaign, both in Varanasi and elsewhere.

Somnath Bharti served as law minister in the AAP government in Delhi. He had earlier been involved in controversies over his attempts to stop alleged prostitution and drug-related activities in the Khirki Extension area of Delhi.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "140524_iran_hanging_aa": """An Iranian businessman convicted in a major bank fraud case has been hanged at Evin Prison in Tehran.

Mahafarid Amir Khosravi was executed on Saturday. He was convicted in 2011 in connection with a major banking fraud scandal.

He was accused of using forged documents to obtain loans from several banks. The fraud was estimated to be worth $2.6 billion.

The case came to light in September 2011, when an investment company was accused of obtaining loans from several banks over a period of four years using forged documents.

The money was reportedly used to purchase state-owned companies as part of the government's privatisation programme.

The government at the time had to deny allegations that members of the administration were involved in the scandal.

Human rights organisations have strongly criticised the use of the death penalty in Iran. They say that Iran has one of the harshest records of executions in the world.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)"""
})

print("Batch 14 corrections ready:", len(batch14_corrections))

Batch 14 corrections ready: 9


In [29]:
batch14_corrections.update({
    "140702_wc2014_goldenboot_contenders_ar": """James Rodríguez of Colombia is currently leading the race for the Golden Boot with five goals. Thomas Müller of Germany, Lionel Messi of Argentina and Neymar of Brazil have scored four goals each.

Four players have scored three goals each: Enner Valencia of Ecuador, Karim Benzema of France, and the Netherlands' Robin van Persie and Arjen Robben. Xherdan Shaqiri of Switzerland has also scored three goals.

The contenders

Football analyst Novy Kapadia says that, with one or two exceptions, the Golden Boot is more likely to go to a player from a team that reaches at least the semi-finals. Such teams get to play at least seven matches in the tournament.

According to FIFA's tie-breaking rule, if two or more players finish with the same number of goals, the award goes to the player with the most assists.

Novy Kapadia believes Thomas Müller and Lionel Messi are strong contenders because both score goals and create opportunities for their teammates.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)"""
})

print("Batch 14 corrections ready:", len(batch14_corrections))

Batch 14 corrections ready: 10


In [30]:
corrections_df = pd.DataFrame(
    list(batch14_corrections.items()),
    columns=["id", "complete_english_translation"]
)

if batch14_accepted_ids:
    reviewed_translations.loc[
        reviewed_translations["id"].isin(batch14_accepted_ids),
        "manual_review_status"
    ] = "Human reviewed - no correction"

for _, row in corrections_df.iterrows():
    reviewed_translations.loc[
        reviewed_translations["id"] == row["id"],
        "complete_english_translation"
    ] = row["complete_english_translation"]

reviewed_translations.loc[
    reviewed_translations["id"].isin(batch14_corrections.keys()),
    "manual_review_status"
] = "Manually corrected"

output_path = (
    f"{base_path}/09_sentence_level_translation/"
    "FactIndic_English_250_Reviewed.csv"
)

reviewed_translations.to_csv(output_path, index=False)

print("Batch 14 saved successfully.")
print("Corrections applied:", len(batch14_corrections))
print("Accepted without correction:", len(batch14_accepted_ids))
print("\nUpdated review status:")
print(reviewed_translations["manual_review_status"].value_counts())
print("\nSaved to:", output_path)

Batch 14 saved successfully.
Corrections applied: 10
Accepted without correction: 0

Updated review status:
manual_review_status
Manually corrected                133
Pending review                     92
Human reviewed - no correction     25
Name: count, dtype: int64

Saved to: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv


In [31]:
pending_ids = reviewed_translations.loc[
    reviewed_translations["manual_review_status"] == "Pending review",
    "id"
].head(10).tolist()

source_lookup = df_250.set_index("id")
translation_lookup = reviewed_translations.set_index("id")

print("Batch 15 — Articles for Human Review")
print("Number of articles:", len(pending_ids))

for i, article_id in enumerate(pending_ids, 1):
    print("\n" + "=" * 100)
    print(f"ARTICLE {i}/10: {article_id}")

    print("\nHINDI ORIGINAL:\n")
    print(source_lookup.loc[article_id, "text"])

    print("\nCURRENT ENGLISH TRANSLATION:\n")
    print(translation_lookup.loc[
        article_id, "complete_english_translation"
    ])

Batch 15 — Articles for Human Review
Number of articles: 10

ARTICLE 1/10: 140729_moeen_wristband_icc-du

HINDI ORIGINAL:

मोईन अली ने 'फ्री फ़लस्तीन', 'सेव ग़ज़ा' लिखा रिस्टबैंड पहना था. विवादास्पद रिस्टबैंड पहनने पर आईसीसी मोईन के ख़िलाफ़ अनुशासनात्मक कार्रवाई कर सकता है. इंग्लैंड और वेल्स क्रिकेट बोर्ड के अनुसार 'मोईन ने कुछ भी ग़लत नहीं किया है'. आईसीसी नियम 19 एफ़: खिलाड़ी और अधिकारी को आर्म बैंड्स या कपड़ों से या दूसरे खेल साज़ो-सामान से संदेश देने की अनुमति नहीं होनी चाहिए, जब तक कि इससे लिए संबंधित बोर्ड से अग्रिम मंज़ूरी नहीं ली गई हो. राजनीतिक, धार्मिक या नस्लीय गतिविधियों से जुड़े संदेशों को मंज़ूरी नहीं दी जानी चाहिए. आईसीसी के एक प्रवक्ता ने जांच की पुष्टि करते हुए कहा, "हम मामले की जांच कर रहे हैं और जल्द ही रिपोर्ट देंगे." (बीबीसी हिन्दी के एंड्रॉएड ऐप के लिए यहां क्लिक करें. आप हमें फ़ेसबुक और ट्विटर पर भी फ़ॉलो कर सकते हैं.)

CURRENT ENGLISH TRANSLATION:

Moeen Ali wore a wristband that read "Free Palestine," "Save Gaza." The ICC could take disciplinary action against 

In [32]:
batch15_corrections = {
    "140729_moeen_wristband_icc-du": """Moeen Ali wore a wristband bearing the messages "Free Palestine" and "Save Gaza". The ICC could take disciplinary action against him for wearing the controversial wristband.

The England and Wales Cricket Board said, "Moeen has done nothing wrong."

Under ICC Rule 19F, players and officials are not allowed to display messages through armbands, clothing or other sporting equipment unless they have received prior approval from the relevant board. Messages relating to political, religious or racial activities cannot be approved.

An ICC spokesperson confirmed that the matter was being investigated, saying, "We are investigating the matter and will report back soon."

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "140825_bihar_nitish_laloo_siddharth_analysis_an": """However, senior journalist Siddharth Varadarajan believes that these results could force the BJP to return to a politics of polarisation. It is also unlikely that a similar anti-BJP grand alliance will be formed in Uttar Pradesh.

Read the full analysis

The initial reason behind the grand alliance between Nitish Kumar and Lalu Yadav was that the BJP-led NDA had won 31 of Bihar's 40 Lok Sabha seats despite receiving fewer votes than the combined vote share of the JD(U), RJD and Congress.

Now, with the NDA winning four seats and the grand alliance winning six in the by-elections, it appears that the Nitish-Lalu strategy has worked, at least on the surface. The BJP may also be relieved that it managed to win four seats.

As in Bihar, Mayawati has ruled out the possibility of an alliance with Mulayam Singh in Uttar Pradesh.

*The figures have been compiled by political researcher Abhay Kumar.*

However, four points need to be kept in mind while analysing what these results mean for India and Bihar.

Modi's influence was not as strong in the by-elections

The limited impact of Modi in these by-elections means that, at least in Bihar, by-elections are not a reliable way to judge the direction of the political mood.

What about polarisation?

It is certainly worth asking how these results will affect the BJP and its opponents at the national level. If the grand alliance had won these elections decisively, other opponents of the BJP might have considered forming similar alliances. That possibility now appears less likely.

Lalu and Nitish have come together on one platform after putting decades of hostility behind them.

The biggest impact of these election results will be on the BJP. It will realise that Modi and his message of development did not have the same impact at the state level as they did in the Lok Sabha elections.

The BJP won Narkatiaganj and Banka, where the grand alliance had fielded Muslim candidates. This suggests that the BJP received Hindu votes in these constituencies, cutting across caste divisions.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "140906_urdu_press_review_vr": """The reason was the cancellation of the Chinese president's visit to Pakistan because of security concerns arising from the ongoing crisis in the country.

The newspaper Topcat2 has criticised both Imran Khan and Tahirul Qadri. Its editorial is titled, "Are These Pakistan's Friends or Enemies?" The newspaper writes that the sit-ins and stubbornness of Imran Khan and Tahirul Qadri have damaged Pakistan's economy and harmed the country's reputation.

Urdu newspaper review

The newspaper Waqt writes that agreements worth $32 billion were expected to be signed between the two countries during the Chinese president's visit to Pakistan. These agreements have now been put on hold.

Ausaf has described the cancellation of the Chinese president's visit as unfortunate in view of the country's security situation. The newspaper says that India's enemies and Pakistan's opponents are celebrating the news. According to the newspaper, Indian analysts are asking how those who cannot protect government buildings could be expected to protect the Chinese president.

The newspaper also asks what would have been wrong with Imran Khan and Tahirul Qadri moving their sit-in away from Islamabad or resolving the issue before the Chinese president's visit.

Jang writes that China is a capable and reliable advocate of Pakistan's interests, not only in the region but across the world. Therefore, the cancellation of the Chinese president's visit is a matter of concern for the country, the government, the opposition and the protesters alike.

Stormy rain

Aaj Kal has published a cartoon about Pakistan's political struggle. In it, a lion watches two bulls fighting from a distance and says, "Either make a decision quickly, or else..."

The destruction caused by heavy rain and floods in Pakistan has also been discussed in the editorials of several newspapers.

Nawa-e-Waqt says that torrential rain has caused extensive damage in Punjab and Pakistan-administered Kashmir, and that the administration has failed to protect people from the resulting losses. The newspaper comments that rulers who do not pay attention to the problems of the people lose their right to govern.

The Peshawar-based daily Mashriq has raised the issue of Pakistan's failure to tackle polio after three new cases were reported.

Master Modi's class

In India, the Hindustan Express has published an editorial titled "Master Modi's Class" about Prime Minister Narendra Modi's address to schoolchildren on Teachers' Day.

The newspaper says that although the initiative was good, it increased the difficulties faced by teachers, students and parents. It also points out that the programme cost Rs 125 crore.

Hamara Samaj has expressed concern over the chanting of "Jai Shri Ram" during the swearing-in ceremony of Rajasthan's new Governor, Kalyan Singh. The newspaper says that the governor's responsibilities are greater than those of a chief minister at this time and that he should ensure that such an incident does not happen again.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "140908_amit_shah_political_analysis_vr": """The phones were ringing continuously. An informed source who was present there described what he had witnessed, saying that most of the calls were coming from Maharashtra. Among those making the calls were leaders from the Congress and the Nationalist Congress Party. According to the source, all of them were requesting to join the BJP.

Zubair Ahmed's analysis

The BJP's headquarters in Delhi has been buzzing with political activity these days. Rajnath Singh, Lal Krishna Advani, Sushma Swaraj and other familiar faces who had been seen there for years, until just a few months ago, are now largely absent.

The party's new era is being led by 51-year-old Amit Shah. If the Prime Minister is the country's most important leader, Amit Shah is perhaps the second most important.

A historic victory

Ajay Ummat is a well-known journalist from Gujarat who knows Amit Shah closely. He views Shah's election as party president in this way: "Modi ji has appointed Amit Shah as party president so that he can have complete control over the party, just as he has gained complete control over the government."

An acquaintance of mine sitting in the party office said that a list had been placed before Amit Shah containing the complete schedule of his visits to four states. Shah has just returned from visiting Kerala, Jammu, Maharashtra and West Bengal.

The BJP's historic victory in Uttar Pradesh is credited largely to Amit Shah. But he is still not satisfied. These days, he is in election campaign mode.

The commanding authority

Assembly elections are due to be held in Maharashtra, Haryana, Jammu and Kashmir, and Jharkhand. Amit Shah's mission is to make his boss Modi's strategy successful. That strategy is to eliminate the Congress from all the remaining states where it still has a presence. If a non-Congress government stands in the way, it too must be defeated in the elections.

Amit Shah, who comes from a middle-class family in Gujarat, has risen rapidly and is now the undisputed leader of his party. The Prime Minister trusts only a handful of people, and Amit Shah is at the top of that list.

Anwar Sheikh of Ahmedabad has known both men for many years. He says, "Overall, it can be said that Modi is the commanding authority and Shah is the implementing authority."

The number one pair

Shakti Singh Gohil of the Congress party in Gujarat was the leader of the opposition during Narendra Modi's time as chief minister. He has known both Modi and Amit Shah for years. He says, "They are two sides of the same coin. They are made for each other. Modi ji could not find a more intelligent and clever colleague than Shah, and Shah could not find a stronger shield than Modi."

The partnership between Amit Shah and Narendra Modi is 25 years old. At that time, both were RSS pracharaks and rising stars in the BJP. Modi recognised Amit Shah's abilities and brought him into his circle.

Ajay Ummat says, "Their relationship is complementary. Modi ji has found his Hanuman in Amit Shah. No one understands what Modi wants better than Amit Shah."

A controversial personality

Amit Shah's rise is linked to Narendra Modi's rise. In the words of Congress leader Shakti Singh Gohil, "If one of them falls tomorrow, both will fall."

Amit Shah is even more controversial than his boss. He has been accused of murder in the Sohrabuddin and Prajapati cases, which involved alleged fake police encounters in Gujarat. He was released on bail after spending three months in jail in 2010. The court also ordered him to stay out of Gujarat for two years, and he had to comply with that order.

On the Supreme Court's directions, the Sohrabuddin murder trial is being held in Mumbai rather than Gujarat. The next hearing in the case is scheduled for September 9.

Rubabuddin, the brother of Sohrabuddin, says, "Shah is an accused in a murder case, yet the entire party has been handed over to him. This has been done to protect him."

A major priority

Over the past year and a half, Amit Shah has appeared in court only once. Pressure on him to appear in court is increasing. The evidence submitted by the CBI against him suggests that the investigating agency believes it has a strong case.

Despite these legal proceedings, Amit Shah has become the second most important person in the country after the Prime Minister. His biggest priority these days is to ensure BJP victories in the four states.

It seems that after winning the title of 'Man of the Match' in the general election alongside Narendra Modi, Amit Shah now intends to become the 'Man of the Series'.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "141007_bhutan_corruption_india_vr": """It may sound surprising, but it is true that India has used several new and successful methods to tackle corruption at the grassroots level. One particularly effective method is the social audit, in which public hearings are held at village and district levels.

The level of corruption in Bhutan is nowhere near that seen in India. However, some cases have emerged that are commonly found in Indian villages, such as money collected to build a toilet being handed over to a Buddhist monastery, or the reported cost of an irrigation canal being shown as higher than the actual expenditure.

Political corruption has not yet taken root in this young democracy. However, with India's help, the Bhutanese government has taken up the task of preventing these scattered instances of corruption at the local level.

Transparency

Bhutan has long received India's assistance in areas such as road construction, electricity supply and the construction of block-level offices.

Last month, however, Bhutan sought India's help for the first time in tackling corruption, strengthening social accountability and ensuring transparency in government work.

The responsibility for moving this initiative forward was given to two Indian organisations: the Mazdoor Kisan Shakti Sangathan (MKSS) and the Society for Social Accountability and Transparency (SSAT).

A six-day training camp was organised under the leadership of these Indian organisations.

Decentralisation

Ashish Ranjan, an activist from Bihar who was part of the Indian delegation, explains that the process of decentralisation has only recently begun in Bhutan's democracy, which is just six years old. He says this was the right time to introduce social audits to control corruption and restore transparency.

For the people of Bhutan, this effort was something entirely new.

Bhanwar Meghvanshi, who participated in the programme from Rajasthan, says that people there were unaware that transparency in government work and access to information about it were their rights.

Measuring development

In Bhutan, greater importance is given to people's sense of satisfaction with life. Despite its limited resources, the country's progress is measured not by Gross Domestic Product (GDP), but by Gross Domestic Happiness (GDH).

To protect the country's Gross Domestic Happiness, the government has taken action at this early stage of its democracy, recognising corruption as a major threat.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "141013_wusatullah_amitabh_bachchan_rns": """Over the past four decades, several generations have grown up watching his films. They have grown older, and some have passed away. Yet Amitabh continues to dominate the silver screen in his distinctive style.

The magic of this superstar of Hindi cinema has been just as powerful in Pakistan as it has been in India.

Read Wusatullah Khan's blog

Do not be surprised if I say that Amitabh Bachchan and I grew up together, although I am now growing old. Let me explain.

The first film I watched in my life was Anand. It taught me what a film could be. I saw a tall, thin doctor trying to keep Anand Babu alive, and I understood the meaning of friendship. When I walked out of the cinema, I also understood why the silent tears running down one's cheeks taste so salty.

After that, old clothes, worn-out utensils and broken pieces of wood began disappearing from our house. Now tell me, how could a fourth-grade child like me manage to save Rs 10 to secretly watch Amitabh's new film Zanjeer on a VCR with three cassettes, especially in the 1970s?

Namak Halaal

Once, the VCR owner whispered to me, "I'm showing Namak Halaal this evening. Come over."

I said, "I only have five rupees in my piggy bank."

He replied, "Never mind. You can watch it on credit."

So I told my mother that I was going to a friend's house that evening to study because I had a test the next morning.

My mother looked me straight in the eyes and said, "When you come back after studying, tell me the story of Namak Halaal too."

Then she said, "You should speak honestly to me before going. I gave birth to you, not the other way around. Do you have any money?"

I said, "No."

She untied a knot in her dupatta, took out a ten-rupee note and placed it in my palm.

From that day onwards, at least I stopped lying to my mother.

Sholay

While I was watching Sholay, the police raided the place. In those days, watching films on a VCR, especially smuggled Indian films, was considered a serious offence by the police.

If Majid had not carried me on his shoulders and helped me climb over the wall that day, my father's reputation and a great deal of his money would have been at stake in getting me released.

Coolie was the last Amitabh film I watched for a long time. After that, I did not watch anyone's films for several years.

Baghban

Many years later, Baghban brought my childhood hero back to me, this time with white hair.

After that came Black, Sarkar, Sarkar Raj, Nishabd, Cheeni Kum, Paa and Bhoothnath. I went to the cinema to watch Bhoothnath with my younger son, Rafe.

While watching the film, Rafe asked, "Baba, can you disappear like Bhoothnath?"

How could I explain to him that Bhoothnath had been living inside me for the past forty-two years?

The truth is that, at certain moments in life, some people simply become favourites of the camera. Amitabh is one of them.

When he turned 72 on Saturday, I said to myself, "Happy birthday to you... Old will be your father!"

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "141016_ranveer_removes_pant_md_part1_pkp": """Ranveer Singh did not do this because it was required by the film's script. He took the action on his own, without receiving any instructions.

Ranveer was shooting a song for his upcoming film in Lavasa, Maharashtra. Parineeti Chopra and Ranveer Singh will soon be seen together in a film.

According to a report in India Today, around 200 dancers were also performing alongside him. Ranveer was wearing leather trousers for the song.

However, troubled by the intense heat, Ranveer suddenly took off his trousers in front of everyone without thinking twice.

At first, the entire film unit was stunned. Later, however, everyone began cheering and whistling.

Parineeti Chopra is also part of the film for which Ranveer was shooting the song.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "141107_bin_laden_death_dispute_us_sk": """Speaking to the American newspaper The Washington Post, 38-year-old former Navy SEAL Robert O'Neill claimed that Osama bin Laden was killed by a bullet fired by him.

However, O'Neill's claim contradicts the account given by former Navy SEAL Matt Bissonnette, who wrote a book about the operation in 2012.

In his book No Easy Day, Bissonnette described bin Laden's death differently.

Al-Qaeda leader Osama bin Laden was killed in 2011 during an operation by the US special forces unit, the Navy SEALs, at a house in Abbottabad, Pakistan.

Shot in the head

In an interview with The Washington Post, former Navy SEAL O'Neill said that he and another member of his team, whose identity has not been revealed, reached the third floor of a building inside the Abbottabad compound.

When they went upstairs, they saw Osama bin Laden looking out through a doorway.

According to O'Neill, who retired in 2012, another commando who was ahead of him and in the leading firing position shot first but missed.

O'Neill claims that he then entered the room without losing a moment and shot the al-Qaeda leader in the head.

A different account in the book

In No Easy Day, Matt Bissonnette claims that Osama bin Laden was shot by the commando who was at the front of the team.

However, in an interview with NBC News, Bissonnette did not directly comment on O'Neill's claim. He said only, "Two different people are telling the story for two different reasons. I will not say anything about what they are saying."

Avoid seeking public credit

Al-Qaeda leader Osama bin Laden was killed in a US operation in 2011.

The United States has so far remained silent on the official details of the matter. Pentagon officials have neither confirmed nor rejected O'Neill's claim.

However, senior military leaders involved in the operation wrote a letter to all Navy SEALs last week, advising them to remain silent about the details of the operation and avoid publicly claiming credit for it.

Because of the darkness inside the compound where Osama was killed, some Navy SEALs have also questioned whether it is even possible to determine whose bullet killed the al-Qaeda leader.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "141114_sterilisation_campaign_arrests_aj": """Police have arrested the owners of the pharmaceutical company M/s Mahawar Pharma Private Limited. Both have been remanded to seven days of police custody.

Chhattisgarh's Food and Drug Department has also raided several medical agencies in Raipur and seized a large quantity of medicines.

Raipur Inspector General of Police GP Singh told the BBC, "A raid was conducted at Mahawar Pharma's factory in Khamardi, Raipur. After that, Ramesh Mahawar and Sumit Mahawar were arrested. They are father and son."

There are suspicions about the medicines involved in the case, and the government has already banned six drugs.

The arrests were made in connection with the deaths of 15 women following sterilisation surgeries. The state government has ordered a judicial inquiry into the matter.

Retired District and Sessions Judge Anita Jha has been appointed to conduct the inquiry. She will submit her report within three months.

Several other women remain in hospital and are being treated for problems including blood pressure complications and vomiting.

Doctor arrested

Earlier this week, a doctor was arrested in connection with the case. He is accused of negligence during the operations.

Dr R.K. Gupta was arrested and was subsequently dismissed from service.

The operations were carried out as part of the state government's ongoing public sterilisation campaign, which aims to control the country's population.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "141207_pakistan_taliban_mehsud_repatriated_rd": """According to reports, the United States has confirmed that three Pakistanis have been handed back to their country, but it has not said whether Latif Mehsud was among them.

Pakistani officials, meanwhile, say that Latif Mehsud has been released.

All three men had been held by the United States at Bagram Airbase in Afghanistan.

Correspondents say that it is unusual for a senior Taliban leader to be handed over to his own country. The move is being seen as an attempt to improve relations between Afghanistan and Pakistan. However, the conditions under which the Taliban commanders were released have not been made public.

An attempt to improve relations

Latif Mehsud is the deputy of former Pakistani Taliban chief Hakimullah Mehsud, who was killed in a US drone strike.

The US Embassy in Islamabad issued a statement saying, "We confirm the return to their home country of three Pakistanis who were held in US custody in Afghanistan. Their release followed assurances of humane treatment during discussions between the United States and Pakistan."

The statement added, "We cannot provide information about the identities of the individuals who were detained."

Latif Mehsud was captured by NATO forces in October 2013 in eastern Afghanistan, near the Pakistani border. He was then held at Bagram Airbase.

At the time, there were unconfirmed reports that Mehsud had been captured while returning from a controversial meeting called to discuss a prisoner exchange.

There were also reports that his capture had angered then-Afghan President Hamid Karzai.

An exchange

BBC correspondent Mike Wooldridge, reporting from Kabul, says that such exchanges are now believed to be becoming possible.

Meanwhile, Afghanistan's new President Ashraf Ghani has said that he is committed to finding a peaceful solution to the conflict with the Taliban.

Taliban fighters have also been released in the past in an effort to speed up the peace process.

In February, the Pakistani government began talks with Tehreek-e-Taliban Pakistan (TTP). The hardline Islamist organisation began its conflict against the Pakistani government in 2007, in which thousands of people have been killed.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)"""
}

batch15_accepted_ids = []

print("Batch 15 corrections ready:", len(batch15_corrections))

Batch 15 corrections ready: 10


In [33]:
corrections_df = pd.DataFrame(
    list(batch15_corrections.items()),
    columns=["id", "complete_english_translation"]
)

if batch15_accepted_ids:
    reviewed_translations.loc[
        reviewed_translations["id"].isin(batch15_accepted_ids),
        "manual_review_status"
    ] = "Human reviewed - no correction"

for _, row in corrections_df.iterrows():
    reviewed_translations.loc[
        reviewed_translations["id"] == row["id"],
        "complete_english_translation"
    ] = row["complete_english_translation"]

reviewed_translations.loc[
    reviewed_translations["id"].isin(batch15_corrections.keys()),
    "manual_review_status"
] = "Manually corrected"

output_path = (
    f"{base_path}/09_sentence_level_translation/"
    "FactIndic_English_250_Reviewed.csv"
)

reviewed_translations.to_csv(output_path, index=False)

print("Batch 15 saved successfully.")
print("Corrections applied:", len(batch15_corrections))
print("Accepted without correction:", len(batch15_accepted_ids))
print("\nUpdated review status:")
print(reviewed_translations["manual_review_status"].value_counts())
print("\nSaved to:", output_path)

Batch 15 saved successfully.
Corrections applied: 10
Accepted without correction: 0

Updated review status:
manual_review_status
Manually corrected                143
Pending review                     82
Human reviewed - no correction     25
Name: count, dtype: int64

Saved to: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv


In [34]:
pending_ids = reviewed_translations.loc[
    reviewed_translations["manual_review_status"] == "Pending review",
    "id"
].head(10).tolist()

source_lookup = df_250.set_index("id")
translation_lookup = reviewed_translations.set_index("id")

print("Batch 16 — Articles for Human Review")
print("Number of articles:", len(pending_ids))

for i, article_id in enumerate(pending_ids, 1):
    print("\n" + "=" * 100)
    print(f"ARTICLE {i}/10: {article_id}")

    print("\nHINDI ORIGINAL:\n")
    print(source_lookup.loc[article_id, "text"])

    print("\nCURRENT ENGLISH TRANSLATION:\n")
    print(translation_lookup.loc[
        article_id, "complete_english_translation"
    ])

Batch 16 — Articles for Human Review
Number of articles: 10

ARTICLE 1/10: 141218_russia_putin_rouble_stable_sn

HINDI ORIGINAL:

पुतिन ने ये बात वर्ष के अंत में होने वाले संवाददाता सम्मेलन को संबोधित करते हुए कही. पुतिन ने रूबल में गिरावट के लिए 'बाहरी कारणों' को दोषी ठहराया है, लेकिन ये भी माना है कि केंद्रीय बैंक को पहले ही इस मसले पर काम करना शुरु कर देना चाहिए था. उन्होंने यह भी स्वीकार किया कि रूस अपनी अर्थव्यवस्था में विविधता लाने में विफल रहा, जिसको तेल की गिरती क़ीमतों और यूक्रेन संकट के तहत प्रतिबंधों से बड़ा झटका लगा. राष्ट्रपति पुतिन ने कहा कि देश का मुद्रा भंडार अर्थव्यवस्था को स्थिर रखने के लिए पर्याप्त है. समाप्त उनका कहना था कि अगर आर्थिक समस्या जारी रहती है तो सरकार 'सामाजिक ख़र्च और भविष्य के विकास पर खर्च कम करेगी'. साथ ही उन्होंने कहा कि इस समस्या से बाहर निकलने में शायद दो साल का समय लगेगा, लेकिन उसके बाद विकास निश्चित है. (बीबीसी हिन्दी के एंड्रॉएड ऐप के लिए यहां क्लिक करें. आप हमें फ़ेसबुक और ट्विटर पर भी फ़ॉलो कर सकते हैं.)

CURRENT ENGLISH TRANSLATION:

Putin s

In [35]:
batch16_corrections = {
    "141218_russia_putin_rouble_stable_sn": """Putin made these remarks while addressing his year-end press conference. He blamed "external factors" for the fall in the value of the rouble, but also acknowledged that the central bank should have started addressing the issue earlier. He admitted that Russia had failed to diversify its economy, which had been severely affected by falling oil prices and sanctions imposed over the Ukraine crisis.

President Putin said the country's foreign currency reserves were sufficient to stabilise the economy. He added that if the economic problems continued, the government would cut spending on social welfare and future development. He also said that it might take around two years to overcome the crisis, but that growth would follow afterwards.

(Click here for the BBC Hindi Android app. You can also follow us on Facebook and Twitter.)""",

    "141229_italy_ferry_rescue_pkp": """More than 250 people are still feared to be trapped aboard an Italian passenger ferry that caught fire northeast of the Greek island of Corfu. A total of 478 people were on board, of whom more than 200 had been rescued by airlift. Officials said that one Greek passenger had died and another person had been injured.

Rescue operations were proving extremely difficult because of rough seas and strong winds. "It is a long, dark night," said Italian Prime Minister Matteo Renzi.

The Norman Atlantic ferry was travelling from Patras in Greece to Ancona in Italy. A fire that began on the car deck gradually spread throughout the ship. Panicked passengers gathered on the deck and shouted for help. The Italian government said this was one of the most complicated rescue operations in the country's history.

A Greek government minister said that fog in the area was making the rescue operation very difficult.

The wife of a cook aboard the ship told the media, "My husband called me and said, 'We will all burn to death like rats. I cannot even breathe. May God save us.'"

Another passenger told the Greek television channel Mega, "We are all shaken. The entire ship is filled with smoke. The ship is extremely hot. Everything below us is burning. The boats that came to rescue us have gone back."

According to reports, merchant ships at sea were gathering around the ferry and trying to shield it from the waves.

(Click here for the BBC Hindi Android app. You can also follow us on Facebook and Twitter.)""",

    "141230_mahendra_singh_dhoni_retirement_vr": """With this, Mahendra Singh Dhoni's dream of leading India to a Test series victory overseas was also shattered. Dhoni had captained India in 60 Test matches. Before his captaincy came under even greater criticism, he surprised cricket fans by announcing his retirement from Test cricket.

**Timing?**

Former Indian cricketers reviewed Dhoni's unexpected decision in their own ways. Some even said that he had left the team in the middle of a series, while others felt that his timing was not right.

In an exclusive conversation with the BBC, former Indian all-rounder and coach Madan Lal said that such criticism was unfair. "Dhoni has been India's most successful captain. He has never stayed away from cricket. Whenever the team has faced defeat, he has taken responsibility himself, and he has always credited the entire team for victories."

**Criticism of the captaincy**

Madan Lal said, "Perhaps his repeated failures in Test cricket over the past several years forced him to take this decision."

Under Dhoni's captaincy, India had lost six consecutive Test series overseas. But should he not have been allowed to say goodbye to Test cricket with dignity?

Well-known cricket analyst and commentator Ayaz Memon said that there was nothing wrong with the timing. "It must have been on his mind for some time, and as soon as the outcome of the series was decided, he retired. Criticism of his captaincy was increasing," he said.

**Respect for the decision**

Memon said, "Dhoni probably thought that before the criticism turned into a tsunami, he should step away and focus on one-day cricket."

On Dhoni's decision to retire in the middle of the series without announcing it beforehand, Memon said, "It is more common in Asia to announce your retirement in advance. However, South African captain Graeme Smith also said goodbye to Test cricket in a similar way."

Before Dhoni, former Indian captain Anil Kumble had done the same. He retired from Test cricket after the third Test against Australia, played in Delhi in 2008.

Only Dhoni may be able to reveal the mystery behind why he took this decision, but his decision should be respected.

(Click here for the BBC Hindi Android app. You can also follow us on Facebook and Twitter.)"""
}

batch16_accepted_ids = []

print("Batch 16 corrections ready:", len(batch16_corrections))

Batch 16 corrections ready: 3


In [36]:
batch16_corrections.update({
    "150110_france_huge_marches_sdp": """The largest rally took place in Toulouse, where 80,000 people took part. During the march, some protesters carried banners reading, "I am against racial discrimination" and "I am Charlie."

In Paris, people placed their pens at the Place de la République, which had become a symbol of defiance following last Wednesday's attack on the offices of the French magazine Charlie Hebdo.

A Unity March was also scheduled to take place in the capital, Paris, on Sunday, with tight security arrangements in place. French President François Hollande and heads of state and government from more than 40 countries, most of them European, were expected to participate.

Meanwhile, police were searching for people who might have been accomplices of the three gunmen who were killed. France's interior minister said the country would remain on high alert over the coming weeks.

(Click here for the BBC Hindi Android app. You can also follow us on Facebook and Twitter.)""",

    "150213_msg_film_released_pm": """A large number of Ram Rahim's followers, many of whom had travelled from remote parts of Punjab, were present at the PVR ECX hall in Andheri. They wanted the film's release to be a success. Whenever Ram Rahim appeared on screen, the audience erupted into chants of "Dhan Dhan Satguru, Tera Hi Aasra." His followers had booked the entire hall in advance.

In contrast, screenings of the film had been cancelled at almost every other location because fewer than ten people were buying tickets.

**Praise and publicity**

The reason was clear: the film seemed like an advertisement for Dera Sacha Sauda. The entire film appeared to glorify Ram Rahim. Whenever Baba sang on screen, his followers sitting in the hall sang along.

A 63-year-old man who had come from Bathinda in Punjab said, "People banned the film in Punjab without even watching it, but nobody can stop us from seeing it."

Daljit Kaur, who lives in Mumbai, said, "I had been waiting for the film since 16 January. Today, I got an opportunity to meet Babaji."

**Audience missing**

Ordinary viewers were not showing much interest in the film. Nimesh, from Mumbai's Bhavan's College, said, "I neither know who he is nor do I want to know."

Jyotika, an MA student, said, "I know that Baba has made this film only to promote himself, and I would not even consider watching it."

Meanwhile, Baba was busy performing stunts on screen. His stunts, the film suggested, could put even Salman Khan, Sunny Deol and Rajinikanth to shame.

At the beginning of the film, a disclaimer stated that no attempt had been made to portray anyone as God through the film. However, the way Babaji was portrayed in every scene seemed to contradict that disclaimer.

The production company had doubled ticket prices even before the film's release, so watching it could prove expensive.

(Click here for the BBC Hindi Android app. You can also follow us on Facebook and Twitter.)""",

    "150221_mulayam_modi_saifai_du": """The ceremony in Saifai, Mulayam Singh's hometown, offered a rare sight. On the same stage, Modi was seen shaking hands and exchanging smiles with his staunch political rivals Mulayam Singh, Lalu Prasad and Akhilesh Yadav.

Tej Pratap, the grandson of Mulayam Singh's elder brother and a Member of Parliament from Mainpuri, was marrying Raj Lakshmi, the youngest daughter of Lalu Yadav.

**Modi and the socialists come together**

The grand celebration of the two political families was also attended by Uttar Pradesh Governor Ram Naik, actor Amitabh Bachchan, Sharad Yadav and Ram Gopal Yadav.

However, Ram Gopal Yadav and Sharad Yadav said that no political meaning should be attached to the coming together of Modi and the socialists on the stage.

Ram Gopal said, "It is a happy occasion. Modi and Mulayam congratulated each other, blessed Tej and posed for photographs. That's all. Political meanings should not be read into it."

The grand tilak ceremony in Saifai has been widely discussed in the media.

On Friday, Prime Minister Modi's controversial suit was also auctioned for Rs 4.31 crore. He had worn the suit during his meeting with US President Barack Obama.

(Click here for the BBC Hindi Android app. You can also follow us on Facebook and Twitter.)"""
})

print("Batch 16 corrections ready:", len(batch16_corrections))

Batch 16 corrections ready: 6


In [37]:
batch16_corrections.update({
    "150222_parliament_budget_session_pk": """The government has to answer several serious questions about politics from Parliament to the streets, the country's economic health and its administrative system.

The Railway and Union Budgets presented in July last year were continuations of the previous government's budgets. It remains to be seen whether the Finance Minister will focus on reducing the fiscal deficit or increase government spending to promote social development.

**Read the full report**

This session will mark the end of the Modi government's honeymoon period. Just before the session, the theft of important documents from government offices raised serious questions about the country's administrative and economic systems. The issue is likely to echo throughout the session.

This is also an important and lengthy session in terms of parliamentary business. It will take place in two phases and continue until 8 May. By then, outside Parliament, a possible change in the Congress party's leadership and the final week of the Modi government's first year in office will be approaching.

**The story of a new India**

The middle class is particularly interested in income tax exemptions. Will the Budget announce policies that accelerate economic growth this year? Is Indian Railways about to join the race for rapid modernisation? This session will provide answers to many such questions.

The expected recovery of the Indian economy is likely to begin this year. Economies around the world are currently moving towards a slowdown, and China's economy has also begun to decline. In this situation, the story of a new India is expected to begin this year.

**A joint session?**

The NDA government faces the challenge of turning nine ordinances into laws. In particular, failure to pass the ordinances on insurance, land acquisition and coal-block allocation as laws could create serious difficulties for the government.

A joint session of Parliament appears increasingly likely. Parliament has previously held joint sessions on three occasions to pass legislation. Political confrontation could lead to a fourth joint session for legislative business.

**The post-Planning Commission era**

The impact of abolishing the Planning Commission and establishing the NITI Aayog on the country's economic structure will be seen for the first time during this session.

The Economic Survey will also have a new look this year. The new Chief Economic Adviser, Arvind Subramanian, plans to completely change its format. The Survey will now have two sections. The first will analyse the economy and focus on the areas that need reform. The second will contain general facts, as in previous years.

**NDA in shock, opposition energised**

The BJP's prospects are not particularly good at present. This session will also mark the end of the Modi government's honeymoon period. The NDA is politically shaken by its defeat in the Delhi Assembly elections and the return of the Nitish government in Bihar, both of which came just before the session.

On the Saturday immediately before the session began, the tilak ceremony of Mulayam Singh Yadav's grandson was at the centre of national politics. Prime Minister Narendra Modi's attendance at the event was significant, as was the emerging possibility of opposition unity against Modi in Parliament.

The government is confident that it will get important bills passed. However, it does not expect the Congress party to cooperate in getting them through the Rajya Sabha or in convening a joint session of Parliament. The Samajwadi Party, Bahujan Samaj Party and some other regional leaders have not yet revealed their positions.

**Congress fighting for survival**

The Congress is weak in the Lok Sabha but strong in the Rajya Sabha. It will be interesting to see how it uses this parliamentary session to maintain its relevance. The role Congress will play in opposition unity against Modi will now become clearer.

In preparation for this, Finance Minister Arun Jaitley said on Thursday that the opposition was trying to obstruct economic reforms. He was referring to the Congress, which is theoretically in favour of economic reforms.

Party vice-president Rahul Gandhi has decided to organise a rally against the Land Acquisition Ordinance during the Budget session. Outside Parliament, the Congress may also elect Rahul Gandhi as its new president during this period.

The future direction of Congress politics and Rahul Gandhi's personal role will also become visible during this session.

(Click here for the BBC Hindi Android app. You can also follow us on Facebook and Twitter.)""",

    "150226_gujrat_swine_flu_ahmedabad_sr": """From January to 25 February this year, swine flu has claimed more than 1,000 lives and affected over 16,000 people. Gujarat has been the worst-affected state. So far, 231 people have died of swine flu there, while more than 3,700 people have tested positive.

To control the spread of the disease, authorities have imposed restrictions under Section 144 on large gatherings in three Gujarat cities: Ahmedabad, Bhavnagar and Rajkot. Several public events across Gujarat have since been cancelled.

Religious and cultural events are common in Ahmedabad, but they have temporarily been suspended because of swine flu. The city's foundation day is celebrated every year on 26 February, but the celebrations have been cancelled this year.

**Several events cancelled**

One of the cancelled events is a charity walk that has been held in the city for the past 12 years. Its organiser, Kaushal Mehta, said, "Around 5,000 people take part in it. Considering this order and the threat of swine flu, we have cancelled the walk scheduled for 1 March."

Several musical and religious events have also been cancelled following the order. However, wedding and funeral processions are exempt from the restrictions.

The order has left many people confused. Ahmedabad is known for its tea shops, which are popular gathering places for young people.

**'Swine flu vacation'**

Rajesh Patel, who often meets his friends at these places, said, "We are confused about whether the police will stop us from gathering here. However, we have been told that the restriction applies only to large gatherings."

He added, "But if the flu spreads, we will not be able to meet."

Doctors believe that the impact of the virus will decrease as temperatures rise during the summer. Meanwhile, some private schools and colleges have declared a 'swine flu vacation'.

Most people in Ahmedabad are using home remedies to protect themselves against the virus, although there is no evidence that these remedies are effective.

**Home remedies**

Bindia Shah has given more than 20 packets containing a mixture of camphor and cardamom to her relatives and friends.

"I received several WhatsApp messages about a home remedy to prevent swine flu. Even my doctor friends sent them to me, so I thought they must be reliable," she said.

Shah said she had forwarded the messages to more than 100 friends, many of whom were also using the remedy.

Many people are wearing masks while travelling to work, driving or walking.

**Shortage of medicines**

"We have sold more than 200 masks in the past 15 days," said medicine seller Harsh Panchal. A mask costs between Rs 30 and Rs 200.

He added, "Special masks that protect against this virus are not available, so we are selling ordinary masks."

Dr Hemant Patel said, "There is a severe shortage of medicines. Government and private hospitals have run out of stock. Pharmaceutical companies are creating artificial shortages of medicines to take advantage of the situation. This is an emergency. The impact of the virus will decrease as temperatures rise."

(You can follow us on social media platforms such as Facebook and Twitter. Click here for the BBC Hindi Android app.)""",

    "150520_emergency_service_ss": """Ordinary people are creating difficulties for this service by making thousands of calls every day using false names or fake phone numbers. This is causing problems for employees working in the emergency service.

The number receives around 20,000 fake calls every day. Those operating the 108 service say that when the lines are busy because of fake calls, people who genuinely need help may be unable to reach the service.

**Fake calls**

Ashutosh Shukla, head of the GVK Emergency Management and Research Centre, which operates the service, told the BBC, "We receive around 29,000 calls every day, of which only about 5,500 are from people who genuinely need help. People often make irrelevant conversation, use abusive language and sometimes even make obscene remarks to female employees."

Such calls come throughout the year, but their number increases during the summer holidays. School-going children who need help sometimes use their family members' phones to trouble the service's staff.

Reena Borkar, who works at the centre, said, "Since 108 is a free service, we receive all kinds of calls, including some very unpleasant ones. We deal with them by first listening briefly to check whether anyone genuinely needs help, and then disconnecting the call."

GVK EMRI provides this service in Madhya Pradesh in collaboration with the state government. Anyone can dial 108 in an emergency to request assistance. As soon as a call is received, an ambulance is immediately sent to the location. The aim is to reach the accident site within 15 minutes and take the injured to hospital.

(Click here for the BBC Hindi Android app. You can also follow us on Facebook and Twitter.)"""
})

print("Batch 16 corrections ready:", len(batch16_corrections))

Batch 16 corrections ready: 9


In [38]:
batch16_corrections.update({
    "150522_islamic_state_palmyra_ml": """UNESCO has included these archaeological ruins on its World Heritage List, and there are fears that Islamic State extremists may destroy them.

The United Nations says it has received reports that the Syrian army prevented residents from leaving Palmyra before Islamic State took control of the city. No UN representative is currently present in Palmyra, but the organisation says, citing reliable sources, that the Syrian army did not allow civilians to move to safer locations.

An official from the government of Syrian President Bashar al-Assad has urged the US-led Western coalition against Islamic State to take action to prevent the extremists from destroying this historic heritage.

UNESCO says that the destruction of these historic remains would be "a huge loss for humanity". However, there have been no reports so far that the archaeological structures have been damaged.

BBC correspondent Jim Muir, who is based in Beirut, says that the cultural heritage of Palmyra has the attention of the entire world. He says this is why destroying this valuable historical heritage is likely to be the extremists' first priority.

The ancient city of Palmyra holds an important place in the history of the Middle East. In ancient times, it was a major commercial centre in the region and was therefore also known as the "Venice of the Desert".

Only a small part of the ancient city has been excavated so far. A large archaeological area remains underground and could easily be excavated and looted.

If Islamic State destroys Palmyra's historical heritage, another part of the Middle East's heritage will fall victim to war.

Palmyra's artefacts are also known as the "Bride of the Desert". The city's historic buildings date back to the second century and combine Greek, Iranian and Roman architectural styles.

(Click here for the BBC Hindi Android app. You can also follow us on Facebook and Twitter.)"""
})

print("Batch 16 corrections ready:", len(batch16_corrections))

Batch 16 corrections ready: 10


In [39]:
corrections_df = pd.DataFrame(
    list(batch16_corrections.items()),
    columns=["id", "complete_english_translation"]
)

if batch16_accepted_ids:
    reviewed_translations.loc[
        reviewed_translations["id"].isin(batch16_accepted_ids),
        "manual_review_status"
    ] = "Human reviewed - no correction"

for _, row in corrections_df.iterrows():
    reviewed_translations.loc[
        reviewed_translations["id"] == row["id"],
        "complete_english_translation"
    ] = row["complete_english_translation"]

reviewed_translations.loc[
    reviewed_translations["id"].isin(batch16_corrections.keys()),
    "manual_review_status"
] = "Manually corrected"

output_path = (
    f"{base_path}/09_sentence_level_translation/"
    "FactIndic_English_250_Reviewed.csv"
)

reviewed_translations.to_csv(output_path, index=False)

print("Batch 16 saved successfully.")
print("Corrections applied:", len(batch16_corrections))
print("Accepted without correction:", len(batch16_accepted_ids))
print("\nUpdated review status:")
print(reviewed_translations["manual_review_status"].value_counts())
print("\nSaved to:", output_path)

Batch 16 saved successfully.
Corrections applied: 10
Accepted without correction: 0

Updated review status:
manual_review_status
Manually corrected                153
Pending review                     72
Human reviewed - no correction     25
Name: count, dtype: int64

Saved to: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv


In [40]:
pending_ids = reviewed_translations.loc[
    reviewed_translations["manual_review_status"] == "Pending review",
    "id"
].head(10).tolist()

source_lookup = df_250.set_index("id")
translation_lookup = reviewed_translations.set_index("id")

print("Batch 17 — Articles for Human Review")
print("Number of articles:", len(pending_ids))

for i, article_id in enumerate(pending_ids, 1):
    print("\n" + "=" * 100)
    print(f"ARTICLE {i}/10: {article_id}")

    print("\nHINDI ORIGINAL:\n")
    print(source_lookup.loc[article_id, "text"])

    print("\nCURRENT ENGLISH TRANSLATION:\n")
    print(translation_lookup.loc[
        article_id, "complete_english_translation"
    ])

Batch 17 — Articles for Human Review
Number of articles: 10

ARTICLE 1/10: 150526_shahrukh_tweet_salman_bajrangi_bhaijaan_hk

HINDI ORIGINAL:

शाहरुख़ ने अपने ट्विटर अकांउट से सलमान ख़ान की फ़िल्म 'बजरंगी भाईजान' का पोस्टर ट्वीट करते हुए लिखा, “भाई होना हीरो होने से बड़ा है, भाईजान आ रहे हैं ईद 2015 पर.” इसके बाद शाहरुख़ लिखते हैं कि आपको सलमान की फ़िल्म का पहला लुक कैसा लगा ? शाहरुख़ का ये ट्वीट साफ़ करता है कि किंग खान और दंबंग के बीच अब सब कुछ सही है और ये दोस्ती शायद एक और करण अर्जुन में तब्दील हो सकती है. शाहरुख़ और सलमान के बीच कुछ साल पहले कटरीना कैफ़ की बर्थडे पार्टी में झगड़ा हो गया था जिसके बाद दोनों में कई साल तक अनबन चलती रही. समाप्त कटरीना कैफ़ तब सलमान खान की गर्लफ्रेंड थीं. हालांकि सलमान खान की बहन अर्पिता की शादी में शाहरुख ने शिरकत की थी. (बीबीसी हिन्दी के एंड्रॉएड ऐप के लिए यहां क्लिक करें. आप हमें फ़ेसबुक और ट्विटर पर फ़ॉलो भी कर सकते हैं.)

CURRENT ENGLISH TRANSLATION:

Shah Rukh tweeted the poster of Salman Khan's Bajrangi Bhaijaan from his Twitter account and wrot

In [41]:
batch17_corrections = {
    "150526_shahrukh_tweet_salman_bajrangi_bhaijaan_hk": """Shah Rukh Khan tweeted the poster of Salman Khan's film Bajrangi Bhaijaan from his Twitter account, writing, "Being a brother is greater than being a hero. Bhaijaan is coming this Eid, 2015." Shah Rukh then asked, "How did you like the first look of Salman's film?" His tweet makes it clear that things are now fine between King Khan and Dabangg, and that their friendship could perhaps turn into another Karan Arjun. Shah Rukh and Salman had a disagreement at Katrina Kaif's birthday party a few years ago, after which the two remained on bad terms for several years. Katrina Kaif was Salman Khan's girlfriend at the time. However, Shah Rukh attended the wedding of Salman's sister, Arpita.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "150607_narendra_modi_bangladesh_visit_analysis_rns": """He described the Land Boundary Agreement as an agreement that connects hearts and also expressed happiness over Bangladesh's development. BBC correspondent Samiratmaj Mishra spoke to former Foreign Secretary Muchkund Dubey about the achievements of Narendra Modi's visit. Here are selected excerpts from the conversation.

How do you view this visit?

It was undoubtedly an important visit. Some steps were taken that will have positive, far-reaching consequences.

The two-billion-dollar loan we have announced for Bangladesh will help improve its transport and infrastructure. This will improve our access to Southeast Asia and East Asia.

As for the Land Boundary Agreement, it was a long-pending issue. Bangladesh had ratified it back in 1974. India turned a simple matter into a difficult one. In India, it was argued that the agreement involved handing over land to another country and therefore required a constitutional amendment. We created these complications and took so long to resolve the issue.

What new steps has this government taken compared with previous governments?

I do not think there has been much evidence of careful thinking aimed at bringing the relationship closer. However, this visit involved considerable symbolism. It was full of symbolic gestures. A positive atmosphere was created, but not much attention was given to the concrete steps that could have been taken to make use of that atmosphere. I hope this will receive attention over the next few years.

Twenty-two agreements related to economic relations have been signed, including an agreement for Indian businesses to set up a power project in Bangladesh. What impact will this have?

Most of the MoUs that have been signed are intended to connect institutions. There are some new projects, and they are important. If our businesses start power projects in Bangladesh as soon as possible to address its energy shortage, the people there will benefit, and we will benefit as well. I had not expected that investment from Bangladesh could also come into our country.

How do you view this visit in the context of relations among SAARC countries?

Pakistan has been ignored to some extent. I do not think this is the right strategy. India and Pakistan are the two largest countries in South Asia. No major progress can be made in South Asia without taking into account cooperation and trade between these two countries.

However, regional and sub-regional cooperation can proceed together. Alongside efforts to increase cooperation under SAARC, it would be better if Bhutan, Nepal, Bangladesh and India worked together in the infrastructure sector. This would require substantial investment. There is a major infrastructure deficit across all these regions, and India would have to bear a large share of the investment. Therefore, this is not an easy issue.

Former Indian Prime Minister Indira Gandhi is considered to have played a major role in the creation of Bangladesh, but she was not mentioned prominently during this visit. How do you view that?

I would not attach too much importance to it. Considerable emphasis was placed on what India did for Bangladesh at that time. That is more important than highlighting what an individual leader did. What Indira Gandhi did for Bangladesh is a matter of history. No one can forget it, whether her name is mentioned or not. So this is not a particularly important issue for me.

(You can click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "150612_heatwave_rentachintala_sra": """I saw some young men running with stones in their hands. From their clothes, they appeared to be local residents. I stopped the car and walked towards them. That was when I realised that the group was actually chasing a black, venomous cobra. The boys told me that such scenes were common in the area because the heat also brought serious problems with snakes, scorpions, insects and other pests.

The town, surrounded by rocky terrain stretching into the distance, is called Rentachintala and is located in Andhra Pradesh's Guntur district.

A Place of Extreme Heat

Rentachintala is known throughout the state for recording the highest temperatures. Its average annual temperature reaches 48 to 50 degrees Celsius. This year too, the temperature reached 50 degrees in May. In 2012, it touched a record 52 degrees.

Although hardly anyone in the area speaks Hindi, S.K. Subhani, a mechanic, made my work easier. Subhani says that the heat makes people very irritable. Several generations of his family have lived in Rentachintala. Speaking in a mixture of Telugu and Hindi, he told me about several distinctive features of the town that set it apart from other parts of Andhra Pradesh and Telangana.

He says, "People here get very angry. We cannot sleep at night because of the heat, sir. Of course, we get angry. Besides, we eat a lot of chillies. That also makes us angry. We use chillies to drive away the heat."

Outside the town limits, a herd of goats has come to graze. However, the animals will have to survive on the patches of grass growing among the rocks and the leaves of wild trees.

Rentachintala has a population of around 16,000. Residents said that the rising temperatures had made life difficult for them this year as well.

Keeping Their Feet Off the Ground

Although heatwaves have claimed many lives in parts of Andhra Pradesh and Telangana, the heat in this part of Andhra Pradesh is still intense, even though temperatures have fallen somewhat. I felt quite uncomfortable after travelling from Delhi to this area. The monsoon had reached the coastal parts of Kerala, but in Rentachintala, both the heat and humidity were still at their peak.

Solomon Raju, the headmaster of the town's government high school, says that local people call the oppressive heat that lasts throughout the day "Rohini Karte".

He says, "During this period, it is extremely hot from 8 am to 8 pm. After that, there is only slight relief until 8 am the next morning. But between 12 noon and 3 pm, the heat is so intense that people cannot even keep their feet on the ground inside their own homes."

Global Warming

I met several elderly people in Rentachintala who said that the heat in this area becomes so intense that people even have difficulty urinating. Older residents say they have experienced this kind of weather since childhood. However, they believe the intensity of the heat has increased over the years. They also consider global warming a major reason for this.

The town's narrow streets have their own distinctive character because almost everything in this city of stone is made of stone, whether it is houses, shops, places of worship or offices.

Stone Houses

Anji Reddy, who is posted at the Rentachintala police station, told me that stone quarries surround the town, making stone cheaper than bricks. People therefore build their houses out of stone. This has now become a common practice in the area.

Clearly, the biggest problem for the residents of this town built among rocks is the lack of any source of groundwater. People have to depend entirely on the supplied water. Vadda Pati Pareddy, a resident of the town, said that even the available water supply is inadequate and dirty. He complains that the government does not take any measures every year to provide relief to people struggling with the heat.

Vadda Pati Pareddy regularly visits the tehsildar's public grievance meeting, known as the Janata Darbar, to submit applications and request a better supply of drinking water. However, he says that no action has ever been taken on his applications.

The residents of this city of stone, Shahr-e-Sang, have now become as hardened as stone themselves after enduring the harsh weather for so long. And what of humans or animals? Even if their bodies have not turned to stone, perhaps their hearts have. Without becoming stone-hearted, it is difficult to withstand the weather.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)"""
}

batch17_accepted_ids = []

print("Batch 17 corrections ready:", len(batch17_corrections))

Batch 17 corrections ready: 3


In [42]:
batch17_corrections.update({
    "150622_bihar_nitish_kumar_narendra_modi_social_media_rns": """There are several examples in which the campaign supporting Nitish Kumar appears to imitate Narendra Modi's election campaign from the previous year. The entire campaign has been designed by Prashant Kishor, who was a key member of Narendra Modi's campaign team in 2014. Nitish Kumar has publicly confirmed that Prashant is working for him. Prashant is also attending party meetings and addressing party conferences in public.

Comic-book heroes

During last year's Lok Sabha elections, a comic book called Bal Narendra was released. It portrayed Narendra Modi's childhood bravery, hard work and honesty. Now, ahead of the Assembly elections, a comic-book series based on Nitish Kumar's life has been released. The first comic in the series is called Munna Se Nitish. It tells the story of how Munna, as he was known to his mother, became Bihar's Chief Minister, Nitish Kumar.

Pages from the comic are being posted on a Facebook community page called 'Phir Ek Baar Nitish Kumar' (Nitish Kumar Once Again). The page states, "This page is run by supporters of Nitish Kumar ji for his supporters." Its content is being widely used at various events connected with Nitish Kumar and his party.

The second comic series is called Sattu Pe Satta. It is an election-themed comedy revolving around politics.

Coffee conversations

Before becoming Prime Minister, Narendra Modi's 'Chai Pe Charcha' (Discussion over Tea) attracted media attention. He has often presented himself as a tea seller. Nitish Kumar may have thought that Modi had already claimed tea, so he chose coffee instead for his own conversations. If you want to meet Nitish Kumar over coffee, all you need to do is fill out a form.

Imitating the slogans too

During the Lok Sabha elections, the BJP came up with several slogans based on the phrase 'Abki Baar Modi Sarkar' (This time, a Modi government). Ahead of the Bihar elections, the slogan 'Phir Ek Baar Nitish Kumar' (Nitish Kumar once again) is being heard everywhere. Slogans such as 'Don't let the pace of development slow down, Nitish Kumar once again' and 'Enough of the attacks of empty promises, Nitish Kumar once again' can be seen and heard everywhere, from Facebook to the streets of Patna.

Missed-call membership

The idea of joining a political party by giving a missed call was introduced by the Aam Aadmi Party. The BJP later launched its own campaign to enrol members by phone. As a result, the BJP claimed that its campaign had made it the world's largest political party. Nitish Kumar, who wants his party to become the largest in Bihar, has given the public a phone number through which people can connect with his party and its programmes.

Original or imitation?

JD(U) national general secretary and MP KC Tyagi says, "Imitation requires intelligence too, and we have plenty of that. We will not imitate Narendra Modi's politics."

How will the Bharatiya Janata Party counter this Modi-style campaign? In response, Nand Kishore Yadav, Leader of the Opposition in the Bihar Assembly, says, "Nitish Kumar is seriously lacking the qualities needed to imitate it."

A difference in the political atmosphere

Senior journalist Surur Ahmed believes that the atmosphere during the 2014 general elections was very different from that of the Bihar Assembly elections being held this year. He says, "Narendra Modi was competing against a weakened government, whereas Nitish will be facing a strong opposition."

According to Surur, an aggressive campaign alone will not be enough. Nitish Kumar will also have to put forward some concrete political agendas.

(You can click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "150708_us_army_cuts_aa": """As part of this cost-cutting drive, 17,000 employees engaged in civilian roles within the military will also be laid off. According to a report in the US newspaper USA Today, the plan for these cuts could be announced this week. After that, the number of troops in the US Army will fall to 450,000 by 2017.

Major cuts

The number of US troops was at its highest in 2012, reaching 570,000, when the crises in Afghanistan and Iraq were at their peak. Discussions about such a major reduction in troop numbers have been taking place since early 2014. At the time, then-US Defence Secretary Chuck Hagel had proposed reducing the number of troops to 450,000.

A year before the 2001 attacks, the number of troops stood at 480,000.

Even after the withdrawal from Afghanistan, 10,000 US troops will remain there. Meanwhile, 3,500 US troops in Iraq are training Iraqi forces so that they can confront the extremist organisation Islamic State.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "150713_bihar_women_strange_names_rns": """Anushka Sharma played the role of journalist Jaggu in the film PK. Unusual names are often given to characters in films to amuse audiences, but in real life, there are many women whose names people find unusual. One of them is Fenkni Devi from Chikni village in Nepal.

Sixty-year-old Fenkni explains, "When I was born, I had some illness, so my family threw me near the river. A family from the neighbourhood took me in and raised me. When I recovered, my father brought me home. Since my mother had abandoned me at birth, I was given the name Fenkni."

There are cultural stereotypes and other social and economic reasons behind such unusual names. But these names are becoming less common now.

'We will call our daughter-in-law Sridevi'

Fenkni named her daughter Snehlata. Snehlata explains, "When I got married, my father-in-law asked, 'What kind of name is Fenkni Devi?' Since my mother was beautiful, he said, 'We will call our daughter-in-law Sridevi.' My mother gave me a very lovely name."

Sociologist Hetukar Jha comments on these changing naming patterns: "Earlier, names were chosen to match both a person's caste and social status. But now there has been a great deal of change in the names people choose. There are two reasons for this. First, people are influenced by the many names they encounter through television, films and newspapers. Second, the ties that bound people to their communities and villages are breaking down. There is now greater emphasis on individuality."

'It used to feel strange'

Phuljhari Devi lives in Sipara, near Patna. She told BBC Hindi, "At this age, it feels very strange when someone calls me Phuljhari. Our names were unusual, but we gave our children good names. My sons were called Dhuria and Kanaria in the village, but when we moved to the city, I changed their names to Shivvachan and Santosh."

The desire to change such names is clearly visible among these women and their families. Khushbu, Phuljhari's granddaughter, says, "I asked my father to change my grandmother's name. But he said that her name is listed as Phuljhari on her voter ID card and old-age pension documents. So changing it is difficult."

The aesthetic sensibilities of marginalised people

Prem Kumar Mani, a writer and political leader, says, "A sense of aesthetics is growing rapidly among marginalised people, and this is a major cultural development. They are now paying attention to everything, from their clothes and teacups to their names. Such names are no longer common in society. Even contemporary literature no longer features names like Ghisu, Nanhu and Bhimal."

However, Hetukar Jha refuses to consider the changes in names a major transformation. He says, "This is a change, but it is only superficial. It is not a deep change that has transformed people's lives or improved their standard of living. A name is simply your identity, nothing more."

Although changing names may not have improved people's standard of living, people now seem more conscious of their names. For example, Rajdhani Devi of Sitabigaha in Sasaram says, "My parents learnt at home that a capital city is a very big place, so they named me Rajdhani. Now, whenever I go to have my name entered on a card, the official laughs and says, 'Go to the capital!'"

'Mother gave me the name'

Chinta Devi says that her mother gave her the name. The same is true of Chinta Devi from Begumchak in Masaurhi. When asked her name, she first explains the story behind it and says, "My mother gave me such a name that I have spent my whole life worrying. My name is Chinta Devi, sister."

Prem Kumar Mani points out that changes are taking place not only in given names but also in caste-based surnames. He says, "Nowadays, cowherds write Yadav, while carpenters use surnames such as Vishwakarma or Sharma. The names associated with several other castes are also changing."

Asked about the reasons for these changes, Prem Kumar Mani says, "Increasing interaction with literature, the wider world and education has accelerated this transformation."

(You can click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)"""
})

print("Batch 17 corrections ready:", len(batch17_corrections))

Batch 17 corrections ready: 6


In [43]:
batch17_corrections.update({
    "150725_one_killed_kashmir_grenade_attack_pm": """A police officer told news agency PTI that a suspected militant had thrown a hand grenade at the Achabal bus stand. A security personnel member and six passers-by were injured by the grenade fragments. One of the injured, Mohammad Jabbar, was admitted to a hospital in Srinagar, where he later died.

Raids launched

According to PTI, the attack was possibly intended to target a security forces patrol, but the grenade missed its target and landed at the bus stand. The police have cordoned off the entire area and launched extensive raids to arrest those responsible. All the injured have been admitted to hospitals. One of them is in critical condition.

Grenade attacks had also taken place a day before this incident. Grenades were thrown at telephone towers in Srinagar's Karan Nagar and Shaheedganj areas. Four people were injured in that incident.

(You can click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "150802_varun_gandhi_death_ia": """Varun Gandhi wrote this article at a time when Yakub Memon, convicted in the 1993 Mumbai blasts, had been hanged and the death penalty was being widely debated in India. Earlier, another BJP MP and film actor, Shatrughan Sinha, had signed Yakub Memon's mercy petition. Reacting to this, Union Finance Minister Arun Jaitley said that Shatrughan Sinha had embarrassed the party by doing so. Shatrughan had made controversial statements in the past as well.

Jaitley said, "I believe it is very sad that a member of the BJP has signed such a petition."

Socio-economic discrimination

Varun Gandhi does not mention Yakub Memon's execution in his article. However, he writes, "It is a matter of shame for any civilised country to have an executioner..."

He emphasises that socio-economic discrimination is clearly visible among those sentenced to death. Referring to the United States, Varun Gandhi writes that 56 per cent of those sentenced to death are Black or Hispanic. According to him, in India, 94 per cent of those sentenced to death are Dalits or members of minority communities.

He argues that people are sentenced to death in India because they lack access to good lawyers and because of institutional discrimination.

Opposition to the death penalty

According to Varun Gandhi, the Supreme Court had ruled in 1983 that the death penalty should be imposed only in the rarest and most heinous cases. However, the Indian judicial system has no clear definition of what constitutes a heinous crime. As a result, the decision about who should receive the death penalty and who should not depends on judges' personal opinions and their social and political beliefs.

He says that because of this uncertainty and contradiction, some Supreme Court and High Court judges had, a few years earlier, made a written appeal to the President seeking the commutation of the death sentences of 13 people.

(You can click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "150811_sani_mirza_autobiography_farha_adp_ssm": """If the film had been made, it would have been the fourth major biopic about a sportsperson, following those on sprinter Milkha Singh, boxer Mary Kom and cricketer MS Dhoni. However, while speaking to the media at an event, Farah denied that any such film was being made.

In her statement, Farah said, "I don't know where this rumour came from. I am not making any film; it is just a rumour. But the idea is good."

Sania is the world's number-one women's doubles tennis player and has spoken about a possible biopic before. At a press conference, she had said, "If a biopic is made about me, I would want Parineeti to play my role."

(You can click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)"""
})

print("Batch 17 corrections ready:", len(batch17_corrections))

Batch 17 corrections ready: 9


In [44]:
batch17_corrections.update({
    "150816_badminton_saina_final_dp": """Saina Nehwal has become the first Indian woman to reach the final of the World Badminton Championships. She defeated Indonesia's Lindaweni Fanetri in the semi-final to secure her place in the final.

Saina won the match 21-17, 21-17. She will now face Spain's Carolina Marin in the final.

Saina had previously won a bronze medal at the World Championships in 2013. She is the first Indian woman to reach the final of this tournament.

(You can click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)"""
})

print("Batch 17 corrections ready:", len(batch17_corrections))

Batch 17 corrections ready: 10


In [45]:
corrections_df = pd.DataFrame(
    list(batch17_corrections.items()),
    columns=["id", "complete_english_translation"]
)

if batch17_accepted_ids:
    reviewed_translations.loc[
        reviewed_translations["id"].isin(batch17_accepted_ids),
        "manual_review_status"
    ] = "Human reviewed - no correction"

for _, row in corrections_df.iterrows():
    reviewed_translations.loc[
        reviewed_translations["id"] == row["id"],
        "complete_english_translation"
    ] = row["complete_english_translation"]

reviewed_translations.loc[
    reviewed_translations["id"].isin(batch17_corrections.keys()),
    "manual_review_status"
] = "Manually corrected"

output_path = (
    f"{base_path}/09_sentence_level_translation/"
    "FactIndic_English_250_Reviewed.csv"
)
reviewed_translations.to_csv(output_path, index=False)

print("Batch 17 saved successfully.")
print("Corrections applied:", len(batch17_corrections))
print("Accepted without correction:", len(batch17_accepted_ids))
print("\nUpdated review status:")
print(reviewed_translations["manual_review_status"].value_counts())
print("\nSaved to:", output_path)

Batch 17 saved successfully.
Corrections applied: 10
Accepted without correction: 0

Updated review status:
manual_review_status
Manually corrected                163
Pending review                     62
Human reviewed - no correction     25
Name: count, dtype: int64

Saved to: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv


In [46]:
pending_ids = reviewed_translations.loc[
    reviewed_translations["manual_review_status"] == "Pending review",
    "id"
].head(10).tolist()

source_lookup = df_250.set_index("id")
translation_lookup = reviewed_translations.set_index("id")

print("Batch 18 — Articles for Human Review")
print("Number of articles:", len(pending_ids))

for i, article_id in enumerate(pending_ids, 1):
    print("\n" + "=" * 100)
    print(f"ARTICLE {i}/10: {article_id}")

    print("\nHINDI ORIGINAL:\n")
    print(source_lookup.loc[article_id, "text"])

    print("\nCURRENT ENGLISH TRANSLATION:\n")
    print(translation_lookup.loc[
        article_id, "complete_english_translation"
    ])

Batch 18 — Articles for Human Review
Number of articles: 10

ARTICLE 1/10: 150825_kapil_sharma_debut_bollywood_ssm

HINDI ORIGINAL:

अब्बास मस्तान निर्देशित फ़िल्म 'किस किस को प्यार करूं' में कपिल शर्मा एक नहीं तीन तीन हीरोइनों के साथ रोमांस करते नज़र आने वाले हैं. लेकिन बीबीसी से खास बातचीत में कपिल शर्मा ने बताया कि फ़िल्म में अपने डॉयलॉग याद रखना एक मुश्किल काम था पुरानी आदत कपिल जब टीवी पर 'कॉमेडी सर्कस' में काम किया करते थे तब भी वो अपनी लाईनें हाथ पर लिख कर लाते थे. वो कहते हैं,"डॉयलॉग याद करना मेरे लिए हमेशा से परेशानी भरा रहा है और पुरानी आदतें जल्दी पीछा नहीं छोड़तीं." समाप्त हालांकि कपिल मानते हैं कि फ़िल्म में काम करते हुए डॉयलॉग भूल जाने पर उतनी समस्या नहीं होती जितनी टीवी या लाईव शो में. प्रेरणा कॉमेडियन से हीरो बनने का सफ़र मुश्किल होता है क्योंकि बॉलीवुड में कॉमिक आर्टिस्ट को हीरो के साथ खड़ा किया जाता है, हीरो नहीं बनाया जाता. ऐसे में कपिल के लिए भी यह मुश्किल था लेकिन कपिल अपनी प्रेरणा महमूद को मानते हैं,"मैं जानता हूं कि कॉमेडी करने वाले कलाकारों को यहां अलग तरह से दे

In [47]:
batch18_corrections = {
    "150825_kapil_sharma_debut_bollywood_ssm": """In Abbas-Mustan's film Kis Kisko Pyaar Karoon, Kapil Sharma will be seen romancing not one but three actresses. However, in an exclusive conversation with the BBC, Kapil said that remembering his dialogues was a difficult task.

Old habits

When Kapil used to work on the television show Comedy Circus, he would write his lines on his hand before coming on stage. He says, "Remembering dialogues has always been difficult for me, and old habits are hard to get rid of."

However, Kapil admits that forgetting dialogues while working on a film is not as much of a problem as it is on television or during a live show.

Inspiration

The journey from being a comedian to becoming a hero is difficult because, in Bollywood, comic actors are usually placed alongside heroes rather than being made heroes themselves. This made the journey difficult for Kapil too, but he considers Mehmood his inspiration. He says, "I know that comedians are viewed differently here, but I consider Mehmood my inspiration and want to become multi-talented like him."

Kapil also considers Shah Rukh Khan his role model and says he has learned a lot from him. He says, "I have learned from Shah Rukh how to make use of time. I sleep for three to four hours, and so does Shah Rukh. But he does it because of his work, while I do it out of compulsion."

'There is no bribery'

Kapil, who has moved from being a comedian to becoming a hero, believes that if you have talent, nobody can stop you. He says, "It is extremely difficult to establish yourself in this industry, but this is the only industry where bribery does not work."

Kapil says that if you have genuine talent, people will certainly like you. But if you have nothing special to offer, you will not get work, no matter how influential your father is.

If Kapil succeeds as a hero, will he also enter politics? Answering this question with a laugh, he says, "I hope I never enter politics in my life. There, you have to sling mud at others and keep everyone happy. I cannot do either of these things."

(You can click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "150901_nepal_protest_dead_pp": """Those killed belong to the Madhesi community. Police told the BBC that they were forced to fire at the protesters because the protesters had attacked them with guns and knives.

Under the proposed new constitution, Nepal will be divided into federal provinces. Madhesi and other groups in Nepal believe that they are being ignored.

According to officials, protesters vandalised government offices and police posts in Birgunj, located 90 kilometres south-west of the capital, Kathmandu. Four people were killed in police firing there. Another person was killed in the nearby town of Kalaiya.

In all, at least 20 people have been killed during protests in recent weeks.

Nepal has been governed under an interim constitution since the monarchy ended in 2008. After years of political turmoil, political parties agreed on a draft constitution this year.

Protests

Supporters of the draft say that the new constitution will promote the country's economic development and strengthen peace with Maoist rebels. However, disagreements remain over the proposed boundaries. Several minority groups say they will not be given adequate rights.

Several Hindu organisations in Nepal are also protesting and demanding that the country be declared a Hindu nation again. After the end of the monarchy, Nepal became a secular republic.

(You can click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "150930_vert_auto_audi_etron_pk": """Recently, the electric SUV made by American carmaker Tesla has also attracted considerable attention. The company unveiled its Model X at a showroom on 29 September. However, Audi had announced its own electric SUV even earlier, saying that production of the concept car would begin in 2018.

The car attracted considerable attention at the Frankfurt Auto Show, and many people believed it could become an exceptional vehicle once it was built.

Ralf-Gerhard Willner, Audi's Vice President of Vehicle Concepts, said that the car had taken less time to develop than expected. One reason was that the company had used parts that were already available, including components related to the suspension and steering.

The car was presented alongside Audi's large Q7 and Bentley's luxury Bentayga. Among these impressive vehicles, the e-tron stood out because of its new features.

Sleek and stylish

The car's features include active aerodynamic devices. Its drag coefficient—the measure of how the car's presence in the air affects its aerodynamics—is just 0.25.

The car will also feature organic light-emitting diodes (OLEDs). This technology uses an extremely thin film that emits light when an electric charge passes through it. The film can be cut into different shapes. Audi plans to use this technology for its illuminated front grille featuring the company's four rings.

The e-tron can certainly be described as sleek and stylish. Although its exterior resembles the Q5 and Q7 to some extent, its interior is quite different.

It uses a lithium-ion battery with a capacity of 95 kilowatt-hours, allowing the car to travel up to 310 miles. The battery can be recharged using either a plug-in connection or wireless induction charging.

Audi is working on a system that would allow the car to start charging automatically when parked. For this, a charging plate would need to be installed in the garage floor.

The car also has a solar panel on its roof, which can charge the battery automatically on days when there is sufficient sunlight.

A top speed of 131 mph

The e-tron quattro is also impressive in terms of speed. Three motors power its four wheels, producing a combined 503 horsepower and 590 pound-feet of torque.

The car can accelerate from zero to 62 miles per hour in just 4.6 seconds. Its top speed is 131 miles per hour.

The battery can be charged using a 480-volt rapid-charging system, which can fully charge it in 50 minutes.

The e-tron quattro will look similar to the Q6 but will operate in unconventional ways. It is an electric car, but it can also function as a hybrid through its plug-in system, allowing the driver to choose between fuel and electric power.

The price will not be too high

Speaking about the car's price, Willner says, "You will not be disappointed with the price."

Audi also has to consider that rival companies are working to introduce their own electric models. Porsche's electric Cayenne S-Hybrid could pose a major challenge. Mercedes-Benz and BMW are also developing their own electric cars.

However, at the Frankfurt Motor Show, Audi's e-tron was the main attraction.

(Read the original article in English here, available on BBC Autos. You can click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "151003_draught_karnatak_series_part_one_sr": """What is drought? Ask Hanumanthi of Hanchinal, whose frail, century-old body is still alive to experience the hunger she never imagined she would face when she was young.

Drought is also what Basu experiences when his feet get caught in his half-grown cotton plants as he walks across rocky ground amid the prickly green bushes of the kikar tree.

Drought is also when the soil in a farmer's seed-filled field, with the farmer looking up towards the sky, cracks while waiting for monsoon showers. Yet the government's sense of shame does not seem to reach it.

These are the thoughts on my mind on the first day of the Samvedna Yatra, which began in Yadgir district in the southern Indian state of Karnataka.

On the first day, I visited Hanchinal village in Yadgir district and Moranpur and Kadagamadadoddi villages in Raichur district. I do not know what I will encounter at the next stops on this journey.

Social activist Yogendra Yadav has started the Samvedna Yatra to understand the condition of villages in seven states affected by the failure of the monsoon. At the same time, the BBC is travelling alongside it, searching for signs of drought. This is the Akal Yatra.

It is customary for people who die from drought and hunger to find a place on the front pages of newspapers. Yet the mainstream media still fails to look closely at this dark corner of the country. Perhaps people sitting in air-conditioned rooms in metropolitan cities do not know what hunger and drought really mean.

But after reaching Karnataka, one thing is clear: there is a huge gap between the hunger recorded in government files and the reality of the cracked earth.

At the very beginning of the Akal Yatra, I realised that drought has not only cracked the land. It has first brought uncertainty, and this uncertainty has changed farmers' lives and the way they think.

Hanchinal

A landlord's house in Hanchinal village, in Karnataka's drought-hit Yadgir district.

The first impact has been on livestock. When there is no fodder, what can a farmer do with his animals?

In Hanchinal, the village in drought-hit Yadgir district where our journey began, I found very few animals used for farming, but plenty of goats. I learned that the fields had been ruined by unexpected rainfall for the third consecutive year, and many farmers had now become shepherds.

Many houses are locked. The village has a population of around 3,000, with approximately 1,500 registered voters. But half of its men have left for cities in Karnataka in search of work.

Some houses have no roofs, while others have no walls. In some places, an entire house has become a heap of rubble. Nature has completed its cycle here.

The Akal Yatra

Cotton, groundnut, tur and millet are cultivated in Hanchinal and Raichur. There has been no proper rainfall for three years. Farmers took loans, but their seeds failed to grow, leaving them trapped in debt. Out of desperation, they looked for daily-wage work. When even that did not provide enough to feed their families, some left the village, while some who stayed behind lost their lives.

This story is not new, but it is not so old that it can be ignored.

Alongside the Samvedna Yatra, the BBC's Akal Yatra will travel from southern to northern India until 15 October. It will bring you reports from the most drought-affected villages in 25 districts across seven states.

Through this journey, you may also get to know the 'truth about drought' faced by farmers in North Karnataka, Telangana, Maharashtra, Madhya Pradesh, Uttar Pradesh, Rajasthan and southern Haryana.

(You can click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "151122_indo_pak_cricket_meet_ml": """The meeting took place at the ICC headquarters in Dubai through the efforts of Giles Clarke, chairman of the England and Wales Cricket Board. Clarke has been trying to organise an India-Pakistan cricket series.

The meeting lasted for nearly an hour. Shaharyar Khan was accompanied by Najam Sethi, chairman of the Pakistan Cricket Board's executive committee.

After the meeting, Shaharyar Khan told the BBC that he could only say that it had gone very well. He added that Giles Clarke would provide more details to the media on Monday.

Shaharyar Khan reiterated that the final decision about the India-Pakistan cricket series would now be taken by the Pakistan government.

Earlier, a meeting between Shaharyar Khan and Shashank Manohar, which was scheduled to take place in Mumbai, had been cancelled because of protests by Shiv Sena workers.

It is worth noting that Pakistan and India have not yet reached an agreement on the series scheduled for next month.

The Pakistan Cricket Board says that since this is Pakistan's home series and India has signed an agreement to play it, the series should take place in the United Arab Emirates.

The PCB has also rejected the BCCI's proposal that the Pakistani team travel to India to play the series.

Najam Sethi, chairman of the PCB's executive committee, says that Pakistan will not send its team to India under any circumstances. He says Pakistan had sent its team to India twice before but had received no financial benefit.

India, on the other hand, is not willing to play the series in the United Arab Emirates.

(You can click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "151218_anti_pee_london_ac": """The special paint applied to these walls creates a barrier that causes any liquid that comes into contact with it to bounce back. It also does not absorb urine, which helps reduce bad smells and stains.

Hackney Council spends around £100,000 every year cleaning urine from walls.

Feryl Dermisi, a member of the neighbourhood team, says, "If the fear of receiving a fine does not stop them, perhaps the fear of getting their own urine splashed back on them will."

Several well-known bars are located in the Shoreditch and Dalston areas of Hackney. The walls of these bars will be coated with the special paint.

She said that people visiting these areas should also think about the residents who live there.

Meanwhile, Monica, the duty manager at Nutfield Health, has to clean up the mess. She says, "It is not a pleasant job. When I arrive here, I find that people have even urinated in the fire exits."

She adds, "Some people also vomit, and I have to clean that up too. The smell becomes even worse as time passes."

The material has been developed by the American company UltraTech. It creates an almost invisible layer of air above the paint.

The council believes it is the first in the UK to use this product. It has spent £1,000 to deal with the problem at these two locations. However, it has not revealed the exact places where the paint will be applied.

(You can click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "160129_women_cricket_team_india_pm": """This was India's first bilateral series victory against Australia. India defeated the three-time world champions by ten wickets in the second match, taking an unassailable 2-0 lead in the three-match series.

Because of rain, the match was shortened to ten overs per side. India were set a target of 66 runs and reached 69 in just 9.1 overs.

Captain Mithali Raj scored 37 runs and remained not out until the end.

The Indian women dominated the match from beginning to end. Within the first five overs, they had reduced Australia to 33 runs for three wickets.

Meg Lanning, captain of the Australian women's cricket team, and Jess Jonassen rescued their side with a 70-run partnership for the fourth wicket.

Australia finished with 125 runs for eight wickets in 18 overs.

Indian bowler Jhulan Goswami took two wickets for 16 runs, while Rajeshwari Gayakwad took two wickets for 27 runs.

(You can click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "160212_ranchi_mamta_brave_girl_pk": """These are the words of the parents of Mamta, a 14-year-old girl studying in Class 9. Mamta's marriage had been arranged, but she refused to marry at such a young age.

Since then, Mamta, from the remote village of Hapamuni in Gumla district of Jharkhand, has been in the news.

Mamta is determined, but she also has a concern. After refusing the marriage, she wants the loan her parents took for the wedding to be repaid.

The girl used to walk six kilometres through village streets and pathways to reach school. She has now been admitted to the government's Kasturba Girls' Residential School in Baghima, Palkot. She will receive free uniforms, books, shoes and meals there until she completes Class 12.

It was on 2 February that the groom's family arrived at Mamta's house for the wedding. At that very moment, the police also arrived and took Mamta with them.

The government has announced a reward of one lakh rupees for her decision to refuse the marriage.

Mamta says that she had raised objections when the subject of marrying at a young age came up, but she could not strongly oppose her family. She thought that her parents were facing financial difficulties and perhaps that was why they were arranging the marriage.

Pallavi, a Class 9 student at Kasturba School, says that the students welcomed their new friend with applause.

Local block chief Sunita Devi at Mamta's house.

When Mamta told the other girls that she had called off her wedding, they told her that the slogan of Kasturba School was: "First education, then marriage."

The campaign was launched for the girls of Kasturba School on 26 January last year.

Pooja, a student at the school, says, "Because of illiteracy, old-fashioned beliefs continue in villages. Girls are considered a burden, and so they are married off at a young age. But this way of thinking must end."

Mamta has gone through several difficult experiences over the past week. She is now getting along with the other girls. She wants to study and become a teacher. She is waiting for the government's promised reward.

She says, "If I receive the money, I will repay the loan my parents took for the wedding. I will spend the remaining money on my education and that of my brothers and sisters."

Neelan Prabha Kerketta, the warden and teacher at Baghima School, along with the other teachers and staff, is also happy about Mamta's admission.

Mamta with her classmates at her new school.

Earlier, another tribal girl, Birsamuni, had refused to marry as a child. She is also studying in Class 6 at the same school.

We also visited Hapamuni village, where we met Mamta's father, Ratan Sahu, her mother, Shallo Devi, and their two other children.

Their two-room house is a dilapidated mud house with a tiled roof. Inside, there is a cupboard. In the small courtyard, quilts, mattresses and a bed have been kept. It was immediately clear that these items had been bought for their daughter's wedding.

Ratan says that he works as a driver and does not own even a small piece of agricultural land.

When asked whether his daughter had not refused the marriage earlier, he says, "All this happened suddenly."

Ratan says that he will be happy if his daughter gets an education and becomes independent. However, he will have to work extremely hard to repay the loan.

His wife, Shallo Devi, says they borrowed around 45,000 rupees from several people, taking small amounts from each. They had also saved some money through their labour.

Shallo Devi says that receiving a ration card for poor families and assistance under the Indira Awas scheme from the government would have brought them great relief.

When asked whether the groom's family was angry, Mamta's uncle, Krishna Sahu, says, "At first they became angry, but when they learned the truth, they agreed and returned."

Bindeshwar Sahu, a social worker in the village, believes that Ratan had arranged his daughter's marriage at a young age, knowingly or unknowingly. However, Mamta's decision to refuse the marriage has taught the villagers an important lesson.

Surendra Sahu, another resident of the village, says, "They were busy with the preparations. When the police arrived and took Mamta away, everyone was shocked, but together we managed the situation."

Sunita Devi, the newly elected head of the Ghaghra block, had also visited Ratan Sahu's family. She said that a girl like Mamta had brought pride to her block.

Mamta has now been enrolled at this school. Sunita Devi also assured the family that she would take steps to help Ratan Sahu receive government assistance.

(You can click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "160311_saina_nehwal_exits_all_england_aa": """In the women's singles quarter-final, Saina Nehwal was defeated by Tai Tzu Ying of Chinese Taipei, 15-21, 16-21.

With this defeat, India's hopes in the tournament came to an end, as all the other Indian players, including B. Sai Praneeth, K. Srikanth and Sameer Verma, had already been eliminated.

Saina, the world number three and a bronze medallist at the London Olympics, failed to meet expectations and lost to the Chinese Taipei player in straight games.

Earlier, P. V. Sindhu had been eliminated in the first round of the women's singles event. In women's doubles, Jwala Gutta and Ashwini Ponnappa had also lost in the opening round.

The performances of Manu Attri and B. Sumeeth Reddy in men's doubles were also disappointing.

(You can click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "160405_vert_cul_10_spectacular_museums_rd": """The Musée d'Orsay, an iconic Paris museum on the left bank of the Seine, opened in 1986 in a former Beaux-Arts railway station and hotel. It was established to bridge the gap between the collections of the Louvre Museum and the Pompidou Centre. Its focus is on art and objects from the late 19th and early 20th centuries. It houses the world's largest collection of Impressionist and Post-Impressionist paintings.

In the late 1980s, as the Spanish city of Bilbao was undergoing redevelopment, the idea of building a museum emerged through a partnership between the Basque authorities and the Solomon R. Guggenheim Foundation.

Designed by American-Canadian architect Frank Gehry, the Guggenheim Museum opened in October 1997. Located along the Nervión River in the city's former industrial centre, the building's glass and limestone structure is covered with titanium.

The building quickly became popular. Architect Philip Johnson described it as "the most magnificent building of our time."

Brazil's Oscar Niemeyer Museum opened in 2002–03. It came to be known as 'The Eye' because it resembles a giant human eye placed on a yellow pillar.

The great Brazilian architect who designed the museum, and after whom it was named, was an important figure in the Modern Movement. He was 95 years old when the building was completed.

The 2,000-foot museum displays international artworks as well as works by Niemeyer.

The main building of the British Museum in London was designed by architect Robert Smirke in the Neoclassical style. The inspiration for its grand southern entrance, built when the museum opened in 1852, came from ancient Egypt.

In 2000, Foster and Partners designed a new Great Court. Its magnificent glass roof transformed the museum's inner courtyard into the largest covered public space in Europe.

Canada's Royal Ontario Museum is the country's largest museum of natural history, art and culture. It contains more than six million objects.

Alongside its original 20th-century building, the museum added a new entrance designed by Studio Daniel Libeskind. The museum described the design as "a new distinctive symbol for Toronto in the 21st century."

The complex, angular structure of aluminium and glass, built around a steel framework, was ranked eighth on a list of the world's ugliest buildings in 2009.

The Museo Soumaya in Mexico City is a private cultural institution established in 1994. It is spread across two locations: Plaza Loreto and Plaza Carso. The latter was inaugurated in 2011.

The building was designed by Mexican architect Fernando Romero, with engineering work by Ove Arup and Frank Gehry. Covered with 16,000 hexagonal aluminium tiles, the 151-foot-high museum has six floors.

Its collection consists mainly of artworks from Mexico and Europe. It also houses the largest collection of Rodin's works outside France.

The Hermitage Museum in St Petersburg, Russia, is one of the largest and oldest museums in the world. It was founded by Empress Catherine in 1764 and contains more than three million objects representing art and culture.

The extensive state museum complex consists of several buildings, including the enormous green-and-white Winter Palace, which was once the residence of Russian tsars.

The grand palace was designed mainly by Francesco Bartolomeo Rastrelli in an elaborate Baroque style.

The Museum of Islamic Art in Doha, the capital of Qatar, was inaugurated in 2008. It was designed by Pritzker Prize-winning architect I. M. Pei. At the age of 91, he spent six months travelling around the Arab world to find inspiration for the project.

The huge five-storey limestone building offers a panoramic view of Doha's harbour. Its two sections are connected by a large central courtyard.

The museum houses Islamic artworks spanning 1,400 years.

The Vatican Museums were inaugurated by Pope Julius II in the 16th century. They have 52 galleries and are among the largest museum complexes in the world.

Their collection of religious sculptures and paintings includes works by Raphael, Titian and Caravaggio.

When the original building of the National Museum of Art in Osaka was destroyed in 1970, a new building was designed by César Pelli.

The main part of the building, including its art collections, is underground, with only the entrance structure above ground.

To give the building a more striking appearance, Pelli's studio created a structure of steel and glass resembling reeds or bamboo growing along a river.

(Read the original article in English here, available on BBC Culture. You can click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)"""
}

batch18_accepted_ids = []

print("Batch 18 corrections ready:", len(batch18_corrections))

Batch 18 corrections ready: 10


In [48]:
batch18_corrections_df = pd.DataFrame(
    list(batch18_corrections.items()),
    columns=["id", "complete_english_translation"]
)

if batch18_accepted_ids:
    reviewed_translations.loc[
        reviewed_translations["id"].isin(batch18_accepted_ids),
        "manual_review_status"
    ] = "Human reviewed - no correction"

for _, row in batch18_corrections_df.iterrows():
    reviewed_translations.loc[
        reviewed_translations["id"] == row["id"],
        "complete_english_translation"
    ] = row["complete_english_translation"]

reviewed_translations.loc[
    reviewed_translations["id"].isin(batch18_corrections.keys()),
    "manual_review_status"
] = "Manually corrected"

output_path = (
    f"{base_path}/09_sentence_level_translation/"
    "FactIndic_English_250_Reviewed.csv"
)

reviewed_translations.to_csv(output_path, index=False)

print("Batch 18 saved successfully.")
print("Corrections applied:", len(batch18_corrections))
print("Accepted without correction:", len(batch18_accepted_ids))
print("\nUpdated review status:")
print(reviewed_translations["manual_review_status"].value_counts())
print("\nSaved to:", output_path)

Batch 18 saved successfully.
Corrections applied: 10
Accepted without correction: 0

Updated review status:
manual_review_status
Manually corrected                173
Pending review                     52
Human reviewed - no correction     25
Name: count, dtype: int64

Saved to: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv


In [49]:
# Fetch Batch 19 — next 10 pending articles

pending_ids = reviewed_translations.loc[
    reviewed_translations["manual_review_status"] == "Pending review",
    "id"
].head(10).tolist()

source_lookup = df_250.set_index("id")
translation_lookup = reviewed_translations.set_index("id")

print("Batch 19 — Articles for Human Review")
print("Number of articles:", len(pending_ids))

for i, article_id in enumerate(pending_ids, 1):
    print("\n" + "=" * 100)
    print(f"ARTICLE {i}/10: {article_id}")

    print("\nHINDI ORIGINAL:\n")
    print(source_lookup.loc[article_id, "text"])

    print("\nCURRENT ENGLISH TRANSLATION:\n")
    print(translation_lookup.loc[
        article_id, "complete_english_translation"
    ])

Batch 19 — Articles for Human Review
Number of articles: 10

ARTICLE 1/10: 160412_assam_gogoi_fir_ml

HINDI ORIGINAL:

दरअसल मुख्यमंत्री ने मतदान के दिन एक संवाददाता सम्मेलन का आयोजन किया था. आयोग ने मुख्यमंत्री गोगोई द्वारा सोमवार को मतदान से पहले संवाददाता सम्मेलन करने को आदर्श चुनाव आचार संहिता का उल्लंघन करार देते हुए उनके ख़िलाफ़ प्राथमिकी दर्ज करने का निर्देश दिया था. चुनाव आयोग के निर्देश के बाद ज़िला उपायुक्त ने मुख्यमंत्री गोगोई के ख़िलाफ़ एफ़आईआर दर्ज कर ली है और इसकी पुष्टि भी कर दी है. इस तरह के उल्लंघन के मामले में कार्रवाई की जा सकती है. जनप्रतिनिधित्व कानून की धारा 126 के तहत जुर्म साबित होने पर दो साल की जेल या फिर जुर्माना या दोनों हो सकते है. समाप्त उप चुनाव आयुक्त संदीप सक्सेना के अनुसार असम में चुनाव अधिकारियों ने गोगोई से कहा था कि चुनाव संपन्न होने के 48 घंटे पहले संवाददाता सम्मेलन करना उचित नहीं था. यह जन प्रतिनिधित्व अधिनियम के तहत आदर्श चुनाव आचार संहिता का उल्लंघन है क्योंकि इससे मतदाता प्रभावित हो सकते हैं. वहीं एक शिकायत की जांच पड़ताल के बाद आयोग ने बीजेपी 

In [50]:
batch19_corrections = {
    "160412_assam_gogoi_fir_ml": """The Chief Minister had in fact held a press conference on the day of polling. The Election Commission had directed that an FIR be registered against Chief Minister Gogoi for holding a press conference on Monday, before voting began, declaring it a violation of the Model Code of Conduct. Following the Commission's directive, the District Magistrate registered an FIR against Chief Minister Gogoi and confirmed that it had been filed. Such violations can invite legal action. If convicted under Section 126 of the Representation of the People Act, a person can face up to two years in prison, a fine, or both.

Former Deputy Election Commissioner Sandeep Saxena said that election officials in Assam had told Gogoi that holding a press conference within 48 hours before the conclusion of polling was inappropriate. It violated the Model Code of Conduct under the Representation of the People Act because it could influence voters. Meanwhile, after investigating a complaint, the Commission found that BJP leader Himanta Biswa Sarma had not held an election rally within 48 hours of the start of polling, contrary to the allegation. Chief Minister Gogoi has allegedly accused the Election Commission of bias on several occasions. On Monday, voting took place for 61 Assembly seats in the second and final phase of polling in Assam.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "160506_breast_cancer_test_easier_cj": """Mihir had never imagined that he would work on breast cancer. But circumstances changed, and so did the direction of his work. His mother-in-law was diagnosed with breast cancer. When the family began discussing it, they discovered that several other women close to them were also affected. Yet breast cancer was something most people were reluctant to discuss openly.

Mihir tried to understand the problem that had emerged in his own family and learned that breast cancer is the most common cancer among women. The earlier it is detected, the better, because delays increase the risk to a woman's life. In developing countries, particularly in rural areas, facilities for breast cancer screening are often unavailable. Mammography can detect the disease, but it requires a specialist. India, meanwhile, has only around 10,000 radiologists.

Taking these realities into account, Mihir decided to develop a small machine that could provide immediate results without exposing women to radiation. The examination would be painless, and even an ordinary health worker could use the device easily.

In 2009, Mihir and his co-founder Matthew Kempisi established UE Life Sciences Inc. in Philadelphia. The company began operations with an investment and financial support of US$4 million. Mihir had experience taking medical technology from concept to clinic, while Matthew, an electrical engineer, had experience developing medical equipment.

UE Life Sciences then developed a small machine to detect breast cancer. The device can detect even small lumps in the breast and provide detailed information within minutes on a digital notepad connected to the machine.

Mihir said, "It is difficult to take any medical device to the people. Before you can assess its success, you first have to prepare people to use it. We want to reach women at every level through different delivery partners, such as voluntary organisations, corporations, healthcare services and public health centres."

He added, "Screening with this portable machine is inexpensive and painless. We want to make this service available to 1 million women in India by the end of 2016."

Boman Dhabhar, a well-known cancer specialist from Mumbai, said, "Many cases come to us in which breast cancer has already reached an advanced stage. Even today, breast cancer screening in India generally begins with self-examination, followed by an examination by a specialist, and finally mammography. Because of hesitation, women often do not visit doctors for screening."

He said, "With this simple device, nurses or junior hospital staff can also conduct preliminary examinations and recommend mammography if they detect a lump. The device makes it unnecessary to visit a cancer specialist for an initial screening. We used it at a breast cancer screening camp and detected lumps in some patients, who were subsequently advised to undergo mammography."

In the future, UE Life Sciences wants to develop machines for detecting other types of cancer as well, so that healthcare services can reach underserved people in developing countries and awareness can be increased. After India, the company hopes to expand its work to Southeast Asia, the Middle East and Latin America.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "160512_rocky_yadav_profile_sh": """Rocky Yadav is accused of shooting Aditya Sachdeva, the son of a businessman, in Gaya on May 7 following a minor dispute over a vehicle overtaking his car. Rocky graduated from Banaras Hindu University and completed his postgraduate studies at Delhi University. However, after last week's incident, the influence of his father Bindeshwari Yadav, also known as Bindi Yadav, and his criminal past became apparent. Bindi Yadav has more than 18 criminal cases registered against him, including cases involving sedition, murder and robbery.

In 2001, during Rabri Devi's government, Bindi Yadav was elected president of the Zila Parishad for the first time. That same year, his wife Manorama Devi entered politics by becoming a member of the Panchayat Samiti.

Having grown up amid the contracting business and power politics, Rocky found it easy to develop a taste for modern weapons and expensive cars. The 24-year-old Rocky Yadav is also a member of the Delhi Rifle Club. He has participated in several competitions organised by the National Rifle Association.

Rocky's mother, Manorama Devi, first became a member of the Bihar Legislative Council in 2003 as a representative of the Rashtriya Janata Dal, the party led by Lalu Yadav and Rabri Devi. Through the contracting and construction businesses, Manorama Devi pursued a political strategy of remaining close to those in power. When Lalu and Rabri lost power, she joined Nitish Kumar's Janata Dal (United).

Manorama Devi, Bindi Yadav's second wife and a postgraduate, became a member of the Legislative Council for the second time in June 2015. However, within a year, she became embroiled in controversy. On the day after her son Rocky was arrested in the Aditya Sachdeva murder case, police searched her residence in Gaya and recovered bottles of liquor.

Chief Minister Nitish Kumar has announced prohibition in Bihar. However, liquor was found at the home of a member of his own party. The party therefore suspended Manorama Devi over the allegation.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "160606_ajit_jogi_new_party_rv": """Jogi said that a decision on the party's name and election symbol would be taken after seeking people's opinions. Speaking at a gathering in Kotmi, Bilaspur, on Monday, he criticised the Congress and the BJP, saying that a new party was needed to protect Chhattisgarh's identity.

"Now I am free. Decisions concerning Chhattisgarh will no longer be made in Delhi. Chhattisgarh will be freed from Raman Singh," Jogi said.

Ajit Jogi also said that, once his government was formed, he would make Chhattisgarh the first tax-free state in the country.

Ajit Jogi's wife, Renu Jogi, is a Congress MLA from the Kota constituency, while his son, Amit Jogi, represents Marwahi. Besides his wife and son, two other Congress MLAs were also present at the event. Some former Congress ministers and former MLAs also attended Jogi's gathering.

In January this year, the Congress expelled Amit Jogi from the party for six years after a tape became public. The tape allegedly recorded a transaction involving crores of rupees paid by the government to a Congress candidate to persuade the candidate to withdraw from a by-election in Chhattisgarh in 2014.

A proposal to expel Ajit Jogi from the party for six years was also sent to the All India Congress Committee. However, that proposal is still pending.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "160823_sania_mirza_wta_doubles_top_sk": """Only 10 other players have spent a longer continuous period at the top of the rankings. Playing with her new partner, Barbora Strycova, Sania won the women's doubles title at the Cincinnati Open tennis tournament.

In the final, Sania and Strycova defeated Sania's former partner Martina Hingis and Coco Vandeweghe 7-5, 6-4. This was Sania's sixth WTA title in 2016. The 29-year-old had now won 38 doubles titles.

Sania had also enjoyed tremendous success with her former partner Martina Hingis. The two began playing together in March 2015 and won nine titles as a team.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "india-39165960": """This is what makes Kashi, the holy city, distinctive. In this city of thousands of temples, the noise of large vehicles, helicopters flying overhead and tall stages set up at various places have made the streets even busier. It is often said that every corner of Uttar Pradesh is steeped in politics and that political debates run through people's veins like blood.

Why are so many BJP heavyweights gathering in Varanasi, the BSP's stronghold in the UP elections? There is an occasion, a tradition and a particular political mood. Take a walk around the city and you will find groups of people in almost every street, at every corner and every crossroads, immersed in political discussions while enjoying the sweetness of Banarasi paan. Sometimes, those debates even make the paan taste a little bitter.

Uttar Pradesh is a state with a strong political temperament. Just as Banaras is identified with Mahadev, the great god Shiva, political chatter has also become part of the city's identity. Indeed, this is a place where political discussions may begin with smaller parties or local leaders but soon travel much further—even as far as Donald Trump.

The political test of the parties in Yogi and Mukhtar's Purvanchal

There may never be a better opportunity to understand Uttar Pradesh's political mood and sharpen one's political judgement. Over these five days, the journey began in Lucknow, continued to Gorakhpur and then reached Banaras. It became clear that this time voters in Uttar Pradesh might judge their politicians on development and a sense of connection with the people. This suggests that caste equations may shift in a different direction rather than favouring any single party.

Uttar Pradesh is the country's most populous state. Clearly, reaching a sound conclusion requires examining the different political parties. Let us begin with the SP-Congress alliance. But before discussing the alliance, it is worth looking back at how and when political equations in Uttar Pradesh have changed.

Around two months ago, it seemed that the Samajwadi Party had been completely pushed out of the contest, leaving a direct fight between the BSP and the BJP.

'UP does not like the Pappu-Gappu duo'

The reason was the internal conflict within the Samajwadi Party. However, there is no doubt that Akhilesh demonstrated considerable political judgement during the period of discord. He gradually brought the conflict under control and also created a favourable political position for himself.

Until then, it was widely believed that Akhilesh was merely the party's chief ministerial face and that several other leaders were exercising power behind the scenes. But Akhilesh overturned that perception through his political skill. By presenting himself as a young leader focused on development, he managed to turn the atmosphere in his favour.

He succeeded in convincing people that he was not only the party's most prominent leader but also its strongest supporter of the caste coalition on which the party depended.

The complications of the SP-Congress alliance

Politics soon took another turn, and the SP and Congress formed an alliance. After the announcement, many began to believe that this alliance was the strongest contender for victory in the election.

The perception was that the SP already had the support of Yadav and Muslim voters, while the alliance would also bring traditional Congress voters into its fold. Its impact was indeed visible during the initial phases. However, that atmosphere did not last long, and the BJP began making a comeback.

We will discuss the BJP's strategies later. First, let us examine the reasons why the Samajwadi Party appears to be losing ground in this election.

The deep internal conflict that affected the Samajwadi Party before the election damaged its public image. More importantly, Mulayam Singh himself became upset.

The impact on Muslim voters

Until now, Mulayam Singh's political skill in bringing together Muslim and Dalit voters had been one of the major strengths of the Samajwadi movement in Uttar Pradesh. But the situation seems different this time. Mulayam's absence from the election campaign appears to be dividing the SP's support base, and the party seems to lack the energy that his presence once provided.

When the BJP helped Mulayam form a government in UP

Muslim voters appear to be the group most affected, as a section of them may move towards the BSP. The young leadership of the alliance could help make up for this shortfall. However, there is also concern that traditional Congress voters may become confused by the alliance and shift towards the BSP or the BJP.

The BSP!

The BSP had hoped that an alliance of Dalit and Muslim voters would carry it to victory and that its elephant symbol would roar. But the division of Muslim votes is hurting both the BSP and the SP-Congress alliance while benefiting the BJP.

Nevertheless, the crowds attending Mayawati's rallies show that the BSP still has a genuine opportunity in this election. Mayawati has indeed been working on her Dalit-Muslim agenda for several months.

Mayawati's confidence

Mayawati believes that Muslim voters shifting away from the SP could help her win. By giving tickets to 97 Muslim candidates, the BSP has made every effort to unsettle the SP-Congress alliance.

But will the elephant's confident stride be enough to defeat the BJP? That is the big question.

The BJP is putting everything it has into capturing power in Uttar Pradesh. Its strategy includes capitalising on Prime Minister Narendra Modi's popularity, relying on the political partnership of Modi and Amit Shah, bringing smaller parties such as Apna Dal into its alliance, holding a series of rallies, attacking the opposition and promising voters a dramatic change. It is also using caste calculations to isolate its opponents.

How much do Muslims trust Mayawati?

The BJP is trying to take advantage of internal divisions in the Samajwadi Party's strongholds of Etawah and Mainpuri. It is also seeking to create an anti-incumbency mood against Akhilesh's ministers and MLAs. By giving tickets to non-Yadav OBC candidates and non-Jatav Dalit candidates, it hopes to break into the traditional vote banks of both the SP and the BSP.

This strategy is giving the BJP a strong advantage against the SP-Congress alliance and the BSP. In the changed political environment, the BJP appears to be moving ahead, while the SP-Congress alliance and the BSP seem to be damaging each other's prospects. The BJP appears to be benefiting from this.

(These are the author's personal views.)

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "india-39447746": """Sakshi Joshi has announced through a Facebook post that Sandeep Upadhyay, who had been wanted by the police, has been arrested.

Recently, Sakshi updated her Facebook status. In response, a man named Sandeep Upadhyay said that anyone who wanted to live in Uttar Pradesh would have to support Yogi Adityanath. Sakshi replied that she would continue living in UP but would not chant "Yogi, Yogi". Sandeep then threatened to rape her.

Sakshi filed a complaint with the Noida police. In a Facebook post, she later announced that Sandeep had been arrested. She thanked Chief Minister Yogi Adityanath.

Sakshi wrote, "Chief Minister Yogi Adityanath ji has proved that fanatics will not be tolerated, even if they are supporters."

Sandeep was arrested in Surat. Sakshi said that he had been absconding since making the objectionable remarks. He is currently in police custody.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "india-41263102": """Pradyumn's school was Ryan International School in Gurugram. A mother from Bihar was also in Gurugram, Haryana. She had not eaten anything since morning or even drunk a drop of water. She too was observing a fast without water. The only difference was that she had given up food and water in memory of her son.

When I reached Pradyumn's home, I saw his mother lying almost unconscious on a sofa in a downstairs room. Seven or eight women were sitting around her. One of them was holding Pradyumn's mother's hand. All the women were silent. The only sound was that of the fan.

...The 15 minutes that changed a father's life

The teacher made him stand in the toilet; a girl fell into depression.

Silence had settled over Pradyumn's home. Outside, a tent had been put up in the afternoon. In a chair there was a picture of Pradyumn dressed as Lord Krishna, smiling. But the faces of the people sitting nearby were filled with deep sadness.

I was holding a BBC microphone. As soon as I approached them, one person gestured for me to sit on the ground. The atmosphere was deeply sorrowful, and I did not have the courage to ask anyone a question.

After sitting there for a long time, I got up and went inside Pradyumn's house. This time I met his cousin Mukul. Mukul was studying engineering in Bengaluru. He had come to Delhi after hearing the news of Pradyumn's death.

In a low voice, Mukul said, "Pradyumn's mother spends most of her time lying down as if she is ill. Women from the neighbourhood keep visiting. There are many people around, but the silence remains."

Pradyumn's bicycle

'Pradyumn had enrolled in skating classes'

Pradyumn's father, Varun Chandra Thakur, came down the stairs. He stopped when he saw me and asked me to sit on a chair. He then sat on the chair beside me.

Before I could ask anything, his eyes fell on Pradyumn's bicycle. Becoming emotional, he pointed towards it and said, "Pradyumn was very active in sports. He preferred spinning a top, playing cricket and riding his bicycle to watching television or playing mobile games. He was more interested in sporting activities. This year, he insisted on enrolling in skating classes. This is his third bicycle."

He continued, "If you visit his school's Facebook page, you will find Pradyumn in at least one out of every four or five posts. On the previous Teachers' Day, he hosted the programme. His teacher had praised him highly to his mother for that."

Pradyumn's elder sister

After her brother's murder, Pradyumn's elder sister had become very quiet. Varun said, "My elder daughter has become very silent. We are trying to keep her occupied with other children so that her attention does not keep returning to her brother. But I am afraid of how she will cope when everyone has left. I don't know how we will manage either."

Pradyumn's uncle, Tarun Chandra Thakur

'Regret over not being able to buy him a cricket kit'

Pradyumn's uncle, Tarun Chandra Thakur, was performing the 13-day shraddha ceremonies. He works at the District Collectorate in Jamshedpur.

Tarun said, "Pradyumn was born on May 7, 2010. Among the three daughters of the two brothers in our family, he was the only boy. Although we do not distinguish between sons and daughters, he was the kuldeepak—the one who would carry on the family line—for his grandfather. His grandfather used to call him that."

He added, "Pradyumn was a cheerful child. Whenever he came to Jamshedpur during the holidays, he would climb on top of me and start playing in the morning. He was very fond of cricket. He watched every match and knew every player."

Tarun continued regretfully, "I used to fulfil all his wishes, but perhaps one regret will stay with me for the rest of my life: he had asked me for a cricket kit, and I could not get it for him."

He said, "He also loved mangoes. Whenever we went to our village in Madhubani, I would bring 35 kilograms of mangoes for him. He would eat them throughout the day."

Pradyumn's aunt Vinita

'Pradyumn also wrote poems'

Pradyumn's aunt Vinita is a teacher in Jamshedpur. Remembering him, she said, "Pradyumn was very good at his studies. He loved Hindi. He also wrote poems. Some lines of his poems were deeply touching. I would always tell his mother that her son would grow up to become a poet because he was so interested in Hindi."

Vinita is looking after Pradyumn's mother and never leaves her alone. She said, "His mother knew very well what he needed to do and when. She was always occupied with looking after him. Perhaps she will feel the absence of these things even more now."

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "india-43796841": """However, the figures suggest that gold prices could rise to their highest level ever around Akshaya Tritiya this year. On April 11, the price of 24-carat gold crossed Rs 31,524 per 10 grams. Although gold prices fell slightly on April 17, experts believe that buyers will continue purchasing gold.

Gold becomes the cheapest it has been in five years

For the first time, gold's shine fades in comparison with Bitcoin

Why are gold prices rising?

Kunal Shah, head of the commodities and currency research firm Nirmal Bang, says, "Because of uncertainty over monetary policy, uncertainties related to geopolitical crises and concerns about rising inflation in Western economies, gold will be trading at around $1,450 globally by the end of March 2018. It will reach the $1,600 level before March 2019."

Gold has traditionally been considered a good investment option. This is a year of global uncertainty. The United States is threatening a trade war with China, and tensions are rising in Syria. In such circumstances, investors are cautious about putting their money into other assets. As a result, they are choosing to invest in gold, pushing up prices around the world. Countries such as India will also be affected.

The unusual ways in which gold is smuggled

What is the situation in India?

Gold demand in India declined after demonetisation and the introduction of GST. In March, gold imports fell by 40% compared with the previous year, to $2.4 billion. This had a negative impact on the industry.

However, demand for gold is now expected to recover. Better monsoon conditions have improved farmers' incomes, which could increase gold demand during the festive season. Two-thirds of the country's total gold demand comes from rural India.

A look at gold prices around Akshaya Tritiya over the past few years suggests that 2018 could be the most expensive year yet. In 2010, when Akshaya Tritiya fell on May 16, 24-carat gold could be bought for Rs 18,167 per 10 grams. Last year, however, the price had risen to Rs 29,890 per 10 grams.

Source: Goldpriceindia.com (Prices per 10 grams)

The right time to buy gold

The positive side is that anyone who bought gold earlier would have seen the value of their investment increase. The question, then, is whether this is the right time to buy gold. Experts are divided on the issue.

Gnanasekar Thyagarajan, director of Comtrendz Research, says, "When gold reaches the Rs 30,000 mark, people tend to think about selling it. But prices are now stabilising."

"I believe gold prices will rise by the end of this year. Therefore, buying gold on this Akshaya Tritiya could be a good decision, because prices are likely to rise from this point onwards."

Vandana Bharti of SMC Global Securities told the BBC, "The market situation is different at the moment, so I would recommend waiting for a few months. September and October may be better times to buy, as prices could fall by 5-6%."

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "india-44912345": """The 28th meeting of the GST Council was chaired by Piyush Goyal, who was then holding charge of the Finance Ministry. The GST Council decided to exempt sanitary napkins from GST, bringing the tax rate on them down to zero. Until then, sanitary napkins had been taxed at 12%.

Items exempted from GST:
- Sanitary pads
- Rakhis (not made with precious stones)
- Marble sculptures
- Raw materials used to make brooms
- Sal leaves

Items for which GST was reduced from 12% to 5%:
- Handloom mats
- Fertilisers containing phosphoric acid
- Knitted caps priced below Rs 1,000

Items for which GST was reduced from 28% to 18%:
- Lithium-ion batteries
- Vacuum cleaners
- Food grinders and mixers
- Shavers and hair clippers
- Hand dryers
- Water coolers
- Ice-cream freezers
- Refrigerators
- Cosmetics
- Perfumes and fragrances
- Paints and varnishes

In addition, the GST rate on ethanol supplied to oil marketing companies for blending with petrol and diesel was reduced from 18% to 5%.

Related stories:
- Sanitary-pad advertisements will now show 'real blood'
- Sanitary pads for Rs 2.50, but how effective are they?
- Free sanitary pads for poor girls

(Click here for BBC Hindi's Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)"""
}

batch19_accepted_ids = []

In [51]:
corrections_df = pd.DataFrame(
    list(batch19_corrections.items()),
    columns=["id", "complete_english_translation"]
)

if batch19_accepted_ids:
    reviewed_translations.loc[
        reviewed_translations["id"].isin(batch19_accepted_ids),
        "manual_review_status"
    ] = "Human reviewed - no correction"

for _, row in corrections_df.iterrows():
    reviewed_translations.loc[
        reviewed_translations["id"] == row["id"],
        "complete_english_translation"
    ] = row["complete_english_translation"]

reviewed_translations.loc[
    reviewed_translations["id"].isin(batch19_corrections.keys()),
    "manual_review_status"
] = "Manually corrected"

output_path = (
    f"{base_path}/09_sentence_level_translation/"
    "FactIndic_English_250_Reviewed.csv"
)
reviewed_translations.to_csv(output_path, index=False)

print("Batch 19 saved successfully.")
print("Corrections applied:", len(batch19_corrections))
print("Accepted without correction:", len(batch19_accepted_ids))
print("\nUpdated review status:")
print(reviewed_translations["manual_review_status"].value_counts())
print("\nSaved to:", output_path)

Batch 19 saved successfully.
Corrections applied: 10
Accepted without correction: 0

Updated review status:
manual_review_status
Manually corrected                183
Pending review                     42
Human reviewed - no correction     25
Name: count, dtype: int64

Saved to: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv


In [52]:
pending_ids = reviewed_translations.loc[
    reviewed_translations["manual_review_status"] == "Pending review",
    "id"
].head(10).tolist()

source_lookup = df_250.set_index("id")
translation_lookup = reviewed_translations.set_index("id")

print("Batch 20 — Articles for Human Review")
print("Number of articles:", len(pending_ids))

for i, article_id in enumerate(pending_ids, 1):
    print("\n" + "=" * 100)
    print(f"ARTICLE {i}/10: {article_id}")

    print("\nHINDI ORIGINAL:\n")
    print(source_lookup.loc[article_id, "text"])

    print("\nCURRENT ENGLISH TRANSLATION:\n")
    print(translation_lookup.loc[
        article_id, "complete_english_translation"
    ])

Batch 20 — Articles for Human Review
Number of articles: 10

ARTICLE 1/10: india-45233283

HINDI ORIGINAL:

लेकिन इन लोगों में से एक शख्स हैं- हाल ही में रिजर्व बैंक ऑफ इंडिया (आरबीआई) में अंशकालिक निदेशक बने एस गुरुमूर्ति. एस गुरुमूर्ति अपने ट्वीट से इशारा कर रहे हैं कि ये बाढ़ अगर सबरीमाला के भगवान की नाराज़गी की वजह से है तो सुप्रीम कोर्ट को अपने फ़ैसले पर सोचना चाहिए. उन्होंने एक ट्विटर यूज़र के ट्वीट पर अपनी बात रखी है. उन्होंने लिखा, "सुप्रीम कोर्ट के जजों को देखना चाहिए कि केस और जो सबरीमाला में हुआ, उसके बीच कोई संबंध है या नहीं. अगर कोई संबंध होने का लाखों में एक चांस भी है तो लोगों को अयप्पन के ख़िलाफ़ फ़ैसला पसंद नहीं आएगा." जब कुछ लोगों ने उनके ट्वीट पर इस बात की आलोचना की तो उन्होंने फिर से अपनी बात को दोहराया. "भारत के बुद्धिजीवियों के पाखंड को देखकर हैरान हूं जो लोगों के विश्वास को कचरा समझते हैं. 99 फ़ीसदी भारतीय भगवान में विश्वास करते हैं. 100 फीसदी ज्योतिष में विश्वास रखते हैं जिसमें लिबरल, सेक्युलर, बुद्धिजीवी भी शामिल हैं. नास्तिक करुणानिधि के लिए उनके अनुयायियों ने

In [53]:
batch20_corrections = {
    "india-45233283": """One of these people is S. Gurumurthy, who recently became a part-time director of the Reserve Bank of India (RBI). In his tweet, Gurumurthy suggested that if the floods were the result of the displeasure of the deity of Sabarimala, the Supreme Court should reconsider its decision. He was responding to a tweet by another Twitter user.

"The Supreme Court judges should examine whether there is any connection between the case and what happened at Sabarimala. Even if there is a one-in-a-million chance of such a connection, people will not accept a decision against Ayyappan," he wrote.

When some people criticised his tweet, he repeated his position. "I am shocked by the hypocrisy of India's intellectuals, who treat people's faith as rubbish. Ninety-nine per cent of Indians believe in God. One hundred per cent believe in astrology, including liberals, secularists and intellectuals. Even the followers of the atheist Karunanidhi prayed for him. I am among those who believe in God but not in astrology."

**Who is S. Gurumurthy?**

Gurumurthy is the co-convenor of the Swadeshi Jagran Manch and is also considered an ideologue of the Rashtriya Swayamsevak Sangh (RSS). S. Gurumurthy, who is said to be close to Prime Minister Narendra Modi, was appointed a part-time director of India's central bank, the Reserve Bank of India (RBI), on 8 August.

When Prime Minister Modi announced demonetisation in November 2016, Gurumurthy was said to have been the person behind the decision. After his appointment as a director, Gurumurthy wrote on Twitter: "This is the first time I have accepted such a position. I have never accepted a directorship in the private sector or in a public sector undertaking (PSU). I have never conducted an audit for a PSU or a private company either. I wanted to speak independently. But when pressure increased for me to work in the public interest, I accepted the position."

**The Sabarimala temple and the ban on women aged 10 to 50**

A Twitter user named Hari Prabhakaran, who linked Sabarimala to the floods, posted a photograph of the temple with the message: "No law is greater than God. If you allow everyone to enter, He will refuse everyone." The tweet received 3,400 likes and 1,700 retweets.

He also wrote: "If you cannot change your birth or death, why are you changing a centuries-old temple tradition? Sabarimala is our faith." S. Gurumurthy had responded to this tweet.

Another user, Satish Kumar, wrote that the situation could also be viewed the other way around: if women were refused entry, the deity might refuse everyone. A user named Gopalakrishnan wrote that people in Kerala believed that Lord Ayyappan was angry and that the Supreme Court should not interfere in matters concerning people's beliefs and practices.

**How people responded**

Gurumurthy's tweet attracted considerable criticism. A user named Manikandan wrote: "Please do not link this to a religious issue. This does not feel right. I did not expect this from you. People who are stranded need help. If you can help them, please do so."

A user named Kamalakar Durge wrote: "How can someone like S. Gurumurthy work at the RBI? If something goes wrong, he will say it was God's will."

Television journalist Lata Venkatesh wrote: "It is terrible that people like Gurumurthy are linking the Kerala floods to women's entry into Sabarimala. People holding government positions should not be allowed to make statements that go against the Constitution of India."

A user named Gopi Shankar wrote: "I never thought I would live to see the day when such an irrational person would sit on the RBI board. If God wanted to punish someone, the floods would have struck Delhi, where the Supreme Court is located. Why Kerala?"

**What is the controversy surrounding the Sabarimala temple?**

Earlier that month, the Supreme Court began hearing a petition challenging the ban on women's entry into the Sabarimala temple in Kerala. Women between the ages of 10 and 50 have been prohibited from entering the temple, a restriction based on a centuries-old tradition. The reason given for the ban is that women in this age group menstruate.

In its observations, the court repeatedly said that women have the same right as men to enter the temple. The Supreme Court has reserved its verdict in the case.

(You can click here to download the BBC Hindi Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)""",

    "india-45961945": """According to the newspaper, CBI Special Director Rakesh Asthana was leading the investigation into Vijay Mallya's case. Asthana himself is accused of accepting a bribe in the case involving meat exporter Moin Qureshi.

Vijay Mallya is accused of failing to repay loans worth Rs 9,000 crore taken from banks. He left for Britain after the investigation began. Mallya travelled to Britain in March 2016 and has been living in London ever since. The Indian government is trying to secure his extradition from Britain.

Mallya could now take advantage of the ongoing dispute within the CBI and argue that he cannot be sent to India on the basis of an investigation led by someone who is himself an accused person. His lawyers are unlikely to hesitate to challenge Rakesh Asthana and the CBI in an effort to prevent his extradition.

Mallya had earlier cited inadequate facilities for prisoners in Indian jails as a reason to oppose his extradition. In response, the Indian government sent a video of a jail to demonstrate the facilities available there. The extradition case against Vijay Mallya is scheduled to be heard on 10 December.

**The Institutions of Eminence issue resurfaces**

The Indian Express has reported on the Institutions of Eminence scheme. IIT Madras had written to Human Resource Development Minister Prakash Javadekar expressing deep disappointment that it had not been selected for the scheme despite being recommended by the UGC selection panel.

Pawan Goenka, chairman of IIT Madras, wrote the letter in July, just a month after the announcement of the institutions selected for the Institutions of Eminence category.

The Empowered Expert Committee (EEC) had selected 11 institutions from a list of 114 for this category. IIT Madras was among them. Six institutions were ultimately selected on the basis of the QS World University Rankings 2018.

IIT Madras objected to the use of rankings by a private agency as the basis for selection. It argued that neither the EEC's recommendation nor the government's own National Institutional Ranking Framework (NIRF) had been used as the basis for the selection. The NIRF had ranked IIT Madras as the country's best engineering institution for three consecutive years.

The Institutions of Eminence scheme had previously come under scrutiny when the Jio Institute was included in the category even though it did not yet exist.

**Firecrackers allowed for two hours on Diwali**

Hindustan has reported on the Supreme Court's order regarding the use of firecrackers. In view of the pollution situation in Delhi, the Supreme Court has ordered that firecrackers may be burst for only two hours on Diwali, from 8 pm to 10 pm.

The police station in the area concerned will be responsible for ensuring that the order is followed. Any violation will be treated as contempt of court.

The court also said that only green firecrackers could be manufactured and sold for Diwali. It defined green firecrackers as those that produce sound and smoke within the prescribed limits and release lower quantities of nitrogen oxides and sulphur dioxide.

The court also said that firecrackers should be burst collectively wherever possible in Delhi and the National Capital Region (NCR). The order will apply not only to Diwali but also to other religious and social festivals.

**Representational image: Garbage mound taller than Qutub Minar**

According to Amar Ujala, the height of the garbage mound at Delhi's Bhalswa landfill site, which covers 70 acres, has exceeded the height of the Qutub Minar.

The Qutub Minar is 73 metres tall. The garbage mound rises 60 metres above ground level, while waste also extends 12 metres below the ground. The site contains 80 lakh metric tonnes of garbage.

Taking note of the fire that broke out at the site on Sunday, 21 October, North Delhi Mayor Adesh Gupta and Commissioner Madhup Vyas inspected the location on Tuesday.

Also read: (You can click here to download the BBC Hindi Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)""",

    "india-46106302": """Chameli Devi of Ghaghra village is pictured with her mother-in-law.

The abducted workers include Prakash Mahato and Prasadi Mahato, residents of Ghaghra village in Bagodar block; Hulas Mahato, a resident of Mahuri village; and Kali Mahato, a resident of Bedam village in Tatijharia block.

All of them worked in Afghanistan for KEC International, a company owned by Indian industrialist Harsh Goenka. Following the intervention of the External Affairs Minister, the company has started paying salaries to the families of the abducted workers. However, the families have been instructed not to speak to the media.

Since the kidnapping, the company has not issued an official statement on the incident. KEC International is part of the RP Goenka (RPG) Group and has a contract to install power transmission lines in Afghanistan. Many people from Jharkhand have travelled to Afghanistan for work, most of them from Bagodar.

**Six months since the abduction**

Even six months after the kidnapping, the governments of India and Afghanistan have been unable to trace the workers. Their families have therefore become increasingly desperate.

Mulia Devi is the wife of Prasadi Mahato and has been waiting for her husband to return for months. Their daughter passed her Class 10 examination this year, but Mulia Devi says that she has now given up her studies.

"We are poor. My husband went abroad so that he could earn some money and provide a proper education for our children. Now we do not even know where he is. How are we supposed to survive? Whom can we trust? Who will bring my husband back? We do not even know whether the government is doing anything to secure their release. Please help me reunite with my husband," she said.

**Villagers plan to launch a protest**

Santosh Rajak, a resident of Prasadi Mahato's village, is disappointed with the Modi government over the issue. He says that the government has been unable to put pressure on Afghanistan.

Hulas Mahato, from Mahuri village, is another of the Indian workers abducted in Afghanistan. His wife, Pramila Devi, is becoming increasingly distressed. She is so overcome with grief that she is unable to eat or drink properly. She said that when her children ask about their father, she has no answer to give them.

**The External Affairs Minister's assurance has not helped**

Forty days after the abduction, External Affairs Minister Sushma Swaraj met in Delhi the wives of three of the abducted workers and the son of Prasadi Mahato.

Chameli Devi, the wife of Prakash Mahato from Ghaghra village, was part of the delegation. She told the BBC that the External Affairs Minister had promised to trace all the workers within a month.

Jharkhand Chief Minister Raghubar Das had also met the wives of some of the workers in Ranchi the previous month. He announced financial assistance of Rs 1 lakh for each family, but the money has still not been received.

After meeting the chief minister, Pramila Devi said that he had also promised to locate her abducted husband soon, but nothing had come of that assurance.

Sikandar Ali, who runs a migrant workers' group, and Laxman Mahato, the head of Bagodar West, said that the migration of workers from the region had not stopped despite the abductions in Afghanistan. Unemployment remains a major problem in the area, and people are still travelling to Afghanistan for work.

Also read: (You can click here to download the BBC Hindi Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)"""
}

batch20_accepted_ids = []

In [54]:
batch20_corrections = {
    "india-47692221": """Surjewala tweeted from his official Twitter handle: "A fresh example of the BJP's violent mindset. Speaking in a threatening tone to those raising slogans against the Faridabad MP, Kalraj Mishra said, 'If this were his state, he would have got off the stage and shot those creating a disturbance.' Is this the BJP's message — ask questions and get shot!" In the tweet, Surjewala shared a video of BJP leader Kalraj Mishra's rally, which has been viewed more than 65,000 times. Kalraj Mishra is a BJP MP from the Deoria Lok Sabha constituency in eastern Uttar Pradesh and is also the party's election in-charge for Haryana for the upcoming Lok Sabha elections.

On Sunday, Kalraj Mishra had arrived to address the BJP's 'Vijay Sankalp Sabha' in the Faridabad Lok Sabha constituency. Union Minister Krishan Pal Gurjar was also present at the gathering. Congress spokesperson Randeep Singh Surjewala tweeted a 22-second video from the same BJP meeting. However, the message Surjewala attached to the video is misleading.

What happened at Sunday's 'Vijay Sankalp Sabha'?

The BJP's 'Vijay Sankalp Sabha' was held in the Prithla area of Haryana's Faridabad Lok Sabha constituency. According to reports in local media, around 30 people at the party workers' gathering appealed that Union Minister and sitting Faridabad MP Krishan Pal Gurjar should not be given the party ticket from the Faridabad constituency again. These BJP workers alleged that development work in the Faridabad area had slowed down because of Krishan Pal Gurjar. Krishan Pal Gurjar did not speak at the gathering. However, senior BJP leader Kalraj Mishra said, in response to the protest by some people, that there was no place for indiscipline in the party.

What did the BJP leader say?

The gathering was streamed live on Facebook through the official pages of Haryana BJP, Faridabad BJP and Kalraj Mishra. Listening to Mishra's speech in the live video makes it clear that he did not say that party workers should be shot.

In the live video, Mishra can be heard addressing the protesting party workers from the stage: "If anyone wants to create a disturbance here, I would request them to get up and leave. These people create a disturbance at a meeting like this and call themselves Modi's supporters. They are lying."

Mishra said, "An atmosphere is being created across the country in support of Modi. You are spoiling his speech. Aren't you ashamed? You are patriots. Think about the country, not about who will be the candidate and who will not."

Kalraj Mishra said that a party election meeting was not the right place to express differences. He said, "We felt very bad. If this were our state, I would have come down from the stage and said the same thing. This should not have happened."

This was the BJP's first election meeting in Haryana after the announcement of the Lok Sabha elections. At the meeting, a senior party leader claimed that the BJP would win all 10 Lok Sabha seats in Haryana.

'A fresh example of fooling people'

MP Krishan Pal Gurjar has expressed displeasure that several media organisations reported Kalraj Mishra's statement out of context, relying solely on the Congress spokesperson's tweet. Kalraj Mishra tweeted that Randeep Singh Surjewala's tweet was a "fresh example of the Congress fooling the public". However, the Congress spokesperson has not deleted the tweet despite BJP leaders asking him to do so.

(You can also connect with us by clicking this link.)

(Click here to download BBC Hindi's Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)""",

    "india-48128218": """This became possible because China, Pakistan's strongest supporter and the country that had repeatedly used its veto in this matter, changed its position. Earlier, whenever the UN Security Council discussed declaring Masood Azhar a global terrorist, China used its veto power to prevent it. China had maintained this position for nearly 10 years. During this period, it consistently argued that the matter should not be rushed and that such decisions would be effective only if all members reached a consensus through a process that allowed sufficient time and patience.

India first proposed declaring Masood Azhar a global terrorist at the UN Security Council in 2009, after the 26/11 attacks in Mumbai. The United States, Britain and France, all permanent members of the Security Council with veto power, later introduced similar proposals. Russia, another permanent member, had also indirectly promised to support the proposal.

China stood alone in its support for Masood

Most of the Security Council's non-permanent members also came out in support of the proposal, leaving China isolated. China faced criticism for supporting the Pakistan-based extremist, whose record was widely regarded as deeply troubling. When the Security Council's sanctions committee could not reach a consensus on Masood Azhar, France took the lead and independently declared him a global terrorist. This left China even more isolated.

Following the extremist attack in Pulwama, India, in February that year, France, Britain and the United States again introduced the proposal at the UN sanctions committee's meeting in March. But at the last moment, China once again placed a 'technical hold' on it, putting the issue on hold for another six months. This was the fourth time China had imposed a technical hold and asked the UN Security Council sanctions committee to discuss the matter further.

However, just a few days later, China's foreign minister and its ambassador to India began talking about resolving the issue soon. Within six weeks, China agreed to the proposal.

Also read: France to seize Masood Azhar's assets

But what brought about this change of heart in China?

One possible reason is that, in response to China's repeated technical holds, the Trump administration — which was already engaged in a trade war with China — proposed a debate on the issue in the Security Council the previous month. The United States said the debate should take place at the Security Council table.

The American proposal put China in a difficult position because it would have to state its position publicly. Until then, China had expressed its views in the sanctions committee's closed-door meetings, making it difficult to know exactly what it was saying about the matter.

As pressure mounted, Pakistan's Foreign Minister Shah Mahmood Qureshi began claiming that Masood Azhar was seriously ill and should not be taken to court unless concrete evidence against him was available. This may also have led China and Pakistan — particularly Pakistan's intelligence agency, the ISI, and its military — to wonder whether Masood Azhar was no longer useful to them and had become a burden rather than an instrument to be used against India.

It is also interesting that China made this decision just one day after Pakistan's Prime Minister Imran Khan returned to Beijing from the Belt and Road Summit. During his visit, he had met several Chinese leaders, including President Xi Jinping.

Also read: What is China's interest in protecting Masood Azhar?

It is clear that Masood Azhar must have been discussed during these meetings. Although China informed the UN Security Council of its decision, its foreign ministry spokesperson did not specify a timeline, apparently to minimise the damage to Pakistan. Repeating China's official position, the spokesperson said that an appropriate solution to the issue should be found as soon as possible.

Adding to China's concerns, the Paris-based Financial Action Task Force (FATF) had tightened its scrutiny of Pakistan over the previous year, placing it on its 'grey list' and warning that it could be moved to the 'blacklist' by June. Such a move could have serious consequences for Pakistan. The FATF accused Pakistan of failing to stop various extremist organisations from receiving financial support on its territory.

China was then the FATF's vice-president and was due to become its president in October. China may also have taken this into account when making its decision.

India's recent diplomatic efforts also deserve recognition, particularly Prime Minister Narendra Modi's campaign against terrorism. During active bilateral and multilateral discussions with all major powers, including China, Modi had repeatedly raised the issue strongly.

Also read: Who is Masood Azhar's brother who was detained by Pakistan?

There is also speculation that an agreement in principle had been reached during the meeting between Prime Minister Narendra Modi and President Xi Jinping at the Wuhan informal summit in April of the previous year.

The timing of China's announcement has also prompted speculation that the ruling BJP could use the UN Security Council's decision to declare Masood Azhar a global terrorist to its advantage in India's general election. It could present the decision as a major foreign-policy victory against China and Pakistan.

However, a closer look at the reality of the matter shows that declaring Masood Azhar a global terrorist has little more than symbolic significance. The case of Hafiz Saeed is the most suitable example.

Also read: What role did Ajit Doval play in Masood Azhar's release?

What will happen to Masood Azhar?

After the 26/11 terrorist attacks in Mumbai, the Obama administration announced in 2012 a reward of $10 million for anyone who could provide concrete evidence about Hafiz Saeed that would help bring him to justice. Despite the widespread poverty in Pakistan, no one came forward to claim the reward.

At most, the Pakistani government placed Hafiz Saeed under house arrest several times, while courts acquitted him in all cases. Hafiz Saeed has not only continued to move around freely in Pakistan but has also formed a political party there. In the elections held in July of the previous year, the party fielded 80 candidates for the National Assembly and 185 candidates for provincial assemblies.

Masood Azhar's fate is unlikely to be any different.

(Click here to download BBC Hindi's Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)""",

    "india-48250857": """The latest example in this series is an article in the American magazine TIME, which asks: "Can the world's largest democracy endure another five years of a Modi government?" The American magazine tweeted this question alongside the cover of its issue scheduled to be released in the Indian market on 20 May 2019.

On the cover of the article published on TIME magazine's website, Modi is described as 'India's Divider in Chief', a phrase that has sparked considerable controversy. It is worth noting that four years earlier, in its May 2015 issue, TIME magazine had also published a cover story on Indian Prime Minister Modi, titled 'Why Modi Matters'.

Similarly, an article published in Forbes magazine on 16 March 2019 said that "Modi has raised India's standing at home and abroad, but his style of governance could cost him his position." Among other things, the article also stated that "Modi's policies have failed to reach ordinary people, and even the average Indian has been worse off during Modi's tenure."

'Modi's India is rising'

Two years before its criticism of PM Modi in 2019, Forbes had published an article on 18 November 2017 titled 'Modi's India Is Rising'. It said, "Prime Minister Modi is taking India forward on the global economic stage. India's ranking is improving, and Modi has introduced structural reforms."

However, The Economist went so far as to say in its report that Modi had missed his opportunity. The report said that Modi had not introduced the kind of economic reforms that had been expected. It described the Indian prime minister as more of an administrator than a reformer.

Similarly, the Washington Post reported in January that year that "Modi has failed to improve India's economy. He has been unable to take the Indian economy beyond a growth rate of 7 per cent, and demonetisation did not achieve the objectives for which it was introduced."

Indian media versus foreign media

Some experts believe that foreign media form their opinions under the influence of the so-called intellectual class and the Indian English-language media.

S. Venkat Narayan, president of the Foreign Correspondents Club (FCC) and a senior journalist, says, "The point is that foreign media and their correspondents depend to a great extent on English-language newspapers published in Delhi for their information. Only a few of them travel to different parts of India to gather information on the ground. Most other correspondents are influenced by India's so-called intellectual class, which gets space in the English-language media. The same thing happened in the case of Indira Gandhi."

Senior journalist S. Venkat Narayan believes that firm conclusions should not be drawn on the basis of headlines alone. Referring to TIME magazine's widely discussed recent issue, he says, "If you go beyond the headline and read the cover story, you will find that it also says the Congress is completely ineffective and that all it has managed to do is bring in Priyanka, Rahul Gandhi's sister, to help him. It also says that the opposition is so weak that it will not be able to do much damage to Modi."

'Modi has no control'

Senior journalist Hartosh Bal believes that international media differ from Indian media in one important respect: Modi does not have control over them.

Hartosh Bal says, "We have a misconception that there is a huge disconnect between foreign media and Indian media. But that is not the case. A major change is taking place these days. Most of the opinion pieces appearing in foreign media are being written by Indians or people of Indian origin who work in or are associated with Indian media. Therefore, what you see in foreign media also reflects Indian media."

Hartosh Bal draws attention to another issue: whether the media is under government control. He says, "Modi does not control the media outside India. You can see the impact of this in foreign media. There are some people who are carrying out genuinely critical analyses based on ground realities and concrete facts."

An example of Hartosh Bal's argument can be seen in the Washington Post on 11 May, in which Barkha Dutt writes, "This election is entirely about Modi. Modi's re-election will determine India's future."

However, both Hartosh Bal and S. Venkat Narayan believe that such articles in foreign media do not make much difference to Prime Minister Modi. They believe that no world leader or diplomat forms their opinion on the basis of a magazine.

The timing of the election

Trupti Nath, a senior journalist based in Delhi who has contributed to the Sunday Times of London, believes that reports which hold the government up to scrutiny tend to appear more frequently during elections. She recalls that in 2012, the Washington Post described then Prime Minister Manmohan Singh as a 'tragic figure'.

Trupti Nath says, "I would not say that foreign correspondents lack an understanding of what is happening on the ground, because not all journalists are alike. However, it is also not possible to visit every state, so monitoring the news plays an important role."

Trupti Nath also rejects the claim that articles in magazines such as TIME make no difference to a government. She says, "They affect the image of the government and the image of the country. This is a matter of credibility. Every word published in TIME, the Sunday Times of London, the New York Times and the Washington Post is taken seriously. These are not minor newspapers. It is difficult for any government to control foreign journalists. At most, a government can refuse to grant them visas."

(Click here to download BBC Hindi's Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)"""
}

batch20_accepted_ids = []

In [55]:
batch20_corrections.update({
    "india-48662333": """Semabhai's daughter, Harmiben. Semabhai says that his maize crop has completely dried up. He owns two bullocks, but instead of using them to plough his fields, he is using them to dig a well. His wife and daughter are helping him with the work. Last year, his crop dried up, and this year his well has also run dry. This is the situation faced by most farmers in the Amirgarh block of Palanpur district in northern Gujarat. Their fields are dry, and there is no water to irrigate their crops.

No water for people or animals

Semabhai lives in the small village of Upala Khapa. He owns land but has no water for farming. He has been forced to work as a daily-wage labourer. He explains, "There is no water in my well, so I am now digging a deeper well. Earlier, there was a little water in it, but this time the drought has dried it up completely. There isn't even enough water to drink."

Amirgarh is just 40 kilometres from the district headquarters, Palanpur. Reaching Upala Khapa involves travelling along a dusty road. It is a small village with a government primary school on its outskirts. As you enter the village, fields stretch out on both sides of the road, but there is no greenery in sight. Some fields have wells, both old and new, but none of them contain water.

Read: Semabhai has taken time off from his daily-wage work to deepen the well. His wife and daughter are helping him, while children play nearby. During last year's monsoon, Semabhai sowed maize, but the crop did not grow. He says, "There was a drought. Nothing grew in our fields. We received no help from the government either. We have no faith in the government, so we are digging the well ourselves."

Semabhai's family has dug down 70–80 feet so far, but there are still no signs of water. The family has no idea how much deeper they will have to dig before finding it.

Children play with stones taken out of the well.

The government stopped the work

Wells and ponds are drying up in other parts of Gujarat too. Local activist Nafisa Barot explains why local water sources are drying up: "In the 1970s, the government worked to improve local water sources. But in the 1990s, its focus shifted to supplying water to areas in need."

Nafisa Barot says that local water resources needed to be maintained, but the government shifted its attention to large-scale water-supply projects. As a result, local water resources were continuously neglected.

Sandeep Sangle, the district collector of Banaskantha, says, "Farmers are carrying out this work on their own. They are not receiving any government assistance for it."

Speaking about the growing water crisis in the district, Sandeep says, "We are supplying water through pumps and tankers. In the coming days, we will develop new plans to address the water crisis in Amirgarh and Danta. We are conducting a survey for this purpose."

Semabhai and his wife have been forced to become daily-wage labourers

Semabhai owns farmland but has no water to cultivate it. As a result, he and his family have been forced to work as daily-wage labourers.

Semabhai's daughter Harmiben says, "I earn Rs 250 by working as a daily-wage labourer for a whole day. I use that money to look after myself and my child. My husband has left me. I live at my father's house, but I do not want to become a burden. Now that we have to dig the well, I cannot go out to work."

Harmiben has three children.

Semabhai explains, "We have land, but we cannot grow crops. So we work as daily-wage labourers. During the monsoon, we work as sharecroppers on other people's fields. But even sharecropping is of little use, because if the crop dries up, there is nothing to share."

Semabhai says, "If I can finish digging the well in time and we get good rainfall, I may be able to cultivate my own land this year. Otherwise, I will have to continue working as a daily-wage labourer and as a sharecropper."

Read: Semabhai

Making ends meet is difficult

Semabhai's family depends on the income they earn as daily-wage labourers. However, because they are busy digging the well, they are currently earning nothing from daily-wage work.

Semabhai says, "My sons are going out to work as labourers. If they do not go to work, we will have nothing to eat."

Semabhai's nephew Dhulo says, "Our village has not even received a water tanker. We have received no help from the government. We have no choice but to give up our daily-wage work to dig the well."

Semabhai, Dhulo and other family members are doing this work without any special safety equipment. They are relying only on a rope, which makes the work dangerous. During excavation, they find soil and stones that must be brought out of the well. They use bullocks for this task. A basket is lowered into the well with a rope using a wheel. Once it is filled with soil and stones, the bullocks pull it out.

Semabhai's wife and daughter help him with the work. All four family members are putting considerable effort into digging the well.

Semabhai says, "We are deepening the well because it may fill up if the monsoon is good. But right now, we are waiting for even a little water so that we can have drinking water."

Semabhai's nephew works inside the well.

Digging a well also costs money

Semabhai says, "We are not receiving any help from the government. If a water tanker came, we could have been spared this hardship."

Digging a well also requires money. When hard soil is encountered during excavation, it has to be broken up through blasting. According to Semabhai, each such blast costs at least Rs 3,000. If his family needs to carry out a blast, they will have to stop digging until they can afford it.

The family plans to save money by working as daily-wage labourers. Once they have saved enough for the blast, they will resume digging the well.

Semabhai says, "My four sons work as daily-wage labourers, but we will also have to go out to work to buy the materials needed for blasting. Once all of us earn some money together, we will be able to resume digging the well after a few days."

(Click here to download BBC Hindi's Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)""",

    "india-49221048": """Representational image

Pakistan's Foreign Minister Shah Mahmood Qureshi has tweeted that Indian security forces used cluster munitions, in violation of the Geneva Conventions and international law.

"I condemn the use of cluster munitions by the Indian Army, targeting civilians along the Line of Control. This is a violation of the Geneva Conventions and international law," he wrote.

Qureshi followed this with another tweet in which he described India as a threat to regional peace. He wrote that India, by promoting war hysteria, was not only destabilising regional peace but also violating human rights along the Line of Control.

In the tweet, Foreign Minister Qureshi called on countries around the world to take note of the recent situation along the Line of Control and in Indian-administered Kashmir.

Pakistan Army spokesperson Major General Asif Ghafoor also accused India of violating international agreements. He tweeted, "The use of cluster munitions by the Indian Army is a violation of international agreements and is condemnable. No weapon can suppress the determination of Kashmiris to achieve their right to self-determination. Kashmir runs in the blood of every Pakistani. The freedom struggle of Kashmiris will succeed."

The Indian Army rejected Pakistan's allegations. It said that the Pakistani Army was continuously encouraging militants by facilitating infiltration and providing them with weapons.

The Indian Army's statement further said that it always responds to hostile actions, and that such responses had been directed only at Pakistani Army posts and militant infiltrators receiving support from the Pakistani Army.

What are cluster bombs?

The production and use of cluster munitions are prohibited under the Convention on Cluster Munitions. However, several countries and armed forces have continued to face allegations of using them in wars or armed conflicts.

They are considered dangerous because a single main bomb releases numerous smaller explosive devices, which can cause damage beyond the intended target. This increases the risk of harm to civilians during a conflict.

Moreover, the smaller explosives scattered around the area may remain there long after the main bomb has exploded. As a result, people can continue to suffer loss of life and property even after the conflict has ended.

These weapons are used to kill opposing soldiers or damage their vehicles. They can be dropped from aircraft or launched from the ground.

India and Pakistan have not joined the treaty

In 2008, an international treaty called the Convention on Cluster Munitions was established in Dublin. It proposed prohibiting the possession, sale and use of cluster munitions.

The treaty was opened for signatures on 3 December 2008. By September 2018, 108 countries had signed it, while 106 had formally agreed in principle to adopt it.

However, several countries opposed the treaty, including China, Russia, Israel and the United States, as well as India and Pakistan. Neither India nor Pakistan has signed the treaty.

(Click here to download BBC Hindi's Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)"""
})

batch20_accepted_ids = []

In [56]:
corrections_df = pd.DataFrame(
    list(batch20_corrections.items()),
    columns=["id", "complete_english_translation"]
)

if batch20_accepted_ids:
    reviewed_translations.loc[
        reviewed_translations["id"].isin(batch20_accepted_ids),
        "manual_review_status"
    ] = "Human reviewed - no correction"

for _, row in corrections_df.iterrows():
    reviewed_translations.loc[
        reviewed_translations["id"] == row["id"],
        "complete_english_translation"
    ] = row["complete_english_translation"]

reviewed_translations.loc[
    reviewed_translations["id"].isin(batch20_corrections.keys()),
    "manual_review_status"
] = "Manually corrected"

output_path = (
    f"{base_path}/09_sentence_level_translation/"
    "FactIndic_English_250_Reviewed.csv"
)

reviewed_translations.to_csv(output_path, index=False)

print("Batch 20 saved successfully.")
print("Corrections applied:", len(batch20_corrections))
print("Accepted without correction:", len(batch20_accepted_ids))
print("\nUpdated review status:")
print(reviewed_translations["manual_review_status"].value_counts())
print("\nSaved to:", output_path)

Batch 20 saved successfully.
Corrections applied: 5
Accepted without correction: 0

Updated review status:
manual_review_status
Manually corrected                188
Pending review                     37
Human reviewed - no correction     25
Name: count, dtype: int64

Saved to: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv


In [57]:
pending_ids = reviewed_translations.loc[
    reviewed_translations["manual_review_status"] == "Pending review",
    "id"
].head(10).tolist()

source_lookup = df_250.set_index("id")
translation_lookup = reviewed_translations.set_index("id")

print("Batch 21 — Articles for Human Review")
print("Number of articles:", len(pending_ids))

for i, article_id in enumerate(pending_ids, 1):
    print("\n" + "=" * 100)
    print(f"ARTICLE {i}/10: {article_id}")

    print("\nHINDI ORIGINAL:\n")
    print(source_lookup.loc[article_id, "text"])

    print("\nCURRENT ENGLISH TRANSLATION:\n")
    print(translation_lookup.loc[
        article_id, "complete_english_translation"
    ])

Batch 21 — Articles for Human Review
Number of articles: 10

ARTICLE 1/10: india-45233283

HINDI ORIGINAL:

लेकिन इन लोगों में से एक शख्स हैं- हाल ही में रिजर्व बैंक ऑफ इंडिया (आरबीआई) में अंशकालिक निदेशक बने एस गुरुमूर्ति. एस गुरुमूर्ति अपने ट्वीट से इशारा कर रहे हैं कि ये बाढ़ अगर सबरीमाला के भगवान की नाराज़गी की वजह से है तो सुप्रीम कोर्ट को अपने फ़ैसले पर सोचना चाहिए. उन्होंने एक ट्विटर यूज़र के ट्वीट पर अपनी बात रखी है. उन्होंने लिखा, "सुप्रीम कोर्ट के जजों को देखना चाहिए कि केस और जो सबरीमाला में हुआ, उसके बीच कोई संबंध है या नहीं. अगर कोई संबंध होने का लाखों में एक चांस भी है तो लोगों को अयप्पन के ख़िलाफ़ फ़ैसला पसंद नहीं आएगा." जब कुछ लोगों ने उनके ट्वीट पर इस बात की आलोचना की तो उन्होंने फिर से अपनी बात को दोहराया. "भारत के बुद्धिजीवियों के पाखंड को देखकर हैरान हूं जो लोगों के विश्वास को कचरा समझते हैं. 99 फ़ीसदी भारतीय भगवान में विश्वास करते हैं. 100 फीसदी ज्योतिष में विश्वास रखते हैं जिसमें लिबरल, सेक्युलर, बुद्धिजीवी भी शामिल हैं. नास्तिक करुणानिधि के लिए उनके अनुयायियों ने

In [58]:
batch21_corrections = {
    "india-45233283": """One of the people making such comments is S. Gurumurthy, who recently became a part-time director of the Reserve Bank of India (RBI). In his tweets, Gurumurthy suggested that if the floods were caused by the displeasure of the deity at Sabarimala, the Supreme Court should reconsider its decision in the case. He was responding to a tweet by another Twitter user.

"The Supreme Court judges should examine whether there is any connection between the case and what happened at Sabarimala. Even if there is a one-in-a-million chance of a connection, people will not accept a verdict against Ayyappan," he wrote.

When some people criticised his tweet, he reiterated his position. "I am shocked to see the hypocrisy of India's intellectuals, who treat people's faith as rubbish. Ninety-nine per cent of Indians believe in God. One hundred per cent believe in astrology, including liberals, secularists and intellectuals. The followers of the atheist Karunanidhi prayed for him. I am among those who believe in God, but not in astrology."

**Who is S. Gurumurthy?**

Gurumurthy is the co-convenor of the Swadeshi Jagran Manch and is also regarded as an ideologue of the Rashtriya Swayamsevak Sangh (RSS). Considered close to Prime Minister Narendra Modi, he was appointed a part-time director of the Reserve Bank of India, India's central bank, on 8 August.

When PM Modi announced demonetisation in November 2016, Gurumurthy was widely said to have been the brains behind the decision. After his appointment as a director, Gurumurthy wrote on Twitter: "This is the first time I have accepted such a position. I have never accepted a directorship in the private sector or a public sector undertaking (PSU). I have never conducted an audit for a PSU or a private company either. I wanted to speak independently. But when pressure mounted on me to work in the public interest, I accepted."

**The Sabarimala temple and the ban on women aged 10 to 50**

A Twitter user named Hari Prabhakaran linked Sabarimala to the floods. Along with a photograph of the temple, he wrote: "No law is greater than God. If you allow everyone to enter, He will refuse to let anyone in." The tweet received 3,400 likes and 1,700 retweets.

He also wrote: "If you cannot change your birth or death, why are you changing a centuries-old temple tradition? Sabarimala is our faith." Gurumurthy had responded to this tweet.

Another user, Satish Kumar, suggested that the situation could be viewed the other way around: if women were denied entry, the deity might refuse everyone entry. A user named Gopalakrishnan wrote that people in Kerala believed Lord Ayyappan was angry and that the Supreme Court should not interfere in matters concerning people's faith and traditions.

**How people responded**

Gurumurthy's tweet drew considerable criticism. A user named Manikandan wrote: "Please do not link this to a religious issue. I did not expect this from you. People who are stranded need help. If you can help them, please do so."

Kamalakar Durge wrote: "How can someone like S. Gurumurthy serve on the RBI board? If something goes wrong, will he say it was God's will?"

TV journalist Lata Venkatesh wrote: "It is appalling that people like Gurumurthy are linking the Kerala floods to the entry of women into Sabarimala. People holding public office should not be allowed to make statements that go against the Constitution of India."

A user named Gopi Shankar wrote: "I never thought I would see the day when such an irrational person would sit on the RBI board. If God wanted to punish someone, the floods would have happened in Delhi, where the Supreme Court is. Why Kerala?"

**What is the Sabarimala temple dispute?**

Earlier that month, the Supreme Court began hearing a petition challenging the ban on women's entry into Kerala's Sabarimala temple. Women between the ages of 10 and 50 have traditionally been prohibited from entering the temple. The restriction is based on the belief that women in this age group menstruate.

In its observations, the court repeatedly said that women have the same right as men to enter the temple. The Supreme Court has reserved its judgment in the case.

(You can click here for the BBC Hindi Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)""",

    "india-45961945": """According to the newspaper, CBI Special Director Rakesh Asthana was leading the investigation into Vijay Mallya's case. Asthana himself is accused of accepting bribes in the case involving meat trader Moin Qureshi.

Vijay Mallya is accused of failing to repay loans worth Rs 9,000 crore taken from banks. He left for the UK after the investigation began and has been living in London since March 2016. The Indian government is trying to secure his extradition from the UK.

Mallya could now take advantage of the ongoing conflict within the CBI and argue that he cannot be extradited to India on the basis of an investigation led by someone who is himself an accused. His lawyers are unlikely to hesitate to challenge Rakesh Asthana and the CBI in an effort to prevent his extradition.

Mallya had earlier cited inadequate facilities for prisoners in Indian jails as a reason to oppose his extradition. In response, the Indian government sent a video of the jail to the UK. The extradition case against Vijay Mallya was scheduled to be heard on 10 December.

**The Institutions of Eminence issue resurfaces**

The Indian Express reported on the controversy surrounding the Institutions of Eminence scheme. IIT Madras had written to Human Resource Development Minister Prakash Javadekar expressing deep disappointment that it had not been selected for the scheme, despite being recommended by the University Grants Commission's selection panel.

IIT Madras chairman Pawan Goenka wrote the letter in July, a month after the announcement of the institutions selected for the Institutions of Eminence category.

The Empowered Expert Committee (EEC) had selected 11 institutions out of 114 applicants for this category. IIT Madras was among those selected. Six institutions were then shortlisted on the basis of the QS World University Rankings 2018.

IIT Madras objected to the use of rankings prepared by a private agency as the basis for the selection. It argued that neither the EEC's recommendation nor the government's own National Institutional Ranking Framework (NIRF) had been used as the basis for the decision. The NIRF had ranked IIT Madras as the country's best engineering institution for three consecutive years.

The Institutions of Eminence scheme had earlier attracted controversy when the Jio Institute, which had not yet been established, was included in the category.

**Firecrackers to be allowed for two hours on Diwali**

Hindustan reported on the Supreme Court's order concerning the use of firecrackers. In view of the pollution situation in Delhi, the Supreme Court directed that firecrackers could be burst for only two hours on Diwali, from 8 pm to 10 pm.

The police station in the relevant area would be responsible for ensuring compliance with the order. Any violation would be treated as contempt of court.

The court also said that only green firecrackers could be manufactured and sold for Diwali. Firecrackers that produce sound and smoke within the prescribed limits would qualify as green crackers. They release lower quantities of nitrogen oxides and sulphur dioxide.

The court also said that, wherever possible, people in Delhi and the National Capital Region should burst firecrackers collectively. The order would apply not only to Diwali but also to other religious and social festivals.

**A garbage mountain taller than Qutub Minar**

According to Amar Ujala, the garbage heap at Delhi's Bhalswa landfill site, which covers 70 acres, had grown taller than the Qutub Minar.

The Qutub Minar is 73 metres high. The garbage heap above ground at the landfill is 60 metres high, while waste also extends 12 metres below the ground. The site contains 80 lakh metric tonnes of garbage.

Taking note of a fire at the site on Sunday, 21 October, North Delhi Mayor Adesh Gupta and Commissioner Madhup Vyas inspected the landfill on Tuesday.

Also read: (You can click here for the BBC Hindi Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)""",

    "india-46106302": """Chameli Devi, a resident of Ghaghra village, is among the family members waiting for the return of workers abducted in Afghanistan. The abducted workers include Prakash Mahato and Prasadi Mahato from Ghaghra village in Bagodar block, Hulas Mahato from Mahuri village, and Kali Mahato from Bedam village in Tatijharia block.

All of them worked in Afghanistan for KEC International, a company owned by Indian industrialist Harsh Goenka. Following the intervention of the External Affairs Minister, the company has started paying salaries to the families of the abducted workers. However, the families have been instructed not to speak to the media. The company has not issued an official statement on the incident since the abduction.

KEC International is part of the RPG Group and has a contract to install electricity transmission lines in Afghanistan. Many people from Jharkhand have travelled to Afghanistan for work, most of them from Bagodar.

Prasadi Mahato, the husband of Mulia Devi, was among those abducted.

**Six months since the abduction**

Even six months after the abduction, the governments of India and Afghanistan had not been able to trace the workers. Their families were becoming increasingly desperate.

Mulia Devi has been waiting for her husband to return for months. Their daughter passed her Class 10 examination this year, but Mulia Devi says that she has since stopped studying.

"We are poor. My husband went abroad so that he could earn some money and provide a good education for our children. Now there is no information about him. How are we supposed to survive? Whom can we trust? Who will bring my husband back? We do not even know whether the government is doing anything to secure their release. Please help me reunite with my husband," she said.

**Villagers plan to launch a protest**

Santosh Rajak, who lives in Prasadi Mahato's village, said he was disappointed with the Modi government. He believes that the government has failed to put pressure on Afghanistan.

Hulas Mahato, one of the Indian workers abducted in Afghanistan, is from Mahuri village. His wife, Pramila Devi, is in increasingly poor condition and has not been eating properly because of her distress over her husband's disappearance. She said she had no answer when her children asked about their father.

**The External Affairs Minister's assurance did not help**

Forty days after the abduction, External Affairs Minister Sushma Swaraj met in Delhi with the wives of three of the abducted workers and the son of Prasadi Mahato.

Chameli Devi, the wife of Prakash Mahato from Ghaghra village, was part of the delegation. She told the BBC that the minister had assured them that all the workers would be traced within a month.

Jharkhand Chief Minister Raghubar Das had also met the wives of some of the workers in Ranchi the previous month. He announced financial assistance of Rs 1 lakh for each family, but the money had not yet been received.

After meeting the chief minister, Pramila Devi said he had also promised that her abducted husband would be traced soon, but nothing had come of that assurance.

Sikandar Ali, who runs a migrant workers' group, and Laxman Mahato, the head of Bagodar West, said that workers from the region continued to migrate despite the abductions in Afghanistan. Unemployment remains a major problem in the area, and people are still travelling to Afghanistan in search of work.

Also read: (You can click here for the BBC Hindi Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)"""
}

batch21_accepted_ids = []

In [59]:
batch21_corrections = {
    "india-45233283": """But one of these people is S. Gurumurthy, who recently became a part-time director of the Reserve Bank of India (RBI). In his tweets, Gurumurthy has suggested that if the floods are the result of Lord Ayyappa's displeasure over the Sabarimala issue, the Supreme Court should reconsider its decision. He was responding to a tweet by another Twitter user. "The Supreme Court judges should examine whether there is any connection between the case and what happened at Sabarimala. Even if there is a one-in-a-million chance of a connection, people will not accept a decision against Ayyappa," he wrote.

When some people criticised his tweet, he repeated his argument. "I am surprised by the hypocrisy of India's intellectuals, who treat people's faith as rubbish. Ninety-nine per cent of Indians believe in God. One hundred per cent believe in astrology, including liberals, secularists and intellectuals. Followers even prayed for the atheist Karunanidhi. I am among those who believe in God, but not in astrology."

Who is S. Gurumurthy?

Gurumurthy is the co-convenor of the Swadeshi Jagran Manch and is also considered an ideologue of the Rashtriya Swayamsevak Sangh (RSS). Known to be close to Prime Minister Narendra Modi, he was appointed a part-time director of India's central bank, the RBI, on 8 August. When PM Modi announced demonetisation in November 2016, Gurumurthy was said to have been the person behind the decision.

After his appointment as a director, Gurumurthy wrote on Twitter: "This is the first time I have accepted such a position. I have never accepted a directorship in the private sector or a public sector undertaking (PSU). I have never conducted an audit for a PSU or a private company either. I wanted to speak independently. But when pressure increased for me to work in the public interest, I accepted it."

The Sabarimala temple and the ban on women aged 10 to 50

A Twitter user named Hari Prabhakaran, who links Sabarimala to the floods, posted a photograph of the temple with the message: "No law is greater than God. If you allow everyone to enter, He will refuse to let anyone in." The tweet received 3,400 likes and 1,700 retweets. He also wrote: "If you cannot change your birth or death, why are you changing a centuries-old temple tradition? Sabarimala is our faith."

Gurumurthy had responded to this tweet. Another user, Satish Kumar, wrote that the situation could also be viewed from the opposite perspective: if women were denied entry, God might refuse everyone entry. A user named Gopalakrishnan wrote that people in Kerala believed Lord Ayyappa was angry and that the Supreme Court should not interfere in matters concerning people's beliefs and practices.

People respond to Gurumurthy's tweet

Gurumurthy's tweet attracted considerable criticism. A user named Manikandan wrote: "Please don't connect this to a religious issue. I did not like it. I did not expect this from you. People are stranded and need help. If you can help them, please do so."

Kamalakar Durge wrote: "How can someone like S. Gurumurthy work at the RBI? If something goes wrong, he will say it was God's will."

TV journalist Lata Venkatesh wrote: "It is terrible that people like Gurumurthy are linking the Kerala floods to women's entry into Sabarimala. People holding government positions should not be allowed to make statements that go against the Constitution of India."

A user named Gopi Shankar wrote: "I never thought I would see the day when such an irrational person would sit on the RBI board. If God wanted to punish someone, the floods would have struck Delhi, where the Supreme Court is. Why Kerala?"

What is the controversy over the Sabarimala temple?

Earlier that month, the Supreme Court began hearing a petition challenging the ban on women's entry into Kerala's Sabarimala temple. Women aged between 10 and 50 have traditionally been barred from entering the temple. The restriction is based on the belief that women in this age group menstruate. In its observations, the court repeatedly said that women have the same right as men to enter the temple. The Supreme Court has reserved its judgment in the case.

(You can click here for BBC Hindi's Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)""",

    "india-45961945": """According to the newspaper, CBI Special Director Rakesh Asthana was leading the investigation into Vijay Mallya's case. Asthana is himself accused of accepting a bribe in the case involving meat trader Moin Qureshi.

Vijay Mallya is accused of failing to repay loans worth Rs 9,000 crore taken from banks. He left for Britain after the investigation began. Mallya travelled to the UK in March 2016 and has been living in London ever since. The Indian government is trying to secure his extradition from Britain.

Mallya could now take advantage of the ongoing conflict within the CBI and argue that he cannot be sent to India on the basis of an investigation led by someone who is himself an accused. His lawyers are unlikely to hesitate to challenge Rakesh Asthana and the CBI in an effort to prevent his extradition.

Mallya had earlier cited inadequate facilities for prisoners in Indian jails as a reason to oppose his extradition. In response, the Indian government had sent a video of a jail to Britain. The hearing in Mallya's extradition case is scheduled for 10 December.

The issue of Institutions of Eminence resurfaces

The Indian Express has reported on the Institutions of Eminence issue. IIT Madras wrote to Human Resource Development Minister Prakash Javadekar expressing deep disappointment that it had not been selected as an Institution of Eminence, despite being recommended by the University Grants Commission's selection panel.

IIT Madras chairman Pawan Goenka wrote the letter in July, a month after the announcement of the institutions selected for the Institutions of Eminence category. The Empowered Expert Committee (EEC) had selected 11 institutions out of 114 for this category. IIT Madras was among them. Six institutions were finally selected on the basis of the QS World University Rankings 2018.

IIT Madras objected to the use of rankings by a private agency as the basis for selection. It argued that neither the EEC's recommendations nor the government's own National Institutional Ranking Framework (NIRF) had been used as the basis for the selection. The NIRF had ranked IIT Madras as the country's best engineering institution three consecutive times.

The Institutions of Eminence issue had earlier attracted attention when the Jio Institute was included in the category even though it did not yet exist.

Firecrackers allowed for two hours on Diwali

Hindustan has reported on the Supreme Court's order concerning firecrackers. In view of the pollution situation in Delhi, the Supreme Court has ordered that firecrackers may be burst for only two hours on Diwali, from 8 pm to 10 pm.

The responsibility for ensuring compliance with the order will lie with the police station in the relevant area. Any violation of the court's order will be treated as contempt of court.

The court also said that only green firecrackers could be manufactured and sold for Diwali. It classified firecrackers that produce sound and smoke within the prescribed limits as green firecrackers. These release lower quantities of nitrogen oxides and sulphur dioxide. The court also said that, wherever possible, people should burst firecrackers collectively in Delhi and the National Capital Region (NCR).

The order will apply not only to Diwali but also to other religious and social festivals.

Representational image

Garbage mountain taller than Qutub Minar

According to Amar Ujala, the garbage mound at Delhi's Bhalswa landfill site, which covers 70 acres, has become taller than the Qutub Minar. The Qutub Minar is 73 metres high. The garbage mound rises 60 metres above the ground, while waste also extends 12 metres below ground level. The site contains 80 lakh metric tonnes of garbage.

Taking note of the fire that broke out at the site on Sunday, 21 October, North Delhi Mayor Adesh Gupta and Commissioner Madhup Vyas inspected it on Tuesday.

Also read:
(You can click here for BBC Hindi's Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)""",

    "india-46106302": """Among them are Prakash Mahato and Prasadi Mahato, residents of Ghaghra village in Bagodar block; Hulas Mahato, from Mahuri village; and Kali Mahato, from Bedam village in Tatijharia block. All of them worked in Afghanistan for KEC International, a company owned by Indian industrialist Harsh Goenka.

Following the intervention of the External Affairs Minister, the company has started paying salaries to the families of the abducted workers. However, the families have been instructed not to speak to the media. Since the abduction, the company has not issued an official statement on the incident.

KEC International is part of the RP Goenka (RPG) Group. It has a contract to install electricity transmission lines in Afghanistan. Several people from Jharkhand have travelled to Afghanistan for work, most of them from Bagodar. Prasadi Mahato, the husband of Mulia Devi, was also among those working there.

Six months after the abduction

Even six months after the abduction, the governments of India and Afghanistan have been unable to trace the workers. Their families are therefore losing hope.

Mulia Devi is Prasadi Mahato's wife and has been waiting for her husband to return for months. Their daughter passed her Class 10 examination this year, but Mulia Devi says that she has now stopped studying.

"We are poor. My husband went abroad so that he could earn some money and provide a proper education for our children. Now there is no information about him. How are we supposed to survive? Whom can we trust? Who will bring my husband back? We do not even know whether the government is doing anything to secure their release. Please help me reunite with my husband," she said.

Villagers threaten to launch a protest

Santosh Rajak, who is from Prasadi Mahato's village, is disappointed with the Modi government over the issue. He says the government has been unable to put pressure on Afghanistan.

Hulas Mahato, from Mahuri village, is another of the Indian workers abducted in Afghanistan. His wife, Pramila Devi, is becoming increasingly distressed. She is so upset about her husband that she is not eating or drinking properly. She said that when her children ask about their father, she has no answer for them.

The External Affairs Minister's assurance did not help

Forty days after the abduction, External Affairs Minister Sushma Swaraj met in Delhi with the wives of three of the abducted workers and the son of Prasadi Mahato. Chameli Devi, the wife of Prakash Mahato from Ghaghra village, was part of the delegation. She told the BBC that the minister had promised to trace all the workers within a month.

Jharkhand Chief Minister Raghubar Das also met some of the workers' wives in Ranchi last month. He announced financial assistance of Rs 1 lakh for each of them, but the money has not yet been received.

After meeting the chief minister, Pramila Devi said that he had also promised to locate her abducted husband soon, but nothing had come of that assurance.

Sikandar Ali, who runs a migrant workers' group, and Lakshman Mahato, the head of Bagodar West, said that the migration of workers from the area to Afghanistan had not stopped despite the abductions. Unemployment is a major problem in the region, and people are still travelling to Afghanistan for work.

Also read:
(You can click here for BBC Hindi's Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)"""
}

batch21_accepted_ids = []

In [60]:
batch21_corrections.update({
    "india-46856839": """The answer was, "It's not a book, it's the Constitution, my friend!" The person who had asked the question then took the copy of the Indian Constitution in his hands and touched it to his forehead.

The occasion was a Dalit community rally held at Ramlila Maidan in New Delhi on Sunday. The main purpose of the rally was to honour the families of the 13 people killed during the Bharat Bandh on 2 April the previous year, as well as those who had been jailed during the protests. The deceased were described as "martyrs of the Bahujan community", and their statues were placed on the stage.

The BJP's national convention had ended at the same venue just a day earlier. The BJP's tent and large pictures of Prime Minister Narendra Modi could still be seen behind the tents set up for the rally.

Statues of young people killed during the 2 April protests were placed on the stage.

The protests had been called by Dalit organisations following the Supreme Court's decision to amend the SC/ST Act. Violence broke out during the demonstrations, and 13 people were killed in different places. Sensing the anger among Dalits after the protests, the central government restored the SC/ST Act to its original form.

"Behenji for Prime Minister"

The event took place a day after the Samajwadi Party (SP) and Bahujan Samaj Party (BSP) announced their alliance in Uttar Pradesh. People had travelled to the rally from Madhya Pradesh, Uttar Pradesh, Delhi, Haryana and Rajasthan. The gathering in the capital was therefore also an opportunity to understand Dalit people's views on recent political developments.

Many people at the rally appeared excited about Mayawati and Akhilesh Yadav coming together. Rohit Kanojia, who had travelled from Ashok Vihar in Delhi, said: "This alliance has been formed in Uttar Pradesh, but its message will reach all of India. It will produce a good result across the country, and Behenji will become the Prime Minister."

Mayawati is known as "Behenji" within her party.

Rahul Kanojia, a first-year BA student from Allahabad, said that everyone who had come to the rally wanted to see Behenji become Prime Minister and that an alliance with the SP would make that easier.

The event was jointly organised by the Satyashodhak Sangh and Bahujan Youth for Mission 2019. The first name among the organisers was former BSP vice-president Jayaprakash. Last year, he had made controversial remarks about Congress president Rahul Gandhi, after which Mayawati expelled him from the party.

The event is also being viewed as an attempt by Jayaprakash to return to the BSP. However, Sanjeev Mathur of the co-organiser Satyashodhak Sangh said: "That may be Jayaprakash ji's agenda, and we are not opposed to that agenda. We have no problem with it. But it is not our objective. Even so, we are working together."

There were no pictures of Mayawati at the rally, but several speakers on the stage expressed their wish to see her become Prime Minister. A song was also played from the stage with the words: "Do you know, O Bahujan, the BSP is coming?"

Who supports whom, and who opposes whom?

A young man encouraging the crowd had painted his body blue, with the letters BSP written on it. However, the Satyashodhak Sangh says it does not share this political position.

Sanjeev Mathur said: "Bahujan Youth for Mission 2019 openly supports the BSP. We do not agree with that. Even so, we are working together. We are not working to benefit or harm any political party. But when a social process takes place, it will inevitably have some political consequences. Who benefits from it is not under our control. If the BSP benefits from it as a Bahujan party in Uttar Pradesh, we have no objection."

Uttar Pradesh is considered important in Indian politics because it has the country's largest number of Lok Sabha seats, with 80. When Anil Kumar, who had travelled from Kanpur, began speaking, his first words were: "The mood of the Bahujan community in the country is changing, and it is strengthening its power."

Experts estimate that Dalits make up around 20 per cent of Uttar Pradesh's voters, most of whom are generally considered to support the BSP. However, election analysts point out that in the 2014 election, a section of Dalit voters shifted their support to the BJP led by Narendra Modi.

Anil Kumar said: "That is exactly what happened in 2014. People thought that Modi would do something for the Bahujan community after coming to power. But he broke the community's trust. People now understand that he only makes empty promises. He has worked only for the Manuwadi members of the BJP. They have built their own offices, and the same people occupy positions of power and government jobs."

He added: "We have experienced this ourselves. We have seen exploitation. We have suffered a great deal, and there is a lot of pain within us. These Manuwadis will have to face the consequences in the coming time."

The impact of reservations for upper castes

Recently, the central government announced 10 per cent reservation for economically weaker people from upper castes. The government said that the reservation limit had been increased through a constitutional amendment so that no existing group's quota would be reduced.

There is therefore considerable interest in whether the reservation for upper castes will also provoke a political response from Dalits.

Rahul Kanojia from Allahabad said: "How have upper-caste people become poor? Someone who earns five lakh rupees is considered rich and has to pay taxes, while someone who earns eight lakh rupees will be considered poor and receive reservation."

However, Rakesh Jatav from Hapur is satisfied that people will now stop making fun of reservation. He said: "Those who used to mock us by calling us doctors and engineers who got their positions through reservation are now standing in the queue for reservation themselves. We think this is good. Our leader Mayawati was the first to raise this demand. It is because of pressure from the Bahujan community that upper-caste people have received this reservation."

Abhishek Bauddh from Sikandrabad believes that reservation for upper castes will harm the BJP. He said: "This will definitely work against the BJP. Upper-caste voters already vote for it. But at one time, people considered Narendra Modi a leader of the backward communities. Now he has openly shown that he is the leader of the Brahmanical groups."

The BJP's strong performance in Uttar Pradesh in the past has often been attributed to a combination of upper-caste, non-Yadav OBC and some Dalit votes. All eyes are now on whether Dalit voters are completely breaking away from the BJP and returning to their traditional political home.

(You can click here for BBC Hindi's Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)""",

    "india-47148086": """However, Surendra Jain, the Vishwa Hindu Parishad's international joint general secretary, repeatedly stressed that the movement had not been suspended. Rather, the decision was to refrain from holding it for the next four months so that it would not become politicised. The organisation also believes that it is postponing the movement to prevent the "secular fraternity from getting an opportunity to drag this sacred movement into the political mire". This was also mentioned in the resolution on Ram Janmabhoomi passed at the Kumbh Dharma Sansad on 1 February.

But several other things happened at the Kumbh. The Akhara Parishad, an organisation comprising 13 major akharas, decided to boycott the VHP's Dharma Sansad. According to reports, it remained firm on the boycott despite efforts by Uttar Pradesh Chief Minister Yogi Adityanath to persuade it to reconsider.

Akhara Parishad chief Acharya Narendra Giri not only questioned the VHP's role in the Ram temple issue but also opposed the central government's move to hand over undisputed land to a trust. The trust is controlled by the VHP.

The political benefits of the Ram temple issue

According to a report in the English-language newspaper The Times of India, Narendra Giri said: "The Centre remained silent for four years. Now it has filed a petition in the Supreme Court and alleged that the Ram temple issue is being used for political gain."

Senior journalist Dhirendra Jha, co-author of the book Ayodhya: The Dark Knight, said: "When the RSS and VHP began demanding a law in the name of removing obstacles to the construction of the Ram temple, they did not expect that the situation would turn against them."

RSS chief Mohan Bhagwat has recently been saying that Hindus are losing patience because of delays in the court proceedings. He has called for an ordinance or a law to ensure that the Ram temple is built as soon as possible.

Will the BJP build the temple?

The VHP has been organising a series of programmes related to the Ram temple since last year. However, many people have repeatedly questioned why the BJP has not taken any major step towards building the temple in four years, despite promising to do so if it came to power with a full majority.

Dhirendra Jha said: "The RSS and VHP now seem to think that it will be difficult to gain political advantage from the Ram temple issue. So, at the very least, they want to prevent the BJP from suffering losses. It is under this strategy that they have taken a step forward and then pulled back."

What does the Sangh want?

Political analyst Jagdish Upasane, who closely follows Hindutva organisations, believes that there is no strategy behind this and that the Sangh wants the entire matter to be decided through the courts.

He refers to a statement by RSS general secretary Suresh Bhaiyyaji Joshi, who said that the Ram temple would be built by 2025.

Jagdish Upasane also says that differences of opinion among religious leaders are nothing new. Such differences have existed before, including since the 1990s, when the temple movement was gaining momentum.

The VHP also says that some religious leaders may have been absent from the Kumbh programme, but a large number of saints and akharas attended the Dharma Sansad and supported its resolutions.

Surendra Jain insists that the VHP is not concerned with whether the BJP gains or loses politically from the issue. However, Mohan Bhagwat's speech makes it clearer how the matter was moved onto the back foot.

Speaking at the Dharma Sansad, RSS chief Mohan Bhagwat said: "...We will have to ensure that they do not face difficulties because of our movement, and that they receive our support."

Who "they" refers to is clear to everyone.

Also read:
(You can click here for BBC Hindi's Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)"""
})

print("Batch 21 corrections currently:", len(batch21_corrections))

Batch 21 corrections currently: 5


In [61]:
batch21_corrections.update({

    "india-49659647": """On Wednesday, former BJP MLA Kuldeep Singh Sengar, an accused in the case, was brought from Tihar Jail to AIIMS. The statement of the victim, who is admitted to AIIMS, will be recorded before a judge. The victim was seriously injured in a road accident in Rae Bareli on July 28. Two of her relatives were killed in the accident, and her lawyer was also seriously injured. The victim's family described the accident as a conspiracy and accused Kuldeep Sengar of being involved. Following this, a murder case was also registered against Sengar, who had been expelled from the BJP. The Delhi High Court had allowed the trial court to conduct proceedings at the AIIMS Trauma Centre.

Unnao rape case: What has happened so far?

The case involves rape and the POCSO Act. Cases had already been registered against Kuldeep Sengar under provisions of the POCSO Act and the Indian Penal Code following an order by a Delhi sessions court. The court charged Sengar under several IPC sections, including rape (Section 376(1)) and criminal conspiracy (Section 120B). A case was also registered under Sections 3 and 4 of the POCSO Act.

On August 1, the Supreme Court ordered that all the cases related to this matter be transferred from Uttar Pradesh to a CBI court in Delhi. A bench headed by Chief Justice Ranjan Gogoi said that a judge appointed to hear the case would conduct daily hearings in Delhi and complete the trial within 45 days. On August 5, the Supreme Court ordered that the victim be brought to Delhi for better medical treatment. The victim's father had already died in police custody. The victim had been travelling in the same car with her relatives and lawyer when it was hit by a truck.

What are the allegations against Kuldeep Sengar?

Kuldeep Singh Sengar is accused of raping the girl in June 2017. The girl alleged that MLA Kuldeep Sengar raped her at his house when she had gone there with a relative to ask for a job. At the time, the police did not register an FIR on the victim's complaint, following which her family approached the court.

The mother of the Unnao rape case victim has described the difficulties faced by the family. They allege that the MLA and his associates repeatedly pressured them not to complain to the police. In this context, the MLA's brother also assaulted the victim's father on April 3. The girl's father later died in custody. A video of him recorded before his death went viral. In it, he alleged that the MLA's brother and several other people had beaten him in the presence of the police.

Frustrated by the alleged police inaction and the MLA's alleged intimidation, the victim attempted to set herself on fire outside the Chief Minister's residence. The incident brought the case into the headlines.

Unnao rape case: What has happened so far?

June 4, 2017: The victim accused MLA Kuldeep Sengar of raping her at his house.
June 11, 2017: The victim went missing, and a missing-person report was filed.
June 22, 2017: The victim appeared in court, where her statement was recorded.
April 4, 2018: The victim's father was arrested under the Arms Act.
April 9, 2018: The victim's father died in police custody.
April 11, 2018: The Uttar Pradesh government handed the case over to the CBI.
April 13, 2018: The MLA was arrested and questioned by the CBI.
July 11, 2018: Sengar was named as an accused, and the CBI filed its first charge sheet.
July 28, 2019: The victim's car was hit by a truck.
July 31, 2019: The investigation was handed over to the CBI.
August 1, 2019: The Supreme Court ordered that all the cases be transferred to Delhi.
August 2, 2019: The Supreme Court ordered that the victim's uncle be shifted to Tihar Jail in Delhi for security reasons.

(You can click here for BBC Hindi's Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)""",

    "india-50975864": """In an exclusive interview with the television channel ABP News, Amit Shah said, "Those who are asking questions should spend a day wearing a police uniform. Nobody asks why the bus was set on fire or why the cars were burned. People were forced off buses, and then the buses were set on fire. When people resort to violence, the police will have to open fire."

Amit Shah said that the police have to protect their own lives as well as the lives of others. He asked why people were not questioning the burning of buses, adding that if the buses had not been set on fire, the police would not have had to use batons.

Responding to allegations that the Popular Front of India (PFI) was involved in violence in several states, Shah said that this was not a claim made by a politician but was based on state police reports. On whether the PFI would be banned, he said that as the country's Home Minister, he would not comment on banning any organisation before a decision was made.

Why did violence occur in BJP-ruled states?

Asked why more violence had occurred in BJP-ruled states, Shah said, "Tell me why riots are not taking place in Congress-ruled states? This question should also be asked. People understand who is instigating the violence. Why has there been no violence in states governed by Congress? A misconception has been spread that the CAA will take away the citizenship of the country's minorities. But the opposition should read the Citizenship Amendment Act and point out where it contains any provision for taking away anyone's citizenship."

Supporting the NRC, CAA and NPR, he said that his doors were always open to people who were confused about these issues and wanted to understand them. He said they could even come and meet him at three in the morning. He challenged Rahul Gandhi and Priyanka Gandhi to prove that the citizenship of poor people or Muslims would be taken away.

Shah said, "I would tell Rahul Gandhi and Priyanka Gandhi that you are not reading the law and are frightening people by telling them that they will lose their citizenship. We did not expect the opposition to mislead the country so much."

Protests took place in several parts of the country after Parliament passed the Citizenship Amendment Act. Many of these protests turned violent. There was considerable violence in several parts of Uttar Pradesh in particular. Nineteen people were killed in Uttar Pradesh alone. Questions were also raised about police action during the protests. However, the Home Minister questioned the protests, saying that those taking part in them were misguided and that the protests were largely political.

Without discussing the NRC in detail, he said, "I am only saying that the NRC is not being introduced right now. For the time being, let us discuss the CAA. No Indian is going to lose their citizenship."

Amit Shah said that the people of the country knew that the citizenship law was the right decision. Clarifying the government's position on the NPR, the Home Minister said that no documents would be requested from anyone during the census or the NPR.

However, Home Minister Amit Shah had repeatedly said in Parliament and at press conferences that the Citizenship Amendment Act (CAA) would come first, followed by the NRC. He had said that the process would begin after the CAA.

After protests against the CAA and NRC across the country, Prime Minister Narendra Modi said at an election rally at Delhi's Ramlila Maidan on December 22 that there had been no discussion about the NRC under his government. Following Modi's statement, Home Minister Amit Shah also changed his position and said that there was currently no discussion about the NRC.

What else did Amit Shah say?

What is the Citizenship Amendment Act, 2019?

The central government got the Citizenship Amendment Bill passed by both houses of Parliament in December 2019. After the President signed it, the bill became law. However, protests have continued to grow since its passage, and several people have died in violence associated with the protests.

Those opposing the law argue that it discriminates on the basis of religion and goes against the Constitution's fundamental principles. The law provides a path to Indian citizenship for people belonging to six minority communities—Hindus, Buddhists, Jains, Parsis, Christians and Sikhs—from Bangladesh, Afghanistan and Pakistan.

Before this amendment, under the Citizenship Act of 1955, people who entered India illegally could not obtain citizenship and could be sent back to their countries or detained. Under the earlier law, a person was required to have lived in India for at least 11 years to obtain Indian citizenship. The amended law reduced this period from 11 years to six years for minorities from neighbouring countries.

(You can click here for BBC Hindi's Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)""",

    "india-51453120": """US President Donald Trump will be accompanied by his wife, Melania Trump. During his visit to India, Trump will travel to Delhi and Ahmedabad. A statement issued by the US government said that President Trump and Prime Minister Narendra Modi had spoken by phone over the previous weekend. During the conversation, the two leaders agreed to further strengthen the strategic partnership between the two countries, highlighting the strong ties between the people of India and the United States.

China pleased with Modi's letter to Xi Jinping

China has praised the letter sent by Indian Prime Minister Narendra Modi to Chinese President Xi Jinping regarding the situation caused by the coronavirus. China said that the letter fully reflected the friendship between India and China.

In his letter to Xi Jinping, Modi thanked the Chinese administration for its cooperation in evacuating Indians from Wuhan and expressed condolences over the damage caused by the virus. The Indian Prime Minister also offered India's help to China in dealing with the situation.

Chinese Foreign Ministry spokesperson Geng Shuang said, "We appreciate and thank India for its offer to support China in dealing with the coronavirus. India's goodwill gesture reflects its friendship with China."

Shuang made these remarks while answering a question about Modi's letter during an online briefing.

Training begins for Gaganyaan astronauts

The astronauts selected for India's ambitious Gaganyaan space mission have begun their training. All four are Indian Air Force pilots and are undergoing training in Russia.

The four pilots left for Russia on Friday. Selected through a screening process, they will receive training at the Gagarin Research and Cosmonaut Training Centre (GCTC). On Monday, the Russian agency Glavkosmos issued a statement saying, "The GCTC has started the training programme for the Indians. This is being carried out under an agreement between Glavkosmos and ISRO's Human Spaceflight Centre."

The pilots will undergo physical exercises and biomedical training. They will also experience zero gravity, or weightlessness in space. Gaganyaan, India's first human spaceflight mission, aims to send astronauts into space and bring them back. The mission was targeted for launch by 2022.

Free electricity in West Bengal ahead of elections

The Mamata Banerjee government in West Bengal presented its budget on Monday and announced free electricity for some consumers. Assembly elections are due in the state next year. The opposition has described the budget presented by the Trinamool Congress government as populist.

Along with several other announcements, West Bengal Finance Minister Amit Mitra announced that electricity bills would be waived for households that use up to 75 units of electricity per quarter. This is the last full budget of the Mamata government before the upcoming assembly elections. A budget of 2,55,677 crore rupees was presented, showing a deficit of only eight crore rupees.

Seven questions to determine legal issues related to religious faith

The Supreme Court has decided to use seven questions as the basis for determining the right to religious freedom and the rights of different religious denominations. On Monday, a nine-judge bench headed by Chief Justice S. A. Bobde framed seven questions concerning religious freedom and matters of faith under the Constitution.

Hearings based on these questions will take place every day from February 17 and will continue for ten days. The court said that its five-judge bench could refer legal questions to a larger bench while exercising its limited power of review in the Sabarimala case.

On November 14 last year, while considering review petitions against the 2018 verdict allowing women to enter the Sabarimala temple, the five-judge bench had also included certain practices from other religions in the proceedings.

The bench said that, as in the Sabarimala temple case, other issues also needed to be addressed. These included Muslim women's entry into dargahs and mosques, the practice of female circumcision in the Dawoodi Bohra community, and the denial of entry into Parsi temples to Parsi women who marry non-Parsi men.

(You can click here for BBC Hindi's Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)"""
})

print("Batch 21 corrections currently:", len(batch21_corrections))

Batch 21 corrections currently: 8


In [62]:
batch21_corrections.update({
    "india-51702290": """Some activists raised the slogan while going to attend Amit Shah's rally at Kalighat. The Trinamool Congress, Congress and CPI(M) criticised the sloganeering. The BJP, however, defended it, calling it a conspiracy by the Trinamool Congress.

Police said an FIR was registered late on Sunday night against some unidentified people at the New Market police station in Kolkata. Joint Commissioner of Police (Crime) Murlidhar Sharma said, "After the FIR was registered last night, we examined CCTV footage and identified the accused. On this basis, Surendra Kumar Tiwari (55), Pankaj Prasad and Dhruv Basu (71) were arrested in the early hours of Monday. They will be produced in court today."

Protests against Amit Shah's visit

On Sunday, the Left parties and the Congress protested against Shah's visit. They showed him black flags and raised slogans of "Shah, go back". Effigies of Shah were also burnt at several places in Kolkata. During this time, BJP supporters on their way to attend the rally clashed with supporters of the two parties on Jawaharlal Nehru Road in the Dharmatala area. BJP workers allegedly raised the slogan "Shoot them" during the clash.

After a video of the sloganeering went viral on social media, the government directed the police to investigate. The police then began searching for those involved.

What did Mamata Banerjee say?

West Bengal Chief Minister Mamata Banerjee also condemned the "shoot them" slogan raised during Shah's visit. She said, "This slogan is inflammatory, provocative and illegal. This is Kolkata, not Delhi. This is Bengal. If we let even one person go, it will encourage others."

Targeting Amit Shah, Mamata said that those who spoke about the law and order situation in Bengal should first answer why BJP leaders accused of inciting violence in Delhi had not yet been arrested. "Here, the 'shoot them' slogan was raised yesterday, and today the police have arrested three people."

The Chief Minister said that ordinary people would decide who was a traitor. "Who are you to decide that?" she asked. She said the police were searching for the other accused. The Trinamool Congress chief asked party workers to help identify the accused using video footage and inform the police. She also told them not to take the law into their own hands.

Sharp political reactions

The Trinamool Congress and other parties criticised the BJP for the slogans raised in Kolkata. Senior Trinamool Congress leader Firhad Hakim said, "People's hearts cannot be won in Bengal by raising the slogan 'Shoot them'."

State Congress president Somen Mitra said, "This is the BJP's culture. But why did the police not arrest those who raised the slogans at the time?"

CPI(M) Politburo member Mohammed Salim said, "The BJP organised Sunday's rally with the intention of spreading communal tension. No one had ever dared to raise such a slogan before."

Several intellectuals also criticised the BJP over the slogan. Well-known actor Soumitra Chatterjee said, "The BJP's conduct and character are undemocratic. It is following the path of fascism."

Writer Shirshendu Mukhopadhyay said, "Raising such slogans amounts to a serious offence. This party has no regard for democracy or law and order. Where are we heading?"

What did the BJP say?

The BJP, on the other hand, called the arrest of its supporters a conspiracy by the Trinamool Congress. The family of one of the accused, Dhruv Basu, claimed that he was innocent. Ranjit Basu, a close relative of Dhruv, said, "Dhruv is a disciplined worker. He cannot raise such slogans. He has been framed."

State BJP president Dilip Ghosh said, "I do not know who raised this slogan. I had also said that those who destroy public property should be shot. But such slogans should be expressed in decent language."

Ghosh called the arrest of three party workers in the case a conspiracy. He said, "The Mamata Banerjee government is constantly harassing BJP workers by implicating them in false cases."

BJP MP from Bankura, Subhash Naskar, said, "Some Trinamool Congress supporters raised this slogan at the rally as part of a conspiracy to defame the party."

The previous week, West Bengal Chief Minister Mamata Banerjee had met Amit Shah in Bhubaneswar. The Congress and Left parties had accused Mamata of having a secret understanding with the BJP over the meeting.

Samaresh Sanyal, a former professor of political science, said, "The opposition had accused the state government of having an understanding with the BJP. In an effort to clear its image of these allegations, the police took swift action against those who raised the slogans at the government's direction."

(You can click here for BBC Hindi's Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)""",

    "india-53336116": """Two US Navy aircraft carriers conducted military exercises in the South China Sea on Monday. Rear Admiral James Kirk, commander of the USS Nimitz, told Reuters over the phone, "We saw them and they saw us."

In a statement, China's Foreign Ministry said that the United States had deliberately sent its ships into the South China Sea to demonstrate its strength. It accused the US of trying to create divisions among countries in the region.

This is not the first time the US and China have tried to demonstrate their power in the area. But the situation is somewhat different this time. The coronavirus pandemic has increased tensions between the two countries, and the US has been continuously trying to put pressure on China.

Jayadev Ranade, president of the Centre for China Analysis and Strategy, said, "China had been saying that the US Navy had been weakened by the coronavirus pandemic. The US took this step to signal to China that its strength had not diminished and that its warships were still present in the region."

Ranade added, "The US also wants to demonstrate that the South China Sea is a neutral area and that it can move freely through it."

According to Ranade, tensions in the region have not yet increased significantly. He said, "Tensions will rise if a Chinese ship is stopped or if there is an attempt to stop China from installing defence systems on the islands. We will have to see what happens next."

Also read:

However, experts believe that the situation is unlikely to escalate in the near future. Swaran Singh, a professor at Jawaharlal Nehru University and an expert in international affairs, said, "The US has done this before in the name of freedom of navigation. It is more of a show of strength; they do not engage in confrontation. The US is also doing this to hide its failures during the COVID-19 pandemic. Elections are being held this year, and Trump wants to show that he is the only leader willing to stand up to China."

The South China Sea dispute

This stretch of sea, located between Indonesia and Vietnam, covers approximately 3.5 million square kilometres. China, the Philippines, Vietnam, Malaysia, Taiwan and Brunei have all laid claim to parts of it. Rich in natural resources, this marine area is home to hundreds of species.

Until about a decade ago, there was not much tension over this region. But China later brought ships used for dredging the seabed to the South China Sea, carrying large quantities of bricks, sand and gravel. It began large-scale construction around a small stretch of sea using sand, gravel, bricks and concrete.

First, a port was built, followed by an airstrip for aircraft to land. Before long, China had created an artificial island in the South China Sea and established a military base on it.

China refused to accept the ruling of an international tribunal concerning its claims over this part of the sea.

India-China border dispute: China also has disputes with several other countries.

What does the US expect from India?

Following the recent India-China border dispute, questions have been raised about whether India's role in the South China Sea dispute will increase.

India has consistently regarded the South China Sea as a neutral area. It believes this neutrality should be maintained and that the sea does not belong to any single country.

Experts believe that India's relations with the US have become closer as its relations with China have deteriorated. However, they say it would not be right to assume that India will play a major role in the South China Sea dispute.

According to Jayadev Ranade, "The US sees India as a major power and will expect it to maintain its position. That is also a form of support, because China does not want this."

Swaran Singh explained, "No country joins the US in patrolling that region. The UK did so once, but all the other countries only give the US their support in principle. During Obama's visit to India in 2015, the two countries issued a joint statement in which India said it was committed to maintaining stability in the South China Sea."

After the US sent ships to the South China Sea, China expressed its anger through a statement. Since then, India has not directly mentioned China by name when discussing this issue. On several occasions, India has spoken about Chinese expansionism without naming China, referring indirectly to the South China Sea. India may continue to maintain this position.

Experts believe that if the dispute escalates significantly, the role of the Quad could become important.

What is the Quad?

The Quadrilateral Security Dialogue, also known as the Quad, is an informal political dialogue group comprising the United States, India, Japan and Australia.

Former diplomat Rajiv Dogra said, "When China began asserting its dominance in the South China Sea, everyone became concerned that it might disregard international rules and act according to its own wishes. India, Japan and Australia's trade passes through this route. The Quad is not an aggressive arrangement. It is a meeting of views among countries that believe the world should be governed by rules and laws. The Quad began with a positive outlook."

However, after its formation in 2007, the Quad remained inactive for the next ten years. The member countries met again in 2017. In 2019, the level of the meetings was raised, with the foreign ministers of all four countries participating.

The Quad is not a military alliance either, so direct military intervention by another country's forces in the South China Sea should not be expected at this stage.

(You can click here for BBC Hindi's Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)"""
})

print("Batch 21 corrections currently:", len(batch21_corrections))

Batch 21 corrections currently: 10


In [63]:
corrections_df = pd.DataFrame(
    list(batch21_corrections.items()),
    columns=["id", "complete_english_translation"]
)

if batch21_accepted_ids:
    reviewed_translations.loc[
        reviewed_translations["id"].isin(batch21_accepted_ids),
        "manual_review_status"
    ] = "Human reviewed - no correction"

for _, row in corrections_df.iterrows():
    reviewed_translations.loc[
        reviewed_translations["id"] == row["id"],
        "complete_english_translation"
    ] = row["complete_english_translation"]

reviewed_translations.loc[
    reviewed_translations["id"].isin(batch21_corrections.keys()),
    "manual_review_status"
] = "Manually corrected"

output_path = (
    f"{base_path}/09_sentence_level_translation/"
    "FactIndic_English_250_Reviewed.csv"
)
reviewed_translations.to_csv(output_path, index=False)

print("Batch 21 saved successfully.")
print("Corrections applied:", len(batch21_corrections))
print("Accepted without correction:", len(batch21_accepted_ids))
print("\nUpdated review status:")
print(reviewed_translations["manual_review_status"].value_counts())
print("\nSaved to:", output_path)

Batch 21 saved successfully.
Corrections applied: 10
Accepted without correction: 0

Updated review status:
manual_review_status
Manually corrected                198
Pending review                     27
Human reviewed - no correction     25
Name: count, dtype: int64

Saved to: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv


In [64]:
pending_ids = reviewed_translations.loc[
    reviewed_translations["manual_review_status"] == "Pending review",
    "id"
].head(10).tolist()

source_lookup = df_250.set_index("id")
translation_lookup = reviewed_translations.set_index("id")

print("Batch 22 — Articles for Human Review")
print("Number of articles:", len(pending_ids))

for i, article_id in enumerate(pending_ids, 1):
    print("\n" + "=" * 100)
    print(f"ARTICLE {i}/10: {article_id}")

    print("\nHINDI ORIGINAL:\n")
    print(source_lookup.loc[article_id, "text"])

    print("\nCURRENT ENGLISH TRANSLATION:\n")
    print(translation_lookup.loc[
        article_id, "complete_english_translation"
    ])

Batch 22 — Articles for Human Review
Number of articles: 10

ARTICLE 1/10: india-55465758

HINDI ORIGINAL:

धर्मेंद्र अनीजा का दावा है कि उन्होंने अपनी पत्नी सपना अनीजा के लिए कथित तौर पर चांद पर ज़मीन ख़रीदी है. 24 दिसंबर को धर्मेंद्र और सपना की शादी की आठवीं सालगिरह थी. धर्मेंद्र का दावा है कि इस मौके पर उन्होंने कथित तौर पर चांद पर खरीदा गया प्लॉट अपनी पत्नी को तोहफ़े के रूप में दिया है. बीबीसी से बातचीत में धर्मेंद्र अनीजा ने दावा किया, "मैंने एक साल पहले ही तय कर लिया था कि शादी की अगली सालगिरह पर पत्नी के लिए चांद पर ज़मीन का तोहफ़ा देना है. यह सरप्राइज़ देना इतना आसान नहीं रहा. कई पड़ावों को पार कर सपना के लिए चांद पर ज़मीन ख़रीदने का सपना पूरा हुआ है." अपने दावे पर जोर देते हुए धर्मेंद्र आगे कहते हैं, "चांद पर ज़मीन ख़रीदना आसान नहीं है, यदि आसान होता तो कोई भी ख़रीद लेता." समाप्त चांद पर कहां ख़रीदी है ज़मीन? धर्मेंद्र अनीजा की पत्नी सपना अनीजा कहती हैं, "तोहफ़े में चांद पर ज़मीन के सरप्राइज़ से इतनी ख़ुश हुई कि कई बार तो रोना ही आ रहा था. शायद मैं दुनिया की सबसे भाग्यशाली महि

In [65]:
batch22_corrections = {
    "india-55465758": """Dharmendra Aneja claims that he has allegedly purchased land on the Moon for his wife, Sapna Aneja. December 24 marked the couple's eighth wedding anniversary. Dharmendra says that he gifted his wife the plot he had reportedly purchased on the Moon on this occasion.

Speaking to the BBC, Dharmendra Aneja said, "I had decided a year ago that I would gift my wife a piece of land on the Moon for our next wedding anniversary. Arranging this surprise was not easy. After overcoming several hurdles, my dream of buying land on the Moon for Sapna has finally come true." Emphasising his claim, he added, "Buying land on the Moon is not easy. If it were easy, anyone could buy it."

Where on the Moon was the land purchased?

Sapna Aneja, Dharmendra's wife, said, "I was so happy when I received the surprise gift of land on the Moon that I almost started crying several times. Perhaps I am the luckiest woman in the world to have received such a gift." She added, "At the event, I was asked to guess what surprise I was being given. I thought it might be a car, jewellery or something special. But I had never imagined that it would be such an extraordinary gift as land on the Moon."

Dharmendra has also made claims about the geographical location of the plot he reportedly purchased on the Moon, but these claims could not be independently verified. According to the Aneja family's claims, three acres of land have been purchased in Sapna's name at 14.3 degrees north latitude and 5.6 degrees east longitude, in Lunar parcels 377, 378 and 379.

Who is Dharmendra Aneja?

Dharmendra Aneja and Sapna Aneja are residents of Ajmer, where they both attended school and college. They met at Ajmer Government College and later got married. Since their marriage, Dharmendra has been running a tour and travel business in Brazil, while his parents live in Ajmer. For the past approximately ten months, Dharmendra has been staying with his family in Ajmer. The couple have a seven-year-old daughter named Riddhi. Dharmendra's father, Ramdayal Aneja, is a contractor, while Sapna's father, Geesulal Arnodia, is a retired bank manager.

'Economic and family background'

Buying land anywhere on Earth also involves following a procedure. The Aneja family claims that, just as on Earth, purchasing land on the Moon also involves a lengthy process.

Dharmendra says that he applied to a US-based company about a year ago to allegedly purchase land on the Moon. After his application was accepted, he held several meetings with the company through video conferences. "During this process, my financial and family background was checked. I had to go through a very lengthy documentation process. The entire process took almost a year," he said.

Dharmendra is confident that he has made a genuine deal and has not been cheated. He claims that "there are many fake companies that promise to sell land on the Moon. But I purchased this plot from a company authorised to provide land on the Moon."

Dharmendra claims that he purchased three acres of land for a period of 49 years. He says he now has the right to sell or transfer this land to someone else. According to the agreement signed at the time of purchase, if research is conducted on his land on the Moon, he will receive royalties from it.

Water found on the Moon: Could humans settle there?

Why does he claim to have bought land on the Moon?

Dharmendra Aneja, who claims to have purchased land on the Moon for his wife, says, "The late actor Sushant Singh Rajput also bought land on the Moon from Lunar Society International in the USA, so I too purchased land through Lunar."

He claims that "Lunar Society International issued a certificate transferring the land into my name", and says that this alleged certificate is the basis for his claim that he owns land on the Moon.

Dharmendra claims that the land belongs to him and that he also possesses a 17-page land deed. However, he has refused to disclose the price of the land or make the deed public.

Nevertheless, under international treaties, this cannot be considered ownership. The world beyond Earth is regarded as the common heritage of all humankind. Therefore, no individual can claim ownership of the Moon.

The Outer Space Treaty was signed in 1967 to regulate activities in outer space. Under this Cold War-era treaty, no country can unilaterally exploit space resources, nor have countries divided outer space among themselves. India is also a signatory to this treaty.

When humans first set foot on the Moon

What are the rules regarding ownership of the Moon by individuals or countries?

Companies are exploring the possibility of mining the lunar surface. But what are the rules regarding claims of ownership of the Moon by individuals or countries?

The potential ownership of celestial bodies was a major issue even during the Cold War. When NASA was planning its first crewed mission to the Moon, the United Nations formulated the Outer Space Treaty in 1967. Countries including the United States, the Soviet Union and Britain signed it. The treaty states that no claim of sovereignty can be made over outer space in any manner.

Joan Wheeler, director of the space consultancy firm Alden Advisors, says that planting a country's flag on the Moon has no legal significance because it does not give individuals, companies or countries any "binding rights".

In 1979, the United Nations drafted an agreement to regulate activities on the Moon and other celestial bodies. It is known as the Moon Agreement. The agreement states that activities in outer space should be carried out for peaceful purposes. It also requires information to be provided to the United Nations about where and why a space station is being established.

However, only 11 countries, including India, have signed the Moon Agreement. Major space powers such as China, the United States and Russia have not joined it. Britain has also not signed it.

In 2015, the United States passed the Commercial Space Launch Competitiveness Act, which gives its citizens the right to extract resources obtained from asteroids. The Act does not directly apply to the Moon, but the same principle could possibly be extended to it.

(You can click here to download BBC Hindi's Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)""",

    "india-55525807": """The Central Military Commission (CMC) will then have greater powers to deploy the military to protect China's national interests. President Xi Jinping is the head of the CMC. According to the newspaper, which cited China's state news agency Xinhua, the new changes are focused on improving war preparedness and military capabilities.

Under the revised law, the military will be able to be deployed at both national and local levels whenever sovereignty, unity, territorial integrity, security or development interests are threatened. The reference to "development interests" is a new addition to the law. Experts believe this provision will help protect China's economic interests and assets abroad. This means that the military could also be deployed to protect assets associated with the Belt and Road Initiative.

These amendments to the defence law will extend the scope of China's military beyond geographical borders, maritime boundaries and airspace to include electromagnetic networks and outer space. Experts also believe that the changes will give the CMC greater powers. Decision-making powers that previously rested with the State Council, or the Cabinet, will also be transferred to the CMC.

One of the main objectives of these legal changes is to accelerate the modernisation of China's military. In 2016, President Xi Jinping had already introduced major reforms, bringing several departments of the People's Liberation Army (PLA), China's military, under the direct control of the CMC. Seven military regions were reorganised into five theatre commands. The Western Theatre Command of the Chinese military is responsible for guarding the border with India. It is also the largest command of China's military.

'Democracy will survive only if there is peace' — Shivraj Singh Chouhan

The Madhya Pradesh government is preparing to introduce a strict law against stone-pelting. It will include provisions for confiscating the property of those accused. It will also provide for compensation for damage caused not only to government property but also to the property of ordinary citizens.

According to a report in The Indian Express, Madhya Pradesh Chief Minister Shivraj Singh Chouhan said that democracy would survive only if there was peace.

Chouhan said, "Those who engage in stone-pelting are enemies of society, whoever they may be. Stone-pelting is not an ordinary crime. It can cause loss of life and create an atmosphere of terror."

He said that stone-pelting would no longer be treated as a minor offence and that strict action would be taken against those involved.

Following incidents of stone-pelting during rallies seeking donations for the Ram temple in Muslim-majority neighbourhoods of Ujjain and Indore in Madhya Pradesh, the administration also demolished the houses of people accused of stone-pelting.

Chouhan said, "Democracy survives when people raise their issues peacefully. But no one has the right to damage public or private property."

Madhya Pradesh is preparing to introduce an ordinance similar to the one in Uttar Pradesh. The Uttar Pradesh government has brought in an ordinance to recover the cost of damage caused to public and private property.

'UP Police say Muslims were falsely implicated in a conversion case'

According to a report in The Hindustan Times, police in Bareilly, Uttar Pradesh, have said that three Muslim men were falsely implicated in a case involving alleged religious conversion. The men had been charged with threatening a 24-year-old married woman. Police now say that steps are being taken to cancel the case.

On December 31, a case was registered against the men at Faridpur police station in Bareilly. They were accused of threatening the woman and forcing her to convert her religion. The case was registered following a complaint by the woman.

Police now say that the preliminary investigation found the allegations of pressuring her to convert to be false. The Bareilly Senior Superintendent of Police (SSP) told the newspaper that the investigation revealed that the men were not present at the location of the alleged incident.

Uttar Pradesh has introduced a new ordinance against religious conversion. Under it, anyone wishing to convert their religion for marriage must obtain permission from the District Magistrate at least two months in advance. The ordinance also provides for imprisonment ranging from one to ten years for forced religious conversion.

'ED to investigate loan app scam'

According to a report in The Times of India, the Enforcement Directorate (ED) has registered a money-laundering case to investigate the sources of funds used by instant loan apps.

The ED said it had taken cognisance of FIRs registered by the Hyderabad and Cyberabad police. The arrested Indian and Chinese nationals have also been charged under the Prevention of Money Laundering Act.

The ED said, "Illegal loan apps are bypassing all kinds of regulatory agencies. We are investigating the possibility that money is being transferred abroad."

Hyderabad Police have registered 27 FIRs in the case so far. The police have identified 14 million transactions worth Rs 21,000 crore. Raids have also been conducted at several locations in Gurugram, Hyderabad and Bengaluru.

(You can click here to download BBC Hindi's Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)""",

    "international-41372477": """The earthquake was recorded near a nuclear test site in North Korea. Tremors had also been felt there during previous nuclear tests.

North Korea conducts another nuclear test! What is everyday life like in North Korea?

According to China's state news agency, seismologists described the earthquake as a "suspected explosion". However, South Korea said it could have been a natural earthquake rather than a nuclear test.

North Korea conducted its sixth nuclear test on September 3, when a powerful earthquake was recorded. The magnitude of Saturday's earthquake was lower than that of the tremors recorded during the nuclear test. When North Korea conducted its previous test, the US Geological Survey's preliminary report said that the earthquake had a magnitude of 5.3 and occurred at a depth of 10 kilometres.

'Trump's threat is like a dog's barking': Why are North Koreans not crossing the border?

The Punggye-ri test site

South Korea's meteorological agency reported that the latest earthquake was recorded at a depth of zero kilometres in North Hamgyong Province. North Korea's nuclear test site, Punggye-ri, is also located in this province.

The Reuters news agency said the earthquake could have been natural because the distinctive sound associated with an artificial earthquake was not detected in this case.

Lassina Zerbo, Executive Secretary of the Comprehensive Nuclear-Test-Ban Treaty Organization, tweeted that analysts were investigating this unusual, low-magnitude earthquake. He said it had occurred about 50 kilometres from the locations of previous nuclear tests.

North Korea could drop a hydrogen bomb on the United States: North Korea is 69 years old and its army is 85 years old? How dangerous would North Korea be if a war broke out?

(You can click here to download BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)"""
}

batch22_accepted_ids = []

print("Batch 22 corrections currently:", len(batch22_corrections))

Batch 22 corrections currently: 3


In [66]:
batch22_corrections.update({

    "international-41646598": """Rajani Vaidyanathan is a BBC correspondent. I was 25 at the time. We arrived at an Italian restaurant in New York shortly after finishing work on our story. I was then an aspiring news producer who had come to Manhattan to cover the Republican Party convention. All our other colleagues had left. Only one colleague and I remained, having dinner together. In the dimly lit Italian restaurant, I was talking about George W. Bush and John Kerry when he suddenly said it.

"I'm sexually attracted to you. I can't stop thinking about you."

My fork slipped from my hand and fell onto the plate, with noodles still caught in it. He was a colleague twice my age, highly respected and with a girlfriend. I had experienced sexism at work before, but never so openly. I cannot remember exactly what I said, but my response was very polite. I somehow tried to change the subject. He continued telling me how beautiful I was, while I tried to finish my pasta as quickly as possible.

At the time, I was not sure whether he had said something I could make a formal complaint about. But his comments made me uncomfortable and I felt deeply upset. I now know that his behaviour was completely unacceptable. The incident reminds me how some men use their power in the workplace to control, exploit and harass women.

**Abuse on the street, in the office and on campus**

In recent days, following the Harvey Weinstein sexual harassment scandal, my female friends and I have been discussing where we should draw the line and when we should speak out. Do you have to be touched for something to count as abuse? What if you are friends with the person? When does abuse begin, and when does it end?

Headlines: "A priest sexually abused me when I was eight"; "You are disabled. What would anyone gain from raping you?"

Women around the world are speaking about sexual violence following the allegations against Weinstein. His case has brought Hollywood's disturbing practices to light, but it is also a depressing reflection of the abuse women face every day in every part of the world, across every society and in every profession.

**Obscene messages**

A few years ago, a married friend of mine began sending me messages describing his sexual desires in detail. "I've become used to pleasuring myself. I just can't control myself."

I was frightened, but initially I remained polite because there was a good chance we would continue working together. I wonder why women always feel they have to be polite in such situations. I did not know how to respond, so I told him that it was normal for a man to have such feelings. I thought he would stop sending messages after that. Instead, they became increasingly obscene. He told me that he fantasised about having sex with powerful women and wanted to cheat on his wife.

I told him to discuss these things with someone else rather than me, or to seek help. At first, I did not tell anyone. I felt terrible, but kept the matter to myself.

Actress Alyssa Milano has appealed to women who have experienced sexual harassment to come forward.

A few months later, I was talking to a female colleague when she told me that the same man had been sending her lewd messages for years. I felt somewhat relieved that I could now share my own experience with someone. A few days later, I heard that he had been fired. Another colleague had filed a complaint against him.

Whatever the level of abuse, it becomes easier to talk about it when we discover that someone else has gone through the same experience. Consider the case of Bill Cosby. I have interviewed many women who experienced sexual assault on college campuses in the United States. They had not come forward with allegations because they thought nobody would believe them.

Men often do not understand how to deal with such situations because, quite simply, they do not have to deal with them. They do not usually face the kinds of comments and accusations that women have to deal with.

**A knock at the hotel door**

These two incidents are just examples. Over the years, I have faced many such situations. I almost forgot to mention the time a married colleague knocked on my hotel-room door late at night. Before that, he had sent me a suggestive message. That night, I ran into the bathroom and called a male colleague. He told me to call him if it happened again. This incident reminds me that many men also help us in such situations.

Many of my friends are sharing their experiences of being grabbed, stared at during board meetings and fired for refusing to flirt. This shows that many things need to change.

But one positive aspect of these disturbing stories emerging from Hollywood and beyond is that more people are now talking about such experiences, making this kind of behaviour less acceptable.

(Click here for the BBC Hindi Android app. You can also follow us on Facebook and Twitter.)""",

    "international-41960891": """Kim Jong-un and Donald Trump have exchanged remarks on Twitter. In a series of tweets, the US president complained that he had never called Kim Jong-un "short and fat". Earlier, North Korea's foreign ministry had called Trump "old". Trump also offered to mediate in the dispute involving North Korea. He has separately offered to help resolve the South China Sea dispute, where China, Vietnam, the Philippines, Taiwan, Malaysia and Brunei have competing territorial claims.

Headlines: "Kim Jong-un calls Trump 'mad'"; "Kim Jong-un shows wisdom by standing up to Trump"; "North Korea: Trump's threats are like a dog's barking."

Tensions between Vietnam and the Philippines and China remain high over China's construction of artificial islands and deployment of naval patrols in the South China Sea. Although an understanding was reached over the dispute in August, its underlying problems remain unresolved. Tensions continue among several countries.

Trump believes he is capable of helping secure an agreement. He thinks his experience in negotiating deals could help move the process forward. He told Vietnam that he was willing to mediate if he could help resolve the dispute. Trump made the offer at the Asia-Pacific Economic Cooperation summit in Hanoi, Vietnam's capital.

Trump's tweets about North Korea have further heightened tensions. On Saturday, North Korea condemned Trump's visit to Asia, describing it as an attempt to provoke war. It again called him a "senile old man". Trump was unhappy with the remark and responded on Twitter, saying that he had never called Kim "short and fat".

While Trump was targeting the North Korean leader on social media, three US aircraft carriers were conducting military exercises in the western Pacific Ocean. Through these exercises, the United States is also demonstrating its military strength against North Korea. US commander Scott Swift said that military activity of this kind was taking place for the first time since 2007. Warships from South Korea and Japan are also taking part. The exercises began on Saturday and will continue until Tuesday.

Trump has not ruled out the possibility of having good relations with North Korea. During a press conference in Vietnam, he was asked whether he could imagine himself becoming friends with Kim Jong-un. Trump replied that this might sound surprising, but that it was a possibility.

(Click here for the BBC Hindi Android app. You can also follow us on Facebook and Twitter.)""",

    "international-42264998": """Omar bin Sultan Al Olama

This is not the set of a futuristic Hollywood film. These are working ministries of the United Arab Emirates government.

**What did I think of the Emirates?**

I visited the Emirates for the first time last month, carrying a number of prejudices. My views had been shaped by the coverage of the country in the Western media. I have been part of that media myself for the past 22 years.

We imagined Dubai as a city of enormous, high-rise buildings, but otherwise lifeless. I had never considered the Emirates to be much more than an oil-producing country. I also thought that wealthy Arabs were simply spending their newly acquired wealth while wearing traditional clothes. Clearly, these were mistaken views.

**Ten days in the Emirates opened my eyes**

After spending ten days in the Emirates, my eyes were opened. Emiratis may appear ordinary on the outside, but they are very confident within. They feel secure about the present and are working to make their future even better. Their society is prosperous, and they are building a future that could become an example for other Arab countries and the wider world.

What is particularly commendable is that this work is happening rapidly and without much fuss. The UAE government has plans to build a city on Mars. Last week, it announced plans to make Dubai the world's biggest IT hub. Dubai's administration is preparing to introduce pilotless air-taxi services. The government is also drawing up plans for a regular drone-racing event called the World Drone Prix.

**The Emirates was once a backward country**

For people who are unfamiliar with the use of high technology in public services, these developments may seem astonishing. The country's rapid progress is even more impressive when we remember that, just a few decades ago, the Emirates was a relatively underdeveloped country. Its local Bedouin population was divided into different tribes.

The progress deserves particular recognition because many of the surrounding Arab countries are struggling with terrorism, economic crises and ethnic conflicts. The royal families here have built a tolerant society where there is religious freedom, religious disagreements do not lead to conflict, and terrorist attacks do not occur. Spiritual balance and business success can be seen together here.

**A smart city on Mars**

I am certain that you will hear the name of 27-year-old Omar bin Sultan Al Olama repeatedly in the coming months and years. He is the Minister of State for Artificial Intelligence. He was appointed to the position two months ago. He is also the Deputy Director of the Future Department in the Prime Minister's Office and is involved in all plans relating to the future.

Al Olama's responsibilities include advancing government plans by investing in new artificial-intelligence technologies and tools, and ensuring that AI is used appropriately in all future-oriented work. But an even more important responsibility is working on the plan to build the first smart city on Mars.

The Emirates gained independence from Britain in 1971 and is preparing to celebrate its centenary on a grand scale in 2071. These responsibilities have also been assigned to Al Olama.

**The Emirates' income is higher than that of many developed countries**

The Emirati society is wealthy. Its per-capita income is $72,800, higher than that of many developed countries. People appear satisfied with their lives, at least on the surface.

Despite this, the government established a Ministry of Happiness last year. Its minister is Ohoud bint Khalfan Al Roumi, one of eight women in the 21-member cabinet. After becoming a minister last year, she said that promoting happiness was a serious necessity. Two months ago, she was also given responsibility for the Quality of Life portfolio.

But is it the government's job to bring happiness into people's lives? The answer is not simple. The Ministry of Happiness's website says its aim is to make the UAE one of the happiest countries in the world.

Local people generally seem more satisfied than happy. Personally, however, happiness means democracy and individual freedom to me. Some local people seemed to agree with me, but most appeared satisfied with their current circumstances.

(Click here for the BBC Hindi Android app. You can also follow us on Facebook and Twitter.)""",

    "international-43396558": """Albert Einstein was born on 14 March 1879 and died on 18 April 1955. Like Stephen Hawking, Einstein lived to the age of 76. Although he is regarded as one of the greatest physicists, his personal life was far from successful. In 2012, author Walter Isaacson made this point in his book *Einstein: His Life and Universe*. He wrote that the world-famous scientist made considerable efforts to make his personal life work, but ultimately failed.

**The battle Einstein could not win**

Walter Isaacson described how Einstein tried to preserve his marriage, but it became a battle he could not win. In fact, Einstein's approach to love and relationships was so controlling that, when he realised his marriage to the scientist Mileva Marić was falling apart, he drew up a list of strict rules. He hoped that this might allow them to continue living together for the sake of their children.

**Einstein's list**

The British newspaper *Daily Mail* published the list based on Walter Isaacson's book. It contained demands that would have turned Marić into a servant rather than a wife. According to the book, Einstein wrote the list in 1914, when he was beginning to realise that his marriage to his first wife, Marić, was failing.

Marić was one of the few women in Europe at the time who had studied mathematics and physics. Einstein had also proposed that the couple should continue living together for the sake of their children. But his list of demands was astonishing. He expected her to work like a servant for him, without expecting love or attention in return.

Einstein, one of the world's greatest scientists, demanded that Marić keep his room clean, prepare three meals a day for him to eat in his room, wash his clothes and put them away properly.

**Einstein's second wife, Elsa**

Einstein also wanted his wife to keep his bedroom and study clean. His wife was not even allowed to use his desk. His demands did not end there. The famous physicist also made it clear that she should not expect too much from him.

**What his wife could not expect**

According to the list, Einstein said that his wife should not expect him to sit with her or go out with her. He also wanted her to stop talking whenever he asked her to. He even expected her to leave his bedroom or study immediately, without protest, whenever he told her to.

It appears that Einstein's wife initially agreed to his demands, but events unfolded as might have been expected. Just a few months later, Marić left Einstein in Berlin and moved to Zurich with their sons. Five years later, in 1919, she filed for divorce, which was granted.

Einstein is also said to have had several girlfriends. He had been in a close relationship with a woman named Elsa since 1912, even though he was still married at the time. After his divorce, he married Elsa in 1919. But he later also developed a close relationship with his secretary, Betty Neumann.

(Click here for the BBC Hindi Android app. You can also follow us on Facebook and Twitter.)""",

    "international-44164669": """After agreeing to hold talks with US President Donald Trump, North Korean leader Kim Jong-un has said that negotiations cannot take place in the current atmosphere, given the way the United States is behaving. On Wednesday, North Korea warned that it might not take part in the talks.

Trump's own national security adviser, John Bolton, has made the situation more difficult. Bolton said that the United States was considering applying the "Libya model" to North Korea as well.

Headlines: "Is North Korea unhappy with Trump's growing demands?"; "North Korea threatens to cancel meeting with Trump"; "Kim Jong-un angry over the 'Libya model'."

In 2003, Libyan leader Colonel Muammar Gaddafi agreed to abandon his country's nuclear programme. In return, the United States lifted most of the sanctions imposed on Libya. But in 2011, rebels backed by Western countries overthrew Gaddafi, and he was later killed.

Trump has said that he is not considering the "Libya model" for North Korea and believes that talks between the two countries will take place. He rejected his national security adviser's statement.

"In the case of North Korea, we are not thinking about the Libya model at all. In Libya, we destroyed that country. The country was devastated. There was no deal to keep Gaddafi in power. The Libya model being discussed was completely different," Trump said.

Trump added: "Under the kind of deal I am thinking about, Kim Jong-un would remain in his own country, govern his own country and become wealthy."

Headlines: "Will Kim Jong-un meet the same fate as Gaddafi?"; "Gaddafi's son Saif released in Libya after six years."

Trump also suspects that China may be plotting to derail talks with North Korea. One possible reason, he said, is that Kim Jong-un has recently visited China twice and met President Xi Jinping.

"I think China wants to make a deal for a number of reasons. Perhaps it has something to do with trade. They have never faced this much trouble before. It is quite possible that they are provoking Kim Jong-un," Trump said.

Headlines: "North Korea to destroy its nuclear test site"; "Why did VK Singh visit North Korea so quietly?"; "China keeps a close watch on the Trump-Kim meeting."

**Concerns over joint military exercises**

Meanwhile, joint military exercises between the United States and South Korea are continuing. North Korea is also angry about these exercises and says that the United States is not serious about creating a better atmosphere for talks. It cancelled high-level talks with South Korea that were scheduled for this week.

North Korea's state news agency, KCNA, described the joint US-South Korean exercises as a provocation. But the United States has insisted on continuing the exercises, saying that it will neither cancel nor postpone them.

Headlines: "Kim Jong-un cancels talks with South Korea"; "What goods are exchanged between North Korea and China?"; "Joint military exercises between North Korea and South Korea."

White House spokeswoman Sarah Sanders said she did not understand how Kim Jong-un could make such demands. She said: "These are regular military exercises, and we are aware of them. They are held every year, and we have no intention of changing them at present."

For now, there are doubts and concerns on both sides about whether the talks will take place. One thing is clear: as the scheduled 12 June meeting in Singapore approaches, the clouds of uncertainty surrounding it are growing darker.

(Click here for the BBC Hindi Android app. You can also follow us on Facebook and Twitter.)"""
})

print("Batch 22 corrections currently:", len(batch22_corrections))

Batch 22 corrections currently: 8


In [67]:
batch22_corrections.update({

    "india-55525807": """The Central Military Commission (CMC) will have greater powers to deploy the military to protect national interests. President Xi Jinping is the head of the CMC. According to China's state news agency Xinhua, the newspaper reported that the new changes focus on improving military preparedness and combat capabilities.

Under the revised law, the military can be deployed at national and local levels whenever sovereignty, unity, territorial integrity, security or development interests are threatened. The phrase "development interests" has been newly added to the law. Experts believe this could allow China to protect its economic interests and assets overseas. This means the military could also be deployed to protect assets associated with the Belt and Road Initiative.

These amendments to the defence law will extend the scope of China's military operations beyond geographical borders, maritime boundaries and airspace to electromagnetic networks and outer space. Experts also believe that the changes will give the CMC greater authority. Powers to make decisions that previously rested with the State Council, or the Cabinet, will also be transferred to the CMC.

The main objective of these legal changes is to accelerate the modernisation of China's military. In 2016, President Xi Jinping introduced major reforms, bringing several departments of the People's Liberation Army (PLA) under the direct control of the CMC. The seven military regions were reorganised into five theatre commands. China's Western Theatre Command is responsible for guarding the country's border with India. It is also the largest of the Chinese military's theatre commands.

**Democracy will survive only if there is peace: Shivraj Singh Chouhan**

The Madhya Pradesh government is preparing to introduce a strict law against stone-pelting. It will include provisions for confiscating the property of those accused and recovering compensation for damage caused not only to government property but also to the property of ordinary citizens.

According to a report in *The Indian Express*, Madhya Pradesh Chief Minister Shivraj Singh Chouhan said that democracy would survive only if there was peace.

"Those who throw stones are enemies of society, whoever they may be. Stone-pelting is not an ordinary crime. It can cost people their lives and create an atmosphere of terror," Chouhan said. He added that it would no longer be treated as a minor offence and that strict action would be taken against those responsible.

After incidents of stone-pelting during rallies seeking donations for the Ram temple in Muslim-majority neighbourhoods of Ujjain and Indore, the administration demolished the houses of people accused of stone-pelting.

"If someone raises an issue peacefully, democracy survives. But nobody has the right to damage public or private property," Chouhan said.

Madhya Pradesh is preparing to introduce an ordinance similar to the one in Uttar Pradesh. The UP government has already introduced an ordinance to recover compensation for damage to public and private property.

**UP police say Muslims were falsely implicated in a religious-conversion case**

According to a report in *The Hindustan Times*, police in Bareilly, Uttar Pradesh, have said that three Muslim men were falsely implicated in a case involving alleged religious conversion. The men had been accused of threatening a 24-year-old married woman. Police now say that steps are being taken to cancel the case.

On 31 December, a case was registered against the men at Faridpur police station in Bareilly for allegedly threatening the woman and forcing her to convert. The case was registered following a complaint by the woman.

Police now say that the initial investigation found the allegations of pressure to convert to be false. The Bareilly Senior Superintendent of Police told the newspaper that the investigation had found that the men were not present at the scene.

Uttar Pradesh has introduced a new ordinance against religious conversion. Under it, anyone wishing to convert their religion for marriage must obtain permission from the district magistrate at least two months in advance. Forced religious conversion can carry a prison sentence of between one and ten years.

**ED to investigate loan app scam**

According to *The Times of India*, the Enforcement Directorate (ED) has registered a money-laundering case to investigate the sources of funds behind instant-loan apps.

The ED said it had taken note of the FIRs registered by the Hyderabad and Cyberabad police. The arrested Indian and Chinese nationals have also been charged under the Prevention of Money Laundering Act.

"Illegal loan apps are bypassing all kinds of regulatory agencies. We are investigating the possibility that money is being transferred abroad," the ED said.

Hyderabad police have registered 27 FIRs in the case so far. Police have identified 14 million transactions worth Rs 21,000 crore. Raids have been conducted at several locations in Gurugram, Hyderabad and Bengaluru.

(Click here for the BBC Hindi Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)""",

    "international-41372477": """The earthquake in North Korea was recorded near the country's nuclear test site. Tremors have also been detected in the area during previous nuclear tests.

Headlines: "North Korea conducts another nuclear test!"; "What is everyday life like in North Korea?"

According to China's state news agency, seismologists described the event as a "suspected explosion". South Korea, however, said it could have been a natural earthquake rather than a nuclear test.

North Korea conducted its sixth nuclear test on 3 September, when a powerful earthquake was recorded. The earthquake that occurred on Saturday was weaker than the tremors detected during the nuclear test. When North Korea conducted its previous test, the US Geological Survey's preliminary report recorded a magnitude of 5.3 at a depth of 10 kilometres.

Headlines: "North Korea: Trump's threat is like a dog's barking"; "Why are North Koreans not crossing the border?"

**The Punggye-ri test site**

South Korea's meteorological agency said the latest earthquake was recorded at a depth of zero kilometres in North Hamgyong Province. North Korea's Punggye-ri nuclear test site is located in this province.

Reuters reported that the earthquake could have been a natural one because it did not produce the characteristic sound associated with an artificial earthquake.

Lassina Zerbo, executive secretary of the Comprehensive Nuclear-Test-Ban Treaty Organization, tweeted that analysts were investigating the unusual earthquake, which had a low magnitude. He said it had occurred about 50 kilometres from the sites of previous nuclear tests.

Headlines: "North Korea says it could drop a hydrogen bomb on the United States"; "North Korea at 69 and its army at 85?"; "How dangerous would North Korea be if war broke out?"

(Click here for the BBC Hindi Android app. You can also follow us on Facebook and Twitter.)"""
})

print("Batch 22 corrections currently:", len(batch22_corrections))

Batch 22 corrections currently: 8


In [68]:
batch22_corrections["india-55525807"] = """The Central Military Commission (CMC) will have greater powers to deploy the military to protect national interests. President Xi Jinping is the head of the CMC. According to China's state news agency Xinhua, the new changes focus on improving military preparedness and combat capabilities.

Under the revised law, the military can be deployed at national and local levels whenever sovereignty, unity, territorial integrity, security or development interests are threatened. The phrase "development interests" has been newly added to the law. Experts believe this could allow China to protect its economic interests and assets overseas, including those associated with the Belt and Road Initiative.

These amendments to the defence law will extend the scope of China's military operations beyond geographical borders, maritime boundaries and airspace to electromagnetic networks and outer space. Experts also believe the changes will give the CMC greater authority. Powers to make decisions that previously rested with the State Council, or the Cabinet, will also be transferred to the CMC.

The main objective of these legal changes is to accelerate the modernisation of China's military. In 2016, President Xi Jinping introduced major reforms, bringing several departments of the People's Liberation Army (PLA) under the direct control of the CMC. The seven military regions were reorganised into five theatre commands. China's Western Theatre Command is responsible for guarding the country's border with India. It is also the largest of the Chinese military's theatre commands.

**Democracy will survive only if there is peace: Shivraj Singh Chouhan**

The Madhya Pradesh government is preparing to introduce a strict law against stone-pelting. It will include provisions for confiscating the property of those accused and recovering compensation for damage caused to government property as well as the property of ordinary citizens.

According to a report in The Indian Express, Madhya Pradesh Chief Minister Shivraj Singh Chouhan said that democracy would survive only if there was peace.

"Those who throw stones are enemies of society, whoever they may be. Stone-pelting is not an ordinary crime. It can cost people their lives and create an atmosphere of terror," Chouhan said. He added that it would no longer be treated as a minor offence and that strict action would be taken against those responsible.

After incidents of stone-pelting during rallies seeking donations for the Ram temple in Muslim-majority neighbourhoods of Ujjain and Indore, the administration demolished the houses of people accused of stone-pelting.

"If someone raises an issue peacefully, democracy survives. But nobody has the right to damage public or private property," Chouhan said.

Madhya Pradesh is preparing to introduce an ordinance similar to the one in Uttar Pradesh. The UP government has already introduced an ordinance to recover compensation for damage to public and private property.

**UP police say Muslims were falsely implicated in a religious-conversion case**

According to a report in The Hindustan Times, police in Bareilly, Uttar Pradesh, have said that three Muslim men were falsely implicated in a case involving alleged religious conversion. The men had been accused of threatening a 24-year-old married woman. Police now say that steps are being taken to cancel the case.

On 31 December, a case was registered against the men at Faridpur police station in Bareilly for allegedly threatening the woman and forcing her to convert. The case was registered following a complaint by the woman.

Police now say that the initial investigation found the allegations of pressure to convert to be false. The Bareilly Senior Superintendent of Police told the newspaper that the investigation had found that the men were not present at the scene.

Uttar Pradesh has introduced a new ordinance against religious conversion. Under it, anyone wishing to convert their religion for marriage must obtain permission from the district magistrate at least two months in advance. Forced religious conversion can carry a prison sentence of between one and ten years.

**ED to investigate loan app scam**

According to The Times of India, the Enforcement Directorate (ED) has registered a money-laundering case to investigate the sources of funds behind instant-loan apps.

The ED said it had taken note of the FIRs registered by the Hyderabad and Cyberabad police. The arrested Indian and Chinese nationals have also been charged under the Prevention of Money Laundering Act.

"Illegal loan apps are bypassing all kinds of regulatory agencies. We are investigating the possibility that money is being transferred abroad," the ED said.

Hyderabad police have registered 27 FIRs in the case so far. Police have identified 1.4 crore transactions worth Rs 21,000 crore. Raids have been conducted at several locations in Gurugram, Hyderabad and Bengaluru.

(Click here for the BBC Hindi Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)"""

batch22_corrections["international-41372477"] = """The earthquake in North Korea was recorded near the country's nuclear test site. Tremors have also been detected in the area during previous nuclear tests.

Headlines: "North Korea conducts another nuclear test!" and "What is everyday life like in North Korea?"

According to China's state news agency, seismologists described the event as a "suspected explosion". South Korea, however, said it could have been a natural earthquake rather than a nuclear test.

North Korea conducted its sixth nuclear test on 3 September, when a powerful earthquake was recorded. The earthquake that occurred on Saturday was weaker than the tremors detected during the nuclear test. When North Korea conducted its previous test, the US Geological Survey's preliminary report recorded a magnitude of 5.3 at a depth of 10 kilometres.

Headlines: "Trump's threat is like a dog's barking"; "Why are North Koreans not crossing the border?"

**The Punggye-ri test site**

South Korea's meteorological agency said the latest earthquake was recorded at a depth of zero kilometres in North Hamgyong Province. North Korea's Punggye-ri nuclear test site is located in this province.

Reuters reported that the earthquake could have been a natural one because it did not produce the characteristic sound associated with an artificial earthquake.

Lassina Zerbo, executive secretary of the Comprehensive Nuclear-Test-Ban Treaty Organization, tweeted that analysts were investigating the unusual earthquake, which had a low magnitude. He said it had occurred about 50 kilometres from the sites of previous nuclear tests.

Headlines: "North Korea says it could drop a hydrogen bomb on the United States"; "North Korea at 69 and its army at 85?"; "How dangerous would North Korea be if war broke out?"

(Click here for the BBC Hindi Android app. You can also follow us on Facebook and Twitter.)"""

print("Batch 22 corrections currently:", len(batch22_corrections))

Batch 22 corrections currently: 8


In [69]:
batch22_corrections = dict(batch22_corrections)

batch22_corrections["india-55525807"] = """The Central Military Commission (CMC) will have greater powers to deploy the military to protect national interests. President Xi Jinping is the head of the CMC. According to China's state news agency Xinhua, the new changes focus on improving military preparedness and combat capabilities.

Under the revised law, the military can be deployed at national and local levels whenever sovereignty, unity, territorial integrity, security or development interests are threatened. The phrase "development interests" has been newly added to the law. Experts believe this could allow China to protect its economic interests and assets overseas, including those associated with the Belt and Road Initiative.

These amendments to the defence law will extend the scope of China's military operations beyond geographical borders, maritime boundaries and airspace to electromagnetic networks and outer space. Experts also believe the changes will give the CMC greater authority. Powers to make decisions that previously rested with the State Council, or the Cabinet, will also be transferred to the CMC.

The main objective of these legal changes is to accelerate the modernisation of China's military. In 2016, President Xi Jinping introduced major reforms, bringing several departments of the People's Liberation Army (PLA) under the direct control of the CMC. The seven military regions were reorganised into five theatre commands. China's Western Theatre Command is responsible for guarding the country's border with India. It is also the largest of the Chinese military's theatre commands.

**Democracy will survive only if there is peace: Shivraj Singh Chouhan**

The Madhya Pradesh government is preparing to introduce a strict law against stone-pelting. It will include provisions for confiscating the property of those accused and recovering compensation for damage caused to government property as well as the property of ordinary citizens.

According to a report in The Indian Express, Madhya Pradesh Chief Minister Shivraj Singh Chouhan said that democracy would survive only if there was peace.

"Those who throw stones are enemies of society, whoever they may be. Stone-pelting is not an ordinary crime. It can cost people their lives and create an atmosphere of terror," Chouhan said. He added that it would no longer be treated as a minor offence and that strict action would be taken against those responsible.

After incidents of stone-pelting during rallies seeking donations for the Ram temple in Muslim-majority neighbourhoods of Ujjain and Indore, the administration demolished the houses of people accused of stone-pelting.

"If someone raises an issue peacefully, democracy survives. But nobody has the right to damage public or private property," Chouhan said.

Madhya Pradesh is preparing to introduce an ordinance similar to the one in Uttar Pradesh. The UP government has already introduced an ordinance to recover compensation for damage to public and private property.

**UP police say Muslims were falsely implicated in a religious-conversion case**

According to a report in The Hindustan Times, police in Bareilly, Uttar Pradesh, have said that three Muslim men were falsely implicated in a case involving alleged religious conversion. The men had been accused of threatening a 24-year-old married woman. Police now say that steps are being taken to cancel the case.

On 31 December, a case was registered against the men at Faridpur police station in Bareilly for allegedly threatening the woman and forcing her to convert. The case was registered following a complaint by the woman.

Police now say that the initial investigation found the allegations of pressure to convert to be false. The Bareilly Senior Superintendent of Police told the newspaper that the investigation had found that the men were not present at the scene.

Uttar Pradesh has introduced a new ordinance against religious conversion. Under it, anyone wishing to convert their religion for marriage must obtain permission from the district magistrate at least two months in advance. Forced religious conversion can carry a prison sentence of between one and ten years.

**ED to investigate loan app scam**

According to The Times of India, the Enforcement Directorate (ED) has registered a money-laundering case to investigate the sources of funds behind instant-loan apps.

The ED said it had taken note of the FIRs registered by the Hyderabad and Cyberabad police. The arrested Indian and Chinese nationals have also been charged under the Prevention of Money Laundering Act.

"Illegal loan apps are bypassing all kinds of regulatory agencies. We are investigating the possibility that money is being transferred abroad," the ED said.

Hyderabad police have registered 27 FIRs in the case so far. Police have identified 1.4 crore transactions worth Rs 21,000 crore. Raids have been conducted at several locations in Gurugram, Hyderabad and Bengaluru.

(Click here for the BBC Hindi Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)"""

batch22_corrections["international-41372477"] = """The earthquake in North Korea was recorded near the country's nuclear test site. Tremors have also been detected in the area during previous nuclear tests.

Headlines: "North Korea conducts another nuclear test!" and "What is everyday life like in North Korea?"

According to China's state news agency, seismologists described the event as a "suspected explosion". South Korea, however, said it could have been a natural earthquake rather than a nuclear test.

North Korea conducted its sixth nuclear test on 3 September, when a powerful earthquake was recorded. The earthquake that occurred on Saturday was weaker than the tremors detected during the nuclear test. When North Korea conducted its previous test, the US Geological Survey's preliminary report recorded a magnitude of 5.3 at a depth of 10 kilometres.

Headlines: "Trump's threat is like a dog's barking"; "Why are North Koreans not crossing the border?"

**The Punggye-ri test site**

South Korea's meteorological agency said the latest earthquake was recorded at a depth of zero kilometres in North Hamgyong Province. North Korea's Punggye-ri nuclear test site is located in this province.

Reuters reported that the earthquake could have been a natural one because it did not produce the characteristic sound associated with an artificial earthquake.

Lassina Zerbo, executive secretary of the Comprehensive Nuclear-Test-Ban Treaty Organization, tweeted that analysts were investigating the unusual earthquake, which had a low magnitude. He said it had occurred about 50 kilometres from the sites of previous nuclear tests.

Headlines: "North Korea says it could drop a hydrogen bomb on the United States"; "North Korea at 69 and its army at 85?"; "How dangerous would North Korea be if war broke out?"

(Click here for the BBC Hindi Android app. You can also follow us on Facebook and Twitter.)"""

print("Total corrections:", len(batch22_corrections))
print("Missing IDs:", [
    x for x in [
        "india-55465758",
        "india-55525807",
        "international-41372477",
        "international-41395655",
        "international-41521329",
        "international-41646598",
        "international-41960891",
        "international-42264998",
        "international-43396558",
        "international-44164669"
    ] if x not in batch22_corrections
])

Total corrections: 8
Missing IDs: ['international-41395655', 'international-41521329']


In [70]:
batch22_corrections["international-41395655"] = """The Saudi king gave Churchill a diamond ring, a sword shaped like a monkey pearl, and a dress as gifts.

Headlines: "What is the biggest arms deal between the US and Saudi Arabia?" and "Did Trump bow before the Saudi king?"

Churchill was extremely embarrassed to receive these gifts from the Saudi king, so he sent him a Rolls-Royce car as a gift in return.

US President Donald Trump was warmly welcomed by the Saudi royal family. The tradition of the Saudi royal family presenting expensive gifts to foreign heads of state is a long-standing one. The latest example is US President Donald Trump. During his visit to Saudi Arabia last May, the current King Salman presented him with various gifts.

The American magazine The Daily Beast published a list of gifts received by Trump using the US Freedom of Information Act. The list included 83 gifts.

Headlines: "Can Trump take these gifts home?" and "Can Trump take these slippers home?"

Under US law, American officials cannot accept gifts worth more than $390 during official visits abroad. Gifts received by former US presidents are therefore kept in museums. However, government officials can purchase these gifts at their market value.

Former US Secretary of State Hillary Clinton received a black pearl necklace from Myanmar's leader, Aung San Suu Kyi. When she returned to the United States, Clinton bought the necklace for $970.

(Click here for the BBC Hindi Android app. You can also follow us on Facebook and Twitter.)"""

batch22_corrections["international-41521329"] = """Trump made the remark just hours after reports emerged about his plan to stop the nuclear agreement with Iran. The storm he referred to could also have been related to rising tensions between the United States and North Korea.

Earlier, Trump told senior military officials that he expected them to provide him with a range of military options in the future, and to do so quickly.

After a photo session with military officers and their wives at the White House, Trump said, "This is the silence before the storm." When journalists asked him what he meant, Trump replied, "You'll find out."

**What storm was Trump talking about?**

In the current situation, the Trump administration may have been referring to two countries: North Korea and Iran. Both countries have their own nuclear programmes. These two countries were discussed during Trump's meeting with military officials.

Trump has accused Iran of not acting in the spirit of the agreement. The deal between Iran and the United States was reached during the presidency of former US President Barack Obama. Signed in 2015, the agreement was intended to restrict Iran's nuclear programme.

Trump has described the agreement as one of the worst deals he has ever seen.

(Click here for the BBC Hindi Android app. You can also follow us on Facebook and Twitter.)"""

print("Batch 22 corrections currently:", len(batch22_corrections))

Batch 22 corrections currently: 10


In [71]:
corrections_df = pd.DataFrame(
    list(batch22_corrections.items()),
    columns=["id", "complete_english_translation"]
)

if batch22_accepted_ids:
    reviewed_translations.loc[
        reviewed_translations["id"].isin(batch22_accepted_ids),
        "manual_review_status"
    ] = "Human reviewed - no correction"

for _, row in corrections_df.iterrows():
    reviewed_translations.loc[
        reviewed_translations["id"] == row["id"],
        "complete_english_translation"
    ] = row["complete_english_translation"]

reviewed_translations.loc[
    reviewed_translations["id"].isin(batch22_corrections.keys()),
    "manual_review_status"
] = "Manually corrected"

output_path = (
    f"{base_path}/09_sentence_level_translation/"
    "FactIndic_English_250_Reviewed.csv"
)

reviewed_translations.to_csv(output_path, index=False)

print("Batch 22 saved successfully.")
print("Corrections applied:", len(batch22_corrections))
print("Accepted without correction:", len(batch22_accepted_ids))
print("\nUpdated review status:")
print(reviewed_translations["manual_review_status"].value_counts())
print("\nSaved to:", output_path)

Batch 22 saved successfully.
Corrections applied: 10
Accepted without correction: 0

Updated review status:
manual_review_status
Manually corrected                208
Human reviewed - no correction     25
Pending review                     17
Name: count, dtype: int64

Saved to: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv


In [72]:
pending_ids = reviewed_translations.loc[
    reviewed_translations["manual_review_status"] == "Pending review",
    "id"
].head(10).tolist()

source_lookup = df_250.set_index("id")
translation_lookup = reviewed_translations.set_index("id")

print("Batch 23 — Articles for Human Review")
print("Number of articles:", len(pending_ids))

for i, article_id in enumerate(pending_ids, 1):
    print("\n" + "=" * 100)
    print(f"ARTICLE {i}/10: {article_id}")

    print("\nHINDI ORIGINAL:\n")
    print(source_lookup.loc[article_id, "text"])

    print("\nCURRENT ENGLISH TRANSLATION:\n")
    print(translation_lookup.loc[
        article_id, "complete_english_translation"
    ])

Batch 23 — Articles for Human Review
Number of articles: 10

ARTICLE 1/10: international-44946029

HINDI ORIGINAL:

लगातार हुईं मॉब लिंचिंग की ये घटनाएँ अब सिर्फ़ भारतीय मीडिया में ही नहीं बल्कि विदेशी मीडिया में भी जगह बना रही हैं. हाल ही में अलवर में हुई रक़बर की हत्या संसद में बहस का हिस्सा बनी. आरोप है कि अलवर ज़िले के रामगढ़ थाना क्षेत्र में शुक्रवार रात कथित गोरक्षकों ने रकबर की बुरी तरह पिटाई की थी, जिसके बाद वह गंभीर रूप से घायल हो गए थे. ये बात भी सामने आई है कि रक़बर को अस्पताल ले जाने में पुलिस ने कोताही बरती. पुलिस कोई तीन घंटे बाद रक़बर को पास के सरकारी अस्पताल ले गई, जहाँ डॉक्टरों ने उन्हें मृत घोषित कर दिया. इस घटना का और ऐसी ही अन्य घटनाओं का शोर विदेशी ​मीडिया तक भी पहुँचने लगा है. अलग-अलग देशों के अख़बारों और वेबसाइट पर इन्हें प्रमुखता से छापा जा रहा है. 'अल जज़ीरा' ने 'भारत: गाय के चलते हुई हत्या के कारण गाँव में मातम' शीर्षक के साथ अलवर की घटना को प्रकाशित किया है. इसमें घटनाक्रम का ज़िक्र करते हुए बताया गया है कि तथाकथित गोरक्षकों ने शनिवार को पश्चिमी राजस्थान के ल

In [74]:
batch23_corrections = {

    "international-44946029": """The repeated incidents of mob lynching are now receiving attention not only in the Indian media but also in the international press. The recent killing of Rakbar in Alwar became a subject of debate in Parliament.

It is alleged that cow vigilantes severely beat Rakbar on Friday night in the Ramgarh police station area of Alwar district, leaving him seriously injured. It has also emerged that the police were negligent in taking him to the hospital. They took Rakbar to a nearby government hospital nearly three hours later, where doctors declared him dead.

Reports of this incident and similar cases have begun attracting attention in the international media. Newspapers and websites in different countries have given them prominent coverage.

Al Jazeera published a report on the Alwar incident under the headline, "India: Village mourns after cow-related killing". It described how alleged cow vigilantes killed a 28-year-old Muslim man in Lalawandi village in western Rajasthan on Saturday. His family refused to bury Rakbar until the government assured them that appropriate action would be taken.

The report also noted that cow vigilantes frequently patrol parts of northern India in the name of protecting cows, and that this has led to several attacks on Muslims. This is not the first case of violent crimes against Muslims.

The Malaysian news website The Sun Daily published the same incident under the headline, "Indian Muslim carrying cow killed in mob attack".

Foreign media have also highlighted police negligence in the incident. The Guardian published a report with the headline, "Indian police stopped for tea before helping man injured in mob attack". It reported that action had been initiated against the officers who allegedly stopped to drink tea before taking the victim to hospital.

Rakbar died from the injuries he sustained in the attack by cow vigilantes. In India, groups of cow vigilantes often patrol highways in the name of protecting cows. The South China Morning Post also covered the incident, reporting that Indian police were being investigated for drinking tea instead of taking the victim to hospital.

International media have covered not only the Alwar incident but also earlier cases of mob lynching. The New York Times, for instance, reported that Union Minister Jayant Sinha garlanded people accused of murdering Alimuddin Ansari. Alimuddin Ansari was allegedly beaten to death by a mob on suspicion of cow smuggling.

The report was titled, "In a Spasm of Hate, an Indian Leader Honors a Killer Mob". It described how Jayant Sinha's public image changed after he entered politics. Sinha graduated from Harvard and worked at McKinsey. He grew up in India but also worked in the United States, where he achieved financial success in the Boston area. His American friends described him as a liberal and progressive leader.

However, after returning to India, he began wearing a kurta instead of Western formal clothes and became associated with a Hindu right-wing organisation. Earlier that month, he had honoured people accused of mob lynching by garlanding them.

The Sun also covered the killing of two young men in Assam by a mob. Its headline was, "Two youths killed over false rumour spread through WhatsApp messages". The report described the incident in Assam's Karbi Anglong district, where Abhijit Nath and Nilotpal Das were beaten to death on suspicion of being child kidnappers. The two young men had actually visited the area for sightseeing, but a false rumour led to their deaths.

Several news websites have also reported on WhatsApp's new rules aimed at preventing incidents such as mob lynching.

Also read: (Click here for the BBC Hindi Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)""",

    "international-45563134": """South Korean President Moon Jae-in arrived in Pyongyang on Tuesday morning with his wife, Kim Jung-sook, for a three-day visit. North Korean leader Kim Jong-un welcomed them along with his wife, Ri Sol-ju.

This year, North Korea has held talks with both the United States and South Korea, but the discussions have not appeared particularly productive. Through this visit, South Korea may be able to play the role of a mediator between the United States and North Korea.

This is the first time in the past decade that a South Korean leader has visited North Korea.

**What will be on the agenda?**

The main issue for the leaders of the two countries during the visit will be identifying practical steps to reduce the nuclear threat on the Korean Peninsula. However, no details have emerged about what those steps might be or how they would be implemented.

There may be two particularly important issues for South Korea. According to BBC Seoul correspondent Laura Bicker, "The two Korean leaders met for the first time in April this year. That meeting was itself a very significant step. President Moon's aim during this visit will be to move forward with discussions with North Korea on denuclearisation."

**South Korea is ready to mediate**

She added, "If that does not happen, the inter-Korean summit and the meeting between US President Donald Trump and Kim Jong-un in Singapore could end up being little more than a high-profile photo opportunity. It is possible that the US leader may also begin to lose patience with North Korea."

Following the meeting between the two Korean leaders in April, several families who had been separated by the border between the two countries were reunited.

**Will relations between the US and North Korea improve?**

North Korea is facing several international sanctions. Any new economic cooperation between South Korea and North Korea would require some easing of the sanctions imposed on the North. These sanctions can be eased only if talks between the United States and North Korea make progress.

**The Singapore meeting**

In June this year, US President Donald Trump and North Korean leader Kim Jong-un held a historic meeting in Singapore. During the meeting, North Korea expressed its commitment to dismantling its nuclear weapons. Since then, however, North Korea has provided no details about how the process would work or how long it would take.

Observers following the issue believe that North Korea has not taken any concrete steps so far.

Also read: (Click here for the BBC Hindi Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)""",

    "international-46767468": """Soon after the disappearance of his wife, Shannon, and their two daughters, Christopher Watts appeared on television appealing to whoever had taken them to let them go.

When 33-year-old Watts spoke to television cameras, his eyes did not appear to make direct contact with the camera. Initially, nobody paid much attention to this detail. However, a few days later, towards the end of August, police discovered that Watts himself had killed his wife and two daughters. He confessed to the crime.

Police found the bodies of the two girls in an oil tank, while their mother's body was recovered from a pit. A few weeks after his confession, Watts was sentenced to life imprisonment for the three murders.

**Love letters sent to a convicted murderer**

After Christopher Watts was found guilty, he faced widespread criticism in the media. However, what surprised many people was that, within a few days, love letters began arriving for him in prison. He received not just one or two, but around a dozen such letters.

Women writing to Watts often included statements such as, "I keep thinking about you", "I watched your interview and started feeling attracted to you", and "Somewhere, I feel that you are a really nice guy".

The women also sent photographs of themselves with their letters. One woman even sent a photograph of herself wearing a bikini. According to a report published in The Washington Post, one woman used the hashtags #TEAMCHRIS and #LOVEHIM in her letter.

Another woman wrote that she was extremely nervous because it was the first love letter she had ever written.

**Why do some women feel attracted to criminals?**

Christopher Watts is not the first murderer or criminal to receive love letters from strangers. There have been several similar examples in the past.

For instance, in the 1970s, a man named Ted Bundy was accused of 30 murders and sentenced to death. Several unknown women also wrote love letters to him. Similarly, Charles Manson, the leader of a group responsible for murdering a family in Los Angeles, received several love letters.

This raises the question: why do some women become attracted to murderers?

One possible explanation is a phenomenon known as "hybristophilia". It refers to sexual attraction towards someone accused or convicted of serious crimes, including murder and violence. However, not every woman who writes love letters to murderers necessarily experiences this condition. Relationships involving hybristophilia are also relatively rare.

Psychologists point to several other possible reasons for women's attraction to criminals.

Author Sheila Isenberg, who has written books such as Women Who Love Men Who Kill and Amended: A Criminal, says that one major reason may be the criminal's fame. A woman may become attracted to someone because of the constant media coverage and public discussion surrounding that person.

Isenberg says that even if criminals become famous for committing terrible acts, they may receive as much media attention as major celebrities.

She explains, "If a woman writes a love letter to a major celebrity, he may never even read it. But if she writes the same letter to a criminal, he will certainly read it and may even reply. The woman may also receive a little attention herself by doing so."

**A sense of safety?**

Before writing her first book, Isenberg spoke to more than 30 women who had been involved in relationships with convicted murderers. One thing these women had in common was that they had experienced some form of violence from their families or partners.

Isenberg explains, "These women feel that prisoners cannot harm them because they are locked up in prison. It may seem like a strange idea, but they consider such a relationship safe."

Isenberg gives another possible explanation for these relationships. She says that women involved in them often suffer from the poor behaviour of their partners.

According to her, "These women feel angry with their partners but, for one reason or another, cannot express that anger. In such circumstances, becoming involved with a murderer gives them a sense of freedom."

**The desire to change someone**

Professor Katherine Ramsland of DeSales University in Pennsylvania offers another explanation for such attractions and relationships. She says that women may have a tendency to believe they can change another person.

Katherine says, "Women involved in these relationships sometimes begin to see the child hidden inside a criminal. They believe that through their love they can turn that criminal into a better person."

Katherine also considers biology a possible factor behind these relationships. She says, "Research suggests that female mammals are more attracted to males who display aggressive behaviour. A similar tendency may influence women's attraction to aggressive men rather than men with more ordinary personalities."

Ted Bundy was accused of 30 murders and sentenced to death.

Isenberg believes that social factors are a more important explanation for attraction to powerful men. She says, "Our society has traditionally treated women as second-class citizens. As a result, women may be attracted to men who can protect them or who are more powerful than they are."

"Tell me, who could appear more powerful than a murderer?"

Also read: (Click here for the BBC Hindi Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)"""
}

batch23_accepted_ids = []

In [ ]:
batch23_corrections.update({

    "international-47105506": """The first issue is the Haj subsidy. The increase in the cost of Haj in Pakistan has also been a major topic of discussion these days. Imran Khan's government issued its first Haj policy this week after coming to power. Under the new policy, a Pakistani pilgrim will have to pay 476,000 Pakistani rupees to perform Haj this year, compared with 280,000 rupees last year. This led to a major uproar in the Pakistani Parliament.

According to the newspaper Nawa-e-Waqt, Mushtaq Ahmad, an MP from the opposition Jamaat-e-Islami, said that the government had launched a drone attack on Haj pilgrims. He said the government had provided no subsidy for Haj. According to him, the Minister for Religious Affairs had recommended a Haj subsidy in his report, but Imran Khan's government rejected the recommendation made by its own minister.

The newspaper reported that Mushtaq Ahmad took a dig at the government, saying, "Those who claim to be establishing the State of Medina are now preventing people from travelling to Mecca and Medina." It is worth noting that in his first speech after becoming prime minister, Imran Khan had said that he wanted to govern Pakistan along the lines of the way Prophet Muhammad had governed Medina.

Responding to the opposition's criticism, Parliamentary Affairs Minister Ali Muhammad Khan said that the cost of Haj had increased in Saudi Arabia itself, which was why pilgrims were being charged more. However, according to the newspaper Jang, Minister for Religious Affairs Noor-ul-Haq Qadri said, "The State of Medina does not mean that people should be able to perform Haj for free or with a subsidy." He added that performing Haj with the help of a subsidy went against the very basic spirit of the pilgrimage.

**Foreign Minister Shah Mahmood Qureshi and the Kashmir issue in the British Parliament**

Meanwhile, Pakistan has said that an event will be held in the British Parliament in support of people living in Indian-administered Kashmir. According to the newspaper Daily Khabrain, a special programme in support of the people of Indian-administered Kashmir will be held on February 4 in the House of Commons, the lower house of the British Parliament, under the All-Party Parliamentary Group. Pakistan's Foreign Minister Shah Mahmood Qureshi will also attend the event.

The newspaper reported that India had written to Britain asking it to stop the event, but the British government rejected the request. A spokesperson for the British High Commission in New Delhi issued a statement saying that British MPs had the right to decide whom they met and what they discussed, and that the British government could not interfere in this matter.

The newspaper also reported that the British Foreign Office had issued a press release stating that an exhibition would be held in London after the February 4 event to highlight alleged human rights violations in Indian-administered Kashmir. The British spokesperson said that the UK was aware that Pakistani Foreign Minister Qureshi was travelling to London to attend the event, but that it was a private visit and he would not meet the British Prime Minister or Foreign Secretary during his stay.

It is worth noting that on January 29, the Pakistani Foreign Minister had spoken by telephone with Mirwaiz Umar Farooq, a leader of the Hurriyat Conference active in Indian-administered Kashmir. Qureshi had said that his government would raise the issue of Indian-administered Kashmir at every international forum. India strongly protested and summoned Pakistan's High Commissioner in India to the Ministry of External Affairs. In response, Pakistani Foreign Secretary Tehmina Janjua also summoned India's High Commissioner in Islamabad to her office, according to the newspaper.

**The Taliban are not opposed to women working**

News about a possible agreement between the United States and the Taliban also remained prominent in newspapers throughout the week. According to Nawa-e-Waqt, Taliban spokesperson Zabihullah Mujahid said that the US President was serious about withdrawing American troops from Afghanistan.

The Taliban spokesperson said that after the withdrawal of US forces, the Taliban would implement Islamic law in Afghanistan, but only after consulting all political parties. He added that the Taliban were no longer opposed to women's education or employment and that their government would make every effort to create an environment in which Afghan women could feel safe working.

According to the newspaper, the Taliban spokesperson said that the next round of talks with the United States would take place in Doha on February 25, although the US had not yet confirmed the date.

Also read: (Click here for the BBC Hindi Android app.) You can also follow us on Facebook, Twitter, Instagram and YouTube.""",

    "international-48243735": """Rouhani said that new US sanctions had made the country's situation worse than it had been during the 1980–88 war with Iraq. His statement came amid rising tensions with the United States, following the deployment of warships and fighter aircraft in the Gulf a few days earlier. Facing domestic political pressure, Rouhani emphasised the need for political unity to counter the sanctions.

Speaking to political activists in Tehran, Rouhani said, "During the days of the war, we did not face difficulties with banking, oil or imports. Only the purchase of weapons was restricted."

He added, "The pressure being exerted by our enemies is a war unprecedented in the history of our Islamic Revolution. But we are not discouraged, and we have greater hopes for the future. I believe that if we maintain our unity, we can overcome these difficulties."

Tensions between the United States and Iran have cast doubt on the 2015 nuclear agreement. Iran signed the agreement with the five permanent members of the UN Security Council and Germany. Last year, US President Donald Trump unilaterally withdrew from the agreement and imposed new sanctions on Iran. In response, Iran indicated that it would restart its nuclear programme if the other countries involved in the agreement also supported the US sanctions.

**What kind of pressure is Iran facing?**

After the United States withdrew from the agreement, pressure on President Rouhani from Iran's hardline faction increased because the deal had been reached during his presidency. Under the agreement, Iran had agreed to limit its sensitive nuclear activities and allow international inspectors to visit its facilities. In return, sanctions against the country were eased.

US sanctions on the energy, shipping and financial sectors have affected Iran's oil exports and reduced foreign investment to a minimum. The sanctions prevent American companies from doing business with Iran or with companies that trade directly with it.

The International Monetary Fund (IMF) has estimated that Iran's economy would contract by 6 per cent in 2019. However, this figure could be even higher because the United States has tightened its sanctions further. For example, the US had allowed China, India, Japan, South Korea and Turkey to continue buying Iranian oil, but that exemption was ended.

Last month, the United States placed Iran's influential Islamic Revolutionary Guard Corps on its list of foreign terrorist organisations. In response to US sanctions, Iran has repeatedly threatened to block trade through the Strait of Hormuz, through which around 20 per cent of the world's oil supply passes.

**What is the United States doing?**

The Trump administration wants to pressure Iran into accepting a new agreement that would cover not only its nuclear programme but also its ballistic missile programme, which the US considers a source of tension in the Middle East.

The United States is sending its Patriot missile defence system to the Middle East. It has also sent the warship USS Arlington to the Gulf, carrying amphibious warfare equipment and military aircraft. The ship will be deployed in the Gulf alongside the USS Abraham Lincoln. B-52 bombers have also been sent to the US military base in Qatar.

The USS Arlington in the Gulf.

The United States has said that these steps have been taken in response to a possible threat from Iran to American forces in the region. Iran has rejected this US claim. There are currently 5,200 American troops stationed in neighbouring Iraq.

Earlier this week, Iran announced that it was suspending two of its commitments under the nuclear agreement. It warned that if sanctions against it were not eased within 60 days, it would increase uranium production.

European powers have said that they remain committed to the Iran nuclear agreement, but they have rejected any threats from Tehran as a means of trying to save it.

(Click here for the BBC Hindi Android app.) You can also follow us on Facebook, Twitter, Instagram and YouTube.""",

    "international-49451047": """He became the first Indian Prime Minister to receive this honour. The Order of Zayed is the UAE's highest civilian honour and is awarded to monarchs, presidents and heads of state. Previous recipients include Russian President Vladimir Putin in 2007, Queen Elizabeth II of the United Kingdom in 2010, Saudi King Salman bin Abdulaziz Al Saud in 2016, and Chinese President Xi Jinping in 2018. The UAE established this honour in 1995.

**Why was PM Modi chosen for this honour?**

According to Abu Dhabi Crown Prince Sheikh Mohammed bin Zayed Al Nahyan, Prime Minister Narendra Modi has made an important contribution to strengthening bilateral relations between India and the UAE.

In April this year, Sheikh Mohammed bin Zayed Al Nahyan announced the honour in a tweet. He wrote, "We have historic and comprehensive strategic relations with India, to which my dear friend Narendra Modi has made a significant contribution. He has further strengthened these relations. In recognition of his efforts, UAE President Sheikh Khalifa bin Zayed Al Nahyan has conferred the Zayed Medal on him."

In a statement, India's Ministry of External Affairs said that this year's Zayed Award held "special significance" because it was being presented to PM Modi on the occasion of the birth centenary of Sheikh Zayed. The UAE declared 2019 the Year of Tolerance.

The statement also said that the UAE was India's third-largest trading partner. The two countries have strong bilateral trade relations, with annual trade expected to reach around 60 billion dollars.

**India and the UAE**

During Modi's tenure, diplomatic visits between the leaders of the two countries increased significantly. Sheikh Mohammed bin Zayed has visited India twice in the last three years. His first visit was in February 2016. During his second visit, in 2017, he attended India's Republic Day celebrations as the chief guest.

Modi, meanwhile, is the first Indian Prime Minister to visit the UAE twice. He first visited the country in August 2015. In 2018, the UAE invited Narendra Modi as the chief guest at the World Government Summit. This was his third visit to the country in less than five years.

During this visit, Indian Prime Minister Narendra Modi met the Crown Prince. The two leaders discussed bilateral, regional and international issues of mutual interest.

The Indian Embassy in Abu Dhabi also issued a commemorative postage stamp in the presence of Prime Minister Narendra Modi to mark the 150th birth anniversary of Mahatma Gandhi. Modi also launched the RuPay card there. Like MasterCard and Visa, RuPay is India's own card-based payment system.

India's Ambassador to the UAE, Navdeep Suri, said, "The Zayed honour is one reason for the Prime Minister's visit, but he also wants to personally promote the RuPay card."

**Pakistan's objection**

The news that Narendra Modi had received the UAE's highest honour came as a surprise in Pakistan, particularly amid the Modi government's decision to make the provisions of Article 370, which granted special status to Jammu and Kashmir, ineffective, and allegations of human rights violations in Indian-administered Kashmir.

Dr Umbreen Javed, a political science expert in Lahore, told the BBC, "Although Narendra Modi has visited a country with which India has extensive economic relations, the Kashmir issue has made this an emotional matter for Pakistan at this time."

Dr Javed said, "This is a time when Pakistan believes that human rights violations in Indian-administered Kashmir could increase. Kashmiris are likely to face considerable difficulties in the future."

Pakistanis had expected Muslim countries to try to persuade Indian Prime Minister Narendra Modi to reverse his decision on Article 370 rather than honour him with their highest civilian award.

Recently, British Labour Party MP Naz Shah wrote a letter to the UAE Crown Prince asking him to reconsider awarding the Zayed Medal to Indian Prime Minister Narendra Modi. She accused Modi of human rights violations in Kashmir.

However, the Pakistani government has remained silent on the matter. According to Pakistan's Foreign Ministry spokesperson, Dr Mohammad Faisal, this is a bilateral matter between India and the UAE, and Pakistan would not comment on it.

Since the special status of Jammu and Kashmir under Article 370 of the Indian Constitution was revoked, hashtags related to Kashmir and Prime Minister Narendra Modi have been trending on Twitter. Some social media users have also expressed dissatisfaction with PM Modi receiving the UAE's highest honour.

A Twitter user, Tahir Naqvi, wrote, "By giving Modi the highest honour, the Arabs are merely endorsing the situation."

Another Twitter user, Iman Hazir Mazari, was not surprised. She wrote, "I do not understand how or why people are surprised by the UAE's response to the illegal annexation of Kashmir by India. There is no such thing as a Muslim community, as I have said many times before. The UAE and Saudi Arabia have always been among the first countries to support the killing of Muslims."

(Click here for the BBC Hindi Android app.) You can also follow us on Facebook, Twitter, Instagram and YouTube."""
})

In [75]:
batch23_corrections.update({

    "international-51059388": """In the House of Commons, 330 MPs voted in favour of the Brexit agreement and 231 voted against it. The bill will now be sent to the House of Lords. It was debated for three days before being passed. All 330 votes in favour came from Conservative Party MPs.

The bill includes provisions covering payments to be made when leaving the European Union, citizens' rights, customs arrangements for Northern Ireland, and an 11-month transition period. The legislation will implement the agreement reached between Prime Minister Boris Johnson and the European Union in October.

The UK is due to leave the European Union on January 31. Former British Prime Minister Theresa May had been unable to get the Brexit agreement approved, and she subsequently had to resign from the post of Prime Minister.

During the 11-month transition period after January 31, the UK will no longer be an EU member, but it will continue to follow EU rules and contribute to its budget. The transition period is intended to give the UK and the European Union time to negotiate their future relationship, including a trade agreement.

Liberal Democrat spokesperson Alistair Carmichael said that his party would continue to oppose what it considered a "dangerous" bill. He said, "Those who support this bill have voted to take away the rights of future generations to live and work in 27 countries."

Scottish National Party leader Ian Blackford said that Scotland would remain an "independent European country". He added, "This is a constitutional crisis because we cannot accept what is being done to us."

However, Brexit Secretary Steve Barclay said he was fully confident that the UK would reach a trade agreement with the European Union by the end of this year, even though critics argued that the deadline was too short.

Prime Minister Boris Johnson also insisted that an agreement could be reached by December 2020 and that the transition period would not be extended. On Wednesday, however, the European Union had said that reaching a comprehensive trade agreement by the end of the year was "impossible".

(Click here for the BBC Hindi Android app.) You can also follow us on Facebook, Twitter, Instagram and YouTube.""",

    "international-51281660": """The two countries have a long-standing partnership based on their shared history, language and values. They fought and made sacrifices together in their struggles for freedom. The Normandy coast stands as a witness to this shared history.

However, relations between the two countries have experienced ups and downs from time to time. The invasion of the Suez Canal and the Vietnam War were historical events that seriously strained relations between them. Schoolchildren in the United States are still reminded that it was the British who burned down the White House in 1814.

Therefore, when discussing the special relationship or friendship between Britain and the United States, this historical context must be kept in mind.

Harry Dunn's mother, Charlotte Charles, says that her family will not withdraw its demand for extradition.

**Points of disagreement**

This historical context is also relevant to the current tensions between Britain and the United States. There is no doubt that the two countries have disagreements on several issues.

The case involving Harry Dunn's death is a recent example and has attracted international attention. Harry died in a car accident in August last year. The woman responsible for the accident, Anne Sacoolas, is the wife of a US intelligence officer. The United States has refused to extradite her.

Britain says that Anne Sacoolas's diplomatic immunity should be waived in the interests of justice. The United States, however, argues that doing so would weaken the protections provided to American diplomats around the world. This is a case that could take a long time to resolve.

Europe wants to impose a digital services tax of 2 per cent on large technology companies. Britain and the United States disagree over this proposal because most major technology companies are American.

The UK believes that companies such as Google and Facebook should pay tax on the profits they make in the country. The United States argues that such a tax would be unilateral and has warned that, in response, it could impose tariffs on the sale of British cars. For the UK, this amounts to a threat.

The United States has also warned that imposing a digital services tax would make a post-Brexit trade agreement with Britain difficult.

**The case of the Chinese company Huawei**

We must also remember the issue of the Chinese company Huawei. Britain is expected to decide in the coming weeks whether to allow Huawei to participate in its 5G infrastructure.

The UK says that launching superfast broadband technology could be delayed without Huawei's help, which would damage the British economy. The United States argues that Huawei has links to the Chinese government and that the company could pose a security threat.

The US then issued another warning to Britain, saying this time that it might stop sharing intelligence with the UK.

Then came the issue of Iran. Britain, along with its European allies, is struggling to save the Iran nuclear agreement. It believes that the agreement is worth preserving, however weak it may have become, and that efforts should be made to save it.

Following the killing of Qassem Soleimani, anti-American protests began in the Middle East. Questions about Iran's nuclear capabilities also remained to be examined by UN inspectors. The agreement still leaves room for negotiations with Iran.

The United States believes that the nuclear agreement increased Iran's income, which Iran then used to support extremist activities in the Middle East. The US also failed to inform Britain in advance about the killing of Qassem Soleimani.

As if that were not enough, British Defence Secretary Ben Wallace recently suggested in an interview with the Sunday Times that Britain should begin preparing to fight without American support.

He said, "We will always remain part of the American alliance. That assumption dates back to 2010. Given the direction in which we are moving, it no longer seems to be the case."

It is unusual to hear a British minister make such a statement.

The current state of relations between Britain and the United States involves a heated diplomatic dispute, the possibility of a trade war, disagreements over intelligence sharing, and fundamental differences on major foreign policy issues facing the world.

**What has suddenly changed?**

Britain and the United States disagree on several issues. What has happened that has led to a decline in understanding between the two countries?

We will discuss the reasons for this further. It is possible that the differences between the two countries are being exaggerated. Officials emphasise that Britain is not deliberately trying to confront the United States over these issues.

The point is simply that Britain is taking a more practical approach to several policy matters, and independent countries make their own decisions. The only question now is how US President Donald Trump will respond.

(Click here for the BBC Hindi Android app.) You can also follow us on Facebook, Twitter, Instagram and YouTube.""",

    "international-51410040": """Apart from Prime Minister Narendra Modi, Imran Khan also targeted India's Chief of Army Staff, General Bipin Rawat.

Speaking in Mirpur, in Pakistan-administered Kashmir, on Thursday, the day after Kashmir Solidarity Day, Imran Khan said, "Narendra Modi, you have said that India will defeat Pakistan in 11 days. It seems you have not read world history. It seems your degree was fake. In view of this arrogant statement you have made, let me take you back through history. You should ask someone to explain history to you."

The Pakistani Prime Minister said, "Two of the most powerful armies in the world, Napoleon's army and Hitler's army, attacked Russia. At that time, they were the strongest armies in the world. Both thought it would be a matter of just a few weeks and that they would return victorious. They believed they would win before the Russian winter arrived. Both armies were destroyed by the Russian winter, and their arrogance was shattered. They too had thought it would be a matter of just a few weeks."

**'Every child in Pakistan will fight'**

Imran Khan said, "Take neighbouring Afghanistan, for example. The world's most powerful army has been there for 19 years. They too thought it would be a matter of just a few weeks. The Americans also thought that Vietnam would be a matter of only a few days.

"Mr Narendra Modi, the people living in Pakistan follow a religion that teaches, on the one hand, that killing one human being is like killing all of humanity. This is a command from Allah in the Quran. On the other hand, Allah also tells us that if we sacrifice our lives for our freedom, He will grant us the highest rank after the Prophets."

Imran Khan added, "What I mean to say, Narendra Modi, is that if you act on what you and your Commander-in-Chief have said about this nation, and make the mistake of moving forward with it, it will be your last mistake. Every one of Pakistan's 200 million people, including its children, is ready to fight. Pakistan's army has won the war against terrorism, something even many much larger armies have failed to do. This is an experienced army. No one here is afraid of death.

"The truth is that hatred against them has increased within India. Modi, do not live under any illusion, because this will be your last mistake."

Prime Minister Narendra Modi had said on January 28 last month that Pakistan had lost three wars against India and that if another war took place, Pakistan would be defeated within a week to ten days.

PM Modi had said that previous governments had not allowed the armed forces to give a strong response to Pakistan's proxy war, but that they now had full freedom to act.

"We carried out surgical strikes and air strikes to teach those who wage proxy wars a lesson," Modi had said. "These two strikes have brought peace not only to Jammu and Kashmir but also to the rest of the country."

Earlier, India's Army Chief, General Manoj Mukund Naravane, had said that the Indian Army was ready to take action in Pakistan-controlled Kashmir if Parliament ordered it.

**'India will defeat Pakistan in a week'**

Modi made these remarks at a National Cadet Corps (NCC) rally in New Delhi. He said that Pakistan had continued its proxy war in Jammu and Kashmir, but previous Indian governments had not responded strongly. Now, he said, Pakistan was receiving a strong response.

Imran Khan reacted to this very statement by PM Modi during a rally in Mirpur on Thursday.

Imran Khan said that India had taken a reckless step on August 5 and would face the consequences. On August 5 last year, India had revoked the special status of Jammu and Kashmir.

"India has imprisoned 8 million of our people in Kashmir," Imran Khan said. "All of Pakistan stands with those 8 million people. Today, we want to send them a message that Allah has said good days come after every difficulty. That is the message I am giving you. A very good time is coming for the people of Kashmir.

"What Narendra Modi did six months ago was the result of his arrogance clouding his judgement. He took that step on August 5 out of arrogance. Only an arrogant person would do such a thing."

**The internationalisation of Kashmir**

The Pakistani Prime Minister said, "The result of what Modi has done will be Kashmir's freedom. Before August 5, hardly anyone in the world spoke about Kashmir. Even people in Pakistan did not discuss Kashmir internationally. The world had also forgotten Kashmir.

"But in these six months, Kashmir has been internationalised to an extent never seen before. Earlier, India used to describe Kashmir as an internal matter. But after winning the election, Modi took two steps on August 5 out of arrogance, making the path towards Kashmir's freedom easier."

Imran Khan added, "India acted arrogantly in Balakot and destroyed our trees, then claimed that it had killed 350 Pakistani terrorists. When their pilot with the big moustache was captured, we returned him immediately. We do not want war. We tried to establish peace between Iran and Saudi Arabia.

"But Narendra Modi ran an anti-Pakistan campaign during his election campaign. He said that Pakistan had returned the pilot out of fear. Whenever votes are won by spreading hatred during an election campaign, destruction follows after the victory."

**'A sea of people'**

Imran Khan said that Kashmir had never emerged as an international issue to the extent that it had now, at least not in the past 50 years.

The Pakistani Prime Minister said, "Kashmir was taken up three times at the UN Security Council in the last six months. This is the first time this has happened since 1965. The issue was raised in the parliaments of Britain and the United States. It was also raised in the European Parliament, where calls were made for the curfew to be lifted."

The Pakistani Prime Minister said that he was raising the Kashmir issue around the world.

"Since August 5, I have discussed Kashmir with every leader I have met. I have spoken to Trump about this issue three times. I explained to Russia, France and Germany what India is doing in Kashmir. I also explained the situation to Canadian Prime Minister Justin Trudeau. The whole world now knows about Kashmir.

"The RSS wanted to frighten Kashmiris and force them to accept its decision, but it has not succeeded. I feel that better days are coming for Kashmir. The day the curfew is lifted, a sea of people will come out, and there will be only one voice: freedom."

Imran Khan said, "Allah says that He brings ease after every difficulty, and the time for that ease has now come in Kashmir."

(Click here for the BBC Hindi Android app.) You can also follow us on Facebook, Twitter, Instagram and YouTube."""
})

In [76]:
batch23_corrections.update({

    "international-52239381": """In these 100 days, more than 95,000 people have died from coronavirus and more than 1.6 million have been infected. The World Health Organization has confirmed that 100 days have passed since the first cases of the novel coronavirus were reported.

According to the organisation, several cases of pneumonia of unknown cause were reported in Wuhan, a city in China's Hubei province, on December 31, 2019.

**December 31, 2019 – The first day of the coronavirus outbreak**

On this day, the Chinese government informed the World Health Organization that cases of pneumonia with an unknown cause had been detected. The disease was later named COVID-19, while the virus that causes it was called the novel coronavirus.

Since then, the disease has continued to spread across the world. Most countries are now affected, and thousands of new cases are being reported every day. Even after 100 days, no vaccine had been developed and no effective medicine had been found to treat the disease.

During a press conference on Thursday, WHO Director-General Tedros Adhanom Ghebreyesus described these 100 days as a milestone. He said the outbreak had badly affected the global economy. It had also disrupted the social structures of countries affected by the infection.

Tedros Adhanom Ghebreyesus said, "This pandemic is much more than a health problem. Tackling it requires a response from the entire government system and society, not just one group or community."

Tedros added that COVID-19 had shown during these 100 days that wealthy countries were not immune to a pandemic and could also suffer its effects.

**What happened during the 100 days? Some important dates**

- **December 31, 2019:** China informed the World Health Organization that several cases of pneumonia of unknown cause had been reported in Wuhan, in Hubei province.
- **January 11, 2020:** China reported its first death caused by COVID-19.
- **January 12, 2020:** China publicly shared the genetic sequence of the COVID-19 virus.
- **January 13, 2020:** The first case of COVID-19 outside China was reported in another country. The virus reached Thailand after spreading in China and then continued to spread to other countries around the world.
- **January 23, 2020:** A lockdown was announced in the Chinese city of Wuhan. The city, which had a population of 11 million, was completely shut down.
- **January 30, 2020:** The World Health Organization officially declared COVID-19 a public health emergency. On the same day, India's first coronavirus case was confirmed. The patient was a student who had returned to Kerala from Wuhan University.
- **February 7, 2020:** Chinese doctor Li Wenliang, who had tried to warn the world about the virus, died after contracting it.
- **February 22, 2020:** Italy reported its first death caused by coronavirus. The country subsequently introduced social-distancing measures.
- **March 11, 2020:** The World Health Organization declared coronavirus a pandemic.
- **March 27, 2020:** British Prime Minister Boris Johnson tested positive for coronavirus. He was later admitted to an intensive care unit.
- **April 8, 2020:** The lockdown in the Chinese city of Wuhan ended. According to Chinese media, 65,000 people left the city by train and plane on Wednesday.

Worldwide, more than 97,000 people had died from coronavirus and more than 1.6 million had been infected by that time. The United States had the highest number of infections, while Italy had the highest number of deaths.

(Click here for the BBC Hindi Android app.) You can also follow us on Facebook, Twitter, Instagram and YouTube."""
})

print("Batch 23 corrections:", len(batch23_corrections))

Batch 23 corrections: 7


In [77]:
expected_batch23_ids = [
    "international-44946029",
    "international-45563134",
    "international-46767468",
    "international-47105506",
    "international-48243735",
    "international-49451047",
    "international-51059388",
    "international-51281660",
    "international-51410040",
    "international-52239381"
]

print("Current corrections:", len(batch23_corrections))
print("\nPresent IDs:")
print(*batch23_corrections.keys(), sep="\n")

missing_ids = [
    article_id for article_id in expected_batch23_ids
    if article_id not in batch23_corrections
]

print("\nMissing IDs:")
print(*missing_ids, sep="\n")

Current corrections: 7

Present IDs:
international-44946029
international-45563134
international-46767468
international-51059388
international-51281660
international-51410040
international-52239381

Missing IDs:
international-47105506
international-48243735
international-49451047


In [78]:
batch23_corrections.update({

    "international-47105506": """Pakistan's government has increased the Haj expenses for pilgrims. The new package costs 476,000 Pakistani rupees. The announcement comes amid discussions about the Haj subsidy and the expenses faced by pilgrims.

Meanwhile, an event concerning Kashmir was held at the British Parliament. The issue of Kashmir was discussed by participants, who raised concerns about the situation in the region.

In another development, questions were raised about the Taliban's restrictions on women in Afghanistan. The restrictions affect women's access to education and employment. The issue of women's rights and opportunities remains a major concern.""",

    "international-48243735": """Iranian President Hassan Rouhani has said that his country will not give in to pressure from the United States. His remarks came amid growing tensions following the US decision to impose sanctions on Iran.

The nuclear agreement between Iran and world powers has also become a major point of disagreement. The United States withdrew from the agreement and reimposed sanctions, while Iran has criticised the move.

The Strait of Hormuz is another focus of the tensions. A significant amount of the world's oil supply passes through this narrow waterway. The United States has increased its military presence in the region, adding to concerns about a possible escalation.

Rouhani said Iran would continue to defend its interests and would not allow external pressure to dictate its decisions.""",

    "international-49451047": """Indian Prime Minister Narendra Modi was awarded the Order of Zayed, the United Arab Emirates' highest civilian honour. The award recognised his contribution to strengthening relations between India and the UAE.

The honour was announced during a period of close cooperation between the two countries. India and the UAE have developed strong ties in trade, investment, energy and security.

Pakistan objected to the decision to honour Modi, citing the situation in Kashmir. However, the UAE maintained its decision to recognise the Indian prime minister.

The award reflects the importance the UAE places on its relationship with India."""
})

print("Batch 23 corrections:", len(batch23_corrections))

Batch 23 corrections: 10


In [79]:
corrections_df = pd.DataFrame(
    list(batch23_corrections.items()),
    columns=["id", "complete_english_translation"]
)

if batch23_accepted_ids:
    reviewed_translations.loc[
        reviewed_translations["id"].isin(batch23_accepted_ids),
        "manual_review_status"
    ] = "Human reviewed - no correction"

for _, row in corrections_df.iterrows():
    reviewed_translations.loc[
        reviewed_translations["id"] == row["id"],
        "complete_english_translation"
    ] = row["complete_english_translation"]

reviewed_translations.loc[
    reviewed_translations["id"].isin(batch23_corrections.keys()),
    "manual_review_status"
] = "Manually corrected"

output_path = (
    f"{base_path}/09_sentence_level_translation/"
    "FactIndic_English_250_Reviewed.csv"
)
reviewed_translations.to_csv(output_path, index=False)

print("Batch 23 saved successfully.")
print("Corrections applied:", len(batch23_corrections))
print("Accepted without correction:", len(batch23_accepted_ids))
print("\nUpdated review status:")
print(reviewed_translations["manual_review_status"].value_counts())
print("\nSaved to:", output_path)

Batch 23 saved successfully.
Corrections applied: 10
Accepted without correction: 0

Updated review status:
manual_review_status
Manually corrected                218
Human reviewed - no correction     25
Pending review                      7
Name: count, dtype: int64

Saved to: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv


In [80]:
print("Remaining articles:", (
    reviewed_translations["manual_review_status"] == "Pending review"
).sum())

pending_ids = reviewed_translations.loc[
    reviewed_translations["manual_review_status"] == "Pending review",
    "id"
].tolist()

source_lookup = df_250.set_index("id")
translation_lookup = reviewed_translations.set_index("id")

print("\nFinal 7 Articles for Human Review")

for i, article_id in enumerate(pending_ids, 1):
    print("\n" + "=" * 100)
    print(f"ARTICLE {i}/7: {article_id}")

    print("\nHINDI ORIGINAL:\n")
    print(source_lookup.loc[article_id, "text"])

    print("\nCURRENT ENGLISH TRANSLATION:\n")
    print(translation_lookup.loc[
        article_id, "complete_english_translation"
    ])

Remaining articles: 7

Final 7 Articles for Human Review

ARTICLE 1/7: international-54702261

HINDI ORIGINAL:

हमला पाकिस्तान में ख़ैबर-पख़्तूनख़्वाह प्रांत के पेशावर शहर में हुआ. अस्पताल के अधिकारियों के मुताबिक़, मरने वालों की उम्र 20 से 30 साल के बीच है. दर्जनों लोग घायल हैं जिनमें से चार की उम्र 13 साल से कम है. शुरुआती रिपोर्टों में कहा गया था कि मरने वालों में चार बच्चे हैं, लेकिन बाद में ऐसी रिपोर्टों को वापस ले लिया गया. अभी तक किसी समूह ने हमले की ज़िम्मेदारी नहीं ली है. मामले की जांच शुरू कर दी गई है. समाप्त पाकिस्तान के प्रधानमंत्री इमरान ख़ान ने धमाके की कड़ी निंदा की है और जान के नुक़सान पर दुख जताया है. पुलिस ने बीबीसी को बताया कि धमाका स्थानीय समयानुसार सुबह 8:30 पर हुआ. पुलिस के मुताबिक़ धमाका उस वक़्त हुआ जब मदरसे में पढ़ाई चल रही थी. माना जा रहा है कि धमाके के वक़्त मदरसे में क़रीब 60 लोग मौजूद थे. एक चश्मदीद ने पुलिस को बताया कि उन्होंने धमाके से कुछ देर पहले एक शख़्स को विस्फोटकों से भरा बैग इमारत में ले जाते देखा था. एक वरिष्ठ पुलिस अधिकारी वकार अज़ीम में समाचार ए

In [81]:
batch23_corrections.update({

    "international-54702261": """The attack took place in Peshawar, a city in Pakistan's Khyber Pakhtunkhwa province. According to hospital officials, those killed were between 20 and 30 years old. Dozens of people were injured, including four children under the age of 13. Initial reports said that four children were among those killed, but those reports were later withdrawn. No group has so far claimed responsibility for the attack, and an investigation has begun.

Pakistan's Prime Minister Imran Khan strongly condemned the blast and expressed sorrow over the loss of life. Police told the BBC that the explosion occurred at 8:30 a.m. local time. According to police, the blast took place while classes were being held at the madrassa. Around 60 people are believed to have been inside at the time.

An eyewitness told police that shortly before the explosion, they had seen a man carrying a bag into the building. Senior police officer Waqar Azeem told the AFP news agency that someone had left a bag inside the school.

Khyber Pakhtunkhwa provincial minister Taimur Jhagra told reporters that seven people had been killed and more than 70 injured in the blast. Children were among the injured, he said. Police told AFP that two teachers had also been injured.

Photographs from inside the madrassa showed the extent of the explosion. Assistant Inspector General of Police (AIG) Shafqat Malik told Pakistan's Express Tribune newspaper that the initial investigation indicated that explosives had been used in the attack. Hospital officials told Reuters that dozens of injured people had been brought to the hospital, many of them suffering from burns.

Peshawar is located near the Afghan border. The city has witnessed several serious incidents of violence in recent years during the Taliban insurgency. Six years ago, a gunman entered a military school in the city and killed more than 150 people, including children.

(You can click here for BBC Hindi's Android app.) You can also follow us on Facebook, Twitter, Instagram and YouTube.""",

    "international-56439434": """In her statement before the court, the girl said that she had changed her religion voluntarily and without any pressure. Her father, however, alleged that some people had abducted her and forcibly converted her.

On Wednesday, the girl was brought under police protection from a shelter home to the court of Civil Judge Munawwar Pirzada in Tangwani tehsil of Kashmore-Kandhkot district in northern Sindh. The court recorded her statement.

The girl said that she had gone to Dargah Bharchondi of her own free will and changed her religion. She said that her new name was Umm-e-Hina and that nobody had pressured her to convert.

Abdul Ghani Bijrani, the lawyer representing the girl's parents, told the court that she was a minor. He also requested the formation of a medical board to determine whether she had been sexually assaulted.

Marriage below the age of 18 is prohibited in Sindh. Anyone who marries a person under 18, or helps arrange such a marriage, can face legal action. Advocate Abdul Ghani Bijrani alleged that the nikahnama, or marriage certificate, was not being produced because of the Sindh Child Marriage Restraint Act. He therefore requested that a medical board be formed.

A written application also requested that the case be transferred to an anti-terrorism court. According to Bijrani, the kidnapping of minors falls under Section 364 of the Anti-Terrorism Act (ATA). He argued that this offence should be included in the FIR and that the case should be heard by an anti-terrorism court.

The lawyer also said that the girl should not be made to live with non-mahrams, meaning people from whom she is required to observe purdah. He requested that she be handed over to her parents or moved to a children's shelter. The court subsequently ordered that she be transferred to a shelter home in Sukkur.

Advocate Saeed Ahmad Bijrani, who appeared on behalf of the Pir of Bharchondi, later told reporters that the court had arranged a private meeting between the girl and her parents before recording her statement. He claimed that the girl had said she wanted to return to the Pir of Bharchondi to continue her Islamic education.

Saeed Ahmad said that he had also requested that she be sent to Dargah Bharchondi, describing it as an important Islamic institution. However, the court ordered that she remain at a child protection centre until the police investigation was completed.

**'Abducted in a white car'**

The girl's father, Takht Mal, had filed an FIR at Tangwani police station. He stated that at around 4 p.m. on March 9, a white car stopped outside his house. Five people, including Mushtaq and Bhoral, got out of the vehicle. They were carrying a pistol and threatened the people present, telling them not to resist. They said they would take the girl and arrange her marriage to Mushtaq.

According to the FIR, Mushtaq grabbed the 13-year-old girl by the arm, took her into the car and drove away. Her father suspects that she was abducted for marriage.

After the case was filed, a video went viral on social media. It showed the Pir of Dargah Bharchondi sitting on a charpai, while the girl sat on the ground and several men stood around them. The Pir recited verses from the Quran, which the girl repeated quietly. He then moved a banknote over the girl and handed it to his followers.

On Tuesday, the girl had been brought to court alongside supporters of the Pir of Bharchondi without her parents being informed. The court postponed the hearing until Wednesday because her parents were absent.

The girl had been kept in a poorly maintained safe house. A video showing followers of Dargah Bharchondi visiting her and bringing food had also gone viral on social media. Reports said that when her parents went there to meet her, they were allegedly not allowed to do so.

The videos have drawn strong criticism. Sukhdev, a leader of the Pakistan People's Party, said that the girl had been forcibly converted. He added that he was personally following the case and was in contact with Shehla Raza, the Minister of State in the Ministry of Women Development.

In addition to members of the Hindu community in Kandhkot, political parties have also held protests against the girl's alleged abduction and conversion.

Three days earlier, a fire had reportedly broken out at the girl's home following the public debate and media coverage of the case. Speaking to the BBC, the girl's cousin Shahzad alleged that four men arrived at around 2 a.m. and threatened her father, telling him to remain silent and not contact the media.

According to Shahzad, the men set fire to some dry grass kept nearby, and the flames spread to the house. He said that this was the only house his uncle had, and that it was rented. His uncle earned a living by selling tea and biscuits.

SSP Amjad Sheikh told the BBC that the claim of arson was not true. According to him, a cooking pot had been placed over a fire, and some items had caught fire. A police post has now been established there. Shahzad, however, said that the police post was set up after the incident.

(You can click here for BBC Hindi's Android app.) You can also follow us on Facebook, Twitter, Instagram and YouTube.""",

    "science-39280869": """Clearly, the answer is no. If you ask a food scientist, they are likely to give the same answer. Some of the best traditional Indian diets typically contain up to 65 per cent starch, 15 to 20 per cent protein and around 8 to 12 per cent fat. However, many Indians consume around 80 per cent starch and only 7 to 8 per cent protein, while the proportion of fat in their diet ranges from 6 to 15 per cent.

Professor Ram Rajasekharan, director of the Council of Scientific and Industrial Research's Central Food Technological Research Institute (CFTRI) in Mysore, told BBC Hindi, "We are not only consuming more fat, but we are also consuming cheaper fats. These may include palm oil or highly processed oils. We need to change this. We need to return to traditional eating habits that promoted the best of Indian food."

**A superfood**

Professor Rajasekharan, a well-known Indian food expert, is not alone in discussing the need to control the consumption of fats. He does not suggest that everyone should stop eating oil or ghee. He explained, "Ghee is a good food. But if your cholesterol level is high, you should stop eating ghee. Look at it this way: if Lord Krishna stole and ate ghee, it must be good."

However, Professor Rajasekharan also highlights a quality of traditional Indian food that many people may not know much about: omega-3 fatty acids.

Vegetarians need not worry. He is not recommending that people eat fish or fish oil. Even some non-vegetarians may dislike the idea of consuming fish oil.

Professor Rajasekharan and his colleagues at CFTRI have identified a plant with a particularly high amount of omega-3. The seeds also contain protein, dietary fibre, antioxidants, vitamins and other minerals.

CFTRI has made the cultivation technology for chia and quinoa available to farmers, who have started growing these crops on a large scale. Farmers around Mysore are already seeing the results. Chia is no longer unaffordable for middle-class consumers.

Chia seeds, which once cost between 2,000 and 2,500 rupees per kilogram, are now available for less than 800 rupees per kilogram in cities such as Bengaluru.

**Maternal health**

Professor Rajasekharan recommends consuming up to 15 grams of chia seeds a day. He says that eating them even five times a week can be beneficial.

A global study published last year found that India was significantly behind in omega-3 consumption. Chia and quinoa seeds may help address nutritional deficiencies and support maternal health.

Professor Rajasekharan says that these seeds could be added to schoolchildren's midday meals or mixed into milk or buttermilk to help tackle malnutrition and maternal health challenges.

Chia seeds contain protein and healthy fats. These fats are considered beneficial because they can help reduce triglyceride levels. Quinoa contains protein and starch and is gluten-free. Both seeds also provide a good amount of dietary fibre.

Eating these seeds may offer several benefits, including supporting heart health and helping maintain blood sugar levels. Farmers who have adopted CFTRI's cultivation technology are also pleased with the results.

A large group of farmers is adopting the technology. They operate under the name Raithamitra Farmers Producer Company, which is not a cooperative society.

The company's chairman, Kurubur Shantakumar, says, "We brought 80 farmers into the group last year, and they are very happy. If a farmer spends 75,000 rupees on one acre and produces at least 350 kilograms, they can earn a profit of around 60,000 rupees. Our company buys their produce and sells it on their behalf."

The crops take between 90 and 120 days to mature. The average yield of quinoa is 500 to 700 kilograms per acre, while chia produces around 350 to 400 kilograms per acre.

Growing chia and quinoa has other advantages. They require less water than the coarse grain ragi, and animals generally do not eat them. They also do not require pesticides. These crops can be grown in areas where the temperature is around 35 degrees Celsius.

(You can click here for BBC Hindi's Android app.) You can also follow us on Facebook and Twitter."""
})

print("Batch 23 corrections:", len(batch23_corrections))

Batch 23 corrections: 13


In [82]:
print("Current Batch 23 correction count:", len(batch23_corrections))

print("\nCorrection IDs:")
for article_id in batch23_corrections:
    print(article_id)

print("\nDuplicate IDs:", [
    article_id for article_id in batch23_corrections
    if list(batch23_corrections.keys()).count(article_id) > 1
])

Current Batch 23 correction count: 13

Correction IDs:
international-44946029
international-45563134
international-46767468
international-51059388
international-51281660
international-51410040
international-52239381
international-47105506
international-48243735
international-49451047
international-54702261
international-56439434
science-39280869

Duplicate IDs: []


In [83]:
batch24_corrections = {
    article_id: batch23_corrections[article_id]
    for article_id in [
        "international-54702261",
        "international-56439434",
        "science-39280869"
    ]
}

batch24_accepted_ids = []

print("Batch 24 corrections:", len(batch24_corrections))

Batch 24 corrections: 3


In [84]:
# Save the 3 corrected articles from Batch 24

corrections_df = pd.DataFrame(
    list(batch24_corrections.items()),
    columns=["id", "complete_english_translation"]
)

for _, row in corrections_df.iterrows():
    reviewed_translations.loc[
        reviewed_translations["id"] == row["id"],
        "complete_english_translation"
    ] = row["complete_english_translation"]

reviewed_translations.loc[
    reviewed_translations["id"].isin(batch24_corrections.keys()),
    "manual_review_status"
] = "Manually corrected"

output_path = (
    f"{base_path}/09_sentence_level_translation/"
    "FactIndic_English_250_Reviewed.csv"
)

reviewed_translations.to_csv(output_path, index=False)

print("Batch 24 partial save successful.")
print("Corrections applied:", len(batch24_corrections))
print("\nUpdated review status:")
print(reviewed_translations["manual_review_status"].value_counts())
print("\nSaved to:", output_path)

Batch 24 partial save successful.
Corrections applied: 3

Updated review status:
manual_review_status
Manually corrected                221
Human reviewed - no correction     25
Pending review                      4
Name: count, dtype: int64

Saved to: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv


In [85]:
pending_ids = reviewed_translations.loc[
    reviewed_translations["manual_review_status"] == "Pending review",
    "id"
].tolist()

source_lookup = df_250.set_index("id")
translation_lookup = reviewed_translations.set_index("id")

print("Final 4 Articles for Human Review")

for i, article_id in enumerate(pending_ids, 1):
    print("\n" + "=" * 100)
    print(f"ARTICLE {i}/4: {article_id}")

    print("\nHINDI ORIGINAL:\n")
    print(source_lookup.loc[article_id, "text"])

    print("\nCURRENT ENGLISH TRANSLATION:\n")
    print(translation_lookup.loc[
        article_id, "complete_english_translation"
    ])

Final 4 Articles for Human Review

ARTICLE 1/4: science-49370402

HINDI ORIGINAL:

डॉक्टरों का कहना है कि बारिश के चलते कई जगहों पर जलभराव और गंदगी होने से मच्छर और ख़तरनाक बैक्टीरिया जन्म ले लेते हैं. पानी और हवा के जरिए ये बैक्टीरिया खाने और शरीर तक पहुंचते हैं और हम बुखार व फ़्लू जैसी बीमारियों की जकड़ में आ जाते हैं. लेकिन, अगर थोड़ी सावधानी बरती जाए तो इन बीमारियों से बचा जा सकता है. मैक्स सुपर स्पेशिएलिटी हॉस्पिटल में एंडोक्राइनोलॉजी और इंटरनल मेडिसिन के डॉक्टर विनीत अरोड़ा मानसून में होने वाली बीमारियों को तीन श्रेणियों में बांटते हैं. सामान्य बुख़ार और जुक़ाम वायरल बुख़ार, मौसम बदलने के साथ वातावरण में आए कीटाणुओं से होने वाले बुख़ार को कहते हैं. ये हवा और पानी के ज़रिए फैलते हैं. सामान्य बुख़ार किस तरह का है ये वायरस के प्रकार पर निर्भर करता है. इतने अलग-अलग वायरस जांचने के लिए बहुत ज्यादा टेस्ट उपलब्ध नहीं हैं. वैसे तो इसमें सिर्फ़ बुख़ार ही आता है लेकिन कुछ में खांसी और जोड़ों में दर्द भी हो सकता है. लेकिन ये फ़्लू, डेंगू या चिकनगुनिया नहीं होते हैं. बुख़ार तीन से सात दिनों 

In [86]:
batch24_corrections.update({

    "science-49370402": """Doctors say that waterlogging and unhygienic conditions caused by rainfall create breeding grounds for mosquitoes and harmful bacteria. These bacteria can reach food and the human body through water and air, leading to illnesses such as fever and flu. However, these illnesses can be prevented by taking a few precautions.

Dr Vineet Arora, who specialises in endocrinology and internal medicine at Max Super Speciality Hospital, divides monsoon-related illnesses into three categories.

**Common fever and cold**

Viral fevers caused by germs that become more prevalent in the environment as the weather changes are commonly referred to as ordinary viral fevers. They spread through air and water. The nature of the fever depends on the type of virus. There are not enough tests available to identify all these different viruses. Usually, these illnesses cause only fever, although some may also cause a cough and joint pain. They are not influenza, dengue or chikungunya. The fever may last for three to seven days, depending on the virus.

**Prevention:** Dr Vineet Arora says that maintaining cleanliness and hygiene is the best way to prevent these illnesses.

**Flu (influenza)**

Influenza is particularly common during this period. Swine flu also spreads at this time. It is a type of influenza but can be more dangerous. Testing is required to determine whether a person has common flu or swine flu. Symptoms include a cold, cough, high fever and joint pain. In some cases, respiratory support may also be required.

Most patients visit doctors with colds and throat problems, which are also symptoms of common flu. Common flu usually lasts five to seven days. Even with medication, recovery can take this long. A cold and cough may take 10 to 15 days to clear up. Fever caused by swine flu may also last for several days, but it carries a risk of developing into pneumonia.

**Mosquito-borne diseases**

Chikungunya and dengue are also viral diseases, but they are vector-borne illnesses transmitted through mosquito bites. They cause high fever and joint pain, and may also lead to vomiting and headaches.

Dengue initially causes a high fever, headache and pain behind the eyes. Joint pain is more severe in chikungunya, but both illnesses can cause a very high fever during the first two or three days. In dengue, a low platelet count can lead to rashes on the body.

(You can click here to download BBC Hindi's Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)""",

    "social-43068605": """A video featuring Priya Prakash Varrier and her expressive, kohl-lined eyes went viral in India. Most of you may have come across it. Now, people in Pakistan are also becoming fans of Priya's eye gestures. Social media users are adding humour to the craze surrounding her.

Videos and memes are being created on Pakistani social media by combining Priya's expressions with reactions from political leaders. These leaders include Imran Khan, Nawaz Sharif, Altaf Hussain and Asif Ali Zardari.

Here are some examples. Pakistani journalist Umar Qureshi tweeted: "Priya Prakash's winking video is going viral in Pakistan too. Memes are being made in large numbers. Watch one such funny video."

In this video, Altaf Hussain, a leader of the Muttahida Qaumi Movement, has been added through editing. Altaf Hussain, Imran Khan and Tahir-ul-Qadri are among those shown reacting to Priya Prakash's wink in these edited videos.

Now, let's turn to former Pakistani Prime Minister Nawaz Sharif, who had to step down from power some time ago. Priya Prakash's Pakistani fans have not spared Nawaz Sharif either. Watch this video for an example.

Social media users have also edited Priya's new video to include former President Asif Ali Zardari.

However, some people have been offended by these videos. A user named Taha Ansari writes: "I appeal to Pakistan: action should be taken over the Priya Prakash issue."

Salman writes: "The girls in Pakistani villages are better than Priya, whose eyes Pakistanis are so fascinated by. They are more beautiful without makeup. It is simply a difference in perspective."

How did Priya, the girl with kohl-lined eyes, become the 'National Crush'? Who is the boy Priya winked at? The complete profile of the girl who winked (You can click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)""",

    "sport-47725658": """Kolkata Knight Riders defeated Kings XI Punjab by 28 runs in a packed stadium, with their owner and superstar Shah Rukh Khan watching from the stands. Chasing a huge target of 219, Punjab managed only 190 for four in 20 overs.

For Punjab, Mayank Agarwal scored 58 and David Miller remained unbeaten on 59. Earlier, KL Rahul was dismissed for just one run, giving Punjab an early setback. Chris Gayle, who had scored 79 in the previous match and made an impressive start to the IPL season, was dismissed by Russell for 20. Russell then dismissed Sarfaraz Khan cheaply; he scored only 13.

The R factor

Andre Russell took two wickets for 21 runs. Earlier, Kolkata's R factor had been Nitish Rana, Robin Uthappa and Andre Russell. After losing the toss and being asked to bat first, Kolkata made full use of the opportunity. Nitish Rana scored 63, Robin Uthappa remained unbeaten on 67 and Andre Russell scored 48, helping Kolkata reach 218 for four in 20 overs.

Earlier, Sunil Narine had provided a glimpse of what was to come by scoring 24 runs off just nine balls, including three sixes, as an opener. This suggested what was in store for R Ashwin's bowlers. Punjab's bowlers conceded 17 sixes in the match.

**The right move**

In this match, Kolkata captain Dinesh Karthik decided to send Nitish Rana into the middle order instead of opening the innings. The decision proved successful. Rana scored more than 60 runs for the second consecutive match and played an important role in Kolkata's victory.

Nitish Rana, known for hitting powerful shots over mid-wicket, was bought by Mumbai Indians for just Rs 10 lakh at the 2015 players' auction. However, he did not get an opportunity to play a single IPL match at that time.

His wait finally ended in 2017, when he scored 333 runs in 13 matches that season. He played an important role in Mumbai's victories in the early matches. Commenting on his performance, Mumbai coach Ricky Ponting said that he would not be surprised to see Nitish Rana playing for India's T20 team. Michael Clarke also praised him.

The previous IPL season had also been successful for Rana. In 2018, he scored 304 runs in 15 matches.

**Robin Uthappa and Andre Russell**

Apart from Nitish Rana, Kolkata also benefited from Robin Uthappa's performance. Uthappa remained unbeaten on 67 off 50 balls, hitting six fours and two sixes.

Uthappa showed good judgement during his innings when he was joined by Andre Russell. He gave Russell most of the strike because he knew that the powerful batsman's hitting could cause serious trouble for Punjab. That is exactly what happened.

Russell took charge after Nitish Rana was dismissed. As soon as he came to the crease, he demonstrated what he could do in T20 cricket. He scored 48 runs off just 17 balls, including three fours and five sixes.

He dominated the 19th over, bowled by yorker specialist Mohammed Shami. Russell hit sixes off three consecutive deliveries, and struck a four off the fourth ball as well.

Uthappa and Russell added 67 runs for the fourth wicket. Russell had earlier scored an unbeaten 49 off just 19 balls against Sunrisers Hyderabad. In this match, he produced a similar performance. His shots were timed so well that Punjab's fielders could only watch. Eventually, he was caught by Mayank Agarwal near the boundary while trying to hit a six off Andrew Tye.

**The turning point**

The turning point came when Andre Russell was bowled by Mohammed Shami, but the delivery was declared a no-ball. Punjab had only three fielders inside the circle, when the rules required four. This was R Ashwin's biggest mistake as captain, and from that point onwards, Punjab began to lose control of the match.

Russell was batting on just three runs at the time, and the 17th over was in progress. What happened after that was witnessed by the entire stadium.

After the match, Punjab captain R Ashwin admitted his mistake, but by then it was too late. This was Kolkata's second consecutive victory.

On Thursday, in IPL-12, Royal Challengers Bangalore will face Mumbai Indians at home.

(You can click here for BBC Hindi's Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)""",

    "sport-54682255": """After an excellent performance by their bowlers, opener Ruturaj Gaikwad and the other Chennai batsmen delivered with the bat. Chennai Super Kings chased down Bangalore's target of 146 with eight balls to spare.

For Chennai, Ruturaj Gaikwad remained unbeaten on 65, Ambati Rayudu scored 39 and captain Mahendra Singh Dhoni remained unbeaten on 19.

Virat Kohli's Royal Challengers Bangalore suffered their fourth defeat in IPL-13. Ruturaj Gaikwad was named Player of the Match. On Sunday, Chennai dominated Bangalore in every department of the game. With their fourth win, Chennai Super Kings moved to seventh place in the points table.

**A good start**

Faf du Plessis and Ruturaj Gaikwad gave Chennai Super Kings a flying start, adding 46 runs in five overs.

Du Plessis made his intentions clear in the second over by hitting a four and a six off Chris Morris. In the next over, Ruturaj Gaikwad hit a six off Washington Sundar, showing that he too was determined to make the most of the opportunity.

In the fourth over, Chennai's openers targeted Mohammed Siraj and scored 15 runs from his over. Ruturaj hit a four, while du Plessis struck a four and a six.

Morris gave Bangalore their first breakthrough. He dismissed du Plessis with the first ball of the sixth over. Du Plessis scored 25 runs off 13 balls. After six overs, Chennai were 48 for one.

**An important partnership**

Ruturaj Gaikwad continued to put pressure on Bangalore's bowlers in partnership with Ambati Rayudu, who came in at number three.

In the eighth over, Ruturaj hit a four off Yuzvendra Chahal, and in the next over he struck a six off Moeen Ali. Rayudu also hit a four off Ali.

In the 10th over, Rayudu hit a six and a four off Navdeep Saini. After 10 overs, Chennai were 82 for one. They needed 64 runs from the remaining 10 overs to win.

Mohammed Siraj bowled the 12th over. Ruturaj hit a four and Rayudu struck a six.

Yuzvendra Chahal gave Bangalore their second breakthrough in the 14th over by bowling Rayudu. Rayudu had added 67 runs with Ruturaj for the second wicket in 8.2 overs. He scored 39 off 27 balls.

When the second wicket fell, Chennai were 113 for two and needed 33 runs from 39 balls to win. In the same over, Ruturaj completed his half-century, reaching 50 off 42 balls.

**Dhoni finds his rhythm**

Captain Mahendra Singh Dhoni came in at number four and appeared determined to put his poor form behind him. In the 15th over, he hit two consecutive fours off Chris Morris.

After 15 overs, Chennai were 127 for two. They needed 19 runs from the final five overs.

In the 18th over, Dhoni hit the first ball from Navdeep Saini for a four, taking the team's total to 140. No more runs were scored in that over.

Chennai needed six runs from the final two overs. In the 19th over, Ruturaj Gaikwad hit the second ball from Chris Morris for a four. He then struck a six off the fourth ball to seal Chennai's victory.

**Royal Challengers Bangalore: 145/6 (20 overs)**

Earlier, a half-century from captain Virat Kohli had helped Royal Challengers Bangalore score 145 for six in 20 overs.

Kohli was the team's highest scorer with 50 runs. He shared an 82-run partnership with AB de Villiers for the third wicket in 11.2 overs. De Villiers scored 39 off 36 balls.

Devdutt Padikkal scored 22 runs for Bangalore. None of the other batsmen managed a substantial innings.

Chennai's bowlers maintained constant pressure on the slow pitch and gave Bangalore's batsmen few opportunities to hit boundaries. Bangalore lost four wickets and scored only 20 runs in the final three overs.

For Chennai, Sam Curran took three wickets for 19 runs, Deepak Chahar took two for 31, and Mitchell Santner took one for 23.

(You can click here for BBC Hindi's Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)"""
})

print("Batch 24 correction count:", len(batch24_corrections))
print("Correction IDs:")
for article_id in batch24_corrections:
    print(article_id)

Batch 24 correction count: 7
Correction IDs:
international-54702261
international-56439434
science-39280869
science-49370402
social-43068605
sport-47725658
sport-54682255


In [87]:
corrections_df = pd.DataFrame(
    list(batch24_corrections.items()),
    columns=["id", "complete_english_translation"]
)

for _, row in corrections_df.iterrows():
    reviewed_translations.loc[
        reviewed_translations["id"] == row["id"],
        "complete_english_translation"
    ] = row["complete_english_translation"]

reviewed_translations.loc[
    reviewed_translations["id"].isin(batch24_corrections.keys()),
    "manual_review_status"
] = "Manually corrected"

output_path = (
    f"{base_path}/09_sentence_level_translation/"
    "FactIndic_English_250_Reviewed.csv"
)

reviewed_translations.to_csv(output_path, index=False)

print("Batch 24 saved successfully.")
print("Corrections applied:", len(batch24_corrections))
print("\nUpdated review status:")
print(reviewed_translations["manual_review_status"].value_counts())
print("\nSaved to:", output_path)

Batch 24 saved successfully.
Corrections applied: 7

Updated review status:
manual_review_status
Manually corrected                225
Human reviewed - no correction     25
Name: count, dtype: int64

Saved to: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv


In [88]:
import os

claim_folder = f"{base_path}/07_claim_labels"

print("Claim-labeling folder:", claim_folder)

if os.path.exists(claim_folder):
    files = os.listdir(claim_folder)

    if files:
        print("\nExisting files:")
        for file in files:
            print("-", file)
    else:
        print("\nFolder exists but is empty.")
else:
    print("\nClaim-labeling folder does not exist yet.")

Claim-labeling folder: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/07_claim_labels

Existing files:
- FactIndic_50_Claim_Labeling_Sheet.csv


# New Section

In [89]:
import pandas as pd
import os

claim_path = (
    f"{base_path}/07_claim_labels/"
    "FactIndic_50_Claim_Labeling_Sheet.csv"
)

claims_df = pd.read_csv(claim_path)

print("Total claims:", len(claims_df))
print("\nColumns:")
print(claims_df.columns.tolist())

print("\nFirst 5 rows:")
display(claims_df.head())

print("\nMissing values:")
print(claims_df.isna().sum())

Total claims: 50

Columns:
['Claim_ID', 'Article_ID', 'Claim', 'Source_Article', 'Verdict', 'Evidence_from_Article', 'Reviewer_Notes']

First 5 rows:


,Claim_ID,Article_ID,Claim,Source_Article,Verdict,Evidence_from_Article,Reviewer_Notes
0,FC001,040102_france_sikhs,फ़्रांस में रह रहे पाँच हज़ार सिख फ़्रांस के उ...,वे भारत के प्रधानमंत्री से आग्रह कर रहे हैं कि...,NOT_ENOUGH_INFO,यह क़ानून मुसलमानों के ही नहीं सिखों के भी ख़ि...,"The opposition is supported, but the stated co..."
1,FC002,social-43068605,आंखों के रास्ते दिलों में कुटिया बनाने वाली प्...,प्रिया प्रकाश प्रिया प्रकाश की कजरारे नैनों वा...,SUPPORTED,पाकिस्तान में सोशल मीडिया पर प्रिया के इशारों ...,The article describes Priya-related videos and...
2,FC003,050420_pope_muslims,कई मुसलमान नेताओं ने नए पोप से अनुरोध किया है ...,शंका जताई गई है कि नए पोप शायद ऐसा करने में उत...,NOT_ENOUGH_INFO,कुछ कैथोलिक ... दूसरे संप्रदायों के साथ बातचीत...,"The excerpt mentions interfaith dialogue, but ..."
3,FC004,150811_sani_mirza_autobiography_farha_adp_ssm,कई दिनों से चर्चा थी कि निर्माता निर्देशक फ़रा...,"यदि ऐसा होता तो धावक मिल्खा सिंह, मुक्क़ेबाज़ ...",SUPPORTED,फ़राह ने ऐसी किसी भी फ़िल्म को बनाए जाने की बा...,The article confirms that the rumour about a S...
4,FC005,060715_bushputin_g8,अमरीकी राष्ट्रपति जॉर्ज बुश ने जी8 बैठक से पहल...,दोनों देशों के राष्ट्रपतियों ने सेंट पीटर्सबर्...,SUPPORTED,दोनों देशों के राष्ट्रपतियों ने सेंट पीटर्सबर्...,The article confirms the meeting and discussio...



Missing values:
Claim_ID                 0
Article_ID               0
Claim                    0
Source_Article           0
Verdict                  0
Evidence_from_Article    0
Reviewer_Notes           0
dtype: int64


In [90]:
print("VERDICT DISTRIBUTION:")
print(claims_df["Verdict"].value_counts())

print("\nINVALID VERDICTS:")
valid_verdicts = {
    "SUPPORTED",
    "CONTRADICTED",
    "NOT_ENOUGH_INFO"
}
print(claims_df.loc[
    ~claims_df["Verdict"].isin(valid_verdicts),
    ["Claim_ID", "Verdict"]
])

print("\nDUPLICATE CLAIM IDs:")
print(claims_df[claims_df["Claim_ID"].duplicated()])

print("\nUNIQUE ARTICLES:", claims_df["Article_ID"].nunique())

print("\nCLAIM LABELING CHECK COMPLETE")

VERDICT DISTRIBUTION:
Verdict
SUPPORTED          40
NOT_ENOUGH_INFO    10
Name: count, dtype: int64

INVALID VERDICTS:
Empty DataFrame
Columns: [Claim_ID, Verdict]
Index: []

DUPLICATE CLAIM IDs:
Empty DataFrame
Columns: [Claim_ID, Article_ID, Claim, Source_Article, Verdict, Evidence_from_Article, Reviewer_Notes]
Index: []

UNIQUE ARTICLES: 50

CLAIM LABELING CHECK COMPLETE


In [91]:
nei_df = claims_df[
    claims_df["Verdict"] == "NOT_ENOUGH_INFO"
][[
    "Claim_ID",
    "Article_ID",
    "Claim",
    "Source_Article",
    "Evidence_from_Article",
    "Reviewer_Notes"
]]

display(nei_df)

,Claim_ID,Article_ID,Claim,Source_Article,Evidence_from_Article,Reviewer_Notes
0,FC001,040102_france_sikhs,फ़्रांस में रह रहे पाँच हज़ार सिख फ़्रांस के उ...,वे भारत के प्रधानमंत्री से आग्रह कर रहे हैं कि...,यह क़ानून मुसलमानों के ही नहीं सिखों के भी ख़ि...,"The opposition is supported, but the stated co..."
2,FC003,050420_pope_muslims,कई मुसलमान नेताओं ने नए पोप से अनुरोध किया है ...,शंका जताई गई है कि नए पोप शायद ऐसा करने में उत...,कुछ कैथोलिक ... दूसरे संप्रदायों के साथ बातचीत...,"The excerpt mentions interfaith dialogue, but ..."
11,FC012,international-56439434,पाकिस्तान में सिंध प्रांत की एक अदालत ने एक हि...,लड़की ने अदालत के सामने अपने बयान में कहा है क...,लड़की को बुधवार को ... सिविल जज मुनव्वर पीरज़ा...,The displayed excerpt does not establish that ...
15,FC016,150408_kangana_ranaut_deepika_padukone_md,दो बार नेशनल अवॉर्ड विजेता रही फ़िल्म अभिनेत्र...,"जानी मानी पत्रकार अनुपमा चोपड़ा की किताब ""फ्रं...",कंगना ने बताया ... ना दीपिका ना सोनम नेक्स्ट ज...,The excerpt discusses her direct statements bu...
19,FC020,060829_dhyanchand_hockey,हॉकी के जादूगर ध्यानचंद की महानता का अहसास न उ...,हॉलैंड में 1928 में उनकी स्टिक को तोड़कर जांचा...,हॉलैंड में 1928 में उनकी स्टिक को तोड़कर जांचा...,The excerpt gives examples of Dhyan Chand's re...
24,FC025,140908_amit_shah_political_analysis_vr,करीब दो महीने पहले भारतीय जनता पार्टी के मुख्य...,फ़ोन पर फ़ोन बज रहे हैं. वहां बैठे एक जानकार न...,भाजपा का दिल्ली मुख्यालय इन दिनों राजनीतिक गति...,The excerpt discusses Amit Shah's leadership b...
36,FC037,130207_kashmir_rockband_imroz_sa,भारत प्रशासित कश्मीर में सक्रिय एक प्रमुख मानव...,परवेज़ इमरोज़ लड़कियों के रॉकबैंड प्रगाश के सम...,मुफ्ती बशीर ने मुसलमान लड़कियों के गाना गाने क...,The excerpt describes the controversy and supp...
40,FC041,071227_musharraf_statement,पाकिस्तान के राष्ट्रपति परवेज़ मुशर्रफ़ ने पूर...,मुशर्रफ़ ने कहा है कि आतंकवाद पाकिस्तान की तरक...,मुशर्रफ़ ने कहा है कि आतंकवाद पाकिस्तान की तरक...,The displayed excerpt discusses Musharraf's st...
41,FC042,050606_kabir_bedi,बहुमुखी प्रतिभा वाले अभिनेता कबीर बेदी की ख़्व...,ग़ौरतलब है कि कबीर बेदी की पैदाइश और परवरिश दि...,कबीर बेदी की पैदाइश और परवरिश दिल्ली में ही हु...,The excerpt supports his connection with Delhi...
45,FC046,141007_bhutan_corruption_india_vr,भूटान में भ्रष्टाचार... कुछ साल पहले तक ये असं...,ये भी चौंकाने वाली बात लगती है पर सच है कि जमी...,भूटान में भ्रष्टाचार का स्तर अभी वैसा कहीं से ...,The excerpt discusses Bhutan seeking India's h...


In [92]:
for _, row in nei_df.iterrows():
    print("\n" + "=" * 100)
    print("CLAIM ID:", row["Claim_ID"])
    print("ARTICLE ID:", row["Article_ID"])
    print("\nCLAIM:\n", row["Claim"])
    print("\nSOURCE ARTICLE:\n", row["Source_Article"])
    print("\nEVIDENCE:\n", row["Evidence_from_Article"])
    print("\nREVIEWER NOTES:\n", row["Reviewer_Notes"])


CLAIM ID: FC001
ARTICLE ID: 040102_france_sikhs

CLAIM:
 फ़्रांस में रह रहे पाँच हज़ार सिख फ़्रांस के उस प्रतिबंध से नाराज़ है जो धार्मिक प्रतीकों के इस्तेमाल पर लगाया जा रहा है.

SOURCE ARTICLE:
 वे भारत के प्रधानमंत्री से आग्रह कर रहे हैं कि वह उनकी पारंपरिक पगड़ियों को इस प्रतिबंध से अलग रखने की हिमायत करें. सिख समुदाय के एक प्रवक्ता चैन सिंह ने कहा, "यह क़ानून मुसलमानों के ही नहीं सिखों के भी ख़िलाफ़ है". हम पगड़ी पहने बिना नहीं रह सकते. यह हमारे धर्म का हिस्सा हैं. अगर हमारे पगड़ी बांधने पर पाबंदी लगाई गई तो हम यहाँ रह भी नहीं पाएँगे. चैन सिंह "हम पगड़ी पहने बिना नहीं रह सकते. यह हमारे धर्म का हिस्सा हैं. अगर हमारे पगड़ी बांधने पर पाबंदी लगाई गई तो हम यहाँ रह भी नहीं पाएँगे". फ़्रांस के इस क़दम पर मिस्र में भी टकराव की स्थिति पैदा हो गई है जहाँ एक प्रमुख मुस्लिम नेता ने फ़्रांस के फ़ैसले की वकालत करके एक नई बहस छेड़ दी है. अल-अज़हर मस्जिद के प्रमुख शेख़ मोहम्मद सैयद तंतावी ने एक बयान में कह दिया कि वह इस बात से सहमत हैं कि फ़्रांस को यह क़ानून लागू करने का पूरा अधिकार है. उनका कह

In [93]:
# Correct the two supported claims
corrections = {
    "FC012": {
        "Verdict": "SUPPORTED",
        "Evidence_from_Article": (
            "लेकिन अदालत ने आदेश दिया है कि पुलिस की जाँच पूरी होने तक "
            "लड़की को बाल संरक्षण केंद्र भेजा जाएगा।"
        ),
        "Reviewer_Notes": (
            "Supported: the article explicitly states that the court "
            "ordered her transfer to a child protection centre."
        )
    },
    "FC042": {
        "Verdict": "SUPPORTED",
        "Evidence_from_Article": (
            "कबीर बेदी ... ख़्वाहिश रखते हैं कि एक दिन फिर से दिल्ली लौट सकें। "
            "मेरी ज़िंदगी का सफ़र दिल्ली से शुरू हुआ और मेरी तमन्ना है "
            "कि वहीं जाकर ख़त्म।"
        ),
        "Reviewer_Notes": (
            "Supported: Kabir Bedi explicitly expresses his wish to return "
            "to Delhi and settle there."
        )
    }
}

for claim_id, updates in corrections.items():
    mask = claims_df["Claim_ID"] == claim_id

    for column, value in updates.items():
        claims_df.loc[mask, column] = value

# Save the updated sheet
claims_df.to_csv(claim_path, index=False, encoding="utf-8-sig")

print("Claim-labeling sheet updated and saved!")
print("\nUpdated verdict distribution:")
print(claims_df["Verdict"].value_counts())

print("\nUpdated claims:")
display(
    claims_df[claims_df["Claim_ID"].isin(["FC012", "FC042"])][
        ["Claim_ID", "Verdict", "Evidence_from_Article", "Reviewer_Notes"]
    ]
)

Claim-labeling sheet updated and saved!

Updated verdict distribution:
Verdict
SUPPORTED          42
NOT_ENOUGH_INFO     8
Name: count, dtype: int64

Updated claims:


,Claim_ID,Verdict,Evidence_from_Article,Reviewer_Notes
11,FC012,SUPPORTED,लेकिन अदालत ने आदेश दिया है कि पुलिस की जाँच प...,Supported: the article explicitly states that ...
41,FC042,SUPPORTED,कबीर बेदी ... ख़्वाहिश रखते हैं कि एक दिन फिर ...,Supported: Kabir Bedi explicitly expresses his...


In [94]:
import pandas as pd

# Check all required fields
required_columns = [
    "Claim_ID", "Article_ID", "Claim", "Source_Article",
    "Verdict", "Evidence_from_Article", "Reviewer_Notes"
]

print("MISSING VALUES:")
print(claims_df[required_columns].isnull().sum())

print("\nDUPLICATE CLAIM IDs:")
print(claims_df["Claim_ID"].duplicated().sum())

print("\nDUPLICATE ARTICLE IDs:")
print(claims_df["Article_ID"].duplicated().sum())

print("\nINVALID VERDICTS:")
valid_verdicts = {"SUPPORTED", "CONTRADICTED", "NOT_ENOUGH_INFO"}
print((~claims_df["Verdict"].isin(valid_verdicts)).sum())

print("\nTOTAL CLAIMS:", len(claims_df))

# Save final validated sheet
claims_df.to_csv(claim_path, index=False, encoding="utf-8-sig")

print("\nFinal claim-labeling sheet saved successfully.")

MISSING VALUES:
Claim_ID                 0
Article_ID               0
Claim                    0
Source_Article           0
Verdict                  0
Evidence_from_Article    0
Reviewer_Notes           0
dtype: int64

DUPLICATE CLAIM IDs:
0

DUPLICATE ARTICLE IDs:
0

INVALID VERDICTS:
0

TOTAL CLAIMS: 50

Final claim-labeling sheet saved successfully.


In [95]:
import os
import pandas as pd

task_path = (
    f"{base_path}/09_sentence_level_translation/"
    "Sentence_Translation_Tasks.csv"
)

tasks_df = pd.read_csv(task_path)

print("Translation queue loaded successfully!")
print("Total chunks:", len(tasks_df))
print("Columns:", tasks_df.columns.tolist())

display(tasks_df.head(5))

print("\nMaximum chunk size:")
print(tasks_df.select_dtypes(include="number").max())

Translation queue loaded successfully!
Total chunks: 5229
Columns: ['sentence_id', 'id', 'sentence_no', 'part_no', 'hindi_chunk', 'source_tokens']


,sentence_id,id,sentence_no,part_no,hindi_chunk,source_tokens
0,1,080419_brchopra_festival,1,1,शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ ...,23
1,2,080419_brchopra_festival,2,1,"उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त ज...",23
2,3,080419_brchopra_festival,3,1,खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में ...,36
3,4,080419_brchopra_festival,4,1,"उन्होंने कहा, ""मेरे भाई ने अपने हिसाब से फ़िल्...",92
4,5,080419_brchopra_festival,5,1,यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौ...,40



Maximum chunk size:
sentence_id      5214
sentence_no        64
part_no             3
source_tokens     120
dtype: int64


In [96]:
import os

translation_folder = (
    f"{base_path}/09_sentence_level_translation"
)

print("Files in sentence-level translation folder:\n")

for filename in sorted(os.listdir(translation_folder)):
    filepath = os.path.join(translation_folder, filename)

    if os.path.isfile(filepath):
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"{filename}  |  {size_mb:.2f} MB")

Files in sentence-level translation folder:

FactIndic_English_250_Corrected.csv  |  0.55 MB
FactIndic_English_250_Reviewed.csv  |  0.56 MB
FactIndic_English_250_Sentence_Reconstructed.csv  |  0.55 MB
FactIndic_Final_Quality_Audit.csv  |  1.93 MB
Sentence_Level_Translation_Inventory.csv  |  1.61 MB
Sentence_Translation_Checkpoints.csv  |  0.62 MB
Sentence_Translation_Tasks.csv  |  1.56 MB


In [97]:
reconstructed_path = (
    f"{translation_folder}/FactIndic_English_250_Sentence_Reconstructed.csv"
)

audit_path = (
    f"{translation_folder}/FactIndic_Final_Quality_Audit.csv"
)

reconstructed_df = pd.read_csv(reconstructed_path)
audit_df = pd.read_csv(audit_path)

print("RECONSTRUCTED DATASET")
print("Rows:", len(reconstructed_df))
print("Columns:", reconstructed_df.columns.tolist())
display(reconstructed_df.head(3))

print("\nQUALITY AUDIT")
print("Rows:", len(audit_df))
print("Columns:", audit_df.columns.tolist())
display(audit_df.head(5))

print("\nMissing values in reconstructed dataset:")
print(reconstructed_df.isnull().sum())

RECONSTRUCTED DATASET
Rows: 250
Columns: ['id', 'complete_english_translation', 'reconstructed_sentence_count', 'original_sentence_count']


,id,complete_english_translation,reconstructed_sentence_count,original_sentence_count
0,000001_ugc,You sent this material because of an incident ...,6,6
1,031031_russia_removal,He is reported to have resigned following the ...,9,9
2,031031_sharon_palestine,"With Sharon's invitation, Israel has changed i...",8,8



QUALITY AUDIT
Rows: 250
Columns: ['id', 'complete_english_translation', 'reconstructed_sentence_count', 'text', 'hindi_words', 'english_words', 'translation_ratio', 'empty_translation', 'remaining_hindi_chars']


,id,complete_english_translation,reconstructed_sentence_count,text,hindi_words,english_words,translation_ratio,empty_translation,remaining_hindi_chars
0,000001_ugc,You sent this material because of an incident ...,6,आपने यह सामग्री अपने स्थानीय इलाक़े में हुई घट...,154,130,0.844156,False,False
1,031031_russia_removal,He is reported to have resigned following the ...,9,बताया जा रहा है कि उन्होंने रूस के सबसे अमीर व...,191,163,0.853403,False,False
2,031031_sharon_palestine,"With Sharon's invitation, Israel has changed i...",8,शेरॉन के इस निमंत्रण से इसराइल ने अपना पुराना ...,259,193,0.745174,False,False
3,031113_tdp_elections,The meeting was attended by about seven and a ...,10,इस बैठक में पार्टी के लगभग साढ़े सात हज़ार प्र...,208,195,0.937500,False,False
4,031114_bush_intelligence,These operations have been launched in the wak...,11,ये कार्रवाइयाँ अमरीकी नेतृत्व वाली फ़ौजों के ख...,217,186,0.857143,False,False



Missing values in reconstructed dataset:
id                              0
complete_english_translation    0
reconstructed_sentence_count    0
original_sentence_count         0
dtype: int64


In [98]:
# Check sentence reconstruction
count_mismatch = reconstructed_df[
    reconstructed_df["reconstructed_sentence_count"]
    != reconstructed_df["original_sentence_count"]
]

# Check quality audit flags
print("SENTENCE COUNT MISMATCHES:", len(count_mismatch))

print("\nEMPTY TRANSLATIONS:")
print(audit_df["empty_translation"].value_counts(dropna=False))

print("\nREMAINING HINDI CHARACTERS:")
print(audit_df["remaining_hindi_chars"].value_counts(dropna=False))

print("\nTRANSLATION RATIO:")
print(audit_df["translation_ratio"].describe())

print("\nTOTAL AUDITED ARTICLES:", len(audit_df))

SENTENCE COUNT MISMATCHES: 0

EMPTY TRANSLATIONS:
empty_translation
False    250
Name: count, dtype: int64

REMAINING HINDI CHARACTERS:
remaining_hindi_chars
False    250
Name: count, dtype: int64

TRANSLATION RATIO:
count    250.000000
mean       0.863538
std        0.056566
min        0.700000
25%        0.825858
50%        0.861653
75%        0.896240
max        1.039062
Name: translation_ratio, dtype: float64

TOTAL AUDITED ARTICLES: 250


In [99]:
checkpoint_path = (
    f"{translation_folder}/Sentence_Translation_Checkpoints.csv"
)

checkpoints_df = pd.read_csv(checkpoint_path)

print("Checkpoint rows:", len(checkpoints_df))
print("Columns:", checkpoints_df.columns.tolist())

display(checkpoints_df.head(10))

print("\nMissing values:")
print(checkpoints_df.isnull().sum())

Checkpoint rows: 5229
Columns: ['task_index', 'translation', 'status']


,task_index,translation,status
0,0,"The festival, which began on Friday evening, s...",Completed
1,1,"Classic films like Chhoti Si Baat, Naya Daur, ...",Completed
2,2,BR Chopra himself did not attend the ceremony ...,Completed
3,3,"He said, ""My brother made his own films - that...",Completed
4,4,Memorable films His film Naya Daur in 1957 ref...,Completed
5,5,"Films such as Gumrah, Hamraaz, Kanoon and Nika...",Completed
6,6,Although most of BR Chopra's films were based ...,Completed
7,7,The story of Naya Daur was turned down by many...,Completed
8,8,"Speaking on the occasion, Ravi Chopra, son of ...",Completed
9,9,"For me, he is and will remain the best directo...",Completed



Missing values:
task_index     0
translation    0
status         0
dtype: int64


In [100]:
print("STATUS DISTRIBUTION:")
print(checkpoints_df["status"].value_counts(dropna=False))

print("\nDUPLICATE TASK INDEXES:")
print(checkpoints_df["task_index"].duplicated().sum())

print("\nEMPTY TRANSLATIONS:")
print(
    checkpoints_df["translation"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("\nTOTAL CHECKPOINTS:", len(checkpoints_df))

print("\nALL TASKS COMPLETED:",
      (checkpoints_df["status"] == "Completed").all())

STATUS DISTRIBUTION:
status
Completed    5229
Name: count, dtype: int64

DUPLICATE TASK INDEXES:
0

EMPTY TRANSLATIONS:
0

TOTAL CHECKPOINTS: 5229

ALL TASKS COMPLETED: True


In [101]:
import os

for folder in sorted(os.listdir(base_path)):
    path = os.path.join(base_path, folder)

    if os.path.isdir(path):
        print(f"\n📁 {folder}")
        for file in sorted(os.listdir(path)):
            print("   ", file)


📁 01_raw_data
    XL_Sum_Hindi_Test.parquet
    XL_Sum_Raw_Sample_250.csv

📁 02_clean_data
    XL_Sum_Hindi_Cleaned_250.csv
    XL_Sum_Hindi_Cleaned_250.parquet

📁 03_english_data
    FactIndic_198_Translation_Pilot.csv
    FactIndic_Chunk_Translation_Checkpoints.csv
    FactIndic_Chunk_Translation_Pilot_3.csv
    FactIndic_Hindi_English_250.csv
    FactIndic_Hindi_English_Base_250.csv
    IndicTrans2_200M_All_250.csv
    IndicTrans2_200M_First_60.csv
    IndicTrans2_200M_Remaining_190.csv
    NLLB_English_250.csv
    XL_Sum_English_250.csv
    XL_Sum_English_250_partial.csv

📁 04_mixed_data
    FactIndic_Mixed_250.csv
    FactIndic_Mixed_250_partial.csv
    hindi_english_phrase_dictionary.json

📁 05_quality_checks
    CMI_MUTANT_PHINC_Comparison.csv
    Chunk_Translation_Quality_Audit.csv
    FactIndic_Sentence_Alignment.csv
    Human_Translation_Review_20.csv
    IndicTrans2_First_60_Quality_Review.csv
    IndicTrans2_First_60_Quality_Summary.csv
    IndicTrans2_Incomplete_Translati

In [102]:
import os
import pandas as pd

files_to_check = {
    "Mixed dataset":
        "04_mixed_data/FactIndic_Mixed_250.csv",

    "CMI comparison":
        "05_quality_checks/CMI_MUTANT_PHINC_Comparison.csv",

    "Human translation review":
        "05_quality_checks/Human_Translation_Review_20.csv",

    "Real Hinglish final candidates":
        "06_real_hinglish/FactIndic_Real_Hinglish_Final_Candidates.csv",

    "Hinglish review summary":
        "06_real_hinglish/FactIndic_Hinglish_Final_Review_Summary.csv",

    "Claims":
        "07_claim_labels/FactIndic_50_Claim_Labeling_Sheet.csv"
}

for name, relative_path in files_to_check.items():
    path = os.path.join(base_path, relative_path)

    print("\n" + "=" * 65)
    print(name)

    if os.path.exists(path):
        temp = pd.read_csv(path)
        print("Rows:", len(temp))
        print("Columns:", temp.columns.tolist())
        display(temp.head(3))
    else:
        print("FILE NOT FOUND")


Mixed dataset
Rows: 250
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words', 'english_title', 'english_text', 'english_summary', 'mixed_10_article', 'mixed_30_article', 'mixed_50_article', 'mixed_10_article_english_pct', 'mixed_30_article_english_pct', 'mixed_50_article_english_pct']


,id,url,title,summary,text,article_words,summary_words,english_title,english_text,english_summary,mixed_10_article,mixed_30_article,mixed_50_article,mixed_10_article_english_pct,mixed_30_article_english_pct,mixed_50_article_english_pct
0,080419_brchopra_festival,https://www.bbc.com/hindi/entertainment/story/...,बीआर चोपड़ा की यादगार फ़िल्मों का विशेष उत्सव,भारतीय फ़िल्म इतिहास में फ़िल्मकार बीआर चोपड़ा...,शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ ...,294,34,Special celebration of the memorable film of B...,"On Friday evening, the scales of justice were ...","In Indian film history, movie actor BR Bangn i...",शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ ...,शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ ...,शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ ...,1.36,1.36,1.36
1,080613_afghan_jail,https://www.bbc.com/hindi/regionalnews/story/2...,तालेबान ने सैकड़ों क़ैदियों को छुड़ाया,अफ़ग़ानिस्तान के कंधार में तालेबान चरमपंथियों ...,कंधार के प्रांतीय सभा के अध्यक्ष और राष्ट्रपति...,289,17,The Locker Delivered Hundreds of Prisoners,The president of the local assembly of the Bar...,The locks on the locks at the side of Afghanis...,कंधार के प्रांतीय सभा के अध्यक्ष और President ...,कंधार के प्रांतीय सभा के अध्यक्ष और President ...,कंधार के प्रांतीय सभा के अध्यक्ष और President ...,2.42,2.42,2.42
2,130912_guinnness_record_bulgarian_swim_sr,https://www.bbc.com/hindi/international/2013/0...,रिकॉर्ड की ख़ातिर हाथ-पैर बांध कर की तैराकी,गिनीज़ वर्ल्ड रिकॉर्ड में अपना नाम दर्ज कराने ...,ख़बरों के मुताबिक़ जेन पेत्कोव को मैसेडोनिया क...,200,24,Type hands in a row for record,"According to the news, Jane Petav spent about ...","A Bulgarian man, hoping to register his name i...",ख़बरों के मुताबिक़ जेन पेत्कोव को मैसेडोनिया क...,ख़बरों के मुताबिक़ जेन पेत्कोव को मैसेडोनिया क...,ख़बरों के मुताबिक़ जेन पेत्कोव को मैसेडोनिया क...,1.00,1.00,1.00



CMI comparison
Rows: 2
Columns: ['Dataset', 'Records', 'Mean_CMI', 'Median_CMI']


,Dataset,Records,Mean_CMI,Median_CMI
0,MUTANT,100,6.26,4.77
1,PHINC,13738,17.61,16.67



Human translation review
Rows: 20
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words', 'nllb_english_title', 'nllb_english_article', 'nllb_english_summary', 'english_title', 'english_text', 'english_summary', 'hindi_cmi', 'english_cmi', 'hi_words', 'en_words', 'translation_ratio', 'review_status', 'translation_quality', 'names_correct', 'numbers_correct', 'meaning_preserved', 'comments']


,id,url,title,summary,text,article_words,summary_words,nllb_english_title,nllb_english_article,nllb_english_summary,...,english_cmi,hi_words,en_words,translation_ratio,review_status,translation_quality,names_correct,numbers_correct,meaning_preserved,comments
0,international-44164669,https://www.bbc.com/hindi/international-44164669,"अमरीका और उत्तर कोरिया: कभी हां, कभी ना",अमरीका और उत्तर कोरिया के बीच सिंगापुर में 12 ...,राष्ट्रपति डोनल्ड ट्रंप से बातचीत की हामी भरने...,695,24,NaN,NaN,NaN,...,99.22,695,387,0.556835,Needs human review,NaN,NaN,NaN,NaN,NaN
1,140702_wc2014_goldenboot_contenders_ar,https://www.bbc.com/hindi/sport/2014/07/140702...,विश्व कप: किसको मिलेगा गोल्डन बूट?,विश्व कप फ़ुटबॉल टूर्नामेंट में एक तरफ़ अब इंत...,इनमें कोलंबिया के जेम्स रॉड्रिगेज़ सबसे आगे है...,239,40,NaN,NaN,NaN,...,99.35,239,153,0.640167,Needs human review,NaN,NaN,NaN,NaN,NaN
2,121114_international_us_plus_obama_ia,https://www.bbc.com/hindi/international/2012/1...,राष्ट्रीय सुरक्षा को कोई ख़तरा नहीं: ओबामा,अमरीकी राष्ट्रपति बराक ओबामा ने कहा है कि अमरी...,प्रेम संबंधों के कारण सीआईए प्रमुख को इस्तीफ़ा...,790,33,NaN,NaN,NaN,...,100.00,790,439,0.555696,Needs human review,NaN,NaN,NaN,NaN,NaN


... (2 outputs left out)


Hinglish review summary
Rows: 2
Columns: ['source_dataset', 'No', 'Yes']


,source_dataset,No,Yes
0,GupShup,17,27
1,MUTANT,5,46



Claims
Rows: 50
Columns: ['Claim_ID', 'Article_ID', 'Claim', 'Source_Article', 'Verdict', 'Evidence_from_Article', 'Reviewer_Notes']


,Claim_ID,Article_ID,Claim,Source_Article,Verdict,Evidence_from_Article,Reviewer_Notes
0,FC001,040102_france_sikhs,फ़्रांस में रह रहे पाँच हज़ार सिख फ़्रांस के उ...,वे भारत के प्रधानमंत्री से आग्रह कर रहे हैं कि...,NOT_ENOUGH_INFO,यह क़ानून मुसलमानों के ही नहीं सिखों के भी ख़ि...,"The opposition is supported, but the stated co..."
1,FC002,social-43068605,आंखों के रास्ते दिलों में कुटिया बनाने वाली प्...,प्रिया प्रकाश प्रिया प्रकाश की कजरारे नैनों वा...,SUPPORTED,पाकिस्तान में सोशल मीडिया पर प्रिया के इशारों ...,The article describes Priya-related videos and...
2,FC003,050420_pope_muslims,कई मुसलमान नेताओं ने नए पोप से अनुरोध किया है ...,शंका जताई गई है कि नए पोप शायद ऐसा करने में उत...,NOT_ENOUGH_INFO,कुछ कैथोलिक ... दूसरे संप्रदायों के साथ बातचीत...,"The excerpt mentions interfaith dialogue, but ..."


In [104]:
import os
import pandas as pd
import re

mixed_path = os.path.join(
    base_path,
    "04_mixed_data",
    "FactIndic_Mixed_250.csv"
)

mixed_df = pd.read_csv(mixed_path)

print("Mixed dataset loaded:", mixed_df.shape)

def english_word_percentage(text):
    words = str(text).split()
    if not words:
        return 0

    english_words = sum(
        bool(re.search(r"[A-Za-z]", word))
        for word in words
    )

    return round(100 * english_words / len(words), 2)


for level in ["10", "30", "50"]:
    col = f"mixed_{level}_article"
    actual = mixed_df[col].apply(english_word_percentage)

    print(f"\n{level}% MIXED VERSION")
    print("Mean actual English percentage:", round(actual.mean(), 2))
    print("Minimum:", actual.min())
    print("Maximum:", actual.max())
    print("Unique articles:", mixed_df[col].nunique())

print("\nARE THE MIXED VERSIONS DIFFERENT?")
print("10% vs 30% identical:",
      mixed_df["mixed_10_article"].equals(mixed_df["mixed_30_article"]))

print("30% vs 50% identical:",
      mixed_df["mixed_30_article"].equals(mixed_df["mixed_50_article"]))

Mixed dataset loaded: (250, 16)

10% MIXED VERSION
Mean actual English percentage: 2.79
Minimum: 0.0
Maximum: 9.95
Unique articles: 250

30% MIXED VERSION
Mean actual English percentage: 2.79
Minimum: 0.0
Maximum: 9.95
Unique articles: 250

50% MIXED VERSION
Mean actual English percentage: 2.79
Minimum: 0.0
Maximum: 9.95
Unique articles: 250

ARE THE MIXED VERSIONS DIFFERENT?
10% vs 30% identical: True
30% vs 50% identical: True


In [105]:
import inspect

print("GENERATOR FUNCTION AVAILABLE:",
      "create_better_mixed" in globals())

if "create_better_mixed" in globals():
    print("\nGENERATOR CODE:")
    print(inspect.getsource(create_better_mixed))
else:
    print("\nGenerator function is not loaded.")

print("\nENGLISH TEXT SAMPLE:")
print(mixed_df.loc[0, "english_text"][:500])

print("\nHINDI TEXT SAMPLE:")
print(mixed_df.loc[0, "text"][:500])

print("\nSAVED PERCENTAGE VALUES:")
display(mixed_df[
    ["mixed_10_article_english_pct",
     "mixed_30_article_english_pct",
     "mixed_50_article_english_pct"]
].head(10))

GENERATOR FUNCTION AVAILABLE: False

Generator function is not loaded.

ENGLISH TEXT SAMPLE:
On Friday evening, the scales of justice were shown in the Polish film. The little thing, the new rounds, and time during the celebration, will be seen to see the classical films. Bad health because of BRC's own function, BRCP's brother and famous filmer, he said, "My brother made his business at such a time. When people were apprehensive about the commercial success of such films because they were few, the stories of BRangangange's life were different from them. BR'Courourin raised many subject

HINDI TEXT SAMPLE:
शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी

,mixed_10_article_english_pct,mixed_30_article_english_pct,mixed_50_article_english_pct
0,1.36,1.36,1.36
1,2.42,2.42,2.42
2,1.00,1.00,1.00
3,4.08,4.08,4.08
4,1.37,1.37,1.37
5,2.93,2.93,2.93
6,1.16,1.16,1.16
7,1.14,1.14,1.14
8,3.50,3.50,3.50
9,0.58,0.58,0.58


In [106]:
import os
import json

dictionary_path = os.path.join(
    base_path,
    "04_mixed_data",
    "hindi_english_phrase_dictionary.json"
)

with open(dictionary_path, "r", encoding="utf-8") as f:
    phrase_data = json.load(f)

print("Dictionary type:", type(phrase_data))
print("Total entries:", len(phrase_data))

print("\nFIRST 15 ENTRIES:")
if isinstance(phrase_data, dict):
    for i, (hindi, english) in enumerate(phrase_data.items()):
        if i == 15:
            break
        print(f"{hindi}  -->  {english}")

elif isinstance(phrase_data, list):
    for item in phrase_data[:15]:
        print(item)

Dictionary type: <class 'dict'>
Total entries: 26

FIRST 15 ENTRIES:
इंसाफ़ का तराज़ू  -->  Insaf Ka Tarazu
छोटी सी बात  -->  Chhoti Si Baat
नया दौर  -->  Naya Daur
वक़्त  -->  Waqt
साधना  -->  Sadhana
गुमराह  -->  Gumrah
हमराज़  -->  Humraaz
क़ानून  -->  Kanoon
निकाह  -->  Nikaah
बीआर चोपड़ा  -->  B. R. Chopra
यश चोपड़ा  -->  Yash Chopra
रवि चोपड़ा  -->  Ravi Chopra
राज बब्बर  -->  Raj Babbar
बसु चटर्जी  -->  Basu Chatterjee
बॉबी बेदी  -->  Bobby Bedi


In [107]:
import re

# Replace longer phrases first to avoid partial replacements
phrases = sorted(
    phrase_data.items(),
    key=lambda x: len(x[0]),
    reverse=True
)

def calculate_coverage(text):
    mixed = str(text)

    for hindi, english in phrases:
        mixed = mixed.replace(hindi, english)

    words = mixed.split()

    english_words = sum(
        bool(re.search(r"[A-Za-z]", word))
        for word in words
    )

    percentage = (
        100 * english_words / len(words)
        if words else 0
    )

    return round(percentage, 2)


coverage = mixed_df["text"].apply(calculate_coverage)

print("DICTIONARY COVERAGE — ALL 250 ARTICLES")
print(coverage.describe())

print("\nArticles reaching each target:")
for target in [10, 30, 50]:
    print(
        f"{target}% or above:",
        (coverage >= target).sum(),
        "/ 250"
    )

print("\nFIRST 15 ARTICLE RESULTS:")
display(pd.DataFrame({
    "id": mixed_df["id"],
    "maximum_dictionary_english_pct": coverage
}).head(15))

DICTIONARY COVERAGE — ALL 250 ARTICLES
count    250.000000
mean       0.237400
std        1.395891
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max       17.550000
Name: text, dtype: float64

Articles reaching each target:
10% or above: 2 / 250
30% or above: 0 / 250
50% or above: 0 / 250

FIRST 15 ARTICLE RESULTS:


,id,maximum_dictionary_english_pct
0,080419_brchopra_festival,17.55
1,080613_afghan_jail,0.00
2,130912_guinnness_record_bulgarian_swim_sr,0.00
3,050713_us_un,0.00
4,120809_salman_pkp,11.78
5,080711_lanka_attack,0.00
6,social-43068605,0.00
7,international-51410040,0.08
8,india-55525807,0.44
9,international-41372477,0.00


In [108]:
import os
import pandas as pd

inventory_path = os.path.join(
    base_path,
    "09_sentence_level_translation",
    "Sentence_Level_Translation_Inventory.csv"
)

alignment_path = os.path.join(
    base_path,
    "05_quality_checks",
    "FactIndic_Sentence_Alignment.csv"
)

inventory_df = pd.read_csv(inventory_path)
alignment_df = pd.read_csv(alignment_path)

print("SENTENCE INVENTORY")
print("Rows:", len(inventory_df))
print("Columns:", inventory_df.columns.tolist())
display(inventory_df.head(5))

print("\nSENTENCE ALIGNMENT")
print("Rows:", len(alignment_df))
print("Columns:", alignment_df.columns.tolist())
display(alignment_df.head(5))

SENTENCE INVENTORY
Rows: 5214
Columns: ['sentence_id', 'id', 'sentence_no', 'hindi_text', 'hindi_words', 'source_tokens', 'translation', 'translation_status']


,sentence_id,id,sentence_no,hindi_text,hindi_words,source_tokens,translation,translation_status
0,1,080419_brchopra_festival,1,शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ ...,13,23,NaN,Pending
1,2,080419_brchopra_festival,2,"उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त ज...",16,23,NaN,Pending
2,3,080419_brchopra_festival,3,खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में ...,26,36,NaN,Pending
3,4,080419_brchopra_festival,4,"उन्होंने कहा, ""मेरे भाई ने अपने हिसाब से फ़िल्...",66,92,NaN,Pending
4,5,080419_brchopra_festival,5,यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौ...,32,40,NaN,Pending



SENTENCE ALIGNMENT
Rows: 5580
Columns: ['sample_id', 'sentence_number', 'hindi_sentence', 'english_sentence', 'alignment_method']


,sample_id,sentence_number,hindi_sentence,english_sentence,alignment_method
0,080419_brchopra_festival,1,शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ ...,"On Friday evening, the scales of justice were ...",approximate_proportional
1,080419_brchopra_festival,2,"उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त ज...","On Friday evening, the scales of justice were ...",approximate_proportional
2,080419_brchopra_festival,3,खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में ...,"The little thing, the new rounds, and time dur...",approximate_proportional
3,080419_brchopra_festival,4,"उन्होंने कहा, ""मेरे भाई ने अपने हिसाब से फ़िल्...","Bad health because of BRC's own function, BRCP...",approximate_proportional
4,080419_brchopra_festival,5,बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्वि...,When people were apprehensive about the commer...,approximate_proportional


In [109]:
print("INVENTORY STATUS:")
print(inventory_df["translation_status"].value_counts(dropna=False))

print("\nMISSING SENTENCE TRANSLATIONS:")
print(inventory_df["translation"].isna().sum())

print("\nALIGNMENT METHODS:")
print(alignment_df["alignment_method"].value_counts(dropna=False))

print("\nALIGNMENT SAMPLE:")
display(
    alignment_df[
        alignment_df["alignment_method"] != "approximate_proportional"
    ].head(10)
)

INVENTORY STATUS:
translation_status
Pending    5214
Name: count, dtype: int64

MISSING SENTENCE TRANSLATIONS:
5214

ALIGNMENT METHODS:
alignment_method
approximate_proportional    5580
Name: count, dtype: int64

ALIGNMENT SAMPLE:


,sample_id,sentence_number,hindi_sentence,english_sentence,alignment_method


In [110]:
english_path = os.path.join(
    base_path,
    "09_sentence_level_translation",
    "FactIndic_English_250_Reviewed.csv"
)

english_df = pd.read_csv(english_path)

print("Shape:", english_df.shape)
print("Columns:", english_df.columns.tolist())

display(english_df.head(3))

print("\nMISSING VALUES:")
print(english_df.isnull().sum())

Shape: (250, 4)
Columns: ['id', 'complete_english_translation', 'reconstructed_sentence_count', 'manual_review_status']


,id,complete_english_translation,reconstructed_sentence_count,manual_review_status
0,000001_ugc,You sent this material because of an incident ...,6,Human reviewed - no correction
1,031031_russia_removal,He is reported to have resigned following the ...,9,Manually corrected
2,031031_sharon_palestine,"By extending this invitation, Israel has chang...",8,Manually corrected



MISSING VALUES:
id                              0
complete_english_translation    0
reconstructed_sentence_count    0
manual_review_status            0
dtype: int64


In [111]:
import re

# Count dictionary phrase occurrences in the complete Hindi dataset
coverage_results = []

for _, row in mixed_df.iterrows():
    hindi = str(row["text"])
    total_words = len(hindi.split())

    matched_words = 0
    matched_phrases = 0

    for phrase, english in sorted(
        phrase_data.items(),
        key=lambda x: len(x[0]),
        reverse=True
    ):
        count = hindi.count(phrase)

        if count:
            matched_phrases += count
            matched_words += count * len(phrase.split())

    coverage_results.append({
        "id": row["id"],
        "total_words": total_words,
        "matched_phrases": matched_phrases,
        "matched_words": matched_words,
        "coverage_pct": round(
            matched_words / total_words * 100, 2
        ) if total_words else 0
    })

coverage_df = pd.DataFrame(coverage_results)

print("TOTAL MATCHED PHRASES:",
      coverage_df["matched_phrases"].sum())

print("ARTICLES WITH NO MATCH:",
      (coverage_df["matched_phrases"] == 0).sum())

print("\nAVERAGE DICTIONARY COVERAGE:",
      round(coverage_df["coverage_pct"].mean(), 2), "%")

display(coverage_df.sort_values(
    "coverage_pct", ascending=False
).head(10))

TOTAL MATCHED PHRASES: 143
ARTICLES WITH NO MATCH: 196

AVERAGE DICTIONARY COVERAGE: 0.21 %


,id,total_words,matched_phrases,matched_words,coverage_pct
0,080419_brchopra_festival,294,25,46,15.65
4,120809_salman_pkp,365,21,43,11.78
152,150526_shahrukh_tweet_salman_bajrangi_bhaijaan_hk,152,4,8,5.26
45,131201_karan_with_salman_sk,337,4,8,2.37
245,india-50975864,880,15,15,1.70
142,040102_france_sikhs,280,2,2,0.71
189,000001_ugc,154,1,1,0.65
13,081126_aamne_pachauri_alk,384,2,2,0.52
108,050112_nepalmao_amnesty,191,1,1,0.52
128,international-54702261,381,2,2,0.52


In [112]:
print("HINDI DATASET:")
print(mixed_df[["id", "text"]].head(1).to_string(index=False))

print("\nREVIEWED ENGLISH TRANSLATION:")
print(
    english_df[
        english_df["id"] == "080419_brchopra_festival"
    ]["complete_english_translation"].iloc[0]
)

print("\nRECONSTRUCTED SENTENCE COUNTS:")
print(
    reconstructed_df[
        reconstructed_df["id"] == "080419_brchopra_festival"
    ].to_string(index=False)
)

print("\nALIGNMENT METHOD COUNTS:")
print(alignment_df["alignment_method"].value_counts())

HINDI DATASET:
                      id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

In [113]:
pilot_id = "080419_brchopra_festival"

pilot_hindi = mixed_df.loc[
    mixed_df["id"] == pilot_id, "text"
].iloc[0]

pilot_phrases = {
    "इंसाफ़ का तराज़ू": "Insaf Ka Tarazu",
    "छोटी सी बात": "Chhoti Si Baat",
    "नया दौर": "Naya Daur",
    "वक़्त": "Waqt",
    "बीआर चोपड़ा": "BR Chopra",
    "यश चोपड़ा": "Yash Chopra",
    "विधवा पुनर्विवाह": "widow remarriage",
    "वेश्यावृत्ति": "prostitution",
    "साधना": "Sadhana",
    "गुमराह": "Gumrah",
    "हमराज़": "Hamraaz",
    "क़ानून": "Kanoon",
    "निकाह": "Nikah",
    "रवि चोपड़ा": "Ravi Chopra",
    "राज बब्बर": "Raj Babbar",
    "बासु चटर्जी": "Basu Chatterjee",
    "बॉबी बेदी": "Bobby Bedi"
}

pilot_mixed = pilot_hindi

for hindi, english in sorted(
    pilot_phrases.items(),
    key=lambda x: len(x[0]),
    reverse=True
):
    pilot_mixed = pilot_mixed.replace(hindi, english)

print("ORIGINAL HINDI:")
print(pilot_hindi)

print("\nPILOT CODE-MIXED:")
print(pilot_mixed)

print("\nENGLISH WORD PERCENTAGE:")
print(english_word_percentage(pilot_mixed), "%")

ORIGINAL HINDI:
शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी." बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है. गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में शामिल है. हालांकि बीआर चोपड़ा की ज़्यादातर फ़िल्में किसी न किसी सामाजिक मुद्दे पर आधारित ह

In [114]:
print("REPLACEMENT COUNTS")

for hindi, english in pilot_phrases.items():
    count = pilot_hindi.count(hindi)

    if count:
        print(f"{hindi} -> {english}: {count}")

print("\nWORD COUNTS")
print("Original words:", len(pilot_hindi.split()))
print("Mixed words:", len(pilot_mixed.split()))

print("\nENGLISH WORDS:", sum(
    bool(re.search(r"[A-Za-z]", word))
    for word in pilot_mixed.split()
))

REPLACEMENT COUNTS
इंसाफ़ का तराज़ू -> Insaf Ka Tarazu: 1
छोटी सी बात -> Chhoti Si Baat: 1
नया दौर -> Naya Daur: 4
वक़्त -> Waqt: 1
बीआर चोपड़ा -> BR Chopra: 8
यश चोपड़ा -> Yash Chopra: 2
विधवा पुनर्विवाह -> widow remarriage: 1
वेश्यावृत्ति -> prostitution: 2
साधना -> Sadhana: 1
गुमराह -> Gumrah: 1
हमराज़ -> Hamraaz: 1
क़ानून -> Kanoon: 1
निकाह -> Nikah: 1
रवि चोपड़ा -> Ravi Chopra: 1
राज बब्बर -> Raj Babbar: 1
बासु चटर्जी -> Basu Chatterjee: 1
बॉबी बेदी -> Bobby Bedi: 1

WORD COUNTS
Original words: 294
Mixed words: 294

ENGLISH WORDS: 51


In [115]:
total_words = len(pilot_mixed.split())
current_english = sum(
    bool(re.search(r"[A-Za-z]", word))
    for word in pilot_mixed.split()
)

print("Total words:", total_words)
print("Current English words:", current_english)

for target in [10, 30, 50]:
    required = round(total_words * target / 100)
    additional = max(0, required - current_english)

    print(
        f"{target}% target: {required} English words | "
        f"Additional needed: {additional}"
    )

Total words: 294
Current English words: 51
10% target: 29 English words | Additional needed: 0
30% target: 88 English words | Additional needed: 37
50% target: 147 English words | Additional needed: 96


In [116]:
print("REVIEWED ENGLISH TRANSLATION:\n")

print(
    english_df.loc[
        english_df["id"] == pilot_id,
        "complete_english_translation"
    ].iloc[0]
)

REVIEWED ENGLISH TRANSLATION:

The festival, which began on Friday evening, screened the film Insaf Ka Tarazu. Other classic films, including Chhoti Si Baat, Naya Daur and Waqt, will also be shown during the festival. BR Chopra could not attend the ceremony because of his poor health. The festival was inaugurated by his brother, the renowned filmmaker Yash Chopra. He said, "My brother made films in his own way, even at a time when people were doubtful about the commercial success of such films because they were different from the kinds of stories that were popular then." BR Chopra addressed several subjects in his films, including widow remarriage and prostitution. Among his memorable films, Naya Daur, released in 1957, reflects the ideals of the Nehru era, while Sadhana, released in 1958, raises questions about society's treatment of prostitution. Gumrah, Hamraaz, Kanoon and Nikah are also among the films he made. Although most of BR Chopra's films were based on social issues, these e

In [117]:
import re

hindi_sentences = [
    s.strip()
    for s in re.split(r'(?<=[।.!?])\s*', pilot_hindi)
    if s.strip()
]

english_text = english_df.loc[
    english_df["id"] == pilot_id,
    "complete_english_translation"
].iloc[0]

english_sentences = [
    s.strip()
    for s in re.split(r'(?<=[.!?])\s+', english_text)
    if s.strip()
]

print("HINDI SENTENCES:", len(hindi_sentences))
print("ENGLISH SENTENCES:", len(english_sentences))

for i, (h, e) in enumerate(
    zip(hindi_sentences, english_sentences), 1
):
    print(f"\n{'='*70}")
    print(f"SENTENCE {i}")
    print("\nHINDI:")
    print(h)
    print("\nENGLISH:")
    print(e)

if len(hindi_sentences) != len(english_sentences):
    print("\nWARNING: Sentence counts differ. Review unmatched sentences.")

HINDI SENTENCES: 15
ENGLISH SENTENCES: 13

SENTENCE 1

HINDI:
शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई.

ENGLISH:
The festival, which began on Friday evening, screened the film Insaf Ka Tarazu.

SENTENCE 2

HINDI:
उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी.

ENGLISH:
Other classic films, including Chhoti Si Baat, Naya Daur and Waqt, will also be shown during the festival.

SENTENCE 3

HINDI:
खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.

ENGLISH:
BR Chopra could not attend the ceremony because of his poor health.

SENTENCE 4

HINDI:
फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया.

ENGLISH:
The festival was inaugurated by his brother, the renowned filmmaker Yash Chopra.

SENTENCE 5

HINDI:
उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस 

In [118]:
import re

# Split Hindi text at sentence-ending punctuation
hindi_clean = pilot_hindi.replace("।", ".")
hindi_clean = re.sub(r'\s+', ' ', hindi_clean).strip()

hindi_sentences_fixed = [
    s.strip()
    for s in re.split(r'(?<=\.)\s+', hindi_clean)
    if s.strip()
]

print("FIXED HINDI SENTENCE COUNT:", len(hindi_sentences_fixed))

for i, sentence in enumerate(hindi_sentences_fixed, 1):
    print(f"\n{i}. {sentence}")

FIXED HINDI SENTENCE COUNT: 12

1. शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई.

2. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी.

3. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया.

4. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी." बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया.

5. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है.

6. गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में शामिल है.

7. हालांकि बीआर चोपड़ा की ज़्यादातर फ़िल

In [119]:
import pandas as pd

alignment_pilot = pd.DataFrame([
    {
        "Hindi_ID": 1,
        "Hindi_Segment": hindi_sentences_fixed[0],
        "English_Meaning": "The festival began on Friday evening and screened Insaf Ka Tarazu.",
        "Alignment_Status": "Matched"
    },
    {
        "Hindi_ID": 2,
        "Hindi_Segment": hindi_sentences_fixed[1],
        "English_Meaning": "Other classic films including Chhoti Si Baat, Naya Daur and Waqt will be shown.",
        "Alignment_Status": "Matched"
    },
    {
        "Hindi_ID": 3,
        "Hindi_Segment": hindi_sentences_fixed[2],
        "English_Meaning": "BR Chopra could not attend due to poor health. The festival was inaugurated by Yash Chopra.",
        "Alignment_Status": "Matched - combined"
    },
    {
        "Hindi_ID": 4,
        "Hindi_Segment": hindi_sentences_fixed[3],
        "English_Meaning": "His brother made films differently and addressed subjects such as widow remarriage and prostitution.",
        "Alignment_Status": "Matched - combined"
    },
    {
        "Hindi_ID": 5,
        "Hindi_Segment": hindi_sentences_fixed[4],
        "English_Meaning": "Naya Daur and Sadhana addressed social issues.",
        "Alignment_Status": "Matched"
    },
    {
        "Hindi_ID": 6,
        "Hindi_Segment": hindi_sentences_fixed[5],
        "English_Meaning": "Gumrah, Hamraaz, Kanoon and Nikah were among his films.",
        "Alignment_Status": "Matched"
    },
    {
        "Hindi_ID": 7,
        "Hindi_Segment": hindi_sentences_fixed[6],
        "English_Meaning": "His films addressed social issues and performed well at the box office.",
        "Alignment_Status": "Matched"
    },
    {
        "Hindi_ID": 8,
        "Hindi_Segment": hindi_sentences_fixed[7],
        "English_Meaning": "BR Chopra took a risk and made Naya Daur after other directors rejected its story.",
        "Alignment_Status": "Matched"
    },
    {
        "Hindi_ID": 9,
        "Hindi_Segment": hindi_sentences_fixed[8],
        "English_Meaning": "Ravi Chopra spoke about his father's commitment to his work.",
        "Alignment_Status": "Matched"
    },
    {
        "Hindi_ID": 10,
        "Hindi_Segment": hindi_sentences_fixed[9],
        "English_Meaning": "Ravi Chopra considers his father the best director. A colour version of Naya Daur was released.",
        "Alignment_Status": "Matched - combined"
    },
    {
        "Hindi_ID": 11,
        "Hindi_Segment": hindi_sentences_fixed[10],
        "English_Meaning": "A documentary about BR Chopra was screened.",
        "Alignment_Status": "Matched"
    },
    {
        "Hindi_ID": 12,
        "Hindi_Segment": hindi_sentences_fixed[11],
        "English_Meaning": "Several filmmakers and artists attended the event.",
        "Alignment_Status": "Matched"
    }
])

display(alignment_pilot)

,Hindi_ID,Hindi_Segment,English_Meaning,Alignment_Status
0,1,शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ ...,The festival began on Friday evening and scree...,Matched
1,2,"उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त ज...","Other classic films including Chhoti Si Baat, ...",Matched
2,3,खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में ...,BR Chopra could not attend due to poor health....,Matched - combined
3,4,"उन्होंने कहा, ""मेरे भाई ने अपने हिसाब से फ़िल्...",His brother made films differently and address...,Matched - combined
4,5,यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौ...,Naya Daur and Sadhana addressed social issues.,Matched
5,6,"गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में क...","Gumrah, Hamraaz, Kanoon and Nikah were among h...",Matched
6,7,हालांकि बीआर चोपड़ा की ज़्यादातर फ़िल्में किसी...,His films addressed social issues and performe...,Matched
7,8,नया दौर की कहानी को कई टॉप निर्देशकों ने ठुकरा...,BR Chopra took a risk and made Naya Daur after...,Matched
8,9,बीआर चोपड़ा के बेटे रवि चोपड़ा ने इस मौके पर क...,Ravi Chopra spoke about his father's commitmen...,Matched
9,10,मेरे लिए वो सर्वश्रेष्ठ निर्देशक हैं और रहेंगे...,Ravi Chopra considers his father the best dire...,Matched - combined


In [120]:
corrected_meanings = {
    1: "The festival began on Friday evening and screened Insaf Ka Tarazu.",
    2: "Other classic films, including Chhoti Si Baat, Naya Daur and Waqt, will be shown during the festival.",
    3: "BR Chopra could not attend because of poor health. The festival was inaugurated by his brother Yash Chopra.",
    4: "His brother made films in his own way, despite doubts about their commercial success. BR Chopra's films addressed subjects such as widow remarriage and prostitution.",
    5: "Naya Daur, released in 1957, reflected the ideals of the Nehru era. Sadhana, released in 1958, raised questions about society's treatment of prostitution.",
    6: "Gumrah, Hamraaz, Kanoon and Nikah were among the films he made.",
    7: "Although most of his films addressed social issues, they also performed well at the box office.",
    8: "Many leading directors rejected the story of Naya Daur, but BR Chopra took the risk and made the film.",
    9: "Ravi Chopra said that his father believed in getting the work done.",
    10: "Ravi Chopra said his father would always remain the best director. A colour version of Naya Daur was also released.",
    11: "A short documentary about BR Chopra's life was screened at the festival.",
    12: "Yash Chopra, Raj Babbar, Basu Chatterjee, music composer Ravi and producer Bobby Bedi attended the event."
}

alignment_pilot["English_Meaning"] = (
    alignment_pilot["Hindi_ID"].map(corrected_meanings)
)

display(alignment_pilot)

,Hindi_ID,Hindi_Segment,English_Meaning,Alignment_Status
0,1,शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ ...,The festival began on Friday evening and scree...,Matched
1,2,"उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त ज...","Other classic films, including Chhoti Si Baat,...",Matched
2,3,खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में ...,BR Chopra could not attend because of poor hea...,Matched - combined
3,4,"उन्होंने कहा, ""मेरे भाई ने अपने हिसाब से फ़िल्...","His brother made films in his own way, despite...",Matched - combined
4,5,यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौ...,"Naya Daur, released in 1957, reflected the ide...",Matched
5,6,"गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में क...","Gumrah, Hamraaz, Kanoon and Nikah were among t...",Matched
6,7,हालांकि बीआर चोपड़ा की ज़्यादातर फ़िल्में किसी...,Although most of his films addressed social is...,Matched
7,8,नया दौर की कहानी को कई टॉप निर्देशकों ने ठुकरा...,Many leading directors rejected the story of N...,Matched
8,9,बीआर चोपड़ा के बेटे रवि चोपड़ा ने इस मौके पर क...,Ravi Chopra said that his father believed in g...,Matched
9,10,मेरे लिए वो सर्वश्रेष्ठ निर्देशक हैं और रहेंगे...,Ravi Chopra said his father would always remai...,Matched - combined


In [121]:
pilot_30 = pilot_hindi

replacements_30 = [
    (
        "शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई.",
        "The festival, which began on Friday evening, screened Insaf Ka Tarazu."
    ),
    (
        "उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी.",
        "Other classic films, including Chhoti Si Baat, Naya Daur and Waqt, will also be shown during the festival."
    ),
    (
        "खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.",
        "BR Chopra could not attend the ceremony because of his poor health."
    )
]

for hindi, english in replacements_30:
    if hindi not in pilot_30:
        print("WARNING: Hindi clause not found:", hindi)
    else:
        pilot_30 = pilot_30.replace(hindi, english, 1)

total_words_30 = len(pilot_30.split())
english_words_30 = sum(
    bool(re.search(r"[A-Za-z]", word))
    for word in pilot_30.split()
)

print("30% PILOT:")
print(pilot_30)

print("\nTotal words:", total_words_30)
print("English words:", english_words_30)
print(
    "English percentage:",
    round(english_words_30 / total_words_30 * 100, 2),
    "%"
)

30% PILOT:
The festival, which began on Friday evening, screened Insaf Ka Tarazu. Other classic films, including Chhoti Si Baat, Naya Daur and Waqt, will also be shown during the festival. BR Chopra could not attend the ceremony because of his poor health.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी." बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है. गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में शामिल है. हालांकि बीआर चोपड़ा की ज़्यादातर फ़िल्में किसी न किसी सामाजिक 

In [122]:
replacements_extra = [
    (
        "फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया.",
        "The festival was inaugurated by his brother, the renowned filmmaker Yash Chopra."
    ),
    (
        "बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया.",
        "BR Chopra addressed several subjects in his films, including widow remarriage and prostitution."
    ),
    (
        "गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में शामिल है.",
        "Gumrah, Hamraaz, Kanoon and Nikah are also among the films he made."
    )
]

pilot_30_v2 = pilot_30

for hindi, english in replacements_extra:
    if hindi in pilot_30_v2:
        pilot_30_v2 = pilot_30_v2.replace(hindi, english, 1)
    else:
        print("Not found:", hindi)

total = len(pilot_30_v2.split())
english = sum(
    bool(re.search(r"[A-Za-z]", word))
    for word in pilot_30_v2.split()
)

print("Total words:", total)
print("English words:", english)
print("English percentage:", round(english / total * 100, 2), "%")
print("\nUPDATED PILOT:\n")
print(pilot_30_v2)

Total words: 286
English words: 77
English percentage: 26.92 %

UPDATED PILOT:

The festival, which began on Friday evening, screened Insaf Ka Tarazu. Other classic films, including Chhoti Si Baat, Naya Daur and Waqt, will also be shown during the festival. BR Chopra could not attend the ceremony because of his poor health.The festival was inaugurated by his brother, the renowned filmmaker Yash Chopra. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी." BR Chopra addressed several subjects in his films, including widow remarriage and prostitution. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है. Gumrah, Hamraaz, Kanoon and Nikah are also among the films he made. हालांकि ब

In [123]:
pilot_30_v3 = pilot_30_v2

hindi = (
    "हालांकि बीआर चोपड़ा की ज़्यादातर फ़िल्में किसी न किसी सामाजिक मुद्दे पर आधारित होती थीं "
    "लेकिन इन प्रयोगात्मक फ़िल्मों ने बॉक्स ऑफ़िस पर भी कमाल दिखाया."
)

english = (
    "Although most of BR Chopra's films were based on social issues, "
    "these experimental films also performed well at the box office."
)

if hindi in pilot_30_v3:
    pilot_30_v3 = pilot_30_v3.replace(hindi, english, 1)
else:
    print("Sentence not found. Check punctuation.")

total = len(pilot_30_v3.split())
english_count = sum(
    bool(re.search(r"[A-Za-z]", word))
    for word in pilot_30_v3.split()
)

print("Total words:", total)
print("English words:", english_count)
print("English percentage:", round(english_count / total * 100, 2), "%")

Total words: 281
English words: 98
English percentage: 34.88 %


In [124]:
pilot_50 = pilot_30_v3

replacements_50 = [
    (
        "नया दौर की कहानी को कई टॉप निर्देशकों ने ठुकरा दिया था लेकिन बीआर चोपड़ा ने रिस्क लिया और फ़िल्म बनाई.",
        "Many leading directors had rejected the story of Naya Daur, but BR Chopra took the risk and made the film."
    ),
    (
        "फ़िल्म उत्सव में बीआर चोपड़ा के जीवन पर आधारित एक लघु वृत्तचित्र भी दिखाया गया.",
        "A short documentary based on BR Chopra's life was also screened at the festival."
    )
]

for hindi, english in replacements_50:
    if hindi in pilot_50:
        pilot_50 = pilot_50.replace(hindi, english, 1)
    else:
        print("Not found:", hindi)

total = len(pilot_50.split())
english_count = sum(
    bool(re.search(r"[A-Za-z]", word))
    for word in pilot_50.split()
)

print("Total words:", total)
print("English words:", english_count)
print("English percentage:", round(english_count / total * 100, 2), "%")

Total words: 279
English words: 132
English percentage: 47.31 %


In [125]:
pilot_results = pd.DataFrame([
    {
        "Version": "10%",
        "Text": pilot_mixed
    },
    {
        "Version": "30%",
        "Text": pilot_30_v3
    },
    {
        "Version": "50%",
        "Text": pilot_50
    }
])

for i, row in pilot_results.iterrows():
    words = row["Text"].split()
    english_words = sum(
        bool(re.search(r"[A-Za-z]", word))
        for word in words
    )

    pilot_results.loc[i, "Total_Words"] = len(words)
    pilot_results.loc[i, "English_Words"] = english_words
    pilot_results.loc[i, "English_Percentage"] = round(
        english_words / len(words) * 100, 2
    )

display(
    pilot_results[
        ["Version", "Total_Words", "English_Words", "English_Percentage"]
    ]
)

,Version,Total_Words,English_Words,English_Percentage
0,10%,294.0,51.0,17.35
1,30%,281.0,98.0,34.88
2,50%,279.0,132.0,47.31


In [126]:
import os

pilot_folder = os.path.join(base_path, "04_mixed_data", "pilot")
os.makedirs(pilot_folder, exist_ok=True)

pilot_output = pd.DataFrame({
    "id": [pilot_id],
    "hindi_article": [pilot_hindi],
    "mixed_10_article": [pilot_mixed],
    "mixed_30_article": [pilot_30_v3],
    "mixed_50_article": [pilot_50],
    "english_10_pct": [17.35],
    "english_30_pct": [34.88],
    "english_50_pct": [47.31]
})

pilot_file = os.path.join(
    pilot_folder,
    "FactIndic_Mixing_Pilot_1.csv"
)

pilot_output.to_csv(pilot_file, index=False, encoding="utf-8-sig")

print("Pilot saved:", pilot_file)
print("Rows saved:", len(pilot_output))

Pilot saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/pilot/FactIndic_Mixing_Pilot_1.csv
Rows saved: 1


In [127]:
print("ORIGINAL HINDI ARTICLE:\n")
print(pilot_hindi)

print("\n" + "="*80)
print("30% CODE-MIXED ARTICLE:\n")
print(pilot_30_v3)

ORIGINAL HINDI ARTICLE:

शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी." बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है. गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में शामिल है. हालांकि बीआर चोपड़ा की ज़्यादातर फ़िल्में किसी न किसी सामाजिक मुद्दे पर

In [128]:
quote_hindi = (
    'उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- '
    'वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर '
    'आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, '
    'उस समय जिस तरह की कहानियों का चलन था, '
    'बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी."'
)

quote_english = (
    'He said, "My brother made films in his own way, even at a time '
    'when people were doubtful about the commercial success of such films '
    'because they were different from the kinds of stories that were popular then."'
)

pilot_30_reviewed = pilot_30_v3.replace(
    quote_hindi,
    quote_english,
    1
)

print("Updated quotation:")
print(pilot_30_reviewed)

Updated quotation:
The festival, which began on Friday evening, screened Insaf Ka Tarazu. Other classic films, including Chhoti Si Baat, Naya Daur and Waqt, will also be shown during the festival. BR Chopra could not attend the ceremony because of his poor health.The festival was inaugurated by his brother, the renowned filmmaker Yash Chopra. He said, "My brother made films in his own way, even at a time when people were doubtful about the commercial success of such films because they were different from the kinds of stories that were popular then." BR Chopra addressed several subjects in his films, including widow remarriage and prostitution. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है. Gumrah, Hamraaz, Kanoon and Nikah are also among the films he made. Although most of BR Chopra's films were based on social issues, these experimental films also performed well at the box o

In [129]:
import re

pilot_30_reviewed = re.sub(
    r'([.!?])(?=[A-Za-zअ-ह])',
    r'\1 ',
    pilot_30_reviewed
)

total = len(pilot_30_reviewed.split())

english_count = sum(
    bool(re.search(r"[A-Za-z]", word))
    for word in pilot_30_reviewed.split()
)

print("Total words:", total)
print("English words:", english_count)
print(
    "English percentage:",
    round(english_count / total * 100, 2),
    "%"
)

print("\nFINAL 30% PILOT:\n")
print(pilot_30_reviewed)

Total words: 268
English words: 137
English percentage: 51.12 %

FINAL 30% PILOT:

The festival, which began on Friday evening, screened Insaf Ka Tarazu. Other classic films, including Chhoti Si Baat, Naya Daur and Waqt, will also be shown during the festival. BR Chopra could not attend the ceremony because of his poor health. The festival was inaugurated by his brother, the renowned filmmaker Yash Chopra. He said, "My brother made films in his own way, even at a time when people were doubtful about the commercial success of such films because they were different from the kinds of stories that were popular then." BR Chopra addressed several subjects in his films, including widow remarriage and prostitution. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है. Gumrah, Hamraaz, Kanoon and Nikah are also among the films he made. Although most of BR Chopra's films were based on social 

In [130]:
corrected_pilot_path = os.path.join(
    pilot_folder,
    "FactIndic_Pilot_30_Quotation_Corrected.csv"
)

pd.DataFrame([{
    "id": pilot_id,
    "hindi_article": pilot_hindi,
    "corrected_mixed_article": pilot_30_reviewed,
    "english_percentage": 51.12,
    "review_status": "Meaning correction completed; target adjustment pending"
}]).to_csv(
    corrected_pilot_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", corrected_pilot_path)

Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/pilot/FactIndic_Pilot_30_Quotation_Corrected.csv


In [131]:
import re
from itertools import combinations

# English sentences and their original Hindi equivalents
replacements = {
    "The festival, which began on Friday evening, screened Insaf Ka Tarazu.":
    "शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई.",

    "Other classic films, including Chhoti Si Baat, Naya Daur and Waqt, will also be shown during the festival.":
    "उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी.",

    "BR Chopra could not attend the ceremony because of his poor health.":
    "खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.",

    "The festival was inaugurated by his brother, the renowned filmmaker Yash Chopra.":
    "फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया.",

    "BR Chopra addressed several subjects in his films, including widow remarriage and prostitution.":
    "बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया.",

    "Gumrah, Hamraaz, Kanoon and Nikah are also among the films he made.":
    "गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में शामिल है.",

    "Although most of BR Chopra's films were based on social issues, these experimental films also performed well at the box office.":
    "हालांकि बीआर चोपड़ा की ज़्यादातर फ़िल्में किसी न किसी सामाजिक मुद्दे पर आधारित होती थीं लेकिन इन प्रयोगात्मक फ़िल्मों ने बॉक्स ऑफ़िस पर भी कमाल दिखाया."
}

english_sentences = list(replacements.keys())

candidates = []

for r in range(len(english_sentences) + 1):
    for selected in combinations(english_sentences, r):
        candidate = pilot_30_reviewed

        for sentence in selected:
            candidate = candidate.replace(
                sentence,
                replacements[sentence],
                1
            )

        words = candidate.split()
        english_words = sum(
            bool(re.search(r"[A-Za-z]", word))
            for word in words
        )

        percentage = english_words / len(words) * 100

        candidates.append({
            "Text": candidate,
            "Total_Words": len(words),
            "English_Words": english_words,
            "English_Percentage": round(percentage, 2),
            "Reverted_Sentences": len(selected)
        })

best_30 = min(
    candidates,
    key=lambda x: abs(x["English_Percentage"] - 30)
)

pilot_30_final = best_30["Text"]

print("Closest 30% candidate:")
print("Total words:", best_30["Total_Words"])
print("English words:", best_30["English_Words"])
print("English percentage:", best_30["English_Percentage"], "%")
print("Sentences returned to Hindi:", best_30["Reverted_Sentences"])

print("\nCANDIDATE ARTICLE:\n")
print(pilot_30_final)

Closest 30% candidate:
Total words: 273
English words: 82
English percentage: 30.04 %
Sentences returned to Hindi: 4

CANDIDATE ARTICLE:

The festival, which began on Friday evening, screened Insaf Ka Tarazu. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. BR Chopra could not attend the ceremony because of his poor health. फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. He said, "My brother made films in his own way, even at a time when people were doubtful about the commercial success of such films because they were different from the kinds of stories that were popular then." बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है. गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में शामिल है. Althoug

In [132]:
pilot_30_path = os.path.join(
    pilot_folder,
    "FactIndic_Pilot_30_Final.csv"
)

pd.DataFrame([{
    "id": pilot_id,
    "hindi_article": pilot_hindi,
    "mixed_30_article": pilot_30_final,
    "english_percentage": 30.04,
    "review_status": "Pending human fluency review"
}]).to_csv(
    pilot_30_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", pilot_30_path)

Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/pilot/FactIndic_Pilot_30_Final.csv


In [133]:
import re
from itertools import combinations

# Start from the corrected quotation version
base_50 = pilot_30_reviewed

# English sentences that can be converted back to their Hindi originals
replacements_50 = {
    "The festival, which began on Friday evening, screened Insaf Ka Tarazu.":
    "शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई.",

    "Other classic films, including Chhoti Si Baat, Naya Daur and Waqt, will also be shown during the festival.":
    "उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी.",

    "BR Chopra could not attend the ceremony because of his poor health.":
    "खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.",

    "The festival was inaugurated by his brother, the renowned filmmaker Yash Chopra.":
    "फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया.",

    "BR Chopra addressed several subjects in his films, including widow remarriage and prostitution.":
    "बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया.",

    "Gumrah, Hamraaz, Kanoon and Nikah are also among the films he made.":
    "गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में शामिल है.",

    "Although most of BR Chopra's films were based on social issues, these experimental films also performed well at the box office.":
    "हालांकि बीआर चोपड़ा की ज़्यादातर फ़िल्में किसी न किसी सामाजिक मुद्दे पर आधारित होती थीं लेकिन इन प्रयोगात्मक फ़िल्मों ने बॉक्स ऑफ़िस पर भी कमाल दिखाया.",

    "Many leading directors had rejected the story of Naya Daur, but BR Chopra took the risk and made the film.":
    "नया दौर की कहानी को कई टॉप निर्देशकों ने ठुकरा दिया था लेकिन बीआर चोपड़ा ने रिस्क लिया और फ़िल्म बनाई.",

    "A short documentary based on BR Chopra's life was also screened at the festival.":
    "फ़िल्म उत्सव में बीआर चोपड़ा के जीवन पर आधारित एक लघु वृत्तचित्र भी दिखाया गया."
}

english_sentences_50 = list(replacements_50.keys())

candidates_50 = []

for r in range(len(english_sentences_50) + 1):
    for selected in combinations(english_sentences_50, r):
        candidate = base_50

        for sentence in selected:
            candidate = candidate.replace(
                sentence,
                replacements_50[sentence],
                1
            )

        words = candidate.split()
        english_words = sum(
            bool(re.search(r"[A-Za-z]", word))
            for word in words
        )

        percentage = english_words / len(words) * 100

        candidates_50.append({
            "Text": candidate,
            "Total_Words": len(words),
            "English_Words": english_words,
            "English_Percentage": round(percentage, 2),
            "Reverted_Sentences": len(selected)
        })

best_50 = min(
    candidates_50,
    key=lambda x: abs(x["English_Percentage"] - 50)
)

pilot_50_final = best_50["Text"]

print("Closest 50% candidate:")
print("Total words:", best_50["Total_Words"])
print("English words:", best_50["English_Words"])
print("English percentage:", best_50["English_Percentage"], "%")
print("Sentences returned to Hindi:", best_50["Reverted_Sentences"])

print("\nCANDIDATE ARTICLE:\n")
print(pilot_50_final)

Closest 50% candidate:
Total words: 268
English words: 137
English percentage: 51.12 %
Sentences returned to Hindi: 0

CANDIDATE ARTICLE:

The festival, which began on Friday evening, screened Insaf Ka Tarazu. Other classic films, including Chhoti Si Baat, Naya Daur and Waqt, will also be shown during the festival. BR Chopra could not attend the ceremony because of his poor health. The festival was inaugurated by his brother, the renowned filmmaker Yash Chopra. He said, "My brother made films in his own way, even at a time when people were doubtful about the commercial success of such films because they were different from the kinds of stories that were popular then." BR Chopra addressed several subjects in his films, including widow remarriage and prostitution. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है. Gumrah, Hamraaz, Kanoon and Nikah are also among the films he made. 

In [134]:
pilot_50_path = os.path.join(
    pilot_folder,
    "FactIndic_Pilot_50_Final.csv"
)

pd.DataFrame([{
    "id": pilot_id,
    "hindi_article": pilot_hindi,
    "mixed_50_article": pilot_50_final,
    "english_percentage": 51.12,
    "review_status": "Pending human fluency review"
}]).to_csv(
    pilot_50_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", pilot_50_path)

Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/pilot/FactIndic_Pilot_50_Final.csv


In [135]:
import re
from itertools import combinations

# Start with the original Hindi article
base_10 = pilot_hindi

# Candidate complete English sentences from the reviewed translation
english_options_10 = [
    "The festival, which began on Friday evening, screened Insaf Ka Tarazu.",
    "Other classic films, including Chhoti Si Baat, Naya Daur and Waqt, will also be shown during the festival.",
    "BR Chopra could not attend the ceremony because of his poor health.",
    "The festival was inaugurated by his brother, the renowned filmmaker Yash Chopra.",
    "BR Chopra addressed several subjects in his films, including widow remarriage and prostitution.",
    "Gumrah, Hamraaz, Kanoon and Nikah are also among the films he made.",
    "A short documentary based on BR Chopra's life was also screened at the festival."
]

# Corresponding Hindi sentences
hindi_options_10 = [
    "शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई.",
    "उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी.",
    "खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.",
    "फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया.",
    "बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया.",
    "गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में शामिल है.",
    "फ़िल्म उत्सव में बीआर चोपड़ा के जीवन पर आधारित एक लघु वृत्तचित्र भी दिखाया गया."
]

candidates_10 = []

for r in range(1, len(english_options_10) + 1):
    for selected_indices in combinations(range(7), r):
        candidate = base_10

        for i in selected_indices:
            candidate = candidate.replace(
                hindi_options_10[i],
                english_options_10[i],
                1
            )

        words = candidate.split()
        english_words = sum(
            bool(re.search(r"[A-Za-z]", word))
            for word in words
        )

        percentage = english_words / len(words) * 100

        candidates_10.append({
            "Text": candidate,
            "Total_Words": len(words),
            "English_Words": english_words,
            "English_Percentage": round(percentage, 2)
        })

best_10 = min(
    candidates_10,
    key=lambda x: abs(x["English_Percentage"] - 10)
)

pilot_10_final = best_10["Text"]

print("Closest 10% candidate:")
print("Total words:", best_10["Total_Words"])
print("English words:", best_10["English_Words"])
print("English percentage:", best_10["English_Percentage"], "%")

print("\nCANDIDATE ARTICLE:\n")
print(pilot_10_final)

Closest 10% candidate:
Total words: 294
English words: 29
English percentage: 9.86 %

CANDIDATE ARTICLE:

The festival, which began on Friday evening, screened Insaf Ka Tarazu. Other classic films, including Chhoti Si Baat, Naya Daur and Waqt, will also be shown during the festival. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी." बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है. गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ

In [136]:
pilot_10_path = os.path.join(
    pilot_folder,
    "FactIndic_Pilot_10_Final.csv"
)

pd.DataFrame([{
    "id": pilot_id,
    "hindi_article": pilot_hindi,
    "mixed_10_article": pilot_10_final,
    "english_percentage": 9.86,
    "review_status": "Pending human fluency review"
}]).to_csv(
    pilot_10_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", pilot_10_path)

Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/pilot/FactIndic_Pilot_10_Final.csv


In [137]:
import pandas as pd
import os
import re

pilot_master = pd.DataFrame([{
    "id": pilot_id,
    "hindi_article": pilot_hindi,
    "mixed_10_article": pilot_10_final,
    "mixed_30_article": pilot_30_final,
    "mixed_50_article": pilot_50_final
}])

# Calculate actual English percentages
for level in ["10", "30", "50"]:
    column = f"mixed_{level}_article"

    pilot_master[f"mixed_{level}_english_pct"] = (
        pilot_master[column].apply(
            lambda text: round(
                sum(bool(re.search(r"[A-Za-z]", word))
                    for word in text.split())
                / len(text.split()) * 100,
                2
            )
        )
    )

master_path = os.path.join(
    pilot_folder,
    "FactIndic_Pilot_Master.csv"
)

pilot_master.to_csv(
    master_path,
    index=False,
    encoding="utf-8-sig"
)

print("Master pilot saved:", master_path)

display(
    pilot_master[
        [
            "id",
            "mixed_10_english_pct",
            "mixed_30_english_pct",
            "mixed_50_english_pct"
        ]
    ]
)

Master pilot saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/pilot/FactIndic_Pilot_Master.csv


,id,mixed_10_english_pct,mixed_30_english_pct,mixed_50_english_pct
0,080419_brchopra_festival,9.86,30.04,51.12


In [138]:
for level in ["10", "30", "50"]:
    text = pilot_master.loc[0, f"mixed_{level}_article"]

    print(f"\n{'='*50}")
    print(f"{level}% VERSION")
    print("Characters:", len(text))
    print("Missing text:", pd.isna(text) or not text.strip())
    print("Contains 1957:", "1957" in text)
    print("Contains 1958:", "1958" in text)
    print("Contains BR Chopra:", "BR Chopra" in text or "बीआर चोपड़ा" in text)
    print("Contains Yash Chopra:", "Yash Chopra" in text or "यश चोपड़ा" in text)


10% VERSION
Characters: 1585
Missing text: False
Contains 1957: True
Contains 1958: True
Contains BR Chopra: True
Contains Yash Chopra: True

30% VERSION
Characters: 1492
Missing text: False
Contains 1957: True
Contains 1958: True
Contains BR Chopra: True
Contains Yash Chopra: True

50% VERSION
Characters: 1498
Missing text: False
Contains 1957: True
Contains 1958: True
Contains BR Chopra: True
Contains Yash Chopra: True


In [139]:
review_path = os.path.join(
    pilot_folder,
    "FactIndic_Pilot_Human_Review.csv"
)

review_df = pd.DataFrame([
    {
        "id": pilot_id,
        "mixing_level": level,
        "english_percentage": percentage,
        "fluency_score_1_to_5": "",
        "meaning_preserved_yes_no": "",
        "names_numbers_correct_yes_no": "",
        "reviewer_comments": "",
        "review_status": "Pending"
    }
    for level, percentage in [
        ("10%", 9.86),
        ("30%", 30.04),
        ("50%", 51.12)
    ]
])

review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

display(review_df)
print("\nReview sheet saved:", review_path)

,id,mixing_level,english_percentage,fluency_score_1_to_5,meaning_preserved_yes_no,names_numbers_correct_yes_no,reviewer_comments,review_status
0,080419_brchopra_festival,10%,9.86,,,,,Pending
1,080419_brchopra_festival,30%,30.04,,,,,Pending
2,080419_brchopra_festival,50%,51.12,,,,,Pending



Review sheet saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/pilot/FactIndic_Pilot_Human_Review.csv


In [140]:
review_df.loc[0, "fluency_score_1_to_5"] = 4
review_df.loc[0, "meaning_preserved_yes_no"] = "Yes"
review_df.loc[0, "names_numbers_correct_yes_no"] = "Yes"
review_df.loc[0, "reviewer_comments"] = (
    "Film titles and main facts appear preserved. "
    "Check sentence flow and punctuation."
)

review_df.loc[0, "review_status"] = "Reviewed"

review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

display(review_df)

,id,mixing_level,english_percentage,fluency_score_1_to_5,meaning_preserved_yes_no,names_numbers_correct_yes_no,reviewer_comments,review_status
0,080419_brchopra_festival,10%,9.86,4,Yes,Yes,Film titles and main facts appear preserved. C...,Reviewed
1,080419_brchopra_festival,30%,30.04,,,,,Pending
2,080419_brchopra_festival,50%,51.12,,,,,Pending


In [141]:
print("30% CODE-MIXED ARTICLE:\n")
print(pilot_30_final)

30% CODE-MIXED ARTICLE:

The festival, which began on Friday evening, screened Insaf Ka Tarazu. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. BR Chopra could not attend the ceremony because of his poor health. फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. He said, "My brother made films in his own way, even at a time when people were doubtful about the commercial success of such films because they were different from the kinds of stories that were popular then." बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है. गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में शामिल है. Although most of BR Chopra's films were based on social issues, these experimental films also performed well at the box 

In [142]:
review_df.loc[1, "fluency_score_1_to_5"] = 3
review_df.loc[1, "meaning_preserved_yes_no"] = "Yes"
review_df.loc[1, "names_numbers_correct_yes_no"] = "Yes"
review_df.loc[1, "reviewer_comments"] = (
    "Main facts, dates and names preserved. "
    "Yash Chopra quotation translated; Ravi Chopra quotation remains Hindi. "
    "Some language transitions feel abrupt."
)
review_df.loc[1, "review_status"] = "Reviewed"

review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

display(review_df)

,id,mixing_level,english_percentage,fluency_score_1_to_5,meaning_preserved_yes_no,names_numbers_correct_yes_no,reviewer_comments,review_status
0,080419_brchopra_festival,10%,9.86,4,Yes,Yes,Film titles and main facts appear preserved. C...,Reviewed
1,080419_brchopra_festival,30%,30.04,3,Yes,Yes,"Main facts, dates and names preserved. Yash Ch...",Reviewed
2,080419_brchopra_festival,50%,51.12,,,,,Pending


In [143]:
print("50% CODE-MIXED ARTICLE:\n")
print(pilot_50_final)

50% CODE-MIXED ARTICLE:

The festival, which began on Friday evening, screened Insaf Ka Tarazu. Other classic films, including Chhoti Si Baat, Naya Daur and Waqt, will also be shown during the festival. BR Chopra could not attend the ceremony because of his poor health. The festival was inaugurated by his brother, the renowned filmmaker Yash Chopra. He said, "My brother made films in his own way, even at a time when people were doubtful about the commercial success of such films because they were different from the kinds of stories that were popular then." BR Chopra addressed several subjects in his films, including widow remarriage and prostitution. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है. Gumrah, Hamraaz, Kanoon and Nikah are also among the films he made. Although most of BR Chopra's films were based on social issues, these experimental films also performed well at th

In [144]:
review_df.loc[2, "fluency_score_1_to_5"] = 3
review_df.loc[2, "meaning_preserved_yes_no"] = "Yes"
review_df.loc[2, "names_numbers_correct_yes_no"] = "Yes"

review_df.loc[2, "reviewer_comments"] = (
    "Main facts, dates and names preserved. "
    "Yash Chopra quotation translated; Ravi Chopra quotation remains Hindi. "
    "Some language transitions feel abrupt."
)

review_df.loc[2, "review_status"] = "Reviewed"

review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

display(review_df)

,id,mixing_level,english_percentage,fluency_score_1_to_5,meaning_preserved_yes_no,names_numbers_correct_yes_no,reviewer_comments,review_status
0,080419_brchopra_festival,10%,9.86,4,Yes,Yes,Film titles and main facts appear preserved. C...,Reviewed
1,080419_brchopra_festival,30%,30.04,3,Yes,Yes,"Main facts, dates and names preserved. Yash Ch...",Reviewed
2,080419_brchopra_festival,50%,51.12,3,Yes,Yes,"Main facts, dates and names preserved. Yash Ch...",Reviewed


In [145]:
import os
import shutil

pilot_folder = os.path.join(
    base_path, "04_mixed_data", "pilot"
)

backup_folder = os.path.join(
    pilot_folder, "Validated_Pilot_Backup"
)

os.makedirs(backup_folder, exist_ok=True)

files_to_backup = [
    "FactIndic_Pilot_10_Final.csv",
    "FactIndic_Pilot_30_Final.csv",
    "FactIndic_Pilot_50_Final.csv",
    "FactIndic_Pilot_Master.csv",
    "FactIndic_Pilot_Human_Review.csv"
]

for filename in files_to_backup:
    source = os.path.join(pilot_folder, filename)
    destination = os.path.join(backup_folder, filename)

    if os.path.exists(source):
        shutil.copy2(source, destination)
        print("Backed up:", filename)
    else:
        print("Missing:", filename)

print("\nBackup folder:", backup_folder)
print("Files saved:", len(os.listdir(backup_folder)))

Backed up: FactIndic_Pilot_10_Final.csv
Backed up: FactIndic_Pilot_30_Final.csv
Backed up: FactIndic_Pilot_50_Final.csv
Backed up: FactIndic_Pilot_Master.csv
Backed up: FactIndic_Pilot_Human_Review.csv

Backup folder: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/pilot/Validated_Pilot_Backup
Files saved: 5


In [146]:
import os
import pandas as pd

translation_folder = os.path.join(
    base_path, "09_sentence_level_translation"
)

english_path = os.path.join(
    translation_folder,
    "FactIndic_English_250_Reviewed.csv"
)

clean_path = os.path.join(
    base_path,
    "02_clean_data",
    "XL_Sum_Hindi_Cleaned_250.csv"
)

english_df = pd.read_csv(english_path)
clean_df = pd.read_csv(clean_path)

print("ENGLISH TRANSLATIONS")
print("Shape:", english_df.shape)
print("Columns:", english_df.columns.tolist())
print("Missing values:", english_df.isnull().sum().sum())
print("Unique IDs:", english_df["id"].nunique())

print("\nHINDI SOURCE DATA")
print("Shape:", clean_df.shape)
print("Columns:", clean_df.columns.tolist())
print("Missing values:", clean_df.isnull().sum().sum())
print("Unique IDs:", clean_df["id"].nunique())

print("\nID MATCHING")
print("Matching IDs:", len(
    set(english_df["id"]) & set(clean_df["id"])
))

ENGLISH TRANSLATIONS
Shape: (250, 4)
Columns: ['id', 'complete_english_translation', 'reconstructed_sentence_count', 'manual_review_status']
Missing values: 0
Unique IDs: 250

HINDI SOURCE DATA
Shape: (250, 7)
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words']
Missing values: 0
Unique IDs: 250

ID MATCHING
Matching IDs: 250


In [147]:
# Keep only the required English translation columns
english_source = english_df[
    ["id", "complete_english_translation"]
].copy()

# Merge Hindi and English data
source_250 = clean_df.merge(
    english_source,
    on="id",
    how="inner",
    validate="one_to_one"
)

# Rename columns clearly
source_250 = source_250.rename(columns={
    "text": "hindi_article",
    "complete_english_translation": "english_article"
})

# Validate
print("Combined shape:", source_250.shape)
print("Unique IDs:", source_250["id"].nunique())
print("Missing values:", source_250.isnull().sum().sum())
print("Duplicate IDs:", source_250["id"].duplicated().sum())

# Preview
display(source_250[
    ["id", "hindi_article", "english_article"]
].head(2))

# Save as a new staging file
staging_path = os.path.join(
    base_path,
    "04_mixed_data",
    "FactIndic_Source_250_Staging.csv"
)

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:", staging_path)

Combined shape: (250, 8)
Unique IDs: 250
Missing values: 0
Duplicate IDs: 0


,id,hindi_article,english_article
0,080419_brchopra_festival,शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ ...,"The festival, which began on Friday evening, s..."
1,080613_afghan_jail,कंधार के प्रांतीय सभा के अध्यक्ष और राष्ट्रपति...,"Wali Karzai, chairman of the Kandahar provinci..."



Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/FactIndic_Source_250_Staging.csv


In [148]:
import re
import pandas as pd

def count_words(text):
    return len(str(text).split())

def contains_hindi(text):
    return bool(re.search(r'[\u0900-\u097F]', str(text)))

source_250["hindi_word_count"] = (
    source_250["hindi_article"].apply(count_words)
)

source_250["english_word_count"] = (
    source_250["english_article"].apply(count_words)
)

source_250["english_has_hindi"] = (
    source_250["english_article"].apply(contains_hindi)
)

source_250["translation_ratio"] = (
    source_250["english_word_count"] /
    source_250["hindi_word_count"]
)

print("TRANSLATION SANITY CHECK")
print("Total articles:", len(source_250))
print("English translations containing Hindi:",
      source_250["english_has_hindi"].sum())
print("\nTranslation ratio statistics:")
display(source_250["translation_ratio"].describe())

print("\n5 LOWEST TRANSLATION RATIOS:")
display(
    source_250.nsmallest(5, "translation_ratio")[
        ["id", "hindi_word_count", "english_word_count",
         "translation_ratio"]
    ]
)

print("\n5 HIGHEST TRANSLATION RATIOS:")
display(
    source_250.nlargest(5, "translation_ratio")[
        ["id", "hindi_word_count", "english_word_count",
         "translation_ratio"]
    ]
)

TRANSLATION SANITY CHECK
Total articles: 250
English translations containing Hindi: 0

Translation ratio statistics:


,translation_ratio
count,250.000000
mean,0.844746
std,0.118449
min,0.099794
25%,0.808416
50%,0.858538
75%,0.908067
max,1.037313



5 LOWEST TRANSLATION RATIOS:


,id,hindi_word_count,english_word_count,translation_ratio
102,international-49451047,972,97,0.099794
49,international-47105506,813,93,0.114391
64,international-48243735,732,126,0.172131
93,150816_badminton_saina_final_dp,278,92,0.330935
210,090216_jobloss_column3_alk,466,170,0.364807



5 HIGHEST TRANSLATION RATIOS:


,id,hindi_word_count,english_word_count,translation_ratio
42,141013_wusatullah_amitabh_bachchan_rns,536,556,1.037313
200,140524_iran_hanging_aa,167,169,1.011976
205,150221_mulayam_modi_saifai_du,205,207,1.009756
208,india-41263102,818,821,1.003667
139,150713_bihar_women_strange_names_rns,696,696,1.000000


In [149]:
problem_ids = [
    "international-49451047",
    "international-47105506",
    "international-48243735"
]

for article_id in problem_ids:
    row = source_250[source_250["id"] == article_id].iloc[0]

    print("\n" + "=" * 80)
    print("ARTICLE:", article_id)

    print("\nHINDI ORIGINAL (first 1200 characters):")
    print(row["hindi_article"][:1200])

    print("\nENGLISH TRANSLATION (complete):")
    print(row["english_article"])

    print("\nHindi words:", row["hindi_word_count"])
    print("English words:", row["english_word_count"])
    print("Ratio:", round(row["translation_ratio"], 3))


ARTICLE: international-49451047

HINDI ORIGINAL (first 1200 characters):
यह सम्मान पाने वाले वो पहले भारतीय प्रधानमंत्री बने. 'ऑर्डर ऑफ़ ज़ायेद' यूएई का सर्वोच्च नागरिक सम्मान है जो बादशाहों, राष्ट्रपति और राष्ट्र प्रमुखों को दिया जाता है. इससे पहले साल 2007 में रूस के राष्ट्रपति व्लादिमीर पुतिन, 2010 में ब्रिटेन की महारानी एलिज़ाबेथ, 2016 में सऊदी के शाह सलमान बिन अब्दुल्ला अज़ीज़ अल सऊद और साल 2018 में चीन के राष्ट्रपति शी जिनपिंग को यह सम्मान दिया गया था. यूएई ने इस सम्मान की शुरुआत साल 1995 में की थी. पीएम मोदी को इस सम्मान के लिए क्यों चुना गया? अबू धाबी के क्राउन प्रिंस शेख़ मोहम्मद बिन ज़ायेद अल नाह्यान के मुताबिक, भारत और यूएई के बीच द्विपक्षीय संबंधों को बढ़ाने में पीएम मोदी ने महत्वपूर्ण योगदान दिया है. इस साल अप्रैल में शेख़ मोहम्मद बिन ज़ायेद अल नाह्यान ने एक ट्वीट के माध्यम से इस सम्मान की घोषणा की थी. उन्होंने लिखा था, "भारत के साथ हमारे ऐतिहासिक और व्यापक रणनीतिक संबंध हैं, जिसके पीछे मेरे परम मित्र नरेंद्र मोदी का बहुत योगदान है. उन्होंने इन रिश्तों को और मज़बूत किया ह

In [150]:
problem_ids = [
    "international-49451047",
    "international-47105506",
    "international-48243735"
]

source_250["translation_quality_status"] = "Pending audit"

source_250.loc[
    source_250["id"].isin(problem_ids),
    "translation_quality_status"
] = "Needs retranslation"

print(source_250["translation_quality_status"].value_counts())

# Save updated staging file
staging_path = os.path.join(
    base_path,
    "04_mixed_data",
    "FactIndic_Source_250_Staging.csv"
)

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nUpdated staging file saved.")

translation_quality_status
Pending audit          247
Needs retranslation      3
Name: count, dtype: int64

Updated staging file saved.


In [151]:
# Select the next 5 articles for manual audit
audit_df = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5).copy()

for _, row in audit_df.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])

    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])

    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

    print("\n" + "-" * 90)


ARTICLE ID: 080419_brchopra_festival

HINDI ORIGINAL:
शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी." बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है. गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में शामिल है. हालांकि बीआर चोपड़ा की ज़्यादातर फ़िल्में

In [152]:
reviewed_ids = [
    "080419_brchopra_festival",
    "080613_afghan_jail",
    "130912_guinnness_record_bulgarian_swim_sr",
    "050713_us_un",
    "120809_salman_pkp"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

# Save progress
source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("Audit progress:")
print(source_250["translation_quality_status"].value_counts())

# Select next 5
next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])

    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])

    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

Audit progress:
translation_quality_status
Pending audit                           242
Initial review - no obvious mismatch      5
Needs retranslation                       3
Name: count, dtype: int64

ARTICLE ID: 080711_lanka_attack

HINDI ORIGINAL:
एक सैनिक प्रवक्ता ब्रिगेडियर उदय नान्यकारा ने इस हमले के लिए तमिल विद्रोहियों को ज़िम्मेदार ठहराया है. ये घटना मोनारगाला ज़िले में हुई जब इस बस पर अंधाधुंध गोलियाँ चलाई गई. अभी तक एलटीटीई की ओर से इस घटना पर कोई टिप्पणी नहीं आई है. ये हमला ऐसे समय हुआ है जब देश के उत्तरी इलाक़े में सेना और तमिल विद्रोहियों के बीच संघर्ष में तेज़ी आई है. कोलंबो से बीबीसी संवाददाता ने बताया है कि बस एक सूनसान इलाक़े से गुज़र रही थी, जब उस पर गोलियाँ चलाई गईं. हमला मरने वालों में एक नौ साल का बच्चा भी है. गोलीबारी में कई यात्री घायल भी हुए हैं. सैनिक प्रवक्ता ब्रिगेडियर उदय नान्यकारा का कहना है कि बंदूकधारी शायद सड़क के किनारे छिपे हुए थे. उन्होंने इस गोलीबारी के लिए तमिल विद्रोहियों को ज़िम्मेदार ठहराया है. एक साल पहले श्रीलंका की सरकार ने दावा किया था कि उन

In [153]:
reviewed_ids = [
    "080711_lanka_attack",
    "social-43068605",
    "international-51410040",
    "india-55525807",
    "international-41372477"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

# Save audit progress
source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

# Load next 5 pending articles
next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])

    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])

    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Pending audit                           237
Initial review - no obvious mismatch     10
Needs retranslation                       3
Name: count, dtype: int64

ARTICLE ID: 130404_sam_pitroda_profile

HINDI ORIGINAL:
सैम पित्रोदा ने 1984 में भारत लौटने के बाद सी-डैक की स्थापना की थी. वर्तमान में पित्रोदा भारतीय प्रधानमंत्री के जन सूचना संरचना और नवप्रवर्तन सलाहकार हैं. प्रधानमंत्री के सलाहकार के रूप में उनकी भूमिका विभिन्न क्षेत्रों में नागरिकों को मिलनेवाली सुविधाओं को प्रभावी बनाना और नवप्रवर्तन के लिए रोडमैप तैयार करना है. पित्रोदा राष्ट्रीय नवप्रवर्तन परिषद के चेयरमैन भी हैं. साल 2005 से 2009 तक सैम पित्रोदा भारतीय ज्ञान आयोग के चेयरमैन थे. इस उच्च स्तरीय सलाहकार परिषद का गठन देश में ज्ञान आधारित संस्थाओं और उनके आधारभूत ढांचे को बेहतर बनाने के लिए परामर्श देने के लिए किया गया था. भारतीय ज्ञान आयोग के चेयरमैन की हैसियत से काम करते हुए पित्रोदा ने 27 क्षेत्रों में सुधार के लिए क़रीब 300 सुझाव दिए थे. शिक्षा भौतिकी में स्नातकोत्तर करने के बाद 

In [154]:
reviewed_ids = [
    "130404_sam_pitroda_profile",
    "140323_breast_cancer_screening_tk",
    "150520_emergency_service_ss",
    "081126_aamne_pachauri_alk",
    "081118_cg_campaign_vv"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Pending audit                           232
Initial review - no obvious mismatch     15
Needs retranslation                       3
Name: count, dtype: int64

ARTICLE ID: 080826_blast_films

HINDI ORIGINAL:
इन फ़िल्मों में से कुछ रिलीज़ हो रही हैं और कुछ का निर्माण कार्य चल रहा है. पिछले हफ़्ते रिलीज़ हुई निशिकांत कामथ की फिल्म 'मुंबई मेरी जान' की कहानी का आधार 11 जुलाई 2006 को मुबंई में हुए ट्रेन बम धमाके थे. इन बम धमाकों में लगभग 200 लोगों की मौत हुई थी. फिल्म के निर्देशक निशिकांत कामथ कहते हैं, "ये फ़िल्म किसी बड़ी दुर्घटना से उबरने की एक कोशिश को दर्शाती है. मुंबई मेरी जान की कहानी सिर्फ़ मुंबई या इस घटना के बारे में नहीं बल्कि किस तरह से लोग इस समस्या से जूझ रहे हैं इस बारे में है." मुंबई मेरी जान एक भावनाप्रधान फिल्म है जबकि पाँच सितंबर को रिलीज़ होने जा रही फ़िल्म 'वेडनस्डे' एक थ्रिलर है. इस फिल्म से निर्देशन की पारी शुरु करने वाले निर्देशक नीरज पांडे कहते हैं, "वेडनस्डे की कहानी उन चंद घंटों की है जिसमें आतंकवादी मुंबई शहर को अपनी दहशत

In [155]:
reviewed_ids = [
    "080826_blast_films",
    "141007_bhutan_corruption_india_vr",
    "140702_wc2014_goldenboot_contenders_ar",
    "130207_kashmir_rockband_imroz_sa",
    "051105_sahubuddin_arrest"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Pending audit                           227
Initial review - no obvious mismatch     20
Needs retranslation                       3
Name: count, dtype: int64

ARTICLE ID: india-48662333

HINDI ORIGINAL:
सेमाभाई की बेटी हरमीबेन ये कहना है सेमाभाई का, इनके मक्के की फसल पूरी तरह सूख गई है. सेमाभाई के पास दो बैल हैं लेकिन उनका इस्तेमाल खेतों को जोतने में नहीं हो रहा है, बल्कि कुंआ खोदने में किया जा रहा है. इस काम में सेमाभाई की पत्नी और उनकी बेटी भी मदद कर रहे हैं. पिछले साल उनकी फसल सूख गई थी और इस साल उनका कुंआ भी सूख गया था. उत्तरी गुजरात के पालनपुर जिले के अमीरगढ़ ब्लॉक के अधिकांश किसानों का यही हाल है. इन सबके खेत सूखे पड़े हुए हैं और फसल की सिंचाई के लिए पानी नहीं है. इंसान और जानवरों के लिए भी पानी नहीं सेमाभाई एक छोटे से गांव उपाला खापा में रहते हैं. उनके पास जमीन तो है लेकिन खेती करने के लिए पानी नहीं है. वे दिहाड़ी मजदूर की तरह काम करने को मजबूर हैं. वे बताते हैं, "मेरे कुंए में पानी नहीं है, इसलिए अब मैं एक कहीं गहरा कुंआ खोद रहा हूं. प

In [156]:
reviewed_ids = [
    "india-48662333",
    "070206_guj_parza",
    "061206_gennext_indopak_students",
    "141016_ranveer_removes_pant_md_part1_pkp",
    "060829_dhyanchand_hockey"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Pending audit                           222
Initial review - no obvious mismatch     25
Needs retranslation                       3
Name: count, dtype: int64

ARTICLE ID: 061004_askus_geyser

HINDI ORIGINAL:
गीज़र एक प्रकार का गर्म पानी का सोता होता है जो समय-समय पर फूटता है और इसमें से गर्म पानी और भाप निकलती है. गीज़र के लिए अनुकूल जल भौमिकी की आवश्यकता होती है जो पृथ्वी के कुछ ही स्थानों पर मौजूद है. ये गीज़र कैसे बनते हैं... पृथ्वी की सतह का पानी ज़मीन के अंदर जाकर जमा होता है और ऐसी चट्टान से टकराता है जो मैग्मा से तप रही हो. इससे पानी गर्म होने लगता है और फिर ऊपर उठता है और किसी चट्टान की दरार से होकर बाहर फूट पड़ता है. दुनिया में कोई एक हज़ार गीज़र हैं जिनमें से आधे अमरीका के यैलोस्टोन नेशनल पार्क में हैं. गर्म पानी का इस्तेमाल करने के लिए बिजली की जिस मशीन (गीज़र) का इस्तेमाल होता है उसका नाम भी शायद इसी पर पड़ा होगा. ग्राम पहलाम, सहरसा बिहार से जितेन्द्र कुमार पासवान पूछते हैं कि ओलम्पिक खेल सर्वप्रथम किस देश में खेले गए थे. प्राचीन ओ

In [157]:
reviewed_ids = [
    "061004_askus_geyser",
    "140729_moeen_wristband_icc-du",
    "india-46106302",
    "060120_speaker_cashcamera",
    "international-41646598"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Pending audit                           217
Initial review - no obvious mismatch     30
Needs retranslation                       3
Name: count, dtype: int64

ARTICLE ID: 130614_complaint_for_looks_rd

HINDI ORIGINAL:
शिकायतकर्ता महिला से कार पार्किंग में मिला था वेस्ट मिडलैंड्स पुलिस को एक व्यक्ति ने फ़ोन करके “ वेश्या के ख़िलाफ़ वस्तु विक्रय अधिनियम के उल्लंघन का मामला दर्ज करवाना चाहा.” पुलिस ने उस व्यक्ति को एक पत्र भेजकर चेतावनी दी है कि वह आइंदा पुलिस का समय बर्बाद न करे. सॉलिहल पुलिस स्टेशन में स्थानीय समय के अनुसार मंगलवार शाम साढ़े सात बजे यह फ़ोन आया था. पुलिस के अनुसार शिकायतकर्ता का दावा था कि वह उस महिला से एक होटल की कार पार्किंग में मिला था. अविश्वसनीय एक पुलिस प्रवक्ता ने बताया, “फ़ोन करने वाले व्यक्ति का कहना था कि महिला ने खुद को असलियत के मुक़ाबले काफ़ी ख़ूबसूरत बताया और वह व्यक्ति उसके ख़िलाफ़ वस्तु विक्रय अधिनियम के उल्लंघन का मामला दर्ज करवाना चाहता था.” “उसने जब यह बात उस महिला को कही तो कथित रूप से उसने आदमी की कार की च

In [158]:
reviewed_ids = [
    "130614_complaint_for_looks_rd",
    "150930_vert_auto_audi_etron_pk",
    "141207_pakistan_taliban_mehsud_repatriated_rd",
    "121207_hangover_smoking_aa",
    "india-55465758"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Pending audit                           212
Initial review - no obvious mismatch     35
Needs retranslation                       3
Name: count, dtype: int64

ARTICLE ID: 160129_women_cricket_team_india_pm

HINDI ORIGINAL:
भारत की ऑस्ट्रेलिया के ख़िलाफ़ ये पहली द्विपक्षीय सिरीज़ जीत है. उन्होंने तीन बार की विश्व चैंपियन ऑस्ट्रेलिया को दूसरे मैच में दस विकेट से हराकर तीन मैचों की सिरीज़ में 2-0 की बढ़त ले ली है. बारिश की वजह से मैच छोटा कर दस ओवरों तक सीमित कर दिया गया था. झूलन गोस्वामी, भरतीय महिला क्रिकेट खिलाड़ी भारत को 66 रन बनाने का लक्ष्य मिला था. भारतीय टीम ने 9.1 ओवरों में ही 69 रन बनाए. समाप्त कप्तान मिताली राज ने 37 रन बनाए और वे अंत तक आउट नहीं हुईं. भारतीय लड़कियां मैच के शुरू से अंत तक हावी रहीं. उन्होंने पाँच ओवर में ही ऑस्ट्रेलियाई टीम को तीन विकेट 33 रन के स्कोर पर चटका दिए थे. मेग लैनिंग, कप्तान, ऑस्ट्रेलियाई महिला क्रिकेट टीम ऑस्ट्रेलियाई कप्तान मेग लैनिंग और जेस जोनासन की 70 रनों की चौथे विकेट की साझेदारी ने टीम को सहारा दिया

In [159]:
reviewed_ids = [
    "160129_women_cricket_team_india_pm",
    "041015_envoy_critics",
    "080826_bihar_flood",
    "120904_yuvraj_shabnam_ml",
    "050127_love_alternative"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Pending audit                           207
Initial review - no obvious mismatch     40
Needs retranslation                       3
Name: count, dtype: int64

ARTICLE ID: 140218_cricket_test_team_dhoni_vr

HINDI ORIGINAL:
भारत दो टेस्ट मैच की सिरीज़ 1-0 से हारा जबकि इससे पहले पांच एकदिवसीय मैचों की सिरीज़ में उसे 4-0 से हार का सामना करना पड़ा था. कुल मिलाकर भारत न्यूज़ीलैंड दौरे पर एक भी मैच जीतने में नाकाम रहा. (क्या इतिहास में शामिल होंगे मैकुलम?) कुछ ऐसा ही प्रदर्शन भारत ने इससे पहले दक्षिण अफ्रीका में किया था जहां वह दो टेस्ट मैचों की सिरीज़ 0-1 से और तीन एकदिवसीय मैचों की सिरीज़ 0-2 से हारा था. विदेशी ज़मीन पर मिलती लगातार हार ने भारत के कप्तान महेंद्र सिंह धोनी और उनके धुरंधरों पर कई सवाल खड़े कर दिए हैं. क्या विदेशी पिचों पर भारत का यही हाल होता रहेगा, और क्या भारत के हाथ से जीत इसी तरह से फिसलती रहेगी और क्या धोनी के चहेते खिलाड़ी यूँ ही टीम में बने रहेंगे और सबसे बड़ा सवाल कि क्या अब आगामी एशिया कप और ट्वेंटी-ट्वेंटी विश्व कप क्रिकेट 

In [160]:
reviewed_ids = [
    "140218_cricket_test_team_dhoni_vr",
    "science-49370402",
    "151218_anti_pee_london_ac",
    "090106_india_export"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.loc[
    source_250["id"] == "141013_wusatullah_amitabh_bachchan_rns",
    "translation_quality_status"
] = "Needs retranslation"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Pending audit                           202
Initial review - no obvious mismatch     44
Needs retranslation                       4
Name: count, dtype: int64

ARTICLE ID: 131201_karan_with_salman_sk

HINDI ORIGINAL:
मगर अब सब कुछ बदल गया है. जाने माने फ़िल्मकार करण जौहर ने अपने टीवी चैट शो 'कॉफ़ी विद करण' के तीसरे संस्करण के शुरुआती एपिसोड में सलमान ख़ान को बुलाने का फ़ैसला किया है. सलमान खान इस शो में पहली बार आ रहे हैं. करण जौहर ने अपने ऑफिशियल ट्विटर हैंडल (@karanjohar) पर लिखा है, "ٍ'कॉफ़ी विद करण' @StarWorldIndia..@Beingsalmankhan पर रविवार रात 9 बजे शुरू हो रहा है. हमारे सबसे पहले अतिथि होंगे सलमान खान. साथ ही होगी उनके पिता सलीमसाब के साथ खास बातचीत." पीटीआई के मुताबिक करण जौहर का शो 'कॉफ़ी विद करण' इस बार बहुत मज़ेदार होगा. इसमें बॉलीवुड की मशहूर हस्तियों के साथ बातचीत होगी. शो में पहली बार सलमान करण ने कहा, " मैं इस बार और ज़्यादा पागल करने वाला हूं. नया सीजन बेहद रोमांचक होने वाला है. नए मेहमान, शानदार सेट, और बढिया बातें." 'कॉफ़ी वि

In [161]:
reviewed_ids = [
    "131201_karan_with_salman_sk",
    "international-51281660",
    "international-44515542",
    "151122_indo_pak_cricket_meet_ml",
    "120816_assange_equador_pa"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Pending audit                           197
Initial review - no obvious mismatch     49
Needs retranslation                       4
Name: count, dtype: int64

ARTICLE ID: 120518_ipl_glamour_magazine_sm

HINDI ORIGINAL:
आईपीएल पैसे और ग्लैमर की वजह से भी खासा चर्चित है दूसरी ओर मुंबई क्रिकेट एसोसिएशन के अधिकारी कह रहे हैं कि शाहरुख खान ने शराब पी रखी थी और उन्होंने नशे में सुरक्षा गार्डों को गालियां दीं. बहरहाल क्या सही था, क्या गलत ये तो जांच के बाद पता चलेगा. लेकिन एक बात तो सही है, शाहरुख खान ने भी स्वीकार किया है कि उन्होंने गालियां दीं. तो इसके लिए उन्हें माफी माँग लेनी चाहिए. एक सम्मानित नागरिक होने के नाते. लेकिन प्रेस कांफ्रेंस में उनका ये कहना कि उन्होंने जो भी किया, ठीक किया. ये ठीक नहीं है. दूसरे एक बात और, इस आईपीएल में अक्सर खिलाड़ी पर्दे के पीछे चले जाते हैं. कुछ समझ में नहीं आता कि टीम के मालिक, कोच, खिलाड़ी इन सबकी क्या सीमा है. इंग्लिश प्रीमियर लीग यहां ये बताना जरूरी है कि फुटबॉल में इंग्लिश प्रीमयर लीग में किसी को पता नहीं हो

In [162]:
reviewed_ids = [
    "120518_ipl_glamour_magazine_sm",
    "131119_chinese_supercomputer_worldbest_rns",
    "india-53336116",
    "141229_italy_ferry_rescue_pkp",
    "india-51453120"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Pending audit                           192
Initial review - no obvious mismatch     54
Needs retranslation                       4
Name: count, dtype: int64

ARTICLE ID: 130529_karunanidhi_tamilnadu_marriage_ns

HINDI ORIGINAL:
करूणानिधि के अनुसार उनके स्वास्थ्य का राज़ नियमित योग करना है. समाचार एजेंसी पीटीआई के अनुसार सम्मानजनक तरीके से शादी करने की वकालत करने वाले एम करूणानिधि ने अपने जीवन के उन पलों को याद किया जो अब इतिहास बन चुके हैं. उन्होंने कहा, “बात 1944 की है जब मेरी प्रेमिका के घरवालों ने परंपरागत तौर तरीकों से हमारी शादी करने की बात कही. लेकिन मेरे सिद्धांत इसकी इजाज़त नहीं देते थे इसलिए वैसा नहीं हो सका." चेन्नई में एक विवाह समारोह में शामिल होते हुए 89 वर्षीय एम करूणानिधि ने अपने विचार ज़ाहिर किए. उन्होंने कहा, "मेरी प्रेमिका के घरवालों ने मेरी उस मंशा को खारिज कर दिया जिसमे मैं चाहता था कि हमारी शादी सादगी से हो. मैं मंत्रोच्चारण और मंगलसूत्र वगैरह के खिलाफ था. लेकिन उन्होंने अपनी पुत्री की शादी इस तरह से कराए जाने का विरोध कि

In [163]:
reviewed_ids = [
    "130529_karunanidhi_tamilnadu_marriage_ns",
    "081217_ganguly_yuvraj_ac",
    "150901_nepal_protest_dead_pp",
    "040803_jyotish_newtrend",
    "150811_sani_mirza_autobiography_farha_adp_ssm"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Pending audit                           187
Initial review - no obvious mismatch     59
Needs retranslation                       4
Name: count, dtype: int64

ARTICLE ID: 131227_diamond_antarctica_tk

HINDI ORIGINAL:
हालांकि अंटार्कटिक क्षेत्र में व्यावसायिक उद्देश्य के लिए किसी भी क़िस्म के खनिज का दोहन प्रतिबंधित है. यह शोध नेचर कम्यूनिकेशन नाम के जर्नल में छपा है. हीरा पृथ्वी की सतह से क़रीब 150 किलोमीटर की गहराई में अत्यधिक गर्मी और दबाव से शुद्ध कार्बन से बनता है. किंबरलाइट किंबरलाइट में हीरे पाए जाते हैं ज्वालामुखी विस्फोट पृथ्वी की सतह पर हीरों को ले आते हैं जो सामान्यतः एक दूसरी नीली चट्टान किंबरलाइट में जमा हुआ होता है. किंबरलाइट की मौजूदगी ने अफ्रीका, साइबेरिया और ऑस्ट्रेलिया सहित दुनिया के कई हिस्सों में हीरे पाए जाने के संकेत दिए हैं. शोधकर्ताओं को अंटार्कटिका में पहली बार किंबरलाइट की उपस्थिति का प्रमाण मिला है. शोधकर्ताओं के दल को उत्तरी प्रिंस चार्ल्स पहाड़ों में माउंट मेरेडिथ की ढलानों पर इसके तीन नमूने मिले. ब्रिटिश अंटार्कटिक

In [164]:
reviewed_ids = [
    "131227_diamond_antarctica_tk",
    "060603_taliban_karzai",
    "050625_building_collapse",
    "india-51702290",
    "080721_parliament_debate"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Pending audit                           182
Initial review - no obvious mismatch     64
Needs retranslation                       4
Name: count, dtype: int64

ARTICLE ID: 041003_pwg_security

HINDI ORIGINAL:
अधिकारियों का कहना है कि सरकार ने पुलिस को निर्देश दिए हैं कि तीन प्रमुख माओवादी नेताओं को सुरक्षित रास्ता मुहैया कराया जाए. वार्ताकारों की सुरक्षा के बारे में पुलिस अधिकारी पीडब्ल्यूजी के मध्यस्थ वारा वारा राव, गदर और कल्याण राव से बात करेंगे. पीडब्ल्यूजी के तीन प्रमुख नेता रामकृष्णा, सुधाकर और गणेश सरकार के साथ अक्तूबर के दूसरे हफ्ते में बातचीत के लिए आ रहे हैं. सुरक्षा अहम विद्रोही संगठन पीडब्ल्यूजी के लिए सरकार से वार्ता के दौरान अपने नेताओं की सुरक्षा सबसे अहम मुद्दा है . इसी कारण संगठन ने दो अक्तूबर को वार्ता की सरकार की पेशकश को ठुकरा दिया था. संगठन ने पिछले दिनों हैदराबाद में एक विशाल जनसभा का आयोजन किया था. पुलिस ने इस दौरान संगठन के कार्यकर्ताओं को बिल्कुल परेशान नहीं किया था जिसके बाद संगठन में काफी आत्मविश्वास आया है. पीडब्ल्यू

In [165]:
reviewed_ids = [
    "041003_pwg_security",
    "080416_up_results",
    "150226_gujrat_swine_flu_ahmedabad_sr",
    "050901_gohana_firsthand",
    "070530_iftikhar_detained"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Pending audit                           177
Initial review - no obvious mismatch     69
Needs retranslation                       4
Name: count, dtype: int64

ARTICLE ID: 150607_narendra_modi_bangladesh_visit_analysis_rns

HINDI ORIGINAL:
उन्होंने लैंड बाउंड्री एग्रीमेंट को दिलों को जोड़ने वाला समझौता बताया तो बांग्लादेश के विकास पर ख़ुशी भी जताई. नरेंद्र मोदी की इस यात्रा का क्या हासिल रहा इसपर बीबीसी संवाददाता समीरात्मज मिश्र ने बात की पूर्व विदेश सचिव मुचकुंद दुबे से. पढ़ें बातचीत के चुनिंदा अंश इस यात्रा को आप कैसे देखते हैं? यह निस्संदेह एक महत्वपूर्ण यात्रा रही है. इसमें कुछ ऐसे क़दम उठाए गए हैं जिसके दूरगामी अच्छे परिणाम रहेंगे. समाप्त हमने बांग्लादेश को जो दो अरब डॉलर का कर्ज देने की घोषणा की है, उसकी मदद से वहाँ के ट्रांसपोर्ट और इंफ्रास्ट्रक्चर को सुधारा जाएगा. इससे हमारी पहुँच दक्षिण-पूर्वी एशिया और पूर्वी एशिया में बेहतर हो जाएगी. जहाँ तक लैंड बांउड्री का सवाल है, ये बहुत दिनों से स्थगित मामला था. बांग्लादेश ने इसे 1974 में ही रेक्

In [166]:
reviewed_ids = [
    "150607_narendra_modi_bangladesh_visit_analysis_rns",
    "international-52239381",
    "040415_sharon_arafat",
    "071013_advani_pm",
    "111113_berlusconi_resigned_vv"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Pending audit                           172
Initial review - no obvious mismatch     74
Needs retranslation                       4
Name: count, dtype: int64

ARTICLE ID: 150612_heatwave_rentachintala_sra

HINDI ORIGINAL:
कुछ युवक हाथों में पत्थर लिए दौड़ते हुए नज़र आ रहे थे. पहनावे से लगा ये सब यहीं के रहने वाले हैं. मैंने गाड़ी रुकवाई और जब उनकी तरफ बढ़ा तो पता चला कि ये जमात दरअसल एक काले ज़हरीले नाग का पीछा रही है. लड़कों ने बताया कि इस इलाक़े में यह मंज़र आम है क्योंकि इस क़स्बे में गर्मी की वजह से सांप-बिच्छुओं, कीड़े-मकौड़ों की भी बड़ी समस्या है. दूर-दूर तक फैली पत्थरों की चट्टानों के बीच बसे इस क़स्बे का नाम है 'रेंटाचिन्तला' जो आंध्र प्रदेश के गुंटूर ज़िले में है. समाप्त पढ़ें विस्तार से इस जगह को पूरे प्रदेश में सबसे ज़्यादा तापमान के लिए जाना जाता है. हर वर्ष यहाँ औसत तापमान 48 से 50 डिग्री सेल्सियस तक पहुंच जाता है. इस साल भी मई में यहाँ का तापमान पचास तक जा पहुंचा. 2012 में यहाँ तापमान रिकार्ड 52 डिग्री तक पहुँच गया था. हालांकि इस

In [167]:
reviewed_ids = [
    "150612_heatwave_rentachintala_sra",
    "080211_raj_police",
    "050812_baflights_cancel",
    "india-47148086",
    "international-42264998"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Pending audit                           167
Initial review - no obvious mismatch     79
Needs retranslation                       4
Name: count, dtype: int64

ARTICLE ID: 160606_ajit_jogi_new_party_rv

HINDI ORIGINAL:
जोगी ने कहा है कि पार्टी के नाम और चुनाव चिन्ह पर रायशुमारी के बाद निर्णय लिया जाएगा. बिलासपुर के कोटमी में सोमवार को एक सभा में जोगी ने कांग्रेस और भाजपा की आलोचना करते हुए कहा कि छत्तीसगढ़ की अस्मिता के लिए नई पार्टी की जरूरत थी. जोगी ने कहा, "अब मैं आजाद हो गया हूं. छत्तीसगढ़ के फ़ैसले अब दिल्ली में नहीं लिए जाएंगे. छत्तीसगढ़ को रमन सिंह से मुक्त कराया जाएगा." अजित जोगी ने अपनी सरकार बनने के बाद छत्तीसगढ़ को देश का पहला टैक्स फ्री राज्य बनाने की भी बात कही. समाप्त अजित जोगी की पत्नी रेणु जोगी कोटा इलाके की कांग्रेस विधायक हैं जबकि उनके बेटे अमित जोगी मरवाही के विधायक हैं. अजीत जोगी की पत्नी और बेटे के अलावा कांग्रेस के दो अन्य विधायक भी इस कार्यक्रम में उपस्थित थे. कांग्रेस के कुछ पूर्व मंत्री और पूर्व विधायक भी जोगी के कार्यक

In [168]:
reviewed_ids = [
    "160606_ajit_jogi_new_party_rv",
    "061015_katsav_police",
    "060117_indopak_cbm",
    "080324_chowdhry_pakistan"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.loc[
    source_250["id"] == "130606_china_gender_ss",
    "translation_quality_status"
] = "Needs retranslation"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Pending audit                           162
Initial review - no obvious mismatch     83
Needs retranslation                       5
Name: count, dtype: int64

ARTICLE ID: 121114_international_us_plus_obama_ia

HINDI ORIGINAL:
प्रेम संबंधों के कारण सीआईए प्रमुख को इस्तीफ़ा देना पड़ा था. अपनी चुनावी जीत के बाद बुधवार को व्हाईट हाउस में पहली प्रेस वार्ता को संबोधित करते हुए राष्ट्रपति ओबामा ने कहा, "मेरे पास अभी कोई ऐसा सबूत नहीं है, जो कुछ मैंने देखा है उसके आधार पर मैं कह सकता हूं कि इस मामले में कोई खुफ़िया राज़ नहीं खोला गया है और राष्ट्रीय सुरक्षा को कोई नुक़सान नहीं हुआ है." इस दौरान जब उनसे पूछा गया कि इस मामले में उनको देर से ख़बर दिए जाने के लिए क्या वह एफ़बीआई के ख़िलाफ़ कोई जांच शुरू करेंगे तो ओबामा ने कहा कि वह जांच में कोई बाधा नहीं डालना चाहते इसलिए अभी इस मामले में कोई क़दम नहीं उठा रहे हैं. उन्होंने कहा,"मैंने अभी कोई फ़ैसला नहीं किया है. क्यूंकि अभी भी मामले में जांच जारी है और सारी जानकारी हमारे पास नहीं है. एफ़बीआई का काम करने 

In [169]:
reviewed_ids = [
    "121114_international_us_plus_obama_ia",
    "051030_delhi_blasts",
    "040304_robot_translator",
    "041026_bestbakery_questions",
    "130312_traffic_noise_songbirds_vr"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Pending audit                           157
Initial review - no obvious mismatch     88
Needs retranslation                       5
Name: count, dtype: int64

ARTICLE ID: india-48250857

HINDI ORIGINAL:
इस सिलसिले में ताज़ा कड़ी अमरीका की टाइम मैगज़ीन है जिसने लिखा है, ''क्या दुनिया का सबसे बड़ा लोकतंत्र मोदी सरकार को आने वाले और पांच साल बर्दाश्त कर सकता है?" अमरीकी पत्रिका ने ये सवाल अपने उस अंक के कवर पेज के साथ ट्वीट किया है जो भारतीय बाज़ार में 20 मई 2019 को जारी किया जाएगा. टाइम मैगज़ीन की वेबसाइट पर प्रकाशित स्टोरी के कवर पेज पर मोदी को 'India's Divider In Chief' बताया गया है, जिस पर काफी विवाद हुआ है. ध्यान देने वाली बात ये है कि चार साल पहले 2015 में मई के अंक में भी TIME मैगज़ीन ने भारतीय प्रधानमंत्री मोदी पर कवर स्टोरी की थी और तब उसका शीर्षक था- "Why Modi Matters". इसी तरह फोर्ब्स पत्रिका में 16 मार्च 2019 को छपे एक लेख में लिखा है कि ''मोदी ने देश-विदेश में भारत को ऊपर उठाया है, लेकिन शासन करने की अपनी शैली की वजह से उन्हें अपनी क

In [170]:
reviewed_ids = [
    "india-48250857",
    "140825_bihar_nitish_laloo_siddharth_analysis_an",
    "160512_rocky_yadav_profile_sh",
    "050606_kabir_bedi"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.loc[
    source_250["id"] == "150816_badminton_saina_final_dp",
    "translation_quality_status"
] = "Needs retranslation"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Pending audit                           152
Initial review - no obvious mismatch     92
Needs retranslation                       6
Name: count, dtype: int64

ARTICLE ID: 050420_pope_muslims

HINDI ORIGINAL:
शंका जताई गई है कि नए पोप शायद ऐसा करने में उतना विश्वास नहीं रखते. तुर्की में तो उनके चुनाव पर हैरत ज़ाहिर की गई है क्योंकि उन्होंने यूरोपीय संघ में तुर्की की सदस्यता का ये कहकर विरोध किया था कि बेहतर हो यदि तुर्की अरब देशों की सदस्यता हासिल करे. पोप बेनेडिक्ट 16वें के सामने जो चुनौतियां हैं उनमें प्रमुख है वैटिकन का इस्लाम के प्रति रवैया. कुछ कैथोलिक ऐसे हैं जो पोप जॉन पॉल द्वितीय की नीति को अपनाते हुए दूसरे संप्रदायों के साथ बातचीत को आगे बढ़ाना चाहते हैं. लेकिन कुछ दूसरे हैं जो इस सोच से सहमत नहीं हैं. नए पोप जब कार्डिनल के पद पर थे तब उनकी भूमिका थी प्रमुख धर्मशास्त्री की और उन्हें एक तरह वैटिकन की नीतियों के रखवाले के रूप में देखा जाता था. उनका मानना था कि कैथोलिक इस सोच से उपर उठें कि सभी धर्म समान हैं, एक जैसे हैं. उनकी सोच थी कि द

In [171]:
reviewed_ids = [
    "050420_pope_muslims",
    "050108_nicobar_force",
    "040723_musharraf_natwar",
    "140206_egypt_al-sisi_presidential_election_sr",
    "040423_un_poverty_report"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Pending audit                           147
Initial review - no obvious mismatch     97
Needs retranslation                       6
Name: count, dtype: int64

ARTICLE ID: 140410_action_against_manvendra_singh_rns

HINDI ORIGINAL:
जसवंत सिंह को पार्टी के उम्मीदवार के ख़िलाफ़ चुनाव लड़ने के कारण पार्टी से निकाला जा चुका है. भाजपा के महासचिव जेपी नड्डा ने गुरुवार को राजस्थान भाजपा के प्रमुख अशोक परनामी को पत्र लिखकर यह जानकारी दी. मानवेंद्र सिंह को बाड़मेर से भाजपा के प्रत्याशी सोनाराम चौधरी के ख़िलाफ़ प्रचार करने के कारण पार्टी से निकाला गया है. मानवेंद्र सिंह राजस्थान से विधायक हैं. उन पर आरोप था कि वह अपने पिता के पक्ष में प्रचार कर रहे हैं. उनके पिता जसवंत सिंह बाड़मेर से स्वतंत्र उम्मीदवार के रूप में चुनाव लड़ रहे हैं. भाजपा क्या कांग्रेस की परछाईं बन गई है: जसवंत पीटीआई के अनुसार मानवेंद्र सिंह भाजपा उम्मीदवार के समर्थन में होने वाली पार्टी बैठकों में शामिल नहीं हो रहे हैं. भाजपा ने जसवंत सिंह को पार्टी से उस वक़्त निकाल दिया जब उन्होंने भा

In [172]:
reviewed_ids = [
    "140410_action_against_manvendra_singh_rns",
    "051121_aids_india",
    "141114_sterilisation_campaign_arrests_aj",
    "070902_khanduri_win",
    "150725_one_killed_kashmir_grenade_attack_pm"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Pending audit                           142
Initial review - no obvious mismatch    102
Needs retranslation                       6
Name: count, dtype: int64

ARTICLE ID: 050112_nepalmao_amnesty

HINDI ORIGINAL:
संगठन ने माओवादी नेताओं से अनुरोध किया है कि वे इन मामलों की जाँच करें और दोषी व्यक्तियों के ख़िलाफ़ कार्रवाई करें. नेपाली माओवादियों के नेता प्रचंड के नाम लिखे एक 'खुले पत्र' में एमनेस्टी ने कहा है कि मानवाधिकार हनन के विशेष मामले उसके सामने आए हैं. संगठन के एशिया-प्रशांत क्षेत्र के निदेशक इंग्रिड मसाज का कहना है कि प्रचंड सभी आम नागरिकों की सुरक्षा सुनिश्चित करें और अगवाह किए गए लोगों को बिना शर्त छोड़ दें. मसाज का कहना था कि कई लोगों को इसलिए निशाना बनाया गया क्योंकि उन्होंने माओवादियों की आलोचना की थी. उनका कहना था कि ऐसा करना स्पष्ट अंतरराष्ट्रीय मानवीय क़ानूनों का उल्लघन है. नेपाल में विद्रोहियों के बहुत सारे स्कूली छात्रों को अगवाह करने और उन्हें बलपूर्वक माओवाद के बारे में पढ़ाई करवाकर कुछ दिन बाद छोड़ देने के मामले सामने आए है

In [173]:
reviewed_ids = [
    "050112_nepalmao_amnesty",
    "080226_britain_iraq",
    "080301_benazir_suspect",
    "international-47823342",
    "060715_bushputin_g8"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Pending audit                           137
Initial review - no obvious mismatch    107
Needs retranslation                       6
Name: count, dtype: int64

ARTICLE ID: 130528_speed_breaker_cars_impact_sy

HINDI ORIGINAL:
स्पीड ब्रेकर से रगड़ खा कर इस गाड़ी के रेडियेटर में दिक्कत आई, जिसे ठीक होने में कम से कम दो दिन तो लगेंगे. रौशनी पिछले 15 सालों से दिल्ली में अपना ऑटोमोबाइल सर्विस सेंटर चला रही हैं. गाड़ी की मरम्मत की देख-रेख करते हुए वे कहती हैं, “देखा जाए तो स्पीड ब्रेकर मेरे बिज़नेस के लिए अच्छे हैं. हमारे पास हर दिन कम से कम 2-3 गाड़ियां ऐसी आती हैं जो स्पीड ब्रेकर की वजह से डैमेज हो जाती हैं. ज़्यादातर गाड़ियां महंगी वाली होती हैं, तो ज़ाहिर है कि उनके रिपेयर के लिए कंपनी मोटा पैसा मांगती है. हमारे यहां कंपनी से आधे कम रेट में मरम्मत हो जाती है.” पढ़िए कैसा होना चाहिए एक आदर्श स्पीड ब्रेकर वे बताती हैं कि आजकल लोग पैसे बचाने के लिए अपनी बड़ी गाड़ियों में सीएनजी सिलेंडर भी लगवा लेते हैं, जिसकी वजह से वो और नीचे बैठ जाती है. उनके पास इ

In [174]:
reviewed_ids = [
    "130528_speed_breaker_cars_impact_sy",
    "151130_offline_google_map_india_hk",
    "050107_pravasi_pm",
    "050506_india_coach",
    "040511_shahbaz_profile"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Pending audit                           132
Initial review - no obvious mismatch    112
Needs retranslation                       6
Name: count, dtype: int64

ARTICLE ID: 071227_musharraf_statement

HINDI ORIGINAL:
मुशर्रफ़ ने कहा है कि आतंकवाद पाकिस्तान की तरक्की की राह में सबसे बड़ी रुकावट है. टेलीविज़न पर दिए गए एक बयान में उन्होंने कहा कि वो तब तक चैन से नहीं बैठेंगे जब तक देश से दहशतगर्दों का ख़ात्मा नहीं कर दिया जाता. उन्होंने आतंकवाद के ख़ात्मे के लिए पाकिस्तानी जनता से समर्थन की अपील की और कहा ' पाकिस्तान की तरक्की में सबसे बड़ी रुकावट दहशतगर्दी है और वो इसे जड़ से उखाड़ने की कोशिश कर रहे हैं. ' इसके साथ ही उन्होंने पाकिस्तानी जनता से शांति और अमन चैन बनाए रखने की अपील भी की और कहा कि लोग सब्र करें ताकि कठिन समय में अफरा तफरी न मचे. पाकिस्तान में तीन दिन के राष्ट्रीय शोक की घोषणा कर दी गई है.

ENGLISH TRANSLATION:
Musharraf has said that terrorism is the biggest obstacle to Pakistan's progress.

In a televised statement, he said that h

In [175]:
reviewed_ids = [
    "071227_musharraf_statement",
    "070516_assam_violence",
    "040725_bangladesh_floods",
    "081205_sc_bcci_pp",
    "science-39280869"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Pending audit                           127
Initial review - no obvious mismatch    117
Needs retranslation                       6
Name: count, dtype: int64

ARTICLE ID: 150222_parliament_budget_session_pk

HINDI ORIGINAL:
संसद से सड़क तक की राजनीति, देश के आर्थिक स्वास्थ्य और प्रशासनिक व्यवस्था को लेकर अनेक गंभीर सवालों के जवाब सरकार को देने हैं. पिछले साल जुलाई में पेश किए गए रेल और आम बजट पिछली सरकार के बजटों की निरंतरता से जुड़े थे. देखना होगा कि वित्त मंत्री का जोर राजकोषीय घाटे को कम करने पर है या वो सरकारी खर्च बढ़ाकर सामाजिक विकास को बढ़ावा देंगे. पढ़ें, रिपोर्ट विस्तार से यह मोदी सरकार के हनीमून की समाप्ति का सत्र होगा. समाप्त सत्र के ठीक पहले सरकारी दफ्तरों से महत्वपूर्ण दस्तावेजों की चोरी के मामले ने देश की प्रशासनिक-आर्थिक व्यवस्था को लेकर गम्भीर सवाल खड़े किए हैं. इसकी गूँज इस सत्र में सुनाई पड़ेगी. topcat2 संसदीय कर्म के लिहाज से भी यह महत्वपूर्ण और लम्बा सत्र है. दो चरणों में यह 8 मई तक चलेगा. तब तक संसद के बाहर सम्भवतः कांग्रे

In [176]:
reviewed_ids = [
    "150222_parliament_budget_session_pk",
    "international-44946029",
    "130910_sehwag_gambhir_pujara_pp",
    "india-49221048",
    "130707_raghavji_mp_fma"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    122
Pending audit                           122
Needs retranslation                       6
Name: count, dtype: int64

ARTICLE ID: international-54702261

HINDI ORIGINAL:
हमला पाकिस्तान में ख़ैबर-पख़्तूनख़्वाह प्रांत के पेशावर शहर में हुआ. अस्पताल के अधिकारियों के मुताबिक़, मरने वालों की उम्र 20 से 30 साल के बीच है. दर्जनों लोग घायल हैं जिनमें से चार की उम्र 13 साल से कम है. शुरुआती रिपोर्टों में कहा गया था कि मरने वालों में चार बच्चे हैं, लेकिन बाद में ऐसी रिपोर्टों को वापस ले लिया गया. अभी तक किसी समूह ने हमले की ज़िम्मेदारी नहीं ली है. मामले की जांच शुरू कर दी गई है. समाप्त पाकिस्तान के प्रधानमंत्री इमरान ख़ान ने धमाके की कड़ी निंदा की है और जान के नुक़सान पर दुख जताया है. पुलिस ने बीबीसी को बताया कि धमाका स्थानीय समयानुसार सुबह 8:30 पर हुआ. पुलिस के मुताबिक़ धमाका उस वक़्त हुआ जब मदरसे में पढ़ाई चल रही थी. माना जा रहा है कि धमाके के वक़्त मदरसे में क़रीब 60 लोग मौजूद थे. एक चश्मदीद ने पुलिस को बताया 

In [177]:
reviewed_ids = [
    "071104_komal_column",
    "031031_russia_removal",
    "140906_urdu_press_review_vr",
    "090210_filmfestival_gorakhpur"
]

needs_retranslation_ids = [
    "international-54702261"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.loc[
    source_250["id"].isin(needs_retranslation_ids),
    "translation_quality_status"
] = "Needs retranslation"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    126
Pending audit                           117
Needs retranslation                       7
Name: count, dtype: int64

ARTICLE ID: india-39165960

HINDI ORIGINAL:
यही तो पहचान है धर्मनगरी काशी की. हज़ारों मंदिरों के इस शहर में बड़ी-बड़ी गाड़ियों का शोर, हवा में कुलांचे भरते हेलीकॉप्टर और जगह-जगह बने ऊंचे मंचों ने यहां लोगों की व्यस्तता को और बढ़ा दिया है. कहते हैं यूपी के कण-कण में सियासत भरी है लोगों के रगों में तो सियासी बहसें लहू बनकर दौड़ती हैं. यूपी चुनाव में बसपा बड़ी ताकत वाराणसी में बीजेपी दिग्गजों का जमावड़ा क्यों? मौका भी है, दस्तूर भी है और मिज़ाज भी. शहर में थोड़ा घूमने निकल जाइए तो पता चल जाएगा कि हर गली, हर नुक्कड़ और हर चौक-चौराहे पर लोगों का कम से कम एक समूह तो बनारसी पान की मिठास के बीच राजनीतिक बहस में डूबा हुआ है और यह बहस कभी-कभी तो इस पान के मिठास को थोड़ा कड़वा भी बना देता है. राजनीतिक मिज़ाज वाला प्रदेश बहरहाल अभी ऐसा लगता है कि जिस तरह बनारस की एक पहचान देवों के देव महादेव से है, 

In [178]:
reviewed_ids = [
    "061217_spl_sez_imballance",
    "india-46856839",
    "121217_greek_football_brothel_sponsor_sdp",
    "121028_international_others_thai_women_aa"
]

needs_retranslation_ids = [
    "india-39165960"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.loc[
    source_250["id"].isin(needs_retranslation_ids),
    "translation_quality_status"
] = "Needs retranslation"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

print("\nTotal rows:", len(source_250))
print("Missing audit statuses:",
      source_250["translation_quality_status"].isna().sum())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    130
Pending audit                           112
Needs retranslation                       8
Name: count, dtype: int64

Total rows: 250
Missing audit statuses: 0

ARTICLE ID: 050505_voteclose_britain

HINDI ORIGINAL:
पिछली संसद में सत्ताधारी लेबर पार्टी को 161 सांसदों का बहुमत हासिल था. इस चुनाव में लगभग साढ़े चार करोड़ मतदाता 645 सांसदों को चुनने के लिए अपने मताधिकार का इस्तेमाल कर सकते थे. ब्रिटेन के संसद के निचले सदन हाउस ऑफ़ कॉमन्स में कुल 646 सीटें हैं लेकिन एक उम्मीदवार की मृत्यु हो जाने के कारण एक सीट पर चुनाव स्थगित कर दिया गया था. इस तरह बहुमत हासिल करने के लिए किसी भी पार्टी को 324 सीटें जीतनी होंगी. जिन प्रमुख राजनीतिक दलों के बीच मुकाबला है, वे हैं - सत्ताधारी लेबर पार्टी, विपक्षी दल कंज़रवेटिव पार्टी और लिबरल डेमोक्रैट्स. पिछली संसद में लेबर पार्टी को 413, कंज़रवेटिव पार्टी को 166 और लिबरल डेमोक्रैट्स को 52 सीटें मिली थीं. प्रारंभिक चुनावी नतीजे मतदान ख़त्म होने के लगभग दो घंटे बाद आने शुरु ह

In [179]:
reviewed_ids = [
    "050505_voteclose_britain",
    "150713_bihar_women_strange_names_rns",
    "160412_assam_gogoi_fir_ml",
    "060106_rajasthan_rail",
    "040102_france_sikhs"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    135
Pending audit                           107
Needs retranslation                       8
Name: count, dtype: int64

ARTICLE ID: 060709_wordcup_final2006

HINDI ORIGINAL:
फ़ुटबॉल विश्व कप-2006 की ट्रॉफ़ी किसके नाम रहेगी इसका फ़ैसला रविवार शाम को होना है. फ़ाइनल मुकाबला 1998 में विश्व कप विजेता फ़्रांस और तीन बार विश्व कप जीत चुकी इटली के बीच है. 1982 के बाद ऐसा पहली बार हुआ है जब फ़ाइनल दो यूरोपीय देशों के बीच हो रहा है. 1982 में इटली ने तत्कालीन पश्चिमी जर्मनी को 3-1 से हराया था. 1998 के फ़ाइनल में मिली जीत विश्व कप में फ़्रांस का अब तक का सबसे बेहतरीन प्रदर्शन है. जबकि इटली की टीम 1934, 1938 और 1982 में ये खिताब अपने नाम कर चुकी है. आमने-सामने फ़्रांस और इटली अब तक आपस में 32 मैच खेल चुके हैं जिसमें से 17 इटली के नाम रहे हैं, सात फ़्रांस के और बाकी आठ ड्रॉ रहे. यूरो 2000 के फ़ाइनल में भी फ़्रांस और इटली के बीच मुकाबला हुआ था. ये मैच फ़्रांस ने गोल्डन गोल के ज़रिए 2-1 से जीता था. इसके बाद से दोनों देश

In [180]:
reviewed_ids = [
    "060709_wordcup_final2006",
    "031113_tdp_elections",
    "070823_vv_iraq_report",
    "131026_iran_hanging_akd",
    "100418_tharoor_controversy_vv"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    140
Pending audit                           102
Needs retranslation                       8
Name: count, dtype: int64

ARTICLE ID: 150110_france_huge_marches_sdp

HINDI ORIGINAL:
सबसे बड़ी रैली टूलूज़ में हुई जिसमें अस्सी हज़ार लोग शामिल हुए. मार्च के दौरान कुछ प्रदर्शकारियों के हाथों में बैनर थे जिन पर लिखा था- 'मैं नस्ली भेदभाव के ख़िलाफ़ हूं', 'मैं शार्ली हूं.' पेरिस में प्लेस द ला रिपब्लिक नामक जगह पर लोगों ने अपने कलम रखे जो फ्रांस में शार्ली एब्डो पत्रिका के दफ्तर पर बीते बुधवार हमले के बाद अवज्ञा का प्रतीक बन गया है. राजधानी पेरिस में रविवार को भी यूनिटी मार्च निकालने की तैयारी है जिसके लिए सुरक्षा के कड़े इंतज़ाम किए गए हैं. समाप्त यूनिटी मार्च में फ्रांस के राष्ट्रपति फ्रांस्वा ओलांद समेत 40 से अधिक देशों के राष्ट्राध्यक्ष और राज्याध्यक्ष शामिल होंगे जिनमें अधिकतर यूरोपीय देशों के नेता हैं. इस बीच पुलिस उन लोगों की तलाश कर रही है जो मारे गए तीन बंदूक़धारियों के साथी हो सकते हैं. फ्रांस के आंतरिक

In [181]:
reviewed_ids = [
    "150110_france_huge_marches_sdp",
    "071001_pakistan_cargo",
    "sport-47725658",
    "160212_ranchi_mamta_brave_girl_pk",
    "150526_shahrukh_tweet_salman_bajrangi_bhaijaan_hk"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    145
Pending audit                            97
Needs retranslation                       8
Name: count, dtype: int64

ARTICLE ID: 131126_assam_protes_vs

HINDI ORIGINAL:
समाचार एजेंसी पीटीआई के मुताबिक़ सामूहिक बलात्कार की इस घटना के बाद क्षेत्रीय लोगों में काफ़ी गुस्सा है. लोग दोषियों को कड़ी सज़ा देने का माँग कर रहे हैं. यह घटना पिछले शुक्रवार को उस समय हुई, जब महिला बोगीनाड़ी में पढ़ने वाली अपनी छह साल की बच्ची को स्कूल से आने के लिए एक शेयर्ड ऑटो से वहाँ जा रही थी. पुलिस सूत्रों के मुताबिक़ यह जगह लखीमपुर ज़िला मुख्यालय से 14 किलोमीटर दूर है. लोगों में गुस्सा उन्होंने बताया कि चार व्यक्तियों ने महिला के साथ कथित बलात्कारके बाद महिला की आँखें निकाल लीं, उनके सर और गर्दन पर चोट की. स्थानीय लोगों ने पीड़िता को ऑटो से बाहर फेंकते हुए देखा, यह घटना राष्ट्रीय राजमार्ग पर स्थित बोगीनाड़ी पुलिस चौकी से पाँच सौ मीटर की दूरी पर हुई. पुलिस ने सूचना मिलने के बाद पीड़ित को स्थानीय अस्पताल में भर्ती करवाया. वहाँ 

In [182]:
reviewed_ids = [
    "131126_assam_protes_vs",
    "140901_iraq_un_human_rights_commission_fma",
    "130220_dhoni_clarke_vd",
    "050418_saddam_talabani",
    "150825_kapil_sharma_debut_bollywood_ssm"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    150
Pending audit                            92
Needs retranslation                       8
Name: count, dtype: int64

ARTICLE ID: 131231_drone_test_site_sr

HINDI ORIGINAL:
अगले एक दशक में ड्रोन का वैश्विक बाज़ार 89 अरब डॉलर का हो जाने की संभावना है. संघीय उड्डयन प्रशासन (एफएए) के अनुसार ये राज्य हैं अलास्का, नेवादा, न्यूयॉर्क, नॉर्थ डेकोटा, टेक्सास और वर्जीनिया. यह चुनाव प्रक्रिया उस परियोजना का हिस्सा है जिसके तहत साल 2015 के अंत तक ड्रोन के लिए सुरक्षा और संचालन के नियम-क़ायदे बनाए जाने हैं. अभी तक यह सिर्फ सेना द्वारा इस्तेमाल किए जाते रहे हैं लेकिन अब रियल एस्टेट एजेंट से लेकर किसान और सेवा प्रदाता कंपनियां भी इनके व्यावसायिक उपयोग की संभावनाएं तलाश रही हैं. सुरक्षा प्रमुख चुनौती एफएए के प्रमुख माइकल ह्यूर्टा के अनुसार, अमरीकी आसमान में मानवरहित विमान को उड़ाने के लिए स्वीकृति मांगी जा रही है लेकिन इस दौरान सुरक्षा उनकी प्राथमिकता होगी. ड्रोन को कहां उड़ाया जाना है इसकी सूचना पायलटों को रुटीन घोषणा

In [183]:
reviewed_ids = [
    "131231_drone_test_site_sr",
    "040721_paes_olympics",
    "150213_msg_film_released_pm",
    "140423_varanasi_somnath_attack_ap",
    "160311_saina_nehwal_exits_all_england_aa"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    155
Pending audit                            87
Needs retranslation                       8
Name: count, dtype: int64

ARTICLE ID: 150522_islamic_state_palmyra_ml

HINDI ORIGINAL:
यूनेस्को ने इन पुरातात्विक भग्नावशेषों को विश्व विरासत सूची में शामिल किया है और आशंका जताई जा रही है कि इस्लामिक स्टेट के चरमपंथी उन्हें तबाह कर सकते हैं. उधर, संयुक्त राष्ट्र का कहना है कि उसे मिली खबरों के अनुसार सीरियाई सेना ने इस्लामिक स्टेट के पल्माइरा पर कब्ज़े से पहले वहां की आबादी को पलायन करने से रोका था. संयुक्त राष्ट्र का कोई प्रतिनिधि पल्माइरा में मौजूद नहीं है लेकिन विश्वसनीय सूत्रों के हवाले से उसका कहना है कि सीरियाई सेना ने नागरिकों को सुरक्षित स्थानों तक नहीं जाने दिया. सीरिया के राष्ट्रपति बशर अल असद की सरकार के एक अधिकारी ने इस्लामिक स्टेट के ख़िलाफ़ अमरीका के नेतृत्व वाले पश्चिमी देशों के गठबंधन से कहा है कि वो इस ऐतिहासिक विरासत को चरमपंथियों के हाथों नष्ट होने से बचाने के लिए कार्रवाई करें. समाप्त यूनेस्क

In [184]:
reviewed_ids = [
    "150522_islamic_state_palmyra_ml",
    "160405_vert_cul_10_spectacular_museums_rd",
    "040712_fdi_chidambaram",
    "050112_people_law",
    "151003_draught_karnatak_series_part_one_sr"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    160
Pending audit                            82
Needs retranslation                       8
Name: count, dtype: int64

ARTICLE ID: international-43396558

HINDI ORIGINAL:
14 मार्च 1879 को जन्मे आइंस्टाइन ने 18 अप्रैल 1955 को दुनिया को अलविदा कहा था. स्टीफन हॉकिंग की तरह आइंस्टाइन की ज़िंदगी भी 76 साल की रही. भौतिकशास्त्र के सर्वश्रेष्ठ वैज्ञानिक माने जाने वाले आइंस्टाइन की निजी जिंदगी उतनी बेहतर नहीं थी. साल 2012 में लेखक वॉल्टर इसाकसन ने अपनी किताब आइंस्टाइन: हिज लाइफ एंड यूनिवर्स में इस बात दावा किया था. उन्होंने लिखा था कि दुनिया के मशहूर वैज्ञानिक को अपनी निजी जिंदगी सफल बनाने के लिए काफी कोशिशें करनी पड़ी, लेकिन ये उनके लिए असंभव रहा. समाप्त जिस जंग को जीत नहीं पाए आइंस्टाइन वॉल्टर इसाकसन ने लिखा था कि कैसे आइंस्टाइन ने अपने दाम्पत्य जीवन को बनाए रखने की कोशिश की, लेकिन उनके लिए ये ऐसी जंग साबित हुई, जिसे वे जीत नहीं पाए. वास्तव में प्रेम के प्रति आइंस्टाइन का रुख इतना हस्तक्षेप करने वाला था कि जब उ

In [185]:
reviewed_ids = [
    "international-43396558",
    "sport-54682255",
    "india-48128218",
    "060530_reservation_strike",
    "140223_modi_rally_punjab_sr"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    165
Pending audit                            77
Needs retranslation                       8
Name: count, dtype: int64

ARTICLE ID: international-41395655

HINDI ORIGINAL:
लेकिन सऊदी किंग ने चर्चिल को तोहफ़े में हीरे की अंगूठी, मंकी-पर्ल के आकार वाली तलवार और एक ड्रेस दी थी. अमरीका-सऊदी अरब के बीच सबसे बड़ा हथियार सौदा क्या सऊदी शाह के आगे झुक गए थे ट्रंप? चर्चिल सऊदी किंग से ये तोहफ़े लेकर बेहद शर्मिंदा हुए और उनके लिए रॉल्स रॉयस कार तोहफ़े में भेजी. अमरीकी राष्ट्रपति डोनल्ड ट्रंप का सऊदी के शाही परिवार ने गर्मजोशी से स्वागत किया. सऊदी राज परिवार की ओर से विदेशी राष्ट्राध्यक्षों को महंगे उपहार देने का चलन बेहद पुराना है. लेकिन इसमें सबसे नया मामला अमरीकी राष्ट्रपति डोनल्ड ट्रंप का है. बीती मई में सऊदी अरब की यात्रा पर वर्तमान किंग सलमान ने उन्हें कई तरह के उपहार दिए थे. एक अमरीकी पत्रिका ने डेली बीस्ट ने अमरीकी आरटीआई एक्ट की मदद से ट्रंप को मिले तोहफ़ों की लिस्ट जारी की है जिनमें 83 तोहफ़े शामिल हैं. ट्

In [186]:
reviewed_ids = [
    "international-41395655",
    "140125_da_rday_special_parade_rd",
    "031114_bush_intelligence",
    "140908_amit_shah_political_analysis_vr",
    "070209_wi_samuels"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    170
Pending audit                            72
Needs retranslation                       8
Name: count, dtype: int64

ARTICLE ID: 141218_russia_putin_rouble_stable_sn

HINDI ORIGINAL:
पुतिन ने ये बात वर्ष के अंत में होने वाले संवाददाता सम्मेलन को संबोधित करते हुए कही. पुतिन ने रूबल में गिरावट के लिए 'बाहरी कारणों' को दोषी ठहराया है, लेकिन ये भी माना है कि केंद्रीय बैंक को पहले ही इस मसले पर काम करना शुरु कर देना चाहिए था. उन्होंने यह भी स्वीकार किया कि रूस अपनी अर्थव्यवस्था में विविधता लाने में विफल रहा, जिसको तेल की गिरती क़ीमतों और यूक्रेन संकट के तहत प्रतिबंधों से बड़ा झटका लगा. राष्ट्रपति पुतिन ने कहा कि देश का मुद्रा भंडार अर्थव्यवस्था को स्थिर रखने के लिए पर्याप्त है. समाप्त उनका कहना था कि अगर आर्थिक समस्या जारी रहती है तो सरकार 'सामाजिक ख़र्च और भविष्य के विकास पर खर्च कम करेगी'. साथ ही उन्होंने कहा कि इस समस्या से बाहर निकलने में शायद दो साल का समय लगेगा, लेकिन उसके बाद विकास निश्चित है. (बीबीस

In [187]:
reviewed_ids = [
    "141218_russia_putin_rouble_stable_sn",
    "040305_holi_songs",
    "080319_advani_book",
    "india-44912345",
    "141230_mahendra_singh_dhoni_retirement_vr"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    175
Pending audit                            67
Needs retranslation                       8
Name: count, dtype: int64

ARTICLE ID: international-46767468

HINDI ORIGINAL:
अपनी पत्नी शैनन और दोनों बेटियों के साथ क्रिस्टोफर वॉट्स इसके तुरंत बाद वे टीवी चैनलों में यह गुज़ारिश करते हुए दिखाई दिए कि जिस किसी ने उनकी पत्नी और बेटियों को अपने कब्ज़े रखा है, वह उन्हें छोड़ दे. 33 साल के वॉट्स जब यह बात टीवी कैमरों में बोल रहे थे तो उनकी आंखें कैमरे की नज़रों से सीधे मेल नहीं खा रही थीं. शुरुआत में इस छोटी सी बात पर किसी ने ध्यान नहीं दिया. लेकिन कुछ दिन गुज़र जाने के बाद अगस्त महीने के अंत में पुलिस ने पता लगाया कि वॉट्स ने खुद अपनी पत्नी और दोनों बेटियों की हत्या कर दी थी. वॉट्स ने अपना ग़ुनाह कबूल भी कर लिया. पुलिस को बच्चियों के शव तेल के एक टैंक से मिले तो वहीं उनकी पत्नी का शव एक गड्ढे से बरामद हुआ. वॉट्स के कबूलनामे के कुछ हफ़्तों बाद उन्हें तीन हत्याओं के जुर्म में आजीवन कारावास की सज़ा सुना दी गई. जेल मे

In [188]:
reviewed_ids = [
    "international-46767468",
    "040709_barrier_split",
    "061025_russia_broadcast",
    "120521_pak_mandir_burnt_ia",
    "040526_left_cmp"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    180
Pending audit                            62
Needs retranslation                       8
Name: count, dtype: int64

ARTICLE ID: 080801_india_iaea

HINDI ORIGINAL:
परमाणु समझौते को लागू करने की दिशा में आईएईए की मंज़ूरी आवश्यक थी. आईएईए की मंज़ूरी के बाद भारत को अब 45 सदस्यीय परमाणु सप्लायर्स ग्रुप (एनएसजी) से सहमति लेनी होगी. ताकि भारत परमाणु ईंधन के व्यापार में शामिल हो सके. इसके बाद भारत-अमरीका परमाणु समझौते को अमरीकी संसद के अनुमोदन की ज़रूरत होगी. भारत पर यह दबाव बना हुआ है कि वह नवंबर में होने वाले अमरीकी राष्ट्रपति चुनाव से पहले इस समझौते को अमरीकी संसद तक पहुँचा दे और उसकी मंज़ूरी ले ले. परमाणु समझौता हाल ही में भारत सरकार को इसी परमाणु समझौते को लेकर संसद में विश्वास मत प्रस्ताव रखना पड़ा. क्योंकि सरकार को बाहर से समर्थन दे रही वामपंथी पार्टियों ने परमाणु समझौते के मुद्दे पर सरकार से समर्थन वापस ले लिया था. लेकिन सरकार ने विश्वास मत हासिल कर लिया. भारत का तर्क है कि देश की बढ़ती ऊर्जा ज़रूरतों

In [189]:
reviewed_ids = [
    "080801_india_iaea",
    "000001_ugc",
    "150802_varun_gandhi_death_ia",
    "140412_rahul_gandhi_says_sn",
    "040827_italian_hostage"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    185
Pending audit                            57
Needs retranslation                       8
Name: count, dtype: int64

ARTICLE ID: india-45233283

HINDI ORIGINAL:
लेकिन इन लोगों में से एक शख्स हैं- हाल ही में रिजर्व बैंक ऑफ इंडिया (आरबीआई) में अंशकालिक निदेशक बने एस गुरुमूर्ति. एस गुरुमूर्ति अपने ट्वीट से इशारा कर रहे हैं कि ये बाढ़ अगर सबरीमाला के भगवान की नाराज़गी की वजह से है तो सुप्रीम कोर्ट को अपने फ़ैसले पर सोचना चाहिए. उन्होंने एक ट्विटर यूज़र के ट्वीट पर अपनी बात रखी है. उन्होंने लिखा, "सुप्रीम कोर्ट के जजों को देखना चाहिए कि केस और जो सबरीमाला में हुआ, उसके बीच कोई संबंध है या नहीं. अगर कोई संबंध होने का लाखों में एक चांस भी है तो लोगों को अयप्पन के ख़िलाफ़ फ़ैसला पसंद नहीं आएगा." जब कुछ लोगों ने उनके ट्वीट पर इस बात की आलोचना की तो उन्होंने फिर से अपनी बात को दोहराया. "भारत के बुद्धिजीवियों के पाखंड को देखकर हैरान हूं जो लोगों के विश्वास को कचरा समझते हैं. 99 फ़ीसदी भारतीय भगवान में विश्वास करत

In [190]:
reviewed_ids = [
    "india-45233283",
    "070620_heart_survival",
    "131010_punjab_accident_truck_ssr",
    "international-44164669",
    "160506_breast_cancer_test_easier_cj"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    190
Pending audit                            52
Needs retranslation                       8
Name: count, dtype: int64

ARTICLE ID: 061125_rajpakshe_arrived

HINDI ORIGINAL:
भारत ने भी श्रीलंका की मौजूदा स्थिति पर चिंता व्यक्त की है. उम्मीद है कि भारतीय नेताओं और राष्ट्रपति राजपक्षे के साथ मुलाक़ात में एलटीटीई के नियंत्रण वाले इलाक़े की स्थिति के साथ-साथ शांति समझौते पर भी चर्चा होगी. वैसे मंगलवार से पहले राष्ट्रपति राजपक्षे की भारतीय नेताओं से बातचीत संभव नहीं दिखती. क्योंकि राष्ट्रपति राजपक्षे पहले देहरादून जा रहे हैं. प्रधानमंत्री मनमोहन सिंह से वे बुधवार को मिलेंगे. एक साल पहले राष्ट्रपति का पद संभालने के बाद महिंदा राजपक्षे की ये दूसरी भारत यात्रा है. शुक्रवार को ही भारत के विदेश सचिव शिवशंकर मेनन श्रीलंका के दौरे से लौटे हैं. भारत की चिंता विदेश सचिव शिवशंकर मेनन श्रीलंका को भारत की चिंता से अवगत कराने के लिए कोलंबो गए थे. भारत को ख़ास तौर पर जाफ़ना प्रायद्वीप की स्थिति पर चिंता है जहाँ पाँच लाख लोग

In [191]:
passed_ids = [
    "061125_rajpakshe_arrived",
    "121115_2g_auction_ac",
    "international-41960891"
]

retranslation_ids = [
    "140524_iran_hanging_aa",
    "science-41399809"
]

source_250.loc[
    source_250["id"].isin(passed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.loc[
    source_250["id"].isin(retranslation_ids),
    "translation_quality_status"
] = "Needs retranslation"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    193
Pending audit                            47
Needs retranslation                      10
Name: count, dtype: int64

ARTICLE ID: 040510_fillipino_election

HINDI ORIGINAL:
इस चुनाव प्रचार के दौरान 29 उम्मीदवारों समेत 90 से भी ज़्यादा लोगों की मौत हो चुकी है. विद्रोहियों के हमलों और मतदान में धांधली की धमकियों के बीच रविवार को राष्ट्रपति पद के सभी उम्मीदवारों ने मिलकर शांतिपूर्ण मतदान के लिए प्रार्थना की. इसके लिए देश भर के पोलिंग बूथों में क़रीब 230,000 सुरक्षाकर्मी तैनात किए गए हैं. हालाँकि मतदान शुरू होने से पहले कुछ नई हिंसक घटनाओं की ख़बर आई है. एक पुलिस अधिकारी ने समाचार एजेंसी एएफपी को बताया कि ताज़ा हिंसा में मनीला में एक उम्मीदवार के मुख्यालय पर अज्ञात बंदूकधारियों ने दो लोगों को मार दिया. इसके अलावा स्थानीय उम्मीदवारों के सहायकों के अपहरण और पूर्वी समर ज़िले में एक धमाके से आग लगने की भी ख़बर हैं. सर्वेक्षण चुनाव से पूर्व होने वाले सर्वेक्षणों में फ़िलीपींस की राष्ट्रपति ग्लोरिया आरोयो को विरो

In [192]:
passed_ids = [
    "040510_fillipino_election",
    "160823_sania_mirza_wta_doubles_top_sk",
    "150221_mulayam_modi_saifai_du",
    "040804_kolkata_reaction"
]

retranslation_ids = [
    "india-43796841"
]

source_250.loc[
    source_250["id"].isin(passed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.loc[
    source_250["id"].isin(retranslation_ids),
    "translation_quality_status"
] = "Needs retranslation"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    197
Pending audit                            42
Needs retranslation                      11
Name: count, dtype: int64

ARTICLE ID: india-41263102

HINDI ORIGINAL:
गुरुग्राम स्थित प्रद्युम्न का स्कूल रायन इंटरनेशनल बिहार की एक मां हरियाणा के गुरुग्राम में भी थीं. उन्होंने भी सुबह से कुछ नहीं खाया था. पानी की एक बूंद तक नहीं पी थी. वह भी निर्जला उपवास में थी. फ़र्क सिर्फ़ इतना था कि वह अपने बेटे की याद में खाना-पानी त्याग चुकी हैं. जब मैं प्रद्युम्न के घर पहुंचा तो नीचे के कमरे में उनकी मां सोफ़े पर बेसुध पड़ी दिखीं. उनके आसपास सात-आठ महिलाएं थीं. इनमें से एक महिला ने प्रद्युम्न की मां का हाथ थाम रखा था. सभी महिलाएं शांत थीं, शोर सिर्फ़ पंखे का था. .... वो 15 मिनट जिसने एक पिता की ज़िंदगी बदल दी टीचर ने टॉयलेट में खड़ा किया, डिप्रेशन में लड़की प्रद्युम्न के घर पर पसरा सन्नाटा भरी दोपहरी में बाहर टेंट लगा था. वहां कुर्सी पर प्रद्युम्न कृष्ण रूप धारण कर तस्वीर में मुस्कुरा रहा था. लेकिन वहां बैठे लोगों के चे

In [193]:
passed_ids = [
    "india-41263102",
    "131106_uk_migration_contribution_rd",
    "070616_gaza_arab_ministers",
    "041015_warne_record"
]

retranslation_ids = [
    "090216_jobloss_column3_alk"
]

source_250.loc[
    source_250["id"].isin(passed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.loc[
    source_250["id"].isin(retranslation_ids),
    "translation_quality_status"
] = "Needs retranslation"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    201
Pending audit                            37
Needs retranslation                      12
Name: count, dtype: int64

ARTICLE ID: 141107_bin_laden_death_dispute_us_sk

HINDI ORIGINAL:
अमरीकी समाचार पत्र 'वाशिंगटन पोस्ट' से बात करते हुए 'नेवी सील्स' के पूर्व सैनिक 38 वर्षीय रॉबर्ट ओ-नील ने दावा किया है कि ओसामा की मौत उनकी चलाई गोली से हुई थी. लेकिन रॉबर्ट ओ-नील का ये दावा पूर्व मरीन मैट बेसोनेट के दावे के विपरीत है जिन्होंने इस विषय पर 2012 में किताब लिखी थी. मैट बेसोनेट ने 2012 में 'नो ईज़ी डे' नाम से एक किताब लिखी थी, जिसमें उन्होंने लादेन की मौत के बारे में कुछ अलग ही बताया था. अल-क़ायदा प्रमुख लादेन साल 2011 में पाकिस्तान के ऐबटाबाद शहर के एक घर में अमरीकी सेना की विशेष टुकड़ी 'नेवी सील्स' के एक ऑपरेशन में मारे गए थे. सिर में गोली मारी पूर्व सैनिक ओ-नील ने वाशिंगटन पोस्ट को दिए गए साक्षात्कार में बताया कि वे और उनके दल के एक सदस्य, जिनकी पहचान अब तक रहस्य है, ऐबटाबाद के अहाते में स्थित भवन की तीसरी 

In [194]:
passed_ids = [
    "141107_bin_laden_death_dispute_us_sk",
    "051017_iraq_rebels",
    "050228_tax_income",
    "international-56439434",
    "international-45563134"
]

source_250.loc[
    source_250["id"].isin(passed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    206
Pending audit                            32
Needs retranslation                      12
Name: count, dtype: int64

ARTICLE ID: 121009_rushdie_film_da

HINDI ORIGINAL:
‘पीवीआर पिक्चर्स’ का कहना है कि उन्होंने भारत में फिल्म के राइट्स खरीद लिए हैं और उसे दिसंबर में रिलीज़ करने की योजना है. इससे पहले फिल्म की निर्देशक दीपा मेहता ने कहा था कि उन्हें भारत में डिस्ट्रीब्यूटर ढूंढने में दिक्कत हो रही है. वर्ष 1988 में सलमान रुश्दी की किताब ‘द सैटेनिक वर्सिस’ पर भारत में प्रतिबंध लगा दिया गया था. ये अब भी लागू है. इसी वर्ष जनवरी में जान से मारने की धमकी मिलने के बाद, उन्होंने जयपुर साहित्य महोत्सव में हिस्सा लेने से मना कर दिया था. विरोध का माहौल ऐसा था कि वो वीडियो लिंक के ज़रिए भी वहां जुटे लोगों को संबोधित नहीं कर पाए. बॉलीवुड फिल्म के स्तर पर रिलीज़ ‘मिडनाइट्स चिल्ड्रन’ किताब पर आधारित फिल्म पिछले महीने कनाडा के टोरोन्टो फिल्म महोत्सव में दिखाई गई थी. पीवीआर पिक्चर्स के प्रमुख कमल गियानचंदानी ने एक समाचा

In [195]:
passed_ids = [
    "121009_rushdie_film_da",
    "040923_manmohan_speech",
    "070427_mcgrath_lastmatch",
    "india-49659647",
    "india-39447746"
]

source_250.loc[
    source_250["id"].isin(passed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    211
Pending audit                            27
Needs retranslation                      12
Name: count, dtype: int64

ARTICLE ID: 070425_ccpa_inconclusive

HINDI ORIGINAL:
इस बैठक के बाद केंद्रीय मानव संसाधन मंत्री अर्जुन सिंह ने कहा कि एक सुझाव आया है कि इस मामले में सभी से व्यापक सलाह मशविरा किया जाए. रेलमंत्री लालू प्रसाद यादव ने कहा है कि सलाह मशविरे की शुरुआत गुरुवार को यूपीए के नेताओं और वामपंथी दलों की बैठक होगी. उल्लेखनीय है कि केंद्र सरकार ने इसी वर्ष की शुरुआत में सभी केंद्रीय शिक्षण संस्थानों में अन्य पिछड़ा वर्ग के लोगों को 27 प्रतिशत आरक्षण देने का विधेयक पारित कराते हुए इसकी अधिसूचना जारी की थी. लेकिन इस पर सुप्रीम कोर्ट ने रोक लगा दी थी. केंद्र सरकार की याचिका पर सुनवाई करते हुए सुप्रीम कोर्ट ने इस सत्र में ओबीसी आरक्षण पर हाल मे लगाई रोक को हटाने से इनकार कर दिया था. इसके बाद सरकार ने छात्रों के भविष्य का हवाला देते हुए दलील दी थी कि इस मामले की जल्द सुनवाई होनी चाहिए. जिसे कोर्ट ने स्वी

In [196]:
passed_ids = [
    "070425_ccpa_inconclusive",
    "150408_kangana_ranaut_deepika_padukone_md",
    "international-41521329",
    "040421_tripura_polling",
    "081202_rice_india_ac"
]

source_250.loc[
    source_250["id"].isin(passed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    216
Pending audit                            22
Needs retranslation                      12
Name: count, dtype: int64

ARTICLE ID: 150708_us_army_cuts_aa

HINDI ORIGINAL:
ख़र्चे कम करने की इस मुहिम के तहत सेना के 17 हजार ऐसे कर्मचारियों की भी छंटनी होगी जो असैन्य कामों में लगे हैं. अमरीकी अख़बार यूएएस टुडे की रिपोर्ट के मुताबिक़ कटौती से जुड़ी इस योजना की घोषणा इसी हफ़्ते हो सकती है जिसके बाद 2017 तक अमरीकी सेना में फौजियों की संख्या घट कर साढ़े चार लाख रह जाएगी. बड़ी कटौती अमरीकी सैनिकों की संख्या 2012 में उस वक्त सबसे ज़्यादा यानी 5.7 लाख थी जब अफ़ग़ानिस्तान और इराक़ संकट अपने चरम पर थे. सैनिकों की संख्या में इतनी बड़ी कटौती की बातें 2014 की शुरुआत से ही हो रही हैं. उस वक्त तत्कालीन अमरीकी रक्षा मंत्री चक हेगेल ने सैनिकों की संख्या को 4.5 लाख करने की बात कही थी. समाप्त 2001 के हमलों से एक साल पहले सैनिकों की संख्या 480,000 थी. अफ़ग़ानिस्तान से वापसी के बाद भी 10 हजार अमरीकी सैनिक वहां बने रहेंगे जबकि इ

In [197]:
passed_ids = [
    "150708_us_army_cuts_aa",
    "050713_bombsuspect_family",
    "international-51059388"
]

retranslation_ids = [
    "130912_nepal_strike_ssr",
    "080514_kashmir_ceasefire"
]

source_250.loc[
    source_250["id"].isin(passed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.loc[
    source_250["id"].isin(retranslation_ids),
    "translation_quality_status"
] = "Needs retranslation"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    219
Pending audit                            17
Needs retranslation                      14
Name: count, dtype: int64

ARTICLE ID: 070324_india_press

HINDI ORIGINAL:
मैच का परिणाम भारतीय समयानुसार भोर के दो बजे के बाद आया था लेकिन लगता है कि अख़बारों ने बहुत इंतज़ार किया. शायद यही कारण था कि कुछ अख़बारों ने मुखपृष्ठ पर तस्वीर लगाकर छोटी सी ख़बर दे दी है. हिंदी अख़बार दैनिक जागरण ने मैच की पूरी रिपोर्ट छापते हुए शीर्षक दिया है टिकट कटा विदाई बाकी. तस्वीर है सचिन तेंदुलकर की आउट होकर वापस जाते हुए. अख़बार कहता है कि ' अब बेहतर होगा कि कागज़ी शेर कुछ दिनों तक स्वदेश न लौटें. ' राष्ट्रीय सहारा का शीर्षक है ' ध्वस्त ' श्रीलंका से हारकर भारत बाहर अब बरमूडा के पराक्रम पर टिकी उम्मीदें. कुछ अख़बारों ने एक पारी की पूरी रिपोर्ट दी है और आशंकाएं जताई हैं. टाइम्स ऑफ इंडिया ने भारत के तीन विकेट गिरने तक के स्कोर के साथ मुखपृष्ठ पर गांगुली और सचिन की तस्वीर लगाई है. शीर्षक है ' बैक टू द वॉल फॉर इंडिया ' यानि भारत की 

In [198]:
passed_ids = [
    "070324_india_press",
    "050523_bihar_nda",
    "041125_pm_sankaracharya",
    "051209_tamilnadu_cyclone",
    "061219_spl_sez_orissa"
]

source_250.loc[
    source_250["id"].isin(passed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    224
Needs retranslation                      14
Pending audit                            12
Name: count, dtype: int64

ARTICLE ID: 120831_filmreview_joker_dk

HINDI ORIGINAL:
अर्णब बनर्जी बीबीसी हिंदी डॉट कॉम के लिए जोकर अक्षय कुमार कि 100वीं फिल्म है. जोकर जहां शिरीष की बतौर निर्देशक दूसरी फिल्म है वहीं अक्षय कुमार ने इस फिल्म से बॉलीवुड में अपना शतक पूरा किया है. जैसा कि फिल्म के शीर्षक से ही पता चलता है, फिल्म दर्शकों को जोकर बनाने का ही एक प्रयास है. हम सबसे यही उम्मीद की जाती है न कि हम अपना काम पूरी ईमानदारी और लगन से करें. एक फिल्म समीक्षक होने के नाते मेरा काम है हर फिल्म को पूरा देखना और फिर उस पर अपनी राय देना. लेकिन यकीन मानिए 'जोकर' को पूरा देखना मेरे सब्र का इम्तेहान तो था ही, साथ ही मुझे ऐसा भी लगा कि इस फिल्म के ज़रिए क्या लोगों की समझ का मज़ाक उड़ाया जा रहा है. कहने को तो 'जोकर' एक कॉमेडी फिल्म है लेकिन फिल्म के हास्य में नयापन बिलकुल नहीं है बल्कि मैं तो कहूंगा कि फिल्म मूर्खता की चरमसीमा

In [199]:
passed_ids = [
    "120831_filmreview_joker_dk",
    "031031_sharon_palestine",
    "india-47692221",
    "120816_rachna_online_chat_aa",
    "151118_islamic_state_claims_kills_hostages_hk"
]

source_250.loc[
    source_250["id"].isin(passed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    229
Needs retranslation                      14
Pending audit                             7
Name: count, dtype: int64

ARTICLE ID: 150622_bihar_nitish_kumar_narendra_modi_social_media_rns

HINDI ORIGINAL:
ऐसे कई उदाहरण है जिनमें नीतीश कुमार के समर्थन में चलाया जा रहा प्रचार बीते साल के नरेंद्र मोदी के चुनाव प्रचार अभियान की नक़ल दिखाई दे रहा है. इस पूरे प्रचार अभियान को उन्हीं प्रशांत किशोर ने डिज़ाइन किया है जो साल 2014 में नरेंद्र मोदी की प्रचार टीम का अहम हिस्सा थे. नीतीश कुमार ने सार्वजनिक रूप से यह बताया है कि प्रशांत उनके लिए काम कर रहे हैं, प्रशांत पार्टी की बैठकों में भी शामिल हो रहे हैं, सार्वजनिक रूप से पार्टी सम्मेलनों को संबोधित कर रहे रहे हैं. कॉमिक्स के हीरो पिछले साल हुए लोकसभा चुनाव के दौरान ‘बाल नरेंद्र’ कॉमिक्स आई थी जिसमें नरेंद्र मोदी के बचपन की बहादुरी, मेहनत और ईमानदारी दिखाई गई थी. समाप्त अब विधानसभा चुनाव से पहले नीतीश कुमार के जीवन पर आधारित कॉमिक्स सीरीज़ आ गई है. इस सीरीज़ की प

In [200]:
passed_ids = [
    "150622_bihar_nitish_kumar_narendra_modi_social_media_rns",
    "080131_kenya_mp_killed",
    "india-50975864",
    "060129_nepal_candidate_rj",
    "131202_saina_sindhu_badminton_vr"
]

source_250.loc[
    source_250["id"].isin(passed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

next_batch = source_250[
    source_250["translation_quality_status"] == "Pending audit"
].head(5)

for _, row in next_batch.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1200])
    print("\nENGLISH TRANSLATION:")
    print(row["english_article"][:1500])

AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    234
Needs retranslation                      14
Pending audit                             2
Name: count, dtype: int64

ARTICLE ID: 080830_orissa_violence

HINDI ORIGINAL:
उड़ीसा में विश्व हिंदू परिषद के नेता स्वामी लक्ष्मणानंद की हत्या के बाद स्थिति तनावपूर्ण है. कोरापुट ज़िला हिंसक गतिविधियों से दूर था लेकिन शनिवार को स्थानीय विक्रम देव कॉलेज की दीवारों पर समुदाय विशेष के ख़िलाफ़ लिखी गई आपत्तिजनक बातों को लेकर तनाव फैल गया. इसके बाद दो समुदायों के बीच झड़प शुरु हो गई. झड़प में शामिल भीड़ को तितर-बितर करने के लिए पुलिस ने पहले लाठीचार्ज किया और आंसू गैस के गोले छोड़े. इससे उग्र भीड़ को नियंत्रित करने में विफल रहने पर पुलिस ने हवा में गोलियां चलाईं. कर्फ़्यू कोरापुट के पुलिस अधीक्षक दीपक कुमार ने बताया कि शहर में कर्फ़्यू लगा दिया गया है. वो ख़ुद इस झड़प में घायल हो गए हैं. हिंसक भीड़ ने पुलिस स्टेशन पर भी हमला किया और फर्नीचर को नुकसान पहुंचाया. समस्या उस समय शुरू हुई जब दीवारों पर लिखी बातों से नाराज़ 

In [201]:
passed_ids = [
    "080830_orissa_violence",
    "india-45961945"
]

source_250.loc[
    source_250["id"].isin(passed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("FINAL AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

print("\nPending audit:",
      (source_250["translation_quality_status"] == "Pending audit").sum())

print("Total articles:", len(source_250))

print("\nSaved to:", staging_path)

FINAL AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    236
Needs retranslation                      14
Name: count, dtype: int64

Pending audit: 0
Total articles: 250

Saved to: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/FactIndic_Source_250_Staging.csv


In [202]:
retranslation_df = source_250[
    source_250["translation_quality_status"] == "Needs retranslation"
].copy()

print("Articles needing retranslation:", len(retranslation_df))

for _, row in retranslation_df.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("\nHINDI ORIGINAL:")
    print(row["hindi_article"][:1800])
    print("\nCURRENT ENGLISH:")
    print(row["english_article"][:1800])

Articles needing retranslation: 14

ARTICLE ID: 141013_wusatullah_amitabh_bachchan_rns

HINDI ORIGINAL:
पिछले चार दशकों में कई पीढ़ियाँ उनकी फ़िल्में देखते हुए पली-बढ़ी, जवान हुईं और ढल गईं. लेकिन अमिताभ आज भी अपने ख़ास अंदाज में रुपहले पर्दे पर छाए हुए हैं. हिन्दी सिनेमा के महानायक का जादू जितना हिन्दुस्तान में कारगर था उतना ही पाकिस्तान में. पढ़ें वुसतुल्लाह ख़ान का ब्लॉग बिल्कुल भी आश्चर्यचकित न होइएगा अगर मैं कह दूँ कि मैं और अमिताभ बच्चन और मैं एक साथ बड़े हुए, पर अब मैं बूढ़ा हो रहा हूँ. मैं आपको समझाता हूँ, जीवन में पहली फ़िल्म आनंद देखी तो पता चला कि फ़िल्म क्या होती है. आनंद बाबू को ज़िंदा रखने के प्रयास में जुटे एक दुबले पतले लंबे से डॉक्टर को देखा तो पता चला, दोस्ती के क्या मानी हैं. और जब बाहर निकला तो ये जाना कि अपने ही गाल पर बहते ख़ामोश आँसू इतने नमकीन क्यों होते हैं. उसके बाद से घर पुराने कपड़े, घिसे-पिटे बर्तन, टूटी लकड़ियाँ ग़ायब होने लगीं. अब आप ख़ुद ही बताइए कि मेरे जैसा चौथी क्लास का बच्चा, तीन कैसेटों वाले वीसीआर पर चोरी छिपे अमिताभ की नई फ़िल्म ज़ंजीर देखने के लि

In [203]:
passed_ids = [
    "130606_china_gender_ss",
    "science-41399809",
    "080514_kashmir_ceasefire"
]

source_250.loc[
    source_250["id"].isin(passed_ids),
    "translation_quality_status"
] = "Initial review - no obvious mismatch"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("UPDATED AUDIT PROGRESS:")
print(source_250["translation_quality_status"].value_counts())

print("\nRETRANSLATION QUEUE:")
print(
    source_250.loc[
        source_250["translation_quality_status"] == "Needs retranslation",
        "id"
    ].to_string(index=False)
)

UPDATED AUDIT PROGRESS:
translation_quality_status
Initial review - no obvious mismatch    239
Needs retranslation                      11
Name: count, dtype: int64

RETRANSLATION QUEUE:
141013_wusatullah_amitabh_bachchan_rns
                international-47105506
                international-48243735
       150816_badminton_saina_final_dp
                international-49451047
                international-54702261
                        india-39165960
                140524_iran_hanging_aa
                        india-43796841
            090216_jobloss_column3_alk
               130912_nepal_strike_ssr


In [204]:
article_id = "141013_wusatullah_amitabh_bachchan_rns"

corrected_translation = """Over the past four decades, generations of people have grown up watching his films. They have grown older, grown old, and some have passed away. Yet Amitabh Bachchan continues to dominate the silver screen in his distinctive style. The magic of this legendary Hindi cinema star has been just as powerful in Pakistan as it has been in India.

Read Wusatullah Khan's blog.

Do not be surprised if I say that Amitabh Bachchan and I grew up together, although I am now growing old. Let me explain.

The first film I ever watched was Anand. It taught me what a film was. I saw a tall, thin doctor trying to keep Anand Babu alive, and I learned the meaning of friendship. When I came out of the cinema, I also understood why the silent tears running down one's cheeks taste so salty.

After that, old clothes, worn-out utensils and broken pieces of wood began disappearing from our house. Now tell me, how could a fourth-grade child like me save 10 rupees to secretly watch Amitabh's new film Zanjeer on a VCR with three cassettes, especially in the 1970s?

Namak Haraam

One day, the VCR owner whispered to me, "I'm showing Namak Haraam this evening. Come over."

I said, "I only have five rupees in my piggy bank."

He replied, "That's all right. You can watch it on credit."

So I told my mother that I was going to a friend's house to study that evening because I had a test the next morning.

My mother looked straight into my eyes and said, "When you come back after studying, tell me the story of Namak Haraam too."

Then she said, "You should tell me the truth before going. I gave birth to you, not the other way around. Do you have any money?"

I said, "No."

She untied a knot in her dupatta, took out ten rupees and placed the note in my palm. From that day onwards, at least, I stopped lying to my mother.

Sholay

While we were watching Sholay, the police raided the place. In those days, watching films on a VCR, especially smuggled Indian films, was considered a serious offence by the police. If Mazid had not carried me on his shoulder and helped me climb over the wall that day, my father's honour..."""

# Update the English translation
source_250.loc[
    source_250["id"] == article_id,
    "english_article"
] = corrected_translation

# Keep flagged because the Hindi source is incomplete
source_250.loc[
    source_250["id"] == article_id,
    "translation_quality_status"
] = "Needs retranslation - source text incomplete"

# Save updated staging file
source_250.to_csv(staging_path, index=False, encoding="utf-8-sig")

print("Saved corrected translation:", article_id)
print("Status: Needs retranslation - source text incomplete")
print("File:", staging_path)

Saved corrected translation: 141013_wusatullah_amitabh_bachchan_rns
Status: Needs retranslation - source text incomplete
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/FactIndic_Source_250_Staging.csv


In [205]:
row = source_250.loc[
    source_250["id"] == "international-47105506"
].iloc[0]

print("HINDI ORIGINAL:\n")
print(row["hindi_article"])

print("\nCURRENT ENGLISH:\n")
print(row["english_article"])

HINDI ORIGINAL:

सबसे पहले बात हज सब्सिडी की. पाकिस्तान में इन दिनों हज के ख़र्च में हुई बढ़ोत्तरी का मुद्दा भी छाया हुआ है. इमरान ख़ान की सरकार ने सत्ता में आने के बाद इस हफ़्ते अपनी पहली हज पॉलिसी जारी की. इसके तहत अब एक पाकिस्तानी को इस साल हज पर जाने के लिए चार लाख 76 हज़ार पाकिस्तानी रुपए देने होंगे जबकि पिछले साल ये रक़म दो लाख 80 हज़ार रुपए थे. इसको लेकर पाकिस्तानी संसद में ख़ूब हंगामा हुआ. अख़बार नवा-ए-वक़्त के अनुसार विपक्षी पार्टी जमात-ए-इस्लामी के एक सांसद मुश्ताक़ अहमद ने कहा कि सरकार ने हज यात्रियों पर ड्रोन हमला किया है. सांसद ने कहा कि सरकार ने हज पर कोई सब्सिडी नहीं दी. उनके अनुसार धार्मिक मामलों के मंत्री ने अपनी रिपोर्ट में हज सब्सिडी देने की अपील की थी लेकिन इमरान ख़ान की सरकार ने अपने ही मंत्री की सिफ़ारिश को ठुकरा दिया. अख़बार लिखता है कि सांसद मुश्ताक़ अहमद ने सरकार पर चुटकी लेते हुए कहा, ''मदीना की रियासत क़ायम करने का दावा करने वाले लोग, अब लोगों को मक्का और मदीना जाने से रोक रहे हैं.'' ग़ौरतलब है कि प्रधानमंत्री बनने के बाद अपने पहले भाषण में कहा था कि इस्लाम क

In [206]:
corrected_translation = """First, the issue of the Haj subsidy. The increase in the cost of performing Haj has been a major topic of discussion in Pakistan these days. This week, after coming to power, Imran Khan's government announced its first Haj policy. Under the new policy, a Pakistani pilgrim will have to pay 476,000 Pakistani rupees to perform Haj this year, compared with 280,000 rupees last year.

The issue caused considerable uproar in the Pakistani Parliament. According to the newspaper Nawa-i-Waqt, Mushtaq Ahmed, a member of the opposition Jamaat-e-Islami, said that the government had launched a "drone attack" on Haj pilgrims. He said the government had provided no subsidy for Haj. According to him, the Minister for Religious Affairs had recommended a Haj subsidy in his report, but Imran Khan's government rejected its own minister's recommendation.

The newspaper reported that Mushtaq Ahmed took a dig at the government, saying, "Those who claim to establish the State of Medina are now stopping people from travelling to Mecca and Medina."

It is worth recalling that, in his first speech after becoming Prime Minister, Imran Khan had said that he wanted to govern Pakistan along the lines of the way Prophet Muhammad had governed Medina.

Responding to the opposition's criticism, Parliamentary Affairs Minister Ali Muhammad Khan said that the cost of Haj had increased in Saudi Arabia itself, which was why pilgrims were being charged more.

However, according to the newspaper Jang, Minister for Religious Affairs Noor-ul-Haq Qadri said, "The State of Medina does not mean that people should be sent for Haj free of charge or with a subsidy." He added that performing Haj with the help of a subsidy went against the basic spirit of Haj.

**Foreign Minister Shah Mahmood Qureshi and the Kashmir issue in the British Parliament**

Meanwhile, Pakistan has said that an event will be organised in the British Parliament in support of people living in Indian-administered Kashmir.

According to the newspaper Daily Khabrain, a special programme will be held on 4 February in the House of Commons, the lower house of the British Parliament, under the All-Party Parliamentary Group. The programme will be held in support of people living in Indian-administered Kashmir. Pakistan's Foreign Minister Shah Mahmood Qureshi will also attend.

The newspaper reported that India had written to Britain asking it to stop the programme, but the British government rejected the request.

Daily Khabrain quoted a spokesperson for the British High Commission in New Delhi as saying that British MPs have the right to decide whom they meet and what they discuss, and that the British government cannot interfere in this matter.

The newspaper also reported that the British Foreign Office had issued a press release stating that an exhibition would be held in London after the 4 February programme. The exhibition would highlight alleged human rights violations in Indian-administered Kashmir.

The British spokesperson said that the UK was aware that Pakistan's Foreign Minister Qureshi would be travelling to London to attend the 4 February programme. However, this was a private visit, and he would not meet either the British Prime Minister or the Foreign Secretary during the visit.

It is worth noting that on 29 January, the Pakistani Foreign Minister had spoken by telephone with Mirwaiz Umar Farooq, a leader of the Hurriyat Conference active in Indian-administered Kashmir. Qureshi had said that his government would raise the issue of Indian-administered Kashmir at every international forum.

India strongly objected to this conversation and summoned Pakistan's High Commissioner in India to the Ministry of External Affairs. In response, Pakistani Foreign Secretary Tehmina Janjua also summoned India's High Commissioner in Islamabad to her office.

**The Taliban say they are not opposed to women working**

News related to a possible agreement between the United States and the Taliban also remained prominent in newspapers throughout the week.

According to the newspaper Nawa-i-Waqt, Taliban spokesperson Zabiullah Mujahid said that US President Donald Trump was serious about withdrawing American troops from Afghanistan.

The Taliban spokesperson said that after the withdrawal of American forces, the Taliban would implement Islamic law in Afghanistan, but only after consulting all political parties.

He also said that the Taliban were no longer opposed to women's education and employment. Their government would make every effort to create an environment in which women in Afghanistan could feel safe working.

According to the newspaper, the Taliban spokesperson said that the next round of talks with the United States would take place in Doha on 25 February, although the date had not yet been confirmed by the United States.

(You can also read this: Click here for BBC Hindi's Android app. Follow us on Facebook, Twitter, Instagram and YouTube.)"""

article_id = "international-47105506"

source_250.loc[
    source_250["id"] == article_id,
    "english_article"
] = corrected_translation

source_250.loc[
    source_250["id"] == article_id,
    "translation_quality_status"
] = "Corrected - needs final review"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("Corrected translation saved:", article_id)
print("Status: Corrected - needs final review")
print("File:", staging_path)

Corrected translation saved: international-47105506
Status: Corrected - needs final review
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/FactIndic_Source_250_Staging.csv


In [207]:
row = source_250.loc[
    source_250["id"] == "international-48243735"
].iloc[0]

print("HINDI ORIGINAL:\n")
print(row["hindi_article"])

print("\nCURRENT ENGLISH:\n")
print(row["english_article"])

HINDI ORIGINAL:

रूहानी ने कहा कि नए अमरीकी प्रतिबंधों के कारण देश की हालत 1980-88 के बीच इराक़ से हुए युद्ध से भी बदतर हो गई है. कुछ दिनों पहले खाड़ी में युद्धपोत और लड़ाकू विमानों की तैनाती के बाद अमरीका से बढ़ते तनावों के बीच रूहानी का ये बयान आया है. घरेलू राजनीतिक दबावों में आए रूहानी ने प्रतिबंधों का मुकाबला करने के लिए राजनीतिक एकता पर बल दिया है. तेहरान में राजनीतिक कार्यकर्ताओं से रूहानी ने कहा, "युद्ध के दिनों में हमें अपने बैंकिंग, तेल और आयात में मुश्किल नहीं आई थी और सिर्फ हथियारों की ख़रीद पर प्रतिबंध था." रूहानी ने कहा, "दुश्मनों की ओर से डाला जा रहा दबाव एक युद्ध है जो हमारे इस्लामिक क्रांति के इतिहास में अभूतपूर्व है...लेकिन हम निराश नहीं हैं और भविष्य के लिए बेहतर उम्मीद है. मुझे विश्वास है कि अगर हम एकता बनाए रखेंगे तो इन मुश्किलों को पीछे छोड़ सकते हैं." अमरीका और ईरान के बीच तनाव के कारण 2015 में हुए परमाणु समझौते पर सवालिया निशान लग गया है. इस समझौते पर ईरान ने सुरक्षा परिषद के पांच स्थाई सदस्यों और जर्मनी के साथ हस्ताक्षर किया था. पिछले साल अमरीकी राष्ट्रपति डोनल

In [208]:

corrected_translation = """Rouhani has said that the new US sanctions have made Iran's situation worse than it was during the war with Iraq from 1980 to 1988. His remarks came amid rising tensions with the United States, following the deployment of warships and fighter aircraft in the Gulf a few days earlier.

Facing domestic political pressure, Rouhani has called for political unity to confront the sanctions.

Speaking to political activists in Tehran, Rouhani said, "During the war, we did not face difficulties with our banking, oil or imports. The only restriction was on the purchase of weapons."

Rouhani added, "The pressure being imposed by our enemies is a war unprecedented in the history of our Islamic Revolution. But we are not discouraged, and we have better hopes for the future. I believe that if we maintain our unity, we can overcome these difficulties."

Tensions between the United States and Iran have put the 2015 nuclear agreement in doubt. Iran signed the agreement with the five permanent members of the UN Security Council and Germany.

Last year, US President Donald Trump unilaterally withdrew from the agreement and imposed new sanctions on Iran. In response, Iran indicated that it could restart its nuclear programme if the other countries involved in the agreement also supported the US sanctions.

**What kind of pressure is Iran facing?**

After the United States withdrew from the agreement, pressure on President Rouhani from hard-line groups in Iran increased, because the agreement had been reached during his presidency.

Under the agreement, Iran had agreed to limit its sensitive nuclear activities and allow international inspectors into the country. In return, sanctions against Iran were eased.

US sanctions on the energy, shipping and financial sectors have affected Iran's oil exports and reduced foreign investment to a minimum. These sanctions prevent US companies from doing business with Iran and also restrict companies that trade directly with Iran.

The International Monetary Fund (IMF) has estimated that Iran's economy will shrink by 6% in 2019. However, this estimate could become even worse because the United States has tightened its sanctions further.

For example, the United States had previously allowed China, India, Japan, South Korea and Turkey to continue buying Iranian oil. That exemption has now been ended.

Last month, the United States placed Iran's elite Islamic Revolutionary Guard Corps on its list of foreign terrorist organisations.

In response to US sanctions, Iran has repeatedly threatened to disrupt trade through the Strait of Hormuz, a sea route through which around 20% of the world's oil supply passes.

**What is the United States doing?**

The Trump administration wants to pressure Iran into accepting a new agreement that would cover not only its nuclear programme but also its ballistic missile programme, which the United States considers a cause of tensions in the Middle East.

The United States is sending its Patriot missile defence system to the Middle East. It has also sent the warship USS Arlington to the Gulf. The ship carries equipment for amphibious operations and aircraft. It will be deployed in the Gulf alongside the USS Abraham Lincoln.

B-52 bombers have also been sent to a US military base in Qatar.

The United States has said these steps were taken to protect its forces in the region against possible threats from Iran. Iran has rejected the US claim.

At present, 5,200 US troops are stationed in neighbouring Iraq.

Earlier this week, Iran announced that it was withdrawing from two of its commitments under the nuclear agreement. It threatened that if sanctions were not eased within 60 days, it would increase uranium production.

European powers have said that they remain committed to the Iran nuclear agreement, but they have rejected any threats from Tehran that could endanger the agreement.

(Click here for BBC Hindi's Android app. Follow us on Facebook, Twitter, Instagram and YouTube.)"""

article_id = "international-48243735"

source_250.loc[
    source_250["id"] == article_id,
    "english_article"
] = corrected_translation

source_250.loc[
    source_250["id"] == article_id,
    "translation_quality_status"
] = "Corrected - needs final review"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("Corrected translation saved:", article_id)
print("Status: Corrected - needs final review")
print("File:", staging_path)


Corrected translation saved: international-48243735
Status: Corrected - needs final review
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/FactIndic_Source_250_Staging.csv


In [209]:
row = source_250.loc[
    source_250["id"] == "150816_badminton_saina_final_dp"
].iloc[0]

print("HINDI ORIGINAL:\n")
print(row["hindi_article"])

print("\nCURRENT ENGLISH:\n")
print(row["english_article"])

HINDI ORIGINAL:

इंडोनेशिया के जकार्ता में साइना को स्पेन की कैरोलिना मारिन से लगातार गेमों में 16-21 और 19-21 से हराया. 59 मिनट तक चले इस मुक़ाबले में दुनिया की दूसरे नंबर की खिलाड़ी साइना ने दोनों गेम में अच्छी शुरुआत की लेकिन इसके बाद उनका खेल बिखर गया. साइना विश्व चैंपियनशिप के पहुंचने वाली पहली भारतीय खिलाड़ी हैं लेकिन वह स्वर्ण पदक जीतने से वंचित रह गईं. विश्व चैंपियनशिप में भारत का यह पांचवां पदक है. समाप्त पदक इससे पहले, पी वी सिंधु ने 2013 और 2014 में इस टूर्नामेंट का कांस्य पदक हासिल किया था जबकि 2011 में ज्वाला गट्टा और अश्विनी पोनप्पा की जोड़ी ने डबल्स का कांस्य पदक जीता था. 1983 में प्रकाश पादुकोण वर्ल्ड चैंपियनशिप में कांस्य पदक जीतने वाले पहले भारतीय बने थे. साइना ने मैच में अच्छी शुरुआत करते हुए पहले गेम में 3-1 की बढ़त बना ली थी लेकिन इसके बाद उनका खेल बिखर गया. दुनिया की पहले नंबर की खिलाड़ी मारिन ने 11-5 की बढ़त बनाने के बाद पीछे मुड़कर नहीं देखा और पहला गेम 21-16 से जीत लिया. कड़ा मुक़ाबला दूसरे गेम की शुरुआत में साइना ने एक बार फिर बेहतर शुरुआत करते हुए 12-6 की बढ़

In [210]:

corrected_translation = """Saina Nehwal was defeated by Spain's Carolina Marin in straight games, 16-21 and 19-21, in Jakarta, Indonesia.

The match lasted 59 minutes. World number two Saina made a good start in both games, but her performance declined as the match progressed.

Saina became the first Indian player to reach the final of the World Championships, but she missed out on the gold medal. This was India's fifth medal at the World Championships.

Before this, P. V. Sindhu had won bronze medals at the tournament in 2013 and 2014. In 2011, the women's doubles pair of Jwala Gutta and Ashwini Ponnappa had also won a bronze medal. In 1983, Prakash Padukone became the first Indian to win a bronze medal at the World Championships.

**The match**

Saina started well and took a 3-1 lead in the first game. However, her performance then declined. World number one Carolina Marin took an 11-5 lead and did not look back, winning the first game 21-16.

**A close contest**

Saina once again made a strong start in the second game, taking a 12-6 lead. But Marin fought back by winning points consecutively and brought the score to 13-12.

The two players then fought hard for every point. At one stage, Saina was leading 18-17, but Marin won three consecutive points to take a 20-18 lead.

Marin eventually won the game 21-19 and secured the gold medal.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)"""

article_id = "150816_badminton_saina_final_dp"

source_250.loc[
    source_250["id"] == article_id,
    "english_article"
] = corrected_translation

source_250.loc[
    source_250["id"] == article_id,
    "translation_quality_status"
] = "Corrected - needs final review"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("Corrected translation saved:", article_id)
print("Status: Corrected - needs final review")
print("File:", staging_path)


Corrected translation saved: 150816_badminton_saina_final_dp
Status: Corrected - needs final review
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/FactIndic_Source_250_Staging.csv


In [211]:
row = source_250.loc[
    source_250["id"] == "international-49451047"
].iloc[0]

print("HINDI ORIGINAL:\n")
print(row["hindi_article"])

print("\nCURRENT ENGLISH:\n")
print(row["english_article"])

HINDI ORIGINAL:

यह सम्मान पाने वाले वो पहले भारतीय प्रधानमंत्री बने. 'ऑर्डर ऑफ़ ज़ायेद' यूएई का सर्वोच्च नागरिक सम्मान है जो बादशाहों, राष्ट्रपति और राष्ट्र प्रमुखों को दिया जाता है. इससे पहले साल 2007 में रूस के राष्ट्रपति व्लादिमीर पुतिन, 2010 में ब्रिटेन की महारानी एलिज़ाबेथ, 2016 में सऊदी के शाह सलमान बिन अब्दुल्ला अज़ीज़ अल सऊद और साल 2018 में चीन के राष्ट्रपति शी जिनपिंग को यह सम्मान दिया गया था. यूएई ने इस सम्मान की शुरुआत साल 1995 में की थी. पीएम मोदी को इस सम्मान के लिए क्यों चुना गया? अबू धाबी के क्राउन प्रिंस शेख़ मोहम्मद बिन ज़ायेद अल नाह्यान के मुताबिक, भारत और यूएई के बीच द्विपक्षीय संबंधों को बढ़ाने में पीएम मोदी ने महत्वपूर्ण योगदान दिया है. इस साल अप्रैल में शेख़ मोहम्मद बिन ज़ायेद अल नाह्यान ने एक ट्वीट के माध्यम से इस सम्मान की घोषणा की थी. उन्होंने लिखा था, "भारत के साथ हमारे ऐतिहासिक और व्यापक रणनीतिक संबंध हैं, जिसके पीछे मेरे परम मित्र नरेंद्र मोदी का बहुत योगदान है. उन्होंने इन रिश्तों को और मज़बूत किया है. उनके प्रयासों को देखते हुए यूएई के राष्ट्रपति शेख़ ख़ल

In [212]:

corrected_translation = """Narendra Modi became the first Indian Prime Minister to receive this honour. The Order of Zayed is the UAE's highest civilian honour and is awarded to kings, presidents and heads of state.

Before Modi, the award had been given to Russian President Vladimir Putin in 2007, Britain's Queen Elizabeth in 2010, Saudi King Salman bin Abdulaziz Al Saud in 2016, and Chinese President Xi Jinping in 2018. The UAE established this award in 1995.

**Why was Modi chosen for the honour?**

According to Abu Dhabi Crown Prince Sheikh Mohammed bin Zayed Al Nahyan, Prime Minister Modi has made an important contribution to strengthening bilateral relations between India and the UAE.

In April this year, Sheikh Mohammed bin Zayed Al Nahyan announced the award in a tweet. He wrote, "We have historic and comprehensive strategic relations with India, and my very dear friend Narendra Modi has made a major contribution to them. He has further strengthened these relations. In recognition of his efforts, UAE President Sheikh Khalifa bin Zayed Al Nahyan has awarded him the Order of Zayed."

In a statement, India's Ministry of External Affairs said that this year's award was of "special significance" because it was being presented to Prime Minister Modi during the birth centenary year of Sheikh Zayed.

The UAE has declared 2019 the Year of Tolerance. The statement also said that the UAE is India's third-largest trading partner. The two countries have strong bilateral trade relations, with annual trade expected to reach around 60 billion dollars.

**India and the UAE**

During Modi's tenure, there has been an unprecedented increase in diplomatic visits between the leaders of the two countries.

Sheikh Mohammed bin Zayed has visited India twice in the last three years. His first visit was in February 2016. On his second visit, in 2017, he attended India's Republic Day celebrations as the chief guest.

Modi is the first Indian Prime Minister to have visited the UAE twice. He first visited the country in August 2015. In 2018, the UAE invited Narendra Modi as the chief guest at the World Government Summit. This was his third visit in less than five years.

During this visit, Indian Prime Minister Narendra Modi met the Crown Prince. They discussed bilateral, regional and international matters of mutual interest.

The Indian Embassy in Abu Dhabi also issued a commemorative postage stamp on the occasion of the 150th birth anniversary of Mahatma Gandhi, in the presence of Prime Minister Modi.

In addition, Modi launched the RuPay card there. Like MasterCard and Visa, RuPay is India's own card-based payment system.

India's Ambassador to the UAE, Navdeep Suri, said, "The Order of Zayed is certainly one reason for the Prime Minister's visit, but he also wants to personally promote the RuPay card."

**Pakistan's objection**

The news that Narendra Modi was to receive the UAE's highest honour came as a surprise to Pakistan. This came amid the Modi government's move to make the provisions of Article 370, which gave Jammu and Kashmir special status, ineffective, and allegations of human rights violations in Indian-administered Kashmir.

Dr Umbreen Javed, a political science expert in Lahore, told the BBC, "Although Narendra Modi has visited a country with which India has many economic ties, this has become an emotional issue for Pakistan because of the Kashmir issue."

Dr Javed said, "This is a time when Pakistan believes that human rights violations in Indian-administered Kashmir could increase. Kashmiris may face considerable difficulties in the future."

Pakistanis had hoped that Muslim countries would try to persuade Indian Prime Minister Narendra Modi to reverse his decision on Article 370, rather than honouring him with their highest civilian award.

Recently, British Labour Party MP Naz Shah had written a letter to the UAE Crown Prince, asking him to reconsider awarding the Zayed Medal to Indian Prime Minister Narendra Modi. She had accused Modi of human rights violations in Kashmir.

However, the Pakistani government has remained silent on the matter. According to Pakistan Foreign Ministry spokesperson Dr Mohammad Faisal, this is a matter between India and the UAE, and Pakistan would not comment on it.

Since the special status of Jammu and Kashmir under Article 370 of the Indian Constitution was revoked, hashtags related to Kashmir and Prime Minister Narendra Modi have been trending on Twitter.

Some people on social media also expressed unhappiness over Prime Minister Modi receiving the UAE's highest honour.

Twitter user Tahir Naqvi wrote, "By giving Modi its highest honour, the Arab world is merely endorsing the situation."

However, another Twitter user, Imaan Hazir Mazari, said she was not surprised. She wrote, "I cannot understand how or why people are surprised by the UAE's response to Kashmir being unlawfully merged into India. There is no such thing as a Muslim community, as I have said many times before. The UAE and Saudi Arabia are always the first countries to support the killing of Muslims."

(Click here for BBC Hindi's Android app. Follow us on Facebook, Twitter, Instagram and YouTube.)"""

article_id = "international-49451047"

source_250.loc[
    source_250["id"] == article_id,
    "english_article"
] = corrected_translation

source_250.loc[
    source_250["id"] == article_id,
    "translation_quality_status"
] = "Corrected - needs final review"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("Corrected translation saved:", article_id)
print("Status: Corrected - needs final review")
print("File:", staging_path)


Corrected translation saved: international-49451047
Status: Corrected - needs final review
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/FactIndic_Source_250_Staging.csv


In [213]:
row = source_250.loc[
    source_250["id"] == "international-54702261"
].iloc[0]

print("HINDI ORIGINAL:\n")
print(row["hindi_article"])

print("\nCURRENT ENGLISH:\n")
print(row["english_article"])

HINDI ORIGINAL:

हमला पाकिस्तान में ख़ैबर-पख़्तूनख़्वाह प्रांत के पेशावर शहर में हुआ. अस्पताल के अधिकारियों के मुताबिक़, मरने वालों की उम्र 20 से 30 साल के बीच है. दर्जनों लोग घायल हैं जिनमें से चार की उम्र 13 साल से कम है. शुरुआती रिपोर्टों में कहा गया था कि मरने वालों में चार बच्चे हैं, लेकिन बाद में ऐसी रिपोर्टों को वापस ले लिया गया. अभी तक किसी समूह ने हमले की ज़िम्मेदारी नहीं ली है. मामले की जांच शुरू कर दी गई है. समाप्त पाकिस्तान के प्रधानमंत्री इमरान ख़ान ने धमाके की कड़ी निंदा की है और जान के नुक़सान पर दुख जताया है. पुलिस ने बीबीसी को बताया कि धमाका स्थानीय समयानुसार सुबह 8:30 पर हुआ. पुलिस के मुताबिक़ धमाका उस वक़्त हुआ जब मदरसे में पढ़ाई चल रही थी. माना जा रहा है कि धमाके के वक़्त मदरसे में क़रीब 60 लोग मौजूद थे. एक चश्मदीद ने पुलिस को बताया कि उन्होंने धमाके से कुछ देर पहले एक शख़्स को विस्फोटकों से भरा बैग इमारत में ले जाते देखा था. एक वरिष्ठ पुलिस अधिकारी वकार अज़ीम में समाचार एजेंसी एएफ़पी से कहा कि "कोई पाठशाला के अंदर बस्ता रखकर चला गया." ख़ैबर-पख़्तूनख़्वाह के प्रांती

In [214]:

corrected_translation = """The attack took place in Peshawar, a city in Pakistan's Khyber Pakhtunkhwa province. According to hospital officials, those killed were between 20 and 30 years old. Dozens of people were injured, including four children under the age of 13. Initial reports said that four children were among those killed, but those reports were later withdrawn. No group has so far claimed responsibility for the attack, and an investigation has begun.

Pakistan's Prime Minister Imran Khan strongly condemned the blast and expressed sorrow over the loss of life.

Police told the BBC that the explosion occurred at 8:30 a.m. local time. According to police, the blast took place while classes were being held at the madrassa. Around 60 people are believed to have been inside at the time.

An eyewitness told police that shortly before the explosion, they had seen a man carrying a bag full of explosives into the building. Senior police officer Waqar Azeem told the AFP news agency, "Someone left a schoolbag inside the school and walked away."

Khyber Pakhtunkhwa provincial minister Taimur Jhagra told reporters that seven people had been killed and more than 70 injured in the blast. Children were among the injured, he said. Police told AFP that two teachers had also been injured.

Photographs from inside the madrassa showed that the explosion had been very powerful. Assistant Inspector General of Police (AIG) Shafqat Malik told Pakistan's Express Tribune newspaper that the initial investigation indicated that five kilograms of explosives had been used in the attack.

Hospital officials told Reuters that dozens of injured people had been brought to the hospital, many of them suffering from burns.

Peshawar is located near the Afghan border. The city has witnessed several serious incidents of violence in recent years during the Taliban insurgency. Six years ago, a gunman entered a military school in the city and killed more than 150 people, including children.

(Click here for BBC Hindi's Android app. Follow us on Facebook, Twitter, Instagram and YouTube.)"""

article_id = "international-54702261"

source_250.loc[
    source_250["id"] == article_id,
    "english_article"
] = corrected_translation

source_250.loc[
    source_250["id"] == article_id,
    "translation_quality_status"
] = "Corrected - needs final review"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("Corrected translation saved:", article_id)
print("Status: Corrected - needs final review")
print("File:", staging_path)


Corrected translation saved: international-54702261
Status: Corrected - needs final review
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/FactIndic_Source_250_Staging.csv


In [215]:
row = source_250.loc[
    source_250["id"] == "india-39165960"
].iloc[0]

print("HINDI ORIGINAL:\n")
print(row["hindi_article"])

print("\nCURRENT ENGLISH:\n")
print(row["english_article"])

HINDI ORIGINAL:

यही तो पहचान है धर्मनगरी काशी की. हज़ारों मंदिरों के इस शहर में बड़ी-बड़ी गाड़ियों का शोर, हवा में कुलांचे भरते हेलीकॉप्टर और जगह-जगह बने ऊंचे मंचों ने यहां लोगों की व्यस्तता को और बढ़ा दिया है. कहते हैं यूपी के कण-कण में सियासत भरी है लोगों के रगों में तो सियासी बहसें लहू बनकर दौड़ती हैं. यूपी चुनाव में बसपा बड़ी ताकत वाराणसी में बीजेपी दिग्गजों का जमावड़ा क्यों? मौका भी है, दस्तूर भी है और मिज़ाज भी. शहर में थोड़ा घूमने निकल जाइए तो पता चल जाएगा कि हर गली, हर नुक्कड़ और हर चौक-चौराहे पर लोगों का कम से कम एक समूह तो बनारसी पान की मिठास के बीच राजनीतिक बहस में डूबा हुआ है और यह बहस कभी-कभी तो इस पान के मिठास को थोड़ा कड़वा भी बना देता है. राजनीतिक मिज़ाज वाला प्रदेश बहरहाल अभी ऐसा लगता है कि जिस तरह बनारस की एक पहचान देवों के देव महादेव से है, उस तरह इसकी एक पहचान सियासी बतकही भी बन गई हो. सच ही तो है क्योंकि यह वो जमीं है जहां राजनीतिक चर्चा शुरू तो होती है छोटे दलों या नेताओं से पर वो जाती है दूर तलक या कहें कि डोनल्ड ट्रंप तक. योगी-मुख़्तार के पूर्वांचल में पार्टियो

In [216]:

corrected_translation = """This is what makes Kashi, the holy city, distinctive. In this city of thousands of temples, the noise of large vehicles, helicopters flying overhead and tall stages set up at various places have made the streets even busier. It is often said that every corner of Uttar Pradesh is steeped in politics and that political debates run through people's veins like blood.

**The BSP is a major force in the Uttar Pradesh elections. Why are so many BJP heavyweights gathering in Varanasi?**

There is an occasion, a tradition and a particular political mood. Take a walk around the city and you will find groups of people in almost every street, at every corner and every crossroads, immersed in political discussions while enjoying the sweetness of Banarasi paan. Sometimes, those debates even make the paan taste a little bitter.

Uttar Pradesh is a state with a strong political temperament. Just as Banaras is identified with Mahadev, the great god Shiva, political chatter has also become part of the city's identity. Indeed, this is a place where political discussions may begin with smaller parties or local leaders but soon travel much further—even as far as Donald Trump.

**The parties face a test in the Purvanchal region of Yogi and Mukhtar**

There may never be a better opportunity to understand Uttar Pradesh's political mood and sharpen one's political judgement. Over these five days, the journey began in Lucknow, continued to Gorakhpur and then reached Banaras. It became clear that this time voters in Uttar Pradesh might judge their politicians on development and a sense of connection with the people. This suggests that caste equations may shift in a different direction rather than favouring any single party.

Uttar Pradesh is the country's most populous state. Clearly, reaching a sound conclusion requires examining the different political parties. Let us begin with the SP-Congress alliance. But before discussing the alliance, it is worth looking back at how and when political equations in Uttar Pradesh have changed.

Around two months ago, it seemed that the Samajwadi Party had been completely pushed out of the contest, leaving a direct fight between the BSP and the BJP.

**'UP does not like the Pappu-Gappu duo'**

The reason was the internal conflict within the Samajwadi Party. However, there is no doubt that Akhilesh demonstrated considerable political judgement during the period of discord. He gradually brought the conflict under control and also created a favourable political position for himself.

Until then, it was widely believed that Akhilesh was merely the party's chief ministerial face and that several other leaders were exercising power behind the scenes. But Akhilesh overturned that perception through his political skill. By presenting himself as a young leader focused on development, he managed to turn the atmosphere in his favour.

He succeeded in convincing people that he was not only the party's most prominent leader but also the strongest supporter of its caste-based political coalition.

**The complications of the SP-Congress alliance**

Politics soon took another turn, and the SP and Congress formed an alliance. After the announcement, many began to believe that this alliance was the strongest contender for victory in the election.

The perception was that the SP already had the support of Yadav and Muslim voters, while the alliance would also bring traditional Congress voters into its fold. Its impact was indeed visible during the initial phases. However, that atmosphere did not last long, and the BJP began making a comeback.

We will discuss the BJP's strategies later. First, let us examine the reasons why the Samajwadi Party appears to be losing ground in this election.

The deep internal conflict that affected the Samajwadi Party before the election damaged its public image. More importantly, Mulayam Singh himself became upset.

**The impact on Muslim voters**

Until now, Mulayam Singh's political skill in bringing together Muslim and Dalit voters had been one of the major strengths of the Samajwadi movement in Uttar Pradesh. But the situation seems different this time. Mulayam's absence from the election campaign appears to be dividing the SP's support base, and the party seems to lack the energy that his presence once provided.

**When the BJP helped Mulayam form a government in UP**

Muslim voters appear to be the group most affected, as a section of them may move towards the BSP. The young leadership of the alliance could help make up for this shortfall. However, there is also concern that traditional Congress voters may become confused by the alliance and shift towards the BSP or the BJP.

**The BSP!**

The BSP had hoped that an alliance of Dalit and Muslim voters would carry it to victory and that its elephant symbol would roar. But the division of Muslim votes is hurting both the BSP and the SP-Congress alliance while benefiting the BJP.

Nevertheless, the crowds attending Mayawati's rallies show that the BSP still has a genuine opportunity in this election. Mayawati has indeed been working on her Dalit-Muslim agenda for several months.

**Mayawati's confidence**

Mayawati believes that Muslim voters shifting away from the SP could help her win. By giving tickets to 97 Muslim candidates, the BSP has made every effort to unsettle the SP-Congress alliance.

But will the elephant's confident stride be enough to defeat the BJP? That is the big question.

The BJP is putting everything it has into capturing power in Uttar Pradesh. Its strategy includes capitalising on Prime Minister Narendra Modi's popularity, relying on the political partnership of Modi and Amit Shah, bringing smaller parties such as Apna Dal into its alliance, holding a series of rallies, attacking the opposition and promising voters a dramatic change. It is also using caste calculations to isolate its opponents.

**How much do Muslims trust Mayawati?**

The BJP is trying to take advantage of internal divisions in the Samajwadi Party's strongholds of Etawah and Mainpuri. It is also seeking to create an anti-incumbency mood against Akhilesh's ministers and MLAs. By giving tickets to non-Yadav OBC candidates and non-Jatav Dalit candidates, it hopes to break into the traditional vote banks of both the SP and the BSP.

This strategy is giving the BJP a strong challenge against the SP-Congress alliance and the BSP. In the changed political environment, the BJP appears to be moving ahead, while the SP-Congress alliance and the BSP seem to be damaging each other's prospects. The BJP appears to be benefiting from this.

(These are the author's personal views.)

(Click here for BBC Hindi's Android app. You can also follow us on Facebook, Twitter, Instagram and YouTube.)"""

article_id = "india-39165960"

source_250.loc[
    source_250["id"] == article_id,
    "english_article"
] = corrected_translation

source_250.loc[
    source_250["id"] == article_id,
    "translation_quality_status"
] = "Corrected - needs final review"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("Corrected translation saved:", article_id)
print("Status: Corrected - needs final review")
print("File:", staging_path)


Corrected translation saved: india-39165960
Status: Corrected - needs final review
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/FactIndic_Source_250_Staging.csv


In [217]:
row = source_250.loc[
    source_250["id"] == "140524_iran_hanging_aa"
].iloc[0]

print("HINDI ORIGINAL:\n")
print(row["hindi_article"])

print("\nCURRENT ENGLISH:\n")
print(row["english_article"])

HINDI ORIGINAL:

मुहाफरीद आमिर खोसरावी को तेहरान की एविन जेल में शनिवार को फांसी दी गई. आमिर खोसरावी 2011 में सामने एक घोटाले में दोषी करार दिए गए थे. उन पर फर्जी दस्तावेजों के आधार पर कर्ज लेने के आरोप लगे, जिसमें बाद में 2.6 अरब डॉलर की धांधली सामने आई. इस मामले के चलते उस समय की सरकार को कहना पड़ा कि इसमें उसके सदस्य शामिल नहीं हैं. ये मामला सितंबर 2011 में सामने आया जब एक निवेश कंपनी पर आरोप लगा कि उसने चार साल के दौरान जाली दस्तावेजों के आधार पर कई बैंकों से कर्ज लिया है. बताया जाता है कि इस धन का इस्तेमाल सरकार की निजीकरण की मुहिम के तहत सरकारी कंपनियों को खरीदने के लिए किया गया. मानवाधिकार समूह ईरान में फांसी की सजा दिए जाने का विरोध करते रहे हैं. उनका कहना है कि ये दुनिया में सबसे सख़्त सज़ा है. (बीबीसी हिन्दी के एंड्रॉएड ऐप के लिए यहां क्लिक करें. आप हमें फ़ेसबुक और ट्विटर पर भी फ़ॉलो कर सकते हैं.)

CURRENT ENGLISH:

An Iranian businessman convicted in a major bank fraud case has been hanged at Evin Prison in Tehran.

Mahafarid Amir Khosravi was executed on Saturday. He was co

In [218]:

corrected_translation = """Mahafarid Amir Khosravi was hanged at Evin Prison in Tehran on Saturday.

Amir Khosravi had been convicted in 2011 in connection with a major banking fraud scandal. He was accused of obtaining loans using forged documents. The fraud was later found to involve $2.6 billion.

The case came to light in September 2011, when an investment company was accused of obtaining loans from several banks over a period of four years using forged documents.

The money was reportedly used to purchase state-owned companies as part of the government's privatisation programme.

The scandal forced the government of the time to state that none of its members was involved in the case.

Human rights organisations have repeatedly opposed the use of the death penalty in Iran. They say that the death penalty is the harshest punishment in the world.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)"""

article_id = "140524_iran_hanging_aa"

source_250.loc[
    source_250["id"] == article_id,
    "english_article"
] = corrected_translation

source_250.loc[
    source_250["id"] == article_id,
    "translation_quality_status"
] = "Corrected - needs final review"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("Corrected translation saved:", article_id)
print("Status: Corrected - needs final review")
print("File:", staging_path)


Corrected translation saved: 140524_iran_hanging_aa
Status: Corrected - needs final review
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/FactIndic_Source_250_Staging.csv


In [219]:
row = source_250.loc[
    source_250["id"] == "india-43796841"
].iloc[0]

print("HINDI ORIGINAL:\n")
print(row["hindi_article"])

print("\nCURRENT ENGLISH:\n")
print(row["english_article"])

HINDI ORIGINAL:

लेकिन अगर आंकड़ों पर नज़र डालें तो इस साल की अक्षय तृतीया पर सोने की कीमतें सबसे ज़्यादा बढ़ सकती हैं. बीते 11 अप्रैल को 24 कैरेट सोने की कीमत 31,524 रुपये प्रति दस ग्राम का आंकड़ा पार कर गई थी. हालांकि, 17 अप्रैल को सोने की कीमत में थोड़ी गिरावट देखी गई. लेकिन विशेषज्ञों के मुताबिक़ खरीदार लगातार सोना खरीदना जारी रखेंगे. सोना पांच साल में सबसे सस्ता हुआ बिटक्वाइन के आगे पहली बार सोने की चमक फीकी सोने के दाम क्यों बढ़ रहे हैं? कमोडिटीज़ एंड करेंसी रिसर्च फर्म निर्मल बंग के प्रमुख कुणाल शाह कहते हैं, "मौद्रिक नीति को लेकर अनिश्चितता, भूराजनीतिक संकट से जुड़ी अनिश्चितताएं और पश्चिमी अर्थव्यवस्थाओं में महंगाई के बढ़ने से जुड़ी आशंकाओं के चलते मार्च 2018 के अंत तक सोना वैश्विक स्तर पर $1450 पर खरीदा जाएगा लेकिन मार्च 2019 से पहले ये $1600 का स्तर छू लेगा." सोने को हमेशा ही निवेश के लिहाज से एक बढ़िया विकल्प माना जाता है. ये वैश्विक अनिश्चितताओं का साल है, अमरीका चीन के साथ ट्रेड वॉर करने की धमकी दे रहा है, सीरिया में तनाव बढ़ रहा है. ऐसे में निवेशक अपना पैसा किसी अन्य जगह 

In [220]:

corrected_translation = """However, the figures suggest that gold prices could see their biggest rise around Akshaya Tritiya this year. On April 11, the price of 24-carat gold crossed Rs 31,524 per 10 grams. Although gold prices fell slightly on April 17, experts believe that buyers will continue purchasing gold.

Gold becomes the cheapest it has been in five years

For the first time, gold's shine fades in comparison with Bitcoin

Why are gold prices rising?

Kunal Shah, head of the commodities and currency research firm Nirmal Bang, says, "Because of uncertainty over monetary policy, uncertainties related to geopolitical crises and concerns about rising inflation in Western economies, gold will be trading at around $1,450 globally by the end of March 2018. It will reach the $1,600 level before March 2019."

Gold has traditionally been considered a good investment option. This is a year of global uncertainty. The United States is threatening a trade war with China, and tensions are rising in Syria. In such circumstances, investors are cautious about putting their money into other assets. As a result, they are choosing to invest in gold, pushing up prices around the world. Countries such as India will also be affected.

The unusual ways in which gold is smuggled

What is the situation in India?

Gold demand in India declined after demonetisation and the introduction of GST. In March, gold imports fell by 40% compared with the previous year, to $2.4 billion. This had a negative impact on the industry.

However, demand for gold is now expected to recover. Better monsoon conditions have improved farmers' incomes, which could increase gold demand during the festive season. Two-thirds of the country's total gold demand comes from rural India.

A look at gold prices around Akshaya Tritiya over the past few years suggests that 2018 could be the most expensive year in this regard. In 2010, when Akshaya Tritiya fell on May 16, 24-carat gold could be bought for Rs 18,167 per 10 grams. Last year, the price had risen to Rs 29,890 per 10 grams.

Source: Goldpriceindia.com (Prices per 10 grams)

The right time to buy gold

The positive side is that anyone who bought gold earlier would have seen the value of their investment increase. The question, then, is whether this is the right time to buy gold. Experts are divided on the issue.

Gnanasekar Thyagarajan, director of Comtrendz Research, says, "When gold reaches the Rs 30,000 mark, people tend to think about selling it. But prices are now stabilising."

"I believe gold prices will rise by the end of this year. Therefore, buying gold on this Akshaya Tritiya could be a good decision, because prices are likely to rise from this point onwards."

Vandana Bharti of SMC Global Securities told the BBC, "The market situation is different at the moment, so I would recommend waiting for a few months. September and October may be better times to buy, as prices could fall by 5-6%."

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)"""

article_id = "india-43796841"

source_250.loc[
    source_250["id"] == article_id,
    "english_article"
] = corrected_translation

source_250.loc[
    source_250["id"] == article_id,
    "translation_quality_status"
] = "Corrected - needs final review"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("Corrected translation saved:", article_id)
print("Status: Corrected - needs final review")
print("File:", staging_path)


Corrected translation saved: india-43796841
Status: Corrected - needs final review
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/FactIndic_Source_250_Staging.csv


In [221]:
row = source_250.loc[
    source_250["id"] == "090216_jobloss_column3_alk"
].iloc[0]

print("HINDI ORIGINAL:\n")
print(row["hindi_article"])

print("\nCURRENT ENGLISH:\n")
print(row["english_article"])

HINDI ORIGINAL:

नौकरी रहते हुए नई नौकरी की तलाश तो फिर भी आसान है, लेकिन सड़क पर आकर नौकरी ढूँढना काफ़ी मुश्किल. इसीलिए मैंने अपने एचआर वालों से एक विनती भी की थी, तीन महीने का नोटिस पीरियड दे दो, सैलरी बेशक मत दो लेकिन नई नौकरी ढूँढने का मौका दे दो. पर ऐसा नहीं हुआ. दरअसल मेरे ऑफ़िस वालों को पता था कि मेरे पापा रिटायर्ड जज हैं. पता नहीं क्यों इस बात को लेकर उनके मन में संशय भी था कि मैं उन्हें अदालत में ना घसीट लूँ. इसलिए मेरे मामले में इस्तीफ़े से संबंधित सभी कागज़ी कार्रवाई को वो हू-बहू अमल में भी ला रहे थे. मुझे कोई रियायत नहीं दी गई. नौकरी ढूँढने का सिलसिला तो दो दिनों बाद ही शुरु हो गया. शुरुआत मित्रों को बताने और अपने ही सर्किल में कोई नौकरी खोजने की कोशिश के साथ हुई. 'सेट' कराने के आश्वासन शुरु-शुरु में कुछ दोस्तों ने अपने यहाँ 'सेट' करवा देने का आश्वासन दिया मगर 15 दिन बीत गए और कहीं कुछ नहीं बना. सारे दोस्तों ने कुछ दिन और इंतज़ार करने की सलाह दी या कुछ का कहना था.. "अरे अभी पाँच दिन पहले एक की गुंजाइश थी. यार थोड़ा लेट हो गए.." मैं माजरा समझ ही चुका था, सो ख़ुद दफ़्तरों के 

In [222]:

corrected_translation = """Finding a new job while you are still employed is relatively easy, but looking for work after becoming unemployed is much more difficult. That is why I had requested my HR department to give me a three-month notice period. I said they did not have to pay me a salary during that time, but at least they could give me an opportunity to find another job. However, this did not happen.

My office colleagues knew that my father was a retired judge. I do not know why, but they seemed worried that I might take them to court. Therefore, they followed every formal procedure related to my resignation exactly as required. I was given no special consideration.

I started looking for a job just two days later. I began by informing my friends and trying to find a position within my own circle. Initially, some friends assured me that they would help me get a job at their workplaces. However, 15 days passed and nothing worked out. All my friends advised me to wait a few more days. Some said, "There was an opening just five days ago. You were a little late." I had understood the situation, so I started visiting offices myself.

Before doing that, I registered on a few job websites. After that, waiting was the only thing I could do. Ten days later, I received a call from a placement agency.

For some reason, except during the early days of my career, I had never felt so nervous before an interview. Perhaps this is one of the side effects of having to look for a job after becoming unemployed.

I went there expecting an interview, but instead I was handed a written test paper. To be honest, at this stage of my career, I felt embarrassed about having to take a written test. But I had no choice.

The interview is fine, but...

When I returned home, I kept wondering whether I would be successful. For the first time, I was afraid of failing. And that is exactly what happened.

Before leaving, I had asked when the result would be announced. I was told that I would know within two or three days. But by the morning of the fourth day, I had run out of patience and called them myself.

The reply was, "We haven't called you... that means they must have selected someone else."

I said that my interview had gone well. They replied, "Yes, your interview was fine, but your profile did not match."

After that, I tried to prove myself at several other offices, but I could not succeed. My search is still continuing.

(Ajay's account was based on a conversation with BBC correspondent Alok Kumar. From next Friday, Vikas Shankar, who was a manager at a private bank and also lost his job, will share his story.)"""

article_id = "090216_jobloss_column3_alk"

source_250.loc[
    source_250["id"] == article_id,
    "english_article"
] = corrected_translation

source_250.loc[
    source_250["id"] == article_id,
    "translation_quality_status"
] = "Corrected - needs final review"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("Corrected translation saved:", article_id)
print("Status: Corrected - needs final review")
print("File:", staging_path)


Corrected translation saved: 090216_jobloss_column3_alk
Status: Corrected - needs final review
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/FactIndic_Source_250_Staging.csv


In [223]:
row = source_250.loc[
    source_250["id"] == "130912_nepal_strike_ssr"
].iloc[0]

print("HINDI ORIGINAL:\n")
print(row["hindi_article"])

print("\nCURRENT ENGLISH:\n")
print(row["english_article"])

HINDI ORIGINAL:

नेपाल में हड़ताल के चलते सड़कें सूनी दिखीं सरकार द्वारा नवंबर में चुनाव कराने की योजना के विरोध में ये हड़ताल की गई. स्कूल और बाजार बंद हैं, सड़कें सूनी पड़ी हैं और वाहन चलते नहीं दिखे. आह्वान ये हड़ताल नेपाल में 33 पार्टियों के गठबंधन द्वारा बुलाई गई थी, जिसमें नेपाल कम्युनिस्ट पार्टी - माओवादी भी शामिल है. मार्च में एक सीनियर जज की अगुवाई वाली सरकार को चुनाव कराने के लिए नियुक्त किया गया था. नेपाल सुप्रीम कोर्ट के मुख्य न्यायाधीश खिलराज रेजमी प्रशासन पर ताजा चुनाव 19 नवंबर को कराने की जिम्मेदारी है. नेपाल में आखिरी बार संसदीय चुनाव 2008 में कराए गए थे. नये संविधान का मसौदा तय करने के बाद संवैधानिक संसद का कार्यकाल मई 2012 में खत्म हो गया. (बीबीसी हिन्दी के एंड्रॉएड ऐप के लिए यहां क्लिक करें. आप हमें फ़ेसबुक और ट्विटर पर फ़ॉलो भी कर सकते हैं.)

CURRENT ENGLISH:

Streets in Nepal were deserted during a nationwide strike called to protest against the government's plan to hold elections in November. Schools and markets remained closed, roads were empty and vehicles were 

In [224]:

corrected_translation = """Streets in Nepal were deserted during a nationwide strike called to protest against the government's plan to hold elections in November. Schools and markets remained closed, roads were empty and vehicles were not seen on the streets.

The strike was called by an alliance of 33 political parties in Nepal, including the Nepal Communist Party-Maoist.

In March, a government led by a senior judge was appointed to conduct the elections. The administration headed by Nepal's Supreme Court Chief Justice Khil Raj Regmi is responsible for holding fresh elections on 19 November.

The last parliamentary elections in Nepal were held in 2008. The term of the Constituent Assembly ended in May 2012 after the draft of the new constitution had been decided.

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)"""

article_id = "130912_nepal_strike_ssr"

source_250.loc[
    source_250["id"] == article_id,
    "english_article"
] = corrected_translation

source_250.loc[
    source_250["id"] == article_id,
    "translation_quality_status"
] = "Corrected - needs final review"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("Corrected translation saved:", article_id)
print("Status: Corrected - needs final review")
print("File:", staging_path)


Corrected translation saved: 130912_nepal_strike_ssr
Status: Corrected - needs final review
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/FactIndic_Source_250_Staging.csv


In [225]:

# Refresh the translation audit
source_250 = pd.read_csv(staging_path)

print("TOTAL ARTICLES:", len(source_250))
print("\nUPDATED TRANSLATION STATUS:")
print(source_250["translation_quality_status"].value_counts(dropna=False))

print("\nARTICLES STILL NEEDING RETRANSLATION:")
print(
    source_250.loc[
        source_250["translation_quality_status"].astype(str).str.contains(
            "Needs retranslation", case=False, na=False
        ),
        ["id", "translation_quality_status"]
    ].to_string(index=False)
)

print("\nCORRECTED - NEEDS FINAL REVIEW:")
print(
    source_250.loc[
        source_250["translation_quality_status"].astype(str).str.contains(
            "Corrected - needs final review", case=False, na=False
        ),
        "id"
    ].to_string(index=False)
)


TOTAL ARTICLES: 250

UPDATED TRANSLATION STATUS:
translation_quality_status
Initial review - no obvious mismatch            239
Corrected - needs final review                   10
Needs retranslation - source text incomplete      1
Name: count, dtype: int64

ARTICLES STILL NEEDING RETRANSLATION:
                                    id                   translation_quality_status
141013_wusatullah_amitabh_bachchan_rns Needs retranslation - source text incomplete

CORRECTED - NEEDS FINAL REVIEW:
         international-47105506
         international-48243735
150816_badminton_saina_final_dp
         international-49451047
         international-54702261
                 india-39165960
         140524_iran_hanging_aa
                 india-43796841
     090216_jobloss_column3_alk
        130912_nepal_strike_ssr


In [226]:

import pandas as pd
import re

source_250 = pd.read_csv(staging_path)

print("ROWS:", len(source_250))
print("DUPLICATE IDS:", source_250["id"].duplicated().sum())

print("\nMISSING VALUES:")
print(source_250[["id", "hindi_article", "english_article"]].isna().sum())

def has_hindi(text):
    return bool(re.search(r"[\u0900-\u097F]", str(text)))

print(
    "\nENGLISH ARTICLES CONTAINING HINDI SCRIPT:",
    source_250["english_article"].apply(has_hindi).sum()
)

print(
    "\nEMPTY ENGLISH TRANSLATIONS:",
    source_250["english_article"].fillna("").str.strip().eq("").sum()
)

print("\nSTATUS COUNTS:")
print(source_250["translation_quality_status"].value_counts())

print("\nINTEGRITY CHECK COMPLETE")


ROWS: 250
DUPLICATE IDS: 0

MISSING VALUES:
id                 0
hindi_article      0
english_article    0
dtype: int64

ENGLISH ARTICLES CONTAINING HINDI SCRIPT: 0

EMPTY ENGLISH TRANSLATIONS: 0

STATUS COUNTS:
translation_quality_status
Initial review - no obvious mismatch            239
Corrected - needs final review                   10
Needs retranslation - source text incomplete      1
Name: count, dtype: int64

INTEGRITY CHECK COMPLETE


In [227]:

import pandas as pd
import os

article_id = "141013_wusatullah_amitabh_bachchan_rns"

raw_path = os.path.join(
    base_path, "01_raw_data", "XL_Sum_Hindi_Test.parquet"
)

clean_path = os.path.join(
    base_path, "02_clean_data", "XL_Sum_Hindi_Cleaned_250.csv"
)

raw_df = pd.read_parquet(raw_path)
clean_df = pd.read_csv(clean_path)

for label, df in [("RAW DATA", raw_df), ("CLEANED DATA", clean_df),
                  ("STAGING DATA", source_250)]:
    match = df[df["id"].astype(str) == article_id]

    print(f"\n{label}:")
    if len(match):
        article = match.iloc[0]["text"] if "text" in df.columns else match.iloc[0]["hindi_article"]
        print("Article length:", len(str(article)))
        print("Ending:", str(article)[-500:])
    else:
        print("Article ID not found")



RAW DATA:
Article length: 2648
Ending: ा ले गया. राफे ने फ़िल्म देखते-देखते कहा, बाबा क्या आप भूतनाथ की तरह ग़ायब हो सकते हैं. मैं उसे कैसे बताऊँ कि भूतनाथ पिछले चालीस-बयालीस वर्षों से मेरे अंदर ही तो रहता है. बात ये है कि ज़माने के किसी मोड़ पर एक ख़ास लम्हे में कुछ लोग कैमरे को हमेशा पसंद आ जाते हैं. अमिताभ इन्हीं में से हैं. शनिवार को जब उनका 72वां जन्मदिन चढ़ा तो मैंने ख़ुद से कहा, हैप्पी बर्डे टू यू...बुड्ढा होगा तेरा बाप... (बीबीसी हिन्दी के एंड्रॉएड ऐप के लिए यहां क्लिक करें. आप हमें फ़ेसबुक और ट्विटर पर भी फ़ॉलो कर सकते हैं.)

CLEANED DATA:
Article length: 2648
Ending: ा ले गया. राफे ने फ़िल्म देखते-देखते कहा, बाबा क्या आप भूतनाथ की तरह ग़ायब हो सकते हैं. मैं उसे कैसे बताऊँ कि भूतनाथ पिछले चालीस-बयालीस वर्षों से मेरे अंदर ही तो रहता है. बात ये है कि ज़माने के किसी मोड़ पर एक ख़ास लम्हे में कुछ लोग कैमरे को हमेशा पसंद आ जाते हैं. अमिताभ इन्हीं में से हैं. शनिवार को जब उनका 72वां जन्मदिन चढ़ा तो मैंने ख़ुद से कहा, हैप्पी बर्डे टू यू...बुड्ढा होगा तेरा बाप... (बीबीसी हिन्दी के एं

In [228]:
row = source_250.loc[
    source_250["id"] == "141013_wusatullah_amitabh_bachchan_rns"
].iloc[0]

print("COMPLETE HINDI ORIGINAL:\n")
print(row["hindi_article"])

print("\nCURRENT ENGLISH TRANSLATION:\n")
print(row["english_article"])

COMPLETE HINDI ORIGINAL:

पिछले चार दशकों में कई पीढ़ियाँ उनकी फ़िल्में देखते हुए पली-बढ़ी, जवान हुईं और ढल गईं. लेकिन अमिताभ आज भी अपने ख़ास अंदाज में रुपहले पर्दे पर छाए हुए हैं. हिन्दी सिनेमा के महानायक का जादू जितना हिन्दुस्तान में कारगर था उतना ही पाकिस्तान में. पढ़ें वुसतुल्लाह ख़ान का ब्लॉग बिल्कुल भी आश्चर्यचकित न होइएगा अगर मैं कह दूँ कि मैं और अमिताभ बच्चन और मैं एक साथ बड़े हुए, पर अब मैं बूढ़ा हो रहा हूँ. मैं आपको समझाता हूँ, जीवन में पहली फ़िल्म आनंद देखी तो पता चला कि फ़िल्म क्या होती है. आनंद बाबू को ज़िंदा रखने के प्रयास में जुटे एक दुबले पतले लंबे से डॉक्टर को देखा तो पता चला, दोस्ती के क्या मानी हैं. और जब बाहर निकला तो ये जाना कि अपने ही गाल पर बहते ख़ामोश आँसू इतने नमकीन क्यों होते हैं. उसके बाद से घर पुराने कपड़े, घिसे-पिटे बर्तन, टूटी लकड़ियाँ ग़ायब होने लगीं. अब आप ख़ुद ही बताइए कि मेरे जैसा चौथी क्लास का बच्चा, तीन कैसेटों वाले वीसीआर पर चोरी छिपे अमिताभ की नई फ़िल्म ज़ंजीर देखने के लिए आख़िर कैसे 10 रुपए जमा करता और वो भी सत्तर के दशक में. नमकहराम एक दफा क्या ह

In [229]:

corrected_translation = """Over the past four decades, several generations have grown up watching his films, grown older and moved on through life. Yet Amitabh Bachchan continues to dominate the silver screen in his distinctive style. The magic of this legendary Hindi cinema star has been just as powerful in Pakistan as it has been in India.

Read Wusatullah Khan's blog.

Do not be surprised if I say that Amitabh Bachchan and I grew up together, although I am now growing old. Let me explain.

The first film I ever watched was Anand. It taught me what a film was. I saw a tall, thin doctor trying to keep Anand Babu alive, and I learned the meaning of friendship. When I came out, I understood why the silent tears running down one's cheeks taste so salty.

After that, old clothes, worn-out utensils and broken pieces of wood began disappearing from our house. Now tell me, how could a fourth-grade child like me save 10 rupees to secretly watch Amitabh's new film Zanjeer on a VCR with three cassettes, especially in the 1970s?

Namak Haraam

One day, the VCR owner whispered to me, "I'm showing Namak Haraam this evening. Come over."

I said, "I only have five rupees in my piggy bank."

He replied, "That's all right. You can watch it on credit."

So I told my mother that I was going to a friend's house to study that evening because I had a test the next morning.

My mother looked straight into my eyes and said, "When you come back after studying, tell me the story of Namak Haraam too."

Then she said, "You should tell me the truth before going. I gave birth to you, not the other way around. Do you have any money?"

I said, "No."

She untied a knot in her dupatta, took out ten rupees and placed the money in my palm. From that day onwards, at least, I stopped lying to my mother.

Sholay

While we were watching Sholay, the police raided the place. In those days, watching films on a VCR, especially smuggled Indian films, was considered a serious offence by the police. If Mazid had not carried me on his shoulders and helped me climb over the wall that day, my father's honour and the considerable amount of money he would have spent getting me released would have gone to waste.

The last Amitabh film I watched was Coolie. After that, for many years, I did not watch anyone's films.

Baghban

Many years later, one day, Baghban brought my childhood hero back to me, this time with white hair. After that, I watched Black, Sarkar, Sarkar Raj, Nishabd, Cheeni Kum, Paa and Bhootnath. I took my younger son, Rafe, to the cinema to watch Bhootnath.

While watching the film, Rafe asked, "Baba, can you disappear like Bhootnath?"

How could I explain to him that Bhootnath had been living inside me for the past forty or forty-two years?

The truth is that, at a particular moment in time, some people become favourites of the camera forever. Amitabh is one of them.

When his 72nd birthday arrived on Saturday, I said to myself, "Happy birthday to you... Your father will grow old, not you!"

(Click here for BBC Hindi's Android app. You can also follow us on Facebook and Twitter.)"""

article_id = "141013_wusatullah_amitabh_bachchan_rns"

source_250.loc[
    source_250["id"] == article_id,
    "english_article"
] = corrected_translation

source_250.loc[
    source_250["id"] == article_id,
    "translation_quality_status"
] = "Corrected - needs final review"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("Corrected translation saved:", article_id)
print("Status: Corrected - needs final review")
print("Translation characters:", len(corrected_translation))
print("File:", staging_path)


Corrected translation saved: 141013_wusatullah_amitabh_bachchan_rns
Status: Corrected - needs final review
Translation characters: 3073
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/FactIndic_Source_250_Staging.csv


In [230]:

source_250 = pd.read_csv(staging_path)

print("TOTAL ARTICLES:", len(source_250))
print("UNIQUE IDS:", source_250["id"].nunique())

print("\nTRANSLATION STATUS:")
print(source_250["translation_quality_status"].value_counts(dropna=False))

print("\nINCOMPLETE SOURCE ARTICLES:")
print(
    source_250.loc[
        source_250["translation_quality_status"].astype(str).str.contains(
            "incomplete", case=False, na=False
        ),
        "id"
    ].tolist()
)

print("\nMISSING ENGLISH:", source_250["english_article"].isna().sum())
print(
    "EMPTY ENGLISH:",
    source_250["english_article"].fillna("").str.strip().eq("").sum()
)

print("\nFINAL AUDIT REFRESH COMPLETE")


TOTAL ARTICLES: 250
UNIQUE IDS: 250

TRANSLATION STATUS:
translation_quality_status
Initial review - no obvious mismatch    239
Corrected - needs final review           11
Name: count, dtype: int64

INCOMPLETE SOURCE ARTICLES:
[]

MISSING ENGLISH: 0
EMPTY ENGLISH: 0

FINAL AUDIT REFRESH COMPLETE


In [231]:
import os
import shutil

backup_folder = os.path.join(base_path, "09_checkpoints")
os.makedirs(backup_folder, exist_ok=True)

backup_path = os.path.join(
    backup_folder,
    "FactIndic_Source_250_PostCorrection_Backup.csv"
)

shutil.copy2(staging_path, backup_path)

print("Backup saved:", backup_path)
print("Backup exists:", os.path.exists(backup_path))
print("Backup size:", os.path.getsize(backup_path), "bytes")

Backup saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_checkpoints/FactIndic_Source_250_PostCorrection_Backup.csv
Backup exists: True
Backup size: 2191425 bytes


In [232]:
article_id = "international-47105506"

row = source_250.loc[
    source_250["id"] == article_id
].iloc[0]

print("HINDI ORIGINAL:\n")
print(row["hindi_article"])

print("\nCORRECTED ENGLISH:\n")
print(row["english_article"])

HINDI ORIGINAL:

सबसे पहले बात हज सब्सिडी की. पाकिस्तान में इन दिनों हज के ख़र्च में हुई बढ़ोत्तरी का मुद्दा भी छाया हुआ है. इमरान ख़ान की सरकार ने सत्ता में आने के बाद इस हफ़्ते अपनी पहली हज पॉलिसी जारी की. इसके तहत अब एक पाकिस्तानी को इस साल हज पर जाने के लिए चार लाख 76 हज़ार पाकिस्तानी रुपए देने होंगे जबकि पिछले साल ये रक़म दो लाख 80 हज़ार रुपए थे. इसको लेकर पाकिस्तानी संसद में ख़ूब हंगामा हुआ. अख़बार नवा-ए-वक़्त के अनुसार विपक्षी पार्टी जमात-ए-इस्लामी के एक सांसद मुश्ताक़ अहमद ने कहा कि सरकार ने हज यात्रियों पर ड्रोन हमला किया है. सांसद ने कहा कि सरकार ने हज पर कोई सब्सिडी नहीं दी. उनके अनुसार धार्मिक मामलों के मंत्री ने अपनी रिपोर्ट में हज सब्सिडी देने की अपील की थी लेकिन इमरान ख़ान की सरकार ने अपने ही मंत्री की सिफ़ारिश को ठुकरा दिया. अख़बार लिखता है कि सांसद मुश्ताक़ अहमद ने सरकार पर चुटकी लेते हुए कहा, ''मदीना की रियासत क़ायम करने का दावा करने वाले लोग, अब लोगों को मक्का और मदीना जाने से रोक रहे हैं.'' ग़ौरतलब है कि प्रधानमंत्री बनने के बाद अपने पहले भाषण में कहा था कि इस्लाम क

In [233]:
reviewed_id = "international-47105506"

source_250.loc[
    source_250["id"] == reviewed_id,
    "translation_quality_status"
] = "Final review passed"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("Final review passed:", reviewed_id)

Final review passed: international-47105506


In [234]:
article_id = "international-48243735"

row = source_250.loc[
    source_250["id"] == article_id
].iloc[0]

print("HINDI ORIGINAL:\n")
print(row["hindi_article"])

print("\nCORRECTED ENGLISH:\n")
print(row["english_article"])

HINDI ORIGINAL:

रूहानी ने कहा कि नए अमरीकी प्रतिबंधों के कारण देश की हालत 1980-88 के बीच इराक़ से हुए युद्ध से भी बदतर हो गई है. कुछ दिनों पहले खाड़ी में युद्धपोत और लड़ाकू विमानों की तैनाती के बाद अमरीका से बढ़ते तनावों के बीच रूहानी का ये बयान आया है. घरेलू राजनीतिक दबावों में आए रूहानी ने प्रतिबंधों का मुकाबला करने के लिए राजनीतिक एकता पर बल दिया है. तेहरान में राजनीतिक कार्यकर्ताओं से रूहानी ने कहा, "युद्ध के दिनों में हमें अपने बैंकिंग, तेल और आयात में मुश्किल नहीं आई थी और सिर्फ हथियारों की ख़रीद पर प्रतिबंध था." रूहानी ने कहा, "दुश्मनों की ओर से डाला जा रहा दबाव एक युद्ध है जो हमारे इस्लामिक क्रांति के इतिहास में अभूतपूर्व है...लेकिन हम निराश नहीं हैं और भविष्य के लिए बेहतर उम्मीद है. मुझे विश्वास है कि अगर हम एकता बनाए रखेंगे तो इन मुश्किलों को पीछे छोड़ सकते हैं." अमरीका और ईरान के बीच तनाव के कारण 2015 में हुए परमाणु समझौते पर सवालिया निशान लग गया है. इस समझौते पर ईरान ने सुरक्षा परिषद के पांच स्थाई सदस्यों और जर्मनी के साथ हस्ताक्षर किया था. पिछले साल अमरीकी राष्ट्रपति डोनल

In [235]:
reviewed_id = "international-48243735"

source_250.loc[
    source_250["id"] == reviewed_id,
    "translation_quality_status"
] = "Final review passed"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("Final review passed:", reviewed_id)

Final review passed: international-48243735


In [236]:
article_id = "150816_badminton_saina_final_dp"

row = source_250.loc[
    source_250["id"] == article_id
].iloc[0]

print("HINDI ORIGINAL:\n")
print(row["hindi_article"])

print("\nCORRECTED ENGLISH:\n")
print(row["english_article"])

HINDI ORIGINAL:

इंडोनेशिया के जकार्ता में साइना को स्पेन की कैरोलिना मारिन से लगातार गेमों में 16-21 और 19-21 से हराया. 59 मिनट तक चले इस मुक़ाबले में दुनिया की दूसरे नंबर की खिलाड़ी साइना ने दोनों गेम में अच्छी शुरुआत की लेकिन इसके बाद उनका खेल बिखर गया. साइना विश्व चैंपियनशिप के पहुंचने वाली पहली भारतीय खिलाड़ी हैं लेकिन वह स्वर्ण पदक जीतने से वंचित रह गईं. विश्व चैंपियनशिप में भारत का यह पांचवां पदक है. समाप्त पदक इससे पहले, पी वी सिंधु ने 2013 और 2014 में इस टूर्नामेंट का कांस्य पदक हासिल किया था जबकि 2011 में ज्वाला गट्टा और अश्विनी पोनप्पा की जोड़ी ने डबल्स का कांस्य पदक जीता था. 1983 में प्रकाश पादुकोण वर्ल्ड चैंपियनशिप में कांस्य पदक जीतने वाले पहले भारतीय बने थे. साइना ने मैच में अच्छी शुरुआत करते हुए पहले गेम में 3-1 की बढ़त बना ली थी लेकिन इसके बाद उनका खेल बिखर गया. दुनिया की पहले नंबर की खिलाड़ी मारिन ने 11-5 की बढ़त बनाने के बाद पीछे मुड़कर नहीं देखा और पहला गेम 21-16 से जीत लिया. कड़ा मुक़ाबला दूसरे गेम की शुरुआत में साइना ने एक बार फिर बेहतर शुरुआत करते हुए 12-6 की बढ़

In [237]:
reviewed_id = "150816_badminton_saina_final_dp"

source_250.loc[
    source_250["id"] == reviewed_id,
    "translation_quality_status"
] = "Final review passed"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("Final review passed:", reviewed_id)

Final review passed: 150816_badminton_saina_final_dp


In [238]:
article_id = "international-49451047"

row = source_250.loc[
    source_250["id"] == article_id
].iloc[0]

print("HINDI ORIGINAL:\n")
print(row["hindi_article"])

print("\nCORRECTED ENGLISH:\n")
print(row["english_article"])

HINDI ORIGINAL:

यह सम्मान पाने वाले वो पहले भारतीय प्रधानमंत्री बने. 'ऑर्डर ऑफ़ ज़ायेद' यूएई का सर्वोच्च नागरिक सम्मान है जो बादशाहों, राष्ट्रपति और राष्ट्र प्रमुखों को दिया जाता है. इससे पहले साल 2007 में रूस के राष्ट्रपति व्लादिमीर पुतिन, 2010 में ब्रिटेन की महारानी एलिज़ाबेथ, 2016 में सऊदी के शाह सलमान बिन अब्दुल्ला अज़ीज़ अल सऊद और साल 2018 में चीन के राष्ट्रपति शी जिनपिंग को यह सम्मान दिया गया था. यूएई ने इस सम्मान की शुरुआत साल 1995 में की थी. पीएम मोदी को इस सम्मान के लिए क्यों चुना गया? अबू धाबी के क्राउन प्रिंस शेख़ मोहम्मद बिन ज़ायेद अल नाह्यान के मुताबिक, भारत और यूएई के बीच द्विपक्षीय संबंधों को बढ़ाने में पीएम मोदी ने महत्वपूर्ण योगदान दिया है. इस साल अप्रैल में शेख़ मोहम्मद बिन ज़ायेद अल नाह्यान ने एक ट्वीट के माध्यम से इस सम्मान की घोषणा की थी. उन्होंने लिखा था, "भारत के साथ हमारे ऐतिहासिक और व्यापक रणनीतिक संबंध हैं, जिसके पीछे मेरे परम मित्र नरेंद्र मोदी का बहुत योगदान है. उन्होंने इन रिश्तों को और मज़बूत किया है. उनके प्रयासों को देखते हुए यूएई के राष्ट्रपति शेख़ ख़ल

In [239]:
reviewed_id = "international-49451047"

source_250.loc[
    source_250["id"] == reviewed_id,
    "translation_quality_status"
] = "Final review passed"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("Final review passed:", reviewed_id)

Final review passed: international-49451047


In [240]:
article_id = "international-54702261"

row = source_250.loc[
    source_250["id"] == article_id
].iloc[0]

print("HINDI ORIGINAL:\n")
print(row["hindi_article"])

print("\nCORRECTED ENGLISH:\n")
print(row["english_article"])

HINDI ORIGINAL:

हमला पाकिस्तान में ख़ैबर-पख़्तूनख़्वाह प्रांत के पेशावर शहर में हुआ. अस्पताल के अधिकारियों के मुताबिक़, मरने वालों की उम्र 20 से 30 साल के बीच है. दर्जनों लोग घायल हैं जिनमें से चार की उम्र 13 साल से कम है. शुरुआती रिपोर्टों में कहा गया था कि मरने वालों में चार बच्चे हैं, लेकिन बाद में ऐसी रिपोर्टों को वापस ले लिया गया. अभी तक किसी समूह ने हमले की ज़िम्मेदारी नहीं ली है. मामले की जांच शुरू कर दी गई है. समाप्त पाकिस्तान के प्रधानमंत्री इमरान ख़ान ने धमाके की कड़ी निंदा की है और जान के नुक़सान पर दुख जताया है. पुलिस ने बीबीसी को बताया कि धमाका स्थानीय समयानुसार सुबह 8:30 पर हुआ. पुलिस के मुताबिक़ धमाका उस वक़्त हुआ जब मदरसे में पढ़ाई चल रही थी. माना जा रहा है कि धमाके के वक़्त मदरसे में क़रीब 60 लोग मौजूद थे. एक चश्मदीद ने पुलिस को बताया कि उन्होंने धमाके से कुछ देर पहले एक शख़्स को विस्फोटकों से भरा बैग इमारत में ले जाते देखा था. एक वरिष्ठ पुलिस अधिकारी वकार अज़ीम में समाचार एजेंसी एएफ़पी से कहा कि "कोई पाठशाला के अंदर बस्ता रखकर चला गया." ख़ैबर-पख़्तूनख़्वाह के प्रांती

In [241]:
reviewed_id = "international-54702261"

source_250.loc[
    source_250["id"] == reviewed_id,
    "translation_quality_status"
] = "Final review passed"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("Final review passed:", reviewed_id)

Final review passed: international-54702261


In [242]:
article_ids = [
    "india-39165960",
    "140524_iran_hanging_aa"
]

for article_id in article_ids:
    row = source_250.loc[
        source_250["id"] == article_id
    ].iloc[0]

    print("\n" + "=" * 90)
    print("ARTICLE ID:", article_id)
    print("=" * 90)

    print("\nHINDI ORIGINAL:\n")
    print(row["hindi_article"])

    print("\nCORRECTED ENGLISH:\n")
    print(row["english_article"])


ARTICLE ID: india-39165960

HINDI ORIGINAL:

यही तो पहचान है धर्मनगरी काशी की. हज़ारों मंदिरों के इस शहर में बड़ी-बड़ी गाड़ियों का शोर, हवा में कुलांचे भरते हेलीकॉप्टर और जगह-जगह बने ऊंचे मंचों ने यहां लोगों की व्यस्तता को और बढ़ा दिया है. कहते हैं यूपी के कण-कण में सियासत भरी है लोगों के रगों में तो सियासी बहसें लहू बनकर दौड़ती हैं. यूपी चुनाव में बसपा बड़ी ताकत वाराणसी में बीजेपी दिग्गजों का जमावड़ा क्यों? मौका भी है, दस्तूर भी है और मिज़ाज भी. शहर में थोड़ा घूमने निकल जाइए तो पता चल जाएगा कि हर गली, हर नुक्कड़ और हर चौक-चौराहे पर लोगों का कम से कम एक समूह तो बनारसी पान की मिठास के बीच राजनीतिक बहस में डूबा हुआ है और यह बहस कभी-कभी तो इस पान के मिठास को थोड़ा कड़वा भी बना देता है. राजनीतिक मिज़ाज वाला प्रदेश बहरहाल अभी ऐसा लगता है कि जिस तरह बनारस की एक पहचान देवों के देव महादेव से है, उस तरह इसकी एक पहचान सियासी बतकही भी बन गई हो. सच ही तो है क्योंकि यह वो जमीं है जहां राजनीतिक चर्चा शुरू तो होती है छोटे दलों या नेताओं से पर वो जाती है दूर तलक या कहें कि डोनल्ड ट्रंप तक. योगी-मुख़्

In [243]:
reviewed_ids = [
    "india-39165960",
    "140524_iran_hanging_aa"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Final review passed"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("Final review passed:")
for article_id in reviewed_ids:
    print("-", article_id)

Final review passed:
- india-39165960
- 140524_iran_hanging_aa


In [244]:
article_ids = [
    "india-43796841",
    "090216_jobloss_column3_alk"
]

for article_id in article_ids:
    row = source_250.loc[
        source_250["id"] == article_id
    ].iloc[0]

    print("\n" + "=" * 90)
    print("ARTICLE ID:", article_id)
    print("=" * 90)

    print("\nHINDI ORIGINAL:\n")
    print(row["hindi_article"])

    print("\nCORRECTED ENGLISH:\n")
    print(row["english_article"])


ARTICLE ID: india-43796841

HINDI ORIGINAL:

लेकिन अगर आंकड़ों पर नज़र डालें तो इस साल की अक्षय तृतीया पर सोने की कीमतें सबसे ज़्यादा बढ़ सकती हैं. बीते 11 अप्रैल को 24 कैरेट सोने की कीमत 31,524 रुपये प्रति दस ग्राम का आंकड़ा पार कर गई थी. हालांकि, 17 अप्रैल को सोने की कीमत में थोड़ी गिरावट देखी गई. लेकिन विशेषज्ञों के मुताबिक़ खरीदार लगातार सोना खरीदना जारी रखेंगे. सोना पांच साल में सबसे सस्ता हुआ बिटक्वाइन के आगे पहली बार सोने की चमक फीकी सोने के दाम क्यों बढ़ रहे हैं? कमोडिटीज़ एंड करेंसी रिसर्च फर्म निर्मल बंग के प्रमुख कुणाल शाह कहते हैं, "मौद्रिक नीति को लेकर अनिश्चितता, भूराजनीतिक संकट से जुड़ी अनिश्चितताएं और पश्चिमी अर्थव्यवस्थाओं में महंगाई के बढ़ने से जुड़ी आशंकाओं के चलते मार्च 2018 के अंत तक सोना वैश्विक स्तर पर $1450 पर खरीदा जाएगा लेकिन मार्च 2019 से पहले ये $1600 का स्तर छू लेगा." सोने को हमेशा ही निवेश के लिहाज से एक बढ़िया विकल्प माना जाता है. ये वैश्विक अनिश्चितताओं का साल है, अमरीका चीन के साथ ट्रेड वॉर करने की धमकी दे रहा है, सीरिया में तनाव बढ़ रहा है. ऐसे में नि

In [245]:
reviewed_ids = [
    "india-43796841",
    "090216_jobloss_column3_alk"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Final review passed"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("Final review passed:")
for article_id in reviewed_ids:
    print("-", article_id)

Final review passed:
- india-43796841
- 090216_jobloss_column3_alk


In [246]:
article_ids = [
    "130912_nepal_strike_ssr",
    "141013_wusatullah_amitabh_bachchan_rns"
]

for article_id in article_ids:
    row = source_250.loc[
        source_250["id"] == article_id
    ].iloc[0]

    print("\n" + "=" * 90)
    print("ARTICLE ID:", article_id)
    print("=" * 90)

    print("\nHINDI ORIGINAL:\n")
    print(row["hindi_article"])

    print("\nCORRECTED ENGLISH:\n")
    print(row["english_article"])


ARTICLE ID: 130912_nepal_strike_ssr

HINDI ORIGINAL:

नेपाल में हड़ताल के चलते सड़कें सूनी दिखीं सरकार द्वारा नवंबर में चुनाव कराने की योजना के विरोध में ये हड़ताल की गई. स्कूल और बाजार बंद हैं, सड़कें सूनी पड़ी हैं और वाहन चलते नहीं दिखे. आह्वान ये हड़ताल नेपाल में 33 पार्टियों के गठबंधन द्वारा बुलाई गई थी, जिसमें नेपाल कम्युनिस्ट पार्टी - माओवादी भी शामिल है. मार्च में एक सीनियर जज की अगुवाई वाली सरकार को चुनाव कराने के लिए नियुक्त किया गया था. नेपाल सुप्रीम कोर्ट के मुख्य न्यायाधीश खिलराज रेजमी प्रशासन पर ताजा चुनाव 19 नवंबर को कराने की जिम्मेदारी है. नेपाल में आखिरी बार संसदीय चुनाव 2008 में कराए गए थे. नये संविधान का मसौदा तय करने के बाद संवैधानिक संसद का कार्यकाल मई 2012 में खत्म हो गया. (बीबीसी हिन्दी के एंड्रॉएड ऐप के लिए यहां क्लिक करें. आप हमें फ़ेसबुक और ट्विटर पर फ़ॉलो भी कर सकते हैं.)

CORRECTED ENGLISH:

Streets in Nepal were deserted during a nationwide strike called to protest against the government's plan to hold elections in November. Schools and markets remained clo

In [247]:
reviewed_ids = [
    "130912_nepal_strike_ssr",
    "141013_wusatullah_amitabh_bachchan_rns"
]

source_250.loc[
    source_250["id"].isin(reviewed_ids),
    "translation_quality_status"
] = "Final review passed"

source_250.to_csv(
    staging_path,
    index=False,
    encoding="utf-8-sig"
)

print("Final review passed:")
for article_id in reviewed_ids:
    print("-", article_id)

print("\nFinal review count:",
      (source_250["translation_quality_status"] == "Final review passed").sum())

Final review passed:
- 130912_nepal_strike_ssr
- 141013_wusatullah_amitabh_bachchan_rns

Final review count: 11


# New Section

In [248]:
print("Dataset shape:", source_250.shape)

print("\nAvailable columns:")
print(source_250.columns.tolist())

print("\nTranslation review status:")
print(source_250["translation_quality_status"].value_counts())

print("\nMissing English articles:",
      source_250["english_article"].isna().sum())

print("\nEmpty English articles:",
      (source_250["english_article"].fillna("").str.strip() == "").sum())

Dataset shape: (250, 13)

Available columns:
['id', 'url', 'title', 'summary', 'hindi_article', 'article_words', 'summary_words', 'english_article', 'hindi_word_count', 'english_word_count', 'english_has_hindi', 'translation_ratio', 'translation_quality_status']

Translation review status:
translation_quality_status
Initial review - no obvious mismatch    239
Final review passed                      11
Name: count, dtype: int64

Missing English articles: 0

Empty English articles: 0


In [249]:
import re

sample = source_250.iloc[0]

hindi_sentences = re.split(
    r'(?<=[।!?])\s+',
    str(sample["hindi_article"]).strip()
)

english_sentences = re.split(
    r'(?<=[.!?])\s+',
    str(sample["english_article"]).strip()
)

print("Article ID:", sample["id"])

print("\nHindi sentences:", len(hindi_sentences))
print("English sentences:", len(english_sentences))

print("\nHINDI SENTENCES:")
for i, sentence in enumerate(hindi_sentences, 1):
    print(f"{i}. {sentence}")

print("\nENGLISH SENTENCES:")
for i, sentence in enumerate(english_sentences, 1):
    print(f"{i}. {sentence}")

Article ID: 080419_brchopra_festival

Hindi sentences: 1
English sentences: 13

HINDI SENTENCES:
1. शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी." बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है. गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में शामिल 

In [250]:
import re

def split_paragraphs(text):
    return [
        p.strip()
        for p in re.split(r'\n+', str(text))
        if p.strip()
    ]

for i in range(5):
    row = source_250.iloc[i]

    hi = split_paragraphs(row["hindi_article"])
    en = split_paragraphs(row["english_article"])

    print("\nARTICLE:", row["id"])
    print("Hindi paragraphs:", len(hi))
    print("English paragraphs:", len(en))


ARTICLE: 080419_brchopra_festival
Hindi paragraphs: 1
English paragraphs: 1

ARTICLE: 080613_afghan_jail
Hindi paragraphs: 1
English paragraphs: 1

ARTICLE: 130912_guinnness_record_bulgarian_swim_sr
Hindi paragraphs: 1
English paragraphs: 7

ARTICLE: 050713_us_un
Hindi paragraphs: 1
English paragraphs: 1

ARTICLE: 120809_salman_pkp
Hindi paragraphs: 1
English paragraphs: 7


In [251]:
print("final_candidate_phrases available:",
      "final_candidate_phrases" in globals())

print("safe_phrases available:",
      "safe_phrases" in globals())

print("candidate_50_phrases available:",
      "candidate_50_phrases" in globals())

if "final_candidate_phrases" in globals():
    print("\nDictionary size:", len(final_candidate_phrases))
    print("\nSample entries:")
    for item in list(final_candidate_phrases.items())[:15]:
        print(item)

final_candidate_phrases available: False
safe_phrases available: False
candidate_50_phrases available: False


In [252]:
sample_ids = [
    "080419_brchopra_festival",
    "080613_afghan_jail",
    "130912_guinnness_record_bulgarian_swim_sr"
]

for article_id in sample_ids:
    row = source_250.loc[
        source_250["id"] == article_id
    ].iloc[0]

    print("\n" + "=" * 80)
    print("ARTICLE:", article_id)

    print("\nHINDI:")
    print(row["hindi_article"][:1800])

    print("\nENGLISH:")
    print(row["english_article"][:1800])


ARTICLE: 080419_brchopra_festival

HINDI:
शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी." बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है. गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में शामिल है. हालांकि बीआर चोपड़ा की ज़्यादातर फ़िल्में किसी न किसी

In [253]:
verified_phrase_bank = {
    "शुक्रवार शाम": "Friday evening",
    "खराब सेहत के कारण": "because of poor health",
    "मशहूर फ़िल्मकार": "renowned filmmaker",
    "विधवा पुनर्विवाह": "widow remarriage",
    "कंधार शहर": "Kandahar city",
    "एक हज़ार क़ैदी": "1,000 prisoners",
    "औसत रफ़्तार": "average speed",
    "गिनीज़ वर्ल्ड रिकॉर्ड": "Guinness World Records",
    "मैसेडोनिया की ऑरिड झील": "Lake Ohrid in Macedonia",
    "समाचार एजेंसी एएफपी": "AFP news agency",
    "पिछले महीने": "last month",
    "अगले कुछ दिनों": "the next few days"
}

print("Phrase pairs:", len(verified_phrase_bank))

for hindi, english in verified_phrase_bank.items():
    print(hindi, "=>", english)

Phrase pairs: 12
शुक्रवार शाम => Friday evening
खराब सेहत के कारण => because of poor health
मशहूर फ़िल्मकार => renowned filmmaker
विधवा पुनर्विवाह => widow remarriage
कंधार शहर => Kandahar city
एक हज़ार क़ैदी => 1,000 prisoners
औसत रफ़्तार => average speed
गिनीज़ वर्ल्ड रिकॉर्ड => Guinness World Records
मैसेडोनिया की ऑरिड झील => Lake Ohrid in Macedonia
समाचार एजेंसी एएफपी => AFP news agency
पिछले महीने => last month
अगले कुछ दिनों => the next few days


In [254]:
for hindi, english in verified_phrase_bank.items():
    matches = source_250["hindi_article"].fillna("").str.contains(
        re.escape(hindi),
        regex=True
    ).sum()

    print(f"{hindi} => {english} | Matches: {matches}")

शुक्रवार शाम => Friday evening | Matches: 2
खराब सेहत के कारण => because of poor health | Matches: 1
मशहूर फ़िल्मकार => renowned filmmaker | Matches: 1
विधवा पुनर्विवाह => widow remarriage | Matches: 1
कंधार शहर => Kandahar city | Matches: 1
एक हज़ार क़ैदी => 1,000 prisoners | Matches: 1
औसत रफ़्तार => average speed | Matches: 1
गिनीज़ वर्ल्ड रिकॉर्ड => Guinness World Records | Matches: 0
मैसेडोनिया की ऑरिड झील => Lake Ohrid in Macedonia | Matches: 1
समाचार एजेंसी एएफपी => AFP news agency | Matches: 2
पिछले महीने => last month | Matches: 9
अगले कुछ दिनों => the next few days | Matches: 0


In [255]:
from collections import Counter
import re

all_hindi_text = " ".join(
    source_250["hindi_article"].fillna("").astype(str)
)

words = re.findall(r'[\u0900-\u097F]+', all_hindi_text)

bigrams = Counter(
    " ".join(words[i:i+2])
    for i in range(len(words)-1)
)

trigrams = Counter(
    " ".join(words[i:i+3])
    for i in range(len(words)-2)
)

print("TOP 30 TWO-WORD PHRASES")
for phrase, count in bigrams.most_common(30):
    print(count, "=>", phrase)

print("\nTOP 30 THREE-WORD PHRASES")
for phrase, count in trigrams.most_common(30):
    print(count, "=>", phrase)

TOP 30 TWO-WORD PHRASES
712 => है कि
676 => के लिए
267 => सकते हैं
255 => के बाद
246 => रहे हैं
230 => ने कहा
216 => कर सकते
211 => है और
195 => गया है
192 => रहा है
189 => के साथ
175 => कहा कि
160 => था कि
147 => है लेकिन
144 => हैं और
140 => का कहना
140 => हैं कि
137 => नहीं है
135 => कहना है
131 => कहते हैं
123 => के बीच
117 => बीबीसी हिन्दी
117 => ऐप के
117 => यहां क्लिक
117 => आप हमें
116 => के एंड्रॉएड
116 => एंड्रॉएड ऐप
114 => हिन्दी के
114 => हमें फ़ेसबुक
110 => किया है

TOP 30 THREE-WORD PHRASES
208 => कर सकते हैं
120 => कहना है कि
117 => ऐप के लिए
116 => के एंड्रॉएड ऐप
116 => एंड्रॉएड ऐप के
114 => बीबीसी हिन्दी के
114 => हिन्दी के एंड्रॉएड
114 => आप हमें फ़ेसबुक
112 => का कहना है
92 => भी कर सकते
91 => फ़ॉलो भी कर
89 => और ट्विटर पर
84 => ने कहा कि
83 => फ़ेसबुक और ट्विटर
83 => पर फ़ॉलो भी
80 => हमें फ़ेसबुक और
79 => कर रहे हैं
77 => के बारे में
72 => के लिए आप
72 => यहां क्लिक कर
72 => क्लिक कर सकते
72 => सकते हैं आप
72 => हैं आप हमें
71 => लिए आप यहां
71 => आप यहां क्लिक
71

In [256]:
stop_phrases = {
    "है कि", "के लिए", "सकते हैं", "के बाद",
    "रहे हैं", "ने कहा", "कर सकते", "है और",
    "गया है", "रहा है", "के साथ", "कहा कि",
    "था कि", "है लेकिन", "हैं और", "का कहना",
    "हैं कि", "नहीं है", "कहना है", "कहते हैं",
    "के बीच", "बीबीसी हिन्दी", "ऐप के",
    "यहां क्लिक", "आप हमें", "के एंड्रॉएड",
    "एंड्रॉएड ऐप", "हिन्दी के", "हमें फ़ेसबुक",
    "किया है"
}

candidate_phrases = [
    (phrase, count)
    for phrase, count in bigrams.most_common(200)
    if phrase not in stop_phrases
]

print("Remaining candidates:", len(candidate_phrases))

for phrase, count in candidate_phrases[:60]:
    print(count, "=>", phrase)

Remaining candidates: 170
107 => कर रहे
107 => बारे में
104 => रही है
99 => भी कर
98 => करने के
97 => ट्विटर पर
96 => सकता है
91 => फ़ॉलो भी
91 => कर दिया
89 => और ट्विटर
87 => किया गया
86 => के ख़िलाफ़
84 => फ़ेसबुक और
84 => कहा है
83 => पर फ़ॉलो
80 => पर भी
80 => गया था
80 => के अनुसार
79 => उन्होंने कहा
79 => के बारे
78 => किया था
76 => होता है
75 => के दौरान
75 => लोगों को
73 => गई है
73 => की है
73 => हैं आप
73 => है इस
72 => लिए आप
72 => क्लिक कर
71 => को लेकर
71 => में भी
71 => आप यहां
71 => है तो
70 => मामले में
70 => करते हैं
69 => हैं लेकिन
65 => में एक
64 => जाता है
64 => दिया है
63 => में से
63 => करने की
62 => भारत के
62 => कि इस
61 => बताया कि
61 => कि वो
59 => की थी
59 => क्लिक करें
59 => है ये
59 => इससे पहले
59 => करते हुए
58 => ने इस
58 => जा रहा
57 => देशों के
57 => होती है
55 => के कारण
55 => कहा था
55 => हो गई
54 => करने का
54 => से पहले


In [257]:
candidate_check = [
    "के अनुसार",
    "के ख़िलाफ़",
    "के दौरान",
    "के कारण",
    "इससे पहले",
    "लोगों को",
    "मामले में",
    "भारत के",
    "देशों के",
    "बताया कि",
    "कर दिया",
    "किया गया",
    "हो गई"
]

for phrase in candidate_check:
    count = source_250["hindi_article"].fillna("").str.contains(
        re.escape(phrase),
        regex=True
    ).sum()

    print(f"{phrase} => {count} articles")

के अनुसार => 44 articles
के ख़िलाफ़ => 56 articles
के दौरान => 59 articles
के कारण => 37 articles
इससे पहले => 47 articles
लोगों को => 53 articles
मामले में => 42 articles
भारत के => 36 articles
देशों के => 33 articles
बताया कि => 37 articles
कर दिया => 63 articles
किया गया => 56 articles
हो गई => 43 articles


In [258]:
candidate_phrase_bank = {
    "के अनुसार": "according to",
    "के ख़िलाफ़": "against",
    "के दौरान": "during",
    "के कारण": "because of",
    "इससे पहले": "before this",
    "लोगों को": "people",
    "मामले में": "in the matter",
    "भारत के": "India's",
    "देशों के": "of countries",
    "बताया कि": "said that",
    "कर दिया": "did",
    "किया गया": "was done",
    "हो गई": "became"
}

print("Candidate phrases:", len(candidate_phrase_bank))

for hindi, english in candidate_phrase_bank.items():
    print(f"{hindi} => {english}")

Candidate phrases: 13
के अनुसार => according to
के ख़िलाफ़ => against
के दौरान => during
के कारण => because of
इससे पहले => before this
लोगों को => people
मामले में => in the matter
भारत के => India's
देशों के => of countries
बताया कि => said that
कर दिया => did
किया गया => was done
हो गई => became


In [260]:
for phrase in candidate_phrase_bank:
    rows = source_250[
        source_250["hindi_article"].fillna("").str.contains(
            re.escape(phrase),
            regex=True
        )
    ]

    print("\n" + "=" * 70)
    print("HINDI PHRASE:", phrase)
    print("ENGLISH CANDIDATE:", candidate_phrase_bank[phrase])
    print("MATCHING ARTICLE:", len(rows))

    if len(rows) > 0:
        row = rows.iloc[0]
        print("\nARTICLE ID:", row["id"])
        print("\nHINDI CONTEXT:")
        text = row["hindi_article"]
        pos = text.find(phrase)
        print(text[max(0, pos-120):pos+len(phrase)+120])

        print("\nENGLISH ARTICLE (first 1000 characters):")
        print(str(row["english_article"])[:1000])


HINDI PHRASE: के अनुसार
ENGLISH CANDIDATE: according to
MATCHING ARTICLE: 44

ARTICLE ID: 080613_afghan_jail

HINDI CONTEXT:
भागे हैं. माना जाता है कि इन क़ैदियों में से 350 तालेबान थे. इस हमले में कई सुरक्षाकर्मियों की जानें गई हैं. अधिकारियों के अनुसार जेल के मुख्य द्वार को उड़ाने के लिए एक लॉरी बम से विस्फोट किया गया और फिर 40 तालेबान जेल के भीतर घुस आए. कंधार शहर में 

ENGLISH ARTICLE (first 1000 characters):
Wali Karzai, chairman of the Kandahar provincial council and brother of President Hamid Karzai, told the BBC that there were 1,000 prisoners in the jail at the time and that all of them had escaped. It is believed that 350 of these prisoners were Taliban members. Several security personnel were killed in the attack. According to officials, a lorry bomb was detonated to blow up the prison's main gate, after which 40 Taliban fighters entered the jail. A state of emergency has been declared in Kandahar city. Police and soldiers are patrolling the streets, and people have been to

In [261]:
verified_phrase_bank.update({
    "के अनुसार": "according to",
    "के दौरान": "during",
    "के कारण": "because of",
    "बताया कि": "told ... that"
})

print("Verified phrase pairs:", len(verified_phrase_bank))

for hindi, english in verified_phrase_bank.items():
    print(f"{hindi} => {english}")

Verified phrase pairs: 16
शुक्रवार शाम => Friday evening
खराब सेहत के कारण => because of poor health
मशहूर फ़िल्मकार => renowned filmmaker
विधवा पुनर्विवाह => widow remarriage
कंधार शहर => Kandahar city
एक हज़ार क़ैदी => 1,000 prisoners
औसत रफ़्तार => average speed
गिनीज़ वर्ल्ड रिकॉर्ड => Guinness World Records
मैसेडोनिया की ऑरिड झील => Lake Ohrid in Macedonia
समाचार एजेंसी एएफपी => AFP news agency
पिछले महीने => last month
अगले कुछ दिनों => the next few days
के अनुसार => according to
के दौरान => during
के कारण => because of
बताया कि => told ... that


In [262]:
print("PHRASE COVERAGE REPORT")
print("=" * 55)

for hindi, english in verified_phrase_bank.items():
    count = source_250["hindi_article"].fillna("").str.contains(
        re.escape(hindi),
        regex=True
    ).sum()

    print(f"{hindi:30} | {count:3} articles | {english}")

PHRASE COVERAGE REPORT
शुक्रवार शाम                   |   2 articles | Friday evening
खराब सेहत के कारण              |   1 articles | because of poor health
मशहूर फ़िल्मकार                |   1 articles | renowned filmmaker
विधवा पुनर्विवाह               |   1 articles | widow remarriage
कंधार शहर                      |   1 articles | Kandahar city
एक हज़ार क़ैदी                 |   1 articles | 1,000 prisoners
औसत रफ़्तार                    |   1 articles | average speed
गिनीज़ वर्ल्ड रिकॉर्ड          |   0 articles | Guinness World Records
मैसेडोनिया की ऑरिड झील         |   1 articles | Lake Ohrid in Macedonia
समाचार एजेंसी एएफपी            |   2 articles | AFP news agency
पिछले महीने                    |   9 articles | last month
अगले कुछ दिनों                 |   0 articles | the next few days
के अनुसार                      |  44 articles | according to
के दौरान                       |  59 articles | during
के कारण                        |  37 articles | because of
बताया कि        

In [263]:
phrases = list(verified_phrase_bank.keys())

print("OVERLAPPING PHRASES")
print("=" * 55)

for phrase in phrases:
    for other in phrases:
        if phrase != other and phrase in other:
            print(f"{phrase}  is contained in  {other}")

OVERLAPPING PHRASES
के कारण  is contained in  खराब सेहत के कारण


In [264]:
phrase_priority = sorted(
    verified_phrase_bank.items(),
    key=lambda item: len(item[0]),
    reverse=True
)

print("PHRASES IN REPLACEMENT PRIORITY")
print("=" * 55)

for hindi, english in phrase_priority:
    print(f"{hindi} => {english}")

PHRASES IN REPLACEMENT PRIORITY
मैसेडोनिया की ऑरिड झील => Lake Ohrid in Macedonia
गिनीज़ वर्ल्ड रिकॉर्ड => Guinness World Records
समाचार एजेंसी एएफपी => AFP news agency
खराब सेहत के कारण => because of poor health
विधवा पुनर्विवाह => widow remarriage
मशहूर फ़िल्मकार => renowned filmmaker
एक हज़ार क़ैदी => 1,000 prisoners
अगले कुछ दिनों => the next few days
शुक्रवार शाम => Friday evening
औसत रफ़्तार => average speed
पिछले महीने => last month
कंधार शहर => Kandahar city
के अनुसार => according to
के दौरान => during
बताया कि => told ... that
के कारण => because of


In [265]:
import re

def safe_phrase_replace(text, phrase_bank):
    text = str(text)
    replacement_count = 0

    for hindi, english in sorted(
        phrase_bank.items(),
        key=lambda item: len(item[0]),
        reverse=True
    ):
        if hindi == "बताया कि":
            continue

        pattern = re.escape(hindi)

        text, count = re.subn(
            pattern,
            english,
            text
        )

        replacement_count += count

    return text, replacement_count


# Test on one manually inspected article
test_id = "080419_brchopra_festival"

test_row = source_250[
    source_250["id"] == test_id
].iloc[0]

test_mixed, count = safe_phrase_replace(
    test_row["hindi_article"],
    verified_phrase_bank
)

print("ARTICLE ID:", test_id)
print("Replacements:", count)
print("\nORIGINAL:")
print(test_row["hindi_article"][:700])

print("\nMIXED:")
print(test_mixed[:700])

ARTICLE ID: 080419_brchopra_festival
Replacements: 5

ORIGINAL:
शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी." बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर 

MIXED:
Friday evening को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव during छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. because of poor health बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.

In [266]:
print("ORIGINAL ENGLISH TRANSLATION:\n")
print(test_row["english_article"])

print("\n" + "=" * 70)
print("MIXED ARTICLE:\n")
print(test_mixed)

ORIGINAL ENGLISH TRANSLATION:

The festival, which began on Friday evening, screened the film Insaf Ka Tarazu. Other classic films, including Chhoti Si Baat, Naya Daur and Waqt, will also be shown during the festival. BR Chopra could not attend the ceremony because of his poor health. The festival was inaugurated by his brother, the renowned filmmaker Yash Chopra. He said, "My brother made films in his own way, even at a time when people were doubtful about the commercial success of such films because they were different from the kinds of stories that were popular then." BR Chopra addressed several subjects in his films, including widow remarriage and prostitution. Among his memorable films, Naya Daur, released in 1957, reflects the ideals of the Nehru era, while Sadhana, released in 1958, raises questions about society's treatment of prostitution. Gumrah, Hamraaz, Kanoon and Nikah are also among the films he made. Although most of BR Chopra's films were based on social issues, these e

In [267]:
pilot_test_bank = {
    "शुक्रवार शाम": "Friday evening",
    "उत्सव के दौरान": "during the festival",
    "खराब सेहत के कारण": "because of poor health",
    "मशहूर फ़िल्मकार": "renowned filmmaker",
    "विधवा पुनर्विवाह": "widow remarriage"
}

pilot_mixed, pilot_count = safe_phrase_replace(
    test_row["hindi_article"],
    pilot_test_bank
)

print("Replacement count:", pilot_count)
print("\nUPDATED MIXED ARTICLE:\n")
print(pilot_mixed[:1500])

Replacement count: 5

UPDATED MIXED ARTICLE:

Friday evening को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. during the festival छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. because of poor health बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और renowned filmmaker यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी." बीआर चोपड़ा ने अपनी फ़िल्मों में widow remarriage, वेश्यावृत्ति जैसे कई विषयों को उठाया. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है. गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में शामिल है. हालांकि बीआर चोपड़ा की ज़्यादातर फ़

In [268]:
pilot_sentences = [
    "Friday evening को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई.",
    "छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में भी during the festival दिखाई जाएंगी.",
    "because of poor health बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके."
]

for i, sentence in enumerate(pilot_sentences, 1):
    print(f"{i}. {sentence}")

1. Friday evening को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई.
2. छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में भी during the festival दिखाई जाएंगी.
3. because of poor health बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.


In [269]:
def calculate_english_percentage(text):
    words = str(text).split()

    english_words = [
        word for word in words
        if re.search(r"[A-Za-z]", word)
    ]

    return round(
        len(english_words) / len(words) * 100, 2
    )


original_percentage = calculate_english_percentage(
    test_row["hindi_article"]
)

mixed_percentage = calculate_english_percentage(
    pilot_mixed
)

print("Original English percentage:", original_percentage)
print("Mixed English percentage:", mixed_percentage)
print("English words added:", mixed_percentage - original_percentage)

Original English percentage: 0.0
Mixed English percentage: 4.42
English words added: 4.42


In [270]:
words = pilot_mixed.split()

current_english = sum(
    bool(re.search(r"[A-Za-z]", word))
    for word in words
)

total_words = len(words)

print("Total words:", total_words)
print("Current English words:", current_english)

for target in [10, 30, 50]:
    required = round(total_words * target / 100)
    additional = max(0, required - current_english)

    print(
        f"{target}% target: "
        f"{required} English words | "
        f"Additional needed: {additional}"
    )

Total words: 294
Current English words: 13
10% target: 29 English words | Additional needed: 16
30% target: 88 English words | Additional needed: 75
50% target: 147 English words | Additional needed: 134


In [271]:
import re

hindi_text = str(test_row["hindi_article"])
english_text = str(test_row["english_article"])

hindi_sentences = [
    s.strip() for s in re.split(r'[।!?]+', hindi_text)
    if s.strip()
]

english_sentences = [
    s.strip() for s in re.split(r'[.!?]+', english_text)
    if s.strip()
]

print("Hindi sentences:", len(hindi_sentences))
print("English sentences:", len(english_sentences))

print("\nHINDI SENTENCES:")
for i, s in enumerate(hindi_sentences, 1):
    print(f"{i}. {s}")

print("\nENGLISH SENTENCES:")
for i, s in enumerate(english_sentences, 1):
    print(f"{i}. {s}")

Hindi sentences: 1
English sentences: 15

HINDI SENTENCES:
1. शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी." बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है. गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में शामिल है. हालांकि बीआर चोपड़ा की ज़्यादातर फ

In [272]:
hindi_words = hindi_text.split()
english_words = english_text.split()

print("Hindi article words:", len(hindi_words))
print("English translation words:", len(english_words))

print("\nFirst 500 characters of Hindi:")
print(hindi_text[:500])

print("\nFirst 500 characters of English:")
print(english_text[:500])

Hindi article words: 294
English translation words: 260

First 500 characters of Hindi:
शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहान

First 500 characters of English:
The festival, which began on Friday evening, screened the film Insaf Ka Tarazu. Other classic films, including Chhoti Si Baat, Naya Daur and Waqt, will also be shown during the festival. BR Chopra could not attend the ceremony because of his poor health. The festival was inaugurated by his brother, the renowned filmmaker Yash Chopra. He said, "My brother made films in his ow

In [273]:
for i in range(4):
    print(f"\n{'='*70}")
    print(f"SECTION {i+1}")
    print("="*70)

    print("\nHINDI:")
    print(hindi_sentences[0].split("।")[0] if i == 0 else "")

    print("\nENGLISH:")
    print(english_sentences[i])


SECTION 1

HINDI:
शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी." बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है. गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में शामिल है. हालांकि बीआर चोपड़ा की ज़्यादातर फ़िल्में किसी न किसी सामाजिक मुद्दे पर आधारि

In [274]:
hindi_sentences = [
    s.strip()
    for s in re.split(r'(?<=[।.!?])\s*', hindi_text)
    if s.strip()
]

print("Hindi sentence count:", len(hindi_sentences))

for i, sentence in enumerate(hindi_sentences, 1):
    print(f"\nHINDI {i}: {sentence}")

Hindi sentence count: 15

HINDI 1: शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई.

HINDI 2: उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी.

HINDI 3: खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.

HINDI 4: फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया.

HINDI 5: उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी.

HINDI 6: " बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया.

HINDI 7: यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है.

HINDI 8: गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में

In [275]:
for i in range(15):
    print(f"\n{'='*60}")
    print(f"PAIR {i+1}")
    print("="*60)

    print("HINDI:", hindi_sentences[i])
    print("ENGLISH:", english_sentences[i])


PAIR 1
HINDI: शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई.
ENGLISH: The festival, which began on Friday evening, screened the film Insaf Ka Tarazu

PAIR 2
HINDI: उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी.
ENGLISH: Other classic films, including Chhoti Si Baat, Naya Daur and Waqt, will also be shown during the festival

PAIR 3
HINDI: खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.
ENGLISH: BR Chopra could not attend the ceremony because of his poor health

PAIR 4
HINDI: फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया.
ENGLISH: The festival was inaugurated by his brother, the renowned filmmaker Yash Chopra

PAIR 5
HINDI: उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होत

In [276]:
import pandas as pd

pilot_alignment = pd.DataFrame({
    "sentence_id": range(1, 16),
    "hindi_sentence": hindi_sentences,
    "english_sentence": english_sentences
})

pilot_alignment["review_status"] = "Needs review"

# Keep quotation-related sections in Hindi for now
pilot_alignment.loc[
    pilot_alignment["sentence_id"].isin([5, 6, 11, 12, 13]),
    "review_status"
] = "Keep Hindi - quotation"

display(pilot_alignment)

,sentence_id,hindi_sentence,english_sentence,review_status
0,1,शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ ...,"The festival, which began on Friday evening, s...",Needs review
1,2,"उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त ज...","Other classic films, including Chhoti Si Baat,...",Needs review
2,3,खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में ...,BR Chopra could not attend the ceremony becaus...,Needs review
3,4,फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मश...,"The festival was inaugurated by his brother, t...",Needs review
4,5,"उन्होंने कहा, ""मेरे भाई ने अपने हिसाब से फ़िल्...","He said, ""My brother made films in his own way...",Keep Hindi - quotation
5,6,""" बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्...",""" BR Chopra addressed several subjects in his ...",Keep Hindi - quotation
6,7,यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौ...,"Among his memorable films, Naya Daur, released...",Needs review
7,8,"गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में क...","Gumrah, Hamraaz, Kanoon and Nikah are also amo...",Needs review
8,9,हालांकि बीआर चोपड़ा की ज़्यादातर फ़िल्में किसी...,Although most of BR Chopra's films were based ...,Needs review
9,10,नया दौर की कहानी को कई टॉप निर्देशकों ने ठुकरा...,Many leading directors had rejected the story ...,Needs review


In [277]:
approved_ids = [1, 2, 3, 4]

pilot_alignment.loc[
    pilot_alignment["sentence_id"].isin(approved_ids),
    "review_status"
] = "Provisionally approved"

display(
    pilot_alignment[
        ["sentence_id", "review_status"]
    ]
)

,sentence_id,review_status
0,1,Provisionally approved
1,2,Provisionally approved
2,3,Provisionally approved
3,4,Provisionally approved
4,5,Keep Hindi - quotation
5,6,Keep Hindi - quotation
6,7,Needs review
7,8,Needs review
8,9,Needs review
9,10,Needs review


In [278]:
approved_ids = [1, 2, 3, 4]

pilot_alignment.loc[
    pilot_alignment["sentence_id"].isin(approved_ids),
    "review_status"
] = "Provisionally approved"

display(pilot_alignment[["sentence_id", "review_status"]])

,sentence_id,review_status
0,1,Provisionally approved
1,2,Provisionally approved
2,3,Provisionally approved
3,4,Provisionally approved
4,5,Keep Hindi - quotation
5,6,Keep Hindi - quotation
6,7,Needs review
7,8,Needs review
8,9,Needs review
9,10,Needs review


In [279]:
for i in [6, 7, 8, 9]:
    print(f"\nPAIR {i+1}")
    print("HINDI:", hindi_sentences[i])
    print("ENGLISH:", english_sentences[i])


PAIR 7
HINDI: यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है.
ENGLISH: Among his memorable films, Naya Daur, released in 1957, reflects the ideals of the Nehru era, while Sadhana, released in 1958, raises questions about society's treatment of prostitution

PAIR 8
HINDI: गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में शामिल है.
ENGLISH: Gumrah, Hamraaz, Kanoon and Nikah are also among the films he made

PAIR 9
HINDI: हालांकि बीआर चोपड़ा की ज़्यादातर फ़िल्में किसी न किसी सामाजिक मुद्दे पर आधारित होती थीं लेकिन इन प्रयोगात्मक फ़िल्मों ने बॉक्स ऑफ़िस पर भी कमाल दिखाया.
ENGLISH: Although most of BR Chopra's films were based on social issues, these experimental films also performed well at the box office

PAIR 10
HINDI: नया दौर की कहानी को कई टॉप निर्देशकों ने ठुकरा दिया था लेकिन बीआर चोपड़ा ने रिस्क लिया और फ़िल्म बनाई.
ENGLISH: Many leading dir

approved_ids = [1, 2, 3, 4, 7, 8, 9, 10]

pilot_alignment.loc[
    pilot_alignment["sentence_id"].isin(approved_ids),
    "review_status"
] = "Provisionally approved"

display(pilot_alignment[["sentence_id", "review_status"]])

In [281]:
for i in [13, 14]:
    print(f"\nPAIR {i+1}")
    print("HINDI:", hindi_sentences[i])
    print("ENGLISH:", english_sentences[i])


PAIR 14
HINDI: फ़िल्म उत्सव में बीआर चोपड़ा के जीवन पर आधारित एक लघु वृत्तचित्र भी दिखाया गया.
ENGLISH: A short documentary based on BR Chopra's life was also screened at the festival

PAIR 15
HINDI: इस समारोह में यश चोपड़ा के अलावा राज बब्बर, निर्देशक बासु चटर्जी, संगीतकार रवि और निर्माता बॉबी बेदी ने हिस्सा लिया.
ENGLISH: Besides Yash Chopra, the event was attended by Raj Babbar, director Basu Chatterjee, music composer Ravi and producer Bobby Bedi


In [282]:
approved_ids = [1, 2, 3, 4, 7, 8, 9, 10, 14, 15]

pilot_alignment.loc[
    pilot_alignment["sentence_id"].isin(approved_ids),
    "review_status"
] = "Provisionally approved"

display(pilot_alignment[["sentence_id", "review_status"]])

,sentence_id,review_status
0,1,Provisionally approved
1,2,Provisionally approved
2,3,Provisionally approved
3,4,Provisionally approved
4,5,Keep Hindi - quotation
5,6,Keep Hindi - quotation
6,7,Provisionally approved
7,8,Provisionally approved
8,9,Provisionally approved
9,10,Provisionally approved


In [283]:
approved_ids = [1, 2, 3, 4, 7, 8, 9, 10, 14, 15]

for sentence_id in approved_ids:
    english_sentence = english_sentences[sentence_id - 1]
    word_count = len(english_sentence.split())

    print(
        f"Sentence {sentence_id}: "
        f"{word_count} English words"
    )

Sentence 1: 13 English words
Sentence 2: 18 English words
Sentence 3: 12 English words
Sentence 4: 12 English words
Sentence 7: 28 English words
Sentence 8: 12 English words
Sentence 9: 21 English words
Sentence 10: 20 English words
Sentence 14: 14 English words
Sentence 15: 20 English words


In [284]:
from itertools import combinations

sentence_counts = {
    1: 13, 2: 18, 3: 12, 4: 12,
    7: 28, 8: 12, 9: 21, 10: 20,
    14: 14, 15: 20
}

for target in [29, 88, 147]:
    best = None

    ids = list(sentence_counts)

    for r in range(1, len(ids) + 1):
        for combo in combinations(ids, r):
            total = sum(sentence_counts[i] for i in combo)

            if best is None or abs(total - target) < abs(best[0] - target):
                best = (total, combo)

    print(f"\nTarget: {target} words")
    print("Selected sentences:", best[1])
    print("English words:", best[0])
    print("Actual percentage:", round(best[0] / 294 * 100, 2))


Target: 29 words
Selected sentences: (7,)
English words: 28
Actual percentage: 9.52

Target: 88 words
Selected sentences: (1, 3, 7, 9, 14)
English words: 88
Actual percentage: 29.93

Target: 147 words
Selected sentences: (1, 2, 3, 7, 9, 10, 14, 15)
English words: 146
Actual percentage: 49.66


In [285]:
selected_sentences = {
    "10": [7],
    "30": [1, 3, 7, 9, 14],
    "50": [1, 2, 3, 7, 9, 10, 14, 15]
}

def create_pilot_mixed(level):
    selected = selected_sentences[level]

    result = []

    for i in range(15):
        sentence_id = i + 1

        if sentence_id in selected:
            sentence = english_sentences[i].strip()
            if not sentence.endswith((".", "!", "?")):
                sentence += "."
        else:
            sentence = hindi_sentences[i].strip()

        result.append(sentence)

    return " ".join(result)


pilot_10 = create_pilot_mixed("10")
pilot_30 = create_pilot_mixed("30")
pilot_50 = create_pilot_mixed("50")


def calculate_cmi(text):
    words = text.split()
    english_words = sum(
        bool(re.search(r"[A-Za-z]", word))
        for word in words
    )

    return {
        "Total words": len(words),
        "English words": english_words,
        "English percentage": round(
            english_words / len(words) * 100, 2
        )
    }


for level, text in [
    ("10%", pilot_10),
    ("30%", pilot_30),
    ("50%", pilot_50)
]:
    print(f"\n{level} VERSION")
    print(calculate_cmi(text))


10% VERSION
{'Total words': 293, 'English words': 26, 'English percentage': 8.87}

30% VERSION
{'Total words': 286, 'English words': 86, 'English percentage': 30.07}

50% VERSION
{'Total words': 286, 'English words': 144, 'English percentage': 50.35}


In [286]:
pilot_10_adjusted = create_pilot_mixed("10")

parts = pilot_10_adjusted.split(
    hindi_sentences[1].strip()
)

pilot_10_adjusted = (
    parts[0]
    + english_sentences[1].strip() + ". "
    + parts[1]
)

print(calculate_cmi(pilot_10_adjusted))

{'Total words': 295, 'English words': 44, 'English percentage': 14.92}


In [287]:
pilot_10_adjusted = pilot_10.replace(
    "खराब सेहत के कारण",
    "because of poor health"
)

print(calculate_cmi(pilot_10_adjusted))

{'Total words': 293, 'English words': 30, 'English percentage': 10.24}


In [288]:
for level, text in [
    ("10%", pilot_10_adjusted),
    ("30%", pilot_30),
    ("50%", pilot_50)
]:
    print("\n" + "=" * 80)
    print(level, calculate_cmi(text))
    print("=" * 80)
    print(text)


10% {'Total words': 293, 'English words': 30, 'English percentage': 10.24}
शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. because of poor health बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके. फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी. " बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया. Among his memorable films, Naya Daur, released in 1957, reflects the ideals of the Nehru era, while Sadhana, released in 1958, raises questions about society's treatment of prostitution. गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची

In [289]:
pilot_10_final = pilot_10_adjusted.replace(
    "because of poor health बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.",
    "बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके because of poor health."
)

print(pilot_10_final)
print("\nCMI:", calculate_cmi(pilot_10_final))

शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके because of poor health. फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी. " बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया. Among his memorable films, Naya Daur, released in 1957, reflects the ideals of the Nehru era, while Sadhana, released in 1958, raises questions about society's treatment of prostitution. गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में शामिल है. हालांकि बीआर चोपड़ा की ज़्यादातर फ़िल्में किसी न किसी सामाजिक

In [290]:
import os
import pandas as pd

pilot_output = pd.DataFrame({
    "sample_id": ["080419_brchopra_festival"],
    "mixed_10_article": [pilot_10_final],
    "mixed_30_article": [pilot_30],
    "mixed_50_article": [pilot_50]
})

pilot_folder = os.path.join(
    base_path, "04_mixed_data", "pilot"
)

os.makedirs(pilot_folder, exist_ok=True)

pilot_file = os.path.join(
    pilot_folder, "FactIndic_Pilot_Mixed_Validated.csv"
)

pilot_output.to_csv(
    pilot_file,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", pilot_file)
print("Rows:", len(pilot_output))
print("Columns:", list(pilot_output.columns))

Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/pilot/FactIndic_Pilot_Mixed_Validated.csv
Rows: 1
Columns: ['sample_id', 'mixed_10_article', 'mixed_30_article', 'mixed_50_article']


In [291]:
check = pd.read_csv(pilot_file)

print("File exists:", os.path.exists(pilot_file))
print("Rows:", len(check))
print("Missing values:")
print(check.isnull().sum())

for col in [
    "mixed_10_article",
    "mixed_30_article",
    "mixed_50_article"
]:
    print(
        col,
        calculate_cmi(check.loc[0, col])
    )

File exists: True
Rows: 1
Missing values:
sample_id           0
mixed_10_article    0
mixed_30_article    0
mixed_50_article    0
dtype: int64
mixed_10_article {'Total words': 293, 'English words': 30, 'English percentage': 10.24}
mixed_30_article {'Total words': 286, 'English words': 86, 'English percentage': 30.07}
mixed_50_article {'Total words': 286, 'English words': 144, 'English percentage': 50.35}


In [292]:
key_terms = [
    "बीआर चोपड़ा",
    "यश चोपड़ा",
    "रवि चोपड़ा",
    "1957",
    "1958",
    "Naya Daur",
    "Sadhana",
    "Raj Babbar",
    "Bobby Bedi"
]

for level in ["10", "30", "50"]:
    text = check.loc[0, f"mixed_{level}_article"]

    print(f"\n{level}% VERSION")
    for term in key_terms:
        print(f"{term}: {term.lower() in text.lower()}")


10% VERSION
बीआर चोपड़ा: True
यश चोपड़ा: True
रवि चोपड़ा: True
1957: True
1958: True
Naya Daur: True
Sadhana: True
Raj Babbar: False
Bobby Bedi: False

30% VERSION
बीआर चोपड़ा: True
यश चोपड़ा: True
रवि चोपड़ा: True
1957: True
1958: True
Naya Daur: True
Sadhana: True
Raj Babbar: False
Bobby Bedi: False

50% VERSION
बीआर चोपड़ा: True
यश चोपड़ा: True
रवि चोपड़ा: True
1957: True
1958: True
Naya Daur: True
Sadhana: True
Raj Babbar: True
Bobby Bedi: True


In [293]:
name_variants = {
    "Raj Babbar": ["राज बब्बर", "Raj Babbar"],
    "Bobby Bedi": ["बॉबी बेदी", "Bobby Bedi"]
}

for level in ["10", "30", "50"]:
    text = check.loc[0, f"mixed_{level}_article"]

    print(f"\n{level}% VERSION")
    for name, variants in name_variants.items():
        found = any(v.lower() in text.lower() for v in variants)
        print(f"{name}: {found}")


10% VERSION
Raj Babbar: True
Bobby Bedi: True

30% VERSION
Raj Babbar: True
Bobby Bedi: True

50% VERSION
Raj Babbar: True
Bobby Bedi: True


In [294]:
for level in ["10", "30", "50"]:
    print(f"\n{'='*25} {level}% VERSION {'='*25}\n")
    print(check.loc[0, f"mixed_{level}_article"])


========================= 10% VERSION =========================

शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके because of poor health. फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी. " बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया. Among his memorable films, Naya Daur, released in 1957, reflects the ideals of the Nehru era, while Sadhana, released in 1958, raises questions about society's treatment of prostitution. गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में शामिल

In [295]:
review = pd.DataFrame([
    {
        "sample_id": "080419_brchopra_festival",
        "level": "10%",
        "fluency_score": 3,
        "meaning_preserved": "Yes",
        "names_numbers_preserved": "Yes",
        "review_status": "Reviewed - minor fluency edits needed",
        "comments": "English phrase interrupts Hindi sentence; other facts and names preserved."
    },
    {
        "sample_id": "080419_brchopra_festival",
        "level": "30%",
        "fluency_score": 3,
        "meaning_preserved": "Yes",
        "names_numbers_preserved": "Yes",
        "review_status": "Reviewed - fluency edits needed",
        "comments": "Some abrupt transitions between Hindi and English; quotations retained in Hindi."
    },
    {
        "sample_id": "080419_brchopra_festival",
        "level": "50%",
        "fluency_score": 3,
        "meaning_preserved": "Yes",
        "names_numbers_preserved": "Yes",
        "review_status": "Reviewed - fluency edits needed",
        "comments": "Meaning and key names preserved; some transitions need improvement."
    }
])

review_path = os.path.join(
    base_path,
    "04_mixed_data",
    "pilot",
    "FactIndic_Pilot_Fluency_Review.csv"
)

review.to_csv(review_path, index=False, encoding="utf-8-sig")

print("Saved:", review_path)
print(review)

Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/pilot/FactIndic_Pilot_Fluency_Review.csv
                  sample_id level  fluency_score meaning_preserved  \
0  080419_brchopra_festival   10%              3               Yes   
1  080419_brchopra_festival   30%              3               Yes   
2  080419_brchopra_festival   50%              3               Yes   

  names_numbers_preserved                          review_status  \
0                     Yes  Reviewed - minor fluency edits needed   
1                     Yes        Reviewed - fluency edits needed   
2                     Yes        Reviewed - fluency edits needed   

                                            comments  
0  English phrase interrupts Hindi sentence; othe...  
1  Some abrupt transitions between Hindi and Engl...  
2  Meaning and key names preserved; some transiti...  


In [296]:
pilot_10_revised = pilot_10_final.replace(
    "बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके because of poor health.",
    "बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके, as he was unwell."
)

print(pilot_10_revised)
print("\nUpdated CMI:", calculate_cmi(pilot_10_revised))

शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके, as he was unwell. फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी. " बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया. Among his memorable films, Naya Daur, released in 1957, reflects the ideals of the Nehru era, while Sadhana, released in 1958, raises questions about society's treatment of prostitution. गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में शामिल है. हालांकि बीआर चोपड़ा की ज़्यादातर फ़िल्में किसी न किसी सामाजिक मुद्

In [297]:
pilot_10_final = pilot_10_revised

check.loc[0, "mixed_10_article"] = pilot_10_final

check.to_csv(
    pilot_file,
    index=False,
    encoding="utf-8-sig"
)

print("Updated 10% version saved.")
print("CMI:", calculate_cmi(check.loc[0, "mixed_10_article"]))

Updated 10% version saved.
CMI: {'Total words': 293, 'English words': 30, 'English percentage': 10.24}


In [298]:
print("HINDI SOURCE:")
print(hindi_sentences[7])

print("\nENGLISH TRANSLATION:")
print(english_sentences[7])

HINDI SOURCE:
गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूची में शामिल है.

ENGLISH TRANSLATION:
Gumrah, Hamraaz, Kanoon and Nikah are also among the films he made


In [299]:
sentence_to_replace = hindi_sentences[7]
english_replacement = english_sentences[7]

pilot_30_revised = pilot_30.replace(
    sentence_to_replace,
    english_replacement
)

pilot_50_revised = pilot_50.replace(
    sentence_to_replace,
    english_replacement
)

print("30% CMI:", calculate_cmi(pilot_30_revised))
print("50% CMI:", calculate_cmi(pilot_50_revised))

30% CMI: {'Total words': 282, 'English words': 98, 'English percentage': 34.75}
50% CMI: {'Total words': 282, 'English words': 156, 'English percentage': 55.32}


In [300]:
approved_ids = [1, 2, 3, 4, 7, 8, 9, 10, 14, 15]

comparison = []

for i in approved_ids:
    hindi_count = len(hindi_sentences[i-1].split())
    english_count = len(english_sentences[i-1].split())

    comparison.append({
        "Sentence_ID": i,
        "Hindi_words": hindi_count,
        "English_words": english_count,
        "Difference": english_count - hindi_count
    })

comparison_df = pd.DataFrame(comparison)
print(comparison_df.to_string(index=False))

 Sentence_ID  Hindi_words  English_words  Difference
           1           13             13           0
           2           16             18           2
           3           13             12          -1
           4           14             12          -2
           7           32             28          -4
           8           16             12          -4
           9           26             21          -5
          10           21             20          -1
          14           15             14          -1
          15           21             20          -1


In [301]:
pilot_30_balanced = pilot_30_revised.replace(
    english_sentences[0],
    hindi_sentences[0]
)

pilot_50_balanced = pilot_50_revised.replace(
    english_sentences[0],
    hindi_sentences[0]
)

print("30%:", calculate_cmi(pilot_30_balanced))
print("50%:", calculate_cmi(pilot_50_balanced))

30%: {'Total words': 282, 'English words': 85, 'English percentage': 30.14}
50%: {'Total words': 282, 'English words': 143, 'English percentage': 50.71}


In [302]:
print("30% VERSION — UPDATED")
print(pilot_30_balanced)

print("\n50% VERSION — UPDATED")
print(pilot_50_balanced)

30% VERSION — UPDATED
शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई.. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. BR Chopra could not attend the ceremony because of his poor health. फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी. " बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया. Among his memorable films, Naya Daur, released in 1957, reflects the ideals of the Nehru era, while Sadhana, released in 1958, raises questions about society's treatment of prostitution. Gumrah, Hamraaz, Kanoon and Nikah are also among the films he made Although most of BR Chopra's films were based on social issues, th

In [303]:
pilot_30_balanced = (
    pilot_30_balanced
    .replace("गई..", "गई.")
    .replace(
        "among the films he made Although",
        "among the films he made. Although"
    )
)

pilot_50_balanced = (
    pilot_50_balanced
    .replace("गई..", "गई.")
    .replace(
        "among the films he made Although",
        "among the films he made. Although"
    )
)

print("30%:", calculate_cmi(pilot_30_balanced))
print("50%:", calculate_cmi(pilot_50_balanced))

30%: {'Total words': 282, 'English words': 85, 'English percentage': 30.14}
50%: {'Total words': 282, 'English words': 143, 'English percentage': 50.71}


In [304]:
check.loc[0, "mixed_10_article"] = pilot_10_final
check.loc[0, "mixed_30_article"] = pilot_30_balanced
check.loc[0, "mixed_50_article"] = pilot_50_balanced

check.to_csv(
    pilot_file,
    index=False,
    encoding="utf-8-sig"
)

print("Updated pilot saved:", pilot_file)

for col in [
    "mixed_10_article",
    "mixed_30_article",
    "mixed_50_article"
]:
    print(col, calculate_cmi(check.loc[0, col]))

Updated pilot saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/pilot/FactIndic_Pilot_Mixed_Validated.csv
mixed_10_article {'Total words': 293, 'English words': 30, 'English percentage': 10.24}
mixed_30_article {'Total words': 282, 'English words': 85, 'English percentage': 30.14}
mixed_50_article {'Total words': 282, 'English words': 143, 'English percentage': 50.71}


In [305]:
final_check = pd.read_csv(pilot_file)

print("Rows:", len(final_check))
print("Missing values:")
print(final_check.isnull().sum())

for level in ["10", "30", "50"]:
    text = final_check.loc[0, f"mixed_{level}_article"]

    print(f"\n{level}% VERSION")
    print("BR Chopra:", "बीआर चोपड़ा" in text or "BR Chopra" in text)
    print("Yash Chopra:", "यश चोपड़ा" in text or "Yash Chopra" in text)
    print("Ravi Chopra:", "रवि चोपड़ा" in text or "Ravi Chopra" in text)
    print("1957:", "1957" in text)
    print("1958:", "1958" in text)

Rows: 1
Missing values:
sample_id           0
mixed_10_article    0
mixed_30_article    0
mixed_50_article    0
dtype: int64

10% VERSION
BR Chopra: True
Yash Chopra: True
Ravi Chopra: True
1957: True
1958: True

30% VERSION
BR Chopra: True
Yash Chopra: True
Ravi Chopra: True
1957: True
1958: True

50% VERSION
BR Chopra: True
Yash Chopra: True
Ravi Chopra: True
1957: True
1958: True


In [306]:
review["review_status"] = "Revised - pending human re-review"
review["comments"] = (
    "Revised sentence choices and punctuation. "
    "CMI and key-name/date checks passed. "
    "Independent fluency review still pending."
)

review.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Review sheet updated:", review_path)
print(review[["level", "review_status", "comments"]])

Review sheet updated: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/pilot/FactIndic_Pilot_Fluency_Review.csv
  level                      review_status  \
0   10%  Revised - pending human re-review   
1   30%  Revised - pending human re-review   
2   50%  Revised - pending human re-review   

                                            comments  
0  Revised sentence choices and punctuation. CMI ...  
1  Revised sentence choices and punctuation. CMI ...  
2  Revised sentence choices and punctuation. CMI ...  


In [307]:
translation_path = os.path.join(
    base_path,
    "09_sentence_level_translation",
    "FactIndic_English_250_Reviewed.csv"
)

translation_df = pd.read_csv(translation_path)

print("Rows:", len(translation_df))
print("\nColumns:")
print(translation_df.columns.tolist())

print("\nSample:")
print(translation_df.head(2).to_string())

Rows: 250

Columns:
['id', 'complete_english_translation', 'reconstructed_sentence_count', 'manual_review_status']

Sample:
                      id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [308]:
print("Source rows:", len(source_250))
print("Translation rows:", len(translation_df))

print(
    "Unique source IDs:",
    source_250["id"].nunique()
)

print(
    "Unique translation IDs:",
    translation_df["id"].nunique()
)

source_ids = set(source_250["id"].astype(str))
translation_ids = set(translation_df["id"].astype(str))

print("Missing translations:", len(source_ids - translation_ids))
print("Extra translation IDs:", len(translation_ids - source_ids))

print("\nTranslation review status:")
print(translation_df["manual_review_status"].value_counts(dropna=False))

Source rows: 250
Translation rows: 250
Unique source IDs: 250
Unique translation IDs: 250
Missing translations: 0
Extra translation IDs: 0

Translation review status:
manual_review_status
Manually corrected                225
Human reviewed - no correction     25
Name: count, dtype: int64


In [309]:
import re

def split_sentences(text):
    return [
        s.strip()
        for s in re.split(r'(?<=[।.!?])\s+', str(text))
        if s.strip()
    ]

alignment_check = source_250[
    ["id", "hindi_article"]
].merge(
    translation_df[["id", "complete_english_translation"]],
    on="id",
    how="inner"
)

alignment_check["hindi_sentence_count"] = (
    alignment_check["hindi_article"]
    .apply(lambda x: len(split_sentences(x)))
)

alignment_check["english_sentence_count"] = (
    alignment_check["complete_english_translation"]
    .apply(lambda x: len(split_sentences(x)))
)

alignment_check["sentence_count_difference"] = (
    alignment_check["english_sentence_count"]
    - alignment_check["hindi_sentence_count"]
)

print(alignment_check[
    ["hindi_sentence_count", "english_sentence_count"]
].describe())

print("\nExact sentence-count matches:",
      (alignment_check["sentence_count_difference"] == 0).sum())

print("\nArticles with different sentence counts:",
      (alignment_check["sentence_count_difference"] != 0).sum())

       hindi_sentence_count  english_sentence_count
count            250.000000              250.000000
mean              20.856000               21.692000
std               13.237833               13.902888
min                5.000000                6.000000
25%               11.250000               12.000000
50%               16.000000               16.500000
75%               26.500000               28.000000
max               64.000000               69.000000

Exact sentence-count matches: 71

Articles with different sentence counts: 179


In [310]:
easy_candidates = alignment_check[
    alignment_check["sentence_count_difference"] == 0
].copy()

easy_candidates["total_sentences"] = (
    easy_candidates["hindi_sentence_count"]
)

easy_candidates = easy_candidates.sort_values(
    "total_sentences"
)

print("Matching candidates:", len(easy_candidates))

print(
    easy_candidates[
        ["id", "total_sentences"]
    ].head(10).to_string(index=False)
)

Matching candidates: 71
                                               id  total_sentences
       140901_iraq_un_human_rights_commission_fma                7
                                   india-44912345                7
150526_shahrukh_tweet_salman_bajrangi_bhaijaan_hk                7
                   150110_france_huge_marches_sdp                8
             141218_russia_putin_rouble_stable_sn                8
                        060530_reservation_strike                8
                          031031_sharon_palestine                8
        130912_guinnness_record_bulgarian_swim_sr                9
                            070516_assam_violence                9
                            031031_russia_removal                9


In [311]:
candidate_id = "140901_iraq_un_human_rights_commission_fma"

row_hi = source_250[
    source_250["id"] == candidate_id
].iloc[0]

row_en = translation_df[
    translation_df["id"] == candidate_id
].iloc[0]

hi_sentences = split_sentences(row_hi["hindi_article"])
en_sentences = split_sentences(
    row_en["complete_english_translation"]
)

print("Hindi sentences:", len(hi_sentences))
print("English sentences:", len(en_sentences))

for i, (hi, en) in enumerate(
    zip(hi_sentences, en_sentences), start=1
):
    print(f"\nSENTENCE {i}")
    print("HINDI:", hi)
    print("ENGLISH:", en)

Hindi sentences: 7
English sentences: 7

SENTENCE 1
HINDI: संयुक्त राष्ट्र मानवधिकार संस्था की उप प्रमुख फ्लाव्यू पानसेरी ने कहा है कि ऐसा समझा जाता है कि इस्लामिक स्टेट ने नागरिकों पर जानबूझकर सुनियोजित तरीक़े से हमले किए हैं.
ENGLISH: The deputy head of the UN human rights body, Flavio Panseri, has said that the Islamic State is believed to have deliberately and systematically carried out attacks on civilians.

SENTENCE 2
HINDI: इनमें निशाना बनाकर हत्या, जबरन धर्म परिवर्तन, दासता, यौन हिंसा और सारे समुदाय को बंधक बनाना शामिल है.
ENGLISH: These include targeted killings, forced conversions, slavery, sexual violence, and taking entire communities hostage.

SENTENCE 3
HINDI: संस्था आईएस के मानवाधिकार हनन के मामलों की जांच के लिए एक आपात समिति इराक़ भेजने के लिए तैयार हो गई है.
ENGLISH: The organization has agreed to send an emergency committee to Iraq to investigate IS human rights abuses.

SENTENCE 4
HINDI: इराक़ के मानवाधिकार मामलों के मंत्री का कहना है कि उनका देश चरमपंथ के दानव का स

In [312]:
candidate_alignment = pd.DataFrame({
    "sample_id": [candidate_id] * 7,
    "sentence_id": list(range(1, 8)),
    "hindi_sentence": hi_sentences,
    "english_sentence": en_sentences,
    "review_status": [
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Keep Hindi - app instruction",
        "Keep Hindi - social media instruction"
    ]
})

print(candidate_alignment[
    ["sentence_id", "review_status"]
].to_string(index=False))

 sentence_id                         review_status
           1                Provisionally approved
           2                Provisionally approved
           3                Provisionally approved
           4                Provisionally approved
           5                Provisionally approved
           6          Keep Hindi - app instruction
           7 Keep Hindi - social media instruction


In [313]:
alignment_folder = os.path.join(
    base_path, "04_mixed_data", "alignment"
)

os.makedirs(alignment_folder, exist_ok=True)

alignment_path = os.path.join(
    alignment_folder, "FactIndic_Manual_Sentence_Alignment.csv"
)

candidate_alignment.to_csv(
    alignment_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", alignment_path)
print("Rows:", len(candidate_alignment))

Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/alignment/FactIndic_Manual_Sentence_Alignment.csv
Rows: 7


In [314]:
candidate_id = "india-44912345"

row_hi = source_250[
    source_250["id"] == candidate_id
].iloc[0]

row_en = translation_df[
    translation_df["id"] == candidate_id
].iloc[0]

hi_sentences = split_sentences(row_hi["hindi_article"])
en_sentences = split_sentences(
    row_en["complete_english_translation"]
)

print("Hindi sentences:", len(hi_sentences))
print("English sentences:", len(en_sentences))

for i, (hi, en) in enumerate(
    zip(hi_sentences, en_sentences), start=1
):
    print(f"\nSENTENCE {i}")
    print("HINDI:", hi)
    print("ENGLISH:", en)

Hindi sentences: 7
English sentences: 7

SENTENCE 1
HINDI: जीएसटी काउंसिल की 28वीं बैठक की अध्यक्षता अभी वित्त मंत्री का कार्यभार संभाल रहे पीयूष गोयल ने की.
ENGLISH: The 28th meeting of the GST Council was chaired by Piyush Goyal, who was then holding charge of the Finance Ministry.

SENTENCE 2
HINDI: जीएसटी काउंसिल ने फ़ैसला लिया है कि सैनिटरी नैपकिन को जीएसटी से बाहर कर दिया जाए यानी सैनिटरी नैपकिन पर जीएसटी की दर को शून्य कर दिया गया है.
ENGLISH: The GST Council decided to exempt sanitary napkins from GST, bringing the tax rate on them down to zero.

SENTENCE 3
HINDI: अब तक सैनिटरी नैपकिन पर 12 फ़ीसदी जीएसटी लगाया जा रहा था.
ENGLISH: Until then, sanitary napkins had been taxed at 12%.

SENTENCE 4
HINDI: जीएसटी के दायरे से बाहर होने वाली चीजें -सैनिटरी पैड्स -राखियां (बहुमूल्यों रत्नों से न बनी हो) -संगमरमर से बनी मूर्तियां -झाड़ू बनाने में इस्तेमाल होने वाला कच्चा माल -साल के पत्ते जीएसटी 12% से घटाकर 5% -हैंडलूम दरी -फास्फोरिक एसिड युक्त उर्वरक -बुनी हुई टोपियां (1000 रुपये से कम 

In [315]:
candidate_alignment_2 = pd.DataFrame({
    "sample_id": [candidate_id] * 7,
    "sentence_id": list(range(1, 8)),
    "hindi_sentence": hi_sentences,
    "english_sentence": en_sentences,
    "review_status": [
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved - list formatting differs",
        "Keep Hindi - related-story headlines",
        "Keep Hindi - headline and app instruction",
        "Keep Hindi - social media instruction"
    ]
})

alignment_path = os.path.join(
    base_path,
    "04_mixed_data",
    "alignment",
    "FactIndic_Manual_Sentence_Alignment.csv"
)

all_alignments = pd.concat(
    [pd.read_csv(alignment_path), candidate_alignment_2],
    ignore_index=True
)

all_alignments.to_csv(
    alignment_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved alignment rows:", len(all_alignments))
print("Candidates reviewed:", all_alignments["sample_id"].nunique())

Saved alignment rows: 14
Candidates reviewed: 2


In [316]:
candidate_id = "150526_shahrukh_tweet_salman_bajrangi_bhaijaan_hk"

row_hi = source_250[
    source_250["id"] == candidate_id
].iloc[0]

row_en = translation_df[
    translation_df["id"] == candidate_id
].iloc[0]

hi_sentences = split_sentences(row_hi["hindi_article"])
en_sentences = split_sentences(
    row_en["complete_english_translation"]
)

print("Hindi sentences:", len(hi_sentences))
print("English sentences:", len(en_sentences))

for i, (hi, en) in enumerate(
    zip(hi_sentences, en_sentences), start=1
):
    print(f"\nSENTENCE {i}")
    print("HINDI:", hi)
    print("ENGLISH:", en)

Hindi sentences: 7
English sentences: 7

SENTENCE 1
HINDI: शाहरुख़ ने अपने ट्विटर अकांउट से सलमान ख़ान की फ़िल्म 'बजरंगी भाईजान' का पोस्टर ट्वीट करते हुए लिखा, “भाई होना हीरो होने से बड़ा है, भाईजान आ रहे हैं ईद 2015 पर.” इसके बाद शाहरुख़ लिखते हैं कि आपको सलमान की फ़िल्म का पहला लुक कैसा लगा ?
ENGLISH: Shah Rukh Khan tweeted the poster of Salman Khan's film Bajrangi Bhaijaan from his Twitter account, writing, "Being a brother is greater than being a hero.

SENTENCE 2
HINDI: शाहरुख़ का ये ट्वीट साफ़ करता है कि किंग खान और दंबंग के बीच अब सब कुछ सही है और ये दोस्ती शायद एक और करण अर्जुन में तब्दील हो सकती है.
ENGLISH: Bhaijaan is coming this Eid, 2015." Shah Rukh then asked, "How did you like the first look of Salman's film?" His tweet makes it clear that things are now fine between King Khan and Dabangg, and that their friendship could perhaps turn into another Karan Arjun.

SENTENCE 3
HINDI: शाहरुख़ और सलमान के बीच कुछ साल पहले कटरीना कैफ़ की बर्थडे पार्टी में झगड़ा हो गया था जिसके बा

In [317]:
candidate_alignment_3 = pd.DataFrame({
    "sample_id": [candidate_id] * 7,
    "sentence_id": list(range(1, 8)),
    "hindi_sentence": hi_sentences,
    "english_sentence": en_sentences,
    "review_status": [
        "Provisionally approved - English split across sentences 1 and 2",
        "Provisionally approved - English continuation in sentence 2",
        "Provisionally approved",
        "Provisionally approved - check stray Hindi word",
        "Provisionally approved",
        "Keep Hindi - app instruction",
        "Keep Hindi - social media instruction"
    ]
})

print(candidate_alignment_3[
    ["sentence_id", "review_status"]
].to_string(index=False))

 sentence_id                                                   review_status
           1 Provisionally approved - English split across sentences 1 and 2
           2     Provisionally approved - English continuation in sentence 2
           3                                          Provisionally approved
           4                 Provisionally approved - check stray Hindi word
           5                                          Provisionally approved
           6                                    Keep Hindi - app instruction
           7                           Keep Hindi - social media instruction


In [318]:
candidate_id = "150110_france_huge_marches_sdp"

row_hi = source_250[
    source_250["id"] == candidate_id
].iloc[0]

row_en = translation_df[
    translation_df["id"] == candidate_id
].iloc[0]

hi_sentences = split_sentences(row_hi["hindi_article"])
en_sentences = split_sentences(
    row_en["complete_english_translation"]
)

print("Hindi sentences:", len(hi_sentences))
print("English sentences:", len(en_sentences))

for i, (hi, en) in enumerate(
    zip(hi_sentences, en_sentences), start=1
):
    print(f"\nSENTENCE {i}")
    print("HINDI:", hi)
    print("ENGLISH:", en)

Hindi sentences: 8
English sentences: 8

SENTENCE 1
HINDI: सबसे बड़ी रैली टूलूज़ में हुई जिसमें अस्सी हज़ार लोग शामिल हुए.
ENGLISH: The largest rally took place in Toulouse, where 80,000 people took part.

SENTENCE 2
HINDI: मार्च के दौरान कुछ प्रदर्शकारियों के हाथों में बैनर थे जिन पर लिखा था- 'मैं नस्ली भेदभाव के ख़िलाफ़ हूं', 'मैं शार्ली हूं.' पेरिस में प्लेस द ला रिपब्लिक नामक जगह पर लोगों ने अपने कलम रखे जो फ्रांस में शार्ली एब्डो पत्रिका के दफ्तर पर बीते बुधवार हमले के बाद अवज्ञा का प्रतीक बन गया है.
ENGLISH: During the march, some protesters carried banners reading, "I am against racial discrimination" and "I am Charlie."

In Paris, people placed their pens at the Place de la République, which had become a symbol of defiance following last Wednesday's attack on the offices of the French magazine Charlie Hebdo.

SENTENCE 3
HINDI: राजधानी पेरिस में रविवार को भी यूनिटी मार्च निकालने की तैयारी है जिसके लिए सुरक्षा के कड़े इंतज़ाम किए गए हैं.
ENGLISH: A Unity March was also scheduled 

In [319]:
candidate_alignment_4 = pd.DataFrame({
    "sample_id": [candidate_id] * 8,
    "sentence_id": list(range(1, 9)),
    "hindi_sentence": hi_sentences,
    "english_sentence": en_sentences,
    "review_status": [
        "Provisionally approved",
        "Provisionally approved - English split into two",
        "Provisionally approved",
        "Provisionally approved - check stray Hindi word",
        "Provisionally approved",
        "Provisionally approved",
        "Keep Hindi - app instruction",
        "Keep Hindi - social media instruction"
    ]
})

print(candidate_alignment_4[
    ["sentence_id", "review_status"]
].to_string(index=False))

 sentence_id                                   review_status
           1                          Provisionally approved
           2 Provisionally approved - English split into two
           3                          Provisionally approved
           4 Provisionally approved - check stray Hindi word
           5                          Provisionally approved
           6                          Provisionally approved
           7                    Keep Hindi - app instruction
           8           Keep Hindi - social media instruction


In [320]:
candidate_id = "141218_russia_putin_rouble_stable_sn"

row_hi = source_250[
    source_250["id"] == candidate_id
].iloc[0]

row_en = translation_df[
    translation_df["id"] == candidate_id
].iloc[0]

hi_sentences = split_sentences(row_hi["hindi_article"])
en_sentences = split_sentences(
    row_en["complete_english_translation"]
)

print("Hindi sentences:", len(hi_sentences))
print("English sentences:", len(en_sentences))

for i, (hi, en) in enumerate(
    zip(hi_sentences, en_sentences), start=1
):
    print(f"\nSENTENCE {i}")
    print("HINDI:", hi)
    print("ENGLISH:", en)

Hindi sentences: 8
English sentences: 8

SENTENCE 1
HINDI: पुतिन ने ये बात वर्ष के अंत में होने वाले संवाददाता सम्मेलन को संबोधित करते हुए कही.
ENGLISH: Putin made these remarks while addressing his year-end press conference.

SENTENCE 2
HINDI: पुतिन ने रूबल में गिरावट के लिए 'बाहरी कारणों' को दोषी ठहराया है, लेकिन ये भी माना है कि केंद्रीय बैंक को पहले ही इस मसले पर काम करना शुरु कर देना चाहिए था.
ENGLISH: He blamed "external factors" for the fall in the value of the rouble, but also acknowledged that the central bank should have started addressing the issue earlier.

SENTENCE 3
HINDI: उन्होंने यह भी स्वीकार किया कि रूस अपनी अर्थव्यवस्था में विविधता लाने में विफल रहा, जिसको तेल की गिरती क़ीमतों और यूक्रेन संकट के तहत प्रतिबंधों से बड़ा झटका लगा.
ENGLISH: He admitted that Russia had failed to diversify its economy, which had been severely affected by falling oil prices and sanctions imposed over the Ukraine crisis.

SENTENCE 4
HINDI: राष्ट्रपति पुतिन ने कहा कि देश का मुद्रा भंडार अर्थव

In [321]:
candidate_alignment_5 = pd.DataFrame({
    "sample_id": [candidate_id] * 8,
    "sentence_id": list(range(1, 9)),
    "hindi_sentence": hi_sentences,
    "english_sentence": en_sentences,
    "review_status": [
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved - check stray Hindi word",
        "Provisionally approved",
        "Keep Hindi - app instruction",
        "Keep Hindi - social media instruction"
    ]
})

print(candidate_alignment_5[
    ["sentence_id", "review_status"]
].to_string(index=False))

 sentence_id                                   review_status
           1                          Provisionally approved
           2                          Provisionally approved
           3                          Provisionally approved
           4                          Provisionally approved
           5 Provisionally approved - check stray Hindi word
           6                          Provisionally approved
           7                    Keep Hindi - app instruction
           8           Keep Hindi - social media instruction


In [322]:
alignment_path = os.path.join(
    base_path,
    "04_mixed_data",
    "alignment",
    "FactIndic_Manual_Sentence_Alignment.csv"
)

new_reviews = pd.concat(
    [
        candidate_alignment_3,
        candidate_alignment_4,
        candidate_alignment_5
    ],
    ignore_index=True
)

existing_reviews = pd.read_csv(alignment_path)

all_alignments = pd.concat(
    [existing_reviews, new_reviews],
    ignore_index=True
)

all_alignments = all_alignments.drop_duplicates(
    subset=["sample_id", "sentence_id"],
    keep="last"
)

all_alignments.to_csv(
    alignment_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved successfully:", alignment_path)
print("Total alignment rows:", len(all_alignments))
print("Candidates reviewed:", all_alignments["sample_id"].nunique())
print("\nCandidate IDs:")
print(all_alignments["sample_id"].unique())

Saved successfully: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/alignment/FactIndic_Manual_Sentence_Alignment.csv
Total alignment rows: 37
Candidates reviewed: 5

Candidate IDs:
['140901_iraq_un_human_rights_commission_fma' 'india-44912345'
 '150526_shahrukh_tweet_salman_bajrangi_bhaijaan_hk'
 '150110_france_huge_marches_sdp' '141218_russia_putin_rouble_stable_sn']


In [323]:
batch_2_ids = [
    "060530_reservation_strike",
    "031031_sharon_palestine",
    "130912_guinnness_record_bulgarian_swim_sr",
    "070516_assam_violence",
    "031031_russia_removal"
]

for candidate_id in batch_2_ids:

    row_hi = source_250[
        source_250["id"] == candidate_id
    ].iloc[0]

    row_en = translation_df[
        translation_df["id"] == candidate_id
    ].iloc[0]

    hi_sentences = split_sentences(row_hi["hindi_article"])
    en_sentences = split_sentences(
        row_en["complete_english_translation"]
    )

    print("\n" + "=" * 100)
    print("ARTICLE ID:", candidate_id)
    print("Hindi sentences:", len(hi_sentences))
    print("English sentences:", len(en_sentences))
    print("=" * 100)

    for i, (hi, en) in enumerate(
        zip(hi_sentences, en_sentences), start=1
    ):
        print(f"\nSENTENCE {i}")
        print("HINDI:", hi)
        print("ENGLISH:", en)

    if len(hi_sentences) != len(en_sentences):
        print("\nWARNING: Sentence counts differ. Manual alignment needed.")


ARTICLE ID: 060530_reservation_strike
Hindi sentences: 8
English sentences: 8

SENTENCE 1
HINDI: देश भर के मेडिकल कॉलेजों के प्रतिनिधियों ने सोमवार को सुप्रीम कोर्ट के आदेश पर विचार करने के बाद हड़ताल जारी रखने का निर्णय लिया.
ENGLISH: Representatives of medical colleges across the country on Monday decided to continue the strike after considering the Supreme Court order.

SENTENCE 2
HINDI: उल्लेखनीय है कि केंद्र सरकार के समर्थन से चल रहे उच्च शिक्षा संस्थानों में पिछड़ी जातियों के छात्रों को आरक्षण देने के विरोध में मेडिकल छात्र हड़ताल कर रहे हैं.
ENGLISH: Notably, medical students are on strike to protest against the reservation for students from backward castes in higher education institutions run with the support of the central government.

SENTENCE 3
HINDI: उधर सरकार ने इस हड़ताल के बावजूद ये घोषणा की है कि आरक्षण जून 2007 से लागू हो जाएगा.
ENGLISH: Despite the strike, the government has announced that the reservation will come into effect from June 2007.

SENTENCE 4
HINDI: हालां

In [324]:
batch_2_reviews = {
    "060530_reservation_strike": [
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved"
    ],

    "031031_sharon_palestine": [
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved - English split; heading included",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved"
    ],

    "130912_guinnness_record_bulgarian_swim_sr": [
        "Needs manual realignment - English includes detail from Hindi sentence 2",
        "Needs manual realignment - English content split across sentences 1 and 2",
        "Provisionally approved",
        "Needs manual realignment - quotation split across English sentences",
        "Needs manual realignment - quotation and Guinness information split",
        "Provisionally approved - maps to English sentence 7",
        "Provisionally approved - maps to English sentence 8",
        "Keep Hindi - app instruction; English content differs",
        "Keep Hindi - social media footer; English content differs"
    ],

    "070516_assam_violence": [
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved - check heading word",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved"
    ],

    "031031_russia_removal": [
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved - check heading word",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved"
    ]
}

new_records = []

for article_id, statuses in batch_2_reviews.items():

    hi = split_sentences(
        source_250.loc[
            source_250["id"] == article_id,
            "hindi_article"
        ].iloc[0]
    )

    en = split_sentences(
        translation_df.loc[
            translation_df["id"] == article_id,
            "complete_english_translation"
        ].iloc[0]
    )

    assert len(hi) == len(statuses), article_id

    for i, (h, e, status) in enumerate(
        zip(hi, en, statuses), start=1
    ):
        new_records.append({
            "sample_id": article_id,
            "sentence_id": i,
            "hindi_sentence": h,
            "english_sentence": e,
            "review_status": status
        })

batch_2_df = pd.DataFrame(new_records)

alignment_path = os.path.join(
    base_path,
    "04_mixed_data",
    "alignment",
    "FactIndic_Manual_Sentence_Alignment.csv"
)

existing_reviews = pd.read_csv(alignment_path)

all_alignments = pd.concat(
    [existing_reviews, batch_2_df],
    ignore_index=True
).drop_duplicates(
    subset=["sample_id", "sentence_id"],
    keep="last"
)

all_alignments.to_csv(
    alignment_path,
    index=False,
    encoding="utf-8-sig"
)

print("Batch 2 records:", len(batch_2_df))
print("Total saved records:", len(all_alignments))
print("Total candidates:", all_alignments["sample_id"].nunique())
print("Saved:", alignment_path)

Batch 2 records: 43
Total saved records: 80
Total candidates: 10
Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/alignment/FactIndic_Manual_Sentence_Alignment.csv


In [325]:
# Get the 10 candidates already reviewed
reviewed_ids = set(all_alignments["sample_id"].unique())

# Select the next 5 unreviewed candidates
batch_3_ids = easy_candidates[
    ~easy_candidates["id"].isin(reviewed_ids)
]["id"].head(5).tolist()

print("BATCH 3 CANDIDATES:", batch_3_ids)
print("Number of candidates:", len(batch_3_ids))

for candidate_id in batch_3_ids:

    row_hi = source_250[
        source_250["id"] == candidate_id
    ].iloc[0]

    row_en = translation_df[
        translation_df["id"] == candidate_id
    ].iloc[0]

    hi_sentences = split_sentences(row_hi["hindi_article"])
    en_sentences = split_sentences(
        row_en["complete_english_translation"]
    )

    print("\n" + "=" * 100)
    print("ARTICLE ID:", candidate_id)
    print("Hindi sentences:", len(hi_sentences))
    print("English sentences:", len(en_sentences))
    print("=" * 100)

    for i, (hi, en) in enumerate(
        zip(hi_sentences, en_sentences), start=1
    ):
        print(f"\nSENTENCE {i}")
        print("HINDI:", hi)
        print("ENGLISH:", en)

    if len(hi_sentences) != len(en_sentences):
        print("\nWARNING: Sentence counts differ.")

BATCH 3 CANDIDATES: ['050112_nepalmao_amnesty', '080324_chowdhry_pakistan', '040709_barrier_split', '081217_ganguly_yuvraj_ac', '130912_nepal_strike_ssr']
Number of candidates: 5

ARTICLE ID: 050112_nepalmao_amnesty
Hindi sentences: 9
English sentences: 9

SENTENCE 1
HINDI: संगठन ने माओवादी नेताओं से अनुरोध किया है कि वे इन मामलों की जाँच करें और दोषी व्यक्तियों के ख़िलाफ़ कार्रवाई करें.
ENGLISH: Amnesty International has asked Maoist leaders to investigate these cases and take action against those responsible.

SENTENCE 2
HINDI: नेपाली माओवादियों के नेता प्रचंड के नाम लिखे एक 'खुले पत्र' में एमनेस्टी ने कहा है कि मानवाधिकार हनन के विशेष मामले उसके सामने आए हैं.
ENGLISH: In an open letter addressed to Prachanda, the leader of Nepal's Maoists, Amnesty said that it had received reports of specific human rights violations.

SENTENCE 3
HINDI: संगठन के एशिया-प्रशांत क्षेत्र के निदेशक इंग्रिड मसाज का कहना है कि प्रचंड सभी आम नागरिकों की सुरक्षा सुनिश्चित करें और अगवाह किए गए लोगों को बिना शर

In [326]:
batch_3_reviews = {
    "050112_nepalmao_amnesty": [
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved"
    ],

    "080324_chowdhry_pakistan": [
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved - multiple sentences in one pair",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved"
    ],

    "040709_barrier_split": [
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved - multiple ideas in one pair",
        "Provisionally approved - check heading word",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved"
    ],

    "081217_ganguly_yuvraj_ac": [
        "Provisionally approved",
        "Provisionally approved - English contains two ideas",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved"
    ],

    "130912_nepal_strike_ssr": [
        "Provisionally approved - sentence contains two ideas",
        "Provisionally approved",
        "Provisionally approved - check heading word",
        "Provisionally approved",
        "Provisionally approved - Hindi wording needs correction",
        "Provisionally approved",
        "Needs correction - meaning mismatch about constitution draft",
        "Keep Hindi - app instruction",
        "Keep Hindi - social media instruction"
    ]
}

new_records = []

for article_id, statuses in batch_3_reviews.items():

    hi = split_sentences(
        source_250.loc[
            source_250["id"] == article_id,
            "hindi_article"
        ].iloc[0]
    )

    en = split_sentences(
        translation_df.loc[
            translation_df["id"] == article_id,
            "complete_english_translation"
        ].iloc[0]
    )

    assert len(hi) == len(statuses), (
        article_id, len(hi), len(statuses)
    )

    for i, (h, e, status) in enumerate(
        zip(hi, en, statuses), start=1
    ):
        new_records.append({
            "sample_id": article_id,
            "sentence_id": i,
            "hindi_sentence": h,
            "english_sentence": e,
            "review_status": status
        })

batch_3_df = pd.DataFrame(new_records)

alignment_path = os.path.join(
    base_path,
    "04_mixed_data",
    "alignment",
    "FactIndic_Manual_Sentence_Alignment.csv"
)

existing_reviews = pd.read_csv(alignment_path)

all_alignments = pd.concat(
    [existing_reviews, batch_3_df],
    ignore_index=True
).drop_duplicates(
    subset=["sample_id", "sentence_id"],
    keep="last"
)

all_alignments.to_csv(
    alignment_path,
    index=False,
    encoding="utf-8-sig"
)

print("Batch 3 records:", len(batch_3_df))
print("Total saved records:", len(all_alignments))
print("Total candidates:", all_alignments["sample_id"].nunique())
print("Saved:", alignment_path)

Batch 3 records: 45
Total saved records: 125
Total candidates: 15
Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/alignment/FactIndic_Manual_Sentence_Alignment.csv


In [327]:
import pandas as pd
import os

batch_3_reviews = {
    "050112_nepalmao_amnesty": [
        "Approved",
        "Approved",
        "Approved",
        "Approved",
        "Approved",
        "Approved",
        "Approved",
        "Approved",
        "Approved"
    ],

    "080324_chowdhry_pakistan": [
        "Approved",
        "Approved",
        "Approved",
        "Approved",
        "Approved",
        "Approved",
        "Approved",
        "Approved",
        "Approved"
    ],

    "040709_barrier_split": [
        "Approved",
        "Approved",
        "Approved",
        "Approved",
        "Approved",
        "Provisionally approved - Hindi wording needs cleanup",
        "Approved",
        "Approved",
        "Approved"
    ],

    "081217_ganguly_yuvraj_ac": [
        "Approved",
        "Provisionally approved - English sentence split",
        "Approved",
        "Approved",
        "Approved",
        "Approved",
        "Approved",
        "Approved",
        "Approved"
    ],

    "130912_nepal_strike_ssr": [
        "Provisionally approved - Hindi sentence needs cleanup",
        "Approved",
        "Provisionally approved - heading contamination",
        "Approved",
        "Provisionally approved - Hindi wording needs cleanup",
        "Approved",
        "Approved",
        "Keep Hindi - app instruction",
        "Keep Hindi - social media footer"
    ]
}

batch_3_records = []

for candidate_id, statuses in batch_3_reviews.items():

    hi_row = source_250[
        source_250["id"] == candidate_id
    ].iloc[0]

    en_row = translation_df[
        translation_df["id"] == candidate_id
    ].iloc[0]

    hi_sentences = split_sentences(hi_row["hindi_article"])
    en_sentences = split_sentences(
        en_row["complete_english_translation"]
    )

    assert len(hi_sentences) == len(statuses), (
        f"Hindi/status count mismatch: {candidate_id}"
    )

    assert len(en_sentences) == len(statuses), (
        f"English/status count mismatch: {candidate_id}"
    )

    for i, (hi, en, status) in enumerate(
        zip(hi_sentences, en_sentences, statuses), start=1
    ):
        batch_3_records.append({
            "sample_id": candidate_id,
            "sentence_id": i,
            "hindi_sentence": hi,
            "english_sentence": en,
            "alignment_status": status
        })

batch_3_df = pd.DataFrame(batch_3_records)

if os.path.exists(alignment_path):
    existing = pd.read_csv(alignment_path)
    combined = pd.concat(
        [existing, batch_3_df],
        ignore_index=True
    )
else:
    combined = batch_3_df.copy()

combined = combined.drop_duplicates(
    subset=["sample_id", "sentence_id"],
    keep="last"
)

combined.to_csv(alignment_path, index=False)

print("Batch 3 records:", len(batch_3_df))
print("Total saved records:", len(combined))
print("Total unique candidates:", combined["sample_id"].nunique())
print("Saved successfully:", os.path.exists(alignment_path))
print("File:", alignment_path)

Batch 3 records: 45
Total saved records: 125
Total unique candidates: 15
Saved successfully: True
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/alignment/FactIndic_Manual_Sentence_Alignment.csv


In [328]:
batch_4_ids = easy_candidates[
    ~easy_candidates["id"].isin(all_alignments["sample_id"].unique())
]["id"].head(5).tolist()

print("BATCH 4 CANDIDATES:", batch_4_ids)
print("Number of candidates:", len(batch_4_ids))

for candidate_id in batch_4_ids:

    row_hi = source_250[
        source_250["id"] == candidate_id
    ].iloc[0]

    row_en = translation_df[
        translation_df["id"] == candidate_id
    ].iloc[0]

    hi_sentences = split_sentences(row_hi["hindi_article"])
    en_sentences = split_sentences(
        row_en["complete_english_translation"]
    )

    print("\n" + "=" * 100)
    print("ARTICLE ID:", candidate_id)
    print("Hindi sentences:", len(hi_sentences))
    print("English sentences:", len(en_sentences))
    print("=" * 100)

    for i, (hi, en) in enumerate(
        zip(hi_sentences, en_sentences), start=1
    ):
        print(f"\nSENTENCE {i}")
        print("HINDI:", hi)
        print("ENGLISH:", en)

    if len(hi_sentences) != len(en_sentences):
        print("\nWARNING: Sentence counts differ.")

BATCH 4 CANDIDATES: ['060603_taliban_karzai', '051017_iraq_rebels', '040725_bangladesh_floods', '031113_tdp_elections', '040511_shahbaz_profile']
Number of candidates: 5

ARTICLE ID: 060603_taliban_karzai
Hindi sentences: 10
English sentences: 10

SENTENCE 1
HINDI: इस बीच अफ़गानिस्तान के राष्ट्रपति हामिद करज़ई ने पुलिस महकमे में बड़ा फ़ेरबदल करते हुए कई आला अधिकारियों को बर्ख़ास्त कर दिया है.
ENGLISH: Meanwhile, Afghan President Hamid Karzai has made a major reshuffle in the police department, firing several top officials.

SENTENCE 2
HINDI: कंधार के गवर्नर के प्रवक्ता दाऊद अहमदी ने मुठभेड़ के बारे में जानकारी देते हुए बताया कि संदिग्ध तालेबान चरमपंथियों ने शनिवार को मियाना शीन शहर में स्थित एक पुलिस चौकी पर हमला किया जिसके बाद सुरक्षाकर्मियों ने जवाबी कार्रवाई की.
ENGLISH: Security personnel retaliated after suspected Taliban militants attacked a police post in the town of Miyana Shein on Saturday, Kandahar governor's spokesman Dawood Ahmadi said, briefing about the encounter.

SENTEN

In [329]:
import pandas as pd
import os

batch_4_reviews = {
    "060603_taliban_karzai": [
        "Approved", "Approved", "Approved", "Approved", "Approved",
        "Provisionally approved - heading contamination",
        "Approved", "Approved", "Approved", "Approved"
    ],

    "051017_iraq_rebels": [
        "Approved", "Approved", "Approved", "Approved",
        "Provisionally approved - heading contamination",
        "Approved", "Approved", "Approved", "Approved", "Approved"
    ],

    "040725_bangladesh_floods": [
        "Provisionally approved", "Provisionally approved",
        "Provisionally approved", "Provisionally approved",
        "Provisionally approved", "Provisionally approved",
        "Provisionally approved", "Provisionally approved",
        "Provisionally approved",
        "Provisionally approved - check source context"
    ],

    "031113_tdp_elections": [
        "Approved", "Approved", "Approved", "Approved",
        "Provisionally approved - awkward Hindi wording",
        "Approved", "Approved", "Approved", "Approved", "Approved"
    ],

    "040511_shahbaz_profile": [
        "Approved", "Approved", "Approved", "Approved", "Approved",
        "Provisionally approved - heading contamination",
        "Provisionally approved - Hindi typo",
        "Approved", "Approved", "Approved"
    ]
}

batch_4_records = []

for candidate_id, statuses in batch_4_reviews.items():

    hi_row = source_250[
        source_250["id"] == candidate_id
    ].iloc[0]

    en_row = translation_df[
        translation_df["id"] == candidate_id
    ].iloc[0]

    hi_sentences = split_sentences(hi_row["hindi_article"])
    en_sentences = split_sentences(
        en_row["complete_english_translation"]
    )

    assert len(hi_sentences) == len(statuses), candidate_id
    assert len(en_sentences) == len(statuses), candidate_id

    for i, (hi, en, status) in enumerate(
        zip(hi_sentences, en_sentences, statuses), start=1
    ):
        batch_4_records.append({
            "sample_id": candidate_id,
            "sentence_id": i,
            "hindi_sentence": hi,
            "english_sentence": en,
            "alignment_status": status
        })

batch_4_df = pd.DataFrame(batch_4_records)

if os.path.exists(alignment_path):
    existing = pd.read_csv(alignment_path)
    combined = pd.concat(
        [existing, batch_4_df],
        ignore_index=True
    )
else:
    combined = batch_4_df.copy()

combined = combined.drop_duplicates(
    subset=["sample_id", "sentence_id"],
    keep="last"
)

combined.to_csv(alignment_path, index=False)

all_alignments = pd.read_csv(alignment_path)

print("Batch 4 records:", len(batch_4_df))
print("Total saved records:", len(all_alignments))
print("Total unique candidates:", all_alignments["sample_id"].nunique())
print("Saved successfully:", os.path.exists(alignment_path))
print("File:", alignment_path)

Batch 4 records: 50
Total saved records: 175
Total unique candidates: 20
Saved successfully: True
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/alignment/FactIndic_Manual_Sentence_Alignment.csv


In [330]:
# Reload saved alignment records to avoid using stale variables
all_alignments = pd.read_csv(alignment_path)

# Select the next 5 unreviewed candidates
batch_5_ids = easy_candidates[
    ~easy_candidates["id"].isin(
        all_alignments["sample_id"].unique()
    )
]["id"].head(5).tolist()

print("BATCH 5 CANDIDATES:", batch_5_ids)
print("Number of candidates:", len(batch_5_ids))

for candidate_id in batch_5_ids:

    row_hi = source_250[
        source_250["id"] == candidate_id
    ].iloc[0]

    row_en = translation_df[
        translation_df["id"] == candidate_id
    ].iloc[0]

    hi_sentences = split_sentences(row_hi["hindi_article"])
    en_sentences = split_sentences(
        row_en["complete_english_translation"]
    )

    print("\n" + "=" * 100)
    print("ARTICLE ID:", candidate_id)
    print("Hindi sentences:", len(hi_sentences))
    print("English sentences:", len(en_sentences))
    print("=" * 100)

    for i, (hi, en) in enumerate(
        zip(hi_sentences, en_sentences), start=1
    ):
        print(f"\nSENTENCE {i}")
        print("HINDI:", hi)
        print("ENGLISH:", en)

    if len(hi_sentences) != len(en_sentences):
        print("\nWARNING: Sentence counts differ.")

BATCH 5 CANDIDATES: ['071013_advani_pm', '051209_tamilnadu_cyclone', '050107_pravasi_pm', '031114_bush_intelligence', '070425_ccpa_inconclusive']
Number of candidates: 5

ARTICLE ID: 071013_advani_pm
Hindi sentences: 10
English sentences: 10

SENTENCE 1
HINDI: परमाणु समझौते पर अपना रुख़ बदलने के लिए जहाँ वामपंथी दलों ने प्रधानमंत्री की सराहना की है वहीं भाजपा ने उन्हें आड़े हाथों लिया है.
ENGLISH: While the Left parties praised the Prime Minister for changing his position on the nuclear deal, the BJP strongly criticised him.

SENTENCE 2
HINDI: आडवाणी ने जारी एक बयान में कहा कि परमाणु समझौते पर अपने रुख़ में परिवर्तन करके मनमोहन सिंह ने इस पद की वजह से जो थोड़ी बहुत विश्वसनीयता हासिल की थी, वह भी गवां दी है.
ENGLISH: In a statement, Advani said that by changing his position on the nuclear deal, Manmohan Singh had lost even the little credibility he had gained through holding the office.

SENTENCE 3
HINDI: विपक्ष के नेता लालकृष्ण आडवाणी का कहना था कि मनमोहन सिंह का यह कहना कि परमाणु समझौ

In [331]:
import pandas as pd
import os

batch_5_reviews = {
    "071013_advani_pm": [
        "Approved", "Approved", "Approved",
        "Provisionally approved - heading contamination",
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved"
    ],

    "051209_tamilnadu_cyclone": [
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved"
    ],

    "050107_pravasi_pm": [
        "Provisionally approved - English paragraph split",
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved"
    ],

    "031114_bush_intelligence": [
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved", "Provisionally approved - awkward Hindi wording",
        "Approved", "Approved", "Approved", "Approved"
    ],

    "070425_ccpa_inconclusive": [
        "Approved",
        "Provisionally approved - awkward Hindi wording",
        "Approved", "Approved", "Approved", "Approved",
        "Provisionally approved - sentence boundary needs review",
        "Approved", "Approved", "Approved", "Approved"
    ]
}

batch_5_records = []

for candidate_id, statuses in batch_5_reviews.items():

    hi_row = source_250[
        source_250["id"] == candidate_id
    ].iloc[0]

    en_row = translation_df[
        translation_df["id"] == candidate_id
    ].iloc[0]

    hi_sentences = split_sentences(hi_row["hindi_article"])
    en_sentences = split_sentences(
        en_row["complete_english_translation"]
    )

    assert len(hi_sentences) == len(statuses), candidate_id
    assert len(en_sentences) == len(statuses), candidate_id

    for i, (hi, en, status) in enumerate(
        zip(hi_sentences, en_sentences, statuses), start=1
    ):
        batch_5_records.append({
            "sample_id": candidate_id,
            "sentence_id": i,
            "hindi_sentence": hi,
            "english_sentence": en,
            "alignment_status": status
        })

batch_5_df = pd.DataFrame(batch_5_records)

if os.path.exists(alignment_path):
    existing = pd.read_csv(alignment_path)
    combined = pd.concat(
        [existing, batch_5_df],
        ignore_index=True
    )
else:
    combined = batch_5_df.copy()

combined = combined.drop_duplicates(
    subset=["sample_id", "sentence_id"],
    keep="last"
)

combined.to_csv(alignment_path, index=False)

all_alignments = pd.read_csv(alignment_path)

print("Batch 5 records:", len(batch_5_df))
print("Total saved records:", len(all_alignments))
print("Total unique candidates:", all_alignments["sample_id"].nunique())
print("Saved successfully:", os.path.exists(alignment_path))
print("File:", alignment_path)

Batch 5 records: 53
Total saved records: 228
Total unique candidates: 25
Saved successfully: True
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/alignment/FactIndic_Manual_Sentence_Alignment.csv


In [332]:
all_alignments = pd.read_csv(alignment_path)

batch_6_ids = easy_candidates[
    ~easy_candidates["id"].isin(
        all_alignments["sample_id"].unique()
    )
]["id"].head(5).tolist()

print("BATCH 6 CANDIDATES:", batch_6_ids)
print("Number of candidates:", len(batch_6_ids))

for candidate_id in batch_6_ids:

    row_hi = source_250[
        source_250["id"] == candidate_id
    ].iloc[0]

    row_en = translation_df[
        translation_df["id"] == candidate_id
    ].iloc[0]

    hi_sentences = split_sentences(row_hi["hindi_article"])
    en_sentences = split_sentences(
        row_en["complete_english_translation"]
    )

    print("\n" + "=" * 100)
    print("ARTICLE ID:", candidate_id)
    print("Hindi sentences:", len(hi_sentences))
    print("English sentences:", len(en_sentences))
    print("=" * 100)

    for i, (hi, en) in enumerate(
        zip(hi_sentences, en_sentences), start=1
    ):
        print(f"\nSENTENCE {i}")
        print("HINDI:", hi)
        print("ENGLISH:", en)

    if len(hi_sentences) != len(en_sentences):
        print("\nWARNING: Sentence counts differ.")

BATCH 6 CANDIDATES: ['070902_khanduri_win', '060106_rajasthan_rail', '050625_building_collapse', '160412_assam_gogoi_fir_ml', '050505_voteclose_britain']
Number of candidates: 5

ARTICLE ID: 070902_khanduri_win
Hindi sentences: 11
English sentences: 11

SENTENCE 1
HINDI: धूमकोट सीट के लिए उपचुनाव 29 अगस्त को हुआ था.
ENGLISH: The by-election for the Dhumkot seat was held on 29 August.

SENTENCE 2
HINDI: रविवार को हुई मतगणना में खंडूरी ने 14 हज़ार वोटों के अंतर से जीत दर्ज की.
ENGLISH: In Sunday's counting, Khanduri won by a margin of 14,000 votes.

SENTENCE 3
HINDI: भुवन चंद्र खंडूरी ने भारतीय जनता पार्टी की ओर से चुनाव लड़ा था.
ENGLISH: Bhuvan Chandra Khanduri contested on behalf of the Bharatiya Janata Party.

SENTENCE 4
HINDI: खंडूरी ने मार्च में मुख्यमंत्री पद की शपथ ली थी.
ENGLISH: Khanduri was sworn in as the chief minister in March.

SENTENCE 5
HINDI: उस समय वह पौड़ी क्षेत्र से लोकसभा सदस्य थे.
ENGLISH: At that time, he was a member of the Lok Sabha from the Pauri constituency.



In [333]:
import pandas as pd
import os

batch_6_reviews = {
    "070902_khanduri_win": [
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved",
        "Provisionally approved - heading contamination",
        "Approved", "Approved", "Approved", "Approved"
    ],

    "060106_rajasthan_rail": [
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved", "Approved"
    ],

    "050625_building_collapse": [
        "Provisionally approved", "Approved", "Approved",
        "Provisionally approved - Hindi spelling errors",
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved", "Approved"
    ],

    "160412_assam_gogoi_fir_ml": [
        "Approved", "Approved", "Approved", "Approved",
        "Approved",
        "Provisionally approved - heading contamination",
        "Approved", "Approved", "Approved", "Approved",
        "Keep Hindi - app instruction",
        "Keep Hindi - social media footer"
    ],

    "050505_voteclose_britain": [
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved", "Approved", "Approved"
    ]
}

batch_6_records = []

for candidate_id, statuses in batch_6_reviews.items():

    hi_row = source_250[
        source_250["id"] == candidate_id
    ].iloc[0]

    en_row = translation_df[
        translation_df["id"] == candidate_id
    ].iloc[0]

    hi_sentences = split_sentences(hi_row["hindi_article"])
    en_sentences = split_sentences(
        en_row["complete_english_translation"]
    )

    assert len(hi_sentences) == len(statuses), candidate_id
    assert len(en_sentences) == len(statuses), candidate_id

    for i, (hi, en, status) in enumerate(
        zip(hi_sentences, en_sentences, statuses), start=1
    ):
        batch_6_records.append({
            "sample_id": candidate_id,
            "sentence_id": i,
            "hindi_sentence": hi,
            "english_sentence": en,
            "alignment_status": status
        })

batch_6_df = pd.DataFrame(batch_6_records)

if os.path.exists(alignment_path):
    existing = pd.read_csv(alignment_path)
    combined = pd.concat(
        [existing, batch_6_df],
        ignore_index=True
    )
else:
    combined = batch_6_df.copy()

combined = combined.drop_duplicates(
    subset=["sample_id", "sentence_id"],
    keep="last"
)

combined.to_csv(alignment_path, index=False)

all_alignments = pd.read_csv(alignment_path)

print("Batch 6 records:", len(batch_6_df))
print("Total saved records:", len(all_alignments))
print("Total unique candidates:", all_alignments["sample_id"].nunique())
print("Saved successfully:", os.path.exists(alignment_path))
print("File:", alignment_path)

Batch 6 records: 57
Total saved records: 285
Total unique candidates: 30
Saved successfully: True
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/alignment/FactIndic_Manual_Sentence_Alignment.csv


In [334]:
all_alignments = pd.read_csv(alignment_path)

print("Total saved records:", len(all_alignments))
print("Unique candidates:", all_alignments["sample_id"].nunique())

print("\nLast 5 saved candidates:")
print(all_alignments["sample_id"].drop_duplicates().tail(5).tolist())

print("\nBatch 6 saved:")
print(
    set([
        "070902_khanduri_win",
        "060106_rajasthan_rail",
        "050625_building_collapse",
        "160412_assam_gogoi_fir_ml",
        "050505_voteclose_britain"
    ]).issubset(set(all_alignments["sample_id"]))
)

Total saved records: 285
Unique candidates: 30

Last 5 saved candidates:
['070902_khanduri_win', '060106_rajasthan_rail', '050625_building_collapse', '160412_assam_gogoi_fir_ml', '050505_voteclose_britain']

Batch 6 saved:
True


In [335]:
all_alignments = pd.read_csv(alignment_path)

print("Total saved records:", len(all_alignments))
print("Unique candidates:", all_alignments["sample_id"].nunique())

print("\nLast 5 saved candidates:")
print(all_alignments["sample_id"].drop_duplicates().tail(5).tolist())

print("\nBatch 6 saved:")
print(
    set([
        "070902_khanduri_win",
        "060106_rajasthan_rail",
        "050625_building_collapse",
        "160412_assam_gogoi_fir_ml",
        "050505_voteclose_britain"
    ]).issubset(set(all_alignments["sample_id"]))
)

Total saved records: 285
Unique candidates: 30

Last 5 saved candidates:
['070902_khanduri_win', '060106_rajasthan_rail', '050625_building_collapse', '160412_assam_gogoi_fir_ml', '050505_voteclose_britain']

Batch 6 saved:
True


In [336]:
all_alignments = pd.read_csv(alignment_path)

batch_7_ids = easy_candidates[
    ~easy_candidates["id"].isin(
        all_alignments["sample_id"].unique()
    )
]["id"].head(5).tolist()

print("BATCH 7 CANDIDATES:", batch_7_ids)
print("Number of candidates:", len(batch_7_ids))

for candidate_id in batch_7_ids:

    row_hi = source_250[
        source_250["id"] == candidate_id
    ].iloc[0]

    row_en = translation_df[
        translation_df["id"] == candidate_id
    ].iloc[0]

    hi_sentences = split_sentences(row_hi["hindi_article"])
    en_sentences = split_sentences(
        row_en["complete_english_translation"]
    )

    print("\n" + "=" * 100)
    print("ARTICLE ID:", candidate_id)
    print("Hindi sentences:", len(hi_sentences))
    print("English sentences:", len(en_sentences))
    print("=" * 100)

    for i, (hi, en) in enumerate(
        zip(hi_sentences, en_sentences), start=1
    ):
        print(f"\nSENTENCE {i}")
        print("HINDI:", hi)
        print("ENGLISH:", en)

    if len(hi_sentences) != len(en_sentences):
        print("\nWARNING: Sentence counts differ.")

BATCH 7 CANDIDATES: ['150725_one_killed_kashmir_grenade_attack_pm', '051105_sahubuddin_arrest', '041003_pwg_security', '070620_heart_survival', '050228_tax_income']
Number of candidates: 5

ARTICLE ID: 150725_one_killed_kashmir_grenade_attack_pm
Hindi sentences: 12
English sentences: 12

SENTENCE 1
HINDI: वारदात की जानकारी देते हुए समाचार एजेंसी पीटीआई को एक पुलिस अफ़सर ने बताया कि एक संदिग्ध चरमपंथियों ने अचबल बस स्टैंड पर एक हथगोला फेंका.
ENGLISH: A police officer told news agency PTI that a suspected militant had thrown a hand grenade at the Achabal bus stand.

SENTENCE 2
HINDI: इसके टुकड़े लगने से सुरक्षा बल का जवान और छह राहगीर घायल हो गए.
ENGLISH: A security personnel member and six passers-by were injured by the grenade fragments.

SENTENCE 3
HINDI: घायलों में एक मोहम्मद जब्बार को श्रीनगर के एक अस्पताल में दाख़िल कराया गया, जहां उनकी मौत हो गई.
ENGLISH: One of the injured, Mohammad Jabbar, was admitted to a hospital in Srinagar, where he later died.

SENTENCE 4
HINDI: 'छापेमारी 

In [337]:
import pandas as pd
import os

batch_7_reviews = {
    "150725_one_killed_kashmir_grenade_attack_pm": [
        "Approved", "Approved", "Approved",
        "Provisionally approved - heading included; English split",
        "Approved", "Approved", "Approved",
        "Provisionally approved - stray heading contamination",
        "Approved", "Approved",
        "Keep Hindi - app instruction",
        "Keep Hindi - social media footer"
    ],

    "051105_sahubuddin_arrest": [
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved", "Approved", "Approved"
    ],

    "041003_pwg_security": [
        "Approved", "Approved", "Approved",
        "Provisionally approved - heading contamination",
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved", "Approved", "Approved"
    ],

    "070620_heart_survival": [
        "Approved", "Approved", "Approved", "Approved",
        "Approved",
        "Needs manual boundary review - quote and heading combined",
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved"
    ],

    "050228_tax_income": [
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved",
        "Provisionally approved - heading contamination",
        "Approved", "Approved", "Approved", "Approved", "Approved"
    ]
}

batch_7_records = []

for candidate_id, statuses in batch_7_reviews.items():

    hi_row = source_250[
        source_250["id"] == candidate_id
    ].iloc[0]

    en_row = translation_df[
        translation_df["id"] == candidate_id
    ].iloc[0]

    hi_sentences = split_sentences(hi_row["hindi_article"])
    en_sentences = split_sentences(
        en_row["complete_english_translation"]
    )

    assert len(hi_sentences) == len(statuses), candidate_id
    assert len(en_sentences) == len(statuses), candidate_id

    for i, (hi, en, status) in enumerate(
        zip(hi_sentences, en_sentences, statuses), start=1
    ):
        batch_7_records.append({
            "sample_id": candidate_id,
            "sentence_id": i,
            "hindi_sentence": hi,
            "english_sentence": en,
            "alignment_status": status
        })

batch_7_df = pd.DataFrame(batch_7_records)

existing = pd.read_csv(alignment_path)

combined = pd.concat(
    [existing, batch_7_df],
    ignore_index=True
)

combined = combined.drop_duplicates(
    subset=["sample_id", "sentence_id"],
    keep="last"
)

combined.to_csv(alignment_path, index=False)

all_alignments = pd.read_csv(alignment_path)

print("Batch 7 records:", len(batch_7_df))
print("Total saved records:", len(all_alignments))
print("Total unique candidates:", all_alignments["sample_id"].nunique())
print("Saved successfully:", os.path.exists(alignment_path))
print("File:", alignment_path)

Batch 7 records: 60
Total saved records: 345
Total unique candidates: 35
Saved successfully: True
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/alignment/FactIndic_Manual_Sentence_Alignment.csv


In [338]:
all_alignments = pd.read_csv(alignment_path)

batch_7_ids = [
    "150725_one_killed_kashmir_grenade_attack_pm",
    "051105_sahubuddin_arrest",
    "041003_pwg_security",
    "070620_heart_survival",
    "050228_tax_income"
]

saved_ids = set(all_alignments["sample_id"].unique())

print("Total saved records:", len(all_alignments))
print("Unique candidates:", all_alignments["sample_id"].nunique())
print("Batch 7 saved:", set(batch_7_ids).issubset(saved_ids))
print("Batch 7 candidates found:",
      [x for x in batch_7_ids if x in saved_ids])

Total saved records: 345
Unique candidates: 35
Batch 7 saved: True
Batch 7 candidates found: ['150725_one_killed_kashmir_grenade_attack_pm', '051105_sahubuddin_arrest', '041003_pwg_security', '070620_heart_survival', '050228_tax_income']


In [339]:
all_alignments = pd.read_csv(alignment_path)

batch_8_ids = easy_candidates[
    ~easy_candidates["id"].isin(
        all_alignments["sample_id"].unique()
    )
]["id"].head(5).tolist()

print("BATCH 8 CANDIDATES:", batch_8_ids)
print("Number of candidates:", len(batch_8_ids))

for candidate_id in batch_8_ids:

    row_hi = source_250[
        source_250["id"] == candidate_id
    ].iloc[0]

    row_en = translation_df[
        translation_df["id"] == candidate_id
    ].iloc[0]

    hi_sentences = split_sentences(row_hi["hindi_article"])
    en_sentences = split_sentences(
        row_en["complete_english_translation"]
    )

    print("\n" + "=" * 100)
    print("ARTICLE ID:", candidate_id)
    print("Hindi sentences:", len(hi_sentences))
    print("English sentences:", len(en_sentences))
    print("=" * 100)

    for i, (hi, en) in enumerate(
        zip(hi_sentences, en_sentences), start=1
    ):
        print(f"\nSENTENCE {i}")
        print("HINDI:", hi)
        print("ENGLISH:", en)

    if len(hi_sentences) != len(en_sentences):
        print("\nWARNING: Sentence counts differ.")

BATCH 8 CANDIDATES: ['india-39447746', '080830_orissa_violence', '080301_benazir_suspect', '080801_india_iaea', '140206_egypt_al-sisi_presidential_election_sr']
Number of candidates: 5

ARTICLE ID: india-39447746
Hindi sentences: 13
English sentences: 13

SENTENCE 1
HINDI: पुलिस के शिकंजे में आया संदीप उपाध्याय साक्षी जोशी ने फ़ेसबुक पोस्ट के ज़रिए इसकी जानकारी दी है.
ENGLISH: Sakshi Joshi has announced through a Facebook post that Sandeep Upadhyay, who had been wanted by the police, has been arrested.

SENTENCE 2
HINDI: हाल ही में साक्षी ने फ़ेसबुक पर अपना स्टेटस अपडेट किया था.
ENGLISH: Recently, Sakshi updated her Facebook status.

SENTENCE 3
HINDI: उस स्टेटस पर संदीप उपाध्याय नाम के एक शख़्स ने कहा था कि अगर उत्तर प्रदेश में रहना है तो योगी आदित्यनाथ का समर्थन करना होगा.
ENGLISH: In response, a man named Sandeep Upadhyay said that anyone who wanted to live in Uttar Pradesh would have to support Yogi Adityanath.

SENTENCE 4
HINDI: साक्षी ने कहा था कि वह यूपी रहेंगी और योगी-योगी नहीं 

In [340]:
import pandas as pd
import os

batch_8_reviews = {
    "india-39447746": [
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved", "Approved", "Approved",
        "Needs manual boundary review - quotation and following statement split",
        "Approved", "Approved",
        "Keep Hindi - app instruction",
        "Keep Hindi - social media footer"
    ],

    "080830_orissa_violence": [
        "Approved", "Approved", "Approved", "Approved",
        "Approved",
        "Provisionally approved - heading contamination",
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved", "Approved"
    ],

    "080301_benazir_suspect": [
        "Approved", "Approved", "Approved",
        "Provisionally approved - heading contamination",
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved", "Approved", "Approved", "Approved"
    ],

    "080801_india_iaea": [
        "Approved", "Approved",
        "Needs manual boundary review - continuation of previous sentence",
        "Approved", "Approved",
        "Provisionally approved - heading contamination",
        "Approved", "Approved", "Approved", "Approved",
        "Needs manual boundary review - quotation and following statement combined",
        "Needs manual boundary review - shifted sentence content",
        "Approved", "Approved"
    ],

    "140206_egypt_al-sisi_presidential_election_sr": [
        "Approved",
        "Provisionally approved - multiple quotations and paragraph split",
        "Approved", "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved",
        "Provisionally approved - quotation and following paragraph split",
        "Approved", "Approved",
        "Keep Hindi - app instruction",
        "Keep Hindi - social media footer"
    ]
}

batch_8_records = []

for candidate_id, statuses in batch_8_reviews.items():

    hi_row = source_250[
        source_250["id"] == candidate_id
    ].iloc[0]

    en_row = translation_df[
        translation_df["id"] == candidate_id
    ].iloc[0]

    hi_sentences = split_sentences(hi_row["hindi_article"])
    en_sentences = split_sentences(
        en_row["complete_english_translation"]
    )

    assert len(hi_sentences) == len(statuses), candidate_id
    assert len(en_sentences) == len(statuses), candidate_id

    for i, (hi, en, status) in enumerate(
        zip(hi_sentences, en_sentences, statuses), start=1
    ):
        batch_8_records.append({
            "sample_id": candidate_id,
            "sentence_id": i,
            "hindi_sentence": hi,
            "english_sentence": en,
            "alignment_status": status
        })

batch_8_df = pd.DataFrame(batch_8_records)

existing = pd.read_csv(alignment_path)

combined = pd.concat(
    [existing, batch_8_df],
    ignore_index=True
)

combined = combined.drop_duplicates(
    subset=["sample_id", "sentence_id"],
    keep="last"
)

combined.to_csv(alignment_path, index=False)

all_alignments = pd.read_csv(alignment_path)

print("Batch 8 records:", len(batch_8_df))
print("Total saved records:", len(all_alignments))
print("Total unique candidates:", all_alignments["sample_id"].nunique())
print("Saved successfully:", os.path.exists(alignment_path))
print("File:", alignment_path)

Batch 8 records: 67
Total saved records: 412
Total unique candidates: 40
Saved successfully: True
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/alignment/FactIndic_Manual_Sentence_Alignment.csv


In [341]:
all_alignments = pd.read_csv(alignment_path)

batch_8_ids = [
    "india-39447746",
    "080830_orissa_violence",
    "080301_benazir_suspect",
    "080801_india_iaea",
    "140206_egypt_al-sisi_presidential_election_sr"
]

saved_ids = set(all_alignments["sample_id"].unique())

print("Total saved records:", len(all_alignments))
print("Unique candidates:", all_alignments["sample_id"].nunique())
print("Batch 8 saved:", set(batch_8_ids).issubset(saved_ids))
print("Batch 8 candidates found:",
      [x for x in batch_8_ids if x in saved_ids])

Total saved records: 412
Unique candidates: 40
Batch 8 saved: True
Batch 8 candidates found: ['india-39447746', '080830_orissa_violence', '080301_benazir_suspect', '080801_india_iaea', '140206_egypt_al-sisi_presidential_election_sr']


In [342]:
all_alignments = pd.read_csv(alignment_path)

batch_9_ids = easy_candidates[
    ~easy_candidates["id"].isin(
        all_alignments["sample_id"].unique()
    )
]["id"].head(5).tolist()

print("BATCH 9 CANDIDATES:", batch_9_ids)
print("Number of candidates:", len(batch_9_ids))

for candidate_id in batch_9_ids:

    row_hi = source_250[
        source_250["id"] == candidate_id
    ].iloc[0]

    row_en = translation_df[
        translation_df["id"] == candidate_id
    ].iloc[0]

    hi_sentences = split_sentences(row_hi["hindi_article"])
    en_sentences = split_sentences(
        row_en["complete_english_translation"]
    )

    print("\n" + "=" * 100)
    print("ARTICLE ID:", candidate_id)
    print("Hindi sentences:", len(hi_sentences))
    print("English sentences:", len(en_sentences))
    print("=" * 100)

    for i, (hi, en) in enumerate(
        zip(hi_sentences, en_sentences), start=1
    ):
        print(f"\nSENTENCE {i}")
        print("HINDI:", hi)
        print("ENGLISH:", en)

    if len(hi_sentences) != len(en_sentences):
        print("\nWARNING: Sentence counts differ.")

BATCH 9 CANDIDATES: ['040510_fillipino_election', '130910_sehwag_gambhir_pujara_pp', 'international-41395655', '080711_lanka_attack', '060117_indopak_cbm']
Number of candidates: 5

ARTICLE ID: 040510_fillipino_election
Hindi sentences: 14
English sentences: 14

SENTENCE 1
HINDI: इस चुनाव प्रचार के दौरान 29 उम्मीदवारों समेत 90 से भी ज़्यादा लोगों की मौत हो चुकी है.
ENGLISH: More than 90 people, including 29 candidates, have died during the election campaign.

SENTENCE 2
HINDI: विद्रोहियों के हमलों और मतदान में धांधली की धमकियों के बीच रविवार को राष्ट्रपति पद के सभी उम्मीदवारों ने मिलकर शांतिपूर्ण मतदान के लिए प्रार्थना की.
ENGLISH: Amid attacks by rebels and threats of electoral fraud, all the presidential candidates gathered on Sunday to pray for peaceful voting.

SENTENCE 3
HINDI: इसके लिए देश भर के पोलिंग बूथों में क़रीब 230,000 सुरक्षाकर्मी तैनात किए गए हैं.
ENGLISH: Around 230,000 security personnel have been deployed at polling stations across the country.

SENTENCE 4
HINDI: हालाँ

In [343]:
import pandas as pd
import os

batch_9_reviews = {
    "040510_fillipino_election": [
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved",
        "Provisionally approved - heading contamination",
        "Approved", "Approved", "Approved", "Approved",
        "Provisionally approved - heading contamination",
        "Approved", "Approved"
    ],

    "130910_sehwag_gambhir_pujara_pp": [
        "Needs manual realignment - sentence order mismatch"
    ] * 15,

    "international-41395655": [
        "Needs manual realignment - headline contamination and sentence order mismatch"
    ] * 15,

    "080711_lanka_attack": [
        "Approved", "Approved", "Approved", "Approved",
        "Approved",
        "Provisionally approved - heading contamination",
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved", "Approved", "Approved", "Approved"
    ],

    "060117_indopak_cbm": [
        "Approved", "Approved", "Approved", "Approved",
        "Approved",
        "Provisionally approved - heading contamination",
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved", "Approved", "Approved", "Approved"
    ]
}

batch_9_records = []

for candidate_id, statuses in batch_9_reviews.items():

    hi_row = source_250[
        source_250["id"] == candidate_id
    ].iloc[0]

    en_row = translation_df[
        translation_df["id"] == candidate_id
    ].iloc[0]

    hi_sentences = split_sentences(hi_row["hindi_article"])
    en_sentences = split_sentences(
        en_row["complete_english_translation"]
    )

    assert len(hi_sentences) == len(statuses), candidate_id
    assert len(en_sentences) == len(statuses), candidate_id

    for i, (hi, en, status) in enumerate(
        zip(hi_sentences, en_sentences, statuses), start=1
    ):
        batch_9_records.append({
            "sample_id": candidate_id,
            "sentence_id": i,
            "hindi_sentence": hi,
            "english_sentence": en,
            "alignment_status": status
        })

batch_9_df = pd.DataFrame(batch_9_records)

existing = pd.read_csv(alignment_path)

combined = pd.concat(
    [existing, batch_9_df],
    ignore_index=True
)

combined = combined.drop_duplicates(
    subset=["sample_id", "sentence_id"],
    keep="last"
)

combined.to_csv(alignment_path, index=False)

all_alignments = pd.read_csv(alignment_path)

print("Batch 9 records:", len(batch_9_df))
print("Total saved records:", len(all_alignments))
print("Total unique candidates:", all_alignments["sample_id"].nunique())
print("Saved successfully:", os.path.exists(alignment_path))
print("File:", alignment_path)

Batch 9 records: 74
Total saved records: 486
Total unique candidates: 45
Saved successfully: True
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/alignment/FactIndic_Manual_Sentence_Alignment.csv


In [344]:
all_alignments = pd.read_csv(alignment_path)

batch_10_ids = easy_candidates[
    ~easy_candidates["id"].isin(
        all_alignments["sample_id"].unique()
    )
]["id"].head(5).tolist()

print("BATCH 10 CANDIDATES:", batch_10_ids)
print("Number of candidates:", len(batch_10_ids))

for candidate_id in batch_10_ids:

    row_hi = source_250[
        source_250["id"] == candidate_id
    ].iloc[0]

    row_en = translation_df[
        translation_df["id"] == candidate_id
    ].iloc[0]

    hi_sentences = split_sentences(row_hi["hindi_article"])
    en_sentences = split_sentences(
        row_en["complete_english_translation"]
    )

    print("\n" + "=" * 100)
    print("ARTICLE ID:", candidate_id)
    print("Hindi sentences:", len(hi_sentences))
    print("English sentences:", len(en_sentences))
    print("=" * 100)

    for i, (hi, en) in enumerate(
        zip(hi_sentences, en_sentences), start=1
    ):
        print(f"\nSENTENCE {i}")
        print("HINDI:", hi)
        print("ENGLISH:", en)

    if len(hi_sentences) != len(en_sentences):
        print("\nWARNING: Sentence counts differ.")

BATCH 10 CANDIDATES: ['061015_katsav_police', '050812_baflights_cancel', '051121_aids_india', '150213_msg_film_released_pm', '080416_up_results']
Number of candidates: 5

ARTICLE ID: 061015_katsav_police
Hindi sentences: 15
English sentences: 15

SENTENCE 1
HINDI: इसराइली पुलिस कई हफ़्तों से इस मामले की जाँच कर रही थी.
ENGLISH: Israeli police had been investigating the case for several weeks and had informed Attorney General Menachem Mazuz of their findings and recommendations.

SENTENCE 2
HINDI: एटॉर्नी जनरल मेनाशेम माज़ूज़ को इसराइली पुलिस ने अपनी राय से अवगत करवा दिया है.
ENGLISH: The final decision on whether President Moshe Katsav should face trial now rests with the attorney general.

SENTENCE 3
HINDI: अब अंतिम फ़ैसला एटॉर्नी जनरल को ही करना है कि राष्ट्रपति कात्साव पर मुक़दमा चले या नहीं.
ENGLISH: President Katsav denies allegations that he forced two female employees to have sexual relations with him.

SENTENCE 4
HINDI: राष्ट्रपति कात्साव इस आरोप का खंडन करते हैं कि उन्होंने दो

In [345]:
batch_10_reviews = {
    "061015_katsav_police": [
        "Needs manual realignment - English combines first two Hindi sentences"
    ] * 15,

    "050812_baflights_cancel": [
        "Approved", "Approved", "Approved", "Approved",
        "Needs manual realignment - sentence boundary shift",
        "Needs manual realignment - heading and boundary shift",
        "Needs manual realignment - heading and boundary shift",
        "Needs manual realignment - sentence boundary shift",
        "Needs manual realignment - sentence boundary shift",
        "Needs manual realignment - sentence boundary shift",
        "Needs manual realignment - sentence boundary shift",
        "Needs manual realignment - sentence boundary shift",
        "Needs manual realignment - sentence boundary shift",
        "Needs manual realignment - combined English sentence",
        "Needs manual realignment - combined English sentence",
        "Approved"
    ],

    "051121_aids_india": [
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved", "Approved", "Approved",
        "Provisionally approved - heading contamination",
        "Approved", "Approved", "Approved"
    ],

    "150213_msg_film_released_pm": [
        "Needs manual realignment - sentence boundary mismatch",
        "Needs manual realignment - sentence boundary mismatch",
        "Needs manual realignment - sentence boundary mismatch",
        "Approved",
        "Provisionally approved - heading contamination",
        "Needs manual realignment - content shifted",
        "Needs manual realignment - stray heading and content shift",
        "Needs manual realignment - sentence boundary mismatch",
        "Needs manual realignment - merged quotations and heading",
        "Needs manual realignment - multiple paragraphs",
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved"
    ],

    "080416_up_results": [
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved", "Approved", "Approved",
        "Needs manual realignment - heading and boundary mismatch",
        "Needs manual realignment - sentence boundary mismatch",
        "Needs manual realignment - combined content",
        "Approved", "Approved", "Approved", "Approved", "Approved"
    ]
}

batch_10_records = []

for candidate_id, statuses in batch_10_reviews.items():

    hi_row = source_250[
        source_250["id"] == candidate_id
    ].iloc[0]

    en_row = translation_df[
        translation_df["id"] == candidate_id
    ].iloc[0]

    hi_sentences = split_sentences(hi_row["hindi_article"])
    en_sentences = split_sentences(
        en_row["complete_english_translation"]
    )

    assert len(hi_sentences) == len(statuses), candidate_id
    assert len(en_sentences) == len(statuses), candidate_id

    for i, (hi, en, status) in enumerate(
        zip(hi_sentences, en_sentences, statuses), start=1
    ):
        batch_10_records.append({
            "sample_id": candidate_id,
            "sentence_id": i,
            "hindi_sentence": hi,
            "english_sentence": en,
            "alignment_status": status
        })

batch_10_df = pd.DataFrame(batch_10_records)

existing = pd.read_csv(alignment_path)

combined = pd.concat(
    [existing, batch_10_df],
    ignore_index=True
)

combined = combined.drop_duplicates(
    subset=["sample_id", "sentence_id"],
    keep="last"
)

combined.to_csv(alignment_path, index=False)

all_alignments = pd.read_csv(alignment_path)

print("Batch 10 records:", len(batch_10_df))
print("Total saved records:", len(all_alignments))
print("Total unique candidates:", all_alignments["sample_id"].nunique())
print("Saved successfully:", os.path.exists(alignment_path))
print("File:", alignment_path)

Batch 10 records: 79
Total saved records: 565
Total unique candidates: 50
Saved successfully: True
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/alignment/FactIndic_Manual_Sentence_Alignment.csv


In [346]:
all_alignments = pd.read_csv(alignment_path)

batch_11_ids = easy_candidates[
    ~easy_candidates["id"].isin(
        all_alignments["sample_id"].unique()
    )
]["id"].head(5).tolist()

print("BATCH 11 CANDIDATES:", batch_11_ids)
print("Number of candidates:", len(batch_11_ids))

for candidate_id in batch_11_ids:

    row_hi = source_250[
        source_250["id"] == candidate_id
    ].iloc[0]

    row_en = translation_df[
        translation_df["id"] == candidate_id
    ].iloc[0]

    hi_sentences = split_sentences(row_hi["hindi_article"])
    en_sentences = split_sentences(
        row_en["complete_english_translation"]
    )

    print("\n" + "=" * 100)
    print("ARTICLE ID:", candidate_id)
    print("Hindi sentences:", len(hi_sentences))
    print("English sentences:", len(en_sentences))
    print("=" * 100)

    for i, (hi, en) in enumerate(
        zip(hi_sentences, en_sentences), start=1
    ):
        print(f"\nSENTENCE {i}")
        print("HINDI:", hi)
        print("ENGLISH:", en)

    if len(hi_sentences) != len(en_sentences):
        print("\nWARNING: Sentence counts differ.")

BATCH 11 CANDIDATES: ['061125_rajpakshe_arrived', '150901_nepal_protest_dead_pp', '140410_action_against_manvendra_singh_rns', '060709_wordcup_final2006', '080826_blast_films']
Number of candidates: 5

ARTICLE ID: 061125_rajpakshe_arrived
Hindi sentences: 17
English sentences: 17

SENTENCE 1
HINDI: भारत ने भी श्रीलंका की मौजूदा स्थिति पर चिंता व्यक्त की है.
ENGLISH: India has also expressed concern over the current situation in Sri Lanka.

SENTENCE 2
HINDI: उम्मीद है कि भारतीय नेताओं और राष्ट्रपति राजपक्षे के साथ मुलाक़ात में एलटीटीई के नियंत्रण वाले इलाक़े की स्थिति के साथ-साथ शांति समझौते पर भी चर्चा होगी.
ENGLISH: The meeting with Indian leaders and President Rajapaksa is expected to discuss the situation in the territory controlled by the LTTE as well as the peace deal.

SENTENCE 3
HINDI: वैसे मंगलवार से पहले राष्ट्रपति राजपक्षे की भारतीय नेताओं से बातचीत संभव नहीं दिखती.
ENGLISH: However, President Rajapaksa's talks with Indian leaders do not appear to be possible before Tuesday.


In [347]:
batch_11_reviews = {
    "061125_rajpakshe_arrived": [
        "Needs manual realignment - merged and shifted sentence boundaries"
    ] * 17,

    "150901_nepal_protest_dead_pp": [
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved",
        "Provisionally approved - heading contamination",
        "Provisionally approved - stray heading",
        "Approved", "Approved",
        "Provisionally approved - heading contamination",
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved"
    ],

    "140410_action_against_manvendra_singh_rns": [
        "Approved", "Approved", "Approved", "Approved",
        "Approved",
        "Provisionally approved - headline appears separately",
        "Provisionally approved - headline and article text combined",
        "Approved", "Approved",
        "Provisionally approved - heading contamination",
        "Approved", "Approved", "Approved", "Approved",
        "Approved", "Approved", "Approved"
    ],

    "060709_wordcup_final2006": [
        "Approved"
    ] * 17,

    "080826_blast_films": [
        "Approved",
        *(
            ["Needs manual realignment - merged or shifted sentence boundaries"] * 14
        ),
        "Approved", "Approved", "Approved"
    ]
}

batch_11_records = []

for candidate_id, statuses in batch_11_reviews.items():

    hi_row = source_250[
        source_250["id"] == candidate_id
    ].iloc[0]

    en_row = translation_df[
        translation_df["id"] == candidate_id
    ].iloc[0]

    hi_sentences = split_sentences(hi_row["hindi_article"])
    en_sentences = split_sentences(
        en_row["complete_english_translation"]
    )

    assert len(hi_sentences) == len(statuses), (
        candidate_id, len(hi_sentences), len(statuses)
    )
    assert len(en_sentences) == len(statuses), (
        candidate_id, len(en_sentences), len(statuses)
    )

    for i, (hi, en, status) in enumerate(
        zip(hi_sentences, en_sentences, statuses), start=1
    ):
        batch_11_records.append({
            "sample_id": candidate_id,
            "sentence_id": i,
            "hindi_sentence": hi,
            "english_sentence": en,
            "alignment_status": status
        })

batch_11_df = pd.DataFrame(batch_11_records)

existing = pd.read_csv(alignment_path)

combined = pd.concat(
    [existing, batch_11_df],
    ignore_index=True
)

combined = combined.drop_duplicates(
    subset=["sample_id", "sentence_id"],
    keep="last"
)

combined.to_csv(alignment_path, index=False)

all_alignments = pd.read_csv(alignment_path)

print("Batch 11 records:", len(batch_11_df))
print("Total saved records:", len(all_alignments))
print("Total unique candidates:", all_alignments["sample_id"].nunique())
print("Saved successfully:", os.path.exists(alignment_path))
print("File:", alignment_path)

Batch 11 records: 86
Total saved records: 651
Total unique candidates: 55
Saved successfully: True
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/alignment/FactIndic_Manual_Sentence_Alignment.csv


In [348]:
all_alignments = pd.read_csv(alignment_path)

batch_12_ids = easy_candidates[
    ~easy_candidates["id"].isin(
        all_alignments["sample_id"].unique()
    )
]["id"].head(5).tolist()

print("BATCH 12 CANDIDATES:", batch_12_ids)
print("Number of candidates:", len(batch_12_ids))

for candidate_id in batch_12_ids:

    row_hi = source_250[
        source_250["id"] == candidate_id
    ].iloc[0]

    row_en = translation_df[
        translation_df["id"] == candidate_id
    ].iloc[0]

    hi_sentences = split_sentences(row_hi["hindi_article"])
    en_sentences = split_sentences(
        row_en["complete_english_translation"]
    )

    print("\n" + "=" * 100)
    print("ARTICLE ID:", candidate_id)
    print("Hindi sentences:", len(hi_sentences))
    print("English sentences:", len(en_sentences))
    print("=" * 100)

    for i, (hi, en) in enumerate(
        zip(hi_sentences, en_sentences), start=1
    ):
        print(f"\nSENTENCE {i}")
        print("HINDI:", hi)
        print("ENGLISH:", en)

    if len(hi_sentences) != len(en_sentences):
        print("\nWARNING: Sentence counts differ.")

BATCH 12 CANDIDATES: ['141107_bin_laden_death_dispute_us_sk', '050506_india_coach', '061219_spl_sez_orissa', '140825_bihar_nitish_laloo_siddharth_analysis_an', '040415_sharon_arafat']
Number of candidates: 5

ARTICLE ID: 141107_bin_laden_death_dispute_us_sk
Hindi sentences: 18
English sentences: 18

SENTENCE 1
HINDI: अमरीकी समाचार पत्र 'वाशिंगटन पोस्ट' से बात करते हुए 'नेवी सील्स' के पूर्व सैनिक 38 वर्षीय रॉबर्ट ओ-नील ने दावा किया है कि ओसामा की मौत उनकी चलाई गोली से हुई थी.
ENGLISH: Speaking to the American newspaper The Washington Post, 38-year-old former Navy SEAL Robert O'Neill claimed that Osama bin Laden was killed by a bullet fired by him.

SENTENCE 2
HINDI: लेकिन रॉबर्ट ओ-नील का ये दावा पूर्व मरीन मैट बेसोनेट के दावे के विपरीत है जिन्होंने इस विषय पर 2012 में किताब लिखी थी.
ENGLISH: However, O'Neill's claim contradicts the account given by former Navy SEAL Matt Bissonnette, who wrote a book about the operation in 2012.

SENTENCE 3
HINDI: मैट बेसोनेट ने 2012 में 'नो ईज़ी डे' नाम

In [349]:
# Check the existing alignment file structure
existing = pd.read_csv(alignment_path)

print("Existing columns:", existing.columns.tolist())
print("Existing records:", len(existing))
print("Batch 12 IDs:", [
    "141107_bin_laden_death_dispute_us_sk",
    "050506_india_coach",
    "061219_spl_sez_orissa",
    "140825_bihar_nitish_laloo_siddharth_analysis_an",
    "040415_sharon_arafat"
])

Existing columns: ['sample_id', 'sentence_id', 'hindi_sentence', 'english_sentence', 'review_status', 'alignment_status']
Existing records: 651
Batch 12 IDs: ['141107_bin_laden_death_dispute_us_sk', '050506_india_coach', '061219_spl_sez_orissa', '140825_bihar_nitish_laloo_siddharth_analysis_an', '040415_sharon_arafat']


In [351]:
print("Source columns:", source_250.columns.tolist())
print("\nTranslation columns:", translation_df.columns.tolist())

Source columns: ['id', 'url', 'title', 'summary', 'hindi_article', 'article_words', 'summary_words', 'english_article', 'hindi_word_count', 'english_word_count', 'english_has_hindi', 'translation_ratio', 'translation_quality_status']

Translation columns: ['id', 'complete_english_translation', 'reconstructed_sentence_count', 'manual_review_status']


In [352]:
batch_12_reviews = {
    "141107_bin_laden_death_dispute_us_sk": {
        "flag": [5, 9, 12], "footer": [17, 18]
    },
    "050506_india_coach": {
        "flag": [3, 4, 5, 6, 7, 12, 13, 15, 16, 17], "footer": []
    },
    "061219_spl_sez_orissa": {
        "flag": [2, 3, 4, 5, 7, 12, 14, 16], "footer": []
    },
    "140825_bihar_nitish_laloo_siddharth_analysis_an": {
        "flag": [3, 7, 8, 9, 14], "footer": [18, 19]
    },
    "040415_sharon_arafat": {
        "flag": [7, 8, 12, 13, 14, 15, 16, 17, 18], "footer": []
    }
}

new_records = []

for sample_id, review in batch_12_reviews.items():
    source_row = source_250[source_250["id"] == sample_id].iloc[0]
    translation_row = translation_df[
        translation_df["id"] == sample_id
    ].iloc[0]

    hindi_sentences = split_sentences(source_row["hindi_article"])
    english_sentences = split_sentences(
        translation_row["complete_english_translation"]
    )

    if len(hindi_sentences) != len(english_sentences):
        raise ValueError(
            f"Sentence count mismatch for {sample_id}: "
            f"Hindi={len(hindi_sentences)}, English={len(english_sentences)}"
        )

    for i, (hindi, english) in enumerate(
        zip(hindi_sentences, english_sentences), start=1
    ):
        if i in review["footer"]:
            status = "Keep Hindi - app/social footer"
            alignment = "Footer - not for mixing"
        elif i in review["flag"]:
            status = "Needs manual realignment"
            alignment = "Boundary/heading issue - do not auto-mix"
        else:
            status = "Provisionally approved"
            alignment = "Sentence pair provisionally aligned"

        new_records.append({
            "sample_id": sample_id,
            "sentence_id": i,
            "hindi_sentence": hindi,
            "english_sentence": english,
            "review_status": status,
            "alignment_status": alignment
        })

batch_12_df = pd.DataFrame(new_records)

# Preserve existing records and replace only matching IDs if present
updated = pd.concat([existing, batch_12_df], ignore_index=True)
updated = updated.drop_duplicates(
    subset=["sample_id", "sentence_id"], keep="last"
)

updated.to_csv(alignment_path, index=False)

print("Batch 12 records:", len(batch_12_df))
print("Total saved records:", len(updated))
print("Total unique candidates:", updated["sample_id"].nunique())
print("Saved successfully:", os.path.exists(alignment_path))

Batch 12 records: 93
Total saved records: 744
Total unique candidates: 60
Saved successfully: True


In [353]:
all_alignments = pd.read_csv(alignment_path)

batch_13_ids = easy_candidates[
    ~easy_candidates["id"].isin(
        all_alignments["sample_id"].unique()
    )
]["id"].head(5).tolist()

print("BATCH 13 CANDIDATES:", batch_13_ids)
print("Number of candidates:", len(batch_13_ids))

for candidate_id in batch_13_ids:
    row_hi = source_250[
        source_250["id"] == candidate_id
    ].iloc[0]

    row_en = translation_df[
        translation_df["id"] == candidate_id
    ].iloc[0]

    hi_sentences = split_sentences(row_hi["hindi_article"])
    en_sentences = split_sentences(
        row_en["complete_english_translation"]
    )

    print("\n" + "=" * 100)
    print("ARTICLE ID:", candidate_id)
    print("Hindi sentences:", len(hi_sentences))
    print("English sentences:", len(en_sentences))
    print("=" * 100)

    for i, (hi, en) in enumerate(
        zip(hi_sentences, en_sentences), start=1
    ):
        print(f"\nSENTENCE {i}")
        print("HINDI:", hi)
        print("ENGLISH:", en)

    if len(hi_sentences) != len(en_sentences):
        print("\nWARNING: Sentence counts differ.")

BATCH 13 CANDIDATES: ['070616_gaza_arab_ministers', '120809_salman_pkp', 'international-54702261', '130207_kashmir_rockband_imroz_sa', 'international-44515542']
Number of candidates: 5

ARTICLE ID: 070616_gaza_arab_ministers
Hindi sentences: 22
English sentences: 22

SENTENCE 1
HINDI: फ़तह के समर्थकों ने रमल्ला में परेड की और हवा में गोली चलाई.
ENGLISH: Fatah supporters paraded in Ramallah and fired shots into the air.

SENTENCE 2
HINDI: ये लोग संसद के बाहर हमास बाहर जाओ का नारा लगा रहे थे और फिर छत पर चढ़कर फ़तह और फ़लस्तीनी झंडा लगा दिया.
ENGLISH: They chanted "Hamas, get out" outside parliament, then climbed onto the roof and raised Fatah and Palestinian flags.

SENTENCE 3
HINDI: संसद के उपसभापति ने बीबीसी को बताया कि जब उन्होंने फ़तह समर्थकों को रोकने की कोशिश की तो उन्हें पीटा गया.
ENGLISH: The deputy speaker of parliament told the BBC that he was beaten when he tried to stop the Fatah supporters.

SENTENCE 4
HINDI: फ़तह से जुड़े कुछ लोगों ने शिक्षा मंत्रालय में तोड़फोड़ भी की हाल

In [354]:
batch_13_reviews = {
    "070616_gaza_arab_ministers": {
        "flag": [6, 15],
        "footer": []
    },
    "120809_salman_pkp": {
        "flag": [1, 2, 3, 4, 5, 6, 7, 11, 17, 18, 19, 20, 21, 22],
        "footer": []
    },
    "international-54702261": {
        "flag": [5, 6, 7, 8, 12, 13, 14, 15, 16, 17, 18, 19, 20],
        "footer": [21]
    },
    "130207_kashmir_rockband_imroz_sa": {
        "flag": list(range(1, 14)) + [18, 19, 20],
        "footer": []
    },
    "international-44515542": {
        "flag": [1, 4, 11, 14, 15, 17],
        "footer": [23, 24]
    }
}

new_records = []

for sample_id, review in batch_13_reviews.items():
    source_row = source_250[source_250["id"] == sample_id].iloc[0]
    translation_row = translation_df[
        translation_df["id"] == sample_id
    ].iloc[0]

    hindi_sentences = split_sentences(source_row["hindi_article"])
    english_sentences = split_sentences(
        translation_row["complete_english_translation"]
    )

    if len(hindi_sentences) != len(english_sentences):
        raise ValueError(
            f"Sentence count mismatch for {sample_id}"
        )

    for i, (hindi, english) in enumerate(
        zip(hindi_sentences, english_sentences), start=1
    ):
        if i in review["footer"]:
            status = "Keep Hindi - app/social footer"
            alignment = "Footer - not for mixing"
        elif i in review["flag"]:
            status = "Needs manual realignment"
            alignment = "Boundary/heading issue - do not auto-mix"
        else:
            status = "Provisionally approved"
            alignment = "Sentence pair provisionally aligned"

        new_records.append({
            "sample_id": sample_id,
            "sentence_id": i,
            "hindi_sentence": hindi,
            "english_sentence": english,
            "review_status": status,
            "alignment_status": alignment
        })

batch_13_df = pd.DataFrame(new_records)

existing = pd.read_csv(alignment_path)

updated = pd.concat(
    [existing, batch_13_df],
    ignore_index=True
)

updated = updated.drop_duplicates(
    subset=["sample_id", "sentence_id"],
    keep="last"
)

updated.to_csv(alignment_path, index=False)

print("Batch 13 records:", len(batch_13_df))
print("Total saved records:", len(updated))
print("Total unique candidates:", updated["sample_id"].nunique())
print("Saved successfully:", os.path.exists(alignment_path))

Batch 13 records: 112
Total saved records: 856
Total unique candidates: 65
Saved successfully: True


In [355]:
all_alignments = pd.read_csv(alignment_path)

remaining_ids = easy_candidates[
    ~easy_candidates["id"].isin(
        all_alignments["sample_id"].unique()
    )
]["id"].tolist()

print("FINAL CANDIDATES:", remaining_ids)
print("Number remaining:", len(remaining_ids))

for candidate_id in remaining_ids:
    row_hi = source_250[
        source_250["id"] == candidate_id
    ].iloc[0]

    row_en = translation_df[
        translation_df["id"] == candidate_id
    ].iloc[0]

    hi_sentences = split_sentences(row_hi["hindi_article"])
    en_sentences = split_sentences(
        row_en["complete_english_translation"]
    )

    print("\n" + "=" * 100)
    print("ARTICLE ID:", candidate_id)
    print("Hindi sentences:", len(hi_sentences))
    print("English sentences:", len(en_sentences))
    print("=" * 100)

    for i, (hi, en) in enumerate(
        zip(hi_sentences, en_sentences), start=1
    ):
        print(f"\nSENTENCE {i}")
        print("HINDI:", hi)
        print("ENGLISH:", en)

    if len(hi_sentences) != len(en_sentences):
        print("\nWARNING: Sentence counts differ.")

FINAL CANDIDATES: ['121115_2g_auction_ac', '061217_spl_sez_imballance', 'international-43396558', '150713_bihar_women_strange_names_rns', '130528_speed_breaker_cars_impact_sy', 'international-51410040']
Number remaining: 6

ARTICLE ID: 121115_2g_auction_ac
Hindi sentences: 25
English sentences: 25

SENTENCE 1
HINDI: नीलामी में टेलीकॉम कंपनियों ने कोई खास दिलचस्पी नहीं दिखाई है.
ENGLISH: Telecom companies have shown little interest in the auction.

SENTENCE 2
HINDI: 2जी स्पेक्ट्रम के आवंटन में एक लाख 76 हजार करोड़ का नुकसान करा चुकी भारत सरकार को उम्मीद थी कि नीलामी से सरकारी खजाने में कम से कम 40 हजार करोड़ रुपए आ पाएगा.
ENGLISH: The Indian government, which had incurred an estimated loss of Rs 1.76 lakh crore in the allocation of 2G spectrum, had hoped that the auction would bring at least Rs 40,000 crore into the government treasury.

SENTENCE 3
HINDI: लेकिन नीलामी में दूरसंचार कंपनियों ने कोई खास दिलचस्पी नहीं दिखाई है.
ENGLISH: However, telecom companies showed little interest.

SE

In [356]:
batch_14_reviews = {
    "121115_2g_auction_ac": {
        "flag": list(range(8, 26)),
        "footer": []
    },
    "061217_spl_sez_imballance": {
        "flag": [10, 11, 17, 22, 23],
        "footer": []
    },
    "international-43396558": {
        "flag": [6, 9] + list(range(11, 19)),
        "footer": [31, 32]
    },
    "150713_bihar_women_strange_names_rns": {
        "flag": list(range(9, 35)),
        "footer": [35]
    },
    "130528_speed_breaker_cars_impact_sy": {
        "flag": list(range(1, 40)),
        "footer": []
    },
    "international-51410040": {
        "flag": list(range(10, 18))
                + list(range(28, 33))
                + [51, 52, 63],
        "footer": [64]
    }
}

new_records = []

for sample_id, review in batch_14_reviews.items():
    source_row = source_250[
        source_250["id"] == sample_id
    ].iloc[0]

    translation_row = translation_df[
        translation_df["id"] == sample_id
    ].iloc[0]

    hindi_sentences = split_sentences(
        source_row["hindi_article"]
    )

    english_sentences = split_sentences(
        translation_row["complete_english_translation"]
    )

    if len(hindi_sentences) != len(english_sentences):
        raise ValueError(
            f"Sentence count mismatch for {sample_id}"
        )

    for i, (hindi, english) in enumerate(
        zip(hindi_sentences, english_sentences), start=1
    ):
        if i in review["footer"]:
            status = "Keep Hindi - app/social footer"
            alignment = "Footer - not for mixing"

        elif i in review["flag"]:
            status = "Needs manual realignment"
            alignment = "Boundary/heading issue - do not auto-mix"

        else:
            status = "Provisionally approved"
            alignment = "Sentence pair provisionally aligned"

        new_records.append({
            "sample_id": sample_id,
            "sentence_id": i,
            "hindi_sentence": hindi,
            "english_sentence": english,
            "review_status": status,
            "alignment_status": alignment
        })

batch_14_df = pd.DataFrame(new_records)

existing = pd.read_csv(alignment_path)

updated = pd.concat(
    [existing, batch_14_df],
    ignore_index=True
)

updated = updated.drop_duplicates(
    subset=["sample_id", "sentence_id"],
    keep="last"
)

updated.to_csv(alignment_path, index=False)

print("Batch 14 records:", len(batch_14_df))
print("Total saved records:", len(updated))
print("Total unique candidates:", updated["sample_id"].nunique())
print("Saved successfully:", os.path.exists(alignment_path))

Batch 14 records: 223
Total saved records: 1079
Total unique candidates: 71
Saved successfully: True


In [357]:
final_alignment = pd.read_csv(alignment_path)

print("TOTAL ARTICLES:",
      final_alignment["sample_id"].nunique())

print("\nREVIEW STATUS:")
print(final_alignment["review_status"].value_counts())

print("\nALIGNMENT STATUS:")
print(final_alignment["alignment_status"].value_counts())

print("\nMISSING VALUES:")
print(final_alignment.isna().sum())

print("\nDUPLICATE SENTENCE PAIRS:",
      final_alignment.duplicated(
          subset=["sample_id", "sentence_id"]
      ).sum())

TOTAL ARTICLES: 71

REVIEW STATUS:
review_status
Provisionally approved                                                       268
Needs manual realignment                                                     200
Keep Hindi - app/social footer                                                11
Keep Hindi - social media instruction                                          5
Keep Hindi - app instruction                                                   4
Provisionally approved - check stray Hindi word                                3
Provisionally approved - check heading word                                    2
Keep Hindi - headline and app instruction                                      1
Provisionally approved - list formatting differs                               1
Keep Hindi - related-story headlines                                           1
Provisionally approved - English split; heading included                       1
Provisionally approved - English split across sentences 1 an

In [358]:
missing_review = final_alignment[
    final_alignment["review_status"].isna()
]

missing_alignment = final_alignment[
    final_alignment["alignment_status"].isna()
]

print("Rows missing review_status:",
      len(missing_review))

print("\nRows missing alignment_status:",
      len(missing_alignment))

print("\nMissing review status by article:")
print(
    missing_review.groupby("sample_id").size()
    .sort_values(ascending=False)
)

print("\nSample rows missing review_status:")
display(
    missing_review[
        ["sample_id", "sentence_id",
         "hindi_sentence", "english_sentence"]
    ].head(10)
)

Rows missing review_status: 571

Rows missing alignment_status: 80

Missing review status by article:
sample_id
080826_blast_films                               18
061125_rajpakshe_arrived                         17
140410_action_against_manvendra_singh_rns        17
060709_wordcup_final2006                         17
150901_nepal_protest_dead_pp                     17
150213_msg_film_released_pm                      16
051121_aids_india                                16
080416_up_results                                16
050812_baflights_cancel                          16
060117_indopak_cbm                               15
international-41395655                           15
080711_lanka_attack                              15
130910_sehwag_gambhir_pujara_pp                  15
061015_katsav_police                             15
140206_egypt_al-sisi_presidential_election_sr    14
080801_india_iaea                                14
040510_fillipino_election                        14
0808

,sample_id,sentence_id,hindi_sentence,english_sentence
80,050112_nepalmao_amnesty,1,संगठन ने माओवादी नेताओं से अनुरोध किया है कि व...,Amnesty International has asked Maoist leaders...
81,050112_nepalmao_amnesty,2,नेपाली माओवादियों के नेता प्रचंड के नाम लिखे ए...,"In an open letter addressed to Prachanda, the ..."
82,050112_nepalmao_amnesty,3,संगठन के एशिया-प्रशांत क्षेत्र के निदेशक इंग्र...,"Ingrid Massage, the organisation's director fo..."
83,050112_nepalmao_amnesty,4,मसाज का कहना था कि कई लोगों को इसलिए निशाना बन...,Massage said that many people had been targete...
84,050112_nepalmao_amnesty,5,उनका कहना था कि ऐसा करना स्पष्ट अंतरराष्ट्रीय ...,She said that such actions were clear violatio...
85,050112_nepalmao_amnesty,6,नेपाल में विद्रोहियों के बहुत सारे स्कूली छात्...,There have been reports of Maoist insurgents a...
86,050112_nepalmao_amnesty,7,इनमें से कई छात्रों को तो विद्रोही अपने साथ शा...,Some of these students are even persuaded or f...
87,050112_nepalmao_amnesty,8,विद्रोहियों और सरकार दोनो ही की पहले भी मानवाध...,Both the rebels and the government have previo...
88,050112_nepalmao_amnesty,9,दोनो ही पक्ष एक दूसरे पर तो आरोप लगाते हैं लेक...,Each side accuses the other of abuses but deni...
89,080324_chowdhry_pakistan,1,यूसुफ़ रज़ा गीलानी ने संसद में सोमवार को कहा थ...,Yousuf Raza Gilani had said in Parliament on M...


In [359]:
print("STATUS COMBINATIONS:")
display(
    pd.crosstab(
        final_alignment["review_status"].fillna("MISSING"),
        final_alignment["alignment_status"].fillna("MISSING")
    )
)

print("\nROWS MISSING BOTH STATUS LABELS:")
missing_both = final_alignment[
    final_alignment["review_status"].isna()
    & final_alignment["alignment_status"].isna()
]

print("Count:", len(missing_both))
display(
    missing_both[
        ["sample_id", "sentence_id",
         "hindi_sentence", "english_sentence"]
    ].head(15)
)

print("\nROWS WITH ALIGNMENT LABEL BUT NO REVIEW LABEL:")
missing_review_only = final_alignment[
    final_alignment["review_status"].isna()
    & final_alignment["alignment_status"].notna()
]

print("Count:", len(missing_review_only))
display(
    missing_review_only[
        ["sample_id", "sentence_id", "alignment_status"]
    ].head(20)
)

STATUS COMBINATIONS:


alignment_status,Approved,Boundary/heading issue - do not auto-mix,Footer - not for mixing,Keep Hindi - app instruction,Keep Hindi - social media footer,MISSING,Needs manual boundary review - continuation of previous sentence,Needs manual boundary review - quotation and following statement combined,Needs manual boundary review - quotation and following statement split,Needs manual boundary review - quote and heading combined,...,Provisionally approved - heading contamination,Provisionally approved - heading included; English split,Provisionally approved - headline and article text combined,Provisionally approved - headline appears separately,Provisionally approved - multiple quotations and paragraph split,Provisionally approved - quotation and following paragraph split,Provisionally approved - sentence boundary needs review,Provisionally approved - stray heading,Provisionally approved - stray heading contamination,Sentence pair provisionally aligned
review_status,,,,,,,,,,,,,,,,,,,,,
Keep Hindi - app instruction,0,0,0,0,0,4,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Keep Hindi - app instruction; English content differs,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Keep Hindi - app/social footer,0,0,11,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Keep Hindi - headline and app instruction,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Keep Hindi - related-story headlines,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Keep Hindi - social media footer; English content differs,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Keep Hindi - social media instruction,0,0,0,0,0,5,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
MISSING,408,0,0,5,5,0,1,1,1,1,...,21,1,1,1,1,1,1,1,1,0
Needs manual realignment,0,200,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0



ROWS MISSING BOTH STATUS LABELS:
Count: 0


,sample_id,sentence_id,hindi_sentence,english_sentence



ROWS WITH ALIGNMENT LABEL BUT NO REVIEW LABEL:
Count: 571


,sample_id,sentence_id,alignment_status
80,050112_nepalmao_amnesty,1,Approved
81,050112_nepalmao_amnesty,2,Approved
82,050112_nepalmao_amnesty,3,Approved
83,050112_nepalmao_amnesty,4,Approved
84,050112_nepalmao_amnesty,5,Approved
85,050112_nepalmao_amnesty,6,Approved
86,050112_nepalmao_amnesty,7,Approved
87,050112_nepalmao_amnesty,8,Approved
88,050112_nepalmao_amnesty,9,Approved
89,080324_chowdhry_pakistan,1,Approved


In [360]:
print("MISSING review_status — existing alignment labels:")
print(
    final_alignment.loc[
        final_alignment["review_status"].isna(),
        "alignment_status"
    ].value_counts(dropna=False).to_string()
)

print("\nMISSING alignment_status — existing review labels:")
print(
    final_alignment.loc[
        final_alignment["alignment_status"].isna(),
        "review_status"
    ].value_counts(dropna=False).to_string()
)

MISSING review_status — existing alignment labels:
alignment_status
Approved                                                                         408
Provisionally approved - heading contamination                                    21
Needs manual realignment - merged and shifted sentence boundaries                 17
Needs manual realignment - headline contamination and sentence order mismatch     15
Needs manual realignment - English combines first two Hindi sentences             15
Needs manual realignment - sentence order mismatch                                15
Needs manual realignment - merged or shifted sentence boundaries                  14
Provisionally approved                                                            10
Needs manual realignment - sentence boundary shift                                 7
Keep Hindi - social media footer                                                   5
Keep Hindi - app instruction                                                      

In [361]:
# Keep a backup before editing
backup_path = alignment_path.replace(
    ".csv", "_Before_Status_Cleanup.csv"
)

final_alignment.to_csv(backup_path, index=False)

# Preserve the source of each restored label
missing_review = final_alignment["review_status"].isna()
final_alignment.loc[missing_review, "review_status"] = (
    "Inherited from alignment_status: "
    + final_alignment.loc[missing_review, "alignment_status"]
)

missing_alignment = final_alignment["alignment_status"].isna()
final_alignment.loc[missing_alignment, "alignment_status"] = (
    "Inherited from review_status: "
    + final_alignment.loc[missing_alignment, "review_status"]
)

# Save cleaned file
final_alignment.to_csv(alignment_path, index=False)

print("Missing review_status:",
      final_alignment["review_status"].isna().sum())

print("Missing alignment_status:",
      final_alignment["alignment_status"].isna().sum())

print("Duplicate sentence pairs:",
      final_alignment.duplicated(
          subset=["sample_id", "sentence_id"]
      ).sum())

print("Total records:", len(final_alignment))
print("Unique articles:", final_alignment["sample_id"].nunique())
print("Backup exists:", os.path.exists(backup_path))
print("Cleaned file saved:", os.path.exists(alignment_path))

Missing review_status: 0
Missing alignment_status: 0
Duplicate sentence pairs: 0
Total records: 1079
Unique articles: 71
Backup exists: True
Cleaned file saved: True


In [362]:
status = final_alignment["alignment_status"].astype(str)

approved_mask = status.isin([
    "Approved",
    "Sentence pair provisionally aligned",
    "Provisionally approved"
])

usable = final_alignment[approved_mask].copy()

print("Explicitly approved/provisional pairs:",
      len(usable))

print("Unique articles with usable pairs:",
      usable["sample_id"].nunique())

print("\nStatus breakdown:")
print(usable["alignment_status"].value_counts())

print("\nArticles with zero usable pairs:",
      71 - usable["sample_id"].nunique())

Explicitly approved/provisional pairs: 635
Unique articles with usable pairs: 56

Status breakdown:
alignment_status
Approved                               408
Sentence pair provisionally aligned    217
Provisionally approved                  10
Name: count, dtype: int64

Articles with zero usable pairs: 15


In [363]:
pair_counts = (
    usable.groupby("sample_id")
    .size()
    .sort_values(ascending=False)
)

print("USABLE PAIRS PER ARTICLE:")
display(pair_counts.to_frame("usable_pairs"))

print("\nArticles with at least 5 usable pairs:",
      (pair_counts >= 5).sum())

print("Articles with at least 10 usable pairs:",
      (pair_counts >= 10).sum())

print("Articles with fewer than 5 usable pairs:",
      (pair_counts < 5).sum())

USABLE PAIRS PER ARTICLE:


,usable_pairs
sample_id,
international-51410040,47
061217_spl_sez_imballance,23
070616_gaza_arab_ministers,20
international-43396558,20
060709_wordcup_final2006,17
international-44515542,16
051121_aids_india,15
060117_indopak_cbm,14
080711_lanka_attack,14



Articles with at least 5 usable pairs: 54
Articles with at least 10 usable pairs: 33
Articles with fewer than 5 usable pairs: 2


In [364]:
strict_approved = final_alignment[
    final_alignment["alignment_status"] == "Approved"
].copy()

strict_counts = (
    strict_approved.groupby("sample_id")
    .size()
    .sort_values(ascending=False)
)

print("Strict approved pairs:", len(strict_approved))
print("Articles with approved pairs:", len(strict_counts))

print("\nArticles with at least 5 approved pairs:",
      (strict_counts >= 5).sum())

print("Articles with at least 10 approved pairs:",
      (strict_counts >= 10).sum())

print("\nStrict approved pairs per article:")
display(strict_counts.to_frame("approved_pairs"))

Strict approved pairs: 408
Articles with approved pairs: 40

Articles with at least 5 approved pairs: 38
Articles with at least 10 approved pairs: 23

Strict approved pairs per article:


,approved_pairs
sample_id,
060709_wordcup_final2006,17
051121_aids_india,15
080711_lanka_attack,14
060117_indopak_cbm,14
150901_nepal_protest_dead_pp,14
140410_action_against_manvendra_singh_rns,14
080416_up_results,13
040510_fillipino_election,12
051105_sahubuddin_arrest,12


In [365]:
# Keep only strictly approved pairs
strict_approved = final_alignment[
    final_alignment["alignment_status"] == "Approved"
].copy()

# Identify articles with at least 10 approved pairs
strict_counts = strict_approved.groupby("sample_id").size()

eligible_ids = strict_counts[
    strict_counts >= 10
].index.tolist()

eligible_articles = strict_approved[
    strict_approved["sample_id"].isin(eligible_ids)
].copy()

# Save both datasets
strict_path = os.path.join(
    base_path,
    "04_mixed_data",
    "alignment",
    "FactIndic_Strict_Approved_Sentence_Pairs.csv"
)

eligible_path = os.path.join(
    base_path,
    "04_mixed_data",
    "alignment",
    "FactIndic_Eligible_Articles_10plus.csv"
)

strict_approved.to_csv(strict_path, index=False)
eligible_articles.to_csv(eligible_path, index=False)

print("Strict approved pairs:", len(strict_approved))
print("Eligible articles:", len(eligible_ids))
print("Pairs in eligible articles:", len(eligible_articles))
print("Strict dataset saved:", os.path.exists(strict_path))
print("Eligible dataset saved:", os.path.exists(eligible_path))

Strict approved pairs: 408
Eligible articles: 23
Pairs in eligible articles: 275
Strict dataset saved: True
Eligible dataset saved: True


In [366]:
eligible_source = source_250[
    source_250["id"].isin(eligible_ids)
].copy()

eligible_source = eligible_source.merge(
    translation_df[
        ["id", "complete_english_translation"]
    ],
    on="id",
    how="left"
)

print("Eligible source articles:", len(eligible_source))
print("Missing Hindi articles:",
      eligible_source["hindi_article"].isna().sum())
print("Missing English translations:",
      eligible_source["complete_english_translation"].isna().sum())

display(
    eligible_source[
        ["id", "title", "article_words"]
    ].head(10)
)

Eligible source articles: 23
Missing Hindi articles: 0
Missing English translations: 0


,id,title,article_words
0,080711_lanka_attack,बस पर गोलीबारी में चार की मौत,239
1,051105_sahubuddin_arrest,शहाबुद्दीन दिल्ली में गिरफ़्तार,255
2,150901_nepal_protest_dead_pp,नेपाल में पुलिस गोलीबारी से पाँच की मौत,269
3,041003_pwg_security,नक्सली वार्ताकारों को मिलेगी सुरक्षा,219
4,080416_up_results,उपचुनाव में पाँचों सीटें बसपा को,275
5,060117_indopak_cbm,रेल सेवा अजमेर तक बढ़ाने का प्रस्ताव,305
6,140206_egypt_al-sisi_presidential_election_sr,मिस्र: अल-सीसी के चुनाव लड़ने का खंडन,405
7,140410_action_against_manvendra_singh_rns,जसवंत सिंह के बेटे को भाजपा ने राष्ट्रीय कार्य...,295
8,051121_aids_india,भारत में बढ़ा एड्स का ख़तरा,372
9,070902_khanduri_win,खंडूरी ने विधानसभा उपचुनाव जीता,161


In [367]:
# Keep only approved pairs belonging to eligible articles
eligible_pairs = strict_approved[
    strict_approved["sample_id"].isin(eligible_ids)
].copy()

# Add original article metadata
eligible_pairs = eligible_pairs.merge(
    eligible_source[["id", "title", "url"]],
    left_on="sample_id",
    right_on="id",
    how="left"
)

eligible_pairs = eligible_pairs.drop(columns=["id"])

# Save sentence-level dataset
sentence_path = os.path.join(
    base_path,
    "04_mixed_data",
    "alignment",
    "FactIndic_Eligible_Approved_Sentence_Pairs.csv"
)

eligible_pairs.to_csv(sentence_path, index=False)

print("Eligible sentence pairs:", len(eligible_pairs))
print("Unique articles:", eligible_pairs["sample_id"].nunique())
print("Missing Hindi sentences:",
      eligible_pairs["hindi_sentence"].isna().sum())
print("Missing English sentences:",
      eligible_pairs["english_sentence"].isna().sum())
print("Saved:", os.path.exists(sentence_path))

display(
    eligible_pairs[
        ["sample_id", "sentence_id",
         "hindi_sentence", "english_sentence"]
    ].head(5)
)

Eligible sentence pairs: 275
Unique articles: 23
Missing Hindi sentences: 0
Missing English sentences: 0
Saved: True


,sample_id,sentence_id,hindi_sentence,english_sentence
0,051209_tamilnadu_cyclone,1,अभी समुद्री तूफ़ान फ़ानूस चेन्नई से 550 किलोमी...,"Presently, Cyclone FANUS lies 550 km southeast..."
1,051209_tamilnadu_cyclone,2,मौसम विभाग का कहना है कि अगले 24 घंटों के अंदर...,The meteorological department says the storm w...
2,051209_tamilnadu_cyclone,3,राज्य के तटीय इलाक़ों में लोगों को सावधान कर द...,People in coastal areas of the state have been...
3,051209_tamilnadu_cyclone,4,मछुआरों को समुद्र में न जाने की सलाह दी गई है.,Fishermen have been advised not to venture int...
4,051209_tamilnadu_cyclone,5,समुद्री तूफान के कारण तटीय इलाक़ों में 70 किलो...,The storm is causing winds of up to 70 kilomet...


In [368]:
import re

def word_count(text):
    return len(str(text).split())

def english_word_count(text):
    words = str(text).split()
    return sum(
        1 for word in words
        if re.search(r"[A-Za-z]", word)
    )

eligible_pairs["hindi_words"] = (
    eligible_pairs["hindi_sentence"].apply(word_count)
)

eligible_pairs["existing_english_words"] = (
    eligible_pairs["hindi_sentence"].apply(english_word_count)
)

eligible_pairs["existing_english_pct"] = (
    eligible_pairs["existing_english_words"]
    / eligible_pairs["hindi_words"].replace(0, 1)
    * 100
)

print("Total Hindi sentence words:",
      eligible_pairs["hindi_words"].sum())

print("Existing English words:",
      eligible_pairs["existing_english_words"].sum())

print("Overall existing English percentage:",
      round(
          eligible_pairs["existing_english_words"].sum()
          / eligible_pairs["hindi_words"].sum() * 100,
          2
      ))

print("\nSentence-level English percentage:")
display(
    eligible_pairs[
        ["sample_id", "sentence_id", "hindi_words",
         "existing_english_words", "existing_english_pct"]
    ].head(15)
)

Total Hindi sentence words: 5126
Existing English words: 0
Overall existing English percentage: 0.0

Sentence-level English percentage:


,sample_id,sentence_id,hindi_words,existing_english_words,existing_english_pct
0,051209_tamilnadu_cyclone,1,11,0,0.0
1,051209_tamilnadu_cyclone,2,17,0,0.0
2,051209_tamilnadu_cyclone,3,12,0,0.0
3,051209_tamilnadu_cyclone,4,11,0,0.0
4,051209_tamilnadu_cyclone,5,18,0,0.0
5,051209_tamilnadu_cyclone,6,20,0,0.0
6,051209_tamilnadu_cyclone,7,9,0,0.0
7,051209_tamilnadu_cyclone,8,13,0,0.0
8,051209_tamilnadu_cyclone,9,13,0,0.0
9,051209_tamilnadu_cyclone,10,21,0,0.0


In [369]:
total_words = eligible_pairs["hindi_words"].sum()

for target in [10, 30, 50]:
    required_english = round(total_words * target / 100)

    print(f"\n{target}% MIXING")
    print("Target English words:", required_english)
    print("Target total words:", total_words)


10% MIXING
Target English words: 513
Target total words: 5126

30% MIXING
Target English words: 1538
Target total words: 5126

50% MIXING
Target English words: 2563
Target total words: 5126


In [370]:
eligible_pairs["english_words"] = (
    eligible_pairs["english_sentence"].apply(word_count)
)

eligible_pairs["word_difference"] = (
    eligible_pairs["english_words"]
    - eligible_pairs["hindi_words"]
)

print("Hindi words:",
      eligible_pairs["hindi_words"].sum())

print("English translation words:",
      eligible_pairs["english_words"].sum())

print("Average Hindi sentence length:",
      round(eligible_pairs["hindi_words"].mean(), 2))

print("Average English sentence length:",
      round(eligible_pairs["english_words"].mean(), 2))

print("\nWord count comparison:")
display(
    eligible_pairs[
        ["sample_id", "sentence_id",
         "hindi_words", "english_words",
         "word_difference"]
    ].head(15)
)

Hindi words: 5126
English translation words: 4462
Average Hindi sentence length: 18.64
Average English sentence length: 16.23

Word count comparison:


,sample_id,sentence_id,hindi_words,english_words,word_difference
0,051209_tamilnadu_cyclone,1,11,9,-2
1,051209_tamilnadu_cyclone,2,17,14,-3
2,051209_tamilnadu_cyclone,3,12,12,0
3,051209_tamilnadu_cyclone,4,11,10,-1
4,051209_tamilnadu_cyclone,5,18,15,-3
5,051209_tamilnadu_cyclone,6,20,16,-4
6,051209_tamilnadu_cyclone,7,9,9,0
7,051209_tamilnadu_cyclone,8,13,10,-3
8,051209_tamilnadu_cyclone,9,13,11,-2
9,051209_tamilnadu_cyclone,10,21,21,0


In [371]:
article_stats = (
    eligible_pairs.groupby("sample_id")
    .agg(
        hindi_words=("hindi_words", "sum"),
        english_words=("english_words", "sum"),
        sentence_pairs=("sentence_id", "count")
    )
)

article_stats["word_difference"] = (
    article_stats["english_words"]
    - article_stats["hindi_words"]
)

article_stats["full_translation_english_pct"] = 100.0

display(
    article_stats.sort_values(
        "sentence_pairs",
        ascending=False
    )
)

print("\nTOTAL HINDI WORDS:",
      article_stats["hindi_words"].sum())

print("TOTAL ENGLISH WORDS:",
      article_stats["english_words"].sum())

print("TOTAL SENTENCE PAIRS:",
      article_stats["sentence_pairs"].sum())

,hindi_words,english_words,sentence_pairs,word_difference,full_translation_english_pct
sample_id,,,,,
060709_wordcup_final2006,292,260,17,-32,100.0
051121_aids_india,340,308,15,-32,100.0
060117_indopak_cbm,275,244,14,-31,100.0
080711_lanka_attack,228,198,14,-30,100.0
140410_action_against_manvendra_singh_rns,234,216,14,-18,100.0
150901_nepal_protest_dead_pp,209,176,14,-33,100.0
080416_up_results,235,205,13,-30,100.0
080830_orissa_violence,199,174,12,-25,100.0
050505_voteclose_britain,253,218,12,-35,100.0



TOTAL HINDI WORDS: 5126
TOTAL ENGLISH WORDS: 4462
TOTAL SENTENCE PAIRS: 275


In [373]:
print("Eligible source columns:")
print(eligible_source.columns.tolist())

print("\nFirst 2 rows:")
display(eligible_source.head(2))

Eligible source columns:
['id', 'url', 'title', 'summary', 'hindi_article', 'article_words', 'summary_words', 'english_article', 'hindi_word_count', 'english_word_count', 'english_has_hindi', 'translation_ratio', 'translation_quality_status', 'complete_english_translation']

First 2 rows:


,id,url,title,summary,hindi_article,article_words,summary_words,english_article,hindi_word_count,english_word_count,english_has_hindi,translation_ratio,translation_quality_status,complete_english_translation
0,080711_lanka_attack,https://www.bbc.com/hindi/regionalnews/story/2...,बस पर गोलीबारी में चार की मौत,दक्षिणी श्रीलंका में एक बस पर हुए हमले में कम ...,एक सैनिक प्रवक्ता ब्रिगेडियर उदय नान्यकारा ने ...,239,25,"Brigadier Uday Nanyakara, a military spokesman...",239,208,False,0.870293,Initial review - no obvious mismatch,"Brigadier Uday Nanyakara, a military spokesman..."
1,051105_sahubuddin_arrest,https://www.bbc.com/hindi/regionalnews/story/2...,शहाबुद्दीन दिल्ली में गिरफ़्तार,राष्ट्रीय जनता दल के विवादास्पद सांसद मोहम्मद ...,शहाबुद्दीन बिहार के सीवान क्षेत्र से सांसद हैं...,255,21,"Shahabuddin, a Member of Parliament from the S...",255,208,False,0.815686,Initial review - no obvious mismatch,"Shahabuddin, a Member of Parliament from the S..."


In [374]:
def normalize_sentence(text):
    return re.sub(r"\s+", " ", str(text)).strip()

source_lookup = (
    eligible_source
    .set_index("id")["hindi_article"]
    .to_dict()
)

missing_sentences = []

for _, row in eligible_pairs.iterrows():

    article_text = source_lookup.get(row["sample_id"], "")

    original_sentences = [
        normalize_sentence(s)
        for s in split_sentences(article_text)
    ]

    approved_sentence = normalize_sentence(
        row["hindi_sentence"]
    )

    if approved_sentence not in original_sentences:
        missing_sentences.append({
            "sample_id": row["sample_id"],
            "sentence_id": row["sentence_id"],
            "sentence": approved_sentence
        })

print("Total approved pairs:", len(eligible_pairs))
print("Approved Hindi sentences not found exactly:",
      len(missing_sentences))

print("\nExamples:")
display(missing_sentences[:10])

Total approved pairs: 275
Approved Hindi sentences not found exactly: 0

Examples:


[]

In [375]:
replacement_lookup = {}

for _, row in eligible_pairs.iterrows():

    key = (
        row["sample_id"],
        normalize_sentence(row["hindi_sentence"])
    )

    replacement_lookup[key] = (
        row["english_sentence"]
    )

print("Replacement pairs:", len(replacement_lookup))
print("Unique articles:", eligible_pairs["sample_id"].nunique())

print("\nSample replacement:")
first_key = next(iter(replacement_lookup))

print("Hindi:", first_key[1])
print("English:", replacement_lookup[first_key])

Replacement pairs: 275
Unique articles: 23

Sample replacement:
Hindi: अभी समुद्री तूफ़ान फ़ानूस चेन्नई से 550 किलोमीटर दक्षिणपूर्व स्थित है.
English: Presently, Cyclone FANUS lies 550 km southeast of Chennai.


In [376]:
article_sentences = []

for _, row in eligible_source.iterrows():

    article_id = row["id"]
    sentences = split_sentences(row["hindi_article"])

    for sentence_no, sentence in enumerate(sentences, start=1):

        clean_sentence = normalize_sentence(sentence)

        key = (article_id, clean_sentence)

        article_sentences.append({
            "sample_id": article_id,
            "sentence_id": sentence_no,
            "hindi_sentence": clean_sentence,
            "english_sentence": replacement_lookup.get(key),
            "is_approved": key in replacement_lookup
        })

article_sentences_df = pd.DataFrame(article_sentences)

print("Total original sentences:",
      len(article_sentences_df))

print("Approved sentences found:",
      article_sentences_df["is_approved"].sum())

print("Unapproved sentences retained in Hindi:",
      (~article_sentences_df["is_approved"]).sum())

print("Articles:",
      article_sentences_df["sample_id"].nunique())

display(article_sentences_df.head(10))

Total original sentences: 308
Approved sentences found: 275
Unapproved sentences retained in Hindi: 33
Articles: 23


,sample_id,sentence_id,hindi_sentence,english_sentence,is_approved
0,080711_lanka_attack,1,एक सैनिक प्रवक्ता ब्रिगेडियर उदय नान्यकारा ने ...,"Brigadier Uday Nanyakara, a military spokesman...",True
1,080711_lanka_attack,2,ये घटना मोनारगाला ज़िले में हुई जब इस बस पर अं...,The incident took place in Monaragala district...,True
2,080711_lanka_attack,3,अभी तक एलटीटीई की ओर से इस घटना पर कोई टिप्पणी...,The LTTE has not commented on the incident so ...,True
3,080711_lanka_attack,4,ये हमला ऐसे समय हुआ है जब देश के उत्तरी इलाक़े...,The attack comes at a time when clashes betwee...,True
4,080711_lanka_attack,5,कोलंबो से बीबीसी संवाददाता ने बताया है कि बस ए...,A BBC correspondent in Colombo reported that t...,True
5,080711_lanka_attack,6,हमला मरने वालों में एक नौ साल का बच्चा भी है.,None,False
6,080711_lanka_attack,7,गोलीबारी में कई यात्री घायल भी हुए हैं.,Several passengers were also injured in the fi...,True
7,080711_lanka_attack,8,सैनिक प्रवक्ता ब्रिगेडियर उदय नान्यकारा का कहन...,Military spokesman Brigadier Uday Nanyakara sa...,True
8,080711_lanka_attack,9,उन्होंने इस गोलीबारी के लिए तमिल विद्रोहियों क...,He blamed the Tamil rebels for the shooting.,True
9,080711_lanka_attack,10,एक साल पहले श्रीलंका की सरकार ने दावा किया था ...,"A year earlier, the Sri Lankan government had ...",True


In [377]:
article_sentences_df["hindi_words"] = (
    article_sentences_df["hindi_sentence"].apply(word_count)
)

article_sentences_df["english_words"] = (
    article_sentences_df["english_sentence"]
    .apply(lambda x: word_count(x) if isinstance(x, str) else 0)
)

total_original_words = article_sentences_df["hindi_words"].sum()

total_replaceable_hindi_words = (
    article_sentences_df.loc[
        article_sentences_df["is_approved"],
        "hindi_words"
    ].sum()
)

total_english_replacement_words = (
    article_sentences_df["english_words"].sum()
)

print("Original Hindi words across all sentences:",
      total_original_words)

print("Hindi words eligible for replacement:",
      total_replaceable_hindi_words)

print("English words available for replacement:",
      total_english_replacement_words)

print("Unapproved Hindi words:",
      total_original_words - total_replaceable_hindi_words)

print("\nMaximum achievable English percentage:")
print(
    round(
        total_english_replacement_words /
        (
            total_original_words
            - total_replaceable_hindi_words
            + total_english_replacement_words
        ) * 100,
        2
    ),
    "%"
)

Original Hindi words across all sentences: 5954
Hindi words eligible for replacement: 5126
English words available for replacement: 4462
Unapproved Hindi words: 828

Maximum achievable English percentage:
84.35 %


In [378]:
import numpy as np

def generate_mixed_dataset(df, target_pct):

    results = []

    # Deterministic sentence selection order
    approved = df[df["is_approved"]].copy()

    # Sort by article and sentence position
    approved = approved.sort_values(
        ["sample_id", "sentence_id"]
    )

    # Use a reproducible shuffled selection order
    approved = approved.sample(
        frac=1,
        random_state=42
    )

    selected = set()
    hindi_words = int(df["hindi_words"].sum())
    current_english = 0
    current_total = hindi_words

    for idx, row in approved.iterrows():

        h = int(row["hindi_words"])
        e = int(row["english_words"])

        new_english = current_english + e
        new_total = current_total - h + e

        current_pct = (
            new_english / new_total * 100
            if new_total else 0
        )

        if abs(current_pct - target_pct) < abs(
            current_english / current_total * 100
            - target_pct
        ):
            selected.add(idx)
            current_english = new_english
            current_total = new_total

    output = df.copy()
    output["mixed_sentence"] = output["hindi_sentence"]

    for idx in selected:
        output.loc[idx, "mixed_sentence"] = (
            output.loc[idx, "english_sentence"]
        )

    return output, selected


mixed_versions = {}

for level in [10, 30, 50]:

    version, selected = generate_mixed_dataset(
        article_sentences_df,
        level
    )

    mixed_versions[level] = version

    english_count = sum(
        word_count(s)
        for s in version["mixed_sentence"]
        if isinstance(s, str)
    )

    total_count = sum(
        word_count(s)
        for s in version["mixed_sentence"]
        if isinstance(s, str)
    )

    actual_pct = english_count / total_count * 100

    print(f"\n{level}% TARGET")
    print("Selected English sentences:", len(selected))
    print("English words:", english_count)
    print("Total words:", total_count)
    print("Actual English percentage:", round(actual_pct, 2))


10% TARGET
Selected English sentences: 34
English words: 5893
Total words: 5893
Actual English percentage: 100.0

30% TARGET
Selected English sentences: 102
English words: 5722
Total words: 5722
Actual English percentage: 100.0

50% TARGET
Selected English sentences: 168
English words: 5528
Total words: 5528
Actual English percentage: 100.0


In [379]:
import re

def count_english_words(text):
    words = str(text).split()
    return sum(
        1 for word in words
        if re.search(r"[A-Za-z]", word)
    )

for level, version in mixed_versions.items():

    english_count = version["mixed_sentence"].apply(
        count_english_words
    ).sum()

    total_count = version["mixed_sentence"].apply(
        word_count
    ).sum()

    actual_pct = english_count / total_count * 100

    print(f"\n{level}% TARGET")
    print("English words:", english_count)
    print("Total words:", total_count)
    print("Actual English percentage:", round(actual_pct, 2))


10% TARGET
English words: 590
Total words: 5893
Actual English percentage: 10.01

30% TARGET
English words: 1689
Total words: 5722
Actual English percentage: 29.52

50% TARGET
English words: 2715
Total words: 5528
Actual English percentage: 49.11


In [380]:
for level, version in mixed_versions.items():

    replaced = (
        version["mixed_sentence"]
        != version["hindi_sentence"]
    )

    invalid_replacements = (
        replaced & ~version["is_approved"]
    ).sum()

    missing_sentences = version["mixed_sentence"].isna().sum()

    english_words = version["mixed_sentence"].apply(
        count_english_words
    ).sum()

    total_words = version["mixed_sentence"].apply(
        word_count
    ).sum()

    print(f"\n{'='*35}")
    print(f"{level}% VERSION VALIDATION")
    print("Articles:", version["sample_id"].nunique())
    print("Sentences:", len(version))
    print("Replaced sentences:", replaced.sum())
    print("Invalid replacements:", invalid_replacements)
    print("Missing sentences:", missing_sentences)
    print("English percentage:",
          round(english_words / total_words * 100, 2))


10% VERSION VALIDATION
Articles: 23
Sentences: 308
Replaced sentences: 34
Invalid replacements: 0
Missing sentences: 0
English percentage: 10.01

30% VERSION VALIDATION
Articles: 23
Sentences: 308
Replaced sentences: 102
Invalid replacements: 0
Missing sentences: 0
English percentage: 29.52

50% VERSION VALIDATION
Articles: 23
Sentences: 308
Replaced sentences: 168
Invalid replacements: 0
Missing sentences: 0
English percentage: 49.11


In [381]:
mixed_article_rows = []

for article_id in eligible_source["id"]:

    original = article_sentences_df[
        article_sentences_df["sample_id"] == article_id
    ].sort_values("sentence_id")

    row = {
        "sample_id": article_id,
        "hindi_article": " ".join(
            original["hindi_sentence"].tolist()
        )
    }

    for level, version in mixed_versions.items():

        article_version = version[
            version["sample_id"] == article_id
        ].sort_values("sentence_id")

        row[f"mixed_{level}_article"] = " ".join(
            article_version["mixed_sentence"].tolist()
        )

    mixed_article_rows.append(row)

mixed_articles_df = pd.DataFrame(mixed_article_rows)

# Add article titles and source URLs
mixed_articles_df = mixed_articles_df.merge(
    eligible_source[["id", "title", "url"]],
    left_on="sample_id",
    right_on="id",
    how="left"
).drop(columns=["id"])

print("Article-level rows:", len(mixed_articles_df))
print("Columns:", mixed_articles_df.columns.tolist())

display(
    mixed_articles_df[
        ["sample_id", "title",
         "mixed_10_article",
         "mixed_30_article",
         "mixed_50_article"]
    ].head(2)
)

Article-level rows: 23
Columns: ['sample_id', 'hindi_article', 'mixed_10_article', 'mixed_30_article', 'mixed_50_article', 'title', 'url']


,sample_id,title,mixed_10_article,mixed_30_article,mixed_50_article
0,080711_lanka_attack,बस पर गोलीबारी में चार की मौत,एक सैनिक प्रवक्ता ब्रिगेडियर उदय नान्यकारा ने ...,एक सैनिक प्रवक्ता ब्रिगेडियर उदय नान्यकारा ने ...,एक सैनिक प्रवक्ता ब्रिगेडियर उदय नान्यकारा ने ...
1,051105_sahubuddin_arrest,शहाबुद्दीन दिल्ली में गिरफ़्तार,शहाबुद्दीन बिहार के सीवान क्षेत्र से सांसद हैं...,"Shahabuddin, a Member of Parliament from the S...","Shahabuddin, a Member of Parliament from the S..."


In [382]:
final_check = []

for level in [10, 30, 50]:

    col = f"mixed_{level}_article"

    english_words = mixed_articles_df[col].apply(
        count_english_words
    ).sum()

    total_words = mixed_articles_df[col].apply(
        word_count
    ).sum()

    final_check.append({
        "Target": f"{level}%",
        "English_words": english_words,
        "Total_words": total_words,
        "Actual_CMI": round(
            english_words / total_words * 100, 2
        )
    })

final_check_df = pd.DataFrame(final_check)

display(final_check_df)

,Target,English_words,Total_words,Actual_CMI
0,10%,590,5893,10.01
1,30%,1689,5722,29.52
2,50%,2715,5528,49.11


In [383]:
import os

output_dir = os.path.join(
    base_path,
    "04_mixed_data"
)

os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "FactIndic_Mixed_23_Validated.csv"
)

mixed_articles_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

# Read it back to verify the saved file
saved_mixed = pd.read_csv(
    output_path,
    encoding="utf-8-sig"
)

print("Saved successfully:", os.path.exists(output_path))
print("File:", output_path)
print("Rows:", len(saved_mixed))
print("Columns:", saved_mixed.columns.tolist())

print("\nMissing values:")
print(saved_mixed.isna().sum())

print("\nSaved file preview:")
display(saved_mixed.head(2))

Saved successfully: True
File: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/04_mixed_data/FactIndic_Mixed_23_Validated.csv
Rows: 23
Columns: ['sample_id', 'hindi_article', 'mixed_10_article', 'mixed_30_article', 'mixed_50_article', 'title', 'url']

Missing values:
sample_id           0
hindi_article       0
mixed_10_article    0
mixed_30_article    0
mixed_50_article    0
title               0
url                 0
dtype: int64

Saved file preview:


,sample_id,hindi_article,mixed_10_article,mixed_30_article,mixed_50_article,title,url
0,080711_lanka_attack,एक सैनिक प्रवक्ता ब्रिगेडियर उदय नान्यकारा ने ...,एक सैनिक प्रवक्ता ब्रिगेडियर उदय नान्यकारा ने ...,एक सैनिक प्रवक्ता ब्रिगेडियर उदय नान्यकारा ने ...,एक सैनिक प्रवक्ता ब्रिगेडियर उदय नान्यकारा ने ...,बस पर गोलीबारी में चार की मौत,https://www.bbc.com/hindi/regionalnews/story/2...
1,051105_sahubuddin_arrest,शहाबुद्दीन बिहार के सीवान क्षेत्र से सांसद हैं...,शहाबुद्दीन बिहार के सीवान क्षेत्र से सांसद हैं...,"Shahabuddin, a Member of Parliament from the S...","Shahabuddin, a Member of Parliament from the S...",शहाबुद्दीन दिल्ली में गिरफ़्तार,https://www.bbc.com/hindi/regionalnews/story/2...


In [384]:
import os
import pandas as pd

output_path = os.path.join(
    base_path,
    "04_mixed_data",
    "FactIndic_Mixed_23_Validated.csv"
)

saved_mixed = pd.read_csv(output_path)

print("File exists:", os.path.exists(output_path))
print("Rows:", len(saved_mixed))
print("Unique articles:", saved_mixed["sample_id"].nunique())
print("Missing values:", saved_mixed.isna().sum().sum())

for level in [10, 30, 50]:
    col = f"mixed_{level}_article"

    english = saved_mixed[col].apply(count_english_words).sum()
    total = saved_mixed[col].apply(word_count).sum()

    print(f"{level}% target → Actual CMI: {english/total*100:.2f}%")

File exists: True
Rows: 23
Unique articles: 23
Missing values: 0
10% target → Actual CMI: 10.01%
30% target → Actual CMI: 29.52%
50% target → Actual CMI: 49.11%


In [385]:
cmi_rows = []

for label, col in [
    ("Original Hindi", "hindi_article"),
    ("10% Target", "mixed_10_article"),
    ("30% Target", "mixed_30_article"),
    ("50% Target", "mixed_50_article")
]:

    english_words = saved_mixed[col].apply(
        count_english_words
    ).sum()

    total_words = saved_mixed[col].apply(
        word_count
    ).sum()

    cmi_rows.append({
        "Version": label,
        "Articles": len(saved_mixed),
        "English_words": english_words,
        "Total_words": total_words,
        "Actual_CMI": round(
            english_words / total_words * 100, 2
        )
    })

cmi_summary = pd.DataFrame(cmi_rows)

cmi_path = os.path.join(
    base_path,
    "05_quality_checks",
    "FactIndic_CMI_Summary_23.csv"
)

os.makedirs(os.path.dirname(cmi_path), exist_ok=True)

cmi_summary.to_csv(
    cmi_path,
    index=False,
    encoding="utf-8-sig"
)

display(cmi_summary)

print("\nSaved:", os.path.exists(cmi_path))
print("Path:", cmi_path)

,Version,Articles,English_words,Total_words,Actual_CMI
0,Original Hindi,23,0,5954,0.00
1,10% Target,23,590,5893,10.01
2,30% Target,23,1689,5722,29.52
3,50% Target,23,2715,5528,49.11



Saved: True
Path: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/FactIndic_CMI_Summary_23.csv


In [386]:
methodology = """
FactIndic Controlled Code-Mixing Generation
Person B - Data Preparation

1. Source data:
- 250 Hindi news articles were initially prepared.
- 23 articles were selected for controlled mixing.
- Selection was based on at least 10 strictly approved
  Hindi-English sentence pairs per article.

2. Sentence alignment:
- 275 strictly approved sentence pairs were used.
- All 275 Hindi sentences were found exactly in their
  corresponding original Hindi articles.
- 33 additional sentences had no approved English pair
  and were retained in Hindi.

3. Generation:
- Original article sentence order was preserved.
- Selected approved Hindi sentences were replaced with
  their corresponding reviewed English translations.
- No random word replacement was used.
- Three versions were generated using deterministic
  sentence selection.

4. Final CMI results:
- Original Hindi: 0.00%
- 10% target: 10.01%
- 30% target: 29.52%
- 50% target: 49.11%

5. Validation:
- 23 unique articles in each version.
- 308 sentences across the 23 articles.
- Zero invalid replacements.
- Zero missing sentences.
- Zero missing values in the saved article dataset.

6. Limitations:
- Mixing is performed at sentence level, not word level.
- Actual CMI may differ slightly from target percentages.
- The 23-article dataset is a validated subset, not the
  complete 250-article mixed dataset.
- Human fluency review is still required.
"""

method_path = os.path.join(
    base_path,
    "08_reports",
    "FactIndic_Mixing_Methodology.txt"
)

os.makedirs(os.path.dirname(method_path), exist_ok=True)

with open(method_path, "w", encoding="utf-8") as f:
    f.write(methodology)

print("Methodology saved:", os.path.exists(method_path))
print("Path:", method_path)

Methodology saved: True
Path: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/08_reports/FactIndic_Mixing_Methodology.txt


In [387]:
review_sample = saved_mixed.sample(
    n=5,
    random_state=42
)[["sample_id", "hindi_article",
   "mixed_10_article",
   "mixed_30_article",
   "mixed_50_article"]].copy()

display(review_sample)

,sample_id,hindi_article,mixed_10_article,mixed_30_article,mixed_50_article
15,031114_bush_intelligence,ये कार्रवाइयाँ अमरीकी नेतृत्व वाली फ़ौजों के ख...,ये कार्रवाइयाँ अमरीकी नेतृत्व वाली फ़ौजों के ख...,ये कार्रवाइयाँ अमरीकी नेतृत्व वाली फ़ौजों के ख...,These operations have been launched in the wak...
9,070902_khanduri_win,धूमकोट सीट के लिए उपचुनाव 29 अगस्त को हुआ था. ...,धूमकोट सीट के लिए उपचुनाव 29 अगस्त को हुआ था. ...,धूमकोट सीट के लिए उपचुनाव 29 अगस्त को हुआ था. ...,धूमकोट सीट के लिए उपचुनाव 29 अगस्त को हुआ था. ...
0,080711_lanka_attack,एक सैनिक प्रवक्ता ब्रिगेडियर उदय नान्यकारा ने ...,एक सैनिक प्रवक्ता ब्रिगेडियर उदय नान्यकारा ने ...,एक सैनिक प्रवक्ता ब्रिगेडियर उदय नान्यकारा ने ...,एक सैनिक प्रवक्ता ब्रिगेडियर उदय नान्यकारा ने ...
8,051121_aids_india,इस रिपोर्ट के अनुसार अभी भी गर्भवती महिलाओं को...,इस रिपोर्ट के अनुसार अभी भी गर्भवती महिलाओं को...,"According to the report, pregnant women still ...","According to the report, pregnant women still ..."
17,070620_heart_survival,जर्मनी और स्विट्जरलैंड के शोधकर्ताओं ने 1676 म...,जर्मनी और स्विट्जरलैंड के शोधकर्ताओं ने 1676 म...,जर्मनी और स्विट्जरलैंड के शोधकर्ताओं ने 1676 म...,जर्मनी और स्विट्जरलैंड के शोधकर्ताओं ने 1676 म...


In [388]:
sample_id = review_sample.iloc[0]["sample_id"]

for level in [10, 30, 50]:

    print("\n" + "=" * 60)
    print(f"ARTICLE: {sample_id} | {level}% MIXING")
    print("=" * 60)

    article = review_sample.iloc[0][
        f"mixed_{level}_article"
    ]

    for i, sentence in enumerate(
        split_sentences(article), start=1
    ):
        print(f"{i}. {sentence}")


ARTICLE: 031114_bush_intelligence | 10% MIXING
1. ये कार्रवाइयाँ अमरीकी नेतृत्व वाली फ़ौजों के ख़िलाफ़ बढ़ रहे हमलों के बाद शुरु की गई है.
2. विद्रोहियों की धरपकड़ के लिए अमरीकी अभियान तीसरी रात भी जारी रहा.
3. ऐसी कार्रवाइयाँ इराक़ी शहर तिकरित में भी हुई हैं.
4. तिकरित में इससे पहले अमरीकी फ़ौजों ने एक संदिग्ध समूह पर मिसाइलों से हमला किया था.
5. अमरीकी फ़ौज का कहना है कि ये लोग अमरीकी सेना पर रॉकेटों से हमला करने की योजना बना रहे थे.
6. इसमें सात इराक़ियों की मौत हुई है.
7. एक अमरीकी अधिकारी ने कहा कि तिकरित के आसापस मारे गए छापों से संकेत मिलता है कि विद्रोही काफ़ी सुनियोजित तरीक़े से योजनाएँ बना रहे हैं.
8. नासिरिया के इतालवी पुलिस अड्डे पर बुधवार को हुए आत्मघाती हमले के बाद दक्षिणी शहर बसरा में भी सुरक्षा कड़ी कर दी गई है.
9. अमरीकी सेना का कहना है कि बृहस्पतिवार को समारा के निकट हुए हमले में उसके दो सैनिक मारे गए और तीन अन्य घायल हो गए.
10. दूसरी ओर अमरीकी राष्ट्रपति जॉर्ज डब्ल्यू बुश ने कहा है कि अमरीका को ख़ुफ़िया सूचनाएँ एकत्र करने के लिए और इराक़ियों की मदद लेनी चाहिए ताकि अ

In [389]:
article_id = "031114_bush_intelligence"

source_row = eligible_source[
    eligible_source["id"] == article_id
].iloc[0]

print("ORIGINAL HINDI ARTICLE:")
print(source_row["hindi_article"])

print("\nREVIEWED FULL ENGLISH TRANSLATION:")
print(source_row["complete_english_translation"])

ORIGINAL HINDI ARTICLE:
ये कार्रवाइयाँ अमरीकी नेतृत्व वाली फ़ौजों के ख़िलाफ़ बढ़ रहे हमलों के बाद शुरु की गई है. विद्रोहियों की धरपकड़ के लिए अमरीकी अभियान तीसरी रात भी जारी रहा. ऐसी कार्रवाइयाँ इराक़ी शहर तिकरित में भी हुई हैं. तिकरित में इससे पहले अमरीकी फ़ौजों ने एक संदिग्ध समूह पर मिसाइलों से हमला किया था. अमरीकी फ़ौज का कहना है कि ये लोग अमरीकी सेना पर रॉकेटों से हमला करने की योजना बना रहे थे. इसमें सात इराक़ियों की मौत हुई है. एक अमरीकी अधिकारी ने कहा कि तिकरित के आसापस मारे गए छापों से संकेत मिलता है कि विद्रोही काफ़ी सुनियोजित तरीक़े से योजनाएँ बना रहे हैं. नासिरिया के इतालवी पुलिस अड्डे पर बुधवार को हुए आत्मघाती हमले के बाद दक्षिणी शहर बसरा में भी सुरक्षा कड़ी कर दी गई है. अमरीकी सेना का कहना है कि बृहस्पतिवार को समारा के निकट हुए हमले में उसके दो सैनिक मारे गए और तीन अन्य घायल हो गए. दूसरी ओर अमरीकी राष्ट्रपति जॉर्ज डब्ल्यू बुश ने कहा है कि अमरीका को ख़ुफ़िया सूचनाएँ एकत्र करने के लिए और इराक़ियों की मदद लेनी चाहिए ताकि अमरीकी नेतृत्व वाले बलों पर हमला करने वालों के ख़िलाफ़ क

In [390]:
review_record = pd.DataFrame([{
    "sample_id": "031114_bush_intelligence",
    "reviewed_levels": "10%, 30%, 50%",
    "sentence_1_issue": "Hindi grammatical construction needs review",
    "sentence_7_issue": "Source typo: आसापस",
    "sentence_8_status": "Meaning appears consistent",
    "sentence_11_status": "English translation matches original Hindi meaning",
    "overall_status": "Needs minor language review",
    "reviewer_notes": (
        "Sentence-level switching is clear. "
        "Original Hindi issues should be recorded separately "
        "from translation errors."
    )
}])

review_path = os.path.join(
    base_path,
    "05_quality_checks",
    "FactIndic_Mixed_Human_Review.csv"
)

if os.path.exists(review_path):
    existing_reviews = pd.read_csv(review_path)
    existing_reviews = existing_reviews[
        existing_reviews["sample_id"] != "031114_bush_intelligence"
    ]
    review_record = pd.concat(
        [existing_reviews, review_record],
        ignore_index=True
    )

review_record.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Review record saved:", os.path.exists(review_path))
print("Review records:", len(review_record))

Review record saved: True
Review records: 1


In [391]:
article_id = "070902_khanduri_win"

for level in [10, 30, 50]:

    print("\n" + "=" * 60)
    print(f"ARTICLE: {article_id} | {level}% MIXING")
    print("=" * 60)

    article = saved_mixed.loc[
        saved_mixed["sample_id"] == article_id,
        f"mixed_{level}_article"
    ].iloc[0]

    for i, sentence in enumerate(
        split_sentences(article), start=1
    ):
        print(f"{i}. {sentence}")


ARTICLE: 070902_khanduri_win | 10% MIXING
1. धूमकोट सीट के लिए उपचुनाव 29 अगस्त को हुआ था.
2. रविवार को हुई मतगणना में खंडूरी ने 14 हज़ार वोटों के अंतर से जीत दर्ज की.
3. भुवन चंद्र खंडूरी ने भारतीय जनता पार्टी की ओर से चुनाव लड़ा था.
4. खंडूरी ने मार्च में मुख्यमंत्री पद की शपथ ली थी.
5. उस समय वह पौड़ी क्षेत्र से लोकसभा सदस्य थे.
6. बाद में उन्होंने लोकसभा की सदस्यता से इस्तीफ़ा दे दिया था.
7. बहुमत हैरानी की बात ये रही कि कांग्रेस विधायक लेफ़्टिनेंट जनरल (सेवानिवृत्त) टीपीएस रावत ने खंडूरी के लिए अपनी सीट खाली की.
8. वे धूमाकोट से दो बार विधायक रहे हैं.
9. फरवरी में हुए चुनाव में राज्य विधानसभा की 70 सीटों में से 69 पर हुए चुनाव में भाजपा को 34 सीटें मिलीं थी जो बहुमत से एक कम थी.
10. बाद में विश्वास मत के दौरान खंडूरी के पक्ष में 39 मत पड़े.
11. Three Uttarakhand Kranti Dal (Ukrand) MLAs and as many independents had voted in favour of the trust vote.

ARTICLE: 070902_khanduri_win | 30% MIXING
1. धूमकोट सीट के लिए उपचुनाव 29 अगस्त को हुआ था.
2. In Sunday's counting, Khanduri won by

In [392]:
article_id = "070902_khanduri_win"

source_row = eligible_source[
    eligible_source["id"] == article_id
].iloc[0]

print("ORIGINAL HINDI ARTICLE:\n")
print(source_row["hindi_article"])

print("\nREVIEWED ENGLISH TRANSLATION:\n")
print(source_row["complete_english_translation"])

ORIGINAL HINDI ARTICLE:

धूमकोट सीट के लिए उपचुनाव 29 अगस्त को हुआ था. रविवार को हुई मतगणना में खंडूरी ने 14 हज़ार वोटों के अंतर से जीत दर्ज की. भुवन चंद्र खंडूरी ने भारतीय जनता पार्टी की ओर से चुनाव लड़ा था. खंडूरी ने मार्च में मुख्यमंत्री पद की शपथ ली थी. उस समय वह पौड़ी क्षेत्र से लोकसभा सदस्य थे. बाद में उन्होंने लोकसभा की सदस्यता से इस्तीफ़ा दे दिया था. बहुमत हैरानी की बात ये रही कि कांग्रेस विधायक लेफ़्टिनेंट जनरल (सेवानिवृत्त) टीपीएस रावत ने खंडूरी के लिए अपनी सीट खाली की. वे धूमाकोट से दो बार विधायक रहे हैं. फरवरी में हुए चुनाव में राज्य विधानसभा की 70 सीटों में से 69 पर हुए चुनाव में भाजपा को 34 सीटें मिलीं थी जो बहुमत से एक कम थी. बाद में विश्वास मत के दौरान खंडूरी के पक्ष में 39 मत पड़े. उत्तराखंड क्रांति दल (उक्रांद) के तीन और इतने ही निर्दलीय विधायकों ने विश्वास मत के समर्थन में मतदान किया था.

REVIEWED ENGLISH TRANSLATION:

The by-election for the Dhumkot seat was held on 29 August. In Sunday's counting, Khanduri won by a margin of 14,000 votes. Bhuvan Chandra Khanduri co

In [393]:
new_review = pd.DataFrame([{
    "sample_id": "070902_khanduri_win",
    "reviewed_levels": "10%, 30%, 50%",
    "sentence_7_issue": "Awkward source phrase: बहुमत हैरानी की बात",
    "sentence_8_issue": "Check Dhumakot / धूमाकोट spelling consistency",
    "sentence_9_issue": "Hindi source wording is repetitive; verify translation",
    "sentence_11_status": "Originally Hindi; English translation preserves meaning",
    "overall_status": "Needs minor source-language review",
    "reviewer_notes": (
        "Sentence-level switching is clear. "
        "Election date, vote margin, party and trust-vote figures "
        "are retained in the reviewed versions."
    )
}])

existing_reviews = pd.read_csv(review_path)

existing_reviews = existing_reviews[
    existing_reviews["sample_id"] != "070902_khanduri_win"
]

existing_reviews = pd.concat(
    [existing_reviews, new_review],
    ignore_index=True
)

existing_reviews.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", os.path.exists(review_path))
print("Total reviewed articles:", len(existing_reviews))
print(existing_reviews["sample_id"].tolist())

Saved: True
Total reviewed articles: 2
['031114_bush_intelligence', '070902_khanduri_win']


In [394]:
article_id = "080711_lanka_attack"

for level in [10, 30, 50]:

    print("\n" + "=" * 60)
    print(f"ARTICLE: {article_id} | {level}% MIXING")
    print("=" * 60)

    article = saved_mixed.loc[
        saved_mixed["sample_id"] == article_id,
        f"mixed_{level}_article"
    ].iloc[0]

    for i, sentence in enumerate(
        split_sentences(article), start=1
    ):
        print(f"{i}. {sentence}")


ARTICLE: 080711_lanka_attack | 10% MIXING
1. एक सैनिक प्रवक्ता ब्रिगेडियर उदय नान्यकारा ने इस हमले के लिए तमिल विद्रोहियों को ज़िम्मेदार ठहराया है.
2. ये घटना मोनारगाला ज़िले में हुई जब इस बस पर अंधाधुंध गोलियाँ चलाई गई.
3. अभी तक एलटीटीई की ओर से इस घटना पर कोई टिप्पणी नहीं आई है.
4. ये हमला ऐसे समय हुआ है जब देश के उत्तरी इलाक़े में सेना और तमिल विद्रोहियों के बीच संघर्ष में तेज़ी आई है.
5. A BBC correspondent in Colombo reported that the bus was passing through a deserted area when shots were fired at it.
6. हमला मरने वालों में एक नौ साल का बच्चा भी है.
7. गोलीबारी में कई यात्री घायल भी हुए हैं.
8. Military spokesman Brigadier Uday Nanyakara said the gunmen were probably hiding beside the road.
9. उन्होंने इस गोलीबारी के लिए तमिल विद्रोहियों को ज़िम्मेदार ठहराया है.
10. एक साल पहले श्रीलंका की सरकार ने दावा किया था कि उन्होंने विद्रोहियों को पूर्वी हिस्से से खदेड़ दिया है लेकिन इस इलाक़े में छिटपुट हिंसा की घटनाएँ जारी हैं.
11. In January of the same year, rebels had also targeted 

In [395]:
new_review = pd.DataFrame([{
    "sample_id": "080711_lanka_attack",
    "reviewed_levels": "10%, 30%, 50%",
    "sentence_2_issue": "Hindi grammar needs review",
    "sentence_6_issue": "Malformed Hindi phrase: हमला मरने वालों में",
    "sentence_9_issue": "Repeated attribution; check source consistency",
    "sentence_12_status": "English version is clearer than Hindi source wording",
    "sentence_15_status": "Verify reported figures against source",
    "overall_status": "Needs source-language and factual review",
    "reviewer_notes": (
        "English replacements are generally readable. "
        "Sentence-level switching is visible. "
        "Original Hindi language issues must be distinguished "
        "from translation errors."
    )
}])

existing_reviews = pd.read_csv(review_path)

existing_reviews = existing_reviews[
    existing_reviews["sample_id"] != "080711_lanka_attack"
]

existing_reviews = pd.concat(
    [existing_reviews, new_review],
    ignore_index=True
)

existing_reviews.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", os.path.exists(review_path))
print("Total reviewed articles:", len(existing_reviews))

Saved: True
Total reviewed articles: 3


In [396]:
article_id = "051121_aids_india"

for level in [10, 30, 50]:

    print("\n" + "=" * 60)
    print(f"ARTICLE: {article_id} | {level}% MIXING")
    print("=" * 60)

    article = saved_mixed.loc[
        saved_mixed["sample_id"] == article_id,
        f"mixed_{level}_article"
    ].iloc[0]

    for i, sentence in enumerate(
        split_sentences(article), start=1
    ):
        print(f"{i}. {sentence}")


ARTICLE: 051121_aids_india | 10% MIXING
1. इस रिपोर्ट के अनुसार अभी भी गर्भवती महिलाओं को संक्रमण का ख़तरा बहुत अधिक बना हुआ है.
2. It also says that the number of people living with HIV has increased worldwide.
3. यह रिपोर्ट दुनिया भर में सोमवार को जारी की गई.
4. भारत में 51 लाख लोग एचआईवी वायरस से संक्रमित हैं.
5. यूएनएड्स के अनुसार दक्षिण और पश्चिमी राज्यों तमिलनाडु, आंध्र प्रदेश, कर्नाटक और महाराष्ट्र में संक्रमण की दर में स्थिरता आई है.
6. लेकिन बिहार और उत्तर प्रदेश जैसे ग़रीब और सघन जनसंख्या वाले उत्तरी भारत के राज्यों में गर्भवती महिलाओं में संक्रमण का ख़तरा बहुत अधिक है.
7. The report said that more than one per cent of women in Tamil Nadu, Andhra Pradesh, Karnataka and Maharashtra, as well as in the northeastern states of Nagaland and Manipur, were found to be infected with HIV.
8. यूएनएड्स का कहना है कि इन महिलाओं को वायरस उनके पति से ही मिल रहा है और अब इसका दायरा शहरी इलाक़ों के बाहर भी बढ़ रहा है.
9. रिपोर्ट में कहा गया है कि इन पुरुषों को एड्स वेश्याओं से असुरक्षित यौन 

In [397]:
new_review = pd.DataFrame([{
    "sample_id": "051121_aids_india",
    "reviewed_levels": "10%, 30%, 50%",
    "sentence_12_issue": "Hindi grammar is malformed; needs source review",
    "sentence_13_issue": "Awkward Hindi construction; needs source review",
    "sentence_7_status": "Verify reported percentages and state names against source",
    "sentence_14_status": "English version preserves reported percentages and years",
    "sentence_16_status": "English version is readable and preserves main meaning",
    "overall_status": "Needs source-language review",
    "reviewer_notes": (
        "English replacements are generally readable. "
        "Historical health statistics are treated as source claims, "
        "not current medical facts."
    )
}])

existing_reviews = pd.read_csv(review_path)

existing_reviews = existing_reviews[
    existing_reviews["sample_id"] != "051121_aids_india"
]

existing_reviews = pd.concat(
    [existing_reviews, new_review],
    ignore_index=True
)

existing_reviews.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", os.path.exists(review_path))
print("Total reviewed articles:", len(existing_reviews))

Saved: True
Total reviewed articles: 4


In [398]:
article_id = "070620_heart_survival"

for level in [10, 30, 50]:

    print("\n" + "=" * 60)
    print(f"ARTICLE: {article_id} | {level}% MIXING")
    print("=" * 60)

    article = saved_mixed.loc[
        saved_mixed["sample_id"] == article_id,
        f"mixed_{level}_article"
    ].iloc[0]

    for i, sentence in enumerate(
        split_sentences(article), start=1
    ):
        print(f"{i}. {sentence}")


ARTICLE: 070620_heart_survival | 10% MIXING
1. जर्मनी और स्विट्जरलैंड के शोधकर्ताओं ने 1676 मरीज़ों पर किए अध्ययन में पाया कि इलाज के तीन साल बाद तक मोटापे से जूझ रहे मरीज़ों की मौत की संख्या सामान्य वज़न वाले लोगों की मौत की संख्या से आधी थी.
2. However, the study, published in the European Heart Journal, did not explain why obese patients had a lower risk of death.
3. विशेषज्ञों ने ये भी बताया कि मोटे लोगों में दिल की बीमारी होने की संभावना भी अधिक होती है.
4. अस्पताल में भर्ती अस्थिर एंजिना और दिल के दौरे की बीमारी से परेशान 1676 मरीज़ों पर ये अध्ययन किया गया.
5. शीर्ष शोधकर्ता डॉ हींज़ ब्यूटनर ने कहा, "इसमें शक़ नहीं कि मोटे लोगों को मधुमेह, हाइपरटेंशन और हृदय धमनी की बीमारी होने की संभावना अधिक होती है.
6. हमारे अध्ययन से पता चला कि यदि एक बार हृदय धमनी का इलाज ठीक से हो जाए तो सामान्य वज़न वाले लोगों की तुलना में मोटे लोगों के स्वस्थ रहने की संभावना बढ़ जाती है." वजहें सामान्य वज़न वालों की तुलना में मोटे लोग अधिक युवा दिखते हैं और उन्हें दिल की बीमारियों की कुछ दवाएँ देकर अस्पत

In [399]:
new_review = pd.DataFrame([{
    "sample_id": "070620_heart_survival",
    "reviewed_levels": "10%, 30%, 50%",
    "sentence_1_issue": "Verify study figures against source",
    "sentence_5_6_issue": "Quotation split across languages and sentence boundaries",
    "sentence_6_issue": "Awkward joining of the phrase beginning वजहें",
    "sentence_11_12_issue": "Quotation formatting and closing quotation mark need review",
    "overall_status": "Needs minor language and formatting review",
    "reviewer_notes": (
        "Main study details are understandable. "
        "Preserve the original meaning and check quotation boundaries. "
        "Historical research claims are not current medical advice."
    )
}])

existing_reviews = pd.read_csv(review_path)

existing_reviews = existing_reviews[
    existing_reviews["sample_id"] != "070620_heart_survival"
]

existing_reviews = pd.concat(
    [existing_reviews, new_review],
    ignore_index=True
)

existing_reviews.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", os.path.exists(review_path))
print("Total reviewed articles:", len(existing_reviews))
print(existing_reviews["sample_id"].tolist())

Saved: True
Total reviewed articles: 5
['031114_bush_intelligence', '070902_khanduri_win', '080711_lanka_attack', '051121_aids_india', '070620_heart_survival']


In [400]:
final_reviews = pd.read_csv(review_path)

print("File exists:", os.path.exists(review_path))
print("Total review records:", len(final_reviews))
print("Unique articles:", final_reviews["sample_id"].nunique())

print("\nREVIEWED ARTICLES:")
print(final_reviews["sample_id"].tolist())

print("\nMissing values:")
print(final_reviews.isnull().sum())

print("\nOverall review status:")
print(final_reviews["overall_status"].value_counts())

File exists: True
Total review records: 5
Unique articles: 5

REVIEWED ARTICLES:
['031114_bush_intelligence', '070902_khanduri_win', '080711_lanka_attack', '051121_aids_india', '070620_heart_survival']

Missing values:
sample_id               0
reviewed_levels         0
sentence_1_issue        3
sentence_7_issue        3
sentence_8_status       4
sentence_11_status      3
overall_status          0
reviewer_notes          0
sentence_8_issue        4
sentence_9_issue        3
sentence_2_issue        4
sentence_6_issue        3
sentence_12_status      4
sentence_15_status      4
sentence_12_issue       4
sentence_13_issue       4
sentence_7_status       4
sentence_14_status      4
sentence_16_status      4
sentence_5_6_issue      4
sentence_11_12_issue    4
dtype: int64

Overall review status:
overall_status
Needs minor language review                   1
Needs minor source-language review            1
Needs source-language and factual review      1
Needs source-language review           

In [401]:
hinglish_path = os.path.join(
    base_path,
    "06_real_hinglish",
    "FactIndic_Real_Hinglish_Final_Candidates.csv"
)

print("File exists:", os.path.exists(hinglish_path))

hinglish_df = pd.read_csv(hinglish_path)

print("Rows:", len(hinglish_df))
print("Columns:", hinglish_df.columns.tolist())

display(hinglish_df.head(5))

File exists: True
Rows: 144
Columns: ['source_dataset', 'source_config', 'text', 'script_type', 'selection_method', 'reply_to_text', 'likely_hinglish']


,source_dataset,source_config,text,script_type,selection_method,reply_to_text,likely_hinglish
0,MUTANT,Mankibaat,"मेरे प्यारे देशवासियो, आप सबको नमस्कार। 2015 –...",Mixed Devanagari-Latin,Script-based candidate filter,NaN,True
1,MUTANT,Mankibaat,अनेक लोगों के अथक प्रयास का परिणाम है कि देश ब...,Mixed Devanagari-Latin,Script-based candidate filter,NaN,True
2,MUTANT,Mankibaat,स्वच्छता की बात एक प्रकार से घर-घर में गूंज रह...,Mixed Devanagari-Latin,Script-based candidate filter,NaN,True
3,MUTANT,Mankibaat,"मेरे प्यारे नौजवान साथियो, 15 अगस्त को लाल किल...",Mixed Devanagari-Latin,Script-based candidate filter,NaN,True
4,MUTANT,Mankibaat,Start-up के संबंध में हमारे यहाँ एक सोच बंधी-ब...,Mixed Devanagari-Latin,Script-based candidate filter,NaN,True


In [402]:
summary_path = os.path.join(
    base_path,
    "06_real_hinglish",
    "FactIndic_Hinglish_Final_Review_Summary.csv"
)

print("Summary exists:", os.path.exists(summary_path))

hinglish_summary = pd.read_csv(summary_path)

print("\nReview summary:")
display(hinglish_summary)

print("\nColumns:")
print(hinglish_summary.columns.tolist())

Summary exists: True

Review summary:


,source_dataset,No,Yes
0,GupShup,17,27
1,MUTANT,5,46



Columns:
['source_dataset', 'No', 'Yes']


In [403]:
folder = os.path.join(base_path, "06_real_hinglish")

print("Files in real Hinglish folder:\n")

for filename in os.listdir(folder):
    print(filename)

Files in real Hinglish folder:

MUTANT_Hinglish_100_Candidates.csv
GupShup_100_Candidates.csv
FactIndic_Real_Hinglish_200_Candidates.csv
FactIndic_Hinglish_200_Screened.csv
FactIndic_Hinglish_144_Likely.csv
FactIndic_Real_Hinglish_Final_Candidates.csv
FactIndic_Hinglish_Manual_Review.csv
FactIndic_Hinglish_95_Strict_Candidates.csv
FactIndic_Hinglish_20_Review.csv
FactIndic_Hinglish_Review_21_40.csv
FactIndic_Hinglish_Review_41_60.csv
FactIndic_Hinglish_Review_61_80.csv
FactIndic_Hinglish_Review_81_95.csv
FactIndic_Hinglish_95_Human_Review.csv
FactIndic_Hinglish_73_Human_Reviewed.csv
FactIndic_Hinglish_Final_Review_Summary.csv


In [404]:
review_95_path = os.path.join(
    base_path,
    "06_real_hinglish",
    "FactIndic_Hinglish_95_Human_Review.csv"
)

hinglish_reviews = pd.read_csv(review_95_path)

print("Rows:", len(hinglish_reviews))
print("Columns:", hinglish_reviews.columns.tolist())

display(hinglish_reviews.head(3))

Rows: 95
Columns: ['source_dataset', 'source_config', 'text', 'script_type', 'selection_method', 'reply_to_text', 'likely_hinglish', 'human_verified', 'language_quality', 'meaning_clear', 'review_notes', 'latin_word_count', 'total_word_count', 'latin_word_percentage', 'latin_words', 'devanagari_words', 'roman_hindi_words', 'total_language_words', 'verified_hinglish', 'english_words_used_naturally']


,source_dataset,source_config,text,script_type,selection_method,reply_to_text,likely_hinglish,human_verified,language_quality,meaning_clear,review_notes,latin_word_count,total_word_count,latin_word_percentage,latin_words,devanagari_words,roman_hindi_words,total_language_words,verified_hinglish,english_words_used_naturally
0,MUTANT,Mankibaat,"मेरे प्यारे नौजवान साथियो, 15 अगस्त को लाल किल...",Mixed Devanagari-Latin,Script-based candidate filter,NaN,True,NaN,NaN,Yes,Natural English technical terms,42,163,25.77,42,133,0,175,Yes,Yes
1,MUTANT,Mankibaat,Start-up के संबंध में हमारे यहाँ एक सोच बंधी-ब...,Mixed Devanagari-Latin,Script-based candidate filter,NaN,True,NaN,NaN,Yes,English technology and payment terms,18,192,9.38,18,180,0,198,Yes,Yes
2,MUTANT,Mankibaat,"प्यारे नौजवान साथियो, 12 जनवरी स्वामी विवेकानं...",Mixed Devanagari-Latin,Script-based candidate filter,NaN,True,NaN,NaN,Yes,English youth festival and event terminology,30,233,12.88,30,201,0,231,Yes,Yes


In [405]:
reviewed_texts = set(
    hinglish_reviews["text"].dropna().astype(str)
)

unreviewed = hinglish_df[
    ~hinglish_df["text"].astype(str).isin(reviewed_texts)
].copy()

print("Already reviewed:", len(reviewed_texts))
print("Unreviewed candidates:", len(unreviewed))

next_batch = unreviewed.head(5).copy()

display(
    next_batch[
        ["source_dataset", "source_config", "text"]
    ]
)

Already reviewed: 95
Unreviewed candidates: 49


,source_dataset,source_config,text
0,MUTANT,Mankibaat,"मेरे प्यारे देशवासियो, आप सबको नमस्कार। 2015 –..."
1,MUTANT,Mankibaat,अनेक लोगों के अथक प्रयास का परिणाम है कि देश ब...
2,MUTANT,Mankibaat,स्वच्छता की बात एक प्रकार से घर-घर में गूंज रह...
9,MUTANT,Mankibaat,"प्यारे देशवासियो, 26 जनवरी का पर्व बहुत उमंग औ..."
10,MUTANT,Mankibaat,"मेरे प्यारे देशवासियो, एक काम के लिये मुझे आपक..."


In [406]:
for i, row in next_batch.iterrows():
    print(f"\n{'='*70}")
    print("INDEX:", i)
    print("TEXT:\n", row["text"])


INDEX: 0
TEXT:
 मेरे प्यारे देशवासियो, आप सबको नमस्कार। 2015 – एक प्रकार से मेरी इस वर्ष की आख़िरी ‘मन की बात’। अगले ‘मन की बात’ 2016 में होगी। अभी-अभी हम लोगों ने क्रिसमस का पर्व मनाया और अब नये वर्ष के स्वागत की तैयारियाँ चल रही हैं। भारत विविधताओं से भरा हुआ है। त्योहारों की भी भरमार लगी रहती है। एक त्योहार गया नहीं कि दूसरा आया नहीं। एक प्रकार से हर त्योहार दूसरे त्योहार की प्रतीक्षा को छोड़कर चला जाता है। कभी-कभी तो लगता है कि भारत एक ऐसा देश है, जहाँ पर ‘त्योहार Driven Economy’ भी है। समाज के ग़रीब तबक़े के लोगों की आर्थिक गतिविधि का वो कारण बन जाता है। मेरी तरफ़ से सभी देशवासियों को क्रिसमस की भी बहुत-बहुत शुभकामनायें और 2016 के नववर्ष की भी बहुत-बहुत शुभकामनायें। 2016 का वर्ष आप सभी के लिए ढेरों खुशियाँ ले करके आये। नया उमंग, नया उत्साह, नया संकल्प आपको नयी ऊंचाइयों पर पहुंचाए। दुनिया भी संकटों से मुक्त हो, चाहे आतंकवाद हो, चाहे ग्लोबल वार्मिंग हो, चाहे प्राकृतिक आपदायें हों, चाहे मानव सृजित संकट हो। मानव जाति सुखचैन की ज़िंदगी पाये, इससे बढ़कर के खुशी क्या हो सकती है|

INDEX: 1
TEXT

In [407]:
new_reviews = next_batch.copy()

new_reviews["human_verified"] = "Yes"
new_reviews["language_quality"] = "Good"
new_reviews["meaning_clear"] = "Yes"
new_reviews["review_notes"] = (
    "Hindi-dominant code-mixed text; "
    "English expressions are contextually understandable."
)
new_reviews["verified_hinglish"] = "Yes"
new_reviews["english_words_used_naturally"] = "Yes"

updated_reviews = pd.concat(
    [hinglish_reviews, new_reviews],
    ignore_index=True
)

updated_reviews.to_csv(review_95_path, index=False)

check = pd.read_csv(review_95_path)

print("Total reviewed:", len(check))
print("Unique reviewed texts:", check["text"].nunique())
print("\nVerified Hinglish:")
print(check["verified_hinglish"].value_counts(dropna=False))

Total reviewed: 100
Unique reviewed texts: 100

Verified Hinglish:
verified_hinglish
Yes    78
No     22
Name: count, dtype: int64


In [408]:
translation_review_path = os.path.join(
    base_path,
    "05_quality_checks",
    "Human_Translation_Review_20.csv"
)

translation_review = pd.read_csv(translation_review_path)

print("Rows:", len(translation_review))
print("Columns:", translation_review.columns.tolist())
display(translation_review.head())

Rows: 20
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words', 'nllb_english_title', 'nllb_english_article', 'nllb_english_summary', 'english_title', 'english_text', 'english_summary', 'hindi_cmi', 'english_cmi', 'hi_words', 'en_words', 'translation_ratio', 'review_status', 'translation_quality', 'names_correct', 'numbers_correct', 'meaning_preserved', 'comments']


,id,url,title,summary,text,article_words,summary_words,nllb_english_title,nllb_english_article,nllb_english_summary,...,english_cmi,hi_words,en_words,translation_ratio,review_status,translation_quality,names_correct,numbers_correct,meaning_preserved,comments
0,international-44164669,https://www.bbc.com/hindi/international-44164669,"अमरीका और उत्तर कोरिया: कभी हां, कभी ना",अमरीका और उत्तर कोरिया के बीच सिंगापुर में 12 ...,राष्ट्रपति डोनल्ड ट्रंप से बातचीत की हामी भरने...,695,24,NaN,NaN,NaN,...,99.22,695,387,0.556835,Needs human review,NaN,NaN,NaN,NaN,NaN
1,140702_wc2014_goldenboot_contenders_ar,https://www.bbc.com/hindi/sport/2014/07/140702...,विश्व कप: किसको मिलेगा गोल्डन बूट?,विश्व कप फ़ुटबॉल टूर्नामेंट में एक तरफ़ अब इंत...,इनमें कोलंबिया के जेम्स रॉड्रिगेज़ सबसे आगे है...,239,40,NaN,NaN,NaN,...,99.35,239,153,0.640167,Needs human review,NaN,NaN,NaN,NaN,NaN
2,121114_international_us_plus_obama_ia,https://www.bbc.com/hindi/international/2012/1...,राष्ट्रीय सुरक्षा को कोई ख़तरा नहीं: ओबामा,अमरीकी राष्ट्रपति बराक ओबामा ने कहा है कि अमरी...,प्रेम संबंधों के कारण सीआईए प्रमुख को इस्तीफ़ा...,790,33,NaN,NaN,NaN,...,100.00,790,439,0.555696,Needs human review,NaN,NaN,NaN,NaN,NaN
3,040511_shahbaz_profile,https://www.bbc.com/hindi/regionalnews/story/2...,शहबाज़ शरीफ़:राजनीति और निर्वासन,मियाँ मोहम्मद शहबाज़ शरीफ़ का संबंध पाकिस्तनी ...,उनके बड़े भाई मियाँ नवाज़ शरीफ़ पाकिस्तान के द...,254,21,NaN,NaN,NaN,...,95.71,254,163,0.641732,Needs human review,NaN,NaN,NaN,NaN,NaN
4,science-41399809,https://www.bbc.com/hindi/science-41399809,मोटापा कम करने का ये तरीका हो सकता है ख़तरनाक,मोटापा कम करने या दूसरे शब्दों में कहें तो शरी...,डॉक्टरों ने लिपोसक्शन को लेकर कुछ गंभीर चिंताए...,410,24,NaN,NaN,NaN,...,100.00,410,214,0.521951,Needs human review,NaN,NaN,NaN,NaN,NaN


In [409]:
for i in range(2):
    row = translation_review.iloc[i]

    print("\n" + "=" * 80)
    print("ARTICLE ID:", row["id"])
    print("TITLE:", row["title"])
    print("\nHINDI SUMMARY:\n", row["summary"])
    print("\nENGLISH SUMMARY:\n", row["english_summary"])
    print("\nHINDI ARTICLE:\n", row["text"])
    print("\nENGLISH TRANSLATION:\n", row["english_text"])


ARTICLE ID: international-44164669
TITLE: अमरीका और उत्तर कोरिया: कभी हां, कभी ना

HINDI SUMMARY:
 अमरीका और उत्तर कोरिया के बीच सिंगापुर में 12 जून को होने वाली बातचीत कभी हां और कभी ना के बीच झूल रही है.

ENGLISH SUMMARY:
 The conversation between the United States and North Korea is ever going up in Singapore on June 12, and never is there.

HINDI ARTICLE:
 राष्ट्रपति डोनल्ड ट्रंप से बातचीत की हामी भरने के बाद उत्तर कोरियाई नेता किम जोंग उन ने कहा है कि अमरीका जिस तरह की चालें चल रहा है, उस माहौल में बातचीत नहीं हो सकती. बुधवार को उत्तर कोरिया ने चेतावनी दी कि वो हो सकता है कि वो इस बातचीत का हिस्सा ना बने. ट्रंप की मुश्किल उनके ही राष्ट्रीय सुरक्षा सलाहकार जॉन बोल्टन ने बढ़ा दी है. जॉन बोल्टन का कहना है कि अमरीका उत्तर कोरिया में भी "लीबिया मॉडल" अपनाने की सोच रहा है. क्या ट्रंप की बढ़ती मांगों से नाराज़ है उत्तर कोरिया उत्तर कोरिया की ट्रंप के साथ बैठक रद्द करने की धमकी 'लीबिया मॉडल' पर नाराज़ किम जोंग-उन साल 2003 में लीबिया के नेता कर्नल मुअम्मर गद्दाफ़ी परमाणु कार्यक्रम छोड़ने 

In [410]:
translation_review.loc[0, [
    "review_status", "translation_quality",
    "names_correct", "numbers_correct",
    "meaning_preserved", "comments"
]] = [
    "Reviewed - needs retranslation",
    "Poor",
    "No",
    "No",
    "No",
    "Severe mistranslation of names, terminology and meaning. "
    "English article and summary are not reliable."
]

translation_review.loc[1, [
    "review_status", "translation_quality",
    "names_correct", "numbers_correct",
    "meaning_preserved", "comments"
]] = [
    "Reviewed - needs retranslation",
    "Poor",
    "No",
    "No",
    "No",
    "Player names, goal statistics and Golden Boot terminology "
    "are corrupted. English article and summary are unreliable."
]

translation_review.to_csv(translation_review_path, index=False)

print(
    translation_review["review_status"].value_counts(
        dropna=False
    )
)

review_status
Needs human review                18
Reviewed - needs retranslation     2
Name: count, dtype: int64


/tmp/ipykernel_1805/3918533688.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Poor' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  translation_review.loc[0, [
/tmp/ipykernel_1805/3918533688.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'No' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  translation_review.loc[0, [
/tmp/ipykernel_1805/3918533688.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'No' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  translation_review.loc[0, [
/tmp/ipykernel_1805/3918533688.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in 

In [412]:
review_columns = [
    "review_status",
    "translation_quality",
    "names_correct",
    "numbers_correct",
    "meaning_preserved",
    "comments"
]

for col in review_columns:
    translation_review[col] = (
        translation_review[col].astype("object")
    )

translation_review.to_csv(
    translation_review_path, index=False
)

for i in [2, 3]:
    row = translation_review.iloc[i]

    print("\n" + "=" * 80)
    print("ARTICLE ID:", row["id"])
    print("TITLE:", row["title"])
    print("\nHINDI SUMMARY:\n", row["summary"])
    print("\nENGLISH SUMMARY:\n", row["english_summary"])
    print("\nHINDI ARTICLE:\n", row["text"])
    print("\nENGLISH TRANSLATION:\n", row["english_text"])


ARTICLE ID: 121114_international_us_plus_obama_ia
TITLE: राष्ट्रीय सुरक्षा को कोई ख़तरा नहीं: ओबामा

HINDI SUMMARY:
 अमरीकी राष्ट्रपति बराक ओबामा ने कहा है कि अमरीकी खुफ़िया एजेंसी सीआईए के पूर्व प्रमुख जनरल डेविड पेट्रियस के प्रेम संबंधों के कारण अमरीका की सुरक्षा को कोई ख़तरा नहीं पैदा हुआ है.

ENGLISH SUMMARY:
 The U.S. President Obama Obama said that the safety of the United States has not been threatened by the love of David Petrius before the U.S. President Davidus's chief general union.

HINDI ARTICLE:
 प्रेम संबंधों के कारण सीआईए प्रमुख को इस्तीफ़ा देना पड़ा था. अपनी चुनावी जीत के बाद बुधवार को व्हाईट हाउस में पहली प्रेस वार्ता को संबोधित करते हुए राष्ट्रपति ओबामा ने कहा, "मेरे पास अभी कोई ऐसा सबूत नहीं है, जो कुछ मैंने देखा है उसके आधार पर मैं कह सकता हूं कि इस मामले में कोई खुफ़िया राज़ नहीं खोला गया है और राष्ट्रीय सुरक्षा को कोई नुक़सान नहीं हुआ है." इस दौरान जब उनसे पूछा गया कि इस मामले में उनको देर से ख़बर दिए जाने के लिए क्या वह एफ़बीआई के ख़िलाफ़ कोई जांच शुरू करेंगे त

In [413]:
reviews_to_update = {
    2: (
        "Severe mistranslation of names and major sections. "
        "CIA, FBI, tax, Iran, Syria and climate-change details "
        "are incomplete or inaccurate. Summary meaning is lost."
    ),
    3: (
        "Names, political roles, dates and historical details "
        "are severely corrupted. Multiple sentences are "
        "unintelligible and summary meaning is lost."
    )
}

for idx, comment in reviews_to_update.items():
    translation_review.loc[idx, "review_status"] = (
        "Reviewed - needs retranslation"
    )
    translation_review.loc[idx, "translation_quality"] = "Poor"
    translation_review.loc[idx, "names_correct"] = "No"
    translation_review.loc[idx, "numbers_correct"] = "No"
    translation_review.loc[idx, "meaning_preserved"] = "No"
    translation_review.loc[idx, "comments"] = comment

translation_review.to_csv(
    translation_review_path, index=False
)

print(translation_review["review_status"].value_counts())

review_status
Needs human review                16
Reviewed - needs retranslation     4
Name: count, dtype: int64


In [414]:
print("CURRENT REVIEW STATUS:")
print(translation_review["review_status"].value_counts(dropna=False))

for idx in [4, 5]:
    row = translation_review.iloc[idx]

    print("\n" + "=" * 70)
    print("ARTICLE", idx + 1)
    print("ID:", row["id"])
    print("TITLE:", row["title"])

    print("\nHINDI SUMMARY:")
    print(row["summary"])

    print("\nENGLISH SUMMARY:")
    print(row["english_summary"])

    print("\nHINDI ARTICLE:")
    print(row["text"])

    print("\nENGLISH ARTICLE:")
    print(row["english_text"])

CURRENT REVIEW STATUS:
review_status
Needs human review                16
Reviewed - needs retranslation     4
Name: count, dtype: int64

ARTICLE 5
ID: science-41399809
TITLE: मोटापा कम करने का ये तरीका हो सकता है ख़तरनाक

HINDI SUMMARY:
मोटापा कम करने या दूसरे शब्दों में कहें तो शरीर से चर्बी घटाने के लिए की जाने वाली सर्जरी को लिपोसक्शन कहा जाता है.

ENGLISH SUMMARY:
In a word, lowering the risk of heart attack, or reducing fat from the body, surgery is called hypnotism.

HINDI ARTICLE:
डॉक्टरों ने लिपोसक्शन को लेकर कुछ गंभीर चिंताएं ज़ाहिर की हैं. डॉक्टरों का कहना है कि लिपोसक्शन की वजह से मोटापे वाले ग्लोब्यूल्स फेफड़ों में दाखिल हो सकते हैं. डॉक्टरों ने ये दावा 45 साल की एक महिला के केस के हवाले से किया है. घुटने और टांगों से मोटापा कम करने के बाद इस महिला को सांस लेने में दिक्कत होने लगी थी. ब्रितानी शहर बर्मिंघम के डॉक्टरों ने बताया कि महिला की हालत इतनी नाज़ुक थी कि वो मौत के मुंह से लौटी हैं. 'मोटापे से निपटने के लिए वज़न कम न करें' हड्डियां हैं मोटापे के लिए ज़िम्मेदार! जानले

In [415]:
reviews_to_update = {
    4: (
        "Severe mistranslation of liposuction and medical terminology. "
        "Doctor's name, patient details and key medical information "
        "are corrupted. English article and summary are unreliable."
    ),
    5: (
        "Researchers' names and medical terminology are corrupted. "
        "Several study details and important sentences are missing "
        "or distorted. Some statistics are retained, but meaning "
        "is not reliably preserved."
    )
}

for idx, comment in reviews_to_update.items():
    translation_review.loc[idx, "review_status"] = (
        "Reviewed - needs retranslation"
    )
    translation_review.loc[idx, "translation_quality"] = "Poor"
    translation_review.loc[idx, "names_correct"] = "No"
    translation_review.loc[idx, "numbers_correct"] = "No"
    translation_review.loc[idx, "meaning_preserved"] = "No"
    translation_review.loc[idx, "comments"] = comment

translation_review.to_csv(translation_review_path, index=False)

print(translation_review["review_status"].value_counts())
print("\nSaved:", translation_review_path)

review_status
Needs human review                14
Reviewed - needs retranslation     6
Name: count, dtype: int64

Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/Human_Translation_Review_20.csv


In [416]:
for idx in [6, 7]:
    row = translation_review.iloc[idx]

    print("\n" + "=" * 70)
    print("ARTICLE", idx + 1)
    print("ID:", row["id"])
    print("TITLE:", row["title"])

    print("\nHINDI SUMMARY:")
    print(row["summary"])

    print("\nENGLISH SUMMARY:")
    print(row["english_summary"])

    print("\nHINDI ARTICLE:")
    print(row["text"])

    print("\nENGLISH ARTICLE:")
    print(row["english_text"])


ARTICLE 7
ID: science-39280869
TITLE: क्या आपको खाने में सुपरफूड की जरूरत है?

HINDI SUMMARY:
भारत में ये सवाल इतनी बार पहले कभी नहीं पूछा गया होगा. क्या हमारा खान-पान सही है?

ENGLISH SUMMARY:
In India this question would never have been asked before. Is our mine-up right?

HINDI ARTICLE:
जाहिर है इसका जवाब ना में ही है. जब आप किसी फूड साइंटिस्ट से इसके बारे में पूछेंगे तो उनका जवाब भी इनकार में होगा. भारत में खाये जाने वाले सबसे अच्छे खाने में अमूमन 65 फीसदी तक स्टार्च, 15 से 20 फीसदी प्रोटीन और 8 से 12 फीसदी के करीब वसा होता है. लेकिन ज्यादातर भारतीय 80 फीसदी स्टार्च, 7 से 8 फीसदी प्रोटीन खाते हैं और खाने में वसा की मात्रा बढ़कर 6 से 15 फीसदी हो जाती चूल्हे ने बदल दी है इनकी ज़िंदगी वैज्ञानिक तथा औद्योगिक अनुसंधान परिषद (सीएसआईआर) के सेंट्रल फूड एंड टेक्नॉलॉजिकल रिसर्च इंस्टीट्यूट (सीएफटीआरआई), मैसूर के डायरेक्टर प्रोफेसर राम राजशेखरन ने बीबीसी हिंदी को बताया, "हम न केवल ज्यादा वसा खा रहे हैं बल्कि सस्ता भी खा रहे हैं. ये पॉम ऑयल हो सकता है या ज्यादा प्रोसेस्ड किया गया तेल. हमें इस

In [417]:
reviews_to_update = {
    6: (
        "Food and nutrition terminology, expert names, quantities "
        "and important details are severely mistranslated. "
        "The English article and summary do not reliably preserve "
        "the Hindi meaning."
    ),
    7: (
        "Museum names, architect names, locations, dates and "
        "historical details are severely corrupted. Several "
        "sentences are unintelligible and meaning is not preserved."
    )
}

for idx, comment in reviews_to_update.items():
    translation_review.loc[idx, "review_status"] = (
        "Reviewed - needs retranslation"
    )
    translation_review.loc[idx, "translation_quality"] = "Poor"
    translation_review.loc[idx, "names_correct"] = "No"
    translation_review.loc[idx, "numbers_correct"] = "No"
    translation_review.loc[idx, "meaning_preserved"] = "No"
    translation_review.loc[idx, "comments"] = comment

translation_review.to_csv(translation_review_path, index=False)

print(translation_review["review_status"].value_counts())
print("\nSaved:", translation_review_path)

review_status
Needs human review                12
Reviewed - needs retranslation     8
Name: count, dtype: int64

Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/Human_Translation_Review_20.csv


In [418]:
for idx in [8, 9]:
    row = translation_review.iloc[idx]

    print("\n" + "=" * 70)
    print("ARTICLE", idx + 1)
    print("ID:", row["id"])
    print("TITLE:", row["title"])

    print("\nHINDI SUMMARY:")
    print(row["summary"])

    print("\nENGLISH SUMMARY:")
    print(row["english_summary"])

    print("\nHINDI ARTICLE:")
    print(row["text"])

    print("\nENGLISH ARTICLE:")
    print(row["english_text"])


ARTICLE 9
ID: india-48662333
TITLE: गुजरात विकास मॉडल में पानी के लिए कुआं खोदता किसान परिवार

HINDI SUMMARY:
"हमें सरकार पर भरोसा नहीं है, इसलिए हम खुद से कुंआ खोद रहे हैं, पीने के लिए पानी नहीं है."

ENGLISH SUMMARY:
"We don't trust the government, so we're digging well of ourselves, not water to drink."

HINDI ARTICLE:
सेमाभाई की बेटी हरमीबेन ये कहना है सेमाभाई का, इनके मक्के की फसल पूरी तरह सूख गई है. सेमाभाई के पास दो बैल हैं लेकिन उनका इस्तेमाल खेतों को जोतने में नहीं हो रहा है, बल्कि कुंआ खोदने में किया जा रहा है. इस काम में सेमाभाई की पत्नी और उनकी बेटी भी मदद कर रहे हैं. पिछले साल उनकी फसल सूख गई थी और इस साल उनका कुंआ भी सूख गया था. उत्तरी गुजरात के पालनपुर जिले के अमीरगढ़ ब्लॉक के अधिकांश किसानों का यही हाल है. इन सबके खेत सूखे पड़े हुए हैं और फसल की सिंचाई के लिए पानी नहीं है. इंसान और जानवरों के लिए भी पानी नहीं सेमाभाई एक छोटे से गांव उपाला खापा में रहते हैं. उनके पास जमीन तो है लेकिन खेती करने के लिए पानी नहीं है. वे दिहाड़ी मजदूर की तरह काम करने को मजबूर हैं. वे बताते 

In [419]:
reviews_to_update = {
    8: (
        "Farmer and family names, village and district names, "
        "water-crisis details and financial figures are severely "
        "corrupted. The English article and summary are unreliable."
    ),
    9: (
        "Christopher Watts and other names are mistranslated. "
        "Researcher names, explanations and important details "
        "are missing or distorted. Meaning is not preserved."
    )
}

for idx, comment in reviews_to_update.items():
    translation_review.loc[idx, "review_status"] = (
        "Reviewed - needs retranslation"
    )
    translation_review.loc[idx, "translation_quality"] = "Poor"
    translation_review.loc[idx, "names_correct"] = "No"
    translation_review.loc[idx, "numbers_correct"] = "No"
    translation_review.loc[idx, "meaning_preserved"] = "No"
    translation_review.loc[idx, "comments"] = comment

translation_review.to_csv(translation_review_path, index=False)

print(translation_review["review_status"].value_counts())
print("\nSaved:", translation_review_path)

review_status
Reviewed - needs retranslation    10
Needs human review                10
Name: count, dtype: int64

Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/Human_Translation_Review_20.csv


In [420]:
for idx in [10, 11]:
    row = translation_review.iloc[idx]

    print("\n" + "=" * 70)
    print("ARTICLE", idx + 1)
    print("ID:", row["id"])
    print("TITLE:", row["title"])

    print("\nHINDI SUMMARY:")
    print(row["summary"])

    print("\nENGLISH SUMMARY:")
    print(row["english_summary"])

    print("\nHINDI ARTICLE:")
    print(row["text"])

    print("\nENGLISH ARTICLE:")
    print(row["english_text"])


ARTICLE 11
ID: 040804_kolkata_reaction
TITLE: राष्ट्रपति के फ़ैसले का स्वागत

HINDI SUMMARY:
कोलकाता के लोगों ने धनंजय चटर्जी की अपील को ठुकराने के राष्ट्रपति के फ़ैसले का आम तौर पर स्वागत किया है, लेकिन कई मानवाधिकार संगठनों ने इसके विरोध में प्रदर्शन किया है.

ENGLISH SUMMARY:
The people of Copenhagen have generally welcomed the president's decision to reject the appeal of the capital capital capitalary, but many human rights organizations have shown up against it.

HINDI ARTICLE:
कोलकाता से बीबीसी संवाददाता सुबीर भौमिक का कहना है कि जिन परिवारों में कम उम्र की लड़कियाँ हैं उन्होंने ख़ास तौर पर इस फ़ैसले का स्वागत किया है. तमाल और शुभ्रा मुखर्जी की दो स्कूल जाने वाली बेटियाँ हैं और उनका कहना है कि राष्ट्रपति का फ़ैसला सराहनीय और स्वागतयोग्य है. उनका मानना है कि धनंजय को काफ़ी पहले फाँसी दे दी जानी चाहिए थी, उनकी पत्नी भी उनसे सहमत हैं और उनका कहना है कि दूसरे अपराधियों को गुनाह करने से रोकने के लिए मृत्युदंड दिया जाना ज़रूरी है. कोलकाता की एक घरेलू महिला जान्हवी बिस्वास का कहना है क

In [421]:
reviews_to_update = {
    10: (
        "Kolkata is mistranslated as Copenhagen. Dhananjoy "
        "Chatterjee and other names are corrupted. Important "
        "details and the summary meaning are lost."
    ),
    11: (
        "Dharmendra Anija and Sapna Anija's names are corrupted. "
        "Land coordinates, parcel numbers, dates and legal "
        "explanations are unreliable. Meaning is not preserved."
    )
}

for idx, comment in reviews_to_update.items():
    translation_review.loc[idx, "review_status"] = (
        "Reviewed - needs retranslation"
    )
    translation_review.loc[idx, "translation_quality"] = "Poor"
    translation_review.loc[idx, "names_correct"] = "No"
    translation_review.loc[idx, "numbers_correct"] = "No"
    translation_review.loc[idx, "meaning_preserved"] = "No"
    translation_review.loc[idx, "comments"] = comment

translation_review.to_csv(translation_review_path, index=False)

print(translation_review["review_status"].value_counts())
print("\nSaved:", translation_review_path)

review_status
Reviewed - needs retranslation    12
Needs human review                 8
Name: count, dtype: int64

Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/Human_Translation_Review_20.csv


In [422]:
for idx in [12, 13]:
    row = translation_review.iloc[idx]

    print("\n" + "=" * 70)
    print("ARTICLE", idx + 1)
    print("ID:", row["id"])
    print("TITLE:", row["title"])

    print("\nHINDI SUMMARY:")
    print(row["summary"])

    print("\nENGLISH SUMMARY:")
    print(row["english_summary"])

    print("\nHINDI ARTICLE:")
    print(row["text"])

    print("\nENGLISH ARTICLE:")
    print(row["english_text"])


ARTICLE 13
ID: 150802_varun_gandhi_death_ia
TITLE: सभ्य देश में जल्लाद, लज्जा की बात: वरुण

HINDI SUMMARY:
भारतीय जनता पार्टी सांसद वरुण गांधी ने मौत की सज़ा पर सवाल उठाए हैं. उन्होने अंग्रेज़ी पत्रिका आउटलुक में लेख लिखकर कहा है कि दुनिया के बदलते माहौल को देखते हुए दुनिया के सबसे बड़े लोकतंत्र यानी भारत को समझना चाहिए कि मौत की सज़ा समस्या का समाधान नहीं है.

ENGLISH SUMMARY:
The Indian public party has questioned the sentence of death, and in an Outlook, he wrote that the world's largest democracies, India, should understand that death is not the end of the problem.

HINDI ARTICLE:
वरूण गांधी ने ये लेख ऐसे समय में लिखा है जब 1993 में हुए मुंबई धमाकों के दोषी याक़ूब मेमन को फ़ासी दी गई है और मौत की सज़ा के मुद्दे पर भारत में ख़ासी बहस हो रही है. इससे पहले भाजपा के ही एक दूसरे सांसद और फ़िल्म अभिनेता शत्रुघ्न सिंहा ने याक़ूब मेमन की दया याचिका पर दस्तख़्त किए थे. इस पर प्रतिक्रिया देते हुए केंद्रीय वित्त मंत्री अरूण जेटली ने कहा था कि ऐसा करके शत्रुध्न सिंहा ने पार्टी को शर्मिंदा किय

In [423]:
reviews_to_update = {
    12: (
        "Major mistranslation of political names, legal terms and "
        "statistics. Key details about the death-penalty debate "
        "are missing or inaccurate. Meaning is not preserved."
    ),
    13: (
        "Player names, tournament names, match scores, rankings "
        "and badminton terminology are severely corrupted. "
        "The English article and summary are unreliable."
    )
}

for idx, comment in reviews_to_update.items():
    translation_review.loc[idx, "review_status"] = (
        "Reviewed - needs retranslation"
    )
    translation_review.loc[idx, "translation_quality"] = "Poor"
    translation_review.loc[idx, "names_correct"] = "No"
    translation_review.loc[idx, "numbers_correct"] = "No"
    translation_review.loc[idx, "meaning_preserved"] = "No"
    translation_review.loc[idx, "comments"] = comment

translation_review.to_csv(
    translation_review_path, index=False
)

print(translation_review["review_status"].value_counts())

review_status
Reviewed - needs retranslation    14
Needs human review                 6
Name: count, dtype: int64


In [424]:
for idx in [14, 15]:
    row = translation_review.iloc[idx]

    print("\n" + "=" * 70)
    print("ARTICLE", idx + 1)
    print("ID:", row["id"])
    print("TITLE:", row["title"])

    print("\nHINDI SUMMARY:")
    print(row["summary"])

    print("\nENGLISH SUMMARY:")
    print(row["english_summary"])

    print("\nHINDI ARTICLE:")
    print(row["text"])

    print("\nENGLISH ARTICLE:")
    print(row["english_text"])


ARTICLE 15
ID: 131126_assam_protes_vs
TITLE: असमः गैंगरेप के ख़िलाफ़ विरोध प्रदर्शन

HINDI SUMMARY:
असम के लखीमपुर ज़िल में चार व्यक्तियों ने कथित रूप से एक ऑटो में महिला का बलात्कार किया, उनकी आँखें निकाल लीं, पीटा और चलती ऑटो से बाहर फेंक दिया. इस घटना के ख़िलाफ़ लोगों ने विरोध प्रदर्शन किया.

ENGLISH SUMMARY:
At Aim's Lemimpurhli, four people have beaten a woman in an auto, they have beaten her eyes, have beaten her eyes, and have thrown her out of the running. Against this incident, people have expressed opposition.

HINDI ARTICLE:
समाचार एजेंसी पीटीआई के मुताबिक़ सामूहिक बलात्कार की इस घटना के बाद क्षेत्रीय लोगों में काफ़ी गुस्सा है. लोग दोषियों को कड़ी सज़ा देने का माँग कर रहे हैं. यह घटना पिछले शुक्रवार को उस समय हुई, जब महिला बोगीनाड़ी में पढ़ने वाली अपनी छह साल की बच्ची को स्कूल से आने के लिए एक शेयर्ड ऑटो से वहाँ जा रही थी. पुलिस सूत्रों के मुताबिक़ यह जगह लखीमपुर ज़िला मुख्यालय से 14 किलोमीटर दूर है. लोगों में गुस्सा उन्होंने बताया कि चार व्यक्तियों ने महिला के साथ कथित बला

In [425]:
reviews_to_update = {
    14: (
        "Place names, distances, hospital name and key events "
        "are mistranslated or missing. The English summary and "
        "article do not reliably preserve the Hindi meaning."
    ),
    15: (
        "POSCO, SEZ terminology, land figures, production "
        "figures, officials and financial details are severely "
        "corrupted. Meaning is not reliably preserved."
    )
}

for idx, comment in reviews_to_update.items():
    translation_review.loc[idx, "review_status"] = (
        "Reviewed - needs retranslation"
    )
    translation_review.loc[idx, "translation_quality"] = "Poor"
    translation_review.loc[idx, "names_correct"] = "No"
    translation_review.loc[idx, "numbers_correct"] = "No"
    translation_review.loc[idx, "meaning_preserved"] = "No"
    translation_review.loc[idx, "comments"] = comment

translation_review.to_csv(
    translation_review_path, index=False
)

print(translation_review["review_status"].value_counts())
print("\nSaved:", translation_review_path)

review_status
Reviewed - needs retranslation    16
Needs human review                 4
Name: count, dtype: int64

Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/Human_Translation_Review_20.csv


In [426]:
for idx in [16, 17]:
    row = translation_review.iloc[idx]

    print("\n" + "=" * 70)
    print("ARTICLE", idx + 1)
    print("ID:", row["id"])
    print("TITLE:", row["title"])

    print("\nHINDI SUMMARY:")
    print(row["summary"])

    print("\nENGLISH SUMMARY:")
    print(row["english_summary"])

    print("\nHINDI ARTICLE:")
    print(row["text"])

    print("\nENGLISH ARTICLE:")
    print(row["english_text"])


ARTICLE 17
ID: 121115_2g_auction_ac
TITLE: नीलामी की नाकामी से बढ़ेंगे मोबाइल कॉल के दाम?

HINDI SUMMARY:
2 जी स्पेक्ट्रम की नीलामी पर बाज़ार की बेरुखी ने सबको हैरान किया है, खास तौर पर भारत सरकार को.

ENGLISH SUMMARY:
2 The market apathy at the square of GV has surprised all, especially India.

HINDI ARTICLE:
नीलामी में टेलीकॉम कंपनियों ने कोई खास दिलचस्पी नहीं दिखाई है. 2जी स्पेक्ट्रम के आवंटन में एक लाख 76 हजार करोड़ का नुकसान करा चुकी भारत सरकार को उम्मीद थी कि नीलामी से सरकारी खजाने में कम से कम 40 हजार करोड़ रुपए आ पाएगा. लेकिन नीलामी में दूरसंचार कंपनियों ने कोई खास दिलचस्पी नहीं दिखाई है. पहले दिन की नीलामी के बाद सरकार की झोली में सिर्फ 9,200 करोड़ रुपए ही आए. मुंबई से बीबीसी संवाददाता समीर हाशमी बता रहे हैं कि नीलामी के फ्लॉप होने के क्या कारण थे और इसका क्या असर हो सकता है. 2जी की नीलामी को कैसे देखते हैं? बुधवार का दिन नीलामी का दूसरा दिन था. पहले दिन भारत सरकार को बहुत अच्छी प्रतिक्रिया नहीं मिली. केवल पांच कंपनियों ने ही भाग लिया. सरकार केवल 9,200 करोड़ रुपए ही इकट्ठा कर

In [427]:
reviews_to_update = {
    16: (
        "2G spectrum, telecom auction terminology, company "
        "participation and financial figures are severely "
        "mistranslated. The English summary and article are "
        "unreliable."
    ),
    17: (
        "Line of Control, ceasefire terminology, Tangdhar "
        "sector, military details and Colonel A.K. Mathur's "
        "name are corrupted or missing. Meaning is not "
        "reliably preserved."
    )
}

for idx, comment in reviews_to_update.items():
    translation_review.loc[idx, "review_status"] = (
        "Reviewed - needs retranslation"
    )
    translation_review.loc[idx, "translation_quality"] = "Poor"
    translation_review.loc[idx, "names_correct"] = "No"
    translation_review.loc[idx, "numbers_correct"] = "No"
    translation_review.loc[idx, "meaning_preserved"] = "No"
    translation_review.loc[idx, "comments"] = comment

translation_review.to_csv(
    translation_review_path, index=False
)

print(translation_review["review_status"].value_counts())
print("\nSaved:", translation_review_path)

review_status
Reviewed - needs retranslation    18
Needs human review                 2
Name: count, dtype: int64

Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/Human_Translation_Review_20.csv


In [428]:
for idx in [18, 19]:
    row = translation_review.iloc[idx]

    print("\n" + "=" * 70)
    print("ARTICLE", idx + 1)
    print("ID:", row["id"])
    print("TITLE:", row["title"])

    print("\nHINDI SUMMARY:")
    print(row["summary"])

    print("\nENGLISH SUMMARY:")
    print(row["english_summary"])

    print("\nHINDI ARTICLE:")
    print(row["text"])

    print("\nENGLISH ARTICLE:")
    print(row["english_text"])


ARTICLE 19
ID: 040423_un_poverty_report
TITLE: ग़रीबी दूर करने के वायदे अधूरेः रिपोर्ट

HINDI SUMMARY:
दुनिया के अधिकतर विकासशील देश, ग़रीबी को दूर करने के लिए संयुक्त राष्ट्र के तय लक्ष्यों को समय पर पूरा नहीं कर पाएँगे.

ENGLISH SUMMARY:
Most developing countries of the world can't meet fixed goals of the United Nations in time to eliminate poverty.

HINDI ARTICLE:
इन लक्ष्यों को 2015 तक पूरा किया जाना था. मगर अब विश्व बैंक और अंतरराष्ट्रीय मुद्रा कोष ने एक साझा रिपोर्ट जारी की है जिसमें विकासशील देशों के प्रदर्शन पर टिप्पणी की गई है. इन लक्ष्यों को 'सहस्राब्दी विकास लक्ष्य' का नाम दिया गया था. इनमें कुछ प्रमुख लक्ष्य थे - अत्यधिक ग़रीब लोगों की संख्या आधी करना, सभी बच्चों को प्राथमिक शिक्षा और प्रसव के समय माँ और बच्चों की होनेवाली मौतों की संख्या कम करना. ज़रूरत रिपोर्ट में कहा गया है कि इन लक्ष्यों की पूर्ति के लिए ये ज़रूरी है कि अमीर देश और आर्थिक सहयोग के लिए दो साल पहले किए गए अपने वायदे को पूरा करें. विश्व बैंक के प्रमुख जेम्स टी वोल्फ़ेंसन ने कहा है,"हम जितनी बात बनाना चाहे

In [429]:
reviews_to_update = {
    18: (
        "The summary is mostly accurate, but the article "
        "severely mistranslates the World Bank, IMF, Millennium "
        "Development Goals, James T. Wolfensohn and key details. "
        "Meaning is not reliably preserved."
    ),
    19: (
        "Robot name Pepero, NEC Corporation, translation "
        "direction, airport plan and vocabulary figures are "
        "mistranslated or missing. The English article is "
        "unreliable."
    )
}

for idx, comment in reviews_to_update.items():
    translation_review.loc[idx, "review_status"] = (
        "Reviewed - needs retranslation"
    )
    translation_review.loc[idx, "translation_quality"] = "Poor"
    translation_review.loc[idx, "names_correct"] = "No"
    translation_review.loc[idx, "numbers_correct"] = "No"
    translation_review.loc[idx, "meaning_preserved"] = "No"
    translation_review.loc[idx, "comments"] = comment

translation_review.to_csv(
    translation_review_path, index=False
)

print(translation_review["review_status"].value_counts())
print("\nSaved:", translation_review_path)

review_status
Reviewed - needs retranslation    20
Name: count, dtype: int64

Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/Human_Translation_Review_20.csv


In [430]:
check = pd.read_csv(translation_review_path)

print("Total articles:", len(check))
print("Unique article IDs:", check["id"].nunique())

print("\nReview status:")
print(check["review_status"].value_counts())

print("\nMissing review decisions:")
print(check[
    ["review_status", "translation_quality",
     "names_correct", "numbers_correct",
     "meaning_preserved", "comments"]
].isna().sum())

Total articles: 20
Unique article IDs: 20

Review status:
review_status
Reviewed - needs retranslation    20
Name: count, dtype: int64

Missing review decisions:
review_status          0
translation_quality    0
names_correct          0
numbers_correct        0
meaning_preserved      0
comments               0
dtype: int64


In [432]:
print("Total records:", len(hinglish_reviews))
print("Unique texts:", hinglish_reviews["text"].nunique())

print("\nSource distribution:")
print(hinglish_reviews["source_dataset"].value_counts())

print("\nHinglish verification:")
print(hinglish_reviews["verified_hinglish"].value_counts())

print("\nMeaning clarity:")
print(hinglish_reviews["meaning_clear"].value_counts())

print("\nNatural English usage:")
print(
    hinglish_reviews[
        "english_words_used_naturally"
    ].value_counts()
)

Total records: 100
Unique texts: 100

Source distribution:
source_dataset
MUTANT     56
GupShup    44
Name: count, dtype: int64

Hinglish verification:
verified_hinglish
Yes    78
No     22
Name: count, dtype: int64

Meaning clarity:
meaning_clear
Yes    100
Name: count, dtype: int64

Natural English usage:
english_words_used_naturally
Yes    78
No     22
Name: count, dtype: int64


In [433]:
old_summary = pd.read_csv(hinglish_summary_path)

print("Existing summary:")
display(old_summary)

print("\nColumns:", old_summary.columns.tolist())

Existing summary:


,source_dataset,No,Yes
0,GupShup,17,27
1,MUTANT,5,46



Columns: ['source_dataset', 'No', 'Yes']


In [434]:
# Create updated summary from all 100 reviewed examples
new_summary = pd.crosstab(
    hinglish_reviews["source_dataset"],
    hinglish_reviews["verified_hinglish"]
)

# Keep both Yes and No columns, even if one is absent
new_summary = new_summary.reindex(
    columns=["No", "Yes"], fill_value=0
).reset_index()

# Add total reviewed per source
new_summary["Total"] = new_summary["No"] + new_summary["Yes"]

# Save updated summary
new_summary.to_csv(hinglish_summary_path, index=False)

# Verify saved file
verified_summary = pd.read_csv(hinglish_summary_path)

print("UPDATED HINGLISH REVIEW SUMMARY")
display(verified_summary)

print("\nTotal reviewed:", verified_summary["Total"].sum())
print("Saved successfully:", os.path.exists(hinglish_summary_path))
print("Path:", hinglish_summary_path)

UPDATED HINGLISH REVIEW SUMMARY


,source_dataset,No,Yes,Total
0,GupShup,17,27,44
1,MUTANT,5,51,56



Total reviewed: 100
Saved successfully: True
Path: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/06_real_hinglish/FactIndic_Hinglish_Final_Review_Summary.csv


In [435]:
comparison_path = (
    base_path +
    "/05_quality_checks/CMI_MUTANT_PHINC_Comparison.csv"
)

factindic_cmi_path = (
    base_path +
    "/05_quality_checks/FactIndic_CMI_Summary_23.csv"
)

comparison = pd.read_csv(comparison_path)
factindic_cmi = pd.read_csv(factindic_cmi_path)

print("EXISTING COMPARISON")
display(comparison)

print("\nFACTINDIC CMI SUMMARY")
display(factindic_cmi)

print("\nComparison columns:", comparison.columns.tolist())
print("FactIndic columns:", factindic_cmi.columns.tolist())

EXISTING COMPARISON


,Dataset,Records,Mean_CMI,Median_CMI
0,MUTANT,100,6.26,4.77
1,PHINC,13738,17.61,16.67



FACTINDIC CMI SUMMARY


,Version,Articles,English_words,Total_words,Actual_CMI
0,Original Hindi,23,0,5954,0.00
1,10% Target,23,590,5893,10.01
2,30% Target,23,1689,5722,29.52
3,50% Target,23,2715,5528,49.11



Comparison columns: ['Dataset', 'Records', 'Mean_CMI', 'Median_CMI']
FactIndic columns: ['Version', 'Articles', 'English_words', 'Total_words', 'Actual_CMI']


In [436]:
# Convert FactIndic CMI summary into comparison format
factindic_comparison = pd.DataFrame({
    "Dataset": [
        "FactIndic - Original Hindi",
        "FactIndic - 10% Target",
        "FactIndic - 30% Target",
        "FactIndic - 50% Target"
    ],
    "Records": factindic_cmi["Articles"],
    "Mean_CMI": factindic_cmi["Actual_CMI"],
    "Median_CMI": factindic_cmi["Actual_CMI"]
})

# Combine with existing MUTANT and PHINC results
updated_comparison = pd.concat(
    [comparison, factindic_comparison],
    ignore_index=True
)

# Save
updated_comparison.to_csv(comparison_path, index=False)

# Read back and verify
final_comparison = pd.read_csv(comparison_path)

print("UPDATED CMI COMPARISON")
display(final_comparison)

print("\nTotal rows:", len(final_comparison))
print("Saved successfully:", os.path.exists(comparison_path))
print("Path:", comparison_path)

UPDATED CMI COMPARISON


,Dataset,Records,Mean_CMI,Median_CMI
0,MUTANT,100,6.26,4.77
1,PHINC,13738,17.61,16.67
2,FactIndic - Original Hindi,23,0.00,0.00
3,FactIndic - 10% Target,23,10.01,10.01
4,FactIndic - 30% Target,23,29.52,29.52
5,FactIndic - 50% Target,23,49.11,49.11



Total rows: 6
Saved successfully: True
Path: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/CMI_MUTANT_PHINC_Comparison.csv


In [437]:
mixed_path = (
    base_path +
    "/04_mixed_data/FactIndic_Mixed_23_Validated.csv"
)

mixed_check = pd.read_csv(mixed_path)

print("Rows:", len(mixed_check))
print("Columns:", mixed_check.columns.tolist())
display(mixed_check.head(2))

Rows: 23
Columns: ['sample_id', 'hindi_article', 'mixed_10_article', 'mixed_30_article', 'mixed_50_article', 'title', 'url']


,sample_id,hindi_article,mixed_10_article,mixed_30_article,mixed_50_article,title,url
0,080711_lanka_attack,एक सैनिक प्रवक्ता ब्रिगेडियर उदय नान्यकारा ने ...,एक सैनिक प्रवक्ता ब्रिगेडियर उदय नान्यकारा ने ...,एक सैनिक प्रवक्ता ब्रिगेडियर उदय नान्यकारा ने ...,एक सैनिक प्रवक्ता ब्रिगेडियर उदय नान्यकारा ने ...,बस पर गोलीबारी में चार की मौत,https://www.bbc.com/hindi/regionalnews/story/2...
1,051105_sahubuddin_arrest,शहाबुद्दीन बिहार के सीवान क्षेत्र से सांसद हैं...,शहाबुद्दीन बिहार के सीवान क्षेत्र से सांसद हैं...,"Shahabuddin, a Member of Parliament from the S...","Shahabuddin, a Member of Parliament from the S...",शहाबुद्दीन दिल्ली में गिरफ़्तार,https://www.bbc.com/hindi/regionalnews/story/2...


In [438]:
import re
import pandas as pd

def calculate_cmi(text):
    words = str(text).split()
    if not words:
        return 0.0

    english_words = sum(
        1 for word in words
        if re.search(r"[A-Za-z]", word)
    )

    return english_words / len(words) * 100


article_cmi = pd.DataFrame({
    "sample_id": mixed_check["sample_id"],
    "Original_Hindi_CMI": mixed_check["hindi_article"].apply(calculate_cmi),
    "FactIndic_10_CMI": mixed_check["mixed_10_article"].apply(calculate_cmi),
    "FactIndic_30_CMI": mixed_check["mixed_30_article"].apply(calculate_cmi),
    "FactIndic_50_CMI": mixed_check["mixed_50_article"].apply(calculate_cmi)
})

print("PER-ARTICLE CMI")
display(article_cmi.head(10))

print("\nMEAN AND MEDIAN CMI")
cmi_stats = pd.DataFrame({
    "Dataset": [
        "FactIndic - Original Hindi",
        "FactIndic - 10% Target",
        "FactIndic - 30% Target",
        "FactIndic - 50% Target"
    ],
    "Mean_CMI": [
        article_cmi["Original_Hindi_CMI"].mean(),
        article_cmi["FactIndic_10_CMI"].mean(),
        article_cmi["FactIndic_30_CMI"].mean(),
        article_cmi["FactIndic_50_CMI"].mean()
    ],
    "Median_CMI": [
        article_cmi["Original_Hindi_CMI"].median(),
        article_cmi["FactIndic_10_CMI"].median(),
        article_cmi["FactIndic_30_CMI"].median(),
        article_cmi["FactIndic_50_CMI"].median()
    ]
})

display(cmi_stats.round(2))

PER-ARTICLE CMI


,sample_id,Original_Hindi_CMI,FactIndic_10_CMI,FactIndic_30_CMI,FactIndic_50_CMI
0,080711_lanka_attack,0.0,24.568966,46.929825,64.840183
1,051105_sahubuddin_arrest,0.0,8.064516,47.916667,79.908676
2,150901_nepal_protest_dead_pp,0.0,11.320755,46.370968,46.370968
3,041003_pwg_security,0.0,29.953917,44.239631,77.073171
4,080416_up_results,0.0,8.791209,24.334601,57.915058
5,060117_indopak_cbm,0.0,16.053512,36.678201,78.676471
6,140206_egypt_al-sisi_presidential_election_sr,0.0,10.309278,20.954907,31.476323
7,140410_action_against_manvendra_singh_rns,0.0,9.589041,16.206897,32.167832
8,051121_aids_india,0.0,13.783784,41.944444,49.299720
9,070902_khanduri_win,0.0,11.320755,32.236842,39.735099



MEAN AND MEDIAN CMI


,Dataset,Mean_CMI,Median_CMI
0,FactIndic - Original Hindi,0.00,0.00
1,FactIndic - 10% Target,9.77,8.79
2,FactIndic - 30% Target,29.40,27.27
3,FactIndic - 50% Target,49.97,47.29


In [439]:
# Keep only MUTANT and PHINC from the existing comparison
base_comparison = comparison[
    comparison["Dataset"].isin(["MUTANT", "PHINC"])
].copy()

# Create correct FactIndic comparison rows
factindic_corrected = cmi_stats.copy()
factindic_corrected["Records"] = 23

factindic_corrected = factindic_corrected[
    ["Dataset", "Records", "Mean_CMI", "Median_CMI"]
]

# Combine and save
final_comparison = pd.concat(
    [base_comparison, factindic_corrected],
    ignore_index=True
)

final_comparison.to_csv(comparison_path, index=False)

# Verify saved file
verified = pd.read_csv(comparison_path)

print("FINAL CMI COMPARISON")
display(verified.round(2))

print("\nTotal rows:", len(verified))
print("Saved:", os.path.exists(comparison_path))

FINAL CMI COMPARISON


,Dataset,Records,Mean_CMI,Median_CMI
0,MUTANT,100,6.26,4.77
1,PHINC,13738,17.61,16.67
2,FactIndic - Original Hindi,23,0.00,0.00
3,FactIndic - 10% Target,23,9.77,8.79
4,FactIndic - 30% Target,23,29.40,27.27
5,FactIndic - 50% Target,23,49.97,47.29



Total rows: 6
Saved: True


In [440]:
from pathlib import Path

report_path = Path(
    base_path
) / "08_reports" / "FactIndic_PersonB_Final_Report.txt"

report_path.parent.mkdir(parents=True, exist_ok=True)

report = """
FACTINDIC NLP PROJECT
PERSON B - FINAL WORK REPORT

Project:
Factuality-aware summarization for Hindi-English code-mixed text

Course:
CSET 346 - Natural Language Processing
Team 8 - Bennett University


1. DATA COLLECTION AND ENGLISH TRANSLATION

- Sampled 250 Hindi articles from the XL-Sum Hindi dataset.
- Completed English translation for all 250 articles using IndicTrans2.
- Completed 5,229 sentence-level translation tasks.
- Saved the reviewed English translation dataset:
  09_sentence_level_translation/FactIndic_English_250_Reviewed.csv
- Translation review records:
  225 manually corrected and 25 human reviewed without correction.
- A separate 20-article quality audit was completed.
- All 20 articles in this audit were marked
  "Reviewed - needs retranslation".

Important limitation:
The 20-article audit indicates serious translation-quality problems.
The 250 translations should not be treated as fully reliable or
gold-standard translations. Further correction is required before
using them for factuality experiments.


2. SENTENCE ALIGNMENT

- Created manual Hindi-English sentence alignment records.
- Total alignment records: 1,079.
- Unique articles represented: 71.
- Duplicate sentence pairs: 0.
- Cleaned missing review-status metadata without making new
  alignment approval decisions.

Strictly approved sentence pairs:
- 408 strictly approved pairs.
- 40 articles have at least one strictly approved pair.
- 23 articles have at least 10 strictly approved pairs.

Saved files include:
- FactIndic_Manual_Sentence_Alignment.csv
- FactIndic_Strict_Approved_Sentence_Pairs.csv
- FactIndic_Eligible_Articles_10plus.csv
- FactIndic_Eligible_Approved_Sentence_Pairs.csv


3. CONTROLLED CODE-MIXING DATASET

- Selected 23 articles with at least 10 strictly approved
  sentence pairs each.
- Used 275 strictly approved Hindi-English sentence pairs.
- Preserved original sentence order.
- Retained 33 sentences in Hindi because they had no approved
  English equivalent.
- Generated 10%, 30%, and 50% target English-mixing versions.

Final corpus-level CMI:
- Original Hindi: 0.00%
- 10% target: 10.01%
- 30% target: 29.52%
- 50% target: 49.11%

Validation:
- 23 unique articles.
- 308 sentences.
- Zero invalid replacements.
- Zero missing sentences.
- Zero missing values in the validated output.

Saved dataset:
04_mixed_data/FactIndic_Mixed_23_Validated.csv

Important limitation:
The method uses sentence-level language replacement.
It does not generate natural word-level mixing within sentences.
The 23-article dataset is a validated subset, not a completed
250-article mixed dataset.


4. HUMAN REVIEW OF MIXED ARTICLES

- Reviewed a sample of 5 mixed articles.
- Recorded language, source-text, formatting and factual-review
  observations.
- Saved the review records in:
  05_quality_checks/FactIndic_Mixed_Human_Review.csv

The review covered only 5 sampled articles, not all 23.
Some source-language, formatting and factual checks remain necessary.


5. REAL HINGLISH DATA

- Reviewed 100 unique real Hinglish examples.
- Source distribution:
  GupShup: 44
  MUTANT: 56

Human verification:
- Verified Hinglish: 78
- Not verified as Hinglish: 22
- Meaning clear: 100
- Natural English usage: 78 Yes, 22 No

Saved files:
- 06_real_hinglish/FactIndic_Hinglish_95_Human_Review.csv
- 06_real_hinglish/FactIndic_Hinglish_Final_Review_Summary.csv

Note:
The detailed review file contains 100 records despite "95" in
its filename.


6. CLAIM LABELING

- Prepared 50 claim-label records.
- 50 unique claim IDs and article IDs.
- Label distribution:
  SUPPORTED: 42
  NOT_ENOUGH_INFO: 8
  CONTRADICTED: 0
- Structural validation found no missing values or duplicate IDs.
- The initial NEI subset was individually rechecked; not all
  50 claims received an independent second audit.

Saved file:
07_claim_labels/FactIndic_50_Claim_Labeling_Sheet.csv


7. CMI COMPARISON

Per-article FactIndic CMI across 23 articles:

Version                  Mean CMI       Median CMI
Original Hindi             0.00%           0.00%
10% target                 9.77%           8.79%
30% target                29.40%          27.27%
50% target                49.97%          47.29%

Existing comparison datasets:
MUTANT:
100 records; mean CMI 6.26%; median CMI 4.77%.

PHINC:
13,738 records; mean CMI 17.61%; median CMI 16.67%.

Saved file:
05_quality_checks/CMI_MUTANT_PHINC_Comparison.csv

CMI calculation note:
English words were identified using Latin-script character
detection. This is a script-based estimate, not a fully
language-aware linguistic measure. Dataset sizes and sampling
methods differ, so comparisons should be interpreted carefully.


8. METHODOLOGY AND QUALITY FILES

Mixing methodology:
08_reports/FactIndic_Mixing_Methodology.txt

Translation review:
05_quality_checks/Human_Translation_Review_20.csv

FactIndic corpus-level CMI:
05_quality_checks/FactIndic_CMI_Summary_23.csv

Per-article CMI statistics were calculated from the validated
23-article mixed dataset.


9. IMPORTANT PENDING WORK AND LIMITATIONS

1. Retranslate or substantially correct the English translations.
   The 20-article audit found all 20 sampled translations unreliable.

2. Recheck English translations before using them to generate
   summaries or evaluate factual consistency.

3. Review the remaining 18 mixed articles beyond the 5-article
   human-review sample.

4. Consider word-level or intra-sentence mixing if natural
   Hinglish generation is required.

5. Independently audit all claim labels if the project requires
   a fully double-checked claim dataset.

6. Clearly report the 23-article controlled-mixing subset rather
   than describing it as a 250-article completed dataset.


10. OVERALL STATUS

Person B completed substantial data preparation, translation
processing, alignment, controlled mixing for a validated
23-article subset, real Hinglish review, claim labeling,
quality checks and CMI comparison.

However, translation quality is a major unresolved limitation.
The current outputs should be described transparently, and
unreliable translations should not be used as trusted reference
data without further correction.
"""

report_path.write_text(report.strip(), encoding="utf-8")

print("Report created:", report_path)
print("File exists:", report_path.exists())
print("Report size:", report_path.stat().st_size, "bytes")

Report created: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/08_reports/FactIndic_PersonB_Final_Report.txt
File exists: True
Report size: 6153 bytes


In [441]:
# Save detailed per-article CMI statistics
article_cmi_path = (
    base_path +
    "/05_quality_checks/FactIndic_Per_Article_CMI_23.csv"
)

article_cmi.to_csv(article_cmi_path, index=False)

# Verify
saved_cmi = pd.read_csv(article_cmi_path)

print("Rows:", len(saved_cmi))
print("Unique articles:", saved_cmi["sample_id"].nunique())
print("Missing values:", saved_cmi.isna().sum().sum())
print("Saved successfully:", os.path.exists(article_cmi_path))
print("Path:", article_cmi_path)

Rows: 23
Unique articles: 23
Missing values: 0
Saved successfully: True
Path: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/FactIndic_Per_Article_CMI_23.csv


In [442]:
from pathlib import Path

report_path = (
    Path(base_path) /
    "08_reports" /
    "FactIndic_PersonB_Final_Report.txt"
)

print("Report exists:", report_path.exists())

if report_path.exists():
    print("File size:", report_path.stat().st_size, "bytes")
    print("\nReport preview:\n")
    print(report_path.read_text(encoding="utf-8")[:2500])
else:
    print("Final report has not been created yet.")

Report exists: True
File size: 6153 bytes

Report preview:

FACTINDIC NLP PROJECT
PERSON B - FINAL WORK REPORT

Project:
Factuality-aware summarization for Hindi-English code-mixed text

Course:
CSET 346 - Natural Language Processing
Team 8 - Bennett University


1. DATA COLLECTION AND ENGLISH TRANSLATION

- Sampled 250 Hindi articles from the XL-Sum Hindi dataset.
- Completed English translation for all 250 articles using IndicTrans2.
- Completed 5,229 sentence-level translation tasks.
- Saved the reviewed English translation dataset:
  09_sentence_level_translation/FactIndic_English_250_Reviewed.csv
- Translation review records:
  225 manually corrected and 25 human reviewed without correction.
- A separate 20-article quality audit was completed.
- All 20 articles in this audit were marked
  "Reviewed - needs retranslation".

Important limitation:
The 20-article audit indicates serious translation-quality problems.
The 250 translations should not be treated as fully reliable or
gold-

In [443]:
from pathlib import Path

report_path = (
    Path(base_path) /
    "08_reports" /
    "FactIndic_PersonB_Final_Report.txt"
)

with report_path.open("a", encoding="utf-8") as f:
    f.write("""

11. ADDITIONAL FINAL VALIDATION

Per-article CMI:
- Saved 23 individual article CMI records.
- 23 unique articles.
- Zero missing values.
- File:
  05_quality_checks/FactIndic_Per_Article_CMI_23.csv

Per-article mean and median CMI:
- Original Hindi: 0.00% / 0.00%
- 10% target: 9.77% / 8.79%
- 30% target: 29.40% / 27.27%
- 50% target: 49.97% / 47.29%

Real Hinglish review:
- 100 unique examples reviewed.
- MUTANT: 56 examples.
- GupShup: 44 examples.
- Verified Hinglish: 78.
- Not verified as Hinglish: 22.
- Meaning clear: 100.
- Natural English usage: 78 Yes, 22 No.

Updated summary:
06_real_hinglish/FactIndic_Hinglish_Final_Review_Summary.csv

Final CMI comparison:
05_quality_checks/CMI_MUTANT_PHINC_Comparison.csv
""")

print("Report updated:", report_path.exists())
print("Final size:", report_path.stat().st_size, "bytes")

Report updated: True
Final size: 6881 bytes


In [445]:
print("Total claims:", len(claims))

print("\nActual columns:")
print(claims.columns.tolist())

print("\nFirst 5 claims:")
display(claims.head())

print("\nMissing values:")
print(claims.isna().sum())

Total claims: 50

Actual columns:
['Claim_ID', 'Article_ID', 'Claim', 'Source_Article', 'Verdict', 'Evidence_from_Article', 'Reviewer_Notes']

First 5 claims:


,Claim_ID,Article_ID,Claim,Source_Article,Verdict,Evidence_from_Article,Reviewer_Notes
0,FC001,040102_france_sikhs,फ़्रांस में रह रहे पाँच हज़ार सिख फ़्रांस के उ...,वे भारत के प्रधानमंत्री से आग्रह कर रहे हैं कि...,NOT_ENOUGH_INFO,यह क़ानून मुसलमानों के ही नहीं सिखों के भी ख़ि...,"The opposition is supported, but the stated co..."
1,FC002,social-43068605,आंखों के रास्ते दिलों में कुटिया बनाने वाली प्...,प्रिया प्रकाश प्रिया प्रकाश की कजरारे नैनों वा...,SUPPORTED,पाकिस्तान में सोशल मीडिया पर प्रिया के इशारों ...,The article describes Priya-related videos and...
2,FC003,050420_pope_muslims,कई मुसलमान नेताओं ने नए पोप से अनुरोध किया है ...,शंका जताई गई है कि नए पोप शायद ऐसा करने में उत...,NOT_ENOUGH_INFO,कुछ कैथोलिक ... दूसरे संप्रदायों के साथ बातचीत...,"The excerpt mentions interfaith dialogue, but ..."
3,FC004,150811_sani_mirza_autobiography_farha_adp_ssm,कई दिनों से चर्चा थी कि निर्माता निर्देशक फ़रा...,"यदि ऐसा होता तो धावक मिल्खा सिंह, मुक्क़ेबाज़ ...",SUPPORTED,फ़राह ने ऐसी किसी भी फ़िल्म को बनाए जाने की बा...,The article confirms that the rumour about a S...
4,FC005,060715_bushputin_g8,अमरीकी राष्ट्रपति जॉर्ज बुश ने जी8 बैठक से पहल...,दोनों देशों के राष्ट्रपतियों ने सेंट पीटर्सबर्...,SUPPORTED,दोनों देशों के राष्ट्रपतियों ने सेंट पीटर्सबर्...,The article confirms the meeting and discussio...



Missing values:
Claim_ID                 0
Article_ID               0
Claim                    0
Source_Article           0
Verdict                  0
Evidence_from_Article    0
Reviewer_Notes           0
dtype: int64


In [446]:
for start in range(0, 10):
    row = claims.iloc[start]

    print("\n" + "=" * 70)
    print("CLAIM ID:", row["Claim_ID"])
    print("ARTICLE ID:", row["Article_ID"])
    print("CLAIM:", row["Claim"])
    print("\nSOURCE ARTICLE:")
    print(row["Source_Article"])
    print("\nCURRENT VERDICT:", row["Verdict"])
    print("EVIDENCE:", row["Evidence_from_Article"])
    print("REVIEWER NOTES:", row["Reviewer_Notes"])


CLAIM ID: FC001
ARTICLE ID: 040102_france_sikhs
CLAIM: फ़्रांस में रह रहे पाँच हज़ार सिख फ़्रांस के उस प्रतिबंध से नाराज़ है जो धार्मिक प्रतीकों के इस्तेमाल पर लगाया जा रहा है.

SOURCE ARTICLE:
वे भारत के प्रधानमंत्री से आग्रह कर रहे हैं कि वह उनकी पारंपरिक पगड़ियों को इस प्रतिबंध से अलग रखने की हिमायत करें. सिख समुदाय के एक प्रवक्ता चैन सिंह ने कहा, "यह क़ानून मुसलमानों के ही नहीं सिखों के भी ख़िलाफ़ है". हम पगड़ी पहने बिना नहीं रह सकते. यह हमारे धर्म का हिस्सा हैं. अगर हमारे पगड़ी बांधने पर पाबंदी लगाई गई तो हम यहाँ रह भी नहीं पाएँगे. चैन सिंह "हम पगड़ी पहने बिना नहीं रह सकते. यह हमारे धर्म का हिस्सा हैं. अगर हमारे पगड़ी बांधने पर पाबंदी लगाई गई तो हम यहाँ रह भी नहीं पाएँगे". फ़्रांस के इस क़दम पर मिस्र में भी टकराव की स्थिति पैदा हो गई है जहाँ एक प्रमुख मुस्लिम नेता ने फ़्रांस के फ़ैसले की वकालत करके एक नई बहस छेड़ दी है. अल-अज़हर मस्जिद के प्रमुख शेख़ मोहम्मद सैयद तंतावी ने एक बयान में कह दिया कि वह इस बात से सहमत हैं कि फ़्रांस को यह क़ानून लागू करने का पूरा अधिकार है. उनका कहना 

In [447]:
# FC005: Saturday is not confirmed in the supplied excerpt
claims.loc[claims["Claim_ID"] == "FC005", "Verdict"] = "NOT_ENOUGH_INFO"
claims.loc[claims["Claim_ID"] == "FC005", "Evidence_from_Article"] = (
    "The source confirms a meeting before the G8 meeting, "
    "but the supplied excerpt does not establish that it took place on Saturday."
)
claims.loc[claims["Claim_ID"] == "FC005", "Reviewer_Notes"] = (
    "The meeting and discussions are supported; the specific day stated in the claim is not established."
)

# FC008: Only Arafat's welcome is explicit in the supplied excerpt
claims.loc[claims["Claim_ID"] == "FC008", "Verdict"] = "NOT_ENOUGH_INFO"
claims.loc[claims["Claim_ID"] == "FC008", "Evidence_from_Article"] = (
    "The source states that Palestinian leader Yasser Arafat welcomed the court's decision."
)
claims.loc[claims["Claim_ID"] == "FC008", "Reviewer_Notes"] = (
    "Arafat's welcome is supported, but the supplied excerpt does not establish the broader plural claim."
)

# FC009: The magnitude 3.4 is not established in the supplied excerpt
claims.loc[claims["Claim_ID"] == "FC009", "Verdict"] = "NOT_ENOUGH_INFO"
claims.loc[claims["Claim_ID"] == "FC009", "Evidence_from_Article"] = (
    "The source attributes a report of a suspected explosion to Chinese seismologists, "
    "but the supplied excerpt does not state a magnitude of 3.4."
)
claims.loc[claims["Claim_ID"] == "FC009", "Reviewer_Notes"] = (
    "The reported event is mentioned, but the claimed magnitude is not established in the supplied excerpt."
)

claims.to_csv(claim_path, index=False, encoding="utf-8-sig")

print("Saved successfully.")
print("\nUpdated verdicts:")
display(claims[claims["Claim_ID"].isin(["FC005", "FC008", "FC009"])]
        [["Claim_ID", "Verdict", "Reviewer_Notes"]])
print("\nCurrent verdict distribution:")
print(claims["Verdict"].value_counts())

Saved successfully.

Updated verdicts:


,Claim_ID,Verdict,Reviewer_Notes
4,FC005,NOT_ENOUGH_INFO,The meeting and discussions are supported; the...
7,FC008,NOT_ENOUGH_INFO,"Arafat's welcome is supported, but the supplie..."
8,FC009,NOT_ENOUGH_INFO,"The reported event is mentioned, but the claim..."



Current verdict distribution:
Verdict
SUPPORTED          39
NOT_ENOUGH_INFO    11
Name: count, dtype: int64


In [448]:
for start in range(10, 20):
    row = claims.iloc[start]

    print("\n" + "=" * 70)
    print("CLAIM ID:", row["Claim_ID"])
    print("ARTICLE ID:", row["Article_ID"])
    print("CLAIM:", row["Claim"])
    print("\nSOURCE ARTICLE:")
    print(row["Source_Article"])
    print("\nCURRENT VERDICT:", row["Verdict"])
    print("EVIDENCE:", row["Evidence_from_Article"])
    print("REVIEWER NOTES:", row["Reviewer_Notes"])


CLAIM ID: FC011
ARTICLE ID: 140524_iran_hanging_aa
CLAIM: ईरान में सरकारी मीडिया के अनुसार एक कारोबारी को अरबों की हेराफेरी करने के जुर्म में फांसी दे दी गई है.

SOURCE ARTICLE:
मुहाफरीद आमिर खोसरावी को तेहरान की एविन जेल में शनिवार को फांसी दी गई. आमिर खोसरावी 2011 में सामने एक घोटाले में दोषी करार दिए गए थे. उन पर फर्जी दस्तावेजों के आधार पर कर्ज लेने के आरोप लगे, जिसमें बाद में 2.6 अरब डॉलर की धांधली सामने आई. इस मामले के चलते उस समय की सरकार को कहना पड़ा कि इसमें उसके सदस्य शामिल नहीं हैं. ये मामला सितंबर 2011 में सामने आया जब एक निवेश कंपनी पर आरोप लगा कि उसने चार साल के दौरान जाली दस्तावेजों के आधार पर कई बैंकों से कर्ज लिया है. बताया जाता है कि इस धन का इस्तेमाल सरकार की निजीकरण की मुहिम के तहत सरकारी कंपनियों को खरीदने के लिए किया गया. मानवाधिकार समूह ईरान में फांसी की सजा दिए जाने का विरोध करते रहे हैं. उनका कहना है कि ये दुनिया में सबसे सख़्त सज़ा है. (बीबीसी हिन्दी के एंड्रॉएड ऐप के लिए यहां क्लिक करें. आप हमें फ़ेसबुक और ट्विटर पर भी फ़ॉलो कर सकते हैं.)

CURRENT VERDICT: S

In [449]:
updates = {
    "FC013": "NOT_ENOUGH_INFO",
    "FC014": "NOT_ENOUGH_INFO",
    "FC019": "SUPPORTED"
}

notes = {
    "FC013": "Heavy rain is forecast for Tamil Nadu and southern Andhra Pradesh, but Puducherry is not mentioned in the supplied source.",
    "FC014": "The article confirms Surjewala made the allegation, but says the viral tweet misrepresented Mishra's actual speech. The exact claim wording is not established.",
    "FC019": "The board spokesperson explicitly states that an investigation will be conducted and appropriate action considered."
}

for claim_id, verdict in updates.items():
    mask = claims["Claim_ID"] == claim_id
    claims.loc[mask, "Verdict"] = verdict
    claims.loc[mask, "Reviewer_Notes"] = notes[claim_id]

claims.to_csv(claim_path, index=False)

print("Saved successfully.")
print(claims["Verdict"].value_counts())
display(claims[claims["Claim_ID"].isin(updates)][
    ["Claim_ID", "Verdict", "Reviewer_Notes"]
])

Saved successfully.
Verdict
SUPPORTED          37
NOT_ENOUGH_INFO    13
Name: count, dtype: int64


,Claim_ID,Verdict,Reviewer_Notes
12,FC013,NOT_ENOUGH_INFO,Heavy rain is forecast for Tamil Nadu and sout...
13,FC014,NOT_ENOUGH_INFO,The article confirms Surjewala made the allega...
18,FC019,SUPPORTED,The board spokesperson explicitly states that ...


In [450]:
for start in range(20, 30):
    row = claims.iloc[start]

    print("\n" + "=" * 70)
    print("CLAIM ID:", row["Claim_ID"])
    print("ARTICLE ID:", row["Article_ID"])
    print("CLAIM:", row["Claim"])
    print("\nSOURCE ARTICLE:")
    print(row["Source_Article"])
    print("\nCURRENT VERDICT:", row["Verdict"])
    print("EVIDENCE:", row["Evidence_from_Article"])
    print("REVIEWER NOTES:", row["Reviewer_Notes"])


CLAIM ID: FC021
ARTICLE ID: 131202_saina_sindhu_badminton_vr
CLAIM: "चीनी खिलाड़ी को हराने के बाद फाइनल में कदम रखने से पहले मैं आत्मविश्वास से भरी थी और मैं जानती थी कि अगर मैं गलती नहीं करती हूं तो ख़िताब जीत सकती हूं."

SOURCE ARTICLE:
बीते शनिवार को मकाऊ ओपन ग्रां प्री गोल्ड बैडमिंटन टूर्नामेंट अपने नाम करने वाली भारत की उभरती खिलाड़ी पीवी सिंधु का यह कहना उनके बारे में काफी कुछ बताता है. फाइनल में सिंधु ने कनाडा की मिशेल ली को 21-15, 21-15 से मात दी जबकि सेमीफाइनल में उन्होंने चीन की क्वालिफायर किन जिनजिंग को 21-13, 18-21, 21-19 से हराया था. (अंतरराष्ट्रीय सितारे नदारद) पीवी सिंधु की क्षमता पर तो शायद ही किसी को कोई शक हो क्योंकि इसी साल वह विश्व बैडमिंटन चैंपियनशिप में कांस्य पदक जीत चुकी है. ऐसी महत्वपूर्ण उपलब्धि हासिल करने वाली वह पहली भारतीय महिला बैडमिंटन खिलाड़ी हैं. इसके बावजूद पीवी सिंधु को अभी विश्व बैडमिंटन में लंबा रास्ता तय करना है. मकाऊ ओपन में उन्हें शीर्ष वरीयता दी गई थी. उन्होंने अपनी वरीयता के साथ न्याय भी किया. इससे पहले सिंधु ने मलेशियन ओपन भी जीता था. मकाऊ ओप

In [451]:
updates = {
    "FC021": "NOT_ENOUGH_INFO",
    "FC023": "NOT_ENOUGH_INFO",
    "FC025": "NOT_ENOUGH_INFO",
    "FC026": "NOT_ENOUGH_INFO",
    "FC027": "NOT_ENOUGH_INFO",
    "FC028": "NOT_ENOUGH_INFO",
    "FC030": "NOT_ENOUGH_INFO"
}

notes = {
    "FC021": "The source confirms Sindhu's match victories, but does not establish the quoted statement about her confidence before the final.",
    "FC023": "The article describes offline Google Maps, but the supplied text does not establish the claimed India launch announcement and timing.",
    "FC025": "The article discusses Amit Shah's leadership but does not establish when he assumed the party president role.",
    "FC026": "The article confirms Salman Khan was invited to the show, but does not establish the claimed response about Shah Rukh Khan.",
    "FC027": "The supplied source confirms the India visit but does not state the claimed dates of 24–25 February.",
    "FC028": "The source supports some details about Pitroda's career, but does not establish the full name, birth date and birthplace stated in the claim.",
    "FC030": "The source confirms extensive flooding in Bangladesh, but does not establish the claim that millions were homeless or displaced."
}

for claim_id, verdict in updates.items():
    mask = claims["Claim_ID"] == claim_id
    claims.loc[mask, "Verdict"] = verdict
    claims.loc[mask, "Reviewer_Notes"] = notes[claim_id]

claims.to_csv(claim_path, index=False)

print("Saved successfully.")
print(claims["Verdict"].value_counts())
display(claims[claims["Claim_ID"].isin(updates)][
    ["Claim_ID", "Verdict", "Reviewer_Notes"]
])

Saved successfully.
Verdict
SUPPORTED          31
NOT_ENOUGH_INFO    19
Name: count, dtype: int64


,Claim_ID,Verdict,Reviewer_Notes
20,FC021,NOT_ENOUGH_INFO,"The source confirms Sindhu's match victories, ..."
22,FC023,NOT_ENOUGH_INFO,"The article describes offline Google Maps, but..."
24,FC025,NOT_ENOUGH_INFO,The article discusses Amit Shah's leadership b...
25,FC026,NOT_ENOUGH_INFO,The article confirms Salman Khan was invited t...
26,FC027,NOT_ENOUGH_INFO,The supplied source confirms the India visit b...
27,FC028,NOT_ENOUGH_INFO,The source supports some details about Pitroda...
29,FC030,NOT_ENOUGH_INFO,The source confirms extensive flooding in Bang...


In [452]:
for start in range(30, 40):
    row = claims.iloc[start]

    print("\n" + "=" * 70)
    print("CLAIM ID:", row["Claim_ID"])
    print("ARTICLE ID:", row["Article_ID"])
    print("CLAIM:", row["Claim"])
    print("\nSOURCE ARTICLE:")
    print(row["Source_Article"])
    print("\nCURRENT VERDICT:", row["Verdict"])
    print("EVIDENCE:", row["Evidence_from_Article"])
    print("REVIEWER NOTES:", row["Reviewer_Notes"])


CLAIM ID: FC031
ARTICLE ID: international-41960891
CLAIM: अमरीकी राष्ट्रपति डोनल्ड ट्रंप ने एक बार फिर से उत्तर कोरिया के शासक किम जोंग-उन पर ज़ुबानी हमला किया है.

SOURCE ARTICLE:
किम जोंग-उन और डोनल्ड ट्रंप ट्रंप ने ट्विटर के ज़रिए किम जोंग-उन को निशाना बनाया. ट्विटर का शिकायती लहजे में अमरीकी राष्ट्रपति ने कहा कि उन्होंने कभी किम जोंग-उन को 'नाटा और मोटा' नहीं कहा. इससे पहले ट्रंप को उत्तर कोरियाई विदेश मंत्रालय ने 'बूढ़ा' कहा था. ट्रंप ने ट्विटर पर इसे लेकर एक साथ कई ट्वीट किए हैं. उन्होंने उत्तर कोरिया को लेकर मध्यस्थता का भी प्रस्ताव रखा था. ट्रंप ने दक्षिण चीन सागर के विवाद को भी सुलझाने को लेकर मध्यस्थता की बात कही है. दक्षिण चीन सागर में चीन, वियतनाम, फ़िलीपींस, ताइवान, मलेशिया और ब्रुनेई अलग-अलगे दावे पेश करते हैं. पहली बार सामने आए किम ने ट्रंप को कहा 'पागल' डोनल्ड ट्रंप को धमका कर किम जोंग उन ने दिखाई समझदारी! ट्रंप की धमकी, कुत्ते के भौंकने जैसीः उत्तर कोरिया दक्षिण चीन सागर में चीन की तरफ़ से कृत्रिम द्वीप और नौसैनिक गश्ती दल बनाए जाने को लेकर वियतनाम और फ़िलीपींस से भार

In [453]:
updates = {
    "FC032": "NOT_ENOUGH_INFO",
    "FC035": "NOT_ENOUGH_INFO",
    "FC037": "SUPPORTED",
    "FC039": "NOT_ENOUGH_INFO"
}

notes = {
    "FC032": "The study links smoking with worse hangovers after heavy drinking, but does not establish that smoking increases intoxication itself.",
    "FC035": "The source defines a geyser, but the supplied text does not establish the questioner's name and location.",
    "FC037": "The Coalition of Civil Society's leader explicitly states that they are filing a public-interest petition against Mufti Bashir in the High Court.",
    "FC039": "The source identifies the book as Advani's autobiography but does not state its title in the supplied excerpt."
}

for claim_id, verdict in updates.items():
    mask = claims["Claim_ID"] == claim_id
    claims.loc[mask, "Verdict"] = verdict
    claims.loc[mask, "Reviewer_Notes"] = notes[claim_id]

claims.to_csv(claim_path, index=False)

print("Saved successfully.")
print(claims["Verdict"].value_counts())
display(claims[claims["Claim_ID"].isin(updates)][
    ["Claim_ID", "Verdict", "Reviewer_Notes"]
])

Saved successfully.
Verdict
SUPPORTED          29
NOT_ENOUGH_INFO    21
Name: count, dtype: int64


,Claim_ID,Verdict,Reviewer_Notes
31,FC032,NOT_ENOUGH_INFO,The study links smoking with worse hangovers a...
34,FC035,NOT_ENOUGH_INFO,"The source defines a geyser, but the supplied ..."
36,FC037,SUPPORTED,The Coalition of Civil Society's leader explic...
38,FC039,NOT_ENOUGH_INFO,The source identifies the book as Advani's aut...


In [454]:
for start in range(40, 50):
    row = claims.iloc[start]

    print("\n" + "=" * 70)
    print("CLAIM ID:", row["Claim_ID"])
    print("ARTICLE ID:", row["Article_ID"])
    print("CLAIM:", row["Claim"])
    print("\nSOURCE ARTICLE:")
    print(row["Source_Article"])
    print("\nCURRENT VERDICT:", row["Verdict"])
    print("EVIDENCE:", row["Evidence_from_Article"])
    print("REVIEWER NOTES:", row["Reviewer_Notes"])


CLAIM ID: FC041
ARTICLE ID: 071227_musharraf_statement
CLAIM: पाकिस्तान के राष्ट्रपति परवेज़ मुशर्रफ़ ने पूर्व प्रधानमंत्री बेनज़ीर भुट्टो की हत्या को 'आतंकवादी कार्रवाई' करार दिया है.

SOURCE ARTICLE:
मुशर्रफ़ ने कहा है कि आतंकवाद पाकिस्तान की तरक्की की राह में सबसे बड़ी रुकावट है. टेलीविज़न पर दिए गए एक बयान में उन्होंने कहा कि वो तब तक चैन से नहीं बैठेंगे जब तक देश से दहशतगर्दों का ख़ात्मा नहीं कर दिया जाता. उन्होंने आतंकवाद के ख़ात्मे के लिए पाकिस्तानी जनता से समर्थन की अपील की और कहा ' पाकिस्तान की तरक्की में सबसे बड़ी रुकावट दहशतगर्दी है और वो इसे जड़ से उखाड़ने की कोशिश कर रहे हैं. ' इसके साथ ही उन्होंने पाकिस्तानी जनता से शांति और अमन चैन बनाए रखने की अपील भी की और कहा कि लोग सब्र करें ताकि कठिन समय में अफरा तफरी न मचे. पाकिस्तान में तीन दिन के राष्ट्रीय शोक की घोषणा कर दी गई है.

CURRENT VERDICT: NOT_ENOUGH_INFO
EVIDENCE: मुशर्रफ़ ने कहा है कि आतंकवाद पाकिस्तान की तरक्की की राह में सबसे बड़ी रुकावट है।
REVIEWER NOTES: The displayed excerpt discusses Musharraf's statement agai

In [455]:
updates = {
    "FC045": "NOT_ENOUGH_INFO",
    "FC047": "NOT_ENOUGH_INFO",
    "FC048": "NOT_ENOUGH_INFO",
    "FC049": "NOT_ENOUGH_INFO",
    "FC050": "NOT_ENOUGH_INFO"
}

notes = {
    "FC045": "The source confirms Sandeep Upadhyay's arrest in Surat, but does not explicitly establish that Uttar Pradesh Police made the arrest.",
    "FC047": "The source reports the executions and the killing of 17 border guards, but explicitly says their direct connection is unclear.",
    "FC048": "The source reports the swine flu outbreak, deaths and public restrictions, but does not establish that health officials appealed for people to remain calm.",
    "FC049": "The source mentions gifts exchanged between Churchill and the Saudi king, but does not establish the exact 1945 visit or the claimed £100 gift bottle.",
    "FC050": "The supplied article discusses Yuvraj's cancer treatment and his mother's experience, but does not contain the specific statement in the claim."
}

for claim_id, verdict in updates.items():
    mask = claims["Claim_ID"] == claim_id
    claims.loc[mask, "Verdict"] = verdict
    claims.loc[mask, "Reviewer_Notes"] = notes[claim_id]

claims.to_csv(claim_path, index=False)

print("Saved successfully.")
print("\nFinal verdict distribution:")
print(claims["Verdict"].value_counts())

print("\nUpdated records:")
display(claims[claims["Claim_ID"].isin(updates)][
    ["Claim_ID", "Verdict", "Reviewer_Notes"]
])

Saved successfully.

Final verdict distribution:
Verdict
NOT_ENOUGH_INFO    26
SUPPORTED          24
Name: count, dtype: int64

Updated records:


,Claim_ID,Verdict,Reviewer_Notes
44,FC045,NOT_ENOUGH_INFO,The source confirms Sandeep Upadhyay's arrest ...
46,FC047,NOT_ENOUGH_INFO,The source reports the executions and the kill...
47,FC048,NOT_ENOUGH_INFO,"The source reports the swine flu outbreak, dea..."
48,FC049,NOT_ENOUGH_INFO,The source mentions gifts exchanged between Ch...
49,FC050,NOT_ENOUGH_INFO,The supplied article discusses Yuvraj's cancer...


In [456]:
print("TOTAL ROWS:", len(claims))
print("UNIQUE CLAIM IDs:", claims["Claim_ID"].nunique())
print("MISSING VALUES:\n", claims.isna().sum())
print("\nVERDICT COUNTS:")
print(claims["Verdict"].value_counts())

print("\nDUPLICATE CLAIM IDs:")
display(claims[claims["Claim_ID"].duplicated(keep=False)])

print("\nFC016 CURRENT LABEL:")
display(claims[claims["Claim_ID"] == "FC016"][
    ["Claim_ID", "Verdict", "Evidence_from_Article", "Reviewer_Notes"]
])

print("\nCLAIM ID RANGE:")
print(claims["Claim_ID"].iloc[0], "to", claims["Claim_ID"].iloc[-1])

TOTAL ROWS: 50
UNIQUE CLAIM IDs: 50
MISSING VALUES:
 Claim_ID                 0
Article_ID               0
Claim                    0
Source_Article           0
Verdict                  0
Evidence_from_Article    0
Reviewer_Notes           0
dtype: int64

VERDICT COUNTS:
Verdict
NOT_ENOUGH_INFO    26
SUPPORTED          24
Name: count, dtype: int64

DUPLICATE CLAIM IDs:


,Claim_ID,Article_ID,Claim,Source_Article,Verdict,Evidence_from_Article,Reviewer_Notes



FC016 CURRENT LABEL:


,Claim_ID,Verdict,Evidence_from_Article,Reviewer_Notes
15,FC016,NOT_ENOUGH_INFO,कंगना ने बताया ... ना दीपिका ना सोनम नेक्स्ट ज...,The excerpt discusses her direct statements bu...



CLAIM ID RANGE:
FC001 to FC050


In [457]:
report_path = base_path + "/08_reports/FactIndic_PersonB_Final_Report.txt"

with open(report_path, "r", encoding="utf-8") as f:
    report_text = f.read()

for i, line in enumerate(report_text.splitlines()):
    if any(word in line.lower() for word in ["claim", "supported", "not_enough_info", "contradicted"]):
        print(f"{i+1}: {line}")

120: 6. CLAIM LABELING
122: - Prepared 50 claim-label records.
123: - 50 unique claim IDs and article IDs.
125:   SUPPORTED: 42
126:   NOT_ENOUGH_INFO: 8
127:   CONTRADICTED: 0
130:   50 claims received an independent second audit.
133: 07_claim_labels/FactIndic_50_Claim_Labeling_Sheet.csv
192: 5. Independently audit all claim labels if the project requires
193:    a fully double-checked claim dataset.
203: 23-article subset, real Hinglish review, claim labeling,


In [458]:
report_text = report_text.replace(
    "SUPPORTED: 42\n  NOT_ENOUGH_INFO: 8\n  CONTRADICTED: 0",
    "SUPPORTED: 24\n  NOT_ENOUGH_INFO: 26\n  CONTRADICTED: 0"
)

report_text = report_text.replace(
    "50 claims received an independent second audit.",
    "All 50 claims were independently audited against their source articles. Final labels: 24 SUPPORTED, 26 NOT_ENOUGH_INFO, and 0 CONTRADICTED."
)

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print("Report updated:", __import__("os").path.exists(report_path))
print("\nUpdated claim-labeling section:")

for i, line in enumerate(report_text.splitlines()):
    if 119 <= i <= 134:
        print(line)

Report updated: True

Updated claim-labeling section:
6. CLAIM LABELING

- Prepared 50 claim-label records.
- 50 unique claim IDs and article IDs.
- Label distribution:
  SUPPORTED: 24
  NOT_ENOUGH_INFO: 26
  CONTRADICTED: 0
- Structural validation found no missing values or duplicate IDs.
- The initial NEI subset was individually rechecked; not all
  All 50 claims were independently audited against their source articles. Final labels: 24 SUPPORTED, 26 NOT_ENOUGH_INFO, and 0 CONTRADICTED.

Saved file:
07_claim_labels/FactIndic_50_Claim_Labeling_Sheet.csv




In [459]:
report_text = report_text.replace(
    "- 50 unique claim IDs and article IDs.",
    "- 50 unique claim IDs."
)

report_text = report_text.replace(
    "- The initial NEI subset was individually rechecked; not all\n"
    "  All 50 claims were independently audited against their source articles. Final labels: 24 SUPPORTED, 26 NOT_ENOUGH_INFO, and 0 CONTRADICTED.",
    "- All 50 claims were independently audited against their source articles."
)

report_text = report_text.replace(
    "Independently audit all claim labels if the project requires\n"
    "   a fully double-checked claim dataset.",
    "Claim-label audit completed: all 50 claims checked against source articles."
)

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print("Clean report saved.")
print("\nClaim-labeling section:")
lines = report_text.splitlines()
start = next(i for i, line in enumerate(lines) if "6. CLAIM LABELING" in line)
for line in lines[start:start+14]:
    print(line)

print("\nCheck for outdated audit task:")
print("Old task remains:", "Independently audit all claim labels" in report_text)

Clean report saved.

Claim-labeling section:
6. CLAIM LABELING

- Prepared 50 claim-label records.
- 50 unique claim IDs.
- Label distribution:
  SUPPORTED: 24
  NOT_ENOUGH_INFO: 26
  CONTRADICTED: 0
- Structural validation found no missing values or duplicate IDs.
- All 50 claims were independently audited against their source articles.

Saved file:
07_claim_labels/FactIndic_50_Claim_Labeling_Sheet.csv


Check for outdated audit task:
Old task remains: False


In [460]:
import pandas as pd
import os

review_path = base_path + "/05_quality_checks/FactIndic_Mixed_Human_Review.csv"
mixed_path = base_path + "/04_mixed_data/FactIndic_Mixed_23_Validated.csv"

reviews = pd.read_csv(review_path)
mixed = pd.read_csv(mixed_path)

print("Total mixed articles:", len(mixed))
print("Review records:", len(reviews))
print("Reviewed unique articles:", reviews["sample_id"].nunique())

reviewed_ids = set(reviews["sample_id"].dropna())
pending = mixed[~mixed["sample_id"].isin(reviewed_ids)]

print("\nRemaining articles:", len(pending))
display(pending[["sample_id", "title"]])

Total mixed articles: 23
Review records: 5
Reviewed unique articles: 5

Remaining articles: 18


,sample_id,title
1,051105_sahubuddin_arrest,शहाबुद्दीन दिल्ली में गिरफ़्तार
2,150901_nepal_protest_dead_pp,नेपाल में पुलिस गोलीबारी से पाँच की मौत
3,041003_pwg_security,नक्सली वार्ताकारों को मिलेगी सुरक्षा
4,080416_up_results,उपचुनाव में पाँचों सीटें बसपा को
5,060117_indopak_cbm,रेल सेवा अजमेर तक बढ़ाने का प्रस्ताव
6,140206_egypt_al-sisi_presidential_election_sr,मिस्र: अल-सीसी के चुनाव लड़ने का खंडन
7,140410_action_against_manvendra_singh_rns,जसवंत सिंह के बेटे को भाजपा ने राष्ट्रीय कार्य...
10,080301_benazir_suspect,बेनज़ीर हत्याकांड में मेहसूद पर आरोपपत्र
11,050107_pravasi_pm,दोहरी नागरिकता का दायरा व्यापक हुआ
12,050505_voteclose_britain,ब्रिटेन के आम चुनाव में मतदान समाप्त


In [461]:
sample_id = "051105_sahubuddin_arrest"

row = mixed[mixed["sample_id"] == sample_id].iloc[0]

print("ARTICLE ID:", row["sample_id"])
print("TITLE:", row["title"])

for col in ["hindi_article", "mixed_10_article",
            "mixed_30_article", "mixed_50_article"]:
    print("\n" + "=" * 70)
    print(col.upper())
    print("=" * 70)
    print(row[col])

ARTICLE ID: 051105_sahubuddin_arrest
TITLE: शहाबुद्दीन दिल्ली में गिरफ़्तार

HINDI_ARTICLE
शहाबुद्दीन बिहार के सीवान क्षेत्र से सांसद हैं और पीटीआई के अनुसार उन्हें दिल्ली स्थित उनके सांसद निवास से बिहार पुलिस के एक दल ने गिरफ़्तार किया. इसके पहले उनकी गिरफ़्तारी को लेकर बिहार सरकार ने देशव्यापी अलर्ट घोषित कर दिया था. बिहार की पुलिस ने सभी राज्यों की पुलिस को सूचित किया था कि शहाबुद्दीन के ख़िलाफ़ ग़ैरज़मानती वारंट है और वे जहाँ भी देखे जाएँ उन्हें गिरफ़्तार कर लिया जाए. ग़ौरतलब है कि इसके पहले जब बिहार पुलिस उन्हें गिरफ़्तार करने दिल्ली पहुँची थी तब वो दिल्ली में ही थे और संसद के मानसून सत्र में हिस्सा ले रहे थे. लेकिन इसके बाद वो लापता हो गए. पर इस बीच शहाबुद्दीन लगातार पत्रकारों से बात करते रहे हैं. उन्होंने बातचीत में कहा था कि वो क़ानून से भाग नहीं रहे हैं और उन्होंने अपने ऊपर लगे आरोपों के ख़िलाफ़ अदालत का दरवाज़ा खटखटाया है. वैसे तो शहाबुद्दीन के ख़िलाफ़ आठ ग़ैरज़मानती वारंट है लेकिन जिस मामले में पुलिस उनके ख़िलाफ़ गिरफ़्तारी वारंट लेकर घूम रही थी वह हथियारों से संबंधित है. उल

In [462]:
review_row = {
    "sample_id": "051105_sahubuddin_arrest",
    "Fluency_10": "Good",
    "Meaning_10": "Yes",
    "Fluency_30": "Good",
    "Meaning_30": "Yes",
    "Fluency_50": "Good",
    "Meaning_50": "Yes",
    "Reviewer_Notes": "All three versions preserve the main facts, names, locations and the eight-warrant detail. English sentences are understandable. Some transitions between Hindi and English are abrupt, and 'externed from the district' is formal. Sentence-level mixing does not create natural within-sentence Hinglish."
}

reviews = pd.concat([reviews, pd.DataFrame([review_row])], ignore_index=True)
reviews = reviews.drop_duplicates(subset=["sample_id"], keep="last")
reviews.to_csv(review_path, index=False)

print("Saved:", review_path)
print("Total reviewed:", reviews["sample_id"].nunique())
display(reviews[reviews["sample_id"] == "051105_sahubuddin_arrest"])

Saved: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/FactIndic_Mixed_Human_Review.csv
Total reviewed: 6


,sample_id,reviewed_levels,sentence_1_issue,sentence_7_issue,sentence_8_status,sentence_11_status,overall_status,reviewer_notes,sentence_8_issue,sentence_9_issue,...,sentence_16_status,sentence_5_6_issue,sentence_11_12_issue,Fluency_10,Meaning_10,Fluency_30,Meaning_30,Fluency_50,Meaning_50,Reviewer_Notes
5,051105_sahubuddin_arrest,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Good,Yes,Good,Yes,Good,Yes,"All three versions preserve the main facts, na..."


In [463]:
sample_id = "150901_nepal_protest_dead_pp"

row = mixed[mixed["sample_id"] == sample_id].iloc[0]

print("ARTICLE ID:", row["sample_id"])
print("TITLE:", row["title"])

for col in ["hindi_article", "mixed_10_article",
            "mixed_30_article", "mixed_50_article"]:
    print("\n" + "=" * 70)
    print(col.upper())
    print("=" * 70)
    print(row[col])

ARTICLE ID: 150901_nepal_protest_dead_pp
TITLE: नेपाल में पुलिस गोलीबारी से पाँच की मौत

HINDI_ARTICLE
मारे गए लोग मधेशी समुदाय के हैं. पुलिस ने बीबीसी को बताया है कि उन्हें प्रदर्शनकारियों पर गोलियाँ चलाने को मजबूर होना पड़ा क्योंकि उन्होंने बंदूकों और चाकुओं से हमला किया था. नए संविधान के प्रस्ताव के मुताबिक़ नेपाल को संघीय प्रांतों में बाँटा जाएगा. नेपाल के मधेशी और अन्य ग्रुपों का मानना है कि उनकी अनदेखी की जा रही है. अधिकारियों के मुताबिक़ राजधानी काठमांडू से 90 किलोमीटर दक्षिण-पश्चिम में स्थित बीरगंज में प्रदर्शनकारियों ने सरकारी कार्यालयों और पुलिस चौकियों में तोड़-फोड़ की. यहाँ पुलिस की गोली से चार लोग मारे गए. मौत जबकि नज़दीकी शहर कलैया में एक अन्य व्यक्ति की मौत हो गई. समाप्त इन सबको मिलाकर हाल के हफ़्तों में प्रदर्शन के दौरान कम से कम 20 लोगों की मौत हो चुकी है. वर्ष 2008 में राजशाही की समाप्ति के बाद से ही नेपाल अंतरिम संविधान के आधार पर काम कर रहा है. वर्षों तक चली उठापटक के बाद राजनीतिक दल इस साल संविधान के मसौदे पर सहमत हुए हैं. प्रदर्शन इस मसौदे के समर्थक ये कहते हैं कि

In [464]:
review_row = {
    "sample_id": "150901_nepal_protest_dead_pp",
    "Fluency_10": "Fair",
    "Meaning_10": "Yes",
    "Fluency_30": "Fair",
    "Meaning_30": "Yes",
    "Fluency_50": "Fair",
    "Meaning_50": "Yes",
    "Reviewer_Notes": "Translated sentences preserve the main meaning, names and numerical details. However, Hindi fragments such as 'मौत' and 'समाप्त' remain in the mixed text and interrupt readability. The 30% and 50% versions are nearly identical, so their actual mixing levels should be checked."
}

reviews = pd.concat([reviews, pd.DataFrame([review_row])], ignore_index=True)
reviews = reviews.drop_duplicates(subset=["sample_id"], keep="last")
reviews.to_csv(review_path, index=False)

print("Saved successfully.")
print("Total reviewed:", reviews["sample_id"].nunique())
display(reviews[reviews["sample_id"] == "150901_nepal_protest_dead_pp"][
    ["sample_id", "Fluency_10", "Meaning_10",
     "Fluency_30", "Meaning_30", "Fluency_50",
     "Meaning_50", "Reviewer_Notes"]
])

Saved successfully.
Total reviewed: 7


,sample_id,Fluency_10,Meaning_10,Fluency_30,Meaning_30,Fluency_50,Meaning_50,Reviewer_Notes
6,150901_nepal_protest_dead_pp,Fair,Yes,Fair,Yes,Fair,Yes,Translated sentences preserve the main meaning...


In [465]:
next_ids = [
    "041003_pwg_security",
    "080416_up_results",
    "060117_indopak_cbm",
    "140206_egypt_al-sisi_presidential_election_sr",
    "140410_action_against_manvendra_singh_rns"
]

for sample_id in next_ids:
    row = mixed[mixed["sample_id"] == sample_id].iloc[0]

    print("\n" + "#" * 90)
    print("ARTICLE ID:", row["sample_id"])
    print("TITLE:", row["title"])

    for col in ["hindi_article", "mixed_10_article",
                "mixed_30_article", "mixed_50_article"]:
        print("\n" + "=" * 70)
        print(col.upper())
        print("=" * 70)
        print(row[col])


##########################################################################################
ARTICLE ID: 041003_pwg_security
TITLE: नक्सली वार्ताकारों को मिलेगी सुरक्षा

HINDI_ARTICLE
अधिकारियों का कहना है कि सरकार ने पुलिस को निर्देश दिए हैं कि तीन प्रमुख माओवादी नेताओं को सुरक्षित रास्ता मुहैया कराया जाए. वार्ताकारों की सुरक्षा के बारे में पुलिस अधिकारी पीडब्ल्यूजी के मध्यस्थ वारा वारा राव, गदर और कल्याण राव से बात करेंगे. पीडब्ल्यूजी के तीन प्रमुख नेता रामकृष्णा, सुधाकर और गणेश सरकार के साथ अक्तूबर के दूसरे हफ्ते में बातचीत के लिए आ रहे हैं. सुरक्षा अहम विद्रोही संगठन पीडब्ल्यूजी के लिए सरकार से वार्ता के दौरान अपने नेताओं की सुरक्षा सबसे अहम मुद्दा है . इसी कारण संगठन ने दो अक्तूबर को वार्ता की सरकार की पेशकश को ठुकरा दिया था. संगठन ने पिछले दिनों हैदराबाद में एक विशाल जनसभा का आयोजन किया था. पुलिस ने इस दौरान संगठन के कार्यकर्ताओं को बिल्कुल परेशान नहीं किया था जिसके बाद संगठन में काफी आत्मविश्वास आया है. पीडब्ल्यूजी के निकटवर्ती लोगों के अनुसार जब तक तीनों नेता हैदराबाद नहीं पहुंच

In [466]:
new_reviews = [
    {
        "sample_id": "041003_pwg_security",
        "Fluency_10": "Good",
        "Meaning_10": "Yes",
        "Fluency_30": "Good",
        "Meaning_30": "Yes",
        "Fluency_50": "Good",
        "Meaning_50": "Yes",
        "Reviewer_Notes": "Main facts, names and October dates are preserved across all versions. English sentences are understandable, although Hindi-English transitions are sometimes abrupt."
    },
    {
        "sample_id": "080416_up_results",
        "Fluency_10": "Fair",
        "Meaning_10": "Yes",
        "Fluency_30": "Fair",
        "Meaning_30": "Yes",
        "Fluency_50": "Fair",
        "Meaning_50": "Yes",
        "Reviewer_Notes": "Main election results, candidate names and vote margins in translated sentences are preserved. Several paragraphs remain in Hindi, including in the 50% version. Mixing coverage and transitions are uneven."
    },
    {
        "sample_id": "060117_indopak_cbm",
        "Fluency_10": "Good",
        "Meaning_10": "Yes",
        "Fluency_30": "Good",
        "Meaning_30": "Yes",
        "Fluency_50": "Good",
        "Meaning_50": "Yes",
        "Reviewer_Notes": "Translated sentences preserve the main details, names, dates, locations and diplomatic issues. English is understandable, though some transitions are abrupt."
    },
    {
        "sample_id": "140206_egypt_al-sisi_presidential_election_sr",
        "Fluency_10": "Fair",
        "Meaning_10": "Yes",
        "Fluency_30": "Fair",
        "Meaning_30": "Yes",
        "Fluency_50": "Fair",
        "Meaning_50": "Yes",
        "Reviewer_Notes": "Translated sentences preserve the main reported statements and political details. Much of the article remains in Hindi, and transitions are uneven. The article reports a denial of the newspaper's claim, not confirmation of Al-Sisi's candidacy."
    },
    {
        "sample_id": "140410_action_against_manvendra_singh_rns",
        "Fluency_10": "Fair",
        "Meaning_10": "Yes",
        "Fluency_30": "Fair",
        "Meaning_30": "Yes",
        "Fluency_50": "Fair",
        "Meaning_50": "Yes",
        "Reviewer_Notes": "Main political details and names are mostly preserved in translated sentences. Several paragraphs remain in Hindi. The source opening itself is confusing about Jaswant Singh, and the mixed versions retain this ambiguity."
    }
]

reviews = pd.concat(
    [reviews, pd.DataFrame(new_reviews)],
    ignore_index=True
)

reviews = reviews.drop_duplicates(
    subset=["sample_id"],
    keep="last"
)

reviews.to_csv(review_path, index=False)

print("Saved successfully:", review_path)
print("Total reviewed:", reviews["sample_id"].nunique())
print("Unique reviewed IDs:", reviews["sample_id"].nunique())

display(
    reviews[reviews["sample_id"].isin(
        [r["sample_id"] for r in new_reviews]
    )][[
        "sample_id", "Fluency_10", "Meaning_10",
        "Fluency_30", "Meaning_30",
        "Fluency_50", "Meaning_50", "Reviewer_Notes"
    ]]
)

Saved successfully: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/FactIndic_Mixed_Human_Review.csv
Total reviewed: 12
Unique reviewed IDs: 12


,sample_id,Fluency_10,Meaning_10,Fluency_30,Meaning_30,Fluency_50,Meaning_50,Reviewer_Notes
7,041003_pwg_security,Good,Yes,Good,Yes,Good,Yes,"Main facts, names and October dates are preser..."
8,080416_up_results,Fair,Yes,Fair,Yes,Fair,Yes,"Main election results, candidate names and vot..."
9,060117_indopak_cbm,Good,Yes,Good,Yes,Good,Yes,Translated sentences preserve the main details...
10,140206_egypt_al-sisi_presidential_election_sr,Fair,Yes,Fair,Yes,Fair,Yes,Translated sentences preserve the main reporte...
11,140410_action_against_manvendra_singh_rns,Fair,Yes,Fair,Yes,Fair,Yes,Main political details and names are mostly pr...


In [467]:
next_ids = [
    "080301_benazir_suspect",
    "050107_pravasi_pm",
    "050505_voteclose_britain",
    "060106_rajasthan_rail",
    "060709_wordcup_final2006"
]

for sample_id in next_ids:
    row = mixed[mixed["sample_id"] == sample_id].iloc[0]

    print("\n" + "#" * 90)
    print("ARTICLE ID:", row["sample_id"])
    print("TITLE:", row["title"])

    for col in ["hindi_article", "mixed_10_article",
                "mixed_30_article", "mixed_50_article"]:
        print("\n" + "=" * 70)
        print(col.upper())
        print("=" * 70)
        print(row[col])


##########################################################################################
ARTICLE ID: 080301_benazir_suspect
TITLE: बेनज़ीर हत्याकांड में मेहसूद पर आरोपपत्र

HINDI_ARTICLE
इस मामले की जाँच कर रहे प्रमुख अधिकारी चौधरी अब्दुल मजीद ने बताया कि इस मामले में मेहसूद के अलावा चार अन्य लोगों के ख़िलाफ़ भी मामला दर्ज किया गया है. अदालत में मामला दर्ज करने के बाद जज ने पाँचों संदिग्ध अभियुक्तों की गिरफ़्तारी का ग़ैर ज़मानती वारंट भी जारी कर दिया है. 27 दिसंबर को रावलपिंडी में एक आत्मघाती हमले में पूर्व प्रधानमंत्री बेनज़ीर भुट्टो की मौत हो गई थी. प्रभाव बैतुल्लाह मेहसूद अफ़ग़ानिस्तान की सीमा से लगे दक्षिणी वज़ीरिस्तान इलाक़े में रहते हैं. ये इलाक़ा तालेबान समर्थक चरमपंथियों के प्रभाव वाला है. हालाँकि मेहसूद पहले ही इस मामले में अपनी भूमिका से इनकार कर चुके हैं. आरोप ये भी है कि बैतुल्लाह मेहसूद चरमपंथी संगठन अल क़ायदा से जुड़े हुए हैं. पाकिस्तान की सरकार पहले ही बेनज़ीर भुट्टो की हत्या के मामले में बैतुल्लाह मेहसूद की भूमिका पर सवाल उठा चुकी है. राष्ट्रपति परवेज़ मुशर्रफ़ भी बे

In [468]:
new_reviews = [
    {
        "sample_id": "080301_benazir_suspect",
        "Fluency_10": "Fair",
        "Meaning_10": "Yes",
        "Fluency_30": "Good",
        "Meaning_30": "Yes",
        "Fluency_50": "Good",
        "Meaning_50": "Yes",
        "Reviewer_Notes": "Names, five suspects and the December 27 date are preserved. The 10% version contains no English replacement. Higher-level versions have understandable English, although some sentences are long."
    },
    {
        "sample_id": "050107_pravasi_pm",
        "Fluency_10": "Fair",
        "Meaning_10": "Yes",
        "Fluency_30": "Good",
        "Meaning_30": "Yes",
        "Fluency_50": "Good",
        "Meaning_50": "Yes",
        "Reviewer_Notes": "The main announcement, 16 countries, estimate of more than 25 million overseas Indians and conference details are preserved. The 10% version contains no English replacement. Higher-level versions are understandable."
    },
    {
        "sample_id": "050505_voteclose_britain",
        "Fluency_10": "Good",
        "Meaning_10": "Yes",
        "Fluency_30": "Good",
        "Meaning_30": "Yes",
        "Fluency_50": "Good",
        "Meaning_50": "Yes",
        "Reviewer_Notes": "Translated passages preserve the voter estimate, 645 contested seats, 324-seat majority, party seat counts and historical turnout figures. English is readable and the main numerical details are consistent with the source."
    },
    {
        "sample_id": "060106_rajasthan_rail",
        "Fluency_10": "Good",
        "Meaning_10": "Yes",
        "Fluency_30": "Good",
        "Meaning_30": "Yes",
        "Fluency_50": "Good",
        "Meaning_50": "Yes",
        "Reviewer_Notes": "The February 1, 2006 start date, December 2004 restoration decision and historical details are preserved in translated passages. English is readable, with some sentence-level language switching."
    },
    {
        "sample_id": "060709_wordcup_final2006",
        "Fluency_10": "Good",
        "Meaning_10": "Yes",
        "Fluency_30": "Good",
        "Meaning_30": "Yes",
        "Fluency_50": "Good",
        "Meaning_50": "Yes",
        "Reviewer_Notes": "Translated passages preserve match history, scores, player names, retirement details and World Cup statistics. English is understandable, although language switching remains sentence-level."
    }
]

reviews = pd.concat(
    [reviews, pd.DataFrame(new_reviews)],
    ignore_index=True
)

reviews = reviews.drop_duplicates(
    subset=["sample_id"],
    keep="last"
)

reviews.to_csv(review_path, index=False)

print("Saved successfully:", review_path)
print("Total reviewed:", reviews["sample_id"].nunique())

display(
    reviews[reviews["sample_id"].isin(
        [r["sample_id"] for r in new_reviews]
    )][[
        "sample_id", "Fluency_10", "Meaning_10",
        "Fluency_30", "Meaning_30",
        "Fluency_50", "Meaning_50", "Reviewer_Notes"
    ]]
)

Saved successfully: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/FactIndic_Mixed_Human_Review.csv
Total reviewed: 17


,sample_id,Fluency_10,Meaning_10,Fluency_30,Meaning_30,Fluency_50,Meaning_50,Reviewer_Notes
12,080301_benazir_suspect,Fair,Yes,Good,Yes,Good,Yes,"Names, five suspects and the December 27 date ..."
13,050107_pravasi_pm,Fair,Yes,Good,Yes,Good,Yes,"The main announcement, 16 countries, estimate ..."
14,050505_voteclose_britain,Good,Yes,Good,Yes,Good,Yes,Translated passages preserve the voter estimat...
15,060106_rajasthan_rail,Good,Yes,Good,Yes,Good,Yes,"The February 1, 2006 start date, December 2004..."
16,060709_wordcup_final2006,Good,Yes,Good,Yes,Good,Yes,"Translated passages preserve match history, sc..."


In [469]:
reviewed_ids = set(reviews["sample_id"].unique())

pending_ids = [
    sid for sid in mixed["sample_id"].tolist()
    if sid not in reviewed_ids
]

print("Remaining articles:", len(pending_ids))
print("Next batch:", pending_ids[:5])

for sample_id in pending_ids[:5]:
    row = mixed[mixed["sample_id"] == sample_id].iloc[0]

    print("\n" + "#" * 90)
    print("ARTICLE ID:", row["sample_id"])
    print("TITLE:", row["title"])

    for col in ["hindi_article", "mixed_10_article",
                "mixed_30_article", "mixed_50_article"]:
        print("\n" + "=" * 70)
        print(col.upper())
        print("=" * 70)
        print(row[col])

Remaining articles: 6
Next batch: ['080801_india_iaea', '040510_fillipino_election', '050228_tax_income', 'india-39447746', '051209_tamilnadu_cyclone']

##########################################################################################
ARTICLE ID: 080801_india_iaea
TITLE: परमाणु समझौते को आईएईए की मंज़ूरी

HINDI_ARTICLE
परमाणु समझौते को लागू करने की दिशा में आईएईए की मंज़ूरी आवश्यक थी. आईएईए की मंज़ूरी के बाद भारत को अब 45 सदस्यीय परमाणु सप्लायर्स ग्रुप (एनएसजी) से सहमति लेनी होगी. ताकि भारत परमाणु ईंधन के व्यापार में शामिल हो सके. इसके बाद भारत-अमरीका परमाणु समझौते को अमरीकी संसद के अनुमोदन की ज़रूरत होगी. भारत पर यह दबाव बना हुआ है कि वह नवंबर में होने वाले अमरीकी राष्ट्रपति चुनाव से पहले इस समझौते को अमरीकी संसद तक पहुँचा दे और उसकी मंज़ूरी ले ले. परमाणु समझौता हाल ही में भारत सरकार को इसी परमाणु समझौते को लेकर संसद में विश्वास मत प्रस्ताव रखना पड़ा. क्योंकि सरकार को बाहर से समर्थन दे रही वामपंथी पार्टियों ने परमाणु समझौते के मुद्दे पर सरकार से समर्थन वापस ले लिया था. लेकिन 

In [470]:
new_reviews = [
    {
        "sample_id": "080801_india_iaea",
        "Fluency_10": "Fair",
        "Meaning_10": "Yes",
        "Fluency_30": "Good",
        "Meaning_30": "Yes",
        "Fluency_50": "Good",
        "Meaning_50": "Yes",
        "Reviewer_Notes": "The 45-member NSG, trust vote and 14 of 22 nuclear facilities are preserved. The final English sentence about civilian use is awkwardly phrased. The 10% version has limited English coverage."
    },
    {
        "sample_id": "040510_fillipino_election",
        "Fluency_10": "Fair",
        "Meaning_10": "Yes",
        "Fluency_30": "Good",
        "Meaning_30": "Yes",
        "Fluency_50": "Good",
        "Meaning_50": "Yes",
        "Reviewer_Notes": "Translated passages preserve the reported deaths, 230,000 security personnel, candidate names, polling estimates and election issues. The 10% version contains no English replacement."
    },
    {
        "sample_id": "050228_tax_income",
        "Fluency_10": "Good",
        "Meaning_10": "Yes",
        "Fluency_30": "Good",
        "Meaning_30": "Yes",
        "Fluency_50": "Good",
        "Meaning_50": "Yes",
        "Reviewer_Notes": "The Rs 10,000 cash withdrawal and Rs 10 tax calculation are preserved. The income-tax thresholds remain in Hindi in all versions, so English coverage is incomplete."
    },
    {
        "sample_id": "india-39447746",
        "Fluency_10": "Fair",
        "Meaning_10": "Yes",
        "Fluency_30": "Good",
        "Meaning_30": "Yes",
        "Fluency_50": "Good",
        "Meaning_50": "Yes",
        "Reviewer_Notes": "The translated passages preserve the names, location of arrest and reported events. The 10% version contains no English replacement. The higher-level versions are understandable."
    },
    {
        "sample_id": "051209_tamilnadu_cyclone",
        "Fluency_10": "Fair",
        "Meaning_10": "Yes",
        "Fluency_30": "Good",
        "Meaning_30": "Yes",
        "Fluency_50": "Good",
        "Meaning_50": "Yes",
        "Reviewer_Notes": "The 550 km distance, 70 km per hour wind speed and 75,000 evacuees are preserved. The source repeats the evacuation figure, and the mixed versions retain this repetition. The 10% version contains no English replacement."
    }
]

reviews = pd.concat(
    [reviews, pd.DataFrame(new_reviews)],
    ignore_index=True
)

reviews = reviews.drop_duplicates(
    subset=["sample_id"],
    keep="last"
)

reviews.to_csv(review_path, index=False)

print("Saved successfully:", review_path)
print("Total reviewed:", reviews["sample_id"].nunique())

display(
    reviews[reviews["sample_id"].isin(
        [r["sample_id"] for r in new_reviews]
    )][[
        "sample_id", "Fluency_10", "Meaning_10",
        "Fluency_30", "Meaning_30",
        "Fluency_50", "Meaning_50", "Reviewer_Notes"
    ]]
)

Saved successfully: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/FactIndic_Mixed_Human_Review.csv
Total reviewed: 22


,sample_id,Fluency_10,Meaning_10,Fluency_30,Meaning_30,Fluency_50,Meaning_50,Reviewer_Notes
17,080801_india_iaea,Fair,Yes,Good,Yes,Good,Yes,"The 45-member NSG, trust vote and 14 of 22 nuc..."
18,040510_fillipino_election,Fair,Yes,Good,Yes,Good,Yes,Translated passages preserve the reported deat...
19,050228_tax_income,Good,Yes,Good,Yes,Good,Yes,"The Rs 10,000 cash withdrawal and Rs 10 tax ca..."
20,india-39447746,Fair,Yes,Good,Yes,Good,Yes,"The translated passages preserve the names, lo..."
21,051209_tamilnadu_cyclone,Fair,Yes,Good,Yes,Good,Yes,"The 550 km distance, 70 km per hour wind speed..."


In [471]:
reviewed_ids = set(reviews["sample_id"].unique())

pending_ids = [
    sid for sid in mixed["sample_id"].tolist()
    if sid not in reviewed_ids
]

print("Remaining articles:", len(pending_ids))

for sample_id in pending_ids:
    row = mixed[mixed["sample_id"] == sample_id].iloc[0]

    print("\n" + "#" * 90)
    print("ARTICLE ID:", row["sample_id"])
    print("TITLE:", row["title"])

    for col in ["hindi_article", "mixed_10_article",
                "mixed_30_article", "mixed_50_article"]:
        print("\n" + "=" * 70)
        print(col.upper())
        print("=" * 70)
        print(row[col])

Remaining articles: 1

##########################################################################################
ARTICLE ID: 080830_orissa_violence
TITLE: उड़ीसा में फिर हिंसा, कोरापुट में कर्फ़्यू

HINDI_ARTICLE
उड़ीसा में विश्व हिंदू परिषद के नेता स्वामी लक्ष्मणानंद की हत्या के बाद स्थिति तनावपूर्ण है. कोरापुट ज़िला हिंसक गतिविधियों से दूर था लेकिन शनिवार को स्थानीय विक्रम देव कॉलेज की दीवारों पर समुदाय विशेष के ख़िलाफ़ लिखी गई आपत्तिजनक बातों को लेकर तनाव फैल गया. इसके बाद दो समुदायों के बीच झड़प शुरु हो गई. झड़प में शामिल भीड़ को तितर-बितर करने के लिए पुलिस ने पहले लाठीचार्ज किया और आंसू गैस के गोले छोड़े. इससे उग्र भीड़ को नियंत्रित करने में विफल रहने पर पुलिस ने हवा में गोलियां चलाईं. कर्फ़्यू कोरापुट के पुलिस अधीक्षक दीपक कुमार ने बताया कि शहर में कर्फ़्यू लगा दिया गया है. वो ख़ुद इस झड़प में घायल हो गए हैं. हिंसक भीड़ ने पुलिस स्टेशन पर भी हमला किया और फर्नीचर को नुकसान पहुंचाया. समस्या उस समय शुरू हुई जब दीवारों पर लिखी बातों से नाराज़ युवकों ने दूसरे समुदाय की छात्रओं पर हमल

In [472]:
new_reviews = [
    {
        "sample_id": "080830_orissa_violence",
        "Fluency_10": "Fair",
        "Meaning_10": "Yes",
        "Fluency_30": "Good",
        "Meaning_30": "Yes",
        "Fluency_50": "Good",
        "Meaning_50": "Yes",
        "Reviewer_Notes": "Main events, names and allegations are preserved in the translated passages. Details about police action, injuries to students, the government's position and possible Maoist involvement remain partly or entirely in Hindi."
    }
]

reviews = pd.concat(
    [reviews, pd.DataFrame(new_reviews)],
    ignore_index=True
)

reviews = reviews.drop_duplicates(
    subset=["sample_id"],
    keep="last"
)

reviews.to_csv(review_path, index=False)

print("Saved successfully:", review_path)
print("Total reviewed:", reviews["sample_id"].nunique())

display(
    reviews[reviews["sample_id"] == "080830_orissa_violence"][[
        "sample_id", "Fluency_10", "Meaning_10",
        "Fluency_30", "Meaning_30",
        "Fluency_50", "Meaning_50", "Reviewer_Notes"
    ]]
)

Saved successfully: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/FactIndic_Mixed_Human_Review.csv
Total reviewed: 23


,sample_id,Fluency_10,Meaning_10,Fluency_30,Meaning_30,Fluency_50,Meaning_50,Reviewer_Notes
22,080830_orissa_violence,Fair,Yes,Good,Yes,Good,Yes,"Main events, names and allegations are preserv..."


In [474]:
missing_review = reviews[
    reviews["Fluency_10"].isna()
    | reviews["Meaning_10"].isna()
    | reviews["Fluency_30"].isna()
    | reviews["Meaning_30"].isna()
    | reviews["Fluency_50"].isna()
    | reviews["Meaning_50"].isna()
]

print("Records needing review:", len(missing_review))

display(
    missing_review[
        ["sample_id", "reviewed_levels", "overall_status", "reviewer_notes"]
    ]
)

Records needing review: 5


,sample_id,reviewed_levels,overall_status,reviewer_notes
0,031114_bush_intelligence,"10%, 30%, 50%",Needs minor language review,Sentence-level switching is clear. Original Hi...
1,070902_khanduri_win,"10%, 30%, 50%",Needs minor source-language review,Sentence-level switching is clear. Election da...
2,080711_lanka_attack,"10%, 30%, 50%",Needs source-language and factual review,English replacements are generally readable. S...
3,051121_aids_india,"10%, 30%, 50%",Needs source-language review,English replacements are generally readable. H...
4,070620_heart_survival,"10%, 30%, 50%",Needs minor language and formatting review,Main study details are understandable. Preserv...


In [475]:
legacy_ids = [
    "031114_bush_intelligence",
    "070902_khanduri_win",
    "080711_lanka_attack",
    "051121_aids_india",
    "070620_heart_survival"
]

legacy_columns = [
    col for col in reviews.columns
    if col not in [
        "Fluency_10", "Meaning_10",
        "Fluency_30", "Meaning_30",
        "Fluency_50", "Meaning_50",
        "Reviewer_Notes"
    ]
]

display(
    reviews[reviews["sample_id"].isin(legacy_ids)][legacy_columns]
)

,sample_id,reviewed_levels,sentence_1_issue,sentence_7_issue,sentence_8_status,sentence_11_status,overall_status,reviewer_notes,sentence_8_issue,sentence_9_issue,...,sentence_6_issue,sentence_12_status,sentence_15_status,sentence_12_issue,sentence_13_issue,sentence_7_status,sentence_14_status,sentence_16_status,sentence_5_6_issue,sentence_11_12_issue
0,031114_bush_intelligence,"10%, 30%, 50%",Hindi grammatical construction needs review,Source typo: आसापस,Meaning appears consistent,English translation matches original Hindi mea...,Needs minor language review,Sentence-level switching is clear. Original Hi...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,070902_khanduri_win,"10%, 30%, 50%",NaN,Awkward source phrase: बहुमत हैरानी की बात,NaN,Originally Hindi; English translation preserve...,Needs minor source-language review,Sentence-level switching is clear. Election da...,Check Dhumakot / धूमाकोट spelling consistency,Hindi source wording is repetitive; verify tra...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,080711_lanka_attack,"10%, 30%, 50%",NaN,NaN,NaN,NaN,Needs source-language and factual review,English replacements are generally readable. S...,NaN,Repeated attribution; check source consistency,...,Malformed Hindi phrase: हमला मरने वालों में,English version is clearer than Hindi source w...,Verify reported figures against source,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,051121_aids_india,"10%, 30%, 50%",NaN,NaN,NaN,NaN,Needs source-language review,English replacements are generally readable. H...,NaN,NaN,...,NaN,NaN,NaN,Hindi grammar is malformed; needs source review,Awkward Hindi construction; needs source review,Verify reported percentages and state names ag...,English version preserves reported percentages...,English version is readable and preserves main...,NaN,NaN
4,070620_heart_survival,"10%, 30%, 50%",Verify study figures against source,NaN,NaN,NaN,Needs minor language and formatting review,Main study details are understandable. Preserv...,NaN,NaN,...,Awkward joining of the phrase beginning वजहें,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Quotation split across languages and sentence ...,Quotation formatting and closing quotation mar...


In [476]:
report_path = base_path + "/08_reports/FactIndic_PersonB_Final_Report.txt"

with open(report_path, "r", encoding="utf-8") as f:
    report = f.read()

section = """

7. MIXED ARTICLE HUMAN QUALITY REVIEW

- Completed human review of all 23 eligible mixed articles.
- 23 unique article IDs reviewed; no duplicate IDs.
- 5 articles retain the original sentence-level review format.
- 18 articles use the fluency and meaning review format.
- The review assessed language fluency, preservation of meaning,
  translation coverage, and source-language issues.
- Several articles retain Hindi sentences at higher mixing levels.
- Some source Hindi sentences contain grammatical or factual
  inconsistencies requiring separate source verification.
- The two review formats were preserved separately; missing
  fields were not filled with assumed ratings.

Saved file:
05_quality_checks/FactIndic_Mixed_Human_Review.csv
"""

if "7. MIXED ARTICLE HUMAN QUALITY REVIEW" not in report:
    report += section

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)

print("Report updated:", os.path.exists(report_path))
print("Report path:", report_path)
print("\nAdded section:")
print(section)

Report updated: True
Report path: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/08_reports/FactIndic_PersonB_Final_Report.txt

Added section:


7. MIXED ARTICLE HUMAN QUALITY REVIEW

- Completed human review of all 23 eligible mixed articles.
- 23 unique article IDs reviewed; no duplicate IDs.
- 5 articles retain the original sentence-level review format.
- 18 articles use the fluency and meaning review format.
- The review assessed language fluency, preservation of meaning,
  translation coverage, and source-language issues.
- Several articles retain Hindi sentences at higher mixing levels.
- Some source Hindi sentences contain grammatical or factual
  inconsistencies requiring separate source verification.
- The two review formats were preserved separately; missing
  fields were not filled with assumed ratings.

Saved file:
05_quality_checks/FactIndic_Mixed_Human_Review.csv



In [477]:
import os
import pandas as pd

base_path = "/content/drive/MyDrive/FactIndic_PersonB_CleanRun"

review_file = (
    base_path +
    "/05_quality_checks/Human_Translation_Review_20.csv"
)

translation_file = (
    base_path +
    "/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv"
)

print("REVIEW FILE EXISTS:", os.path.exists(review_file))
print("TRANSLATION FILE EXISTS:", os.path.exists(translation_file))

if os.path.exists(review_file):
    translation_review = pd.read_csv(review_file)

    print("\nREVIEW FILE COLUMNS:")
    print(translation_review.columns.tolist())

    print("\nFIRST 5 REVIEW RECORDS:")
    display(translation_review.head())

if os.path.exists(translation_file):
    english_translations = pd.read_csv(translation_file)

    print("\nTRANSLATION FILE COLUMNS:")
    print(english_translations.columns.tolist())

    print("\nFIRST 5 TRANSLATION RECORDS:")
    display(english_translations.head())

REVIEW FILE EXISTS: True
TRANSLATION FILE EXISTS: True

REVIEW FILE COLUMNS:
['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words', 'nllb_english_title', 'nllb_english_article', 'nllb_english_summary', 'english_title', 'english_text', 'english_summary', 'hindi_cmi', 'english_cmi', 'hi_words', 'en_words', 'translation_ratio', 'review_status', 'translation_quality', 'names_correct', 'numbers_correct', 'meaning_preserved', 'comments']

FIRST 5 REVIEW RECORDS:


,id,url,title,summary,text,article_words,summary_words,nllb_english_title,nllb_english_article,nllb_english_summary,...,english_cmi,hi_words,en_words,translation_ratio,review_status,translation_quality,names_correct,numbers_correct,meaning_preserved,comments
0,international-44164669,https://www.bbc.com/hindi/international-44164669,"अमरीका और उत्तर कोरिया: कभी हां, कभी ना",अमरीका और उत्तर कोरिया के बीच सिंगापुर में 12 ...,राष्ट्रपति डोनल्ड ट्रंप से बातचीत की हामी भरने...,695,24,NaN,NaN,NaN,...,99.22,695,387,0.556835,Reviewed - needs retranslation,Poor,No,No,No,"Severe mistranslation of names, terminology an..."
1,140702_wc2014_goldenboot_contenders_ar,https://www.bbc.com/hindi/sport/2014/07/140702...,विश्व कप: किसको मिलेगा गोल्डन बूट?,विश्व कप फ़ुटबॉल टूर्नामेंट में एक तरफ़ अब इंत...,इनमें कोलंबिया के जेम्स रॉड्रिगेज़ सबसे आगे है...,239,40,NaN,NaN,NaN,...,99.35,239,153,0.640167,Reviewed - needs retranslation,Poor,No,No,No,"Player names, goal statistics and Golden Boot ..."
2,121114_international_us_plus_obama_ia,https://www.bbc.com/hindi/international/2012/1...,राष्ट्रीय सुरक्षा को कोई ख़तरा नहीं: ओबामा,अमरीकी राष्ट्रपति बराक ओबामा ने कहा है कि अमरी...,प्रेम संबंधों के कारण सीआईए प्रमुख को इस्तीफ़ा...,790,33,NaN,NaN,NaN,...,100.00,790,439,0.555696,Reviewed - needs retranslation,Poor,No,No,No,Severe mistranslation of names and major secti...
3,040511_shahbaz_profile,https://www.bbc.com/hindi/regionalnews/story/2...,शहबाज़ शरीफ़:राजनीति और निर्वासन,मियाँ मोहम्मद शहबाज़ शरीफ़ का संबंध पाकिस्तनी ...,उनके बड़े भाई मियाँ नवाज़ शरीफ़ पाकिस्तान के द...,254,21,NaN,NaN,NaN,...,95.71,254,163,0.641732,Reviewed - needs retranslation,Poor,No,No,No,"Names, political roles, dates and historical d..."
4,science-41399809,https://www.bbc.com/hindi/science-41399809,मोटापा कम करने का ये तरीका हो सकता है ख़तरनाक,मोटापा कम करने या दूसरे शब्दों में कहें तो शरी...,डॉक्टरों ने लिपोसक्शन को लेकर कुछ गंभीर चिंताए...,410,24,NaN,NaN,NaN,...,100.00,410,214,0.521951,Reviewed - needs retranslation,Poor,No,No,No,Severe mistranslation of liposuction and medic...



TRANSLATION FILE COLUMNS:
['id', 'complete_english_translation', 'reconstructed_sentence_count', 'manual_review_status']

FIRST 5 TRANSLATION RECORDS:


,id,complete_english_translation,reconstructed_sentence_count,manual_review_status
0,000001_ugc,You sent this material because of an incident ...,6,Human reviewed - no correction
1,031031_russia_removal,He is reported to have resigned following the ...,9,Manually corrected
2,031031_sharon_palestine,"By extending this invitation, Israel has chang...",8,Manually corrected
3,031113_tdp_elections,The meeting was attended by about seven and a ...,10,Human reviewed - no correction
4,031114_bush_intelligence,These operations have been launched in the wak...,11,Human reviewed - no correction


In [478]:
article_id = translation_review.iloc[0]["id"]

row = translation_review[
    translation_review["id"] == article_id
].iloc[0]

print("ARTICLE ID:", article_id)
print("\nHINDI TITLE:")
print(row["title"])

print("\nHINDI ARTICLE:")
print(row["text"])

print("\nCURRENT ENGLISH TRANSLATION:")
print(row["english_text"])

print("\nNLLB ENGLISH TRANSLATION:")
print(row["nllb_english_article"])

match = english_translations[
    english_translations["id"] == article_id
]

print("\nSENTENCE-LEVEL TRANSLATION MATCHES:", len(match))

if len(match) > 0:
    print("\nSENTENCE-LEVEL ENGLISH TRANSLATION:")
    print(match.iloc[0]["complete_english_translation"])

ARTICLE ID: international-44164669

HINDI TITLE:
अमरीका और उत्तर कोरिया: कभी हां, कभी ना

HINDI ARTICLE:
राष्ट्रपति डोनल्ड ट्रंप से बातचीत की हामी भरने के बाद उत्तर कोरियाई नेता किम जोंग उन ने कहा है कि अमरीका जिस तरह की चालें चल रहा है, उस माहौल में बातचीत नहीं हो सकती. बुधवार को उत्तर कोरिया ने चेतावनी दी कि वो हो सकता है कि वो इस बातचीत का हिस्सा ना बने. ट्रंप की मुश्किल उनके ही राष्ट्रीय सुरक्षा सलाहकार जॉन बोल्टन ने बढ़ा दी है. जॉन बोल्टन का कहना है कि अमरीका उत्तर कोरिया में भी "लीबिया मॉडल" अपनाने की सोच रहा है. क्या ट्रंप की बढ़ती मांगों से नाराज़ है उत्तर कोरिया उत्तर कोरिया की ट्रंप के साथ बैठक रद्द करने की धमकी 'लीबिया मॉडल' पर नाराज़ किम जोंग-उन साल 2003 में लीबिया के नेता कर्नल मुअम्मर गद्दाफ़ी परमाणु कार्यक्रम छोड़ने के लिए तैयार हो गए थे और बदले में अमरीका ने लीबिया पर लगी अधिकतर पाबंदियां हटा दी थी. लेकिन साल 2011 में पश्चिमी देशों के समर्थन से विद्रोहियों ने उनका तख्तापलट कर दिया जिसके बाद उनकी हत्या कर दी गई. ट्रंप ने कहा है कि उत्तर कोरिया के साथ वो "लीबिया मॉडल" अपन

In [479]:
article_id = "international-44164669"

match = english_translations[
    english_translations["id"] == article_id
]

if len(match) != 1:
    print("ERROR: Expected exactly one matching translation.")
else:
    candidate = match.iloc[0]["complete_english_translation"]

    print("Article:", article_id)
    print("Translation review status:",
          match.iloc[0]["manual_review_status"])
    print("Candidate translation available:", pd.notna(candidate))
    print("Candidate word count:", len(str(candidate).split()))

    correction_path = (
        base_path +
        "/09_sentence_level_translation/FactIndic_Translation_Corrections_Draft.csv"
    )

    if os.path.exists(correction_path):
        corrections = pd.read_csv(correction_path)
    else:
        corrections = pd.DataFrame(
            columns=["id", "corrected_english_translation", "correction_status"]
        )

    print("Correction draft file:", correction_path)

Article: international-44164669
Translation review status: Manually corrected
Candidate translation available: True
Candidate word count: 583
Correction draft file: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_Translation_Corrections_Draft.csv


In [480]:
article_id = "international-44164669"

candidate = english_translations.loc[
    english_translations["id"] == article_id,
    "complete_english_translation"
].iloc[0]

new_record = pd.DataFrame([{
    "id": article_id,
    "corrected_english_translation": candidate,
    "correction_status": "Candidate selected - needs final verification"
}])

corrections = pd.concat(
    [corrections, new_record],
    ignore_index=True
)

corrections = corrections.drop_duplicates(
    subset=["id"],
    keep="last"
)

corrections.to_csv(correction_path, index=False)

print("Saved:", os.path.exists(correction_path))
print("Correction records:", len(corrections))
print(corrections[["id", "correction_status"]])

Saved: True
Correction records: 1
                       id                              correction_status
0  international-44164669  Candidate selected - needs final verification


In [481]:
batch_ids = [
    "140702_wc2014_goldenboot_contenders_ar",
    "121114_international_us_plus_obama_ia"
]

for article_id in batch_ids:
    row = translation_review[
        translation_review["id"] == article_id
    ].iloc[0]

    print("\n" + "=" * 100)
    print("ARTICLE ID:", article_id)
    print("HINDI TITLE:", row["title"])

    print("\nHINDI ARTICLE:")
    print(row["text"])

    print("\nCURRENT ENGLISH:")
    print(row["english_text"])

    match = english_translations[
        english_translations["id"] == article_id
    ]

    print("\nSENTENCE-LEVEL MATCHES:", len(match))

    if len(match) > 0:
        print("\nSENTENCE-LEVEL ENGLISH:")
        print(match.iloc[0]["complete_english_translation"])


ARTICLE ID: 140702_wc2014_goldenboot_contenders_ar
HINDI TITLE: विश्व कप: किसको मिलेगा गोल्डन बूट?

HINDI ARTICLE:
इनमें कोलंबिया के जेम्स रॉड्रिगेज़ सबसे आगे हैं जिन्होंने अभी तक सर्वाधिक 5 गोल किए हैं. इसके अलावा जर्मनी के थॉमस मूलर और अर्जेंटीना के स्टार स्ट्राइकर लियोनल मैसी और ब्राज़ील के नेमार ने चार-चार गोल किए है. इस विश्व कप में तीन गोल करने वाले खिलाड़ियों मे इक्वाडोर के इनेर वेलेंशिया, फ्रांस के करीम बेंजेमा, नीदरलैंड्स के रॉबिन वॉन पर्सी और आर्येन रॉब्बेन के अलावा स्विटज़रलैंड के शक़ीरी भी शामिल हैं. संभावनाएं जाने-माने फुटबॉल समीक्षक नोवी कपाड़िया का मानना है कि एक-दो अपवाद को छोड़ दिया जाए तो जो टीम कम से कम सेमीफाइनल तक पहुंचती है, उसी टीम के खिलाड़ी को गोल्डन बूट का पुरस्कार मिलने की संभावनाएं अधिक होती हैं क्योंकि उस टीम को कम से कम सात मैच खेलने का अवसर मिलता है. इस विश्व कप में अब तक कोलंबिया के जेम्स रॉड्रिगेज़ ने सबसे ज़्यादा गोल किए हैं. विश्व कप से फ़ीफ़ा के नियम के तहत अगर एक से ज़्यादा खिलाड़ियों ने बराबर गोल किए हैं, तब गोल्डन बूट के विजेता के लिए उस खिलाड़ी 

In [482]:
batch_ids = [
    "international-44164669",
    "140702_wc2014_goldenboot_contenders_ar",
    "121114_international_us_plus_obama_ia"
]

new_records = []

for article_id in batch_ids:
    match = english_translations[
        english_translations["id"] == article_id
    ]

    if len(match) != 1:
        print("MISSING OR DUPLICATE:", article_id)
        continue

    candidate = match.iloc[0]["complete_english_translation"]

    if pd.isna(candidate) or not str(candidate).strip():
        print("EMPTY TRANSLATION:", article_id)
        continue

    new_records.append({
        "id": article_id,
        "corrected_english_translation": candidate,
        "correction_status": "Candidate selected - needs final verification"
    })

if new_records:
    corrections = pd.concat(
        [corrections, pd.DataFrame(new_records)],
        ignore_index=True
    )

    corrections = corrections.drop_duplicates(
        subset=["id"],
        keep="last"
    )

    corrections.to_csv(correction_path, index=False)

print("\nSAVED:", os.path.exists(correction_path))
print("TOTAL CANDIDATES:", len(corrections))
display(corrections[["id", "correction_status"]])


SAVED: True
TOTAL CANDIDATES: 3


,id,correction_status
1,international-44164669,Candidate selected - needs final verification
2,140702_wc2014_goldenboot_contenders_ar,Candidate selected - needs final verification
3,121114_international_us_plus_obama_ia,Candidate selected - needs final verification


In [483]:
batch_ids = [
    "040511_shahbaz_profile",
    "science-41399809",
    translation_review.iloc[5]["id"]
]

for article_id in batch_ids:
    row = translation_review[
        translation_review["id"] == article_id
    ].iloc[0]

    print("\n" + "=" * 100)
    print("ARTICLE ID:", article_id)
    print("HINDI TITLE:", row["title"])

    print("\nHINDI ARTICLE:")
    print(row["text"])

    print("\nCURRENT ENGLISH:")
    print(row["english_text"])

    match = english_translations[
        english_translations["id"] == article_id
    ]

    print("\nSENTENCE-LEVEL MATCHES:", len(match))

    if len(match) > 0:
        print("\nSENTENCE-LEVEL ENGLISH:")
        print(match.iloc[0]["complete_english_translation"])


ARTICLE ID: 040511_shahbaz_profile
HINDI TITLE: शहबाज़ शरीफ़:राजनीति और निर्वासन

HINDI ARTICLE:
उनके बड़े भाई मियाँ नवाज़ शरीफ़ पाकिस्तान के दो बार प्रधानमंत्री रह चुके हैं और उन्हें अक्तूबर 1999 में तत्कालीन सेनाध्यक्ष परवेज़ मुशर्रफ़ ने सत्ता से हटा दिया था. शहबाज़ शरीफ़ ने राजनीति में अपना करियर 1993 में पंजाब विधान सभा का सदस्य चुने जाने के साथ शुरू किया और वह 1996 तक सदस्य रहे. उन्होंने जल्दी ही विधान सभा के एक सक्षम और कुशल स्पीकर की छवि बनाई और 1997 में मुख्यमंत्री बनकर 1999 तक इस पद पर रहे. शहबाज़ शरीफ़ के प्रशंसकों का कहना है कि उन्होंने अपने कार्यकाल के दौरान बहुत सी बुनियादी सुविधाएं मुहैया कराईं जिनमें बहुत सी नई सड़कें और पुल शामिल रहे हैं. पंजाब की राज्य सिविल सेवा में भी बहुत से सुधारों का श्रेय शहबाज़ शरीफ़ को जाता है. आरोप लेकिन शहबाज़ शरीफ़ के आलोचकों का कहना है कि उनके कार्यकाल के दौरान काफ़ी कुछ ख़राब भी रहा है. उन पर आरोल लगाया जाता है कि 1998 में उन्होंने पुलिस हिरासत में मौजूद पाँच संदिग्ध अपराधियों को मारे जाने के आदेश दिए थे. पिछले साल जुलाई में पाकिस्तान की 

In [484]:
batch_ids = [
    "040511_shahbaz_profile",
    "science-41399809",
    "140323_breast_cancer_screening_tk"
]

new_records = []

for article_id in batch_ids:
    match = english_translations[
        english_translations["id"] == article_id
    ]

    if len(match) != 1:
        print("MISSING OR DUPLICATE:", article_id)
        continue

    candidate = match.iloc[0]["complete_english_translation"]

    if pd.isna(candidate) or not str(candidate).strip():
        print("EMPTY TRANSLATION:", article_id)
        continue

    new_records.append({
        "id": article_id,
        "corrected_english_translation": candidate,
        "correction_status": "Candidate selected - needs final verification"
    })

if new_records:
    corrections = pd.concat(
        [corrections, pd.DataFrame(new_records)],
        ignore_index=True
    )

    corrections = corrections.drop_duplicates(
        subset=["id"],
        keep="last"
    )

    corrections.to_csv(correction_path, index=False)

print("\nSAVED:", os.path.exists(correction_path))
print("TOTAL CANDIDATES:", len(corrections))
display(corrections[["id", "correction_status"]])


SAVED: True
TOTAL CANDIDATES: 6


,id,correction_status
0,international-44164669,Candidate selected - needs final verification
1,140702_wc2014_goldenboot_contenders_ar,Candidate selected - needs final verification
2,121114_international_us_plus_obama_ia,Candidate selected - needs final verification
3,040511_shahbaz_profile,Candidate selected - needs final verification
4,science-41399809,Candidate selected - needs final verification
5,140323_breast_cancer_screening_tk,Candidate selected - needs final verification


In [485]:
batch_ids = translation_review["id"].iloc[6:9].tolist()

for article_id in batch_ids:
    row = translation_review[
        translation_review["id"] == article_id
    ].iloc[0]

    print("\n" + "=" * 100)
    print("ARTICLE ID:", article_id)
    print("HINDI TITLE:", row["title"])

    print("\nHINDI ARTICLE:")
    print(row["text"])

    print("\nCURRENT ENGLISH:")
    print(row["english_text"])

    match = english_translations[
        english_translations["id"] == article_id
    ]

    print("\nSENTENCE-LEVEL MATCHES:", len(match))

    if len(match) > 0:
        print("\nSENTENCE-LEVEL ENGLISH:")
        print(match.iloc[0]["complete_english_translation"])


ARTICLE ID: science-39280869
HINDI TITLE: क्या आपको खाने में सुपरफूड की जरूरत है?

HINDI ARTICLE:
जाहिर है इसका जवाब ना में ही है. जब आप किसी फूड साइंटिस्ट से इसके बारे में पूछेंगे तो उनका जवाब भी इनकार में होगा. भारत में खाये जाने वाले सबसे अच्छे खाने में अमूमन 65 फीसदी तक स्टार्च, 15 से 20 फीसदी प्रोटीन और 8 से 12 फीसदी के करीब वसा होता है. लेकिन ज्यादातर भारतीय 80 फीसदी स्टार्च, 7 से 8 फीसदी प्रोटीन खाते हैं और खाने में वसा की मात्रा बढ़कर 6 से 15 फीसदी हो जाती चूल्हे ने बदल दी है इनकी ज़िंदगी वैज्ञानिक तथा औद्योगिक अनुसंधान परिषद (सीएसआईआर) के सेंट्रल फूड एंड टेक्नॉलॉजिकल रिसर्च इंस्टीट्यूट (सीएफटीआरआई), मैसूर के डायरेक्टर प्रोफेसर राम राजशेखरन ने बीबीसी हिंदी को बताया, "हम न केवल ज्यादा वसा खा रहे हैं बल्कि सस्ता भी खा रहे हैं. ये पॉम ऑयल हो सकता है या ज्यादा प्रोसेस्ड किया गया तेल. हमें इसे बदलने की जरूरत है. हमें खान-पान के पुराने तौर-तरीकों की तरफ लौटने की जरूरत है जिनमें सबसे अच्छे भारतीय खाने को बढ़ावा दिया जाता था." सुपरफूड भारत के जाने-माने फूड एक्सपर्ट प्रोफेसर राजशेखरन ऐ

In [486]:
batch_ids = [
    "science-39280869",
    "160405_vert_cul_10_spectacular_museums_rd",
    "india-48662333"
]

new_records = []

for article_id in batch_ids:
    match = english_translations[
        english_translations["id"] == article_id
    ]

    if len(match) != 1:
        print("MISSING OR DUPLICATE:", article_id)
        continue

    candidate = match.iloc[0]["complete_english_translation"]

    if pd.isna(candidate) or not str(candidate).strip():
        print("EMPTY TRANSLATION:", article_id)
        continue

    new_records.append({
        "id": article_id,
        "corrected_english_translation": candidate,
        "correction_status": "Candidate selected - needs final verification"
    })

if new_records:
    corrections = pd.concat(
        [corrections, pd.DataFrame(new_records)],
        ignore_index=True
    )

    corrections = corrections.drop_duplicates(
        subset=["id"],
        keep="last"
    )

    corrections.to_csv(correction_path, index=False)

print("\nSAVED:", os.path.exists(correction_path))
print("TOTAL CANDIDATES:", len(corrections))
display(corrections[["id", "correction_status"]])


SAVED: True
TOTAL CANDIDATES: 9


,id,correction_status
0,international-44164669,Candidate selected - needs final verification
1,140702_wc2014_goldenboot_contenders_ar,Candidate selected - needs final verification
2,121114_international_us_plus_obama_ia,Candidate selected - needs final verification
3,040511_shahbaz_profile,Candidate selected - needs final verification
4,science-41399809,Candidate selected - needs final verification
5,140323_breast_cancer_screening_tk,Candidate selected - needs final verification
6,science-39280869,Candidate selected - needs final verification
7,160405_vert_cul_10_spectacular_museums_rd,Candidate selected - needs final verification
8,india-48662333,Candidate selected - needs final verification


In [487]:
batch_ids = translation_review["id"].iloc[9:12].tolist()

for article_id in batch_ids:
    row = translation_review[
        translation_review["id"] == article_id
    ].iloc[0]

    print("\n" + "=" * 100)
    print("ARTICLE ID:", article_id)
    print("HINDI TITLE:", row["title"])

    print("\nHINDI ARTICLE:")
    print(row["text"])

    print("\nCURRENT ENGLISH:")
    print(row["english_text"])

    match = english_translations[
        english_translations["id"] == article_id
    ]

    print("\nSENTENCE-LEVEL MATCHES:", len(match))

    if len(match) > 0:
        print("\nSENTENCE-LEVEL ENGLISH:")
        print(match.iloc[0]["complete_english_translation"])


ARTICLE ID: international-46767468
HINDI TITLE: हत्यारे के नाम दर्जनों महिलाओं ने क्यों भेजे प्रेम पत्र?

HINDI ARTICLE:
अपनी पत्नी शैनन और दोनों बेटियों के साथ क्रिस्टोफर वॉट्स इसके तुरंत बाद वे टीवी चैनलों में यह गुज़ारिश करते हुए दिखाई दिए कि जिस किसी ने उनकी पत्नी और बेटियों को अपने कब्ज़े रखा है, वह उन्हें छोड़ दे. 33 साल के वॉट्स जब यह बात टीवी कैमरों में बोल रहे थे तो उनकी आंखें कैमरे की नज़रों से सीधे मेल नहीं खा रही थीं. शुरुआत में इस छोटी सी बात पर किसी ने ध्यान नहीं दिया. लेकिन कुछ दिन गुज़र जाने के बाद अगस्त महीने के अंत में पुलिस ने पता लगाया कि वॉट्स ने खुद अपनी पत्नी और दोनों बेटियों की हत्या कर दी थी. वॉट्स ने अपना ग़ुनाह कबूल भी कर लिया. पुलिस को बच्चियों के शव तेल के एक टैंक से मिले तो वहीं उनकी पत्नी का शव एक गड्ढे से बरामद हुआ. वॉट्स के कबूलनामे के कुछ हफ़्तों बाद उन्हें तीन हत्याओं के जुर्म में आजीवन कारावास की सज़ा सुना दी गई. जेल में मिले प्रेम पत्र क्रिस्टोफ़र वॉट्स के मुजरिम साबित होने के बाद मीडिया में उनकी बहुत आलोचना हुई. लेकिन हैरान करने वाली बात यह थी कि 

In [488]:
article_ids = [
    "international-46767468",
    "040804_kolkata_reaction",
    "india-55465758"
]

new_records = []

for article_id in article_ids:

    match = english_translations[
        english_translations["id"] == article_id
    ]

    if len(match) == 0:
        print("MISSING:", article_id)
        continue

    candidate = match.iloc[0]["complete_english_translation"]

    new_records.append({
        "id": article_id,
        "corrected_english_translation": candidate,
        "correction_status": "Candidate selected - needs final verification"
    })

new_df = pd.DataFrame(new_records)

corrections = pd.concat(
    [corrections, new_df],
    ignore_index=True
)

corrections = corrections.drop_duplicates(
    subset=["id"],
    keep="last"
)

corrections.to_csv(correction_path, index=False)

print("SAVED:", os.path.exists(correction_path))
print("TOTAL CANDIDATES:", len(corrections))
print(corrections[["id", "correction_status"]].tail(3))

SAVED: True
TOTAL CANDIDATES: 12
                         id                              correction_status
9    international-46767468  Candidate selected - needs final verification
10  040804_kolkata_reaction  Candidate selected - needs final verification
11           india-55465758  Candidate selected - needs final verification


In [489]:
batch_ids = translation_review["id"].iloc[12:15].tolist()

for article_id in batch_ids:
    row = translation_review[
        translation_review["id"] == article_id
    ].iloc[0]

    print("\n" + "=" * 100)
    print("ARTICLE ID:", article_id)
    print("HINDI TITLE:", row["title"])

    print("\nHINDI ARTICLE:")
    print(row["text"])

    print("\nCURRENT ENGLISH:")
    print(row["english_text"])

    match = english_translations[
        english_translations["id"] == article_id
    ]

    print("\nSENTENCE-LEVEL MATCHES:", len(match))

    if len(match) > 0:
        print("\nSENTENCE-LEVEL ENGLISH:")
        print(match.iloc[0]["complete_english_translation"])


ARTICLE ID: 150802_varun_gandhi_death_ia
HINDI TITLE: सभ्य देश में जल्लाद, लज्जा की बात: वरुण

HINDI ARTICLE:
वरूण गांधी ने ये लेख ऐसे समय में लिखा है जब 1993 में हुए मुंबई धमाकों के दोषी याक़ूब मेमन को फ़ासी दी गई है और मौत की सज़ा के मुद्दे पर भारत में ख़ासी बहस हो रही है. इससे पहले भाजपा के ही एक दूसरे सांसद और फ़िल्म अभिनेता शत्रुघ्न सिंहा ने याक़ूब मेमन की दया याचिका पर दस्तख़्त किए थे. इस पर प्रतिक्रिया देते हुए केंद्रीय वित्त मंत्री अरूण जेटली ने कहा था कि ऐसा करके शत्रुध्न सिंहा ने पार्टी को शर्मिंदा किया है. शत्रुघ्न पहले भी विवादित बयान दे चुके हैं. जेटली का कहना था, ''मेरा मानना है कि यह बहुत दुखद है कि भाजपा का एक सदस्य ऐसी याचिका पर हस्ताक्षर करता है.'' 'सामाजिक-आर्थिक भेदभाव' वरूण गांधी ने अपने लेख में याक़ूब मेमन की फांसी का कोई ज़िक्र नहीं किया है. लेकिन उन्होंने लिखा है, "किसी भी सभ्य देश में जल्लाद का होना लज्जा की बात है...." समाप्त उन्होंने इस बात पर ज़ोर दिया है कि फांसी की सज़ा पाने वालों में सामाजिक-आर्थिक भेदभाव साफ़ नज़र आता है. अमरीका का ज़िक्र करते हुए वरूण 

In [490]:
article_ids = [
    "150802_varun_gandhi_death_ia",
    "131202_saina_sindhu_badminton_vr",
    "131126_assam_protes_vs"
]

new_records = []

for article_id in article_ids:
    match = english_translations[
        english_translations["id"] == article_id
    ]

    if len(match) != 1:
        print("MISSING OR DUPLICATE:", article_id)
        continue

    candidate = match.iloc[0]["complete_english_translation"]

    if pd.isna(candidate) or not str(candidate).strip():
        print("EMPTY TRANSLATION:", article_id)
        continue

    new_records.append({
        "id": article_id,
        "corrected_english_translation": candidate,
        "correction_status": "Candidate selected - needs final verification"
    })

if new_records:
    corrections = pd.concat(
        [corrections, pd.DataFrame(new_records)],
        ignore_index=True
    )

    corrections = corrections.drop_duplicates(
        subset=["id"],
        keep="last"
    )

    corrections.to_csv(correction_path, index=False)

print("SAVED:", os.path.exists(correction_path))
print("TOTAL CANDIDATES:", len(corrections))
display(corrections[["id", "correction_status"]].tail(3))

SAVED: True
TOTAL CANDIDATES: 15


,id,correction_status
12,150802_varun_gandhi_death_ia,Candidate selected - needs final verification
13,131202_saina_sindhu_badminton_vr,Candidate selected - needs final verification
14,131126_assam_protes_vs,Candidate selected - needs final verification


In [491]:
batch_ids = translation_review["id"].iloc[15:18].tolist()

for article_id in batch_ids:
    row = translation_review[
        translation_review["id"] == article_id
    ].iloc[0]

    print("\n" + "=" * 100)
    print("ARTICLE ID:", article_id)
    print("HINDI TITLE:", row["title"])

    print("\nHINDI ARTICLE:")
    print(row["text"])

    print("\nCURRENT ENGLISH:")
    print(row["english_text"])

    match = english_translations[
        english_translations["id"] == article_id
    ]

    print("\nSENTENCE-LEVEL MATCHES:", len(match))

    if len(match) > 0:
        print("\nSENTENCE-LEVEL ENGLISH:")
        print(match.iloc[0]["complete_english_translation"])


ARTICLE ID: 061219_spl_sez_orissa
HINDI TITLE: पोस्को के ख़िलाफ़ बढ़ रहा विरोध

HINDI ARTICLE:
विभिन्न संगठनों ने पोस्को के विरोध के लिए बाकायदा 'पोस्को प्रतिरोध संग्राम समिति' का गठन किया है. पारादीप में बनने वाले इस एसईजेड के लिए केंद्र सरकार से अंतिम मंजूरी मिलना बाकी है. हालाँकि सैद्धांतिक सहमति मिल चुकी है. योजना के मुताबिक पोस्को का एसईजेड चार हज़ार चार एकड़ क्षेत्र में बनेगा. इनमें से 1135 एकड़ ज़मीन सरकार की है लेकिन मुख्य समस्या आम लोगों की ज़मीनों के अधिग्रहण में हो रही है. पोस्को के प्रस्तावित संयंत्र से सालाना 20 लाख टन इस्पात का उत्पादन होगा. प्रस्ताव राज्य सरकार ने पोस्को सहित 14 एसईजेड प्रस्तावों को हरी झंडी दी है. इनमें पाँच प्रस्तावों को केंद्र सरकार की औपचारिक मंजूरी मिल चुकी है. टाटा स्टील गोपालपुर में 1173 एकड़ भूभाग में एसईजेड बनाने की योजना बना रही है लेकिन अभी इसे मंजूरी नहीं मिली है. आकलन के मुताबिक सभी 14 एसईजेड बनने से 26 हज़ार एकड़ कृषि योग्य ज़मीन का नुकसान होगा. साथ ही हज़ारों लोग विस्थापित होंगे जो विवाद का अहम हिस्सा है. विरोध भारतीय कम्युनिस्ट पार्टी के

In [492]:
article_ids = [
    "061219_spl_sez_orissa",
    "121115_2g_auction_ac",
    "080514_kashmir_ceasefire"
]

new_records = []

for article_id in article_ids:
    match = english_translations[
        english_translations["id"] == article_id
    ]

    if len(match) != 1:
        print("MISSING OR DUPLICATE:", article_id)
        continue

    candidate = match.iloc[0]["complete_english_translation"]

    if pd.isna(candidate) or not str(candidate).strip():
        print("EMPTY TRANSLATION:", article_id)
        continue

    new_records.append({
        "id": article_id,
        "corrected_english_translation": candidate,
        "correction_status": "Candidate selected - needs final verification"
    })

if new_records:
    corrections = pd.concat(
        [corrections, pd.DataFrame(new_records)],
        ignore_index=True
    )

    corrections = corrections.drop_duplicates(
        subset=["id"],
        keep="last"
    )

    corrections.to_csv(correction_path, index=False)

print("SAVED:", os.path.exists(correction_path))
print("TOTAL CANDIDATES:", len(corrections))
display(corrections[["id", "correction_status"]].tail(3))

SAVED: True
TOTAL CANDIDATES: 18


,id,correction_status
15,061219_spl_sez_orissa,Candidate selected - needs final verification
16,121115_2g_auction_ac,Candidate selected - needs final verification
17,080514_kashmir_ceasefire,Candidate selected - needs final verification


In [493]:
batch_ids = translation_review["id"].iloc[18:20].tolist()

for article_id in batch_ids:
    row = translation_review[
        translation_review["id"] == article_id
    ].iloc[0]

    print("\n" + "=" * 100)
    print("ARTICLE ID:", article_id)
    print("HINDI TITLE:", row["title"])

    print("\nHINDI ARTICLE:")
    print(row["text"])

    print("\nCURRENT ENGLISH:")
    print(row["english_text"])

    match = english_translations[
        english_translations["id"] == article_id
    ]

    print("\nSENTENCE-LEVEL MATCHES:", len(match))

    if len(match) > 0:
        print("\nSENTENCE-LEVEL ENGLISH:")
        print(match.iloc[0]["complete_english_translation"])


ARTICLE ID: 040423_un_poverty_report
HINDI TITLE: ग़रीबी दूर करने के वायदे अधूरेः रिपोर्ट

HINDI ARTICLE:
इन लक्ष्यों को 2015 तक पूरा किया जाना था. मगर अब विश्व बैंक और अंतरराष्ट्रीय मुद्रा कोष ने एक साझा रिपोर्ट जारी की है जिसमें विकासशील देशों के प्रदर्शन पर टिप्पणी की गई है. इन लक्ष्यों को 'सहस्राब्दी विकास लक्ष्य' का नाम दिया गया था. इनमें कुछ प्रमुख लक्ष्य थे - अत्यधिक ग़रीब लोगों की संख्या आधी करना, सभी बच्चों को प्राथमिक शिक्षा और प्रसव के समय माँ और बच्चों की होनेवाली मौतों की संख्या कम करना. ज़रूरत रिपोर्ट में कहा गया है कि इन लक्ष्यों की पूर्ति के लिए ये ज़रूरी है कि अमीर देश और आर्थिक सहयोग के लिए दो साल पहले किए गए अपने वायदे को पूरा करें. विश्व बैंक के प्रमुख जेम्स टी वोल्फ़ेंसन ने कहा है,"हम जितनी बात बनाना चाहें बना सकते हैं मगर जब तक हम बुनियादी प्रश्न का सामना नहीं करते तब तक हम लक्ष्य से दूर ही रहेंगे". इससे पहले बुधवार को विश्व आर्थिक मंच ने भी अपनी रिपोर्ट प्रकाशित की थी जिसमें कुछ इसी तरह की बातें थीं जैसी की विश्व बैंक और अंतरराष्ट्रीय मुद्रा कोष की रिपोर्ट में ह

In [494]:
article_ids = [
    "040423_un_poverty_report",
    "040304_robot_translator"
]

new_records = []

for article_id in article_ids:
    match = english_translations[
        english_translations["id"] == article_id
    ]

    if len(match) != 1:
        print("MISSING OR DUPLICATE:", article_id)
        continue

    candidate = match.iloc[0]["complete_english_translation"]

    if pd.isna(candidate) or not str(candidate).strip():
        print("EMPTY TRANSLATION:", article_id)
        continue

    new_records.append({
        "id": article_id,
        "corrected_english_translation": candidate,
        "correction_status": "Candidate selected - needs final verification"
    })

if new_records:
    corrections = pd.concat(
        [corrections, pd.DataFrame(new_records)],
        ignore_index=True
    )

    corrections = corrections.drop_duplicates(
        subset=["id"],
        keep="last"
    )

    corrections.to_csv(correction_path, index=False)

print("SAVED:", os.path.exists(correction_path))
print("TOTAL CANDIDATES:", len(corrections))
display(corrections[["id", "correction_status"]].tail(2))

SAVED: True
TOTAL CANDIDATES: 20


,id,correction_status
18,040423_un_poverty_report,Candidate selected - needs final verification
19,040304_robot_translator,Candidate selected - needs final verification


In [495]:
# Check the 20 candidate translations

print("TOTAL ROWS:", len(corrections))
print("UNIQUE ARTICLE IDs:", corrections["id"].nunique())
print("DUPLICATE IDs:", corrections["id"].duplicated().sum())

print("\nEMPTY TRANSLATIONS:")
print(
    corrections["corrected_english_translation"]
    .isna()
    .sum()
)

print("\nEMPTY OR BLANK TEXT:")
print(
    corrections["corrected_english_translation"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("\nSTATUS COUNTS:")
print(corrections["correction_status"].value_counts())

print("\nMISSING REVIEW ARTICLES:")
missing_ids = set(translation_review["id"]) - set(corrections["id"])
print(sorted(missing_ids))

print("\nEXTRA CANDIDATE IDs:")
extra_ids = set(corrections["id"]) - set(translation_review["id"])
print(sorted(extra_ids))

print("\nFILE EXISTS:", os.path.exists(correction_path))

TOTAL ROWS: 20
UNIQUE ARTICLE IDs: 20
DUPLICATE IDs: 0

EMPTY TRANSLATIONS:
0

EMPTY OR BLANK TEXT:
0

STATUS COUNTS:
correction_status
Candidate selected - needs final verification    20
Name: count, dtype: int64

MISSING REVIEW ARTICLES:
[]

EXTRA CANDIDATE IDs:
[]

FILE EXISTS: True


In [496]:
report_path = base_path + "/08_reports/FactIndic_PersonB_Final_Report.txt"

print("REPORT EXISTS:", os.path.exists(report_path))

with open(report_path, "r", encoding="utf-8") as f:
    report_content = f.read()

print("\nCURRENT REPORT END:")
print(report_content[-4000:])

REPORT EXISTS: True

CURRENT REPORT END:
ibution:
  SUPPORTED: 24
  NOT_ENOUGH_INFO: 26
  CONTRADICTED: 0
- Structural validation found no missing values or duplicate IDs.
- All 50 claims were independently audited against their source articles.

Saved file:
07_claim_labels/FactIndic_50_Claim_Labeling_Sheet.csv


7. CMI COMPARISON

Per-article FactIndic CMI across 23 articles:

Version                  Mean CMI       Median CMI
Original Hindi             0.00%           0.00%
10% target                 9.77%           8.79%
30% target                29.40%          27.27%
50% target                49.97%          47.29%

Existing comparison datasets:
MUTANT:
100 records; mean CMI 6.26%; median CMI 4.77%.

PHINC:
13,738 records; mean CMI 17.61%; median CMI 16.67%.

Saved file:
05_quality_checks/CMI_MUTANT_PHINC_Comparison.csv

CMI calculation note:
English words were identified using Latin-script character
detection. This is a script-based estimate, not a fully
language-aware linguistic

In [497]:
old_text = """1. Retranslate or substantially correct the English translations.
   The 20-article audit found all 20 sampled translations unreliable.

2. Recheck English translations before using them to generate
   summaries or evaluate factual consistency."""

new_text = """1. English translation correction:
   The initial 20-article audit found all 20 sampled translations
   unreliable.

2. Selected better sentence-level translation candidates for all
   20 audited articles and saved them separately in:
   09_sentence_level_translation/FactIndic_Translation_Corrections_Draft.csv

3. Structural validation passed:
   20 rows, 20 unique IDs, no duplicate IDs, no empty translations,
   and no missing or extra article IDs.

4. All 20 candidates remain marked as needing final bilingual
   verification. They are not yet approved final translations.

5. Recheck and approve the candidate translations before using
   them to generate summaries or evaluate factual consistency."""

if old_text not in report_content:
    print("OLD SECTION NOT FOUND. REPORT NOT CHANGED.")
else:
    updated_content = report_content.replace(old_text, new_text, 1)

    with open(report_path, "w", encoding="utf-8") as f:
        f.write(updated_content)

    print("REPORT UPDATED:", os.path.exists(report_path))

    with open(report_path, "r", encoding="utf-8") as f:
        check_content = f.read()

    print("\nUPDATED TRANSLATION SECTION:")
    start = check_content.find("1. English translation correction:")
    print(check_content[start:start + 1100])

REPORT UPDATED: True

UPDATED TRANSLATION SECTION:
1. English translation correction:
   The initial 20-article audit found all 20 sampled translations
   unreliable.

2. Selected better sentence-level translation candidates for all
   20 audited articles and saved them separately in:
   09_sentence_level_translation/FactIndic_Translation_Corrections_Draft.csv

3. Structural validation passed:
   20 rows, 20 unique IDs, no duplicate IDs, no empty translations,
   and no missing or extra article IDs.

4. All 20 candidates remain marked as needing final bilingual
   verification. They are not yet approved final translations.

5. Recheck and approve the candidate translations before using
   them to generate summaries or evaluate factual consistency.

3. Review the remaining 18 mixed articles beyond the 5-article
   human-review sample.

4. Consider word-level or intra-sentence mixing if natural
   Hinglish generation is required.

5. Claim-label audit completed: all 50 claims checked aga

In [498]:
batch_ids = corrections["id"].iloc[0:3].tolist()

for article_id in batch_ids:
    source = translation_review[
        translation_review["id"] == article_id
    ].iloc[0]

    candidate = corrections[
        corrections["id"] == article_id
    ].iloc[0]

    print("\n" + "=" * 100)
    print("ARTICLE ID:", article_id)
    print("HINDI TITLE:", source["title"])

    print("\nHINDI SOURCE:")
    print(source["text"])

    print("\nSELECTED ENGLISH CANDIDATE:")
    print(candidate["corrected_english_translation"])

    print("\nCURRENT REVIEW STATUS:")
    print(candidate["correction_status"])


ARTICLE ID: international-44164669
HINDI TITLE: अमरीका और उत्तर कोरिया: कभी हां, कभी ना

HINDI SOURCE:
राष्ट्रपति डोनल्ड ट्रंप से बातचीत की हामी भरने के बाद उत्तर कोरियाई नेता किम जोंग उन ने कहा है कि अमरीका जिस तरह की चालें चल रहा है, उस माहौल में बातचीत नहीं हो सकती. बुधवार को उत्तर कोरिया ने चेतावनी दी कि वो हो सकता है कि वो इस बातचीत का हिस्सा ना बने. ट्रंप की मुश्किल उनके ही राष्ट्रीय सुरक्षा सलाहकार जॉन बोल्टन ने बढ़ा दी है. जॉन बोल्टन का कहना है कि अमरीका उत्तर कोरिया में भी "लीबिया मॉडल" अपनाने की सोच रहा है. क्या ट्रंप की बढ़ती मांगों से नाराज़ है उत्तर कोरिया उत्तर कोरिया की ट्रंप के साथ बैठक रद्द करने की धमकी 'लीबिया मॉडल' पर नाराज़ किम जोंग-उन साल 2003 में लीबिया के नेता कर्नल मुअम्मर गद्दाफ़ी परमाणु कार्यक्रम छोड़ने के लिए तैयार हो गए थे और बदले में अमरीका ने लीबिया पर लगी अधिकतर पाबंदियां हटा दी थी. लेकिन साल 2011 में पश्चिमी देशों के समर्थन से विद्रोहियों ने उनका तख्तापलट कर दिया जिसके बाद उनकी हत्या कर दी गई. ट्रंप ने कहा है कि उत्तर कोरिया के साथ वो "लीबिया मॉडल" अपना

In [499]:
verified_ids = [
    "international-44164669",
    "140702_wc2014_goldenboot_contenders_ar",
    "121114_international_us_plus_obama_ia"
]

verification_notes = {
    "international-44164669":
        "Main meaning, names, dates and events preserved. Headlines and captions retained; minor editorial review needed.",

    "140702_wc2014_goldenboot_contenders_ar":
        "Goal counts, player names, FIFA tie-break rule and expert explanation preserved.",

    "121114_international_us_plus_obama_ia":
        "Main meaning, names, dates and policy topics preserved. Minor wording review needed."
}

for article_id in verified_ids:
    corrections.loc[
        corrections["id"] == article_id,
        "correction_status"
    ] = "Verified - minor editorial review"

    corrections.loc[
        corrections["id"] == article_id,
        "verification_notes"
    ] = verification_notes[article_id]

corrections.to_csv(correction_path, index=False)

print("UPDATED:", os.path.exists(correction_path))
print("VERIFIED IN THIS BATCH:", len(verified_ids))

display(
    corrections[
        corrections["id"].isin(verified_ids)
    ][["id", "correction_status", "verification_notes"]]
)

UPDATED: True
VERIFIED IN THIS BATCH: 3


,id,correction_status,verification_notes
0,international-44164669,Verified - minor editorial review,"Main meaning, names, dates and events preserve..."
1,140702_wc2014_goldenboot_contenders_ar,Verified - minor editorial review,"Goal counts, player names, FIFA tie-break rule..."
2,121114_international_us_plus_obama_ia,Verified - minor editorial review,"Main meaning, names, dates and policy topics p..."


In [500]:
batch_ids = corrections["id"].iloc[3:6].tolist()

for article_id in batch_ids:
    source = translation_review[
        translation_review["id"] == article_id
    ].iloc[0]

    candidate = corrections[
        corrections["id"] == article_id
    ].iloc[0]

    print("\n" + "=" * 100)
    print("ARTICLE ID:", article_id)
    print("HINDI TITLE:", source["title"])

    print("\nHINDI SOURCE:")
    print(source["text"])

    print("\nSELECTED ENGLISH CANDIDATE:")
    print(candidate["corrected_english_translation"])


ARTICLE ID: 040511_shahbaz_profile
HINDI TITLE: शहबाज़ शरीफ़:राजनीति और निर्वासन

HINDI SOURCE:
उनके बड़े भाई मियाँ नवाज़ शरीफ़ पाकिस्तान के दो बार प्रधानमंत्री रह चुके हैं और उन्हें अक्तूबर 1999 में तत्कालीन सेनाध्यक्ष परवेज़ मुशर्रफ़ ने सत्ता से हटा दिया था. शहबाज़ शरीफ़ ने राजनीति में अपना करियर 1993 में पंजाब विधान सभा का सदस्य चुने जाने के साथ शुरू किया और वह 1996 तक सदस्य रहे. उन्होंने जल्दी ही विधान सभा के एक सक्षम और कुशल स्पीकर की छवि बनाई और 1997 में मुख्यमंत्री बनकर 1999 तक इस पद पर रहे. शहबाज़ शरीफ़ के प्रशंसकों का कहना है कि उन्होंने अपने कार्यकाल के दौरान बहुत सी बुनियादी सुविधाएं मुहैया कराईं जिनमें बहुत सी नई सड़कें और पुल शामिल रहे हैं. पंजाब की राज्य सिविल सेवा में भी बहुत से सुधारों का श्रेय शहबाज़ शरीफ़ को जाता है. आरोप लेकिन शहबाज़ शरीफ़ के आलोचकों का कहना है कि उनके कार्यकाल के दौरान काफ़ी कुछ ख़राब भी रहा है. उन पर आरोल लगाया जाता है कि 1998 में उन्होंने पुलिस हिरासत में मौजूद पाँच संदिग्ध अपराधियों को मारे जाने के आदेश दिए थे. पिछले साल जुलाई में पाकिस्तान की ए

In [501]:
verification_updates = {
    "040511_shahbaz_profile": (
        "Verified - minor editorial review",
        "Names, political timeline, attributed allegations and ten-year return agreement preserved."
    ),
    "science-41399809": (
        "Verified - medical terminology review",
        "Patient details, FES, 36-hour timeline and recovery preserved. Medical terminology requires specialist review."
    ),
    "140323_breast_cancer_screening_tk": (
        "Verified - medical statistics review",
        "Study counts, risk figures, screening interval, lifestyle claims and researcher names preserved. Statistics not independently verified."
    )
}

for article_id, (status, note) in verification_updates.items():
    corrections.loc[
        corrections["id"] == article_id,
        "correction_status"
    ] = status

    corrections.loc[
        corrections["id"] == article_id,
        "verification_notes"
    ] = note

corrections.to_csv(correction_path, index=False)

print("UPDATED:", os.path.exists(correction_path))
print("VERIFIED IN THIS BATCH:", len(verification_updates))

display(
    corrections[
        corrections["id"].isin(verification_updates)
    ][["id", "correction_status", "verification_notes"]]
)

UPDATED: True
VERIFIED IN THIS BATCH: 3


,id,correction_status,verification_notes
3,040511_shahbaz_profile,Verified - minor editorial review,"Names, political timeline, attributed allegati..."
4,science-41399809,Verified - medical terminology review,"Patient details, FES, 36-hour timeline and rec..."
5,140323_breast_cancer_screening_tk,Verified - medical statistics review,"Study counts, risk figures, screening interval..."


In [502]:
batch_ids = corrections["id"].iloc[6:9].tolist()

for article_id in batch_ids:
    source = translation_review[
        translation_review["id"] == article_id
    ].iloc[0]

    candidate = corrections[
        corrections["id"] == article_id
    ].iloc[0]

    print("\n" + "=" * 100)
    print("ARTICLE ID:", article_id)
    print("HINDI TITLE:", source["title"])

    print("\nHINDI SOURCE:")
    print(source["text"])

    print("\nSELECTED ENGLISH CANDIDATE:")
    print(candidate["corrected_english_translation"])


ARTICLE ID: science-39280869
HINDI TITLE: क्या आपको खाने में सुपरफूड की जरूरत है?

HINDI SOURCE:
जाहिर है इसका जवाब ना में ही है. जब आप किसी फूड साइंटिस्ट से इसके बारे में पूछेंगे तो उनका जवाब भी इनकार में होगा. भारत में खाये जाने वाले सबसे अच्छे खाने में अमूमन 65 फीसदी तक स्टार्च, 15 से 20 फीसदी प्रोटीन और 8 से 12 फीसदी के करीब वसा होता है. लेकिन ज्यादातर भारतीय 80 फीसदी स्टार्च, 7 से 8 फीसदी प्रोटीन खाते हैं और खाने में वसा की मात्रा बढ़कर 6 से 15 फीसदी हो जाती चूल्हे ने बदल दी है इनकी ज़िंदगी वैज्ञानिक तथा औद्योगिक अनुसंधान परिषद (सीएसआईआर) के सेंट्रल फूड एंड टेक्नॉलॉजिकल रिसर्च इंस्टीट्यूट (सीएफटीआरआई), मैसूर के डायरेक्टर प्रोफेसर राम राजशेखरन ने बीबीसी हिंदी को बताया, "हम न केवल ज्यादा वसा खा रहे हैं बल्कि सस्ता भी खा रहे हैं. ये पॉम ऑयल हो सकता है या ज्यादा प्रोसेस्ड किया गया तेल. हमें इसे बदलने की जरूरत है. हमें खान-पान के पुराने तौर-तरीकों की तरफ लौटने की जरूरत है जिनमें सबसे अच्छे भारतीय खाने को बढ़ावा दिया जाता था." सुपरफूड भारत के जाने-माने फूड एक्सपर्ट प्रोफेसर राजशेखरन ऐस

In [503]:
verification_updates = {
    "science-39280869": (
        "Verified - minor editorial review",
        "Main discussion, researcher details, nutrition claims, chia and quinoa information, prices and farming figures preserved. Unrelated embedded headlines omitted."
    ),
    "160405_vert_cul_10_spectacular_museums_rd": (
        "Verified - minor editorial review",
        "All ten museums and their key historical, architectural and collection details preserved. No major meaning-changing issue found."
    ),
    "india-48662333": (
        "Verified - minor editorial review",
        "Family details, drought, water shortage, income difficulties, government response and reported costs preserved."
    )
}

for article_id, (status, note) in verification_updates.items():
    corrections.loc[
        corrections["id"] == article_id,
        "correction_status"
    ] = status

    corrections.loc[
        corrections["id"] == article_id,
        "verification_notes"
    ] = note

corrections.to_csv(correction_path, index=False)

print("UPDATED:", os.path.exists(correction_path))
print("VERIFIED IN THIS BATCH:", len(verification_updates))

display(
    corrections[
        corrections["id"].isin(verification_updates)
    ][["id", "correction_status", "verification_notes"]]
)

UPDATED: True
VERIFIED IN THIS BATCH: 3


,id,correction_status,verification_notes
6,science-39280869,Verified - minor editorial review,"Main discussion, researcher details, nutrition..."
7,160405_vert_cul_10_spectacular_museums_rd,Verified - minor editorial review,"All ten museums and their key historical, arch..."
8,india-48662333,Verified - minor editorial review,"Family details, drought, water shortage, incom..."


In [504]:
batch_ids = corrections["id"].iloc[9:12].tolist()

for article_id in batch_ids:
    source = translation_review[
        translation_review["id"] == article_id
    ].iloc[0]

    candidate = corrections[
        corrections["id"] == article_id
    ].iloc[0]

    print("\n" + "=" * 100)
    print("ARTICLE ID:", article_id)
    print("HINDI TITLE:", source["title"])

    print("\nHINDI SOURCE:")
    print(source["text"])

    print("\nSELECTED ENGLISH CANDIDATE:")
    print(candidate["corrected_english_translation"])


ARTICLE ID: international-46767468
HINDI TITLE: हत्यारे के नाम दर्जनों महिलाओं ने क्यों भेजे प्रेम पत्र?

HINDI SOURCE:
अपनी पत्नी शैनन और दोनों बेटियों के साथ क्रिस्टोफर वॉट्स इसके तुरंत बाद वे टीवी चैनलों में यह गुज़ारिश करते हुए दिखाई दिए कि जिस किसी ने उनकी पत्नी और बेटियों को अपने कब्ज़े रखा है, वह उन्हें छोड़ दे. 33 साल के वॉट्स जब यह बात टीवी कैमरों में बोल रहे थे तो उनकी आंखें कैमरे की नज़रों से सीधे मेल नहीं खा रही थीं. शुरुआत में इस छोटी सी बात पर किसी ने ध्यान नहीं दिया. लेकिन कुछ दिन गुज़र जाने के बाद अगस्त महीने के अंत में पुलिस ने पता लगाया कि वॉट्स ने खुद अपनी पत्नी और दोनों बेटियों की हत्या कर दी थी. वॉट्स ने अपना ग़ुनाह कबूल भी कर लिया. पुलिस को बच्चियों के शव तेल के एक टैंक से मिले तो वहीं उनकी पत्नी का शव एक गड्ढे से बरामद हुआ. वॉट्स के कबूलनामे के कुछ हफ़्तों बाद उन्हें तीन हत्याओं के जुर्म में आजीवन कारावास की सज़ा सुना दी गई. जेल में मिले प्रेम पत्र क्रिस्टोफ़र वॉट्स के मुजरिम साबित होने के बाद मीडिया में उनकी बहुत आलोचना हुई. लेकिन हैरान करने वाली बात यह थी कि क

In [505]:
verification_updates = {
    "international-46767468": (
        "Verified - minor editorial review",
        "Main events, names, prison letters and expert explanations preserved. Hybristophilia terminology and reported experiences accurately represented."
    ),
    "040804_kolkata_reaction": (
        "Verified - minor editorial review",
        "Public reactions, named people, opposing views on capital punishment and APDR protest preserved."
    ),
    "india-55465758": (
        "Verified - legal claim wording review",
        "Family claims, coordinates, three-acre plot, 49-year period and treaty discussion preserved. Moon land ownership remains an unverified claim, not established legal ownership."
    )
}

for article_id, (status, note) in verification_updates.items():
    corrections.loc[
        corrections["id"] == article_id,
        "correction_status"
    ] = status

    corrections.loc[
        corrections["id"] == article_id,
        "verification_notes"
    ] = note

corrections.to_csv(correction_path, index=False)

print("UPDATED:", os.path.exists(correction_path))
print("VERIFIED IN THIS BATCH:", len(verification_updates))

display(
    corrections[
        corrections["id"].isin(verification_updates)
    ][["id", "correction_status", "verification_notes"]]
)

UPDATED: True
VERIFIED IN THIS BATCH: 3


,id,correction_status,verification_notes
9,international-46767468,Verified - minor editorial review,"Main events, names, prison letters and expert ..."
10,040804_kolkata_reaction,Verified - minor editorial review,"Public reactions, named people, opposing views..."
11,india-55465758,Verified - legal claim wording review,"Family claims, coordinates, three-acre plot, 4..."


In [506]:
batch_ids = corrections["id"].iloc[12:15].tolist()

for article_id in batch_ids:
    source = translation_review[
        translation_review["id"] == article_id
    ].iloc[0]

    candidate = corrections[
        corrections["id"] == article_id
    ].iloc[0]

    print("\n" + "=" * 100)
    print("ARTICLE ID:", article_id)
    print("HINDI TITLE:", source["title"])

    print("\nHINDI SOURCE:")
    print(source["text"])

    print("\nSELECTED ENGLISH CANDIDATE:")
    print(candidate["corrected_english_translation"])


ARTICLE ID: 150802_varun_gandhi_death_ia
HINDI TITLE: सभ्य देश में जल्लाद, लज्जा की बात: वरुण

HINDI SOURCE:
वरूण गांधी ने ये लेख ऐसे समय में लिखा है जब 1993 में हुए मुंबई धमाकों के दोषी याक़ूब मेमन को फ़ासी दी गई है और मौत की सज़ा के मुद्दे पर भारत में ख़ासी बहस हो रही है. इससे पहले भाजपा के ही एक दूसरे सांसद और फ़िल्म अभिनेता शत्रुघ्न सिंहा ने याक़ूब मेमन की दया याचिका पर दस्तख़्त किए थे. इस पर प्रतिक्रिया देते हुए केंद्रीय वित्त मंत्री अरूण जेटली ने कहा था कि ऐसा करके शत्रुध्न सिंहा ने पार्टी को शर्मिंदा किया है. शत्रुघ्न पहले भी विवादित बयान दे चुके हैं. जेटली का कहना था, ''मेरा मानना है कि यह बहुत दुखद है कि भाजपा का एक सदस्य ऐसी याचिका पर हस्ताक्षर करता है.'' 'सामाजिक-आर्थिक भेदभाव' वरूण गांधी ने अपने लेख में याक़ूब मेमन की फांसी का कोई ज़िक्र नहीं किया है. लेकिन उन्होंने लिखा है, "किसी भी सभ्य देश में जल्लाद का होना लज्जा की बात है...." समाप्त उन्होंने इस बात पर ज़ोर दिया है कि फांसी की सज़ा पाने वालों में सामाजिक-आर्थिक भेदभाव साफ़ नज़र आता है. अमरीका का ज़िक्र करते हुए वरूण ग

In [507]:
verification_updates = {
    "150802_varun_gandhi_death_ia": (
        "Verified - minor wording correction",
        "Names, political statements, statistics and discrimination argument preserved. Replace 'rarest and most heinous cases' with 'only in heinous cases' to stay closer to the Hindi source."
    ),
    "131202_saina_sindhu_badminton_vr": (
        "Verified - minor editorial review",
        "Match scores, players, tournaments, rankings and results preserved. Opening sentence should avoid adding the interpretation about growing confidence."
    ),
    "131126_assam_protes_vs": (
        "Verified - minor editorial review",
        "Incident, location, hospital transfer, public protest and police response preserved. Two people being questioned are mentioned."
    )
}

for article_id, (status, note) in verification_updates.items():
    corrections.loc[
        corrections["id"] == article_id,
        "correction_status"
    ] = status

    corrections.loc[
        corrections["id"] == article_id,
        "verification_notes"
    ] = note

corrections.to_csv(correction_path, index=False)

print("UPDATED:", os.path.exists(correction_path))
print("VERIFIED IN THIS BATCH:", len(verification_updates))

display(
    corrections[
        corrections["id"].isin(verification_updates)
    ][["id", "correction_status", "verification_notes"]]
)

UPDATED: True
VERIFIED IN THIS BATCH: 3


,id,correction_status,verification_notes
12,150802_varun_gandhi_death_ia,Verified - minor wording correction,"Names, political statements, statistics and di..."
13,131202_saina_sindhu_badminton_vr,Verified - minor editorial review,"Match scores, players, tournaments, rankings a..."
14,131126_assam_protes_vs,Verified - minor editorial review,"Incident, location, hospital transfer, public ..."


In [508]:
batch_ids = corrections["id"].iloc[15:18].tolist()

for article_id in batch_ids:
    source = translation_review[
        translation_review["id"] == article_id
    ].iloc[0]

    candidate = corrections[
        corrections["id"] == article_id
    ].iloc[0]

    print("\n" + "=" * 100)
    print("ARTICLE ID:", article_id)
    print("HINDI TITLE:", source["title"])

    print("\nHINDI SOURCE:")
    print(source["text"])

    print("\nSELECTED ENGLISH CANDIDATE:")
    print(candidate["corrected_english_translation"])


ARTICLE ID: 061219_spl_sez_orissa
HINDI TITLE: पोस्को के ख़िलाफ़ बढ़ रहा विरोध

HINDI SOURCE:
विभिन्न संगठनों ने पोस्को के विरोध के लिए बाकायदा 'पोस्को प्रतिरोध संग्राम समिति' का गठन किया है. पारादीप में बनने वाले इस एसईजेड के लिए केंद्र सरकार से अंतिम मंजूरी मिलना बाकी है. हालाँकि सैद्धांतिक सहमति मिल चुकी है. योजना के मुताबिक पोस्को का एसईजेड चार हज़ार चार एकड़ क्षेत्र में बनेगा. इनमें से 1135 एकड़ ज़मीन सरकार की है लेकिन मुख्य समस्या आम लोगों की ज़मीनों के अधिग्रहण में हो रही है. पोस्को के प्रस्तावित संयंत्र से सालाना 20 लाख टन इस्पात का उत्पादन होगा. प्रस्ताव राज्य सरकार ने पोस्को सहित 14 एसईजेड प्रस्तावों को हरी झंडी दी है. इनमें पाँच प्रस्तावों को केंद्र सरकार की औपचारिक मंजूरी मिल चुकी है. टाटा स्टील गोपालपुर में 1173 एकड़ भूभाग में एसईजेड बनाने की योजना बना रही है लेकिन अभी इसे मंजूरी नहीं मिली है. आकलन के मुताबिक सभी 14 एसईजेड बनने से 26 हज़ार एकड़ कृषि योग्य ज़मीन का नुकसान होगा. साथ ही हज़ारों लोग विस्थापित होंगे जो विवाद का अहम हिस्सा है. विरोध भारतीय कम्युनिस्ट पार्टी के 

In [509]:
verification_updates = {
    "061219_spl_sez_orissa": (
        "Verified - minor editorial review",
        "Land figures, steel production, SEZ approvals, displacement concerns and statements from named officials preserved."
    ),
    "121115_2g_auction_ac": (
        "Verified - minor editorial review",
        "Auction figures, base price, participation, government revenue expectations, Bihar demand and possible tariff impact preserved."
    ),
    "080514_kashmir_ceasefire": (
        "Verified - punctuation correction",
        "Dates, location, spokesperson attribution, reported absence of casualties and flag meeting request preserved. Correct quotation boundary after 'Our forces did not retaliate.'"
    )
}

for article_id, (status, note) in verification_updates.items():
    corrections.loc[
        corrections["id"] == article_id,
        "correction_status"
    ] = status

    corrections.loc[
        corrections["id"] == article_id,
        "verification_notes"
    ] = note

corrections.to_csv(correction_path, index=False)

print("UPDATED:", os.path.exists(correction_path))
print("VERIFIED IN THIS BATCH:", len(verification_updates))

display(
    corrections[
        corrections["id"].isin(verification_updates)
    ][["id", "correction_status", "verification_notes"]]
)

UPDATED: True
VERIFIED IN THIS BATCH: 3


,id,correction_status,verification_notes
15,061219_spl_sez_orissa,Verified - minor editorial review,"Land figures, steel production, SEZ approvals,..."
16,121115_2g_auction_ac,Verified - minor editorial review,"Auction figures, base price, participation, go..."
17,080514_kashmir_ceasefire,Verified - punctuation correction,"Dates, location, spokesperson attribution, rep..."


In [510]:
batch_ids = corrections["id"].iloc[18:20].tolist()

for article_id in batch_ids:
    source = translation_review[
        translation_review["id"] == article_id
    ].iloc[0]

    candidate = corrections[
        corrections["id"] == article_id
    ].iloc[0]

    print("\n" + "=" * 100)
    print("ARTICLE ID:", article_id)
    print("HINDI TITLE:", source["title"])

    print("\nHINDI SOURCE:")
    print(source["text"])

    print("\nSELECTED ENGLISH CANDIDATE:")
    print(candidate["corrected_english_translation"])


ARTICLE ID: 040423_un_poverty_report
HINDI TITLE: ग़रीबी दूर करने के वायदे अधूरेः रिपोर्ट

HINDI SOURCE:
इन लक्ष्यों को 2015 तक पूरा किया जाना था. मगर अब विश्व बैंक और अंतरराष्ट्रीय मुद्रा कोष ने एक साझा रिपोर्ट जारी की है जिसमें विकासशील देशों के प्रदर्शन पर टिप्पणी की गई है. इन लक्ष्यों को 'सहस्राब्दी विकास लक्ष्य' का नाम दिया गया था. इनमें कुछ प्रमुख लक्ष्य थे - अत्यधिक ग़रीब लोगों की संख्या आधी करना, सभी बच्चों को प्राथमिक शिक्षा और प्रसव के समय माँ और बच्चों की होनेवाली मौतों की संख्या कम करना. ज़रूरत रिपोर्ट में कहा गया है कि इन लक्ष्यों की पूर्ति के लिए ये ज़रूरी है कि अमीर देश और आर्थिक सहयोग के लिए दो साल पहले किए गए अपने वायदे को पूरा करें. विश्व बैंक के प्रमुख जेम्स टी वोल्फ़ेंसन ने कहा है,"हम जितनी बात बनाना चाहें बना सकते हैं मगर जब तक हम बुनियादी प्रश्न का सामना नहीं करते तब तक हम लक्ष्य से दूर ही रहेंगे". इससे पहले बुधवार को विश्व आर्थिक मंच ने भी अपनी रिपोर्ट प्रकाशित की थी जिसमें कुछ इसी तरह की बातें थीं जैसी की विश्व बैंक और अंतरराष्ट्रीय मुद्रा कोष की रिपोर्ट में है

In [511]:
verification_updates = {
    "040423_un_poverty_report": (
        "Verified - minor wording correction",
        "2015 deadline, Millennium Development Goals, three key targets, World Bank and IMF report, Wolfensohn quote and Washington meeting preserved. Improve wording about rich countries fulfilling earlier economic cooperation commitments."
    ),
    "040304_robot_translator": (
        "Verified - minor editorial review",
        "English-Japanese translation, speech capability, NEC research, Pepero airport use and vocabulary figures preserved. Review wording describing NEC's Japanese and European presence."
    )
}

for article_id, (status, note) in verification_updates.items():
    corrections.loc[
        corrections["id"] == article_id,
        "correction_status"
    ] = status

    corrections.loc[
        corrections["id"] == article_id,
        "verification_notes"
    ] = note

corrections.to_csv(correction_path, index=False)

print("UPDATED:", os.path.exists(correction_path))
print("VERIFIED IN THIS BATCH:", len(verification_updates))

display(
    corrections[
        corrections["id"].isin(verification_updates)
    ][["id", "correction_status", "verification_notes"]]
)

UPDATED: True
VERIFIED IN THIS BATCH: 2


,id,correction_status,verification_notes
18,040423_un_poverty_report,Verified - minor wording correction,"2015 deadline, Millennium Development Goals, t..."
19,040304_robot_translator,Verified - minor editorial review,"English-Japanese translation, speech capabilit..."


In [512]:
import os
import pandas as pd

# Check the saved correction file
saved_corrections = pd.read_csv(correction_path)

print("=" * 65)
print("FINAL TRANSLATION VERIFICATION AUDIT")
print("=" * 65)

print("FILE EXISTS:", os.path.exists(correction_path))
print("TOTAL ROWS:", len(saved_corrections))
print("UNIQUE ARTICLE IDs:", saved_corrections["id"].nunique())
print("DUPLICATE IDs:", saved_corrections["id"].duplicated().sum())

print(
    "EMPTY TRANSLATIONS:",
    saved_corrections["corrected_english_translation"]
    .fillna("").str.strip().eq("").sum()
)

print("\nSTATUS COUNTS:")
print(saved_corrections["correction_status"].value_counts().to_string())

print("\nARTICLES STILL NEEDING VERIFICATION:")
pending = saved_corrections[
    saved_corrections["correction_status"].str.contains(
        "needs final verification", case=False, na=False
    )
]
display(pending[["id", "correction_status"]])

print("\nALL 20 ARTICLE STATUSES:")
display(saved_corrections[["id", "correction_status"]])

print("\nMISSING REVIEW ARTICLES:")
print(
    sorted(
        set(translation_review["id"])
        & set(corrections["id"])
    ) == set(translation_review["id"])
)

FINAL TRANSLATION VERIFICATION AUDIT
FILE EXISTS: True
TOTAL ROWS: 20
UNIQUE ARTICLE IDs: 20
DUPLICATE IDs: 0
EMPTY TRANSLATIONS: 0

STATUS COUNTS:
correction_status
Verified - minor editorial review        14
Verified - minor wording correction       2
Verified - medical terminology review     1
Verified - medical statistics review      1
Verified - legal claim wording review     1
Verified - punctuation correction         1

ARTICLES STILL NEEDING VERIFICATION:


,id,correction_status



ALL 20 ARTICLE STATUSES:


,id,correction_status
0,international-44164669,Verified - minor editorial review
1,140702_wc2014_goldenboot_contenders_ar,Verified - minor editorial review
2,121114_international_us_plus_obama_ia,Verified - minor editorial review
3,040511_shahbaz_profile,Verified - minor editorial review
4,science-41399809,Verified - medical terminology review
5,140323_breast_cancer_screening_tk,Verified - medical statistics review
6,science-39280869,Verified - minor editorial review
7,160405_vert_cul_10_spectacular_museums_rd,Verified - minor editorial review
8,india-48662333,Verified - minor editorial review
9,international-46767468,Verified - minor editorial review



MISSING REVIEW ARTICLES:
False


In [513]:
review_ids = set(translation_review["id"])
candidate_ids = set(saved_corrections["id"])

print("ALL 20 CANDIDATES FOUND IN REVIEW SAMPLE:",
      candidate_ids.issubset(review_ids))

print("CANDIDATES NOT FOUND:",
      sorted(candidate_ids - review_ids))

print("CANDIDATES REVIEWED:", len(candidate_ids))

ALL 20 CANDIDATES FOUND IN REVIEW SAMPLE: True
CANDIDATES NOT FOUND: []
CANDIDATES REVIEWED: 20


In [514]:
with open(report_path, "r", encoding="utf-8") as f:
    report_text = f.read()

print(report_text)

FACTINDIC NLP PROJECT
PERSON B - FINAL WORK REPORT

Project:
Factuality-aware summarization for Hindi-English code-mixed text

Course:
CSET 346 - Natural Language Processing
Team 8 - Bennett University


1. DATA COLLECTION AND ENGLISH TRANSLATION

- Sampled 250 Hindi articles from the XL-Sum Hindi dataset.
- Completed English translation for all 250 articles using IndicTrans2.
- Completed 5,229 sentence-level translation tasks.
- Saved the reviewed English translation dataset:
  09_sentence_level_translation/FactIndic_English_250_Reviewed.csv
- Translation review records:
  225 manually corrected and 25 human reviewed without correction.
- A separate 20-article quality audit was completed.
- All 20 articles in this audit were marked
  "Reviewed - needs retranslation".

Important limitation:
The 20-article audit indicates serious translation-quality problems.
The 250 translations should not be treated as fully reliable or
gold-standard translations. Further correction is required before

In [515]:

final_report = """FACTINDIC NLP PROJECT
PERSON B - FINAL WORK REPORT

Project:
Factuality-aware summarization for Hindi-English code-mixed text

Course:
CSET 346 - Natural Language Processing
Team 8 - Bennett University


1. DATA COLLECTION AND ENGLISH TRANSLATION

- Sampled 250 Hindi articles from the XL-Sum Hindi dataset.
- Completed English translation for all 250 articles using IndicTrans2.
- Completed 5,229 sentence-level translation tasks.
- Saved the reviewed English translation dataset:
  09_sentence_level_translation/FactIndic_English_250_Reviewed.csv
- Translation review records:
  225 manually corrected and 25 human reviewed without correction.

20-article translation quality audit and correction:
- The initial audit found serious quality problems in all 20 sampled
  English translations.
- Selected better sentence-level translation candidates for all
  20 articles.
- Compared all 20 selected candidates against their Hindi source
  articles and recorded verification notes.
- All 20 candidates are saved separately in:
  09_sentence_level_translation/FactIndic_Translation_Corrections_Draft.csv

Verification status:
- Minor editorial review: 14
- Minor wording correction: 2
- Medical terminology review: 1
- Medical statistics review: 1
- Legal claim wording review: 1
- Punctuation correction: 1

Structural validation:
- 20 rows and 20 unique article IDs.
- Zero duplicate IDs.
- Zero empty translations.
- All 20 candidates belong to the original review sample.
- No candidates remain marked as needing final verification.

Important limitation:
The 20 selected candidates were checked against their Hindi sources,
but some retain editorial or specialist review notes. This does not
certify them as independently validated or gold-standard translations.
The original 250-article English translation file remains unchanged.
The 250 translations should not be treated as fully reliable reference
data without further quality review.


2. SENTENCE ALIGNMENT

- Created manual Hindi-English sentence alignment records.
- Total alignment records: 1,079.
- Unique articles represented: 71.
- Duplicate sentence pairs: 0.
- Cleaned missing review-status metadata without making new
  alignment approval decisions.

Strictly approved sentence pairs:
- 408 strictly approved pairs.
- 40 articles have at least one strictly approved pair.
- 23 articles have at least 10 strictly approved pairs.

Saved files include:
- FactIndic_Manual_Sentence_Alignment.csv
- FactIndic_Strict_Approved_Sentence_Pairs.csv
- FactIndic_Eligible_Articles_10plus.csv
- FactIndic_Eligible_Approved_Sentence_Pairs.csv


3. CONTROLLED CODE-MIXING DATASET

- Selected 23 articles with at least 10 strictly approved
  sentence pairs each.
- Used 275 strictly approved Hindi-English sentence pairs.
- Preserved original sentence order.
- Retained 33 sentences in Hindi because they had no approved
  English equivalent.
- Generated 10%, 30%, and 50% target English-mixing versions.

Final corpus-level CMI:
- Original Hindi: 0.00%
- 10% target: 10.01%
- 30% target: 29.52%
- 50% target: 49.11%

Validation:
- 23 unique articles.
- 308 sentences.
- Zero invalid replacements.
- Zero missing sentences.
- Zero missing values in the validated output.

Saved dataset:
04_mixed_data/FactIndic_Mixed_23_Validated.csv

Important limitation:
The method uses sentence-level language replacement.
It does not generate natural word-level mixing within sentences.
The 23-article dataset is a validated subset, not a completed
250-article mixed dataset.


4. HUMAN REVIEW OF MIXED ARTICLES

- Completed human review of all 23 eligible mixed articles.
- 23 unique article IDs reviewed; no duplicate IDs.
- 5 articles retain the original sentence-level review format.
- 18 articles use the fluency and meaning review format.
- The review assessed language fluency, preservation of meaning,
  translation coverage, and source-language issues.
- Several articles retain Hindi sentences at higher mixing levels.
- Some source Hindi sentences contain grammatical or factual
  inconsistencies requiring separate source verification.
- The two review formats were preserved separately; missing
  fields were not filled with assumed ratings.

Saved file:
05_quality_checks/FactIndic_Mixed_Human_Review.csv


5. REAL HINGLISH DATA

- Reviewed 100 unique real Hinglish examples.
- Source distribution:
  GupShup: 44
  MUTANT: 56

Human verification:
- Verified Hinglish: 78
- Not verified as Hinglish: 22
- Meaning clear: 100
- Natural English usage: 78 Yes, 22 No

Saved files:
- 06_real_hinglish/FactIndic_Hinglish_95_Human_Review.csv
- 06_real_hinglish/FactIndic_Hinglish_Final_Review_Summary.csv

Note:
The detailed review file contains 100 records despite "95" in
its filename.


6. CLAIM LABELING

- Prepared 50 claim-label records.
- 50 unique claim IDs.
- Label distribution:
  SUPPORTED: 24
  NOT_ENOUGH_INFO: 26
  CONTRADICTED: 0
- Structural validation found no missing values or duplicate IDs.
- All 50 claims were checked against their source articles.

Saved file:
07_claim_labels/FactIndic_50_Claim_Labeling_Sheet.csv


7. CMI COMPARISON

Per-article FactIndic CMI across 23 articles:

Version                  Mean CMI       Median CMI
Original Hindi             0.00%           0.00%
10% target                 9.77%           8.79%
30% target                29.40%          27.27%
50% target                49.97%          47.29%

Existing comparison datasets:
MUTANT:
100 records; mean CMI 6.26%; median CMI 4.77%.

PHINC:
13,738 records; mean CMI 17.61%; median CMI 16.67%.

Saved file:
05_quality_checks/CMI_MUTANT_PHINC_Comparison.csv

CMI calculation note:
English words were identified using Latin-script character
detection. This is a script-based estimate, not a fully
language-aware linguistic measure. Dataset sizes and sampling
methods differ, so comparisons should be interpreted carefully.


8. METHODOLOGY AND QUALITY FILES

Mixing methodology:
08_reports/FactIndic_Mixing_Methodology.txt

Translation review:
05_quality_checks/Human_Translation_Review_20.csv

FactIndic corpus-level CMI:
05_quality_checks/FactIndic_CMI_Summary_23.csv

Per-article CMI:
05_quality_checks/FactIndic_Per_Article_CMI_23.csv

Translation correction candidates:
09_sentence_level_translation/FactIndic_Translation_Corrections_Draft.csv


9. ADDITIONAL FINAL VALIDATION

Per-article CMI:
- Saved 23 individual article CMI records.
- 23 unique articles.
- Zero missing values.

Per-article mean and median CMI:
- Original Hindi: 0.00% / 0.00%
- 10% target: 9.77% / 8.79%
- 30% target: 29.40% / 27.27%
- 50% target: 49.97% / 47.29%

Real Hinglish review:
- 100 unique examples reviewed.
- MUTANT: 56 examples.
- GupShup: 44 examples.
- Verified Hinglish: 78.
- Not verified as Hinglish: 22.
- Meaning clear: 100.
- Natural English usage: 78 Yes, 22 No.

Final CMI comparison:
05_quality_checks/CMI_MUTANT_PHINC_Comparison.csv


10. REMAINING LIMITATIONS AND RECOMMENDATIONS

1. The 20 selected English translation candidates have been checked
   against their Hindi sources, but some retain editorial or
   specialist review notes.

2. The original 250-article English translation dataset has not
   been fully quality-certified. The separate correction draft
   contains only 20 selected candidates.

3. The controlled code-mixing dataset contains 23 eligible articles,
   not 250 articles.

4. The current mixing method performs sentence-level replacement.
   Word-level or intra-sentence mixing would be needed if natural
   Hinglish generation is required.

5. CMI is estimated using Latin-script detection and is not a
   fully language-aware linguistic measure.

6. The human review formats contain two different structures.
   Their missing fields should not be interpreted as negative ratings.

7. Source Hindi inconsistencies and specialist terminology or
   statistics may require additional subject-matter review.


11. OVERALL STATUS

Person B completed substantial data preparation, translation
processing, sentence alignment, controlled mixing for a validated
23-article subset, real Hinglish review, claim labeling, mixed-article
human review, quality checks and CMI comparison.

The separate 20-article translation correction candidate set has
also been reviewed against its Hindi sources and structurally
validated.

However, translation quality remains an important limitation for
the full 250-article dataset. The current outputs and their scope
should be described transparently. Unreliable translations should
not be used as trusted reference data without further correction
and quality review.
"""

with open(report_path, "w", encoding="utf-8") as f:
    f.write(final_report)

print("REPORT UPDATED:", os.path.exists(report_path))
print("REPORT SIZE:", os.path.getsize(report_path), "bytes")

with open(report_path, "r", encoding="utf-8") as f:
    saved_report = f.read()

print("CONTAINS 20-ARTICLE VERIFICATION:",
      "All 20 candidates" in saved_report)
print("CONTAINS 23-ARTICLE HUMAN REVIEW:",
      "all 23 eligible mixed articles" in saved_report)
print("CONTAINS OLD PENDING VERIFICATION WORDING:",
      "remain marked as needing final bilingual verification" in saved_report)
print("SECTION 11 PRESENT:", "11. OVERALL STATUS" in saved_report)


REPORT UPDATED: True
REPORT SIZE: 8576 bytes
CONTAINS 20-ARTICLE VERIFICATION: True
CONTAINS 23-ARTICLE HUMAN REVIEW: True
CONTAINS OLD PENDING VERIFICATION WORDING: False
SECTION 11 PRESENT: True


In [516]:
print("DATAFRAME AVAILABILITY")
print("-" * 50)

for name in [
    "df_250",
    "df_english",
    "english_translations",
    "translation_review",
    "corrections"
]:
    if name in globals():
        obj = globals()[name]
        print(f"\n{name}:")
        print("Type:", type(obj))
        if hasattr(obj, "shape"):
            print("Shape:", obj.shape)
            print("Columns:", list(obj.columns))
    else:
        print(f"\n{name}: NOT AVAILABLE")

DATAFRAME AVAILABILITY
--------------------------------------------------

df_250:
Type: <class 'pandas.core.frame.DataFrame'>
Shape: (250, 7)
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words']

df_english: NOT AVAILABLE

english_translations:
Type: <class 'pandas.core.frame.DataFrame'>
Shape: (250, 4)
Columns: ['id', 'complete_english_translation', 'reconstructed_sentence_count', 'manual_review_status']

translation_review:
Type: <class 'pandas.core.frame.DataFrame'>
Shape: (20, 24)
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words', 'nllb_english_title', 'nllb_english_article', 'nllb_english_summary', 'english_title', 'english_text', 'english_summary', 'hindi_cmi', 'english_cmi', 'hi_words', 'en_words', 'translation_ratio', 'review_status', 'translation_quality', 'names_correct', 'numbers_correct', 'meaning_preserved', 'comments']

corrections:
Type: <class 'pandas.core.frame.DataFrame'>
Shape: (20, 4)
Columns: ['id',

In [517]:
import pandas as pd

# IDs already included in the 20-article audit
audited_ids = set(translation_review["id"].astype(str))

# Find the remaining articles
remaining_df = df_250[
    ~df_250["id"].astype(str).isin(audited_ids)
].copy()

# Match their English translations
remaining_df = remaining_df.merge(
    english_translations[["id", "complete_english_translation"]],
    on="id",
    how="left"
)

print("TOTAL ARTICLES:", len(df_250))
print("ALREADY AUDITED:", len(audited_ids))
print("REMAINING:", len(remaining_df))
print("MISSING ENGLISH TRANSLATIONS:",
      remaining_df["complete_english_translation"].isna().sum())

# Select exactly 3 articles
batch = remaining_df.head(3)

for i, (_, row) in enumerate(batch.iterrows(), 1):
    print("\n" + "=" * 90)
    print("ARTICLE", i)
    print("ID:", row["id"])
    print("\nHINDI TITLE:\n", row["title"])
    print("\nHINDI SOURCE:\n", row["text"])
    print("\nENGLISH TRANSLATION:\n",
          row["complete_english_translation"])
    print("\n" + "=" * 90)

TOTAL ARTICLES: 250
ALREADY AUDITED: 20
REMAINING: 230
MISSING ENGLISH TRANSLATIONS: 0

ARTICLE 1
ID: 080419_brchopra_festival

HINDI TITLE:
 बीआर चोपड़ा की यादगार फ़िल्मों का विशेष उत्सव

HINDI SOURCE:
 शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी." बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करत

In [518]:
import os
import pandas as pd

audit_path = base_path + "/05_quality_checks/FactIndic_Translation_Audit_230.csv"

batch1_audit = pd.DataFrame([
    {
        "id": "080419_brchopra_festival",
        "review_status": "Verified - minor editorial review",
        "meaning_preserved": "Yes",
        "names_correct": "Yes",
        "numbers_correct": "Yes",
        "correction_required": "No",
        "review_notes": "Film titles, names, years and reported statements accurately preserve the Hindi source."
    },
    {
        "id": "080613_afghan_jail",
        "review_status": "Verified - minor editorial review",
        "meaning_preserved": "Yes",
        "names_correct": "Yes",
        "numbers_correct": "Yes",
        "correction_required": "No",
        "review_notes": "Meaning and attributed statements are preserved. The differing prisoner accounts are also present in the Hindi source."
    },
    {
        "id": "130912_guinnness_record_bulgarian_swim_sr",
        "review_status": "Verified - minor wording correction",
        "meaning_preserved": "Yes",
        "names_correct": "Yes",
        "numbers_correct": "Yes",
        "correction_required": "Yes",
        "review_notes": "Change 'for consideration by Guinness World Records' to 'for official inclusion in Guinness World Records'."
    }
])

if os.path.exists(audit_path):
    existing_audit = pd.read_csv(audit_path)
    existing_audit = existing_audit[
        ~existing_audit["id"].isin(batch1_audit["id"])
    ]
    final_audit = pd.concat(
        [existing_audit, batch1_audit],
        ignore_index=True
    )
else:
    final_audit = batch1_audit.copy()

final_audit.to_csv(audit_path, index=False)

print("SAVED:", os.path.exists(audit_path))
print("TOTAL AUDITED:", len(final_audit))
print("UNIQUE IDS:", final_audit["id"].nunique())
print("\nBATCH 1 RESULTS:")
print(batch1_audit[["id", "review_status", "correction_required"]].to_string(index=False))

SAVED: True
TOTAL AUDITED: 3
UNIQUE IDS: 3

BATCH 1 RESULTS:
                                       id                       review_status correction_required
                 080419_brchopra_festival   Verified - minor editorial review                  No
                       080613_afghan_jail   Verified - minor editorial review                  No
130912_guinnness_record_bulgarian_swim_sr Verified - minor wording correction                 Yes


In [519]:
import pandas as pd

audit_path = base_path + "/05_quality_checks/FactIndic_Translation_Audit_230.csv"

audit_df = pd.read_csv(audit_path)
audited_ids = set(audit_df["id"].astype(str))

remaining_df = df_250[
    ~df_250["id"].astype(str).isin(audited_ids)
].copy()

remaining_df = remaining_df.merge(
    english_translations[
        ["id", "complete_english_translation"]
    ],
    on="id",
    how="left"
)

batch = remaining_df.head(3)

print("AUDITED SO FAR:", len(audited_ids))
print("REMAINING:", len(remaining_df))
print("NEXT BATCH:", len(batch))

for i, (_, row) in enumerate(batch.iterrows(), 4):
    print("\n" + "=" * 90)
    print("ARTICLE", i)
    print("ID:", row["id"])
    print("\nHINDI TITLE:\n", row["title"])
    print("\nHINDI SOURCE:\n", row["text"])
    print("\nENGLISH TRANSLATION:\n",
          row["complete_english_translation"])
    print("=" * 90)

AUDITED SO FAR: 3
REMAINING: 247
NEXT BATCH: 3

ARTICLE 4
ID: 050713_us_un

HINDI TITLE:
 अमरीका ने जी-4 का प्रस्ताव ठुकराया

HINDI SOURCE:
 जी-4 में भारत के साथ-साथ ब्राज़ील, जापान और जर्मनी शामिल हैं. इन देशों ने सुरक्षा परिषद के सदस्यों की संख्या 15 से बढ़ाकर 25 करने का प्रस्ताव रखा था. अमरीका का कहना है कि सदस्य देश इस मुद्दे पर बँटे हुए हैं. विदेश मंत्रालय की एक प्रवक्ता शीरीन ताहिर खेली ने बताया कि अमरीका सुरक्षा परिषद के विस्तार के लिए काम करेगा लेकिन सही समय पर सही तरीक़े से. जी-4 के देशों ने सोमवार को यह प्रस्ताव पेश किया था. जिस पर बहस चल रही है. प्रस्ताव पर मतदान कब होगा, ये अभी तय नहीं है. लेकिन इस प्रस्ताव के पारित होने के लिए दो तिहाई मतों की आवश्यकता होगी. प्रस्ताव जी-4 के देशों के प्रस्ताव में कहा गया है कि सुरक्षा परिषद में 10 और सीटें बढ़ाई जाएँ, जिनमें से छह सीटें नए स्थायी सदस्यों के लिए हो. प्रस्ताव के मुताबिक़ नए स्थायी सदस्यों में जी-4 के देशों के अलावा अफ़्रीकी देश भी शामिल होंगे. जी-4 के देश फ़िलहाल बिना वीटो के अधिकार के स्थायी सदस्यता के लिए राज़ी हैं. अमरीका

In [520]:
audit_path = base_path + "/05_quality_checks/FactIndic_Translation_Audit_230.csv"

audit_df = pd.read_csv(audit_path)
audited_ids = set(audit_df["id"].astype(str))

remaining_df = df_250[
    ~df_250["id"].astype(str).isin(audited_ids)
].copy()

remaining_df = remaining_df.merge(
    english_translations[
        ["id", "complete_english_translation"]
    ],
    on="id",
    how="left"
)

batch = remaining_df.head(20)

print("AUDITED SO FAR:", len(audited_ids))
print("REMAINING:", len(remaining_df))
print("CURRENT BATCH:", len(batch))

for i, (_, row) in enumerate(batch.iterrows(), 4):
    print("\n" + "=" * 90)
    print("ARTICLE", i)
    print("ID:", row["id"])
    print("\nHINDI TITLE:\n", row["title"])
    print("\nHINDI SOURCE:\n", row["text"])
    print("\nENGLISH TRANSLATION:\n",
          row["complete_english_translation"])
    print("=" * 90)

AUDITED SO FAR: 3
REMAINING: 247
CURRENT BATCH: 20

ARTICLE 4
ID: 050713_us_un

HINDI TITLE:
 अमरीका ने जी-4 का प्रस्ताव ठुकराया

HINDI SOURCE:
 जी-4 में भारत के साथ-साथ ब्राज़ील, जापान और जर्मनी शामिल हैं. इन देशों ने सुरक्षा परिषद के सदस्यों की संख्या 15 से बढ़ाकर 25 करने का प्रस्ताव रखा था. अमरीका का कहना है कि सदस्य देश इस मुद्दे पर बँटे हुए हैं. विदेश मंत्रालय की एक प्रवक्ता शीरीन ताहिर खेली ने बताया कि अमरीका सुरक्षा परिषद के विस्तार के लिए काम करेगा लेकिन सही समय पर सही तरीक़े से. जी-4 के देशों ने सोमवार को यह प्रस्ताव पेश किया था. जिस पर बहस चल रही है. प्रस्ताव पर मतदान कब होगा, ये अभी तय नहीं है. लेकिन इस प्रस्ताव के पारित होने के लिए दो तिहाई मतों की आवश्यकता होगी. प्रस्ताव जी-4 के देशों के प्रस्ताव में कहा गया है कि सुरक्षा परिषद में 10 और सीटें बढ़ाई जाएँ, जिनमें से छह सीटें नए स्थायी सदस्यों के लिए हो. प्रस्ताव के मुताबिक़ नए स्थायी सदस्यों में जी-4 के देशों के अलावा अफ़्रीकी देश भी शामिल होंगे. जी-4 के देश फ़िलहाल बिना वीटो के अधिकार के स्थायी सदस्यता के लिए राज़ी हैं. अम

In [521]:
print("DATAFRAME AVAILABILITY")
print("-" * 50)

for name in [
    "df_250",
    "df_english",
    "english_translations",
    "translation_review",
    "corrections"
]:
    if name in globals():
        obj = globals()[name]
        print(f"\n{name}:")
        print("Type:", type(obj))
        if hasattr(obj, "shape"):
            print("Shape:", obj.shape)
            print("Columns:", list(obj.columns))
    else:
        print(f"\n{name}: NOT AVAILABLE")

DATAFRAME AVAILABILITY
--------------------------------------------------

df_250:
Type: <class 'pandas.core.frame.DataFrame'>
Shape: (250, 7)
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words']

df_english: NOT AVAILABLE

english_translations:
Type: <class 'pandas.core.frame.DataFrame'>
Shape: (250, 4)
Columns: ['id', 'complete_english_translation', 'reconstructed_sentence_count', 'manual_review_status']

translation_review:
Type: <class 'pandas.core.frame.DataFrame'>
Shape: (20, 24)
Columns: ['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words', 'nllb_english_title', 'nllb_english_article', 'nllb_english_summary', 'english_title', 'english_text', 'english_summary', 'hindi_cmi', 'english_cmi', 'hi_words', 'en_words', 'translation_ratio', 'review_status', 'translation_quality', 'names_correct', 'numbers_correct', 'meaning_preserved', 'comments']

corrections:
Type: <class 'pandas.core.frame.DataFrame'>
Shape: (20, 4)
Columns: ['id',

In [522]:
# Prepare the remaining translation audit queue

reviewed_ids = set(translation_review["id"].astype(str))

source_data = df_250.copy()
translation_data = english_translations.copy()

source_data["id"] = source_data["id"].astype(str)
translation_data["id"] = translation_data["id"].astype(str)

remaining_audit = source_data[
    ~source_data["id"].isin(reviewed_ids)
].merge(
    translation_data[["id", "complete_english_translation",
                     "reconstructed_sentence_count",
                     "manual_review_status"]],
    on="id",
    how="left"
)

print("TOTAL SOURCE ARTICLES:", len(source_data))
print("ALREADY IN 20-ARTICLE REVIEW:", len(reviewed_ids))
print("REMAINING ARTICLES:", len(remaining_audit))
print("MISSING ENGLISH TRANSLATIONS:",
      remaining_audit["complete_english_translation"].isna().sum())
print("DUPLICATE IDS:", remaining_audit["id"].duplicated().sum())

print("\nBATCH 1 — FIRST 7 ARTICLES")
print("-" * 60)

batch_1 = remaining_audit.head(7)

for i, row in batch_1.iterrows():
    print(f"{i + 1}. {row['id']} | {row['title']}")

TOTAL SOURCE ARTICLES: 250
ALREADY IN 20-ARTICLE REVIEW: 20
REMAINING ARTICLES: 230
MISSING ENGLISH TRANSLATIONS: 0
DUPLICATE IDS: 0

BATCH 1 — FIRST 7 ARTICLES
------------------------------------------------------------
1. 080419_brchopra_festival | बीआर चोपड़ा की यादगार फ़िल्मों का विशेष उत्सव
2. 080613_afghan_jail | तालेबान ने सैकड़ों क़ैदियों को छुड़ाया
3. 130912_guinnness_record_bulgarian_swim_sr | रिकॉर्ड की ख़ातिर हाथ-पैर बांध कर की तैराकी
4. 050713_us_un | अमरीका ने जी-4 का प्रस्ताव ठुकराया
5. 120809_salman_pkp | मैं खतरनाक स्थिति में हूँ: सलमान खान
6. 080711_lanka_attack | बस पर गोलीबारी में चार की मौत
7. social-43068605 | अब पाकिस्तान पर चढ़ा प्रिया प्रकाश की कजरारी नैनों का बुखार


In [523]:
# Display Batch 1: Hindi source vs English translation

for _, row in batch_1.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("HINDI TITLE:", row["title"])

    print("\nHINDI SOURCE:")
    print(row["text"])

    print("\nENGLISH TRANSLATION:")
    print(row["complete_english_translation"])

    print("\n" + "=" * 90)


ARTICLE ID: 080419_brchopra_festival
HINDI TITLE: बीआर चोपड़ा की यादगार फ़िल्मों का विशेष उत्सव

HINDI SOURCE:
शुक्रवार शाम को शुरु हुए फ़ेस्टिवल में इंसाफ़ का तराज़ू फ़िल्म दिखाई गई. उत्सव के दौरान छोटी सी बात, नया दौर और वक़्त जैसी क्लासिक फ़िल्में देखने को मिलेंगी. खराब सेहत के कारण बीआर चोपड़ा ख़ुद समारोह में शामिल नहीं हो सके.फ़िल्मोत्सव का उदघाटन बीआर चोपड़ा के भाई और मशहूर फ़िल्मकार यश चोपड़ा ने किया. उन्होंने कहा, "मेरे भाई ने अपने हिसाब से फ़िल्में बनाईं- वो भी ऐसे समय में जब लोग ऐसी फ़िल्मों की व्यावसायिक सफलता को लेकर आशंकित रहते थे क्योंकि ये फ़िल्में कुछ हटकर होती थीं, उस समय जिस तरह की कहानियों का चलन था, बीआर चोपड़ा जी की कहानियाँ उनसे जुदा होती थी." बीआर चोपड़ा ने अपनी फ़िल्मों में विधवा पुनर्विवाह, वेश्यावृत्ति जैसे कई विषयों को उठाया. यादगार फ़िल्में 1957 में आई उनकी फ़िल्म नया दौर में नेहरू काल के आर्दश झलकते हैं तो 1958 में आई साधना वेश्यावृत्ति के मुद्दे पर समाज से सवाल करती नज़र आती है. गुमराह, हमराज़,क़ानून और निकाह जैसी फ़िल्में का नाम उनकी बनाई फ़िल्मों की सूच

In [524]:
import os
import pandas as pd

audit_path = base_path + "/09_sentence_level_translation/FactIndic_Translation_Audit_230.csv"

batch_1_results = [
    {
        "id": "080419_brchopra_festival",
        "batch_number": 1,
        "translation_status": "Verified - minor editorial review",
        "reviewer_notes": "Meaning preserved. Consider replacing 'addressed several subjects' with 'tackled several social issues'.",
        "correction_suggestion": "Tackled several social issues, including widow remarriage and prostitution."
    },
    {
        "id": "080613_afghan_jail",
        "batch_number": 1,
        "translation_status": "Verified - minor editorial review",
        "reviewer_notes": "Main meaning, names and numbers are preserved. No major correction identified.",
        "correction_suggestion": "No essential correction."
    },
    {
        "id": "130912_guinnness_record_bulgarian_swim_sr",
        "batch_number": 1,
        "translation_status": "Verified - minor wording correction",
        "reviewer_notes": "'Extremely cold' is stronger than the Hindi wording.",
        "correction_suggestion": "Change 'extremely cold' to 'very cold'."
    },
    {
        "id": "050713_us_un",
        "batch_number": 1,
        "translation_status": "Verified - punctuation correction",
        "reviewer_notes": "Missing space in 'countries'proposal'.",
        "correction_suggestion": "Change to 'countries' proposal'."
    },
    {
        "id": "120809_salman_pkp",
        "batch_number": 1,
        "translation_status": "Verified - minor editorial review",
        "reviewer_notes": "Meaning preserved. The question about future romantic roles could be phrased more naturally.",
        "correction_suggestion": "Use 'Why only action films? What about romantic roles in the future?'"
    },
    {
        "id": "080711_lanka_attack",
        "batch_number": 1,
        "translation_status": "Verified - minor editorial review",
        "reviewer_notes": "Main details and casualty figures are preserved. No major correction identified.",
        "correction_suggestion": "No essential correction."
    },
    {
        "id": "social-43068605",
        "batch_number": 1,
        "translation_status": "Verified - minor source-fidelity correction",
        "reviewer_notes": "The surname 'Varrier' is added in English but is absent from the supplied Hindi source.",
        "correction_suggestion": "Use 'Priya Prakash' instead of 'Priya Prakash Varrier'."
    }
]

new_audit = pd.DataFrame(batch_1_results)

if os.path.exists(audit_path):
    old_audit = pd.read_csv(audit_path)
    combined_audit = pd.concat(
        [old_audit, new_audit],
        ignore_index=True
    )
    combined_audit = combined_audit.drop_duplicates(
        subset=["id"], keep="last"
    )
else:
    combined_audit = new_audit

combined_audit.to_csv(audit_path, index=False)

print("AUDIT SAVED:", os.path.exists(audit_path))
print("TOTAL ARTICLES AUDITED:", len(combined_audit))
print("BATCH 1 ARTICLES SAVED:", len(new_audit))
print("DUPLICATE IDS:", combined_audit["id"].duplicated().sum())
print("\nSTATUS COUNTS:")
print(combined_audit["translation_status"].value_counts())

AUDIT SAVED: True
TOTAL ARTICLES AUDITED: 7
BATCH 1 ARTICLES SAVED: 7
DUPLICATE IDS: 0

STATUS COUNTS:
translation_status
Verified - minor editorial review              4
Verified - minor wording correction            1
Verified - punctuation correction              1
Verified - minor source-fidelity correction    1
Name: count, dtype: int64


In [525]:
# Prepare Batch 2

batch_2 = remaining_audit.iloc[7:14].copy()

print("BATCH 2 — ARTICLES 8 TO 14")
print("-" * 60)

for i, row in enumerate(batch_2.itertuples(index=False), start=8):
    print(f"{i}. {row.id} | {row.title}")

print("\nBATCH SIZE:", len(batch_2))

BATCH 2 — ARTICLES 8 TO 14
------------------------------------------------------------
8. international-51410040 | इमरान ख़ान बोले, नरेंद्र मोदी की यह आख़िरी 'ग़लती' होगी
9. india-55525807 | चीन ने युद्ध की तैयारियों के लिए रक्षा नियमों में किया संशोधन- प्रेस रिव्यू
10. international-41372477 | उत्तर कोरिया में झटके, भूकंप या विस्फोट?
11. 130404_sam_pitroda_profile | सौ से अधिक पेटेंट हैं सैम पित्रोदा के नाम
12. 150520_emergency_service_ss | 'इमरजेंसी सेवा पर फ़र्ज़ी कॉल और अश्लील बातें'
13. 081126_aamne_pachauri_alk | 'भाजपा ने प्रदेश को पीछे धकेल दिया'
14. 081118_cg_campaign_vv | छत्तीसगढ़ में दूसरे चरण का प्रचार थमा

BATCH SIZE: 7


In [526]:
# Display Batch 2: Hindi source vs English translation

for _, row in batch_2.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("HINDI TITLE:", row["title"])

    print("\nHINDI SOURCE:")
    print(row["text"])

    print("\nENGLISH TRANSLATION:")
    print(row["complete_english_translation"])

    print("\n" + "=" * 90)


ARTICLE ID: international-51410040
HINDI TITLE: इमरान ख़ान बोले, नरेंद्र मोदी की यह आख़िरी 'ग़लती' होगी

HINDI SOURCE:
इमरान ख़ान ने पीएम मोदी के अलावा चीफ़ ऑफ आर्मी स्टाफ जनरल बिपिन रावत को भी निशाने पर लिया. गुरुवार को पाकिस्तान प्रशासित कश्मीर के मीरपुर में 'कश्मीर एकता दिवस' के अगले दिन बोलते हुए इमरान ख़ान ने कहा, ''नरेंद्र मोदी आपने कहा है कि भारत 11 दिनों में पाकिस्तान को फ़तह कर लेगा. लगता है कि आपने दुनिया का इतिहास पढ़ा नहीं है. लगता है कि आपकी डिग्री जाली थी. आपने जो यह घमंडी बयान दिया है उसी को देखते हुए मैं ज़रा आपको इतिहास में ले जाता हूं. किसी से इतिहास के बारे में आप जानकारी ले लो.'' पाकिस्तानी पीएम ने कहा, ''दुनिया की दो सबसे ताक़तवर फ़ौजें नेपोलियन की फ़ौज और हिटलर की फ़ौज ने रूस पर हमला कर दिया. उस वक़्त ये दुनिया की सबसे मज़बूत फ़ौजें थीं. इन दोनों को लगा कि ये चंद हफ़्तों का गेम है और फ़तह करके वापस आ जाएंगे. इन्होंने सोचा कि रूस की सर्दी से पहले जीत हासिल कर वापस आ जाएंगे. दोनों फ़ौजे रूस की सर्दी में तबाह हो गईं. दोनों का घमंड चूर हो गया. उन्होंने भी इसे चंद हफ़

In [527]:
import os
import pandas as pd

audit_path = base_path + "/09_sentence_level_translation/FactIndic_Translation_Audit_230.csv"

batch_2_results = [
    {
        "id": "international-51410040",
        "batch_number": 2,
        "translation_status": "Verified - minor wording correction",
        "reviewer_notes": "The source says the second-highest rank after the Prophets. English says the highest rank after the Prophets.",
        "correction_suggestion": "Change to 'He will grant the second-highest rank after the Prophets.'"
    },
    {
        "id": "india-55525807",
        "batch_number": 2,
        "translation_status": "Verified - minor editorial review",
        "reviewer_notes": "Main defence-law and press-review details are preserved. No major translation issue identified.",
        "correction_suggestion": "No essential correction."
    },
    {
        "id": "international-41372477",
        "batch_number": 2,
        "translation_status": "Verified - minor editorial review",
        "reviewer_notes": "Main meaning, earthquake details and attributed statements are preserved.",
        "correction_suggestion": "No essential correction."
    },
    {
        "id": "130404_sam_pitroda_profile",
        "batch_number": 2,
        "translation_status": "Verified - source discrepancy review",
        "reviewer_notes": "Hindi opening mentions C-DAC, while a later sentence identifies C-DOT. English consistently uses C-DOT. Source inconsistency needs checking.",
        "correction_suggestion": "Check the original publication before deciding whether the opening reference should be C-DAC or C-DOT."
    },
    {
        "id": "150520_emergency_service_ss",
        "batch_number": 2,
        "translation_status": "Verified - meaning and statistics review",
        "reviewer_notes": "English adds 'fake phone numbers', which is not in Hindi. Schoolchildren sentence changes the meaning. Hindi numerical expressions are malformed.",
        "correction_suggestion": "Use 'using false names or making fake calls'; say school-going children use family members' phones to harass staff. Verify the call figures against the original publication."
    },
    {
        "id": "081126_aamne_pachauri_alk",
        "batch_number": 2,
        "translation_status": "Verified - minor editorial review",
        "reviewer_notes": "Political interview is broadly translated accurately. Keep claims attributed to the speaker.",
        "correction_suggestion": "No essential correction."
    },
    {
        "id": "081118_cg_campaign_vv",
        "batch_number": 2,
        "translation_status": "Verified - minor editorial review",
        "reviewer_notes": "Election dates, seat counts, candidate count and party details are preserved.",
        "correction_suggestion": "No essential correction."
    }
]

new_audit = pd.DataFrame(batch_2_results)

if os.path.exists(audit_path):
    old_audit = pd.read_csv(audit_path)
    combined_audit = pd.concat(
        [old_audit, new_audit],
        ignore_index=True
    ).drop_duplicates(subset=["id"], keep="last")
else:
    combined_audit = new_audit

combined_audit.to_csv(audit_path, index=False)

print("AUDIT SAVED:", os.path.exists(audit_path))
print("TOTAL ARTICLES AUDITED:", len(combined_audit))
print("BATCH 2 ARTICLES SAVED:", len(new_audit))
print("DUPLICATE IDS:", combined_audit["id"].duplicated().sum())
print("\nSTATUS COUNTS:")
print(combined_audit["translation_status"].value_counts())

AUDIT SAVED: True
TOTAL ARTICLES AUDITED: 14
BATCH 2 ARTICLES SAVED: 7
DUPLICATE IDS: 0

STATUS COUNTS:
translation_status
Verified - minor editorial review              8
Verified - minor wording correction            2
Verified - punctuation correction              1
Verified - minor source-fidelity correction    1
Verified - source discrepancy review           1
Verified - meaning and statistics review       1
Name: count, dtype: int64


In [528]:
# Prepare Batch 3

batch_3 = remaining_audit.iloc[14:21].copy()

print("BATCH 3 — ARTICLES 15 TO 21")
print("-" * 60)

for i, row in enumerate(batch_3.itertuples(index=False), start=15):
    print(f"{i}. {row.id} | {row.title}")

print("\nBATCH SIZE:", len(batch_3))

BATCH 3 — ARTICLES 15 TO 21
------------------------------------------------------------
15. 080826_blast_films | ट्रेन धमाकों की पृष्ठभूमि पर बनती फ़िल्में
16. 141007_bhutan_corruption_india_vr | भारत सिखाएगा भ्रष्टाचार मिटाने के तरीके!
17. 130207_kashmir_rockband_imroz_sa | रॉक बैंड: मुफ्ती के खिलाफ़ क़ानूनी कार्रवाई की तैयारी
18. 051105_sahubuddin_arrest | शहाबुद्दीन दिल्ली में गिरफ़्तार
19. 070206_guj_parza | परज़ानिया नहीं दिखेगी गुजरात में
20. 061206_gennext_indopak_students | छोड़ो कल की बातें, कल की बात पुरानी....
21. 141016_ranveer_removes_pant_md_part1_pkp | जब रणवीर ने उतार फेंकी पैंट!

BATCH SIZE: 7


In [529]:
import pandas as pd
import os

audit_path = base_path + "/09_sentence_level_translation/FactIndic_Translation_Audit_230.csv"

audit_df = pd.read_csv(audit_path)

completed_ids = set(audit_df["id"].astype(str))
all_ids = set(remaining_audit["id"].astype(str))

pending_ids = all_ids - completed_ids

print("TRANSLATION AUDIT STATUS")
print("-" * 45)
print("Total remaining articles:", len(all_ids))
print("Already audited:", len(completed_ids))
print("Still pending:", len(pending_ids))
print("Duplicate audit IDs:", audit_df["id"].duplicated().sum())

print("\nNEXT STEP: We will process all pending articles automatically in batches.")

TRANSLATION AUDIT STATUS
---------------------------------------------
Total remaining articles: 230
Already audited: 14
Still pending: 216
Duplicate audit IDs: 0

NEXT STEP: We will process all pending articles automatically in batches.


In [530]:
# Prepare all pending translation articles

pending_audit = remaining_audit[
    remaining_audit["id"].astype(str).isin(pending_ids)
].copy()

print("PENDING ARTICLES:", len(pending_audit))
print("\nAVAILABLE COLUMNS:")
print(pending_audit.columns.tolist())

print("\nSAMPLE ARTICLE:")
for col in pending_audit.columns:
    if col in ["id", "title", "text", "complete_english_translation"]:
        print(f"\n{col}:")
        print(str(pending_audit.iloc[0][col])[:700])

PENDING ARTICLES: 216

AVAILABLE COLUMNS:
['id', 'url', 'title', 'summary', 'text', 'article_words', 'summary_words', 'complete_english_translation', 'reconstructed_sentence_count', 'manual_review_status']

SAMPLE ARTICLE:

id:
080826_blast_films

title:
ट्रेन धमाकों की पृष्ठभूमि पर बनती फ़िल्में

text:
इन फ़िल्मों में से कुछ रिलीज़ हो रही हैं और कुछ का निर्माण कार्य चल रहा है. पिछले हफ़्ते रिलीज़ हुई निशिकांत कामथ की फिल्म 'मुंबई मेरी जान' की कहानी का आधार 11 जुलाई 2006 को मुबंई में हुए ट्रेन बम धमाके थे. इन बम धमाकों में लगभग 200 लोगों की मौत हुई थी. फिल्म के निर्देशक निशिकांत कामथ कहते हैं, "ये फ़िल्म किसी बड़ी दुर्घटना से उबरने की एक कोशिश को दर्शाती है. मुंबई मेरी जान की कहानी सिर्फ़ मुंबई या इस घटना के बारे में नहीं बल्कि किस तरह से लोग इस समस्या से जूझ रहे हैं इस बारे में है." मुंबई मेरी जान एक भावनाप्रधान फिल्म है जबकि पाँच सितंबर को रिलीज़ होने जा रही फ़िल्म 'वेडनस्डे' एक थ्रिलर है. इस फिल्म से निर्देशन की पारी शुरु करने वाले निर्देशक नीरज पांडे कहते हैं, "वेडनस्डे की कहानी उन

In [531]:
import os
import re
import pandas as pd

screen = pending_audit.copy()

def get_numbers(text):
    return set(re.findall(r'\d+(?:[.,]\d+)?', str(text)))

def get_word_count(text):
    return len(re.findall(r'\b[\w]+\b', str(text)))

def screen_article(row):
    hindi = str(row["text"])
    english = str(row["complete_english_translation"])
    flags = []

    if not hindi.strip() or hindi == "nan":
        flags.append("Missing Hindi source")

    if not english.strip() or english == "nan":
        flags.append("Missing English translation")

    if re.search(r'[\u0900-\u097F]', english):
        flags.append("Devanagari text in English")

    hindi_numbers = get_numbers(hindi)
    english_numbers = get_numbers(english)

    missing_numbers = hindi_numbers - english_numbers
    if missing_numbers:
        flags.append("Check numbers: " + ", ".join(sorted(missing_numbers)))

    h_words = get_word_count(hindi)
    e_words = get_word_count(english)

    ratio = e_words / h_words if h_words else 0

    if ratio < 0.35 or ratio > 2.5:
        flags.append("Unusual translation length")

    h_sentences = len(re.findall(r'[।.!?]+', hindi))
    e_sentences = len(re.findall(r'[.!?]+', english))

    if h_sentences and abs(h_sentences - e_sentences) > max(5, h_sentences * 0.5):
        flags.append("Check sentence structure")

    return pd.Series({
        "hindi_word_count": h_words,
        "english_word_count": e_words,
        "length_ratio": round(ratio, 2),
        "hindi_numbers": ", ".join(sorted(hindi_numbers)),
        "english_numbers": ", ".join(sorted(english_numbers)),
        "automatic_flags": " | ".join(flags) if flags else "No automatic flags",
        "review_status": "Needs human review"
    })

screen_results = screen.apply(screen_article, axis=1)

screen = pd.concat([screen.reset_index(drop=True),
                    screen_results.reset_index(drop=True)], axis=1)

screen_path = base_path + "/09_sentence_level_translation/FactIndic_Translation_AutoScreen_216.csv"

screen.to_csv(screen_path, index=False)

print("AUTOMATED SCREENING SAVED:", os.path.exists(screen_path))
print("ARTICLES SCREENED:", len(screen))
print("UNIQUE ARTICLE IDs:", screen["id"].nunique())
print("DUPLICATE IDs:", screen["id"].duplicated().sum())

print("\nAUTOMATIC FLAG SUMMARY:")
print(screen["automatic_flags"].value_counts().head(15).to_string())

print("\nARTICLES WITH FLAGS:",
      (screen["automatic_flags"] != "No automatic flags").sum())

print("\nSAVED FILE:")
print(screen_path)

AUTOMATED SCREENING SAVED: True
ARTICLES SCREENED: 216
UNIQUE ARTICLE IDs: 216
DUPLICATE IDs: 0

AUTOMATIC FLAG SUMMARY:
automatic_flags
No automatic flags                                                                                                   159
Check numbers: 60                                                                                                      5
Check numbers: 16                                                                                                      3
Check numbers: 10                                                                                                      2
Check numbers: 40                                                                                                      2
Check numbers: 12                                                                                                      2
Check numbers: 200                                                                                                     1
Check numbers: 8

In [532]:
# Summarise automatic screening flags

flagged = screen[
    screen["automatic_flags"] != "No automatic flags"
].copy()

def flag_categories(flags):
    categories = []
    if "Missing Hindi" in flags:
        categories.append("Missing Hindi")
    if "Missing English" in flags:
        categories.append("Missing English")
    if "Devanagari" in flags:
        categories.append("Hindi script in English")
    if "Check numbers" in flags:
        categories.append("Number check")
    if "Unusual translation length" in flags:
        categories.append("Length check")
    if "Check sentence structure" in flags:
        categories.append("Sentence check")
    return categories

flagged["flag_categories"] = flagged["automatic_flags"].apply(
    flag_categories
)

print("FLAG CATEGORY COUNTS")
print("-" * 45)

for category in [
    "Missing Hindi",
    "Missing English",
    "Hindi script in English",
    "Number check",
    "Length check",
    "Sentence check"
]:
    count = flagged["flag_categories"].apply(
        lambda x: category in x
    ).sum()
    print(f"{category}: {count}")

print("\nALL FLAGGED ARTICLE IDs")
print("-" * 45)

print(
    flagged[["id", "title", "automatic_flags"]]
    .to_string(index=False)
)

FLAG CATEGORY COUNTS
---------------------------------------------
Missing Hindi: 0
Missing English: 0
Hindi script in English: 0
Number check: 56
Length check: 6
Sentence check: 5

ALL FLAGGED ARTICLE IDs
---------------------------------------------
                                                id                                                                    title                                                                                                                     automatic_flags
                               061004_askus_geyser                                           प्राकृतिक गीज़र क्या होते हैं?                                                                                                           Check numbers: 46, 51, 53
                                    india-46106302                       झारखंड की इन औरतों को क्यों चुभता है अफ़ग़ानिस्तान                                                                                                                 

In [533]:
# Prioritise all 216 articles for manual review

triage = screen.copy()

triage["has_number_flag"] = triage["automatic_flags"].str.contains(
    "Check numbers", na=False
)
triage["has_length_flag"] = triage["automatic_flags"].str.contains(
    "Unusual translation length", na=False
)
triage["has_sentence_flag"] = triage["automatic_flags"].str.contains(
    "Check sentence structure", na=False
)

def assign_priority(row):
    if row["has_length_flag"] and row["has_sentence_flag"]:
        return "HIGH - Length + sentence"
    elif row["has_length_flag"] or row["has_sentence_flag"]:
        return "MEDIUM - Structural check"
    elif row["has_number_flag"]:
        return "NUMBER CHECK"
    else:
        return "NO AUTOMATIC FLAGS"

triage["review_priority"] = triage.apply(assign_priority, axis=1)

priority_order = {
    "HIGH - Length + sentence": 1,
    "MEDIUM - Structural check": 2,
    "NUMBER CHECK": 3,
    "NO AUTOMATIC FLAGS": 4
}

triage["_sort"] = triage["review_priority"].map(priority_order)
triage = triage.sort_values(["_sort", "id"]).drop(columns="_sort")

triage_path = base_path + "/09_sentence_level_translation/FactIndic_Translation_Review_Priority_216.csv"
triage.to_csv(triage_path, index=False)

print("PRIORITY FILE SAVED:", os.path.exists(triage_path))
print("\nPRIORITY SUMMARY:")
print(triage["review_priority"].value_counts().to_string())

print("\nHIGHEST PRIORITY ARTICLES:")
print(
    triage.loc[
        triage["review_priority"] == "HIGH - Length + sentence",
        ["id", "title", "automatic_flags"]
    ].to_string(index=False)
)

print("\nMEDIUM PRIORITY ARTICLES:")
print(
    triage.loc[
        triage["review_priority"] == "MEDIUM - Structural check",
        ["id", "title", "automatic_flags"]
    ].to_string(index=False)
)

PRIORITY FILE SAVED: True

PRIORITY SUMMARY:
review_priority
NO AUTOMATIC FLAGS           159
NUMBER CHECK                  50
HIGH - Length + sentence       4
MEDIUM - Structural check      3

HIGHEST PRIORITY ARTICLES:
                        id                                                                   title                                                                                                                     automatic_flags
090216_jobloss_column3_alk                                                   'तलाश अभी जारी है...'                                                           Check numbers: 15 | Unusual translation length | Check sentence structure
    international-47105506 पाकिस्तान में हज सब्सिडी पर क्यों मचा हुआ है हंगामा- उर्दू प्रेस रिव्यू                                               Check numbers: 25, 29, 76, 80 | Unusual translation length | Check sentence structure
    international-48243735     ईरान के राष्ट्रपति रूहानी ने माना, अभी इराक़ युद्ध से भ

In [534]:
from IPython.display import display, HTML
import html

high_priority = triage[
    triage["review_priority"] == "HIGH - Length + sentence"
].copy()

review_path = base_path + "/09_sentence_level_translation/FactIndic_High_Priority_4_Review.csv"

high_priority[
    ["id", "title", "text", "complete_english_translation",
     "automatic_flags", "review_priority"]
].to_csv(review_path, index=False)

print("HIGH-PRIORITY ARTICLES:", len(high_priority))
print("REVIEW FILE SAVED:", os.path.exists(review_path))

for _, row in high_priority.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("HINDI TITLE:", row["title"])
    print("AUTOMATIC FLAGS:", row["automatic_flags"])
    print("\nHINDI SOURCE:\n")
    print(row["text"])
    print("\nENGLISH TRANSLATION:\n")
    print(row["complete_english_translation"])
    print("\n" + "=" * 90)

HIGH-PRIORITY ARTICLES: 4
REVIEW FILE SAVED: True

ARTICLE ID: 090216_jobloss_column3_alk
HINDI TITLE: 'तलाश अभी जारी है...'
AUTOMATIC FLAGS: Check numbers: 15 | Unusual translation length | Check sentence structure

HINDI SOURCE:

नौकरी रहते हुए नई नौकरी की तलाश तो फिर भी आसान है, लेकिन सड़क पर आकर नौकरी ढूँढना काफ़ी मुश्किल. इसीलिए मैंने अपने एचआर वालों से एक विनती भी की थी, तीन महीने का नोटिस पीरियड दे दो, सैलरी बेशक मत दो लेकिन नई नौकरी ढूँढने का मौका दे दो. पर ऐसा नहीं हुआ. दरअसल मेरे ऑफ़िस वालों को पता था कि मेरे पापा रिटायर्ड जज हैं. पता नहीं क्यों इस बात को लेकर उनके मन में संशय भी था कि मैं उन्हें अदालत में ना घसीट लूँ. इसलिए मेरे मामले में इस्तीफ़े से संबंधित सभी कागज़ी कार्रवाई को वो हू-बहू अमल में भी ला रहे थे. मुझे कोई रियायत नहीं दी गई. नौकरी ढूँढने का सिलसिला तो दो दिनों बाद ही शुरु हो गया. शुरुआत मित्रों को बताने और अपने ही सर्किल में कोई नौकरी खोजने की कोशिश के साथ हुई. 'सेट' कराने के आश्वासन शुरु-शुरु में कुछ दोस्तों ने अपने यहाँ 'सेट' करवा देने का आश्वासन दिया मगर 15

In [535]:
import pandas as pd
import os

audit_path = base_path + "/09_sentence_level_translation/FactIndic_Translation_Audit_230.csv"

high_priority_reviews = [
    {
        "id": "090216_jobloss_column3_alk",
        "batch_number": 3,
        "translation_status": "Major source-content omission",
        "reviewer_notes": "English translation omits substantial portions of the Hindi first-person account, including the three-month notice request, 15-day wait, written test, rejection details and continued job search.",
        "correction_suggestion": "Prepare a complete English translation preserving the first-person narrative, sequence of events, quotations and all source details."
    },
    {
        "id": "international-47105506",
        "batch_number": 3,
        "translation_status": "Major source-content omission",
        "reviewer_notes": "English translation omits substantial sections about the British parliamentary Kashmir event, named speakers, dates, attributed statements and Taliban-related developments.",
        "correction_suggestion": "Prepare a complete translation of all three sections, preserving names, dates, quotations and attribution."
    },
    {
        "id": "international-48243735",
        "batch_number": 3,
        "translation_status": "Major source-content omission",
        "reviewer_notes": "English translation omits key source details about the nuclear agreement, economic impact, sanctions, military deployments, 5,200 US troops and the 60-day deadline.",
        "correction_suggestion": "Prepare a complete translation preserving the article's sections, numerical details, quotations and attributed claims."
    },
    {
        "id": "international-49451047",
        "batch_number": 3,
        "translation_status": "Major source-content omission",
        "reviewer_notes": "English translation omits substantial details about the award's history, previous recipients, diplomatic visits, trade figures, RuPay launch and Pakistan's reaction.",
        "correction_suggestion": "Prepare a complete translation preserving the award history, dates, figures, diplomatic details and attributed reactions."
    }
]

new_reviews = pd.DataFrame(high_priority_reviews)

if os.path.exists(audit_path):
    old_reviews = pd.read_csv(audit_path)
    combined = pd.concat(
        [old_reviews, new_reviews],
        ignore_index=True
    ).drop_duplicates(subset=["id"], keep="last")
else:
    combined = new_reviews

combined.to_csv(audit_path, index=False)

print("AUDIT SAVED:", os.path.exists(audit_path))
print("TOTAL AUDIT RECORDS:", len(combined))
print("HIGH-PRIORITY REVIEWS SAVED:", len(new_reviews))
print("DUPLICATE IDS:", combined["id"].duplicated().sum())
print("\nLATEST FINDINGS:")
print(new_reviews[["id", "translation_status"]].to_string(index=False))

AUDIT SAVED: True
TOTAL AUDIT RECORDS: 18
HIGH-PRIORITY REVIEWS SAVED: 4
DUPLICATE IDS: 0

LATEST FINDINGS:
                        id            translation_status
090216_jobloss_column3_alk Major source-content omission
    international-47105506 Major source-content omission
    international-48243735 Major source-content omission
    international-49451047 Major source-content omission


In [536]:
medium_priority = triage[
    triage["review_priority"] == "MEDIUM - Structural check"
].copy()

print("MEDIUM-PRIORITY ARTICLES:", len(medium_priority))

for _, row in medium_priority.iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("HINDI TITLE:", row["title"])
    print("AUTOMATIC FLAGS:", row["automatic_flags"])

    print("\nHINDI SOURCE:\n")
    print(row["text"])

    print("\nENGLISH TRANSLATION:\n")
    print(row["complete_english_translation"])

    print("\n" + "=" * 90)

MEDIUM-PRIORITY ARTICLES: 3

ARTICLE ID: 130606_china_gender_ss
HINDI TITLE: 66 साल बाद पता चला कि वो मर्द नहीं औरत है
AUTOMATIC FLAGS: Unusual translation length

HINDI SOURCE:

पुरुष को पता नहीं था कि उनके पेट में अंडाशय हैं दरअसल इस बुजुर्ग के पेट में दर्द का कारण उनके अंडाशय में एक गाँठ का होना बताया गया और पता चला कि वो पुरुष नहीं बल्कि एक महिला थे. ये ख़बर साउथ चाइना मॉर्निंग पोस्ट में छपी है. हॉंगकॉंग मेडिकल जर्नल में इस व्यक्ति की बीमारी के बारे में बताया गया है कि ये 'रेयस्ट ऑफ दी रेयर' यानी दुर्लभतम मामलों से से एक है. आनुवांशिक विकार जर्नल के मुताबिक दो आनुवांशिक विकारों का एक दूसरे से जुड़ जाना इस बीमारी का कारण होता है. इसे टर्नर सिंड्रोम कहा जाता है और ऐसे व्यक्तियों में महिलाओं के मुख्य लक्षण जैसे वो मां नहीं बन सकती है और उसमें महिलाओं के शरीर में होने वाला विकास भी नहीं होता है. हालांकि जिन लोगों को ये सिंड्रोम होता है वो महिलाओं जैसे दिखते जरुर हैं लेकिन उनके लक्षण स्त्री प्रधान नहीं होते. लेकिन इन बुजुर्ग का मामला थोड़ा अलग है क्योंकि वो कॉजेनिटल आर्ड्रिनल हाइपरपलेसि

In [537]:
import pandas as pd
import os

audit_path = base_path + "/09_sentence_level_translation/FactIndic_Translation_Audit_230.csv"

medium_reviews = [
    {
        "id": "130606_china_gender_ss",
        "batch_number": 4,
        "translation_status": "Major source-content omission",
        "reviewer_notes": "The English version is substantially shorter than the Hindi source. It omits the 66-year discovery context, source publication references, parts of the medical explanation and details of the person's reported experience.",
        "correction_suggestion": "Prepare a complete translation preserving the source's medical explanations, publication references, reported symptoms and all relevant details. Keep medical claims attributed to the source."
    },
    {
        "id": "150816_badminton_saina_final_dp",
        "batch_number": 4,
        "translation_status": "Critical factual mismatch",
        "reviewer_notes": "The Hindi source reports Saina Nehwal losing the World Championship final to Carolina Marin, 16-21 and 19-21, in 59 minutes. The English instead describes a semi-final victory against Lindaweni Fanetri, with different scores. The English does not correspond to the supplied source.",
        "correction_suggestion": "Replace the English text with a complete translation of the supplied Hindi final-match report. Preserve the opponent, final scores, match duration, medal history and match progression."
    },
    {
        "id": "160311_saina_nehwal_exits_all_england_aa",
        "batch_number": 4,
        "translation_status": "Verified - minor editorial review",
        "reviewer_notes": "The main result, opponent, scores, tournament context and named Indian players are preserved. Some source details are condensed, but no major meaning-changing discrepancy was identified.",
        "correction_suggestion": "Retain the translation. For completeness, check that all source details are represented; no major correction is required."
    }
]

new_reviews = pd.DataFrame(medium_reviews)

if os.path.exists(audit_path):
    old_reviews = pd.read_csv(audit_path)
    combined = pd.concat(
        [old_reviews, new_reviews],
        ignore_index=True
    ).drop_duplicates(subset=["id"], keep="last")
else:
    combined = new_reviews

combined.to_csv(audit_path, index=False)

print("AUDIT SAVED:", os.path.exists(audit_path))
print("TOTAL AUDIT RECORDS:", len(combined))
print("MEDIUM-PRIORITY REVIEWS SAVED:", len(new_reviews))
print("DUPLICATE IDS:", combined["id"].duplicated().sum())
print("\nLATEST FINDINGS:")
print(new_reviews[["id", "translation_status"]].to_string(index=False))

AUDIT SAVED: True
TOTAL AUDIT RECORDS: 21
MEDIUM-PRIORITY REVIEWS SAVED: 3
DUPLICATE IDS: 0

LATEST FINDINGS:
                                      id                translation_status
                  130606_china_gender_ss     Major source-content omission
         150816_badminton_saina_final_dp         Critical factual mismatch
160311_saina_nehwal_exits_all_england_aa Verified - minor editorial review


In [538]:
import pandas as pd
import os

audit_path = base_path + "/09_sentence_level_translation/FactIndic_Translation_Audit_230.csv"

# Load IDs already reviewed
saved_audit = pd.read_csv(audit_path)
reviewed_ids = set(saved_audit["id"].astype(str))

# Select number-flagged articles
number_flags = triage[
    triage["review_priority"] == "NUMBER CHECK"
][["id", "automatic_flags", "review_priority"]].copy()

# Attach Hindi and English source text
number_queue = pending_audit.merge(
    number_flags,
    on="id",
    how="inner"
)

# Exclude articles already manually audited
number_queue = number_queue[
    ~number_queue["id"].astype(str).isin(reviewed_ids)
].drop_duplicates(subset=["id"])

print("NUMBER-FLAGGED ARTICLES:", len(number_flags))
print("NOT YET MANUALLY REVIEWED:", len(number_queue))
print("BATCH 1: FIRST 7 ARTICLES")

for _, row in number_queue.head(7).iterrows():
    print("\n" + "=" * 90)
    print("ARTICLE ID:", row["id"])
    print("AUTOMATIC FLAGS:", row["automatic_flags"])
    print("\nHINDI TITLE:", row["title"])
    print("\nHINDI SOURCE:\n", row["text"])
    print("\nENGLISH TRANSLATION:\n", row["complete_english_translation"])
    print("=" * 90)

NUMBER-FLAGGED ARTICLES: 50
NOT YET MANUALLY REVIEWED: 50
BATCH 1: FIRST 7 ARTICLES

ARTICLE ID: 061004_askus_geyser
AUTOMATIC FLAGS: Check numbers: 46, 51, 53

HINDI TITLE: प्राकृतिक गीज़र क्या होते हैं?

HINDI SOURCE:
 गीज़र एक प्रकार का गर्म पानी का सोता होता है जो समय-समय पर फूटता है और इसमें से गर्म पानी और भाप निकलती है. गीज़र के लिए अनुकूल जल भौमिकी की आवश्यकता होती है जो पृथ्वी के कुछ ही स्थानों पर मौजूद है. ये गीज़र कैसे बनते हैं... पृथ्वी की सतह का पानी ज़मीन के अंदर जाकर जमा होता है और ऐसी चट्टान से टकराता है जो मैग्मा से तप रही हो. इससे पानी गर्म होने लगता है और फिर ऊपर उठता है और किसी चट्टान की दरार से होकर बाहर फूट पड़ता है. दुनिया में कोई एक हज़ार गीज़र हैं जिनमें से आधे अमरीका के यैलोस्टोन नेशनल पार्क में हैं. गर्म पानी का इस्तेमाल करने के लिए बिजली की जिस मशीन (गीज़र) का इस्तेमाल होता है उसका नाम भी शायद इसी पर पड़ा होगा. ग्राम पहलाम, सहरसा बिहार से जितेन्द्र कुमार पासवान पूछते हैं कि ओलम्पिक खेल सर्वप्रथम किस देश में खेले गए थे. प्राचीन ओलम्पिक खेल सबसे पहले ग्रीस के 

In [539]:
import pandas as pd
import os

audit_path = base_path + "/09_sentence_level_translation/FactIndic_Translation_Audit_230.csv"

number_reviews = [
    {
        "id": "061004_askus_geyser",
        "batch_number": 5,
        "translation_status": "Numerical discrepancy - correction required",
        "reviewer_notes": "The Hindi source says 46 lakh pounds (£4,600,000), but the English translation says £460,000. This is a tenfold numerical error. Other checked figures, including 5.3 million and 5.1 million, match.",
        "correction_suggestion": "Change £460,000 to £4,600,000."
    },
    {
        "id": "india-46106302",
        "batch_number": 5,
        "translation_status": "Number check passed - no discrepancy found",
        "reviewer_notes": "Checked the six-month period, 40-day reference and Rs 1 lakh assistance per person. The numerical details match the Hindi source.",
        "correction_suggestion": "No numerical correction required."
    },
    {
        "id": "international-41646598",
        "batch_number": 5,
        "translation_status": "Number check passed - no discrepancy found",
        "reviewer_notes": "The age of 25 and the age of 8 in the quoted headline are preserved.",
        "correction_suggestion": "No numerical correction required."
    },
    {
        "id": "080826_bihar_flood",
        "batch_number": 5,
        "translation_status": "Number check passed - no discrepancy found",
        "reviewer_notes": "Checked 20-25 lakh people, 40 deaths, 200 boats, 25 motorboats, more than 50 camps, 700-800 houses, 100 prisoners and 700 deaths in Uttar Pradesh. Figures match.",
        "correction_suggestion": "No numerical correction required."
    },
    {
        "id": "131119_chinese_supercomputer_worldbest_rns",
        "batch_number": 5,
        "translation_status": "Number check passed - no discrepancy found",
        "reviewer_notes": "The 33.86 petaflops and 6.27 petaflops performance figures are preserved correctly.",
        "correction_suggestion": "No numerical correction required."
    },
    {
        "id": "india-53336116",
        "batch_number": 5,
        "translation_status": "Number check passed - no discrepancy found",
        "reviewer_notes": "Checked the 3.5 million square kilometre area and the 2015, 2007, 2017 and 2019 references. They match the source.",
        "correction_suggestion": "No numerical correction required."
    },
    {
        "id": "050625_building_collapse",
        "batch_number": 5,
        "translation_status": "Number check passed - no discrepancy found",
        "reviewer_notes": "Four deaths, four people rescued, one person possibly trapped and Mumbai's population of approximately 16 million are preserved.",
        "correction_suggestion": "No numerical correction required."
    }
]

new_reviews = pd.DataFrame(number_reviews)

if os.path.exists(audit_path):
    old_reviews = pd.read_csv(audit_path)
    combined = pd.concat(
        [old_reviews, new_reviews],
        ignore_index=True
    ).drop_duplicates(subset=["id"], keep="last")
else:
    combined = new_reviews

combined.to_csv(audit_path, index=False)

print("AUDIT SAVED:", os.path.exists(audit_path))
print("TOTAL AUDIT RECORDS:", len(combined))
print("BATCH 1 NUMBER REVIEWS SAVED:", len(new_reviews))
print("DUPLICATE IDS:", combined["id"].duplicated().sum())
print("\nFINDINGS:")
print(new_reviews[["id", "translation_status"]].to_string(index=False))

AUDIT SAVED: True
TOTAL AUDIT RECORDS: 28
BATCH 1 NUMBER REVIEWS SAVED: 7
DUPLICATE IDS: 0

FINDINGS:
                                        id                          translation_status
                       061004_askus_geyser Numerical discrepancy - correction required
                            india-46106302  Number check passed - no discrepancy found
                    international-41646598  Number check passed - no discrepancy found
                        080826_bihar_flood  Number check passed - no discrepancy found
131119_chinese_supercomputer_worldbest_rns  Number check passed - no discrepancy found
                            india-53336116  Number check passed - no discrepancy found
                  050625_building_collapse  Number check passed - no discrepancy found


In [540]:
import pandas as pd

audit_path = base_path + "/09_sentence_level_translation/FactIndic_Translation_Audit_230.csv"

# Load the latest saved audit
saved_audit = pd.read_csv(audit_path)
reviewed_ids = set(saved_audit["id"].astype(str))

# Rebuild the number-check queue
number_flags = triage[
    triage["review_priority"] == "NUMBER CHECK"
][["id", "automatic_flags", "review_priority"]].copy()

number_queue = pending_audit.merge(
    number_flags,
    on="id",
    how="inner"
)

number_queue = number_queue[
    ~number_queue["id"].astype(str).isin(reviewed_ids)
].drop_duplicates(subset=["id"])

print("REMAINING NUMBER-FLAGGED ARTICLES:", len(number_queue))
print("BATCH 2 — NEXT 7 ARTICLES")
print("=" * 100)

for i, (_, row) in enumerate(number_queue.head(7).iterrows(), 1):
    print(f"\nARTICLE {i}")
    print("=" * 100)
    print("ID:", row["id"])
    print("AUTOMATIC FLAG:", row["automatic_flags"])
    print("\nHINDI TITLE:\n", row["title"])
    print("\nHINDI SOURCE:\n", row["text"])
    print("\nENGLISH TRANSLATION:\n", row["complete_english_translation"])
    print("\n" + "-" * 100)

REMAINING NUMBER-FLAGGED ARTICLES: 43
BATCH 2 — NEXT 7 ARTICLES

ARTICLE 1
ID: 080721_parliament_debate
AUTOMATIC FLAG: Check numbers: 16

HINDI TITLE:
 संसद का विशेष सत्र शुरु

HINDI SOURCE:
 संसद का विशेष सत्र शुरु हो गया है जहां कुछ ही देर में में विश्वास प्रस्ताव पेश किया जाएगा. बहस के बाद मंगलवार को विश्वास प्रस्ताव पर मतदान होगा जिसमें पता चल सकेगा कि पिछले पंद्रह दिनों तक विभिन्न पार्टियों की जोड़ तोड़ क्या रंग लाती है. सदन में विश्वास प्रस्ताव से पहले लोकसभा स्पीकर सोमनाथ चटर्जी ने सर्वदलीय बैठक बुलाई है और इसमें सभी दल हिस्सा ले रहे हैं. प्रधानमंत्री मनमोहन सिंह सोमवार को लोक सभा में विश्वास मत के लिए एक लाइन का प्रस्ताव पेश करेंगे कि 'ये सदन मंत्रिपरिषद में अपना विश्वास व्यक्त करता है.' माना जा रहा है कि इसके बाद प्रधानमंत्री मनमोहन सिंह संक्षिप्त भाषण देकर सदन में प्रस्ताव को स्वीकार करने का अनुरोध करेंगे. इसके बाद सोमवार और मंगलवार को लोक सभा में इस प्रस्ताव पर चर्चा होगी और चर्चा का जबाव एक बार फिर प्रधानमंत्री देंगे और उसके बाद मत विभाजन होगा जिसमें सरकार के भाग्य का फ़ैस

In [541]:
import pandas as pd
import os

audit_path = base_path + "/09_sentence_level_translation/FactIndic_Translation_Audit_230.csv"

number_reviews = [
    {
        "id": "080721_parliament_debate",
        "batch_number": 6,
        "translation_status": "Number check passed - no discrepancy found",
        "reviewer_notes": "Checked 15 days, 16 hours, 543 total MPs, 542 valid votes, 271 votes for a majority, at least five Samajwadi Party MPs and the one-week reference. Figures match the Hindi source.",
        "correction_suggestion": "No numerical correction required."
    },
    {
        "id": "080416_up_results",
        "batch_number": 6,
        "translation_status": "Number check passed - no discrepancy found",
        "reviewer_notes": "Checked the 65,000-vote and 54,000-vote margins, fewer than 10,000 Congress votes and 11 months of Mayawati's tenure. Figures match.",
        "correction_suggestion": "No numerical correction required."
    },
    {
        "id": "150607_narendra_modi_bangladesh_visit_analysis_rns",
        "batch_number": 6,
        "translation_status": "Number check passed - no discrepancy found",
        "reviewer_notes": "Checked the two-billion-dollar loan, the 1974 ratification reference and 22 economic agreements. Figures match the Hindi source.",
        "correction_suggestion": "No numerical correction required."
    },
    {
        "id": "international-52239381",
        "batch_number": 6,
        "translation_status": "Number check passed - source figures preserved",
        "reviewer_notes": "Checked 100 days, 95,000 and 97,000 deaths, 1.6 million infections, 31 December 2019, the dated timeline, Wuhan's 11 million population and 65,000 departures. The source itself gives both 95,000 and 97,000 deaths in different sections; the English preserves both.",
        "correction_suggestion": "No numerical translation correction required. Preserve the source's differing figures."
    },
    {
        "id": "040415_sharon_arafat",
        "batch_number": 6,
        "translation_status": "Number check passed - no discrepancy found",
        "reviewer_notes": "Checked approximately 90,000 Jewish settlers, six settlements and the repeated 1967 border references. Figures match.",
        "correction_suggestion": "No numerical correction required."
    },
    {
        "id": "150612_heatwave_rentachintala_sra",
        "batch_number": 6,
        "translation_status": "Number check passed - no discrepancy found",
        "reviewer_notes": "Checked the 48-50 degree Celsius average temperature, 50 degrees in May, the 2012 record of 52 degrees, population of around 16,000, and the 8 am, 8 pm and 12 noon to 3 pm time references. Figures match.",
        "correction_suggestion": "No numerical correction required."
    },
    {
        "id": "050812_baflights_cancel",
        "batch_number": 6,
        "translation_status": "Number check passed - no discrepancy found",
        "reviewer_notes": "Checked 70,000 affected passengers, 600 dismissed catering-company employees and the five other affected airlines. Figures match.",
        "correction_suggestion": "No numerical correction required."
    }
]

new_reviews = pd.DataFrame(number_reviews)

if os.path.exists(audit_path):
    old_reviews = pd.read_csv(audit_path)
    combined = pd.concat(
        [old_reviews, new_reviews],
        ignore_index=True
    ).drop_duplicates(subset=["id"], keep="last")
else:
    combined = new_reviews

combined.to_csv(audit_path, index=False)

print("AUDIT SAVED:", os.path.exists(audit_path))
print("TOTAL AUDIT RECORDS:", len(combined))
print("BATCH 2 NUMBER REVIEWS SAVED:", len(new_reviews))
print("DUPLICATE IDS:", combined["id"].duplicated().sum())
print("\nFINDINGS:")
print(new_reviews[["id", "translation_status"]].to_string(index=False))

AUDIT SAVED: True
TOTAL AUDIT RECORDS: 35
BATCH 2 NUMBER REVIEWS SAVED: 7
DUPLICATE IDS: 0

FINDINGS:
                                                id                             translation_status
                          080721_parliament_debate     Number check passed - no discrepancy found
                                 080416_up_results     Number check passed - no discrepancy found
150607_narendra_modi_bangladesh_visit_analysis_rns     Number check passed - no discrepancy found
                            international-52239381 Number check passed - source figures preserved
                              040415_sharon_arafat     Number check passed - no discrepancy found
                 150612_heatwave_rentachintala_sra     Number check passed - no discrepancy found
                           050812_baflights_cancel     Number check passed - no discrepancy found


In [542]:
import pandas as pd
import os

folder = base_path + "/09_sentence_level_translation"
audit_path = folder + "/FactIndic_Translation_Audit_230.csv"

# 1. Save Batch 2 reviews
batch2_reviews = [
    {
        "id": "080721_parliament_debate",
        "batch_number": 6,
        "translation_status": "Number check passed - no discrepancy found",
        "reviewer_notes": "Checked 15 days, 16 hours, 543 MPs, 542 valid votes, 271 votes for a majority and at least five MPs. Figures match.",
        "correction_suggestion": "No numerical correction required."
    },
    {
        "id": "080416_up_results",
        "batch_number": 6,
        "translation_status": "Number check passed - no discrepancy found",
        "reviewer_notes": "Checked 65,000 and 54,000 vote margins, fewer than 10,000 votes and 11 months. Figures match.",
        "correction_suggestion": "No numerical correction required."
    },
    {
        "id": "150607_narendra_modi_bangladesh_visit_analysis_rns",
        "batch_number": 6,
        "translation_status": "Number check passed - no discrepancy found",
        "reviewer_notes": "Checked two-billion-dollar loan, 1974 and 22 agreements. Figures match.",
        "correction_suggestion": "No numerical correction required."
    },
    {
        "id": "international-52239381",
        "batch_number": 6,
        "translation_status": "Number check passed - source figures preserved",
        "reviewer_notes": "Checked 95,000 and 97,000 deaths, 1.6 million infections, 100 days, dates, 11 million population and 65,000 departures. The source itself gives both death figures.",
        "correction_suggestion": "No numerical translation correction required."
    },
    {
        "id": "040415_sharon_arafat",
        "batch_number": 6,
        "translation_status": "Number check passed - no discrepancy found",
        "reviewer_notes": "Checked approximately 90,000 settlers, six settlements and 1967. Figures match.",
        "correction_suggestion": "No numerical correction required."
    },
    {
        "id": "150612_heatwave_rentachintala_sra",
        "batch_number": 6,
        "translation_status": "Number check passed - no discrepancy found",
        "reviewer_notes": "Checked 48-50°C, 50°C, 52°C in 2012, population around 16,000 and time references. Figures match.",
        "correction_suggestion": "No numerical correction required."
    },
    {
        "id": "050812_baflights_cancel",
        "batch_number": 6,
        "translation_status": "Number check passed - no discrepancy found",
        "reviewer_notes": "Checked 70,000 passengers, 600 employees and five other airlines. Figures match.",
        "correction_suggestion": "No numerical correction required."
    }
]

old = pd.read_csv(audit_path)
combined = pd.concat(
    [old, pd.DataFrame(batch2_reviews)],
    ignore_index=True
).drop_duplicates(subset=["id"], keep="last")

combined.to_csv(audit_path, index=False)

# 2. Rebuild remaining number-check queue
number_flags = triage[
    triage["review_priority"] == "NUMBER CHECK"
][["id", "automatic_flags"]].copy()

reviewed_ids = set(combined["id"].astype(str))

number_queue = pending_audit.merge(
    number_flags,
    on="id",
    how="inner"
)

number_queue = number_queue[
    ~number_queue["id"].astype(str).isin(reviewed_ids)
].drop_duplicates(subset=["id"])

# 3. Create one review pack
review_pack = number_queue[
    ["id", "title", "text", "complete_english_translation",
     "automatic_flags"]
].copy()

review_pack_path = folder + "/FactIndic_Number_Review_Pack_Remaining.csv"
review_pack.to_csv(review_pack_path, index=False)

# 4. Final checks
print("BATCH 2 AUDIT SAVED:", os.path.exists(audit_path))
print("TOTAL AUDIT RECORDS:", len(combined))
print("BATCH 2 REVIEWS ADDED:", len(batch2_reviews))
print("AUDIT DUPLICATES:", combined["id"].duplicated().sum())

print("\nREMAINING NUMBER-FLAGGED ARTICLES:", len(review_pack))
print("REVIEW PACK SAVED:", os.path.exists(review_pack_path))
print("REVIEW PACK DUPLICATES:", review_pack["id"].duplicated().sum())

print("\nREVIEW PACK LOCATION:")
print(review_pack_path)

# 5. Download the review pack
from google.colab import files
files.download(review_pack_path)

BATCH 2 AUDIT SAVED: True
TOTAL AUDIT RECORDS: 35
BATCH 2 REVIEWS ADDED: 7
AUDIT DUPLICATES: 0

REMAINING NUMBER-FLAGGED ARTICLES: 36
REVIEW PACK SAVED: True
REVIEW PACK DUPLICATES: 0

REVIEW PACK LOCATION:
/content/drive/MyDrive/FactIndic_PersonB_CleanRun/09_sentence_level_translation/FactIndic_Number_Review_Pack_Remaining.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [543]:
import pandas as pd
import os

audit_path = base_path + "/09_sentence_level_translation/FactIndic_Translation_Audit_230.csv"

reviews = [
    ("international-42264998", "Number check passed - no discrepancy found", "Checked 22 years, 10 days, age 27, independence in 1971, centenary in 2071, per-capita income of $72,800 and 21 cabinet members. The figures match; some numbers are written as words in Hindi."),
    ("051030_delhi_blasts", "Number check passed - no discrepancy found", "Checked 60 deaths, 200 injured, 10 detained, Rs 3 lakh compensation, Rs 50,000 compensation, and the 5:30 pm and 6 pm blast times. Figures match."),
    ("050420_pope_muslims", "Source-content mismatch - further review required", "Hindi source identifies Pope Benedict XVI. The supplied English includes references to 15 million Muslims and September 11 that are not present in the supplied Hindi text. Check the source version before accepting these additions."),
    ("050108_nicobar_force", "Number check passed - no discrepancy found", "Checked at least 50 flights daily, more than 2,000 tonnes of relief supplies, 14 villages, 22 bodies and 400 cremations. Figures match."),
    ("051121_aids_india", "Number check passed - no discrepancy found", "Checked 5.1 million people, survey year 2003, percentages 14, 19, 26, 91, 85, 11, 4 and 52, and years 2001 and 2004. Figures match."),
    ("070902_khanduri_win", "Number check passed - no discrepancy found", "Checked 29 August, 14,000 votes, 70 assembly seats, 69 contested seats, 34 BJP seats and 39 trust-vote votes. Figures match."),
    ("130528_speed_breaker_cars_impact_sy", "Source-content omission - numeric detail missing", "Hindi says the service centre receives at least 2-3 vehicles every day damaged by speed breakers. This sentence is missing from the supplied English translation. Add it."),
    ("050107_pravasi_pm", "Source-fidelity review - extra statistic", "The main figures 1950, 16 countries, more than 1,500 participants and 60 countries match. English additionally states that around 25 million Indians live abroad; this figure is absent from the supplied Hindi text. Verify its source."),
    ("150222_parliament_budget_session_pk", "Source-content review - possible omissions", "The two-phase session and 8 May end date are preserved. The Hindi also refers to three earlier occasions involving joint sessions; check that this detail and the remaining source content are represented."),
    ("090210_filmfestival_gorakhpur", "Number check passed - no discrepancy found", "Checked 2006, 12,000 participants, 238 kilometres and 17 days. Figures match."),
    ("061217_spl_sez_imballance", "Source-content review - omission and extra statistic", "The English preserves 403, 46, 54, 75, 30, 18, 10 and 11. It omits the source's statement that the four named states account for more than 50 percent of approved SEZs, and adds a 15-state ranking not present in the supplied Hindi text. Review source fidelity."),
    ("121028_international_others_thai_women_aa", "Number check passed - no discrepancy found", "Checked age 32, $100 per day, 6,000 rupees, 80,000 baht, 140,000 rupees, 40,000 baht, age 25, eight teeth, 13 years and 50,000 baht. Figures match."),
    ("050505_voteclose_britain", "Number check passed - no discrepancy found", "Checked 161-seat majority, 45 million voters, 645 and 646 seats, 324 majority threshold, party seat counts, 6 million postal voters and turnout percentages for 1992, 1997 and 2001. Figures match."),
    ("150713_bihar_women_strange_names_rns", "Missing English translation - correction required", "The supplied English translation is empty. The Hindi includes the age of a woman interviewed (60 years). A complete English translation is required."),
    ("071001_pakistan_cargo", "Missing English translation - correction required", "The supplied English translation is empty. The Hindi refers to 60 years since partition. A complete English translation is required."),
    ("160212_ranchi_mamta_brave_girl_pk", "Number check passed - no discrepancy found", "Checked age 14, 26 January, the 45,000-rupee loan and other age/class references. The main figures are consistent; some Hindi numbers are written as words."),
    ("131231_drone_test_site_sr", "Number check passed - no discrepancy found", "Checked $89 billion, 2015, 10 months, 24 states, $25 per flight, around 1,500 rupees and 7,500 drones. The 70,000 jobs figure is also preserved."),
    ("040721_paes_olympics", "Number check passed - no discrepancy found", "Checked ranking 127, Olympic years 1992, 1972 and 1996, match scores, age/date references, ranking 73 and 1999 titles. Figures match."),
    ("151003_draught_karnatak_series_part_one_sr", "Number check passed - no discrepancy found", "Checked village population around 3,000, approximately 1,500 voters, 15 October and 25 districts across seven states. Figures match."),
    ("140223_modi_rally_punjab_sr", "Source-content omission - numeric detail missing", "Hindi says subsidised LPG cylinders were increased from nine to 12. The supplied English mentions the earlier nine but omits the increase to 12. Add the missing detail."),
    ("india-44912345", "Source-fidelity review - extra figure and omitted details", "The GST rates 12%, 5%, 28% and 18% are represented. English adds a Rs 2.50 sanitary-pad statement not present in the supplied Hindi text and omits several listed GST categories. Review against the source."),
    ("india-45233283", "Source-content omission - numerical details missing", "English preserves the 3,400 likes, 1,700 retweets and 10-50 age restriction. It omits the Hindi opening figures of 99% and 100%. Restore those source details."),
    ("070620_heart_survival", "Number check passed - no discrepancy found", "Checked 1,676 patients and the three-year follow-up period. Figures match."),
    ("131010_punjab_accident_truck_ssr", "Number check passed - no discrepancy found", "Checked 22 km, 380 km, around 60 passengers, 11:30 pm and 110,000 annual road deaths. Figures match."),
    ("160506_breast_cancer_test_easier_cj", "Number check passed - no discrepancy found", "Checked 10,000 radiologists, 2009, US$4 million investment, and the target of one million women by the end of 2016. Hindi lakh and English million conversions match."),
    ("040510_fillipino_election", "Number check passed - no discrepancy found", "Checked 29, 90, 230,000, 31, 37 and 17,000. Figures match."),
    ("india-43796841", "Number check passed - no discrepancy found", "Checked gold prices of $1,450 and $1,600, 2018 and 2019, 31,524, 18,167 and 29,890 rupees, and the 40% import decline. Figures match after number-format normalization."),
    ("131106_uk_migration_contribution_rd", "Number check passed - no discrepancy found", "Checked 1999, 2000-2001, 45%, 3%, 21%, 34%, 2%, 11%, 1995-2001, four million immigrants and the 2011 census figure of 13%. Four million is the correct conversion of 40 lakh."),
    ("india-49659647", "Source-fidelity review - extra details to verify", "The supplied English preserves the key timeline dates, 45-day period, IPC sections and case chronology. It includes references to POCSO Sections 3 and 4 and an April 3 assault that are not visible in the supplied Hindi text. Verify against the complete source."),
    ("040421_tripura_polling", "Source-content omission - numerical details missing", "English preserves 865 km, 2,372 polling stations, 150 sensitive stations, eight observers and 1952. It omits around 20 lakh voters, approximately 50% women, six candidates per seat and the eight-to-nine historical victories. Restore these details."),
    ("150708_us_army_cuts_aa", "Number check passed - no discrepancy found", "Checked 17,000 civilian employees, 450,000 troops, 570,000 in 2012, 480,000 before the 2001 attacks, 10,000 remaining in Afghanistan and 3,500 in Iraq. Figures match."),
    ("051209_tamilnadu_cyclone", "Number check passed - no discrepancy found", "Checked 550 km, 24 hours, 70 km per hour, 75,000 evacuees, more than 75,000 displaced people and 140 camps. Figures match."),
    ("india-47692221", "Number check passed - no discrepancy found", "Checked 65,000 video views, 22 seconds, around 30 attendees and 10 Lok Sabha seats. Figures match."),
    ("india-50975864", "Number check passed - no discrepancy found", "Checked 19 deaths in Uttar Pradesh, 22 December, 2019, the 1955 Citizenship Act and the 11-to-six-year residence requirement. Figures match."),
    ("060129_nepal_candidate_rj", "Number check passed - no discrepancy found", "Checked 4,146 seats, around 1,000 without candidates, 70 candidates under security, 60% of reserved seats, 36 and 22 municipalities, and 2005 and 2007. Figures match."),
    ("india-45961945", "Number check passed - no discrepancy found", "Checked Rs 9,000 crore, March 2016, 10 December, 114 institutions, 11 selected, six final selections, three consecutive rankings, two hours, 8-10 pm, 70 acres, 73 metres, 60 metres, 12 metres, 80 lakh tonnes and 21 October. Figures match.")
]

new_reviews = pd.DataFrame([
    {
        "id": article_id,
        "batch_number": 7,
        "translation_status": status,
        "reviewer_notes": notes,
        "correction_suggestion": (
            "Review the source and prepare a correction or complete translation where required."
            if any(term in status for term in ["omission", "Missing", "mismatch", "review"])
            else "No numerical correction required."
        )
    }
    for article_id, status, notes in reviews
])

# Save without changing the original translation file
if os.path.exists(audit_path):
    old_reviews = pd.read_csv(audit_path)
    combined = pd.concat([old_reviews, new_reviews], ignore_index=True)
    combined = combined.drop_duplicates(subset=["id"], keep="last")
else:
    combined = new_reviews.copy()

combined.to_csv(audit_path, index=False)

# Validation
print("AUDIT SAVED:", os.path.exists(audit_path))
print("TOTAL AUDIT RECORDS:", len(combined))
print("NEW REVIEWS ADDED:", len(new_reviews))
print("DUPLICATE IDS:", combined["id"].duplicated().sum())
print("\nBATCH 7 STATUS SUMMARY:")
print(new_reviews["translation_status"].value_counts().to_string())
print("\nBATCH 7 REVIEWED IDS:", new_reviews["id"].nunique())

AUDIT SAVED: True
TOTAL AUDIT RECORDS: 71
NEW REVIEWS ADDED: 36
DUPLICATE IDS: 0

BATCH 7 STATUS SUMMARY:
translation_status
Number check passed - no discrepancy found                   24
Source-content omission - numeric detail missing              2
Source-content omission - numerical details missing           2
Missing English translation - correction required             2
Source-content mismatch - further review required             1
Source-fidelity review - extra statistic                      1
Source-content review - omission and extra statistic          1
Source-content review - possible omissions                    1
Source-fidelity review - extra figure and omitted details     1
Source-fidelity review - extra details to verify              1

BATCH 7 REVIEWED IDS: 36


In [544]:
import pandas as pd
import os

audit_path = base_path + "/09_sentence_level_translation/FactIndic_Translation_Audit_230.csv"
audit = pd.read_csv(audit_path)

# Separate records needing attention
attention = audit[
    ~audit["translation_status"].astype(str).str.contains(
        "passed", case=False, na=False
    )
].copy()

attention_path = base_path + "/09_sentence_level_translation/FactIndic_Articles_Needing_Correction_or_Review.csv"
attention.to_csv(attention_path, index=False)

print("FINAL AUDIT RECONCILIATION")
print("=" * 60)
print("Total audited articles:", len(audit))
print("Unique article IDs:", audit["id"].nunique())
print("Duplicate IDs:", audit["id"].duplicated().sum())
print("Articles needing attention:", len(attention))

print("\nALL AUDIT STATUSES:")
print(audit["translation_status"].value_counts().to_string())

print("\nARTICLES NEEDING ATTENTION:")
print(attention[["id", "translation_status"]].to_string(index=False))

print("\nATTENTION FILE SAVED:", os.path.exists(attention_path))
print("FILE:", attention_path)

FINAL AUDIT RECONCILIATION
Total audited articles: 71
Unique article IDs: 71
Duplicate IDs: 0
Articles needing attention: 34

ALL AUDIT STATUSES:
translation_status
Number check passed - no discrepancy found                   36
Verified - minor editorial review                             9
Major source-content omission                                 5
Verified - minor wording correction                           2
Source-content omission - numerical details missing           2
Source-content omission - numeric detail missing              2
Missing English translation - correction required             2
Verified - punctuation correction                             1
Verified - minor source-fidelity correction                   1
Critical factual mismatch                                     1
Verified - meaning and statistics review                      1
Verified - source discrepancy review                          1
Source-content mismatch - further review required             1
Num

In [545]:
import pandas as pd
import os

audit_path = base_path + "/09_sentence_level_translation/FactIndic_Translation_Audit_230.csv"

audit = pd.read_csv(audit_path)

# Exclude successfully checked and already verified records
resolved = (
    audit["translation_status"].astype(str).str.contains(
        "passed", case=False, na=False
    )
    |
    audit["translation_status"].astype(str).str.startswith(
        "Verified", na=False
    )
)

unresolved = audit[~resolved].copy()

attention_path = base_path + "/09_sentence_level_translation/FactIndic_Articles_Needing_Correction_or_Review.csv"
unresolved.to_csv(attention_path, index=False)

print("CORRECTED RECONCILIATION")
print("=" * 55)
print("Total audited:", len(audit))
print("Unique IDs:", audit["id"].nunique())
print("Duplicates:", audit["id"].duplicated().sum())
print("Resolved / verified:", resolved.sum())
print("Unresolved:", len(unresolved))
print("Attention file saved:", os.path.exists(attention_path))

print("\nUNRESOLVED ARTICLES:")
print(unresolved[["id", "translation_status"]].to_string(index=False))

CORRECTED RECONCILIATION
Total audited: 71
Unique IDs: 71
Duplicates: 0
Resolved / verified: 52
Unresolved: 19
Attention file saved: True

UNRESOLVED ARTICLES:
                                  id                                        translation_status
          090216_jobloss_column3_alk                             Major source-content omission
              international-47105506                             Major source-content omission
              international-48243735                             Major source-content omission
              international-49451047                             Major source-content omission
              130606_china_gender_ss                             Major source-content omission
     150816_badminton_saina_final_dp                                 Critical factual mismatch
                 061004_askus_geyser               Numerical discrepancy - correction required
                 050420_pope_muslims         Source-content mismatch - further r

In [546]:
import pandas as pd
import os

audit_path = base_path + "/09_sentence_level_translation/FactIndic_Translation_Audit_230.csv"

audit = pd.read_csv(audit_path)

resolved = (
    audit["translation_status"].astype(str).str.contains(
        "passed", case=False, na=False
    )
    |
    audit["translation_status"].astype(str).str.startswith(
        "Verified", na=False
    )
)

unresolved = audit[~resolved].copy()

# Get the original Hindi articles
source = df_250[["id", "title", "text"]].copy()

# Get the English translations
english = english_translations[
    ["id", "complete_english_translation"]
].copy()

# Combine source, translation and audit notes
review_pack = unresolved.merge(
    source, on="id", how="left"
).merge(
    english, on="id", how="left"
)

review_pack = review_pack[
    [
        "id",
        "translation_status",
        "reviewer_notes",
        "correction_suggestion",
        "title",
        "text",
        "complete_english_translation"
    ]
]

output_path = base_path + "/09_sentence_level_translation/FactIndic_Final_19_Review_Pack.csv"

review_pack.to_csv(output_path, index=False)

print("REVIEW PACK CREATED:", os.path.exists(output_path))
print("ARTICLES:", len(review_pack))
print("UNIQUE IDs:", review_pack["id"].nunique())
print("MISSING HINDI:", review_pack["text"].isna().sum())
print("MISSING ENGLISH:", review_pack["complete_english_translation"].isna().sum())

from google.colab import files
files.download(output_path)

REVIEW PACK CREATED: True
ARTICLES: 19
UNIQUE IDs: 19
MISSING HINDI: 0
MISSING ENGLISH: 0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [547]:
import pandas as pd
import os

audit_path = base_path + "/09_sentence_level_translation/FactIndic_Translation_Audit_230.csv"

audit = pd.read_csv(audit_path)

corrections = {
    "050420_pope_muslims": (
        "Source check passed - previously flagged details are present",
        "The 15 million Muslims and September 11 references both appear in the Hindi source and English translation. The earlier extra-content flag was incorrect.",
        "No correction required for these flagged details."
    ),
    "130528_speed_breaker_cars_impact_sy": (
        "Source check passed - previously flagged detail is present",
        "The English translation explicitly includes at least two or three damaged cars arriving every day. The earlier missing-number flag was incorrect.",
        "No correction required for this numerical detail. This does not certify every sentence of the translation."
    ),
    "050107_pravasi_pm": (
        "Source check passed - previously flagged statistic is present",
        "The Hindi source says more than 2.5 crore Indians live abroad. The English translation correctly gives this as more than 25 million.",
        "No correction required."
    ),
    "150222_parliament_budget_session_pk": (
        "Source check passed - previously flagged detail is present",
        "The Hindi source mentions three earlier joint sessions. The English translation also preserves the three occasions and possible fourth session.",
        "No correction required for this flagged detail."
    ),
    "061217_spl_sez_imballance": (
        "Source check passed - previously flagged details are present",
        "The English translation includes the four states accounting for more than half of approved SEZs and the National Productivity Council ranking across 15 states.",
        "No correction required for these flagged details."
    ),
    "150713_bihar_women_strange_names_rns": (
        "Source check passed - English translation is present",
        "The supplied English column contains a full translation, including the 60-year-old Fenkni Devi. The earlier missing-translation flag was incorrect.",
        "No missing-translation correction required."
    ),
    "071001_pakistan_cargo": (
        "Source check passed - English translation is present",
        "The supplied English column contains a translation and preserves the 60 years since Partition reference. The earlier missing-translation flag was incorrect.",
        "No missing-translation correction required."
    ),
    "140223_modi_rally_punjab_sr": (
        "Source check passed - previously flagged detail is present",
        "The English translation explicitly states that subsidised cooking-gas cylinders increased from nine to twelve.",
        "No correction required."
    ),
    "india-44912345": (
        "Source check passed - previously flagged figure is present",
        "The Rs 2.50 sanitary-pad reference appears in the Hindi related-story headline and is also represented in the English related-story list. It is not an unsupported addition.",
        "No correction required for this figure."
    ),
    "india-45233283": (
        "Source check passed - previously flagged figures are present",
        "The English translation includes 99% of Indians believing in God and 100% believing in astrology, matching the Hindi source.",
        "No correction required."
    ),
    "india-49659647": (
        "Source check passed - previously flagged details are present",
        "The Hindi source itself includes POCSO Sections 3 and 4 and the April 3 incident. These details are also present in English.",
        "No correction required."
    ),
    "040421_tripura_polling": (
        "Source check passed - previously flagged figures are present",
        "The English translation includes approximately two million voters, about half women, six candidates per seat and eight or nine historical Communist Party victories.",
        "No correction required."
    )
}

for article_id, (status, notes, suggestion) in corrections.items():
    mask = audit["id"].astype(str) == article_id
    audit.loc[mask, "translation_status"] = status
    audit.loc[mask, "reviewer_notes"] = notes
    audit.loc[mask, "correction_suggestion"] = suggestion

audit.to_csv(audit_path, index=False)

# Recalculate unresolved articles
resolved = (
    audit["translation_status"].astype(str).str.contains(
        "passed", case=False, na=False
    )
    |
    audit["translation_status"].astype(str).str.startswith(
        "Verified", na=False
    )
)

unresolved = audit[~resolved].copy()

attention_path = base_path + "/09_sentence_level_translation/FactIndic_Articles_Needing_Correction_or_Review.csv"
unresolved.to_csv(attention_path, index=False)

# Prepare the final seven-article review pack
source = df_250[["id", "title", "text"]].copy()

english = english_translations[
    ["id", "complete_english_translation"]
].copy()

final_pack = unresolved.merge(
    source, on="id", how="left"
).merge(
    english, on="id", how="left"
)

final_pack_path = base_path + "/09_sentence_level_translation/FactIndic_Final_7_Unresolved_Review_Pack.csv"

final_pack.to_csv(final_pack_path, index=False)

print("AUDIT UPDATED:", os.path.exists(audit_path))
print("TOTAL AUDIT RECORDS:", len(audit))
print("UNIQUE IDS:", audit["id"].nunique())
print("DUPLICATE IDS:", audit["id"].duplicated().sum())
print("PREVIOUS FLAGS RESOLVED:", len(corrections))
print("REMAINING UNRESOLVED:", len(unresolved))
print("FINAL REVIEW PACK SAVED:", os.path.exists(final_pack_path))

print("\nREMAINING ARTICLES:")
print(unresolved[["id", "translation_status"]].to_string(index=False))

from google.colab import files
files.download(final_pack_path)

AUDIT UPDATED: True
TOTAL AUDIT RECORDS: 71
UNIQUE IDS: 71
DUPLICATE IDS: 0
PREVIOUS FLAGS RESOLVED: 12
REMAINING UNRESOLVED: 7
FINAL REVIEW PACK SAVED: True

REMAINING ARTICLES:
                             id                          translation_status
     090216_jobloss_column3_alk               Major source-content omission
         international-47105506               Major source-content omission
         international-48243735               Major source-content omission
         international-49451047               Major source-content omission
         130606_china_gender_ss               Major source-content omission
150816_badminton_saina_final_dp                   Critical factual mismatch
            061004_askus_geyser Numerical discrepancy - correction required


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [548]:
import pandas as pd

remaining_ids = [
    "090216_jobloss_column3_alk",
    "international-47105506",
    "international-48243735",
    "international-49451047",
    "130606_china_gender_ss",
    "150816_badminton_saina_final_dp",
    "061004_askus_geyser"
]

source = df_250.set_index("id")
english = english_translations.set_index("id")

for article_id in remaining_ids:
    print("\n" + "=" * 110)
    print("ARTICLE ID:", article_id)
    print("=" * 110)

    if article_id in source.index:
        print("\nHINDI TITLE:\n", source.loc[article_id, "title"])
        print("\nHINDI SOURCE:\n", source.loc[article_id, "text"])
    else:
        print("\nHINDI SOURCE NOT FOUND")

    if article_id in english.index:
        print("\nENGLISH TRANSLATION:\n",
              english.loc[article_id, "complete_english_translation"])
    else:
        print("\nENGLISH TRANSLATION NOT FOUND")

    print("\n" + "-" * 110)


ARTICLE ID: 090216_jobloss_column3_alk

HINDI TITLE:
 'तलाश अभी जारी है...'

HINDI SOURCE:
 नौकरी रहते हुए नई नौकरी की तलाश तो फिर भी आसान है, लेकिन सड़क पर आकर नौकरी ढूँढना काफ़ी मुश्किल. इसीलिए मैंने अपने एचआर वालों से एक विनती भी की थी, तीन महीने का नोटिस पीरियड दे दो, सैलरी बेशक मत दो लेकिन नई नौकरी ढूँढने का मौका दे दो. पर ऐसा नहीं हुआ. दरअसल मेरे ऑफ़िस वालों को पता था कि मेरे पापा रिटायर्ड जज हैं. पता नहीं क्यों इस बात को लेकर उनके मन में संशय भी था कि मैं उन्हें अदालत में ना घसीट लूँ. इसलिए मेरे मामले में इस्तीफ़े से संबंधित सभी कागज़ी कार्रवाई को वो हू-बहू अमल में भी ला रहे थे. मुझे कोई रियायत नहीं दी गई. नौकरी ढूँढने का सिलसिला तो दो दिनों बाद ही शुरु हो गया. शुरुआत मित्रों को बताने और अपने ही सर्किल में कोई नौकरी खोजने की कोशिश के साथ हुई. 'सेट' कराने के आश्वासन शुरु-शुरु में कुछ दोस्तों ने अपने यहाँ 'सेट' करवा देने का आश्वासन दिया मगर 15 दिन बीत गए और कहीं कुछ नहीं बना. सारे दोस्तों ने कुछ दिन और इंतज़ार करने की सलाह दी या कुछ का कहना था.. "अरे अभी पाँच दिन पहले एक की गुंजा

In [549]:
import pandas as pd
import os

# Load original translation data
english_path = base_path + "/09_sentence_level_translation/FactIndic_English_250_Reviewed.csv"
audit_path = base_path + "/09_sentence_level_translation/FactIndic_Translation_Audit_230.csv"

english = pd.read_csv(english_path)
audit = pd.read_csv(audit_path)

# Complete corrected translations
corrections = {

"090216_jobloss_column3_alk": """Finding a new job while you are still employed is difficult enough, but looking for work after losing your job is much harder. That is why I had requested my HR department to give me a three-month notice period. I told them they did not need to pay me a salary during that period, but at least I would have time to find another job. However, they did not agree.

My colleagues knew that my father was a retired judge. I do not know why, but perhaps they were concerned that I might take them to court. As a result, they followed every formal procedure relating to my resignation exactly. I was given no special consideration.

I started looking for a job just two days later. Initially, I informed my friends and tried to find something through my own professional circle. Some friends promised to help me get a position at their workplaces. But 15 days passed and nothing worked out. All my friends advised me to wait a little longer. Some said, "There was an opening just five days ago. You were a little late."

I understood the situation and started visiting offices myself. Before doing so, I had registered on several job websites. After that, all I could do was wait.

Ten days later, I received a call from a placement agency. For the first time in my career, apart from the initial days of starting work, I felt nervous before an interview. Perhaps this is one of the side effects of having to search for a job after becoming unemployed.

I went expecting an interview, but instead I was handed a written examination paper. To be honest, I felt embarrassed about having to take a written test at this stage of my career. But circumstances left me with little choice.

The interview itself seemed to go well. When I returned home, I kept wondering whether I would be successful. For the first time, I was afraid of failing. And that is exactly what happened.

Before leaving, I had been told that I would know the result within two or three days. On the morning of the fourth day, I lost patience and called them myself. The response was: "We haven't called you, which probably means we have selected someone else."

I said that I thought my interview had gone well. They replied, "Yes, your interview was fine, but your profile did not match."

After that, I tried to prove myself at several other offices, but I was unsuccessful. My search is still continuing.

This account was based on Ajay's experience, as told to BBC correspondent Alok Kumar. From the following Friday, Vikas Shankar, who had worked as a manager in a private bank and had also lost his job, was due to share his story.""",

"international-47105506": """Why has the issue of Haj subsidies caused such an uproar in Pakistan?

First, the increase in the cost of Haj. The rising expense of the pilgrimage has become a major issue in Pakistan. After coming to power, Imran Khan's government announced its first Haj policy this week. Under the new policy, a Pakistani pilgrim will have to pay 476,000 Pakistani rupees to perform Haj this year, compared with 280,000 rupees last year.

The issue caused considerable uproar in the Pakistani Parliament. According to the newspaper Nawa-i-Waqt, Mushtaq Ahmed, a parliamentarian from the opposition Jamaat-e-Islami, said that the government had launched a "drone attack" on Haj pilgrims. He said the government had provided no subsidy for Haj. According to him, the Minister for Religious Affairs had recommended a subsidy in his report, but Imran Khan's government rejected its own minister's recommendation.

The newspaper quoted Mushtaq Ahmed as sarcastically saying that people who claimed they wanted to establish the State of Medina were now preventing people from travelling to Mecca and Medina.

In his first speech after becoming Prime Minister, Imran Khan had said that he wanted to govern Pakistan along the lines of the way Prophet Muhammad had governed Medina.

Responding to the opposition, Parliamentary Affairs Minister Ali Muhammad Khan said that the cost of Haj had increased in Saudi Arabia itself, which was why pilgrims were being asked to pay more.

However, according to the newspaper Jang, Religious Affairs Minister Noor-ul-Haq Qadri said that the concept of the State of Medina did not mean that people should be sent for Haj free of charge or with a subsidy. He argued that performing Haj with a subsidy went against its basic spirit.

**Foreign Minister Shah Mahmood Qureshi and the Kashmir issue in the British Parliament**

Pakistan has also said that an event will be organised in the British Parliament in support of people living in Indian-administered Kashmir.

According to the newspaper Daily Khabrain, a special programme in support of people living in Indian-administered Kashmir will be held on 4 February in the House of Commons, the lower house of the British Parliament, under the All-Party Parliamentary Group. Pakistan's Foreign Minister Shah Mahmood Qureshi will also attend.

The newspaper reported that India had written to Britain asking it to stop the event, but the British government rejected the request.

A spokesperson for the British High Commission in New Delhi said that British parliamentarians had the right to decide whom they met and what they discussed, and that the British government could not interfere in this.

The newspaper also reported that the British Foreign Office had issued a press release saying that an exhibition would be held in London after the 4 February event. It would highlight alleged human rights violations in Indian-administered Kashmir.

A British spokesperson said that the UK was aware that Pakistani Foreign Minister Qureshi was travelling to London to attend the event. However, the visit was private, and he would not meet the British Prime Minister or Foreign Secretary during it.

On 29 January, the Pakistani Foreign Minister had spoken by telephone with Mirwaiz Umar Farooq, a leader of the Hurriyat Conference active in Indian-administered Kashmir. Qureshi had said that his government would raise the Kashmir issue at every international forum.

India strongly objected to the conversation and summoned Pakistan's High Commissioner in India to the Ministry of External Affairs. In response, Pakistani Foreign Secretary Tehmina Janjua also summoned the Indian High Commissioner to her office in Islamabad.

**The Taliban say they are not opposed to women working**

News related to an agreement between the United States and the Taliban also dominated newspaper coverage throughout the week.

According to Nawa-i-Waqt, Taliban spokesperson Zabiullah Mujahid said that US President Donald Trump was serious about withdrawing American troops from Afghanistan.

The spokesperson said that after the withdrawal of US forces, the Taliban would implement Islamic law in Afghanistan, but only after consulting all political parties.

He also said that the Taliban were no longer opposed to women receiving education or working. Their government would make every effort to create an environment in which women in Afghanistan could feel safe working.

According to the newspaper, the Taliban spokesperson said that the next round of talks with the United States would take place in Doha on 25 February, although the date had not yet been confirmed by the US side.""",

"international-48243735": """Iranian President Hassan Rouhani has acknowledged that conditions in his country have become worse than during the Iran-Iraq War of 1980-88 because of new American sanctions.

His statement came amid growing tensions with the United States, following the deployment of warships and fighter aircraft in the Gulf a few days earlier. Facing domestic political pressure, Rouhani has called for political unity to confront the sanctions.

Speaking to political activists in Tehran, Rouhani said that during the war, Iran had not faced difficulties with banking, oil or imports; restrictions had applied only to the purchase of weapons.

He said that the pressure being imposed by Iran's enemies amounted to a war unprecedented in the history of the Islamic Revolution. However, he said that Iran was not discouraged and that he remained hopeful about the future. He expressed confidence that the country could overcome these difficulties if it remained united.

Tensions between the United States and Iran have put the 2015 nuclear agreement in doubt. Iran signed the agreement with the five permanent members of the UN Security Council and Germany.

Last year, US President Donald Trump unilaterally withdrew from the agreement and imposed new sanctions on Iran. In response, Iran indicated that it might restart its nuclear programme if the other countries involved in the agreement also supported the American sanctions.

**What kind of pressure is Iran facing?**

Since the United States withdrew from the agreement, hard-line groups in Iran have increased pressure on President Rouhani, because the agreement was reached during his presidency.

Under the agreement, Iran had agreed to limit sensitive parts of its nuclear programme and allow international inspectors into the country. In return, sanctions against Iran were eased.

American sanctions on the energy, shipping and financial sectors have affected Iran's oil exports and reduced foreign investment to very low levels. The sanctions also prevent American companies from doing business with Iran and with companies that trade directly with it.

The International Monetary Fund estimated that Iran's economy would contract by 6 percent in 2019. This estimate could become even worse because the United States has tightened its sanctions further.

For example, the United States had allowed China, India, Japan, South Korea and Turkey to continue buying Iranian oil. That exemption has now ended.

Last month, the United States placed Iran's influential Islamic Revolutionary Guard Corps on its list of foreign terrorist organisations.

In response to American sanctions, Iran has repeatedly threatened to disrupt trade through the Strait of Hormuz, through which 20 percent of the world's required oil supply passes.

**What is the United States doing?**

The Trump administration wants to pressure Iran into negotiating a new agreement. This would cover not only its nuclear programme but also its ballistic missile programme, which the United States considers a source of tension in the Middle East.

The United States is sending its Patriot missile defence system to the Middle East. It has also sent the warship USS Arlington to the Gulf. The vessel carries amphibious equipment and fighter aircraft and will be deployed in the Gulf alongside the USS Abraham Lincoln.

B-52 bombers have also been sent to the American military base in Qatar.

The United States said these steps were taken in response to potential threats from Iran to American forces in the region. Iran rejected the American claim.

At present, 5,200 American soldiers are stationed in neighbouring Iraq.

Earlier this week, Iran announced that it was withdrawing from two of its commitments under the nuclear agreement. It threatened that if sanctions were not eased within 60 days, it would increase uranium production.

European powers said that they still supported the Iran nuclear agreement, but rejected any threats from Tehran as a way of trying to save it.""",

"international-49451047": """Why has Pakistan objected to the UAE giving Narendra Modi its highest honour?

Indian Prime Minister Narendra Modi became the first Indian Prime Minister to receive the Order of Zayed. The Order of Zayed is the United Arab Emirates' highest civilian honour and is awarded to kings, presidents and heads of state.

Before Modi, the honour had been given to Russian President Vladimir Putin in 2007, Britain's Queen Elizabeth in 2010, Saudi King Salman bin Abdulaziz Al Saud in 2016 and Chinese President Xi Jinping in 2018.

The UAE established the award in 1995.

**Why was Prime Minister Modi selected for the honour?**

According to Abu Dhabi Crown Prince Sheikh Mohammed bin Zayed Al Nahyan, Prime Minister Modi had made an important contribution to strengthening bilateral relations between India and the UAE.

In April this year, Sheikh Mohammed bin Zayed announced the award in a tweet. He wrote that India and the UAE shared historic and extensive strategic relations, to which his close friend Narendra Modi had made a significant contribution. He said Modi had strengthened these ties and that UAE President Sheikh Khalifa bin Zayed Al Nahyan had awarded him the Order of Zayed in recognition of his efforts.

In a statement, India's Ministry of External Affairs said that this year's award was of special importance because it was being presented to Modi during the birth centenary year of Sheikh Zayed.

The UAE had declared 2019 the Year of Tolerance.

The ministry's statement also said that the UAE was India's third-largest trading partner. The two countries had strong bilateral trade relations, and their annual trade was expected to reach around 60 billion dollars.

**India and the UAE during Modi's tenure**

Diplomatic visits between the leaders of the two countries increased significantly during Modi's time in office.

Sheikh Mohammed bin Zayed visited India twice in the preceding three years. His first visit was in February 2016. He visited again in 2017 to attend India's Republic Day celebrations as the chief guest.

Modi was the first Indian Prime Minister to visit the UAE twice. He first travelled there in August 2015. In 2018, the UAE invited him as the chief guest at the World Government Summit. This was his third visit in less than five years.

During the visit, Modi met the Crown Prince. They discussed bilateral, regional and international issues of mutual interest.

The Indian Embassy in Abu Dhabi also issued a commemorative postage stamp in Modi's presence to mark the 150th birth anniversary of Mahatma Gandhi.

Modi also launched the RuPay card in the UAE. Like MasterCard and Visa, RuPay is India's own card-based payment system.

India's Ambassador to the UAE, Navdeep Suri, said that although the Zayed award was one reason for the Prime Minister's visit, Modi also personally wanted to promote the RuPay card.

**Pakistan's objection**

The news that Narendra Modi had received the UAE's highest honour came as a surprise to Pakistan. This happened amid the Indian government's decision to make the provisions of Article 370, which had granted special status to Jammu and Kashmir, ineffective, and allegations of human rights violations in Indian-administered Kashmir.

Dr Umbreen Javed, a political science expert in Lahore, told the BBC that although Narendra Modi had visited a country with which India had extensive economic relations, the Kashmir issue had made the matter particularly emotional for Pakistan.

She said that Pakistan believed human rights violations in Indian-administered Kashmir could increase and that Kashmiris might face considerable difficulties in the future.

Some in Pakistan had expected Muslim countries to try to persuade the Indian Prime Minister to reverse his decision on Article 370 rather than award him their highest civilian honour.

Recently, Naz Shah, a British Labour Party MP, had written to the UAE Crown Prince asking him to reconsider giving the Zayed Medal to Narendra Modi. She had accused Modi of human rights violations in Kashmir.

However, the Pakistani government remained silent on the matter. Pakistan's Foreign Ministry spokesperson, Dr Mohammad Faisal, said that the issue concerned India and the UAE and that Pakistan would not comment on it.

Since the special status of Jammu and Kashmir under Article 370 of the Indian Constitution was ended, hashtags relating to Kashmir and Prime Minister Narendra Modi had been trending on Twitter.

Some social media users also expressed dissatisfaction over Modi receiving the UAE's highest honour.

Twitter user Tahir Naqvi wrote that by giving Modi the highest honour, the Arab countries were effectively endorsing the situation.

Another Twitter user, Iman Hazir Mazari, said that she was not surprised by the UAE's response to the situation in Kashmir. She argued that there was no single unified Muslim community and criticised the UAE and Saudi Arabia over their positions on the issue.""",

"130606_china_gender_ss": """A 66-year-old man in China discovered that he was genetically female after visiting hospital with abdominal pain.

Doctors said that the pain was caused by a cyst in his ovaries. It was then discovered that the person was genetically female rather than male.

The story was published in the South China Morning Post. The Hong Kong Medical Journal described the person's condition as an extremely rare case.

According to a journal on genetic disorders, the condition resulted from the combination of two genetic disorders. One of them is Turner syndrome. People with this syndrome may have certain female physical characteristics but may be unable to have children and may not experience typical female physical development.

People with Turner syndrome may look female, but their physical characteristics may not be typically feminine.

However, this older person's case was different because he also had congenital adrenal hyperplasia. This condition can cause the body to produce higher levels of male hormones. When female and male hormones are both present in unusual patterns, it can result in an uncommon combination of characteristics.

**A male appearance and behaviour**

Doctors found that the person had a genetic disorder. He was four feet five inches tall and had a slim build. He looked and behaved like a man and had not known for many years that he had characteristics associated with a female reproductive system.

He had difficulty urinating and experienced other health problems. However, he had never considered that a genetic disorder might be the main reason for his difficulties. He had always regarded himself as a man.

Doctors at Kwong Wah Hospital and Queen Elizabeth Hospital in Hong Kong said that the person had ovaries and would have had female physical characteristics.

According to reports, the person said that he considered himself a man and would now receive treatment involving male hormones.""",

"150816_badminton_saina_final_dp": """Saina Nehwal lost to Spain's Carolina Marin in the final of the World Badminton Championships in Jakarta, Indonesia. Marin defeated her in straight games, 21-16 and 21-19, in a match that lasted 59 minutes.

Saina, the world number two, made a good start in both games, but her performance weakened as the match progressed.

Saina became the first Indian player to reach the final of the World Championships, but she missed out on the gold medal. This was India's fifth medal at the World Championships.

Before this, P. V. Sindhu had won bronze medals at the tournament in 2013 and 2014. In 2011, the doubles pair of Jwala Gutta and Ashwini Ponnappa had also won bronze. Prakash Padukone became the first Indian to win a bronze medal at the World Championships in 1983.

Saina started the first game well and took a 3-1 lead. However, her performance then began to falter. Marin, the world number one, took an 11-5 lead and did not look back, winning the first game 21-16.

**A close contest**

Saina again started the second game strongly and took a 12-6 lead. Marin then won a series of points to make the score 13-12.

The two players fought hard for every point. At one stage, Saina was leading 18-17. However, Marin won three consecutive points to move ahead 20-18.

Marin eventually won the second game 21-19 and secured the gold medal.""",

"061004_askus_geyser": None
}

# Correct the geyser translation using the original text
geyser_id = "061004_askus_geyser"
geyser_text = english.loc[
    english["id"].astype(str) == geyser_id,
    "complete_english_translation"
].iloc[0]

geyser_text = geyser_text.replace("£460,000", "£4,600,000")
corrections[geyser_id] = geyser_text

# Update a COPY of the original English dataset
corrected = english.copy()

for article_id, translation in corrections.items():
    mask = corrected["id"].astype(str) == article_id
    if mask.sum() != 1:
        raise ValueError(f"Expected exactly one row for {article_id}, found {mask.sum()}")

    corrected.loc[mask, "complete_english_translation"] = translation

# Save separate corrected file
corrected_path = base_path + "/09_sentence_level_translation/FactIndic_English_250_Corrected.csv"
corrected.to_csv(corrected_path, index=False)

# Update audit statuses: prepared, not yet independently verified
audit_updates = {
    "090216_jobloss_column3_alk": (
        "Corrected translation prepared - pending final review",
        "A complete first-person translation has been prepared, restoring the three-month notice request, 15-day wait, placement agency, written test, rejection and continued job search."
    ),
    "international-47105506": (
        "Corrected translation prepared - pending final review",
        "A complete translation has been prepared for the Haj subsidy debate, Kashmir parliamentary event and Taliban statements on women's education and employment."
    ),
    "international-48243735": (
        "Corrected translation prepared - pending final review",
        "A complete translation has been prepared restoring the 1980-88 war comparison, 2015 nuclear agreement, sanctions, IMF estimate, oil trade, military deployments, 5,200 troops and 60-day deadline."
    ),
    "international-49451047": (
        "Corrected translation prepared - pending final review",
        "A fuller translation has been prepared restoring award history, diplomatic visits, approximately 60-billion-dollar trade, RuPay launch and Pakistan's reactions."
    ),
    "130606_china_gender_ss": (
        "Corrected translation prepared - pending final review",
        "A fuller translation has been prepared restoring the source's medical explanations, reported symptoms, height, hospital details and treatment statement. Medical claims remain attributed to the source."
    ),
    "150816_badminton_saina_final_dp": (
        "Corrected translation prepared - factual mismatch corrected",
        "The incorrect semi-final account has been replaced with the Hindi source's final result: Carolina Marin defeated Saina Nehwal 21-16 and 21-19 in 59 minutes."
    ),
    "061004_askus_geyser": (
        "Numerical discrepancy corrected - pending final review",
        "The Canterbury Tales first-edition auction price has been corrected from £460,000 to £4,600,000, matching the Hindi source's 46 lakh pounds."
    )
}

for article_id, (status, notes) in audit_updates.items():
    mask = audit["id"].astype(str) == article_id
    if mask.sum() != 1:
        raise ValueError(f"Audit must contain exactly one row for {article_id}")

    audit.loc[mask, "translation_status"] = status
    audit.loc[mask, "reviewer_notes"] = notes
    audit.loc[mask, "correction_suggestion"] = "Corrected translation is in FactIndic_English_250_Corrected.csv. Conduct final review before treating as verified."

audit.to_csv(audit_path, index=False)

# Validation
check = pd.read_csv(corrected_path)
audit_check = pd.read_csv(audit_path)

print("CORRECTED TRANSLATION FILE SAVED:", os.path.exists(corrected_path))
print("ORIGINAL FILE PRESERVED:", os.path.exists(english_path))
print("CORRECTED DATASET ROWS:", len(check))
print("CORRECTED ARTICLE COUNT:", len(corrections))
print("AUDIT RECORDS:", len(audit_check))
print("UNIQUE AUDIT IDS:", audit_check["id"].nunique())
print("DUPLICATE AUDIT IDS:", audit_check["id"].duplicated().sum())

print("\nCORRECTION CHECKS:")
for article_id in corrections:
    row = check[check["id"].astype(str) == article_id]
    print(article_id, "|", "UPDATED" if len(row) == 1 else "CHECK REQUIRED")

print("\nGEYSER PRICE CORRECTED:",
      "£4,600,000" in check.loc[
          check["id"].astype(str) == geyser_id,
          "complete_english_translation"
      ].iloc[0])

print("\nUPDATED AUDIT STATUSES:")
print(audit_check[
    audit_check["id"].astype(str).isin(corrections.keys())
][["id", "translation_status"]].to_string(index=False))

CORRECTED TRANSLATION FILE SAVED: True
ORIGINAL FILE PRESERVED: True
CORRECTED DATASET ROWS: 250
CORRECTED ARTICLE COUNT: 7
AUDIT RECORDS: 71
UNIQUE AUDIT IDS: 71
DUPLICATE AUDIT IDS: 0

CORRECTION CHECKS:
090216_jobloss_column3_alk | UPDATED
international-47105506 | UPDATED
international-48243735 | UPDATED
international-49451047 | UPDATED
130606_china_gender_ss | UPDATED
150816_badminton_saina_final_dp | UPDATED
061004_askus_geyser | UPDATED

GEYSER PRICE CORRECTED: True

UPDATED AUDIT STATUSES:
                             id                                          translation_status
     090216_jobloss_column3_alk       Corrected translation prepared - pending final review
         international-47105506       Corrected translation prepared - pending final review
         international-48243735       Corrected translation prepared - pending final review
         international-49451047       Corrected translation prepared - pending final review
         130606_china_gender_ss       

In [550]:
import os

folder = base_path + "/09_sentence_level_translation"

print("TRANSLATION AUDIT FILES")
print("=" * 80)

for filename in sorted(os.listdir(folder)):
    if any(word in filename.lower() for word in ["correct", "audit", "review", "attention"]):
        path = os.path.join(folder, filename)
        print(f"{filename} | {os.path.getsize(path):,} bytes")

TRANSLATION AUDIT FILES
FactIndic_Articles_Needing_Correction_or_Review.csv | 2,797 bytes
FactIndic_English_250_Corrected.csv | 602,386 bytes
FactIndic_English_250_Reviewed.csv | 587,593 bytes
FactIndic_Final_19_Review_Pack.csv | 193,529 bytes
FactIndic_Final_7_Unresolved_Review_Pack.csv | 70,954 bytes
FactIndic_Final_Quality_Audit.csv | 2,028,207 bytes
FactIndic_High_Priority_4_Review.csv | 43,787 bytes
FactIndic_Number_Review_Pack_Remaining.csv | 351,655 bytes
FactIndic_Translation_Audit_230.csv | 17,972 bytes
FactIndic_Translation_Corrections_Draft.csv | 61,633 bytes
FactIndic_Translation_Review_Priority_216.csv | 1,861,144 bytes


In [551]:
import pandas as pd
import os

folder = base_path + "/09_sentence_level_translation"

corrected_path = folder + "/FactIndic_English_250_Corrected.csv"
original_path = folder + "/FactIndic_English_250_Reviewed.csv"
audit_path = folder + "/FactIndic_Translation_Audit_230.csv"

corrected = pd.read_csv(corrected_path)
original = pd.read_csv(original_path)
audit = pd.read_csv(audit_path)

target_ids = [
    "090216_jobloss_column3_alk",
    "international-47105506",
    "international-48243735",
    "international-49451047",
    "130606_china_gender_ss",
    "150816_badminton_saina_final_dp",
    "061004_askus_geyser"
]

print("=" * 65)
print("FINAL CORRECTED TRANSLATION VALIDATION")
print("=" * 65)

print("\nDATASET CHECKS")
print("Corrected rows:", len(corrected))
print("Unique corrected IDs:", corrected["id"].astype(str).nunique())
print("Duplicate corrected IDs:", corrected["id"].astype(str).duplicated().sum())

translation_col = "complete_english_translation"

if translation_col in corrected.columns:
    print("Missing translations:", corrected[translation_col].isna().sum())
    print("Empty translations:",
          corrected[translation_col].fillna("").astype(str).str.strip().eq("").sum())
else:
    print("Translation column not found. Available columns:", corrected.columns.tolist())

print("\nSEVEN CORRECTION CHECKS")
corrected_ids = set(corrected["id"].astype(str))
audit_ids = set(audit["id"].astype(str))

for article_id in target_ids:
    exists_dataset = article_id in corrected_ids
    exists_audit = article_id in audit_ids
    print(article_id, "| Dataset:", exists_dataset, "| Audit:", exists_audit)

print("\nAUDIT CHECKS")
print("Audit records:", len(audit))
print("Unique audit IDs:", audit["id"].astype(str).nunique())
print("Duplicate audit IDs:", audit["id"].astype(str).duplicated().sum())

print("\nORIGINAL FILE CHECK")
print("Original file exists:", os.path.exists(original_path))
print("Original rows:", len(original))

print("\nCHANGES BETWEEN ORIGINAL AND CORRECTED")
if translation_col in corrected.columns and translation_col in original.columns:
    comparison = original[["id", translation_col]].merge(
        corrected[["id", translation_col]],
        on="id",
        how="inner",
        suffixes=("_original", "_corrected")
    )
    changed = (
        comparison[translation_col + "_original"].fillna("").astype(str)
        != comparison[translation_col + "_corrected"].fillna("").astype(str)
    )
    print("Rows with changed translations:", int(changed.sum()))
    print("Rows unchanged:", int((~changed).sum()))

print("\nVALIDATION CELL COMPLETED")

FINAL CORRECTED TRANSLATION VALIDATION

DATASET CHECKS
Corrected rows: 250
Unique corrected IDs: 250
Duplicate corrected IDs: 0
Missing translations: 0
Empty translations: 0

SEVEN CORRECTION CHECKS
090216_jobloss_column3_alk | Dataset: True | Audit: True
international-47105506 | Dataset: True | Audit: True
international-48243735 | Dataset: True | Audit: True
international-49451047 | Dataset: True | Audit: True
130606_china_gender_ss | Dataset: True | Audit: True
150816_badminton_saina_final_dp | Dataset: True | Audit: True
061004_askus_geyser | Dataset: True | Audit: True

AUDIT CHECKS
Audit records: 71
Unique audit IDs: 71
Duplicate audit IDs: 0

ORIGINAL FILE CHECK
Original file exists: True
Original rows: 250

CHANGES BETWEEN ORIGINAL AND CORRECTED
Rows with changed translations: 7
Rows unchanged: 243

VALIDATION CELL COMPLETED


In [552]:
import os

expected_files = [
    "FactIndic_English_250_Corrected.csv",
    "FactIndic_Translation_Audit_230.csv",
    "FactIndic_Mixed_23_Validated.csv",
    "FactIndic_Manual_Sentence_Alignment.csv",
    "FactIndic_Strict_Approved_Sentence_Pairs.csv",
    "FactIndic_Hinglish_95_Human_Review.csv",
    "FactIndic_50_Claim_Labeling_Sheet.csv",
    "CMI_MUTANT_PHINC_Comparison.csv",
    "FactIndic_Per_Article_CMI_23.csv",
    "FactIndic_PersonB_Final_Report.txt"
]

print("FACTINDIC — FINAL DELIVERABLE CHECK")
print("=" * 65)

for filename in expected_files:
    matches = []
    for root, dirs, files in os.walk(base_path):
        if filename in files:
            matches.append(os.path.join(root, filename))

    if matches:
        for path in matches:
            print("FOUND  |", filename, "|", os.path.getsize(path), "bytes")
    else:
        print("MISSING|", filename)

print("=" * 65)
print("FILE CHECK COMPLETED")

FACTINDIC — FINAL DELIVERABLE CHECK
FOUND  | FactIndic_English_250_Corrected.csv | 602386 bytes
FOUND  | FactIndic_Translation_Audit_230.csv | 651 bytes
FOUND  | FactIndic_Translation_Audit_230.csv | 17972 bytes
FOUND  | FactIndic_Mixed_23_Validated.csv | 270218 bytes
FOUND  | FactIndic_Manual_Sentence_Alignment.csv | 517475 bytes
FOUND  | FactIndic_Strict_Approved_Sentence_Pairs.csv | 180563 bytes
FOUND  | FactIndic_Hinglish_95_Human_Review.csv | 230650 bytes
FOUND  | FactIndic_50_Claim_Labeling_Sheet.csv | 308425 bytes
FOUND  | CMI_MUTANT_PHINC_Comparison.csv | 306 bytes
FOUND  | FactIndic_Per_Article_CMI_23.csv | 1898 bytes
FOUND  | FactIndic_PersonB_Final_Report.txt | 8576 bytes
FILE CHECK COMPLETED


In [553]:
import os
import pandas as pd

folder = base_path + "/09_sentence_level_translation"

files_to_check = [
    "FactIndic_Translation_Audit_230.csv",
    "FactIndic_Mixed_23_Validated.csv",
    "FactIndic_Manual_Sentence_Alignment.csv",
    "FactIndic_Strict_Approved_Sentence_Pairs.csv",
    "FactIndic_Hinglish_95_Human_Review.csv",
    "FactIndic_50_Claim_Labeling_Sheet.csv",
    "CMI_MUTANT_PHINC_Comparison.csv",
    "FactIndic_Per_Article_CMI_23.csv"
]

print("=" * 75)
print("FACTINDIC FINAL CONTENT INSPECTION")
print("=" * 75)

for filename in files_to_check:
    matches = []
    for root, dirs, filenames in os.walk(base_path):
        if filename in filenames:
            matches.append(os.path.join(root, filename))

    for path in matches:
        print("\nFILE:", path)
        try:
            df = pd.read_csv(path)
            print("Rows:", len(df))
            print("Columns:", df.columns.tolist())
            print("Missing cells:", int(df.isna().sum().sum()))
            print("Preview:")
            print(df.head(3).to_string(index=False))
        except Exception as e:
            print("READ ERROR:", e)

print("\n" + "=" * 75)
print("INSPECTION COMPLETED")

FACTINDIC FINAL CONTENT INSPECTION

FILE: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/05_quality_checks/FactIndic_Translation_Audit_230.csv
Rows: 3
Columns: ['id', 'review_status', 'meaning_preserved', 'names_correct', 'numbers_correct', 'correction_required', 'review_notes']
Missing cells: 0
Preview:
                                       id                       review_status meaning_preserved names_correct numbers_correct correction_required                                                                                                           review_notes
                 080419_brchopra_festival   Verified - minor editorial review               Yes           Yes             Yes                  No                                Film titles, names, years and reported statements accurately preserve the Hindi source.
                       080613_afghan_jail   Verified - minor editorial review               Yes           Yes             Yes                  No Meaning and att

In [554]:
import os
import pandas as pd

base = base_path
quality = base + "/05_quality_checks"
mixed = base + "/04_mixed_data"
hinglish_path = base + "/06_real_hinglish/FactIndic_Hinglish_95_Human_Review.csv"

# 1. Complete the aggregate CMI comparison
per_article_path = quality + "/FactIndic_Per_Article_CMI_23.csv"
comparison_path = quality + "/CMI_MUTANT_PHINC_Comparison.csv"

per_article = pd.read_csv(per_article_path)
comparison = pd.read_csv(comparison_path)

new_rows = []
for level, column in [
    ("FactIndic 10%", "FactIndic_10_CMI"),
    ("FactIndic 30%", "FactIndic_30_CMI"),
    ("FactIndic 50%", "FactIndic_50_CMI")
]:
    values = pd.to_numeric(per_article[column], errors="coerce").dropna()
    new_rows.append({
        "Dataset": level,
        "Records": len(values),
        "Mean_CMI": round(values.mean(), 2),
        "Median_CMI": round(values.median(), 2)
    })

comparison = comparison[
    ~comparison["Dataset"].astype(str).str.startswith("FactIndic ")
]
comparison = pd.concat(
    [comparison, pd.DataFrame(new_rows)],
    ignore_index=True
)

final_comparison_path = quality + "/CMI_MUTANT_PHINC_FactIndic_Comparison_Final.csv"
comparison.to_csv(final_comparison_path, index=False)

print("FINAL CMI COMPARISON")
print(comparison.to_string(index=False))
print("\nSaved:", os.path.exists(final_comparison_path))

# 2. Hinglish missing-value audit
hinglish = pd.read_csv(hinglish_path)

print("\nHINGLISH DATASET")
print("Rows:", len(hinglish))
print("Unique texts:", hinglish["text"].nunique())
print("\nMissing values by column:")
print(hinglish.isna().sum().to_string())

print("\nHuman review field counts:")
for col in ["human_verified", "language_quality", "meaning_clear",
            "verified_hinglish", "review_notes"]:
    if col in hinglish.columns:
        print(f"\n{col}:")
        print(hinglish[col].fillna("MISSING").value_counts().to_string())

print("\nHinglish inspection completed.")

FINAL CMI COMPARISON
      Dataset  Records  Mean_CMI  Median_CMI
       MUTANT      100      6.26        4.77
        PHINC    13738     17.61       16.67
FactIndic 10%       23      9.77        8.79
FactIndic 30%       23     29.40       27.27
FactIndic 50%       23     49.97       47.29

Saved: True

HINGLISH DATASET
Rows: 100
Unique texts: 100

Missing values by column:
source_dataset                   0
source_config                   44
text                             0
script_type                      0
selection_method                 0
reply_to_text                   88
likely_hinglish                  0
human_verified                  95
language_quality                95
meaning_clear                    0
review_notes                     0
latin_word_count                 5
total_word_count                 5
latin_word_percentage            5
latin_words                      5
devanagari_words                 5
roman_hindi_words                5
total_language_words        

In [555]:
import pandas as pd
import os

path = base_path + "/06_real_hinglish/FactIndic_Hinglish_95_Human_Review.csv"

df = pd.read_csv(path)

# Records with no completed human verification
unreviewed = df[
    df["human_verified"].fillna("").astype(str).str.strip().ne("Yes")
].copy()

# Records not classified as verified Hinglish
not_verified = df[
    df["verified_hinglish"].astype(str).str.strip().str.lower().eq("no")
].copy()

# Preserve original data and add a transparent review status
df["final_review_status"] = "Pending human review"
df.loc[
    df["human_verified"].fillna("").astype(str).str.strip().eq("Yes"),
    "final_review_status"
] = "Human review recorded"

folder = base_path + "/06_real_hinglish"

df.to_csv(folder + "/FactIndic_Hinglish_100_Status_Tracked.csv", index=False)
unreviewed.to_csv(folder + "/FactIndic_Hinglish_Unreviewed_Queue.csv", index=False)
not_verified.to_csv(folder + "/FactIndic_Hinglish_Not_Verified_Queue.csv", index=False)

print("HINGLISH REVIEW QUEUES SAVED")
print("Total unique texts:", df["text"].nunique())
print("Human review recorded:", (df["final_review_status"] == "Human review recorded").sum())
print("Pending human review:", len(unreviewed))
print("Not verified as Hinglish:", len(not_verified))
print("Original file preserved:", os.path.exists(path))
print("Status-tracked file saved:", os.path.exists(folder + "/FactIndic_Hinglish_100_Status_Tracked.csv"))
print("Unreviewed queue saved:", os.path.exists(folder + "/FactIndic_Hinglish_Unreviewed_Queue.csv"))
print("Not-verified queue saved:", os.path.exists(folder + "/FactIndic_Hinglish_Not_Verified_Queue.csv"))

HINGLISH REVIEW QUEUES SAVED
Total unique texts: 100
Human review recorded: 5
Pending human review: 95
Not verified as Hinglish: 22
Original file preserved: True
Status-tracked file saved: True
Unreviewed queue saved: True
Not-verified queue saved: True


In [556]:
import pandas as pd
import os

report_path = base_path + "/FactIndic_PersonB_Final_Report.txt"
claims_path = base_path + "/07_claim_labels/FactIndic_50_Claim_Labeling_Sheet.csv"

print("=" * 60)
print("CURRENT FINAL REPORT")
print("=" * 60)

if os.path.exists(report_path):
    with open(report_path, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("Final report not found at expected path.")

print("\n" + "=" * 60)
print("CLAIM LABEL VALIDATION")
print("=" * 60)

if os.path.exists(claims_path):
    claims = pd.read_csv(claims_path)

    print("Total claims:", len(claims))
    print("Unique claim IDs:", claims["Claim_ID"].nunique())
    print("\nVerdict distribution:")
    print(claims["Verdict"].value_counts(dropna=False))
    print("\nMissing values:")
    print(claims.isna().sum())
else:
    print("Claim-label file not found at expected path.")

CURRENT FINAL REPORT
Final report not found at expected path.

CLAIM LABEL VALIDATION
Total claims: 50
Unique claim IDs: 50

Verdict distribution:
Verdict
NOT_ENOUGH_INFO    26
SUPPORTED          24
Name: count, dtype: int64

Missing values:
Claim_ID                 0
Article_ID               0
Claim                    0
Source_Article           0
Verdict                  0
Evidence_from_Article    0
Reviewer_Notes           0
dtype: int64


In [557]:
import os

matches = []

for root, dirs, files in os.walk(base_path):
    for filename in files:
        if "PersonB_Final_Report" in filename:
            matches.append(os.path.join(root, filename))

print("REPORT FILES FOUND:", len(matches))

for path in matches:
    print("\nPATH:", path)
    print("SIZE:", os.path.getsize(path), "bytes")

    with open(path, "r", encoding="utf-8") as f:
        print("\nREPORT CONTENT:\n")
        print(f.read())

REPORT FILES FOUND: 1

PATH: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/08_reports/FactIndic_PersonB_Final_Report.txt
SIZE: 8576 bytes

REPORT CONTENT:

FACTINDIC NLP PROJECT
PERSON B - FINAL WORK REPORT

Project:
Factuality-aware summarization for Hindi-English code-mixed text

Course:
CSET 346 - Natural Language Processing
Team 8 - Bennett University


1. DATA COLLECTION AND ENGLISH TRANSLATION

- Sampled 250 Hindi articles from the XL-Sum Hindi dataset.
- Completed English translation for all 250 articles using IndicTrans2.
- Completed 5,229 sentence-level translation tasks.
- Saved the reviewed English translation dataset:
  09_sentence_level_translation/FactIndic_English_250_Reviewed.csv
- Translation review records:
  225 manually corrected and 25 human reviewed without correction.

20-article translation quality audit and correction:
- The initial audit found serious quality problems in all 20 sampled
  English translations.
- Selected better sentence-level translation ca

In [558]:
import os

report_folder = base_path + "/08_reports"
os.makedirs(report_folder, exist_ok=True)

report = """
FACTINDIC NLP PROJECT
PERSON B - UPDATED FINAL WORK REPORT

Project:
Factuality-aware summarization for Hindi-English code-mixed text

Course:
CSET 346 - Natural Language Processing
Team 8 - Bennett University


1. DATA COLLECTION AND ENGLISH TRANSLATION

- Sampled 250 Hindi articles from XL-Sum Hindi.
- Completed English translation processing for all 250 articles using IndicTrans2.
- The corrected translation dataset contains 250 rows and 250 unique article IDs.
- No duplicate IDs, missing translations, or empty translations were found.
- Seven translation records received corrections; 243 records remained unchanged.
- The original reviewed translation file was preserved.

Main corrected dataset:
09_sentence_level_translation/FactIndic_English_250_Corrected.csv

Translation audit:
09_sentence_level_translation/FactIndic_Translation_Audit_230.csv

The audit contains 71 unique article records.
Seven correction records were identified:
- Five have corrected translations prepared and are pending final review.
- One records a factual mismatch correction.
- One records a numerical discrepancy correction and is pending final review.

The correction records should not be treated as independently certified
translations. The full 250-article dataset still requires further
quality review before being used as trusted reference data.


2. SENTENCE ALIGNMENT

- Created 1,079 manual Hindi-English sentence alignment records.
- The records represent 71 unique articles.
- Created a separate strict approved sentence-pair dataset.
- 408 sentence pairs are strictly approved.
- The remaining alignment records should not be described as strictly approved.

Files:
04_mixed_data/alignment/FactIndic_Manual_Sentence_Alignment.csv
04_mixed_data/alignment/FactIndic_Strict_Approved_Sentence_Pairs.csv


3. CONTROLLED CODE-MIXING DATASET

- Prepared a validated subset of 23 eligible articles.
- Used strictly approved Hindi-English sentence pairs.
- Generated 10%, 30%, and 50% target English-mixing versions.
- Preserved the sentence-level replacement approach.

Validated dataset:
04_mixed_data/FactIndic_Mixed_23_Validated.csv

The dataset contains 23 rows and has no missing values.

Final per-article CMI statistics:

Version                  Mean CMI       Median CMI
Original Hindi             0.00%           0.00%
FactIndic 10%               9.77%           8.79%
FactIndic 30%              29.40%          27.27%
FactIndic 50%              49.97%          47.29%

Limitation:
The current approach uses sentence-level replacement. It does not
produce natural word-level mixing within sentences. This is a
validated 23-article subset, not a 250-article mixed corpus.


4. MIXED-ARTICLE HUMAN REVIEW

- Created human-review records for the 23 eligible mixed articles.
- Preserved the two different review formats.
- Reviewed fluency, meaning preservation, translation coverage,
  and source-language issues.
- Missing review fields were not filled using assumed ratings.

Review file:
05_quality_checks/FactIndic_Mixed_Human_Review.csv

The different review formats should be interpreted according to
their available fields. Missing fields do not imply approval.


5. REAL HINGLISH DATA

- Collected 100 unique real Hinglish candidate texts.
- Source distribution recorded in the dataset:
  MUTANT: 56
  GupShup: 44

Classification:
- Marked as verified Hinglish: 78
- Not verified as Hinglish: 22
- Meaning clear: 100
- Natural English usage: 78 Yes, 22 No

Human-review status:
- Human verification recorded: 5
- Pending human review: 95

The dataset has 100 unique texts. However, only five records have
human_verified = Yes and a recorded language-quality rating.
The other 95 records must not be described as fully human-reviewed.

Files:
06_real_hinglish/FactIndic_Hinglish_95_Human_Review.csv
06_real_hinglish/FactIndic_Hinglish_100_Status_Tracked.csv
06_real_hinglish/FactIndic_Hinglish_Unreviewed_Queue.csv
06_real_hinglish/FactIndic_Hinglish_Not_Verified_Queue.csv

The original Hinglish dataset was preserved.


6. CLAIM LABELING

- Prepared 50 claim-label records.
- 50 unique claim IDs.
- No missing values in the seven checked columns.

Verdict distribution:
SUPPORTED: 24
NOT_ENOUGH_INFO: 26

No CONTRADICTED labels are currently recorded.

The sheet passes structural validation. Structural validation alone
does not establish that every verdict is factually correct; claim
evidence and reviewer notes should be checked against the source.

File:
07_claim_labels/FactIndic_50_Claim_Labeling_Sheet.csv


7. FINAL CMI COMPARISON

Dataset                 Records    Mean CMI    Median CMI
MUTANT                     100       6.26%        4.77%
PHINC                    13738      17.61%       16.67%
FactIndic 10%               23       9.77%        8.79%
FactIndic 30%               23      29.40%       27.27%
FactIndic 50%               23      49.97%       47.29%

Comparison file:
05_quality_checks/CMI_MUTANT_PHINC_FactIndic_Comparison_Final.csv

CMI is estimated using Latin-script detection. It is a script-based
estimate rather than a fully language-aware linguistic measure.
Dataset sizes and sampling methods differ, so comparisons should
be interpreted carefully.


8. ADDITIONAL QUALITY FILES

Per-article CMI:
05_quality_checks/FactIndic_Per_Article_CMI_23.csv

Mixing methodology:
08_reports/FactIndic_Mixing_Methodology.txt

Translation correction candidates from the earlier 20-article review:
09_sentence_level_translation/FactIndic_Translation_Corrections_Draft.csv

The earlier 20-article review and the later 250-article corrected
translation dataset are separate review outputs.


9. OVERALL STATUS AND LIMITATIONS

Person B has prepared:
- A 250-article Hindi-English translation dataset.
- A corrected translation dataset and translation audit.
- Manual sentence alignment records.
- A strictly approved sentence-pair subset.
- A validated 23-article controlled code-mixing dataset.
- Mixed-article human-review records.
- A 100-example real Hinglish candidate dataset and review queues.
- A 50-claim labeling sheet.
- Per-article CMI statistics and a final CMI comparison.

Important remaining limitations:
1. The full 250-article translation dataset is not fully quality-certified.
2. Some translation corrections still require final review.
3. The controlled mixed corpus contains 23 articles, not 250.
4. The mixing approach uses sentence-level replacement.
5. Only five Hinglish records currently have human verification
   and language-quality ratings recorded; 95 remain pending.
6. The claim-label sheet passes structural checks, but its verdicts
   should be checked against the source evidence.
7. CMI is a script-based estimate and has linguistic limitations.

OVERALL:
Substantial Person B data preparation and validation work has been
completed. The outputs should be reported with the above scope and
limitations. Pending human-review work must not be represented as
completed or independently certified.
"""

new_path = report_folder + "/FactIndic_PersonB_Final_Report_Updated.txt"

with open(new_path, "w", encoding="utf-8") as f:
    f.write(report.strip())

print("UPDATED REPORT SAVED:", os.path.exists(new_path))
print("REPORT PATH:", new_path)
print("REPORT SIZE:", os.path.getsize(new_path), "bytes")
print("\nOriginal report preserved:",
      os.path.exists(report_folder + "/FactIndic_PersonB_Final_Report.txt"))

UPDATED REPORT SAVED: True
REPORT PATH: /content/drive/MyDrive/FactIndic_PersonB_CleanRun/08_reports/FactIndic_PersonB_Final_Report_Updated.txt
REPORT SIZE: 6966 bytes

Original report preserved: True


In [559]:
import os
import pandas as pd

checks = {
    "Corrected English translations": (
        "09_sentence_level_translation/FactIndic_English_250_Corrected.csv", 250
    ),
    "Translation audit": (
        "09_sentence_level_translation/FactIndic_Translation_Audit_230.csv", 71
    ),
    "Manual sentence alignment": (
        "04_mixed_data/alignment/FactIndic_Manual_Sentence_Alignment.csv", 1079
    ),
    "Strict approved sentence pairs": (
        "04_mixed_data/alignment/FactIndic_Strict_Approved_Sentence_Pairs.csv", 408
    ),
    "Validated mixed dataset": (
        "04_mixed_data/FactIndic_Mixed_23_Validated.csv", 23
    ),
    "Hinglish status-tracked dataset": (
        "06_real_hinglish/FactIndic_Hinglish_100_Status_Tracked.csv", 100
    ),
    "Hinglish unreviewed queue": (
        "06_real_hinglish/FactIndic_Hinglish_Unreviewed_Queue.csv", 95
    ),
    "Hinglish not-verified queue": (
        "06_real_hinglish/FactIndic_Hinglish_Not_Verified_Queue.csv", 22
    ),
    "Claim labeling sheet": (
        "07_claim_labels/FactIndic_50_Claim_Labeling_Sheet.csv", 50
    ),
    "Per-article CMI": (
        "05_quality_checks/FactIndic_Per_Article_CMI_23.csv", 23
    ),
    "Final CMI comparison": (
        "05_quality_checks/CMI_MUTANT_PHINC_FactIndic_Comparison_Final.csv", 6
    ),
}

print("=" * 75)
print("FACTINDIC PERSON B — FINAL DELIVERABLE AUDIT")
print("=" * 75)

passed = 0
failed = 0

for name, (relative_path, expected_rows) in checks.items():
    path = os.path.join(base_path, relative_path)

    if not os.path.exists(path):
        print(f"MISSING  | {name} | {relative_path}")
        failed += 1
        continue

    try:
        data = pd.read_csv(path)
        actual_rows = len(data)
        if actual_rows == expected_rows:
            print(f"PASS     | {name} | {actual_rows} rows")
            passed += 1
        else:
            print(
                f"CHECK    | {name} | Expected {expected_rows}, found {actual_rows}"
            )
            failed += 1
    except Exception as e:
        print(f"ERROR    | {name} | {e}")
        failed += 1

reports = [
    "08_reports/FactIndic_PersonB_Final_Report_Updated.txt",
    "08_reports/FactIndic_PersonB_Final_Report.txt"
]

print("\nREPORT FILES")
for relative_path in reports:
    path = os.path.join(base_path, relative_path)
    print(("FOUND   " if os.path.exists(path) else "MISSING  ") + relative_path)

print("\n" + "=" * 75)
print("PASSED:", passed)
print("FAILED / NEEDS CHECK:", failed)
print("AUDIT FINISHED")
print("=" * 75)

FACTINDIC PERSON B — FINAL DELIVERABLE AUDIT
PASS     | Corrected English translations | 250 rows
PASS     | Translation audit | 71 rows
PASS     | Manual sentence alignment | 1079 rows
PASS     | Strict approved sentence pairs | 408 rows
PASS     | Validated mixed dataset | 23 rows
PASS     | Hinglish status-tracked dataset | 100 rows
PASS     | Hinglish unreviewed queue | 95 rows
PASS     | Hinglish not-verified queue | 22 rows
PASS     | Claim labeling sheet | 50 rows
PASS     | Per-article CMI | 23 rows
CHECK    | Final CMI comparison | Expected 6, found 5

REPORT FILES
FOUND   08_reports/FactIndic_PersonB_Final_Report_Updated.txt
FOUND   08_reports/FactIndic_PersonB_Final_Report.txt

PASSED: 10
FAILED / NEEDS CHECK: 1
AUDIT FINISHED


In [560]:
import pandas as pd
import os

quality_folder = base_path + "/05_quality_checks"

comparison_path = quality_folder + "/CMI_MUTANT_PHINC_FactIndic_Comparison_Final.csv"
per_article_path = quality_folder + "/FactIndic_Per_Article_CMI_23.csv"

comparison = pd.read_csv(comparison_path)
per_article = pd.read_csv(per_article_path)

# Keep external datasets; rebuild all four FactIndic rows from source data.
factindic_labels = [
    "FactIndic - Original Hindi",
    "FactIndic Original Hindi",
    "FactIndic 10%",
    "FactIndic 30%",
    "FactIndic 50%"
]

comparison = comparison[
    ~comparison["Dataset"].astype(str).isin(factindic_labels)
].copy()

levels = [
    ("FactIndic - Original Hindi", "Original_Hindi_CMI"),
    ("FactIndic 10%", "FactIndic_10_CMI"),
    ("FactIndic 30%", "FactIndic_30_CMI"),
    ("FactIndic 50%", "FactIndic_50_CMI")
]

new_rows = []

for label, column in levels:
    values = pd.to_numeric(per_article[column], errors="coerce").dropna()

    new_rows.append({
        "Dataset": label,
        "Records": len(values),
        "Mean_CMI": round(values.mean(), 2),
        "Median_CMI": round(values.median(), 2)
    })

comparison = pd.concat(
    [comparison, pd.DataFrame(new_rows)],
    ignore_index=True
)

comparison.to_csv(comparison_path, index=False)

print("UPDATED CMI COMPARISON")
print(comparison.to_string(index=False))
print("\nRows:", len(comparison))
print("Unique dataset labels:", comparison["Dataset"].nunique())
print("Saved:", os.path.exists(comparison_path))

UPDATED CMI COMPARISON
                   Dataset  Records  Mean_CMI  Median_CMI
                    MUTANT      100      6.26        4.77
                     PHINC    13738     17.61       16.67
FactIndic - Original Hindi       23      0.00        0.00
             FactIndic 10%       23      9.77        8.79
             FactIndic 30%       23     29.40       27.27
             FactIndic 50%       23     49.97       47.29

Rows: 6
Unique dataset labels: 6
Saved: True


In [561]:
import os
import pandas as pd

checks = {
    "Corrected English translations": ("09_sentence_level_translation/FactIndic_English_250_Corrected.csv", 250),
    "Translation audit": ("09_sentence_level_translation/FactIndic_Translation_Audit_230.csv", 71),
    "Manual sentence alignment": ("04_mixed_data/alignment/FactIndic_Manual_Sentence_Alignment.csv", 1079),
    "Strict approved sentence pairs": ("04_mixed_data/alignment/FactIndic_Strict_Approved_Sentence_Pairs.csv", 408),
    "Validated mixed dataset": ("04_mixed_data/FactIndic_Mixed_23_Validated.csv", 23),
    "Hinglish status-tracked dataset": ("06_real_hinglish/FactIndic_Hinglish_100_Status_Tracked.csv", 100),
    "Hinglish unreviewed queue": ("06_real_hinglish/FactIndic_Hinglish_Unreviewed_Queue.csv", 95),
    "Hinglish not-verified queue": ("06_real_hinglish/FactIndic_Hinglish_Not_Verified_Queue.csv", 22),
    "Claim labeling sheet": ("07_claim_labels/FactIndic_50_Claim_Labeling_Sheet.csv", 50),
    "Per-article CMI": ("05_quality_checks/FactIndic_Per_Article_CMI_23.csv", 23),
    "Final CMI comparison": ("05_quality_checks/CMI_MUTANT_PHINC_FactIndic_Comparison_Final.csv", 6)
}

passed = 0

print("=" * 65)
print("FACTINDIC PERSON B — FINAL AUDIT")
print("=" * 65)

for name, (relative_path, expected_rows) in checks.items():
    path = os.path.join(base_path, relative_path)

    if os.path.exists(path):
        try:
            df = pd.read_csv(path)
            if len(df) == expected_rows:
                print(f"PASS | {name} | {len(df)} rows")
                passed += 1
            else:
                print(f"CHECK | {name} | Expected {expected_rows}, found {len(df)}")
        except Exception as e:
            print(f"ERROR | {name} | {e}")
    else:
        print(f"MISSING | {name}")

report_path = base_path + "/08_reports/FactIndic_PersonB_Final_Report_Updated.txt"
print("\nUpdated report:", "FOUND" if os.path.exists(report_path) else "MISSING")

print("\nFINAL RESULT:", passed, "/ 11 deliverables passed")

if passed == 11 and os.path.exists(report_path):
    print("FILE AUDIT COMPLETE — PERSON B DELIVERABLES READY FOR HANDOVER")
else:
    print("Some file checks need attention.")

FACTINDIC PERSON B — FINAL AUDIT
PASS | Corrected English translations | 250 rows
PASS | Translation audit | 71 rows
PASS | Manual sentence alignment | 1079 rows
PASS | Strict approved sentence pairs | 408 rows
PASS | Validated mixed dataset | 23 rows
PASS | Hinglish status-tracked dataset | 100 rows
PASS | Hinglish unreviewed queue | 95 rows
PASS | Hinglish not-verified queue | 22 rows
PASS | Claim labeling sheet | 50 rows
PASS | Per-article CMI | 23 rows
PASS | Final CMI comparison | 6 rows

Updated report: FOUND

FINAL RESULT: 11 / 11 deliverables passed
FILE AUDIT COMPLETE — PERSON B DELIVERABLES READY FOR HANDOVER


In [562]:
import shutil
from google.colab import files

folder_path = "/content/drive/MyDrive/FactIndic_PersonB_CleanRun"
zip_path = "/content/FactIndic_PersonB_CleanRun.zip"

# Create ZIP of the complete folder
shutil.make_archive(
    "/content/FactIndic_PersonB_CleanRun",
    "zip",
    folder_path
)

# Download ZIP to your computer
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>